# Vector add with an unmasked fast path for wide loads

A GPU moves memory fastest with **128-bit** accesses: 4 `float32` or 8 `float16` values per instruction per thread. Triton emits them only when it can prove that each thread's elements are contiguous, aligned, and either all in bounds or all out. A mask like `offs < n` with an `n` that is not a multiple of 16 breaks that proof, and the compiler falls back to one scalar access per element **for the whole tensor**, not just the tail.

The fix is to launch the bulk of the work without a mask and handle the ragged end separately.

Write `vec_add(x, y)`.

## Inputs
- `x`, `y`: contiguous CUDA tensors of the same shape and dtype, either `float32` or `float16`; any number of elements $n$ (including 0), $n < 2^{31}$.

## Output
- A new tensor of the same shape and dtype, `x + y` (computed in that dtype, as `torch` does).

## Requirements
- One kernel with constexpr parameters `BLOCK` (use 4096) and `EVEN`, and a runtime `start` offset. Program `pid` covers `start + pid * BLOCK + [0, BLOCK)`.
- **Bulk launch**: `n // BLOCK` programs, `start = 0`, `EVEN=True`: loads and stores without a mask. Tell the compiler what you know with `tl.multiple_of` and `tl.max_contiguous` on the offsets.
- **Tail launch**: if `n % BLOCK != 0`, one program with `start = (n // BLOCK) * BLOCK`, `EVEN=False`, masked by `offs < n`.
- Skip any launch that would have zero programs.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from the same addition in PyTorch.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def vec_add(x, y):
    """x, y: float32 or float16 CUDA tensors of one shape. Return x + y: unmasked bulk launch + masked tail."""
    raise NotImplementedError


@triton.jit
def _add_kernel(x_ptr, y_ptr, out_ptr, start, n, BLOCK: tl.constexpr, EVEN: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0], "dtype": "float32"}, {"__nd__": [10.0, 20.0, 30.0], "dtype": "float32"}], "e": {"__nd__": [11.0, 22.0, 33.0], "dtype": "float32"}}, {"a": [{"__nd__": [0.5, -1.25], "dtype": "float16"}, {"__nd__": [0.25, 1.0], "dtype": "float16"}], "e": {"__nd__": [0.75, -0.25], "dtype": "float16"}}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [-0.37002477049827576, 0.9940786957740784, 0.4158574938774109, -0.6181637048721313, 0.6721407771110535, -1.4500495195388794, 0.59348464012146, -0.5618297457695007, 0.6314530968666077, 0.43909063935279846, -0.767248809337616, 0.5283687710762024, 0.3419276773929596, -0.6461430788040161, 2.001150608062744, 0.8019599318504333, -1.1820652484893799, -0.987981915473938, 0.31683167815208435, 0.31204426288604736, -0.7268560528755188, 1.215877890586853, 0.09701711684465408, -0.8626833558082581, -0.3745976984500885, -0.17879340052604675, -1.5533106327056885, -0.8322121500968933, -1.8099119663238525, -0.29355618357658386, 0.6126395463943481, -1.2858506441116333, 0.32208627462387085, -1.1826670169830322, -0.16362570226192474, -0.5161660313606262, -1.2121832370758057, -0.5490984916687012, -1.2254692316055298, 0.2935107350349426, 0.0461755245923996, 1.1999634504318237, -1.4632532596588135, 1.0927759408950806, -0.07875733822584152, 0.40536174178123474, -0.18846678733825684, 1.54696786403656, 1.3860177993774414, 1.0685069561004639, 0.2594567537307739, -2.337815284729004, 0.5563281178474426, -0.032759178429841995, -0.13396959006786346, 0.4345596730709076, 1.1934596300125122, 0.2672702670097351, -0.4664258658885956, -0.7777408361434937, 0.3700708746910095, 1.098569631576538, 0.5479772090911865, 0.45783841609954834, -0.45099467039108276, -1.2358990907669067, 0.9460468888282776, 0.6063642501831055, -1.8276169300079346, -0.7171863913536072, 1.0013699531555176, 1.0677965879440308, -0.6408771872520447, 2.2534637451171875, 1.4017287492752075, 0.9553112983703613, -0.1276785433292389, -1.3912400007247925, 0.8105084300041199, 1.778507947921753, 0.9139626026153564, -1.0778785943984985, 1.1387124061584473, -0.6649229526519775, -2.1980578899383545, 0.32018208503723145, -2.7102773189544678, 0.11305607110261917, 1.012002944946289, 1.038432240486145, 2.7208898067474365, 0.6697626113891602, -0.9833241105079651, 0.6155073642730713, 0.679188072681427, -0.5825828909873962, -1.7499442100524902, -0.373699814081192, -0.694375216960907, -0.3966110646724701, -0.18788385391235352, 0.7539269924163818, 0.5956143736839294, 1.4259463548660278, -0.5251567363739014, 0.9050846099853516, -1.7442854642868042, 0.025639895349740982, -1.5849429368972778, -0.8844839334487915, 1.0437040328979492, -0.9513410925865173, 1.4398325681686401, 1.7167609930038452, 0.5370325446128845, -0.29939261078834534, -0.9823975563049316, -0.8369831442832947, -0.07050628960132599, -0.9916666746139526, -0.09119582176208496, -0.7789248824119568, 0.6146163940429688, -1.8329609632492065, 0.4827103316783905, 0.9553006291389465, -1.003502368927002, 1.470671534538269, 1.5322585105895996, 0.6896921992301941, -1.4922293424606323, 0.2702944874763489, -1.610053300857544, -0.455674409866333, -1.2597442865371704, 0.156612828373909, -0.7461720108985901, 0.3372936248779297, 0.920749306678772, -0.7050688862800598, 0.540473997592926, -0.4106655716896057, -0.2218904346227646, -0.30466926097869873, -0.7505086660385132, -0.5697317123413086, 0.9094322323799133, 1.0412949323654175, -0.03751721233129501, 0.8326123952865601, 0.98857182264328, 0.6496959328651428, -2.108025312423706, -0.08377981185913086, 1.2613272666931152, 0.30640697479248047, -0.10202116519212723, -0.3555801212787628, 0.27722570300102234, -0.36956337094306946, 0.8430964946746826, -0.0951581820845604, -0.008857722394168377, 0.4793161451816559, 1.2191123962402344, -0.15452364087104797, 0.4462410807609558, -0.27865150570869446, -1.2048978805541992, -0.004319578874856234, 0.5400668382644653, 0.3550315499305725, -0.0912574827671051, -0.2596076428890228, -1.2287226915359497, -0.13175757229328156, 1.5802090167999268, -0.23148591816425323, -0.04017642140388489, 1.385007381439209, 0.11485780775547028, 0.2501009404659271, -1.1519781351089478, 0.8145022392272949, 1.3756110668182373, -0.060703206807374954, 1.046583890914917, 0.970615565776825, -2.732088804244995, 1.341553807258606, -1.4025453329086304, 0.4285116195678711, 0.4511874318122864, -0.23717664182186127, -0.36100152134895325, -0.38223540782928467, -0.4725777506828308, -0.4543269872665405, -0.725764274597168, -0.09754803776741028, 0.09922081232070923, -0.2052440047264099, 0.30248039960861206, -0.827082633972168, 0.17917095124721527, 0.8031712770462036, 1.5616438388824463, -1.2695618867874146, 0.7493720650672913, 0.2617315351963043, -1.9592711925506592, -0.6300631761550903, -0.9981675148010254, 0.5131746530532837, 1.469598412513733, 1.2522300481796265, 0.24066340923309326, 1.372960090637207, 0.38532477617263794, 0.2198943793773651, -1.515475869178772, -0.7237181067466736, -0.36980026960372925, -0.7703574895858765, -0.9865668416023254, -0.4763820767402649, 0.9921067357063293, 0.4505114257335663, 0.2644912004470825, -0.35951104760169983, -0.4528141915798187, -0.03178921341896057, -0.742453932762146, 0.1658802479505539, -0.14613555371761322, -0.3054947257041931, 0.15532685816287994, -0.034532900899648666, 0.261385977268219, -0.9491704702377319, -1.006509780883789, -0.8425278067588806, -0.9928058981895447, 1.218929648399353, -0.8859475255012512, 0.31928038597106934, 0.6503829956054688, 1.3706763982772827, -0.24502867460250854, -0.5919913649559021, 0.13388830423355103, 1.724153995513916, -0.30822259187698364, 0.8624104261398315, 0.9104285836219788, -1.1796659231185913, 1.0860037803649902, 0.6899465322494507, -1.5763481855392456, 0.917490541934967, 1.205213189125061, 0.05263439938426018, 0.2577322721481323, 0.6857852935791016, 0.3121580183506012, -0.8688257336616516, -1.1842598915100098, -0.7545560002326965, -0.7361478209495544, 0.5931713581085205, -1.2746644020080566, -0.859021008014679, -0.8847097158432007, 1.2063000202178955, 1.6486905813217163, 0.4217170476913452, 0.7748213410377502, 0.6308270692825317, -0.9474168419837952, -0.6752399802207947, 0.445160835981369, 0.09983386844396591, -2.0980687141418457, -0.9768304824829102, -0.07920221239328384, -1.4615710973739624, 0.5544266700744629, 0.5927782654762268, 1.2248685359954834, -1.1524605751037598, -0.27532100677490234, 0.7684370279312134, -0.9213734269142151, -0.006152240093797445, -0.7299480438232422, -0.07779458165168762, -1.288407325744629, -0.040865492075681686, 1.4751580953598022, 1.0870059728622437, -0.6765217781066895, -0.18792970478534698, -1.1194924116134644, 0.9320544004440308, 0.18645192682743073, 1.0110125541687012, -0.05888402462005615, -0.7134693264961243, 0.6863558292388916, -1.2954390048980713, -0.25916752219200134, 1.0500023365020752, -0.702538251876831, -2.818385124206543, -1.1093050241470337, 1.7618309259414673, -0.43679675459861755, -1.2878785133361816, -0.5825988054275513, 1.2878206968307495, -0.553811252117157, -1.1921441555023193, 1.705419898033142, 0.006309261079877615, 0.70633864402771, -1.587616205215454, 1.3449865579605103, 0.12403396517038345, 1.5147979259490967, 1.1352788209915161, -0.04077593609690666, 0.13184812664985657, -0.0013922646176069975, 2.881549119949341, 0.858081579208374, 1.3647233247756958, 0.7621323466300964, -1.8741531372070312, -1.2698136568069458, 1.5117641687393188, 0.21206854283809662, -0.625609815120697, 0.5869348049163818, 0.21350540220737457, 2.245389938354492, -0.5807538628578186, -0.5951892733573914, 0.31424078345298767, 1.1199697256088257, 0.10369087755680084, 0.7379708886146545, -0.07897618412971497, -0.07107262313365936, -0.16989241540431976, -0.37414541840553284, 0.9180055260658264, -0.4219602048397064, 1.082637071609497, -1.5745058059692383, -2.1605384349823, 2.7086265087127686, -1.1961641311645508, 0.9012508392333984, 0.17870433628559113, 0.9602720141410828, 2.3747072219848633, 0.22802966833114624, 0.823582112789154, -0.3324962258338928, 0.634375274181366, -1.3081237077713013, 1.451825499534607, 0.09531432390213013, -0.9579704999923706, -0.11963702738285065, -0.08487662672996521, 1.3413810729980469, -1.4871068000793457, -0.8456001877784729, 0.7157757878303528, -0.491290420293808, 0.005389414262026548, 0.542784571647644, -0.8848887085914612, 0.3690875768661499, -1.597000002861023, -1.403592824935913, 0.14860041439533234, -0.843212902545929, -0.8826764225959778, -0.9445027709007263, 0.05701061338186264, -0.2705797851085663, 0.022160988301038742, 0.5984236598014832, 1.8775349855422974, -0.32245299220085144, 1.2435808181762695, -0.2905149757862091, -0.9978784322738647, 0.37938135862350464, -0.3124238848686218, 1.671830177307129, -0.9279424548149109, 0.3706122040748596, 0.7983161807060242, 0.5696668028831482, 1.929155945777893, -0.9888536930084229, 0.11837776005268097, 0.21252653002738953, 1.4974684715270996, -1.3984991312026978, -1.5532046556472778, -0.2994076907634735, 0.4060773551464081, -0.4950699508190155, -0.5759199261665344, -0.25604647397994995, 0.45157283544540405, 0.955832302570343, -0.42438584566116333, -1.0523635149002075, 0.5466853380203247, -0.7163819670677185, 1.8714364767074585, -0.0823717713356018, 0.5946707725524902, -0.9094012379646301, 0.4685433804988861, -0.46519991755485535, -0.5323440432548523, 0.2525920867919922, -0.04324483126401901, 0.7744289636611938, -0.012856312096118927, 0.7840846180915833, 0.11233992874622345, -0.4982891380786896, 1.0593630075454712, -0.1464301347732544, -0.3600553274154663, -0.5807677507400513, 0.6065126061439514, -1.1876342296600342, -1.6785374879837036, -0.4107178747653961, -0.6152400374412537, -0.4935760796070099, 0.3591906428337097, 1.8159304857254028, 0.24918389320373535, -0.4219056963920593, -1.2487460374832153, -0.7809147238731384, -1.2148487567901611, -1.5595731735229492, -0.4473439157009125, 0.756018877029419, -0.0029400538187474012, 0.007446118164807558, -0.4245684742927551, -0.04905455559492111, -0.32548344135284424, -1.2983207702636719, 0.48520776629447937, -0.3963974416255951, -0.25734183192253113, -1.491150975227356, 1.2818059921264648, 1.2544548511505127, -1.026675820350647, -1.253718376159668, -1.062463402748108, 0.5855676531791687, 1.1880046129226685, -0.6332449913024902, -1.4320330619812012, -1.153239369392395, 0.352693647146225, -0.012097908183932304, -0.579498291015625, -1.0158276557922363, -1.4279327392578125, 0.04690782353281975, 0.31413665413856506, 0.7467302083969116, -0.8218593597412109, 0.6632279753684998, -0.8420589566230774, -0.25765129923820496, 1.9334187507629395, 1.429296612739563, -0.3611142039299011, 2.1138553619384766, -0.9331784248352051, -0.4540785849094391, -1.8898348808288574, -0.46474966406822205, 0.4373335838317871, 1.2544556856155396, 1.5136706829071045, 0.33921414613723755, 1.317102074623108, -1.6477237939834595, 0.8334899544715881, -0.41339346766471863, 0.0700872614979744, -0.3047561049461365, -0.4339798092842102, 0.47452035546302795, -0.5794656276702881, 0.11923125386238098, 0.15327759087085724, 0.24836121499538422, 1.4308069944381714, 1.7749210596084595, -1.5568469762802124, 1.0260133743286133, -0.7553117871284485, -0.04961054027080536, 1.1485037803649902, -1.856911063194275, 0.6318517923355103, -0.2824929654598236, -0.32213085889816284, -0.01938936486840248, -0.7722261548042297, -1.2063356637954712, -0.45558905601501465, -0.8508535623550415, 0.8378506898880005, -0.8349450826644897, 0.6295223832130432, -1.370800495147705, 0.458631694316864, 1.782618522644043, 1.4723759889602661, -0.05047663301229477, 1.9722524881362915, -3.066455364227295, -0.40463799238204956, -1.8216931819915771, -1.5402582883834839, -1.587424874305725, -1.339093804359436, -1.0236403942108154, -0.2604309618473053, 1.1194127798080444, 0.5454080700874329, -0.39838457107543945, -0.7962685823440552, -0.4540114402770996, 2.5210747718811035, -1.6492478847503662, 1.5089118480682373, 0.33552461862564087, -0.00990100484341383, 3.1058974266052246, 0.4250451326370239, -0.6949827075004578, 0.3485014736652374, 1.0672177076339722, 0.800363302230835, 0.3673955798149109, -0.3664575517177582, -1.059862494468689, 0.2591317892074585, 0.7530665993690491, -0.5711345672607422, -1.2357556819915771, -1.3748011589050293, -1.5361686944961548, -0.3860313594341278, -1.3911601305007935, 0.9727901220321655, -1.815459966659546, -0.9467715620994568, 0.8915889859199524, 0.326621949672699, 0.11219058185815811, -0.4520125687122345, -0.3025002181529999, 0.2821657657623291, 0.8265029191970825, 0.06258834153413773, 0.225181445479393, -1.3450390100479126, 0.6692513227462769, 1.9151978492736816, -0.5982159972190857, 0.33894237875938416, -0.7146478295326233, 0.5118528008460999, 0.450721800327301, -1.7213363647460938, -0.31226396560668945, -0.22344060242176056, -0.6020962595939636, 0.44811758399009705, -0.5578846335411072, -1.2994630336761475, 1.9609565734863281, 0.28511518239974976, 1.5925558805465698, 0.46191534399986267, 0.17982426285743713, 1.0005072355270386, 0.6525012254714966, -0.35726091265678406, 0.110874243080616, -0.47739359736442566, 0.8737949728965759, -1.0651744604110718, -0.09089382737874985, 0.3677404820919037, -0.02588578499853611, 0.2606240212917328, -1.3525058031082153, -2.0398426055908203, 1.517551064491272, -0.5125705003738403, -0.27703189849853516, 0.2112911343574524, 0.6113520264625549, 0.8440424799919128, 0.856823205947876, -0.5331064462661743, -0.14764997363090515, -0.40897470712661743, -1.9071872234344482, -0.2591422498226166, -0.23145602643489838, -1.2183020114898682, -1.8540977239608765, -0.27131006121635437, -0.5808886289596558, -0.3655787408351898, 1.3522632122039795, -0.23843464255332947, -0.2968854308128357, -1.4822697639465332, -0.31353744864463806, 0.8969948291778564, 0.5827953815460205, -0.7536485195159912, 0.8214680552482605, 2.0203323364257812, -1.3556275367736816, -0.2249748557806015, -1.8910421133041382, 1.5753173828125, -1.6836100816726685, 0.27553513646125793, 0.23823212087154388, 1.2461535930633545, -0.08030222356319427, -0.35796624422073364, -1.582040548324585, -1.0851876735687256, -0.17079339921474457, 0.9315333366394043, -1.8059293031692505, 0.8863958716392517, -0.523620069026947, -1.6331173181533813, 0.3705240488052368, 0.42696577310562134, 0.9433311223983765, 0.7416011095046997, -0.6307539939880371, 0.4519558250904083, -0.03555583953857422, 0.17284874618053436, -0.3643229603767395, -0.9537102580070496, -0.15019233524799347, 0.4239244759082794, -1.202461838722229, -0.023232094943523407, -0.8894680738449097, -1.303565502166748, -0.41616296768188477, 0.44271335005760193, 0.8380074501037598, -0.8411705493927002, -0.6475197672843933, -1.003109097480774, 0.36564475297927856, 0.04664383828639984, 0.43692252039909363, 1.3650543689727783, -0.30011242628097534, -0.3960305452346802, 1.0046862363815308, -0.12446010112762451, -0.5631402730941772, 0.8535195589065552, 0.8852158784866333, -1.4056562185287476, -1.0458046197891235, 0.16757388412952423, 1.0665005445480347, 0.7383015155792236, -0.8371489644050598, -0.0062357354909181595, -1.0444477796554565, 1.3683453798294067, 2.3256328105926514, -0.8344547152519226, -0.2109876126050949, -1.0988541841506958, 0.584011435508728, -0.3493495285511017, 0.28092092275619507, -0.5486758351325989, 0.2276405692100525, -1.875609278678894, 0.4650997221469879, -1.8267381191253662, 0.6686553359031677, 1.1913425922393799, 0.4099971055984497, 0.7684752345085144, 1.6192870140075684, 0.12359405308961868, -0.8496811985969543, -1.000797986984253, 0.7090489864349365, -2.1835379600524902, -0.34268391132354736, -0.7916789054870605, -0.17191168665885925, -0.2784576117992401, -1.1075977087020874, 1.074878454208374, 1.0002294778823853, -1.6928842067718506, 0.21232226490974426, 0.05543988570570946, -2.077547550201416, -0.2537110149860382, 0.316167950630188, -0.676699697971344, -0.04330625757575035, 0.7723732590675354, -0.3399220407009125, -0.297294020652771, -0.24967603385448456, -0.8832126259803772, -0.49229171872138977, 0.6172619462013245, -0.06519002467393875, -1.2994771003723145, -0.24857452511787415, 0.04408244043588638, 0.18137192726135254, 0.16372157633304596, -0.4573538303375244, -0.8300043344497681, 0.264740526676178, -0.22487619519233704, 0.728863537311554, -0.8888007998466492, -0.15443983674049377, 1.2538655996322632, -1.3402913808822632, -1.1350950002670288, 0.34252145886421204, 2.0540735721588135, -0.48444226384162903, -0.3781082332134247, 0.01909475401043892, -1.4395703077316284, 0.6020914912223816, 0.23047132790088654, -0.4148936867713928, -1.6351197957992554, 1.2514257431030273, 0.48473814129829407, 0.6605024337768555, 0.757206916809082, -0.330504447221756, 0.017068661749362946, -0.30588653683662415, 2.1611735820770264, -0.8148571848869324, 0.6857414245605469, -1.1625723838806152, 0.7047014236450195, -1.7034282684326172, -0.8582571148872375, 0.45203015208244324, 0.11463450640439987, 0.22979651391506195, -0.6053863167762756, -0.2971048653125763, 0.3254297077655792, 0.946919858455658, -0.087587870657444, -0.026903627440333366, -0.6571388244628906, -1.6727349758148193, -0.1625915765762329, -0.1670992225408554, 0.351527601480484, -0.9898771047592163, -0.8316073417663574, -1.3773781061172485, -1.2019182443618774, -0.5626524090766907, -1.1321607828140259, -1.6972323656082153, -1.0696066617965698, -1.806998610496521, -0.5866749882698059, -1.2394341230392456, -0.13991568982601166, -1.6521859169006348, 0.6908422708511353, -2.349998712539673, 2.0772433280944824, 1.0062059164047241, 0.20595486462116241, 0.7808818221092224, -0.2556535303592682, -1.440231442451477, 1.3551219701766968, 1.0806422233581543, 1.7632982730865479, 0.23711271584033966, -0.04403107613325119, 0.5154411792755127, 0.30505019426345825, 0.41257867217063904, 0.6455170512199402, 1.0845797061920166, -0.040077220648527145, 1.0444506406784058, -1.8200706243515015, -0.19804000854492188, 2.130429983139038, 0.27766525745391846, -1.0762999057769775, 0.08223864436149597, -1.9298124313354492, -0.8446506857872009, 2.45953369140625, 0.8453685641288757, 0.1510145664215088, -0.6865313649177551, 0.4796275794506073, 0.9754119515419006, -0.09950710088014603, 0.7526267170906067, -0.4129866659641266, 1.229522466659546, -0.735930323600769, -1.1272412538528442, -1.5057429075241089, -0.3275719881057739, 0.5884090662002563, 1.0447001457214355, 0.11126451939344406, -0.4448316991329193, -0.2321847826242447, -0.5029768943786621, 0.10641910880804062, -0.03987199068069458, 1.6612684726715088, 2.1203219890594482, -1.1674931049346924, 0.9558720588684082, 0.8048085570335388, -1.0127800703048706, 0.2247329205274582, -1.2458698749542236, -1.4420701265335083, -0.5423204302787781, -0.4365982115268707, -0.14037908613681793, -1.1707448959350586, 0.31952860951423645, 0.22913745045661926, 0.21170629560947418, -0.40007302165031433, 0.7864029407501221, -0.24081040918827057, 0.07488680630922318, -0.3701091706752777, -3.19364333152771, 1.4071531295776367, -1.0484683513641357, 0.3068920075893402, 0.5366355776786804, -0.6765702366828918, -0.6016837358474731, 0.4635780155658722, -0.9525953531265259, 2.941861152648926, 0.07873665541410446, -1.7433420419692993, 0.6124721169471741, 0.869945228099823, 1.1098648309707642, -0.12434892356395721, -1.6847823858261108, 0.335241436958313, 0.6428236961364746, -0.814464271068573, 0.11444620788097382, -0.8443291783332825, 1.520714521408081, 1.1332991123199463, 0.24749833345413208, -0.028351761400699615, -0.5994614958763123, 1.1066054105758667, -0.11685124039649963, 0.21352937817573547, 0.8879318833351135, -0.24717223644256592, -0.14556461572647095, -0.9774456024169922, 0.2431420534849167, -1.1394681930541992, 0.96442049741745, -0.7301173806190491, 1.1645147800445557, 0.7053524851799011, -0.3686788082122803, 0.06794945150613785, -0.597821831703186, 0.31386178731918335, -0.26122018694877625, 0.8265324831008911, 2.0540966987609863, -2.4254164695739746, 1.2035934925079346, 1.0537739992141724, -1.2033981084823608, -0.5339320302009583, 0.12324921786785126, 0.3013564348220825, 0.5709235668182373, 0.8028149008750916, -0.2786867022514343, -0.690058171749115, -0.7053896188735962, -0.43115854263305664, 0.5818141102790833, 0.49987390637397766, -0.10886642336845398, -0.6078745126724243, 0.4056541621685028, -0.050577979534864426, 1.4858506917953491, -1.535563588142395, 0.34288132190704346, 1.2656162977218628, -1.1264406442642212, -0.3552672266960144, -1.1851330995559692, -2.599428176879883, -1.4548639059066772, 1.1153090000152588, 0.4649079144001007, -0.8315039873123169, -0.6571749448776245, 0.13690079748630524, -0.812153697013855, -0.4599311947822571, -1.200410008430481, -0.4034860134124756, -0.45276743173599243, 1.4553371667861938, 0.656671941280365, -1.0303773880004883, 0.059334345161914825, -0.47073736786842346, 0.5456727147102356, 0.5240597724914551, 0.2802054286003113, 0.10934922844171524, 0.2630147635936737, -0.9701905250549316, -0.18546704947948456, 0.03709257021546364, 0.6181604862213135, 0.3938177525997162, -0.3330325186252594, -0.9823684692382812, 1.8283882141113281, 2.0242795944213867, -0.5248327255249023, 0.6997760534286499, 2.6427695751190186, 0.06503424048423767, -0.2832109332084656, 0.6102144122123718, -0.24635349214076996, -0.5293608903884888, -0.3093455731868744, -1.5841089487075806, -0.8656037449836731, 0.13978737592697144, 1.2267379760742188, 0.9738013744354248, -1.268140435218811, 1.1046115159988403, 0.12904559075832367, -0.07456380128860474, -0.16401417553424835, -0.2605554461479187, 2.029287338256836, 0.5095624923706055, -1.4117016792297363, -1.7114661931991577, 0.07863876223564148, -0.033014677464962006, -0.16732335090637207, -0.19739021360874176, -0.43513813614845276, -0.5831651091575623, 0.8798234462738037, -0.403205543756485, -0.03821495547890663, 1.0174373388290405, 0.6773992776870728, 0.47689563035964966, -2.3401174545288086, -0.4890573024749756, 0.30670756101608276, 0.1841573417186737, 0.2753613591194153, -0.573279082775116, 1.352726697921753, 0.38963690400123596, -0.5627957582473755, -1.6686707735061646, -1.2408884763717651, -0.6774348020553589, -1.836242437362671, 0.5139681696891785, 0.4227144420146942, 0.23131227493286133, -0.39247456192970276, -0.6596566438674927, 0.00782735925167799, 0.13372305035591125, -2.643620252609253, -0.280386358499527, -0.32868510484695435, 0.15962940454483032, -1.0840446949005127, -1.7975844144821167, 0.9494380950927734, 0.2801201641559601, -1.3093053102493286, -1.3587223291397095, -0.25470730662345886, -0.8262704610824585, -0.5221042633056641, 0.9360315799713135, -0.03790935128927231, 0.6458467841148376, 0.9882152080535889, -2.4822564125061035, 1.6028531789779663, 2.254564046859741, 0.3664443790912628, -0.6515531539916992, -0.23997758328914642, 3.9509711265563965, -0.43136218190193176, 1.7653234004974365, -0.06345335394144058, -1.1273307800292969, -1.4112083911895752, -0.4370839595794678, -0.3216232657432556, 0.6170474886894226, 0.2970365583896637, 1.981066107749939, -0.5957537293434143, 1.340065598487854, 0.3410550653934479, -0.07800475507974625, -0.7985352873802185, 0.7270923256874084, 0.41317617893218994, 1.2010154724121094, -1.5325552225112915, -0.0012925734044983983, 1.4322243928909302, -0.18972553312778473, 0.708868682384491, -0.2729288339614868, 0.39861854910850525, 0.4402579665184021, -0.4290193021297455, 0.56235271692276, -0.41475218534469604, 0.3430057466030121, -0.06604230403900146, -0.44113847613334656, -0.5894370675086975, 0.8486465215682983, 0.2926304340362549, -0.21683959662914276, 1.228333830833435, -0.6954331994056702, -1.378510594367981, -0.5405221581459045, 1.8295530080795288, 0.08962421864271164, -0.04999268800020218, -0.6995533108711243, 0.5103040933609009, -0.22940590977668762, -0.4217264652252197, 0.8212597966194153, 1.545805811882019, -0.35039222240448, 0.5661038756370544, -0.32033729553222656, -0.5576211810112, -0.1394357681274414, 0.03178724646568298, 0.15854673087596893, -0.49525272846221924, -1.2982572317123413, -1.8505939245224, 1.7882494926452637, 0.28321021795272827, 0.45030492544174194, -0.3040294349193573, 1.074451208114624, -0.648552417755127, 1.4736191034317017, -1.2098878622055054, 0.0018457400146871805, 0.9486825466156006, -1.0329197645187378, -0.7043359875679016, -0.4032685458660126, -0.5707643628120422, -1.7132742404937744, 0.03581327944993973, 1.7783167362213135, -0.2625714838504791, -0.17171265184879303, -0.5591984391212463, -0.09962638467550278, -1.1356253623962402, -0.8360509872436523, 0.3350059390068054, 1.6396955251693726, 2.595327377319336, 0.8692001700401306, 1.5382534265518188, 0.5087119936943054, -1.4104642868041992, -0.6616299152374268, 0.6502439975738525, 0.90982985496521, 0.3929559886455536, 1.727852463722229, 0.13115094602108002, -0.4526435434818268, 0.8306047320365906, 0.11116195470094681, 0.5281176567077637, -0.8128882050514221, -0.29472577571868896, 1.4832481145858765, -0.9402187466621399, -0.8149113059043884, 0.04264112189412117, -0.2893992066383362, -0.05593901127576828, 0.765509307384491, -0.12822094559669495, -0.6215865612030029, 0.8309468030929565, -0.20781998336315155, -0.8085062503814697, 0.0376242958009243, 0.2773571312427521, -0.4127451777458191, 1.6410765647888184, 0.03131379559636116, 1.953125, -1.1652625799179077, 1.526678442955017, 0.5953394770622253, 0.15780280530452728, 1.406180500984192, -0.0015351686161011457, -0.1210746318101883, -0.967720091342926, 1.3218395709991455, -1.3045514822006226, 2.1355342864990234, -0.4987214207649231, -0.7248693704605103, 1.244062900543213, -0.15301305055618286, 0.8379386067390442, -0.13798820972442627, -0.4905034601688385, 0.6376500129699707, -1.7450586557388306, -0.49230191111564636, -0.39129355549812317, -0.6954994797706604, -0.6199360489845276, 1.470968246459961, 0.461270809173584, -1.8789187669754028, 1.8622761964797974, 0.9272100925445557, 1.6911944150924683, -0.34826987981796265, -0.11566929519176483, 0.11727781593799591, 0.6637551784515381, -0.6878739595413208, -0.4190768003463745, -0.8415112495422363, 0.8961008787155151, -0.161015585064888, 0.24807170033454895, -0.0026487386785447598, 0.40767717361450195, -0.49879634380340576, 0.7663748860359192, -1.3374511003494263, -0.3153398931026459, -0.8219308853149414, -0.6433379650115967, 1.2748160362243652, 1.7383557558059692, -0.17307031154632568, -1.5437301397323608, -0.19107572734355927, 2.0159287452697754, 0.5715168714523315, -0.2875292897224426, -0.5310763716697693, 1.7402088642120361, 0.07760874927043915, 1.0219221115112305, -0.015228583477437496, 0.05311606451869011, -0.3391449451446533, -0.7208724021911621, -0.3375003933906555, 0.09778033196926117, -0.90086829662323, 0.41085225343704224, 1.4543957710266113, 0.9083681106567383, -0.14434824883937836, -1.7032747268676758, 1.2227544784545898, 0.37786105275154114, 0.6260605454444885, 1.0665709972381592, 1.947249412536621, 1.9974457025527954, 0.6645638346672058, 0.34525179862976074, -0.34640949964523315, 0.6071821451187134, 0.256681352853775, 0.35650748014450073, -0.46439552307128906, 0.277308851480484, -1.0097383260726929, 1.363396167755127, 0.802351176738739, 0.6272370219230652, 1.337219476699829, -0.8125999569892883, 1.4495306015014648, -0.11625590920448303, 0.15466144680976868, -2.1458234786987305, 0.30430179834365845, 0.06735654175281525, 0.7421422004699707, -0.6523484587669373, -0.7788650393486023, -0.34788617491722107, -0.1548037827014923, 1.2063113451004028, 1.4419975280761719, -0.08307746052742004, -0.030089085921645164, 0.6487734317779541, 1.0191912651062012, 0.6382495164871216, 0.3110232651233673, 0.38998085260391235, 0.46901634335517883, -0.6895774602890015, 0.9284005165100098, -1.0388927459716797, -0.8568933010101318, -1.6799839735031128, -0.7513700127601624, -0.8764746189117432, -0.21467462182044983, 0.1899297684431076, -1.747208833694458, -0.24458320438861847, -1.8488043546676636, -1.2947129011154175, -1.9304758310317993, 0.5752564072608948, 0.3613535761833191, -0.8340801000595093, -1.1253349781036377, -1.1479096412658691, -0.7169365286827087, -1.1895978450775146, -1.9524136781692505, 0.5055578947067261, -1.0902234315872192, -0.4621880054473877, -0.5391988158226013, -0.6250125765800476, -0.3122614026069641, -0.09976273030042648, 0.19416961073875427, 0.11181817948818207, 1.112438678741455, -1.041117548942566, 0.36882898211479187, -1.3099207878112793, -2.0048115253448486, 0.13201239705085754, 0.828911304473877, -0.09510800242424011, 1.9376758337020874, 1.0048447847366333, 1.8948076963424683, -0.10632237792015076, -0.28744491934776306, 1.6249598264694214, -2.596536159515381, -0.6968351602554321, -0.19917963445186615, 0.4978157877922058, -0.6102128624916077, 0.753957986831665, -1.0921006202697754, 1.70490562915802, 1.2771399021148682, 0.36705294251441956, 1.7006078958511353, 1.49813711643219, -1.5210188627243042, -0.24984775483608246, 0.829558789730072, 1.5503416061401367, 1.1924598217010498, 0.6860349178314209, -2.439281940460205, 0.16210120916366577, -1.9313043355941772, -0.06827986240386963, -0.8623721599578857, -0.08126550912857056, 0.5547025203704834, 0.9956935048103333, 0.7228302359580994, 0.9592087864875793, 0.2188514620065689, -0.4052138924598694, 0.9950938820838928, 0.1900010108947754, 0.6351837515830994, -0.7929010987281799, 1.2846397161483765, 1.113700032234192, -0.9233162999153137, -0.6732909679412842, 0.13091765344142914, -1.5282350778579712, 1.4905955791473389, -0.13805024325847626, 0.25456562638282776, -0.5845567584037781, 0.7630957365036011, 1.5141359567642212, -1.4020054340362549, 0.8426723480224609, 1.997751235961914, -0.894512951374054, 0.20956766605377197, 0.6544189453125, 0.31187838315963745, 0.7757341861724854, -1.451670527458191, 1.014411211013794, -0.009758254513144493, -2.0693466663360596, 0.9244192838668823, 1.019004464149475, -0.6602838635444641, -0.38000112771987915, -0.9709100723266602, -1.7990150451660156, -2.204303741455078, 0.18298619985580444, 0.0006852980586700141, -1.2609862089157104, -1.2775853872299194, 0.3244907259941101, -1.6420897245407104, -0.0793285071849823, 0.8218814134597778, -1.1896448135375977, -2.0027809143066406, 0.39215847849845886, 0.025216471403837204, 2.018646001815796, -1.148283839225769, 1.5445303916931152, -1.0013962984085083, 0.7317000031471252, -0.06923522800207138, 0.8099099397659302, 0.6692157983779907, -0.943559467792511, -0.6000661849975586, 1.0278050899505615, 0.6241849660873413, -0.7829285860061646, -0.25472867488861084, -0.6435146927833557, -0.3709251582622528, 0.002567497780546546, -0.5479094982147217, -0.5670750737190247, 1.7987418174743652, -0.45397716760635376, 1.2402245998382568, -0.7918469309806824, -1.141623616218567, 0.8615084290504456, 0.48483818769454956, 0.33239036798477173, 1.3212801218032837, -0.6525998711585999, 1.275202751159668, 1.9559110403060913, 0.09807806462049484, 2.6825835704803467, 0.6132454872131348, -0.5521465539932251, -0.1874728500843048, -0.15393473207950592, -0.9395949840545654, 2.1517937183380127, 0.4381008446216583, 1.093631625175476, 0.3569668233394623, 0.9689972996711731, 0.05107801407575607, 0.4333093464374542, -0.0886787474155426, 0.3621360659599304, -0.752861499786377, -1.0570757389068604, -1.3964332342147827, 0.05086720734834671, -0.6031054258346558, -0.6119956970214844, 2.034029960632324, 0.5248963832855225, 1.250319480895996, 0.9137195944786072, -1.0298041105270386, 0.5379263162612915, 1.5062639713287354, -0.09902610629796982, -0.91289222240448, -2.052788019180298, -0.9941977262496948, -0.10716904699802399, 0.20353302359580994, -0.6773553490638733, -1.1183322668075562, 0.687221884727478, 0.1792314052581787, 0.8709326386451721, 0.7295287847518921, 0.48821353912353516, -0.8439426422119141, -1.58029043674469, -0.1385585218667984, -0.8744980096817017, -1.2615201473236084, -0.5602251887321472, 0.17344124615192413, -1.0059183835983276, 0.856671929359436, 0.4524340331554413, -0.08987391740083694, -0.22049562633037567, -0.19422662258148193, 0.5921849012374878, -0.9295344352722168, 0.11271984130144119, 0.3980284631252289, -0.06121860817074776, 2.028439998626709, -0.87846440076828, 0.5869480967521667, 1.1569266319274902, -0.8633518218994141, 0.9889084100723267, 0.7728556394577026, 1.0324524641036987, 0.23809590935707092, 0.4860342741012573, 0.28240329027175903, 0.17614080011844635, -0.9309841394424438, 0.7625128030776978, 0.46981266140937805, -1.3342584371566772, -0.6807732582092285, -1.0201598405838013, 2.908567190170288, -0.542717456817627, -0.4768133759498596, -0.874125599861145, -1.0676547288894653, 0.3733707070350647, 0.4870305061340332, -0.20788027346134186, -1.8004815578460693, 1.0493707656860352, 0.09240899980068207, 0.5448607802391052, -0.2394292801618576, -0.8130818009376526, 0.4870506227016449, -0.7042068839073181, 1.0231517553329468, -2.0465140342712402, -0.43518751859664917, -1.6326382160186768, 0.43385815620422363, 0.22318248450756073, -0.697763204574585, -0.35752052068710327, -0.6595275402069092, -0.16371643543243408, 0.7846471667289734, -0.5182704925537109, -0.9955973625183105, 0.06772515177726746, -1.3473224639892578, 0.6409634351730347, 0.37027910351753235, -1.3150479793548584, 0.40720334649086, -0.5374805927276611, 0.9588841199874878, 0.08868392556905746, -0.6710745096206665, 0.5775133371353149, -0.60235196352005, -0.3881642520427704, -0.22421149909496307, -0.45650577545166016, 0.7422119975090027, -1.1077369451522827, -0.34874656796455383, 0.5979123115539551, -0.5314940214157104, 1.1719642877578735, 0.3505646288394928, -0.2998397946357727, 0.470654159784317, -1.7023448944091797, 0.4827599823474884, 0.7686410546302795, 0.279902845621109, -0.6685563921928406, -1.0083030462265015, 0.7258192896842957, 0.5771602392196655, 1.1198524236679077, -0.6640253663063049, -0.6283363103866577, -0.645256519317627, -0.5278674960136414, 1.4124062061309814, 0.022271042689681053, 1.014796257019043, -0.44083651900291443, -0.891934871673584, -2.084775447845459, -0.3857164978981018, 1.2429749965667725, -1.5287580490112305, -0.1459643840789795, -0.3060664236545563, -1.3872826099395752, -0.3750208914279938, 0.3496096730232239, 0.7355219721794128, -0.06359218806028366, 0.5692884922027588, 1.0610685348510742, -0.797909140586853, -2.3974764347076416, 0.19623732566833496, 0.2820148468017578, -0.743983268737793, -1.6225581169128418, -0.7965810894966125, 0.7117608785629272, 0.2646838128566742, 0.47448065876960754, -0.32161882519721985, -0.258861780166626, -1.0385206937789917, 0.9402629733085632, -0.7608152031898499, 0.7903726696968079, -0.7758784294128418, 0.6316808462142944, 1.138008713722229, -0.11950498074293137, -0.9422040581703186, -0.6880509257316589, -0.8773080706596375, 0.9185611605644226, 0.9404597878456116, -0.4012596607208252, -0.14608101546764374, -0.6196106672286987, 1.9378376007080078, -0.37766745686531067, 0.04662987217307091, -0.6594419479370117, -1.1187679767608643, -0.5892592668533325, 0.10810555517673492, -0.02216256968677044, -0.8770895004272461, -0.3161516785621643, 1.019915223121643, 0.25637519359588623, -0.999969482421875, -0.12553910911083221, -0.6467868089675903, 0.9090606570243835, -1.3161262273788452, 0.2214878797531128, -0.5877109169960022, 1.3516892194747925, -0.10283229500055313, 0.9931758642196655, -1.2229243516921997, -0.5550995469093323, 2.170358180999756, -0.6532875895500183, 0.37887367606163025, 0.5963027477264404, -0.9183425903320312, -1.0213587284088135, -0.2435825765132904, 0.9746975302696228, 0.08371440321207047, -1.3772717714309692, -0.05483635887503624, 0.6351538300514221, -0.18559427559375763, -1.4767166376113892, -1.5223164558410645, 1.2423129081726074, 1.7853056192398071, 0.17195585370063782, -1.3820604085922241, -1.1583458185195923, -0.07154256850481033, 0.12269779294729233, 0.40180879831314087, 0.21317218244075775, 0.16763420403003693, 0.28471359610557556, 0.9585670232772827, 2.2603859901428223, 0.6097503900527954, 0.62046217918396, -1.3906574249267578, 0.3430718779563904, -0.6276012659072876, 0.3444017469882965, -1.6653612852096558, -0.40822410583496094, 1.5662410259246826, 1.5107797384262085, -0.7551409602165222, 0.5565645098686218, -0.39051923155784607, -0.0018955768318846822, -0.9507365226745605, 0.4684785306453705, 1.8489505052566528, 1.6183675527572632, 0.8891599178314209, -0.8846181035041809, -1.7198786735534668, 0.025193223729729652, 0.4578874707221985, 1.2704927921295166, 0.11294770985841751, 0.8983560800552368, -0.6372199654579163, -0.1458917260169983, -1.1172841787338257, 1.6637887954711914, -0.11254842579364777, -0.7338742613792419, -1.5341873168945312, 0.7064884901046753, 0.9410720467567444, 0.8197737336158752, -1.2053321599960327, 0.5567451119422913, -0.014227909967303276, -0.5803924798965454, -0.5061301589012146, -0.35915058851242065, 0.08963043242692947, 0.2073846161365509, -0.6474610567092896, 1.1080254316329956, 0.31363794207572937, 0.1346520334482193, -0.10805271565914154, 0.5103679895401001, 1.6035277843475342, 0.8469045758247375, -1.3817119598388672, -0.7833348512649536, 1.086767315864563, 0.5984270572662354, 0.6265089511871338, 1.1278364658355713, 0.532832145690918, -0.9675989747047424, 0.06502252072095871, 0.7142367362976074, 0.6558058261871338, 0.8588013648986816, -1.6537531614303589, 0.861217737197876, -0.08758503943681717, -0.5293375253677368, 0.503174901008606, 1.9713736772537231, 0.38101065158843994, -0.9374404549598694, 0.6978694200515747, 0.8584558963775635, -1.252530574798584, 0.023765750229358673, -0.19819806516170502, -0.2184978425502777, -1.1507340669631958, -0.4653468430042267, 0.18624728918075562, 0.42082053422927856, 1.1011241674423218, -0.019681379199028015, 1.2181037664413452, 0.8131119012832642, -0.2200806587934494, 0.5915319919586182, 1.5306415557861328, -1.32815682888031, -0.30110689997673035, 0.15437425673007965, 0.8892670273780823, -0.9920571446418762, 0.0927143543958664, -0.3969193696975708, -0.8338884711265564, 1.0943084955215454, 0.5240619778633118, -0.9691603183746338, -0.033151306211948395, -0.6847587823867798, -1.4466663599014282, 0.937657356262207, 0.7347232699394226, 0.4448081851005554, 0.015491406433284283, 0.4754807949066162, -0.4188070297241211, -1.8032004833221436, -0.2712016701698303, 1.3846333026885986, -0.6086984276771545, 0.3380380868911743, -1.0779190063476562, 1.5649422407150269, -0.9475283622741699, -0.1779688000679016, -0.8524035215377808, -0.41142991185188293, 1.0813828706741333, -1.0824222564697266, -1.3000584840774536, -2.0849125385284424, -1.5444214344024658, 1.4439172744750977, -1.1931782960891724, 1.041063904762268, -0.6255987882614136, -0.6962447166442871, -0.3079482614994049, -0.362233430147171, -2.1540346145629883, -0.6770392656326294, -0.0181625634431839, -1.3890995979309082, 0.9856764674186707, 0.3351012170314789, 0.06038724258542061, 0.9025658965110779, 0.5745871663093567, -0.7394267320632935, 0.5917652249336243, -1.607115626335144, 0.5786601901054382, -0.5121310353279114, 0.6987766623497009, 0.7140076160430908, 0.26056522130966187, 0.5269522070884705, 1.6266443729400635, -0.7156317234039307, 1.3565963506698608, -1.1033416986465454, -0.13490396738052368, -0.15909510850906372, 0.4746173024177551, 0.06400355696678162, 1.356313705444336, 0.9848989248275757, 0.14924634993076324, -0.3606564700603485, 0.11927338689565659, -3.840118169784546, -0.09239038079977036, 1.0257230997085571, 0.23823201656341553, -0.05328529328107834, -0.9375591278076172, -0.25597307085990906, 0.34776031970977783, -1.668397068977356, 0.49239763617515564, -0.5166773796081543, -0.9631255865097046, -0.3027959167957306, -0.15065965056419373, -0.5037771463394165, 0.5816928148269653, -0.47013533115386963, -0.2738841474056244, -0.9312564134597778, 0.6176760196685791, -0.5642210841178894, -0.5845827460289001, 1.0389456748962402, 0.6204122304916382, 0.5790901184082031, -0.3569972813129425, 0.42987972497940063, 0.5029810667037964, -1.4232773780822754, -0.3791416585445404, 0.0708300843834877, -1.3417763710021973, -0.037052351981401443, 0.3498067855834961, -1.2997374534606934, 1.9829670190811157, -0.5786806344985962, 0.6502841114997864, -0.1726016402244568, -0.29393914341926575, -0.24497278034687042, 0.7404813766479492, -0.6067358255386353, -0.5074144601821899, -0.43024906516075134, -1.0069581270217896, 0.09325697273015976, -0.7484866976737976, 1.4387264251708984, -0.7562899589538574, -0.264286071062088, 1.7351981401443481, 0.8183702230453491, 0.22294305264949799, -0.2542358636856079, -1.3432446718215942, -0.6691660284996033, 0.763117253780365, 1.9343892335891724, -0.18510545790195465, -0.09322711080312729, 1.3473846912384033, -0.49233365058898926, -0.9729325771331787, -1.1126571893692017, -0.8299270272254944, 1.3466356992721558, 0.3080204129219055, -1.02554190158844, 1.3233674764633179, -0.2982977628707886, 0.6324175596237183, -1.0895142555236816, 0.03925419598817825, 0.7353463172912598, 0.155897855758667, -0.11913099884986877, 2.0285844802856445, -0.32507139444351196, 0.4431256353855133, -1.4988484382629395, 0.4779092073440552, 0.7579452991485596, 0.3958795368671417, -0.27484795451164246, 1.5758768320083618, -1.2471920251846313, -0.6685776114463806, -0.932192862033844, -1.1860018968582153, 0.05740625038743019, 1.6868332624435425, 0.26464593410491943, -1.1028501987457275, 1.0174840688705444, 0.005483641754835844, 0.6042699813842773, 1.7197257280349731, 1.1942838430404663, 1.8044888973236084, 0.8228572010993958, -0.7221888899803162, -1.4523574113845825, -0.43736013770103455, 0.7446222305297852, 0.7214995622634888, 0.2685971260070801, 0.25496047735214233, -1.6433032751083374, 0.42566633224487305, 1.1252857446670532, 0.18337514996528625, -0.25827887654304504, -0.9069989919662476, -0.39513877034187317, -0.43486374616622925, 0.3891676366329193, -0.38864684104919434, 0.705194354057312, 1.542171835899353, 0.009476262144744396, 0.3932502567768097, -0.6188388466835022, -0.3831406235694885, -0.6483938694000244, -0.5124534964561462, -1.1273242235183716, 0.5486655831336975, 0.6067125201225281, 1.614101767539978, 1.4905421733856201, -1.3433343172073364, 1.1906002759933472, -0.3639588952064514, 0.6654562950134277, 0.0982663705945015, 0.8303722143173218, 1.0647022724151611, -0.525087833404541, 0.40307867527008057, 1.7037768363952637, -1.28864324092865, -0.3765436112880707, -1.5899171829223633, 0.6059579849243164, -1.0221868753433228, 0.08472739160060883, -0.7610993981361389, -0.32725653052330017, -0.1660539209842682, 0.5770446062088013, -0.380435049533844, -0.14159971475601196, 0.5361953377723694, -0.05341067537665367, -0.5250608325004578, -1.1463786363601685, 0.48822686076164246, 0.658364474773407, 0.6225166320800781, 0.6850059032440186, 1.4772906303405762, 0.4105289876461029, 0.4473978281021118, -1.5195213556289673, -1.4351928234100342, 0.24190062284469604, 1.4632679224014282, 2.292832612991333, 0.5033837556838989, -0.8125561475753784, -0.6098196506500244, -2.1940767765045166, 0.47618111968040466, 0.460071325302124, 0.556057333946228, 0.8864219784736633, 0.7891111969947815, 0.7046117782592773, -0.6043669581413269, 1.0372546911239624, 2.121704578399658, 0.08422666788101196, -1.1538913249969482, -0.42104440927505493, -1.5704514980316162, 1.4351075887680054, -0.4928571879863739, 0.18076230585575104, -1.3251094818115234, 0.0006374355289153755, 0.03696868196129799, -0.7564906477928162, 1.559544563293457, 0.9522161483764648, -0.3995540738105774, 1.876395344734192, -0.6322288513183594, -0.28908127546310425, -1.5976868867874146, 0.8933622241020203, -0.061834149062633514, 2.5881831645965576, 2.172882318496704, 2.103179454803467, 0.2856244444847107, 1.332185983657837, -2.272127389907837, 1.3920649290084839, 0.16233626008033752, 0.06749919801950455, -1.214207410812378, 1.0467606782913208, -2.0824356079101562, 0.8469407558441162, -1.3712471723556519, 0.1061546802520752, 0.9505477547645569, 1.5946593284606934, -0.719141960144043, 0.021366585046052933, 0.7164779901504517, 1.8408758640289307, -0.5469170212745667, -0.7245118021965027, 1.377354383468628, 0.5052433609962463, 0.8021430969238281, 0.5256132483482361, 2.726649761199951, -3.1289970874786377, 0.5427256226539612, 0.5166015625, 1.1662980318069458, 1.1375633478164673, 0.989377498626709, 1.0124565362930298, -0.9983385801315308, 0.9252436757087708, 0.39398831129074097, -0.1763698160648346, -1.2599657773971558, 0.5047674179077148, -0.18571767210960388, 0.15803155303001404, -1.4439125061035156, -0.38377103209495544, 0.15055184066295624, 0.40987569093704224, -0.13487690687179565, -0.1078517884016037, -1.5022363662719727, 0.03705502301454544, 0.4052148461341858, -0.9886150360107422, -1.2232868671417236, 0.027210937812924385, 0.745822548866272, 0.050817180424928665, -0.20390774309635162, -0.5634897351264954, 0.1442928910255432, -0.03599266707897186, 0.9066106677055359, 0.06293690204620361, -0.27379199862480164, -1.0050472021102905, -0.8353534936904907, 0.9124593734741211, 0.864013135433197, 0.06148887053132057, -0.2601853311061859, 1.128756046295166, -0.19585314393043518, -0.009992954321205616, 0.32456979155540466, -1.4328705072402954, -0.979764461517334, -0.22414185106754303, -0.878116250038147, -1.025154948234558, -0.24482882022857666, -0.6207939982414246, 0.5009700059890747, -0.1971653252840042, -0.2451147735118866, 1.5543313026428223, -0.09158296883106232, 0.08370263129472733, -0.5877564549446106, -0.04923728108406067, 2.705855369567871, -0.5430543422698975, -1.1547378301620483, 0.20096363127231598, -0.49069151282310486, -0.16637705266475677, 0.8930012583732605, -0.5983850955963135, 0.5851027369499207, 0.9722487330436707, 1.3891732692718506, -0.03666669875383377, -1.7535854578018188, 2.1013152599334717, 0.6434105038642883, 0.281690776348114, -0.6912518739700317, 0.8472306728363037, 0.3466324508190155, 0.4580908417701721, 1.1956623792648315, 2.746701955795288, -3.0776100158691406, -0.06609141826629639, -0.558922290802002, -1.6919190883636475, -1.2181164026260376, -0.343885600566864, 1.1064903736114502, 2.1596860885620117, -1.214944839477539, 0.8374055624008179, 0.8068237900733948, 0.0035378201864659786, -0.35368847846984863, -3.284823417663574, 1.73934805393219, 0.8422852158546448, -0.5682564973831177, -0.7430092096328735, -0.33325082063674927, 1.9801313877105713, 0.25239297747612, -0.4461415708065033, -1.109411358833313, 0.26526758074760437, -0.1590227335691452, 0.749626100063324, -0.6871950030326843, -0.2209680825471878, -1.4014179706573486, -0.4235248565673828, 1.2294604778289795, -0.12145878374576569, -0.6367092728614807, -1.1062161922454834, 0.7996250987052917, -0.3192839026451111, -0.4251173734664917, 0.42447590827941895, -0.5148874521255493, -0.727313220500946, 0.3757990300655365, 1.2313449382781982, 0.16775915026664734, 0.2223888486623764, -0.5298209190368652, -2.501646041870117, -0.8159387111663818, -1.8295332193374634, -0.3021371364593506, 0.7386085987091064, -0.5284624695777893, -0.47028467059135437, 1.527199149131775, -0.7311538457870483, 0.4432128071784973, 0.35581880807876587, 0.2574414014816284, 0.9597388505935669, 1.3273991346359253, 0.9606337547302246, -1.7072182893753052, -0.44734227657318115, -0.6906805634498596, -2.1567752361297607, 0.15222781896591187, -1.3351774215698242, 1.7512843608856201, -0.03922862559556961, 2.31557297706604, 0.35476768016815186, 1.0771077871322632, -0.8087059855461121, -0.6465980410575867, -1.8195703029632568, 0.6394142508506775, 2.4802486896514893, -0.9946841597557068, 0.24428682029247284, -0.6232333183288574, -0.7760971784591675, -0.2097187489271164, 0.5331480503082275, 1.00835382938385, -0.5847859382629395, -1.2459908723831177, -1.0131527185440063, 0.1673823893070221, -0.5458157062530518, -0.0638137236237526, -0.04330521076917648, -0.34139710664749146, -1.5593886375427246, 0.3260141909122467, 0.394257515668869, 0.6683106422424316, -1.1777647733688354, 1.2252205610275269, 0.5712171196937561, 1.0384562015533447, 2.5095508098602295, -0.5927481055259705, 0.8364208936691284, 1.4039185047149658, -0.862566351890564, 1.6689503192901611, 0.0033605126664042473, 1.4517608880996704, 0.32194697856903076, 1.200268268585205, 0.4788697063922882, -2.5580556392669678, -0.03316245228052139, 0.5111076831817627, 0.4580860137939453, 0.003916746936738491, 0.49236246943473816, -0.524024248123169, -2.376021146774292, 0.059437576681375504, 3.1709072589874268, 0.01915728487074375, 0.24167004227638245, -0.059744760394096375, 0.46782898902893066, 0.7284167408943176, 0.23988255858421326, 1.5912678241729736, 0.7579034566879272, 1.9268620014190674, 0.06245223060250282, -0.6899280548095703, 0.8675478100776672, -0.5601138472557068, 0.6014249324798584, 0.33659932017326355, 2.0423479080200195, 0.1718490719795227, -1.2425695657730103, 1.4324895143508911, 0.6433032155036926, -0.07917429506778717, 0.2543452978134155, -0.5755313038825989, -0.41434338688850403, -0.04373179003596306, -0.4803866446018219, 0.4096827507019043, -1.253833532333374, 0.5982750058174133, 0.44276171922683716, -0.4866126775741577, -1.579224705696106, 2.180058240890503, -0.9719629287719727, 1.7007167339324951, -0.8953599333763123, -0.9307396411895752, -0.7881205081939697, -0.8720196485519409, -0.44034600257873535, -1.7367520332336426, -1.2963937520980835, -1.3328694105148315, 0.6376330852508545, 0.22674360871315002, -0.03999577462673187, 0.2178509682416916, 1.2159702777862549, -0.3213779926300049, 0.04285986348986626, -0.6086937785148621, -0.5384300351142883, 0.634036123752594, -0.7247180938720703, -0.227850541472435, 0.15539854764938354, 1.6349784135818481, -0.9464967250823975, -0.9287574291229248, 0.3952232003211975, 0.547143280506134, -0.6961762309074402, 0.4299328625202179, -0.5838462114334106, -0.07393718510866165, 1.3077951669692993, 0.7203467488288879, 1.5153100490570068, -0.9920318126678467, -0.09128915518522263, 1.6449204683303833, 0.052486468106508255, 0.839979887008667, 0.13024289906024933, -1.3140145540237427, 3.3487799167633057, 0.7404935956001282, -2.211880922317505, -0.08305630832910538, -0.1676928848028183, -0.11962536722421646, -0.31709611415863037, 0.4002057611942291, -0.2406339794397354, 1.3963992595672607, -0.59511798620224, 0.26773539185523987, 0.43292728066444397, 0.3150477409362793, -1.610068440437317, 0.13440778851509094, -0.6769891977310181, -0.33533748984336853, -0.3030954599380493, -0.09274253994226456, 1.0124471187591553, 0.2142290621995926, 1.9709607362747192, 0.3085062503814697, -1.0292779207229614, -0.07178949564695358, 0.9592600464820862, 1.2807170152664185, 0.7314931750297546, 0.7251192331314087, 0.16362904012203217, -1.0295286178588867, -0.5471923351287842, -1.2452656030654907, -0.8721568584442139, -0.49412938952445984, -0.023650603368878365, -0.2541564702987671, 0.023539463058114052, 0.9846181273460388, -0.6363525986671448, -1.1701099872589111, -0.47355666756629944, 1.8293466567993164, -0.08422911167144775, 0.3717136085033417, -0.5309004187583923, -1.1013505458831787, 0.09586518257856369, -0.010286652483046055, -0.8739737868309021, 0.6112537980079651, 2.2394983768463135, -1.0533264875411987, -1.3422867059707642, 1.233277440071106, -1.8653104305267334, -0.9085481762886047, 0.02434934861958027, -0.057839564979076385, 1.0236788988113403, -1.2091020345687866, 0.3659249246120453, 0.31416550278663635, 1.330356240272522, -0.5377033352851868, -0.38995325565338135, 0.389382004737854, -0.8575549721717834, 0.44525307416915894, 0.6510456204414368, -0.5773133039474487, 0.4828425943851471, 1.2271616458892822, -0.7537340521812439, -0.21483570337295532, 0.7215324640274048, -0.68480384349823, 0.8464143872261047, 0.40966546535491943, -1.0420515537261963, -1.1131863594055176, 0.9976800084114075, 0.9714761972427368, -0.8654260039329529, -0.13124030828475952, -0.37156838178634644, -1.1604896783828735, -1.748128890991211, 0.1277940571308136, -0.23981107771396637, 0.45861271023750305, 1.67551851272583, 2.577167272567749, -0.388791024684906, -2.198451280593872, 2.189298629760742, 0.8211702108383179, 0.38959869742393494, -0.04349196329712868, 0.7551069855690002, -0.40975695848464966, -0.9414253830909729, -0.42761701345443726, 1.1369277238845825, 0.17109574377536774, 0.6202328205108643, 0.8789703249931335, -0.8147618174552917, -0.8464343547821045, -0.5830168128013611, -0.40572530031204224, 0.056337758898735046, -0.18074475228786469, -0.2910058796405792, 0.3731715679168701, 0.9474204778671265, 1.0980384349822998, -0.7656787633895874, 0.4214150309562683, -0.6229631304740906, -0.7344812750816345, -1.9643367528915405, 0.5474950671195984, 0.16400793194770813, 0.2492598593235016, 1.0193177461624146, -1.6156716346740723, -1.1745774745941162, -0.5012528896331787, 0.7048685550689697, 0.3266240358352661, -1.3920549154281616, 0.3313513696193695, 0.8909288644790649, -1.3116705417633057, -0.6529427170753479, -0.8415026068687439, 1.2698042392730713, -1.8174388408660889, 1.1252942085266113, 1.2376333475112915, 1.0030337572097778, -0.31527483463287354, 1.101767659187317, 1.9026132822036743, -0.11414764821529388, -0.04575344920158386, -1.6471351385116577, 1.2088215351104736, 0.014006301760673523, 1.0113494396209717, -1.6971276998519897, -0.9924619793891907, -0.05042329803109169, 1.1464567184448242, 0.285563588142395, -0.16759681701660156, 0.4730135202407837, -0.19336442649364471, 1.105175256729126, 0.9656596779823303, 0.49815645813941956, -0.10930034518241882, -1.2124141454696655, 1.3878108263015747, -0.11280977725982666, -0.50478196144104, 0.9219558238983154, -0.8119663596153259, 1.7465370893478394, 1.2784130573272705, -1.9814395904541016, -1.693645715713501, 0.44353869557380676, 0.6977736949920654, 0.4250057637691498, 0.21539044380187988, -0.864526093006134, 0.3271162807941437, 0.03461930900812149, 0.9429371356964111, 1.5160632133483887, 1.0394338369369507, -0.13673238456249237, 0.6072729229927063, 0.3133423328399658, -1.6828906536102295, 2.106175422668457, 0.3112501800060272, -1.572357416152954, 0.2734575867652893, -0.3361576497554779, 2.0103487968444824, -0.27651241421699524, 2.1329855918884277, -0.19665783643722534, 0.3626689314842224, -0.1176825538277626, -0.9162378311157227, -0.7280044555664062, 0.3828565180301666, -0.6450358033180237, 0.6409615874290466, 0.10517597943544388, 1.715549111366272, 0.3248338997364044, 0.6514405608177185, 0.7836556434631348, 0.4697781503200531, 1.4606074094772339, -0.754231870174408, -2.018357753753662, -0.3871135711669922, -0.0012911863159388304, 0.01996931992471218, 0.4611770510673523, -1.3012890815734863, 0.7034462094306946, -0.06089498847723007, -0.9088732600212097, -2.253326892852783, -1.4694689512252808, 1.0360230207443237, 0.4905857443809509, -0.08895666152238846, 1.362525463104248, 0.1968105435371399, 0.812601625919342, -0.7042294144630432, 0.42542606592178345, -1.26591157913208, 0.34762200713157654, 0.454165518283844, 0.3246424198150635, 2.4242770671844482, 0.08714382350444794, -0.4846652150154114, -0.39403533935546875, -1.332184910774231, -0.9515220522880554, -0.054251790046691895, -0.7688317894935608, 0.5996791124343872, 0.843854546546936, 1.7830950021743774, 1.9301280975341797, -0.39436307549476624, -0.062070395797491074, -0.48374900221824646, -1.4557312726974487, -1.4262412786483765, 0.5265631079673767, -0.4711420238018036, 1.766687273979187, -0.0019860181491822004, -1.7579213380813599, 0.10275278985500336, -0.5077154040336609, -1.3480676412582397, 0.10764738917350769, 1.2382581233978271, -1.0418061017990112, -1.0063111782073975, 0.11465205997228622, -0.4675672650337219, 0.17386852204799652, 0.28062763810157776, 0.8920935392379761, -2.0131218433380127, -0.7485659718513489, 1.855744481086731, 1.3792314529418945, 0.11715969443321228, 0.0013996920315548778, -1.168680191040039, -0.49854397773742676, 1.3844949007034302, -0.9667700529098511, -1.0166183710098267, 0.2939266562461853, 0.3981926739215851, 0.7379245758056641, 1.4781609773635864, -0.1530904471874237, -0.21267704665660858, 0.4062085449695587, -0.08997372537851334, -1.4751036167144775, 1.1556396484375, -1.4156644344329834, -0.46549752354621887, -0.7898099422454834, 0.4332308769226074, -0.8256365060806274, -1.6250801086425781, 0.29635995626449585, -0.4489031434059143, -0.5100151300430298, -1.655503749847412, -0.7896320223808289, -0.6388246417045593, -0.26318904757499695, 0.7738189101219177, -1.3884071111679077, -0.07988857477903366, 0.36761096119880676, 1.492416501045227, -0.35703763365745544, -0.35641011595726013, -0.016046492382884026, -1.4533737897872925, -1.7406299114227295, -0.2580397129058838, 0.8299703598022461, 0.9292898178100586, -1.6363612413406372, 0.21134011447429657, 0.10961231589317322, -1.1679579019546509, -0.4570649564266205, -0.8426527380943298, 1.112359881401062, -1.3622909784317017, 0.7910228371620178, -0.856574535369873, -1.513049840927124, 2.1974523067474365, -1.088263750076294, -0.8570980429649353, 0.19779014587402344, 0.6137807965278625, -0.057179756462574005, 0.06473053246736526, 1.0277998447418213, -0.75216144323349, -0.5686194896697998, 1.5464600324630737, 0.4677639305591583, -2.1331589221954346, -2.1170828342437744, -0.4538179039955139, 1.6158503293991089, 1.050032377243042, 0.4789673089981079, 1.9257194995880127, 0.5493472218513489, 1.3180228471755981, 0.6492264270782471, 0.7455216646194458, -0.06638642400503159, -1.8560599088668823, -0.5228816270828247, -0.143870010972023, 1.566176414489746, -0.9217521548271179, -0.22117163240909576, 0.3736000955104828, 0.7290252447128296, 2.228318214416504, 0.0326395109295845, 0.7038261294364929, -0.9164006114006042, 0.18744874000549316, 0.2332914024591446, -0.3149438202381134, 0.5660074949264526, -1.9791977405548096, -0.2681950628757477, 0.05817941576242447, -0.4263133406639099, 0.7196208834648132, 1.0819538831710815, -0.9790789484977722, 0.1268991380929947, 0.13763849437236786, -0.5821155309677124, -2.024463653564453, 1.8635573387145996, 1.0508906841278076, -0.09164201468229294, 0.258289098739624, -0.3195550739765167, 1.2537645101547241, 0.8633291125297546, 1.9801709651947021, 1.3713696002960205, 0.5250017642974854, -1.1513283252716064, 0.543403685092926, -1.3747419118881226, 0.2508902847766876, 0.055227454751729965, -0.13093699514865875, 1.4697840213775635, -0.5400386452674866, 0.5759084224700928, 0.12611688673496246, -1.5101335048675537, -0.7206041216850281, 0.3427754044532776, 1.1792746782302856, -1.5665018558502197, -0.4653051495552063, -1.0142381191253662, -0.5836818814277649, -0.8287569284439087, 0.5964609384536743, -1.342119574546814, 0.11095746606588364, -0.9575664401054382, 0.07304573059082031, 0.22287587821483612, -0.10107085108757019, -0.2387087643146515, -0.02248588763177395, -0.1910647451877594, -0.1310044527053833, 0.7071911692619324, 0.11795927584171295, -1.6674902439117432, -0.59231036901474, -1.2996035814285278, 0.6429564356803894, 0.12168797105550766, 1.3562179803848267, 0.4358351230621338, -2.094874620437622, -1.0110588073730469, 0.8667137622833252, 1.2471942901611328, -2.015655755996704, 0.027401544153690338, 0.5484121441841125, -0.4381197690963745, 0.09448900818824768, -0.09283164143562317, -0.039461541920900345, -0.3961484134197235, 1.633739709854126, 1.3839213848114014, -0.11238652467727661, 0.9772335886955261, -0.005524155218154192, -0.08151127398014069, -0.6574838161468506, 0.2639234960079193, -0.5898639559745789, 1.2610803842544556, 0.901630699634552, -0.4127807915210724, 0.06054258346557617, 0.7805311679840088, 0.21029239892959595, -0.8762632012367249, 0.7006729245185852, -0.2101602703332901, 1.378692388534546, 1.4889276027679443, -0.15806107223033905, 0.9789847135543823, -1.3983336687088013, 0.07685946673154831, -0.42251917719841003, 1.0442043542861938, 0.2830711305141449, -0.6206540465354919, -0.7739043831825256, -1.301113247871399, 0.11411391943693161, 1.1099740266799927, -0.12446720153093338, 0.2680698037147522, -1.6024895906448364, 0.7845877408981323, -1.0608080625534058, 0.21282197535037994, -0.944739580154419, -0.45670145750045776, -0.2838304042816162, 1.2472645044326782, -0.27308931946754456, 0.3131130039691925, 0.12972335517406464, -0.7685896158218384, -0.5600143074989319, -2.1364047527313232, 0.8482169508934021, 1.509333848953247, -0.38561034202575684, -0.8279798030853271, -0.5437827110290527, -1.4487321376800537, -1.0184499025344849, 0.9150010943412781, 0.6304782032966614, 0.9426549673080444, -0.9293034076690674, -0.8976215720176697, -0.957036554813385, 0.31461232900619507, -1.0726854801177979, -0.7063839435577393, -0.8302190899848938, -0.22355477511882782, -2.14351749420166, -1.2359837293624878, 0.17535358667373657, -1.3260565996170044, -0.5635433197021484, -0.18956580758094788, -0.6337470412254333, 0.7709236741065979, -0.38509202003479004, -1.7208579778671265, -0.7060566544532776, -0.2690390646457672, -0.659196674823761, 1.593902587890625, 0.42719048261642456, 0.36109477281570435, 1.3345359563827515, 0.7318828701972961, -0.04106073081493378, 0.5724599361419678, 1.7644217014312744, -0.20520776510238647, 0.33110150694847107, 0.9884322881698608, -1.0564281940460205, -1.0982745885849, -0.10547500103712082, -0.9527378082275391, 0.24573639035224915, -0.47568443417549133, 0.10245446860790253, 0.39091402292251587, -0.8216972947120667, 0.517778754234314, 1.5628387928009033, 0.13024699687957764, -0.8732986450195312, -0.2501215934753418, 0.12438669055700302, 0.11397067457437515, 0.03327781707048416, -0.8462220430374146, 1.558793067932129, 1.8679298162460327, 0.1812012493610382, -0.17900335788726807, -0.13012667000293732, 2.0520946979522705, 0.42796996235847473, 0.7006616592407227, 0.41325268149375916, 1.188030481338501, -0.24833810329437256, -0.8295889496803284, -1.245854139328003, -1.923233151435852, 0.5780714750289917, -1.0962841510772705, -0.33068299293518066, -0.7936904430389404, 2.695213556289673, -2.5728485584259033, -0.06611812859773636, 0.7401018142700195, -1.835685133934021, -0.3736378252506256, 0.5528711676597595, -0.6857844591140747, 1.8244075775146484, 1.915626883506775, 0.9414762854576111, -0.12421482056379318, -0.21758633852005005, 0.13858506083488464, -0.5806622505187988, 1.3539444208145142, -0.024707335978746414, -2.028372049331665, -1.1491215229034424, 0.5749908685684204, 0.5915751457214355, 0.06500244140625, -1.2535861730575562, -1.608856439590454, -0.6194620132446289, 0.8969296216964722, 0.7964832186698914, 0.781476616859436, -0.5039706826210022, -0.21296222507953644, 0.6221503019332886, -0.20290319621562958, -0.4786943793296814, -1.7527153491973877, 0.8711192607879639, 0.4923776686191559, -1.2973599433898926, 0.468066930770874, 0.391670286655426, -0.6226356625556946, 0.28645116090774536, 1.2393773794174194, 0.5410178303718567, -0.8143171072006226, -0.21978995203971863, -1.0478554964065552, -1.0443849563598633, 0.49675872921943665, -0.7591475248336792, -1.0360784530639648, -0.5450328588485718, -1.7823048830032349, 0.30928415060043335, 0.047980181872844696, -1.1834611892700195, 0.9095091223716736, 0.6486111283302307, 0.3290531039237976, 1.332383155822754, 1.0905903577804565, 1.4692882299423218, -0.15442131459712982, 0.4531969726085663, -0.05601091682910919, 1.121319055557251, -0.5696161389350891, 1.4939309358596802, 2.2723042964935303, -1.3127048015594482, 0.9693288207054138, -0.09818527847528458, 2.63161563873291, 0.1437050700187683, 0.7287757396697998, 0.6957316398620605, -1.3539515733718872, 0.16807840764522552, -0.8545151352882385, -1.1358400583267212, -0.03572482615709305, -0.135597825050354, 1.5340508222579956, -0.06439278274774551, -0.1982753574848175, 0.5688802003860474, -0.33907875418663025, 0.2374332994222641, 0.8207929730415344, -0.8618350028991699, 2.3265881538391113, -0.6675724983215332, 0.8409347534179688, -1.1390076875686646, -0.44764330983161926, -1.76784086227417, -1.6045620441436768, 0.5697195529937744, -2.446942090988159, -0.026178814470767975, -2.7547194957733154, 0.7810810208320618, -0.26559194922447205, -0.3900549113750458, 1.050603985786438, -0.33783289790153503, 0.14728504419326782, -0.9052941203117371, 0.2611892819404602, 0.820398211479187, -1.237617015838623, 1.6438404321670532, 1.1103976964950562, -0.9424195289611816, 0.37277141213417053, 3.1322638988494873, -1.8826345205307007, -1.27582848072052, -0.8873333930969238, 1.9290730953216553, 0.24347685277462006, -1.6690185070037842, 1.079390525817871, 0.28237149119377136, 0.5135727524757385, 0.42600005865097046, 0.15007366240024567, -0.86008620262146, -0.34293338656425476, 0.3408719599246979, 0.8308454751968384, 0.7417149543762207, -1.2811411619186401, -1.4968836307525635, 1.0581754446029663, 0.05199804902076721, 0.4416123926639557, 0.3376269042491913, 0.03952696546912193, -0.8945540189743042, -1.8148716688156128, 0.17837898433208466, 0.171553835272789, -0.723685085773468, -0.335680216550827, -0.18027348816394806, -0.16504055261611938, -0.17050383985042572, -0.4894901514053345, 0.18581870198249817, -1.4047738313674927, 1.2689660787582397, -0.822257936000824, 0.43368029594421387, -2.701533555984497, -0.09842869639396667, -1.115883708000183, -2.0785248279571533, 0.4012383222579956, 0.5378098487854004, -0.5079395174980164, 0.49860796332359314, 0.8318160772323608, -0.4291699528694153, -2.3142971992492676, -1.4199721813201904, -0.5524061322212219, 0.1864171028137207, -1.7198735475540161, -0.6923931837081909, -1.0969725847244263, -0.35743290185928345, -0.7508853673934937, -0.10752629488706589, 1.6423568725585938, 0.014622471295297146, -2.4613585472106934, -0.31358036398887634, -0.2778312563896179, -0.5651208758354187, -0.8504908084869385, -1.4504942893981934, -0.6815049648284912, 1.1514278650283813, 0.6234209537506104, 0.5074559450149536, -0.7502495050430298, -1.2020679712295532, 0.442228227853775, 0.13792261481285095, 1.682981014251709, -0.15490607917308807, -0.5921719670295715, 0.3305087983608246, -1.7298579216003418, 0.10298902541399002, 0.3117264211177826, 1.08897066116333, -2.1362156867980957, 1.2796876430511475, -0.06076199188828468, -0.1264076977968216, 0.6168314814567566, -1.0690181255340576, 1.551662564277649, 0.025127651169896126, -0.7834817171096802, -0.1468709409236908, -0.18875928223133087, 0.9354739785194397, 0.5035378336906433, -0.17420634627342224, 1.5277220010757446, -1.0193785429000854, -1.5341143608093262, -1.3459968566894531, 0.5762794017791748, -2.1155362129211426, 1.0191158056259155, -0.31100571155548096, 1.2588366270065308, -0.9875861406326294, 0.24874797463417053, -0.3251171410083771, 0.6826589703559875, -0.9775185585021973, 0.5048763751983643, -2.2515575885772705, 0.38828471302986145, 0.030792903155088425, 1.0140434503555298, -1.1795307397842407, 0.05197734758257866, 0.7408972978591919, -0.4539072513580322, 2.3024652004241943, 0.6049495339393616, -0.9547119736671448, -1.1884535551071167, 1.2081432342529297, 1.129323124885559, 0.9613981246948242, -2.1287808418273926, 0.3544042706489563, -0.16596008837223053, 2.570204019546509, -0.8157434463500977, 0.24267756938934326, -1.2668315172195435, -1.2863355875015259, -0.5503976345062256, -0.2071041464805603, -0.0925491526722908, -1.073743224143982, -0.5084715485572815, 1.337814211845398, 0.5951240062713623, 2.2804219722747803, -0.7433763146400452, -0.6173824667930603, 3.712425470352173, 0.6395114064216614, 0.3105490207672119, 0.14572025835514069, -0.3172807991504669, -1.5712108612060547, -0.9707565307617188, -0.326898992061615, -0.9410703182220459, -1.1788934469223022, 0.8377689123153687, -0.499756783246994, 1.0604056119918823, -0.06915026903152466, -0.9649123549461365, 1.4848501682281494, 1.0778969526290894, 0.8970059752464294, -0.8727900981903076, -0.03944474831223488, 0.023270685225725174, -0.5033420324325562, -0.3331018388271332, 0.2692791223526001, 1.8513232469558716, -1.3093516826629639, -0.7880115509033203, 0.8318045139312744, -0.8904312252998352, 1.4780642986297607, 1.4503616094589233, -1.9665347337722778, -0.09103740751743317, 0.9743472337722778, 0.6118937134742737, 0.3826032876968384, -0.42628246545791626, -0.24699150025844574, 1.034375548362732, -0.10082852840423584, 0.9172440767288208, -0.7897235751152039, 0.5918107032775879, 1.1273428201675415, -0.746163010597229, -0.9033775925636292, 0.4861471354961395, 1.9538891315460205, 0.9217281341552734, 1.2721977233886719, 0.8889191746711731, -0.7645446062088013, 0.5235554575920105, -0.7107534408569336, 0.33005592226982117, 0.7849342823028564, -1.4361430406570435, 0.11258699744939804, -0.2342257797718048, -0.48181289434432983, 0.5412865281105042, -0.8803624510765076, -1.828014612197876, -0.22835321724414825, 0.30876025557518005, 0.5729226469993591, 0.5513465404510498, -1.5321930646896362, -0.31458234786987305, -0.2789110541343689, -1.1166951656341553, -0.5851112604141235, 0.18439878523349762, 1.1590441465377808, -1.4130654335021973, -0.5163005590438843, 0.2217637598514557, 1.3080830574035645, -3.0231895446777344, 1.0791584253311157, -0.28182458877563477, -0.6293185949325562, 0.6456484794616699, 0.41837161779403687, 0.6263924241065979, 0.8013332486152649, -0.30086231231689453, -0.5650250911712646, 0.3191724419593811, -0.472426176071167, 0.07497801631689072, -0.19737540185451508, -0.4795951843261719, 0.295376181602478, 1.4332534074783325, -0.9338611960411072, 0.23950348794460297, -0.27337637543678284, 0.9830873608589172, -0.549507200717926, 1.0123869180679321, -0.5845323801040649, -1.856874942779541, 1.6832460165023804, 1.1300640106201172, 1.1546735763549805, -0.40625494718551636, -0.8705567717552185, -0.2925203740596771, 0.6074217557907104, 0.04858915135264397, 1.855670690536499, -1.0284326076507568, 2.2324438095092773, -0.7584100961685181, -1.5039236545562744, -1.7387360334396362, -0.2920598089694977, 0.7800400853157043, 0.91390061378479, -1.2793203592300415, -0.9908674955368042, -0.882197916507721, 0.18278639018535614, 0.2620098888874054, -1.8884813785552979, -0.06953984498977661, 0.010192236863076687, -0.5406407117843628, -0.852716326713562, -1.1395907402038574, -0.3087649941444397, -1.6068744659423828, -0.7017247676849365, -1.6336584091186523, 0.14267170429229736, -0.17239172756671906, 1.6579947471618652, 1.859271764755249, 1.159212589263916, -0.6434634923934937, -0.8053939938545227, -1.2791528701782227, 0.9356168508529663, -1.2277628183364868, -0.30332571268081665, -0.3898824155330658, 0.12014276534318924, 0.749570369720459, -0.6498509049415588, 0.5977407097816467, -1.772497534751892, 0.4665755033493042, 0.5388720631599426, 0.4962329864501953, 0.3638440668582916, 1.9683451652526855, -0.17480887472629547, 1.3089852333068848, 0.7275658845901489, 0.6989783644676208, -1.3224705457687378, -0.06159986928105354, -0.48314550518989563, 0.41743993759155273, 0.8740974068641663, 0.13331091403961182, 0.09707340598106384, 0.0436822846531868, 0.21069298684597015, 0.7697425484657288, 0.4763202965259552, 0.5106974244117737, 0.39972585439682007, 0.028366317972540855, -0.9064595699310303, -1.1182019710540771, -0.5335993766784668, -0.6801011562347412, -0.5993745923042297, -1.4122294187545776, 2.5185558795928955, -0.10745571553707123, 0.39509791135787964, -0.8479477763175964, -0.03428724408149719, 1.2111736536026, -0.020499613136053085, 0.05548626929521561, -0.5105311870574951, -0.6276838183403015, -0.6705950498580933, 0.3109714984893799, -0.40645092725753784, 1.138673186302185, -0.19306659698486328, -1.7555580139160156, 1.2165755033493042, 0.14067572355270386, -0.3338623642921448, -1.1464804410934448, -0.20697394013404846, -0.2779628336429596, 1.8046174049377441, 0.04423031583428383, -1.1856340169906616, -1.0895631313323975, -1.8202331066131592, -0.09930159896612167, 1.686872959136963, 1.387099027633667, -0.25694164633750916, -1.2367337942123413, -0.7759200930595398, -1.4931708574295044, 0.4387466013431549, 0.5734424591064453, -0.8935333490371704, 0.5143283009529114, -0.5576913356781006, -1.0784493684768677, 0.4334179759025574, 0.28808796405792236, -0.3839665651321411, -0.9504426121711731, -0.8839668035507202, 0.4070444107055664, 1.4368704557418823, 0.09947855770587921, 0.27180784940719604, 2.8973093032836914, 0.4461846351623535, -1.519888162612915, 0.9471395611763, -0.3707493543624878, 0.16450685262680054, 0.44083869457244873, 1.3351436853408813, 0.8182563781738281, 1.4443542957305908, 0.28655901551246643, -0.5025339722633362, -0.5134429931640625, -0.20929521322250366, -0.6103720664978027, -0.7330531477928162, 1.9220192432403564, -2.0819196701049805, -0.9209473729133606, 0.47401559352874756, -0.525791347026825, -0.8172215819358826, 0.385023295879364, 0.11016995459794998, 0.6550447344779968, 0.04791364446282387, -0.08207856863737106, -0.5076684355735779, -0.14529187977313995, -1.1906425952911377, 1.3031904697418213, 1.0105674266815186, 0.31824806332588196, 1.9908108711242676, -0.8277807235717773, 1.7963066101074219, -0.9269804358482361, -2.125274181365967, -0.48910510540008545, -0.14152216911315918, -1.334854006767273, -0.15401692688465118, 0.06918323785066605, 0.12243058532476425, -1.0485137701034546, -0.8068185448646545, -2.3242063522338867, -1.39625084400177, 0.8127559423446655, 0.03853308781981468, -2.9454450607299805, -0.8325705528259277, -1.5135775804519653, -0.3392142355442047, -0.11470291763544083, 0.6192291378974915, 0.8055046200752258, -1.7083773612976074, 0.12641794979572296, -0.09246745705604553, 0.36513346433639526, -1.3301670551300049, 1.075702428817749, -0.11534252017736435, 0.6892576217651367, -0.063633993268013, -0.5506842732429504, -0.020462971180677414, 1.4103373289108276, -0.16889353096485138, 0.13730841875076294, -0.20102502405643463, -1.024405598640442, 0.8983322381973267, -1.0119616985321045, -0.9202942252159119, 0.8648989200592041, 2.4138412475585938, 0.25537192821502686, -0.12397249788045883, -0.26367518305778503, 0.9847679734230042, 1.3440176248550415, -0.10346993058919907, -0.3801940679550171, -0.7157896161079407, -1.2771130800247192, -1.7303786277770996, 2.1376378536224365, -1.0251147747039795, 0.8134379982948303, -0.0933861956000328, 0.18097251653671265, 1.4413114786148071, -0.5111315250396729, -0.39090508222579956, -0.989048421382904, 0.45766618847846985, 1.0030559301376343, -0.41537198424339294, 0.24931971728801727, 0.46724259853363037, 0.39297565817832947, 0.23617099225521088, 0.47332191467285156, 0.5672977566719055, 0.23083674907684326, -0.039724308997392654, -0.928129255771637, -1.6133898496627808, -1.328731656074524, -0.18970300257205963, 0.223188579082489, 0.338051438331604, 1.0531139373779297, 0.7050837874412537, 2.0544896125793457, 1.1274421215057373, 0.8660295009613037, -0.1280924379825592, 1.4173197746276855, 0.06495017558336258, 0.26773208379745483, 2.077794313430786, -0.8411586284637451, 1.2565358877182007, 0.7877407073974609, -0.9433408379554749, 1.0195802450180054, 0.46243003010749817, -1.5061055421829224, 0.1301996260881424, 0.3691944181919098, -1.441780924797058, -1.8834067583084106, 0.8047935366630554, 0.8246335983276367, 0.09483463317155838, -0.6657811999320984, 0.0902416855096817, 0.163908913731575, 1.9685022830963135, -1.6266885995864868, 1.9502366781234741, -0.841344952583313, 0.7928377389907837, 0.5116048455238342, -0.11406498402357101, 0.5957195162773132, -0.7916737794876099, 0.14013786613941193, 0.24216949939727783, 1.21443510055542, 0.5102103352546692, -1.125770092010498, 1.3574209213256836, -0.5700932741165161, -0.2400248497724533, -1.5170974731445312, 1.7622150182724, 0.8710130453109741, 0.27744218707084656, -0.8464585542678833, 1.3488476276397705, 0.12138281017541885, 0.03031841665506363, 0.6630502343177795, 0.29819121956825256, 1.3118925094604492, -0.49452313780784607, -0.7849690914154053, 0.4540840983390808, 1.0142238140106201, 0.46680748462677, 0.7253462672233582, 1.5376036167144775, -0.31414714455604553, 0.11632850021123886, -1.9114603996276855, 1.012377142906189, -1.0753660202026367, 0.05120136961340904, -1.6856622695922852, 0.3134608566761017, -0.3814951777458191, 1.9618902206420898, 1.002554178237915, -1.5401219129562378, 1.8084766864776611, 0.3705679178237915, -1.093837022781372, -0.46497029066085815, 2.3838484287261963, 0.48460203409194946, -0.999100923538208, -0.18768295645713806, -0.7472424507141113, -1.6374160051345825, 0.32499226927757263, 2.406581401824951, 0.21074512600898743, 3.7832725048065186, -0.016515344381332397, 0.9137569069862366, -0.9787196516990662, 0.20966392755508423, 0.604052722454071, 0.3200696110725403, 0.529900312423706, 1.4793894290924072, -0.6840313076972961, -0.6874938011169434, -0.2336382269859314, 0.38116735219955444, -0.8083109259605408, -1.670113205909729, 0.0001951478043338284, -0.0693042054772377, -1.4490816593170166, -0.3423156440258026, -0.22678479552268982, 1.8566639423370361, 0.4656003415584564, -0.15783050656318665, 0.5578976273536682, 1.0800127983093262, -0.4752369523048401, 0.05673526972532272, 0.6229836344718933, 1.1126680374145508, -1.3522731065750122, -1.5384831428527832, -1.0973857641220093, -0.34849822521209717, -0.7629873752593994, 2.455578088760376, 0.5642870664596558, -1.5744028091430664, 0.3015258014202118, 0.06448529660701752, 0.16031542420387268, -0.2470199167728424, -0.4338397979736328, -0.10319963842630386, 0.5936150550842285, 0.7262717485427856, 0.6136641502380371, -0.06998252123594284, 0.8023350238800049, -0.575482189655304, 0.9913786053657532, -0.07738500833511353, 1.191451907157898, -0.53718501329422, -1.4038006067276, 1.8004815578460693, -0.10010571032762527, 0.06986776739358902, 0.10280699282884598, 0.446897953748703, -0.41449788212776184, -2.0741782188415527, 1.0427833795547485, -2.032529354095459, 0.8316263556480408, -0.5487857460975647, -1.8607985973358154, -1.3954530954360962, -0.6972142457962036, 0.9977449774742126, -0.4380367398262024, -0.5634244084358215, 0.4892316162586212, 0.6192468404769897, 0.01615021377801895, 1.9085928201675415, -2.0154941082000732, -1.499828577041626, 0.4079791307449341, 1.2628753185272217, -0.14232054352760315, -0.5479097962379456, 0.9791367053985596, -0.03214915841817856, 2.443777322769165, -0.4073185920715332, 0.7680396437644958, -0.3664900064468384, 0.9540800452232361, 2.0807907581329346, -0.7965706586837769, -2.330968141555786, 1.0252100229263306, -1.3029224872589111, 1.1973800659179688, 1.777870774269104, -0.2293781340122223, 1.465754508972168, 1.201981544494629, 0.45521074533462524, -1.7520450353622437, 0.341193288564682, 1.0963833332061768, -0.00041292665991932154, 0.40173766016960144, -1.1499178409576416, -0.5399792194366455, -1.1490979194641113, 1.524556279182434, 1.1236872673034668, -0.0760318711400032, -0.22817327082157135, 1.0785095691680908, 0.4321919083595276, -1.1916650533676147, 1.4019193649291992, -0.18526040017604828, -2.3894524574279785, -0.13876792788505554, -0.4379671812057495, -1.494800090789795, 0.03361836075782776, -1.7215675115585327, 0.5420176386833191, -0.948800802230835, -0.7493644952774048, 0.1791536659002304, 1.2465488910675049, -0.5255783796310425, -0.6879976987838745, -0.3943617641925812, 0.16076767444610596, -0.6660410165786743, -0.8923123478889465, -0.4907117187976837, -0.5392462611198425, -0.6080686450004578, 1.0252894163131714, 2.69016170501709, 1.1545665264129639, -1.3856782913208008, -0.4284484386444092, 0.24813927710056305, -0.22798626124858856, 0.9710489511489868, 0.18809492886066437, 0.7045765519142151, 1.0192211866378784, -0.8415939211845398, -0.2811044454574585, -0.3924967050552368, -1.8627430200576782, 1.1123626232147217, 0.24464988708496094, 0.5798898339271545, -0.5711957812309265, 0.3640759587287903, 0.15922395884990692, -1.007273554801941, 0.7326156497001648, 0.2192251980304718, -0.7634146809577942, -0.4675500690937042, 1.53665030002594, 0.49861153960227966, -0.26147130131721497, 0.9121266603469849, 1.3299075365066528, 0.9275003671646118, -1.1773419380187988, -1.5033819675445557, 0.5895070433616638, -0.514930009841919, -1.5591503381729126, 0.3412930965423584, -0.11183133721351624, -0.06587258726358414, 1.5141407251358032, 0.8120418190956116, -1.369727373123169, 0.4276924133300781, -1.366758108139038, 1.2918635606765747, 1.5892246961593628, 1.513138771057129, 0.056817155331373215, -1.998979091644287, 0.2860829830169678, -0.43243151903152466, 0.7401922345161438, 0.06805279105901718, -0.5960920453071594, 0.18443332612514496, -0.07317190617322922, -0.08340313285589218, -1.6401828527450562, -0.8894030451774597, -0.2734314799308777, -1.0237491130828857, -0.7391163110733032, 0.18686728179454803, -0.1870184987783432, -0.6336629986763, 1.3587791919708252, -0.5685210824012756, -1.4590120315551758, 0.6288295388221741, -0.967582643032074, -2.065669536590576, -0.252302885055542, -0.9083788394927979, -0.9720858335494995, 0.05606415122747421, -0.44069939851760864, 1.6175507307052612, 0.057745449244976044, -0.7224075198173523, 0.633082926273346, 0.2120446264743805, -1.8464527130126953, -0.8523042798042297, 0.6285116076469421, -0.06851758062839508, -0.1636432409286499, -1.001189112663269, 0.7221278548240662, -0.48043912649154663, -0.5273807048797607, -0.5258504748344421, 0.6963348984718323, 0.8441113233566284, -0.46571141481399536, 1.4383845329284668, 0.06822888553142548, 0.5702852606773376, -0.12860260903835297, 0.255171537399292, -0.25290098786354065, 0.20777879655361176, 0.2847791612148285, -0.7829852104187012, 0.3158833086490631, 1.2344129085540771, -1.1020299196243286, -1.7721593379974365, -1.6604968309402466, 0.42848607897758484, 1.435183048248291, -0.3885003924369812, 0.02365797944366932, -1.319177508354187, 0.1734616905450821, -1.5680711269378662, -1.8841702938079834, -1.0627663135528564, 0.8403976559638977, -2.23260498046875, 0.24233730137348175, -0.5405847430229187, 1.3365830183029175, -0.6213378310203552, 2.066976547241211, -0.8185291290283203, -0.053946249186992645, 0.6857418417930603, 1.5390465259552002, -0.5211955904960632, -1.2124521732330322, 0.3174460232257843, 0.780328094959259, 1.098583459854126, 0.2757634222507477, 0.9420290589332581, 0.3284900486469269, -1.4189852476119995, -0.8281444907188416, -0.5177643895149231, 0.6577406525611877, 0.23103082180023193, -0.09789275377988815, -2.312657117843628, -0.6096081137657166, 0.7320315837860107, 1.0780683755874634, 0.741567850112915, 0.8910303115844727, -1.4392352104187012, -1.0683660507202148, -0.19240401685237885, 1.1357157230377197, 0.8316509127616882, -0.027036402374505997, -0.8738000392913818, 0.06297659873962402, -0.8121762275695801, -0.961733877658844, 1.3224518299102783, -1.0076072216033936, 0.6307726502418518, -0.866929829120636, 1.7085336446762085, -2.6240906715393066, 0.5341657400131226, 0.1902531236410141, -0.7416877746582031, 1.7846441268920898, 0.5508990287780762, -0.25662341713905334, -0.24409787356853485, 0.642815351486206, -1.250230312347412, -1.5639475584030151, -0.8316092491149902, 1.0350325107574463, -0.5206699967384338, 0.9429033994674683, 1.0944750308990479, -0.6177218556404114, 0.36382004618644714, -0.1694151759147644, -0.8012508749961853, 0.10744444280862808, -0.1464112102985382, 0.008166072890162468, 0.8694178462028503, 0.20944491028785706, -1.7663692235946655, -0.21500737965106964, 1.2865209579467773, 0.0065879314206540585, -0.29235848784446716, 0.3670743405818939, 1.1606241464614868, -2.0107367038726807, 1.8167273998260498, 0.1435685157775879, -1.7761236429214478, 1.3590606451034546, 2.3933932781219482, 1.1611483097076416, 0.9221886992454529, 0.5628929138183594, 0.5608359575271606, 0.36341193318367004, 1.5112948417663574, 0.49923911690711975, -0.6690367460250854, 0.36311450600624084, 0.5104591846466064, -2.3121118545532227, 0.2069104164838791, -2.509646415710449, 0.14100144803524017, -1.290545105934143, -1.9402347803115845, -1.001246690750122, 0.7225031852722168, 1.0403766632080078, -0.8097701072692871, 0.8823474645614624, 0.45457354187965393, 1.182004690170288, 1.0744153261184692, 0.6486579775810242, -2.2927584648132324, 0.6592385172843933, -0.2125653326511383, -1.900382399559021, -1.2811310291290283, 0.38198015093803406, -0.22327375411987305, -0.5720452666282654, -0.021043501794338226, 0.183678537607193, -0.7559510469436646, 0.4753817617893219, 0.5848215222358704, 0.7256113886833191, 1.6357567310333252, 1.333781123161316, 1.9040104150772095, 0.45585209131240845, 1.3826526403427124, -0.6944704651832581, 0.7637600898742676, -1.1896841526031494, -0.4225206971168518, 0.7563768029212952, 0.34465035796165466, -2.178570508956909, 0.6884556412696838, -1.0473276376724243, -0.7379905581474304, 0.6465083360671997, -0.5283316373825073, -0.12690526247024536, -1.4805388450622559, 0.09720530360937119, -0.7693246006965637, -0.6739595532417297, 0.7977451682090759, -0.5337210893630981, -1.4051274061203003, -0.09759445488452911, 0.76067054271698, 0.2920236885547638, -2.953479051589966, -0.640083372592926, -0.43559354543685913, -2.0669972896575928, -0.01870216242969036, 0.17585960030555725, -1.187726616859436, 0.4574472904205322, -1.0548877716064453, 0.2811635732650757, 2.246260643005371, -0.5728613138198853, 0.5593541264533997, 0.5078108906745911, 0.007052939385175705, 1.293868899345398, 0.25143373012542725, 2.4708120822906494, -0.1095413938164711, 0.3791913688182831, -0.8951830267906189, 0.5813156366348267, -0.10421493649482727, 0.8459845781326294, -1.0467449426651, 0.6305205821990967, -1.136626124382019, -0.3530668616294861, 2.0614359378814697, 0.3088546395301819, 0.3821119964122772, 2.119680166244507, 0.4902394115924835, -0.6266258358955383, 0.6037762761116028, 1.3865611553192139, -0.5906040072441101, 0.22964699566364288, 0.9174531698226929, -0.4528062045574188, 0.3097214698791504, -1.1535003185272217, -0.04771709069609642, -1.4555264711380005, 0.0173930786550045, 2.3311703205108643, 0.4908095896244049, -0.6741092205047607, -0.709160327911377, -0.2040574550628662, -1.0006109476089478, -0.024463936686515808, -0.9184488654136658, 0.6440000534057617, -0.006662833038717508, -0.3900766670703888, 0.5711822509765625, 0.2939797043800354, 1.94892156124115, -0.7785438895225525, 1.4819090366363525, -0.8368568420410156, 0.6657200455665588, -2.0397605895996094], "dtype": "float32"}, {"__nd__": [1.2248691320419312, 0.03667665272951126, 1.1645355224609375, 0.6781101226806641, -1.5488193035125732, 0.07863481342792511, -0.590228796005249, -1.2396506071090698, -0.6993206739425659, -0.7645642161369324, -1.5291944742202759, -1.736527681350708, 0.3320883512496948, -0.06611952185630798, -1.1498209238052368, 0.4327496588230133, 0.36421725153923035, 0.022296367213129997, -0.6583837866783142, -1.1322205066680908, -1.0438088178634644, 1.6747422218322754, -1.0977224111557007, -1.1893537044525146, 1.2355245351791382, -0.6385915875434875, 0.5915988683700562, -0.4615944027900696, 2.076263904571533, -0.8186275362968445, -1.3196955919265747, -0.5112106204032898, 2.0299925804138184, -0.3471064269542694, -0.385925829410553, -0.22093358635902405, -0.18954706192016602, 0.4563509523868561, 1.134975552558899, 1.6931707859039307, 0.663702130317688, 1.1762471199035645, -1.0458096265792847, -0.5418112277984619, 1.241133451461792, -0.7988930344581604, 0.8174464106559753, -0.4398384094238281, -1.1135046482086182, -0.7040405869483948, 1.7347705364227295, -0.39379048347473145, 0.8404501080513, -0.4224177896976471, -0.8867653608322144, -0.14556966722011566, 1.0162206888198853, 0.5225532650947571, -0.401103138923645, 1.8748376369476318, 0.17479978501796722, 1.397429347038269, -0.13301090896129608, 1.0077369213104248, 0.181749626994133, -0.7270065546035767, -0.1104421392083168, 0.9180655479431152, 0.15325017273426056, -0.4015371799468994, -0.3138881027698517, -0.056253112852573395, 1.5073175430297852, -1.34926176071167, -1.3708460330963135, -2.414175271987915, 1.063486933708191, -0.5015692710876465, 0.06215541809797287, -1.4181185960769653, -0.675132155418396, -1.7411288022994995, -0.7699259519577026, -0.6351388692855835, 1.3691468238830566, -0.8473155498504639, 0.5261052250862122, -0.3344317674636841, -0.03526868671178818, 0.9600269198417664, 0.2542389929294586, -0.44789716601371765, 0.7804582118988037, 0.6868835687637329, 0.9358027577400208, 0.2643551528453827, -0.46296632289886475, 0.17955119907855988, -0.11534833908081055, 1.2174888849258423, 0.8333198428153992, 1.7608238458633423, 0.7317225933074951, -0.2220500409603119, -0.17061839997768402, -2.130110263824463, -0.39887887239456177, -0.6842477321624756, 1.135769248008728, -1.526254415512085, 0.7327048778533936, 0.7667919397354126, -2.080016851425171, -3.0436503887176514, -1.0733636617660522, 1.1502737998962402, 0.530600905418396, 1.0406546592712402, -0.20572149753570557, 1.0778160095214844, -0.38478773832321167, -2.4011409282684326, 0.9246259331703186, -1.031144618988037, -0.26154854893684387, 0.6058578491210938, -2.1801702976226807, 0.40493351221084595, 1.1052175760269165, -1.203892707824707, -0.3229221701622009, -0.4153437614440918, -0.6400051712989807, -1.9674853086471558, 1.020461082458496, 1.3266513347625732, -0.9439083337783813, -0.8956946134567261, -0.5544183850288391, 0.8941599726676941, -0.8251680731773376, -0.6812419295310974, 0.24888035655021667, 2.0155763626098633, 0.0022086570970714092, 2.4156086444854736, 0.28628695011138916, -0.5761909484863281, 0.8960201740264893, -1.135286808013916, 0.4221953749656677, -0.15916167199611664, -0.4213487505912781, 1.4564082622528076, 1.020820140838623, -0.8492008447647095, 0.5717692375183105, -0.9154492616653442, -1.4671428203582764, 0.041944004595279694, -1.0256822109222412, 0.35879960656166077, -0.5164329409599304, -0.24829842150211334, -0.022317489609122276, -0.2389676719903946, -0.019012710079550743, -0.19793108105659485, 0.28726261854171753, 0.09924431890249252, -0.7417694330215454, -0.05829159915447235, 0.8070599436759949, -0.7964173555374146, 0.33210480213165283, -0.07328970730304718, 0.02296363189816475, -0.3277285397052765, -0.6334190368652344, 0.10830635577440262, 0.055275995284318924, -0.20351265370845795, -0.4482399821281433, 1.1119780540466309, -0.22732171416282654, 1.9584654569625854, 0.7144457697868347, -0.5210253596305847, 0.45759353041648865, -0.6975962519645691, 1.3724697828292847, 0.3322400748729706, -1.0145052671432495, -1.0576083660125732, 0.5835683941841125, 0.027415910735726357, -0.28562501072883606, 0.20681488513946533, -0.020732751116156578, 1.1280663013458252, 0.8839832544326782, -0.022669021040201187, 0.8324753642082214, -0.9792489409446716, 0.8219069838523865, 2.119751453399658, 0.7784566879272461, -0.7345549464225769, -1.2256405353546143, -0.784889817237854, -0.3937279284000397, -0.8981658816337585, -0.21243980526924133, 0.6042763590812683, -1.333333134651184, 0.29751649498939514, 1.3038464784622192, -0.5783405900001526, 0.08089990168809891, 0.5710995197296143, -0.2047034204006195, -0.548909068107605, -1.5486589670181274, -0.9529778361320496, 1.4470703601837158, -0.3073647618293762, 0.12451762706041336, 0.6146813035011292, 0.3783372938632965, 1.2495801448822021, 0.5385321974754333, 0.6431615948677063, 0.26459360122680664, 0.6890779137611389, -1.00931978225708, -2.133930206298828, -0.8120675683021545, -1.0167375802993774, -0.1640215814113617, 0.24196231365203857, -1.2101716995239258, -0.23605214059352875, -0.9705438017845154, 0.7180163264274597, -0.700846254825592, -0.11447355151176453, -1.0358821153640747, 0.9929546117782593, -0.02137421816587448, -1.0561165809631348, 0.14645089209079742, 0.35054537653923035, -0.5820411443710327, 0.024993320927023888, -1.0016263723373413, -0.9879687428474426, 0.020479019731283188, 2.6832573413848877, 0.8605912327766418, -0.38063713908195496, -0.4083237648010254, -1.326230764389038, 1.069934606552124, 1.0195856094360352, -1.3185577392578125, 0.7938840985298157, -1.404078722000122, -1.7203633785247803, -2.0462985038757324, -0.9716870784759521, 0.5357844829559326, 1.6262588500976562, 0.6039170622825623, -1.1328810453414917, -0.7715049982070923, -0.21675173938274384, 1.2643704414367676, -0.24362899363040924, -2.1067254543304443, -0.7064560055732727, -0.19405792653560638, -0.6236368417739868, -1.093035101890564, -0.25981584191322327, -0.5974532961845398, 0.9240838289260864, -0.3152957260608673, -1.687124490737915, -0.468271940946579, 0.9458590745925903, -2.091709852218628, 0.6941509246826172, 0.20554664731025696, 1.3294258117675781, 0.4136519730091095, -0.09992097318172455, 0.3676450848579407, 0.7056074142456055, 0.6005988121032715, -1.1864274740219116, 0.695841372013092, -1.16114342212677, 0.35527393221855164, -2.1508188247680664, -0.47702646255493164, 0.694920003414154, -1.007981538772583, 0.8620012998580933, -0.1905582845211029, -1.0249407291412354, -0.35325193405151367, 1.0287476778030396, 0.8391759991645813, 0.12094414234161377, 0.49061575531959534, -0.6355228424072266, 2.3368279933929443, 0.7111539244651794, 0.11849334090948105, -0.7189630270004272, 0.2638484537601471, -0.23338454961776733, -1.0533379316329956, -0.0667504221200943, -0.9770016074180603, -1.9001551866531372, 1.9667469263076782, 1.1677193641662598, 0.5759553909301758, -0.28368422389030457, -0.6082340478897095, -0.6106003522872925, -0.6603639125823975, -0.3087455630302429, -0.2124292552471161, 0.39373260736465454, 1.318953275680542, -1.6550958156585693, -1.1172025203704834, 2.8494977951049805, -1.2855807542800903, 0.03143492713570595, 1.173600673675537, 2.018281936645508, 1.3888561725616455, 0.4600168764591217, -0.17045645415782928, -3.249208450317383, -0.032680585980415344, -1.0468964576721191, 0.7074590921401978, -0.0761539489030838, -1.3759716749191284, -0.060648560523986816, 1.1569744348526, 1.549171805381775, 1.7713007926940918, 0.05005103722214699, 0.12487106025218964, 0.6095819473266602, -1.5187299251556396, 1.127748966217041, -0.30111655592918396, 0.4440991282463074, 0.852554202079773, 1.23469078540802, -0.022170336917042732, 1.0408568382263184, 1.2843849658966064, -0.07233239710330963, -2.6439290046691895, 1.4986456632614136, -0.36667346954345703, 2.811002731323242, 0.20876848697662354, -1.7930864095687866, -2.0983963012695312, 0.5919516682624817, 1.5249383449554443, 0.212646022439003, 0.3533242344856262, -0.10873020440340042, 0.21057546138763428, -0.2794947326183319, -0.636608898639679, 0.7698885202407837, 0.6298510432243347, 1.4682596921920776, -1.2659142017364502, 0.744932234287262, 1.1341354846954346, -0.3419523239135742, 1.3741016387939453, 0.5216890573501587, 1.0178333520889282, -0.8867803812026978, 0.3922150135040283, -0.15452320873737335, -0.32771405577659607, -1.076069951057434, -0.2569509744644165, -2.2865753173828125, 0.1766911745071411, 0.9427640438079834, 0.03578248620033264, -0.048640865832567215, 0.9899961948394775, 1.009458065032959, -0.062328651547431946, -0.6836087703704834, -0.7408132553100586, 1.3781882524490356, -0.6601431965827942, -0.7700344324111938, -0.7377868890762329, -1.4417897462844849, 0.6789447665214539, -1.0534600019454956, 1.2026641368865967, -0.5252322554588318, 0.3264719843864441, 0.8994318246841431, 0.5534335374832153, 0.3594176173210144, 0.5798270106315613, -0.1726490706205368, 0.0571109876036644, -0.9609552621841431, -0.45044752955436707, -1.2538691759109497, -1.135398507118225, -0.5603020787239075, 1.401911735534668, -0.7988487482070923, -1.120753288269043, -0.2645546793937683, 0.28446537256240845, 0.6224882006645203, -0.233587384223938, 1.6336208581924438, -1.070313572883606, -1.128808617591858, 0.5108119249343872, 1.288334846496582, 0.2902025580406189, -0.9839904308319092, -0.6451717019081116, -1.5471752882003784, -0.45395627617836, 0.5164942741394043, 1.4662803411483765, -0.5218991041183472, -0.10081007331609726, 1.3797580003738403, 0.07987333089113235, -0.040239933878183365, -1.414137840270996, -0.5027053356170654, 0.9818838834762573, -1.020205020904541, -0.18830163776874542, -0.35855719447135925, 0.5342555046081543, -0.7543243765830994, -0.8304937481880188, 0.7730380892753601, 1.3633989095687866, -0.8340849280357361, 0.238754004240036, -0.08261510729789734, 0.19360141456127167, 0.3831992745399475, 0.18955709040164948, -0.3517269790172577, -0.29533013701438904, 0.7732224464416504, 0.7379567623138428, 1.5173763036727905, 0.3158605396747589, 0.09980614483356476, -1.1221473217010498, -1.4005954265594482, 0.20080283284187317, 0.6438153982162476, -0.8355065584182739, 1.1409518718719482, 0.7015032768249512, -0.9918655157089233, -1.1709589958190918, 0.5991945862770081, -0.07185523957014084, -1.038926601409912, 0.349600613117218, -0.09881982207298279, -1.004191517829895, -0.004117156844586134, -0.11881770938634872, 0.0889601856470108, 0.10013508051633835, -1.097442626953125, 0.47036486864089966, -0.5882236957550049, -1.3617072105407715, -1.1464053392410278, -0.5644403696060181, 0.87458735704422, -1.2083735466003418, -0.0033938370179384947, -1.48122239112854, 0.9140516519546509, 1.4314489364624023, 0.6453100442886353, 1.0391972064971924, 0.159865140914917, -0.5223104953765869, -0.10082035511732101, 0.6233503818511963, -1.5692739486694336, -0.31631675362586975, -0.0026596307288855314, -0.4392716586589813, 1.573995590209961, -0.556981086730957, 0.3400706946849823, 1.0034089088439941, -2.576528787612915, -1.6309577226638794, -0.7137983441352844, 0.989282488822937, 0.5832359194755554, 0.17317895591259003, -0.1937103569507599, -1.5396454334259033, 1.2043461799621582, -0.4353826642036438, -0.28590455651283264, 2.615229368209839, -1.130587100982666, -0.16376276314258575, 0.8238169550895691, 1.132063865661621, -0.39184579253196716, -0.5215886831283569, 0.35895296931266785, 1.0528614521026611, 0.2983976900577545, 0.6263298392295837, 0.3496120274066925, 0.7401398420333862, -0.7489306926727295, -0.056401945650577545, -0.01184207946062088, 1.5215461254119873, -0.8490942716598511, 0.19562718272209167, 0.3379022181034088, 0.3379567861557007, 0.11741473525762558, 1.0910509824752808, 0.9135882258415222, -0.04129058122634888, 0.9294045567512512, -1.2232887744903564, 0.2489822655916214, 1.7013968229293823, -1.892648458480835, 0.6690512895584106, -1.4086440801620483, 0.16411258280277252, 0.7535143494606018, -0.34823858737945557, 1.627374529838562, 1.4138160943984985, -0.2395457625389099, -0.18840134143829346, 1.0438941717147827, -0.8983330130577087, -1.9050265550613403, -0.4060589075088501, 0.9496713280677795, 0.39947912096977234, 1.7890770435333252, -0.8751355409622192, 0.2060869336128235, 0.25057199597358704, 0.9196619391441345, 0.9034464955329895, -0.03198530152440071, 0.38252562284469604, -0.23640590906143188, -0.11643079668283463, -0.2571544945240021, -0.09451017528772354, -1.2583714723587036, 1.0357728004455566, -2.3688321113586426, 1.4229296445846558, 0.33475714921951294, 0.8947038650512695, 0.7438536882400513, -1.2658705711364746, 0.6847498416900635, 0.6082390546798706, -0.7822656035423279, 1.1576052904129028, 0.001990190241485834, -2.14634370803833, 1.0117837190628052, 0.3553939163684845, 0.40288832783699036, -0.21498925983905792, -0.2538608908653259, -1.079274296760559, 2.1201679706573486, -0.7080795764923096, 0.42936041951179504, 0.07640015333890915, -1.664860725402832, -0.900263249874115, 0.7031607627868652, -0.10954157263040543, 1.3333625793457031, 1.7788339853286743, -0.7430170774459839, 0.3673655092716217, -1.4722431898117065, -0.6221349239349365, -0.7202363014221191, -0.757148027420044, -3.2251179218292236, -0.5334795713424683, -0.5254082083702087, -1.463819146156311, -1.18719482421875, -0.4132538139820099, -0.8746187686920166, 2.2545063495635986, -0.7070354223251343, -1.0656893253326416, -1.2347164154052734, -1.3557188510894775, 0.3344084620475769, -0.43441271781921387, -1.1123007535934448, -1.0342122316360474, -0.1626203954219818, -1.2331706285476685, 0.3324206471443176, -1.8137333393096924, 1.3433068990707397, 0.8771387934684753, -0.05294441059231758, -0.5545679330825806, 0.2755810618400574, -0.8732070922851562, -0.8662863373756409, -1.2357769012451172, 0.33827298879623413, -1.632219910621643, -0.4213453531265259, -2.452333927154541, -0.27160364389419556, -0.9275261163711548, 0.9426988363265991, 1.4262309074401855, 1.339876651763916, 0.4570049047470093, -1.4115800857543945, -1.0004172325134277, 0.07681222259998322, -0.7699323892593384, 0.2646685838699341, -0.7433545589447021, -0.2627761960029602, -1.6826295852661133, -0.2902240455150604, -0.7389802932739258, 0.4874550998210907, -1.8746017217636108, -0.4635973870754242, -1.4236078262329102, -0.9442121386528015, 0.3702230751514435, 1.002898097038269, -0.2540069818496704, 0.10435155779123306, 0.3531428277492523, 0.7296456098556519, 0.19671405851840973, -1.6530089378356934, 0.5631793737411499, 0.2902204692363739, 0.5676696300506592, 0.36411482095718384, 0.6828973293304443, -1.0949628353118896, -0.4101049304008484, -0.29479798674583435, 0.053859662264585495, -0.3141166865825653, -1.1322698593139648, -0.32258346676826477, -0.4557817280292511, 1.7943758964538574, 0.7784774899482727, -1.1612682342529297, -1.8110029697418213, 1.6679826974868774, 0.20322144031524658, -1.805282473564148, -0.3155845105648041, 0.313764363527298, 1.8509948253631592, -0.04925511032342911, -1.3541070222854614, -0.5677629709243774, 0.6361501216888428, -0.5138257741928101, 0.01882113143801689, 0.5662902593612671, -1.0440279245376587, 1.2318625450134277, -0.9935629367828369, -0.7412269711494446, -0.8242000937461853, 1.9248467683792114, 0.435491681098938, 1.1882481575012207, 0.6089240312576294, 0.7025406956672668, -0.4451742470264435, -1.1602792739868164, 0.032941609621047974, 0.27378222346305847, 1.0703377723693848, 1.8073115348815918, -0.45405858755111694, -0.43534761667251587, 1.0198415517807007, -0.5528209209442139, 0.5559282898902893, 1.2129204273223877, 0.5128741264343262, 0.3203337490558624, -0.32996124029159546, -0.18308128416538239, 0.7440385222434998, -0.7414538860321045, -0.08954266458749771, 0.4846968352794647, -0.039993785321712494, 0.7674392461776733, 0.3242897093296051, -1.3776302337646484, -1.0056853294372559, 0.7372552156448364, -0.18719223141670227, 0.22522568702697754, -0.432574987411499, -0.012735452502965927, -0.8548706769943237, 2.0194091796875, 2.0178418159484863, 1.0072321891784668, 0.2572312355041504, -0.15570801496505737, -1.636055827140808, 0.2903546690940857, -0.6685521006584167, -0.6117858290672302, 0.9127678275108337, 0.5252333283424377, -1.238937258720398, 0.3371935486793518, 0.25584733486175537, -0.42072489857673645, 1.2983686923980713, 0.5123454332351685, -2.650831937789917, 0.887077271938324, 0.12815475463867188, -0.20521993935108185, 1.9268698692321777, -0.7329534888267517, 1.7491074800491333, 1.00200355052948, -1.056227445602417, 0.3127995729446411, 0.45085981488227844, 0.9202792048454285, -2.1339001655578613, -1.250028371810913, -1.1433448791503906, -0.0418672189116478, -0.6080734133720398, 1.0863308906555176, 0.1887281835079193, 0.7940203547477722, 0.6003562211990356, -0.5275482535362244, 1.3248381614685059, -1.4127748012542725, -0.597194254398346, 0.5190867185592651, 0.5314606428146362, -0.8928226232528687, 1.8222564458847046, -1.9737228155136108, 1.1210874319076538, 0.3009874224662781, 0.049089860171079636, 0.673337996006012, -0.013686121441423893, 0.3425765931606293, -0.30716586112976074, -0.19310611486434937, 0.5758983492851257, 0.20951354503631592, -0.7666205763816833, 1.3660831451416016, 0.6657350063323975, -0.8809630870819092, -0.6090809106826782, 0.18636390566825867, 0.13173404335975647, 0.46306291222572327, 0.07685073465108871, 2.5290679931640625, 0.09122505784034729, -1.5008026361465454, 1.82711923122406, -0.4903610050678253, -0.0001069671125151217, 1.1965628862380981, 2.3012795448303223, 0.3676855266094208, -0.6131591796875, 1.405369520187378, -2.0945887565612793, -0.2529692053794861, 0.7779297232627869, -0.6236403584480286, -1.7864868640899658, 0.6144016981124878, 0.07826341688632965, 0.005762225016951561, -0.37621065974235535, -0.721752941608429, -0.8903000354766846, 1.846199631690979, -1.0543737411499023, 1.709574580192566, 0.7617211937904358, 2.074399709701538, 0.7355882525444031, 0.14253994822502136, -0.1975429803133011, 0.0835409015417099, 1.2133793830871582, -0.7996611595153809, -0.6505051255226135, -0.4669717252254486, 0.56330406665802, -1.0511322021484375, 2.2351419925689697, 0.9563314914703369, -0.5158920884132385, 0.1903490424156189, -0.3323412239551544, 0.24623748660087585, -0.05682589113712311, -0.23570013046264648, -0.7246763706207275, 0.5243080258369446, -0.4175242781639099, 0.7750765085220337, -1.2455933094024658, 0.7331088185310364, 0.9961664080619812, -0.4152926504611969, 0.4541333317756653, 2.0148661136627197, -0.48482561111450195, -0.31657102704048157, -0.913064181804657, -0.3665132224559784, -1.4212826490402222, -0.3854858875274658, 1.8192107677459717, 0.5997591614723206, -0.42543718218803406, -0.5372536778450012, -2.5020673274993896, -0.33930322527885437, -0.3122786283493042, -0.35911503434181213, 1.478651762008667, -0.75559401512146, -1.0113543272018433, -1.1606773138046265, 0.8728234171867371, 0.09158182889223099, 1.2479296922683716, 0.38693705201148987, 1.3370656967163086, -1.333829402923584, 0.07591234147548676, -2.434509038925171, 0.9993975162506104, -2.1066250801086426, -0.4916141629219055, -1.1413335800170898, -0.1688348352909088, -0.44554251432418823, -0.8179166316986084, 0.3114439845085144, -0.04571980983018875, -0.6418598294258118, 0.08930324018001556, -0.23050661385059357, 1.2292568683624268, -0.21179482340812683, -1.1340304613113403, 0.5580555200576782, 0.928727924823761, -0.13753658533096313, 1.7115626335144043, 1.3868919610977173, -0.2874283194541931, -0.5128533244132996, 0.9716792106628418, 0.6650803089141846, 0.021657723933458328, -0.8888192772865295, -0.20835652947425842, 0.3565353453159332, -1.568886399269104, -0.1884353756904602, 0.3300739526748657, -0.3851916790008545, 0.7702105045318604, 0.23899555206298828, -0.5393352508544922, 0.606970489025116, 0.40507426857948303, 1.3601065874099731, -0.6616581082344055, -0.0818297266960144, -1.559478521347046, -0.43200427293777466, 1.118709683418274, 0.8775773048400879, -0.08756078034639359, 0.5154719948768616, -0.055464379489421844, -0.6272068619728088, 0.033902015537023544, 1.1183158159255981, -0.5657517313957214, 1.1073917150497437, 0.6300262212753296, 2.554473638534546, 0.2838611602783203, 0.11312126368284225, 0.4642702341079712, -0.5258918404579163, 1.7948634624481201, -0.1446521133184433, 1.0262484550476074, -0.05189671367406845, -0.8729904890060425, 2.466564893722534, -0.8725379109382629, -0.4843609929084778, -1.1304506063461304, 0.6141409277915955, -0.48570406436920166, 0.32280921936035156, 0.3740784227848053, -0.01918582245707512, -2.0682802200317383, 1.687504768371582, 0.009659246541559696, -1.5933310985565186, -0.9504360556602478, 0.19060947000980377, 0.5379424095153809, 0.9490989446640015, -0.8214161396026611, -0.5720136761665344, -0.7037525773048401, -0.5684108138084412, -0.33407434821128845, 0.6137457489967346, -2.4962246417999268, -0.0018778567900881171, 0.19962270557880402, 0.2793630361557007, -0.22098253667354584, -0.33823543787002563, -0.260035902261734, -0.05599081516265869, -0.5191947221755981, 0.694011390209198, 0.11231081932783127, -0.039116017520427704, 1.452203392982483, 0.03566005080938339, -1.2767550945281982, -0.1002252846956253, -1.0613222122192383, -0.6446206569671631, 0.09780758619308472, -0.06286051124334335, 0.2749912440776825, -1.0999724864959717, 0.879619836807251, 0.4304637908935547, -0.18246804177761078, -0.22691719233989716, 0.6349719762802124, 0.13162685930728912, 1.785895824432373, -0.021066265180706978, 1.6937018632888794, -0.04561426490545273, 0.7000319361686707, 0.6628319025039673, 0.03413690999150276, 0.09953311830759048, 0.004973800852894783, 0.8356645107269287, 0.15993808209896088, -0.5187351107597351, -0.10362482815980911, 0.7477659583091736, -0.385768324136734, -1.1712706089019775, -0.9537984728813171, 0.20783494412899017, -0.18659137189388275, -0.4900100529193878, 0.8461272120475769, 0.04157111793756485, 1.1799426078796387, -0.08186417818069458, 0.9477819800376892, -1.5076086521148682, -1.3783760070800781, -0.38424965739250183, 0.9731482267379761, 1.985282063484192, -0.3422674238681793, -0.6143392324447632, 0.7395942211151123, 0.6260818839073181, -0.09877274930477142, 0.7640647292137146, -0.09938545525074005, 0.07909920066595078, -0.28008610010147095, -0.8461421728134155, -0.002809234894812107, 1.4215720891952515, 0.8170273900032043, 1.1766926050186157, 1.5972872972488403, -0.9015661478042603, -0.4738607406616211, 0.24119965732097626, 0.2512892186641693, 1.3509637117385864, -0.017547886818647385, -0.8669335842132568, -1.918882966041565, -0.09975035488605499, 0.340450644493103, -0.8198538422584534, 1.4049053192138672, 1.4025338888168335, 1.2117475271224976, 0.06075283885002136, 0.519827663898468, -0.17620240151882172, -0.13502971827983856, -0.6036763787269592, -0.08920774608850479, -0.13852611184120178, 0.8415412306785583, -1.090618371963501, 0.3367578089237213, -1.1489273309707642, 2.2232182025909424, 1.0352718830108643, 1.1067602634429932, 1.6689174175262451, 0.5189796090126038, -0.7543612122535706, 1.289150595664978, -1.6596860885620117, -0.5880192518234253, -1.1157703399658203, 0.02449444681406021, -2.0005674362182617, 0.30211499333381653, -0.647845447063446, -0.08227700740098953, 0.3650215268135071, -1.8984854221343994, -0.6879652142524719, -0.061577215790748596, -0.8791808485984802, -1.0782970190048218, 1.0252212285995483, 0.21583212912082672, -0.7833397388458252, -0.6613260507583618, 0.5474210977554321, 0.2518423795700073, 1.896655797958374, 1.920525312423706, -2.238640308380127, 0.18091586232185364, 1.2813831567764282, 0.22173309326171875, 0.38778460025787354, -0.6242457628250122, -0.19731852412223816, -2.019078016281128, -0.08102811872959137, 0.2095363438129425, 0.2435150146484375, -1.3607540130615234, 0.6513926386833191, 0.07831481099128723, 0.7219333052635193, 0.7041685581207275, -2.694088935852051, -0.19497571885585785, 0.39897239208221436, -1.479440689086914, 1.322058916091919, 0.5958468914031982, -0.8012036085128784, 0.4953571856021881, 0.8932856321334839, -1.0071722269058228, -1.0192166566848755, -0.5059593915939331, -0.7384852170944214, 0.17468899488449097, 0.14932934939861298, -0.431519091129303, -0.1760409027338028, -1.0361818075180054, 2.2639739513397217, -0.0594535730779171, -0.37417104840278625, 1.3139632940292358, -0.05522835627198219, 0.8779714107513428, -0.3044939637184143, -0.5869233012199402, -0.02037705108523369, -0.142490953207016, -0.3044435977935791, -0.8514999747276306, -0.7723487019538879, -1.0837076902389526, -0.5046219229698181, 0.40251976251602173, 2.2429277896881104, -1.729600429534912, -0.23838038742542267, -1.568544864654541, -0.28278374671936035, -0.4324420392513275, 0.3591149151325226, 0.1990961730480194, -0.5190625786781311, 0.9801068305969238, -0.6727344989776611, -0.7358613014221191, 1.3173073530197144, 1.7768542766571045, 1.0683964490890503, 0.08466589450836182, -1.085636854171753, 0.7659867405891418, -0.34608420729637146, -0.0012534635607153177, -0.2555140256881714, 0.6604782938957214, 0.04505253583192825, 1.0989723205566406, -1.0383384227752686, 1.547045350074768, 1.6592962741851807, -1.025765061378479, -0.8268306851387024, -0.5708284974098206, -0.2221759706735611, -1.2624684572219849, -0.27703291177749634, 0.9080177545547485, -0.2473364919424057, 1.2433621883392334, 0.07926592230796814, 0.5548568367958069, -0.059692010283470154, 1.9525938034057617, 0.7609013915061951, -1.5586395263671875, 1.1954126358032227, 1.1890116930007935, 1.0716514587402344, -0.1354031264781952, -1.6446033716201782, -0.6799866557121277, -0.9832645058631897, -0.3413292467594147, -0.271277517080307, -0.7465247511863708, -0.6986633539199829, -1.6025378704071045, 0.35092195868492126, -0.1408490240573883, -1.160848617553711, -0.7261731624603271, -1.6528520584106445, 1.7659872770309448, -1.2813764810562134, 0.5257989764213562, 1.384840965270996, 1.557327389717102, -1.1332491636276245, -1.9025423526763916, 1.8194223642349243, -0.1294335275888443, -0.5025218725204468, -0.5610302686691284, 1.2169750928878784, 1.070083498954773, -1.0071462392807007, -0.5289382338523865, 2.187528133392334, 0.07113165408372879, -0.4750981628894806, 0.10779833048582077, -0.02204364538192749, 1.1396129131317139, 0.905291736125946, 0.8343278169631958, 0.6179155707359314, -0.12258601933717728, -0.16520915925502777, -1.1489613056182861, -2.0228068828582764, -0.9977817535400391, 0.0024621780030429363, -0.9146111011505127, 0.4957292377948761, 0.2248985320329666, -0.24553029239177704, 0.3772597908973694, -0.17877164483070374, 0.7256259322166443, -1.250835657119751, -0.6929587721824646, -1.5333225727081299, -1.2767888307571411, -0.21017195284366608, 1.2790188789367676, -1.0905063152313232, 0.38414597511291504, -1.14444899559021, -0.4300186336040497, -0.5497870445251465, 1.3471248149871826, 1.317221760749817, -1.5679309368133545, -1.1946545839309692, -2.2789089679718018, 0.01894652657210827, 0.6272718906402588, 2.477362871170044, -0.43102818727493286, 1.2602094411849976, -2.153564691543579, -0.6258471608161926, 0.6768409013748169, -0.2033606618642807, -0.8385215401649475, -0.5059106945991516, -0.31611621379852295, -0.11963338404893875, 0.931283712387085, -0.14333508908748627, -0.33401936292648315, -0.2608577609062195, -1.045459508895874, 0.6195693016052246, -1.0714099407196045, -1.17751944065094, 0.4314825236797333, 0.2573676109313965, -1.2204352617263794, 0.11756487190723419, -1.3483167886734009, -2.1645407676696777, 0.9233760237693787, 0.6781562566757202, 0.06414302438497543, 0.257796049118042, 0.3604392111301422, 0.704060435295105, 1.8191399574279785, 0.4970978796482086, -1.4849228858947754, 1.125008225440979, 2.0050606727600098, 1.8291667699813843, -0.39889419078826904, -1.4652068614959717, 1.2173467874526978, -0.4767606258392334, 2.6216371059417725, 0.6278845071792603, -1.562807321548462, 1.0885733366012573, 0.5410669445991516, -2.1691627502441406, 1.8010741472244263, -0.3501720726490021, -1.0893973112106323, -0.7020560503005981, -0.808513879776001, 0.15817445516586304, -1.0320079326629639, 0.16943414509296417, 0.22133351862430573, -1.204365611076355, 1.503779649734497, 1.3298918008804321, 0.3383684754371643, 2.261308193206787, 0.12573760747909546, -0.42059412598609924, 0.7494714260101318, 0.6760233640670776, 0.5559893846511841, -0.20114360749721527, -0.5004090666770935, 0.033875972032547, -0.7893784642219543, 0.7556904554367065, 0.2645401060581207, -0.14313410222530365, 0.05045441910624504, -0.6505128145217896, 1.3845162391662598, -0.8140531182289124, -2.4001784324645996, -0.63851398229599, 0.4854907989501953, 1.7778301239013672, 0.1665540188550949, 0.21823689341545105, 0.27554553747177124, -0.725848376750946, 1.7841840982437134, -1.1143819093704224, 0.7031180262565613, -0.017649037763476372, -0.4951898157596588, -0.6245113015174866, 0.7799367308616638, 0.2331155240535736, 0.6890490055084229, 0.5319229960441589, 1.4473838806152344, -0.3735488951206207, -1.207553744316101, -2.363741159439087, 1.8905665874481201, -1.2544865608215332, 0.4177790582180023, 0.4871845841407776, -0.38731029629707336, 0.14954854547977448, 1.3345189094543457, 0.5143717527389526, -0.18922263383865356, -1.03107488155365, -1.0841150283813477, -1.6372487545013428, -0.6071887016296387, -2.3865647315979004, -0.6119431257247925, 2.0583713054656982, 1.0273773670196533, -1.2890045642852783, -0.3782002329826355, -0.011616706848144531, 0.25858965516090393, -0.5855915546417236, 0.9699507355690002, 1.480233907699585, -0.6805923581123352, -0.10340159386396408, 0.6879216432571411, 0.6868119239807129, -1.4607774019241333, 0.7521658539772034, 1.5471481084823608, -0.17192475497722626, -1.199790358543396, -0.4804513156414032, 0.457657128572464, -0.9843459129333496, -2.090843915939331, 0.13451530039310455, 2.561944007873535, -0.2445632964372635, -0.39207923412323, 1.2204004526138306, 0.10925497114658356, 0.3143689036369324, -1.4103327989578247, -1.7072120904922485, -0.024626117199659348, -0.19043666124343872, 2.0160787105560303, -0.4278678596019745, -0.32928815484046936, 0.12077036499977112, 1.4791911840438843, 1.7812495231628418, 1.241606593132019, 1.2131094932556152, -0.6940973401069641, 1.3088617324829102, 0.7362001538276672, -0.5921825170516968, -1.5437676906585693, 0.3220120072364807, -1.2225638628005981, 0.6873922944068909, 0.3409431576728821, -0.7976881861686707, 0.7816283702850342, -0.4057060480117798, -1.295233130455017, 1.1831409931182861, -0.5997735261917114, -0.623455822467804, -0.5338872671127319, -0.8428363800048828, 2.3567962646484375, -0.6982102990150452, -1.061210036277771, -1.1423310041427612, -1.1170228719711304, -0.2555326223373413, 0.04690369591116905, 0.3113369047641754, 0.1876620203256607, 0.28030312061309814, -0.5934411287307739, -1.5520150661468506, 0.6427127718925476, 0.23476609587669373, -0.038590941578149796, 1.5579509735107422, -1.4182298183441162, -0.7177584171295166, -1.5064623355865479, -2.2230241298675537, 0.39488694071769714, 0.4035133719444275, 0.21902374923229218, -0.9017246961593628, -0.6085389256477356, -0.2592564523220062, -1.4391727447509766, -0.85012286901474, 0.6344037652015686, 1.3982734680175781, -1.143915057182312, -0.643643856048584, 0.6427295804023743, 1.6165400743484497, 0.7801370024681091, -0.014694889076054096, 0.28177687525749207, 0.25612229108810425, -0.3200167715549469, 1.5725784301757812, 0.5679291486740112, -1.5530669689178467, 1.4341742992401123, 0.9586102366447449, 0.08589856326580048, 1.6466840505599976, -0.6095415353775024, 1.5640523433685303, -1.0389856100082397, -0.38375309109687805, 1.1717828512191772, -1.9099940061569214, -0.05520673468708992, -0.4058477580547333, -0.17104032635688782, 1.3649470806121826, 0.08534135669469833, -1.0404527187347412, 0.289458304643631, 2.726511240005493, 0.9096208214759827, -1.3762116432189941, 1.1929020881652832, -0.18932119011878967, -0.9722862243652344, -0.9702873826026917, 0.5612490177154541, 0.8135853409767151, -1.9865316152572632, 0.6573002934455872, 0.6040297150611877, 0.42871150374412537, -1.2931734323501587, 0.3471733033657074, -0.5453943610191345, 0.2743503749370575, 0.014492207206785679, -0.11095554381608963, 0.20594483613967896, -0.6894884705543518, 0.4977487325668335, 0.8486867547035217, 0.7294444441795349, -0.12252195924520493, -0.37110865116119385, 0.02605205588042736, -0.7064234018325806, 0.948189377784729, -0.2729986906051636, 0.46440887451171875, -1.3439195156097412, 0.24883176386356354, -0.024135302752256393, 1.4709618091583252, -1.197235107421875, 0.2581517994403839, 0.9662642478942871, -0.478615403175354, -0.6944121718406677, 0.5610899925231934, -0.1969962865114212, -1.7235502004623413, -1.1233481168746948, 0.3596709370613098, -1.3477157354354858, 0.30643996596336365, -1.0067607164382935, 1.1554884910583496, 0.06551964581012726, 0.3717227280139923, 0.8782934546470642, 1.022026777267456, -0.8930422067642212, 1.0961103439331055, -3.5559232234954834, 1.106075406074524, -0.8960849046707153, 1.7639018297195435, -0.07848872244358063, -0.25993630290031433, 0.17557883262634277, -1.4927648305892944, -1.4230256080627441, 0.02961759828031063, 1.0825010538101196, -2.4608206748962402, -0.541899561882019, 0.8050770163536072, -0.51882404088974, 0.14703579246997833, -0.30311617255210876, 0.4154966175556183, -0.6368823051452637, -0.8063617944717407, 0.5274030566215515, -0.1636686772108078, -0.5808550119400024, 0.04826779291033745, 0.15234458446502686, 1.1598892211914062, -1.746803641319275, -0.7454457879066467, -1.0055642127990723, 0.9662182927131653, -0.4605902135372162, -0.4412010610103607, 0.2672530710697174, -0.4802719056606293, -0.6399369239807129, 1.3565332889556885, -0.10218953341245651, 0.1876583844423294, 1.0152779817581177, -0.8216578364372253, -0.745130181312561, -2.309849739074707, -0.7491199374198914, -0.5243222117424011, -0.1768801212310791, -1.3312655687332153, 1.0135327577590942, 0.2913424074649811, -0.7668014168739319, -1.6069507598876953, 0.8965272903442383, 1.740151286125183, 0.3965655565261841, -0.6584431529045105, -0.6083385944366455, 1.1003482341766357, -0.09808055311441422, -0.1823369264602661, 0.031153665855526924, -2.2102348804473877, -1.4902966022491455, -2.488175868988037, 0.7930947542190552, 0.7817862629890442, -1.0944194793701172, 1.455021858215332, -0.9934503436088562, -1.4077619314193726, 0.46085870265960693, -0.48265594244003296, 0.07984039932489395, 0.6051987409591675, -1.3200265169143677, -0.350705623626709, -0.7127047181129456, 0.6371615529060364, -0.9175736308097839, 0.014604547992348671, -0.7099545001983643, -1.1417770385742188, 0.8232999444007874, 0.1684935837984085, -1.4750882387161255, -0.18379241228103638, -0.9916931390762329, -1.6085232496261597, 0.6201745271682739, 0.6292389631271362, 0.46860581636428833, -1.3192082643508911, 0.9603283405303955, -0.31830450892448425, 0.7769414782524109, 0.26517021656036377, 0.7128582000732422, 0.8645691275596619, 0.7043336033821106, 0.2548978328704834, -0.5073229670524597, -0.10882525891065598, -0.237169548869133, -0.5614045262336731, 0.98475182056427, -1.2036091089248657, 0.5689210891723633, -0.0020248647779226303, 0.23406654596328735, -0.24580471217632294, 0.07429869472980499, 1.9167424440383911, -1.5274807214736938, -2.2204062938690186, 0.33228665590286255, -1.4554554224014282, -0.05369883030653, -0.18025483191013336, -0.7983827590942383, 0.3666272759437561, 0.19519323110580444, 1.9206563234329224, -1.2597229480743408, -0.7019484043121338, -0.3470567464828491, 0.8789441585540771, 0.3047633171081543, -1.5519812107086182, 0.944449782371521, -1.010603904724121, 2.261876106262207, -2.2596724033355713, 1.080251932144165, -1.9822264909744263, 0.6527746319770813, 1.086063265800476, -0.5414279699325562, 0.2964969277381897, -1.1958048343658447, 1.0647321939468384, -1.0125797986984253, 0.7335697412490845, -0.782301664352417, -0.5297145843505859, 0.05532480403780937, -0.12043359875679016, -0.17620237171649933, -0.6783472299575806, -0.7334506511688232, -0.11398281157016754, 1.4048712253570557, 0.6398946642875671, -1.4227700233459473, -1.3724727630615234, 0.369686484336853, 0.0634651631116867, -0.21369920670986176, 0.27148476243019104, -1.1115609407424927, -0.7125666737556458, -0.6658418774604797, 0.9209929704666138, 2.4076550006866455, 1.0527902841567993, -0.36442241072654724, -0.9153894186019897, 0.5535255074501038, 0.8467704653739929, 0.1339743584394455, 0.354190856218338, -0.02805796079337597, -0.5990405678749084, 0.504930317401886, -0.8429555296897888, 0.0660737007856369, -0.01084514707326889, -0.5389451384544373, -0.07930673658847809, 0.3382096290588379, -0.39290934801101685, 0.2901090383529663, -1.1693041324615479, 1.925313115119934, 0.9592505693435669, 0.08112172037363052, -0.8938422203063965, -0.6465240120887756, 0.3703056275844574, 1.5839608907699585, -0.47318035364151, -1.836310625076294, 1.9166496992111206, -0.5420798659324646, -0.5830391049385071, 0.7870090007781982, -1.2081084251403809, -0.21505311131477356, 1.4764933586120605, 0.312574177980423, -0.047515060752630234, -0.31589800119400024, -0.9528552293777466, 1.9038561582565308, -1.6556528806686401, -0.3062187731266022, -0.22493241727352142, -0.907331109046936, 0.28683674335479736, -0.6361597180366516, 1.293838381767273, -0.255910724401474, -0.35047435760498047, -0.40650275349617004, -0.033695291727781296, -0.33868157863616943, -1.5655616521835327, -0.02562965825200081, -0.03487886115908623, 0.10422756522893906, -1.1711944341659546, 1.875754475593567, 1.212072730064392, -0.5808896422386169, 1.7122931480407715, -0.11573652923107147, 1.9526394605636597, 2.257279872894287, 0.6943727731704712, 1.0507367849349976, -0.4745624363422394, -2.508044958114624, -0.814120352268219, -0.43872585892677307, -0.47220367193222046, -1.7920725345611572, 0.7871478199958801, 2.781796932220459, -1.202989101409912, -0.19708040356636047, -1.099361777305603, -1.843382477760315, 0.33331388235092163, 0.5402995944023132, -0.885801374912262, -0.3710035979747772, -0.997169554233551, 0.0024089126382023096, -2.2141988277435303, -0.8974506855010986, -1.8644686937332153, 0.9397814869880676, 1.7925893068313599, -0.48394957184791565, -1.1543093919754028, -0.4358232021331787, 2.0275256633758545, -0.02191140316426754, 0.6389852166175842, -1.165461778640747, -1.1857542991638184, 0.43690454959869385, 0.3001821041107178, 1.732847809791565, -0.1796533316373825, -0.6159264445304871, 1.1256314516067505, -0.23119957745075226, 0.4216614365577698, -0.8080677390098572, 0.549565851688385, 0.4302736818790436, 1.6536710262298584, -1.4792625904083252, 1.1804019212722778, -1.2594356536865234, 0.9352270364761353, -1.3152495622634888, -0.03597576916217804, -0.7730731964111328, -0.9944462180137634, -2.064174175262451, 0.6125374436378479, -0.21360231935977936, -0.22196710109710693, 0.6945615410804749, 1.5159108638763428, 0.9030791521072388, 1.262488603591919, 1.0508525371551514, 0.8494786024093628, 0.37092772126197815, -0.08479554206132889, -0.38785693049430847, 0.14362692832946777, 0.4313761293888092, 0.954393744468689, -0.8925088047981262, 0.46732819080352783, -0.33062776923179626, 1.2971808910369873, -0.32569077610969543, -0.5407565832138062, -0.7987908124923706, -0.9959868788719177, 0.5339998006820679, -1.4169280529022217, 0.9158463478088379, -1.4033143520355225, 1.712972640991211, 1.7009742259979248, -0.8923738598823547, 1.7771070003509521, 2.3089990615844727, 1.5697646141052246, 1.2658933401107788, 0.05748028680682182, 0.7312555909156799, 0.026401987299323082, -1.0798063278198242, 1.5942955017089844, -0.7583380937576294, -0.4011353850364685, 0.8255029320716858, -0.49183863401412964, -0.9631792306900024, -1.3640512228012085, 1.1632124185562134, 0.3162471055984497, 0.8743681907653809, 0.22364991903305054, 1.354414939880371, 1.1137956380844116, 0.7398661375045776, 0.28171780705451965, 1.258202314376831, -1.6435141563415527, -1.6629952192306519, -0.3081551790237427, 1.3534497022628784, -1.0715439319610596, -2.8151721954345703, -0.9597530364990234, -1.5022965669631958, 0.11981060355901718, -1.082901954650879, 0.16447843611240387, 0.20029796659946442, -0.1938055157661438, 0.30278533697128296, 1.075947642326355, -0.3685705363750458, -0.3461858928203583, -0.11887681484222412, -2.196467161178589, -1.1500095129013062, -0.20907482504844666, 0.46098819375038147, 0.8241497278213501, -0.22942423820495605, 1.9493955373764038, 0.4778846204280853, -0.07928227633237839, -0.7678669095039368, 1.2821094989776611, 0.0852418765425682, -0.17697764933109283, 0.20399880409240723, 1.2781974077224731, 0.895110547542572, -0.70452880859375, -0.31459665298461914, 0.767667829990387, -0.5850801467895508, -0.678341269493103, -1.5466285943984985, -2.4508919715881348, -1.4302760362625122, -0.6688274145126343, 0.9254693984985352, -1.8200024366378784, 2.0914831161499023, 0.18351995944976807, -0.5208891034126282, 0.401838093996048, 0.4761073589324951, -1.149185061454773, 1.2649067640304565, 0.11269684880971909, 0.453900009393692, -2.0512068271636963, 0.26734426617622375, 0.6527813076972961, -0.8613741993904114, -0.333281010389328, 0.07247409224510193, -1.0204330682754517, 2.1141364574432373, -0.7258902788162231, -1.275865912437439, -1.550552487373352, -0.21024341881275177, -1.0194342136383057, -1.049997091293335, 0.8420556783676147, -0.43336939811706543, 0.3524596393108368, -0.4655190706253052, 0.4317302703857422, 0.4045310914516449, -0.6513120532035828, -0.41222870349884033, -1.0605040788650513, -0.35350510478019714, 0.7467662692070007, -0.5312930941581726, 0.6277761459350586, 0.04894724115729332, 1.1787316799163818, -1.2527750730514526, 0.2763284742832184, 1.3105242252349854, -1.0403112173080444, -0.5168576836585999, 2.583557605743408, -1.2677465677261353, 1.372470736503601, -3.1042404174804688, 0.39495187997817993, 1.4322589635849, 2.5858678817749023, -0.6366145014762878, -0.1894579976797104, 0.2498461753129959, -0.7150080800056458, 1.1889373064041138, 0.8353549242019653, -0.08301232010126114, 0.24665983021259308, -0.06847669184207916, 1.595875859260559, 1.781639814376831, 0.3952581584453583, 1.1646205186843872, 0.2267797738313675, 0.35788196325302124, -0.09589516371488571, 1.8784576654434204, 1.7991158962249756, 0.025258339941501617, 0.9212508201599121, -1.227323293685913, -0.6699211597442627, 0.37587231397628784, 0.5780501365661621, -0.13729000091552734, 0.32153356075286865, -0.6819702982902527, 2.0103468894958496, 0.7872607111930847, 0.8871850371360779, 0.8005924820899963, -0.4334615468978882, 1.825750470161438, -0.7614580392837524, -0.16366127133369446, 0.5618195533752441, -0.24788570404052734, 0.47024640440940857, -0.18010547757148743, -0.4853843152523041, -1.4115519523620605, 1.284001350402832, -0.45971158146858215, 0.6553525924682617, -1.1109017133712769, -1.6808421611785889, -0.571489155292511, 0.9040006399154663, 0.8480691909790039, 0.2461642473936081, -0.28798291087150574, -1.4504114389419556, -0.5675292611122131, 0.41002514958381653, 0.18671701848506927, 0.7726417183876038, 0.18100687861442566, -0.40623804926872253, -0.14412716031074524, -1.2282518148422241, 0.6582440733909607, 0.6410747766494751, -0.889233410358429, 0.6045591831207275, 1.2137209177017212, 0.9004412293434143, -0.05633211508393288, 2.3313188552856445, 0.9773199558258057, -0.595444917678833, 1.1533054113388062, -0.5796295404434204, 1.1844755411148071, 1.313356876373291, -0.9489713907241821, 1.7288432121276855, -1.2431511878967285, -0.802288830280304, 0.49884170293807983, -2.2097606658935547, -1.002887487411499, 0.43894264101982117, -0.9150248169898987, 0.5205292105674744, -1.7283813953399658, -0.29403313994407654, -0.04620634764432907, -1.1216801404953003, -0.5504702925682068, -0.7483740448951721, 2.1137402057647705, 0.24477262794971466, 0.22538326680660248, 0.3893938660621643, 0.7890833616256714, -1.0635676383972168, -0.8317187428474426, -2.0068612098693848, -0.3888278901576996, 0.8653792142868042, -0.30121731758117676, -1.0754683017730713, -1.5271859169006348, 0.7151139974594116, 0.6759175658226013, 2.1141810417175293, -2.2239441871643066, -0.2633274793624878, -0.10490182042121887, 1.4729931354522705, 2.131713628768921, 0.8455444574356079, 0.9290434122085571, 0.25992661714553833, -0.24383893609046936, -0.1128138080239296, -2.7879159450531006, -0.5507097244262695, -0.9476868510246277, 0.9641883373260498, 0.287847638130188, -1.4134151935577393, 0.0610191710293293, -1.12027108669281, -0.5659863948822021, -0.9191241264343262, 0.016022874042391777, 1.3167457580566406, -1.0470809936523438, 0.23680604994297028, 0.2895974814891815, -1.3500339984893799, 2.3722383975982666, 0.27291038632392883, 0.3840242326259613, -0.2019508183002472, 0.2223229557275772, -3.3696916103363037, -1.5543400049209595, -1.1309964656829834, -0.8574740886688232, 0.9711512923240662, 0.3182681202888489, -0.08765280246734619, 0.21911300718784332, 0.2320408821105957, 0.9733232259750366, -1.005670428276062, 1.4238182306289673, -0.4131334722042084, 0.023861927911639214, 0.19070270657539368, 0.37525686621665955, -1.0327752828598022, -0.6122449040412903, 0.47023409605026245, 2.19551157951355, 1.186959147453308, 0.7515267133712769, 0.6901777982711792, 0.6907468438148499, 0.6644260287284851, 0.051413219422101974, -0.6618682146072388, -0.8932139873504639, -0.24465210735797882, -0.10851594060659409, -2.2059295177459717, 0.12967945635318756, -0.7366434931755066, 1.6768118143081665, 0.33058294653892517, -1.0581135749816895, 0.09809517115354538, -0.16586366295814514, -0.9158087968826294, 0.5782490372657776, -0.0017048302106559277, -1.2596808671951294, 0.41516321897506714, -0.7048977613449097, 0.2598101496696472, -1.1507105827331543, 0.7606933116912842, 0.33115312457084656, 0.6265856027603149, -0.7048681974411011, 0.27919113636016846, -0.7663009166717529, 1.7368478775024414, 1.7874547243118286, 0.7106966972351074, -1.84063720703125, 1.527424931526184, 1.6243799924850464, -0.30984610319137573, -1.1817421913146973, 0.27827462553977966, 0.64278244972229, 1.1135063171386719, 1.2657921314239502, 0.204593688249588, -0.6697875261306763, 1.080790400505066, 0.5960716009140015, -1.3167880773544312, -0.9346091151237488, -0.03961397334933281, -0.443112313747406, -0.26042211055755615, -1.8678079843521118, 0.2543024718761444, -1.7066041231155396, -0.975915253162384, 0.11369464546442032, 0.4840972125530243, -1.6089591979980469, 1.063036322593689, 0.1315675526857376, -0.8503127694129944, 1.150760293006897, 1.0710400342941284, 0.4272426962852478, -0.5522418022155762, 0.838883638381958, 0.8392651677131653, 1.4413139820098877, 0.3247973322868347, 1.7969093322753906, -0.4885031282901764, -0.35887229442596436, -0.6827206015586853, 0.226175919175148, -1.4881017208099365, -0.10208100080490112, -0.11030590534210205, 0.229852557182312, 1.8015531301498413, -1.773032784461975, 0.7948908805847168, -1.0943267345428467, 0.26853182911872864, 0.2691837251186371, 2.041635274887085, -0.41065746545791626, -0.4559197723865509, -0.25499728322029114, 1.0728365182876587, 2.0750083923339844, 0.4064888656139374, 0.9859582185745239, -1.223641037940979, -1.253830075263977, -0.4109039008617401, 0.9602717757225037, 0.5525610446929932, 1.0340607166290283, 0.7746483087539673, -1.1495544910430908, 1.075048565864563, 0.5002442598342896, -0.08831502497196198, 0.4051864743232727, -1.2930552959442139, 0.5581578612327576, -0.08644377440214157, -0.8826344013214111, 0.13848254084587097, -0.8038729429244995, 0.39716070890426636, -0.2986484169960022, -0.4497549533843994, -0.7732686996459961, 0.02770267426967621, 0.083403080701828, -0.0036656013689935207, 0.5973237752914429, -0.6575968265533447, 0.2067776471376419, 0.5281956791877747, 1.2929354906082153, 0.006814439781010151, 0.6481274366378784, -0.5916716456413269, 0.4390505850315094, 1.6711416244506836, 0.3219001591205597, 0.9710335731506348, -0.06612088531255722, 0.6277226805686951, -1.2879961729049683, -0.7554566264152527, -2.2672410011291504, 1.373742938041687, 0.4465486705303192, 1.702184796333313, 1.211146593093872, -1.3337972164154053, 0.11252741515636444, 0.39903321862220764, -0.8081455826759338, -1.4013704061508179, -0.8762885928153992, -0.4716844856739044, 0.9540840983390808, 0.36456719040870667, 0.48546648025512695, 0.7332930564880371, 1.27729070186615, 1.294613003730774, -1.556922435760498, -0.5656256675720215, -0.4667649269104004, -0.43810003995895386, 0.06537765264511108, 1.7075178623199463, 0.6118050217628479, 0.36449146270751953, -1.4002947807312012, 0.061766672879457474, 1.272297739982605, -0.24537567794322968, 0.8317162990570068, -0.7785245776176453, 0.9980075359344482, 2.0220160484313965, 0.3248925805091858, 0.0811953917145729, 2.1098477840423584, -0.7894225120544434, 1.1191248893737793, -0.7444203495979309, -0.9684385061264038, 0.08170054107904434, -0.6353996992111206, 0.523846447467804, 1.5787054300308228, 1.2005735635757446, 0.5600738525390625, 1.201380729675293, -0.044173985719680786, 0.5249173641204834, 0.41132843494415283, -0.1892871856689453, -0.16201868653297424, -0.45754244923591614, 2.02156662940979, 0.08775424212217331, -0.024972591549158096, 1.2700947523117065, 1.802430510520935, -1.3895586729049683, 0.7247572541236877, -0.1811027228832245, 0.35734766721725464, -0.4064834415912628, 1.0178885459899902, 0.6879294514656067, -1.4061330556869507, -3.0282135009765625, 0.4587423801422119, 2.3584113121032715, -0.2217610627412796, -0.03034020960330963, -0.4383930265903473, 0.8519508838653564, 1.06565260887146, 0.38050535321235657, 0.8203054666519165, -0.09946773201227188, 0.04608921706676483, -0.3613508939743042, 0.4744032025337219, -0.6110962629318237, 0.11078372597694397, -0.051251135766506195, -0.5506389737129211, 0.10584671050310135, -1.651810646057129, -1.0642316341400146, 1.7330875396728516, 1.9756529331207275, -0.6599894165992737, -0.5786967873573303, 0.02042367309331894, -1.4472116231918335, 1.06504225730896, -0.4244190752506256, 1.7530101537704468, -0.8849203586578369, -0.4105681777000427, -1.0169459581375122, 0.7406904697418213, 1.3066424131393433, -0.3497263491153717, 1.0577203035354614, 0.16661621630191803, -0.6869953274726868, 0.6032674908638, -0.40490400791168213, -0.47719910740852356, 1.6205203533172607, -2.2800755500793457, 0.9434859156608582, 0.2758711874485016, -0.9053499102592468, -0.21092896163463593, -0.03200072422623634, 1.6784157752990723, -0.5827826261520386, 2.3698441982269287, 1.1919288635253906, 1.4536985158920288, 0.29384228587150574, -0.24182246625423431, 1.5903899669647217, -0.25036191940307617, 1.664711833000183, -0.8143125176429749, 0.3931110203266144, 1.407106876373291, -1.3171465396881104, 0.39777234196662903, 2.534435987472534, -0.7679638266563416, -0.3586835563182831, 0.9965854287147522, 0.8704673051834106, 0.14238665997982025, -1.115552306175232, -0.4450366199016571, -1.4431155920028687, 0.6046440601348877, 0.8947920203208923, 1.8669716119766235, 1.1402217149734497, 0.6927502751350403, 0.2957986891269684, 0.5107227563858032, -0.8403217792510986, -1.9085257053375244, -0.4515501856803894, 1.146864891052246, -0.9999996423721313, -0.7859811782836914, 0.6341858506202698, -0.19969502091407776, 0.12485253810882568, -1.8189035654067993, -0.3760894238948822, 0.2011152058839798, 0.5772738456726074, -0.3124695122241974, 0.8066365718841553, -0.2949941158294678, -0.0018232801230624318, -1.3141517639160156, -1.0560221672058105, -0.7705692648887634, -0.4795590341091156, -0.6857545375823975, -1.554133415222168, -0.06736240535974503, 0.09596719592809677, -0.09501186013221741, 0.24991688132286072, -0.449939489364624, 0.10142185539007187, 1.2485193014144897, -0.7544330954551697, -0.5780093669891357, -0.3799952268600464, -0.9079689979553223, -0.9461547136306763, -2.1970431804656982, 1.5852253437042236, 0.2768799960613251, 0.40798449516296387, -1.6783674955368042, -1.4998475313186646, -0.9905278086662292, 0.21320736408233643, 1.5452582836151123, 0.4055574834346771, 1.2397822141647339, -0.24291661381721497, -0.8505743741989136, 1.0726031064987183, -0.9310960173606873, -1.760286808013916, -0.9220479726791382, 0.4089326560497284, -0.1768600046634674, -1.376309871673584, 0.6269092559814453, 0.3743656873703003, -1.0352219343185425, 0.4004093408584595, 1.7468700408935547, -0.1657118797302246, -0.3250844180583954, -0.23749567568302155, -2.6749234199523926, -0.03781669959425926, -0.5625927448272705, 0.3333025574684143, 0.14988352358341217, 0.3642124533653259, 1.2627818584442139, 0.09899093210697174, 0.3708204925060272, 0.27877870202064514, 0.49796754121780396, -0.2603239119052887, -0.67985999584198, -0.3666709065437317, -0.45291954278945923, -0.04403296858072281, -1.1501230001449585, 0.32621994614601135, 0.640303373336792, -1.155733346939087, -0.9639037251472473, 0.31613197922706604, 0.7667715549468994, 0.2384985089302063, -0.7137004137039185, -1.1355805397033691, -0.3334707021713257, 1.4219428300857544, 1.480186104774475, -1.0950751304626465, -0.7179675102233887, -1.5975128412246704, -0.2840617597103119, -1.6839613914489746, -0.2523292899131775, 0.8962588310241699, 0.5465864539146423, -0.07073843479156494, 0.2962610423564911, 1.0289275646209717, -1.3415322303771973, -0.3142181932926178, -0.20297814905643463, -0.9072359800338745, -0.16027802228927612, -0.6029139161109924, 0.5351830124855042, 0.6169222593307495, -1.0785152912139893, 0.5939874649047852, -2.011949062347412, -1.7201610803604126, 1.2551918029785156, -0.7996290326118469, -0.7295342087745667, -1.2247180938720703, 1.5461101531982422, 0.2832980751991272, -0.6818252205848694, -0.6806960701942444, -0.7974622249603271, 0.3943706452846527, 1.0453472137451172, 1.9423909187316895, -1.1026374101638794, -2.261807680130005, -0.19045542180538177, -0.1769774705171585, 0.7108449339866638, 0.5241783857345581, -0.2920132279396057, -0.2943180501461029, -0.16343292593955994, -0.4525113105773926, 0.7431599497795105, 1.2345237731933594, 0.2653626501560211, -0.6429775953292847, 0.6913976669311523, 0.16851924359798431, 0.4603458642959595, 0.15518565475940704, 0.1178271546959877, 0.047759123146533966, -0.0729432925581932, -0.6412533521652222, -1.8136235475540161, 1.4340766668319702, 3.1281685829162598, 0.791304349899292, 0.4311690926551819, -0.675032377243042, 0.5743566751480103, -0.33998242020606995, 0.10438305139541626, -1.403315544128418, -1.0712766647338867, 0.16686242818832397, 1.0039057731628418, -1.153670072555542, -1.161189079284668, 0.7341358065605164, 0.7933739423751831, 0.6797381639480591, -0.22555992007255554, -1.1587523221969604, 3.134108066558838, 0.3481084406375885, -0.21764132380485535, 0.014040393754839897, 1.5965018272399902, 1.1279847621917725, -2.1642000675201416, 0.11044955253601074, -0.2130645364522934, -1.4389622211456299, 0.6367708444595337, -0.3990260064601898, 0.7686477899551392, 0.3667528033256531, 0.02110004983842373, 0.3030413091182709, -2.255730628967285, 1.5298856496810913, -0.6704134941101074, 1.3462679386138916, -0.26417621970176697, -0.6071406006813049, 1.7917561531066895, 0.8884717226028442, 0.7270151972770691, -0.3123084604740143, 0.13036035001277924, -0.7798649072647095, 1.1056517362594604, -0.454963356256485, 0.05893068388104439, 2.1478617191314697, -1.5192469358444214, 1.130638837814331, 0.6044739484786987, -0.276969313621521, -0.6997732520103455, -0.32764655351638794, -0.4071425497531891, -1.181900143623352, -0.7473438382148743, -1.4351226091384888, 0.4692220985889435, -0.13570401072502136, 1.193986415863037, -1.5528533458709717, 0.35929134488105774, 0.5792850255966187, -0.12025433778762817, -0.1554318219423294, -1.4827672243118286, 1.4106051921844482, -0.35855525732040405, 0.428821861743927, 0.09540899097919464, 0.3821422755718231, -0.6929584741592407, 0.590745747089386, -0.8484736680984497, -0.3990749716758728, 1.2650084495544434, 1.096339464187622, -0.029472796246409416, 0.4449084401130676, 0.6776967644691467, -1.987182855606079, -1.628143310546875, -2.335627555847168, 0.7515929341316223, -1.9861116409301758, 0.3830621540546417, 1.6430847644805908, -0.5293012261390686, -0.30643516778945923, -0.23924458026885986, 0.20536008477210999, -0.10088443756103516, 0.2606201767921448, -0.09232991188764572, 1.1604361534118652, -0.8264966607093811, 1.0589163303375244, 0.41863784193992615, -0.4880385994911194, 0.7384400367736816, 0.18285682797431946, -0.30511996150016785, 0.6416831016540527, -0.45348840951919556, 0.8378787636756897, -1.1754270792007446, -0.48508739471435547, 0.35354480147361755, 0.35785776376724243, 1.5036462545394897, -0.36330658197402954, -0.38078615069389343, -0.6378902792930603, -0.4195098280906677, 0.47297176718711853, -0.46471428871154785, 0.11600695550441742, -0.28952163457870483, -0.4257255792617798, -0.2795793414115906, 0.26617398858070374, 0.004708689171820879, 1.6049684286117554, -1.9779096841812134, 0.1635035127401352, -0.957222580909729, -0.9420126676559448, 0.15896685421466827, 0.4227313697338104, -1.5935245752334595, 0.006972834002226591, 1.3186264038085938, -0.7408990263938904, -0.5562582612037659, 1.3031162023544312, 0.9980000257492065, -1.0647780895233154, 0.5007311105728149, 1.2526205778121948, 0.5867376923561096, 1.1631561517715454, 1.053725004196167, 1.3322545289993286, -0.6860234141349792, 0.20160090923309326, 0.19280730187892914, 0.4551222026348114, 1.0645781755447388, -0.3215624988079071, -0.47926855087280273, -1.2486181259155273, 1.173741340637207, -1.575113296508789, -0.1255715936422348, -0.45593497157096863, -1.0230423212051392, -0.2046455293893814, 0.6760103702545166, -1.0263336896896362, -0.5818238854408264, 0.6280326247215271, -0.858620822429657, -0.8346045613288879, -0.27130642533302307, 0.5199066400527954, -0.14454881846904755, -0.4790292978286743, 2.5686264038085938, -0.15944986045360565, -0.7564836144447327, 0.23007310926914215, -0.07439620792865753, -0.3241782784461975, -0.17315445840358734, 0.030569851398468018, 2.034716844558716, 0.3594142496585846, 0.9380983114242554, 0.8959278464317322, 0.0006584072834812105, -0.35489901900291443, 1.1201328039169312, 0.4627949893474579, 0.5292212963104248, -0.7440697550773621, 0.12453004717826843, -0.2295307219028473, 0.20726388692855835, -2.4365346431732178, 0.014522735960781574, -0.9807660579681396, -1.2588205337524414, 0.45570623874664307, -0.37225955724716187, 1.2131881713867188, 0.9733099341392517, -2.0680460929870605, 0.6239498257637024, 0.9228850603103638, 1.1418503522872925, 1.0428190231323242, -0.9280883073806763, -1.046255111694336, 0.6933891773223877, 0.8231002688407898, 1.2286266088485718, -1.5761960744857788, -0.28819945454597473, 0.01154905091971159, -1.0844969749450684, -1.8077919483184814, 0.08403987437486649, -0.34344998002052307, -0.8689941763877869, -0.4828479588031769, 0.4999019205570221, 0.3134070634841919, 0.424074649810791, -0.8103258609771729, 0.30418238043785095, -0.43456757068634033, -1.237434983253479, 0.3202511668205261, -2.093526840209961, 1.541380763053894, -2.1288769245147705, 1.1918596029281616, 0.14430180191993713, -0.11786363273859024, -0.3548825681209564, 0.7032022476196289, -0.6393623352050781, 0.3765011429786682, 1.0499553680419922, 0.6259576678276062, 0.26655179262161255, -1.237326979637146, 1.6447968482971191, -0.9627799391746521, 1.0523892641067505, -0.44347020983695984, -0.699559211730957, -1.1806769371032715, -0.03941237926483154, -1.0971317291259766, -0.33637163043022156, -1.372243881225586, 0.10322361439466476, 0.5859001278877258, -0.6408506035804749, -2.0037319660186768, -0.16136321425437927, -0.14703573286533356, 1.0817064046859741, 0.17389674484729767, -0.01165724080055952, 1.545899748802185, -1.428237795829773, -0.5327655673027039, -1.439850926399231, -0.6908761858940125, 0.22464439272880554, 2.4035608768463135, 1.964420199394226, 0.6283630728721619, -1.8444318771362305, 0.9162834286689758, -0.30917859077453613, 1.2499032020568848, 0.350909024477005, 1.6391522884368896, 1.2210739850997925, -0.5590938925743103, -0.0203484445810318, 0.29581764340400696, -0.11497890204191208, 0.021268736571073532, -1.048805594444275, 0.36841124296188354, 0.0874037966132164, -0.9169947504997253, -0.9462246298789978, 0.29257509112358093, -1.3975123167037964, -0.22477543354034424, -1.249607801437378, -0.5244134068489075, 1.4244905710220337, -1.02961003780365, -0.25052982568740845, 0.17636790871620178, -1.325116515159607, -0.4011284410953522, 0.7431352734565735, -0.01003176998347044, -0.5284082293510437, 0.7874566316604614, 0.3723333179950714, -0.6993884444236755, 0.9572686553001404, -1.0867780447006226, -0.7138023376464844, -1.2336630821228027, -0.3861274719238281, -1.0185085535049438, 0.07727358490228653, 0.9822806715965271, 2.034878969192505, -1.233953595161438, 1.0361140966415405, -1.500536561012268, -0.24143379926681519, -1.3059699535369873, -0.05431780964136124, 1.0716636180877686, -0.6112006306648254, -1.294620156288147, -1.9550241231918335, 1.8593778610229492, 1.2178832292556763, -0.8393946886062622, -0.5148955583572388, 1.470463752746582, 1.4756975173950195, -0.598400354385376, 0.018193434923887253, -1.0925068855285645, -0.7937745451927185, -1.649871587753296, 0.7033820748329163, 0.6899157166481018, 0.15104961395263672, 1.1497234106063843, -0.3215196132659912, -1.2503914833068848, -0.4369530975818634, -2.5347299575805664, 0.8960772156715393, -0.2722012400627136, -1.925036072731018, -1.7944930791854858, 0.3139151632785797, 0.1367577463388443, -1.722324013710022, -0.3688488006591797, 1.8134922981262207, -1.176651954650879, -0.8023220896720886, -0.8326976299285889, -0.3546386957168579, -1.2673972845077515, 0.11147645115852356, -0.7666692733764648, -1.1980340480804443, -0.8272708654403687, 0.4449828863143921, 0.7477933764457703, 1.6934905052185059, -1.0318217277526855, 0.628421425819397, -0.32730862498283386, 0.3589165508747101, -1.625266194343567, -0.7915661334991455, -0.774205207824707, -0.04695383086800575, 0.029350226745009422, 1.2252672910690308, 0.771772563457489, -0.6360740661621094, 0.16986750066280365, 0.622911274433136, 1.6421586275100708, 0.5821094512939453, 0.5691099166870117, -0.2803221046924591, 0.49365299940109253, 1.706708312034607, -1.4445873498916626, 0.7353304028511047, -0.6168292164802551, -0.13358567655086517, 0.19140642881393433, 1.0069562196731567, -0.8013560771942139, -1.7830084562301636, 1.066190242767334, -0.027390971779823303, 1.0753123760223389, 0.8272650241851807, 0.2529458999633789, -0.3477003276348114, 0.5442684292793274, -0.8487011194229126, 0.13833773136138916, 0.5631434321403503, 1.9278067350387573, -0.6265807747840881, 0.3701501786708832, 2.3643851280212402, -1.3773889541625977, 1.4299144744873047, -0.41003623604774475, 0.9775771498680115, 0.23124325275421143, 1.3027377128601074, 0.8891304731369019, 0.3872040808200836, 1.5466221570968628, 0.19622772932052612, -0.5789610147476196, -0.5408321619033813, -0.7606496810913086, -0.03212858736515045, 1.6324422359466553, 0.3302019536495209, -1.005501389503479, 0.45714884996414185, -0.9935617446899414, 0.9943996071815491, 1.1999034881591797, 1.6675409078598022, 0.334801584482193, -0.02086602710187435, -0.8012638092041016, 0.42013418674468994, 0.41190996766090393, -0.15427079796791077, 1.4663472175598145, -0.7736912965774536, 0.21634703874588013, 1.2907514572143555, 0.2849767208099365, -0.5245658755302429, -0.5027918219566345, 1.3628615140914917, 0.6102184653282166, 0.9903652667999268, 0.48590874671936035, -0.17818208038806915, 0.1811428964138031, 0.7575129866600037, -0.3038765788078308, 0.6457641124725342, 1.2486988306045532, -0.43293389678001404, 1.138418197631836, -1.5607553720474243, -0.022548530250787735, 1.0959893465042114, -0.7975254058837891, -1.7115989923477173, 0.3100420832633972, 0.8271135091781616, 0.1249890923500061, -0.24488301575183868, 0.20612122118473053, -0.4376996159553528, -1.3789877891540527, 2.534416437149048, -0.9793274998664856, -0.7253037095069885, -0.30356428027153015, 1.3247967958450317, 0.34858036041259766, -0.5254010558128357, 2.461974859237671, 0.1254037618637085, 0.2678675949573517, 0.5148994326591492, -1.1326978206634521, -0.19250187277793884, 0.42378130555152893, 1.3332054615020752, -1.5994905233383179, -0.4139588475227356, 1.2354822158813477, 1.1763640642166138, 2.1086437702178955, 1.0004240274429321, -1.0232794284820557, 0.5703347325325012, 0.9825250506401062, -0.2754272222518921, -0.8160892724990845, 0.12565675377845764, 0.6493649482727051, -0.33172357082366943, -0.15475411713123322, -1.6314960718154907, 1.5564614534378052, -0.2926847040653229, -0.0016745672328397632, 0.9830960035324097, -0.13337844610214233, -0.6909094452857971, -0.8177845478057861, -1.510478138923645, -1.0360102653503418, -0.22079536318778992, 1.101203203201294, -0.1700933575630188, 1.701465129852295, 0.4575653374195099, -1.2498191595077515, 0.9585534334182739, 1.2343411445617676, 2.470977306365967, 0.10739370435476303, -0.2771458625793457, 1.8038086891174316, 0.7072688341140747, 2.708101987838745, -1.5351632833480835, -0.25231701135635376, -0.7605172395706177, 2.318281888961792, 1.4690946340560913, -1.3444620370864868, 1.196345567703247, -0.9609801173210144, 0.14922232925891876, -0.7623453736305237, 0.5041873455047607, 0.3974144458770752, 0.9944525957107544, 1.437509298324585, -0.7033067345619202, -0.23027484118938446, 0.9491626620292664, -1.4899309873580933, -0.3849809169769287, 1.242087960243225, 0.7048360705375671, 0.27593109011650085, -1.0353775024414062, -0.23454514145851135, -0.6416984796524048, 0.7873752117156982, 0.6504190564155579, -0.4898918569087982, 0.4937428832054138, -1.167069673538208, 1.1452165842056274, 1.2553664445877075, 0.405071496963501, -0.6666014790534973, 1.3294378519058228, -0.5404450297355652, 1.2403181791305542, -0.238127201795578, -0.17247824370861053, -1.2854478359222412, 0.275760680437088, 1.68947434425354, 1.2550747394561768, 1.109067678451538, -1.3575241565704346, -0.6372829675674438, 1.2922496795654297, 0.9594156742095947, 1.060634970664978, -0.009776972234249115, -0.6805746555328369, 0.03347768262028694, -0.5730747580528259, 0.2932738661766052, 1.5031753778457642, 2.2759647369384766, -1.0375614166259766, 1.7763569355010986, 0.5633466839790344, -0.5288499593734741, -1.3161699771881104, -0.4493285119533539, -0.7247872352600098, -0.876992404460907, -0.9449825882911682, -0.44451993703842163, -1.1217049360275269, 0.45094603300094604, 0.48271286487579346, 1.2334845066070557, 1.0737271308898926, -0.11025529354810715, -0.5758066773414612, -1.4585334062576294, 1.7723891735076904, -0.3938802182674408, 0.7089120745658875, -1.0731709003448486, -0.4628922641277313, 0.6936323046684265, -0.36860930919647217, 1.9324390888214111, 0.41010943055152893, -0.3684430420398712, -2.1950185298919678, 0.3598107099533081, -0.7632084488868713, -0.22249099612236023, 0.8876082897186279, -0.526117205619812, -0.004534604027867317, 1.4566391706466675, -0.06675496697425842, -0.06477871537208557, -0.44971463084220886, 0.5777572989463806, 2.4124629497528076, -0.863251268863678, -0.3503323793411255, 1.4942437410354614, 0.4259326159954071, 0.4667762517929077, 0.6309655904769897, 0.17086976766586304, 2.2428531646728516, 1.01876962184906, -0.536018431186676, 1.9948227405548096, -0.7299755811691284, 0.973302960395813, -1.2625844478607178, -0.3587746322154999, 0.5701470971107483, 0.28038012981414795, -0.38679370284080505, -1.928119421005249, -0.09923727810382843, 1.9473321437835693, -0.3455623686313629, -0.2257629930973053, 2.1756463050842285, -0.6248461604118347, -0.6872132420539856, 1.7028582096099854, 0.05090048536658287, -1.2940483093261719, 0.27912378311157227, -0.3527749180793762, 0.0501607321202755, 1.3573319911956787, -1.2152808904647827, 0.07644466310739517, -0.5207544565200806, 0.5233524441719055, 0.11475180834531784, 1.8422592878341675, -2.1393535137176514, 0.8149755001068115, 1.3453468084335327, 0.07608731091022491, -0.27395835518836975, 0.9184823632240295, -0.4105870723724365, -0.5884656310081482, -0.4267667233943939, 1.2694474458694458, -0.6169157028198242, -1.8181575536727905, -0.9246299862861633, 0.2875545620918274, -0.31558483839035034, 0.5013203024864197, -1.3956440687179565, -0.06145312637090683, -0.003907799255102873, 1.2221908569335938, -1.964030385017395, 0.5108187794685364, 0.8626433610916138, -0.33941859006881714, 1.463762640953064, -0.04491676390171051, -0.8037453293800354, 0.10028571635484695, 0.25131407380104065, 0.1046033576130867, 0.8462238311767578, -0.6801717877388, -0.48731064796447754, -1.2727258205413818, 1.1668633222579956, 0.96290522813797, -0.46003517508506775, -1.9992661476135254, -0.8609085083007812, -0.43265587091445923, 0.35190480947494507, 0.34155258536338806, -0.6727880239486694, 1.1068131923675537, -0.16916704177856445, 1.1831066608428955, 0.45707133412361145, -0.15894576907157898, -2.049647331237793, 0.7863186001777649, -3.2413461208343506, 0.06197880208492279, 0.22301340103149414, 0.11222993582487106, -0.44170284271240234, 0.5578520894050598, 0.0051557160913944244, 0.22669617831707, -0.8470784425735474, 2.6812386512756348, -0.2534448206424713, -0.8934271335601807, -3.043384075164795, -0.8102848529815674, -0.5546286702156067, -0.39169371128082275, -0.5328618884086609, -0.614704966545105, -1.7606550455093384, 0.6260918974876404, -0.511013388633728, -0.8567072153091431, -0.39552855491638184, 0.5302726626396179, 1.3506171703338623, -0.7816520929336548, -0.39999186992645264, 1.2777990102767944, -0.9763404726982117, -1.2588101625442505, -0.6204612851142883, -0.19728249311447144, -0.7718077898025513, -0.6276760697364807, -1.062499761581421, 0.2410438507795334, 0.8438162207603455, 0.6475152373313904, -0.35730859637260437, 0.6504347324371338, 0.6953313946723938, 0.4515989124774933, -0.06717979162931442, 0.7870124578475952, 0.7689995765686035, -0.2990228235721588, -2.6763877868652344, -1.5593549013137817, 0.299198716878891, 0.11196336150169373, 0.6320275664329529, -0.6627816557884216, -0.3201979100704193, -0.4078701436519623, 0.26579490303993225, 0.19911949336528778, 0.7531279921531677, -0.01792812906205654, 0.5433894395828247, -1.7561918497085571, -0.503783643245697, -1.3560302257537842, 0.6663506627082825, 1.3162637948989868, -0.2870771884918213, 0.16538089513778687, -1.1636720895767212, 1.6931339502334595, -0.6985588669776917, -0.871552586555481, 0.700860321521759, 0.6978945136070251, 0.958655834197998, 1.3679672479629517, 0.18090181052684784, 0.7155697345733643, 0.3408255875110626, -1.1651216745376587, -0.897547721862793, 0.6769218444824219, -1.6765177249908447, -0.014626707881689072, 0.07194582372903824, -0.3833138644695282, -1.5618419647216797, 1.3270716667175293, 0.6846039295196533, 1.128012776374817, -0.8362880349159241, 0.15720494091510773, 1.673801302909851, -0.5301207304000854, 0.6295411586761475, -0.2621328830718994, -1.1283693313598633, -0.5412077903747559, -0.7743422985076904, 1.0306167602539062, 0.43152010440826416, 0.6233744621276855, 0.31361523270606995, -0.5732442736625671, -0.7115731835365295, 1.0705533027648926, -0.3085397779941559, 0.7335559129714966, 1.5457795858383179, -1.0343713760375977, 0.8354618549346924, -0.3052828907966614, -0.32954224944114685, 1.9310591220855713, -0.2692732512950897, 0.8632999658584595, -0.05920855700969696, -0.7463107705116272, -0.2847479581832886, 1.356146216392517, -0.1646479070186615, -1.1555780172348022, -1.1523704528808594, 0.7255103588104248, 1.2403864860534668, 0.9594244956970215, 0.2755489647388458, 1.256047248840332, -0.5218316912651062, -1.4913551807403564, -0.07829535752534866, 1.2418336868286133, -1.0171656608581543, -0.5230825543403625, 0.9036886096000671, -1.2053073644638062, 0.19231082499027252, 0.8540761470794678, -0.26218998432159424, -0.8437603712081909, -0.6650815606117249, -0.8486688733100891, 0.4634507894515991, -0.013838998973369598, -0.173085018992424, 1.2005010843276978, -0.9931164979934692, -0.72017902135849, 0.09003114700317383, 0.47889187932014465, -1.7469528913497925, -0.8663192391395569, 0.3817947506904602, -1.3012158870697021, 1.519557237625122, -1.0143886804580688, 0.5440391302108765, -0.955878734588623, 0.12525680661201477, 0.45149654150009155, 0.3708769381046295, 0.9780797958374023, 1.8391127586364746, 0.6089087724685669, -1.653369665145874, -0.5449702739715576, 1.5320488214492798, -0.18954753875732422, 0.37946462631225586, 0.1511961817741394, 1.1835495233535767, 1.4061962366104126, -2.033357620239258, 1.5047829151153564, -1.296998143196106, -0.26061320304870605, -0.8963896632194519, -0.42358437180519104, 0.6611037254333496, 0.8170055150985718, -1.1973317861557007, -1.5191835165023804, -0.3496803939342499, -2.3554611206054688, 0.8140817880630493, 2.795011043548584, 1.2231377363204956, 0.21772503852844238, -0.36155882477760315, -0.740461528301239, 0.27347663044929504, 1.2879574298858643, -0.22193969786167145, 1.9894566535949707, 0.032320283353328705, 0.14503949880599976, 1.0440914630889893, 1.394135594367981, -1.3301671743392944, 1.4822211265563965, 0.4761432111263275, -0.5378766655921936, -1.0552138090133667, 0.4383697509765625, 0.5578620433807373, -0.5152650475502014, -0.3794867694377899, -1.0231993198394775, -1.0322134494781494, 3.0974600315093994, 1.1478853225708008, -0.1357845515012741, 1.0356510877609253, 0.5199613571166992, -0.8740324974060059, 1.3240337371826172, -0.596027672290802, -0.68434739112854, 2.168619394302368, -0.14869804680347443, -0.8862593770027161, 2.08262300491333, -0.7946241497993469, 0.5199093222618103, -1.2586617469787598, -0.4274293780326843, -0.07183361053466797, 1.6464354991912842, -1.722166895866394, -0.6949527859687805, -0.16902366280555725, 1.0877950191497803, -1.2206183671951294, -1.7395365238189697, 0.23342546820640564, 1.376222848892212, 1.1104927062988281, 0.25221818685531616, 3.6702206134796143, 2.2278311252593994, 0.3408547341823578, -1.035923719406128, -1.255348801612854, 0.617256224155426, 0.6253831386566162, 0.6423888802528381, 0.49851691722869873, 0.12611761689186096, -0.17959371209144592, 0.3884720206260681, -2.302110195159912, 0.6582791805267334, 0.3380180895328522, 0.3646596670150757, 0.7786508202552795, 0.61783766746521, -0.48089298605918884, 0.6264335513114929, 0.038844380527734756, 1.6967639923095703, -1.2981046438217163, -0.21896597743034363, -0.5643683671951294, 0.6200298070907593, -1.2432186603546143, -0.017213625833392143, 1.5719190835952759, 2.7959275245666504, -0.3726234436035156, -0.37510156631469727, -0.7994305491447449, 0.09199879318475723, -1.4711971282958984, 0.9751523733139038, 0.19337378442287445, 0.7225459218025208, -0.7249372601509094, -0.8997471928596497, -1.0862222909927368, 0.7563069462776184, -0.7342328429222107, 0.4217277467250824, -1.3380430936813354, -1.7268385887145996, -0.7649097442626953, 1.2348912954330444, 0.07894615083932877, -1.6840381622314453, 2.044395923614502, -2.1400389671325684, 0.031678155064582825, -0.7408701181411743, 0.40216004848480225, -0.8411579728126526, -1.5221247673034668, -0.4394473731517792, 2.0244204998016357, -1.2293345928192139, -0.6093220710754395, -0.0008296399028040469, 0.2305501103401184, -0.570305585861206, 0.6052795648574829, 0.0015641521895304322, 0.12678265571594238, -0.7720968127250671, -0.015234998427331448, -1.7958533763885498, 0.8000398278236389, 0.42820850014686584, -0.83376544713974, 0.621190071105957, -0.2679407596588135, -0.7410861253738403, -0.9539160132408142, -0.4786660373210907, 0.2818620800971985, 0.5366746187210083, 0.8441104888916016, -0.2672695815563202, 0.10332192480564117, -1.5590322017669678, -0.37696245312690735, 0.15123170614242554, -0.6456809043884277, 0.046325329691171646, -1.1398568153381348, 0.4718289375305176, -0.7266249656677246, 0.9231131076812744, 0.03162195906043053, -1.1527372598648071, -1.0647834539413452, 0.44689154624938965, 0.21728792786598206, -1.4086315631866455, 1.6782344579696655, -0.6570447683334351, -1.158164381980896, 0.27949321269989014, 0.7796469330787659, -0.5010991096496582, 0.41360926628112793, 0.2029533088207245, 0.28035977482795715, 1.7422164678573608, -0.3264416456222534, -0.45577794313430786, -0.2288384586572647, -1.2373508214950562, 1.1201869249343872, 0.48577582836151123, 0.40328916907310486, 0.23964335024356842, -0.7306256294250488, 0.9165680408477783, 1.1045842170715332, 0.7524717450141907, 0.9595418572425842, 0.48579999804496765, -1.2210075855255127, 0.46406304836273193, -0.4136159420013428, -0.9053933024406433, 0.3454262614250183, 2.0682108402252197, 0.45774781703948975, 0.27702796459198, -0.43130287528038025, 0.749455451965332, 1.1915791034698486, -1.1861673593521118, -0.2754637598991394, 0.11520084738731384, 0.6528342962265015, 0.9852731823921204, -0.5888655781745911, 1.5594005584716797, -0.24936163425445557, -0.27921241521835327, 0.9649639129638672, 0.05563630908727646, -1.035914421081543, -0.41476279497146606, 0.9712337255477905, 0.25979989767074585, -0.27609652280807495, -1.6988544464111328, 0.002741523552685976, 1.5901916027069092, -0.32986944913864136, -0.23967449367046356, 0.84154212474823, -0.836662232875824, -0.6692934632301331, -0.4184057116508484, 0.15446527302265167, 0.08396481722593307, -1.1032849550247192, -0.010458125732839108, -0.438958078622818, 0.1906706690788269, -0.2722245454788208, 1.088044285774231, 0.40524035692214966, 0.5550844669342041, -0.9247559905052185, -0.06379976868629456, -1.5762056112289429, 0.031440433114767075, -0.5628055930137634, 0.6298127174377441, 0.40622612833976746, 0.7447313666343689, 2.0161566734313965, 0.3177698850631714, 0.4326159954071045, 0.7639074325561523, 1.426374912261963, 1.8102470636367798, -1.4929436445236206, -0.039047278463840485, 1.9384301900863647, -0.42678454518318176, -1.9342666864395142, -0.5213217735290527, 1.2184109687805176, -0.3540600538253784, 0.03348105773329735, 0.8403457403182983, 1.3863662481307983, 0.7634012699127197, -0.6023206114768982, 1.3137136697769165, 0.1178710088133812, -0.28334400057792664, -1.0317295789718628, 0.12730801105499268, -1.0336711406707764, -1.1280573606491089, 0.5725212693214417, 0.38337624073028564, -0.7818073034286499, 0.4121401607990265, -1.1581263542175293, 0.16534124314785004, 0.40295833349227905, 0.22270789742469788, 0.1468798667192459, 1.462750792503357, 0.7183717489242554, 0.3142862617969513, 0.36312583088874817, -0.22945082187652588, -0.09610231220722198, -0.15918181836605072, 0.8500967621803284, -0.1620989888906479, -0.31062284111976624, 0.6484886407852173, -0.8181110620498657, 1.4264531135559082, -1.5020499229431152, 0.231667622923851, 0.9184993505477905, -0.05003824084997177, -0.5937507152557373, 1.7850658893585205, 0.5709445476531982, 0.8458922505378723, 0.3563827574253082, -0.08852467685937881, 0.3407605290412903, 1.0397920608520508, -2.0462424755096436, 0.31450122594833374, -1.7153006792068481, 1.1877297163009644, -0.2680361866950989, 1.1824100017547607, -0.8619260787963867, 0.5409569144248962, -1.496433973312378, 0.4136059582233429, 0.45217591524124146, 0.7520020008087158, 0.26687684655189514, 0.9156268835067749, 1.1360552310943604, 1.2809345722198486, 0.016051005572080612, 0.8239408135414124, 1.5777803659439087, -1.884560465812683, 0.3190913200378418, 0.5272056460380554, 2.6637051105499268, -0.8708820939064026, -1.1996558904647827, 1.0203731060028076, 0.29386335611343384, -0.7247790098190308, 2.083979368209839, 0.1993207335472107, 0.7610859870910645, -0.3781522512435913, 1.364074468612671, 0.6372409462928772, 0.026245776563882828, 0.33688536286354065, -0.4319300055503845, 0.933664858341217, -0.3952043652534485, -0.6265718340873718, -0.8548734188079834, 0.5335668921470642, 0.06502655148506165, -0.21068574488162994, 0.4713963270187378, -1.3321748971939087, 0.5875551104545593, 1.5411614179611206, 1.348689079284668, 0.6867627501487732, -0.9257664680480957, -0.947293221950531, -1.4216623306274414, -1.1494035720825195, -0.37172213196754456, -0.7314538955688477, -1.613173484802246, 0.01989719085395336, 1.211184024810791, 1.5733293294906616, 1.4075506925582886, 0.09453428536653519, 0.22943785786628723, -1.4821540117263794, -0.3996221423149109, 0.2462148815393448, 0.8780655860900879, 1.5054876804351807, 0.7880776524543762, -2.0088677406311035, 0.9978735446929932, 0.94488126039505, 1.0502535104751587, -0.8928417563438416, -0.5755413770675659, -0.5056210160255432, -1.099950909614563, 0.6047642827033997, -0.34171223640441895, 0.6298813819885254, 0.015178358182311058, 0.7703589797019958, -0.04632921889424324, 0.9925364255905151, -0.14336207509040833, -0.165119007229805, -1.8109779357910156, -1.3450255393981934, -0.003169272094964981, -0.8966689705848694, 0.4358408451080322, -0.6411654353141785, -0.3135198950767517, -0.21020998060703278, -0.8386995792388916, -0.897221028804779, -0.01564549095928669, 0.9247234463691711, -0.5713450908660889, -2.164457321166992, -0.1122591644525528, -2.4322168827056885, 1.2058173418045044, 0.04073517769575119, -0.2361249476671219, 0.6126158237457275, 0.0009136372245848179, 0.09653656929731369, 1.4465190172195435, -1.2900779247283936, -0.06098465248942375, -1.4129799604415894, 0.5920870900154114, 1.0765079259872437, -0.7096331715583801, 0.7941653728485107, -0.07869461923837662, 1.1644803285598755, -0.3988569974899292, -0.19432976841926575, -0.433255136013031, 0.46961724758148193, 1.6575340032577515, 0.49446526169776917, -0.5961765050888062, 0.5963838696479797, -0.6893106698989868, 0.08769360184669495, -1.1895287036895752, -0.8019394278526306, 0.8845449090003967, 0.5271785855293274, -0.25673970580101013, 0.7350863814353943, 0.44258734583854675, 0.44998931884765625, -0.3869055211544037, 0.5483517050743103, 1.328089952468872, -1.346968650817871, -1.885107159614563, 1.7103005647659302, -0.34433308243751526, 0.3443905711174011, -0.3109767735004425, 1.2929563522338867, -1.2561780214309692, 0.6778720021247864, -1.622196078300476, 0.6378244161605835, 1.008054494857788, 2.680474042892456, 0.47038331627845764, 0.5246702432632446, 0.7753859758377075, -0.1143021211028099, 0.2920742332935333, -0.6147037148475647, 1.5538369417190552, 1.0805869102478027, 1.335429072380066, 0.08305158466100693, 0.16952717304229736, 0.6064606308937073, 0.617500364780426, -0.7685449123382568, 0.7309862971305847, 1.7807241678237915, -0.8354142904281616, 0.989281177520752, 2.228691339492798, 0.5078545212745667, 0.2976088523864746, -0.3774113655090332, 0.04627034068107605, 2.102619171142578, -0.29702794551849365, -0.9931852221488953, -1.030918836593628, -1.9984824657440186, -0.41974037885665894, 1.181277871131897, -0.45938941836357117, -0.29727959632873535, 0.2686702013015747, 0.11901871860027313, 1.1061168909072876, 0.0875585526227951, 1.535495638847351, 0.49866464734077454, -0.29383426904678345, 1.4020018577575684, 0.8026214838027954, -0.7868015766143799, 0.4480525553226471, -0.253633588552475, -0.8534179329872131, 0.124837726354599, 0.5091502070426941, 1.0094857215881348, -1.776323914527893, -0.738372266292572, 1.8187209367752075, 2.1089444160461426, -0.4515528976917267, -0.050177667289972305, -0.3539041578769684, 0.6746595501899719, -0.900236964225769, 0.11476331204175949, 0.9450810551643372, -0.5799586772918701, -1.0349879264831543, 0.9323189854621887, -0.7582215070724487, -0.41548529267311096, 0.45143768191337585, 0.40192219614982605, -2.3535032272338867, 1.6378839015960693, -0.21713963150978088, -0.19687919318675995, -0.4752199351787567, -0.30210426449775696, 1.3700141906738281, -0.16944605112075806, -1.0533050298690796, 1.9664037227630615, -0.17215421795845032, 0.5114354491233826, 1.6001566648483276, -0.9216254353523254, 1.379796028137207, -0.008648824878036976, -0.3529062867164612, -0.7838969826698303, -1.0391207933425903, -3.167116641998291, -1.879581093788147, 1.605473279953003, -1.6101881265640259, 1.3820912837982178, 0.5281310081481934, 0.23135602474212646, 0.31610581278800964, 0.16881394386291504, -0.16855871677398682, 0.04302171617746353, -1.5780688524246216, -0.3638704717159271, -0.6870109438896179, -0.28594881296157837, 0.12290041893720627, -0.2431933432817459, 0.9743960499763489, -0.08622457832098007, -0.11298161000013351, 0.12688769400119781, -0.683276355266571, -1.9398797750473022, -1.3848448991775513, 1.4256376028060913, 0.5882443189620972, 1.154033899307251, -1.3895446062088013, 0.0021792356856167316, 2.6045849323272705, 0.6630249619483948, 1.267127275466919, 1.0918411016464233, 1.7310343980789185, 0.6053180694580078, 0.25423794984817505, -0.7406628131866455, -1.058224081993103, 0.06877649575471878, -0.19294829666614532, -0.07004797458648682, 0.6929849982261658, 1.4733144044876099, -0.01263743918389082, -0.10012463480234146, 1.2095651626586914, 0.5249732136726379, -0.44694268703460693, -1.9724218845367432, -1.0369552373886108, 0.10715101659297943, 0.7993548512458801, -0.32460451126098633, 0.9250272512435913, -1.4406331777572632, -0.37334075570106506, -1.2388430833816528, 0.8852757811546326, -0.8744663596153259, -0.2794703543186188, -0.38523656129837036, -1.0517832040786743, -0.5730568766593933, -0.2795506417751312, 1.5146288871765137, 1.3839540481567383, 0.1318206936120987, -0.991919219493866, -0.41931238770484924, 0.3301841616630554, -0.7248775362968445, 0.5064091682434082, 0.21167141199111938, 1.1588627099990845, -1.118864893913269, 0.7936220169067383, 1.260799765586853, 1.8497246503829956, 0.5722052454948425, 0.21654266119003296, -1.0920522212982178, 0.6522209644317627, -0.4264663755893707, -0.16481280326843262, 0.6464192867279053, 0.011533479206264019, -0.42117050290107727, 0.3319924771785736, -0.34444910287857056, 0.7603710293769836, -0.02807515859603882, 1.5860041379928589, -1.3817280530929565, -0.9559005498886108, -0.05163330212235451], "dtype": "float32"}], "e": {"__nd__": [0.854844331741333, 1.0307554006576538, 1.5803930759429932, 0.059946417808532715, -0.8766785264015198, -1.3714146614074707, 0.0032558441162109375, -1.8014802932739258, -0.06786757707595825, -0.3254735767841339, -2.296443223953247, -1.2081589698791504, 0.674015998840332, -0.7122626304626465, 0.8513296842575073, 1.234709620475769, -0.8178479671478271, -0.9656855463981628, -0.34155210852622986, -0.8201762437820435, -1.770664930343628, 2.890620231628418, -1.0007052421569824, -2.052037000656128, 0.8609268665313721, -0.8173849582672119, -0.9617117643356323, -1.293806552886963, 0.26635193824768066, -1.112183690071106, -0.7070560455322266, -1.7970612049102783, 2.352078914642334, -1.529773473739624, -0.5495515465736389, -0.7370996475219727, -1.4017302989959717, -0.09274753928184509, -0.09049367904663086, 1.9866814613342285, 0.7098776698112488, 2.3762106895446777, -2.5090627670288086, 0.5509647130966187, 1.1623761653900146, -0.39353129267692566, 0.6289796233177185, 1.107129454612732, 0.27251315116882324, 0.3644663691520691, 1.9942272901535034, -2.7316057682037354, 1.3967782258987427, -0.45517697930336, -1.0207349061965942, 0.2889900207519531, 2.2096803188323975, 0.7898235321044922, -0.867529034614563, 1.0970968008041382, 0.5448706746101379, 2.4959988594055176, 0.41496628522872925, 1.4655753374099731, -0.2692450284957886, -1.9629056453704834, 0.835604727268219, 1.5244297981262207, -1.6743667125701904, -1.1187236309051514, 0.6874818801879883, 1.0115435123443604, 0.8664403557777405, 0.9042019844055176, 0.030882716178894043, -1.4588639736175537, 0.9358084201812744, -1.892809271812439, 0.8726638555526733, 0.3603893518447876, 0.23883044719696045, -2.819007396697998, 0.36878645420074463, -1.300061821937561, -0.8289110660552979, -0.5271334648132324, -2.1841721534729004, -0.2213757038116455, 0.9767342805862427, 1.9984591007232666, 2.9751288890838623, 0.2218654453754425, -0.20286589860916138, 1.3023909330368042, 1.6149908304214478, -0.31822773814201355, -2.2129106521606445, -0.19414861500263214, -0.8097235560417175, 0.8208777904510498, 0.6454359889030457, 2.5147509574890137, 1.3273370265960693, 1.2038962841033936, -0.6957751512527466, -1.2250256538391113, -2.1431643962860107, -0.6586078405380249, -0.4491736888885498, -2.410738468170166, 1.7764089107513428, -0.18454915285110474, -0.6401842832565308, -1.3268893957138062, -0.5363311171531677, 0.8508812189102173, -0.45179665088653564, 0.20367151498794556, -0.27622777223587036, 0.08614933490753174, -0.47598356008529663, -3.180065870285034, 1.5392422676086426, -2.864105701446533, 0.22116178274154663, 1.5611584186553955, -3.1836726665496826, 1.8756051063537598, 2.6374759674072266, -0.5142005085945129, -1.8151514530181885, -0.14504927396774292, -2.25005841255188, -2.423159599304199, -0.23928320407867432, 1.4832642078399658, -1.6900804042816162, -0.5584009885787964, 0.36633092164993286, 0.18909108638763428, -0.2846940755844116, -1.0919075012207031, 0.026989921927452087, 1.7109071016311646, -0.7483000159263611, 1.845876932144165, 1.1957192420959473, 0.46510398387908936, 0.858502984046936, -0.30267441272735596, 1.4107671976089478, 0.490534245967865, -2.529374122619629, 1.3726284503936768, 2.2821474075317383, -0.542793869972229, 0.4697480797767639, -1.2710293531417847, -1.1899170875549316, -0.32761937379837036, -0.1825857162475586, 0.2636414170265198, -0.5252906680107117, 0.23101772367954254, 1.196794867515564, -0.39349132776260376, 0.4272283613681793, -0.4765825867652893, -0.9176352620124817, 0.09492474049329758, -0.20170259475708008, 0.29673993587493896, 0.7158024311065674, -1.0560250282287598, -0.8966178894042969, -0.20504727959632874, 1.6031726598739624, -0.5592144727706909, -0.6735954284667969, 1.4933137893676758, 0.1701337993144989, 0.04658828675746918, -1.6002180576324463, 1.9264802932739258, 1.1482893228530884, 1.8977622985839844, 1.7610297203063965, 0.44959020614624023, -2.2744953632354736, 0.6439575552940369, -0.030075550079345703, 0.7607517242431641, -0.5633178353309631, -1.2947850227355957, 0.2225668728351593, -0.35481950640678406, -0.7582027912139893, -0.2475121021270752, -0.7464970350265503, 1.0305182933807373, 0.9832040667533875, -0.2279130220413208, 1.1349557638168335, -1.8063316345214844, 1.0010778903961182, 2.9229226112365723, 2.3401005268096924, -2.0041167736053467, -0.476268470287323, -0.5231583118438721, -2.352999210357666, -1.528228998184204, -1.2106072902679443, 1.1174509525299072, 0.13626527786254883, 1.5497465133666992, 1.5445098876953125, 0.7946195006370544, 0.46622467041015625, 0.7909939289093018, -1.7201793193817139, -1.2726271152496338, -1.918459177017212, -1.7233352661132812, 0.4605035185813904, -0.7837468385696411, 1.116624355316162, 1.065192699432373, 0.6428284645080566, 0.8900691270828247, 0.08571800589561462, 0.6113723516464233, -0.47786033153533936, 0.854958176612854, -1.1554553508758545, -2.439424991607666, -0.6567407250404358, -1.0512704849243164, 0.0973643958568573, -0.7072081565856934, -2.216681480407715, -1.0785799026489258, -1.96334969997406, 1.936945915222168, -1.5867937803268433, 0.2048068344593048, -0.38549911975860596, 2.363631010055542, -0.2664029002189636, -1.6481080055236816, 0.28033918142318726, 2.0746994018554688, -0.8902637362480164, 0.8874037265777588, -0.09119778871536255, -2.1676347255706787, 1.106482744216919, 3.373203754425049, -0.7157569527626038, 0.5368534326553345, 0.7968894243240356, -1.2735964059829712, 1.3276668787002563, 1.7053709030151367, -1.0063997507095337, -0.07494163513183594, -2.588338613510132, -2.474919319152832, -2.7824463844299316, -0.37851572036743164, -0.738879919052124, 0.7672378420829773, -0.2807926535606384, 0.07341897487640381, 0.877185583114624, 0.20496530830860138, 2.039191722869873, 0.3871980905532837, -3.0541422367095947, -1.3816959857940674, 0.25110292434692383, -0.5238029956817627, -3.191103935241699, -1.236646294593811, -0.6766555309295654, -0.537487268447876, 0.23913094401359558, -1.094346284866333, 0.756596565246582, -0.20660150051116943, -2.3670308589935303, 1.4625879526138306, -0.7158267498016357, 1.3232735395431519, -0.3162960708141327, -0.17771555483341217, -0.9207622408866882, 0.6647419333457947, 2.0757570266723633, -0.09942150115966797, 0.019319593906402588, -1.3490731716156006, -0.7642184495925903, -1.2187644243240356, -0.2905745506286621, 1.7059326171875, -1.0668655633926392, 0.148531973361969, 0.4957975447177887, -2.3203797340393066, -0.6124194860458374, 2.0787501335144043, 0.13663774728775024, -2.6974411010742188, -0.6186892986297607, 1.1263080835342407, 1.9000312089920044, -0.5767245888710022, -0.4641054570674896, 0.5688576698303223, -0.2899627983570099, -1.4255287647247314, 0.6520819664001465, -0.060441162437200546, -0.27066296339035034, -3.487771511077881, 3.3117334842681885, 1.2917532920837402, 2.0907533168792725, 0.8515945672988892, -0.6490100026130676, -0.4787522256374359, -0.661756157875061, 2.572803497314453, 0.6456522941589355, 1.7584559917449951, 2.081085681915283, -3.5292489528656006, -2.3870162963867188, 4.36126184463501, -1.0735121965408325, -0.594174861907959, 1.760535478591919, 2.2317874431610107, 3.6342461109161377, -0.1207369863986969, -0.7656457424163818, -2.9349677562713623, 1.0872890949249268, -0.9432055950164795, 1.445430040359497, -0.15513013303279877, -1.4470442533493042, -0.23054097592830658, 0.7828290462493896, 2.467177391052246, 1.349340558052063, 1.1326881647109985, -1.4496347904205322, -1.5509564876556396, 1.189896583557129, -0.06841516494750977, 0.6001342535018921, 0.6228034496307373, 1.812826156616211, 3.6093978881835938, 0.20585933327674866, 1.8644390106201172, 0.9518887400627136, 0.5620428919792175, -3.952052593231201, 2.9504711627960205, -0.2713591456413269, 1.8530322313308716, 0.08913145959377289, -1.8779630661010742, -0.7570152282714844, -0.895155131816864, 0.6793381571769714, 0.9284217953681946, -0.13796618580818176, -0.10334078967571259, 0.7533600330352783, -1.1643834114074707, -0.26752132177352905, -0.8271114826202393, -0.7737417817115784, 1.6168601512908936, -2.1091270446777344, -0.13774418830871582, 0.18963271379470825, -0.284941703081131, 1.1035218238830566, 0.5438500642776489, 1.6162569522857666, 0.9907546043395996, 0.06976202130317688, 1.0890575647354126, -0.6182290315628052, -2.073948383331299, 0.12243038415908813, -2.598999261856079, 1.84852135181427, 0.01482158899307251, 0.40639469027519226, 0.7496753334999084, 1.5596630573272705, 2.9386138916015625, -1.0511823892593384, -0.5652310252189636, -0.5282866954803467, 2.8756566047668457, -2.0586423873901367, -2.3232390880584717, -1.0371946096420288, -1.0357123613357544, 0.18387481570243835, -1.6293799877166748, 0.9466176629066467, -0.07365942001342773, 1.282304286956787, 0.47504597902297974, -0.4989299774169922, 0.9061029553413391, -0.13655495643615723, 1.6987874507904053, -0.02526078373193741, -0.36628448963165283, -1.3598487377166748, -0.7853257656097412, -1.6005984544754028, -1.0926461219787598, 1.6545038223266602, -0.8420935869216919, -0.3463243246078491, -0.27741098403930664, 1.0685499906539917, 0.7348281145095825, -0.7318764925003052, 2.692983865737915, -1.2167437076568604, -1.4888639450073242, -0.06995582580566406, 1.8948473930358887, -0.8974316716194153, -2.6625280380249023, -1.05588960647583, -2.1624152660369873, -0.9475323557853699, 0.875684916973114, 3.2822108268737793, -0.2727152109146118, -0.5227157473564148, 0.131011962890625, -0.7010414004325867, -1.2550886869430542, -2.9737110137939453, -0.9500492811203003, 1.7379027605056763, -1.023145079612732, -0.18085551261901855, -0.783125638961792, 0.4852009415626526, -1.0798077583312988, -2.128814458847046, 1.258245825767517, 0.9670014381408691, -1.0914267301559448, -1.2523969411849976, 1.1991908550262451, 1.4480562210083008, -0.6434765458106995, -1.0641613006591797, -1.414190411567688, 0.29023751616477966, 1.9612270593643188, 0.10471177101135254, 0.08534324169158936, -0.8373788595199585, 0.4524998068809509, -1.1342452764511108, -1.9800937175750732, -0.8150248527526855, -0.7841173410415649, -0.7885987162590027, 1.455088496208191, 1.4482334852218628, -1.8137248754501343, -0.507731020450592, -0.24286437034606934, -0.3295065462589264, 0.8944921493530273, 1.7788972854614258, -0.4599340260028839, 1.1096638441085815, -0.9372955560684204, -0.5728963017463684, -1.8008747100830078, -0.3646145761013031, -0.6601090431213379, 1.724820613861084, 0.9254469871520996, -1.0224931240081787, 0.17069673538208008, -2.2121641635894775, 1.708077311515808, -1.6217670440673828, 0.06669342517852783, -1.7859785556793213, 0.4800718426704407, 1.905969262123108, 0.06584441661834717, 1.158428430557251, 0.31314271688461304, -0.2739492654800415, 1.3299866914749146, 2.3982715606689453, -3.1261210441589355, 0.7096966505050659, -0.7579714059829712, -0.4888821840286255, 2.722499370574951, -2.4138922691345215, 0.9719225168228149, 0.7209159135818481, -2.8986597061157227, -1.650347113609314, -1.4860244989395142, -0.21705317497253418, 0.12764686346054077, -0.6776745915412903, 0.644140362739563, -2.3745903968811035, 1.8338685035705566, -1.806183099746704, 0.17272713780403137, 4.397848129272461, 0.3417888879776001, -0.21423938870429993, 2.796069383621216, -1.9343914985656738, -0.7964837551116943, -2.3432817459106445, -1.1813052892684937, -0.534563422203064, -1.040696144104004, -0.3973105549812317, 0.08918106555938721, 1.8595526218414307, -0.20352262258529663, -0.4547865092754364, -0.8081106543540955, 1.0675346851348877, 1.6719805002212524, -1.4536206722259521, 1.8468140363693237, 0.6734814047813416, 0.10751373320817947, 4.196948528289795, 1.3386332988739014, -0.7362732887268066, 1.277906060218811, -0.15607106685638428, 1.04934561252594, 2.0687923431396484, -2.259105920791626, -0.3908112049102783, -1.1495122909545898, 0.9171791672706604, 0.18237978219985962, -1.5839942693710327, 0.2525733709335327, -0.12235260009765625, -0.6255770921707153, -1.579561471939087, 2.0166842937469482, -2.7137930393218994, -2.8517980575561523, 0.4855300784111023, 1.2762932777404785, 0.5116696953773499, 1.337064504623413, -1.1776357889175415, 0.4882526993751526, 1.0770748853683472, 0.9822502732276917, 1.128627896308899, -1.3770242929458618, 1.0517768859863281, 1.6787919998168945, -0.7146468162536621, 0.08178788423538208, -0.8091580271720886, -0.7465186715126038, 1.486494541168213, -4.090168476104736, 1.1106656789779663, 0.11131654679775238, 0.2926076054573059, 1.1919713020324707, -1.8237552642822266, -0.614713191986084, 2.5691957473754883, -0.4971504211425781, 2.7501611709594727, 0.46390554308891296, -1.9665194749832153, 2.0122909545898438, 1.0078951120376587, 0.0456274151802063, -0.10411501675844193, -0.7312544584274292, -0.20547932386398315, 1.0549935102462769, -0.7989733815193176, 0.7971009016036987, 0.05051437020301819, -1.4042366743087769, -2.2527689933776855, -1.336681842803955, 1.4080095291137695, 0.8207920789718628, 1.5018020868301392, -0.5317259430885315, 0.978717565536499, -0.6282007098197937, 0.23468828201293945, -1.2533427476882935, -0.9047980308532715, -3.6340925693511963, -2.440666675567627, -0.7845504283905029, -1.6952751874923706, -2.405496835708618, -2.2673516273498535, -1.1459288597106934, 1.6736177206039429, -1.0726141929626465, 0.2865738868713379, -1.4731510877609253, -1.652604341506958, -1.1478612422943115, -0.7479501962661743, -0.21530592441558838, -0.45141685009002686, -0.9162689447402954, -0.41170257329940796, 2.352752923965454, -3.169360876083374, 1.118332028388977, -1.0139033794403076, 1.5223729610443115, -2.238178014755249, 0.5511162281036377, -0.6349749565124512, 0.3798672556877136, -1.3160791397094727, -0.01969325542449951, -3.2142605781555176, -1.5065330266952515, -2.6231272220611572, 0.6599296927452087, -2.7334554195404053, 1.829094648361206, 0.9026108384132385, -0.29324066638946533, 0.8275289535522461, -0.9846143126487732, -0.05708611011505127, 0.8184133172035217, -1.4006863832473755, 0.71662437915802, -0.7789103984832764, -0.08992744982242584, -2.046952486038208, -1.2439342737197876, -0.8891726136207581, 0.9113795757293701, -3.07706356048584, -0.4868294894695282, -2.3130760192871094, -2.2477777004241943, -0.045939892530441284, 1.4456114768981934, 0.5840004682540894, -0.7368189692497253, -0.294376939535141, -0.27346348762512207, 0.5623587965965271, -1.6063650846481323, 1.000101923942566, 1.6552748680114746, 0.26755720376968384, -0.03191572427749634, 1.687583565711975, -1.2194229364395142, -0.9732452034950256, 0.5587215423583984, 0.9390755295753479, -1.7197729349136353, -2.178074359893799, -0.15500958263874054, 0.610718846321106, 2.532677412033081, -0.05867147445678711, -1.1675039529800415, -2.8554506301879883, 3.036328077316284, 2.5288543701171875, -2.639737129211426, -0.5265721082687378, -0.7850898504257202, 2.4350061416625977, -0.3986046314239502, -1.0731861591339111, -1.116438865661621, 0.8637906908988953, -2.389435052871704, 0.4839208424091339, -1.2604478597640991, -0.37537258863449097, 2.4232051372528076, -0.5835658311843872, 0.027248263359069824, 0.7950869202613831, 2.048440933227539, -0.41418951749801636, 0.18745017051696777, 1.317973017692566, -1.4809973239898682, -0.7878581285476685, -1.951958179473877, -0.13897007703781128, -0.004675388336181641, -0.03725993633270264, 2.882189989089966, 0.5461708903312683, -2.1282317638397217, 1.2321637868881226, -0.4973810315132141, -1.5216193199157715, 0.9592094421386719, 0.8290420770645142, -0.35636594891548157, -0.3732675015926361, 0.5892919898033142, 0.4041164815425873, -1.0387479066848755, -0.33921870589256287, -0.3985157907009125, -0.5322855114936829, 1.3847012519836426, 0.25909969210624695, -2.677107334136963, -1.2542598247528076, 0.7813376784324646, -0.0058203041553497314, 0.3889472484588623, -0.8899288177490234, -0.8427397608757019, -0.5901301503181458, 1.7945330142974854, 2.7467052936553955, 0.11843138933181763, 0.10279139876365662, 1.0981576442718506, -2.9763472080230713, -0.8447403311729431, -0.3260306417942047, 1.4422876834869385, 0.4283255636692047, 0.14712509512901306, -1.219842553138733, -1.1023766994476318, 0.857938826084137, -0.19025357067584991, 0.8834750056266785, -1.122774362564087, -1.3994061946868896, 1.3718154430389404, 0.7886571884155273, 0.5519869923591614, 1.5963654518127441, -0.715884804725647, 1.4432209730148315, 3.163177013397217, -1.8710846900939941, 0.998540997505188, -0.7117125988006592, 1.6249806880950928, -3.8373284339904785, -2.108285427093506, -0.691314697265625, 0.07276728749275208, -0.37827688455581665, 0.48094457387924194, -0.10837668180465698, 1.1194500923156738, 1.5472760200500488, -0.6151361465454102, 1.2979345321655273, -2.069913625717163, -2.2699291706085205, 0.3564951419830322, 0.36436140537261963, -0.541295051574707, 0.8323793411254883, -2.805330276489258, -0.2562906742095947, -0.9009308218955994, -0.5135625600814819, -0.4588227868080139, -1.7109185457229614, -0.7270300388336182, -2.114164352416992, -0.7797811031341553, -0.6635357737541199, 0.06959785521030426, -2.418806552886963, 2.0569252967834473, -1.6842637062072754, 1.1962802410125732, 0.3971250057220459, 0.3923187851905823, 0.9126158952713013, 0.20740938186645508, -1.3633806705474854, 3.884190082550049, 1.1718672513961792, 0.26249563694000244, 2.0642318725585938, -0.5343920588493347, 0.5153341889381409, 1.5016131401062012, 2.713858127593994, 1.0132025480270386, 0.4714205265045166, 1.3652923107147217, -1.0501381158828735, -2.0730397701263428, 0.579889714717865, 1.5067896842956543, -1.5088216066360474, -0.46189820766448975, 0.16050206124782562, -1.924050211906433, -1.2208613157272339, 1.7377808094024658, -0.04493147134780884, 1.9972141981124878, -1.7409050464630127, 2.189202070236206, 1.7371331453323364, 1.9748926162719727, 1.4882149696350098, -0.2704467177391052, 1.0319794416427612, -0.652389407157898, 0.08613812923431396, -2.3054041862487793, -0.9780771136283875, 0.12143734097480774, 1.6080042123794556, -0.9398676753044128, 1.790310263633728, 0.7241467237472534, -1.0188689231872559, 0.2967681586742401, -0.372213214635849, 1.907505989074707, 2.0634961128234863, -1.4031932353973389, 0.23119568824768066, 1.3291165828704834, -1.4303042888641357, 0.9998094439506531, -2.4914631843566895, -0.7089613080024719, 0.4538459777832031, -0.8518908619880676, 0.31375426054000854, 0.8441212177276611, -0.1652970016002655, -0.0874335765838623, -0.701357901096344, -0.7665862441062927, -0.6348797082901001, -0.6262962818145752, 1.8940975666046143, 0.22964999079704285, -3.6190805435180664, 0.8698994517326355, -3.5505356788635254, -0.03241121768951416, 0.22435694932937622, -1.0356853008270264, 0.8769680261611938, -0.29201599955558777, -1.9639496803283691, 1.7811838388442993, 0.9515600800514221, -1.651760220527649, 1.8604018688201904, 1.2568823099136353, 2.446930408477783, -1.4581782817840576, -1.608870029449463, -2.0992674827575684, 1.642221212387085, -2.9210894107818604, -0.3771679401397705, -1.9856626987457275, 1.3518797159194946, 0.6877565979957581, -0.5704182982444763, 0.2830922305583954, -0.6451812982559204, 0.46474558115005493, -0.02754800021648407, -0.016977235674858093, 2.1171886920928955, -0.45896705985069275, -1.279595136642456, -0.41939008235931396, 1.1718699932098389, -1.2770047187805176, 2.675983190536499, 0.6567745804786682, 0.8770864605903625, 0.19249916076660156, 0.6030004024505615, 0.7330297827720642, -0.5761641263961792, -0.5749574899673462, -0.46957671642303467, 1.183067798614502, 0.4852102994918823, -2.61385178565979, 1.5336674451828003, 0.6685823202133179, -0.4331876039505005, -0.29493647813796997, -0.41608601808547974, 0.9083269238471985, 0.975997805595398, 2.16292142868042, -0.9403448104858398, -0.7718878984451294, -2.2648682594299316, -0.8631628155708313, 1.700523853302002, 1.3774511814117432, -0.19642719626426697, -0.09240251779556274, 0.35018977522850037, -0.6777848601341248, 1.5197527408599854, -0.4172477722167969, -0.22287040948867798, 2.3730080127716064, -0.4964144229888916, 2.1992063522338867, -0.9012719392776489, -2.48630690574646, -0.990593671798706, 0.5894171595573425, 2.2597713470458984, -0.9761561155319214, 0.3690735101699829, 0.08500408381223679, -1.6851441860198975, 2.006633758544922, -2.0729479789733887, -0.8878470063209534, -1.5832180976867676, 2.0694780349731445, 0.17096787691116333, -0.7075681686401367, 0.4334127604961395, -0.4899231791496277, -1.5226075649261475, 2.211564540863037, 0.2898646891117096, -1.4839818477630615, -0.6874213218688965, -0.7795810699462891, 0.3524753451347351, 0.9861915111541748, -0.20325565338134766, -0.17819592356681824, -1.0367851257324219, -1.5507793426513672, 1.4943138360977173, 2.6380252838134766, -3.021057367324829, 0.6978982090950012, 2.8423922061920166, 0.34439727663993835, -0.5041934847831726, 0.2719789743423462, -0.5063893795013428, -0.5853517055511475, -0.8285403251647949, -0.8900975584983826, -0.7532929182052612, 0.10067135840654373, 2.678941249847412, 1.0094614028930664, -2.544895648956299, 1.0043861865997314, -0.9322766065597534, -0.7191844582557678, -0.06620658934116364, -0.32341596484184265, 2.304278612136841, -0.5904099941253662, -0.5320818424224854, -1.281002402305603, -0.1038292795419693, -0.25993186235427856, 0.46764862537384033, -0.06576335430145264, 1.3507577180862427, -0.6042313575744629, 2.5735254287719727, -0.4488198161125183, 0.6618169546127319, 1.6802692413330078, 0.711536169052124, 0.5764287710189819, -2.335143566131592, 0.3466072082519531, 0.46664565801620483, -0.3345777690410614, 0.17173653841018677, 0.17448687553405762, 0.9669584035873413, -0.781633734703064, -1.5165941715240479, -1.4608358144760132, -1.427479863166809, -1.1674448251724243, -0.990115225315094, 0.5555393099784851, 1.6026570796966553, 0.14944809675216675, 0.5553073883056641, -2.1672654151916504, -1.3705486059188843, -0.2505266070365906, -1.6704720258712769, 1.7048957347869873, -0.670952558517456, -0.45470982789993286, -0.3444504737854004, -1.1715025901794434, 0.8506653308868408, 1.044184923171997, -1.4086908102035522, -1.2796231508255005, -0.5347933769226074, -1.672412633895874, -0.524913489818573, 2.3576035499572754, 0.7791180610656738, 1.8225393295288086, 2.5855026245117188, -3.383822441101074, 1.1289924383163452, 2.4957637786865234, 0.6177335977554321, 0.6994105577468872, -0.2575254738330841, 3.0840375423431396, -2.350245237350464, 1.665573000907898, 0.27699729800224304, -1.9471845626831055, -0.006303071975708008, 0.9654499292373657, 0.8901242613792419, 0.6778002977371216, 0.8168642520904541, 1.804863691329956, -0.7307834625244141, 0.7363892197608948, 0.2518473267555237, -0.21653085947036743, 0.043005943298339844, -0.36352604627609253, 0.7499339580535889, 0.052088141441345215, 0.6906629800796509, 1.0339792966842651, 2.538984775543213, 1.4791918992996216, 1.2278482913970947, -1.0272901058197021, 1.6877691745758057, -1.2194280624389648, -1.0170385837554932, -0.5534176230430603, -0.39025774598121643, -1.6575616598129272, 0.23607268929481506, -1.0889838933944702, -0.6717140674591064, 1.2136681079864502, -1.6058549880981445, -0.9048048257827759, 1.1667566299438477, -1.5746140480041504, -2.4568076133728027, 0.4846990704536438, 2.0453851222991943, -0.693715512752533, -0.7113187313079834, -0.15213221311569214, 0.7621464729309082, 1.6672499179840088, 1.4987988471984863, -1.4173805713653564, 1.7267216444015503, 0.9309909343719482, 0.7878369688987732, 0.06744730472564697, -1.1818668842315674, -0.33675429224967957, -1.9872907400131226, 0.07751861214637756, -0.28571638464927673, -1.0547422170639038, -3.211348056793213, 2.4396421909332275, 0.3615250289440155, 1.1722382307052612, 0.40013912320137024, -1.6196377277374268, -0.843528151512146, 1.872591495513916, -2.689328670501709, 1.3239046335220337, 1.5445294380187988, -1.8341233730316162, -0.2089788019657135, 0.4900170862674713, -1.5779366493225098, -2.7324910163879395, -0.470146119594574, 1.039831519126892, -0.08788248896598816, -0.022383302450180054, -0.9907175302505493, -0.275667279958725, -2.171807289123535, 1.4279229640960693, 0.275552362203598, 1.2655245065689087, 3.9092907905578613, 0.8139718174934387, 2.416224956512451, 0.2042180299758911, -1.9973876476287842, -0.6820069551467896, 0.5077530145645142, 0.6053862571716309, -0.458543986082077, 0.9555037617683411, -0.9525567293167114, -0.9572654962539673, 1.2331244945526123, 2.3540897369384766, -1.2014827728271484, -1.0512685775756836, -1.86327064037323, 1.2004643678665161, -1.372660756111145, -0.45579639077186584, 0.24173729121685028, -0.8084617853164673, 0.924167811870575, 0.09277480840682983, -0.8640822172164917, 0.6957207918167114, 2.6078009605407715, 0.8605764508247375, -0.7238403558731079, -1.048012614250183, 1.0433439016342163, -0.7588293552398682, 1.639823079109192, -0.22420023381710052, 2.613603353500366, -1.12021005153656, 2.6256508827209473, -0.4429989457130432, 1.7048481702804565, 3.065476894378662, -1.027300238609314, -0.9479053020477295, -1.5385485887527466, 1.0996636152267456, -2.5670199394226074, 1.8585014343261719, 0.40929633378982544, -0.9722058773040771, 2.4874250888824463, -0.07374712824821472, 1.392795443534851, -0.19768022000789642, 1.4620903730392456, 1.3985514640808105, -3.3036980628967285, 0.7031106948852539, 0.7977181673049927, 0.376151978969574, -0.7553391456604004, -0.17363512516021729, -0.2187158465385437, -2.8621833324432373, 1.520946979522705, 0.6559325456619263, 0.9446696639060974, -1.0469331741333008, -1.7182071208953857, 0.46819978952407837, 0.5229061841964722, -1.8487225770950317, -1.1452499628067017, -2.494363307952881, 2.66208815574646, -1.442392110824585, 0.7738707065582275, 1.3821922540664673, 1.965004563331604, -1.6320455074310303, -1.1361675262451172, 0.48197126388549805, -0.44477343559265137, -1.3244527578353882, -1.204368233680725, 2.491791248321533, 2.808439254760742, -1.1802165508270264, -2.0726683139801025, 1.9964524507522583, 2.0870604515075684, 0.09641870856285095, -0.17973095178604126, -0.5531200170516968, 2.87982177734375, 0.9829005002975464, 1.8562499284744263, 0.6026870012283325, -0.06946995854377747, -0.5043541193008423, -1.8698337078094482, -2.360307216644287, -0.9000014066696167, -0.8984061479568481, -0.5037588477134705, 1.950124979019165, 1.1332666873931885, -0.3898785412311554, -1.3260149955749512, 1.0439828634262085, 1.1034870147705078, -0.6247751116752625, 0.3736122250556946, 0.4139268398284912, 0.7206568717956543, 0.4543918967247009, 1.6242706775665283, -1.4369158744812012, 0.9913281202316284, -0.8877676725387573, -0.07351115345954895, -1.0141825675964355, 1.6244336366653442, 0.307483434677124, -0.20453476905822754, -0.3923034071922302, -1.6516718864440918, 1.3561660051345825, -0.18532806634902954, 3.926893472671509, -0.5472841262817383, 1.4148708581924438, -4.2993879318237305, -0.3215453624725342, 0.744197428226471, 0.5387815237045288, -1.4908699989318848, -1.284775733947754, -0.6640024185180664, -0.27443715929985046, 2.1375951766967773, 1.2986624240875244, -0.4170968234539032, -0.2909468412399292, -0.3966860771179199, 1.6387605667114258, -0.4331604242324829, -0.866496205329895, 0.8214633464813232, 0.7263839244842529, -1.9100127220153809, 1.0459654331207275, -2.387209415435791, -3.0214340686798096, -0.7566079497337341, -0.07321375608444214, -0.8123316168785095, 0.04312142729759216, 0.5503689646720886, -1.043148398399353, 1.5745567083358765, -1.3517065048217773, -2.7796359062194824, -0.8054676055908203, 2.5803170204162598, 2.1905202865600586, -1.2329742908477783, -2.5905418395996094, 0.06943714618682861, -1.193697214126587, 1.4320392608642578, -1.3245291709899902, -1.0572494268417358, -0.001650094985961914, 0.07887893915176392, -2.7083616256713867, 1.1760616302490234, -0.6624335050582886, -1.1891599893569946, -0.5078864097595215, -0.6966956853866577, 1.270613193511963, -2.0731253623962402, 0.5382631421089172, -1.0885872840881348, -3.209177017211914, 1.6357920169830322, 2.1588029861450195, 0.2432604730129242, 4.198984146118164, 1.130582332611084, 1.4742136001586914, 0.6431490182876587, 0.3885784447193146, 2.1809492111206055, -2.7976796627044678, -1.1972441673278809, -0.16530366241931915, -0.29156267642974854, 0.14547759294509888, 1.0184980630874634, -1.2352347373962402, 1.7553600072860718, 0.6266270875930786, 1.751569151878357, 0.8865547776222229, -0.9020413160324097, -2.1595327854156494, 0.23564304411411285, 2.607388973236084, 1.716895580291748, 1.4106967449188232, 0.9615804553031921, -3.165130376815796, 1.9462852478027344, -3.0456862449645996, 0.6348381638526917, -0.8800212144851685, -0.5764553546905518, -0.06980878114700317, 1.775630235671997, 0.9559457302093506, 1.6482577323913574, 0.7507744431495667, 1.0421700477600098, 0.6215449571609497, -1.0175527334213257, -1.7285573482513428, 1.097665548324585, 0.03015315532684326, 1.5314791202545166, -0.43613171577453613, -1.0606012344360352, 0.2804661989212036, -0.1937161684036255, 2.004967212677002, -0.327272891998291, -0.7765092849731445, -1.6686718463897705, -0.8741530179977417, 0.9069472551345825, -3.7885701656341553, 0.23072922229766846, 4.056122779846191, 0.13286441564559937, -1.0794368982315063, 0.2762187123298645, 0.3002616763114929, 1.034323811531067, -2.037261962890625, 1.9843618869781494, 1.470475673675537, -2.74993896484375, 0.8210176825523376, 1.7069261074066162, 0.02652806043624878, -1.8407785892486572, -0.2187442183494568, -0.2518669366836548, -2.3762285709381104, -1.0168042182922363, -0.47976601123809814, -0.8033291101455688, -2.2619314193725586, -1.7663531303405762, -1.507574439048767, 2.4826154708862305, 0.5773181319236755, -1.5817240476608276, -0.7823804616928101, 0.5014134645462036, 0.3395853638648987, 0.6083132028579712, -2.8554959297180176, 1.5199042558670044, -1.1918330192565918, 2.7477786540985107, -0.49710309505462646, 0.4806217849254608, 0.7899861335754395, 0.5356317162513733, 1.1811833381652832, 2.269411563873291, 1.8372944593429565, -1.4770259857177734, 1.0541330575942993, 0.09268546104431152, -0.963107705116272, -1.5412001609802246, -0.22589749097824097, -1.7896389961242676, 2.4861340522766113, -0.11303400993347168, 0.4425364136695862, -0.010218560695648193, -1.5473296642303467, -0.43372470140457153, 1.6679792404174805, -0.2673831582069397, 0.6978242993354797, -1.1864871978759766, 0.43236637115478516, 4.312707424163818, -0.6001322269439697, 1.6213735342025757, -0.5290855169296265, -1.6691694259643555, -0.4430054724216461, -0.10703103244304657, -0.6282581090927124, 2.3394558429718018, 0.7184039354324341, 0.5001904964447021, -1.195048213005066, 1.6117100715637207, 0.2858441174030304, 0.3947184085845947, 1.469272255897522, -1.056093692779541, -1.4706199169158936, -2.563538074493408, -3.619457244873047, 0.44575414061546326, -0.19959205389022827, -0.392971932888031, 1.1323052644729614, -0.08364254236221313, 0.9910629987716675, -0.5254531502723694, -1.8799269199371338, 1.1723301410675049, 2.9045374393463135, -1.24294114112854, -1.556536078453064, -1.4100584983825684, 0.6223423480987549, 0.6729679703712463, 0.1888381391763687, -0.3955784738063812, -0.8622099757194519, 0.36720511317253113, 1.75180983543396, 1.4388618469238281, -0.8235381841659546, 1.9223878383636475, 0.11466759443283081, -1.494391918182373, 1.5081255435943604, -1.484039545059204, 0.3025321960449219, -1.5992107391357422, -0.21031184494495392, 0.1658644676208496, -1.0533220767974854, 0.39722728729248047, -0.4957216680049896, -0.3915359377861023, 1.1707204580307007, 0.6775262355804443, -1.969987154006958, 0.4021781384944916, 3.124539613723755, 0.848402202129364, 0.6522283554077148, 0.3144376873970032, 0.3976269066333771, 0.18464040756225586, -1.833639144897461, 1.5501574277877808, 1.5864410400390625, -0.9540791511535645, 0.8953962326049805, 1.0900640487670898, 0.711114764213562, -1.1170326471328735, -0.5838108062744141, 0.21711844205856323, 0.7441630363464355, -1.3197662830352783, -0.7917287945747375, -0.8142150044441223, 2.219078779220581, -0.04496872425079346, 0.3718733787536621, -0.1446811556816101, -1.1901767253875732, 0.0022620558738708496, 0.5130825638771057, -0.9143036603927612, -0.8522921800613403, 0.7763720750808716, 0.556817889213562, -0.799058735370636, 0.009402483701705933, -0.8372170925140381, 1.9580124616622925, -1.901442050933838, 1.2813035249710083, -1.0802497863769531, -0.9138029217720032, -2.3270504474639893, 0.994948148727417, 0.026186197996139526, -2.4213132858276367, -1.4808685779571533, -0.29985660314559937, -1.51143217086792, 1.0910871028900146, -1.5250312089920044, 0.15989112854003906, 0.13324479758739471, -0.9755997657775879, 1.519256830215454, 1.392305850982666, -2.208090305328369, 1.503313660621643, -4.0934038162231445, 2.0649595260620117, -0.8074010014533997, 1.092827320098877, 0.4990246295928955, -0.862288236618042, -0.2125854194164276, -1.716976284980774, -1.8795313835144043, 0.7718296051025391, -0.025235891342163086, -2.8095672130584717, 0.056012749671936035, 0.27358299493789673, 0.6531402468681335, 0.4976004362106323, -0.6029559373855591, 0.8861507773399353, -2.3392271995544434, -0.3236018121242523, 1.296044111251831, 0.11623416841030121, -1.2494113445281982, -0.9600352644920349, 0.8781638741493225, 1.7370494604110718, -0.6269512176513672, -1.4094711542129517, -1.63390052318573, 0.32096177339553833, -0.9884576797485352, 0.9712051153182983, 0.289524108171463, 0.5345243215560913, -1.0807734727859497, 0.4645984172821045, -2.186964988708496, -0.1980581134557724, 2.2582530975341797, -2.3504159450531006, -0.8910945653915405, -2.6159162521362305, -2.1364026069641113, -0.8993431329727173, 0.17272955179214478, -0.5957435965538025, 0.94994056224823, 0.8606308698654175, 0.29426711797714233, -2.404860019683838, -1.5009491443634033, 1.936388611793518, 0.6785804033279419, -1.4024264812469482, -2.2308967113494873, 0.3037671446800232, 0.6136803030967712, 0.08234688639640808, 0.5056343078613281, -2.531853675842285, -1.7491583824157715, -3.5266966819763184, 1.7333576679229736, 0.020971059799194336, -0.3040468096733093, 0.6791434288024902, -0.36176949739456177, -0.26975321769714355, 0.34135371446609497, -1.4248600006103516, -0.6082105040550232, -0.27210932970046997, -0.40146535634994507, 0.5897541642189026, -1.113964319229126, 0.49108052253723145, -1.537184238433838, 1.9524421691894531, -1.0876219272613525, -1.0951471328735352, 0.16385799646377563, -0.9502744078636169, -2.064347505569458, -0.07568685710430145, -1.0138556957244873, -2.4856128692626953, 0.3040228486061096, 1.6491541862487793, 0.7249810099601746, -2.3191776275634766, 0.8347892165184021, -0.965091347694397, 1.6860021352767944, -1.0509560108184814, 0.934346079826355, 0.27685821056365967, 2.056022882461548, 0.15206554532051086, 0.4858528971672058, -1.3317495584487915, -0.7922691106796265, 1.6089537143707275, 0.3314642310142517, -0.8247354030609131, 1.1652238368988037, -0.9203674793243408, -0.7872921824455261, -0.48938727378845215, 1.0489962100982666, 2.0004568099975586, -2.904752492904663, -2.275242567062378, 0.9674404859542847, -1.6410497426986694, -1.530415415763855, -1.7025712728500366, 0.44393014907836914, 2.151932954788208, 0.36714908480644226, 0.5385959148406982, -2.4180688858032227, -0.7734909653663635, -0.2243589460849762, 1.2807528972625732, 0.5179355144500732, -1.3843469619750977, 1.229163408279419, -0.05203688144683838, 4.522262096405029, -1.6499220132827759, 1.700714111328125, -3.3728837966918945, 0.9958465099334717, 0.4584619998931885, -0.19702622294425964, -1.3688642978668213, -1.6040289402008057, 2.6309733390808105, 0.4981999397277832, -0.021571218967437744, -0.22573715448379517, -0.9202338457107544, 0.05342922732234001, -1.0711700916290283, 0.29227614402770996, 1.1706032752990723, 0.8849169015884399, 0.7751771211624146, 0.5202531218528748, -1.0799839496612549, -1.3975768089294434, -0.914585292339325, 1.6401792764663696, 0.17641288042068481, 0.6846568584442139, -0.3657352030277252, -1.2574527263641357, -1.8298509120941162, 0.9979469180107117, 0.8084445595741272, 1.6737806797027588, -0.48139703273773193, 0.34206607937812805, 0.02568262815475464, 1.373299241065979, -0.3585616946220398, 0.690719485282898, 0.3399629592895508, -0.6084504127502441, -1.105170726776123, 0.14577972888946533, -0.7533251047134399, 0.2734583020210266, -0.6583061814308167, 0.5690802931785583, 0.23433120548725128, 0.472861647605896, -0.5009620785713196, 0.8004770278930664, 0.43422365188598633, 2.7722177505493164, -0.4224613904953003, -0.7022131085395813, 0.1929250955581665, -0.04809695482254028, 0.9968146085739136, 2.7117972373962402, 0.05965179204940796, -2.8039095401763916, 1.9816721677780151, 0.17215687036514282, 0.07276672124862671, 1.6458103656768799, -2.8618617057800293, 0.6461646556854248, 1.3889082670211792, -0.21676334738731384, 0.4556598365306854, 1.6554756164550781, -0.5718445777893066, 0.9664157032966614, -0.9577834606170654, 0.5522371530532837, -1.4774630069732666, -0.883565366268158, 0.08863867819309235, -0.8546575307846069, 0.14310431480407715, -0.7212575674057007, -0.16422706842422485, 0.01431778073310852, 1.0674288272857666, -0.35836297273635864, -0.3474578857421875, 0.7874822616577148, -0.2549595236778259, 0.6957595348358154, 0.3594471216201782, 0.5475976467132568, 0.9109658002853394, -0.4265154004096985, 2.601560115814209, -1.1077936887741089, 2.045353889465332, 1.8603605031967163, -0.1395156979560852, 2.145045280456543, 0.04949954152107239, -3.477205276489258, -0.8472716808319092, -1.1234846115112305, -1.918869972229004, -0.8544151782989502, 1.5218710899353027, 3.226605176925659, -1.1874977350234985, 0.27840039134025574, -1.5181688070297241, -3.646583080291748, 0.06211221218109131, 1.9249329566955566, -1.4944998025894165, -0.032965511083602905, -2.0750885009765625, 1.5673511028289795, -3.1617271900177, -1.0754194259643555, -2.716872215270996, 0.5283515453338623, 2.873972177505493, -1.5663717985153198, -2.4543678760528564, -2.520735740661621, 0.48310422897338867, 1.4220058917999268, -0.5541930794715881, -0.124397873878479, -1.811353087425232, -0.25934016704559326, -0.007766157388687134, 1.3706144094467163, -2.3336880207061768, -1.2929656505584717, 1.107468843460083, -1.620299220085144, 1.4073379039764404, -0.4729665219783783, 0.6099531054496765, 1.3328396081924438, 2.2282581329345703, -2.218689441680908, 1.7721672058105469, -2.866551399230957, 1.5138871669769287, -1.827380657196045, 0.6628009080886841, -0.05906558036804199, -0.7338809967041016, -1.537221908569336, 2.2391817569732666, -0.9292340278625488, 1.134629249572754, -0.40878015756607056, 1.3810069561004639, 0.743984043598175, 1.7371058464050293, 1.1148561239242554, 2.2057924270629883, 1.3558266162872314, 0.06445080786943436, -0.748513400554657, 0.26290032267570496, -3.4087419509887695, 0.8620033860206604, 0.1332142949104309, 0.7055602073669434, -0.3839130699634552, 0.3596217632293701, -0.5816638469696045, -0.19299626350402832, -2.4671878814697266, -0.5035892724990845, 0.017322421073913574, -2.3800535202026367, 0.6130504608154297, -1.5539740324020386, 1.2091954946517944, 2.2826671600341797, -1.3625092506408691, 1.5032228231430054, 1.3777426481246948, 2.1874406337738037, 0.7016722559928894, -0.5271024703979492, 1.7702012062072754, 0.646814227104187, -0.5007162094116211, 1.2372982501983643, -0.32845836877822876, 0.10184568166732788, -0.5977744460105896, -0.8709802627563477, -0.892349123954773, -2.7058277130126953, 1.1261600255966187, 0.6660538911819458, -0.4253692626953125, 2.2066168785095215, 0.7757343053817749, 1.7640798091888428, 0.5672644972801208, -0.012221336364746094, 1.013229489326477, -0.9030327796936035, -2.269731044769287, -0.8155696392059326, 0.9232006072998047, -2.0785021781921387, -2.7219152450561523, -1.7082397937774658, -0.06357014179229736, -0.636479377746582, -1.3471879959106445, 1.8996765613555908, 1.0186681747436523, 0.029137536883354187, 0.04854947328567505, -0.26729702949523926, -1.0377365350723267, 0.4169313609600067, 1.8155124187469482, -2.381572723388672, -1.2432366609573364, 1.1383098363876343, -0.03134545683860779, -0.1487828493118286, -1.3420814275741577, 1.1194684505462646, 1.8245203495025635, 0.22873812913894653, -1.7934088706970215, 2.6054768562316895, -0.21305587887763977, 0.4554399251937866, -0.8855154514312744, 1.3174515962600708, 1.6304569244384766, -0.548630952835083, -0.4337276518344879, 2.7962522506713867, -0.9101515412330627, -0.23521563410758972, -3.0454769134521484, -1.9729827642440796, -0.6723307371139526, -0.27294787764549255, 0.6506214141845703, -0.2441256046295166, 0.844291090965271, -0.48505765199661255, -1.4530819654464722, -0.7841638326644897, 0.533513605594635, 0.5376482009887695, 1.529552698135376, -0.9901533722877502, 1.471384048461914, -2.0457231998443604, 0.8716142177581787, 2.372507095336914, 0.33290964365005493, 1.471207857131958, 0.8953312635421753, -1.742621898651123, 0.6617790460586548, -1.16325044631958, -0.5312436819076538, -0.8290529251098633, 0.05835370719432831, -0.7644737362861633, -2.693300247192383, 1.2677220106124878, 0.6919163465499878, 0.535834789276123, -0.7237979173660278, -0.47526872158050537, 0.009392321109771729, -1.086175799369812, -0.02306106686592102, -1.4491509199142456, 0.35168924927711487, 2.288938045501709, -0.5218168497085571, 1.021026372909546, -0.569891631603241, 0.7955910563468933, -1.901168942451477, -0.23612502217292786, 0.18320000171661377, -0.4916456341743469, 0.08985483646392822, 4.197659492492676, 0.22279560565948486, 0.02913641929626465, -1.9136401414871216, 0.030992984771728516, 2.097715377807617, 2.6841342449188232, 0.19375771284103394, 0.8752442598342896, -0.2752416729927063, -0.3119294047355652, 2.892714023590088, -0.45328831672668457, -0.4595559239387512, -1.3432573080062866, 0.5374813079833984, 0.5736889839172363, 1.866367220878601, -0.36584123969078064, 0.8373639583587646, 0.060725852847099304, 0.9349265694618225, -0.4763302206993103, 1.7368578910827637, 2.3353111743927, -0.028152335435152054, 0.39618998765945435, -2.373702049255371, -0.18169429898262024, 1.0342367887496948, 1.2005667686462402, 0.5477159023284912, 1.7988241910934448, -0.2714413106441498, 2.457744598388672, -0.7322606444358826, -0.5480077862739563, 1.0424931049346924, 1.02980637550354, 4.1185832023620605, -0.2580742835998535, -0.9762173891067505, -0.04800009727478027, -2.441962480545044, 0.9464275240898132, 0.2799658477306366, 0.07067301869392395, -0.5251299738883972, 2.0731124877929688, 0.2449001967906952, 0.050985634326934814, -0.07364702224731445, 0.44086241722106934, -0.487262487411499, -0.24989068508148193, 0.427024781703949, -1.3242872953414917, 1.1471246480941772, -1.9432686567306519, -0.3867669701576233, -0.9150843620300293, 0.1873544603586197, 0.8096104264259338, -0.5754837989807129, 1.153306484222412, 0.808089017868042, -1.6278059482574463, 2.534639358520508, 0.008845925331115723, -1.1783146858215332, -0.993127703666687, 2.1070830821990967, 0.8386070728302002, 2.531851053237915, 4.5042009353637695, 3.0804994106292725, -0.3098204731941223, 2.4854912757873535, -2.851757049560547, 2.576540470123291, 1.4756931066513062, -0.8814721703529358, 0.5146358013153076, -0.19639050960540771, -2.8847243785858154, 1.3457825183868408, -3.581007957458496, -0.8967328071594238, 1.3894903659820557, 0.6796345114707947, -0.1986127495765686, -1.707014799118042, 0.4224448502063751, 1.794669508934021, -1.6685972213745117, -1.2749820947647095, 0.6289803385734558, 2.618983507156372, 1.0469157695770264, 0.7509965300559998, 3.1160435676574707, -2.339913845062256, -0.5208420157432556, -0.3151171803474426, -0.840563178062439, 0.7487354278564453, 1.8547567129135132, 0.711239218711853, -2.0738067626953125, -0.601942241191864, 1.1091022491455078, 0.4995477497577667, 0.8542152643203735, -1.7191767692565918, -0.4490451514720917, 0.053129732608795166, 0.029080629348754883, 1.747942566871643, 0.9960963129997253, 1.3389191627502441, 0.12504971027374268, -0.35169070959091187, -1.615050196647644, -2.7508609294891357, -0.14549487829208374, -1.9363019466400146, -0.25909852981567383, 0.3150585889816284, -0.6675926446914673, 0.11183635145425797, -1.3241788148880005, -1.1294760704040527, -0.774831235408783, -0.019969793036580086, 2.2233564853668213, -0.9841440916061401, -0.03698594868183136, -0.7154496908187866, -2.18538761138916, 3.2846977710723877, 1.1369235515594482, 0.4455130994319916, -0.4621361494064331, 1.351078987121582, -3.565544843673706, -1.5643329620361328, -0.8064266443252563, -2.290344715118408, -0.008613169193267822, 0.09412626922130585, -0.9657690525054932, -0.806041955947876, -0.012787938117980957, 0.35252922773361206, -0.5047004222869873, 1.2266528606414795, -0.658248245716095, 1.578193187713623, 0.09911973774433136, 0.4589594900608063, -1.6205317974090576, -0.6614822149276733, 3.1760895252227783, 1.6524572372436523, 0.032221317291259766, 0.9524903297424316, 0.19948628544807434, 0.5243697762489319, 1.5574272871017456, -0.54697185754776, -0.07676547765731812, 0.07903474569320679, 1.1445211172103882, -0.14518263936042786, -3.95951509475708, 2.230994701385498, -0.09323298931121826, 1.9585025310516357, -0.36066892743110657, -0.21088290214538574, 0.4447276294231415, 0.292227178812027, 0.27985358238220215, 3.324950933456421, -3.079314947128296, -1.3257722854614258, -0.14375907182693481, -2.3968167304992676, -0.9583062529563904, -1.494596242904663, 1.8671836853027344, 2.4908392429351807, -0.5883592367172241, 0.1325373649597168, 1.086014986038208, -0.7627630829811096, 1.3831593990325928, -1.4973686933517456, 2.450044631958008, -0.9983519911766052, 0.9591684341430664, 0.8813707828521729, -0.643096923828125, 0.798389196395874, 0.5306676030158997, 0.19664087891578674, 0.004094958305358887, 1.531059741973877, 0.04557095468044281, 0.0798385739326477, 0.3935953974723816, 0.37510353326797485, -2.7182059288024902, -1.3581340312957764, 1.1898465156555176, -0.5645710825920105, -0.8971313834190369, -2.9740242958068848, 1.0539275407791138, -2.025887966156006, -1.4010326862335205, 0.538170576095581, -0.030790239572525024, -2.3362724781036377, 1.4388353824615479, 1.3629125356674194, -0.6825536489486694, 1.3731491565704346, 0.5412191152572632, -2.0744032859802246, -1.368180513381958, -0.9906495809555054, 0.5371280312538147, 2.179922580718994, -0.2036651372909546, 1.3266246318817139, 1.038696050643921, -1.0900261402130127, -0.239507794380188, 0.5819947123527527, -1.230660319328308, 0.8576578497886658, 1.2170932292938232, 1.1904863119125366, 0.09433484077453613, -2.2203750610351562, 0.10421031713485718, -3.2511019706726074, 0.4207596480846405, -1.0659936666488647, 3.792919635772705, -0.4498860836029053, 1.8596532344818115, 0.09977039694786072, 2.149944305419922, 1.2663023471832275, -0.2401091754436493, -0.8336120843887329, -0.5842267870903015, 1.2264186143875122, -1.4055880308151245, 1.2045586109161377, -0.07067227363586426, 0.25796353816986084, 0.5649295449256897, -0.6164064407348633, 2.083402395248413, -0.0845416784286499, -1.3343058824539185, -0.6079662442207336, -1.1256729364395142, 0.01234215497970581, -0.15025749802589417, -0.9259396195411682, -0.20291456580162048, -2.3632616996765137, 0.7231749296188354, 0.09560909867286682, 0.21855568885803223, -1.9510334730148315, 1.2529232501983643, 0.6546201705932617, 1.0347906351089478, 3.106874465942383, -1.25034499168396, 1.043198585510254, 1.9321141242980957, 0.43036913871765137, 1.675764799118042, 0.6514879465103149, 0.8600892424583435, 0.7609975337982178, 2.8714098930358887, 0.8007698655128479, -1.587022066116333, -0.09928333759307861, 1.1388304233551025, -0.829910159111023, -0.7515398859977722, -1.7748785018920898, 0.8497186899185181, -1.9294724464416504, 1.761622428894043, 4.382053852081299, -1.314639925956726, 0.3541974425315857, 0.33928847312927246, -0.3403165936470032, -0.6729536652565002, -0.6364060640335083, 1.1195833683013916, 1.7119874954223633, 2.291429281234741, 0.5479187369346619, 0.0433650016784668, 2.144838571548462, 0.7344991564750671, -0.9554975032806396, -0.22902634739875793, 1.5755829811096191, -0.26625096797943115, -1.177191972732544, 3.140007495880127, 1.2551082372665405, 0.28531718254089355, -1.1459494829177856, -0.5137646198272705, 0.8579543828964233, -0.28910747170448303, 0.35132965445518494, -0.36884182691574097, -0.2558259963989258, 2.620290994644165, 0.767654299736023, -0.4054172933101654, 0.5306230783462524, 1.3906357288360596, 0.14716196060180664, 0.9562963843345642, -1.8637983798980713, -0.8490390777587891, -1.4235202074050903, -0.34817320108413696, 1.1383594274520874, -0.536178469657898, -0.736319899559021, -0.13148868083953857, 0.5934591293334961, 0.751660943031311, 0.37133264541625977, 0.028563782572746277, 1.053951621055603, -0.7789204120635986, 2.0644264221191406, -0.5209395289421082, -0.5634026527404785, 1.9041309356689453, 1.0777124166488647, -1.6174092292785645, 0.8801558017730713, 1.4538756608963013, -0.5891490578651428, -1.3352408409118652, 1.413111686706543, 1.2350727319717407, -2.102309226989746, -2.598280668258667, -0.12510383129119873, 2.2844741344451904, 1.0860340595245361, 0.6900065541267395, 1.076917052268982, -0.14008092880249023, 0.9743634462356567, 2.025425910949707, 0.8727919459342957, 0.7405121326446533, 0.17633211612701416, -1.6753654479980469, 3.823183059692383, 0.12939733266830444, -2.1010971069335938, -0.13430744409561157, -0.7183318734169006, -0.013778656721115112, -1.9689067602157593, -0.6640259027481079, 1.4924535751342773, 3.3720521926879883, -1.2551074028015137, -0.31096139550209045, 0.4533509612083435, -1.1321638822555542, -0.5450261831283569, -0.29001128673553467, 1.0760209560394287, -1.2202578783035278, -0.713663637638092, -1.1096885204315186, 1.7531375885009766, 1.5208715200424194, 1.62123441696167, 1.3662265539169312, -0.8626617193222046, -0.758784830570221, 1.5625275373458862, 0.8758130073547363, 0.2542940676212311, 2.345639705657959, -2.1164464950561523, -0.08604270219802856, -0.2713211476802826, -2.1506154537200928, -1.0830857753753662, -0.5261301398277283, 1.6547651290893555, -0.8369390964508057, 2.393383741378784, 2.176547050476074, 0.817345917224884, -0.876267671585083, -0.7153791189193726, 3.419736623764038, -0.3345910310745239, 2.0364253520965576, -1.3452129364013672, -0.7082395553588867, 1.5029720067977905, -1.3274332284927368, -0.47620144486427307, 3.1456897258758545, 1.4715344905853271, -1.4120100736618042, -0.34570127725601196, 2.1037447452545166, -1.722923755645752, -2.0241005420684814, -0.4206872582435608, -1.5009551048278809, 1.628322958946228, -0.3143100142478943, 2.232896566390991, 1.4543871879577637, 2.023106575012207, -0.24190464615821838, 0.12076950073242188, -0.45093977451324463, -2.766080617904663, -0.006297111511230469, 1.797910451889038, -1.57731294631958, -0.3031385838985443, 1.8613474369049072, -0.953429102897644, -0.08998316526412964, -1.0973711013793945, -1.0608932971954346, 1.0475295782089233, 0.9869393110275269, -1.3545210361480713, -0.3065497875213623, 0.7026858925819397, 0.9696528911590576, -2.1795778274536133, -1.1872625350952148, -1.1421376466751099, -1.6400487422943115, -2.4338834285736084, -1.4263393878936768, -0.307173490524292, 0.5545799136161804, 1.580506682395935, 2.8270840644836426, -0.83873051404953, -2.097029447555542, 3.4378180503845215, 0.0667371153831482, -0.1884106695652008, -0.42348718643188477, -0.15286201238632202, -1.3559117317199707, -3.1384685039520264, 1.1576082706451416, 1.41380774974823, 0.5790802240371704, -1.05813467502594, -0.620877206325531, -1.805289626121521, -0.6332269906997681, 0.9622414708137512, -0.0001678168773651123, 1.2961199283599854, -0.42366135120391846, -1.1415802240371704, 1.4457746744155884, 0.01632446050643921, -0.6622483730316162, -1.6877267360687256, 0.8303476572036743, -0.7998231649398804, -2.1107912063598633, -1.3374274969100952, 0.9218607544898987, -0.8712140321731567, 0.6496691703796387, 2.7661876678466797, -1.7813835144042969, -1.499661922454834, -0.7387485504150391, -1.9700548648834229, 0.28880733251571655, -1.9546476602554321, 0.6646538972854614, 1.040812373161316, -0.9474580883979797, 0.609839141368866, -0.7425116896629333, 1.640624761581421, -1.5386601686477661, 1.6232616901397705, 0.9773094654083252, 0.32317376136779785, -0.6819457411766052, 0.6488481163978577, 1.8585803508758545, -1.2642706632614136, 0.2804664969444275, -1.0068317651748657, 0.05308818817138672, -0.9498974084854126, 1.3274813890457153, -0.9303561449050903, -0.7539634704589844, -0.7641237378120422, 0.010876178741455078, -0.047907114028930664, 1.2543460130691528, 1.9531996250152588, -1.2884395122528076, 0.3872077465057373, -0.6318531632423401, 0.21409469842910767, -1.7932617664337158, -1.4647433757781982, 2.284069538116455, 0.4337766766548157, -0.575520396232605, 1.218216896057129, 0.21696120500564575, 0.4050048589706421, 0.9641948938369751, -2.184417724609375, -2.600881576538086, 0.28326067328453064, 0.094859778881073, 0.9601887464523315, 0.8323127031326294, -1.9430413246154785, 0.9211037158966064, -1.9773297309875488, -0.7772239446640015, 2.7712550163269043, 0.23980480432510376, -0.8662666082382202, -0.617445170879364, 1.859452486038208, -1.399592638015747, 1.4243502616882324, -0.36944589018821716, -2.3698196411132812, 0.6678282022476196, 0.7091895341873169, 3.952739715576172, -1.3791497945785522, -0.12882208824157715, -0.3871132731437683, 0.1856914609670639, 0.5931623578071594, -0.39205944538116455, -1.0200176239013672, 0.08853846788406372, -0.8084686994552612, 0.18845027685165405, 0.8483359217643738, 2.950072765350342, 0.5901965498924255, 0.008462965488433838, 1.475053310394287, 0.6382973790168762, 1.9209532737731934, -0.5990462303161621, -1.9005305767059326, -0.3393544554710388, -0.0742344781756401, -0.621284008026123, -1.3524465560913086, 0.1327875852584839, 3.8316147327423096, 0.7304093837738037, -0.47770416736602783, -2.928359270095825, -0.8951122760772705, 0.6960406303405762, 0.5949687957763672, -1.4922722578048706, 0.29124879837036133, 0.36367297172546387, 1.816507339477539, -1.8578994274139404, -0.7357630133628845, -0.5317757725715637, 1.140995979309082, 1.1339037418365479, 0.09908249974250793, 1.2655247449874878, 3.221251964569092, -0.13655677437782288, -0.6116766929626465, -1.3181445598602295, 0.6449797749519348, 1.0737329721450806, -2.9330317974090576, 0.710128664970398, 0.6307899951934814, 0.34413278102874756, 2.566898822784424, -0.793389081954956, 0.7065774202346802, -0.11699619889259338, -1.4346312284469604, -1.1231999397277832, -1.7291674613952637, 1.0587435960769653, 1.0962737798690796, 1.3442819118499756, -2.022097587585449, -0.5043877959251404, 1.2840406894683838, -0.4595959186553955, 0.8346625566482544, 0.9259496927261353, -0.9114457368850708, -1.786176085472107, 1.2203037738800049, -0.9225306510925293, 0.2327992022037506, 2.4284894466400146, -0.6271533966064453, -0.8824830055236816, -0.14409202337265015, 1.57877516746521, 0.6794582009315491, -0.21048685908317566, -0.4057428538799286, -2.3505802154541016, -1.2458877563476562, -0.050627708435058594, -0.4975479543209076, -1.1523224115371704, 1.4879131317138672, -1.154660701751709, 1.0972158908843994, 2.057446002960205, -0.2733447849750519, -0.368108868598938, -1.0765587091445923, 1.320631504058838, -1.8336589336395264, 1.5844614505767822, -1.3202553987503052, -0.08335524797439575, -1.4827684164047241, 1.0239765644073486, -1.6741101741790771, -2.0241551399230957, 1.561368465423584, 0.6474363207817078, -0.539487898349762, -1.2105953693389893, -0.11193525791168213, -2.626007556915283, -1.8913323879241943, -1.5618085861206055, -0.6368141770362854, -2.06600022315979, 0.7506731152534485, 3.1355013847351074, -0.8863388299942017, -0.662845253944397, -0.25529107451438904, -1.2480137348175049, -1.8415143489837646, 0.0025804638862609863, 0.7376404404640198, 2.089725971221924, -2.462857961654663, 1.2702564001083374, 0.5282501578330994, -1.655996561050415, 0.28137508034706116, -0.659795880317688, 0.8072398900985718, -0.7206078767776489, 0.33753442764282227, -0.01869577169418335, -2.688477039337158, 1.712364912033081, -0.734718918800354, -0.49924027919769287, 1.7014364004135132, 0.250474214553833, -0.43796589970588684, -0.5731597542762756, 0.6082900166511536, -0.27918967604637146, -1.0333337783813477, 1.6624670028686523, 0.1782422959804535, -2.558884620666504, -2.3966622352600098, -0.18764391541481018, 1.6205589771270752, 2.655000686645508, -1.4989423751831055, 2.0892229080200195, -0.4078753590583801, 0.3760101795196533, 0.8081932663917542, 1.1682530641555786, -1.659911036491394, -1.849087119102478, 0.795744776725769, -0.8847690224647522, 1.009918212890625, 0.38136404752731323, 0.776828408241272, -0.6911779642105103, 1.2297563552856445, 3.4809389114379883, 0.6193771958351135, 1.8669822216033936, 0.13732439279556274, 1.5197032690048218, -0.45273202657699585, -0.11334291100502014, 0.758814811706543, -1.5240755081176758, 0.7963831424713135, -0.26338309049606323, -0.9055818915367126, -0.5289972424507141, 2.255695343017578, -2.554192304611206, 0.0013275444507598877, -0.31829649209976196, -1.6051578521728516, -2.229109287261963, 2.539567708969116, 0.024556994438171387, -0.6734659075737, 0.8863217234611511, -1.178175926208496, 0.4191599488258362, 0.5920226573944092, 2.500077724456787, 1.2268208265304565, 0.045972466468811035, 1.4172980785369873, 0.3839538097381592, -2.1312255859375, 0.48096340894699097, -0.019168753176927567, -0.45511525869369507, 1.296629548072815, -0.5094687938690186, 2.6106252670288086, 0.48553115129470825, -0.5720351934432983, 0.1753237247467041, 0.34343379735946655, 0.8243756294250488, -0.4463690519332886, -0.002510160207748413, -0.4850168228149414, -1.327751636505127, -0.7042268514633179, 0.366930216550827, -1.1348557472229004, -2.3255772590637207, -0.9430437088012695, -0.9077203273773193, -1.0359447002410889, 0.3546353876590729, -0.6109683513641357, 1.1907023191452026, 0.7822451591491699, -2.1990504264831543, 1.3311409950256348, 1.0408443212509155, -0.5256398916244507, 0.45050865411758423, -2.227691888809204, -0.40329867601394653, 0.8150771260261536, 2.1793181896209717, 1.6644617319107056, -3.6710705757141113, -1.2992582321166992, 0.8782628178596497, 0.16269731521606445, -3.8234477043151855, 0.11144141852855682, 0.20496216416358948, -1.3071138858795166, -0.3883589506149292, 0.4070702791213989, 0.27394551038742065, 0.027926236391067505, 0.8234138488769531, 1.6881037950515747, -0.5469540953636169, -0.2602013945579529, 0.3147270083427429, -2.1750380992889404, 0.8838969469070435, -1.8649533987045288, 0.6019956469535828, 1.4053821563720703, 0.78376704454422, -0.7676633596420288, 0.7637448310852051, 0.14116883277893066, 0.5867935419082642, 0.17369216680526733, 1.3266305923461914, 0.05639152228832245, 0.1413654088973999, 3.1337244510650635, -1.1208410263061523, 2.031373977661133, -1.8418039083480835, -0.6226997375488281, -1.603196144104004, 1.0047919750213623, -0.8140605688095093, -0.9570256471633911, -2.146148204803467, -1.1978896856307983, 0.700014054775238, 0.4691234230995178, -2.1281991004943848, 0.10670658946037292, -1.7495253086090088, 1.8662941455841064, -0.8869113326072693, 0.20116473734378815, 0.6011601686477661, -1.884939193725586, -0.8165959715843201, -0.19258642196655273, -0.9639655351638794, 0.537757396697998, 2.5332841873168945, 1.1958305835723877, 0.06834876537322998, -3.9808366298675537, 1.764500379562378, 1.200155258178711, 0.8642928600311279, -0.47707077860832214, 1.095369577407837, -0.22765815258026123, -1.5775437355041504, 0.8946526646614075, 0.9262958765029907, 0.8276760578155518, -0.9080346822738647, -1.9464271068572998, -0.5886253118515015, 0.40201613306999207, -1.989680290222168, -1.6526086330413818, -0.5376440286636353, -1.6210670471191406, -2.368292808532715, -2.485591411590576, -0.3490598201751709, 0.0984339714050293, -1.5931533575057983, -0.4400956332683563, -0.45737913250923157, -0.554192841053009, -0.7862204313278198, -0.977722704410553, -0.7160884141921997, -0.7974473237991333, 0.12825995683670044, 1.966235876083374, -0.272197961807251, 1.3183634281158447, 0.2477579116821289, 0.018080532550811768, -1.274723768234253, 0.18633246421813965, 0.7459131479263306, -0.12793418765068054, 1.3133821487426758, 3.023311138153076, -2.290381908416748, -0.062160491943359375, -1.6060115098953247, -1.194171667098999, -1.0602335929870605, -0.530002236366272, 1.1741180419921875, -0.22028660774230957, -2.1163175106048584, -1.4372453689575195, 3.4222166538238525, 1.348130226135254, -1.7126933336257935, -0.7650171518325806, 1.5948504209518433, 1.5896681547164917, -0.5651225447654724, -0.8280286192893982, 0.46628618240356445, 1.074155330657959, -1.46867036819458, 0.5243787169456482, 0.5597890615463257, 2.2031443119049072, 1.5776933431625366, 0.37914204597473145, -0.8371387720108032, 0.75107741355896, -2.7830681800842285, 0.06648826599121094, -1.5180554389953613, -3.84826922416687, -1.2164216041564941, -0.7823690176010132, -0.19392524659633636, -2.516014575958252, 2.326364755630493, -0.7593562602996826, -1.2427700757980347, -0.06222027540206909, -2.6683826446533203, -0.7282764911651611, -0.7145261168479919, -0.5743080377578735, 1.0577383041381836, 0.7175928354263306, 0.11420542001724243, 0.3207680583000183, 0.5302070379257202, 1.832075595855713, -1.6124839782714844, 1.9823658466339111, -0.35201597213745117, -1.6694555282592773, -2.774387836456299, -0.2165752649307251, -0.18263006210327148, 0.018048610538244247, -1.2242358922958374, -0.38358914852142334, 0.1523105502128601, 0.2608555555343628, 0.9663507342338562, 1.4043879508972168, 1.1381878852844238, 0.36914724111557007, 1.1912602186203003, -0.4832252860069275, 0.014958620071411133, -0.04600703716278076, -0.5734680891036987, 1.227708101272583, -1.914189100265503, 0.33448123931884766, 0.5830767154693604, 0.38432055711746216, -0.5149049162864685, -0.5436310768127441, 1.607208013534546, -0.8417080640792847, 0.8555223941802979, -0.2205904722213745, -0.7914390563964844, 0.14905840158462524, -0.2148790955543518, -1.8847795724868774, -0.4066951274871826, -1.2191615104675293, 2.237090826034546, -0.5786005854606628, -0.813310980796814, 3.2738943099975586, -0.7287778258323669, 1.758967638015747, 0.9223469495773315, 2.0681674480438232, 1.7005314826965332, 1.1483163833618164, 1.3423274755477905, 0.33119314908981323, 2.667941093444824, -0.373388409614563, 0.9149699211120605, 1.731472134590149, -2.073354482650757, 0.9372002482414246, 1.534256935119629, 2.961817502975464, -0.8617963194847107, 1.1859245300292969, -0.29783010482788086, -0.35955196619033813, 1.3679819107055664, 0.8130257725715637, -0.8010385036468506, -0.05659085512161255, -0.9368616342544556, 1.9541850090026855, 0.347517192363739, -0.35254615545272827, 2.0352272987365723, -1.1127700805664062, 0.4537803530693054, 2.111544370651245, -0.5768582820892334, 1.8020222187042236, -1.1703643798828125, 2.20379638671875, -0.528789222240448, 0.5427219867706299, -1.2819321155548096, -1.7827441692352295, 0.7508624792098999, -1.6894290447235107, -0.3300553858280182, -2.1089553833007812, 2.0297799110412598, -0.6985258460044861, 0.7483632564544678, -0.5101513862609863, -0.36038142442703247, 1.243274450302124, -1.702819585800171, -1.4504096508026123, 1.1304402351379395, -0.4105035066604614, 1.768829584121704, 0.8655146956443787, -0.7362983226776123, -0.06492820382118225, 1.7532761096954346, 0.6517819166183472, -2.2551560401916504, -1.6126370429992676, 1.6255087852478027, 1.568273663520813, -1.3204381465911865, 0.5539894700050354, 2.7443463802337646, 0.638976514339447, 0.6938676834106445, 0.6649730801582336, -1.992784023284912, -0.5354352593421936, 0.7646532654762268, 2.164051055908203, -0.8577755689620972, -1.6951000690460205, -0.2614014148712158, 2.23453950881958, 2.16064190864563, 1.4420363903045654, -0.685652494430542, 0.6098616719245911, 0.087971031665802, -2.090298891067505, -0.6377102732658386, 0.29721057415008545, -0.07432013750076294, -0.6674038171768188, -0.3350276052951813, -1.7965366840362549, 1.3859575986862183, -0.782174825668335, 0.18414413928985596, -0.421677827835083, 1.1355876922607422, -1.513167381286621, -0.38410425186157227, -4.212011814117432, -1.1344389915466309, -1.3366791009902954, -0.9773216247558594, 0.2311449646949768, 2.2392749786376953, -0.05037418007850647, -0.7512111663818359, 1.7903695106506348, 0.8051711916923523, 0.15668010711669922, -1.3125784397125244, -0.8295519948005676, 1.9902257919311523, -1.0126047134399414, 2.0157089233398438, -2.6321358680725098, -0.6097499132156372, -1.5114026069641113, 2.2107555866241455, 3.1114516258239746, -1.3298395872116089, -1.2650129795074463, -1.2745604515075684, -0.12860892713069916, -1.3274662494659424, -0.34630346298217773, -1.0530798435211182, 0.3129476308822632, 2.588937282562256, -0.07988578081130981, 0.27718108892440796, 0.19891315698623657, -2.6919989585876465, 0.057247310876846313, 1.3800106048583984, 2.387817144393921, 0.12102501094341278, -1.627549409866333, 0.09596365690231323, -2.371556282043457, 0.8903642296791077, 0.9621454477310181, 0.5990787744522095, -1.642472743988037, 0.11261796951293945, 1.0844545364379883, 1.1289587020874023, 1.0219030380249023, -1.7356195449829102, 2.8811004161834717, -0.5153173804283142, 0.456836462020874, -0.3849981427192688, -0.3612375259399414, -0.3499738574028015, 0.7792985439300537, 1.5152679681777954, 2.782796859741211, 0.08968913555145264, -2.8916385173797607, -1.983279824256897, 1.8685290813446045, -1.1561205387115479, 2.0797507762908936, -0.32078269124031067, 0.5782619714736938, -0.954108476638794, -0.3243267834186554, -0.03184327483177185, 2.1858344078063965, 1.2984461784362793, -0.5326850414276123, -0.4752006530761719, 0.9516314268112183, -0.4980570673942566, -0.30212652683258057, -1.628859281539917, -0.672809898853302, -0.1360951066017151, -1.3988897800445557, 1.857945203781128, -0.5167554020881653, -0.5037659406661987, -0.7057406902313232, 2.4416277408599854, 2.203050136566162, 0.8511428236961365, -2.704587459564209, -1.1041290760040283, 1.606429100036621, 2.176323890686035, -0.1068313717842102, -0.8304933309555054, -1.7297238111495972, -0.5927032828330994, -0.9190069437026978, 1.725334882736206, 0.3175602853298187, -1.4421862363815308, -2.7034900188446045, 1.697624921798706, -0.16808444261550903, 2.0579309463500977, 0.14423197507858276, -1.1434996128082275, 3.707890748977661, 2.0961506366729736, 0.2437940537929535, 0.08094154298305511, -0.7669954299926758, -0.9934535622596741, 1.4417064189910889, -1.190150260925293, -1.2914026975631714, 0.3153502941131592, 1.2637015581130981, -0.032980531454086304, 1.691371202468872, 0.10171949863433838, 1.2779407501220703, 2.50361967086792, 0.5418785214424133, 2.891828775405884, -1.602765679359436, 0.9338582158088684, -1.2393137216567993, -0.8621166944503784, 0.2370452582836151, 0.549659252166748, 1.4645295143127441, -3.237471103668213, -0.8872488141059875, 2.7791366577148438, -1.2359936237335205, 1.2523013353347778, 3.6260080337524414, -2.5913808345794678, -0.7782506346702576, 2.6772055625915527, 0.6627941727638245, -0.9114450216293335, -0.147158682346344, -0.5997664332389832, 1.0845363140106201, 1.2565034627914429, -0.2980368137359619, -0.7132788896560669, 0.07105624675750732, 1.6506953239440918, -0.6314111948013306, 0.9388816952705383, -1.6532063484191895, 2.768864631652832, 2.2670750617980957, 1.3482850790023804, 0.614960789680481, 0.15393775701522827, 0.11296838521957397, -1.2992191314697266, -0.09671080112457275, 2.054381847381592, -2.053058624267578, -1.7055705785751343, -1.1588557958602905, -0.19425833225250244, 0.2257016897201538, -0.3790421485900879, -3.223658561706543, -0.2898063361644745, 0.3048524558544159, 1.7951135635375977, -1.4126838445663452, -1.021374225616455, 0.5480610132217407, -0.618329644203186, 0.3470674753189087, -0.6300280094146729, -0.619346559047699, 1.259329915046692, -1.161751389503479, -0.4116972088813782, 1.0679875612258911, 0.6279112696647644, -3.510500192642212, -0.1935673952102661, 0.8850387334823608, 0.3335866332054138, 0.18561330437660217, -1.5808944702148438, -0.23451608419418335, 0.36867737770080566, 0.05104249715805054, -0.2234725058078766, -0.35361558198928833, 0.6343870162963867, -0.09418902546167374, 0.9857312440872192, -0.022523850202560425, 0.13643041253089905, -0.6163939237594604, -0.14754259586334229, -3.001842737197876, -0.21139757335186005, 1.2061007022857666, -0.43727725744247437, 0.5706840753555298, -0.026680290699005127, -1.8517192602157593, 1.9099421501159668, 0.2829855680465698, 3.8359122276306152, -0.6596997976303101, -1.763983964920044, -3.335904359817505, -0.20286309719085693, -0.5060395002365112, 1.4639769792556763, -1.5612945556640625, 1.6177388429641724, -2.5190651416778564, -0.877831757068634, -2.2497494220733643, -1.1487669944763184, 0.3845115303993225, 1.4441733360290527, 0.0712968111038208, -1.772519588470459, -1.2821898460388184, 1.460585355758667, -0.7143305540084839, -3.147291660308838, -0.6900011301040649, -0.18709026277065277, -1.312448501586914, -1.4803924560546875, -2.2020905017852783, -0.06772114336490631, -0.7630582451820374, -0.05420953035354614, -1.990967035293579, 0.7931064367294312, 0.5229396820068359, 2.109593629837036, 1.792091965675354, 1.9462250471115112, 0.12553608417510986, -1.104416847229004, -3.955540657043457, -0.6237380504608154, -0.9285640716552734, -0.19136235117912292, 0.24214515089988708, -0.542638897895813, 0.4293724596500397, -1.0577210187911987, 0.8635356426239014, -1.573378086090088, 1.2197034358978271, 0.5209439396858215, 1.03962242603302, -1.392347812652588, 1.4645614624023438, -1.5308390855789185, 1.9753358364105225, 2.0438296794891357, 0.41190117597579956, -1.1570897102355957, -1.2252719402313232, 1.2099884748458862, -0.2811189293861389, 0.0025448203086853027, 0.8341712355613708, 0.7949678897857666, 1.002338171005249, 1.578660249710083, 0.9506443738937378, 1.191890001296997, 0.8515230417251587, -0.7653958201408386, -0.8691813945770264, -0.2295377254486084, -2.794719696044922, -0.5482260584831238, -0.6081553101539612, -0.9826884269714355, -2.974071502685547, 3.845627546310425, 0.5771481990814209, 1.5231106281280518, -1.6842358112335205, 0.12291769683361053, 2.884974956512451, -0.5506203174591064, 0.6850274205207825, -0.7726640701293945, -1.7560532093048096, -1.2118028402328491, -0.46337080001831055, 0.6241658329963684, 1.5701932907104492, 0.43030786514282227, -1.441942811012268, 0.6433312296867371, -0.5708974599838257, 0.7366909384727478, -1.4550201892852783, 0.5265820026397705, 1.2678167819976807, 0.7702460289001465, 0.8796921968460083, -1.4909169673919678, -1.4191054105758667, 0.11082601547241211, -0.368574857711792, 2.550172805786133, 1.3278905153274536, -1.003252387046814, -1.5214817523956299, 0.5802261233329773, -1.6578187942504883, -0.7168314456939697, -0.5789279937744141, -0.1680229902267456, 1.7547147274017334, 0.4017331600189209, -0.8029004335403442, 1.6894652843475342, -0.23374372720718384, -1.8753217458724976, -1.028738021850586, 0.35786688327789307, -0.6101212501525879, 0.9137879014015198, 1.0031671524047852, -0.9334995150566101, 3.0896201133728027, 1.3002607822418213, -1.7820781469345093, 0.10337918996810913, -1.0358309745788574, -0.6841620206832886, 0.9042894840240479, 1.3213046789169312, 0.6451713442802429, 2.644855499267578, -0.7065575122833252, -1.2227129936218262, -0.42341184616088867, 0.269596666097641, -2.3573250770568848, -1.599372386932373, 2.303813934326172, -3.3831355571746826, 0.5986098647117615, -0.5403730869293213, 0.018247783184051514, -1.7731003761291504, 0.5102801322937012, 0.5616664886474609, 1.0259217023849487, 1.0259934663772583, 1.757034182548523, 0.10124033689498901, -1.7986615896224976, -1.7356128692626953, 2.8352394104003906, 0.8210198879241943, 0.6977126598358154, 2.1420071125030518, 0.3557687997817993, 3.202502727508545, -2.9603381156921387, -0.6204912662506104, -1.7861032485961914, -0.40213537216186523, -2.23124361038208, -0.5776013135910034, 0.7302869558334351, 0.9394360780715942, -2.2458455562591553, -2.3260021209716797, -2.673886775970459, -3.751711845397949, 1.6268377304077148, 2.8335440158843994, -1.7223073244094849, -0.6148455142974854, -1.875136375427246, -1.0796757936477661, 0.15877372026443481, 1.907186508178711, 0.5835649371147156, 0.2810792922973633, 0.15873822569847107, 0.052572041749954224, 1.4092249870300293, 0.06396853923797607, -0.2544647455215454, 1.366878628730774, 1.1654008626937866, -0.6015106439590454, -1.605898141860962, 0.417906790971756, 1.968199372291565, -0.6841585636138916, -0.24217835068702698, -1.224224328994751, -2.056619167327881, 3.9957923889160156, 0.1359236240386963, -1.0560787916183472, 1.9005500078201294, 2.933802604675293, -0.618660569190979, 1.2000612020492554, -0.8597028255462646, 0.3004205822944641, 3.512637138366699, -0.2521679699420929, -1.266453504562378, 1.3668334484100342, -2.071737289428711, -1.2104692459106445, 0.8789761066436768, -1.4525442123413086, 0.7416043877601624, 1.5530493259429932, -1.5411944389343262, 0.7463586926460266, -0.6801551580429077, 0.6968899369239807, -2.2096667289733887, -1.2818703651428223, 1.2364814281463623, 0.9608508348464966, 1.3598123788833618, 0.7194607853889465, 4.063196182250977, 2.4640021324157715, 0.8141766786575317, -0.4686259627342224, -1.0245120525360107, 0.5775319337844849, -0.30274611711502075, -0.9710009694099426, -0.8302147388458252, -0.06358538568019867, 0.04359486699104309, 0.7265234589576721, -1.2489962577819824, 1.3633630275726318, 2.392507791519165, 1.492101788520813, 1.6446802616119385, 0.48974522948265076, 0.9364267587661743, 0.6913837194442749, 0.3065764605998993, 3.7745583057403564, -2.139263153076172, 1.0375698804855347, 0.22337234020233154, -0.3233110308647156, -0.2236384153366089, 0.44521641731262207, 0.06581354141235352, 2.9261271953582764, -0.003429025411605835, -1.8168824911117554, -2.6828372478485107, 0.8967923521995544, -0.6465635299682617, 1.0699870586395264, -0.47240740060806274, 0.8127875924110413, -0.5610283613204956, 1.0687551498413086, -2.7129108905792236, 2.7065436840057373, -1.575577735900879, 1.2145655155181885, -0.8264382481575012, -1.8409035205841064, -0.16919022798538208, 0.44321751594543457, 0.2190840244293213, -1.4418686628341675, 3.258831024169922, -1.629828691482544, -1.0940918922424316, 0.6165508031845093, -0.16793322563171387, -1.081182837486267, -3.039222240447998, 1.3227676153182983, 2.8954334259033203, -0.9518923759460449, -1.4557806253433228, 1.348017930984497, 0.35193291306495667, -0.5399871468544006, 1.2683298587799072, 0.29975536465644836, 1.4386751651763916, -1.2666199207305908, -0.8002040982246399, -1.3417692184448242, 1.8142635822296143, 0.8950159549713135, -0.10841917991638184, 2.1587936878204346, -0.5820878744125366, -0.6247576475143433, -2.8653764724731445, 0.5337110757827759, -0.7935039401054382, 0.5878759622573853, -0.8415517807006836, 0.046191275119781494, -0.2781732678413391, 0.40285801887512207, 0.6255917549133301, -1.388890266418457, 1.1627957820892334, 0.41689324378967285, -2.233693838119507, 0.006858646869659424, 1.6572234630584717, 1.407715082168579, -0.9674789905548096, -1.3404202461242676, -1.8120259046554565, -1.1905244588851929, 0.5422801971435547, 0.9979498386383057, 1.8889795541763306, 3.126227855682373, -1.1746797561645508, 1.1932501792907715, -0.1990727186203003, -0.291435182094574, 1.0176620483398438, 0.5230228900909424, 0.8102600574493408, 3.2216057777404785, -1.0104730129241943, -1.1432716846466064, -0.4624766707420349, -0.8561834692955017, 0.31187599897384644, -1.1843373775482178, 0.4034843146800995, 0.17033913731575012, -2.1797072887420654, 0.5742523670196533, 0.877799391746521, 2.609135627746582, 1.4251421689987183, 0.327969491481781, -0.6631099581718445, 1.544075846672058, -0.8888528943061829, -0.84865802526474, 0.9684098958969116, 3.1808788776397705, -0.8945252895355225, -1.2614551782608032, -1.528688669204712, 0.40095722675323486, 0.4285917282104492, 1.2694107294082642, 0.28882330656051636, -1.4592019319534302, 0.9543601274490356, 1.0497584342956543, -0.4285501539707184, 1.3123806715011597, -0.6832014322280884, -0.38241204619407654, 1.5585789680480957, 0.7819080352783203, -0.42225027084350586, -0.4847453236579895, 1.7735687494277954, -0.3156822919845581, 0.7152820825576782, -1.7762393951416016, 1.1941934823989868, 1.053006649017334, -1.7336699962615967, 1.5608071088790894, 0.7414364218711853, -0.7667944431304932, -0.5664864778518677, 0.028492242097854614, -0.260032594203949, -1.990213394165039, -0.0605015754699707, -2.042987585067749, 0.3926682770252228, -0.3581150770187378, -2.133023262023926, -0.30740880966186523, -0.29197388887405396, 1.5528295040130615, -1.362792730331421, -0.6272242069244385, -1.086974024772644, 0.6506872773170471, -0.5466553568840027, 2.538405418395996, -1.6092679500579834, -0.7550972104072571, 2.424135684967041, 1.580645203590393, 0.29029545187950134, 0.2159976363182068, 2.4055116176605225, 1.7780978679656982, 0.9508336782455444, -0.4463658630847931, 2.706469774246216, -0.7932745218276978, -0.9801866412162781, 1.5594689846038818, 0.4218403100967407, -2.685028076171875, 1.0586910247802734, -0.4625767469406128, 2.5837464332580566, 2.5412721633911133, -0.8316987752914429, 2.779468059539795, 1.319852590560913, 0.1718667447566986, -2.7837746143341064, 0.4685012996196747, 0.06271219253540039, -1.1284703016281128, 0.9742588996887207, -0.766541600227356, -1.3217865228652954, -0.7369577884674072, 0.3664299249649048, 1.289028525352478, 0.32692646980285645, -0.005465373396873474, 1.2253894805908203, 1.8949427604675293, -0.4732933044433594, 1.7162055969238281, 0.1778654307126999, -2.618903160095215, -0.23487024009227753, -0.5971490144729614, -0.6447033286094666, -0.12848062813282013, -2.0321903228759766, 1.1905062198638916, -1.7669118642807007, 0.6770886182785034, -1.3228962421417236, 1.478216528892517, 0.39292097091674805, -0.7380359172821045, -0.9881124496459961, 1.9458335638046265, -0.09509646892547607, -0.04642009735107422, -0.1343289613723755, -0.6277709603309631, -0.2673081159591675, 2.0650815963745117, 0.6439192295074463, 1.4690678119659424, -3.1009788513183594, 0.7592812776565552, -0.019896909594535828, 0.954423725605011, 0.1091228723526001, 0.7290518283843994, -0.7918574213981628, 1.432827115058899, -0.38941800594329834, 0.4708975553512573, -0.12561985850334167, -0.9471161365509033, 2.248417854309082, 1.5255844593048096, 0.5959408283233643, 0.25274503231048584, 1.9418563842773438, -1.7253365516662598, -0.6881822347640991, 1.2598212957382202, 2.882930278778076, -1.6342967748641968, -1.6672059297561646, 2.557023525238037, 0.7924748659133911, -0.9862502813339233, 2.9961061477661133, 1.5292282104492188, 1.6885863542556763, -1.5554941892623901, -0.13930749893188477, 1.226747989654541, -0.4886842370033264, -1.2222650051116943, -0.09063690900802612, 0.8218334913253784, -0.461076945066452, 0.8875688910484314, -0.042831599712371826, -0.8361604809761047, 0.49271896481513977, -1.5774438381195068, 1.7632598876953125, 0.2570497989654541, 2.100693941116333, 1.5979785919189453, -0.6502900123596191, 0.972845733165741, -1.3581979274749756, -0.2071009874343872, -1.353609561920166, -1.7454955577850342, -0.1872888058423996, -0.8046258091926575, -1.6965765953063965, -1.6202856302261353, 0.3217809796333313, 1.2998979091644287, 0.38380157947540283, -0.6445820331573486, 0.4163051247596741, -1.6691725254058838, -1.033285140991211, 1.6049940586090088, 0.30954450368881226, 0.04647564888000488, 1.4169071912765503, -2.9764504432678223, -1.067795991897583, 0.6925783753395081, 0.14187467098236084, -1.8649275302886963, -0.5194772481918335, -0.9463204145431519, 0.5175998210906982, 0.6625097393989563, -1.064119815826416, 1.2629642486572266, 0.2272229790687561, -1.0760936737060547, -0.8986334800720215, 1.6210479736328125, -0.2118796557188034, -0.3287622332572937, -2.812167167663574, -0.6228976845741272, -0.4836083948612213, -1.4240496158599854, -0.09000962972640991, 0.05516946315765381, 0.5305914282798767, -0.6759213805198669, 0.5996849536895752, -0.8289921283721924, 0.5546397566795349, 0.796120822429657, -0.3161735534667969, -2.4173583984375, 0.09551963210105896, -2.147437810897827, 0.4228321313858032, 0.3566184937953949, 0.9982879757881165, -0.4894140958786011, -1.7712457180023193, -1.563960313796997, 1.8750051259994507, 0.14510512351989746, -0.44948503375053406, -1.3893219232559204, -0.7270904183387756, 1.2499696016311646, -2.2777042388916016, -1.0900049209594727, -1.14146089553833, 2.004878044128418, -2.6314620971679688, 0.048007532954216, -0.9738398790359497, 1.8062002658843994, 1.036196231842041, 2.5614418983459473, -1.4147056341171265, 0.5424376130104065, -0.0035688281059265137, 1.6267400979995728, -1.7107243537902832, -2.0143916606903076, 1.2019909620285034, 1.3075066804885864, 0.8418437242507935, 1.0108498334884644, 1.3846163749694824, 0.7784793376922607, -1.8058907985687256, -0.27979278564453125, 0.810325562953949, -0.6892279982566833, -1.654076337814331, 1.6124078035354614, -2.6569902896881104, -0.26521754264831543, 0.42105481028556824, 2.3710246086120605, -0.5146101713180542, 1.5689022541046143, -3.061431407928467, -0.43054163455963135, 0.815650463104248, 3.816189765930176, 1.3020342588424683, 0.49763384461402893, -0.09841406345367432, -0.05132552236318588, -0.5201020240783691, -1.5764375925064087, 2.876288890838623, 0.07297968864440918, 1.9662017822265625, -0.7838782668113708, 1.8780608177185059, -2.017630100250244, 1.1516661643981934, -0.5782917737960815, -0.010701477527618408, 3.565368175506592, -0.28451526165008545, 0.732657790184021, 1.9845935106277466, 1.150669813156128, -0.9526214599609375, -1.9413589239120483, -0.7853388786315918, 3.1376516819000244, -0.8176979422569275, -0.050281822681427, 0.06355619430541992, -2.616204261779785, -0.05592033267021179, 1.0118627548217773, -1.260640263557434, -0.18983516097068787, 0.1222589910030365, 0.12718479335308075, 1.9755346775054932, 0.29700344800949097, -0.23087358474731445, 0.2836572527885437, 0.9926866888999939, 1.4085898399353027, 0.5102629661560059, -0.41972723603248596, 1.6086766719818115, -2.2643702030181885, 0.9633094668388367, 0.2684062421321869, -1.2669734954833984, 2.368546485900879, 0.6170693635940552, 0.4227760434150696, 2.7409095764160156, 2.671837329864502, 0.10928305983543396, 0.31323426961898804, 1.1573907136917114, 1.173898696899414, -1.5692737102508545, 0.47787782549858093, 1.4555401802062988, -2.8920705318450928, -0.828077495098114, -1.5773274898529053, -0.6172200441360474, -1.7060303688049316, -1.4887970685958862, -0.5993244647979736, -1.63100004196167, 2.678260564804077, -1.0269097089767456, 0.6854682564735413, -0.020646393299102783, 0.8799004554748535, 2.444429397583008, 0.4792119264602661, -3.3460636138916016, 2.6256422996520996, -0.3847195506095886, -1.3889470100402832, 0.3190256357192993, -0.5396453142166138, 1.156522274017334, -0.5806940793991089, -0.37394979596138, -0.6002184152603149, -1.7950718402862549, -2.691734790802002, -1.2947595119476318, 2.331084728240967, 0.025568604469299316, 2.715872287750244, 2.4321413040161133, 0.6872081160545349, 1.6987584829330444, -0.525656521320343, 0.5952013731002808, -1.1466624736785889, -2.000589609146118, 0.39250633120536804, -0.34236058592796326, -2.4645192623138428, 0.8113560676574707, -1.2905210256576538, 0.23640549182891846, 0.5602837800979614, -0.6413132548332214, -1.7568469047546387e-05, -2.1638152599334717, -1.8426744937896729, -2.1541695594787598, 0.7516780495643616, 1.3859894275665283, 0.6203128099441528, -2.7946720123291016, -0.09541521966457367, 3.365255355834961, 0.955048680305481, -1.6863517761230469, 0.4517577290534973, 1.295440912246704, -1.461679220199585, 0.23553578555583954, -0.5648032426834106, -2.245950698852539, 0.5262237787246704, -1.2478361129760742, 0.21111559867858887, 2.9392457008361816, 0.9004530906677246, 0.5467166900634766, 0.4076862633228302, 1.2166180610656738, 1.8188421726226807, -0.1955089569091797, 0.49839019775390625, -1.1464966535568237, 0.4863423705101013, -0.09582817554473877, 0.25671112537384033, 0.8208123445510864, -0.5946485996246338, -1.4200856685638428, -0.6083225011825562, -0.2513503432273865, -1.227533221244812, 1.7819656133651733, -0.07638192176818848, -0.6696711778640747, 1.5466232299804688, 0.2106887698173523, 0.8880030512809753, 1.9877302646636963, 1.5183818340301514, -1.582523226737976, -0.18966539204120636, 1.2476372718811035, -1.177683711051941, 0.8161306381225586, -0.9418289065361023, 1.1111456155776978, -2.5743913650512695, 0.8110150694847107, 3.5919699668884277, 2.340534210205078, -0.10190397500991821, -0.492617666721344, -1.296109676361084, -0.34838998317718506, -0.45093029737472534, -1.0832617282867432, 1.290419340133667, 0.004870646167546511, -0.8112471699714661, 0.9031747579574585, -0.050469398498535156, 2.7092926502227783, -0.8066190481185913, 3.067913055419922, -2.2185850143432617, -0.290180504322052, -2.0913939476013184], "dtype": "float32"}}, {"a": [{"__nd__": [0.48326489329338074, -0.7211804389953613, 0.41588231921195984, -0.5739192962646484, -0.34623950719833374, -2.0778133869171143, -0.3172813653945923, 0.8657210469245911, -0.7996253371238708, -0.13353419303894043, 1.7474758625030518, 1.2171540260314941, -1.1173275709152222, 1.2462679147720337, -0.2673420310020447, 0.4836564362049103, -0.014048201963305473, -1.6491858959197998, -1.7572818994522095, -0.7535382509231567, -1.1550079584121704, -0.7237728834152222, -0.13306613266468048, 1.2265068292617798, 0.2305092066526413, 1.2107094526290894, -1.3786561489105225, -0.1801043301820755, -1.5719623565673828, 0.8703122735023499, 0.4656153619289398, 0.4907141327857971, -1.0973230600357056, -0.04981006681919098, -0.49526458978652954, -0.022208740934729576, -1.5223740339279175, -0.03291517496109009, -0.4474334716796875, 0.519927978515625, -0.6453513503074646, -0.2930307984352112, 0.37113308906555176, -0.5087856650352478, 0.4153803586959839, -0.2079470008611679, 0.20746180415153503, -1.7924652099609375, 1.3142609596252441, 1.6820871829986572, 0.7675224542617798, -0.7495462894439697, -0.07237833738327026, 0.7563469409942627, 0.8554735779762268, -0.6531068682670593, 0.5406090021133423, 1.4661296606063843, 1.9557956457138062, 0.4495149850845337, -0.32341498136520386, 0.6777169108390808, 0.9533652067184448, -1.3811835050582886, -0.7721478343009949, 0.4070521593093872, -0.22388732433319092, 0.13499890267848969, 0.36884820461273193, -0.5492072105407715, -1.4775128364562988, 0.292782187461853, -1.1813485622406006, -0.7709261178970337, 0.5867502093315125, 1.4978604316711426, 0.7232785224914551, 2.1022510528564453, 1.2031854391098022, -0.49251672625541687, 0.1576528698205948, -0.5415850281715393, 0.8378368020057678, -1.2214233875274658, 0.5014380812644958, 1.0970652103424072, -0.950465977191925, 1.6652593612670898, 0.7910640835762024, -1.0578269958496094, 0.3089342415332794, 0.7443476915359497, -0.8177950978279114, 0.12187685817480087, 1.331787347793579, -0.882805347442627, 0.5737980604171753, -0.6351064443588257, -0.9146061539649963, -1.4146769046783447, -1.0566351413726807, 0.9735317826271057, -0.9656367301940918, 1.508845567703247, 0.17350779473781586, -2.02710223197937, -0.5036005973815918, -0.8879009485244751, 0.009963513351976871, 0.15154968202114105, -0.30249670147895813, -0.8121037483215332, -1.089396357536316, -0.4887881577014923, -0.47879183292388916, 2.0787460803985596, -0.04483223706483841, -0.05035998299717903, 0.710731029510498, 0.18859080970287323, -1.9696389436721802, 1.1315847635269165, 1.013471245765686, 0.7113379240036011, -1.868191123008728, 0.43142011761665344, 0.26829931139945984, 1.2536627054214478, 0.523347020149231, -2.323676586151123, 0.8610715866088867, 0.8246503472328186, 2.205387592315674, 1.1346465349197388, 1.856562614440918, -1.2417917251586914, 0.23268240690231323, 0.11677207052707672, -0.09062755107879639, -1.963202953338623, 1.2223390340805054, 1.5555869340896606, -1.500151515007019, 0.518744945526123, -0.6260367035865784, -1.5848428010940552, -0.2802019715309143, -0.6364507079124451, -0.9675140380859375, -2.0135090351104736, 0.8401381969451904, 0.9606272578239441, -1.91000497341156, -0.9312173128128052, -1.9879484176635742, -0.7001147866249084, -0.44312363862991333, -1.152610182762146, 0.30389705300331116, 0.8018683791160583, -1.4095394611358643, -1.9691768884658813, -0.594234049320221, 0.43832263350486755, 1.8385719060897827, -1.200111746788025, 0.9469679594039917, 0.25321903824806213, -0.021386710926890373, 0.1288701891899109, 0.4541695713996887, 1.7483696937561035, -1.330590009689331, 1.2953383922576904, 1.8054710626602173, 1.0294489860534668, 0.8039478063583374, 0.685179591178894, -0.09075930714607239, -0.6589780449867249, -0.4620967507362366, -0.8389084935188293, 0.8525480031967163, -1.8291031122207642, 0.2832440733909607, 1.4287699460983276, 1.382243037223816, -0.7133863568305969, 0.15348298847675323, -0.541546642780304, -0.20373494923114777, -1.399441123008728, 2.239443778991699, -0.979159951210022, 0.003057460766285658, -0.5568468570709229, -0.045008476823568344, 2.357349157333374, 1.6274869441986084, 2.771247148513794, -0.943389356136322, -0.5569377541542053, -1.3954088687896729, 0.07684507220983505, -1.7572978734970093, 0.6294273138046265, 0.5582964420318604, 0.22784069180488586, 0.5000090003013611, 0.04867924004793167, -0.0413382425904274, -1.1673277616500854, -0.49960705637931824, -1.5798007249832153, -0.18570829927921295, 1.4648768901824951, 0.021386265754699707, -0.31010758876800537, 0.9382150173187256, -0.7458981871604919, 0.29572194814682007, -1.0102550983428955, -0.4834251403808594, 0.17454339563846588, -0.04114919528365135, 0.370344340801239, 1.2617100477218628, 0.23082271218299866, 0.548084020614624, -0.15364240109920502, -0.9988102912902832, -0.5653295516967773, 0.4166162312030792, -0.35055360198020935, -0.3608919382095337, -0.13245202600955963, 0.6116851568222046, 1.0594083070755005, -0.11769454926252365, -1.74336576461792, -1.3866482973098755, -0.511242151260376, 0.4522985816001892, -0.21575386822223663, 0.5384694337844849, 1.1506402492523193, -0.45036524534225464, 0.688096284866333, -0.6068004369735718, 1.458106517791748, 0.9617907404899597, 0.04034275561571121, 1.1812318563461304, 0.021753473207354546, 0.42584019899368286, 1.2513281106948853, 0.11399421095848083, 0.2952338755130768, -0.20959778130054474, -2.120840311050415, -0.4789111912250519, -0.010368045419454575, -0.02328040450811386, -0.5654758214950562, 0.9393250942230225, -0.9160075187683105, -0.06934449821710587, 0.3915090262889862, 1.2352957725524902, 1.6084294319152832, 2.808499574661255, -0.09881212562322617, -1.2143489122390747, -1.5903303623199463, -0.40978503227233887, 0.029100602492690086, 0.561829686164856, 1.3157771825790405, 1.1291829347610474, 0.6408618092536926, -1.701501727104187, -0.4702149033546448, -0.8339658379554749, 0.4538382291793823, 1.3447743654251099, -1.0062583684921265, 1.4575191736221313, -3.1384854316711426, -1.5663528442382812, 0.7269322276115417, -0.9633917212486267, 2.052238702774048, 1.5406510829925537, -0.3753684163093567, -1.2885425090789795, 0.6057853698730469, -0.848837673664093, 0.5071115493774414, -1.8063231706619263, 0.35979846119880676, 0.6352370381355286, 0.6266449689865112, -0.8258235454559326, -0.1817118376493454, 2.2331557273864746, -0.5575158596038818, 0.18677844107151031, -1.0189915895462036, -2.530344009399414, 0.25882482528686523, 0.38586074113845825, -1.027890682220459, 0.1768895387649536, 0.521928071975708, -0.1862088292837143, -0.8645713329315186, 0.1299731284379959, -0.6075111627578735, 1.2847707271575928, -0.2981835901737213, 0.8444227576255798, 1.5606790781021118, 0.642094612121582, -1.1695716381072998, -0.5418838858604431, -0.9501144289970398, -0.746863603591919, 0.09906259179115295, 1.7583057880401611, 0.8436247110366821, -1.0442993640899658, -0.6418735384941101, -0.9612685441970825, -0.07036074250936508, 0.8894429802894592, 0.6025742888450623, 0.16070900857448578, -0.06512264907360077, -0.15288324654102325, -0.4797358214855194, 0.0613420195877552, 1.1965429782867432, 0.621043860912323, -0.24585242569446564, 0.7552827000617981, -0.08328129351139069, 0.3434986174106598, -0.17698363959789276, 1.1481932401657104, -1.3864156007766724, 1.2613730430603027, -1.01338529586792, 0.03322739899158478, -1.1443320512771606, -0.19132846593856812, -1.2252442836761475, 0.0916074588894844, 2.5356030464172363, -0.14981834590435028, 1.8535466194152832, -0.9319175481796265, 1.1351509094238281, 2.6644582748413086, 0.06729938089847565, -0.08956902474164963, -0.6971737742424011, 1.1182804107666016, 0.3119378685951233, -0.17520445585250854, 1.1010234355926514, -0.38962405920028687, -0.31133002042770386, 1.1121785640716553, -0.3883053958415985, -0.7865167260169983, -0.6987916231155396, 1.1454921960830688, 0.5073490738868713, -1.0582177639007568, -0.46076467633247375, -0.5829914212226868, -0.25948241353034973, 2.2580220699310303, 0.4747976064682007, -0.5503097176551819, -0.38025519251823425, -0.23318403959274292, -1.6505521535873413, -1.7064073085784912, 2.06451678276062, 0.8660398721694946, -0.28238019347190857, 0.33526715636253357, 0.7144753336906433, -0.8907259702682495, -0.37578389048576355, 0.7282307147979736, -0.7479547262191772, 0.4482195973396301, -0.6186960935592651, -0.25045377016067505, 0.08294382691383362, 1.1497106552124023, 1.4739376306533813, 2.336298942565918, 0.837158739566803, 1.0111889839172363, 0.955464780330658, 1.8528578281402588, 0.0506754107773304, -1.3794841766357422, 0.39179345965385437, -0.748836874961853, 0.9837918877601624, 0.40965795516967773, 0.9029496312141418, 0.18537256121635437, 0.2639808654785156, -0.7557933330535889, -0.5382480621337891, -0.48107317090034485, -0.19432634115219116, 0.05143182352185249, 0.8928578495979309, -1.0290775299072266, 0.706169605255127, -0.1094663068652153, -0.14399605989456177, -1.1303281784057617, 1.0195989608764648, -1.5484741926193237, 2.6058342456817627, -0.886147677898407, -0.419866144657135, 0.33662962913513184, -0.06989793479442596, -0.05210663750767708, 1.9412710666656494, -0.5448570251464844, -0.12103740870952606, 0.10798303782939911, 1.6165319681167603, 2.2294349670410156, -2.2654900550842285, -0.8801995515823364, 2.1726577281951904, 0.7761598229408264, 0.7112668752670288, -0.5720806121826172, 1.4616408348083496, -0.9058987498283386, 1.8220188617706299, -0.7083683609962463, 1.163254976272583, 0.6536883115768433, 0.11358051747083664, 0.02708793617784977, 0.38609635829925537, 1.6849251985549927, -0.15609224140644073, 1.3517026901245117, -1.4466736316680908, -0.4920617341995239, -0.5823517441749573, 0.6029844880104065, 0.16370020806789398, -0.3932374119758606, -0.5845683813095093, 1.1381253004074097, -0.9160650968551636, -0.07724815607070923, 0.4124859869480133, -3.3527963161468506, 0.5065328478813171, 0.9870103597640991, -1.0680018663406372, 1.3948004245758057, -0.9235143065452576, 0.7039276361465454, 0.7994334101676941, -0.732501208782196, 0.20572833716869354, 0.9512271881103516, -0.6871078014373779, -1.891121745109558, 1.0655324459075928, 0.6506727337837219, -1.0326308012008667, 0.9046593904495239, 1.4485903978347778, -0.9794299602508545, 0.04979110509157181, 1.2437989711761475, -0.9804839491844177, 1.0677911043167114, 0.9214083552360535, 1.3009588718414307, 0.7360668778419495, 1.3400496244430542, -0.3074527084827423, -1.2415556907653809, -1.4963964223861694, -0.3529582917690277, 0.5361031293869019, -0.7460150122642517, -1.1172116994857788, -1.0621272325515747, -0.6162950992584229, -0.04169810935854912, -0.9039574861526489, 1.7266981601715088, -0.28115731477737427, 0.2703067362308502, 0.8743164539337158, 0.9019136428833008, -0.4006790220737457, -0.5508975982666016, 1.9373327493667603, 0.11824177205562592, -0.7854540944099426, -0.46449339389801025, -0.15262778103351593, 0.6999870538711548, -0.43488264083862305, 0.3685712516307831, 0.5061702132225037, 0.4800470769405365, 0.41905564069747925, -1.467633605003357, -0.16014277935028076, 1.028006672859192, 0.18588267266750336, 0.4771629273891449, -1.5181233882904053, -0.6875091195106506, 0.24151970446109772, -1.2394272089004517, -0.384482741355896, 0.6152455806732178, 1.1651479005813599, 0.42142051458358765, 1.3326852321624756, -0.09882302582263947, -1.2292146682739258, -0.3793952465057373, 0.922256350517273, 0.8069363236427307, -0.14389526844024658, -0.13037067651748657, 0.3699250817298889, -1.5925891399383545, 1.0850000381469727, -0.5616236925125122, -0.4286904036998749, 0.007346730213612318, 1.1438403129577637, -0.6903696656227112, -0.9810991287231445, 0.791667103767395, 0.5217692255973816, -0.9521207213401794, 2.151064872741699, 0.959784984588623, 0.2145438939332962, 0.17450782656669617, 1.1231672763824463, -0.27567481994628906, 0.03851071745157242, -0.6724504828453064, 0.023370767012238503, 1.3673524856567383, 0.7108337879180908, 1.1958792209625244, 0.8659869432449341, -0.9763870239257812, 2.2616806030273438, -0.6982607841491699, 0.6454510688781738, -2.1488425731658936, -0.8071166276931763, -0.5638043880462646, 0.027999885380268097, -1.1612006425857544, -0.7334554195404053, 1.5013190507888794, 0.5088381767272949, -0.8529379963874817, -1.3292720317840576, 0.22173449397087097, -1.1397374868392944, 0.1512748748064041, 0.10461743921041489, 0.13903383910655975, 0.31710508465766907, -0.6570686101913452, 0.4608355462551117, -0.4219161868095398, 1.3684929609298706, 0.8592966794967651, -2.072401762008667, -0.22980014979839325, 0.8286316394805908, 1.0597485303878784, -0.12365403771400452, -0.5342727303504944, -1.5441851615905762, 1.0496041774749756, 0.5236783623695374, 1.2607170343399048, 0.9348821043968201, 1.411261796951294, -1.3901689052581787, 0.8224038481712341, -1.1465853452682495, 0.7994996309280396, 0.38161221146583557, -1.1180431842803955, -0.9656907320022583, 0.8132553696632385, 0.8072890043258667, -0.4665070176124573, 0.8960568308830261, -0.6899233460426331, 0.5578076243400574, -0.4507579207420349, -0.3879203498363495, -0.43313461542129517, -1.4912863969802856, 0.5462749600410461, 1.0220662355422974, 1.536422610282898, -0.7893920540809631, -0.8988518714904785, -0.4312620162963867, -0.0660795047879219, -1.1271740198135376, -0.5836421251296997, -0.33875083923339844, -0.6810558438301086, -0.35332050919532776, -2.080451488494873, -0.44459590315818787, 0.040454983711242676, 1.105663537979126, 0.2180044949054718, -0.4294391870498657, 0.479053258895874, -2.2404491901397705, 0.6595479249954224, 1.0920114517211914, -1.0911420583724976, 1.4604758024215698, 1.2943092584609985, -0.8833988308906555, -1.0958553552627563, -0.09476228058338165, 2.162635326385498, -0.17942999303340912, -2.0371646881103516, -1.4042719602584839, 0.8376793265342712, 0.28413328528404236, -0.49170422554016113, 1.3391494750976562, -0.24092301726341248, -0.450919508934021, -0.6189849376678467, 0.19671615958213806, 3.639613151550293, 0.15639346837997437, -0.33878546953201294, -0.880397379398346, -1.1330573558807373, -1.9137911796569824, 0.4834246039390564, -1.6055430173873901, 1.2202287912368774, -1.0197455883026123, 0.8287243247032166, -0.1981458067893982, 0.8684176206588745, -0.20628675818443298, -2.725173234939575, 0.08160459250211716, -0.9367957711219788, -0.6152415871620178, -0.41422969102859497, -0.0041124229319393635, 2.03262996673584, -2.120572805404663, -0.4002920091152191, 0.3437834084033966, -1.0767658948898315, -0.8153998255729675, -0.994993269443512, 1.187962532043457, 0.8973038196563721, -0.31517279148101807, -0.9000333547592163, 0.5077081322669983, 0.5088571310043335, -0.10208725184202194, 0.6421476006507874, 0.27084141969680786, 1.2043174505233765, -1.2386116981506348, -0.44297704100608826, 0.6593571901321411, 0.5517054796218872, 0.32935696840286255, -1.1097561120986938, -0.38238221406936646, -0.16647189855575562, -0.5883795022964478, 0.4830324053764343, 0.7743651866912842, 0.4341682493686676, -0.49490100145339966, 1.3161664009094238, 0.12700141966342926, 0.19524094462394714, 0.957818329334259, -0.804419219493866, -0.7963696718215942, -0.9121764302253723, 2.9148952960968018, -0.8485068082809448, -0.500344455242157, -1.0856022834777832, 1.2650142908096313, -0.3214448392391205, -0.9333150386810303, -1.4288878440856934, 1.3707001209259033, 0.6788358688354492, 0.2909734547138214, -0.16666947305202484, -1.389559030532837, 1.1572461128234863, -0.9465028643608093, -0.01725137233734131, -0.9627450108528137, 0.8174695372581482, -1.8556549549102783, 0.39513447880744934, -0.3420015573501587, 0.2901381254196167, 0.7135117053985596, -0.606657087802887, -0.24446021020412445, -0.9176942110061646, -1.063554286956787, -0.5850608348846436, 0.27962321043014526, 1.2824710607528687, -1.4912362098693848, 0.5031449198722839, -0.7258977293968201, 0.17277826368808746, -0.3153090476989746, 0.881392240524292, 1.4923886060714722, 0.6540182828903198, 2.6974294185638428, 0.09639357030391693, -0.9623481631278992, 0.6133851408958435, 0.6787269711494446, 0.3466182053089142, 1.8776209354400635, 0.9190772175788879, -0.462988018989563, 1.532386064529419, 0.975817084312439, 2.0008459091186523, 0.31835436820983887, -1.1016913652420044, -0.5974804162979126, 0.2248874455690384, 0.2722648084163666, 0.30352991819381714, 0.2678263783454895, -0.9082090258598328, 0.6493481397628784, 0.2226063311100006, -1.3786821365356445, 0.30292269587516785, -1.536217451095581, -1.4621578454971313, -0.39886531233787537, 0.45989376306533813, -1.7937382459640503, -1.0989619493484497, -0.6611785888671875, -0.021918324753642082, -1.4525299072265625, 1.2530938386917114, 0.27091822028160095, -1.1391490697860718, 1.0801104307174683, -1.384405493736267, -0.09270364791154861, 1.6995844841003418, -0.7360063195228577, -0.5596967935562134, -1.2443528175354004, -0.9776690602302551, 1.35218346118927, -0.011010143905878067, -0.059827495366334915, -1.5644361972808838, 0.1386736035346985, 0.2914714515209198, 0.0992894321680069, 0.47561612725257874, -0.013247297145426273, -0.1610589325428009, 0.35231813788414, -0.8499369025230408, 0.07471171021461487, 1.504951000213623, 1.1631858348846436, -1.7441669702529907, -1.8026759624481201, -1.4538394212722778, -0.08169003576040268, 2.1052639484405518, -0.8780652284622192, -0.6546463966369629, 2.374199390411377, -0.2432866096496582, -1.0665764808654785, 0.6086708307266235, -1.2004789113998413, -2.080817937850952, -0.12347760051488876, -1.7640224695205688, -0.948307991027832, -0.06299848109483719, -1.3838629722595215, -1.3052804470062256, -0.6241784691810608, 0.3523801565170288, 1.6133400201797485, 1.9764299392700195, -2.1598942279815674, -0.4315760135650635, -0.6761763095855713, -0.6357048153877258, 1.0210272073745728, 0.2481507658958435, -1.200703740119934, -1.4901604652404785, 0.3648982644081116, 0.43321341276168823, 1.593216061592102, -1.196742296218872, 0.5234568119049072, -0.39251741766929626, 0.33610403537750244, -0.28051814436912537, 1.5528923273086548, 0.856318473815918, -0.2491556853055954, -0.3205825090408325, -1.0581295490264893, -0.20684686303138733, -1.476210117340088, -0.2503204941749573, 0.19405627250671387, -1.7814815044403076, -1.4019299745559692, -1.674971103668213, -0.33610817790031433, 1.8176027536392212, 2.160048007965088, 0.2567269504070282, -0.5905247926712036, -0.5502006411552429, -0.6420038938522339, 0.49413222074508667, 0.056013502180576324, -0.18745633959770203, -0.007082771044224501, 1.6148144006729126, 1.010658860206604, -0.8060145974159241, -0.26320815086364746, 0.622151792049408, 0.6609097123146057, 2.3764266967773438, 2.2105019092559814, -0.6411411762237549, 0.8218389749526978, -0.09342716634273529, 1.0907926559448242, -1.5691237449645996, 0.9827834963798523, 0.6935949325561523, -0.21918392181396484, -1.590378761291504, -1.6714776754379272, -1.4211384057998657, -0.4185654819011688, -0.004237407352775335, 0.12630969285964966, 0.7689464688301086, 2.2993102073669434, 0.16577890515327454, 0.4287932813167572, 0.37570810317993164, 0.7027305364608765, 0.1796390265226364, -1.0667996406555176, 0.6001651883125305, 2.31455397605896, 1.174471378326416, -1.6428853273391724, -0.6723411679267883, 0.013224114663898945, 0.2017320692539215, -1.1111878156661987, -0.11021163314580917, -1.4753661155700684, -1.7157235145568848, -0.4280967712402344, 1.423553466796875, -1.4960857629776, -1.1262556314468384, 0.2329655885696411, -1.2760392427444458, -0.6719607710838318, 0.49768149852752686, 1.5180144309997559, -0.38098281621932983, -0.42393141984939575, -1.6132237911224365, 1.008386492729187, 0.37571126222610474, 0.6783825159072876, -1.062978744506836, -0.2628999948501587, -1.786715030670166, 1.1383329629898071, 0.27563390135765076, -0.3480401337146759, 0.4567142724990845, -0.12757608294487, -0.057808708399534225, 0.0038760402239859104, -1.4188129901885986, -2.5946104526519775, -0.2731112241744995, -0.5999009609222412, -0.28359121084213257, -0.6293429732322693, 0.31707248091697693, 1.9482197761535645, 0.46071431040763855, 0.10243698954582214, -1.1392629146575928, -0.06311838328838348, 0.043596211820840836, -0.8417474031448364, 0.5971845984458923, -1.6676279306411743, 0.1968558430671692, -0.690872311592102, 0.0904860720038414, 0.19484169781208038, 0.23942215740680695, -1.3783940076828003, -0.36468183994293213, -0.3423353433609009, -0.2157338410615921, -0.8913947939872742, -0.7443090677261353, 0.12751974165439606, -0.9244860410690308, 0.20458698272705078, -1.415458083152771, 0.378339558839798, 0.8485642671585083, 0.42190492153167725, -1.1607753038406372, -0.4920194745063782, 0.6655924320220947, -0.6082242131233215, -1.231371521949768, -0.04349891096353531, 0.19470679759979248, -0.38788267970085144, 2.1800429821014404, 0.7724636793136597, -1.0003870725631714, -0.7766091227531433, -0.6986603736877441, -0.018572473898530006, 1.0328408479690552, -0.3075680434703827, 0.7181767821311951, 0.28341278433799744, 0.06413474678993225, 0.5647956132888794, -0.4325208067893982, -0.9305081367492676, -0.6651239395141602, -1.6516309976577759, 0.8705592751502991, -0.6745656728744507, -1.2481797933578491, -0.01986459270119667, 0.1854674071073532, 0.7328740358352661, 0.953942060470581, 0.3210880756378174, 0.8292558193206787, -1.5438997745513916, -0.14195005595684052, 1.1808596849441528, -1.213172435760498, -0.4682404100894928, 1.3194085359573364, 0.3717067539691925, -0.7434502840042114, -0.4160163402557373, -0.5741516947746277, -0.426230251789093, -0.24133437871932983, -1.2928946018218994, 0.28162339329719543, -0.9461672902107239, 0.6889207363128662, 1.0245389938354492, 0.1701202392578125, -1.1578807830810547, 0.9398496747016907, -0.33227837085723877, 0.037995245307683945, -0.7114396095275879, 0.028787462040781975, 0.0922398567199707, 2.191882848739624, -0.5025194883346558, -1.1440370082855225, 0.1736535280942917, -0.6995184421539307, -0.8916138410568237, -1.077560305595398, 0.41814061999320984, -2.4850754737854004, 0.624970555305481, 0.11090369522571564, -0.16813983023166656, 0.7216642498970032, -0.04093161225318909, -1.3447651863098145, 0.34853395819664, -0.4625200927257538, -0.5724227428436279, 1.1207259893417358, -1.4470511674880981, -0.12248431146144867, -0.9978784322738647, 0.3345943093299866, -1.7029763460159302, 0.08893522620201111, -0.6431402564048767, 1.4291926622390747, -0.23831334710121155, 0.22478821873664856, 0.10652083158493042, -0.5524154305458069, -0.9473171830177307, -1.315811038017273, -1.0513726472854614, -0.7331823110580444, 0.11500398069620132, -0.44636648893356323, 0.3423648178577423, -1.6221786737442017, 0.5971764922142029, 0.6597110033035278, 1.7958238124847412, -0.07754763960838318, 1.3386270999908447, -2.1235945224761963, 0.9040249586105347, 0.26506203413009644, 1.2745556831359863, 0.975675642490387, 0.407380074262619, 0.7868404984474182, -0.34196069836616516, -1.280843734741211, -2.7330307960510254, 3.26857328414917, 0.14208947122097015, 0.8062147498130798, 1.2609162330627441, -1.0479315519332886, -0.2222161591053009, 1.2930124998092651, 0.9894261360168457, -0.07718449085950851, 1.0186289548873901, -0.1929590106010437, 0.697236180305481, 0.8199107646942139, -0.5289666652679443, 0.7822444438934326, 0.3672538697719574, -0.2917752265930176, -0.4179728329181671, -2.3363091945648193, 0.4610106348991394, 1.0358842611312866, 0.5408637523651123, -0.48023173213005066, 0.5106614232063293, 0.23177801072597504, -0.4075218439102173, 0.6319003105163574, -0.9870942234992981, -1.299418568611145, 0.4515596926212311, -0.7627937197685242, 1.2402658462524414, 1.402248501777649, 0.4976302981376648, 0.7174010276794434, 0.0607774518430233, -0.4422418475151062, -2.036799430847168, 0.209707573056221, 0.2582012414932251, 0.5424681901931763, 0.801980197429657, -0.003104831324890256, -0.8224583864212036, 0.1685669720172882, -2.832728385925293, -0.3411436676979065, -0.509253203868866, -0.8671531081199646, 0.2611072361469269, 1.9786970615386963, -1.1501808166503906, -0.37674880027770996, -0.04428549483418465, -0.858237087726593, 0.016123268753290176, 0.37539514899253845, 0.9532073736190796, 0.8984556198120117, 0.3697969317436218, 0.13472585380077362, -0.9021836519241333, 0.20698429644107819, 0.9997396469116211, -1.0804438591003418, 0.769256055355072, -3.629317283630371, -0.3657376170158386, -0.18456225097179413, -0.6361594200134277, 0.4401093125343323, -2.029909372329712, -0.49313056468963623, -0.5917028784751892, -1.574022889137268, 1.2704849243164062, 1.2939157485961914, 1.7309887409210205, -0.3274831473827362, -0.3534166216850281, 0.25037911534309387, 0.3501679599285126, -0.957439124584198, 0.3784755766391754, 1.1828620433807373, 0.48376116156578064, -0.2709324061870575, 1.3226109743118286, 0.4715607464313507, -0.5565040111541748, -0.8935795426368713, -0.30297815799713135, 0.539856493473053, 1.8326975107192993, 0.522962749004364, 0.7481855750083923, 0.05307348817586899, -0.30224552750587463, 0.26410388946533203, -1.1876310110092163, -1.6048051118850708, -0.8785040378570557, 0.5815983414649963, -1.2181376218795776, 0.2473694384098053, 0.5272393226623535, -0.9044809341430664, 0.6941165328025818, 1.9720547199249268, -0.4170248508453369, -0.13071812689304352, -0.6999748945236206, 0.3527628779411316, -0.30301138758659363, 0.667885959148407, -0.8302959203720093, 0.009105756878852844, -0.24139034748077393, 1.5282703638076782, -0.3524591326713562, 0.5582848787307739, -1.5124295949935913, 0.020011182874441147, -0.3591363728046417, -0.6407668590545654, -1.095175862312317, -0.8380985260009766, -0.89693284034729, -2.354804277420044, 0.7232966423034668, -0.31240102648735046, -0.5472434163093567, 0.6429659724235535, -0.26799139380455017, 1.0931230783462524, 2.196763277053833, 0.7222700119018555, -2.341787099838257, 0.6101925373077393, -0.5975272059440613, -1.2803351879119873, -0.12605607509613037, -0.18119928240776062, 0.29440838098526, -0.7670315504074097, 2.1163268089294434, 0.861497700214386, -1.8612877130508423, -0.3099319636821747, -0.616710364818573, -0.753439724445343, -1.0517498254776, 0.1671258956193924, -1.8939858675003052, -1.271701455116272, -0.8449041247367859, 0.9495003819465637, 0.47755205631256104, -0.9638279676437378, -0.3928806781768799, -0.06025165319442749, -0.8840449452400208, 0.08688179403543472, 0.22690808773040771, -0.05366317182779312, -0.9387626051902771, -0.2894933223724365, 0.8432209491729736, 0.55267333984375, 1.5168126821517944, -0.6662172079086304, 0.4477190673351288, -0.5181835889816284, -0.7159953713417053, 0.306118369102478, -1.236000657081604, 0.34909695386886597, 0.2308349609375, 0.2728783190250397, -0.8359647393226624, 0.9038835763931274, 0.7371304631233215, 1.102184772491455, -0.38295742869377136, -1.313990831375122, 0.7577881813049316, -0.37731510400772095, -1.3244836330413818, -0.028197966516017914, 0.4882911443710327, 1.038543939590454, 1.0620319843292236, 1.8728538751602173, -0.019836924970149994, 0.018993699923157692, -0.3026914596557617, -0.4046192765235901, 1.209662675857544, 0.48187097907066345, 0.009812582284212112, 2.0682146549224854, 0.5757308602333069, 1.4864935874938965, -0.03753819689154625, -1.1431654691696167, -1.2811367511749268, 0.3137730360031128, -0.1994742751121521, 0.657528817653656, -1.1040407419204712, 0.7097253799438477, -0.2657774090766907, -0.5170941948890686, 0.708361029624939, 0.5133299231529236, 0.6746483445167542, -1.8127126693725586, 0.09953097254037857, 1.32990562915802, -0.3850895166397095, 0.5099807381629944, -1.5219849348068237, 1.4963533878326416, -0.01585143432021141, 0.7934491634368896, -0.826866626739502, -0.09368197619915009, -0.19388264417648315, 1.9521881341934204, -0.9070724844932556, 0.7091788053512573, 1.2026053667068481, -1.5182685852050781, -0.48154348134994507, 0.2928818464279175, -0.14591728150844574, -0.64556884765625, -0.03354741260409355, 0.08541451394557953, 0.9810615181922913, -1.700101613998413, 0.19425348937511444, 1.0226092338562012, 1.3657634258270264, 0.20946791768074036, 0.7866761088371277, 0.004570063669234514, 0.6960132122039795, -0.35514867305755615, -0.28069496154785156, 0.9773684740066528, -1.5013176202774048, 0.4117608368396759, -1.1177493333816528, 0.6859195232391357, 0.4712006747722626, -0.00018432788783684373, 0.37464314699172974, 0.7424527406692505, 0.4096049964427948, 0.36269456148147583, -2.4434800148010254, 1.0373857021331787, 0.6971744298934937, 0.9690185189247131, 0.4984557628631592, -0.11391720175743103, -1.7879642248153687, -0.13777565956115723, -0.9118417501449585, 0.471141517162323, -0.06513402611017227, -1.2135992050170898, -0.20075084269046783, -1.6974738836288452, 0.3145447075366974, 0.9760423302650452, -0.31355345249176025, 0.043298687785863876, 1.8537179231643677, -1.1100376844406128, 0.7546656131744385, -0.09837613999843597, -1.844648838043213, -0.08573362976312637, -0.6854757070541382, -0.6092116236686707, -1.0894584655761719, -0.978765070438385, -1.069821834564209, -0.24790507555007935, -0.3907073736190796, 0.4270041882991791, -0.3511599004268646, 2.257533073425293, -0.7631675601005554, 0.8940417766571045, -0.9949761033058167, -0.3342111110687256, 0.8922913074493408, 0.642274796962738, -1.8242483139038086, -1.0058789253234863, -1.493504524230957, 0.6577317714691162, -0.19603858888149261, 1.4740318059921265, 0.2877146601676941, -1.5611604452133179, -0.5343924760818481, 0.8826101422309875, 0.40420371294021606, -1.103092908859253, -0.8204253315925598, -0.7292565107345581, -0.18885283172130585, 1.3858873844146729, -0.5325204133987427, 1.909185767173767, 0.877083420753479, -0.5791927576065063, -1.5935090780258179, -0.8939986228942871, 0.00764111103489995, -0.03574047237634659, 1.229757308959961, 0.4698152542114258, -1.4200148582458496, -0.10559868067502975, 0.5294882655143738, 0.6295047998428345, 0.9069951772689819, 0.08728528767824173, -1.1373504400253296, -1.6826077699661255, -2.2243430614471436, 1.093536138534546, 0.20623499155044556, -0.0455351397395134, -1.729835867881775, 0.749360203742981, -1.241708755493164, 0.5664921402931213, -0.788215160369873, -1.137054204940796, -0.45575788617134094, 0.713866651058197, -0.6234840154647827, -0.690906822681427, 0.4337298572063446, -1.622544288635254, 1.7466073036193848, 0.23492081463336945, 1.311346411705017, 1.507978081703186, -0.13154321908950806, 1.977325439453125, -1.6513246297836304, -0.06794057786464691, -0.45883816480636597, -0.6812277436256409, 0.7446445226669312, -0.4308600425720215, -1.2711585760116577, -2.1165599822998047, -0.06204789876937866, 1.4147735834121704, 1.3689773082733154, -0.4597671926021576, -1.4826467037200928, 0.9221518635749817, 0.9480436444282532, -0.827107846736908, -1.3226109743118286, 3.1871700286865234, -0.8441774249076843, 1.3386280536651611, 0.1598433554172516, 0.006146638188511133, -0.7329699397087097, 1.8514074087142944, -1.2516512870788574, -2.0006263256073, 0.39161431789398193, 0.3979282081127167, 0.05111132189631462, 1.2820953130722046, 0.7259210348129272, 0.4954909682273865, 0.16533058881759644, -0.6490404605865479, -0.9091701507568359, -0.20506270229816437, 1.4721368551254272, -2.405395030975342, 0.8431355953216553, -0.5937796235084534, 0.7710767388343811, -0.11628484725952148, -0.7416945695877075, 0.8165645599365234, 0.9797754883766174, 0.5596399307250977, -0.9331676959991455, -0.9527564644813538, -0.3751862049102783, -0.7151472568511963, 0.275076299905777, 1.6402256488800049, 2.407252550125122, -2.2380242347717285, 0.1183926984667778, -0.4205344319343567, 1.9523333311080933, -1.6832032203674316, -1.5743606090545654, -0.857880175113678, -2.070410966873169, -0.41771531105041504, 0.3920488953590393, 0.26072585582733154, 0.14962053298950195, 0.4286653399467468, -0.23649245500564575, -0.20707836747169495, -0.5479918718338013, 0.6323622465133667, -1.0571906566619873, -1.6379430294036865, -0.31877601146698, -0.1992463320493698, 0.5295682549476624, 0.8194319605827332, -1.4376270771026611, 0.1942988783121109, 0.01122627779841423, 0.2734537720680237, -0.47565263509750366, 0.05656407028436661, -0.6332518458366394, -0.23396988213062286, -1.2101951837539673, 0.6358650326728821, 0.1247398778796196, 0.14858907461166382, 0.2476215809583664, 1.464640498161316, -0.34243863821029663, -2.0467381477355957, 0.7115929126739502, 1.5493063926696777, 0.913282573223114, -1.3632926940917969, -0.0465785413980484, 0.4332541525363922, -0.7835932970046997, 0.038487765938043594, -0.018659837543964386, 0.6166119575500488, 0.7005224227905273, -0.8759887218475342, 1.1822359561920166, -1.2159225940704346, 1.5481696128845215, 1.6825809478759766, -0.39643222093582153, -1.0102132558822632, 1.9321609735488892, 1.8028358221054077, -2.1249947547912598, -0.01708856038749218, 3.5068178176879883, -1.1473865509033203, -2.5998096466064453, 0.41920819878578186, 0.6836233735084534, 1.160290002822876, 0.9515491127967834, 0.5362166166305542, 0.2332710176706314, -1.3575150966644287, 0.7445457577705383, 1.213412880897522, -1.5512953996658325, 0.4277959167957306, 0.6454934477806091, 0.6828900575637817, -1.6487951278686523, 0.26449593901634216, 0.6823254227638245, -0.3812033534049988, -2.1825201511383057, -1.350790023803711, 1.6462180614471436, 0.12728896737098694, -1.9375284910202026, -0.903226912021637, 0.3997836112976074, 0.11907105147838593, -0.556031346321106, -0.2218494415283203, 0.2795838713645935, 1.3636924028396606, -0.8892689347267151, -2.457613945007324, 0.5089724659919739, -0.5099383592605591, -0.14782048761844635, -1.8411014080047607, 0.23080827295780182, 0.22953660786151886, -2.0624263286590576, 0.6684074401855469, -0.1507023572921753, -0.10505684465169907, -1.6120636463165283, 1.9545050859451294, -0.8336153626441956, -0.4064730405807495, 0.9636207222938538, -1.69668447971344, 1.1211738586425781, 0.3957131803035736, -0.9792535305023193, 1.6710463762283325, -1.0173884630203247, 0.40930989384651184, 0.9649671912193298, 1.512345790863037, -0.4392649829387665, 1.5082825422286987, -0.5969586372375488, 0.5213318467140198, 0.14856208860874176, -0.18014578521251678, 1.6589528322219849, -0.5786100625991821, -0.9701533317565918, -0.8412695527076721, 1.023591160774231, 0.4832545220851898, -1.8843038082122803, -0.695353627204895, 1.1070690155029297, -1.9792060852050781, 0.15738733112812042, -0.8265658617019653, -0.4591943621635437, 0.8039413690567017, 0.5073983073234558, 2.20937180519104, -1.143495798110962, 1.1734317541122437, -0.18921592831611633, 0.35344022512435913, -0.486764520406723, 1.3864792585372925, -0.056921109557151794, 0.31245020031929016, 1.1640197038650513, 0.8922950625419617, 0.5031816959381104, -0.7585760951042175, 0.6036800742149353, -0.07936979830265045, 0.03488481044769287, 0.5205559730529785, 1.1654069423675537, 1.376193881034851, 1.0355063676834106, 1.5143135786056519, -0.6697536706924438, -0.8844001293182373, 1.4026001691818237, -0.6841558218002319, -1.8292959928512573, -0.08340919762849808, -2.3108296394348145, 0.1077529713511467, 0.7676685452461243, 0.6411144137382507, 0.2073422521352768, 0.27390268445014954, -1.2031011581420898, 0.16375519335269928, 1.90580153465271, 1.6643168926239014, 0.43330398201942444, -1.3095126152038574, 0.3430142104625702, 1.8566594123840332, -0.6971868872642517, 1.1489838361740112, 0.39432722330093384, 0.33871644735336304, -0.6610231995582581, 0.6443732976913452, -0.6347655057907104, 1.5687869787216187, -0.18798662722110748, 0.24115517735481262, 0.3719532787799835, -0.4528299570083618, 0.2894217371940613, 1.2412892580032349, 0.4381411075592041, 0.4353649616241455, -1.0407793521881104, -2.066324234008789, -0.39510011672973633, -1.528532862663269, 1.175460934638977, 0.36125174164772034, -0.4747653603553772, 1.1111232042312622, 0.8165767788887024, 0.2932138741016388, -1.2962970733642578, 1.0553523302078247, 2.03603196144104, -0.40186062455177307, -1.5042610168457031, -0.9303602576255798, -1.709769606590271, -1.660531759262085, 0.14804141223430634, 0.598656177520752, -1.2042820453643799, 0.2875466048717499, -0.8997836112976074, 0.9009959697723389, 0.5090713500976562, 0.49071887135505676, -1.1748861074447632, -1.7536982297897339, -0.3141494691371918, 0.20420455932617188, 1.173261046409607, 1.4990788698196411, 1.4257980585098267, -1.0778777599334717, 1.0183382034301758, 1.7044645547866821, -0.706998884677887, -1.34414803981781, 0.7822520136833191, -2.615062952041626, -0.24850346148014069, 0.17700067162513733, 0.08239902555942535, 0.9653728008270264, -1.0004611015319824, 0.1950785219669342, 0.3307429254055023, 0.8224393725395203, 0.7755953669548035, 1.2582775354385376, -0.3355717360973358, -0.6611066460609436, -0.22964204847812653, -0.6908159852027893, -0.03837694972753525, 1.2873895168304443, -0.8102889657020569, -2.2127814292907715, -1.5226280689239502, 1.1616981029510498, -1.2501672506332397, 1.5051056146621704, 0.0859844908118248, -0.24669140577316284, 0.13873575627803802, -0.007442949339747429, -2.1569786071777344, 2.383944511413574, 0.9772750735282898, 2.0539088249206543, -0.9637252688407898, -0.5329899787902832, -0.5942285656929016, -0.8555263876914978, -0.26652029156684875, 0.34303703904151917, -0.4200778901576996, 0.2458125352859497, 0.22223488986492157, 0.2594910264015198, 1.086888074874878, -0.9120883345603943, 0.560991644859314, 1.4294869899749756, 0.37435898184776306, -0.30247798562049866, 2.5884132385253906, -0.3214839994907379, 0.10765694826841354, 2.0814309120178223, -0.14702317118644714, -0.010724147781729698, -0.2526431679725647, 0.9691782593727112, -0.6175628900527954, 1.7014058828353882, 0.36663439869880676, -1.6809968948364258, -0.35780730843544006, 0.34523630142211914, 1.354244589805603, 1.7475416660308838, -1.463884711265564, 1.461320161819458, 0.7766755819320679, -0.6130923628807068, 0.4804331362247467, 1.2135334014892578, -0.2326914668083191, 0.5276280641555786, 1.4617009162902832, -0.30961018800735474, 0.017761411145329475, -0.1746177226305008, -0.24109143018722534, -0.044719427824020386, 1.1661185026168823, -0.08322732895612717, 0.944871723651886, 1.686334252357483, -1.0376137495040894, -0.16974785923957825, 1.0430941581726074, -0.6208289861679077, 0.3443094789981842, 0.1589941680431366, -0.5359004735946655, -1.0416611433029175, 1.3033043146133423, 0.17398439347743988, -1.4444503784179688, 0.9384476542472839, 0.2177419364452362, 0.6464291214942932, 0.8411672711372375, -0.6600222587585449, 0.6653975248336792, -0.35329481959342957, -0.2504217326641083, 0.17442253232002258, -0.1166127547621727, 0.7456443309783936, -0.02525101602077484, 1.0675324201583862, -0.03355661779642105, -1.0149000883102417, 1.706510305404663, -1.956565260887146, 2.4624173641204834, -0.4011167883872986, -0.09208883345127106, -0.5526870489120483, 1.8804097175598145, -0.11492457240819931, 0.7003818154335022, -0.7062561511993408, 0.026774587109684944, -1.2554206848144531, 0.2535961866378784, -2.1729655265808105, -0.1269541233778, -0.6900913715362549, 1.256704568862915, -0.9275841116905212, 0.17070560157299042, -0.43589457869529724, -0.2689136266708374, 1.2903529405593872, -0.2951193153858185, -0.7664119601249695, -1.8273814916610718, -0.4017871022224426, 0.9990519285202026, 0.3772805631160736, -0.2872515916824341, -0.24310383200645447, 0.017153484746813774, -0.9524489641189575, 0.6899783611297607, -0.13011643290519714, -0.45394113659858704, -0.8838163614273071, 0.7135462164878845, -1.3166955709457397, -0.21518337726593018, 0.9482210874557495, -0.25273358821868896, 0.7948742508888245, -0.11442471295595169, -0.2981223165988922, 0.7252826690673828, -0.37754955887794495, 0.08572093397378922, -1.3622069358825684, 2.0831196308135986, -0.22004006803035736, 0.02817697636783123, 0.4830658435821533, 1.0438047647476196, 1.1310267448425293, 0.07873362302780151, -1.0810338258743286, -0.24458818137645721, 0.4712827801704407, -1.0010572671890259, 0.3680321276187897, 0.4767391085624695, -0.400896281003952, -0.29104235768318176, 0.569758951663971, 0.34206414222717285, -0.3760620355606079, -0.410569965839386, -0.3846049904823303, -0.26477327942848206, -0.6698207259178162, 0.026251934468746185, -0.6612405180931091, 1.3242928981781006, -0.5034202337265015, -0.4198918342590332, 0.6988494992256165, 0.3504921495914459, -1.9711189270019531, 2.198723316192627, 0.22239841520786285, 0.18682777881622314, 0.5519464612007141, 0.179520845413208, -0.02126394584774971, 1.4112091064453125, 0.48141756653785706, 0.4901192784309387, 0.2227114588022232, -1.4820470809936523, 0.11044157296419144, 0.05922860652208328, 0.26132577657699585, 0.5656498074531555, -0.08652930706739426, -0.36539992690086365, 1.2032886743545532, -0.5691667795181274, 0.36431628465652466, 0.32272616028785706, 0.03541051968932152, -0.007894020527601242, -1.1596511602401733, 0.30553150177001953, 0.9891347289085388, 0.8217455148696899, -0.8621270060539246, -0.5583586096763611, -1.206919550895691, 1.966346025466919, -0.4227179288864136, 2.119591236114502, -0.4973108470439911, 1.2978405952453613, 1.0278676748275757, 0.5332186818122864, -0.034317582845687866, -0.46507924795150757, -0.6210183501243591, -0.5375226736068726, 0.6899339556694031, -0.42678213119506836, -0.31836438179016113, -0.1633731722831726, -0.39250946044921875, -1.8013551235198975, -1.1322009563446045, -1.0947835445404053, -0.1524427980184555, -0.9684517979621887, -0.2849547266960144, -1.304679036140442, 1.7150884866714478, -0.26301607489585876, 1.1158931255340576, -0.33250248432159424, -0.4281240999698639, -0.3556974530220032, 1.3633856773376465, -0.8746204376220703, 1.6676429510116577, -0.8984053134918213, -0.5110300183296204, 2.0783982276916504, 0.7769635915756226, -0.22834786772727966, 0.2323656976222992, -0.31646084785461426, 0.1814640462398529, -1.6842814683914185, 0.604284405708313, 0.08447834849357605, 1.040226936340332, 0.3511004149913788, -0.5743539333343506, -1.1839910745620728, 0.538545548915863, 0.6758213043212891, -0.1973862498998642, 0.8369243144989014, 0.4539543390274048, -0.3244960308074951, 1.158737301826477, -1.4331910610198975, -0.06146099045872688, -0.457599401473999, 0.5772688388824463, 0.5777263045310974, 0.9871832132339478, -1.7811341285705566, -0.5284277200698853, 0.5895462036132812, 0.679311990737915, 0.10600737482309341, -1.5722217559814453, 1.8634846210479736, -0.3165672719478607, 0.8094006776809692, 0.24112051725387573, 0.2084403783082962, 0.35765302181243896, 0.6302660703659058, 0.9370028972625732, 0.09154456853866577, -0.46949517726898193, 0.12050355970859528, -0.30926942825317383, 0.2319379299879074, 0.02690490521490574, 0.5804719924926758, -0.7561836838722229, 1.3800523281097412, 0.17665958404541016, -0.5441064238548279, -0.18879716098308563, -2.1705846786499023, 1.002851963043213, -0.9991223812103271, 0.085256427526474, -1.8671996593475342, -1.5251775979995728, -1.125351071357727, 0.014737267047166824, -1.0507874488830566, -0.1343597024679184, -0.1789848655462265, 0.32899075746536255, 0.6012324094772339, -1.0961244106292725, -0.44748935103416443, 1.2679227590560913, 0.5198545455932617, 0.6172961592674255, -1.0792721509933472, 1.1774959564208984, 0.07031402736902237, -0.9613008499145508, -0.8892448544502258, -1.1151952743530273, 0.2763865292072296, -0.03431493043899536, -1.2083479166030884, 1.5071123838424683, 0.6303502321243286, 1.8940355777740479, 0.21861791610717773, 0.1474124640226364, 2.131674289703369, -1.699092984199524, -2.0716867446899414, -1.3348567485809326, 0.5676252245903015, -0.4135255217552185, 0.596209704875946, -0.22626227140426636, -0.9921586513519287, -0.8819090127944946, -1.3955256938934326, 0.7211944460868835, -2.337771415710449, 0.22204655408859253, -1.691228985786438, -1.5349187850952148, 0.32620757818222046, 0.361322820186615, -0.5619344115257263, 0.2991935908794403, -0.6826700568199158, 1.9546313285827637, 0.21908298134803772, 0.8342681527137756, -0.8855982422828674, 0.12390503287315369, -1.069231629371643, 0.42877763509750366, -0.896518886089325, 0.35490691661834717, -0.8926626443862915, -0.6104971766471863, 0.9534587264060974, 1.0789490938186646, -0.3461884558200836, -0.17921167612075806, 1.8350768089294434, -0.0061022816225886345, 1.4795217514038086, -0.0025726701132953167, -1.8844070434570312, 0.5666174292564392, 0.42579394578933716, -1.1486393213272095, -1.4936288595199585, 0.44898080825805664, -0.39694806933403015, 0.5768941640853882, -0.4753134548664093, 0.4617886543273926, 0.2495356798171997, -0.5683441758155823, 0.13170596957206726, -0.5983884334564209, 0.5053478479385376, -0.11989866942167282, -0.0037767733447253704, 0.8406991958618164, 2.258293867111206, -0.7498763203620911, -1.450523018836975, -0.3663972318172455, 0.1709779053926468, 0.6089343428611755, 0.11014772206544876, 0.9135669469833374, -0.8380432724952698, -0.3981589674949646, -0.5973153710365295, 0.4206710159778595, -1.130263090133667, 1.3242313861846924, -0.030818726867437363, 1.731210708618164, 2.234937906265259, -0.9224115014076233, -1.7360552549362183, 0.15533855557441711, -1.231451392173767, 0.1931314468383789, -0.6374416947364807, -0.7802222371101379, -0.07550165802240372, 2.424652576446533, 0.9235110878944397, 0.9113782644271851, 0.5519176125526428, 0.4410999119281769, -1.3943603038787842, -0.7741428017616272, -0.04012931138277054, -1.7013615369796753, 0.9198710322380066, 0.95069420337677, 0.9350548386573792, 0.7907360196113586, -0.32019272446632385, 2.3108601570129395, -0.5820057392120361, 0.6518000364303589, -0.7141366600990295, 1.7096353769302368, -0.9767123460769653, 0.44446811079978943, -0.9886406660079956, 0.10846292227506638, -0.15180400013923645, 0.6605209112167358, 0.0378015860915184, 0.10200778394937515, 1.4716728925704956, -1.2304105758666992, -1.0254504680633545, 0.893778383731842, -0.44952407479286194, -1.3602310419082642, 1.4763718843460083, -0.6285825371742249, 0.7675087451934814, -1.4192827939987183, -0.15196265280246735, 0.1802828013896942, -0.13574258983135223, -0.4819185137748718, 1.4135857820510864, -0.5729348659515381, 1.9263418912887573, 1.6940428018569946, -1.608040452003479, 0.00700128311291337, 2.1400697231292725, -0.5687083005905151, -0.7428945899009705, -0.49898722767829895, 0.8916338086128235, 0.2198207676410675, 0.14475183188915253, -0.4215739369392395, 0.9848715662956238, 1.2420350313186646, -0.3023982346057892, -0.4377453327178955, -0.5324647426605225, 1.212762475013733, -0.40927377343177795, -0.33928659558296204, 1.1619423627853394, -1.6292303800582886, -0.4238896667957306, 0.06084361672401428, 0.8690804243087769, 0.2730240523815155, -0.3046955168247223, 0.03501802682876587, -0.006546212360262871, -0.34881365299224854, 0.23599889874458313, 0.9264770746231079, -1.0519275665283203, 0.6397188901901245, 0.023008134216070175, -0.36831802129745483, 0.04651651158928871, 0.5504869222640991, -1.6870231628417969, -0.0027751021552830935, 0.8955382108688354, 1.2688905000686646, -0.7127322554588318, -1.1351230144500732, 0.49273544549942017, 1.5189180374145508, 0.5526801347732544, -0.6154564023017883, -0.8466296792030334, 1.3931896686553955, 0.02823103405535221, -0.11545448005199432, 0.17311890423297882, -0.261076420545578, -0.43889158964157104, 0.5168651342391968, 0.5865556597709656, -2.2523293495178223, -1.4982300996780396, 1.505926489830017, 0.42893972992897034, 0.7411705255508423, 0.9268894791603088, 1.9006121158599854, -1.1615774631500244, 0.6827029585838318, -1.0242855548858643, -0.08312592655420303, 1.2424989938735962, 1.3576291799545288, -0.09410788863897324, -0.5199986100196838, 0.5741641521453857, 0.2755063474178314, -1.743236780166626, 0.06892921030521393, -1.9748539924621582, 1.0970362424850464, -0.05747049301862717, 1.2779264450073242, -0.4141576588153839, 1.8123947381973267, 1.4442311525344849, 0.29974809288978577, 0.9702351689338684, 0.2734980285167694, -0.10873331129550934, 0.9449630379676819, -0.7499564290046692, -2.10713791847229, 0.8984501957893372, -1.207161545753479, -2.768583059310913, -0.11368652433156967, 0.41557571291923523, -0.9409164190292358, -0.9997547268867493, 0.5809789896011353, -0.6970100998878479, 0.6356635093688965, -3.4887893199920654, -0.7186310887336731, 1.534449577331543, -0.1659574806690216, 0.24782083928585052, -1.4729020595550537, 0.4735791087150574, 0.5739458799362183, -0.5820114016532898, -0.9748160243034363, -0.7869260311126709, -0.763195812702179, 0.7936235070228577, -1.238478660583496, -0.06391407549381256, 0.36514678597450256, -0.8032983541488647, 0.46578100323677063, -0.3840305507183075, 0.96387779712677, -2.1581218242645264, 0.3608556091785431, 0.3575579524040222, -0.5874732732772827, -0.8675658106803894, -0.25677183270454407, -1.346435308456421, 0.6879013776779175, -1.1121058464050293, 0.5529821515083313, 1.4231394529342651, -0.5203418135643005, 0.48474037647247314, 0.8222983479499817, 1.060015320777893, -0.3937430679798126, 1.4352118968963623, 0.9748498201370239, 0.8920809030532837, -1.6328855752944946, -3.189751386642456, -0.9916011095046997, -0.670097291469574, 0.22784806787967682, -0.13490824401378632, 0.6784342527389526, -0.14236177504062653, -0.6762314438819885, 1.4556429386138916, 0.35151562094688416, -1.0308709144592285, -0.009186401963233948, 1.0052788257598877, -0.5177581906318665, -0.6916090846061707, 0.10534941405057907, 1.3259806632995605, -0.5442760586738586, -0.1692744493484497, -1.5477842092514038, 1.3696256875991821, 1.0847363471984863, -0.8095741271972656, -0.3082776665687561, 0.7071017026901245, 1.5506644248962402, 0.22977742552757263, 0.834116518497467, 0.8181062340736389, 1.442426085472107, 0.36990886926651, 0.8546271920204163, 0.3058291971683502, 1.0120537281036377, -0.6782456040382385, 0.1914786398410797, 0.09421363472938538, -1.3699086904525757, -0.1293991059064865, -0.22488389909267426, 1.3689249753952026, 1.1099176406860352, -0.3612389862537384, 0.2795391082763672, 1.3094193935394287, 0.2629943788051605, 0.32834917306900024, 0.3932116627693176, 0.003372133243829012, -0.0743367075920105, 0.9036363363265991, -0.45018142461776733, 1.957055687904358, 1.40962553024292, -0.057065028697252274, -1.0298875570297241, -0.7802175879478455, -0.061112985014915466, 1.0266196727752686, -0.6304125785827637, 0.05022606626152992, 0.2128133326768875, 1.1724666357040405, 1.6225131750106812, 0.13643783330917358, -1.3917791843414307, -0.40295711159706116, -1.4642308950424194, -0.7272081971168518, -0.282925009727478, -0.23581072688102722, -1.7899229526519775, 0.7787270545959473, 1.1527140140533447, -0.13695040345191956, 0.2848520278930664, -1.1027015447616577, 0.29670166969299316, 1.2573349475860596, -0.4795123040676117, -0.652607262134552, 1.9995989799499512, 1.2691510915756226, 0.7866283655166626, -1.7341835498809814, 0.008405015803873539, 2.874483585357666, -0.6512795686721802, -0.7379171848297119, 1.5409839153289795, -1.18892502784729, -0.1399812251329422, -0.3814542293548584, -2.362370729446411, 0.8254963755607605, -0.47028493881225586, 0.16995497047901154, 0.18036922812461853, 1.38443124294281, -0.16651730239391327, 0.6856231689453125, 0.47832462191581726, -1.888382077217102, -0.5047023892402649, 1.2863707542419434, -1.1663661003112793, 0.2808719873428345, 0.6156628727912903, 2.637899875640869, 0.04010022059082985, -0.16560296714305878, -1.225571632385254, -0.47695431113243103, -0.5441476106643677, 0.39606133103370667, 1.71855890750885, 0.5945289134979248, 0.29087260365486145, -0.10518177598714828, -0.8669437766075134, -1.2050243616104126, 0.2660769820213318, -0.1265833079814911, -0.7939497232437134, -0.8463393449783325, -1.1765121221542358, -0.9167847037315369, 0.945934534072876, 1.137832760810852, 0.5775554180145264, 0.4684469699859619, 1.9049240350723267, 0.693284809589386, 0.05807330086827278, -1.2887649536132812, -0.3827499747276306, -1.4436473846435547, 1.3385061025619507, 1.9228885173797607, 0.6891559362411499, -1.1392821073532104, 0.2293189913034439, -0.6677794456481934, 0.2562205493450165, -0.7602713704109192, -0.2671761214733124, -0.011286304332315922, -0.22592657804489136, 1.1883755922317505, 0.442630410194397, 0.7998331189155579, -0.5208301544189453, -0.7220975160598755, 0.3343642055988312, -0.41691064834594727, -0.8961595892906189, -1.099361777305603, -0.3284006714820862, 0.8445146679878235, -2.2507481575012207, -0.2915627062320709, 2.1802468299865723, 1.3305453062057495, 0.27893149852752686, 0.7552902698516846, 0.774757444858551, -0.5111395716667175, -0.6435325741767883, 0.6634783148765564, 0.9040853381156921, -1.0034584999084473, 0.5221610069274902, 1.0473732948303223, 1.276720643043518, 1.0800594091415405, -0.176179438829422, -0.7150788903236389, 0.6684186458587646, -0.8270168304443359, 0.9907627105712891, 0.8270182609558105, 1.3470463752746582, 0.4669440984725952, 1.4208136796951294, 0.4263245463371277, 0.01606025919318199, 0.02168962173163891, -0.03970395773649216, -0.2040766179561615, -0.33026042580604553, -0.4784913957118988, 0.5022740364074707, 0.08472643047571182, -0.3469081521034241, 1.074371337890625, -0.9863796234130859, 0.8936771154403687, -0.4344995319843292, -1.3519260883331299, -0.2189711630344391, 1.386499285697937, 0.519691526889801, -1.384576678276062, 0.028063924983143806, -0.02910688892006874, -0.378298819065094, 1.1875712871551514, 0.9246858358383179, 1.1378711462020874, 0.25359219312667847, 0.4345822334289551, 0.8709908127784729, -1.2971007823944092, 0.2965145707130432, 0.31083205342292786, 1.1289911270141602, 0.9242612719535828, 0.7066392302513123, -0.21456074714660645, 0.5374841094017029, 0.2615349292755127, 0.9391428828239441, 0.6158015131950378, -0.14486540853977203, 0.19749745726585388, 0.7609076499938965, -1.0746718645095825, -0.6941606402397156, -0.22545218467712402, -0.32743358612060547, -1.3251084089279175, -0.6055378317832947, 0.5171816945075989, -0.47144895792007446, -1.3678441047668457, -0.1995999813079834, -1.5053980350494385, 1.6064106225967407, -1.8835922479629517, -0.015187067911028862, -0.7991323471069336, -0.033168725669384, -0.36652302742004395, -0.9396833777427673, -1.0673595666885376, 0.596991240978241, -1.6656131744384766, 0.32071056962013245, 0.6985718011856079, -1.1787973642349243, -0.24399042129516602, -0.19547566771507263, 0.4555926024913788, -0.09479714184999466, 2.3964974880218506, -1.004783034324646, 0.38281771540641785, 1.338868498802185, -1.035460352897644, -1.5766997337341309, -0.5301975607872009, -0.01419802475720644, 0.06276439875364304, -2.3091228008270264, -0.8654392957687378, -1.5349315404891968, 0.4203525185585022, 0.27747049927711487, -0.5448785424232483, -0.607282280921936, 0.39402368664741516, 1.2835959196090698, 0.8422536253929138, 0.011252318508923054, 0.18747664988040924, -0.29899558424949646, -1.6031471490859985, 0.5090848803520203, 0.9751867651939392, -1.0210814476013184, -1.4647107124328613, -0.963268518447876, 0.33055490255355835, 0.9650084972381592, 0.68458491563797, -0.7255264520645142, 1.013293743133545, 1.1182962656021118, 0.3860558867454529, -0.04290952906012535, -0.6402163505554199, 0.9495621919631958, -1.399924635887146, -0.498580664396286, 1.2519091367721558, -2.827601671218872, -0.808080792427063, 0.40207669138908386, 1.020284652709961, -0.7821760773658752, 1.2407363653182983, 2.3596417903900146, 1.2747608423233032, 0.014716382138431072, 1.6044872999191284, 0.4419017732143402, 0.7221719622612, -1.09325110912323, -0.7180262207984924, 0.3627975881099701, -0.48085668683052063, -1.4279687404632568, -0.19042226672172546, -1.1178457736968994, 0.35867205262184143, -1.0594767332077026, -0.41447120904922485, -1.5124716758728027, 0.2558567523956299, -0.08658328652381897, -0.9414643049240112, 1.6602826118469238, 0.7770258188247681, -0.2677757441997528, -1.3110370635986328, 0.15858711302280426, 0.8053624033927917, 0.44226598739624023, -0.394893616437912, -0.15712472796440125, -0.2743875980377197, -2.1753368377685547, -1.0734950304031372, 0.4368196129798889, -0.7696902751922607, 0.5572174787521362, -0.6512137055397034, -0.6428159475326538, 1.370807409286499, -0.15033306181430817, -0.47218504548072815, -0.20171864330768585, -0.6490039825439453, -0.9675120115280151, -0.6996312141418457, 0.3535187542438507, 1.2330710887908936, -0.3122228682041168, 0.5423010587692261, -1.3321396112442017, 0.45843377709388733, -0.26999473571777344, -0.6872632503509521, -0.28510773181915283, -1.0360163450241089, 0.8296034932136536, -0.4496211111545563, 0.9231624007225037, 0.8212608695030212, -0.22778235375881195, 0.4460800290107727, 1.1665089130401611, -0.2814043164253235, 0.6775323748588562, 1.44490385055542, 0.5133953094482422, -0.41392338275909424, 1.3597376346588135, -0.6311410069465637, -0.9997806549072266, 0.2878691554069519, -0.31866466999053955, -0.8935625553131104, 0.06560736149549484, 1.3664865493774414, -0.721472978591919, 1.0483403205871582, 1.2094591856002808, -1.362107276916504, -0.4880671501159668, -0.33928054571151733, -0.7495294809341431, 0.2833969295024872, -0.9877181053161621, 0.834639310836792, -0.7054950594902039, 0.16831561923027039, 0.8748130202293396, 1.2950749397277832, -0.3095613718032837, -0.9139158725738525, 0.6585886478424072, 0.015656430274248123, -0.15732818841934204, 0.05916226655244827, -0.48440346121788025, -0.7415060997009277, -2.0893471240997314, -0.7654660940170288, -1.4044175148010254, 0.3667147159576416, -0.9869408011436462, 0.6154850125312805, 0.024707909673452377, 0.8196081519126892, 1.1641333103179932, 0.825452983379364, -1.4023224115371704, -1.51529860496521, -0.22798281908035278, -0.2538476586341858, -0.6706483364105225, 0.5610315799713135, -2.4613547325134277, -0.24030053615570068, 0.9737035036087036, -1.3001874685287476, -0.611447811126709, 0.7844855785369873, -0.059786293655633926, -0.2534885108470917, 1.0373367071151733, 0.05741739645600319, 0.4836508333683014, 0.7357082962989807, -1.4258229732513428, -2.0156455039978027, 1.322675108909607, -0.9831180572509766, -0.9979855418205261, 0.7307577729225159, 1.6184947490692139, -1.6471561193466187, 0.5256636738777161, 1.1575223207473755, 0.5958156585693359, 2.1808888912200928, 0.37618204951286316, 0.3102600872516632, -0.500852108001709, -0.34829509258270264, 1.156832218170166, 0.3848051130771637, 0.7624855041503906, -1.4591130018234253, 0.6345537900924683, -0.8081472516059875, 1.6094486713409424, 0.23062556982040405, 0.5082849860191345, -0.6480167508125305, -0.07170671969652176, 1.4779208898544312, 0.2030276507139206, -1.2934883832931519, 1.4388028383255005, 0.43054699897766113, 0.04133257642388344, -0.07765942811965942, 0.8018932938575745, -0.9004174470901489, 0.5929531455039978, 0.05780261754989624, 0.47546765208244324, -0.17939451336860657, -1.8448303937911987, 0.16676899790763855, 0.799239456653595, -0.5568527579307556, 0.37037986516952515, 2.2619540691375732, -0.016233310103416443, 0.917677640914917, -0.950783908367157, -0.6385614275932312, 0.12628348171710968, -1.3332349061965942, -0.16321195662021637, -0.36437299847602844, 0.6703971028327942, -1.10673189163208, 1.8144021034240723, -0.5095514059066772, 1.299525260925293, -0.30671146512031555, -0.8618980646133423, 0.001280574593693018, 0.5457925200462341, -0.8136323690414429, -0.13577890396118164, 0.9747843742370605, -1.2045317888259888, 1.5644829273223877, -0.6416074633598328, 0.823078989982605, 0.16412481665611267, -0.18476274609565735, -1.4595234394073486, -0.6644600033760071, 0.4718746542930603, 0.8665112853050232, -0.1605479121208191, -0.4465964138507843, 0.955591082572937, -0.7076005339622498, 1.403977632522583, -0.42738428711891174, -1.113750696182251, -0.9047137498855591, -1.5470778942108154, 2.3621716499328613, -0.6268378496170044, 1.3614614009857178, -0.9352648854255676, -1.3890186548233032, -0.06432083249092102, -0.7181264758110046, 1.0123132467269897, -0.9409477710723877, -0.043737929314374924, -0.049793679267168045, -1.0862915515899658, 2.0772194862365723, 1.8956105709075928, 0.5372079610824585, -1.7297890186309814, -1.0918302536010742, -0.41007643938064575, -1.0613449811935425, 0.9960783123970032, 0.7003485560417175, -1.1775403022766113, -0.5307483673095703, 0.6277900338172913, 1.4595134258270264, -2.3861801624298096, 0.4516910910606384, 0.5099430680274963, 1.0983599424362183, 0.30618569254875183, -0.4686774015426636, -1.45353102684021, 0.610913097858429, -0.35910937190055847, -1.001362919807434, -0.7819496393203735, 0.05800432711839676, -0.5417556762695312, -0.7151681780815125, -0.22836433351039886, 0.7005923986434937, -0.6845484375953674, 0.45830070972442627, -1.0112117528915405, -0.10459518432617188, -0.3928263187408447, 0.06999872624874115, -1.3002023696899414, 1.2262547016143799, 1.0137577056884766, 0.42499011754989624, 0.06164020672440529, -0.4056704640388489, 0.39688393473625183, -0.34938931465148926, 1.295032024383545, -1.1293163299560547, -0.39857491850852966, 0.11850735545158386, 0.5663577318191528, -1.1900792121887207, 0.6651156544685364, -0.2919110059738159, 0.4323588013648987, -0.010191112756729126, 1.4517894983291626, 1.0028777122497559, -1.4507898092269897, 0.09028183668851852, 0.46389997005462646, -0.906762957572937, -0.8422330617904663, -1.6768864393234253, 0.08052711933851242, -1.0577980279922485, -0.34032878279685974, -0.35753050446510315, 0.3024752140045166, -1.2533241510391235, -0.07361646741628647, 1.234789490699768, -0.6142077445983887, 0.93949294090271, 0.7970558404922485, -1.9118722677230835, -0.6281299591064453, 1.2986301183700562, -0.8439859747886658, 0.7451930642127991, -1.3388652801513672, -0.2233668565750122, 0.41979336738586426, -0.6572350263595581, 0.7801887392997742, 0.006833474617451429, -1.7789796590805054, 0.49306443333625793, -1.2344715595245361, 0.508778989315033, 0.056529972702264786, 0.764386773109436, -1.213242530822754, -1.5511085987091064, -1.2783293724060059, 0.8302629590034485, 0.525647759437561, -0.11699773371219635, -0.7508542537689209, -0.4582688808441162, -0.7320969104766846, 0.3845188617706299, -0.8187090754508972, -1.3503233194351196, 1.3134148120880127, 2.3230695724487305, -0.6775227785110474, -0.3296384811401367, 0.8101446032524109, 1.219800353050232, -0.8876975178718567, 0.29976221919059753, 1.661198377609253, 0.8881093859672546, -0.008671003393828869, 1.1254205703735352, -2.9982523918151855, -0.08865607529878616, 0.4219927489757538, 0.9449122548103333, -1.0593845844268799, 0.49658796191215515, -0.23910750448703766, 2.3987371921539307, 0.08728142827749252, -0.7605698704719543, 1.1114068031311035, 0.5337131023406982, -0.47990521788597107, -0.44789379835128784, -0.04126286134123802, -0.79501873254776, -0.6764305830001831, 1.0364402532577515, 1.1092733144760132, -0.014542164281010628, 1.5025339126586914, 1.5837453603744507, 0.5121912360191345, -0.029689526185393333, -0.2643863260746002, -0.5909572839736938, -1.0188260078430176, 0.9967775344848633, 0.09445751458406448, 0.7433670163154602, 1.5089406967163086, -2.0742523670196533, -0.3240872919559479, -0.8222066164016724, 1.0821616649627686, 0.2863008975982666, -1.1096001863479614, -2.1192727088928223, 0.41196203231811523, 1.5400283336639404, -1.075548768043518, 0.35709142684936523, 0.4087340235710144, -0.17406530678272247, -0.37693506479263306, 2.3946962356567383, 0.8756592869758606, 0.6468356251716614, -1.266611933708191, -0.03226735070347786, 1.0130418539047241, -0.841838538646698, 1.1653225421905518, -0.5633936524391174, -0.7508594393730164, 1.3581111431121826, -0.8224038481712341, 0.544528603553772, -0.2775389850139618, 1.7129544019699097, 0.3213995397090912, 0.8418832421302795, 0.2442813217639923, 1.0143569707870483, 0.8849039673805237, 1.345108985900879, -0.16968843340873718, 0.19073566794395447, 1.0664286613464355, -0.7081152200698853, 1.331483006477356, -0.07649285346269608, -0.7811904549598694, 0.08884764462709427, 1.9960569143295288, -0.926132321357727, -1.1977441310882568, -0.1489168107509613, -0.4765603244304657, 0.9240097403526306, -0.20208647847175598, -0.9346737265586853, -1.0344146490097046, -0.6202081441879272, -0.23887285590171814, 1.008368730545044, 0.7647154331207275, -0.24559734761714935, -0.28775104880332947, -1.3843497037887573, -1.9867660999298096, -0.028688203543424606, -0.3003556728363037, -0.2872346043586731, 0.6116741895675659, 0.49675682187080383, -0.003256880445405841, -0.4136233925819397, 2.239802837371826, -1.5625613927841187, -0.8948569893836975, -0.6917504072189331, 0.8171004056930542, -1.1109575033187866, -0.4772776663303375, -0.05587810277938843, 0.05148705467581749, 0.17158985137939453, 0.032779015600681305, -1.6579725742340088, 0.23189571499824524, -0.3537951409816742, -0.6098757386207581, -0.6435567736625671, -1.350519061088562, -1.4565284252166748, 0.05838320031762123, 0.774527907371521, 1.2538636922836304, 0.4896584451198578, 0.6495645046234131, 0.19496679306030273, 0.7873620986938477, 0.5690988302230835, -0.42676082253456116, -0.8664408922195435, 0.42292946577072144, -0.146913543343544, -0.897804319858551, -1.8879990577697754, 0.42335230112075806, 0.307365357875824, -0.06417927891016006, 0.2233528196811676, 0.42905813455581665, 1.8163813352584839, -0.5174552202224731, -0.5100497007369995, 2.016418695449829, 2.04880952835083, -0.14317844808101654, 0.2478616088628769, -0.28010818362236023, 0.5143094062805176, -1.4967941045761108, -0.7726077437400818, 0.24807031452655792, -0.9677911996841431, 0.6891918778419495, 0.740558922290802, 0.3606293797492981, 0.39124754071235657, 0.8354817032814026, -0.43495091795921326, -1.1563671827316284, 0.03631662577390671, 0.7463193535804749, 0.33670204877853394, 1.8915863037109375, -1.4305977821350098, 0.2089681178331375, 0.1718347668647766, -1.1214675903320312, 0.5767188668251038, -0.8155831098556519, -1.0739760398864746, 0.9771148562431335, 0.41621506214141846, 1.8898531198501587, -0.0252344012260437, 0.31782266497612, -0.2624865770339966, -0.739354133605957, 0.9670109748840332, 1.526444673538208, -1.3013945817947388, 0.9416213035583496, 0.8256089687347412, 1.0559749603271484, -1.2934261560440063, -0.041930511593818665, 0.29017695784568787, 0.9935809969902039, -1.0349419116973877, -0.2350044548511505, -1.6802111864089966, 1.0212467908859253, -0.1642017811536789, 1.5001063346862793, 0.11478918045759201, -1.4623669385910034, -0.9802263379096985, 0.2200174480676651, 0.6396839022636414, 0.8379269242286682, 0.626168966293335, 1.2641960382461548, -0.4081881642341614, 0.45267266035079956, 0.6966732144355774, 0.7332136631011963, 0.03642640635371208, -1.0394576787948608, -0.25847378373146057, 1.6374424695968628, 0.7444931864738464, -0.1597316861152649, -0.9202708005905151, -0.46917304396629333, 1.0980836153030396, 1.5339527130126953, -0.033147502690553665, -1.247965693473816, 0.33575814962387085, 1.0214946269989014, -0.6629865765571594, 0.059247009456157684, 0.5927251577377319, 0.6806683540344238, 0.7126961350440979, -0.36717021465301514, 0.1609020233154297, -0.42343610525131226, 0.3382069170475006, -1.2530664205551147, -0.286152720451355, 0.8859007954597473, -1.1743324995040894, 1.1999824047088623, 1.8470900058746338, 1.319181203842163, 0.3243960440158844, -0.07990939915180206, -0.2184981256723404, 1.622597098350525, 1.6306519508361816, -0.4440653920173645, -1.143147587776184, 0.22523383796215057, -0.3358898460865021, 0.9459533095359802, -1.1492093801498413, -0.10127867758274078, -1.5143964290618896, 1.047920823097229, -0.6542244553565979, 0.5004058480262756, 0.5927152037620544, 1.1917903423309326, -0.43832290172576904, 1.420189619064331, -0.20138902962207794, 0.8893846273422241, 1.1976779699325562, -1.867977261543274, 1.138773798942566, 1.1194403171539307, -0.07019179314374924, 1.050550937652588, -0.30836358666419983, 0.3003827929496765, -1.3855329751968384, -0.703120768070221, 0.954379141330719, 0.9546833634376526, 1.3523575067520142, 0.1909567415714264, -0.8502176403999329, 0.340242862701416, 1.6048188209533691, -2.0582542419433594, 0.15603524446487427, -0.783675491809845, 1.1381999254226685, 0.31963828206062317, -2.0192480087280273, -0.8999372124671936, -0.18042343854904175, 0.8672336339950562, -0.3711601197719574, 1.6657191514968872, 0.4209486246109009, 0.4330032467842102, -1.5655889511108398, 1.5512118339538574, 1.1637701988220215, 0.15703409910202026, -0.780322253704071, -0.45951223373413086, 0.5201272964477539, -1.301923155784607, -1.0734997987747192, 0.1603385955095291, -0.9453902840614319, -0.6892841458320618, -0.15121164917945862, 0.2974216938018799, -1.8550474643707275, 2.3726251125335693, -0.9008432626724243, -1.3599199056625366, 1.5017999410629272, 0.12841680645942688, 0.2328755110502243, 0.8093695640563965, -0.7878391742706299, -0.7513460516929626, -1.16752028465271, -0.48249363899230957, -0.07040254771709442, -0.3332420587539673, 1.8892490863800049, 0.9151113033294678, -0.9551844000816345, -0.8055675625801086, -0.7683905959129333, -0.5554354190826416, -1.6285449266433716, -1.6041696071624756, -1.5467708110809326, -1.228116512298584, -0.09970371425151825, -0.5680667161941528, 1.5256149768829346, -1.1242694854736328, -0.6993452906608582, -1.4744230508804321, -0.4752868413925171, -1.5988510847091675, -0.31505468487739563, -1.34317946434021, 0.5785689353942871, -0.5668458342552185, -0.33206191658973694, -0.08374917507171631, 0.04731146991252899, 1.1918644905090332, -0.2570652961730957, 0.37428775429725647, -0.3901323080062866, -1.2461037635803223, -1.1319854259490967, -0.15329989790916443, -0.2027069628238678, -0.31601837277412415, 0.6525530815124512, 0.15274202823638916, -1.3477957248687744, -0.46785327792167664, 2.005286693572998, 0.7041265368461609, 0.8861608505249023, 0.0021393192000687122, -0.39020857214927673, -1.1146750450134277, -1.0187095403671265, -0.9387198090553284, 1.3760179281234741, 0.579919159412384, 0.7936654090881348, 0.3289332091808319, -0.312700480222702, -1.1286327838897705, 0.037159547209739685, 0.2101176232099533, -0.6702864170074463, -0.13314692676067352, 0.4710451364517212, -1.094750165939331, -1.5953357219696045, 0.8302615284919739, 0.6920823454856873, 0.37524086236953735, -0.5510618686676025, -0.37457144260406494, -0.7242905497550964, 0.5877635478973389, 0.7594412565231323, 0.6535534262657166, -0.9783309102058411, 0.2767086923122406, 0.5453565716743469, -0.9560954570770264, -1.4335461854934692, 1.1388521194458008, -0.11396989226341248, 0.8289579749107361, 0.7766456604003906, 1.4257380962371826, -0.35929742455482483, -0.3307769000530243, 0.06205061823129654, 0.16902175545692444, 0.09253104776144028, -0.5438095331192017, -3.3333511352539062, 0.744473397731781, -0.5864992141723633, -0.28452974557876587, -1.0205011367797852, -0.9745277166366577, 1.7483618259429932, -0.6107364296913147, 2.104313850402832, -0.30029359459877014, 0.7781165838241577, -0.05415099114179611, -1.4358704090118408, -0.5802095532417297, -0.3688047230243683, -0.09388530254364014, 1.7852510213851929, -1.2413039207458496, 0.0625578984618187, -0.732184886932373, 0.23648716509342194, 0.42446187138557434, -1.2558574676513672, -0.9558583498001099, -0.5304462909698486, -0.22570930421352386, -0.07758288085460663, -0.8940806984901428, -0.40865617990493774, -0.4554264545440674, -1.4168016910552979, 1.0522878170013428, -0.9375545382499695, -0.09297940880060196, -0.6932803392410278, -1.3547956943511963, 0.04701332747936249, 0.5275938510894775, -0.8505345582962036, 1.1659125089645386, -0.7146860957145691, -0.07340645045042038, 0.4110153317451477, -0.3181073069572449, 0.6078633666038513, 0.7001969814300537, -0.9430331587791443, -1.4123587608337402, -0.16520453989505768, 1.2362719774246216, 0.7725924253463745, -0.5580690503120422, -0.19116562604904175, -1.9897352457046509, -0.9931068420410156, -1.1914823055267334, -0.37481677532196045, 0.9852680563926697, 0.9027614593505859, 0.45066410303115845, 0.9153227210044861, 0.5538795590400696, -0.4155403673648834, -0.5291780829429626, 0.9897343516349792, 0.3824983835220337, -0.026411209255456924, -0.40252119302749634, 0.6816968321800232, 0.94296795129776, -0.957306981086731, -0.6883572340011597, -0.4452393651008606, 0.4021470248699188, -0.7927823662757874, -0.5625625252723694, -0.8509255051612854, 0.03656025975942612, -0.5230029225349426, -0.9298416972160339, 1.029606819152832, -0.9951247572898865, -0.8691024780273438, 0.4553743898868561, -1.350731611251831, 2.011956214904785, 0.1529611051082611, 1.0729509592056274, -1.7200345993041992, 0.04357348009943962, 1.7261757850646973, -0.3119823634624481, -0.8497998714447021, 0.6316357851028442, -0.8911487460136414, 2.150362730026245, -1.004711627960205, 0.2754352390766144, -0.504226565361023, -0.2979569435119629, -0.14187152683734894, 0.5192017555236816, 0.9476085901260376, 0.7141930460929871, 0.7753745317459106, -1.543341040611267, -1.3937044143676758, -0.028753962367773056, 0.4792349636554718, 0.20101061463356018, -1.0955984592437744, 0.5779724717140198, 1.070194959640503, -0.8236314654350281, -1.210784912109375, -1.1376606225967407, 0.7935476899147034, 1.8462696075439453, 0.8972144722938538, -1.6231167316436768, 0.45568519830703735, 0.3123294711112976, 1.6691420078277588, 0.4636300504207611, 1.757775068283081, -0.6052243113517761, 0.6699785590171814, 1.3224165439605713, -1.5301649570465088, 0.18462249636650085, -0.651555061340332, -0.6172831654548645, -1.6849515438079834, 0.6616603136062622, -0.48571768403053284, 0.3919239044189453, -1.8553506135940552, -0.8427395224571228, -0.3703930974006653, 0.08238448202610016, -1.9780220985412598, -0.928613007068634, 0.16147024929523468, -0.4825843572616577, -0.21812137961387634, 0.0006039266008883715, 0.4239729046821594, -1.4423412084579468, -0.2673647403717041, -0.790765106678009, -0.40268272161483765, -1.222172498703003, -0.7947789430618286, -0.9020280241966248, 1.1461526155471802, -0.25108975172042847, 2.442315101623535, -0.741265058517456, 0.5350754261016846, -0.817135214805603, -0.016558853909373283, -0.36166074872016907, -1.5806989669799805, 0.3622547686100006, 0.513215184211731, -0.8358927369117737, 1.0060656070709229, -0.3180178999900818, -0.32072263956069946, -0.6989728808403015, 0.603070080280304, 1.109557032585144, -2.128444194793701, -1.441928505897522, 0.3984851837158203, 0.3925548791885376, 1.2057169675827026, 0.8313741683959961, -0.33222195506095886, 2.064272880554199, 0.49033504724502563, -0.31725460290908813, -0.5673359036445618, -0.3678327202796936, 0.4642917513847351, -2.9977447986602783, -0.42491525411605835, 0.16958443820476532, -1.4030052423477173, 0.6699606776237488, -0.04932333901524544, 0.3369969427585602, 0.1230739876627922, -0.6260685920715332, 1.7371159791946411, 1.6732803583145142, 0.1182665005326271, -0.023875724524259567, -1.0311133861541748, -0.4948747754096985, -1.2736974954605103, -0.4545738995075226, -0.036792680621147156, -0.8298172950744629, -0.9554871916770935, -0.31740567088127136, 0.3174808919429779, -0.6676750779151917, -0.111534982919693, 0.5115135908126831, 1.3564916849136353, 0.7445272207260132, -1.381414532661438, 0.8596282601356506, -0.08313613384962082, -1.9016774892807007, 0.4026283919811249, -0.8346105813980103, 1.2167888879776, 1.2447717189788818, -1.0069921016693115, -1.150935411453247, 0.5396435856819153, 0.8361868262290955, -1.2761497497558594, -0.647352933883667, -0.12865184247493744, 0.7918369770050049, -0.19740943610668182, 0.022528689354658127, 1.517764925956726, 0.6609212160110474, 0.6483634114265442, 1.7515349388122559, 0.9968723654747009, 0.0054831658490002155, 1.656939148902893, 0.3683113157749176, 0.20243629813194275, -1.0438661575317383, -0.2199850082397461, -0.183607280254364, 0.2188601791858673, 0.586169958114624, -0.015448547899723053, 0.05471329391002655, 2.046311378479004, 1.3745619058609009, -1.110236644744873, -0.46985217928886414, 0.7139341831207275, -0.5998700261116028, -0.6676035523414612, 0.49106040596961975, 0.804496169090271, -0.16693341732025146, -0.093048095703125, -0.7744154930114746, 1.570698618888855, 0.1728227585554123, -0.40722838044166565, -2.8048481941223145, 0.9259015321731567, 1.018593668937683, -0.5754243731498718, 0.19590361416339874, -0.4398615062236786, -0.749126672744751, 0.33023110032081604, -1.718556523323059, -0.45592978596687317, -0.5748259425163269, -1.9465243816375732, 1.10902738571167, -1.2824242115020752, 0.59898841381073, -1.1152148246765137, -0.357015460729599, -0.16202208399772644, 1.1325629949569702, -0.5364400744438171, 0.9574103355407715, -0.8532115817070007, -0.6688309907913208, -0.8963570594787598, 0.4055730700492859, -1.096976637840271, 1.822957992553711, -0.36853504180908203, 0.7187586426734924, 2.3925182819366455, 0.6598787903785706, 0.6239131689071655, -0.48624131083488464, -3.436939239501953, 0.7271782159805298, 0.8436778783798218, -1.704026699066162, -1.6573346853256226, 1.2145081758499146, -0.7700204253196716, 0.12705522775650024, 0.5734010338783264, -0.7717510461807251, -0.534905195236206, -0.7346022129058838, 0.47985559701919556, -1.1486612558364868, -0.022639626637101173, -0.14877352118492126, -1.0725778341293335, 0.2849777638912201, 1.1099876165390015, 0.020777549594640732, 1.95198655128479, -0.23846817016601562, 1.834882378578186, 0.16677464544773102, -1.1220240592956543, -1.3595209121704102, -1.324079990386963, -0.848664402961731, 0.9931203126907349, -0.9996948838233948, -0.791292130947113, 0.9424811005592346, -0.7320089936256409, 0.9383572936058044, -1.099062442779541, -0.23933064937591553, -0.6673359870910645, -0.849682629108429, 0.3709106743335724, -0.8107193112373352, -0.09967563301324844, -0.12530192732810974, -1.4589494466781616, -1.7217178344726562, -0.4607000946998596, -0.6346821784973145, 0.1764630526304245, 0.6594687700271606, 1.0829840898513794, 0.19635739922523499, 1.0098568201065063, 0.9197697639465332, -0.8754369020462036, -0.0628175139427185, -2.136282205581665, -1.3262264728546143, -0.24952396750450134, 0.3842032551765442, 0.5457130074501038, -0.3109284043312073, 0.27641141414642334, 0.6057535409927368, -0.6496527194976807, 1.0097020864486694, 0.7401390075683594, 0.6273065209388733, -1.7849266529083252, -1.5855966806411743, 1.4779512882232666, -1.431030035018921, -2.0780599117279053, -0.7287245988845825, 0.15228961408138275, 0.7401921153068542, 0.32748472690582275, -0.7020464539527893, 0.516826868057251, -0.3002215325832367, -0.7570671439170837, 1.8617709875106812, -1.2256418466567993, -0.6526793241500854, -0.04775388911366463, 0.9762961864471436, -0.43574970960617065, -0.548485279083252, -1.2171367406845093, 0.21426130831241608, 1.7417579889297485, -1.6342096328735352, -0.8818744421005249, -0.2110387235879898, 3.248823881149292, 0.3681195080280304, 2.010528564453125, 0.7250434756278992, 0.054658882319927216, -0.7998647093772888, 0.9329438209533691, 0.3108576834201813, 2.1820452213287354, -1.75129234790802, -1.4988083839416504, -0.6678555607795715, 0.8912045955657959, 0.7562576532363892, 0.6179751753807068, 2.147861957550049, 1.1773921251296997, -0.8121391534805298, 1.752376914024353, -0.1950150579214096, -0.45571964979171753, 0.8259743452072144, 1.1011600494384766, -0.9675643444061279, -1.560349702835083, 1.4915739297866821, -0.27094566822052, -0.8378798961639404, 0.6611841917037964, 0.9764677286148071, 0.7156433463096619, 0.059941913932561874, -0.2765941917896271, 0.635091245174408, 0.31242990493774414, 0.2091994285583496, 1.0572645664215088, -0.6427189707756042, 0.6501960754394531, 1.6035501956939697, -0.2341509759426117, -0.19457165896892548, 0.9635891914367676, -1.57657790184021, 0.8996344804763794, -1.1903914213180542, 0.19031472504138947, 0.7650161385536194, -0.8294323086738586, 0.24533075094223022, 0.6256586909294128, 0.9463627934455872, 0.7546431422233582, -0.8781792521476746, -0.2771448791027069, -1.7543344497680664, 1.91353440284729, 1.3542765378952026, 0.9474250674247742, -0.690343976020813, 0.5943169593811035, 0.6685635447502136, -1.2345491647720337, -0.3927413523197174, 0.7564135789871216, 0.7168548107147217, -0.9232046604156494, 0.9587090015411377, 2.429982900619507, -0.8537172079086304, 1.152416467666626, 0.17460604012012482, 0.3263666033744812, -0.3231782615184784, 0.2553233504295349, -0.3144036829471588, -0.8293372988700867, 0.22193709015846252, -1.6290079355239868, 0.9684926867485046, 0.23391783237457275, -0.19151028990745544, 0.33648383617401123, 1.0263481140136719, -0.4500064551830292, 0.6942779421806335, -1.4640039205551147, -2.179910182952881, -1.9249002933502197, 1.4811347723007202, -1.3501094579696655, -0.9943178296089172, 1.5556303262710571, 0.44229450821876526, -0.5474629402160645, 0.7962616682052612, -1.5383446216583252, 2.3027379512786865, 1.711747169494629, -0.8227745294570923, -1.3007404804229736, 1.3777399063110352, 1.4106868505477905, -0.0787941962480545, 0.4057472348213196, -0.19194483757019043, 0.7110061645507812, -1.5070277452468872, -1.4503618478775024, 1.4747434854507446, -1.7613500356674194, 1.1011691093444824, -2.097024440765381, 1.1761623620986938, -0.8421810865402222, -0.04151947796344757, -0.1023206114768982, -0.08714559674263, -1.039694905281067, -1.0011329650878906, -1.8727645874023438, -0.22183337807655334, -1.2223753929138184, 1.101341724395752, 1.0197736024856567, -0.054913874715566635, 1.0363051891326904, -0.021224090829491615, 0.7067561149597168, -1.215972661972046, -0.22765487432479858, -0.3595524728298187, -0.3690119683742523, -0.3330347537994385, -0.8200773596763611, 0.14630529284477234, 0.6727205514907837, -0.7774960994720459, 1.4977574348449707, -0.20393621921539307, 1.1330389976501465, -0.3060813248157501, -1.626974105834961, -1.2953194379806519, 0.08258678764104843, 1.2117719650268555, -0.13890407979488373, -0.06540864706039429, 0.770892322063446, 0.9906935691833496, -3.4742655754089355, 0.5543097257614136, 0.41453853249549866, -0.10908198356628418, 1.3802169561386108, 1.0449432134628296, -1.3527066707611084, -1.2845734357833862, -1.5838161706924438, 0.8075011968612671, 0.6006606817245483, -0.01577966846525669, 0.7555567622184753, -0.38439735770225525, 0.3715137839317322, 0.3293670117855072, -0.4793965220451355, -0.41816744208335876, -0.33527296781539917, -0.5832111239433289, 0.34131860733032227, 1.8237062692642212, -1.0536924600601196, 0.662352979183197, 1.633622646331787, -0.05940105766057968, -0.005740574095398188, 0.6195985674858093, 0.9325218200683594, -0.8866948485374451, 0.5004594326019287, 0.9243746399879456, 0.823228120803833, 0.3049478530883789, 0.9970409870147705, 0.07110203057527542, 0.24494726955890656, -1.1053712368011475, -0.004241896793246269, -1.8282660245895386, -0.2701590955257416, -1.4685943126678467, 0.3440324664115906, -0.4701251685619354, 1.1936920881271362, 0.01062548253685236, 0.6941620707511902, -0.06272800266742706, -0.7176366448402405, -0.4695455729961395, -0.4673818051815033, 1.0387922525405884, 1.3077962398529053, 1.652850866317749, 0.04371229186654091, 0.37115877866744995, 0.5123959183692932, 0.033271197229623795, 1.6300525665283203, 1.0476200580596924, -0.6320376992225647, -0.23788584768772125, 1.333052396774292, 1.0924372673034668, 0.1747892200946808, -1.1441128253936768, -1.0426243543624878, 1.3694181442260742, 0.6900936365127563, 0.6601430773735046, 0.10069654881954193, 0.40670520067214966, -1.088015079498291, -1.3172187805175781, -0.9354620575904846, -1.1979442834854126, 1.3402639627456665, 0.913813591003418, 0.04884616658091545, -0.5222124457359314, -1.5775532722473145, 2.091566801071167, 1.172036051750183, -0.490578830242157, 0.13083887100219727, 0.28659263253211975, -0.7663832902908325, 0.917324423789978, 0.18018023669719696, 3.0245156288146973, 0.4266539514064789, -0.20866341888904572, -1.7755520343780518, -0.20961646735668182, 0.23935548961162567, -0.893255889415741, 0.09760477393865585, 0.9482384324073792, -0.2649398148059845, -0.11920277029275894, 0.34357765316963196, 0.027334263548254967, -1.0051430463790894, -0.28461313247680664, 1.8013691902160645, -0.47571247816085815, 0.5439374446868896, -0.45580315589904785, 0.7191565632820129, -0.44837191700935364, 1.176560878753662, -0.41771602630615234, 1.2957173585891724, -2.755009889602661, 0.06587250530719757, 0.762085497379303, 2.1998722553253174, 0.8420024514198303, -0.07181087136268616, -0.5420013666152954, -0.20506328344345093, -0.8681601285934448, -0.7589473128318787, -1.0772169828414917, 0.10998544096946716, -0.6972184181213379, -0.16013935208320618, 0.6360540986061096, -0.5503973960876465, -0.12164738029241562, 1.0096920728683472, -1.4098310470581055, -0.9867419004440308, -0.1289309412240982, 0.4909425675868988, 0.21842584013938904, 0.1992311030626297, 0.13963471353054047, -1.49836266040802, 1.5687687397003174, 0.3057716488838196, -0.1633445918560028, -1.2658259868621826, 1.400559902191162, -0.4415477216243744, 0.41313987970352173, -0.08935254067182541, 0.5080804824829102, -0.18774010241031647, -1.6292047500610352, -1.135613203048706, 0.9457292556762695, -0.39754030108451843, 0.6365719437599182, 0.5163321495056152, -0.21196413040161133, 0.8076221346855164, 0.09181679040193558, -0.6774595975875854, 0.3324393033981323, 0.6048768162727356, -0.28661903738975525, 0.11906204372644424, -0.12259482592344284, -0.1074400395154953, -0.5349175930023193, -0.36966872215270996, -0.38884538412094116, -0.3969584107398987, -0.14366956055164337, -1.4790537357330322, -0.05878272280097008, 1.7530808448791504, 0.2069288194179535, -1.0813137292861938, -0.8498754501342773, 1.7160297632217407, -1.9774606227874756, 0.2611555755138397, 0.38192927837371826, -0.6707087755203247, -1.5106043815612793, 0.07327350229024887, -0.9457443356513977, -1.2630023956298828, -1.7942876815795898, -1.858768343925476, -0.4880410134792328, 0.7350695729255676, -0.6812519431114197], "dtype": "float32"}, {"__nd__": [0.7552886009216309, 0.41863036155700684, -0.9277275800704956, -1.2317759990692139, -1.0586559772491455, 0.25439783930778503, -2.1227023601531982, 0.5301753282546997, 1.1022346019744873, -0.10080412030220032, 0.7969366312026978, -0.5069459676742554, -1.4112837314605713, -0.656311571598053, -0.3984297811985016, -0.5421417355537415, -0.001984005095437169, -0.49248966574668884, 0.10298339277505875, 0.5969032645225525, -1.8967171907424927, -1.5468592643737793, 0.3152754604816437, -1.758924126625061, 2.0313048362731934, 0.903052031993866, 0.07653778791427612, -1.2097007036209106, 1.6357624530792236, -1.9624226093292236, -0.7699077725410461, -1.9456311464309692, -1.2936259508132935, 0.25542858242988586, -0.13267210125923157, 0.22861143946647644, 0.7766832113265991, 0.19158928096294403, 0.14862489700317383, -0.3786716163158417, -0.28842076659202576, 1.083476185798645, 1.0463018417358398, 0.3777291476726532, -1.3042386770248413, -0.9537027478218079, 0.8531448841094971, -2.1706655025482178, 0.8525364398956299, -0.9938914775848389, -0.42966240644454956, -1.4348225593566895, -1.308712363243103, 1.1910805702209473, 1.4202171564102173, 0.37151241302490234, 2.237773895263672, -0.9114142656326294, 0.47487112879753113, -1.8234812021255493, 0.39223217964172363, -0.5093583464622498, 0.656773567199707, -0.42029184103012085, 1.0939440727233887, 1.3546242713928223, 0.645119845867157, 0.3675607740879059, -0.8516783714294434, 0.1456742286682129, 1.629318356513977, 0.5937590003013611, -1.4777779579162598, -1.4138623476028442, 0.44407662749290466, 0.43958863615989685, -1.247668981552124, -0.489692360162735, -0.384125679731369, 0.373066246509552, 0.0003826686879619956, -0.5670386552810669, -0.3150407373905182, 0.11080780625343323, 0.9358954429626465, -1.1065402030944824, 0.09767696261405945, 0.7742350697517395, 0.7243000864982605, 0.8120183348655701, 0.8887684345245361, -0.7516492605209351, -0.09874998033046722, 0.39199742674827576, 0.5614988803863525, 0.22917106747627258, -0.07309581339359283, -0.04885197803378105, -1.181410312652588, -0.2475687712430954, -0.7783305048942566, 0.7467458844184875, 1.982343077659607, 0.17389057576656342, 0.4606212079524994, 0.3020974397659302, 1.1361924409866333, -1.496809482574463, 0.0667768269777298, 0.1325668841600418, -0.9122881293296814, 0.7070545554161072, 0.27303117513656616, 0.42966994643211365, 0.12750551104545593, 2.6095361709594727, -0.6981167197227478, 1.054351806640625, -0.8896390199661255, 0.3693121075630188, 0.9274801015853882, 0.9300026297569275, 1.1205288171768188, 0.4655435085296631, 1.738526463508606, 1.2145230770111084, 3.4533185958862305, 0.2259509116411209, 0.5894888043403625, 0.15955422818660736, -0.5856664180755615, -0.3265080451965332, 0.008613720536231995, -0.1756196767091751, 1.1698077917099, -0.0708857923746109, 1.0253722667694092, -1.5973567962646484, 0.620901346206665, 0.5693347454071045, -1.1610041856765747, 1.1068508625030518, -0.6914586424827576, -0.13016316294670105, -0.572414219379425, 0.2587107717990875, 0.7050179839134216, 0.4734986424446106, 2.1452250480651855, 0.09967780858278275, 0.8318623900413513, 1.3069857358932495, 1.118802785873413, -0.5909895300865173, 0.5635366439819336, -0.46469801664352417, 0.05312535539269447, 0.20693370699882507, 1.2287766933441162, 1.0517001152038574, -0.23773489892482758, 0.4451708495616913, 0.7520594000816345, 0.931340754032135, 0.5122062563896179, -0.6902856230735779, -1.9948811531066895, -2.391894578933716, 0.610817015171051, 0.030285129323601723, -1.046981692314148, 1.6223431825637817, -0.8110072016716003, 2.1891417503356934, 1.0229262113571167, -2.118844509124756, 0.2622540593147278, -2.5639939308166504, -0.06893948465585709, -0.8109362125396729, 0.6646455526351929, -0.23933953046798706, 0.15596666932106018, 0.005893745459616184, -0.6816694736480713, -0.2482547163963318, -0.12019370496273041, 1.6386497020721436, -1.4234137535095215, -0.8775576949119568, -0.09187565743923187, 1.4602373838424683, 0.04471415653824806, 0.9049791097640991, -1.6437538862228394, 0.7937003970146179, 0.5603068470954895, 0.4924357533454895, -0.44289758801460266, 0.26572880148887634, -0.16203634440898895, 1.9714305400848389, 0.5925052165985107, 0.1089530736207962, 0.30098825693130493, -3.356550931930542, -2.1575100421905518, -0.8120512366294861, -2.7472009658813477, 0.8805027604103088, 1.0429799556732178, 0.982791543006897, -1.7265825271606445, -1.2135759592056274, 0.6734291315078735, 1.0065959692001343, 1.1718382835388184, -0.43344607949256897, 0.10514494776725769, -0.8143972754478455, -0.41463083028793335, -0.11317905783653259, 1.4437625408172607, -0.393433153629303, 0.7287805676460266, -0.41033637523651123, -0.5068588852882385, -0.45197805762290955, -0.6637375354766846, 0.8366112112998962, -0.46360325813293457, -0.7885212302207947, 0.7750304341316223, -0.0771605521440506, 0.25578925013542175, 0.774753987789154, 0.40043991804122925, 0.4048366844654083, 0.6350721716880798, 1.3351678848266602, 1.4395860433578491, -0.37352779507637024, -1.48933744430542, 1.7889350652694702, -0.6917679905891418, -0.7858229875564575, 0.41296541690826416, 1.0467424392700195, -1.097647786140442, -0.07983353734016418, -1.1044541597366333, 0.5356544852256775, 0.20768727362155914, -0.47646722197532654, 1.212707757949829, -0.2227230966091156, 0.8613985180854797, 0.8156346082687378, 0.3414197862148285, -0.19454526901245117, -0.5850632190704346, -0.6148464679718018, 1.077695608139038, -0.7033402323722839, -1.3166645765304565, -0.10502811521291733, -1.7558327913284302, 0.37847086787223816, -1.1152997016906738, -0.23831738531589508, 0.7569468021392822, -0.7255503535270691, -1.5693129301071167, 0.25765225291252136, -1.6072288751602173, -0.01692940481007099, -0.12369484454393387, -0.4504835605621338, 1.3447680473327637, -0.06567885726690292, -0.9563625454902649, -0.4638125002384186, 0.4951944351196289, -0.4414055049419403, -0.4067203104496002, -0.645668089389801, -0.21019573509693146, 0.6331902742385864, 1.1417113542556763, 1.2632232904434204, 0.02039995975792408, 0.2244601994752884, -1.394282579421997, 0.18801631033420563, 0.3226650059223175, -1.2969958782196045, -0.049882806837558746, 0.41490688920021057, 0.5654629468917847, -1.0062346458435059, -0.17066256701946259, 1.3037885427474976, -0.6153054237365723, -0.29597026109695435, 0.9074820280075073, -1.9803882837295532, 0.5065909624099731, -0.7037938237190247, -0.04060946777462959, -0.1802394986152649, -0.8033750653266907, 0.0009193088626489043, 2.503138303756714, -1.0517370700836182, 0.04037145897746086, 0.8174682855606079, -0.8922753930091858, -0.3782820403575897, -0.47697630524635315, -0.8524871468544006, -1.433475136756897, 0.5400428175926208, 0.6638224124908447, -0.08356530219316483, -0.6325864791870117, 1.4150745868682861, 1.067191481590271, 0.6090388894081116, 0.4458492696285248, 1.6419377326965332, -1.1602479219436646, 0.32526957988739014, -0.44873639941215515, 1.294552206993103, -1.9767669439315796, 0.3758486807346344, 0.11160966008901596, 1.241925835609436, 0.40852904319763184, -0.4519907832145691, 2.179148850700585e-06, -0.3348517417907715, 0.951069176197052, 0.5831134915351868, -1.25971257686615, -1.8487334251403809, 0.49943459033966064, 0.1392933875322342, -1.7397429943084717, -0.9715255498886108, -0.9659246802330017, -0.9932020902633667, 0.6901753544807434, 0.23121926188468933, -1.1221046447753906, -0.490976482629776, -0.386428564786911, 0.4501494765281677, -0.818817138671875, 1.0335869789123535, 0.7956876754760742, 0.23004986345767975, 1.9782873392105103, 1.6256555318832397, 0.3054827153682709, 0.917599081993103, -0.7565076947212219, -1.881927490234375, -0.5609092712402344, 2.0629546642303467, -0.9627700448036194, -0.809709370136261, -0.08317296952009201, 0.030871955677866936, -0.036041319370269775, 0.8181211352348328, -0.5287032723426819, -0.8319724798202515, -0.3915006220340729, 0.48896324634552, -0.10772785544395447, 0.4523119032382965, -0.05263880267739296, 0.8357027769088745, 0.15530164539813995, 0.7379031181335449, 1.1150145530700684, -0.5008239150047302, 0.5100096464157104, -0.3328995406627655, -1.0096381902694702, -0.5154038667678833, -0.2466113567352295, -1.430309772491455, -1.487281084060669, 1.1598796844482422, -0.5641224980354309, 0.019938137382268906, -1.055430293083191, -0.5412866473197937, 0.5599506497383118, -1.4618208408355713, -1.2131181955337524, -0.7976404428482056, 1.1507439613342285, -0.42210903763771057, 1.2753868103027344, -0.12372113019227982, -0.5028057098388672, 0.24205270409584045, -0.044215261936187744, -1.3672235012054443, -0.14226692914962769, 1.7734057903289795, -0.2138299196958542, 0.048613041639328, 1.1545780897140503, -1.017975091934204, -0.1847323775291443, -0.8628380298614502, 1.4917100667953491, 0.32592183351516724, -0.9321145415306091, 0.6488314867019653, -1.1614247560501099, 1.084333062171936, 1.213962197303772, -0.4810155928134918, 1.1355347633361816, 1.0457125902175903, -0.024439936503767967, -0.0738198310136795, -1.507553219795227, 0.798314094543457, -1.4350498914718628, 0.8793388605117798, 0.03954765200614929, -0.3737392723560333, -1.6233993768692017, -0.31750714778900146, 0.30357202887535095, -0.011896687559783459, 1.0331958532333374, -1.3560125827789307, 1.7156339883804321, 0.30107930302619934, 0.23654784262180328, -0.764543890953064, 1.4519553184509277, 0.7735138535499573, -0.4615131616592407, -0.6457530856132507, 0.6536692976951599, 0.6271429657936096, -1.0709271430969238, -1.022360920906067, -0.5247752070426941, 1.2447962760925293, 0.01135795097798109, 0.2554214596748352, 1.7458692789077759, -0.8338539004325867, -0.883408784866333, -1.9330518245697021, -1.158334732055664, 0.24045659601688385, -0.0037256372161209583, -0.26177385449409485, 1.9818004369735718, 0.932165265083313, -0.17927765846252441, -0.09230291098356247, 0.21225833892822266, 1.7473403215408325, -1.0805248022079468, 1.0036882162094116, 0.3755458891391754, -0.7057580947875977, 0.6611847281455994, 0.9296430349349976, 2.2563705444335938, -0.22447215020656586, -0.4126266837120056, 0.7434293627738953, -0.7653844356536865, -0.5612878799438477, 0.0222734697163105, -0.9289745092391968, -0.6616389751434326, 1.5447750091552734, -0.7756950259208679, -0.2907947897911072, 0.474015474319458, 1.2978169918060303, -0.10252492874860764, -0.4713306725025177, 0.4574906527996063, -1.6520321369171143, -1.1522856950759888, -0.8710275888442993, 0.3145209550857544, 1.2198692560195923, 0.4438086152076721, 1.2688562870025635, -1.2897125482559204, 0.9480003118515015, -0.8922404050827026, -0.5380018949508667, 0.9732832908630371, -0.6297730207443237, -0.7084349989891052, -1.5098545551300049, -0.308901309967041, -1.775999903678894, -0.06972947716712952, 1.1626524925231934, -0.4087360203266144, -1.7231537103652954, -0.6495411396026611, -0.19821429252624512, 0.3776891529560089, -1.8717460632324219, -1.953174352645874, -1.1995270252227783, 0.5380942225456238, -1.0886062383651733, 0.37760117650032043, 0.5829837918281555, -0.06888268887996674, 1.5207140445709229, -0.7947311401367188, -0.7218841910362244, 0.18993443250656128, -0.2997906506061554, 0.7919213771820068, 0.418578177690506, -0.5411372184753418, -0.8073415160179138, -0.5759851336479187, -1.581608533859253, -0.43826791644096375, 1.899484634399414, -0.38445982336997986, -0.003817568998783827, 0.12270122766494751, 1.6145249605178833, -0.8367703557014465, -0.1328616440296173, -0.5108973979949951, 0.3348231911659241, -0.14998818933963776, 0.2664719521999359, -0.7449280023574829, -1.1517974138259888, 1.0981616973876953, 0.22354216873645782, 0.792499840259552, 0.31254109740257263, -0.9698730111122131, 0.4073086977005005, -0.0527510903775692, -1.8198171854019165, -0.49327027797698975, -0.062232472002506256, 1.093293309211731, 0.7821606993675232, 1.6454010009765625, 0.6710917949676514, -0.7730545997619629, -0.6125047206878662, -0.8445119261741638, 0.010007482953369617, -0.12928177416324615, -0.37031564116477966, 0.17670901119709015, -0.5665990710258484, 1.1419265270233154, -1.4680907726287842, -1.1182769536972046, 0.7358633875846863, 0.1445644050836563, -0.5912823677062988, -0.011526091024279594, 0.06852652132511139, 0.41816791892051697, 0.5056663155555725, 1.338270664215088, -0.37786877155303955, 0.5871140956878662, 1.0236965417861938, -1.4875314235687256, -0.7684810757637024, 0.4198063910007477, -1.0404051542282104, -0.29299208521842957, -1.8625880479812622, 0.5787937641143799, -0.07663091272115707, -1.2318471670150757, 0.3100038766860962, 1.31449556350708, 0.7022977471351624, 0.042815640568733215, 1.4737516641616821, -0.2995500862598419, 0.4317936599254608, 0.503181517124176, -0.41966524720191956, -0.4998893141746521, 0.8366042375564575, -1.923174262046814, -1.4867830276489258, -1.8425929546356201, -0.25865650177001953, -0.20916609466075897, 1.1733441352844238, -0.45627185702323914, -1.163872241973877, 0.6254661679267883, -0.33204028010368347, -0.11634119600057602, -2.3446271419525146, -1.0223039388656616, -1.2092992067337036, -0.28650176525115967, -0.26811131834983826, -0.16336117684841156, -1.314560055732727, 1.7344789505004883, -0.04165402799844742, -0.4920275807380676, 1.6791859865188599, -1.922741174697876, -0.5826094746589661, -1.8227535486221313, -1.0967562198638916, 0.571325421333313, -0.31039419770240784, -1.2457339763641357, 0.3649294972419739, 0.6007667779922485, -0.31395211815834045, -1.1931875944137573, 1.6090093851089478, -1.0050877332687378, 0.6876190304756165, -0.15682505071163177, 0.26017484068870544, 0.19706563651561737, -0.7250344753265381, 1.1754146814346313, 0.6225515007972717, -1.419675350189209, 0.9478722810745239, 1.2924258708953857, -2.60546612739563, -0.19851277768611908, -1.3551132678985596, 0.5218154191970825, 0.5504992008209229, -0.10885301232337952, 1.7940014600753784, 0.8461108803749084, -0.4974181056022644, -1.5785011053085327, -1.8020988702774048, 0.568325400352478, 0.45717132091522217, 0.28435462713241577, 2.0477075576782227, 0.10096859186887741, -0.7056009769439697, -0.11004618555307388, 1.4870809316635132, 0.5058480501174927, -1.7537696361541748, 0.7778986096382141, 0.5401965975761414, 0.06575223803520203, -0.960108757019043, 0.7758590579032898, 0.4394177794456482, -1.962974190711975, 1.102135181427002, -0.07648594677448273, 0.21438373625278473, -0.28215277194976807, 1.307562232017517, 1.7544959783554077, -0.5453846454620361, -1.4051482677459717, -0.9853783845901489, 0.64895099401474, -0.7723601460456848, -0.04691941663622856, 0.8645334243774414, 1.221906065940857, -0.9017579555511475, -0.8816723823547363, -0.30458325147628784, 0.3547413647174835, 0.21647889912128448, -0.24188873171806335, 0.6122581362724304, -0.8021463751792908, 1.028808355331421, 0.976080060005188, -0.7962211966514587, -0.1362985223531723, 0.5081983208656311, -2.000800609588623, 1.6395512819290161, 0.5974061489105225, 0.6004849076271057, 0.6819120049476624, 0.6978543996810913, 2.2315075397491455, -1.0513830184936523, -1.865859866142273, 0.37065860629081726, -0.6866623759269714, -1.6238315105438232, -1.0377469062805176, -1.257185697555542, 1.0776760578155518, -0.5786851644515991, -0.2722960412502289, -1.513838529586792, 0.20534563064575195, 1.1297125816345215, 0.8086774349212646, -1.4847121238708496, -0.5094237327575684, 0.3719549775123596, 0.15407808125019073, 0.3992093503475189, 0.060505080968141556, -0.9203038811683655, -0.5709003210067749, -0.9398021697998047, -0.7964873313903809, -0.033388182520866394, 0.4014882445335388, 1.3495644330978394, -1.697687029838562, 1.9075132608413696, 0.6411175727844238, -2.3015189170837402, 0.9531136751174927, -1.7195813655853271, -0.292851060628891, -1.7722400426864624, -0.05687354505062103, -0.3003498911857605, 1.3736674785614014, 2.3942770957946777, 0.7819526195526123, 0.8798328042030334, -2.723008155822754, 1.0629112720489502, -0.46406808495521545, -0.7453017830848694, 0.13741175830364227, -2.010361671447754, -0.21702182292938232, 0.11653341352939606, 1.112785816192627, -1.8758151531219482, -0.810376763343811, 0.8501267433166504, 2.079242706298828, 0.3880528211593628, 0.5744361877441406, 0.17350028455257416, 0.32914575934410095, -0.04759510979056358, -1.1060863733291626, 0.45259639620780945, 0.35888907313346863, 0.9701153039932251, 0.08412623405456543, 0.21166467666625977, -0.5844533443450928, 0.480057954788208, -0.32223647832870483, -0.38967186212539673, -0.6518362760543823, -0.32199567556381226, -0.041949715465307236, 0.7230556607246399, -1.5341538190841675, -1.879615306854248, -0.5315554738044739, 0.5277085304260254, -0.9235025644302368, 0.5133700966835022, 0.3051519989967346, 0.25809967517852783, -0.6463649868965149, -1.892846941947937, 0.7089110016822815, -1.207903504371643, -1.649611473083496, 1.5900007486343384, -0.1807100474834442, 0.2393871396780014, 0.18041571974754333, -0.7830225825309753, -0.5189021825790405, 0.9496005773544312, 1.2931787967681885, 0.21485210955142975, 0.04945903643965721, 0.24644264578819275, -0.26793915033340454, 0.21220390498638153, 0.021807299926877022, -2.3469982147216797, 0.49530625343322754, -0.9126953482627869, 1.196306586265564, 0.8321363925933838, -1.401113748550415, -0.2617597281932831, -0.019632725045084953, 1.0025808811187744, -2.2653281688690186, -0.7875213623046875, -0.06838217377662659, -1.1861040592193604, -0.7175841331481934, 0.9688195586204529, 0.3225823938846588, 1.9215601682662964, -1.4939547777175903, 2.2163453102111816, -1.759311556816101, 0.8322917819023132, 0.12612412869930267, -0.12631754577159882, -0.5324306488037109, -1.1491044759750366, 0.6533218026161194, 1.3122735023498535, -0.04775021970272064, 0.5029645562171936, -0.6331271529197693, 0.38920116424560547, -0.6098620295524597, -0.11300526559352875, 0.5408381819725037, -1.5955134630203247, -0.299581915140152, 1.6879123449325562, -0.515521228313446, 0.19655296206474304, -0.4520339071750641, -2.884216070175171, -1.6007882356643677, -2.1232104301452637, 0.3613458275794983, -0.6875776648521423, -1.323965311050415, 0.07048048079013824, 0.6164606213569641, 1.2343031167984009, 0.3423249423503876, -0.6131448745727539, -0.14635133743286133, -2.0413968563079834, 0.013663608580827713, -0.8023979663848877, -0.7370194792747498, 0.29432082176208496, 1.0054525136947632, -0.6694328188896179, -0.9146921038627625, -0.39556360244750977, -0.648827850818634, -0.5785049796104431, -0.7316863536834717, 2.0287437438964844, 0.3058825433254242, -0.9147546887397766, -1.818153977394104, 0.04959999397397041, 1.1673917770385742, 0.1301136165857315, 0.6464791893959045, 0.26885008811950684, 0.08701752871274948, 0.5474525690078735, 0.8008953928947449, 0.24049341678619385, -0.006246301345527172, -0.5746368169784546, 1.4895350933074951, 0.44770756363868713, -0.06770230829715729, -1.5765916109085083, -2.7149553298950195, -1.6516720056533813, -1.678279995918274, -0.07370208203792572, 0.1373148262500763, 0.37008965015411377, -0.1755276322364807, 0.9673328995704651, -1.3566280603408813, -1.927371621131897, -0.3809855282306671, 0.32142874598503113, -0.8404624462127686, 1.6863007545471191, 0.8768951892852783, -1.3023189306259155, 0.8544638156890869, -0.8018466830253601, -1.0219049453735352, -1.5003520250320435, 0.7488049268722534, 0.4068971574306488, -1.635958194732666, 0.27496498823165894, -0.01983710750937462, -0.7497535347938538, -0.9505313634872437, 0.5663407444953918, -1.1516506671905518, 0.4806307256221771, 1.068788766860962, 0.22101330757141113, 0.7768559455871582, -0.8132770657539368, 0.949154794216156, -1.7014126777648926, 0.7450014352798462, -0.3673836588859558, 0.01514117419719696, -0.6217913627624512, 0.7526155710220337, 0.341116726398468, -0.6639652252197266, 1.2384573221206665, -0.009127506986260414, 0.5954821705818176, -0.041850537061691284, -0.21166430413722992, -0.25044718384742737, 0.9172722697257996, 0.2401687502861023, 0.4955180287361145, 0.41921278834342957, 1.0548489093780518, 0.021916145458817482, 1.2998586893081665, 1.9814807176589966, -0.027465732768177986, 0.9189494252204895, 0.5616990923881531, -0.7260850667953491, -3.056452512741089, -0.7250417470932007, 0.08197734504938126, -1.7338041067123413, 0.8675850629806519, 0.5661049485206604, 1.5453895330429077, 0.8271021246910095, 0.8956803679466248, 1.4899646043777466, 1.3831392526626587, -1.6088122129440308, 0.4748002886772156, -0.5465592741966248, 1.0580569505691528, 1.2336045503616333, -0.9690794944763184, 1.0212602615356445, 1.6885879039764404, 0.44359323382377625, -1.6472609043121338, 0.5385327935218811, -1.2528839111328125, 0.21737439930438995, -0.32243019342422485, 0.7169991135597229, -1.2439024448394775, -0.6974315643310547, -0.0035265327896922827, 1.3976539373397827, 0.45870986580848694, -1.3626761436462402, -0.0648336261510849, 0.5038946270942688, 1.3209091424942017, -1.291432499885559, 1.27016282081604, 0.2410329431295395, 0.047545190900564194, -0.5381367206573486, -1.7909424304962158, -1.4294157028198242, -0.30342602729797363, 0.06187814474105835, -0.7557145953178406, -0.20103198289871216, -1.3819166421890259, 0.04098108038306236, 0.0010935115860775113, -1.5708383321762085, 0.12033061683177948, -0.6383904218673706, 0.8461569547653198, 0.3963981866836548, -0.0846230536699295, -1.3946510553359985, -0.6865785121917725, 1.8407299518585205, -0.7461443543434143, -0.2918144166469574, -0.6210551857948303, -1.2354450225830078, -1.722969651222229, 0.5825024843215942, 1.1265573501586914, -1.4122111797332764, 0.20470570027828217, -1.4291927814483643, -0.9816937446594238, -0.14185954630374908, -1.7233648300170898, -1.4144338369369507, -1.2766739130020142, -0.3313860297203064, -0.5707452893257141, -0.5484203696250916, 0.16482512652873993, 1.4950282573699951, -2.1281661987304688, -0.2341507524251938, -0.2779519259929657, 0.3488040864467621, -0.16269773244857788, 0.5039569139480591, -0.852258026599884, 1.4087026119232178, 1.2385307550430298, -0.20445021986961365, 1.028969645500183, -1.0804169178009033, -2.10599946975708, 1.0975204706192017, -0.5471495985984802, 1.1224473714828491, 0.23867207765579224, -0.12117287516593933, 0.6953190565109253, 1.3836829662322998, -0.36496084928512573, -0.4347778856754303, 1.47443687915802, -1.449405550956726, -0.3765052855014801, -0.4966430068016052, 0.214914470911026, -1.1420183181762695, -0.8057509660720825, -0.56186842918396, -0.23023642599582672, 0.07507956773042679, -1.5214760303497314, 1.586510419845581, 0.00923383329063654, 0.7481553554534912, 0.7213817238807678, -1.1343579292297363, 0.4638339579105377, -0.30440816283226013, 0.02701006829738617, 1.1923717260360718, -0.5841156840324402, 0.8355517387390137, -1.8742316961288452, 0.5470985174179077, -1.0354082584381104, -0.5597512125968933, -0.7013158798217773, 0.9173237681388855, 0.37352174520492554, 0.8558467030525208, -1.2898151874542236, 0.45867249369621277, 1.5863865613937378, -1.3101409673690796, -0.2583971619606018, -0.9028916954994202, -1.0208667516708374, 0.6221433877944946, 0.8699634075164795, -0.4961565136909485, 0.6060176491737366, 0.19877366721630096, -1.565743088722229, -0.21798153221607208, 0.21011973917484283, -1.3028327226638794, 1.086612582206726, -1.9588979482650757, -0.4653998017311096, 1.1533920764923096, -2.089982509613037, -1.207650065422058, -1.1051603555679321, 1.4496281147003174, -0.07699351757764816, -0.5388942956924438, -1.360492467880249, 0.323532372713089, 0.755865216255188, -0.7532280683517456, -0.5516535043716431, 2.0250632762908936, -0.4256877303123474, 1.614016056060791, -1.3710296154022217, -0.055628109723329544, -0.20121803879737854, 0.5508241653442383, -0.23379135131835938, 0.9845705628395081, -3.609314203262329, 0.2628706097602844, 0.669307291507721, 1.1324042081832886, 2.3655776977539062, 1.1839938163757324, 0.6957575082778931, -0.14433591067790985, -0.01129802968353033, -1.1027368307113647, 0.8609742522239685, 1.3128632307052612, -1.06493079662323, 0.12742401659488678, -0.46011465787887573, -0.9934765100479126, -0.8560104966163635, 1.9946258068084717, -0.1108999252319336, -0.6483084559440613, -1.004922866821289, -0.8848320841789246, 0.9066063165664673, 1.4280749559402466, -0.46656981110572815, -1.3377526998519897, -0.88057541847229, -1.1368118524551392, 0.32289835810661316, -0.53369140625, 1.4560272693634033, 0.9099798202514648, 1.6252892017364502, -0.7830810546875, -0.5755867958068848, -0.2623401880264282, 0.33805665373802185, 0.4536111056804657, 0.7667331695556641, -1.1033953428268433, -0.29168763756752014, -1.2175567150115967, 0.9244581460952759, -0.959804892539978, -2.2828590869903564, 1.3504916429519653, 0.21027961373329163, -0.7388534545898438, -0.2537722885608673, -0.582023024559021, -0.8145463466644287, 0.2635657489299774, 2.0408589839935303, 1.5273877382278442, 0.18099619448184967, 0.9065573215484619, 1.3425347805023193, 0.8370779156684875, 0.9316596388816833, -0.3953182101249695, -0.17407196760177612, 0.7369551062583923, 2.06685733795166, 0.02599196694791317, -0.3666691780090332, 0.06489310413599014, 0.4451923370361328, 1.441458821296692, -0.7423199415206909, -1.136250615119934, 2.1683719158172607, 0.652880072593689, -0.6672447919845581, -1.1613643169403076, 1.5800025463104248, 0.29898926615715027, -1.1116893291473389, 0.5205017924308777, -0.08984716981649399, -1.2440004348754883, -1.14303457736969, 0.3928811550140381, 0.4116133153438568, 0.23030561208724976, 0.2577148973941803, 1.0008702278137207, 1.057172179222107, 0.8487810492515564, -0.05378817394375801, -0.7826509475708008, -0.3580334782600403, 0.7366232872009277, -1.0843453407287598, 0.7337248921394348, 1.0030041933059692, -2.5302610397338867, -1.2049901485443115, 1.882935643196106, 1.2120039463043213, 1.6831960678100586, 0.036097194999456406, -2.305887222290039, -1.2535868883132935, 1.7296936511993408, -1.1714849472045898, -0.4417295455932617, -0.16845212876796722, 1.558091402053833, 0.08401714265346527, -0.17207057774066925, 0.3660643994808197, 2.430626153945923, -0.7652145028114319, 0.8355342745780945, 0.4165068566799164, -1.6366040706634521, 1.0029678344726562, 0.8402091860771179, -1.5275928974151611, -1.7086251974105835, -1.89053475856781, -0.2743186056613922, -1.612442970275879, 1.3501511812210083, 0.042633987963199615, -0.20607049763202667, 0.8788864612579346, 1.2256274223327637, -0.7760650515556335, -0.318271666765213, -1.1407275199890137, 0.04589744284749031, 0.586700975894928, -0.0936937928199768, -1.4209579229354858, 0.09568243473768234, -0.1629887968301773, -0.05577319115400314, -0.7751970887184143, -0.6846997737884521, -0.8748655915260315, -0.7127699851989746, 0.46965649724006653, 0.30285942554473877, 0.03493562340736389, -0.9242092967033386, -1.6859900951385498, 1.4088023900985718, 0.286241352558136, -0.1720641851425171, -0.07806800305843353, -0.6350368857383728, -0.9580351114273071, 0.054653532803058624, -0.030842948704957962, -1.7863452434539795, 1.6741917133331299, -1.6640712022781372, -1.209352731704712, 0.23500290513038635, -1.1059788465499878, 1.1293691396713257, 0.1641121357679367, -0.6125122308731079, -1.503609299659729, 0.3017290234565735, 1.4312200546264648, 0.9015523195266724, -1.6950761079788208, 0.0015268810093402863, -0.33558839559555054, 1.7200075387954712, -0.353873074054718, -1.5177605152130127, 0.5024183392524719, 0.5315361618995667, 0.904707133769989, -0.4357604682445526, 0.7759510278701782, -1.1951931715011597, -1.0422409772872925, -1.6792508363723755, 0.9992303252220154, 0.6028612852096558, -0.8133091330528259, -1.3369730710983276, 0.9666796326637268, -0.7022601962089539, -0.4322926998138428, 0.5315344333648682, 1.4904059171676636, -0.4868803918361664, 0.4213440418243408, -0.9184800386428833, -0.24027039110660553, 0.5172666907310486, 0.6502233147621155, -1.4638720750808716, -0.7276859283447266, 0.6416864991188049, -0.06262347102165222, 1.3312175273895264, 1.788108229637146, -1.1580872535705566, 1.2758857011795044, 1.0519241094589233, 1.2657109498977661, -0.7053711414337158, 1.2557052373886108, 0.3035299777984619, 1.5380197763442993, 0.16048790514469147, -1.7537314891815186, 0.23558086156845093, 2.0316925048828125, 0.9681327939033508, 0.038230907171964645, 0.0560658760368824, -0.9961636662483215, 0.7834922075271606, -0.8691251873970032, -0.07595188170671463, -1.1080479621887207, 0.1355576366186142, 0.011466072872281075, 0.058940693736076355, 0.970050036907196, 0.3975900113582611, -0.08345508575439453, 0.892373263835907, -0.29998764395713806, 0.8420319557189941, 0.5844202041625977, -0.10773507505655289, 0.7155085802078247, 1.4951242208480835, 0.45061933994293213, 0.942171037197113, -1.151906967163086, -0.17271439731121063, -0.13037973642349243, -0.5084978938102722, 1.2059820890426636, 0.20147034525871277, 0.28791293501853943, 0.5655968189239502, 0.7859910130500793, -1.9766864776611328, 0.8298024535179138, 0.42278778553009033, 0.18259699642658234, -0.8974084854125977, 0.3599095344543457, 0.5959592461585999, -0.5988181233406067, -0.031710434705019, -1.4739773273468018, 1.6909401416778564, 1.6441975831985474, -0.14635762572288513, -0.12945528328418732, -0.5612962245941162, -0.7572892308235168, 0.10629440099000931, -0.6958029270172119, 0.4845863878726959, 0.5184440016746521, 0.6631414294242859, -0.1286659836769104, -0.5620214939117432, -0.2125350832939148, 0.12319077551364899, -1.2203022241592407, 0.16764222085475922, 0.5054388642311096, 2.1593968868255615, 0.49149924516677856, 0.8904282450675964, -1.271141529083252, 0.2871719002723694, -0.24269194900989532, 0.3188730478286743, -1.1989631652832031, -0.5086947083473206, 1.6956781148910522, 0.81595778465271, 1.1970279216766357, 0.5612199902534485, 0.5278487801551819, -0.35624921321868896, 0.2146385908126831, -0.23669879138469696, 0.6052590608596802, -0.19513912498950958, 2.4406394958496094, -0.15240095555782318, 0.29546600580215454, 1.6515636444091797, 1.8865078687667847, -1.254417061805725, -2.1501142978668213, -1.3728102445602417, 0.23208366334438324, 0.11223580688238144, 0.46234023571014404, -0.0638885349035263, -0.8650064468383789, 1.847514271736145, -1.4537527561187744, -1.0998286008834839, -0.19764462113380432, -1.9110608100891113, 1.5508872270584106, -0.2929321229457855, -1.1762977838516235, 0.3994150459766388, 1.5752519369125366, 0.35543209314346313, 0.9892060160636902, 0.24355749785900116, 0.8217673897743225, -1.4739693403244019, -0.6537668704986572, -0.9818642735481262, 0.13006319105625153, -0.7907615303993225, 0.3863542973995209, 1.0945816040039062, 0.6997238993644714, -0.03510377183556557, 0.8431314826011658, 1.284034252166748, -0.5231118202209473, 0.22427737712860107, -2.168137788772583, 1.2025539875030518, 0.712576687335968, 0.02294343337416649, 0.6176161766052246, -0.5963627696037292, -1.9694595336914062, -0.3947824537754059, 2.60347843170166, 0.36205318570137024, -0.5357930064201355, 2.244967222213745, -2.6254758834838867, 0.08286460489034653, 0.23889698088169098, 0.1234099492430687, -0.01725396327674389, -0.3237384259700775, -0.1357707679271698, -1.6463497877120972, 1.4643491506576538, 0.4382796883583069, 1.5885721445083618, 0.3139495253562927, -0.10624954104423523, 0.9430814385414124, 2.1243410110473633, -0.21793651580810547, 0.1915961354970932, -2.3510751724243164, -1.3612674474716187, -0.3848736882209778, -1.3992239236831665, 0.767879068851471, 3.2223875522613525, 0.2065763771533966, 0.2425152063369751, -1.435413122177124, -0.061407748609781265, -0.5178531408309937, 1.0966894626617432, 0.4066716730594635, -2.3957855701446533, -0.4033213257789612, 0.4037703275680542, -0.5155012011528015, 1.5042191743850708, -1.0745819807052612, -1.8215276002883911, -0.712502121925354, -1.672762155532837, 0.5981649160385132, -1.4578384160995483, 1.140701174736023, 0.3536318838596344, 0.882574737071991, 0.459003210067749, -0.9956371784210205, -0.8444435596466064, 0.3534155786037445, -0.4109490215778351, -0.8341112732887268, 0.11097244173288345, 0.5552472472190857, 0.8007038831710815, -2.0356783866882324, -0.5097029805183411, -0.8433040976524353, 0.5073831081390381, -1.238282561302185, 0.13327302038669586, 0.7194430828094482, 0.15165233612060547, -0.17960867285728455, -0.10199159383773804, -0.07666278630495071, 0.18682582676410675, -0.039305370301008224, 1.1174684762954712, -1.0449531078338623, 0.41148218512535095, 0.32086503505706787, 0.20393872261047363, -0.6612671613693237, -0.4793734550476074, -0.398332804441452, -0.47524312138557434, -2.150689125061035, -0.8854836821556091, 0.042965881526470184, -0.18972796201705933, -0.21629942953586578, 1.1848007440567017, -1.195784091949463, 1.2691411972045898, -0.6890336871147156, -0.36963918805122375, -0.5581008791923523, 1.1397756338119507, 0.9525687098503113, 1.635257363319397, -1.1356604099273682, -0.3504970371723175, 0.1307932734489441, 0.035924557596445084, 1.5454829931259155, 0.7120309472084045, 1.1743190288543701, -1.236140489578247, -0.7251031398773193, 0.45448237657546997, -0.7775498032569885, 1.675215721130371, 1.0407419204711914, 2.5794055461883545, -1.2745035886764526, 0.2719815671443939, -0.6311242580413818, -1.4447047710418701, -0.7644138336181641, -1.9667431116104126, -1.121814489364624, 1.9213814735412598, 0.9868069887161255, -1.4766939878463745, 0.35003259778022766, 0.5913856625556946, 1.957831859588623, -2.0474467277526855, 0.20844578742980957, -0.551602840423584, 0.06250282377004623, 0.241713747382164, -1.113909363746643, 0.31888964772224426, 0.7769421935081482, -1.5644854307174683, 0.3448013365268707, -0.9328014850616455, -0.40006688237190247, -0.24279846251010895, -0.5100147128105164, -1.2183356285095215, -0.6591905951499939, 1.2037333250045776, -0.9746586084365845, 0.06708976626396179, -0.3621073067188263, 1.0172855854034424, 1.1078039407730103, 0.21297840774059296, -1.284661889076233, 0.9118939638137817, 0.6876422166824341, 0.09600619971752167, 0.34119486808776855, -2.151057720184326, -0.48041996359825134, 1.3090746402740479, -1.5958596467971802, 0.878475546836853, 0.395775705575943, 1.7255147695541382, -0.48255714774131775, -1.4503997564315796, -0.5582888722419739, -0.7674939036369324, -0.3729553520679474, -0.8506808280944824, 2.367171049118042, -1.1852773427963257, 0.4160887897014618, -1.5132147073745728, -1.2255058288574219, -1.652484655380249, 1.3129037618637085, -1.0672341585159302, 0.5606135129928589, -0.7007662653923035, 0.16026806831359863, -1.6204369068145752, -1.7015821933746338, 0.020733144134283066, -1.7019985914230347, 0.6656402945518494, 1.1037451028823853, 1.1190578937530518, 0.28647729754447937, -0.16090133786201477, -0.12751711905002594, 1.260334849357605, -1.3213701248168945, -0.8109668493270874, -0.11000651121139526, 0.21486644446849823, -2.2146902084350586, 1.5782324075698853, 0.6816795468330383, 0.44138357043266296, -0.6366191506385803, -1.0037319660186768, -0.8963359594345093, 2.723706007003784, 1.9670354127883911, 0.007179674692451954, 1.8965331315994263, 1.3531543016433716, -0.8251476287841797, 0.9965963959693909, 0.0025542418006807566, -1.3992133140563965, -0.4455728828907013, 0.35177814960479736, -0.8960277438163757, 0.8061895966529846, -0.4975949823856354, 1.2108242511749268, 1.1802316904067993, -1.6983256340026855, 0.6851900219917297, -0.44503048062324524, 0.690231204032898, -0.7761169672012329, 1.4973540306091309, 1.8204196691513062, -0.42992085218429565, 0.6675528883934021, 1.1279053688049316, 0.45955973863601685, -0.8462569713592529, -0.7608889937400818, 1.648878574371338, -0.7089428901672363, 0.715030312538147, 0.9217702150344849, -1.6268165111541748, -0.7855257391929626, -0.6264447569847107, 0.8056452870368958, -0.4626866579055786, 1.209328055381775, 0.6988703012466431, 0.15458400547504425, -1.1112796068191528, 1.0926045179367065, -0.1730828434228897, -0.798844039440155, -0.29397621750831604, -0.39162954688072205, -1.0132499933242798, -1.0779117345809937, 0.18824438750743866, -0.17243130505084991, -1.3536369800567627, -0.7046133279800415, -2.0670435428619385, -0.2083253115415573, -1.149550437927246, 0.31631705164909363, 0.5988153219223022, 0.36741235852241516, 2.556518077850342, -0.9195150136947632, 0.31347328424453735, 1.0910041332244873, 1.096652626991272, 0.43133345246315, -1.4242544174194336, -0.5453340411186218, 0.29174375534057617, 1.0479718446731567, -1.34689462184906, -1.7535820007324219, -0.06950833648443222, 0.6613573431968689, 0.35703420639038086, 0.9080667495727539, -0.07434549927711487, -0.8657617568969727, -1.214038372039795, -0.34514743089675903, 0.7254732251167297, 1.3273682594299316, 0.24310995638370514, 1.6518614292144775, -0.5308379530906677, 0.45432180166244507, -0.4793335199356079, 1.0360572338104248, -0.5550984144210815, 2.4371840953826904, 0.4899302124977112, 0.512545108795166, 0.324042409658432, 0.3415340781211853, -1.512072205543518, 0.4441315233707428, -0.6216151714324951, -1.8846386671066284, -0.46647414565086365, -0.0013086282415315509, 1.026096224784851, 0.5920107960700989, -0.11522692441940308, -0.732424259185791, -0.44636425375938416, -0.45077794790267944, 0.15299652516841888, -0.5134096145629883, 0.010941603220999241, 1.344030499458313, 0.5233473181724548, 1.7832595109939575, 1.1198457479476929, 0.7662461996078491, -2.6850883960723877, -0.6892166137695312, 1.139609932899475, -0.3444947600364685, 0.3285849094390869, 0.2842831313610077, -1.4752812385559082, -1.3321478366851807, 0.4195978343486786, 1.212459921836853, -0.7791081070899963, -0.5115856528282166, 0.4860459864139557, -1.0971777439117432, 0.10404699295759201, 2.134122848510742, 1.4178134202957153, -3.3510217666625977, -2.238513946533203, 0.7615565061569214, 0.26107677817344666, 1.208912968635559, 0.6947779655456543, 0.5075352191925049, -0.16821271181106567, 0.38761311769485474, 0.2499629110097885, 0.3708127737045288, 1.568490982055664, -0.7428584098815918, 0.10733600705862045, -1.8345065116882324, -0.7190114855766296, -0.5244280695915222, 0.9564336538314819, -1.4566363096237183, -0.20698828995227814, 0.705296516418457, 0.9902551770210266, -1.90481698513031, 2.141140937805176, -0.5524179935455322, -1.269129991531372, 0.6723663806915283, -2.2017908096313477, -1.0757434368133545, -0.21564139425754547, 1.0591461658477783, 0.1949828714132309, -0.8562382459640503, 0.24177435040473938, 0.9950330853462219, -2.1963462829589844, 0.6593982577323914, 0.32289808988571167, 0.12440285086631775, -1.6329033374786377, -0.8021913170814514, -0.22822538018226624, 0.33637747168540955, 0.9586148858070374, 0.10717751085758209, 0.3277852535247803, -0.19599001109600067, -0.6671719551086426, 0.9822719693183899, 0.16665203869342804, 1.4951273202896118, -1.8644638061523438, 1.9569236040115356, 0.5349920392036438, 0.6308957934379578, -0.31441056728363037, 0.15090060234069824, 0.8712906241416931, -0.7332839369773865, -1.0202702283859253, 0.9888516068458557, -0.07999306917190552, -0.37955349683761597, -0.4075818955898285, 1.466888427734375, -0.005429156590253115, -1.4205565452575684, -1.3894624710083008, -0.5801188349723816, -0.6200494766235352, -1.9422751665115356, 0.5450059175491333, -0.44359999895095825, 0.768288254737854, 1.0136088132858276, -0.2819531261920929, -0.30991026759147644, -1.3609931468963623, 0.3589932918548584, -0.7420068383216858, -0.1862505078315735, 1.4548859596252441, -1.0045340061187744, -1.0344018936157227, 0.07231330871582031, 0.9669174551963806, -1.743749737739563, -0.32061704993247986, 1.4827213287353516, 1.6611804962158203, 0.4084950089454651, 0.030749382451176643, -0.8196470141410828, -1.320948600769043, -0.02181793749332428, 0.5559014081954956, 1.0827611684799194, 0.13248910009860992, 0.24892471730709076, 0.7737618684768677, 1.2623652219772339, 0.09006202965974808, 0.9730471968650818, -1.531013011932373, 0.9607013463973999, -0.6215067505836487, 0.003901774063706398, 1.9322998523712158, -0.910790205001831, 0.06658142060041428, 0.7247732877731323, -0.3878801763057709, 0.2757183313369751, -1.2515085935592651, -0.0917738527059555, 0.04025774449110031, -0.5576143860816956, 0.2810911238193512, 1.1855378150939941, 0.07539436966180801, 1.1697648763656616, 1.2423204183578491, 1.4675971269607544, 1.3446639776229858, -0.3186179995536804, -0.5444173812866211, 0.1452479213476181, 0.23697535693645477, 0.8330433368682861, 0.45604225993156433, -0.16731466352939606, -0.5439762473106384, -0.7720125317573547, -1.8179631233215332, -0.479591965675354, 2.0364973545074463, -1.86880624294281, -0.2938169538974762, 1.1057195663452148, 0.4965171813964844, 0.607957124710083, -0.8026358485221863, -0.6467604637145996, 0.25556299090385437, -1.5301275253295898, 0.04137643799185753, 0.3432471752166748, 0.41659873723983765, 0.30610617995262146, -0.4398147463798523, 0.0701044425368309, -1.5472328662872314, -0.18746471405029297, 0.22007934749126434, 0.7760290503501892, -0.9402772188186646, 0.5808560252189636, 0.11415468901395798, -0.02990640327334404, -0.057559024542570114, 0.014575461857020855, 0.9698554277420044, 0.22846302390098572, 1.6685583591461182, 0.2788070738315582, 0.08038495481014252, -0.2095794975757599, -0.8011518716812134, -0.27257052063941956, 1.2568840980529785, -1.2969419956207275, -0.4578147232532501, 1.8748420476913452, 0.0037764778826385736, -0.08098974078893661, -0.24237211048603058, 0.5905481576919556, -0.029085585847496986, 1.0898699760437012, 0.18084779381752014, -0.5033639073371887, -1.5723053216934204, 0.22557128965854645, -0.090311199426651, 0.22641350328922272, -0.6808509230613708, 0.42993420362472534, 0.3706982135772705, -0.6462881565093994, 2.061628818511963, -0.20197069644927979, -0.6866491436958313, -0.2026699334383011, 0.84417325258255, 0.8609524965286255, 1.132570743560791, 1.649972915649414, 1.6539089679718018, 0.1391690969467163, -1.7569890022277832, 1.101728081703186, -1.7464303970336914, -0.8478655815124512, 0.5036002397537231, 0.9453721642494202, -1.368210792541504, 0.7891677021980286, 0.5098249316215515, -1.0265651941299438, 0.08331384509801865, -2.335087299346924, -2.632880210876465, 0.7596439123153687, 1.2073999643325806, -0.2603098750114441, 0.6851904392242432, -0.46797558665275574, -1.1482090950012207, 0.9485465288162231, -0.18225379288196564, -0.6674939393997192, -1.1725473403930664, -1.6081171035766602, 0.6246733069419861, -1.3819271326065063, 0.6856626272201538, -0.3305208086967468, 0.7782246470451355, 0.26583555340766907, 1.5602304935455322, -0.5931352972984314, 0.7072243094444275, -0.32791200280189514, 0.28589245676994324, -2.4878134727478027, 0.8780799508094788, 0.8845179080963135, 0.7454788684844971, -0.10084190219640732, -2.147268295288086, -0.087427519261837, -0.08570892363786697, 0.6754632592201233, -0.5030652284622192, 0.1792544275522232, -0.5724123120307922, 1.4758535623550415, 0.28289687633514404, -1.1494654417037964, -1.5842561721801758, -0.4017404019832611, -1.5640400648117065, -1.1809786558151245, 1.3205382823944092, -0.3940684497356415, -0.31935498118400574, -0.6102554202079773, -0.578088641166687, -1.1418488025665283, 0.9285945296287537, -0.22870570421218872, -0.1350763738155365, -1.3665220737457275, -1.0023362636566162, -0.03896920382976532, -0.2761252522468567, 0.8240699172019958, 0.5602440237998962, 2.0306334495544434, 0.6190981268882751, -0.3495040833950043, 1.1528111696243286, 0.11196314543485641, -0.08879507333040237, 0.555513322353363, 0.09860767424106598, -0.5190058946609497, -0.667646586894989, 0.1497960388660431, -0.9729240536689758, 0.48527249693870544, -0.14641320705413818, -1.740641474723816, 1.5605515241622925, -0.8521865606307983, -0.4850424528121948, 0.8579098582267761, 0.10777325183153152, 0.03713514655828476, -1.1475567817687988, -0.7715832591056824, -1.3393083810806274, 0.7631500959396362, -1.3431944847106934, -1.3738009929656982, -0.5855959057807922, 0.8543155193328857, 0.2984064817428589, 1.2790358066558838, 0.302452951669693, 0.3934638202190399, -0.977041482925415, 1.666322946548462, 0.27647554874420166, -0.26696380972862244, 1.4333163499832153, 0.04528101533651352, -1.3116096258163452, 0.8594735264778137, -2.3453330993652344, -0.6507118940353394, 0.14545844495296478, 1.5651369094848633, 0.5695199370384216, 1.1415870189666748, 0.8993625640869141, -1.9379969835281372, -1.2505472898483276, -0.30355679988861084, -0.3153788447380066, -1.3741213083267212, 1.8807852268218994, -0.7516371011734009, -0.8400031328201294, 1.647207260131836, 0.6542271375656128, 1.0806711912155151, 0.3709026277065277, 2.406344413757324, 1.1538150310516357, 0.1979067325592041, 0.3702128827571869, 1.0294533967971802, 0.4030630588531494, -0.6978197693824768, -1.5734641551971436, -0.5414941906929016, -0.5258643627166748, 1.1452915668487549, 0.07778794318437576, 0.8274455666542053, -0.11003132909536362, 2.5476975440979004, -0.9748938679695129, -0.09733759611845016, 1.9315476417541504, 0.7783833742141724, -0.1209278553724289, -0.6402300000190735, 0.28945043683052063, -0.17099136114120483, 1.1992459297180176, -0.8982741832733154, 0.5789507627487183, -0.3086365759372711, -1.4695496559143066, 1.0339943170547485, 0.7346524596214294, 1.4014531373977661, -0.4482998251914978, -0.7268553972244263, 0.6950423717498779, -0.9558508396148682, -0.7591602206230164, -0.6446639895439148, -0.978656530380249, 0.7130623459815979, -0.9844098091125488, -0.7381824851036072, -0.8587790131568909, -0.7118150591850281, -0.20999860763549805, -0.9270766973495483, -0.10595713555812836, -0.17560864984989166, 2.413508176803589, -0.6254479289054871, 0.17926430702209473, 0.2615911662578583, 1.2338413000106812, -0.9956228733062744, -0.8707600235939026, -1.4173210859298706, 0.6405456066131592, 0.8112736344337463, -0.855836033821106, 0.02726997807621956, 2.1207687854766846, 1.2874915599822998, 1.276834487915039, 0.696308970451355, 0.1664009839296341, -1.0195672512054443, -0.4305378496646881, -0.22818759083747864, -0.7522161602973938, -0.13022714853286743, 0.8702020645141602, 0.6300692558288574, 0.6369259357452393, -1.4903490543365479, -0.38987642526626587, 0.19498343765735626, 1.76010000705719, -1.1358332633972168, 1.7043366432189941, -0.9549028873443604, 1.6765514612197876, -0.5830564498901367, -1.4094635248184204, -1.2415375709533691, -1.1005719900131226, 0.4123561382293701, -1.4639811515808105, 0.11187736690044403, 1.42289400100708, 0.18677349388599396, 0.1469789296388626, 0.279288649559021, -0.9284141659736633, -2.1162099838256836, 2.6594338417053223, -0.9831671118736267, -1.0886064767837524, 1.0325559377670288, -0.3141612410545349, 0.021641088649630547, 0.421979159116745, 2.0790798664093018, 0.762876033782959, 1.413642168045044, 1.1936277151107788, -0.602483332157135, 0.2047504335641861, -0.6895637512207031, -0.4609517753124237, 0.10939151048660278, -0.38663557171821594, 0.621577262878418, -0.0725390613079071, -0.07781019806861877, -0.761205792427063, -0.7355650067329407, 0.15415386855602264, -0.9852594137191772, 2.822467803955078, 1.0999771356582642, 1.4404661655426025, -0.0001532513415440917, -0.35358622670173645, 0.07427218556404114, 1.196885347366333, -1.264390468597412, -0.19779174029827118, 0.8555406332015991, -0.630699098110199, 0.006775303278118372, -0.4259985685348511, 1.41190505027771, 0.44839462637901306, 0.23354436457157135, 0.3546907901763916, 0.7880910634994507, -0.2861171364784241, -0.19841791689395905, 1.5006874799728394, 0.44631829857826233, 1.2078897953033447, 0.49846649169921875, 0.18772748112678528, 0.9891757369041443, -0.8359653949737549, -0.4074103832244873, -1.549713134765625, 0.2847616672515869, -1.7606492042541504, 0.48650118708610535, -0.6844894289970398, 0.18931323289871216, -0.32546690106391907, 0.07710355520248413, -1.4196010828018188, 0.3450824022293091, -0.8514171242713928, -0.5214859843254089, -0.30372491478919983, -0.6531032919883728, -0.10395129024982452, 2.61885666847229, -0.2432972937822342, -0.2943030595779419, 0.2739695608615875, -0.5796218514442444, 1.1395186185836792, -0.6499897837638855, 0.18799999356269836, -0.025107190012931824, 0.7334142327308655, 0.42659318447113037, 1.0172228813171387, -0.6298210620880127, -0.8114152550697327, -0.015953198075294495, 0.5282955765724182, -0.46322885155677795, -0.9438368678092957, -1.540785551071167, -0.10812359303236008, 0.5852805376052856, -0.681006908416748, 2.2491443157196045, -0.6685059070587158, -0.5484251976013184, 1.475454568862915, -0.21583016216754913, 0.11898232996463776, -0.6032017469406128, -0.29528430104255676, -0.1811148226261139, -1.619138240814209, -0.17483533918857574, -0.8948444128036499, 0.3124861717224121, 0.05764021351933479, -1.1519497632980347, -0.1584676206111908, -1.6966562271118164, 1.1189175844192505, -0.05519635230302811, -0.5732567310333252, -0.060709305107593536, 0.14183922111988068, 0.09826821088790894, 0.5684914588928223, 0.5305196046829224, -0.08093106746673584, 1.683502197265625, 0.5623013973236084, 0.26917850971221924, -1.506411075592041, -0.736677348613739, 0.9811515808105469, 1.8158724308013916, -0.2991860806941986, 0.17333421111106873, 2.1150619983673096, -0.33518898487091064, 1.9874954223632812, 0.41866928339004517, 0.42042118310928345, 0.43120402097702026, 0.9453441500663757, 0.9653691053390503, -1.1257848739624023, 0.3708682656288147, -0.5183498859405518, 0.36365365982055664, -0.24383147060871124, -0.7410888075828552, 0.5977732539176941, 0.16124571859836578, -0.6724645495414734, 0.39177367091178894, -0.577467679977417, 0.15654967725276947, -1.3223026990890503, -0.887064516544342, 0.8736004829406738, -1.3574237823486328, 1.3318028450012207, -1.1004687547683716, -0.6523413062095642, 1.2320529222488403, -0.382347971200943, 0.25271227955818176, 0.08514677733182907, -0.47652703523635864, -1.6492681503295898, 0.546174943447113, 0.729781985282898, -3.074019193649292, -0.34034767746925354, -1.4981110095977783, 0.35901474952697754, -0.5374348163604736, -0.000857989420183003, 0.18234148621559143, -2.5100743770599365, 0.8107967376708984, 0.6873785853385925, 1.8306355476379395, -0.4827151894569397, 0.7870184183120728, -0.31221601366996765, 0.3377666771411896, -0.036648139357566833, -1.248612642288208, -1.0808541774749756, 0.19731281697750092, 0.8950183987617493, -1.1505601406097412, 0.12884309887886047, 0.17200665175914764, -1.3637027740478516, 1.968000054359436, -2.1464521884918213, 0.5665577054023743, -0.5167129039764404, -1.6568248271942139, 0.8322852253913879, -0.5023935437202454, 0.7550095915794373, 0.7237430214881897, -1.2235723733901978, -0.5759780406951904, -0.2773011326789856, -1.1623421907424927, -0.8702049255371094, -0.9426082372665405, -0.825894296169281, -0.19401995837688446, 1.4625835418701172, -0.2125411480665207, -0.07123658806085587, -0.8620073795318604, 1.36274254322052, 1.1426523923873901, 0.5270491242408752, -0.48587554693222046, -2.059504985809326, -0.8544694781303406, 1.6913797855377197, 0.09424599260091782, -1.462694764137268, 1.862146019935608, 1.0479953289031982, 1.443245530128479, 1.2894214391708374, -2.8693456649780273, -0.7214205861091614, -0.9295084476470947, 0.6667351126670837, -1.4565156698226929, -1.281116247177124, -1.8652467727661133, 1.8632410764694214, 0.7578994631767273, 0.4713152348995209, -0.9748413562774658, 0.650579035282135, 0.8082528114318848, -1.1537563800811768, 0.011191993951797485, 0.715722382068634, -1.5251232385635376, 0.9936776161193848, 0.9653629660606384, 0.9924712777137756, 0.6357880234718323, 1.8750181198120117, -0.35033294558525085, -0.008932496421039104, -1.2748253345489502, -0.5837801098823547, 0.5233438014984131, 0.6943408846855164, -0.4289994537830353, -0.1508757621049881, 0.8199437856674194, 1.9078967571258545, -1.5862077474594116, -0.0271092988550663, 0.5773626565933228, 0.37701216340065, 0.2375742346048355, 1.4473800659179688, 1.0108230113983154, -0.6804209351539612, -1.7521092891693115, -0.7273627519607544, 1.0311410427093506, 0.10787127166986465, -0.48706963658332825, -0.1728009283542633, 0.09879941493272781, -2.2057745456695557, -0.6339113116264343, 0.43339768052101135, -0.6590301990509033, -0.02376396581530571, 1.8717286586761475, 1.2035562992095947, 0.04324463754892349, -0.23084858059883118, 1.0735414028167725, -0.34843143820762634, 0.2388804405927658, 0.8376267552375793, -1.6873915195465088, 0.5796041488647461, -0.5698724389076233, 0.7939482927322388, 1.280674934387207, 0.3747388422489166, 0.3518555760383606, -0.3947775363922119, 1.3214173316955566, 0.9310488700866699, 0.5256763100624084, 0.43715012073516846, -0.48300865292549133, 0.8723049163818359, 0.7932736277580261, 0.3095723092556, -0.2884230613708496, 1.830525517463684, -0.52547687292099, -2.194150686264038, -0.6944958567619324, 1.6035432815551758, 0.15939432382583618, 0.30813494324684143, 0.0038801380433142185, 0.1309640109539032, -0.47234097123146057, 0.7315711975097656, -0.7664784789085388, -0.6349295973777771, 0.02407284639775753, -1.5602456331253052, 0.49855443835258484, -0.226581409573555, -0.2964213490486145, -0.2967592775821686, -1.0614322423934937, 0.9800668954849243, 0.3680262267589569, -0.9977642297744751, 0.3896704614162445, 0.848544180393219, 0.3560049533843994, -0.4895002245903015, 0.618248701095581, -0.48110803961753845, -1.2354261875152588, -0.268248587846756, -1.605687141418457, -1.4685308933258057, -0.48649394512176514, 2.341313123703003, -0.29621899127960205, -1.6580666303634644, -0.6223458051681519, 0.18715745210647583, -0.2900172472000122, 1.3217390775680542, 0.2579621970653534, -0.541722297668457, 0.8801364302635193, -1.0452077388763428, 0.3886939287185669, 1.4638807773590088, -0.6647542715072632, 0.10282766819000244, 1.6700836420059204, 1.165418028831482, -1.2343981266021729, 1.7714924812316895, 0.9613588452339172, 0.16683422029018402, 0.2818814516067505, 1.769951581954956, 0.07454133778810501, -0.6078556776046753, -1.029799222946167, 0.9935587644577026, -1.0280766487121582, -0.7186111807823181, -1.093370795249939, -0.5157111287117004, 1.9561928510665894, 0.11432413756847382, 1.0339564085006714, -0.489215224981308, 0.758642315864563, -1.6780779361724854, -1.0167492628097534, -0.2374226152896881, -1.334604024887085, -0.3597344160079956, -0.4769364893436432, 0.11720369011163712, 1.6250219345092773, -0.5063294768333435, -0.0035573213826864958, -0.16154393553733826, 0.9578604102134705, 0.5981296300888062, -0.7581223249435425, -1.009513020515442, -1.4879769086837769, 0.510117769241333, -0.9917431473731995, 0.7089470028877258, -1.2543433904647827, 0.5941676497459412, 0.25667130947113037, 1.4094573259353638, 0.5473545789718628, -1.5877528190612793, -0.8734925985336304, 0.36003854870796204, 0.19802576303482056, 0.9158851504325867, -1.1553301811218262, 0.5190784931182861, 0.2711941599845886, -1.1183557510375977, -2.0716552734375, -0.9901991486549377, 0.7024955153465271, 0.12442954629659653, -1.5396384000778198, -1.1637628078460693, -0.08158101886510849, 2.2805943489074707, -1.8116742372512817, 0.48978191614151, 0.20489709079265594, -0.6166394948959351, 0.4590378403663635, 2.0734777450561523, 1.682798981666565, 2.0628018379211426, 2.6338083744049072, 0.3607339859008789, -1.3151538372039795, 1.1383939981460571, 0.4568382203578949, 0.47583329677581787, 1.7262927293777466, 0.3342474102973938, -1.3897157907485962, 0.8922443389892578, -0.6333926916122437, 0.08950435370206833, -0.9445406198501587, 0.3777901828289032, -1.9562917947769165, 0.9596319198608398, -1.6595805883407593, -0.019756941124796867, 0.20349645614624023, 0.18259702622890472, -1.2549772262573242, 1.1467983722686768, 1.702229619026184, 0.3998366594314575, -0.5019866824150085, 1.6448591947555542, -1.1132301092147827, 1.4045559167861938, 0.6312486529350281, -0.10294484347105026, 0.031077001243829727, 1.2001121044158936, -0.6135878562927246, -1.1044143438339233, -0.3151543438434601, -1.2597078084945679, 0.8671362400054932, 1.7905442714691162, 0.919803261756897, -0.7842848300933838, -0.47615858912467957, -1.1715636253356934, 1.9009909629821777, 1.612776756286621, 0.02141563408076763, 1.1066395044326782, 1.7279455661773682, 0.07855198532342911, -0.2657894194126129, -0.40976861119270325, -0.8030291199684143, 1.0296038389205933, 0.29864218831062317, 0.12130457162857056, -0.6637726426124573, -0.25657734274864197, -2.1946465969085693, 0.7354699969291687, 1.3436719179153442, 1.329045295715332, -1.261287808418274, 1.873726487159729, -0.7551880478858948, 0.5840333700180054, 0.5703698396682739, 0.19373057782649994, -1.2217991352081299, -0.4366069734096527, 1.2307385206222534, 0.06314180046319962, -0.2674263119697571, 2.193305492401123, 0.6537914276123047, 1.3088674545288086, 1.955350637435913, -0.0674506425857544, -0.13217566907405853, 0.5294350385665894, -0.747821033000946, 0.18622930347919464, -0.5357903838157654, -0.26550936698913574, -0.16786903142929077, -0.6601208448410034, -1.0961028337478638, -0.9036048054695129, -1.0040090084075928, -0.1689041703939438, -0.2526779770851135, 0.6126940846443176, -0.4835531413555145, 0.1550072580575943, 0.12169855833053589, 0.19501730799674988, -0.9145625233650208, 0.4674151837825775, -0.09170646220445633, -0.3530440330505371, -2.650282382965088, 0.6645752787590027, 0.9054750800132751, -0.41331276297569275, -0.4314717650413513, -1.2628939151763916, 1.681868553161621, -0.75953209400177, 0.9014873504638672, 0.6029831171035767, -1.1078741550445557, 2.086324691772461, 0.25763556361198425, -2.3086185455322266, 1.3497740030288696, 0.24230866134166718, 0.5852522850036621, -0.11634915322065353, 0.949583113193512, -0.9497443437576294, 0.9232626557350159, -0.09473038464784622, 1.6594160795211792, 0.17919085919857025, -0.23748810589313507, 0.9759768843650818, 0.022137904539704323, -0.159975066781044, 0.8751296401023865, -1.1991279125213623, -0.46062368154525757, 0.28400689363479614, 0.0027807271108031273, 0.905724048614502, 0.5111821293830872, 0.02518724650144577, 2.385857343673706, 1.369513750076294, 2.17803955078125, 1.3829175233840942, 0.43299469351768494, 1.8180102109909058, 0.7682225704193115, 1.547595739364624, -2.1984059810638428, -0.12684662640094757, 0.15465565025806427, -0.4694909155368805, -1.8165932893753052, -0.03943371772766113, 0.5558729767799377, -0.8222732543945312, -0.2921580970287323, -0.06600068509578705, -1.103947401046753, -2.8670578002929688, 0.26968085765838623, -1.194409728050232, -1.161318063735962, -0.7255670428276062, -1.8745906352996826, 0.5751243233680725, -0.42322683334350586, -1.086962103843689, -1.4619779586791992, -0.790874183177948, -0.05560895428061485, -1.9419366121292114, -0.4116487503051758, -3.252654790878296, -0.23384413123130798, -1.657588005065918, -0.9965639710426331, 1.028570294380188, 0.6799203157424927, -1.5054689645767212, -0.18996207416057587, 1.2738251686096191, -0.8815405964851379, -0.5555682182312012, -0.7942174077033997, -1.8573887348175049, -0.7292183041572571, 0.7003379464149475, 0.052117232233285904, -1.747595191001892, 0.7134060859680176, 0.36406683921813965, -1.0034574270248413, -0.07487355172634125, -0.7122170329093933, -0.427544504404068, -1.2434160709381104, -3.0060434341430664, 2.1127538681030273, -1.522735595703125, -0.9987659454345703, -0.35939016938209534, -0.40111595392227173, 0.629544198513031, 1.7619400024414062, 0.8876670002937317, -0.7758775949478149, 0.1095321774482727, -0.4854190945625305, 1.2070358991622925, -0.46076804399490356, 0.11303570121526718, -0.27251678705215454, -1.117815375328064, 0.3962402641773224, 0.9438439607620239, -0.5637893080711365, -1.821653127670288, 2.2797834873199463, 0.33111104369163513, -2.528576374053955, 1.4789119958877563, -2.7114102840423584, -2.3484086990356445, -2.288578748703003, 1.216795563697815, 0.1398683339357376, 0.5426726937294006, 1.3272359371185303, -1.1461453437805176, -0.4460545480251312, 0.43976426124572754, 0.22189483046531677, -2.653893232345581, -1.068976879119873, 2.4694626331329346, 1.4756017923355103, 0.38357412815093994, 0.40429022908210754, -0.8537295460700989, -0.3412487208843231, -1.5946993827819824, 1.7517460584640503, -0.7656704783439636, 0.14551550149917603, -2.096266031265259, -0.7901080846786499, 0.15393026173114777, 0.2810157835483551, 0.9153692126274109, 3.1140215396881104, 0.5986034870147705, 1.3158289194107056, -1.9878042936325073, -0.8180923461914062, 0.5917484760284424, 2.041466474533081, -0.08041395246982574, 0.8077536225318909, 0.27014198899269104, 0.1635221391916275, -0.39104363322257996, -0.6847241520881653, 0.3099319636821747, 0.42044904828071594, 1.0280802249908447, 1.1190261840820312, 1.606298565864563, -0.3696626126766205, 0.034032758325338364, -0.40357592701911926, -1.6510863304138184, 1.3980566263198853, -0.00222015124745667, -0.8759137988090515, -0.29583632946014404, 1.0646439790725708, 1.3574628829956055, -0.40021559596061707, 0.7802312970161438, 0.10783318430185318, -0.24751417338848114, 0.8321358561515808, -0.9651877880096436, 0.7151062488555908, -1.1118775606155396, -1.075943112373352, 0.18673531711101532, 0.8591738343238831, -1.9273840188980103, -0.5959126353263855, -3.8574750423431396, 0.3543570935726166, -1.3097491264343262, 0.48645028471946716, 0.09555917233228683, -0.9360941648483276, -0.035055290907621384, 1.459994912147522, -0.8902473449707031, -0.46371763944625854, 1.7074592113494873, 1.3776012659072876, 1.5040695667266846, 1.9485905170440674, -1.3437429666519165, -2.3847274780273438, -1.0153435468673706, -0.9816330671310425, -0.9443567395210266, -0.3610464334487915, -0.039856381714344025, 0.44689664244651794, -0.22494906187057495, -0.2658761739730835, 1.8514745235443115, 0.4944929778575897, 0.06160514056682587, 0.09477143734693527, -0.4378865659236908, 0.2612542510032654, 1.75164794921875, 0.8825802206993103, 1.0755395889282227, -0.5138441920280457, 0.005476485472172499, 0.9545032382011414, -1.517217993736267, 1.319281816482544, 0.20493614673614502, 0.6190868020057678, 0.8331021070480347, 0.8875817060470581, 0.12747357785701752, 1.7015578746795654, 0.6041878461837769, -0.6639888882637024, -0.03679278865456581, 0.49253571033477783, -0.6790025234222412, -0.20303595066070557, 0.7476228475570679, -0.8041213154792786, -0.48553264141082764, 0.22881971299648285, 0.0749606117606163, 0.4909099340438843, 0.2733963429927826, 0.7381057739257812, -0.5787549018859863, 0.2898459732532501, -0.22266161441802979, 0.2539019286632538, 0.4850601255893707, 0.40953952074050903, -1.3698654174804688, 0.4546915292739868, 0.35228297114372253, -0.6367299556732178, -0.6734251379966736, -1.258434534072876, -0.16741006076335907, -0.27554237842559814, -3.6545462608337402, 2.515709638595581, 0.33352118730545044, -1.5254302024841309, -0.7219784259796143, -1.1590481996536255, -0.8625839352607727, -1.9138634204864502, 0.22894316911697388, -0.13341496884822845, -0.549607515335083, -0.9773610830307007, -0.049193739891052246, 1.0072803497314453, 0.5432268381118774, -0.3872198164463043, 0.8966290354728699, 0.15853559970855713, -0.1413322389125824, 0.6723687052726746, 1.8725498914718628, 0.0071378666907548904, -1.221899390220642, 0.8517231941223145, -0.7114461660385132, 0.30143073201179504, -0.7318007349967957, -0.9083438515663147, -1.9478662014007568, 1.4473674297332764, -0.9774869084358215, 0.49723103642463684, -1.4405542612075806, 0.2033981829881668, -0.2001129537820816, 0.061906371265649796, -1.6430883407592773, -0.4294620156288147, 0.8124767541885376, 0.44718343019485474, -0.8630126118659973, -1.1648327112197876, 0.8310413956642151, -0.844004213809967, -0.1563718616962433, 0.7266163229942322, -1.7175636291503906, -0.8373665809631348, -0.5381453037261963, 0.4741559624671936, 0.3687233328819275, 0.440886914730072, 1.687512755393982, 0.4901844561100006, -0.29324889183044434, 0.676334023475647, -1.2000658512115479, 1.2080780267715454, 1.5906466245651245, 1.103749394416809, 0.20892909169197083, 1.853479266166687, -0.2356642186641693, 0.5860173106193542, 1.0763672590255737, -1.5050606727600098, 1.9203132390975952, 0.2993369698524475, -2.6140575408935547, 0.2742732763290405, 0.9139111042022705, -0.6921197175979614, 0.8600343465805054, 0.770571231842041, -0.07964514195919037, -0.009117399342358112, -0.21672244369983673, 0.2039269059896469, 0.21509656310081482, 1.928757905960083, 0.8960188031196594, -0.4332810342311859, 0.6065241098403931, -0.43917688727378845, -1.3835952281951904, 1.0801265239715576, 3.796820878982544, 0.11673828214406967, 0.016254661604762077, -1.3226661682128906, 1.5362863540649414, -0.807128369808197, -1.5497907400131226, -0.008996234275400639, -0.1268063485622406, 0.18184195458889008, 0.8213251829147339, 0.8419546484947205, 0.521932065486908, 1.3976430892944336, 0.506814181804657, 2.798494338989258, 1.6074239015579224, 0.08176767081022263, 0.8552709221839905, -0.3813059329986572, 1.0724550485610962, 0.0913175642490387, 1.03683602809906, 0.48354631662368774, -1.6324031352996826, 0.40361639857292175, 0.9293903112411499, -0.7643437385559082, 0.11560814827680588, 0.7680809497833252, 1.38148033618927, 1.0941261053085327, -1.3927536010742188, 0.4181348979473114, 0.5004351735115051, -0.4789862632751465, -0.7274830937385559, -0.755715548992157, -1.2975975275039673, 0.6936694979667664, 0.2666788399219513, -2.488154888153076, -0.690037727355957, -0.9829963445663452, 0.31584686040878296, 0.4980340301990509, 1.736366629600525, -2.484036922454834, 0.19965322315692902, 0.07079186290502548, -1.2676066160202026, 0.032180383801460266, 1.0189512968063354, -0.012985379435122013, -0.9053418040275574, 0.5122969746589661, -2.75105881690979, 2.5685648918151855, 0.11142181605100632, 0.18033362925052643, -0.6554251909255981, -0.3043835461139679, 0.16442397236824036, 0.5884514451026917, 0.9177535176277161, -1.0626740455627441, -1.0106072425842285, -0.05420020967721939, -0.1456839144229889, 0.8189252614974976, 0.7192477583885193, 0.4583507478237152, 0.8725709915161133, -0.7183048129081726, 0.011466155759990215, 1.319154143333435, 0.9176310896873474, -1.7193095684051514, 1.8810442686080933, 1.2901904582977295, -0.22100424766540527, 0.43422773480415344, -0.12150508910417557, -0.8933749794960022, 1.0414706468582153, 0.4255639910697937, -0.11931530386209488, 1.284326195716858, 1.9194657802581787, 1.9240299463272095, -0.5821610689163208, -0.5735845565795898, -1.138250708580017, 1.0382858514785767, -0.018571367487311363, 0.2787952125072479, -1.53487229347229, -0.11060065776109695, -1.6190189123153687, 0.3660145401954651, 1.1844512224197388, 0.29898008704185486, 1.354079008102417, 1.515094518661499, 1.0998516082763672, 1.668876051902771, -0.6199204325675964, 0.5194169282913208, 0.9816822409629822, 1.821374535560608, -0.5580549836158752, -1.2433921098709106, 1.0062469244003296, -0.17590788006782532, 1.2386826276779175, -0.16160878539085388, -1.4449304342269897, -1.2005207538604736, 0.2311638593673706, -0.22022075951099396, -0.4535086452960968, 0.23633795976638794, 1.0212873220443726, -0.4555729627609253, 0.6402584910392761, -0.5565422177314758, 0.024769457057118416, 0.018674762919545174, 0.900802731513977, -1.0024213790893555, 0.45140907168388367, -1.8323345184326172, -0.3717518448829651, 1.2191131114959717, -0.8446788787841797, 1.3161611557006836, -0.5804839730262756, -0.5817946791648865, -0.6586436629295349, -0.4531119167804718, -1.469246745109558, -0.33812856674194336, 0.36491715908050537, 0.6504676342010498, 0.13111552596092224, 0.03601080924272537, 0.575483500957489, -0.7042354941368103, 0.6358435750007629, 0.43225616216659546, 0.2642872631549835, -2.364637613296509, 0.9993757605552673, -1.3279476165771484, -0.1888958215713501, -0.8752930760383606, -0.3792073428630829, 1.5876306295394897, 0.2082686722278595, 0.9813789129257202, 0.2659253180027008, 0.3632100820541382, 0.1592407077550888, -0.2896348237991333, -1.3818978071212769, 0.9739195108413696, -1.3396565914154053, -0.3780440390110016, 1.5517433881759644, 0.9149473905563354, -1.631124496459961, -0.47372761368751526, -0.5620238780975342, -0.5665773153305054, 0.39765676856040955, -1.2093056440353394, 1.2320032119750977, -0.5250936150550842, -0.6915601491928101, 1.320834755897522, 0.6613614559173584, -0.8504055738449097, -0.8817293047904968, 0.2969915568828583, 0.7669078707695007, 0.4991958439350128, 0.445565789937973, -1.6503796577453613, 0.821913480758667, -0.3233861029148102, -1.3771778345108032, 1.6349114179611206, -1.254724383354187, -0.7517785429954529, 1.7668299674987793, 1.1027088165283203, 0.3207113444805145, -0.68693608045578, 0.7360633611679077, 1.5681428909301758, 0.17938531935214996, 0.013994873501360416, 0.40220800042152405, 0.9091202616691589, -0.9690120816230774, 0.08040125668048859, 0.9652500152587891, 0.5397067666053772, 0.25761500000953674, -2.04086971282959, 1.2912486791610718, -1.4124035835266113, 0.11405470222234726, -0.2548860013484955, -1.5002435445785522, 1.7277201414108276, -0.64887535572052, -0.9626415967941284, 0.5194193124771118, 0.06456489115953445, -1.9009265899658203, -0.9386349320411682, -0.26878300309181213, 1.3374170064926147, -0.4645839333534241, -1.2613271474838257, -0.3223591148853302, -0.2124108523130417, 0.3628983795642853, -0.45476552844047546, -0.7556559443473816, -0.05616132915019989, 0.7802391052246094, 2.711613416671753, 1.06065034866333, 0.5313048958778381, -0.7919489741325378, -1.098785638809204, -0.4820999801158905, -0.42825019359588623, -0.8811063766479492, 0.8544754981994629, 1.3089535236358643, 0.20628388226032257, 2.6621651649475098, -0.6642143130302429, -0.6838818192481995, -0.3005674183368683, -0.3151037096977234, 0.18341319262981415, 0.44208353757858276, -0.4794662892818451, 0.6279144287109375, -0.2587694525718689, -0.12009149044752121, 1.0771750211715698, -0.6613897681236267, -0.7985045909881592, 0.5544683337211609, -2.8228111267089844, 0.43378040194511414, 0.12721796333789825, -0.039244309067726135, 0.5667465925216675, -0.8839322328567505, 0.8066021203994751, -1.8522027730941772, -1.7591139078140259, 0.6678513884544373, 0.30500784516334534, 1.3966585397720337, 0.85075443983078, -1.0459822416305542, 0.9634932279586792, -1.0965023040771484, -0.8856372833251953, -0.6925848722457886, -0.6067629456520081, -0.6164950132369995, -0.6897948980331421, -1.0924047231674194, 0.4666960835456848, 0.3870429992675781, -1.1964908838272095, -0.012025111354887486, 0.4563075602054596, -0.07446810603141785, -0.7622795701026917, 0.3804009258747101, 0.2856561839580536, -0.16465531289577484, 1.1540751457214355, 0.7398689985275269, -1.0901237726211548, -1.8671112060546875, 0.13583904504776, 2.1776247024536133, -1.2527925968170166, 2.0684800148010254, -1.0614893436431885, 1.7449231147766113, 1.3044263124465942, -1.388048529624939, 1.2337708473205566, -0.08257275074720383, 1.1383461952209473, -0.523104727268219, 1.4715328216552734, 0.21134492754936218, 0.9369960427284241, -0.9233278632164001, -1.8889809846878052, -2.0882811546325684, 1.2273883819580078, 0.2262992113828659, 0.09882348775863647, 0.9109597206115723, -1.4469048976898193, 0.031944308429956436, -1.434240460395813, 0.849455714225769, -1.1659350395202637, -0.7486777901649475, -0.10191285610198975, 0.23490428924560547, 0.6051518321037292, -0.3303319215774536, 1.4674923419952393, 0.32457253336906433, 1.4172277450561523, 0.2583177387714386, 0.9907759428024292, 0.6695736646652222, -1.144256591796875, 0.5761488676071167, -0.4095301330089569, -0.01993671990931034, -0.6437147855758667, 0.49722355604171753, -0.6055914163589478, 1.0524024963378906, -1.6569563150405884, 0.6265363693237305, -0.32686832547187805, -1.5126439332962036, -0.3508109152317047, 0.30325502157211304, 0.2754877209663391, -0.6812847852706909, 0.08709721267223358, 0.4988957941532135, -0.36065471172332764, 0.8419995307922363, 0.09105827659368515, -0.8401286005973816, -0.36663106083869934, -0.3776993155479431, -0.3713569641113281, 0.3995964825153351, -0.22397364675998688, -0.0998198390007019, 0.05371079593896866, -0.2127063274383545, 1.3572090864181519, 0.21942771971225739, -0.5716075897216797, -0.13409534096717834, 0.6113347411155701, 1.709750771522522, 0.10010147839784622, 0.8946349024772644, -0.5799488425254822, -0.14264099299907684, -0.6185664534568787, 0.0432833768427372, -0.5057140588760376, 0.6181024312973022, -0.17042264342308044, 0.6765639781951904, 1.179426908493042, -0.25493574142456055, 0.5502335429191589, -0.5091506838798523, 0.8095390796661377, 1.8910491466522217, -0.05512117221951485, -0.12501834332942963, -1.771895408630371, 1.526047706604004, 0.23318050801753998, -0.9836307764053345, 1.9248456954956055, 0.15183372795581818, -1.100383996963501, 0.8694397807121277, -0.23236562311649323, -1.4673582315444946, 0.5408920049667358, 0.42256495356559753, 0.09057886898517609, -3.06138014793396, 0.695395827293396, 0.7391389608383179, -1.590049386024475, -1.6062685251235962, -0.7151491045951843, 2.078721523284912, 0.900295615196228, 0.1546529084444046, 0.952305793762207, -0.5948575735092163, 0.11371580511331558, -0.6514188647270203, -1.2197012901306152, -0.2615540027618408, -0.7692530155181885, 1.7431094646453857, 0.9781637191772461, 0.2453364133834839, 0.6266660690307617, 1.3342211246490479, -1.3742440938949585, -1.4324957132339478, -1.4519175291061401, -1.3290743827819824, 1.7533091306686401, 1.360406756401062, -1.4922542572021484, 0.31650662422180176, 0.11858565360307693, 0.2907487750053406, -0.5882931351661682, -0.12423732876777649, 0.7807790637016296, 1.544990062713623, 0.14020082354545593, -0.5580037832260132, -0.1899893432855606, -0.19017621874809265, -1.6894985437393188, 0.83122318983078, 1.4520667791366577, 0.8300952315330505, 2.384901523590088, -1.1663118600845337, 0.9794195294380188, 0.49605345726013184, -0.47709494829177856, -0.592108964920044, -0.8394455313682556, 0.15700267255306244, 1.3077642917633057, -1.2490159273147583, -0.8059121966362, -1.007856011390686, 1.7524030208587646, 0.4825980067253113, 0.16642077267169952, -0.5624642968177795, 1.1750380992889404, -0.32664597034454346, -0.49900224804878235, -0.556309700012207, -2.2942051887512207, 0.08368219435214996, 0.15894430875778198, -0.1067994236946106, 2.259484052658081, -0.1692550629377365, -0.8865702152252197, -1.7329803705215454, 1.7061406373977661, 0.012902108952403069, -0.09090784192085266, 2.1945607662200928, -0.35428357124328613, -1.4510753154754639, 0.6326079368591309, 0.716734766960144, 3.586900472640991, 1.358559250831604, -1.4181796312332153, 0.26431870460510254, -0.41917484998703003, -0.2617586851119995, -0.7295473217964172, -0.47773948311805725, 0.05537642538547516, 0.09359913319349289, -0.8195530772209167, -1.0200366973876953, 1.5514335632324219, 0.6785444617271423, 0.21823111176490784, -0.40417343378067017, 1.6501438617706299, -0.06293042004108429, -1.261782169342041, -1.2218350172042847, 0.9994361996650696, 1.0796107053756714, 0.6865481734275818, 0.16757746040821075, -0.3178620934486389, 0.9605806469917297, 0.3001989424228668, 0.2297244518995285, -1.4908641576766968, 1.1831567287445068, -0.3184152841567993, -0.9322901964187622, 1.5261656045913696, 0.644769549369812, 1.315106749534607, -0.021536359563469887, -0.11091414093971252, 1.1460509300231934, -0.1634155660867691, 1.7466915845870972, 0.13123302161693573, 0.7816881537437439, -0.5923292636871338, -1.148658275604248, 0.4501737952232361, -0.8389292359352112, 0.3320002555847168, 0.29134601354599, 0.8237369060516357, 0.5354218482971191, 0.9130203127861023, 0.5147267580032349, -0.7773429155349731, -1.483879566192627, 0.7625328898429871, 0.4304158687591553, 1.3045073747634888, 0.484185129404068, -0.019536424428224564, 1.6070712804794312, -1.562340259552002, 0.935399055480957, 0.49676772952079773, 0.7820714116096497, 0.46534159779548645, -0.11962608247995377, -0.39274862408638, -0.8258304595947266, 1.1531964540481567, -0.26075980067253113, -0.2208845466375351, -0.4910544753074646, -0.9106495380401611, -0.29180142283439636, -0.37037020921707153, 0.45681998133659363, -0.4486643970012665, -0.3517549932003021, 1.5893915891647339, 0.2969774901866913, -1.469510793685913, 0.1687530130147934, -0.09705460071563721, -0.37396594882011414, 0.06045781821012497, 0.30221566557884216, -0.7676582336425781, -0.870527446269989, 0.7824040055274963, -0.6100118160247803, 0.4554828703403473, -0.9779943227767944, 2.2119767665863037, -0.09749426692724228, 0.729146420955658, -0.8641089200973511, -0.23025071620941162, -1.585071086883545, 0.3075849115848541, -0.32921120524406433, 0.39771464467048645, -1.5487565994262695, -0.9262827038764954, -2.178575038909912, 1.8376853466033936, 0.7531959414482117, 0.6487530469894409, 0.12974169850349426, 0.9953756928443909, -1.0881613492965698, -1.8076542615890503, -1.0420221090316772, 0.6088792681694031, 1.5800926685333252, -1.21993887424469, 0.24321423470973969, -0.9201277494430542, 1.7588088512420654, 0.21937979757785797, -1.8862937688827515, 0.39569753408432007, 1.0781570672988892, 0.5217652916908264, 1.3728326559066772, 1.3076577186584473, -2.8587565422058105, -1.144252896308899, 0.03556263446807861, 0.14563113451004028, -0.040887683629989624, 0.00032716355053707957, -1.2435033321380615, 2.018665075302124, -0.011134721338748932, 1.5146652460098267, -1.5000700950622559, -1.3826773166656494, 0.6603894233703613, -1.75791335105896, -1.3338770866394043, 0.8541572690010071, 0.9625229239463806, -1.1662973165512085, -1.1053050756454468, -2.7886734008789062, 0.15251614153385162, -1.9015491008758545, -0.28505584597587585, 0.4336332380771637, 2.0930166244506836, -0.08598636090755463, 0.36207661032676697, 0.6632401347160339, 0.4502790570259094, -0.6621469855308533, 0.4308791756629944, 0.6763052344322205, -0.6130870580673218, 0.2773243486881256, -0.8255214691162109, -0.34804126620292664, 1.1151220798492432, 0.9229921698570251, 0.8352953791618347, 0.7766015529632568, 0.336250901222229, -0.3030015230178833, -0.9696549773216248, 1.61563241481781, -0.10454796254634857, -1.0341651439666748, -0.050016939640045166, -0.47774970531463623, 0.6483089327812195, -0.5911011695861816, -0.460955411195755, 0.8771494030952454, -2.2824411392211914, 2.1769349575042725, 0.2692455053329468, 0.9613281488418579, 1.475191354751587, -0.5625773072242737, -0.1994219273328781, 0.36829474568367004, 1.4305195808410645, -0.5927569270133972, -2.603208065032959, -1.1960901021957397, -0.06438961625099182, 0.48601284623146057, 1.025631308555603, 0.370383083820343, -0.8168617486953735, -0.19584637880325317, -1.513879656791687, -0.20105060935020447, -2.235053777694702, 0.276352196931839, 1.1751445531845093, 1.1765016317367554, 0.8479831218719482, 0.05169728770852089, 0.4186970591545105, -0.7655565738677979, -1.3380383253097534, 0.32251983880996704, -0.6911800503730774, 0.2830296754837036, -0.17222584784030914, -1.2354710102081299, 1.614966630935669, -0.5125491619110107, -0.3417991101741791, 0.8535360097885132, -0.7631129026412964, -0.7783114910125732, -0.6636502146720886, 0.6021983027458191, 0.6021640300750732, -0.17777293920516968, -0.5930241942405701, -1.1355127096176147, -2.5412023067474365, -1.688240647315979, 0.5750420093536377, 1.2178393602371216, 1.7997239828109741, 1.1318397521972656, 0.2483174055814743, -1.8559287786483765, -1.9656449556350708, 1.6093111038208008, -1.3053605556488037, -0.19844010472297668, -1.355782389640808, -0.446781724691391, -0.6595131158828735, 1.4488691091537476, -0.21914970874786377, 1.2209126949310303, -1.3602396249771118, 1.725684404373169, -0.20877650380134583, -0.06368649005889893, -0.7510045766830444, 0.7695213556289673, -0.9384083151817322, -1.6687464714050293, -0.2413284182548523, 0.2780236005783081, -1.3967574834823608, -0.6902605295181274, -0.816139817237854, 0.9004275798797607, -0.7500386238098145, 0.1443321853876114, -0.0200401209294796, -0.46638238430023193, 0.9754016399383545, -1.121645450592041, 0.3386734127998352, -1.142038345336914, 0.332556813955307, -0.49032098054885864, 0.032454393804073334, 0.3932347595691681, 2.1793251037597656, 0.5391358733177185, 0.4471548795700073, -0.6361503005027771, 0.16629569232463837, 2.906602144241333, -0.2811906039714813, 1.1575970649719238, 0.04941398277878761, 0.45491257309913635, 1.1815191507339478, -1.0001176595687866, 0.19333745539188385, -0.4125615656375885, -1.5568135976791382, 1.431288242340088, 0.35660624504089355, -0.03380078822374344, 0.9497461915016174, 0.542463481426239, -0.1886342465877533, 0.34841659665107727, -0.5128832459449768, 2.1728515625, 1.0644313097000122, 0.7692092657089233, -0.16268478333950043, 1.9240367412567139, -0.38982105255126953, 1.1019784212112427, 0.18500345945358276, -1.626930594444275, 1.0288506746292114, -0.9627462029457092, -0.03580719232559204, 1.1727217435836792, 1.8940250873565674, -0.65386563539505, -0.5026904940605164, -0.06999796628952026, -0.25897401571273804, -0.22521837055683136, 0.8519037961959839, -0.5800396203994751, 1.2310802936553955, -0.3429974913597107, -1.2591582536697388, -0.4902098476886749, 1.6782324314117432, -0.024411167949438095, 0.31930676102638245, -0.19087064266204834, 1.6785635948181152, 1.4169988632202148, 0.09416619688272476, -0.10706031322479248, -0.09354588389396667, 0.9663614630699158, -0.7909477949142456, 0.48734399676322937, -0.6598824262619019, 0.21995341777801514, -0.4611242413520813, -1.7050094604492188, 0.5489488840103149, -0.321416974067688, 0.6539729237556458, 1.2666455507278442, -1.7551422119140625, 0.5471488833427429, 1.4768208265304565, 0.9704447984695435, -1.8170461654663086, 0.5863563418388367, -0.535426139831543, 1.4961254596710205, 0.026020513847470284, 0.07327645272016525, -1.7750898599624634, -0.06906832754611969, 0.2549053728580475, -0.2807571589946747, -1.529233455657959, 0.01819073222577572, 1.2037155628204346, -1.0296434164047241, 2.4417567253112793, -0.6186270117759705, 0.17820072174072266, -0.9383378624916077, 1.2722867727279663, 0.059567030519247055, -0.5443477630615234, -0.5863053202629089, 0.495212584733963, -2.055161714553833, -0.2186756730079651, -0.26954030990600586, 1.9940900802612305, 0.28941869735717773, 1.6139410734176636, -0.5997124910354614, 0.7462801933288574, -1.7394055128097534, 0.44376444816589355, 1.018425464630127, 0.04783378541469574, -1.0428879261016846, 0.15687383711338043, 0.3457900285720825, -0.7331272959709167, -0.47159910202026367, 0.6464468836784363, 1.2530016899108887, -0.1537172496318817, -0.9068878293037415, -0.11674736440181732, 0.5062605738639832, -0.5029760599136353, 1.8117924928665161, 0.378924697637558, 0.9660325646400452, 0.23213647305965424, 1.3916443586349487, -1.42147696018219, 1.241515040397644, 0.012077203020453453, -0.14820614457130432, -1.647994875907898, 0.9585365653038025, 0.8576800227165222, -0.24576574563980103, 0.7533894181251526, 1.118802785873413, 1.4923616647720337, 2.407332181930542, 0.9745720028877258, -0.6398646831512451, -0.7552668452262878, -0.24150514602661133, 1.296875238418579, 0.2905504107475281, -0.4652928411960602, 0.1756947636604309, -0.814483642578125, 0.9752840399742126, -1.319905400276184, -0.39353451132774353, 1.5304152965545654, -0.7437805533409119, 0.6409313678741455, 0.5256208777427673, 0.10112441331148148, -1.3411340713500977, -0.06983129680156708, -0.6543474793434143, -1.5140607357025146, 0.3152705729007721, 0.7585822939872742, 0.618744432926178, -0.19567926228046417, -0.9170240163803101, -1.8086771965026855, -1.4109691381454468, -1.609153151512146, -1.0255398750305176, 1.7052425146102905, -2.2574563026428223, -0.1579308658838272, -1.0192759037017822, 0.05038907006382942, 0.4270419478416443, -0.2129439264535904, -1.1657805442810059, 0.12929557263851166, 0.776840090751648, 0.9583597779273987, 0.9100554585456848, -0.953900158405304, -1.1257234811782837, 0.6702257990837097, 0.5285467505455017, 0.4673975706100464, 0.40389686822891235, -0.09321129322052002, 0.3043211102485657, -0.6281401515007019, -0.46285054087638855, -0.6653247475624084, -0.2340727597475052, -1.250023603439331, 1.8654974699020386, 0.5202654600143433, -0.5415000915527344, -0.28649821877479553, -1.560843825340271, 2.0233776569366455, -0.841972291469574, 0.4057493209838867, -0.5443469882011414, -2.099609375, 0.06349514424800873, -1.132920503616333, 1.6035571098327637, -2.398923635482788, 0.8212961554527283, -1.7231433391571045, 0.32944467663764954, 0.593132734298706, 1.5580766201019287, 1.2498393058776855, 0.9551858305931091, -0.5370020270347595, 0.6233882904052734, -0.6572965979576111, 1.0364484786987305, -0.3262711763381958, 0.8946450352668762, -1.6274471282958984, -1.211735725402832, -1.3257381916046143, 0.7053894400596619, 0.07342339307069778, -0.494037389755249, 0.5341411828994751, -0.8451423645019531, -0.5838323831558228, 0.33424168825149536, 1.7316526174545288, 0.3829604685306549, 0.905685305595398, 0.29707255959510803, 2.0626440048217773, -0.4581480026245117, -0.8567792773246765, -0.6738166809082031, 1.3141285181045532, 0.9557241201400757, 0.18937107920646667, 0.5005236268043518, 0.5540599226951599, 0.5664102435112, 0.7195200324058533, 0.9846896529197693, 0.2080157995223999, -2.0417892932891846, -0.5224267840385437, -1.4400887489318848, 0.9350782632827759, -0.16793932020664215, -0.2341407686471939, 1.1791974306106567, 1.3609025478363037, 1.208500862121582, -2.49070405960083, -0.6015688180923462, -0.6750344038009644, -1.9289146661758423, 0.13861358165740967, -0.11480378359556198, 0.46714460849761963, 1.0060608386993408, -1.1728883981704712, -0.5506126880645752, -0.49073317646980286, 0.6973943114280701, 1.0788054466247559, -1.2813682556152344, -1.0947551727294922, 0.739417314529419, 0.10864599794149399, -0.04324715584516525, -0.34084320068359375, -0.6681024432182312, -0.7722548246383667, 0.1263907104730606, -0.2293650358915329], "dtype": "float32"}], "e": {"__nd__": [1.238553524017334, -0.3025500774383545, -0.5118452310562134, -1.8056952953338623, -1.404895544052124, -1.8234155178070068, -2.43998384475708, 1.3958964347839355, 0.30260926485061646, -0.23433831334114075, 2.544412612915039, 0.7102080583572388, -2.528611183166504, 0.5899563431739807, -0.6657718420028687, -0.05848529934883118, -0.016032207757234573, -2.1416754722595215, -1.6542985439300537, -0.15663498640060425, -3.051725149154663, -2.270632266998291, 0.1822093278169632, -0.5324172973632812, 2.2618141174316406, 2.1137614250183105, -1.3021183013916016, -1.3898050785064697, 0.06380009651184082, -1.0921103954315186, -0.3042924106121063, -1.4549169540405273, -2.390949010848999, 0.20561851561069489, -0.6279367208480835, 0.2064027041196823, -0.7456908226013184, 0.15867410600185394, -0.29880857467651367, 0.14125636219978333, -0.933772087097168, 0.7904453873634338, 1.4174349308013916, -0.1310565173625946, -0.8888583183288574, -1.1616497039794922, 1.0606067180633545, -3.9631307125091553, 2.166797399520874, 0.6881957054138184, 0.3378600478172302, -2.184368848800659, -1.3810906410217285, 1.94742751121521, 2.275690793991089, -0.281594455242157, 2.7783827781677246, 0.5547153949737549, 2.43066668510437, -1.3739662170410156, 0.06881719827651978, 0.16835856437683105, 1.6101387739181519, -1.8014752864837646, 0.3217962384223938, 1.7616764307022095, 0.42123252153396606, 0.5025596618652344, -0.4828301668167114, -0.4035329818725586, 0.15180552005767822, 0.8865411877632141, -2.6591265201568604, -2.184788465499878, 1.0308268070220947, 1.9374490976333618, -0.524390459060669, 1.6125587224960327, 0.8190597295761108, -0.11945047974586487, 0.1580355316400528, -1.108623743057251, 0.5227960348129272, -1.110615611076355, 1.437333583831787, -0.009474992752075195, -0.852789044380188, 2.4394943714141846, 1.515364170074463, -0.2458086609840393, 1.1977026462554932, -0.0073015689849853516, -0.9165450930595398, 0.5138742923736572, 1.8932862281799316, -0.6536343097686768, 0.5007022619247437, -0.6839584112167358, -2.0960164070129395, -1.6622456312179565, -1.834965705871582, 1.7202776670455933, 1.0167063474655151, 1.6827361583709717, 0.634128987789154, -1.72500479221344, 0.6325918436050415, -2.3847103118896484, 0.0767403393983841, 0.28411656618118286, -1.214784860610962, -0.10504919290542603, -0.8163651823997498, -0.05911821126937866, -0.3512863218784332, 4.688282012939453, -0.7429489493370056, 1.0039918422698975, -0.17890799045562744, 0.5579029321670532, -1.042158842086792, 2.061587333679199, 2.134000062942505, 1.1768814325332642, -0.12966465950012207, 1.6459431648254395, 3.7216179370880127, 1.4796136617660522, 1.1128358840942383, -2.1641223430633545, 0.2754051685333252, 0.4981423020362854, 2.214001417160034, 0.9590268731117249, 3.0263705253601074, -1.3126775026321411, 1.2580547332763672, -1.480584740638733, 0.5302737951278687, -1.3938682079315186, 0.061334848403930664, 2.662437915802002, -2.191610097885132, 0.388581782579422, -1.1984509229660034, -1.32613205909729, 0.4248160123825073, -0.16295206546783447, 1.177711009979248, -1.9138312339782715, 1.6720006465911865, 2.267612934112549, -0.791202187538147, -1.5222067832946777, -1.4244117736816406, -1.1648128032684326, -0.38999828696250916, -0.9456764459609985, 1.532673716545105, 1.8535685539245605, -1.647274374961853, -1.5240060091018677, 0.15782535076141357, 1.3696633577346802, 2.350778102874756, -1.890397310256958, -1.0479131937026978, -2.1386754512786865, 0.5894303321838379, 0.15915532410144806, -0.5928121209144592, 3.3707127571105957, -2.141597270965576, 3.484480142593384, 2.828397274017334, -1.089395523071289, 1.06620192527771, -1.8788143396377563, -0.15969878435134888, -1.469914197921753, 0.2025488018989563, -1.0782480239868164, 1.008514642715454, -1.8232094049453735, -0.3984254002571106, 1.1805152893066406, 1.2620493173599243, 0.9252633452415466, -1.2699307203292847, -1.4191043376922607, -0.29561060667037964, 0.060796260833740234, 2.2841579914093018, -0.07418084144592285, -1.640696406364441, 0.23685353994369507, 0.5152983665466309, 2.8497848510742188, 1.1845893859863281, 3.036975860595703, -1.1054257154464722, 1.4144928455352783, -0.8029036521911621, 0.18579813838005066, -1.4563095569610596, -2.727123737335205, -1.5992136001586914, -0.5842105150222778, -2.247191905975342, 0.9291819930076599, 1.0016417503356934, -0.18453621864318848, -2.226189613342285, -2.7933766841888428, 0.4877208471298218, 2.47147274017334, 1.193224549293518, -0.743553638458252, 1.0433599948883057, -1.5602954626083374, -0.11890888214111328, -1.1234341859817505, 0.9603374004364014, -0.2188897579908371, 0.687631368637085, -0.03999203443527222, 0.7548511624336243, -0.2211553454399109, -0.11565351486206055, 0.68296879529953, -1.4624135494232178, -1.3538508415222168, 1.191646695137024, -0.42771416902542114, -0.10510268807411194, 0.6423019766807556, 1.012125015258789, 1.4642449617385864, 0.5173776149749756, -0.40819787979125977, 0.05293774604797363, -0.8847699165344238, -1.037038803100586, 1.57318115234375, -0.15329855680465698, 0.3648172616958618, -0.03739982843399048, 1.7348387241363525, -1.7044482231140137, 1.3782730102539062, -0.14266341924667358, 0.5759972333908081, 1.3889191150665283, -0.45471376180648804, 1.6385478973388672, 1.0286049842834473, 0.9753926992416382, 1.1108684539794922, 0.13182200491428375, -2.315385580062866, -1.063974380493164, -0.6252145171165466, 1.054415225982666, -1.2688159942626953, -0.3773394823074341, -1.0210356712341309, -1.8251773118972778, 0.7699798941612244, 0.1199960708618164, 1.3701120615005493, 3.565446376800537, -0.8243624567985535, -2.7836618423461914, -1.3326780796051025, -2.0170140266418457, 0.012171197682619095, 0.4381348490715027, 0.8652936220169067, 2.4739508628845215, 0.5751829743385315, -2.6578643321990967, -0.9340274333953857, -0.33877140283584595, 0.012432724237442017, 0.938054084777832, -1.6519265174865723, 1.2473233938217163, -2.5052952766418457, -0.424641489982605, 1.9901554584503174, -0.9429917335510254, 2.2766988277435303, 0.14636850357055664, -0.18735210597515106, -0.9658775329589844, -0.6912105083465576, -0.8987205028533936, 0.9220184087753296, -1.2408602237701416, -0.6464362144470215, 0.4645744562149048, 1.9304335117340088, -1.4411289691925049, -0.47768211364746094, 3.1406378746032715, -2.5379042625427246, 0.6933693885803223, -1.722785472869873, -2.570953369140625, 0.07858532667160034, -0.4175143241882324, -1.0269713401794434, 2.680027961730957, -0.5298089981079102, -0.14583736658096313, -0.047103047370910645, -0.7623022794723511, -0.9857932329177856, 0.807794451713562, -1.1506707668304443, -0.5890523791313171, 2.100721836090088, 1.3059170246124268, -1.2531369924545288, -1.1744704246520996, 0.46496015787124634, 0.32032787799835205, 0.7081015110015869, 2.2041549682617188, 2.485562324523926, -2.20454740524292, -0.31660395860671997, -1.41000497341156, 1.224191427230835, -1.0873239040374756, 0.978422999382019, 0.27231866121292114, 1.1768032312393188, 0.2556458115577698, -0.9317265748977661, 0.0613441988825798, 0.8616912364959717, 1.572113037109375, 0.3372610807418823, -0.5044298768043518, -1.9320147037506104, 0.842933177947998, -0.03769025206565857, -0.5915497541427612, -2.357941150665283, 0.295448362827301, -2.006587505340576, 0.723402738571167, -0.9131127595901489, -1.3134331703186035, -1.716220736503601, -0.2948211133480072, 2.985752582550049, -0.9686354994773865, 2.8871335983276367, -0.13622987270355225, 1.3652007579803467, 4.642745494842529, 1.6929548978805542, 0.21591368317604065, 0.2204253077507019, 0.36177271604537964, -1.5699896812438965, -0.7361137270927429, 3.163978099822998, -1.3523941040039062, -1.1210393905639648, 1.0290056467056274, -0.3574334383010864, -0.8225580453872681, 0.11932951211929321, 0.616788923740387, -0.3246234059333801, -1.4497183561325073, 0.028198570013046265, -0.6907192468643188, 0.19282948970794678, 2.20538330078125, 1.3105003833770752, -0.3950080871582031, 0.35764792561531067, 0.8818305134773254, -2.1513760089874268, -1.1963976621627808, 1.7316172122955322, -0.14359831809997559, -0.7977840900421143, 0.08865579962730408, -0.7158344388008118, -2.378006935119629, 0.7840957641601562, 0.16410821676254272, -0.7280166149139404, -0.6072106957435608, -1.159982681274414, 0.3094968795776367, -1.37887704372406, -0.0634075403213501, 0.6762971878051758, 3.4870429039001465, 0.4150497019290924, 2.2865757942199707, 0.8317436575889587, 1.3500521183013916, 0.29272812604904175, -1.4236993789672852, -0.9754300117492676, -0.8911038041114807, 2.757197618484497, 0.19582803547382355, 0.9515626430511475, 1.339950680732727, -0.7539942264556885, -0.9405257105827332, -1.4010860919952393, 1.0106369256973267, 0.13159549236297607, -0.8806827068328857, 1.541689395904541, -2.190502166748047, 1.790502667427063, 1.104495882987976, -0.625011682510376, 0.005206584930419922, 2.0653114318847656, -1.5729141235351562, 2.5320143699645996, -2.3937008380889893, 0.378447949886322, -1.098420262336731, 0.8094409108161926, -0.012558985501527786, 1.5675318241119385, -2.1682562828063965, -0.4385445713996887, 0.41155505180358887, 1.604635238647461, 3.2626309394836426, -3.621502637863159, 0.8354344367980957, 2.4737370014190674, 1.0127077102661133, -0.053277015686035156, 0.8798747062683105, 2.235154628753662, -1.3674118518829346, 1.1762657165527344, -0.054699063301086426, 1.7903978824615479, -0.41723883152008057, -0.9087803959846497, -0.49768728017807007, 1.6308926343917847, 1.6962831020355225, 0.09932921826839447, 3.097571849822998, -2.2805275917053223, -1.375470519065857, -2.5154035091400146, -0.5553502440452576, 0.40415680408477783, -0.39696305990219116, -0.8463422060012817, 3.1199257373809814, 0.016100168228149414, -0.25652581453323364, 0.32018306851387024, -3.140537977218628, 2.253873109817505, -0.09351444244384766, -0.06431365013122559, 1.7703462839126587, -1.6292724609375, 1.3651123046875, 1.7290763854980469, 1.523869276046753, -0.018743813037872314, 0.538600504398346, 0.056321561336517334, -2.656506061553955, 0.5042445659637451, 0.6729462146759033, -1.9616053104400635, 0.2430204153060913, 2.9933652877807617, -1.7551250457763672, -0.24100369215011597, 1.7178144454956055, 0.31733304262161255, 0.9652661681175232, 0.45007768273353577, 1.7584495544433594, -0.9159652590751648, 0.18776392936706543, -1.1784802675247192, -0.9270347356796265, -0.27652716636657715, 0.09085032343864441, 1.8049594163894653, -2.0357275009155273, -0.16921138763427734, -1.9543676376342773, -1.1542969942092896, 0.9315851926803589, -1.5337305068969727, 1.0182631015777588, -1.7910118103027344, -0.038594573736190796, -0.9016834497451782, 0.8321841955184937, 0.76197350025177, -0.9596335887908936, 0.21417903900146484, -0.531299352645874, -0.9836683869361877, -0.08680424094200134, -2.024373769760132, -1.2531872987747192, -1.6344096660614014, 0.9066654443740845, -0.5824360251426697, 0.8576482534408569, 1.0020394325256348, -1.5365163087844849, 1.360571265220642, 0.23327553272247314, -0.5360015034675598, 0.6670973300933838, -1.8179140090942383, 0.1044122576713562, 0.6600978970527649, -1.7805644273757935, -1.191824197769165, 0.03926044702529907, -0.41646063327789307, -0.0168474018573761, 3.2321698665618896, -0.4832828640937805, -1.2330322265625, -0.2566940188407898, 2.5367813110351562, -0.02983403205871582, -0.2767569124698639, -0.6412680745124817, 0.704748272895813, -1.742577314376831, 1.351472020149231, -1.3065516948699951, -1.580487847328186, 1.1055084466934204, 1.367382526397705, 0.10213017463684082, -0.6685580015182495, -0.17820590734481812, 0.9290779232978821, -1.0048718452453613, 0.3312476873397827, 0.4665147066116333, 0.15231141448020935, 1.2678011655807495, 1.9053280353546143, 1.3697261810302734, 0.7096025347709656, -1.445505142211914, -0.5891339778900146, 0.5228405594825745, 0.7208412885665894, 1.0665974617004395, 0.4956713020801544, -0.7996780276298523, 1.6950814723968506, 0.4436657428741455, -0.8226397037506104, -3.2671194076538086, -0.07125324010848999, -0.41923999786376953, -0.5632824897766113, -1.1727267503738403, -0.6649289131164551, 1.9194869995117188, 1.0145044326782227, 0.4853326678276062, -1.7071408033370972, 0.8088486194610596, -0.11604094505310059, -1.336256504058838, -0.6638636589050293, 0.5588402152061462, -0.7233000993728638, -0.9500607252120972, -1.4017524719238281, 0.1568775773048401, 1.2918620109558105, -0.37255048751831055, -1.7623978853225708, 1.0846954584121704, 1.5309293270111084, 1.1025642156600952, 1.35009765625, -0.8338228464126587, -1.112391471862793, 1.5527856349945068, 0.1040131151676178, 0.7608277201652527, 1.7714862823486328, -0.51191246509552, -2.8769519329071045, -1.0201890468597412, -1.405241847038269, 0.5903335213661194, 1.554956316947937, -1.574315071105957, -2.1295628547668457, 1.4387215375900269, 0.4752487242221832, -0.5828481912612915, -1.4485702514648438, -1.7122273445129395, -0.6514915823936462, -0.7372596859931946, -0.6560316681861877, -0.5964958071708679, -2.8058464527130127, 2.2807538509368896, 0.9804121851921082, 1.0443949699401855, 0.8897939324378967, -2.8215930461883545, -1.013871431350708, -1.8888330459594727, -2.2239303588867188, -0.012316703796386719, -0.6491450071334839, -1.9267897605895996, 0.011608988046646118, -1.4796847105026245, -0.7585480213165283, -1.1527326107025146, 2.7146730422973633, -0.7870832681655884, 0.25817984342575073, 0.32222819328308105, -1.9802743196487427, 0.8566135764122009, 0.3669769763946533, 0.08427262306213379, 2.0830273628234863, -0.12536609172821045, 0.06447345018386841, 0.1965705156326294, -2.700228452682495, 1.9641225337982178, -1.5345432758331299, -1.515349268913269, -0.853772759437561, 0.7288262844085693, 2.078134775161743, 0.3544066548347473, 0.8417313694953918, -1.8194241523742676, -2.253018379211426, -0.05065953731536865, 0.6538875102996826, 3.9239678382873535, 2.204101085662842, -0.23781687021255493, -1.585998296737671, -1.2431035041809082, -0.42671024799346924, 0.9892726540565491, -3.3593125343322754, 1.9981274604797363, -0.47954899072647095, 0.8944765329360962, -1.158254623413086, 1.6442766189575195, 0.2331310212612152, -4.68814754486084, 1.1837397813796997, -1.0132817029953003, -0.4008578658103943, -0.696382462978363, 1.3034497499465942, 3.787126064300537, -2.665957450866699, -1.8054403066635132, -0.6415950059890747, -0.42781490087509155, -1.5877599716186523, -1.0419126749038696, 2.0524959564208984, 2.1192097663879395, -1.2169307470321655, -1.7817057371139526, 0.20312488079071045, 0.8635984659194946, 0.11439164727926254, 0.400258868932724, 0.8830995559692383, 0.4021710753440857, -0.20980334281921387, 0.5331029891967773, -0.13686400651931763, 0.4154069423675537, 0.8375552892684937, -3.1105566024780273, 1.2571690082550049, 0.43093425035476685, 0.012105405330657959, 1.1649444103240967, 1.4722195863723755, 2.6656758785247803, -1.5462839603424072, -0.5496934652328491, 0.4976600408554077, -0.4914214313030243, -0.6660131812095642, -1.8421661853790283, -2.053555488586426, 0.16549962759017944, 2.336210250854492, -1.120802879333496, -2.0141830444335938, -0.8802566528320312, 2.3947267532348633, 0.48723259568214417, -2.41802716255188, -1.9383115768432617, 1.7426550388336182, 0.8329139351844788, 0.6901828050613403, -0.10616439580917358, -2.3098628520965576, 0.5863457918167114, -1.8863050937652588, -0.8137387037277222, -0.9961332082748413, 1.218957781791687, -0.506090521812439, -1.302552580833435, 1.565511703491211, 0.9312556982040405, -1.5880072116851807, 0.3464565873146057, -1.9640415906906128, -1.210545301437378, -2.835794448852539, -0.6419343948364258, -0.020726680755615234, 2.6561384201049805, 0.903040885925293, 1.285097599029541, 0.15393507480621338, -2.550229787826538, 0.7476022243499756, 0.41732415556907654, 0.7470868229866028, 0.7914300560951233, 0.6870677471160889, -0.1206282526254654, -0.8458147644996643, 1.7261710166931152, -1.1970882415771484, -0.46375855803489685, 2.727747678756714, 2.9983198642730713, -0.0749351978302002, 2.1068222522735596, 1.1493173837661743, 2.329991579055786, 0.270759254693985, -2.207777738571167, -0.14488402009010315, 0.5837765336036682, 1.242380142211914, 0.38765615224838257, 0.47949105501174927, -1.4926624298095703, 1.1294060945510864, -0.09963014721870422, -1.7683539390563965, -0.3489135801792145, -1.858213186264038, -1.5041075944900513, 0.3241903483867645, -1.0742599964141846, -3.673353672027588, -1.6305174827575684, -0.1334700584411621, -0.9454208612442017, -0.9391598105430603, 1.5582458972930908, 0.5290179252624512, -1.7855141162872314, -0.8127365112304688, -0.6754944920539856, -1.3006072044372559, 0.0499730110168457, 0.8539944291114807, -0.74040687084198, -1.0049656629562378, -0.7972533702850342, 0.5691608786582947, -0.5299123525619507, 0.8897730708122253, -0.2712574005126953, 0.35352569818496704, 0.3409304916858673, 0.34573209285736084, 0.2076769769191742, 0.19895660877227783, -0.13925163447856903, -1.9946800470352173, -0.35463064908981323, -0.8379836082458496, 2.7012577056884766, 1.9953222274780273, -3.1452808380126953, -2.0644357204437256, -1.4734721183776855, 0.9208908677101135, -0.1600642204284668, -1.6655865907669067, -0.7230285406112671, 1.1880953311920166, -0.9608707427978516, -0.09775692224502563, 0.93125319480896, 0.7210812568664551, -3.574772834777832, 2.092867612838745, -3.52333402633667, -0.1160162091255188, 0.06312564760446548, -1.5101804733276367, -1.8377110958099365, -1.7732830047607422, 1.005702018737793, 2.9256134033203125, 1.9286797046661377, -1.6569297313690186, -1.0647032260894775, -0.2869751453399658, -1.2455668449401855, 0.9080219268798828, 0.7889889478683472, -2.796217203140259, -1.789742350578308, 2.0528106689453125, -0.08230781555175781, 1.7897690534591675, -1.6487761735916138, -2.3607592582702637, -1.9933056831359863, -1.7871063947677612, 0.08082768321037292, 0.8653146624565125, -0.46764683723449707, -0.17867520451545715, 0.2958781123161316, 0.17617356777191162, 0.13547807931900024, -2.089354991912842, -0.3966718316078186, -1.8473405838012695, -1.7678178548812866, -2.2043280601501465, -2.4119906425476074, -0.04178735613822937, 2.8230552673339844, 1.4906151294708252, -0.6579651832580566, -0.9860883951187134, -1.199028491973877, -1.2205088138580322, -0.237554132938385, 2.0847573280334473, 0.11842620372772217, -0.9218374490737915, -0.2033395767211914, 1.0602588653564453, 0.36137717962265015, -0.13309453427791595, 1.2686309814453125, 0.9297598004341125, 2.463444232940674, 2.7579545974731445, 0.15975421667099, 1.0623323917388916, -0.09967346489429474, 0.5161558389663696, -0.07958865165710449, 1.4304910898208618, 0.6258926391601562, -1.7957755327224731, -4.305334091186523, -3.3231496810913086, -3.0994184017181396, -0.49226754903793335, 0.13307741284370422, 0.4963993430137634, 0.5934188365936279, 3.2666430473327637, -1.1908491849899292, -1.4985783100128174, -0.005277425050735474, 1.02415931224823, -0.660823404788971, 0.6195011138916016, 1.477060317993164, 1.0122350454330444, 2.028935194015503, -2.4447319507598877, -1.6942460536956787, -1.4871279001235962, 0.9505369663238525, -0.7042906284332275, -1.7461698055267334, -1.2004010677337646, -1.735560655593872, -1.1778502464294434, 0.47302210330963135, -0.9297450184822083, -2.2779064178466797, 0.7135963439941406, -0.2072504758834839, -0.45094746351242065, 1.274537444114685, 0.7047373652458191, 0.5681719779968262, -2.1253440380096436, -0.8682223558425903, 0.6410028338432312, 0.3908524513244629, 0.056591153144836426, -0.31036317348480225, 0.07821673154830933, -2.4506802558898926, 2.3767902851104736, 0.2665064036846161, 0.24744203686714172, 0.4148637354373932, -0.3392403721809387, -0.3082558810710907, 0.9211483001708984, -1.1786441802978516, -2.099092483520508, 0.14610156416893005, 0.45494794845581055, -0.26167505979537964, 0.6705157160758972, 2.298553228378296, 1.920754075050354, 1.3796637058258057, 0.6641360521316528, -1.865347981452942, -3.1195709705352783, -0.6814455389976501, -0.7597700357437134, -1.1366195678710938, -0.8000428676605225, 0.7629607915878296, 0.8545172214508057, 0.9175881743431091, 1.090522050857544, 1.729386806488037, 0.0047452449798583984, -1.973494052886963, 0.1324649453163147, -0.7622931003570557, 0.16666215658187866, 0.48929548263549805, -0.8415597677230835, 0.09677422046661377, 1.8931748867034912, -0.9718648195266724, -1.2689213752746582, 1.3870971202850342, -0.8309789896011353, -0.9434009194374084, -0.814449667930603, 1.3825914859771729, -1.8521265983581543, -1.9288030862808228, -0.047025442123413086, 1.5923607349395752, 0.0708271861076355, 0.8173668384552002, 0.7076300382614136, -0.4964924454689026, 0.5443000197410583, -1.9900928735733032, 1.251590371131897, 1.2738738059997559, -0.2600228488445282, 0.18004006147384644, -1.507529616355896, -1.3652809858322144, 0.26136958599090576, -0.37064266204833984, -1.686222791671753, -0.8661559224128723, -3.0335476398468018, 0.9115403294563293, -0.6734721660614014, -2.8190181255340576, 0.10046602785587311, -0.4529229998588562, 1.579030990600586, 1.3503402471542358, 0.23646502196788788, -0.5653952360153198, -2.230478286743164, 1.6987799406051636, 0.4347153306007385, -1.5049868822097778, -1.0892956256866455, 0.08396351337432861, -1.3512629270553589, -0.1609477996826172, 0.7105410099029541, -1.9863629341125488, -0.22152455151081085, -1.6705272197723389, -2.2745883464813232, 0.13976384699344635, -2.669532060623169, -0.7255131006240845, -0.25213491916656494, -0.1612657904624939, -1.728626012802124, 0.3914293050765991, -0.16745324432849884, 1.533023476600647, -2.8396058082580566, -0.20536328852176666, -0.185712069272995, 2.540686845779419, -0.6652172207832336, -0.6400800943374634, -0.6786044836044312, 0.7091841697692871, 0.34691691398620605, -1.282010555267334, 1.4471102952957153, -3.5654923915863037, -1.4810289144515991, 1.2084242105484009, -0.7152894139289856, 1.844111680984497, 0.19774046540260315, -1.4659380912780762, 1.0438530445098877, 0.9211628437042236, -0.9373835921287537, 0.6859481334686279, 0.027385711669921875, -1.571889877319336, -1.3743836879730225, -0.16204869747161865, -1.4880619049072266, -1.053083062171936, -1.4488911628723145, 0.8673242330551147, -0.46854978799819946, 0.29986777901649475, -1.4149551391601562, 1.034095048904419, -0.9380833506584167, -0.5676556825637817, -0.3299909234046936, -1.8675402402877808, 0.5788379311561584, -0.750774621963501, 0.3693748712539673, -0.4298069477081299, 0.013060808181762695, 1.4952627420425415, -0.078407883644104, 0.46955087780952454, 0.3032188415527344, -2.6833457946777344, 0.20270907878875732, 1.182385802268982, 1.6480774879455566, 1.8315223455429077, -0.8824350833892822, 1.2455129623413086, 1.244425892829895, -2.59098482131958, -2.9914278984069824, 2.3656816482543945, -0.878777265548706, 1.4283580780029297, 2.1308796405792236, -1.5440881252288818, 0.38380149006843567, 1.4917861223220825, -0.5763169527053833, -0.295166015625, 1.2287486791610718, -1.4957916736602783, 1.783848762512207, -1.1389871835708618, -0.994366466999054, 1.9356365203857422, -1.7227286100387573, -1.4994252920150757, -1.5231331586837769, -0.886681079864502, 0.38401710987091064, 0.4969899654388428, -0.8196287155151367, -0.15669935941696167, 1.266526699066162, -0.5214500427246094, -0.9591753482818604, 2.656963586807251, -1.4127819538116455, 0.314597487449646, -0.919469952583313, -0.8184218406677246, 1.0390478372573853, 1.9530726671218872, 0.2638389468193054, 1.7019715309143066, -3.548536777496338, -0.17937123775482178, -1.3674921989440918, 1.3421118259429932, 2.623778820037842, 1.7264620065689087, 1.4977376461029053, -0.14744074642658234, -0.8337563872337341, -0.9341698884963989, -1.9717540740966797, 0.9717195630073547, -1.5741839408874512, -0.739729106426239, -0.19900742173194885, 0.9852205514907837, -2.0061912536621094, 1.6178770065307617, -0.15518541634082794, -1.5065455436706543, -0.9887995719909668, -0.5094369649887085, 1.8598136901855469, 2.3265304565429688, -0.09677287936210632, -1.2030268907546997, -1.7827590703964233, -0.9298275709152222, 1.3226380348205566, -1.6141352653503418, 2.22528338432312, -2.7193374633789062, 1.2595515251159668, -0.9676433205604553, -1.2117462158203125, 0.17776912450790405, -1.6918526887893677, -0.03951945900917053, 0.17503029108047485, -2.6774182319641113, 0.9787973165512085, 0.07635903358459473, 2.655447006225586, -1.2872880697250366, -2.6362757682800293, 1.6008707284927368, 0.5604475736618042, -1.6962926387786865, 0.1247032880783081, 0.6008390188217163, -0.33078518509864807, -0.007366657257080078, 3.3634700775146484, 1.9989484548568726, -0.37550783157348633, 0.012977778911590576, 1.039556622505188, 1.3769344091415405, 2.764357089996338, 0.12764453887939453, 0.5741136074066162, 0.7900285720825195, 1.764611840248108, 0.29009586572647095, -1.5543001890182495, -1.5399119853973389, -0.43331170082092285, 2.023057222366333, -1.9604575634002686, -0.8888812065124512, 2.6956112384796143, -0.25160086154937744, 0.02687174081802368, 0.8106904029846191, 1.162977695465088, 0.16827113926410675, -1.8116642236709595, 0.8732646703720093, -0.3928585648536682, -0.5761144757270813, -1.9733304977416992, 0.40198689699172974, 0.17022296786308289, 1.7585759162902832, -0.0947442352771759, 1.5591551065444946, -0.4552574157714844, 0.8687922358512878, -0.41292455792427063, -1.4234178066253662, -1.453209400177002, -0.10147523880004883, -1.9812781810760498, -1.621079444885254, 1.726300835609436, -2.8426620960235596, -1.7522335052490234, 2.5259015560150146, 0.9440125226974487, 2.7763190269470215, 2.232860565185547, -1.5836172103881836, -3.59537410736084, 2.33988618850708, -1.769012212753296, -1.722064733505249, -0.2945082187652588, 1.37689208984375, 0.3784255385398865, -0.9391021132469177, 2.482391119003296, 3.292123794555664, -2.626502275466919, 0.5256023406982422, -0.20020350813865662, -2.3900437355041504, -0.04878199100494385, 1.0073350667953491, -3.421578884124756, -2.9803266525268555, -2.735438823699951, 0.6751817464828491, -1.1348909139633179, 0.3863232135772705, -0.35024669766426086, -0.26632213592529297, -0.005158483982086182, 1.3125091791152954, -0.5491569638252258, -0.37193483114242554, -2.0794901847839355, -0.2435958832502365, 1.4299218654632568, 0.4589795470237732, 0.0958547592163086, -0.5705347657203674, 0.2847302556037903, -0.5739567875862122, -1.4911924600601196, -0.3785814046859741, -2.1108663082122803, -0.36367303133010864, 0.7004914283752441, 0.575737714767456, -0.8010290861129761, -0.02032572031021118, -0.9488596320152283, 2.5109872817993164, -0.09671607613563538, -1.4860550165176392, 0.6797201633453369, -1.0123519897460938, -2.2825188636779785, 0.02645556628704071, 0.45744818449020386, -0.7478013038635254, 2.7362236976623535, 0.20878267288208008, -1.2291896343231201, 0.2539966106414795, -1.4086703062057495, 0.7247498631477356, 1.373774766921997, -0.13064125180244446, -1.493796706199646, 2.369943618774414, 2.006950855255127, 2.3880457878112793, -1.732614278793335, -1.1416386365890503, -1.616725206375122, 2.033780574798584, -0.5533473491668701, -0.8602316975593567, -0.6016224026679993, 1.2412614822387695, 0.6389297246932983, -0.9528546333312988, 1.4843120574951172, -0.6818632483482361, -0.36759263277053833, -3.4919633865356445, 1.0987613201141357, 1.9327669143676758, -1.1983985900878906, -0.8269923329353333, -0.5553053021430969, 0.7940931916236877, -0.4481441378593445, 1.3249835968017578, 0.6635392904281616, -0.5805623531341553, 0.22746139764785767, 1.033708095550537, -1.1473429203033447, 1.2264454364776611, 1.8528287410736084, -2.98214054107666, -1.2092294692993164, 0.9345683455467224, -0.20854075253009796, 0.6856486797332764, 1.7545608282089233, -1.072672724723816, 2.2569472789764404, -0.6481775045394897, 1.459964394569397, 0.31723809242248535, 2.6214685440063477, 0.5129978656768799, 2.3246958255767822, 0.165057972073555, -1.057718276977539, -0.11956781148910522, 1.750997543334961, 1.9455013275146484, -1.463086724281311, 0.4678267240524292, -2.113913059234619, 1.4694117307662964, -0.3979245126247406, -0.07613620907068253, -0.733404815196991, 0.8780103921890259, 0.4210710823535919, 0.4216352701187134, -1.4734299182891846, 1.4349757432937622, 0.6137193441390991, 1.8613917827606201, 0.19846811890602112, 0.7281147241592407, -1.203544020652771, -0.24551072716712952, -0.1963331699371338, 1.9662656784057617, 0.38548532128334045, -0.2714281678199768, -1.3526577949523926, -1.8701882362365723, 0.18416497111320496, 0.46754443645477295, 0.8924286365509033, 0.24476903676986694, 2.1416308879852295, -0.5444408655166626, 1.540656566619873, -2.0750625133514404, -1.0148463249206543, 0.33705416321754456, -0.502878725528717, -1.506620168685913, -0.7295489311218262, -0.38280582427978516, -1.668639898300171, -0.27961552143096924, -1.8646847009658813, 2.1179442405700684, 1.2930376529693604, 2.111175537109375, -0.8926228284835815, 0.3327455520629883, -1.7522653341293335, -0.22791671752929688, 0.1964883804321289, 1.1268612146377563, -1.3058042526245117, -0.34273749589920044, -1.6221704483032227, 0.09571027755737305, -0.4085736870765686, 1.5972225666046143, -0.9325875639915466, -1.3935182094573975, -0.028953611850738525, 3.0420069694519043, 0.8957029581069946, -0.2126646637916565, -2.091566801071167, -0.4420846104621887, -0.43154478073120117, 1.7047604322433472, -1.7314835786819458, 1.4004909992218018, 2.5727615356445312, 0.2367650270462036, -0.39648115634918213, -0.3327786326408386, 0.5354899168014526, -0.39198967814445496, 1.444395899772644, 0.23311646282672882, -0.8147557973861694, -0.30073779821395874, 2.970127820968628, 0.4771038293838501, 1.2024612426757812, 1.7388489246368408, 0.7491574287414551, -2.9370248317718506, -4.374457359313965, -0.2792741060256958, 0.43831866979599, 0.06670066714286804, -1.2674956321716309, 0.6854716539382935, -2.106715202331543, 2.414006471633911, -2.2419679164886475, -2.2368826866149902, -0.6534025073051453, -1.1971940994262695, 0.9274032115936279, -0.9838389158248901, -0.7425678968429565, -1.2231292724609375, 3.321859359741211, 0.5903528928756714, 2.3005523681640625, 1.7515355348587036, 0.6902241706848145, 0.5033560991287231, -2.305091381072998, -1.0498048067092896, -0.32877498865127563, -1.4719892740249634, 1.1309988498687744, 0.6637215614318848, -0.5714346766471863, -2.1516637802124023, 0.7810835838317871, 2.698807716369629, 0.8458654880523682, -0.23548981547355652, -3.650784492492676, 2.1247057914733887, 1.6606203317642212, -0.8041644096374512, -0.704994797706604, 2.5908071994781494, -2.8136370182037354, 0.9438456296920776, 2.763321876525879, 0.36819982528686523, -1.2687629461288452, 4.09637451171875, -3.877127170562744, -1.9177616834640503, 0.6305112838745117, 0.521338164806366, 0.03385736048221588, 0.9583568572998047, 0.5901502370834351, -1.1508588790893555, 1.6296796798706055, -0.21076077222824097, 0.6794019937515259, 0.10888682305812836, 1.3658872842788696, -1.4623136520385742, 2.9674766063690186, -0.8117161393165588, 0.9626728892326355, -2.467360019683838, -2.102962017059326, 0.43169087171554565, -0.4194484353065491, 1.3275189399719238, 2.289219856262207, -0.7461800575256348, -0.13267099857330322, -2.1505603790283203, 0.213668555021286, 1.1223725080490112, 3.5039420127868652, -1.8313525915145874, -2.277392864227295, -0.8238557577133179, 2.3561036586761475, -2.198704481124878, -0.07014143466949463, -1.932462215423584, -3.8919386863708496, -1.130217432975769, -1.2807133197784424, 0.8588907718658447, -1.3082178831100464, 1.569366455078125, 0.11713942885398865, 0.6754963397979736, -0.08898866176605225, -0.3632749319076538, -1.9016342163085938, -1.2845274209976196, -0.7297250032424927, -1.0333576202392578, 0.6405407190322876, 1.3746792078018188, -0.6369231939315796, -1.8413795232772827, -0.49847671389579773, -0.5698503255844116, 0.031730473041534424, -1.1817184686660767, -0.49997884035110474, 0.4854732155799866, -1.0585428476333618, 0.45625635981559753, 0.02274828404188156, 0.0719262883067131, 0.43444740772247314, 1.425335168838501, 0.7750298380851746, -3.091691255569458, 1.1230751276016235, 1.8701714277267456, 1.1172213554382324, -2.02455997467041, -0.5259519815444946, 0.034921348094940186, -1.2588363885879517, -2.112201452255249, -0.9041435122489929, 0.6595778465270996, 0.510794460773468, -1.0922881364822388, 2.367036819458008, -2.4117066860198975, 2.8173108100891113, 0.993547260761261, -0.7660714387893677, -1.5683140754699707, 3.07193660736084, 2.755404472351074, -0.4897373914718628, -1.152748942375183, 3.156320810317993, -1.0165932178497314, -2.563884973526001, 1.964691162109375, 1.395654320716858, 2.334609031677246, -0.2845913767814636, -0.18888652324676514, 0.6877533793449402, -2.1350648403167725, 2.4197614192962646, 2.254154682159424, 1.028110146522522, -0.8467077016830444, 0.9174749851226807, 0.0517657995223999, -3.0934998989105225, -0.4999178946018219, -1.2844176292419434, -1.5030179023742676, -0.2611386775970459, -0.36398303508758545, 0.16952407360076904, 0.4773215651512146, -1.3461427688598633, 1.0546050071716309, -1.6476631164550781, 0.3275168538093567, -1.10763418674469, -0.15934661030769348, 0.5212976336479187, 0.24978303909301758, -0.5703792572021484, -1.6806716918945312, -1.0555129051208496, -0.16513702273368835, -1.0806219577789307, -2.241168260574341, -0.011990189552307129, -0.2804781198501587, -3.280761957168579, 0.009216845035552979, 1.0530309677124023, -1.0797154903411865, -1.5449738502502441, 1.5923978090286255, 0.18367022275924683, 0.7013309001922607, 1.176599144935608, -2.981346368789673, 2.0330677032470703, 1.08335542678833, -0.8832473158836365, 2.0122413635253906, -3.1684460639953613, -0.0711100697517395, 2.2740418910980225, -0.08351385593414307, 0.43921056389808655, 1.9040582180023193, 1.1285561323165894, 0.038774698972702026, -1.301837682723999, -0.7384346723556519, 0.8914589285850525, -0.9515653848648071, -1.8208341598510742, 1.5259015560150146, -0.16168618202209473, 0.8993433117866516, -3.3975186347961426, -1.920859456062317, -0.5454156398773193, -0.6663023233413696, -0.909846842288971, -0.26595234870910645, -1.1599606275558472, 0.9642094373703003, -1.1130385398864746, 0.5077896118164062, -1.122762680053711, -0.528566837310791, 0.47642436623573303, 1.4571852684020996, 0.6322933435440063, 1.6729565858840942, -0.21782244741916656, 0.18493308126926422, 2.4243545532226562, -0.42907506227493286, -0.30778515338897705, -0.8685826063156128, 0.8185465335845947, -2.294059991836548, 1.6131172180175781, 1.202235460281372, 1.606790542602539, 0.7395747303962708, 0.03177440166473389, 0.6179776191711426, 2.053952217102051, 1.0826352834701538, 1.4097797870635986, 1.2123773097991943, -0.47614169120788574, -0.9085568189620972, -1.3142333030700684, 0.11030721664428711, -0.6315447688102722, 0.19554153084754944, 0.5591204166412354, -0.6221250295639038, -0.3969115614891052, -0.3338397741317749, 3.1166257858276367, 2.8445487022399902, -1.2650216817855835, -0.6243225932121277, -0.10201627016067505, 2.5468907356262207, -1.4733037948608398, 2.6463379859924316, 2.2147469520568848, -0.09120440483093262, 0.006529688835144043, 1.7722786664962769, -0.1752057671546936, 0.7225300073623657, -0.9488756060600281, 1.8900337219238281, -0.3369896113872528, 0.26220035552978516, 1.2111918926239014, -0.38552725315093994, -0.34738463163375854, -0.19107979536056519, -0.2351340651512146, -2.529010772705078, 0.8142279386520386, -0.829662561416626, 1.3300449848175049, -0.7500278949737549, 0.6178391575813293, 0.9380403757095337, 0.017732739448547363, -0.0007623434066772461, -1.6879266500473022, 0.04210233688354492, 0.9581202268600464, -0.2136162370443344, -1.6766923666000366, -2.2839972972869873, -2.4143829345703125, -3.7275753021240234, -0.06028389930725098, -0.5508942604064941, -0.8879649639129639, 0.8863619565963745, -0.5323712825775146, 3.4575140476226807, -0.41044366359710693, 0.8041921854019165, -0.08388197422027588, -0.6570456027984619, 0.11718398332595825, -1.2200498580932617, 0.6279270052909851, 1.7908226251602173, 2.4737699031829834, -2.424772262573242, -0.7352437973022461, 1.6349562406539917, -0.045641541481018066, -0.9871138334274292, 1.6903188228607178, -2.689408540725708, -1.1142652034759521, -1.03703773021698, -0.2627484202384949, 1.6908459663391113, 0.3269071578979492, 0.43818849325180054, 1.9826043844223022, 0.29160141944885254, 1.2299171686172485, 0.7789440155029297, 0.7004854679107666, -1.21620512008667, 2.2075419425964355, -0.20088577270507812, 0.47416815161705017, 1.6114319562911987, -0.4687548875808716, -3.724853515625, -1.0784965753555298, 0.5400829315185547, -3.134805917739868, 1.0386314392089844, 0.0846758633852005, 0.7794048190116882, 0.7307465672492981, -0.12266987562179565, -2.8894028663635254, 1.9375802278518677, 0.5264971256256104, 2.2069053649902344, -1.4771349430084229, -0.5220483541488647, 0.7498019337654114, -0.33217906951904297, 1.5167392492294312, 1.4628827571868896, 0.34616830945014954, -2.4392757415771484, -0.4669817090034485, 1.3991010189056396, 0.7423933148384094, -0.5835034251213074, 0.845274806022644, -0.04579424858093262, -0.9577888250350952, 0.11711984872817993, 3.800873279571533, -1.1005921363830566, -0.4039286971092224, 2.567476987838745, -1.2442009449005127, 0.09332284331321716, 1.8814797401428223, 2.3869917392730713, -3.9685845375061035, -0.5371080636978149, 1.1281908750534058, -1.4199200868606567, 0.8511056900024414, 1.0400142669677734, 1.861779808998108, 1.579329013824463, -1.0762715339660645, 1.7112830877304077, 1.1474883556365967, 0.9553986191749573, -0.2624252736568451, 1.3208694458007812, -2.0671980381011963, -0.19138342142105103, 0.937272846698761, 0.6468234658241272, -1.4388748407363892, -0.38160601258277893, 0.4642050862312317, 0.9455357789993286, -0.7386984825134277, 2.0579135417938232, 0.39245373010635376, 0.41720426082611084, -0.36524736881256104, -2.3715386390686035, -0.03264927864074707, -0.836470365524292, 1.4034556150436401, 0.3539770245552063, -1.3921387195587158, -0.7998868227005005, 2.298337459564209, -2.022361993789673, -0.7850521206855774, 1.2613457441329956, 0.34214478731155396, -0.9864742159843445, 0.03897595405578613, -0.8882476091384888, 1.0017750263214111, 0.6053200960159302, -0.14324422180652618, 0.5022077560424805, -0.3126027584075928, 0.07847237586975098, 0.9570209383964539, 1.2341845035552979, 1.4615707397460938, -2.879364013671875, 3.6634340286254883, -1.4215731620788574, 3.093313217163086, -0.715527355670929, 0.058811768889427185, 0.3186035752296448, 1.1471257209777832, -1.1351947784423828, 1.689233422279358, -0.7862492203712463, -0.35277891159057617, -1.663002610206604, 1.7204846143722534, -2.1783947944641113, -1.5475106239318848, -2.0795538425445557, 0.6765857338905334, -1.5476336479187012, -1.7715696096420288, 0.10911133885383606, -0.7125136256217957, 2.058641195297241, 0.7184895277023315, -1.0483651161193848, -2.137291669845581, -1.7627801895141602, 1.358045220375061, -0.3647262752056122, -0.47350209951400757, 1.2117820978164673, -0.9873805046081543, -1.9868508577346802, 0.762291669845581, 0.8368010520935059, -2.197690963745117, -1.2044334411621094, 2.196267604827881, 0.34448492527008057, 0.1933116316795349, 0.978970468044281, -1.072380542755127, -0.5260743498802185, -0.13624265789985657, 0.2577790915966034, 1.8080438375473022, -0.24506045877933502, 0.33464565873146057, -0.5884450674057007, 3.345484733581543, -0.1299780309200287, 1.001224160194397, -1.0479471683502197, 2.0045061111450195, 0.5095199942588806, 0.08263539522886276, 0.8512660264968872, -1.1553783416748047, 0.5378642082214355, -0.27628397941589355, -0.0198480486869812, 0.7524574398994446, -1.6524049043655396, -0.3828161954879761, 0.6100167036056519, -0.2155502438545227, -0.09497091174125671, 0.7749678492546082, -0.3092106282711029, 0.904991626739502, 0.572499692440033, 1.4938490390777588, 0.6834234595298767, 1.0056748390197754, -1.0478376150131226, -0.2746438980102539, 0.9358248710632324, 1.1835354566574097, -1.5150766372680664, 2.0314085483551025, -0.32157784700393677, -0.5851847529411316, -1.2660167217254639, -0.300071120262146, 2.0152335166931152, -0.45759713649749756, 0.18760061264038086, 1.5958387851715088, 0.7192286252975464, -0.8740899562835693, -0.6921942830085754, -0.5875318646430969, 0.5168887376785278, -0.9644777178764343, -0.04515286907553673, -0.022152751684188843, 1.619887351989746, -0.263060599565506, -0.07549846172332764, 0.39283061027526855, -1.5118223428726196, -0.1953587383031845, -0.9395717978477478, 1.0815606117248535, 0.04885751008987427, 1.4026014804840088, -0.747972309589386, -0.5882650017738342, -1.2644785642623901, 1.9809215068817139, 0.5471374988555908, 2.3480541706085205, 1.1712474822998047, 1.5766476392745972, 1.1082526445388794, 0.3236391842365265, -0.8354694843292236, -0.7376497983932495, 0.6358657479286194, -1.8344646692276, 0.23211923241615295, 1.4480599164962769, -0.31458789110183716, -0.24436292052268982, -0.6348815560340881, -1.210806965827942, -1.1612865924835205, -0.0049135684967041016, 0.028404995799064636, -1.4718157052993774, -1.85725998878479, -1.0791077613830566, 1.6247773170471191, -0.03660257160663605, 0.43504220247268677, 0.0974317193031311, -0.057425886392593384, -1.0019855499267578, 3.4250144958496094, -1.07659113407135, 0.9809938073158264, -1.101075291633606, 0.3331432342529297, 2.9393506050109863, 1.9095343351364136, 1.421625018119812, 1.8862746953964233, -0.17729175090789795, -1.575524926185608, -0.5825533866882324, -1.1421459913253784, -0.7633872032165527, 1.5438271760940552, 1.2964725494384766, -1.9425647258758545, -0.3948233723640442, 1.0483704805374146, -0.3507438898086548, -0.11407240480184555, -1.4981629848480225, -2.1789259910583496, 0.43514788150787354, 2.3661372661590576, -1.6935009956359863, 0.6237294673919678, -0.9255750179290771, -0.5709402561187744, 1.5262727737426758, 0.8049294352531433, -2.4486279487609863, -1.7009750604629517, -1.018570899963379, 1.303985357284546, -1.275919795036316, -0.8865591287612915, 1.532963752746582, 0.4616573750972748, 1.075236201286316, 1.8013510704040527, -0.3846949338912964, 1.0648772716522217, 0.3023540675640106, 1.2228953838348389, -2.396268844604492, 0.4085847735404968, 1.0050214529037476, 0.43620944023132324, 0.1310960352420807, -2.120363473892212, 0.4930444657802582, -0.8418926000595093, 2.0555155277252197, -0.3264056444168091, -0.36485201120376587, -0.7612094879150391, -0.6947311162948608, 1.285748839378357, -2.148587703704834, -1.4989997148513794, -2.268939971923828, -3.0892176628112793, -2.3063297271728516, 1.3352755308151245, -1.4448559284210205, -0.45371466875076294, -0.789240300655365, -0.24909788370132446, -0.5406163930892944, -0.1675298810005188, -0.6761950254440308, 1.1328463554382324, -0.8466675281524658, -0.3850401043891907, -1.118241310119629, 0.9013707041740417, 0.89438396692276, -0.40105682611465454, 1.1413886547088623, -0.4960971474647522, -0.07311755418777466, 1.1184961795806885, -1.0963847637176514, 1.4183173179626465, 1.1858634948730469, 1.9926432371139526, -0.300387978553772, -0.5202341079711914, 2.28147029876709, -2.6720170974731445, -1.5864142179489136, -1.4812699556350708, -1.1730163097381592, 1.1470260620117188, -0.2559768557548523, -0.7113047242164612, -0.1342487931251526, -0.7741357684135437, -1.3583905696868896, -0.4263623356819153, -3.1093547344207764, -1.1172618865966797, -0.9280788898468018, -2.878113269805908, -1.047593355178833, -0.22427308559417725, 0.2923811078071594, 0.5976001024246216, 0.596365749835968, 2.257084369659424, 0.6125468015670776, -0.1427733302116394, 0.7807247042655945, 0.40038058161735535, -1.336195468902588, 1.8620939254760742, -0.8512378931045532, -0.956702709197998, -0.03318911790847778, -2.9558303356170654, 0.30274683237075806, 1.2244075536727905, 1.218948483467102, 0.3903082609176636, 2.976663827896118, 0.8932603001594543, -0.4584752321243286, -1.253119945526123, -2.1879639625549316, 0.2512385845184326, -0.948327362537384, 0.7321459054946899, -2.2452659606933594, -0.39102232456207275, 1.2502591609954834, 1.231121301651001, 0.6053577661514282, 0.8326913118362427, 2.6558799743652344, 0.5854708552360535, 0.32961270213127136, -0.228175550699234, 1.5348012447357178, 0.283164381980896, -0.7015965580940247, -0.7327649593353271, 1.7167997360229492, -1.275740623474121, -0.3052314519882202, -0.2886092960834503, 0.9984234571456909, 0.4989030063152313, 2.6578452587127686, -0.06132692098617554, -0.9353808760643005, 1.533388614654541, 0.18106800317764282, 0.2997431755065918, -1.7704930305480957, 1.6136817932128906, -0.2018100917339325, 2.9304566383361816, 1.3366637229919434, -0.34346073865890503, -2.044691801071167, -1.314211130142212, -0.19745707511901855, 0.9277839064598083, 0.7640114426612854, -1.2285220623016357, -0.8023570775985718, 3.119694948196411, -0.03233975172042847, 0.1522180438041687, -0.09274637699127197, -0.5375566482543945, -0.6812979578971863, -1.7585525512695312, -0.7783117890357971, -2.560140609741211, 0.20805597305297852, 0.740695595741272, 0.00797814130783081, 0.6847788691520691, -0.4958013892173767, 4.724368095397949, -1.207453727722168, 0.8310643434524536, -0.45254549384117126, 2.943476676940918, -1.9723352193832397, -0.42629191279411316, -2.405961751937866, 0.7490085363388062, 0.6594696044921875, -0.19531512260437012, 0.06507156789302826, 2.2227766513824463, 2.759164333343506, 0.046423912048339844, -0.3291414976119995, 1.060179352760315, -1.4690912961959839, -1.7907688617706299, 1.248184323310852, -1.3807986974716187, 0.637281596660614, -0.5490807294845581, 0.47810661792755127, 0.8172087669372559, -1.6260915994644165, -0.8717949390411377, 1.6085692644119263, 1.1871651411056519, 0.7905086278915405, 3.398379325866699, -2.562943458557129, 1.6835527420043945, 1.5570132732391357, -1.9781718254089355, -1.9844322204589844, -1.5995591878890991, 1.3039898872375488, -1.2441604137420654, 0.25662919878959656, 1.0013201236724854, 1.1716450452804565, 1.3890140056610107, -0.02310958504676819, -1.366159439086914, -2.648674726486206, 3.8721961975097656, -1.392440915107727, -1.427893042564392, 2.194498300552368, -1.9433915615081787, -0.4022485911846161, 0.4828227758407593, 2.948160171508789, 1.0359001159667969, 1.108946681022644, 1.2286458015441895, -0.6090295314788818, -0.14406321942806244, -0.45356485247612, 0.4655252993106842, -0.9425360560417175, 0.25308331847190857, 0.644585371017456, -0.44085708260536194, -0.03129368647933006, -0.21071887016296387, -2.4225881099700928, 0.15137876570224762, -0.0897212028503418, 4.091358184814453, 0.3872448801994324, 0.3053431510925293, 0.49258220195770264, 1.1653318405151367, 0.6269522905349731, 0.5814289450645447, -2.111020088195801, 1.195397973060608, 0.8837716579437256, -0.7461535930633545, 0.17989420890808105, -0.6870750188827515, 0.9730134606361389, 0.9652597904205322, 0.8201000094413757, -1.8976385593414307, -0.7101390361785889, 1.2198092937469482, 0.2305218130350113, 2.2418580055236816, 1.3732078075408936, 3.10850191116333, -0.6631109714508057, 0.8704304695129395, -0.03510981798171997, -0.9190913438796997, 0.8350886106491089, -0.1920839548110962, 0.19065377116203308, -2.2806477546691895, 1.0606653690338135, -0.4089830815792084, -1.5539236068725586, -0.25653767585754395, -1.8977503776550293, -0.32256484031677246, 0.2876119017601013, 0.4265093207359314, -0.9356436729431152, 1.5086698532104492, 0.7911278605461121, 0.19579680263996124, 3.5890917778015137, 0.030200734734535217, -0.40303635597229004, 1.2189326286315918, -1.3295782804489136, -0.9676192998886108, 0.24846041202545166, -1.019161581993103, -2.7936902046203613, 0.6197277307510376, 0.842168927192688, 0.07630646228790283, -1.6295757293701172, -0.2304362654685974, -0.7129632830619812, 1.16395902633667, -3.9520182609558105, -1.6624679565429688, -0.0063359737396240234, -0.2740810811519623, 0.8331013917922974, -2.1539089679718018, 2.7227234840393066, -0.09456002712249756, -1.130436658859253, 0.5006385445594788, -1.0027562379837036, -0.6442134976387024, 0.19042176008224487, -1.5337629318237305, -0.24502889811992645, -1.2539914846420288, -0.9781336784362793, -0.4290634095668793, -0.07154437899589539, 1.0215179920196533, -3.3100714683532715, 0.2023879885673523, -1.3390982151031494, 0.5314443111419678, -0.9227621555328369, -0.8300285339355469, -1.4071446657180786, 0.829740583896637, -1.0138375759124756, 1.1214735507965088, 1.9536590576171875, -0.6012728810310364, 2.1682424545288086, 1.3845996856689453, 1.3291938304901123, -1.9001541137695312, 0.6985345482826233, 1.9560014009475708, 2.707953453063965, -1.9320716857910156, -3.0164172649383545, 1.1234608888626099, -1.0052862167358398, 2.215343475341797, 0.28376102447509766, 1.0988554954528809, 0.28884226083755493, 0.2691127061843872, 2.4210119247436523, -0.7742692232131958, -0.6600026488304138, -0.5275362730026245, 1.3689324855804443, -0.7615896463394165, -1.4326978921890259, 0.7031226754188538, 1.4872263669967651, -1.216740608215332, 0.22249922156333923, -2.1252517700195312, 1.5261753797531128, -0.23756635189056396, -1.696638584136963, 0.5653228163719177, -0.6503220796585083, 2.882467269897461, -0.8706912994384766, 0.18177521228790283, 2.050159215927124, 1.0600781440734863, 0.6226211786270142, 0.9397739768028259, -0.17069783806800842, -0.6372144222259521, -0.1320706605911255, 0.9212605953216553, -2.9798054695129395, -1.7102563381195068, -1.6275100708007812, 0.13413085043430328, 0.831490159034729, 1.1090596914291382, -0.17889750003814697, -2.2305352687835693, 2.120216131210327, 0.9503729343414307, 2.158984661102295, -0.08950352668762207, 0.7903905510902405, -0.38655272126197815, 1.2414029836654663, -0.48682957887649536, 0.7084430456161499, 0.32877135276794434, 0.14024779200553894, -0.13486915826797485, -1.9307777881622314, 0.06773011386394501, 1.1986262798309326, -1.9941153526306152, 2.018226146697998, -1.9336388111114502, 1.7390244007110596, 1.1058002710342407, -1.5203869342803955, -0.5594939589500427, -0.9053506851196289, -0.7092213034629822, -0.0034651756286621094, -1.5064973831176758, -0.81178879737854, -2.0672240257263184, -0.3836151361465454, 0.28250908851623535, -1.0795586109161377, -0.5410422682762146, -1.2967214584350586, 1.7592852115631104, 1.0447938442230225, -0.550748884677887, -1.5146145820617676, 3.3623414039611816, 2.4118034839630127, 1.3136775493621826, -2.2200591564178467, -2.051100015640259, 2.0200140476226807, 1.0401002168655396, -0.6436712145805359, 0.07828915119171143, 0.6732209920883179, 0.9080141186714172, 1.0617913007736206, -1.0729492902755737, -2.043849229812622, -1.1917054653167725, -0.7595534920692444, 0.8471043109893799, -0.07208442687988281, -1.4476335048675537, -1.1796236038208008, 2.3415656089782715, -1.1304826736450195, -0.03338715434074402, 0.31152939796447754, -0.5157870650291443, 1.0891247987747192, -0.5380935072898865, 2.649091958999634, 0.7558225989341736, -1.6907261610031128, -0.23189401626586914, 0.4884086549282074, 0.44832366704940796, 1.0318493843078613, 3.5935769081115723, 0.24419596791267395, 0.2819401025772095, -1.3800071477890015, -1.4507238864898682, -0.6816805601119995, 0.9604178667068481, -0.5555827617645264, -0.9448254704475403, -0.026395559310913086, 0.7313846349716187, -2.5029923915863037, 0.9188252091407776, 1.7151954174041748, 0.954567551612854, 0.7060211896896362, 3.352303981781006, 1.7041077613830566, -0.6223476529121399, -3.0408742427825928, -1.1101126670837402, -0.4125063419342041, 1.4463773965835571, 1.4358189105987549, 0.516355037689209, -1.0404826402664185, -1.976455569267273, -1.3016908168792725, 0.6896182298660278, -1.4193015098571777, -0.2909400761127472, 1.8604423999786377, 0.9776297211647034, 1.231620192527771, 0.2117818295955658, 1.8733744621276855, -0.869261622428894, -0.4832170605659485, 1.171990990638733, -2.104302167892456, -0.3165554404258728, -1.669234275817871, 0.4655476212501526, 2.1251895427703857, -1.8760093450546265, 0.06029286980628967, 1.7854692935943604, 2.6519627571105957, 1.2099803686141968, 1.2809665203094482, 1.2119076251983643, -0.9941482543945312, 0.2287723422050476, 1.4567519426345825, 1.2136576175689697, -1.2918815612792969, 2.3526864051818848, 0.5218964219093323, -0.91743004322052, 0.38556355237960815, 1.4273638725280762, -0.5556845664978027, 0.9765535593032837, -0.8231366872787476, 1.1217267513275146, 0.35467728972435, 2.078617572784424, -0.2995343804359436, 0.7858840823173523, 0.45039740204811096, -1.5441854000091553, 0.5202440619468689, -0.26628535985946655, -0.5004979372024536, -0.6270197033882141, -1.5399236679077148, 1.482340931892395, 0.45275264978408813, -1.344672441482544, 1.464041829109192, -0.13783544301986694, 1.249682068824768, -0.9239997863769531, -0.7336773872375488, -0.7000792026519775, 0.15107309818267822, 0.25144293904304504, -2.9902639389038086, -1.4404670000076294, -0.515600860118866, 1.9630143642425537, 0.8913522958755493, -0.7333807945251465, 0.5155253410339355, 0.4407496452331543, 0.14456498622894287, 2.192729949951172, -1.0391385555267334, -0.24520772695541382, 1.1909685134887695, 0.08378338813781738, 1.3129551410675049, 2.170520067214966, -0.8793150186538696, 0.6403117775917053, 1.931618571281433, 2.1045608520507812, -0.618596613407135, 1.6266270875930786, 1.1588562726974487, 0.9277418851852417, -0.792790412902832, 1.0757908821105957, -0.1509108543395996, -0.9352892637252808, -2.354907512664795, 0.38802093267440796, -0.5108949542045593, -1.1900601387023926, -2.461215019226074, -0.7153111100196838, 0.4507948160171509, 1.720734715461731, -0.8496358394622803, -0.5044022798538208, -0.040490031242370605, -1.7112466096878052, -1.3832722902297974, -1.1771060228347778, -2.401963710784912, 0.23725682497024536, -2.142549753189087, 0.43791425228118896, 2.3235936164855957, -1.685126781463623, -0.24754774570465088, -0.3570196032524109, 1.4134529829025269, 0.5033324956893921, 1.638375163078308, -2.014296054840088, -1.1051591634750366, 1.848986268043518, -2.0272035598754883, -0.867752730846405, -1.7845408916473389, 0.5799696445465088, 0.319435715675354, -0.8996654748916626, -0.318084716796875, -3.1226844787597656, -0.4531400799751282, 0.6375090479850769, -0.34685277938842773, 0.30860286951065063, -0.7613065242767334, 1.802674412727356, 1.1134477853775024, -1.1071034669876099, -1.884178638458252, -1.2891947031021118, -0.9006516337394714, 0.633514404296875, -0.5644516348838806, -2.1848442554473877, -1.546291708946228, 1.3173258304595947, -1.4811193943023682, 1.4547903537750244, 0.8894820213317871, -1.3421659469604492, 1.4723315238952637, 3.1917738914489746, 2.068854808807373, 2.0198922157287598, 1.9935920238494873, 1.3102961778640747, -2.715078353881836, 0.6398133039474487, 1.708747386932373, -2.3517684936523438, 0.9182119369506836, 0.7363240718841553, -0.36943113803863525, 0.11006826162338257, 0.6073436737060547, 2.449146032333374, 0.33022022247314453, 0.39250656962394714, -0.3518044948577881, 1.4015337228775024, -0.9374086260795593, -1.1130080223083496, -0.5145297646522522, 0.5453945994377136, -1.7358338832855225, -0.2811703681945801, 1.5118073225021362, -0.7180091142654419, -0.14331462979316711, 0.5853824615478516, -1.5277013778686523, -0.10791575908660889, 0.887105405330658, -0.18952813744544983, -0.9103872776031494, 2.8603947162628174, 0.16343796253204346, -1.3721901178359985, -1.6261913776397705, -1.1011207103729248, 1.6724987030029297, 2.2328102588653564, 0.5249096155166626, -0.9414095878601074, -0.7505462169647217, -3.346900463104248, 0.8274959325790405, 2.0495963096618652, -0.7482746243476868, 1.6638569831848145, 1.0767319202423096, -0.5642639398574829, 1.1050180196762085, -0.5601016879081726, -1.2752141952514648, 0.8278852105140686, -0.35036179423332214, -0.8462074398994446, -1.3634037971496582, 0.09694141149520874, -0.9615755081176758, 0.4232471287250519, 1.8859729766845703, -0.003094315528869629, -0.802854061126709, 1.6037317514419556, -1.4424512386322021, 0.29892563819885254, -0.46564650535583496, 1.02333402633667, -1.6714202165603638, 0.48655542731285095, 2.05199933052063, -0.16464054584503174, 0.17865371704101562, 3.359814405441284, 0.3723871111869812, 1.9863998889923096, 3.400254487991333, 0.4459446668624878, -0.546099066734314, 1.8891726732254028, -1.3789620399475098, -0.8135513663291931, -0.24792122840881348, -0.5841740369796753, -1.061431646347046, -0.594513475894928, 0.27038371562957764, -1.625077724456787, 0.04433131217956543, 1.0405550003051758, -1.6147851943969727, 0.12462693452835083, -0.8228336572647095, -0.59452223777771, 0.40509548783302307, -0.7927007675170898, -0.07992321252822876, -0.23807987570762634, 0.07660915702581406, 0.5217689871788025, -1.3552074432373047, 0.355013906955719, -0.008440792560577393, 0.24527588486671448, -0.4158153235912323, -1.4202220439910889, 1.7410308122634888, -1.2439355850219727, 0.15998125076293945, -1.4863640069961548, -1.8733402490615845, 0.6819071769714355, 0.6243503093719482, -3.2955594062805176, 1.965259075164795, 0.26701655983924866, 1.404860496520996, 1.0477842092514038, 1.775036096572876, -2.3520667552948, -0.5920359492301941, -0.3227131962776184, 1.4055683612823486, -0.491457462310791, 0.3235434889793396, -1.4853777885437012, -0.2181626260280609, 0.8137284517288208, -0.4250578284263611, -1.8105757236480713, 0.32386189699172974, 0.22422060370445251, -0.2507077753543854, 1.9430607557296753, 0.5685995221138, 0.5088380575180054, 3.121565580368042, -0.05630922317504883, 0.16239404678344727, 2.705592632293701, -0.5501233339309692, 0.8200246691703796, 1.4989802837371826, 3.166090488433838, -3.845561981201172, 0.3988170623779297, 1.3121780157089233, 0.12632474303245544, 0.3642956018447876, 0.336748331785202, 0.8661330938339233, -1.3231253623962402, -0.6404532194137573, 1.0908315181732178, -0.7191423177719116, -2.104572296142578, -1.189432144165039, -0.5598559379577637, -1.9694652557373047, 0.8838816285133362, -1.6439650058746338, 1.083409309387207, -1.0712435245513916, -1.158668875694275, 0.015942931175231934, -0.5878465175628662, -1.3490973711013794, -0.5031337738037109, 0.01889824867248535, -3.21132230758667, -0.3115035593509674, -0.8556947112083435, -1.8969814777374268, 1.621523380279541, 0.7377229332923889, -1.0300012826919556, -0.36935657262802124, -0.5710052251815796, -0.7147716283798218, 0.2436712384223938, -1.3510701656341553, -1.487008810043335, 1.532735824584961, 0.6841046214103699, 0.9697948694229126, -2.6983790397644043, 0.07484465837478638, 0.49035030603408813, -2.3366923332214355, -0.23808550834655762, -1.0765900611877441, 0.2428525984287262, -2.3501479625701904, -1.1916413307189941, 1.60320246219635, -0.22321033477783203, -1.3054773807525635, -1.2212882041931152, -0.39983537793159485, 1.1753367185592651, 0.9483076333999634, 0.75188809633255, 0.1989067792892456, -1.0949995517730713, 1.079063892364502, 0.5654284358024597, 0.3623109459877014, 0.27716052532196045, -0.4572795331478119, -2.577338695526123, -0.2682197391986847, 1.4157185554504395, 0.3027219772338867, -1.982201099395752, 1.8331871032714844, 1.2867021560668945, -3.2361769676208496, 2.882889747619629, -3.1387946605682373, -3.4621593952178955, -3.1932926177978516, -0.3302823305130005, 2.502039909362793, -0.08416515588760376, 2.688697338104248, -2.0814101696014404, -1.8350732326507568, 0.3754434287548065, -0.49623164534568787, -1.6415799856185913, -2.0099246501922607, 2.425724744796753, 1.425808072090149, -0.7027174234390259, 2.4815096855163574, 1.0418810844421387, 0.19595924019813538, -3.324488401412964, 0.6599158048629761, -1.1757469177246094, -0.9158294796943665, -1.1001877784729004, -0.08975952863693237, -1.02360999584198, -0.2497325837612152, 1.5431592464447021, 4.573534965515137, -1.787576675415039, 1.7675199508666992, -1.4778611660003662, 0.280267596244812, 0.8979341983795166, 1.5727890729904175, -1.5339449644088745, 1.4186667203903198, -0.08896738290786743, -0.8378407955169678, -1.1729933023452759, -0.6267198324203491, -0.23182371258735657, -0.2947191298007965, 0.7997158765792847, 1.819618582725525, 0.9217501282691956, 0.08863809704780579, -0.9771789908409119, -0.5081710815429688, -2.043912649154663, 1.4680553674697876, -1.3024225234985352, 0.35034090280532837, 0.7179213762283325, 1.4896340370178223, 1.4191031455993652, -0.8058860301971436, 1.1771152019500732, -0.24155613780021667, 1.0475178956985474, -0.2971804738044739, -1.3637627363204956, 0.8336136341094971, -0.5455198287963867, -2.266022205352783, 0.8518509864807129, 0.5672628283500671, -1.4950251579284668, -0.606103777885437, -2.4056854248046875, 1.3572348356246948, -2.7605390548706055, 0.5767320990562439, 0.5594591498374939, -1.8428571224212646, -0.8772883415222168, -0.21689152717590332, -0.8097202181816101, -1.5215156078338623, 1.3671303987503052, 1.0200707912445068, 1.8065447807312012, 0.6952663660049438, -1.417359471321106, -1.1499379873275757, -1.6295512914657593, -0.04214012622833252, -0.14730089902877808, -2.272918701171875, -0.6679863333702087, 1.7455267906188965, -1.0689350366592407, 0.4793168902397156, 0.5126092433929443, 0.2711261212825775, 0.4813985228538513, -0.5624635815620422, 0.3423021733760834, 0.2680877149105072, -0.02733170986175537, 1.3756446838378906, -0.15893197059631348, -0.005065202713012695, 0.06200645864009857, 1.7188899517059326, -2.7304606437683105, -0.2318267822265625, -1.0733932256698608, 1.4493497610092163, 1.3587498664855957, 0.770583987236023, -0.6233806610107422, 1.2432889938354492, -0.12790906429290771, -0.2794700264930725, -0.8555018901824951, -0.8577876091003418, 0.6344122886657715, 2.1200337409973145, 0.07010006904602051, -1.1337597370147705, 0.32461196184158325, 1.448620080947876, -0.8127369284629822, 0.7906721830368042, 1.934594750404358, 1.6262152194976807, -0.5874258875846863, 1.415266513824463, -3.220913887023926, 0.16524586081504822, 0.9070528745651245, 1.3544517755508423, -2.4292500019073486, 0.9512795209884644, 0.11317546665668488, 1.762007236480713, -0.5861437320709229, -2.0190043449401855, 0.9439967274665833, 0.2581707239151001, -4.134451389312744, 2.0678157806396484, 0.2922583222389221, -2.320448875427246, -1.3984090089797974, -0.12260794639587402, 0.24668937921524048, -1.9284056425094604, 1.7314770221710205, 1.450330376625061, -0.037416279315948486, -1.007050633430481, -0.31358006596565247, 0.41632306575775146, -0.47559916973114014, 0.6095577478408813, 0.9910865426063538, 0.9019026160240173, 1.3676084280014038, -1.401883602142334, 1.5484626293182373, -0.8150687217712402, -0.13973772525787354, 1.138024091720581, -1.8210463523864746, -1.8178420066833496, -0.3198387026786804, 0.6316844820976257, -3.0234150886535645, 1.8044588565826416, -0.5687528848648071, 0.3231657147407532, -1.8174893856048584, 2.5980944633483887, 0.6755463480949402, 0.7087420225143433, -2.909700393676758, -0.46172937750816345, 1.8255186080932617, -0.39465510845184326, 0.30230993032455444, -1.7282264232635498, 0.08018195629119873, 0.5141069293022156, -0.9787757396697998, 1.2711448669433594, -1.9951026439666748, 0.8755878210067749, -0.2167457640171051, 1.3160392045974731, 0.6130046844482422, 1.4552438259124756, 2.5724167823791504, 1.8352934122085571, -0.4629373252391815, 0.8670697212219238, -0.1336371898651123, 0.49996280670166016, 2.9221296310424805, 1.0272564888000488, -0.5722613334655762, 1.9423269033432007, 1.760392665863037, -0.3401150107383728, -0.1213768720626831, -1.6539775133132935, 1.4437528848648071, 1.2233467102050781, -2.8161439895629883, -0.6604004502296448, -0.12050354480743408, -1.3123278617858887, 0.6211614608764648, 1.778939962387085, 0.685070276260376, -0.2547147572040558, -0.5044735074043274, -1.1804227828979492, -1.7716695070266724, 1.9000697135925293, 0.5956631302833557, -0.7205156087875366, 1.218198299407959, 0.05757993459701538, -1.3868521451950073, 0.6665031313896179, 6.036623954772949, -1.445823073387146, -0.8786023259162903, -2.0144166946411133, 2.353386878967285, -1.9180858135223389, -2.0270683765411377, -0.06487433612346649, -0.07531929016113281, 0.3534318208694458, 0.854104220867157, -0.8160179257392883, 0.7538278102874756, 1.043847918510437, -0.10306155681610107, 2.154937505722046, 0.25690484046936035, -1.3747607469558716, 0.9136541485786438, 0.39322197437286377, 2.3263187408447266, 0.5809760093688965, 1.6864005327224731, 0.6785131096839905, -0.845041036605835, 0.9727152585983276, 0.5026295185089111, -1.6307846307754517, 0.5385376214981079, 0.6211674213409424, 0.483676016330719, -0.7938729524612427, -0.9694012999534607, 0.725500226020813, 0.43625590205192566, -0.2556334435939789, -0.29842495918273926, 1.0606658458709717, -1.8150527477264404, 0.18361979722976685, 2.283097505569458, -0.4393453598022461, -0.8332161903381348, -0.7351347208023071, 0.03573867678642273, 1.012343406677246, 0.23957252502441406, -3.2566447257995605, 0.44772353768348694, -0.8969993591308594, -0.5784147381782532, 0.7727392911911011, 1.3795807361602783, 0.37826216220855713, -0.06986010074615479, 0.07734605669975281, -3.907425880432129, 2.604881525039673, 0.8577411770820618, 0.5170356631278992, 1.2361611127853394, -1.7349812984466553, 0.37339210510253906, 0.7602862119674683, -0.20371407270431519, -0.4859551787376404, -1.8261903524398804, -1.128176212310791, 0.8314309120178223, 1.235140323638916, 2.609100818634033, 0.4331163465976715, 1.1903936862945557, -0.9807913899421692, -0.7278879880905151, 2.286165237426758, 2.4440758228302, -3.0207042694091797, 2.8226656913757324, 2.1157994270324707, 0.8349707126617432, -0.8591984510421753, -0.16343560814857483, -0.6031980514526367, 2.0350515842437744, -0.609377920627594, -0.3543197512626648, -0.39588499069213867, 2.9407124519348145, 1.7598282098770142, 0.9179452657699585, -0.45879536867141724, -2.6006176471710205, 0.058059513568878174, 0.20144608616828918, 0.9184790849685669, -0.6969453692436218, 0.5155683159828186, -0.35482287406921387, -0.04217362403869629, 1.6371238231658936, 0.9956532716751099, 2.0872926712036133, 1.5515209436416626, 0.06039392948150635, 1.4104022979736328, 1.0175220966339111, 1.2639100551605225, 0.8219505548477173, 0.9011037349700928, -1.0272279977798462, -0.1453084945678711, 2.5401997566223145, -0.20905537903308868, -0.009283065795898438, 0.17414936423301697, -0.4234358072280884, -1.8635072708129883, 0.2904108762741089, 0.37250441312789917, 0.22715970873832703, 0.9490340948104858, 0.6541171073913574, -0.2946709394454956, 0.21682238578796387, -0.21833530068397522, -1.2282969951629639, -0.26747795939445496, 1.7867035865783691, -2.1767539978027344, 1.6513915061950684, 0.014755487442016602, 0.947429358959198, 1.5435091257095337, -0.9245882630348206, 1.0976630449295044, 1.0421130657196045, 1.0488572120666504, -1.1027090549468994, -1.5962594747543335, -1.2440129518508911, -0.674018383026123, 1.3108704090118408, -0.4987417459487915, 0.029836848378181458, -1.4783855676651, 1.6234042644500732, -1.3584599494934082, 1.1362494230270386, 1.02497136592865, 1.4560775756835938, -2.8029603958129883, 2.419565439224243, -1.52933669090271, 0.700488805770874, 0.32238489389419556, -2.2471845149993896, 2.7264044284820557, 1.3277089595794678, 0.9111871123313904, 1.3164762258529663, 0.054846495389938354, 0.4596235156059265, -1.6751677989959717, -2.0850186347961426, 1.9282987117767334, -0.3849732279777527, 0.974313497543335, 1.7427000999450684, 0.06472975015640259, -1.290881633758545, 1.1310912370681763, -2.6202781200408936, -0.4105420708656311, -0.3860187232494354, -0.0711057186126709, 1.5516414642333984, -2.544341564178467, -1.5914974212646484, 1.140411376953125, 1.5285950899124146, -1.2215657234191895, 0.7839898467063904, 0.7179402112960815, 1.199911117553711, -1.0663931369781494, 1.9967776536941528, -0.48660945892333984, 0.9789475798606873, -1.1037083864212036, -1.836690068244934, 2.155038833618164, -2.556647539138794, -1.8252782821655273, 1.927168607711792, 0.15731853246688843, -0.36857280135154724, -0.838147759437561, 1.0334850549697876, -0.28690457344055176, 2.5520105361938477, -0.8868483901023865, -0.957711935043335, 2.4109201431274414, -0.8405952453613281, 0.3132767677307129, 1.7746195793151855, -0.24813240766525269, -0.4937310516834259, -3.2083899974823, 0.8087550401687622, -1.4828060865402222, -0.21918734908103943, 1.634363055229187, -0.5851322412490845, 0.7725357413291931, -1.4544429779052734, -1.731032133102417, -0.036016106605529785, -1.563979983329773, -3.505096197128296, -2.485405683517456, -1.4968994855880737, 1.23771333694458, -1.0326507091522217, 0.2642878293991089, -1.4466285705566406, -0.911756157875061, -1.1115247011184692, -0.9300523996353149, -2.3545069694519043, -0.3712159991264343, -0.5629403591156006, 3.29018235206604, 0.4938045144081116, 0.1992429792881012, -0.8756981492042542, -1.0514742136001587, 0.7097644805908203, -0.6853154897689819, -0.5068186521530151, 0.46434319019317627, 0.06284976005554199, -0.9257015585899353, 2.5088653564453125, -0.8669213056564331, -0.999900221824646, 0.3519856631755829, -0.16236168146133423, -1.1643825769424438, -0.025769740343093872, 1.5258203744888306, 1.3320410251617432, 0.6273913979530334, -0.11795216798782349, 0.6869664192199707, -1.7760648727416992, -1.8172141313552856, -0.3842514753341675, -1.4467931985855103, 1.0136995315551758, 0.9208833575248718, 0.2896888852119446, 0.25404611229896545, -2.0125651359558105, 0.843761682510376, -1.6420851945877075, -2.4294004440307617, 0.5347044467926025, 0.7760529518127441, 0.30190837383270264, -0.7445812821388245, -0.21572071313858032, 1.6555755138397217, -0.7212614417076111, -1.4366991519927979, -1.0671563148498535, -1.3310534954071045, -0.028731465339660645, 0.06964635848999023, -0.4388512969017029, -0.5116348266601562, 0.6637517213821411, -0.6511343121528625, -0.9681205749511719, -0.977238655090332, 1.0643839836120605, -0.8762494325637817, 1.2093589305877686, 1.0623018741607666, 1.2610827684402466, 0.7947777509689331, 0.40909209847450256, -1.0280731916427612, -1.6980894804000854, 0.22837010025978088, 1.6338151693344116, -4.586143493652344, 2.812953472137451, -1.6479885578155518, 1.4603934288024902, 0.2839251756668091, -2.3625762462615967, 2.98213267326355, -0.6933091878890991, 3.2426600456237793, -0.8233983516693115, 2.2496495246887207, 0.15719392895698547, -0.49887436628341675, -1.5035374164581299, -2.2577857971191406, -2.182166576385498, 3.0126395225524902, -1.0150047540664673, 0.16138139367103577, 0.17877483367919922, -1.2104177474975586, 0.4564061760902405, -2.6900978088378906, -0.10640263557434082, -1.6963813304901123, -0.9743871092796326, -0.17949573695659637, -0.6591764092445374, 0.1964956521987915, -0.785758376121521, 0.050690650939941406, 1.3768603801727295, 0.47967320680618286, 0.16533833742141724, 0.29749560356140137, -0.6852220296859741, -1.097243309020996, 1.1037427186965942, -1.260064721107483, 1.1459758281707764, -1.358400821685791, 0.42381709814071655, -0.19457608461380005, 0.7342951893806458, -1.0490930080413818, 1.3267333507537842, -1.2699015140533447, -2.9250025749206543, -0.5160154700279236, 1.5395269393920898, 1.0480802059173584, -1.239353895187378, -0.10406841337680817, -1.4908394813537598, -1.3537615537643433, -0.34948277473449707, -0.2837584912776947, 0.14513945579528809, 0.536130428314209, 0.07296478748321533, 0.543965756893158, 0.953476071357727, -0.6395140290260315, -0.6289979219436646, 1.043445110321045, 0.1697920560836792, 1.3307979106903076, -0.18309347331523895, 0.1100892424583435, 0.8088725805282593, -0.3459722399711609, 1.0213935375213623, -0.34513789415359497, 1.2967818975448608, -1.3727312088012695, -0.7052035331726074, -1.469491958618164, 0.07984364032745361, -1.028717041015625, -0.3117392659187317, 0.8591841459274292, -0.31856077909469604, 0.31032443046569824, 0.20043864846229553, -0.8004980683326721, 1.502805471420288, 0.9625002145767212, 2.9640002250671387, -1.775155782699585, -0.08144485950469971, -0.04571962356567383, 1.2140653133392334, -0.616619348526001, -0.35199499130249023, 1.0336968898773193, 2.302196502685547, -2.105095624923706, 1.1448750495910645, -0.736592173576355, -1.7653151750564575, 0.3990204930305481, 0.9417667388916016, 1.0381875038146973, -2.347187042236328, 1.4707703590393066, -0.8042020797729492, -2.9837536811828613, -1.635022521018982, -0.23591414093971252, 2.2797322273254395, -0.1953028440475464, 0.7326253652572632, 2.02250075340271, -1.4184889793395996, -1.0970691442489624, -1.7890794277191162, -0.42615360021591187, 1.5847156047821045, 0.12796145677566528, 0.11999273300170898, 1.4338488578796387, 0.5576658844947815, 2.2958080768585205, 1.7978512048721313, 0.38353097438812256, -2.037719964981079, -0.7819389700889587, -0.006657838821411133, 0.22314417362213135, 1.5450292825698853, -2.1438093185424805, -0.30077654123306274, -1.5663658380508423, 0.9524090886116028, -1.0740108489990234, 0.2676865756511688, -1.0745716094970703, 0.7022505402565002, -0.23019227385520935, -0.4756193161010742, -2.168011426925659, -1.1187891960144043, -1.5280282497406006, 0.3486388325691223, 1.233945369720459, 0.8306991457939148, 2.8088743686676025, -2.6086530685424805, 0.7120547890663147, -0.2947116494178772, -0.8797776699066162, -1.8142814636230469, -1.6342244148254395, -0.7450253367424011, 2.4539170265197754, -1.500105619430542, 1.6364028453826904, -1.749121069908142, 2.287478446960449, -0.33453720808029175, 0.1498619168996811, -0.924125075340271, -0.40566086769104004, 0.03560879826545715, 0.014212936162948608, -1.392202377319336, -1.2881395816802979, -0.23433570563793182, -0.16177833080291748, -0.8057723045349121, 2.8625540733337402, 0.9403019547462463, -3.015014410018921, -3.1749088764190674, 2.104625701904297, 0.4054569900035858, 1.1148091554641724, 3.025934934616089, -0.6865055561065674, 0.6131975650787354, 1.1229429244995117, 0.3994801640510559, 3.019564628601074, 0.9907265305519104, -0.9538878798484802, -2.733426094055176, -0.8440901041030884, -0.09217424690723419, -2.1325526237487793, 0.19222119450569153, 0.006053086370229721, 0.43059608340263367, -0.696479082107544, -1.6461052894592285, 3.2885494232177734, 2.3518247604370117, 0.33649760484695435, -0.42804914712905884, 0.6190304756164551, -0.5578051805496216, -2.5354795455932617, -1.6764088869094849, 0.9626435041427612, 0.2497934103012085, -0.2689390182495117, -0.1498282104730606, -0.00038120150566101074, 0.2929055690765381, 0.18866395950317383, 0.7412380576133728, -0.13437247276306152, 1.92768394947052, -1.6998298168182373, -0.07266193628311157, 1.443029522895813, -1.2569079399108887, 1.7177351713180542, -0.8561469316482544, 1.10587477684021, 2.390822649002075, -1.1704076528549194, 0.5957561731338501, 0.6708766222000122, 1.6178749799728394, -1.8684790134429932, -1.796011209487915, 0.32152193784713745, -0.0470922589302063, 0.13459081947803497, 0.3138746917247772, 2.3415017127990723, 1.1963430643081665, 1.5613837242126465, 2.266261577606201, 0.21952944993972778, -1.4783964157104492, 2.4194719791412354, 0.7987271547317505, 1.506943702697754, -0.5596810579299927, -0.23952142894268036, 1.423464059829712, -1.343480110168457, 1.521569013595581, 0.4813191890716553, 0.8367847204208374, 2.511652946472168, 1.25493586063385, -1.5029852390289307, -1.295682668685913, 1.8671306371688843, -0.8606297969818115, -0.8884881138801575, 5.930662155151367e-06, -0.10615336894989014, -0.4587348401546478, -0.46341830492019653, -0.317595511674881, 1.1220341920852661, -0.17893223464488983, 1.1821632385253906, -2.507870674133301, -0.5436092615127563, 1.1873466968536377, -0.672478973865509, -0.1780623346567154, -0.379403680562973, -0.4469110071659088, -0.4374271333217621, -2.5890839099884033, 0.32647421956062317, -1.184837818145752, -1.4910415410995483, 0.1310330629348755, 0.9295525550842285, 0.5014941692352295, -0.3860684037208557, -1.2211244106292725, -0.39227280020713806, -0.4525080919265747, -0.228855162858963, 0.6281991004943848, -0.4554969370365143, -2.217587471008301, -1.8226397037506104, -1.7730019092559814, 0.7407087087631226, 2.5761539936065674, 0.2802180051803589, 0.8485003709793091, 3.3878939151763916, -0.42828255891799927, -1.1837410926818848, -1.5282634496688843, -2.8280599117279053, 2.3072710037231445, -0.37626099586486816, -1.4608124494552612, -2.5774624347686768, 2.9733171463012695, -0.5506406426429749, -1.7592384815216064, 0.9690985679626465, 0.30640602111816406, -0.013139903545379639, 0.6382304430007935, 1.787513256072998, -4.007417678833008, -1.1668925285339355, -0.11321088671684265, -0.9269466996192932, 0.24409008026123047, 1.1103147268295288, -1.222725749015808, 3.970651626586914, -0.24960288405418396, 3.3495476245880127, -1.333295464515686, -2.5047013759613037, -0.6991314888000488, -3.081993341445923, -2.1825413703918457, 1.8472776412963867, -0.03717195987701416, -1.9575893878936768, -0.16282397508621216, -3.5206823348999023, 1.0908734798431396, -3.0006115436553955, -0.5243865251541138, -0.23370274901390076, 1.2433340549468994, 0.28492432832717896, -0.44864270091056824, 0.5635644793510437, 0.3249771296977997, -2.12109637260437, -1.2908387184143066, 0.21560513973236084, -1.2477692365646362, 0.4537873864173889, -0.1660526990890503, 0.7349427938461304, 1.3114794492721558, 1.9328489303588867, 1.7550652027130127, -0.09883534908294678, 0.2734333872795105, -2.439283847808838, -2.295881509780884, 1.3661084175109863, 0.2796552777290344, -0.48845213651657104, -0.36094534397125244, -0.2013382911682129, 1.2540624141693115, -1.2407538890838623, 0.5487467050552368, 1.61728835105896, -1.655134677886963, 0.39200830459594727, -1.3163511753082275, 2.439279556274414, 0.044161319732666016, -2.640637159347534, -0.9281465411186218, 0.5205843448638916, 2.1707117557525635, -0.26527220010757446, -3.3052544593811035, -0.6792632341384888, -0.3646111488342285, -0.27105429768562317, 2.887402296066284, -0.8552587628364563, -1.469541072845459, -0.2436002641916275, -0.5375834703445435, -0.6368002891540527, -2.783539056777954, -0.9407845735549927, 1.3894058465957642, 2.918259620666504, -0.7862265110015869, -0.8301771283149719, 0.2076583355665207, 2.483267307281494, -0.9699188470840454, 2.3330483436584473, 0.03386342525482178, 0.3376885652542114, -0.9720905423164368, -0.30252718925476074, 1.9258242845535278, 1.6694960594177246, -2.0930914878845215, -0.6452723741531372, -1.4309685230255127, 0.11289310455322266, 0.09260743856430054, 1.2201734781265259, 2.750025987625122, 0.99961918592453, -1.405163288116455, 0.6168642044067383, -2.7362172603607178, -2.1439602375030518, 1.401016354560852, 2.3189992904663086, 0.8321596384048462, -0.4285099506378174, 1.7398912906646729, -2.1268744468688965, -2.803524971008301, 2.2704954147338867, -0.3288928270339966, 0.5172032117843628, -1.2958405017852783, -0.7233759164810181, -0.024421870708465576, 1.7612990140914917, -0.00995028018951416, 2.278177261352539, -2.0029585361480713, 2.375880479812622, 1.3947737216949463, -0.2978374660015106, -0.9455762505531311, 1.7331105470657349, -2.514986276626587, -0.7691119909286499, -1.4317197799682617, 0.4683383107185364, -0.6317413449287415, -1.5196928977966309, -0.5708090662956238, 1.5260863304138184, 0.1963241696357727, 0.8989753127098083, -0.8982193470001221, -0.7435272932052612, -0.7789328098297119, 0.791888952255249, 1.6929500102996826, -0.1946132779121399, -0.357787162065506, 0.10399597883224487, 0.7010179162025452, -0.841314435005188, 1.7865837812423706, 1.2955493927001953, 1.164009690284729, -1.5593550205230713, 1.1250046491622925, 5.33658504486084, -1.134907841682434, 2.31001353263855, 0.22402001917362213, 0.7812792062759399, 0.858340859413147, -0.7447943091392517, -0.12106622755527496, -1.2418988943099976, -1.334876537322998, -0.19771969318389893, 1.325098991394043, 0.2001170516014099, 0.7582359313964844, 0.8789473176002502, 0.8377138376235962, -0.1015898585319519, 0.18139469623565674, 0.7088476419448853, -1.1154788732528687, -1.1556910276412964, 1.3184499740600586, 0.5739272832870483, -1.384138822555542, 2.6576087474823, 0.6272979974746704, -2.174393653869629, 1.8251123428344727, -2.5010907649993896, 2.2669308185577393, 2.8844690322875977, 1.071250557899475, -1.954606056213379, 0.8750494122505188, 1.340688943862915, -0.33776819705963135, 0.18052886426448822, 0.6599589586257935, 0.13096654415130615, -0.2759474515914917, -1.7933592796325684, 0.21558523178100586, -2.2515599727630615, 2.7794015407562256, -2.1214356422424316, 1.495469093322754, -1.0330517292022705, 1.637044072151184, 1.3146781921386719, 0.007020600140094757, -1.1467552185058594, -1.0946788787841797, -0.906403124332428, -1.0127811431884766, -0.7350313663482666, 0.4414592981338501, 1.2397270202636719, -0.5160381197929382, -0.6687042713165283, 0.5277248024940491, 0.3853391408920288, -0.5619997382164001, 1.0389907360076904, -2.114694595336914, 0.1781369149684906, 1.143786072731018, 0.15036743879318237, -1.6707408428192139, 1.2590768337249756, -1.3129222393035889, 2.993882894515991, -0.17791570723056793, 1.2063153982162476, -2.0811712741851807, -1.6960424184799194, -1.0404140949249268, -0.19817036390304565, -0.3174614906311035, -0.12071334570646286, 1.1383068561553955, -0.2587510943412781, 3.432450294494629, -4.092892646789551, 0.7325104475021362, -0.5237993001937866, 1.1632047891616821, 1.4397839307785034, 0.5005954504013062, -1.939012050628662, -0.7893608808517456, -3.6389780044555664, 0.588825523853302, 0.3311203718185425, 1.978310465812683, 1.0449755191802979, 1.229543685913086, -0.22819870710372925, 1.075647234916687, -2.218801975250244, 0.02559700608253479, 0.6831524968147278, -0.5353773236274719, -0.7015693187713623, 1.9805800914764404, -0.7079024314880371, -0.07077431678771973, 1.1620235443115234, 0.5870458483695984, 1.2472611665725708, 0.4658813178539276, 0.02563399076461792, -1.0034421682357788, 1.0067200660705566, 0.4213985800743103, 2.6350207328796387, 0.6838725805282593, 1.963073492050171, 0.30323851108551025, 1.6365916728973389, -2.526848316192627, 1.237273097038269, -1.8161888122558594, -0.4183652400970459, -3.116589069366455, 1.302569031715393, 0.3875548541545868, 0.9479263424873352, 0.7640148997306824, 1.812964916229248, 1.429633617401123, 1.6896955966949463, 0.5050264596939087, -1.1072465181350708, 0.28352540731430054, 1.066291093826294, 2.949726104736328, 0.3342626988887787, -0.09413406252861023, 0.6880906820297241, -0.7812124490737915, 2.6053366661071777, -0.2722853422164917, -1.0255721807479858, 1.2925294637680054, 0.5892718434333801, 1.7333686351776123, 0.7004101276397705, -1.0429884195327759, -2.383758544921875, 1.2995868921279907, 0.03574615716934204, -0.85391765832901, 0.41596710681915283, 1.1652874946594238, -0.46927064657211304, -1.5128980875015259, -1.8524861335754395, -3.0066213607788086, -0.07070517539978027, -0.695339560508728, -0.9766936898231506, 1.183030128479004, -3.8350095748901367, 1.933635950088501, 0.15276014804840088, -0.44018974900245667, 0.5578808188438416, 0.07364870607852936, -1.9321638345718384, 1.0466200113296509, 0.9570203423500061, 3.982875347137451, 1.3367093801498413, -1.1625635623931885, -2.901275634765625, 0.4606093168258667, 0.7679022550582886, -0.4258583188056946, 0.5015016198158264, 0.8550271391868591, 0.03938129544258118, -0.7473429441452026, -0.11927288770675659, -0.6379904747009277, -1.2392158508300781, -1.5346367359161377, 3.6668667793273926, 0.04455298185348511, 0.0024373531341552734, -0.742301344871521, -0.8416872620582581, 1.5750057697296143, 0.33458858728408813, -0.011966705322265625, 0.751370370388031, -4.854619026184082, 0.1293676495552063, -0.37083500623703003, 3.803429365158081, -1.5569212436676025, 0.7494852542877197, -2.2651448249816895, 0.12438139319419861, -0.27502739429473877, 0.79912930727005, 0.17262232303619385, 1.065171241760254, -1.2342205047607422, 0.46324893832206726, -0.021242499351501465, 0.486051082611084, -0.447918564081192, 1.9043371677398682, -3.037278175354004, -2.1984777450561523, -1.4546691179275513, 1.1963319778442383, 0.2918492257595062, -0.2948063015937805, 0.6737759113311768, -2.3435049057006836, 0.9849363565444946, 0.6400133371353149, 1.5683079957962036, -0.8828655481338501, 2.3062453269958496, -0.14447516202926636, 2.4757838249206543, -0.5475005507469177, -0.34869879484176636, -0.8615567684173584, -0.31507623195648193, -0.17988908290863037, 1.1351003646850586, 0.10298332571983337, 1.1906318664550781, 1.08274245262146, 0.5075559020042419, 1.7923117876052856, 0.2998325824737549, -2.7192487716674805, -0.18998748064041138, -0.8352119326591492, 0.6484591960906982, -0.048877276480197906, -0.35673558712005615, 1.071757435798645, 0.8259849548339844, 0.8388321399688721, -2.879549503326416, -0.9985272288322449, -0.8187039494514465, -3.407968521118164, 0.07983085513114929, 1.6382770538330078, 0.6740734577178955, -0.07525289058685303, -2.022763729095459, 1.1654170751571655, -2.468193769454956, 0.9585498571395874, 1.4607347249984741, -1.952077031135559, -2.6053595542907715, 0.812690794467926, -0.8370983600616455, -1.3062494993209839, -2.1351308822631836, -2.5268707275390625, -1.2602958679199219, 0.861460268497467, -0.9106169939041138], "dtype": "float32"}}, {"a": [{"__nd__": [-0.12054443359375, 0.420654296875, 0.8720703125, -1.34765625, -0.34228515625, -0.81884765625, -1.357421875, 0.689453125, -0.465576171875, 0.6953125, -1.630859375, -2.087890625, -0.06427001953125, -0.10357666015625, 1.453125, 0.359619140625, -1.0615234375, 0.5224609375, 0.2239990234375, -1.0634765625, -0.55078125, 0.00026416778564453125, 0.471923828125, -1.4169921875, 0.8525390625, 0.8125, -0.10650634765625, -1.68359375, 1.654296875, -0.0478515625, 1.0546875, 0.384765625, -2.306640625, -0.128173828125, 1.083984375, -1.42578125, -0.6806640625, -0.208251953125, 1.5029296875, -1.9853515625, -0.525390625, -0.88232421875, -0.78662109375, -1.455078125, 0.1846923828125, 1.0986328125, 0.4208984375, -0.94189453125, -0.62158203125, -0.1990966796875, -1.3974609375, -1.3671875, -0.537109375, 0.37890625, -0.0670166015625, 0.984375, -0.732421875, -0.59716796875, 1.458984375, 0.265625, 1.9443359375, 1.1201171875, 0.67236328125, -0.53173828125, -1.119140625, 0.10003662109375, 0.353515625, 0.91357421875, -1.65234375, -1.7509765625, 0.669921875, -0.103515625, 1.3564453125, -2.302734375, 1.9091796875, -0.1785888671875, -1.7578125, -0.5009765625, -0.76611328125, -1.4052734375, -0.415771484375, -1.3671875, -0.294189453125, 0.822265625, -0.8818359375, 0.038909912109375, -0.1041259765625, -0.515625, 0.58642578125, 0.7998046875, 1.6572265625, -0.818359375, 1.0517578125, 1.056640625, 0.69677734375, -0.37890625, -1.5146484375, 1.67578125, -3.029296875, -0.30029296875, 0.46337890625, 1.8779296875, -1.388671875, -0.390625, -2.162109375, -0.3671875, 0.83984375, -0.1639404296875, -0.259765625, 0.30908203125, -2.650390625, 1.8359375, -1.4599609375, 1.0634765625, 1.3779296875, -1.236328125, 0.63818359375, -0.256103515625, -0.62451171875, 0.026580810546875, 0.75244140625, -0.337890625, -0.00421142578125, 1.189453125, 1.0693359375, -0.53271484375, -1.015625, -0.233154296875, -1.0986328125, -0.5126953125, 0.09765625, -0.22216796875, 0.308349609375, -0.66357421875, 1.0947265625, 0.5400390625, 1.06640625, -1.283203125, -1.0966796875, -0.07489013671875, 1.4580078125, 0.1468505859375, 0.994140625, -0.69091796875, -0.53076171875, 1.0556640625, 0.483642578125, -0.2288818359375, -1.5419921875, -1.724609375, 0.45751953125, -0.339111328125, -0.51416015625, -1.2138671875, -0.8349609375, -0.09185791015625, 1.32421875, 0.16796875, -0.008392333984375, -1.4443359375, 0.2406005859375, 0.294921875, 1.953125, -0.278564453125, 0.1290283203125, -0.73974609375, -0.1739501953125, 0.8525390625, 1.4384765625, 0.509765625, 1.7861328125, 0.8798828125, 0.6044921875, -1.583984375, -1.32421875, -1.8095703125, -0.200927734375, 0.1702880859375, 0.40087890625, -0.10308837890625, -1.3984375, 0.55859375, 0.378173828125, 0.978515625, 0.88916015625, 0.141845703125, 0.76416015625, -1.0654296875, 0.01727294921875, -1.052734375, -0.7470703125, 1.2236328125, -2.84375, -0.3330078125, -0.03460693359375, -0.9794921875, -0.1240234375, -0.18408203125, -0.65966796875, 1.2548828125, 1.337890625, 0.75439453125, 2.46875, 0.2447509765625, -0.283447265625, 0.58935546875, -1.0849609375, -0.43603515625, 1.93359375, 1.9033203125, 0.5087890625, -0.6904296875, 2.29296875, -0.26220703125, -0.88671875, -0.96826171875, -0.93310546875, -1.271484375, 1.3447265625, 0.6298828125, -0.87548828125, 0.340576171875, -0.900390625, 1.349609375, -0.381591796875, 2.11328125, 1.283203125, -0.90576171875, -0.73388671875, -2.8359375, -0.33984375, 0.1998291015625, -0.0594482421875, -1.2646484375, -0.496826171875, 0.214111328125, -1.427734375, 0.2027587890625, 1.17578125, -0.84765625, 0.66943359375, -0.93896484375, -1.7431640625, 1.181640625, -1.015625, 0.40185546875, 0.0011243820190429688, 0.5068359375, -0.51611328125, -0.5830078125, 1.4814453125, -0.6181640625, -0.150146484375, 0.01119232177734375, 1.6220703125, -0.298828125, -1.0869140625, 0.6982421875, 0.6787109375, 1.5830078125, -0.408447265625, -0.44970703125, -1.109375, 3.431640625, -0.1693115234375, -0.353515625, -1.0556640625, 0.54296875, 0.3828125, -1.1513671875, 0.06256103515625, 1.322265625, -1.2099609375, 0.32275390625, 1.146484375, 0.673828125, -1.923828125, -0.272705078125, 0.63232421875, 0.2056884765625, 1.220703125, -0.5830078125, -0.375732421875, -0.8291015625, -0.2254638671875, 0.5009765625, 0.2958984375, -0.72802734375, 0.417236328125, 0.28857421875, -1.748046875, 0.1920166015625, -1.03125, 0.21533203125, -0.97802734375, 0.046630859375, -0.02716064453125, 0.76123046875, -0.365966796875, -1.2021484375, 0.33203125, 0.5107421875, 0.6455078125, -1.55078125, -1.6572265625, -0.4541015625, -0.1507568359375, -0.0513916015625, 0.9140625, -1.53125, 0.00859832763671875, 1.0859375, 0.791015625, 1.7392578125, 0.43896484375, 1.216796875, -0.20703125, 2.279296875, 0.8642578125, -0.260498046875, 2.052734375, 0.91552734375, 0.83935546875, 1.8017578125, 0.334228515625, -0.9736328125, -0.137451171875, 1.1005859375, -0.2125244140625, 0.0938720703125, 1.357421875, -0.1959228515625, 1.755859375, 1.0068359375, 2.130859375, -0.728515625, -0.90478515625, -0.06494140625, -1.248046875, -0.2880859375, 0.2154541015625, 0.35791015625, -2.064453125, -0.2208251953125, 1.0087890625, 0.111083984375, -1.6669921875, -0.67236328125, -0.400634765625, -1.5771484375, -0.420654296875, -0.18896484375, 0.1279296875, -0.96923828125, 0.96923828125, 0.47705078125, 0.65185546875, -0.3349609375, -1.3681640625, -1.0830078125, -0.43212890625, -0.4326171875, 0.43505859375, 0.62890625, -1.9052734375, 1.2890625, 0.36328125, -1.3193359375, 0.423583984375, -0.301513671875, -1.169921875, 0.603515625, 0.1505126953125, 1.0986328125, 0.032257080078125, 1.2578125, -0.60302734375, 0.312255859375, -2.71875, 0.85791015625, -0.86376953125, 0.61181640625, 1.458984375, 0.4814453125, -0.08636474609375, -0.2435302734375, 0.32861328125, -0.413330078125, 0.9306640625, 0.41796875, -0.5078125, 0.133544921875, -1.2880859375, 0.1396484375, 0.1954345703125, 0.0236968994140625, 0.01177978515625, 0.72265625, -3.875, -1.326171875, 0.61669921875, 0.34912109375, 0.9619140625, -1.3212890625, 1.0322265625, 0.2059326171875, -0.298583984375, -0.2587890625, 0.82958984375, 0.4267578125, -0.92529296875, 0.048126220703125, 0.3115234375, -0.28466796875, -0.1519775390625, -1.107421875, 1.1689453125, 0.1563720703125, -0.054931640625, -0.103515625, 0.640625, -0.50341796875, -0.70947265625, 0.7802734375, -1.4208984375, -0.666015625, 0.1690673828125, 1.2900390625, -0.3818359375, 0.5439453125, -0.2919921875, -0.138671875, 0.2308349609375, 1.4033203125, 0.1627197265625, 0.39697265625, 0.35400390625, -0.8330078125, -0.916015625, 1.5869140625, -0.08575439453125, -2.837890625, -0.580078125, -0.005626678466796875, 0.0853271484375, -0.6083984375, 0.61328125, -1.26953125, -0.3720703125, 0.6875, 0.02093505859375, 0.83642578125, -0.1568603515625, 2.2109375, 0.400390625, -0.08837890625, -2.080078125, 0.354736328125, 0.15087890625, 0.5869140625, -0.935546875, 0.94775390625, -0.82763671875, -0.5439453125, 0.08245849609375, -0.60693359375, 0.8525390625, -1.5556640625, -1.5732421875, -0.716796875, 1.380859375, 0.5576171875, -0.880859375, 0.204345703125, 1.427734375, 0.474365234375, -0.08978271484375, 0.414794921875, -1.1220703125, -1.8544921875, 0.1339111328125, 0.234619140625, -0.341796875, 1.6181640625, 1.81640625, 1.7353515625, 0.1014404296875, 1.1728515625, 0.75, -0.68505859375, -0.05841064453125, -0.78369140625, 0.5380859375, 0.8662109375, -2.044921875, -1.486328125, 1.337890625, 0.84912109375, 0.998046875, -1.7578125, 0.67626953125, -1.2451171875, 0.82177734375, 0.04937744140625, 0.62841796875, 1.05078125, -1.84765625, 0.6728515625, -1.427734375, -0.1541748046875, -1.9150390625, -0.560546875, -0.1644287109375, -0.435791015625, -0.02203369140625, -0.72021484375, -1.1650390625, -0.5146484375, 1.3330078125, -0.322998046875, 0.33642578125, -0.5439453125, -1.41015625, -0.13232421875, 0.460693359375, -1.4423828125, -1.5263671875, 1.3994140625, -0.568359375, 0.373046875, -0.188720703125, 0.6875, -1.2744140625, 0.1953125, 0.91455078125, 0.155517578125, 1.05078125, -0.260498046875, 0.2030029296875, 0.378662109375, -0.5400390625, 1.671875, 0.5927734375, 0.63427734375, 0.01274871826171875, -0.619140625, -0.7666015625, 0.31787109375, -1.1767578125, 0.9716796875, 1.994140625, 0.6611328125, 0.900390625, -0.230712890625, 0.367431640625, -1.005859375, -0.396240234375, -1.3369140625, -0.73388671875, -0.28466796875, -0.16455078125, 0.2322998046875, -0.75, 1.1103515625, -1.03125, 0.81396484375, -0.32666015625, -0.62890625, 1.6123046875, 1.357421875, 2.708984375, 1.23828125, 0.266357421875, -1.0400390625, 2.61328125, -0.396484375, -0.6884765625, 0.489013671875, 0.1219482421875, 0.09002685546875, 1.3974609375, 0.1458740234375, -0.381591796875, -2.232421875, -0.388427734375, -1.25390625, 2.541015625, -0.2205810546875, -0.35546875, -0.1820068359375, 1.71484375, 1.076171875, -0.85693359375, 0.01995849609375, -0.4853515625, -0.2354736328125, 1.1064453125, 0.1383056640625, -1.33984375, 0.71826171875, 0.7861328125, -0.036529541015625, -0.484375, -0.34423828125, 1.181640625, 1.70703125, 0.455078125, 0.499755859375, 0.50927734375, 1.7041015625, 1.625, -0.366943359375, 0.317138671875, -0.76416015625, -0.61376953125, 0.5341796875, 1.1552734375, -0.51318359375, -0.09356689453125, 2.34765625, 2.1875, 1.5107421875, -1.6181640625, 1.521484375, 0.1719970703125, -0.0274200439453125, -0.481689453125, 0.11114501953125, 0.90087890625, -0.65185546875, 1.0126953125, 1.3173828125, 0.324462890625, 1.37890625, 0.133056640625, 1.025390625, -0.50390625, -0.08349609375, -1.7373046875, 1.3837890625, 0.296630859375, 0.0200653076171875, -0.185546875, -0.87548828125, -0.49072265625, 0.7421875, -0.1212158203125, -1.0380859375, -1.5380859375, 1.517578125, 0.297119140625, 1.65625, 0.63330078125, 0.01291656494140625, -0.10968017578125, 1.3857421875, -1.6220703125, 2.212890625, -1.603515625, -1.12109375, 0.11651611328125, -0.45361328125, -0.40625, -0.456787109375, -1.091796875, -0.5693359375, -1.4140625, -0.03717041015625, -0.52880859375, -0.6357421875, -0.541015625, -0.806640625, -0.6484375, 0.47119140625, 1.1337890625, 0.921875, 0.47998046875, 0.4716796875, -0.1600341796875, 1.298828125, -0.428466796875, 0.3369140625, 1.13671875, -2.166015625, -0.7197265625, -1.033203125, -0.122314453125, -0.55859375, 0.346435546875, 0.477294921875, -0.051300048828125, 0.3935546875, -0.88037109375, -0.36669921875, 1.23046875, 1.6474609375, -0.0297698974609375, -0.4560546875, -1.1376953125, 0.15185546875, -0.26611328125, -0.427490234375, 0.447021484375, -1.9619140625, 0.232177734375, 2.123046875, -1.02734375, 0.26123046875, 0.1397705078125, 0.1702880859375, -0.7998046875, 0.231689453125, 0.62255859375, -0.77392578125, 0.619140625, 0.049530029296875, 1.6103515625, -0.52099609375, 1.021484375, 1.177734375, 0.67138671875, -2.06640625, 0.8115234375, -0.043731689453125, 0.4853515625, 0.046234130859375, -2.228515625, 0.6845703125, 0.83935546875, -0.322265625, -0.7958984375, -0.646484375, -0.188232421875, -0.38720703125, 0.0281219482421875, -0.413330078125, -1.5, -1.3671875, -1.3232421875, -1.1123046875, -0.2244873046875, -0.387451171875, 0.81396484375, 0.63818359375, -0.5224609375, 0.295654296875, -0.1612548828125, -0.85546875, -1.400390625, 1.068359375, -0.0015411376953125, 0.84375, -1.6123046875, 0.9814453125, -0.625, 0.314453125, 1.9189453125, -0.814453125, -1.6650390625, -1.1015625, -0.46923828125, -0.0281982421875, 0.134521484375, 0.437744140625, 0.1263427734375, 0.41748046875, 0.35546875, 1.6943359375, -0.259521484375, -0.32958984375, -0.29296875, -0.259521484375, 0.11505126953125, 0.1219482421875, -0.447265625, -1.7744140625, -0.3037109375, 0.31396484375, 0.994140625, 0.093994140625, 0.257080078125, -1.2353515625, 1.3330078125, -0.432373046875, -1.51171875, 0.87158203125, -1.3037109375, 0.705078125, 1.0576171875, -1.556640625, -1.4189453125, 1.0634765625, -0.12445068359375, 0.52783203125, -0.61279296875, -0.92431640625, -0.10919189453125, 1.5322265625, 0.837890625, -3.205078125, -1.525390625, -1.265625, -0.56640625, 0.67626953125, -0.330322265625, 0.7421875, -1.296875, -0.67333984375, 0.732421875, 0.002613067626953125, 0.0113067626953125, -0.853515625, -0.79833984375, 1.2998046875, -0.54833984375, 0.53369140625, 0.468505859375, 0.3623046875, -1.552734375, -1.89453125, 2.19921875, 0.12237548828125, 0.32958984375, -1.2333984375, -0.2081298828125, 0.349609375, -1.6953125, 1.6845703125, -0.1986083984375, -1.669921875, -0.6123046875, -1.490234375, 0.110595703125, -0.134033203125, -1.0498046875, -1.2158203125, 1.1767578125, 0.71484375, 1.6728515625, 0.438232421875, -0.152099609375, 0.2763671875, -0.71044921875, -0.85791015625, 0.5263671875, 0.46826171875, -0.51611328125, 1.0615234375, -1.783203125, -0.330078125, -0.67822265625, -0.01309967041015625, 0.12310791015625, 0.79541015625, 0.966796875, 0.361328125, 1.185546875, -0.38916015625, 1.2568359375, 0.87890625, 0.609375, 1.23046875, 0.2471923828125, 0.356689453125, -0.88037109375, -1.1142578125, 1.0966796875, -1.2158203125, 0.55712890625, 2.015625, -0.5927734375, -1.5439453125, 1.3935546875, -0.2269287109375, 0.458984375, 0.281005859375, -1.58203125, 0.217529296875, 0.986328125, -0.4287109375, -1.384765625, -1.0146484375, 1.1162109375, 0.55810546875, 0.03955078125, 0.20703125, 0.4228515625, -0.7490234375, 0.374755859375, -0.032867431640625, -0.19677734375, -1.2421875, -0.1602783203125, 1.7734375, 1.2919921875, -0.302490234375, 0.29296875, -1.380859375, -0.9208984375, 0.3251953125, 0.30859375, 0.591796875, 0.9677734375, -0.21337890625, 0.69873046875, -0.156982421875, -0.7666015625, 1.171875, -0.78466796875, -0.40283203125, -0.826171875, -0.37841796875, 0.456787109375, 0.2734375, -2.45703125, 0.0662841796875, -1.228515625, -1.2958984375, 0.79052734375, 0.448974609375, -1.99609375, 0.302734375, 0.5693359375, 0.955078125, 1.2607421875, 1.4345703125, 0.49169921875, 0.40576171875, 2.283203125, 1.9453125, -0.86669921875, -0.0631103515625, -0.51416015625, -0.57177734375, -1.171875, -0.55908203125, -0.60888671875, -1.830078125, -0.297119140625, 0.43896484375, -0.41552734375, 0.75927734375, -0.177978515625, -1.171875, -1.2978515625, -0.353515625, 3.130859375, 1.1796875, 0.478759765625, -0.08258056640625, -1.0341796875, 0.409912109375, -0.80419921875, -0.86865234375, 0.0958251953125, 1.357421875, -0.470703125, -1.5166015625, -2.044921875, -1.3125, 3.017578125, -1.0107421875, 1.27734375, -0.0160675048828125, -0.95654296875, -0.429443359375, 0.190185546875, 0.71337890625, 0.2548828125, 0.0209503173828125, -1.3955078125, -1.0205078125, 0.38623046875, -1.212890625, 0.81884765625, -1.3642578125, -0.81640625, -0.199462890625, 0.2015380859375, 2.205078125, 0.5546875, 0.64453125, 1.0888671875, -0.39501953125, -0.927734375, 0.8056640625, 1.208984375, -1.7578125, -0.53857421875, -1.462890625, 0.1153564453125, -0.7822265625, 2.212890625, -1.244140625, -2.28125, -0.347900390625, -0.2705078125, -0.17919921875, -0.026458740234375, -0.3759765625, -0.77099609375, 0.264404296875, 1.8583984375, -1.33203125, 0.0670166015625, 0.30224609375, 0.9599609375, -0.1337890625, 0.062042236328125, -0.312255859375, 0.92724609375, 2.21484375, 1.2314453125, -2.048828125, -1.1572265625, -0.1815185546875, -0.227294921875, 0.411865234375, 0.8916015625, -1.1943359375, -0.18896484375, 0.9462890625, -1.171875, 2.302734375, -0.9228515625, -0.2237548828125, -0.8798828125, 1.408203125, -1.458984375, 1.37109375, 0.80859375, -0.0243072509765625, 0.59619140625, 0.1605224609375, -1.4228515625, 0.1923828125, 0.03253173828125, 0.54638671875, 0.2841796875, 0.7216796875, 0.6728515625, -0.194091796875, 0.44970703125, -0.04901123046875, -0.58203125, -0.32958984375, 0.2255859375, -1.263671875, 1.58984375, 1.94921875, 1.611328125, -1.103515625, 2.28125, -0.1429443359375, -0.357666015625, 1.015625, 0.5595703125, 0.947265625, -1.123046875, -0.480712890625, 0.75341796875, -0.29736328125, 0.354736328125, -1.265625, -0.24951171875, -0.76416015625, 0.49951171875, 0.09588623046875, -2.376953125, 0.00562286376953125, 1.0673828125, -0.007190704345703125, -1.4013671875, 1.59765625, 0.2481689453125, 0.81396484375, -1.1240234375, 1.3759765625, 0.11688232421875, 1.1513671875, -0.53271484375, 1.2431640625, -0.81396484375, 0.06707763671875, -0.4443359375, -0.841796875, 1.564453125, -0.18798828125, 0.3505859375, -0.026763916015625, -2.185546875, 0.9072265625, -1.427734375, -0.95068359375, 1.6748046875, 0.95068359375, -1.3076171875, -1.1767578125, -1.2880859375, -0.1993408203125, -0.12481689453125, 0.09625244140625, -0.845703125, -0.52197265625, 0.7578125, 0.52392578125, -1.3857421875, -1.1318359375, -1.0224609375, -0.08538818359375, -0.5947265625, 0.057220458984375, -0.455078125, 0.38134765625, -0.269287109375, -0.73779296875, 0.243408203125, 0.4091796875, -1.72265625, -0.6142578125, -2.072265625, -0.8681640625, -1.7177734375, -1.609375, -0.5068359375, -1.4052734375, 1.8115234375, -0.2239990234375, 0.10009765625, 0.26953125, -0.346923828125, 1.787109375, -0.132080078125, -0.262451171875, -0.70849609375, -0.65087890625, -0.857421875, -0.27490234375, -0.546875, -0.044158935546875, 0.59375, 0.1802978515625, -0.498046875, 0.98486328125, 0.39501953125, -0.475830078125, 0.1390380859375, -0.0595703125, 2.146484375, 0.5751953125, 1.3193359375, -1.74609375, 1.783203125, 1.1328125, 0.3935546875, -0.492919921875, -0.6357421875, 0.72021484375, -2.072265625, 0.251953125, -0.1514892578125, -0.236328125, 2.302734375, 1.623046875, 0.67138671875, -0.54931640625, -1.013671875, -0.051727294921875, -0.0281219482421875, 0.317626953125, 0.70751953125, 1.34375, -1.2841796875, -0.393798828125, 1.3974609375, 0.285400390625, -1.1884765625, -1.2568359375, -1.646484375, -1.056640625, -0.7216796875, 0.31982421875, 0.65771484375, 1.794921875, 0.955078125, -0.69921875, 1.0771484375, 1.345703125, -0.724609375, -0.478515625, -0.6162109375, 0.046112060546875, -0.491943359375, 1.6201171875, -2.201171875, 0.82958984375, 0.82421875, 0.4853515625, 1.6845703125, -0.986328125, 0.47607421875, -0.73828125, 1.0966796875, 0.50244140625, 0.61669921875, -0.06939697265625, 1.7666015625, 0.41845703125, -0.87158203125, -0.163818359375, -2.318359375, -0.70068359375, -0.4658203125, -0.22705078125, -1.5087890625, -0.344970703125, -1.126953125, 0.8671875, 0.677734375, 0.8662109375, 0.80322265625, 0.31787109375, 0.1240234375, -0.6328125, -0.03485107421875, -0.57177734375, -0.6064453125, -0.428466796875, -0.57763671875, 2.05859375, 1.4970703125, -0.96435546875, -0.7666015625, 0.59716796875, -1.5380859375, 1.09375, -0.28955078125, 1.2109375, 0.37744140625, -0.9951171875, -0.55224609375, 0.1419677734375, 1.1796875, -0.452880859375, -1.0283203125, -0.92138671875, 0.166259765625, 0.465087890625, 0.5654296875, 0.8662109375, -0.67138671875, -0.0982666015625, -0.73779296875, -1.6865234375, -0.19091796875, 0.3896484375, 0.5625, 0.349609375, -1.7236328125, 0.12890625, -0.654296875, 0.62060546875, -0.67236328125, -0.5537109375, -0.36865234375, -0.1407470703125, -0.24072265625, 2.171875, 1.2216796875, -0.285400390625, -0.6025390625, -0.6005859375, -0.252197265625, -0.77978515625, 0.68310546875, -0.475830078125, 1.2587890625, 1.203125, -0.7412109375, -0.978515625, 2.18359375, -0.91162109375, -0.81494140625, -0.13134765625, -0.658203125, -1.3671875, 1.005859375, 0.453369140625, 0.348876953125, -0.61962890625, 1.025390625, -0.3603515625, -0.10302734375, -0.377685546875, 0.81884765625, -2.2890625, 0.900390625, 0.513671875, 1.1552734375, -0.05072021484375, 0.87451171875, -0.4384765625, 0.12054443359375, 0.853515625, 1.5576171875, -1.091796875, -1.3076171875, -2.328125, -0.71484375, 0.77783203125, -0.25341796875, -0.36767578125, -0.0011806488037109375, 1.7197265625, -1.6298828125, -0.0341796875, 0.65283203125, 0.8935546875, -1.0830078125, 0.974609375, -1.8984375, 1.322265625, -0.91552734375, -0.28955078125, -0.1910400390625, -1.052734375, 0.517578125, 0.7646484375, -1.302734375, 0.98046875, 0.56298828125, 0.70263671875, 0.2156982421875, 0.449462890625, 1.3515625, 0.546875, 0.1363525390625, -0.60546875, 0.8447265625, -0.2156982421875, -0.79248046875, -0.6533203125, -0.83544921875, 1.296875, 0.6455078125, 0.595703125, 1.818359375, 1.41796875, -1.0439453125, 0.98388671875, -1.138671875, -1.44140625, -0.4658203125, 0.18017578125, 0.0267181396484375, 1.6591796875, -0.0268707275390625, -0.404052734375, 0.414794921875, -0.363037109375, 0.483642578125, -1.197265625, 0.287109375, -0.357666015625, 0.367431640625, 1.1328125, -0.2685546875, -1.548828125, -0.07177734375, -0.373046875, 0.8701171875, 0.04144287109375, -1.3232421875, 2.107421875, 0.6669921875, 0.228271484375, -0.2685546875, -0.69580078125, -1.52734375, 0.06146240234375, -0.58935546875, 0.4970703125, 1.4306640625, 0.260009765625, -0.26708984375, 0.050567626953125, 1.228515625, -0.9013671875, 0.39111328125, -2.287109375, -0.296142578125, 0.765625, 0.043487548828125, 0.453369140625, 0.038055419921875, 1.630859375, 0.26123046875, 0.70654296875, 0.162841796875, 0.39501953125, -0.17724609375, 1.01171875, 1.1279296875, -0.2392578125, -0.421875, -0.406005859375, 0.62890625, 0.58154296875, 0.171875, -0.1265869140625, -1.4775390625, -1.4140625, -1.9189453125, 0.212646484375, -0.67431640625, 1.3212890625, 0.152099609375, -0.73486328125, -0.61279296875, 0.8359375, -0.276611328125, -0.11297607421875, 1.3251953125, -0.58056640625, 1.3134765625, -1.1064453125, 0.08245849609375, 0.634765625, 0.5361328125, -0.69677734375, -0.318359375, -0.94140625, -0.04901123046875, -0.429443359375, -0.1251220703125, -0.035675048828125, 1.2607421875, -0.9599609375, 0.51220703125, -0.826171875, 1.021484375, 0.369873046875, 0.45068359375, -0.96435546875, -0.4638671875, 1.4921875, 1.1064453125, 0.5908203125, 1.7080078125, -1.0068359375, -0.470947265625, 0.2078857421875, 0.496826171875, -0.37353515625, 0.2091064453125, -1.5361328125, 0.082763671875, -0.8505859375, -0.058746337890625, -0.39208984375, -0.57958984375, -1.7880859375, 0.55322265625, 0.43310546875, -1.3642578125, -0.759765625, 0.5146484375, -0.78369140625, 0.72998046875, -1.9404296875, 0.630859375, 1.48046875, -0.615234375, -0.533203125, -1.337890625, -1.076171875, -1.6025390625, -0.06500244140625, 1.3203125, -2.576171875, -0.79150390625, 1.091796875, 0.62353515625, -0.1531982421875, 2.43359375, -0.26611328125, 0.755859375, -0.62548828125, 0.4248046875, -0.82373046875, -1.2509765625, 0.1160888671875, -0.439453125, 0.8876953125, 0.6123046875, 1.439453125, 1.1201171875, -1.1171875, 0.32568359375, 0.1925048828125, -1.421875, -2.123046875, -0.29833984375, 0.89404296875, 0.6572265625, -1.3134765625, -1.013671875, -1.591796875, 0.74462890625, -0.02008056640625, -0.1236572265625, -0.8603515625, -0.630859375, -0.50830078125, -2.265625, -1.05859375, 1.4873046875, 0.88330078125, -0.73779296875, -0.460693359375, -0.392333984375, -0.036376953125, 0.127197265625, -0.65478515625, 0.0682373046875, -0.4365234375, 0.435546875, -0.845703125, 2.384765625, 0.939453125, 1.5966796875, 0.40576171875, -0.407958984375, -1.3251953125, -0.73193359375, -1.1201171875, 0.336181640625, 0.5517578125, 1.2919921875, 0.168212890625, 0.55224609375, -0.2330322265625, 0.32275390625, -2.25390625, 0.6279296875, -2.119140625, -0.410400390625, -1.275390625, -0.1251220703125, -0.258544921875, -0.447998046875, 0.7978515625, -0.1630859375, -0.88427734375, -0.77978515625, 0.77099609375, -0.583984375, -0.377197265625, -0.74755859375, -0.41796875, 0.52734375, -0.1925048828125, -1.2265625, -0.446533203125, 1.087890625, -0.69580078125, 0.2008056640625, 0.60693359375, 0.272216796875, 0.279541015625, 1.630859375, 0.1099853515625, -0.62451171875, -1.1728515625, -0.634765625, -1.9912109375, 0.5380859375, 1.1572265625, 0.09014892578125, -1.9794921875, 1.328125, -0.371826171875, 0.437744140625, 1.3515625, 0.7001953125, 0.2275390625, 1.009765625, 0.78515625, 0.912109375, -0.55322265625, -0.525390625, 0.1502685546875, 2.697265625, 1.0283203125, 0.156982421875, 1.0185546875, 0.2066650390625, 0.77685546875, -0.08294677734375, 0.74560546875, -1.2080078125, 0.97412109375, -1.068359375, -0.11236572265625, 1.54296875, 0.9130859375, 1.8154296875, -0.5048828125, -0.85546875, 1.89453125, 0.144287109375, -0.61962890625, -1.7099609375, -0.318603515625, -0.41015625, 0.5087890625, -0.273681640625, 0.359375, 0.433837890625, 0.099609375, 1.3466796875, -0.81982421875, -0.282958984375, 2.87890625, -0.69482421875, 1.54296875, 0.042449951171875, 1.0390625, -0.050811767578125, 2.234375, 0.27294921875, -0.734375, -0.56787109375, 1.0595703125, 0.85546875, 0.7666015625, -0.216552734375, 1.412109375, -0.310302734375, -0.8017578125, 0.6181640625, -0.826171875, 0.220703125, 0.814453125, -0.01090240478515625, 0.6064453125, 0.027984619140625, -1.3408203125, 0.328369140625, -0.2314453125, -0.1851806640625, -1.595703125, 0.16748046875, -0.79345703125, -0.94580078125, 0.91259765625, -1.8857421875, 0.0736083984375, 0.38818359375, -0.8955078125, -0.08978271484375, -1.357421875, 0.58935546875, 0.32958984375, -0.1783447265625, 0.841796875, -0.451904296875, 0.05657958984375, -1.3037109375, -0.25537109375, 1.1123046875, 0.0970458984375, -0.178466796875, 1.2119140625, 0.92626953125, 0.7724609375, -0.5078125, 0.02410888671875, -0.8857421875, 0.034759521484375, -0.87890625, -0.9921875, -1.025390625, -2.572265625, -0.250732421875, 0.86669921875, -2.228515625, 0.1292724609375, -1.166015625, 2.388671875, 0.857421875, -1.005859375, 0.5302734375, 0.0018587112426757812, 0.62939453125, -1.3857421875, -0.57373046875, 0.2391357421875, 0.9775390625, -3.384765625, -1.1904296875, 0.385986328125, -0.08990478515625, 0.4921875, -0.387939453125, -0.464599609375, -0.3505859375, 0.6201171875, 0.08416748046875, 1.1162109375, -0.009429931640625, 0.186767578125, -1.318359375, 1.330078125, 2.673828125, -0.395751953125, -0.69482421875, 1.2783203125, 1.1279296875, 1.123046875, -0.7734375, 0.70654296875, -0.35595703125, -0.552734375, 0.626953125, -1.021484375, -0.55419921875, 0.1290283203125, 0.6015625, 1.0302734375, -1.1474609375, -2.091796875, -0.48828125, 0.5068359375, 0.654296875, 1.8701171875, -0.417236328125, 0.6162109375, 0.54443359375, -0.51953125, -0.56005859375, -1.560546875, -1.232421875, -2.08203125, -0.5576171875, 0.677734375, 1.8046875, 0.96826171875, 0.248291015625, -0.998046875, 1.5908203125, -0.323486328125, -0.916015625, 0.96337890625, -1.7470703125, 0.72900390625, 0.388427734375, 0.81689453125, -0.34130859375, 0.468505859375, 0.43798828125, -1.654296875, -1.484375, 0.64306640625, -0.630859375, 2.1484375, -0.207275390625, -0.07049560546875, -0.1163330078125, -0.4775390625, -1.0283203125, -1.3115234375, -0.140625, 0.90625, -0.7958984375, 1.9248046875, 1.390625, -0.470703125, 1.923828125, 0.71142578125, -1.1474609375, 0.14501953125, -0.6259765625, 1.404296875, -0.374267578125, -0.5283203125, 0.2442626953125, -0.51025390625, -0.11187744140625, 0.48779296875, 2.142578125, 0.444091796875, 1.306640625, 0.453369140625, 0.068603515625, -0.432861328125, 0.51171875, -0.256591796875, 1.1552734375, -0.104736328125, -0.83740234375, -1.8203125, 0.91064453125, -0.5546875, -0.771484375, -0.44775390625, -2.021484375, -1.00390625, 0.210693359375, -0.0141448974609375, -1.4140625, -0.5390625, 1.14453125, -1.7119140625, -0.50439453125, 0.33154296875, 0.0352783203125, -1.8154296875, 0.36328125, 0.94384765625, 0.63134765625, -1.291015625, -1.453125, 0.990234375, 0.28515625, 0.458740234375, 0.69677734375, -1.18359375, -0.28515625, 1.0712890625, -1.2197265625, -1.1298828125, 0.6083984375, 1.2138671875, 0.81787109375, 0.254638671875, -0.62060546875, 0.0303192138671875, 0.431640625, -0.024017333984375, -0.264404296875, 0.307861328125, 0.7021484375, 1.5478515625, -0.63818359375, 2.263671875, 0.06341552734375, -0.41650390625, 1.0087890625, 0.436767578125, 1.7607421875, 1.2431640625, -1.568359375, 0.046905517578125, -1.0048828125, -0.24609375, -1.26171875, 1.33203125, 0.1295166015625, -0.498046875, 1.2744140625, -1.6494140625, -1.4775390625, -0.04998779296875, -0.83056640625, -0.005725860595703125, 0.1668701171875, -0.345458984375, 0.2158203125, -1.541015625, -0.53662109375, 0.2666015625, 0.4345703125, -0.8525390625, -0.53759765625, -0.57763671875, 0.365234375, -0.78515625, -0.8359375, 0.09002685546875, -0.2998046875, -0.99072265625, -0.09088134765625, 0.73876953125, -1.0615234375, 0.810546875, 0.51513671875, -1.5107421875, -0.005146026611328125, -0.403076171875, -0.62548828125, -1.7138671875, 1.5498046875, -0.1622314453125, 0.1693115234375, 0.451171875, -0.302490234375, 0.7841796875, -0.5322265625, 1.091796875, 0.806640625, 0.42333984375, -0.16552734375, 0.91748046875, 1.7197265625, 1.6044921875, -0.341552734375, -1.6025390625, 1.7705078125, -1.31640625, 0.1278076171875, -2.193359375, 0.63818359375, 0.12237548828125, 0.61376953125, -0.045989990234375, -0.214599609375, 0.00794219970703125, -0.60400390625, -0.86376953125, -1.2294921875, 1.833984375, 0.11822509765625, -1.1728515625, -1.099609375, -0.90576171875, 0.70458984375, -0.77197265625, 0.9677734375, -1.0107421875, -1.0546875, -0.572265625, 1.5712890625, 0.47314453125, 0.216796875, -0.2452392578125, 0.546875, 0.5322265625, 0.71875, -0.703125, -0.1607666015625, -1.4189453125, -2.162109375, -0.2232666015625, 0.9375, -0.356689453125, -1.103515625, -1.783203125, -0.8291015625, -0.650390625, 2.244140625, 1.4423828125, -0.328369140625, -0.307373046875, -0.75537109375, 0.6904296875, -1.3515625, 0.5712890625, -0.7646484375, -0.99462890625, -0.7109375, 0.442626953125, 0.58203125, -0.26953125, -0.0888671875, -0.31787109375, -1.1328125, 1.1044921875, 0.705078125, 1.1123046875, -1.4482421875, -0.826171875, -0.0877685546875, -1.302734375, 0.4521484375, 0.2177734375, 1.484375, -1.75, -2.51171875, -1.7587890625, 0.475830078125, -0.200927734375, 0.65869140625, -1.1953125, 0.59521484375, 0.0185699462890625, 0.94775390625, 0.359130859375, 0.0797119140625, -1.357421875, 1.3076171875, 0.192626953125, -0.412841796875, -0.787109375, 0.90087890625, -0.6591796875, 1.619140625, 0.66259765625, -0.2164306640625, -0.38818359375, 0.5693359375, 0.219970703125, 1.6484375, 0.7080078125, 0.048126220703125, 0.78369140625, -1.423828125, 1.1337890625, 0.74462890625, -0.240234375, -0.55615234375, 1.5791015625, -1.330078125, -0.3193359375, 0.307861328125, 1.4501953125, -0.047210693359375, -1.2197265625, 0.255859375, 1.08203125, -1.205078125, 0.62451171875, -2.134765625, 0.23779296875, -1.2490234375, -0.484375, 1.04296875, 0.70458984375, 0.023468017578125, 1.2158203125, 1.931640625, -0.310546875, 0.93603515625, -0.4638671875, 0.67041015625, 0.98974609375, 1.0849609375, -0.82568359375, -0.17333984375, -0.11724853515625, 1.7958984375, 0.09613037109375, -1.2333984375, 0.80615234375, 0.300537109375, -0.5966796875, -1.3671875, -1.1923828125, 1.759765625, 0.7119140625, 0.00521087646484375, 1.1484375, -0.87451171875, 0.16650390625, -1.8828125, 0.03753662109375, 0.10693359375, 1.373046875, 0.8642578125, 0.57666015625, -0.87353515625, -0.84130859375, 0.138916015625, 0.427490234375, -0.334716796875, -0.282470703125, 0.0159912109375, 2.251953125, 1.4033203125, 1.2451171875, 1.4716796875, 0.9619140625, -0.4052734375, -0.66845703125, -1.4287109375, -0.6279296875, -0.383544921875, -0.347412109375, 0.204833984375, -0.880859375, 1.4345703125, 1.2421875, 0.01212310791015625, -1.458984375, 0.916015625, 0.90625, -0.464111328125, 0.359130859375, 0.34716796875, 0.1815185546875, 0.9296875, -1.46875, -0.68017578125, -1.580078125, 0.95263671875, 0.5537109375, -1.857421875, 0.10113525390625, -0.33203125, -0.2418212890625, -0.451416015625, 0.66162109375, 0.84033203125, -0.32470703125, 1.345703125, -0.814453125, -0.595703125, -2.0625, 0.28564453125, 0.2392578125, -0.79150390625, 1.0, -0.18310546875, -1.4287109375, -0.904296875, 0.004970550537109375, 0.29248046875, 0.35595703125, -0.59130859375, 1.4931640625, 1.837890625, 0.84326171875, -1.23828125, 0.15185546875, 0.658203125, -0.37109375, -1.7421875, -1.873046875, 1.52734375, -1.51171875, 0.401123046875, 0.260986328125, 1.2099609375, 0.82177734375, 0.061309814453125, 1.33203125, 2.587890625, 1.078125, -0.46435546875, 1.201171875, 0.0928955078125, -1.6796875, 0.5244140625, -0.97998046875, -1.0458984375, 0.06878662109375, -0.51416015625, -1.2255859375, 0.313232421875, 1.1630859375, -1.59765625, 0.736328125, 0.73046875, -0.63525390625, -0.281494140625, -0.65673828125, -0.4501953125, -0.2335205078125, -0.353759765625, -1.2216796875, -2.20703125, -0.07135009765625, -1.7626953125, -2.484375, 0.935546875, 0.498046875, -0.08135986328125, -0.3779296875, 0.826171875, 0.194580078125, -0.2237548828125, 0.373779296875, 1.5234375, 1.185546875, 1.392578125, -0.62646484375, 0.387939453125, 0.48291015625, 0.44873046875, -1.05078125, 0.818359375, -0.09979248046875, 0.426025390625, -1.3701171875, -0.30859375, -0.04522705078125, 0.78125, 1.1572265625, 0.07958984375, 0.6298828125, 0.1575927734375, 0.98388671875, -0.06671142578125, -0.7978515625, -0.63916015625, 0.281494140625, -0.63818359375, -0.72802734375, -0.219970703125, 0.6806640625, -0.293212890625, 1.0732421875, 1.8330078125, -0.00792694091796875, 1.2373046875, -0.31884765625, -0.43505859375, -0.299072265625, 0.2391357421875, 2.236328125, 1.953125, 1.671875, -0.1556396484375, 0.62939453125, -0.90283203125, 0.263916015625, -0.2001953125, 1.3076171875, -0.15576171875, 1.517578125, -0.338623046875, 0.89599609375, -0.206787109375, -0.724609375, -0.12890625, 2.623046875, 0.671875, 1.1220703125, 0.626953125, 0.52978515625, -0.480224609375, -0.328369140625, -0.0684814453125, 1.939453125, -1.322265625, -0.982421875, -0.78466796875, 0.65869140625, -0.52783203125, -0.86572265625, -1.7587890625, 1.3818359375, -1.650390625, 0.470458984375, -1.4462890625, -0.007488250732421875, -1.2314453125, -0.448974609375, -0.74462890625, -0.5888671875, -0.68408203125, 0.06640625, -0.99755859375, -0.10113525390625, -1.7734375, -0.1434326171875, -0.3212890625, -1.2177734375, 0.9033203125, -0.03277587890625, -0.2646484375, -1.25, 0.7490234375, -0.427490234375, 0.160888671875, -0.400146484375, 0.4033203125, -0.4404296875, -0.08551025390625, 0.3984375, 1.15234375, -0.3974609375, 1.7236328125, 0.2088623046875, 1.7666015625, -0.018524169921875, -0.41015625, -0.470947265625, -0.272216796875, 0.62841796875, 0.55029296875, -0.90771484375, -1.6396484375, -0.75830078125, -0.1475830078125, -0.919921875, -0.397216796875, -0.1600341796875, -0.489990234375, -0.3134765625, 0.822265625, 0.290771484375, 0.69580078125, 1.29296875, 0.9404296875, -1.134765625, -1.7890625, -0.556640625, -0.57861328125, 0.52685546875, 0.47900390625, 0.6337890625, 1.470703125, -0.08612060546875, 0.317138671875, 0.80322265625, -1.005859375, 0.462646484375, -2.255859375, 0.1590576171875, 0.6328125, -0.912109375, -0.484130859375, 0.39111328125, -0.2265625, 1.013671875, -0.28466796875, 0.11712646484375, -0.272216796875, 0.4951171875, -0.45654296875, -1.1865234375, 0.1346435546875, 0.023773193359375, 0.75634765625, 1.4384765625, -0.25341796875, -1.3046875, 1.1787109375, -1.984375, -0.68798828125, -1.203125, 0.8701171875, -0.61669921875, 0.86865234375, -1.76171875, 0.65087890625, -1.1943359375, 0.69873046875, 0.37109375, 0.004058837890625, 0.9345703125, 0.169189453125, -1.013671875, -1.6572265625, -0.26904296875, 0.68408203125, 0.025970458984375, 0.0054473876953125, 0.4296875, -1.19140625, 1.41015625, -0.67236328125, -0.64599609375, -0.814453125, 2.09765625, 0.63232421875, -0.55126953125, -1.16015625, 0.126953125, 0.19775390625, 1.013671875, -0.7236328125, -0.98486328125, 0.42626953125, -1.533203125, 0.1839599609375, 0.2396240234375, -1.9375, -0.96337890625, 0.221435546875, 0.417724609375, -2.291015625, -0.085205078125, 0.56494140625, -0.1688232421875, 0.37451171875, 0.5166015625, 0.884765625, 1.09375, 1.0869140625, 1.6533203125, -0.18017578125, -0.65576171875, -1.029296875, 0.82666015625, -1.796875, -0.300537109375, 0.1055908203125, -0.499267578125, -1.84765625, -0.98779296875, -1.78515625, 0.33544921875, 1.833984375, 0.58056640625, 1.3017578125, -0.0701904296875, -0.3154296875, 0.1123046875, 0.265625, 0.1695556640625, 0.712890625, 0.435302734375, -0.572265625, 0.177490234375, -1.1962890625, -0.46044921875, 1.9716796875, -0.52978515625, -0.360107421875, -0.61767578125, -0.1873779296875, -0.685546875, 0.46728515625, 0.5439453125, 1.568359375, -0.424072265625, 0.82080078125, -0.177734375, 0.611328125, 0.0179290771484375, 0.6181640625, -0.01399993896484375, 0.09954833984375, -1.615234375, -1.453125, 1.41796875, 0.3056640625, -0.89013671875, -1.0791015625, -0.296875, 1.4697265625, 1.271484375, 0.82666015625, 0.6240234375, -0.285888671875, -0.063232421875, 1.029296875, 0.07244873046875, -1.34375, -0.039031982421875, 0.5087890625, -1.6875, 1.2939453125, -0.79296875, -0.297119140625, -1.4404296875, -0.90283203125, -0.8359375, -0.076904296875, 0.302978515625, -0.08056640625, 0.408203125, -0.71533203125, -1.525390625, 0.269775390625, 0.66845703125, -0.344970703125, 1.576171875, 1.0185546875, 0.7509765625, 0.1004638671875, -0.0794677734375, 1.7353515625, -0.95556640625, 1.2451171875, 2.041015625, 0.18994140625, -2.23046875, 0.53173828125, -0.0168304443359375, -0.63818359375, 0.60302734375, 0.62255859375, -0.892578125, -0.058929443359375, -1.0859375, 0.7451171875, -1.2255859375, -0.92431640625, -1.890625, 0.54052734375, -0.7783203125, 0.055633544921875, -1.72265625, 0.56298828125, 1.6552734375, -0.2086181640625, 0.49072265625, -1.6015625, -0.81201171875, -0.2476806640625, -0.374755859375, -0.52734375, 1.732421875, -0.298095703125, 1.7724609375, 0.041168212890625, 0.257080078125, -0.59033203125, 1.2275390625, -0.1309814453125, -0.84130859375, 0.953125, 0.61474609375, -0.08160400390625, -0.3076171875, 1.51171875, 0.324951171875, 0.375244140625, -1.1376953125, 1.1123046875, -0.548828125, 0.90380859375, 0.51171875, 0.7783203125, 0.1800537109375, 0.72216796875, 0.6005859375, -0.64111328125, -1.15625, 1.3017578125, -1.8408203125, -0.99609375, 0.460693359375, 0.282958984375, 1.58203125, 0.09307861328125, -1.0009765625, 0.332275390625, -0.359619140625, 0.250732421875, -0.509765625, 0.79736328125, -0.66357421875, -0.01444244384765625, 0.70361328125, 1.419921875, -0.142333984375, 1.525390625, 0.9462890625, 0.43798828125, -0.6083984375, 0.49267578125, -1.875, -1.3095703125, -0.76611328125, -1.51953125, 2.86328125, -0.83203125, 0.9287109375, 1.177734375, -0.4365234375, -1.9287109375, 0.026123046875, 2.001953125, -0.1790771484375, 0.5380859375, 0.422119140625, -0.5673828125, -2.099609375, -0.607421875, 1.369140625, 1.2890625, -0.744140625, 3.11328125, 0.6298828125, 0.92431640625, -0.1468505859375, 0.08221435546875, 0.50341796875, 0.10052490234375, 0.67822265625, 0.0865478515625, -0.393798828125, -1.0205078125, -1.703125, -1.7509765625, 1.0625, 0.6396484375, 0.29931640625, -1.16015625, 1.3828125, -0.10308837890625, 1.712890625, -0.376953125, -1.197265625, 0.05401611328125, 1.130859375, -0.498046875, -1.439453125, -0.33544921875, 1.0458984375, -1.2529296875, -0.873046875, 0.87744140625, 1.744140625, -2.109375, 0.4248046875, 0.55810546875, -0.5986328125, 1.49609375, -0.29833984375, -0.6640625, -1.0302734375, -1.0341796875, -0.75732421875, 0.72509765625, 1.5888671875, -1.5869140625, 0.12060546875, 0.313232421875, -0.11090087890625, 1.51953125, 0.6787109375, -0.599609375, 0.1976318359375, -0.73046875, -0.95068359375, 0.46142578125, -1.64453125, 0.2841796875, -1.2265625, 2.404296875, 0.88427734375, -0.066162109375, 1.7548828125, -0.09381103515625, -0.8994140625, 1.3837890625, 0.389404296875, 0.61865234375, -1.77734375, -0.042022705078125, 0.2193603515625, 1.7421875, 0.68994140625, 0.935546875, -1.5517578125, 0.37939453125, -0.2939453125, -0.469482421875, 1.5, 1.158203125, -0.297607421875, -0.0210723876953125, 1.2158203125, -0.06756591796875, -0.4853515625, 0.76513671875, -1.9052734375, 0.6142578125, -1.6015625, -0.194091796875, 1.388671875, -0.53125, 0.91796875, -1.3203125, 0.1634521484375, -0.206298828125, -1.4990234375, 0.0797119140625, 0.2230224609375, -0.441162109375, 0.341552734375, 0.1220703125, 0.447509765625, -0.112060546875, 0.00033473968505859375, -0.2919921875, 1.6083984375, -0.1307373046875, -0.00836944580078125, 0.2132568359375, 0.06396484375, 1.6748046875, -1.39453125, -0.63330078125, -1.64453125, 0.1536865234375, -0.916015625, -1.404296875, 0.97705078125, 0.022216796875, 0.1619873046875, -2.064453125, -2.248046875, 0.79345703125, -0.75, 0.91455078125, -0.284423828125, -1.2197265625, -1.259765625, 0.89404296875, 1.20703125, 2.06640625, 0.0322265625, -1.6611328125, -0.5546875, -1.4619140625, -0.78515625, -0.77685546875, -0.7744140625, -0.473876953125, -0.18310546875, -0.35205078125, 1.2373046875, -0.87548828125, -0.01428985595703125, -0.7177734375, -0.2235107421875, -0.4794921875, -1.1201171875, 0.5830078125, -0.88916015625, 1.5732421875, -1.1044921875, -0.444580078125, -0.01473236083984375, 1.05859375, -0.7763671875, 1.5068359375, -0.5830078125, -0.348388671875, -0.8544921875, -0.053070068359375, -1.12890625, -0.0283355712890625, -0.8291015625, -3.36328125, -0.53125, -1.78515625, 0.332763671875, 0.74951171875, -0.59375, 0.10992431640625, 1.05078125, -0.30908203125, 0.388671875, -0.9091796875, 0.9794921875, -0.1097412109375, -1.0224609375, 0.1326904296875, 2.375, 0.29248046875, 1.5029296875, 0.1148681640625, -0.030609130859375, -0.2037353515625, 3.146484375, -0.75732421875, 0.15625, -2.607421875, -1.708984375, -0.09619140625, -0.7216796875, -0.427001953125, 1.197265625, 0.83154296875, 0.1741943359375, 0.292236328125, 0.83251953125, 0.1195068359375, 0.6123046875, -0.458984375, 0.37060546875, -0.232177734375, 1.5908203125, 0.491455078125, 1.35546875, -2.810546875, -0.80615234375, -0.360107421875, -0.455322265625, 0.62451171875, -1.3125, -0.04547119140625, 0.8203125, 0.383544921875, -1.2763671875, 0.181884765625, 0.46728515625, -0.42919921875, -0.373291015625, 0.865234375, -0.2083740234375, -0.483154296875, 1.0283203125, -0.3896484375, 0.1287841796875, -0.054168701171875, -0.953125, -0.640625, -0.478515625, -1.2421875, 0.67138671875, -1.205078125, -0.44677734375, -0.54296875, 0.393798828125, -0.308837890625, -0.08062744140625, -0.322509765625, 0.62548828125, -1.0166015625, 0.68798828125, -0.482177734375, -0.1436767578125, 1.0439453125, -1.4228515625, -0.296142578125, 0.07781982421875, -0.259765625, 0.2183837890625, -1.2705078125, -0.1688232421875, -0.5439453125, 1.0439453125, 1.7841796875, -0.57373046875, -0.329833984375, 0.226318359375, -0.035552978515625, 0.6826171875, -0.09136962890625, -1.4326171875, -0.273193359375, 1.2880859375, -1.7060546875, 1.6162109375, 0.74560546875, -1.7333984375, 0.67138671875, 0.0999755859375, 0.50439453125, -0.734375, 0.364990234375, -0.041656494140625, -0.06011962890625, -0.48291015625, -0.15625, -0.623046875, 0.90478515625, -0.371826171875, 0.70947265625, 0.7138671875, -0.0675048828125, -0.121826171875, 0.09454345703125, 0.67236328125, -0.58447265625, 0.143310546875, 0.371337890625, -0.17431640625, 1.7255859375, -0.1092529296875, -0.343994140625, 1.169921875, 0.2108154296875, -0.035125732421875, 0.63427734375, -1.888671875, -0.04437255859375, 1.064453125, -0.320068359375, 1.58984375, 0.5478515625, 0.105712890625, -1.015625, 1.1474609375, 0.376953125, 0.73828125, 0.876953125, -2.408203125, -0.358154296875, 0.50341796875, -0.373046875, 2.236328125, 1.4521484375, -0.76220703125, 0.73681640625, 0.087158203125, 0.380126953125, 0.55517578125, -0.00553131103515625, 2.232421875, 1.4775390625, 2.666015625, -1.0263671875, 1.0361328125, -0.27197265625, -1.927734375, 0.349365234375, 0.96484375, 0.2161865234375, 0.13525390625, 0.316162109375, 1.923828125, -0.0645751953125, 0.71826171875, -0.572265625, 1.529296875, 0.1328125, 0.80810546875, 0.9345703125, -2.060546875, 2.060546875, 0.059326171875, 0.355224609375, -0.44970703125, 0.88916015625, -0.285400390625, -0.76513671875, -1.67578125, -1.1611328125, -0.52197265625, 0.1529541015625, -0.95654296875, 0.452880859375, 1.115234375, 0.073974609375, 0.454345703125, -0.8759765625, -0.487060546875, 0.258544921875, 0.69873046875, -0.34521484375, 1.25390625, -0.591796875, -1.451171875, -0.57275390625, 0.07958984375, -0.470458984375, 0.485107421875, 0.5830078125, -0.421142578125, -0.73193359375, 1.283203125, -0.99951171875, -0.93994140625, 2.099609375, 0.62451171875, 0.356201171875, 0.05877685546875, 0.34228515625, -0.057281494140625, 1.6376953125, -0.09234619140625, 0.8671875, -0.0921630859375, 0.95166015625, -1.38671875, 1.3896484375, -1.2119140625, 1.0048828125, -1.60546875, 1.5537109375, 0.90771484375, 0.79736328125, 0.278076171875, -0.56298828125, -0.008331298828125, 1.076171875, -0.422119140625, 0.52783203125, -1.9921875, -1.9716796875, 1.357421875, -0.1673583984375, 1.037109375, 1.275390625, 1.875, 0.08856201171875, 0.428955078125, 0.54833984375, -0.2193603515625, 1.396484375, 0.5126953125, 1.216796875, 0.83984375, 0.72802734375, -0.420166015625, 1.1748046875, 1.052734375, -0.1414794921875, -2.037109375, 0.255859375, 1.2041015625, -0.005764007568359375, -0.4384765625, -0.58642578125, -0.3115234375, -1.197265625, 0.986328125, -0.4462890625, -0.44140625, -0.2430419921875, -1.150390625, 1.765625, -1.44921875, 1.3427734375, 0.1260986328125, -0.188232421875, 0.061248779296875, 0.352294921875, -1.4189453125, 0.5517578125, 0.97412109375, -0.97509765625, 0.4970703125, 1.6142578125, -1.220703125, -1.4677734375, 0.49169921875, 0.0389404296875, 1.1044921875, 0.61962890625, -1.197265625, -0.65185546875, -0.309326171875, -1.2744140625, 0.984375, -0.1549072265625, -0.94140625, -1.0107421875, 0.347900390625, -1.1708984375, -0.92626953125, -0.6376953125, 1.548828125, -0.93896484375, -0.3583984375, 1.35546875, -0.501953125, -1.828125, 1.2861328125, -1.04296875, -1.326171875, -0.52001953125, 0.9091796875, 0.40380859375, 0.82958984375, 0.172119140625, 0.00719451904296875, 1.01171875, -1.1181640625, -0.440185546875, 0.363525390625, 0.04119873046875, -0.2264404296875, -0.63671875, 0.1767578125, -0.5703125, -0.29833984375, 0.2069091796875, 1.6845703125, 0.28369140625, 1.69140625, -0.240966796875, -0.0020351409912109375, 2.263671875, 0.10321044921875, -1.09375, 1.322265625, -0.258056640625, -0.94189453125, 0.50390625, -1.6318359375, -0.48828125, 1.525390625, 1.33203125, 0.2327880859375, 0.97802734375, 0.85302734375, -0.2122802734375, 0.1695556640625, 1.203125, 0.338134765625, 0.1634521484375, 0.41552734375, 0.623046875, 0.35595703125, -0.166748046875, -0.0703125, -0.28955078125, -1.0537109375, 0.8671875, -1.1376953125, -1.0439453125, -0.89599609375, 0.328369140625, 0.1827392578125, 2.078125, -2.328125, 0.73486328125, 0.494384765625, -0.268310546875, -0.0787353515625, -0.8173828125, -1.6337890625, -1.568359375, 0.055999755859375, 1.2353515625, 0.67431640625, 0.7509765625, 0.4619140625, 2.232421875, 0.443603515625, -0.047393798828125, -0.826171875, 1.60546875, -0.0474853515625, 0.3076171875, -0.40966796875, 1.513671875, -0.446533203125, -0.0013322830200195312, 0.7587890625, 1.5986328125, -0.056640625, -0.8310546875, 1.8017578125, 0.2132568359375, 0.316650390625, 0.1600341796875, -1.0869140625, 0.449462890625, -0.93310546875, 1.552734375, 0.1573486328125, 0.763671875, 1.666015625, 0.0247955322265625, 1.3505859375, 1.3564453125, 0.70068359375, -0.0018520355224609375, -1.458984375, 0.99169921875, 0.72021484375, 0.61083984375, 0.0880126953125, -0.47021484375, 1.2607421875, -0.2705078125, 0.2318115234375, -1.765625, 0.43701171875, 0.5732421875, -0.21533203125, 0.751953125, -0.1435546875, 0.837890625, -1.060546875, -1.8212890625, 1.384765625, 1.1240234375, -0.281982421875, 0.39453125, 0.6279296875, 1.2470703125, 0.15869140625, -0.037261962890625, -0.239990234375, -2.869140625, 0.341064453125, -1.0986328125, -1.41796875, 0.01204681396484375, 0.3671875, -1.0673828125, 0.84228515625, -0.1707763671875, 1.421875, 0.68310546875, -1.0498046875, 1.7333984375, 2.298828125, 0.82763671875, -1.216796875, 0.470947265625, -0.537109375, -0.0114898681640625, -1.22265625, -0.3037109375, 0.3525390625, 1.12109375, -0.5283203125, -0.1097412109375, -1.0517578125, 0.412841796875, -0.35498046875, 0.8857421875, 0.36279296875, -0.5888671875, 0.276611328125, 0.1907958984375, 2.16796875, 0.88623046875, -0.75146484375, -1.0419921875, -1.3525390625, -1.2607421875, -0.1759033203125, 0.65966796875, -0.92138671875, -1.2119140625, -1.1396484375, 0.8525390625, 0.40576171875, -0.91552734375, -0.578125, 1.0107421875, 1.046875, 0.1314697265625, -0.1363525390625, 1.107421875, 0.56689453125, -1.8369140625, 1.5654296875, 0.1737060546875, 0.1568603515625, 2.16796875, 0.50537109375, -0.293701171875, -0.60498046875, 3.28515625, 0.0791015625, -0.26708984375, -2.232421875, 0.5830078125, 0.035919189453125, -0.0989990234375, 1.888671875, 0.3037109375, -1.1796875, 1.7177734375, 0.435791015625, -0.133056640625, -0.55810546875, -0.94384765625, 1.3564453125, -0.1671142578125, 1.6259765625, 1.1728515625, -0.86962890625, -0.857421875, -1.3759765625, -0.4462890625, 1.4765625, -0.740234375, 0.64501953125, 1.103515625, -1.044921875, 1.22265625, -0.3515625, -2.595703125, 0.98876953125, -0.88232421875, 0.494140625, -0.7197265625, 1.0126953125, -1.34375, 0.53515625, -0.96630859375, -2.109375, 0.221923828125, -1.22265625, 0.28515625, -0.90478515625, -0.7138671875, -1.390625, 0.56591796875, 0.88232421875, -0.75927734375, 1.55078125, -1.8955078125, 1.46875, -2.45703125, 0.70556640625, -0.1416015625, -0.08343505859375, 1.1240234375, 0.045928955078125, 1.19921875, 0.453125, -1.3984375, -0.52880859375, -0.140625, 0.258544921875, 0.99169921875, 0.343505859375, -0.408935546875, -1.2021484375, -0.6376953125, 0.1943359375, -0.05633544921875, 0.62451171875, 1.0009765625, 0.771484375, 0.00543212890625, 0.68310546875, -1.0771484375, 0.42041015625, 0.21240234375, 0.2030029296875, 0.9189453125, 1.279296875, -0.8154296875, -0.334716796875, 0.41650390625, -0.210693359375, 0.316650390625, 1.1103515625, 0.859375, -1.1376953125, -0.30908203125, 2.36328125, 0.16015625, -0.07958984375, 0.050628662109375, -0.043701171875, 0.89501953125, 0.231201171875, 0.43408203125, -1.2138671875, 0.85302734375, -0.456298828125, 0.5048828125, -0.302734375, -0.426513671875, -0.54296875, -0.822265625, -2.5859375, 0.6787109375, 0.0709228515625, -0.89892578125, -0.424072265625, -0.6884765625, 0.1319580078125, -0.685546875, -0.0199127197265625, -0.40283203125, 1.04296875, 1.1845703125, 0.36328125, 0.5380859375, -1.3564453125, -0.66845703125, 1.0087890625, -0.982421875, -0.1295166015625, -0.5859375, 0.379638671875, 1.099609375, 1.1845703125, 0.7314453125, 0.180908203125, 0.5361328125, -0.07806396484375, 0.2154541015625, -1.369140625, -1.0517578125, 1.5947265625, -0.126708984375, 0.420166015625, -1.279296875, 0.77587890625, -0.60107421875, -0.0246429443359375, 0.19873046875, 0.74755859375, -0.5234375, 0.8125, 0.44970703125, -0.31689453125, -0.65576171875, -0.59228515625, 0.8251953125, -0.285400390625, 0.517578125, -0.208984375, -1.0498046875, 0.471923828125, 0.71240234375, 1.6962890625, 0.396240234375, -1.529296875, 0.1829833984375, -0.70947265625, -0.3740234375, -0.08349609375, -0.252685546875, 1.6552734375, -0.63134765625, 0.86962890625, 0.77294921875, 0.378173828125, -0.345458984375, 1.3291015625, -0.81689453125, -0.689453125, 3.2421875, 0.64453125, 0.005764007568359375, -1.130859375, -0.281005859375, -1.025390625, -0.393798828125, 1.6181640625, -0.50048828125, 0.415283203125, -0.09320068359375, -0.59619140625, 0.62060546875, 0.89794921875, -1.2880859375, 1.0947265625, -0.626953125, -1.7802734375, -1.0859375, 0.79248046875, 0.61669921875, 0.939453125, 1.5517578125, 1.2236328125, 1.5517578125, 0.7783203125, 0.35498046875, -1.3330078125, 0.429443359375, -1.572265625, 1.04296875, -1.48046875, 2.037109375, 1.3115234375, -0.09564208984375, -1.3828125, -0.1402587890625, 0.62939453125, 0.966796875, -0.1734619140625, 0.188232421875, 0.382080078125, -0.045745849609375, 0.86279296875, 1.4921875, 0.99951171875, 0.91943359375, -0.84375, 1.3798828125, 0.396728515625, -0.1456298828125, -0.1258544921875, -0.41259765625, -0.08740234375, 0.16748046875, 1.3681640625, -0.79638671875, -2.7734375, -0.955078125, -0.183837890625, -0.365234375, 1.6201171875, -0.2685546875, 1.83984375, -0.354248046875, 0.329833984375, -0.47509765625, -1.181640625, -1.630859375, 0.71240234375, 0.79833984375, -1.337890625, 0.857421875, -0.6923828125, 1.013671875, -1.1982421875, 1.08984375, 0.3046875, -1.5556640625, 0.433837890625, 1.474609375, 0.435791015625, 0.04876708984375, 0.43603515625, -0.51806640625, -2.814453125, -1.7509765625, -0.80078125, -0.7646484375, 0.8134765625, 0.2381591796875, 0.21728515625, -0.44921875, -1.6806640625, -0.1170654296875, -0.27734375, -0.289794921875, 0.35009765625, -0.422119140625, -1.0908203125, 0.96142578125, -0.81005859375, -0.9208984375, 0.53466796875, 0.5302734375, -0.64501953125, -1.142578125, -0.73974609375, 0.55419921875, -0.08538818359375, -0.1151123046875, -0.0650634765625, 0.337158203125, -0.2113037109375, 0.2607421875, 1.2158203125, -0.9326171875, -0.865234375, 1.1708984375, 0.2509765625, -0.55810546875, 0.662109375, -0.54638671875, -1.0546875, -0.2049560546875, 1.943359375, 0.58056640625, 0.313232421875, -0.06793212890625, -0.48046875, 0.9892578125, 0.93017578125, -1.166015625, 0.986328125, -0.62255859375, 0.70166015625, 0.065673828125, 3.712890625, 0.1461181640625, 0.382568359375, 0.260986328125, -0.232666015625, -0.11212158203125, -0.5234375, -0.36767578125, 0.03875732421875, 1.0439453125, 0.1397705078125, -0.55517578125, 0.437744140625, -0.28759765625, 0.6171875, 0.43017578125, -0.70068359375, 1.0302734375, -0.148193359375, -0.60400390625, 0.8369140625, -0.134521484375, 0.7958984375, 0.1834716796875, 0.87744140625, 0.4638671875, 0.492431640625, -1.21875, 1.4013671875, 1.4052734375, 0.27001953125, 2.333984375, -0.2193603515625, -0.384521484375, 1.1416015625, -0.76953125, -2.140625, 1.4892578125, -0.43408203125, -0.87353515625, -1.8544921875, -1.3359375, -1.708984375, 0.064453125, -1.8701171875, 0.038055419921875, 1.02734375, -0.8505859375, 0.007579803466796875, 1.3759765625, 0.395751953125, -0.6015625, 0.018218994140625, -0.986328125, -0.270263671875, -0.039306640625, -0.60986328125, 0.10235595703125, 0.9150390625, 0.001468658447265625, -0.5166015625, -0.34033203125, -0.131103515625, 1.212890625, 1.16796875, 0.042022705078125, -0.416015625, -0.6044921875, -0.26806640625, -0.252685546875, -1.7978515625, -1.8837890625, 1.556640625, 1.224609375, -0.5048828125, -0.51611328125, -0.353515625, 0.0601806640625, -0.78955078125, -0.83056640625, -0.5244140625, -0.2313232421875, -0.0472412109375, 0.89306640625, 0.27294921875, -1.306640625, 0.84814453125, 0.5634765625, 0.82666015625, -0.3984375, -0.156494140625, -0.038604736328125, -1.7041015625, -0.86962890625, 0.294921875, 0.78466796875, -0.47802734375, -0.453125, 0.7744140625, 1.2900390625, 0.75048828125, 0.80322265625, 0.387451171875, 0.69189453125, 2.404296875, -0.08172607421875, -0.0340576171875, -1.783203125, -0.1685791015625, -0.44091796875, 0.07232666015625, 1.7841796875, -0.56298828125, -0.4384765625, -0.884765625, 1.8271484375, 0.35400390625, 0.337890625, -0.144287109375, -0.4482421875, 0.64404296875, 2.697265625, -1.2470703125, 1.41796875, 1.08203125, -2.4140625, 2.14453125, -2.2890625, -0.2451171875, -2.333984375, 0.7294921875, -0.85205078125, 1.244140625, -0.59765625, 1.3779296875, -0.321533203125, 1.6552734375, 0.6298828125, -0.322021484375, -0.814453125, -1.0654296875, 0.06890869140625, 0.9384765625, 0.5537109375, -1.12109375, -1.171875, 1.3115234375, -0.0228729248046875, 1.5419921875, -0.265380859375, 0.310546875, -0.892578125, 0.92138671875, 0.45361328125, 0.59423828125, -1.427734375, 1.1982421875, -1.6455078125, 0.89208984375, 0.01033782958984375, 2.08203125, 1.5625, 0.84228515625, 1.34765625, 0.05364990234375, -1.3935546875, -0.5869140625, -0.6845703125, 2.005859375, -0.400390625, 0.2353515625, 1.7900390625, -0.417724609375, -0.27099609375, -0.313720703125, 0.81494140625, 1.1435546875, -0.058837890625, 0.0241241455078125, -1.54296875, 0.268798828125, 0.630859375, -1.01953125, -0.1473388671875, 0.77587890625, -0.01398468017578125, 0.92236328125, 0.1336669921875, -0.26025390625, 1.12890625, 0.06817626953125, -0.418212890625, -0.3623046875, -0.189208984375, 1.2470703125, 0.677734375, 0.190185546875, 1.7119140625, 0.52294921875, -1.17578125, 0.7109375, -0.638671875, 1.0576171875, 1.1865234375, -0.7109375, -1.4677734375, -0.81982421875, 0.307861328125, -1.12890625, 1.2705078125, 0.7138671875, 0.99462890625, 0.358642578125, 0.432373046875, -1.3544921875, -1.7705078125, 0.21240234375, -1.5693359375, -0.62451171875, -1.44140625, 0.728515625, -0.69140625, -0.85693359375, 0.4208984375, 0.93017578125, 1.26953125, 0.76513671875, -0.370849609375, -2.255859375, 0.52001953125, -0.6025390625, 1.7998046875, -1.3720703125, 1.3818359375, 0.0226593017578125, -1.02734375, 1.4794921875, 1.2041015625, 0.04095458984375, -0.517578125, -0.2333984375, 1.69921875, -1.607421875, 1.1240234375, -1.2734375, -1.2646484375, -0.1910400390625, -0.75048828125, 0.09588623046875, 0.23388671875, -0.177978515625, 0.5009765625, 0.339599609375, 0.048126220703125, 0.2418212890625, -0.454345703125, -1.48828125, -0.79443359375, -1.27734375, -1.5400390625, 0.68994140625, 0.221435546875, 0.88037109375, 0.57421875, -1.2646484375, 1.869140625, 0.494873046875, -0.245361328125, -0.07373046875, 2.611328125, 0.60546875, 0.90576171875, -2.85546875, 1.666015625, -0.294677734375, -0.2127685546875, 2.009765625, 0.69873046875, 0.54052734375, 0.29638671875, -1.966796875, 0.89599609375, -1.611328125, 0.048919677734375, -0.187255859375, -0.191162109375, 1.6357421875, -0.97998046875, 0.10546875, 0.413818359375, -1.2080078125, 0.5478515625, -0.5546875, -0.357177734375, 1.1201171875, 0.28271484375, 0.305908203125, -0.80517578125, 0.30517578125, -0.04693603515625, -1.306640625, -0.55224609375, 0.342529296875, 0.43408203125, 0.93896484375, -0.278564453125, -0.210205078125, -1.09765625, 0.43212890625, -0.26220703125, -2.21484375, -0.000667572021484375, 0.051239013671875, 0.465576171875, -0.56298828125, -0.72265625, 0.31396484375, 1.6025390625, -0.4072265625, 1.2666015625, 0.8525390625, -0.36376953125, -0.5439453125, -0.427734375, 1.748046875, 2.04296875, -0.257568359375, 0.767578125, -1.2236328125, -0.759765625, 0.7041015625, 0.77294921875, 0.65380859375, 0.12005615234375, 0.77001953125, 0.08599853515625, 0.92626953125, -1.3984375, 0.52587890625, -0.12481689453125, 0.47607421875, 1.62890625, 1.009765625, -1.1533203125, 0.3154296875, -0.94775390625, -1.0078125, -0.1578369140625, -0.057586669921875, 0.445068359375, -1.5908203125, -0.4580078125, 1.439453125, 1.275390625, -2.15625, -0.033966064453125, -1.3779296875, -0.476806640625, -0.342529296875, 1.5556640625, 0.08966064453125, 0.06280517578125, -2.80859375, -2.00390625, -1.13671875, -0.322265625, -1.58203125, -1.37109375, 0.1824951171875, -0.63330078125, -0.3486328125, -0.05584716796875, -0.051544189453125, 1.748046875, 0.372802734375, -1.134765625, 0.384033203125, 0.52685546875, 1.3798828125, -0.11871337890625, -1.376953125, 0.4541015625, -0.448486328125, -0.60595703125, 0.75390625, -0.83154296875, -1.677734375, -1.4775390625, 1.4228515625, -0.060089111328125, 0.70361328125, -0.00415802001953125, -1.048828125, 0.736328125, 0.40625, -1.5849609375, 0.397705078125, 0.375244140625, -0.9990234375, -0.88916015625, -0.12066650390625, -0.07135009765625, 0.265869140625, 0.5654296875, -1.0634765625, 0.40966796875, 0.34228515625, 0.413330078125, 1.2626953125, 1.896484375, -0.625, 0.3759765625, -0.8134765625, -0.02349853515625, -0.791015625, 0.65234375, -0.016387939453125, -0.52001953125, 0.61669921875, -2.3125, -1.4150390625, 0.0037670135498046875, -0.3134765625, -0.1695556640625, 1.37890625, -1.2373046875, -1.408203125, 0.0714111328125, 0.1185302734375, 0.51708984375, 0.150146484375, -2.552734375, -0.86669921875, 0.8720703125, 1.4150390625, -0.3349609375, 0.035308837890625, -2.109375, 0.279296875, 0.478759765625, -1.12890625, -0.441650390625, 0.96923828125, 1.251953125, 0.89990234375, -0.360595703125, -2.48828125, -0.07391357421875, 0.2232666015625, -1.33203125, 0.265869140625, 0.27099609375, -0.646484375, 0.70166015625, 0.39794921875, -0.296630859375, -0.0186614990234375, -0.247314453125, -0.36474609375, 2.287109375, -1.5888671875, 0.071044921875, -1.150390625, -0.253173828125, 0.8505859375, 0.892578125, -0.05316162109375, 0.640625, 0.057464599609375, -0.7216796875, -1.4091796875, -0.384033203125, -0.67236328125, 0.43310546875, -1.41015625, -0.10693359375, 1.646484375, 0.2315673828125, 0.39013671875, 1.4990234375, -0.466552734375, 0.72998046875, -0.0582275390625, 1.666015625, 0.50048828125, 0.176513671875, 1.6923828125, 0.58837890625, 0.71484375, 0.5439453125, 0.383544921875, -0.13232421875, 0.89501953125, 0.298828125, 0.59765625, -1.0205078125, -0.257568359375, -0.97412109375, -1.177734375, -0.37060546875, 1.4365234375, 1.361328125, 0.1767578125, 0.541015625, -0.461669921875, 0.441650390625, -0.478515625, -0.51513671875, -0.8388671875, 1.44140625, -0.482177734375, 2.091796875, 1.44140625, -1.59765625, -0.10809326171875, 0.147705078125, -0.921875, -0.1094970703125, 1.21484375, 0.461669921875, 0.1956787109375, -0.326171875, -0.2315673828125, 0.845703125, -1.81640625, 0.2486572265625, -1.275390625, -0.6044921875, 0.86181640625, -0.43359375, -1.3251953125, -1.2255859375, 0.2734375, -0.625, 0.401611328125, 0.69921875, 0.63037109375, -1.9482421875, 0.7958984375, 0.212890625, -1.3994140625, 2.13671875, -0.007556915283203125, 0.37255859375, 0.560546875, -0.81201171875, -1.2197265625, 0.6259765625, 1.83203125, -1.451171875, 2.013671875, 1.169921875, -1.591796875, -0.94775390625, 1.5634765625, 0.369384765625, -0.369384765625, 0.005764007568359375, -0.27783203125, -2.271484375, -0.53857421875, 0.0205230712890625, 0.454833984375, 0.01372528076171875, 1.3681640625, -0.57568359375, -0.72314453125, 1.3603515625, 1.630859375, 0.2587890625, -0.27099609375, 0.6220703125, 0.54931640625, -0.1387939453125, -1.189453125, 0.28173828125, -0.8525390625, 0.7734375, 0.437255859375, 1.134765625, -1.7470703125, -1.1630859375, 1.2275390625, -0.9423828125, 0.8251953125, -1.9189453125, -0.7724609375, -0.20361328125, 1.0185546875, 1.904296875, -0.0032501220703125, 0.4306640625, -0.37353515625, 0.68310546875, -0.09136962890625, 0.56103515625, 1.1318359375, -1.2587890625, 1.01171875, 0.337646484375, 0.0233917236328125, -0.14697265625, 1.74609375, -0.6318359375, -1.8818359375, 0.5576171875, 0.69580078125, 1.8134765625, -2.306640625, 1.7265625, -0.0293426513671875, 0.5732421875, -1.162109375, -0.806640625, -0.7958984375, 0.71142578125, -0.5810546875, 1.5361328125, 0.83935546875, 1.63671875, 0.279052734375, -1.1728515625, 0.55029296875, -2.478515625, 0.287109375, -0.82568359375, -0.73486328125, 1.826171875, 0.31103515625, -0.0029754638671875, 2.404296875, -1.970703125, 0.253173828125, 1.138671875, -1.2509765625, 0.43994140625, -0.75634765625, -0.04315185546875, -0.6357421875, 1.1142578125, -0.0635986328125, 0.8505859375, 0.24951171875, 0.277099609375, 0.439453125, -0.47265625, 2.19921875, 0.25048828125, -0.52880859375, -1.2275390625, -1.998046875, 0.147705078125, 0.7158203125, 1.61328125, -1.005859375, 0.73779296875, -0.12164306640625, -0.07647705078125, 0.90185546875, 1.169921875, -2.0234375, 1.5009765625, -1.4619140625, -0.31640625, 0.74169921875, -0.452392578125, 1.1484375, -1.4755859375, -0.76220703125, 0.007476806640625, -0.39697265625, 0.7119140625, -1.12890625, 1.455078125, 0.64501953125, 2.06640625, -0.219970703125, 1.240234375, 1.3759765625, -0.86669921875, -0.97216796875, -2.115234375, 0.34228515625, 1.2421875, 0.350341796875, 1.0107421875, 1.1982421875, 0.424560546875, -0.1558837890625, -2.142578125, 0.328369140625, 0.91650390625, 0.09185791015625, -0.99365234375, -1.767578125, 0.07098388671875, -0.9765625, 0.420166015625, 1.12109375, 0.9677734375, 1.3828125, -0.51953125, 0.62255859375, -1.1025390625, 1.150390625, -0.98388671875, -0.6826171875, 1.2314453125, 1.458984375, 0.59326171875, -0.2222900390625, -1.2763671875, 0.8388671875, -1.9228515625, -1.7919921875, 0.491455078125, -0.317626953125, -0.88720703125, 0.330810546875, 0.73779296875, -2.447265625, -0.165771484375, -0.0794677734375, -1.5556640625, -0.43115234375, -0.346435546875, 1.2197265625, 1.5927734375, 0.445068359375, 0.3623046875, -0.21728515625, 2.048828125, -1.951171875, -0.4912109375, 0.234130859375, 0.200439453125, 1.259765625, -0.256591796875, -1.16015625, -0.51220703125, -0.10845947265625, -0.64501953125, 0.86767578125, 1.259765625, -0.4404296875, 0.51904296875, -0.53955078125, -1.24609375, 0.416259765625, -0.385498046875, 1.4462890625, -0.2578125, -0.043731689453125, 0.213623046875, -0.71728515625, 0.88330078125, -1.38671875, -0.2403564453125, 0.4189453125, -0.63818359375, 0.1357421875, -1.87890625, -0.68212890625, 1.0166015625, 0.6875, 1.24609375, 1.2939453125, 0.48828125, -1.3828125, -0.99658203125, 0.214599609375, 1.27734375, -0.6845703125, 1.095703125, 2.267578125, -1.2548828125, 1.0361328125, -1.080078125, -0.73876953125, -1.7734375, 0.55322265625, -0.662109375, 0.97509765625, 0.529296875, -1.197265625, -0.490234375, -0.44287109375, -0.208251953125, 0.31201171875, -0.0675048828125, 1.07421875, 1.1201171875, 0.796875, 2.248046875, -0.10467529296875, 1.4970703125, 0.60400390625, 0.38525390625, -0.662109375, -0.289794921875, -0.381103515625, -0.78759765625, -1.81640625, -0.8759765625, 0.755859375, -0.31640625, 0.59619140625, 0.374267578125, -0.1102294921875, -1.8974609375, 0.01409149169921875, 0.034759521484375, -0.98291015625, 1.611328125, -0.4423828125, -0.54736328125, -1.7548828125, -2.37890625, -1.6201171875, 0.44189453125, 0.85302734375, 1.478515625, 0.25927734375, 0.6103515625, 0.55322265625, -1.2763671875, -0.3623046875, -1.2041015625, 0.67919921875, 0.142333984375, -1.5986328125, -0.40869140625, -1.9404296875, 0.74560546875, 0.2418212890625, 0.02911376953125, -0.7333984375, 0.1966552734375, 0.43994140625, -0.525390625, -0.56982421875, -1.9228515625, 0.90087890625, 0.779296875, -0.1845703125, 1.052734375, -0.6640625, -2.779296875, 2.03515625, -0.291259765625, 1.2890625, 0.89794921875, -0.1126708984375, 0.572265625, 0.857421875, 0.0947265625, -1.0234375, -0.335693359375, -0.459716796875, 0.198974609375, -0.892578125, -0.358154296875, 0.3369140625, 0.7705078125, 0.490234375, -0.630859375, 0.65869140625, -0.716796875, -0.224853515625, -0.146728515625, -0.5009765625, 0.7109375, -0.23828125, 0.1837158203125, 0.8310546875, 1.3076171875, 1.521484375, 0.2880859375, -0.448486328125, 0.9248046875, 1.439453125, 0.62353515625, -0.45166015625, 0.83837890625, 1.390625, -0.007358551025390625, -0.75146484375, -0.1241455078125, 0.452392578125, -0.267333984375, 0.61328125, -0.28515625, -0.4375, 0.587890625, -0.3408203125, 0.0010442733764648438, 1.4638671875, -0.646484375, -1.3486328125, 1.7236328125, 1.064453125, -0.80810546875, -1.193359375, 0.3642578125, -0.68896484375, 0.498291015625, -0.6376953125, 0.364990234375, -1.1611328125, 0.208251953125, -0.3798828125, -1.8173828125, 1.3837890625, 0.880859375, -0.5283203125, -0.515625, 0.1815185546875, -0.49267578125, -1.3427734375, 0.0029506683349609375, -0.53173828125, 1.4091796875, -0.64697265625, -0.78857421875, -0.9443359375, 0.379150390625, 0.057037353515625, -1.2451171875, -0.71728515625, 0.74755859375, 0.74267578125, -0.4833984375, -0.275390625, 1.8876953125, 0.324462890625, 1.9912109375, 0.1593017578125, -1.078125, 0.72509765625, 0.82568359375, -1.0673828125, 1.8369140625, -1.2822265625, 0.222900390625, 0.1668701171875, 0.830078125, -0.380615234375, 0.9794921875, 0.2279052734375, 0.89306640625, 0.509765625, 0.85498046875, -0.41259765625, -0.51806640625, -0.3466796875, 0.0535888671875, 0.65576171875, 0.71240234375, 0.339599609375, 0.88671875, -0.77685546875, -1.4013671875, 1.333984375, -0.54833984375, 0.25390625, 1.03515625, 1.384765625, 0.192626953125, -0.84130859375, -0.03314208984375, 0.428466796875, -1.478515625, -0.16015625, -0.3740234375, -0.486572265625, -1.978515625, 1.5830078125, -0.008544921875, -0.3212890625, -1.0576171875, 1.041015625, 0.34521484375, -0.06695556640625, 0.72998046875, 0.1822509765625, -1.6142578125, 0.0206756591796875, 0.4931640625, -0.78076171875, -1.0400390625, 0.587890625, 0.35791015625, -0.305908203125, 0.89111328125, -0.75927734375, 1.2333984375, 1.4267578125, 0.5166015625, 0.6103515625, 0.0513916015625, -1.1904296875, 0.152587890625, 0.9677734375, -2.72265625, -0.7392578125, -0.6318359375, -0.56640625, 0.970703125, -1.1123046875, 0.58935546875, 0.3984375, 0.049530029296875, -0.2666015625, -0.994140625, 2.2421875, -0.80712890625, -0.26513671875, -0.473876953125, -0.06658935546875, -0.0711669921875, -1.458984375, 0.27294921875, 1.455078125, -1.23828125, -0.1795654296875, -0.2333984375, -0.0210113525390625, 0.395263671875, 0.32470703125, -0.060272216796875, 0.67333984375, -1.693359375, -0.285400390625, 0.73876953125, -0.47802734375, -0.61669921875, -0.087646484375, 0.2998046875, 0.30859375, -1.0556640625, 0.377197265625, 0.89453125, 1.41796875, -0.435302734375, -0.57470703125, 2.11328125, 0.259765625, -0.162109375, -0.311279296875, 0.490234375, -0.31005859375, 0.4306640625, -1.3095703125, -2.537109375, 1.2255859375, -0.98681640625, 0.093017578125, 0.779296875, 0.39892578125, 0.56103515625, -0.55810546875, -0.5390625, -1.7158203125, -2.3046875, 1.3037109375, 1.5458984375, -0.2454833984375, 0.70947265625, -1.517578125, 0.55029296875, 0.41943359375, -0.367431640625, 0.4375, -0.78662109375, 2.728515625, -0.5009765625, 0.6728515625, -0.681640625, -1.125, -1.5517578125, -0.280517578125, -0.45458984375, -0.1268310546875, -0.578125, -0.374755859375, -0.437744140625, 0.69970703125, -0.875, 0.385009765625, 0.363037109375, 1.0322265625, -2.0078125, -1.98046875, -1.2548828125, -0.75048828125, -0.50341796875, 0.247802734375, -2.05078125, 0.65966796875, -0.1209716796875, -0.014739990234375, 0.357177734375, 0.12188720703125, -0.38037109375, -0.3447265625, 0.82568359375, 1.6943359375, -1.875, -1.1494140625, -0.8349609375, -0.33935546875, -0.66015625, -0.61083984375, 0.8564453125, -1.15625, 1.3505859375, -1.2890625, 0.1785888671875, -1.11328125, -0.0928955078125, 1.6943359375, -1.0751953125, 0.65087890625, -0.309814453125, -1.513671875, 0.414794921875, 0.3388671875, -0.8564453125, 0.1453857421875, -1.041015625, 0.7880859375, -1.2666015625, 0.5302734375, -0.16748046875, 0.564453125, -1.5185546875, 1.4072265625, 0.31591796875, -0.5068359375, -0.6171875, 0.432373046875, -0.210205078125, -1.3427734375, 0.202392578125, -1.458984375, 0.380126953125, -0.482177734375, 1.0703125, 1.8388671875, -0.84375, 1.1552734375, -1.712890625, -2.1484375, -1.0146484375, -0.475830078125, -2.7109375, 0.57666015625, 1.4794921875, -0.2861328125, -1.05078125, -0.42333984375, 0.89404296875, 0.0849609375, -1.3818359375, -1.109375, -1.1171875, -0.8583984375, -1.8974609375, -0.95166015625, 0.3369140625, 1.1240234375, 0.5888671875, -0.2125244140625, -0.5078125, 2.05078125, -0.701171875, -0.317138671875, -0.66162109375, -0.06396484375, 0.93310546875, 1.5361328125, 0.2568359375, -0.36181640625, -1.580078125, 0.853515625, 3.109375, 2.0078125, -1.4287109375, -0.2362060546875, -1.0439453125, -1.6796875, 0.33447265625, -1.44921875, 1.3154296875, 0.1412353515625, 0.69775390625, 1.33984375, -0.06536865234375, -1.0107421875, 0.357421875, -0.5234375, -2.2421875, -0.92724609375, 1.5361328125, 0.376708984375, -1.8759765625, -0.46484375, -2.169921875, -1.3447265625, -0.51953125, -1.23046875, -0.705078125, -0.4697265625, -0.94189453125, -0.8349609375, 0.361572265625, 0.09228515625, -0.480224609375, -0.07257080078125, 1.2744140625, -0.34228515625, 0.2060546875, 0.673828125, 2.056640625, -0.371337890625, -0.90869140625, -0.2161865234375, -0.398681640625, 0.8955078125, 0.8994140625, 0.07598876953125, -0.373779296875, 1.6064453125, 0.032745361328125, 0.6611328125, -0.64404296875, 1.0107421875, 0.446044921875, 0.1986083984375, -0.3828125, -2.279296875, -0.3720703125, 1.14453125, -1.4033203125, -0.609375, 1.05859375, 3.337890625, -0.36328125, -0.131591796875, 0.58251953125, 1.865234375, -1.5400390625, 0.77099609375, -1.150390625, 1.36328125, -1.9033203125, -0.30126953125, 1.134765625, -0.5341796875, 0.84521484375, 1.2255859375, 0.76220703125, 0.6806640625, 0.494384765625, 0.2088623046875, -0.7080078125, -1.0546875, -0.5849609375, -0.05316162109375, -1.5419921875, -2.20703125, 0.447998046875, 0.046051025390625, 0.17138671875, 0.8193359375, 1.189453125, -1.2412109375, 1.2431640625, 0.09625244140625, 0.09735107421875, -0.01064300537109375, -0.2039794921875, -1.55859375, -0.748046875, -0.58837890625, 0.1708984375, -0.91943359375, 1.2470703125, 0.1279296875, -0.11187744140625, -1.123046875, 0.372314453125, 0.495849609375, 0.004703521728515625, 0.08465576171875, 1.0185546875, 1.2578125, 2.298828125, -0.2227783203125, -0.5380859375, 1.05859375, 0.124755859375, -1.3564453125, -0.3486328125, -2.486328125, 0.342041015625, -0.0660400390625, 0.28466796875, -0.120361328125, -1.6484375, 1.625, 1.05078125, -1.0693359375, -0.2271728515625, 0.7841796875, 0.87109375, -0.33642578125, 0.65087890625, -0.493408203125, -0.2939453125, -0.67919921875, -0.342529296875, 0.2025146484375, 0.33935546875, 0.76806640625, -2.09375, -0.429931640625, 0.414794921875, 0.1231689453125, 0.8388671875, 0.1531982421875, -0.86376953125, 0.80078125, 0.7138671875, 0.4892578125, 0.343505859375, -0.54248046875, -0.62109375, -0.303466796875, -1.0107421875, -0.1729736328125, -1.134765625, -0.249755859375, 0.08087158203125, 0.37451171875, -0.97607421875, 0.412109375, -0.88037109375, 2.025390625, 1.7578125, 0.939453125, -0.473876953125, 1.1474609375, 0.59130859375, -0.0071258544921875, -0.58642578125, 0.97412109375, 0.951171875, 0.5048828125, 0.7421875, -1.021484375, 0.1923828125, 0.17724609375, 1.4580078125, -0.005352020263671875, 2.44140625, 0.16796875, 1.3828125, -1.3974609375, 0.87060546875, -1.2138671875, -2.017578125, 0.060760498046875, -0.88623046875, -1.4189453125, -0.51416015625, 1.08984375, 0.51416015625, 0.87353515625, 0.2303466796875, -0.1688232421875, -1.865234375, 0.1470947265625, 1.62890625, -0.5517578125, -1.0703125, -0.05902099609375, 1.34375, -1.5556640625, 0.171630859375, -1.1279296875, 0.2198486328125, 0.89990234375, -0.44384765625, -0.2578125, 1.83203125, 0.1478271484375, -0.7470703125, 0.6513671875, 0.6494140625, -0.076171875, 1.1904296875, -1.9306640625, 0.09234619140625, -0.215576171875, 0.57470703125, -0.37890625, -0.60595703125, -1.259765625, 0.54248046875, 0.9052734375, 2.03125, 0.61328125, -1.3291015625, 0.12158203125, 0.35888671875, -1.171875, 0.67919921875, -0.0031108856201171875, 0.5078125, 0.8017578125, 0.9296875, -0.92724609375, 2.837890625, 0.045745849609375, 0.250244140625, 0.763671875, 1.4931640625, -0.21533203125, -0.495849609375, -1.19140625, -1.31640625, -1.0859375, -1.072265625, 0.333740234375, -0.49658203125, 1.673828125, -0.63330078125, -0.6865234375, 0.471435546875, -0.35888671875, -0.387451171875, 1.2109375, 0.59033203125, 0.69482421875, 0.2685546875, -0.7802734375, 1.064453125, -1.318359375, 2.451171875, -1.5107421875, 0.261474609375, -0.2076416015625, -0.6142578125, -0.0018777847290039062, -0.1722412109375, -0.0322265625, 0.7041015625, 0.237060546875, 0.6240234375, -1.0126953125, 0.2568359375, 0.2469482421875, -1.0107421875, 0.09783935546875, 1.318359375, 0.76318359375, -1.34375, 0.409912109375, -2.46875, 1.0234375, -0.484375, -0.50732421875, -0.5654296875, -0.86376953125, 0.2010498046875, 2.36328125, -1.0302734375, -1.345703125, -2.068359375, -0.08319091796875, -0.2919921875, -1.7138671875, 0.56494140625, 0.303955078125, 2.4375, -0.368408203125, -0.5595703125, -0.93994140625, 1.3623046875, 1.11328125, -1.302734375, -1.3984375, -0.46484375, -1.6845703125, -0.61328125, 0.55078125, -0.450439453125, 0.96533203125, -1.212890625, 0.70751953125, 1.396484375, 0.5322265625, -1.3779296875, -2.203125, 0.50390625, 0.2308349609375, 0.11761474609375, -0.50537109375, -0.3671875, -0.366943359375, 1.357421875, 0.486083984375, -0.43603515625, -0.6259765625, -0.6953125, 0.67822265625, 1.5400390625, 0.84814453125, -0.07513427734375, -0.4521484375, -0.1500244140625, -0.51611328125, -1.3046875, 0.394775390625, -0.63037109375, 1.509765625, 1.34375, 0.81298828125, 1.0029296875, 0.61767578125, 0.5029296875, -0.470703125, 1.0185546875, -0.4677734375, 0.52880859375, -1.3427734375, 1.5771484375, 0.39892578125, -0.564453125, -0.951171875, 0.91357421875, -1.6484375, 1.0341796875, -0.0968017578125, 0.487060546875, 1.80859375, 1.6435546875, -0.828125, -0.67578125, 0.2666015625, -0.6298828125, -1.2587890625, -0.420166015625, 1.2021484375, 1.9326171875, -0.51416015625, 1.337890625, -0.1536865234375, 0.0194549560546875, 0.53564453125, -1.06640625, 0.33740234375, 0.7880859375, -0.72119140625, 0.8134765625, 0.383544921875, -0.31640625, 0.18310546875, 0.2320556640625, -1.734375, -0.252197265625, -1.3603515625, -2.142578125, -0.1678466796875, -0.7978515625, -1.6640625, 0.681640625, -0.99755859375, 2.66796875, 0.86572265625, 0.59716796875, -0.6083984375, 1.119140625, -1.24609375, -1.0146484375, 0.077392578125, -0.5625, 0.56201171875, -1.212890625, -0.892578125, -0.394287109375, 0.771484375, 0.861328125, -1.1533203125, 0.459228515625, 1.8623046875, 0.96728515625, 0.630859375, -0.78857421875, -1.8505859375, 0.1966552734375, 1.2802734375, -0.12408447265625, 0.5908203125, -1.2197265625, -0.9716796875, 0.87060546875, 0.78955078125, 1.341796875, 0.2296142578125, 0.30419921875, 0.004184722900390625, 0.70947265625, 1.1962890625, -0.67724609375, -1.052734375, -1.072265625, 0.51416015625, -1.4345703125, -1.2197265625, -0.7158203125, 0.28466796875, -0.388671875, 0.143310546875, -2.27734375, 2.333984375, -0.77001953125, -0.38232421875, 0.302001953125, 0.2900390625, -0.29931640625, -0.0452880859375, -0.64990234375, -0.279541015625, 0.09814453125, -0.421630859375, 0.2283935546875, 1.2666015625, -0.70263671875, -1.3271484375, -0.71533203125, -1.931640625, 1.5400390625, 0.018035888671875, 0.03619384765625, 0.6015625, -0.822265625, 0.8447265625, -0.003467559814453125, 0.48583984375, -0.74365234375, -0.1219482421875, -0.71728515625, 1.341796875, -1.1611328125, 0.375, 1.3935546875, 0.1749267578125, 1.6416015625, -2.078125, -1.6240234375, -2.54296875, -0.47509765625, -0.055419921875, 0.712890625, 0.2437744140625, 1.853515625, 0.2239990234375, -1.056640625, 0.79931640625, -0.04638671875, -0.08367919921875, -0.163818359375, -1.310546875, -0.9091796875, -0.362548828125, -0.7685546875, 0.68994140625, -0.253662109375, 0.150390625, -0.1697998046875, 2.048828125, -1.1240234375, 0.370849609375, -0.1434326171875, 1.2099609375, -0.449462890625, -0.07342529296875, 0.73681640625, 1.40625, 0.493896484375, -0.268310546875, -0.7734375, 0.033050537109375, 1.5283203125, 0.849609375, -0.82763671875, -1.8671875, 0.56640625, 0.4443359375, -1.1318359375, -0.10357666015625, -0.7548828125, 2.296875, -0.7607421875, -1.5146484375, -0.54833984375, 2.39453125, -0.77197265625, -0.55859375, 1.783203125, -1.8779296875, -0.79345703125, 2.453125, -0.38134765625, -0.990234375, -0.98974609375, 0.7177734375, 0.83984375, -2.072265625, -0.7880859375, 0.80908203125, -0.37451171875, 0.239501953125, -0.07647705078125, 0.352294921875, -0.9580078125, -0.401611328125, -0.43798828125, 0.80615234375, -0.4150390625, -0.75830078125, 0.54638671875, -0.46142578125, 0.34033203125, 0.395751953125, 1.1533203125, -0.482666015625, -0.11602783203125, 0.64990234375, 0.467529296875, 1.7021484375, -1.0126953125, -0.533203125, -0.81787109375, 0.49658203125, -0.50244140625, -0.88818359375, 0.3486328125, -0.1546630859375, 0.77294921875, -0.76025390625, -2.05078125, -1.5029296875, 0.306396484375, -1.755859375, 1.23828125, -0.415283203125, -1.0322265625, 0.053802490234375, -0.18359375, 0.25830078125, 0.94775390625, 0.78271484375, 0.49462890625, 1.0146484375, -0.56494140625, 1.1552734375, -1.6845703125, -1.484375, -0.900390625, -0.01053619384765625, -2.068359375, -0.048187255859375, 0.8701171875, 1.1611328125, 0.11468505859375, -0.56201171875, 0.50927734375, -0.56494140625, 1.5556640625, 2.158203125, 2.048828125, -0.7939453125, 2.447265625, 0.92724609375, -1.548828125, 0.406005859375, -2.7421875, 1.880859375, 1.1630859375, -0.85693359375, 0.1302490234375, -1.341796875, -0.0904541015625, -1.5380859375, -0.264404296875, -0.1959228515625, 0.96533203125, 0.60205078125, 0.39013671875, 1.8203125, -0.160888671875, 0.708984375, -0.5732421875, -0.4755859375, -1.314453125, -1.5966796875, 0.46435546875, 1.2060546875, -0.03070068359375, -0.063720703125, -1.0390625, -1.0673828125, 0.2763671875, 0.955078125, 0.23486328125, -0.6318359375, -0.287109375, 0.53857421875, -0.92138671875, -0.467041015625, -0.1358642578125, -0.55078125, 0.7509765625, 0.5732421875, -0.64111328125, 0.85107421875, 1.107421875, -0.1568603515625, 0.90576171875, -0.85791015625, 0.70556640625, 1.494140625, 0.92626953125, -0.732421875, -0.32275390625, -0.638671875, -0.46435546875, 0.381103515625, -1.9931640625, -0.1832275390625, -0.08660888671875, -0.42822265625, 0.7431640625, -0.447509765625, -0.75390625, 0.8173828125, -0.4345703125, 3.2734375, 0.10400390625, -1.908203125, 1.1328125, 0.3076171875, -0.8564453125, -2.291015625, -0.146728515625, -0.88916015625, -0.475830078125, 0.7587890625, -0.1629638671875, -1.126953125, -0.75927734375, -0.5146484375, -0.26806640625, -0.5576171875, -0.82958984375, 0.66748046875, 0.8232421875, -1.9521484375, -0.5908203125, -1.25390625, 0.32861328125, 0.384765625, 0.495361328125, 0.08209228515625, 0.314453125, -0.0751953125, -0.71630859375, -1.150390625, 0.07568359375, 1.0703125, 0.401123046875, 0.02587890625, 0.62255859375, 0.41796875, -0.6318359375, 0.66650390625, 0.139404296875, 0.634765625, -1.3564453125, -0.1326904296875, 0.671875, 0.5322265625, 0.3115234375, 0.32177734375, -0.24853515625, -0.837890625, 0.53125, 0.167724609375, -0.262939453125, 0.303466796875, -0.7158203125, -0.2724609375, -0.8896484375, -0.1378173828125, -0.0306549072265625, 0.1839599609375, -2.951171875, 0.1904296875, -0.197998046875, 0.71630859375, -0.335205078125, -0.505859375, 0.6982421875, -0.046142578125, -0.032257080078125, 0.037200927734375, -0.5751953125, 0.175537109375, -0.308837890625, 1.240234375, 0.61376953125, 2.185546875, -1.5390625, 0.1534423828125, 1.0849609375, 1.4912109375, 0.35498046875, 0.71728515625, 0.939453125, -0.4677734375, -0.446044921875, 0.467529296875, 0.489501953125, 0.73583984375, -1.4501953125, -1.56640625, 0.056640625, 0.98876953125, 0.2890625, 1.2099609375, 0.9716796875, 0.486572265625, 0.0159454345703125, 0.352294921875, -1.3232421875, 0.54931640625, 0.91357421875, 1.0146484375, -1.0283203125, 0.69677734375, -0.865234375, 1.28515625, -1.59765625, 1.115234375, 0.591796875, -0.9765625, -0.1258544921875, -0.53369140625, -0.56689453125, 1.8037109375, -1.607421875, -0.306396484375, -0.6865234375, -0.40185546875, -1.076171875, 0.7900390625, -0.818359375, 1.666015625, 0.1376953125, -0.12353515625, -1.0029296875, -1.5263671875, -1.3974609375, -0.5341796875, -0.66796875, -1.1083984375, -0.402587890625, 0.43896484375, 0.1439208984375, -0.01279449462890625, 0.5283203125, 0.78759765625, 0.70458984375, -0.1541748046875, -1.2744140625, -0.425537109375, 1.1083984375, 0.339111328125, 0.9541015625, -0.41650390625, 0.494140625, 0.640625, 0.3896484375, 0.638671875, 0.0274505615234375, -1.1650390625, -1.2421875, -0.401123046875, 2.63671875, -1.6982421875, 0.1207275390625, 1.671875, 1.33984375, 0.6484375, 1.3232421875, 0.26513671875, -1.51953125, 0.11163330078125, 0.021942138671875, -0.097412109375, -0.9052734375, 0.2666015625, 0.2314453125, 1.6630859375, 0.712890625, -1.421875, 0.2122802734375, -1.9345703125, -0.1156005859375, -1.138671875, -0.322021484375, 0.478515625, -0.441162109375, 1.345703125, 0.342529296875, 0.66064453125, -1.4287109375, -0.5185546875, -0.72802734375, -1.16796875, 0.2587890625, 0.060302734375, 0.4482421875, 0.06536865234375, -0.21533203125, -0.521484375, 0.1572265625, 0.368896484375, -0.219970703125, 0.52685546875, 0.6884765625, 1.2724609375, 1.4814453125, 0.287109375, -0.7744140625, 0.66845703125, 0.9453125, -0.372802734375, -1.1572265625, -0.64990234375, 0.72216796875, 1.22265625, 0.27734375, -1.3056640625, 0.6279296875, -1.2041015625, 0.47998046875, -0.047454833984375, 0.50634765625, 0.18994140625, -0.75, 0.07861328125, -0.42041015625, 0.54296875, 0.76025390625, -2.017578125, -0.5625, -0.0271759033203125, 2.474609375, -0.64599609375, 1.2900390625, -0.0297698974609375, -1.7626953125, 0.2802734375, -0.06365966796875, -1.0634765625, 0.8486328125, 0.93896484375, -2.060546875, -0.025665283203125, 0.81982421875, 1.1884765625, 1.25390625, -0.705078125, 0.626953125, -0.180908203125, 0.414794921875, -0.93408203125, -0.98828125, -1.681640625, 0.3818359375, 0.82666015625, -1.322265625, -0.00494384765625, 1.00390625, 0.99658203125, -0.69580078125, -0.775390625, -0.11456298828125, 1.0673828125, 0.1339111328125, 0.0802001953125, 0.3623046875, -0.7421875, 0.083251953125, 0.6328125, -1.599609375, 1.3212890625, -0.73046875, 0.78515625, 0.1583251953125, 1.7919921875, -0.1939697265625, -2.267578125, 0.4248046875, 0.2210693359375, -1.880859375, -0.71240234375, 0.447265625, 0.28857421875, 0.78955078125, -1.310546875, -0.59765625, -1.8125, 1.2978515625, 0.3525390625, -0.986328125, 0.7431640625, -0.313232421875, -0.9072265625, 0.007617950439453125, -0.287109375, 0.271484375, -0.209228515625, -0.82421875, -2.89453125, -0.7529296875, -1.1806640625, 1.619140625, -0.328125, 0.56494140625, -1.2373046875, 0.71240234375, 0.57568359375, 1.1787109375, -0.8779296875, 0.5361328125, -0.2861328125, 0.81591796875, 1.4775390625, -0.029998779296875, -1.81640625, -0.355712890625, 0.16162109375, -0.304931640625, -1.0390625, -0.87109375, -0.78125, 1.4921875, 1.7138671875, 2.01953125, -0.421630859375, 0.1444091796875, -0.26904296875, -1.2275390625, 0.383056640625, -1.4736328125, 1.5732421875, -0.71435546875, -1.751953125, -0.93505859375, -0.03802490234375, 0.322998046875, 2.34765625, -0.471435546875, 1.0126953125, 0.68359375, 1.6162109375, 2.14453125, -0.75439453125, -0.137451171875, -1.53125, -0.53076171875, 0.46533203125, -1.318359375, -0.24658203125, -0.78173828125, 0.0501708984375, 1.658203125, 0.54296875, -1.2998046875, 1.0244140625, -0.51025390625, 1.5634765625, -1.5166015625, -1.3173828125, 1.3544921875, 0.177734375, -1.494140625, -0.11328125, -0.6904296875, -3.251953125, 0.63916015625, 1.462890625, -0.295166015625, 1.248046875, -1.3935546875, 0.1378173828125, -0.0303802490234375, -0.93505859375, -0.42431640625, 0.5087890625, -0.0173492431640625, 0.021087646484375, -0.430908203125, -0.77587890625, 0.44873046875, 1.775390625, 0.304443359375, -0.73681640625, 0.036041259765625, -0.09771728515625, 0.241455078125, -0.1168212890625, 0.226806640625, -0.92578125, -0.083251953125, 1.412109375, 0.74853515625, 0.14990234375, -3.01171875, 0.29296875, -2.05859375, 1.5732421875, 0.314208984375, 0.916015625, 0.3681640625, 0.5517578125, -0.037994384765625, 0.90625, 1.466796875, 0.060028076171875, -0.2105712890625, 1.1181640625, 0.99755859375, -0.0192718505859375, -0.2047119140625, 1.23046875, -0.833984375, 0.5341796875, 0.71923828125, 0.517578125, -0.453125, 1.1513671875, 1.7529296875, 0.288818359375, 1.04296875, -1.5107421875, -1.0166015625, 0.5439453125, -0.75, -0.18896484375, 0.8212890625, -1.05859375, 1.6806640625, -1.3955078125, -0.443603515625, 0.42822265625, -0.5478515625, 0.892578125, -0.3505859375, -1.0439453125, -0.63134765625, -0.86328125, -0.0310516357421875, 0.74169921875, 0.75439453125, 1.458984375, 1.49609375, -0.145751953125, 1.66796875, -0.97265625, 0.388671875, 1.9990234375, 1.064453125, -0.45361328125, 0.065673828125, -1.4599609375, 0.437744140625, 0.80322265625, -1.6435546875, 0.317626953125, 0.87353515625, 0.12481689453125, 0.357666015625, 0.71142578125, 1.1494140625, -0.0217742919921875, 0.44091796875, 0.4873046875, -0.139892578125, 0.165283203125, -1.4892578125, 1.365234375, 0.65673828125, -0.4521484375, -0.1910400390625, -0.5078125, -0.5947265625, -0.7568359375, 1.162109375, -0.7275390625, -0.2314453125, 0.68408203125, -1.4013671875, -1.103515625, 1.2392578125, 0.4150390625, 0.1474609375, -2.546875, -0.268798828125, 0.20361328125, -0.1995849609375, 0.01739501953125, 0.10772705078125, -1.025390625, 0.5830078125, 0.33251953125, 0.81103515625, -2.326171875, 0.31591796875, -0.97607421875, 1.44140625, 0.9921875, 1.513671875, -0.9765625, -0.43798828125, -1.9951171875, 1.1201171875, 0.257080078125, -1.349609375, 0.30810546875, -1.296875, 1.478515625, 0.537109375, -0.348876953125, 1.248046875, 1.22265625, 0.65771484375, 0.173095703125, 0.1917724609375, -0.1575927734375, -0.44140625, -0.54541015625, 1.3076171875, -0.1629638671875, -0.94287109375, 1.6171875, -0.221435546875, 1.9794921875, -0.6845703125, -0.387451171875, -0.26123046875, 0.208740234375, 0.5888671875, -2.623046875, 1.412109375, 1.4912109375, 0.09014892578125, 0.533203125, -0.18505859375, 0.380126953125, 0.89501953125, -0.231689453125, 1.759765625, -1.4541015625, 1.2978515625, 0.16259765625, -1.5732421875, 1.15234375, 1.4462890625, -0.8134765625, -0.03912353515625, -0.8212890625, 0.7421875, 0.341064453125, 0.6884765625, -0.7734375, 0.8310546875, 0.94189453125, -0.6328125, -0.81640625, 1.5283203125, 0.93798828125, 1.0830078125, 2.345703125, 0.0196533203125, -0.121826171875, -2.125, 0.46826171875, -0.4892578125, 0.8466796875, 1.84375, -0.10302734375, 1.1923828125, -0.93408203125, 0.1666259765625, 1.7001953125, 0.8271484375, -0.231689453125, 0.1571044921875, -0.1845703125, 0.31396484375, -2.646484375, 1.4677734375, 1.0634765625, -0.0195770263671875, 2.53515625, 0.79248046875, -1.06640625, 0.0665283203125, -0.49609375, 0.70947265625, -1.0537109375, -0.59130859375, 1.048828125, -0.52880859375, -0.7998046875, 1.3388671875, -0.07958984375, 0.53515625, -0.1702880859375, -1.4775390625, -0.463134765625, -0.1163330078125, 2.0625, -0.6005859375, -0.26708984375, 1.23046875, 0.285888671875, -0.830078125, -1.8154296875, -0.37060546875, 1.4990234375, -0.87646484375, -0.51904296875, -0.80078125, 0.392578125, 0.3935546875, -0.365234375, -0.8642578125, 0.94580078125, 0.476806640625, 1.083984375, 1.03125, 0.3134765625, -0.71923828125, 0.87890625, -1.169921875, 0.58203125, 0.64599609375, -0.059783935546875, 0.037689208984375, -1.3837890625, -1.025390625, 1.0361328125, 0.89208984375, -0.55419921875, -1.5615234375, -1.5927734375, 0.6845703125, -0.1912841796875, -0.72509765625, -0.057037353515625, 0.83984375, -0.09912109375, 1.138671875, -1.4228515625, 1.4619140625, 0.499755859375, -0.74560546875, -0.3720703125, 0.0386962890625, -0.736328125, 0.09930419921875, 0.5126953125, -0.83056640625, 0.61083984375, 1.224609375, 0.1827392578125, -0.236083984375, -0.3525390625, 0.313720703125, -1.16796875, 0.356689453125, -0.335693359375, 0.38330078125, -0.65185546875, -1.099609375, 2.203125, 1.125, -0.673828125, 0.73779296875, 0.48193359375, -0.1527099609375, -0.50927734375, 0.244384765625, -0.034698486328125, -0.39794921875, 0.51318359375, 0.619140625, -0.412109375, -0.39697265625, -0.04345703125, 1.263671875, -0.338623046875, 1.7451171875, 0.00789642333984375, 0.62841796875, -2.046875, -0.216064453125, 0.2353515625, 0.1390380859375, -1.595703125, 0.48291015625, 0.28173828125, 0.87548828125, 0.25830078125, 0.755859375, 0.218505859375, 0.66552734375, -1.498046875, -0.354736328125, 0.6044921875, -0.7353515625, 0.97802734375, -0.986328125, 0.08721923828125, -0.449462890625, -1.4580078125, -2.24609375, 0.78271484375, 1.0595703125, -0.0640869140625, -0.2239990234375, -1.2646484375, 0.837890625, 0.077392578125, 0.353759765625, -0.7998046875, 1.0, 0.8330078125, -0.046875, 0.2183837890625, -0.96826171875, -0.859375, 0.1728515625, 1.7041015625, -0.0105133056640625, -1.767578125, 0.61962890625, 1.2744140625, 0.0123443603515625, 1.5869140625, -0.75830078125, 0.56787109375, 1.7802734375, 0.32177734375, 2.744140625, 0.494384765625, -0.796875, -0.027374267578125, 0.035491943359375, 0.1427001953125, -0.11602783203125, -1.083984375, 0.7744140625, 1.072265625, 0.849609375, 0.93408203125, 0.328369140625, 2.29296875, 1.0849609375, -1.6923828125, 1.416015625, 0.00759124755859375, 0.355712890625, -1.578125, -1.30859375, -0.85107421875, 0.634765625, -0.9912109375, -0.86279296875, -0.2030029296875, 1.181640625, -0.4169921875, -1.193359375, 0.3720703125, -0.630859375, -1.470703125, -2.99609375, -1.357421875, -1.126953125, 0.81103515625, 0.8349609375, 1.7509765625, 0.6435546875, 0.1578369140625, 1.005859375, -0.56787109375, -0.9765625, -1.0859375, 0.955078125, -1.2138671875, -0.1329345703125, -0.35107421875, -0.87744140625, -0.129638671875, -0.403076171875, -0.69970703125, 1.359375, 1.25, -0.783203125, -0.6640625, -0.6103515625, -1.453125, -1.1142578125, 1.763671875, 0.15380859375, 0.861328125, -2.533203125, 0.828125, -1.0078125, 1.3564453125, -0.61572265625, 1.67578125, 0.203369140625, 0.79638671875, -1.025390625, 0.356201171875, -1.0205078125, 1.02734375, 1.203125, 0.433349609375, -0.048736572265625, -0.85791015625, 1.056640625, -0.182373046875, -0.8212890625, -0.4248046875, 0.99658203125, 1.654296875, -0.5380859375, -1.001953125, 0.310302734375, -0.031951904296875, 1.3505859375, -0.7470703125, -0.417236328125, -0.016448974609375, -0.492431640625, -0.01163482666015625, 0.60693359375, -0.299072265625, -0.205322265625, -0.4765625, -0.6357421875, -1.0029296875, 0.8486328125, 0.1751708984375, 0.0765380859375, 1.359375, 1.060546875, 1.494140625, -0.9228515625, -0.74853515625, 0.70361328125, -0.35791015625, -0.4501953125, -0.434326171875, 0.78466796875, 1.2001953125, -0.64599609375, 0.181884765625, 0.689453125, 0.7138671875, 1.1396484375, -0.2086181640625, -1.677734375, 0.041961669921875, 0.2080078125, -0.2039794921875, 0.278076171875, 1.8369140625, -0.615234375, -0.78857421875, 0.50537109375, -1.1435546875, -1.119140625, 0.65380859375, -1.7021484375, -1.984375, -1.6162109375, -0.54150390625, -0.21533203125, 0.53125, -0.27587890625, 0.755859375, -1.2744140625, -0.62353515625, -0.25537109375, -0.50537109375, 1.0703125, 1.658203125, -0.14501953125, 0.265869140625, 1.623046875, 0.07733154296875, -0.81103515625, -0.06951904296875, -0.56005859375, -0.515625, -0.41845703125, 0.0132293701171875, 2.9921875, -2.009765625, -0.113525390625, -0.50634765625, 1.072265625, -0.6826171875, -1.619140625, -1.09765625, 0.54248046875, -1.19140625, 0.38623046875, -1.61328125, -1.7568359375, -0.07012939453125, -0.439697265625, -1.2041015625, 0.05535888671875, 0.7744140625, -2.384765625, 0.405517578125, -2.98828125, 1.4990234375, -2.234375, -0.279541015625, 0.54638671875, -0.83447265625, 0.53369140625, 0.78759765625, 0.11279296875, -0.8466796875, -0.1533203125, -0.002532958984375, 0.70068359375, -0.139404296875, 0.38916015625, -0.91552734375, -0.1556396484375, 0.5107421875, -0.004215240478515625, -1.8974609375, -0.658203125, -0.49169921875, -0.5625, 0.79345703125, -0.6318359375, -0.845703125, 1.23046875, -1.1435546875, -0.2022705078125, 0.9189453125, 1.0302734375, 0.56982421875, 1.0185546875, -0.30126953125, -1.7978515625, -0.2213134765625, -0.0082550048828125, 0.94091796875, -0.58642578125, -0.29736328125, 0.208740234375, -0.73681640625, 0.12420654296875, 0.65771484375, 0.5185546875, -1.701171875, 0.325927734375, -0.81884765625, 0.5634765625, 0.47412109375, -0.1580810546875, 1.4365234375, 0.56201171875, 1.0615234375, -0.1982421875, -1.3076171875, -0.5830078125, -0.50634765625, 0.85009765625, -0.66845703125, 0.90185546875, -0.697265625, -0.445556640625, 1.4140625, -0.0093536376953125, -0.6201171875, -0.591796875, -1.0517578125, 1.0712890625, 1.478515625, -0.55517578125, 0.6923828125, 2.009765625, 0.2191162109375, -0.5703125, 0.65869140625, -0.75390625, -1.76171875, 0.168701171875, 0.74169921875, -0.193115234375, 0.83154296875, 0.298095703125, 1.8818359375, 0.412353515625, -0.5146484375, 0.260009765625, 1.044921875, -0.052459716796875, -0.78564453125, 1.208984375, 0.365478515625, -0.1893310546875, 1.0546875, 0.41845703125, -0.814453125, -1.3212890625, 1.0693359375, 0.395751953125, 1.4931640625, -0.261474609375, 0.869140625, 1.46484375, 0.274658203125, 0.3115234375, 0.595703125, 0.98681640625, -1.185546875, -1.1416015625, 0.08465576171875, -0.51123046875, 0.1348876953125, 0.250732421875, -0.7783203125, 0.0002338886260986328, 1.0498046875, 0.37353515625, 0.78125, 0.0711669921875, 0.76611328125, -0.10028076171875, 0.388671875, 0.5849609375, -0.95849609375, 0.95068359375, -0.46923828125, -0.9462890625, 0.0771484375, -0.328857421875, 1.1142578125, 0.5419921875, 0.2232666015625, 0.08270263671875, 1.4736328125, 1.044921875, -0.9384765625, 0.5908203125, 0.830078125, -0.52685546875, 0.212646484375, -0.6220703125, 0.62939453125, -1.28515625, 0.0302886962890625, -1.259765625, 0.267578125, 0.958984375, 0.55810546875, 0.7509765625, -0.033416748046875, -1.5576171875, 0.50244140625, -0.468994140625, -0.7265625, 1.6005859375, 0.8017578125, 1.58203125, 0.487548828125, -0.10052490234375, -1.3466796875, -0.55078125, -0.43701171875, 0.244384765625, -0.98681640625, 0.640625, 0.93798828125, -0.2802734375, -1.5634765625, 0.71630859375, 1.1484375, 1.2900390625, 0.80859375, 1.2607421875, -1.12890625, -0.297119140625, -0.08978271484375, 0.415771484375, 1.193359375, 0.59765625, -1.841796875, -1.1279296875, 1.46875, -0.85302734375, -0.06390380859375, 0.0859375, 1.044921875, 1.56640625, -0.55322265625, -0.1453857421875, -0.46728515625, -2.708984375, 0.274169921875, 0.17919921875, -0.3955078125, 1.2666015625, -0.72265625, -1.1884765625, -0.6220703125, -1.451171875, 0.78125, 0.1229248046875, -0.8046875, -0.346923828125, 1.1142578125, 0.72607421875, 1.62890625, -0.833984375, -0.045135498046875, -0.412353515625, -0.54150390625, 0.95654296875, 1.0556640625, -1.625, -1.806640625, 1.7880859375, -0.0987548828125, 2.154296875, -0.65869140625, 1.2412109375, -0.287109375, -0.1566162109375, -0.955078125, 0.55810546875, -1.7890625, -1.4296875, -0.60107421875, 0.437255859375, 1.09375, 1.5625, -0.29248046875, 0.6044921875, -0.552734375, -0.73876953125, 1.0087890625, -0.0031642913818359375, -0.2044677734375, 0.330810546875, -0.478515625, -1.1220703125, -0.238525390625, -1.615234375, 0.35595703125, -0.444580078125, 1.86328125, 0.2366943359375, 0.64990234375, -0.0139923095703125, 0.265869140625, 0.31201171875, -1.384765625, -0.52783203125, 0.9365234375, -0.55126953125, -0.2919921875, 0.6923828125, -1.2060546875, 0.16552734375, -0.53173828125, 1.1796875, 1.66015625, 0.305419921875, 0.405029296875, 1.005859375, 0.6552734375, 0.210205078125, 0.78857421875, 1.2998046875, -1.34375, 0.472412109375, 0.5908203125, -0.7705078125, 0.70068359375, -1.2978515625, -0.0848388671875, -1.041015625, 0.79541015625, 0.451416015625, -2.0625, -1.216796875, -0.02276611328125, 0.4765625, -0.47412109375, -0.75390625, 0.6318359375, -0.9140625, -1.111328125, -0.06591796875, 0.95166015625, -0.0217132568359375, 0.0513916015625, -0.033905029296875, -0.88427734375, 0.374755859375, -1.0751953125, -0.53662109375, -1.3134765625, -0.05938720703125, -0.71875, -0.80126953125, 0.568359375, -0.302734375, 1.021484375, -0.7666015625, -1.8359375, -0.57568359375, 2.73828125, -0.61474609375, -1.419921875, -0.414794921875, 0.104736328125, 1.4521484375, 0.88916015625, -0.0506591796875, -0.37060546875, 1.005859375, -0.261474609375, -0.2047119140625, 1.3642578125, 0.1822509765625, -0.398681640625, 0.396728515625, -0.152587890625, -0.1309814453125, 0.1448974609375, 0.5556640625, -0.09649658203125, 0.7216796875, 0.755859375, -0.9150390625, 0.79931640625, -0.283935546875, 0.59814453125, 0.2069091796875, 0.94775390625, -0.5859375, -0.75634765625, -1.6171875, 0.6376953125, 0.0506591796875, -0.439453125, -1.6943359375, 0.336181640625, 0.3251953125, -0.6220703125, -0.77587890625, 0.0116119384765625, -0.5810546875, -1.2880859375, 0.14111328125, -0.39013671875, -0.444091796875, 1.0068359375, 1.763671875, 0.73486328125, 1.03515625, -1.548828125, 0.68212890625, 0.2744140625, 1.1103515625, 0.364990234375, 0.2236328125, 0.39111328125, 1.146484375, 0.1539306640625, 0.5341796875, -0.9716796875, 0.414306640625, -0.85791015625, -1.0673828125, 1.1865234375, 2.873046875, -1.3544921875, -0.2359619140625, 1.9091796875, -0.79443359375, -0.1317138671875, 0.89599609375, -0.34765625, 0.4873046875, -0.469970703125, -1.01171875, 0.68896484375, 0.357177734375, 0.58935546875, -0.7451171875, -0.09503173828125, -0.2239990234375, -0.346435546875, 0.2259521484375, 0.93310546875, 1.251953125, -0.79345703125, -0.7421875, 1.8173828125, 0.56005859375, -1.1171875, 1.0537109375, 0.09124755859375, 0.1923828125, -1.4970703125, 0.99755859375, 1.1259765625, -0.53125, -0.529296875, 0.032012939453125, 0.6494140625, 0.6044921875, -0.7646484375, 0.042205810546875, -0.5146484375, 0.25537109375, 2.2109375, -1.39453125, -0.88330078125, 2.00390625, -0.01354217529296875, 0.380859375, -0.60302734375, -1.6708984375, -0.416748046875, 0.2125244140625, 0.1551513671875, 0.345458984375, -0.023162841796875, -0.9443359375, -0.284912109375, -0.56005859375, 0.8173828125, 0.93603515625, -0.10137939453125, 1.35546875, 0.00989532470703125, -0.2666015625, 1.765625, 0.1192626953125, 0.74560546875, 1.0322265625, 2.107421875, 0.50244140625, 1.6484375, 1.330078125, -0.8134765625, 1.53515625, -0.69140625, 0.435791015625, 0.60595703125, -1.1845703125, 0.53955078125, 0.7998046875, -1.09765625, -1.5859375, 0.36181640625, -0.03643798828125, -0.04296875, 1.7412109375, -1.5654296875, 0.68359375, -1.126953125, 0.444091796875, 0.60986328125, 1.8603515625, 0.9775390625, -0.46337890625, -0.494384765625, -0.289794921875, -0.78173828125, -0.494384765625, 0.453369140625, -0.49951171875, -0.171875, -0.143798828125, -0.057464599609375, -0.3671875, 0.861328125, -0.1240234375, 1.5166015625, 0.0460205078125, 0.129638671875, -0.6962890625, 0.55712890625, -0.0004429817199707031, 2.43359375, -0.255615234375, 1.28125, 0.7119140625, -0.042572021484375, 0.91357421875, -0.77490234375, -1.037109375, -1.3935546875, 0.379150390625, 1.361328125, 0.94482421875, -0.119384765625, -0.10418701171875, -0.76123046875, -0.5283203125, -2.587890625, 0.572265625, 0.042205810546875, -0.39599609375, -1.525390625, 0.2127685546875, 0.29931640625, 1.640625, -0.079833984375, -0.123046875, -0.88037109375, 1.7783203125, 0.162841796875, -0.48583984375, -0.7548828125, 2.22265625, 2.154296875, -0.72412109375, 0.289306640625, -0.51123046875, 1.162109375, -0.60546875, 0.720703125, -0.3408203125, 0.12451171875, -0.409912109375, -0.8173828125, -0.038299560546875, 2.109375, -0.375732421875, 0.27490234375, -0.59375, -0.11669921875, -0.6650390625, -0.446533203125, -0.47119140625, -1.384765625, -0.568359375, -0.30908203125, 0.91845703125, -0.73486328125, 0.6044921875, -0.94482421875, 2.072265625, 0.204345703125, -0.908203125, -0.279052734375, -0.30078125, -0.5400390625, -0.0537109375, 0.8642578125, 0.23046875, -0.27783203125, 0.1502685546875, -1.0712890625, 0.53515625, 1.3310546875, -0.97412109375, -2.392578125, -0.033843994140625, -0.974609375, -0.27392578125, 0.15625, -0.28125, 0.5849609375, 0.92529296875, -1.56640625, 1.7109375, -1.3515625, 2.287109375, 1.056640625, -0.9111328125, 0.51220703125, -0.82763671875, 1.73046875, 0.260986328125, 1.05078125, 1.7294921875, -1.080078125, -0.53271484375, -0.9775390625, 1.458984375, -0.122314453125, -1.125, -0.79345703125, 0.034149169921875, -2.12890625, 0.5263671875, -0.0106964111328125, -0.2332763671875, -0.168701171875, -0.326416015625, -0.1932373046875, -1.259765625, 2.037109375, 2.28515625, 0.7060546875, -2.408203125, 0.96923828125, -0.75048828125, -0.2783203125, -0.3154296875, -0.424072265625, 0.396240234375, -0.84033203125, -1.5166015625, 0.65478515625, 1.3837890625, 0.336181640625, -0.378662109375, -1.396484375, -1.5947265625, -0.31005859375, 0.387939453125, -0.467529296875, 1.8193359375, -0.94921875, 1.5087890625, 0.52197265625, -0.88671875, -1.041015625, 0.381591796875, 1.03515625, 1.89453125, -0.7958984375, 0.5439453125, 0.1181640625, -0.354248046875, 1.2646484375, -0.2420654296875, 1.8388671875, -1.7099609375, 2.173828125, 1.1435546875, 0.273193359375, 0.79052734375, 1.2607421875, -2.029296875, 0.0765380859375, 0.029693603515625, -1.1767578125, 1.095703125, 1.25, -0.48193359375, -2.033203125, 0.4150390625, -0.48876953125, -0.1754150390625, -1.5849609375, -0.36572265625, -0.038543701171875, 1.728515625, 1.5478515625, 0.0208740234375, 0.701171875, 0.07318115234375, 0.301513671875, -0.78271484375, 1.1328125, -2.0859375, -2.50390625, 0.388916015625, -0.322509765625, 0.0985107421875, 0.060455322265625, -2.3984375, -1.2939453125, -1.056640625, 0.036468505859375, -0.0693359375, -0.055511474609375, 2.3359375, 0.07952880859375, -0.4111328125, 1.2939453125, 0.395751953125, 0.7548828125, 1.1083984375, -0.188720703125, 0.33935546875, 1.5712890625, -0.65234375, -1.4677734375, 0.68017578125, -1.533203125, -1.927734375, 0.137939453125, -0.73779296875, -1.673828125, 0.8544921875, -1.033203125, 0.2568359375, -0.040313720703125, 0.28955078125, 1.5029296875, 1.009765625, 0.77783203125, 0.919921875, 0.482666015625, 1.5703125, -0.01081085205078125, 0.6416015625, -2.51171875, -1.2470703125, -1.7041015625, 1.2109375, -2.3671875, 0.201904296875, 0.55126953125, 1.330078125, -0.1251220703125, 1.4208984375, 0.0350341796875, -0.90869140625, -0.2462158203125, -0.08984375, 0.7958984375, -1.0380859375, -0.31982421875, -1.4296875, 1.158203125, 0.2474365234375, 1.7373046875, 1.1376953125, -0.446044921875, -0.8759765625, -1.1953125, -0.7451171875, -0.4892578125, -0.759765625, 0.1922607421875, -0.8994140625, 0.56298828125, 1.2900390625, -1.099609375, -0.7509765625, 0.40478515625, -0.25146484375, -0.01346588134765625, 1.046875, -0.309326171875, 0.62109375, -1.0673828125, -1.7587890625, -1.1904296875, 1.330078125, 0.385498046875, -2.470703125, 1.208984375, -0.76416015625, -0.049224853515625, 1.5498046875, 0.388427734375, -0.41943359375, 0.414306640625, 1.2958984375, -0.67626953125, 0.51611328125, -0.25, 0.89892578125, 0.10479736328125, -0.6611328125, -0.73974609375, 0.37353515625, 0.357666015625, 3.501953125, -0.373046875, 0.58984375, 0.7255859375, -0.701171875, 0.447509765625, 1.1728515625, -0.513671875, 0.053955078125, 1.6630859375, 0.74267578125, -1.1728515625, 0.6806640625, 0.258544921875, -1.1845703125, 0.99365234375, -0.93310546875, -1.994140625, 1.482421875, 1.43359375, -0.64306640625, -0.1279296875, -0.5908203125, -1.1357421875, 1.0732421875, 0.2169189453125, -1.0, 1.0380859375, -0.019500732421875, 0.5419921875, 1.4189453125, 0.4833984375, -0.143798828125, 0.59033203125, 2.46875, -0.2529296875, 0.55908203125, 1.45703125, 1.1201171875, 0.32861328125, -1.791015625, -1.6201171875, -0.424072265625, -0.1326904296875, -2.009765625, 0.162353515625, -0.425537109375, 0.35302734375, -0.0202178955078125, -0.33447265625, 1.2734375, -0.79150390625, -1.6630859375, 1.1884765625, 2.0703125, 0.80126953125, -0.759765625, -1.05078125, -1.10546875, 0.5927734375, 0.791015625, -0.2083740234375, 0.8544921875, -1.3388671875, 0.61279296875, -1.083984375, 0.9560546875, 0.74755859375, -0.375732421875, 1.60546875, -0.5048828125, 0.1680908203125, 2.27734375, 0.81494140625, -0.6796875, 0.8310546875, -0.364013671875, 0.82861328125, -0.576171875, -0.2174072265625, -1.173828125, -1.33984375, -0.388671875, -0.29931640625, -0.3486328125, -1.1982421875, 0.5966796875, 0.2147216796875, 0.63134765625, -1.7939453125, -0.900390625, -0.319091796875, -0.3955078125, 0.20263671875, 2.345703125, 0.2413330078125, -0.82177734375, -0.7890625, 0.036956787109375, 0.662109375, -2.1171875, -0.93603515625, 1.3251953125, -1.1982421875, -0.8935546875, 0.36474609375, 1.021484375, -0.6279296875, 0.322021484375, 1.7734375, 0.48291015625, 0.499267578125, -0.0958251953125, 0.71044921875, -0.324462890625, -1.341796875, -0.53662109375, -0.9169921875, 0.2294921875, -0.10791015625, -1.4013671875, 2.140625, 0.701171875, -1.2724609375, -0.57763671875, -0.134765625, -1.33203125, -1.2373046875, -2.26171875, -0.224365234375, 0.5390625, 1.5087890625, 0.3232421875, 0.0019664764404296875, 0.378662109375, -0.7607421875, 0.16650390625, -1.3408203125, 0.0682373046875, -0.267822265625, 0.27197265625, -0.05169677734375, 0.80908203125, -0.78466796875, 0.70458984375, 0.8486328125, 0.52099609375, 0.0980224609375, 0.194580078125, 1.673828125, -0.06842041015625, 1.396484375, 0.2147216796875, -1.5888671875, -0.44287109375, 0.72119140625, -1.373046875, -0.7060546875, -0.720703125, -1.29296875, 1.861328125, -1.0478515625, -0.0706787109375, 0.4658203125, -1.2197265625, -1.9267578125, 0.2491455078125, -0.364501953125, -1.5673828125, 0.09698486328125, -1.3603515625, 0.68798828125, 1.10546875, 1.689453125, 0.140869140625, 0.53369140625, 1.6552734375, 0.12310791015625, 1.630859375, 2.166015625, 0.259765625, -1.2705078125, 0.9287109375, 0.0869140625, 1.3984375, -1.5283203125, -1.3427734375, 0.7412109375, -0.9833984375, 1.52734375, -0.3720703125, 0.5673828125, 0.8837890625, -0.62744140625, -2.294921875, 0.92529296875, 1.2548828125, 1.20703125, -0.1473388671875, 0.043701171875, 0.1396484375, 1.78515625, -0.120361328125, -0.990234375, 1.2890625, 0.301025390625, 0.0462646484375, 2.267578125, 0.241943359375, -0.583984375, 0.69775390625, 0.38427734375, -0.71044921875, -1.9482421875, 0.36962890625, 0.8369140625, 0.91845703125, 0.91455078125, 0.0655517578125, -1.3193359375, 0.1123046875, 0.9794921875, -0.14208984375, 1.189453125, -0.67041015625, -0.302978515625, 0.10467529296875, -2.044921875, 0.27392578125, 0.84619140625, -0.412109375, 2.11328125, 0.28955078125, -1.1962890625, 1.7392578125, -0.218017578125, 1.4580078125, 0.52587890625, 0.366455078125, 0.59814453125, -0.42626953125, -0.450439453125, -0.13134765625, 1.474609375, -0.200927734375, -0.9482421875, -0.46875, 0.8466796875, 0.66796875, -0.9462890625, -0.83154296875, -0.2548828125, -0.9921875, 0.2724609375, 0.03515625, 0.0135650634765625, 0.5234375, -0.6640625, 0.62646484375, 0.43701171875, 0.994140625, 0.88818359375, -0.79638671875, 1.09765625, 0.34521484375, -1.1533203125, 0.31005859375, 0.1260986328125, 1.7177734375, -0.93408203125, 0.47705078125, 0.154296875, -1.3818359375, -0.21337890625, 0.60302734375, -1.15625, 1.0087890625, -2.6875, -0.00881195068359375, -0.56103515625, -0.0684814453125, -2.03125, -0.07354736328125, -2.447265625, 0.336669921875, 0.17236328125, -1.7490234375, -0.2392578125, -0.317626953125, 0.8173828125, -1.017578125, -1.2734375, -0.488525390625, 0.52783203125, -0.468505859375, -0.505859375, -0.69189453125, -0.720703125, -1.2890625, 1.3955078125, 0.90771484375, -0.294921875, -0.775390625, -0.332275390625, 0.53662109375, -0.96337890625, -0.642578125, 0.82373046875, 0.2137451171875, -0.207763671875, -0.3955078125, 1.107421875, -0.6796875, 1.04296875, -0.3740234375, -0.95654296875, 0.4912109375, 1.4296875, -0.6123046875, 2.240234375, -1.861328125, -1.400390625, -0.357666015625, -0.00931549072265625, 1.0400390625, 0.09521484375, 0.99267578125, 0.8583984375, -0.6669921875, 0.56298828125, -0.9931640625, 0.260498046875, 0.8935546875, -0.70654296875, 1.046875, -0.133056640625, 0.1571044921875, 0.5, -1.9560546875, -0.39306640625, 0.47412109375, 1.8720703125, -0.0049285888671875, -1.349609375, 1.1865234375, -0.57275390625, 2.556640625, 1.3154296875, 0.0859375, 1.7333984375, 2.060546875, -0.67333984375, -1.4599609375, -2.123046875, 0.35693359375, 1.37890625, -2.171875, 1.3759765625, -2.037109375, -0.40966796875, 0.201416015625, 0.1566162109375, -1.7216796875, 0.07916259765625, -0.94921875, 0.0224151611328125, -0.1373291015625, 0.65625, 0.62841796875, -0.9931640625, -0.22314453125, -1.26171875, -0.146728515625, -0.25244140625, -0.45849609375, -0.57080078125, 0.413818359375, -0.297119140625, 1.8466796875, 1.8349609375, -0.316162109375, -0.7587890625, -0.64501953125, 0.1856689453125, 1.0810546875, -0.517578125, 2.294921875, -1.6044921875, -0.3359375, -0.1968994140625, -0.5595703125, -0.19580078125, 0.6513671875, 0.2230224609375, 0.6220703125, 0.58544921875, 1.64453125, -1.59765625, 0.438232421875, 0.87744140625, 0.583984375, 1.3359375, 0.206298828125, -0.44140625, 1.3095703125, -0.2177734375, 1.2548828125, -1.3134765625, 0.128173828125, 0.818359375, 0.70068359375, 0.509765625, -0.345458984375, 0.5283203125, 1.5927734375, -0.345947265625, 1.75390625, 0.411376953125, 0.1361083984375, -0.362060546875, -0.05328369140625, 1.02734375, 1.7392578125, 0.412841796875, 0.8623046875, 0.89990234375, 0.413330078125, -0.406982421875, -2.419921875, 0.329345703125, 1.060546875, 0.449951171875, 0.0960693359375, 2.12890625, 1.5322265625, -1.5380859375, 0.751953125, 0.399169921875, 1.1826171875, -0.239501953125, -2.3359375, -0.29736328125, -0.53369140625, 0.94140625, 0.1856689453125, 0.97216796875, 1.5595703125, 0.32568359375, 1.34765625, -0.10809326171875, -0.487060546875, 0.197509765625, -0.1748046875, 1.2529296875, 0.10797119140625, 0.142822265625, 1.67578125, -1.8564453125, 0.69970703125, -0.059844970703125, 1.189453125, -1.412109375, 0.61328125, 0.357666015625, 1.451171875, 0.484619140625, -1.3349609375, 1.3759765625, 0.2059326171875, -0.144287109375, 0.10107421875, -0.8955078125, -0.32958984375, -0.6884765625, 2.017578125, 0.2349853515625, -1.5322265625, -0.366455078125, -1.294921875, 0.079833984375, -1.2138671875, 0.363525390625, 0.27734375, 1.095703125, 0.3759765625, 0.1781005859375, 2.349609375, -0.9365234375, 0.580078125, -0.036895751953125, -0.43896484375, 0.74658203125, 0.041168212890625, -1.2548828125, -1.1767578125, 0.76318359375, -1.4931640625, -1.888671875, -0.74658203125, -0.58056640625, 0.482177734375, 0.88427734375, 1.7724609375, -0.72216796875, -1.9921875, 1.20703125, -0.045989990234375, -0.8603515625, -0.4677734375, 0.5947265625, -0.58056640625, -1.0576171875, -0.5576171875, 0.125, -0.125732421875, 0.4423828125, 0.399169921875, -0.85986328125, 0.8525390625, 0.8583984375, 0.00970458984375, 0.72998046875, 0.5810546875, -0.74267578125, 0.245361328125, -0.5751953125, -4.7421875, -0.8486328125, -0.35546875, 0.67724609375, -0.8642578125, -0.5087890625, -0.1982421875, 0.41357421875, 0.4365234375, -0.89111328125, -1.0576171875, 1.51171875, 0.9423828125, 0.22705078125, -0.284423828125, 0.78515625, -1.4375, -0.09381103515625, 0.420654296875, 0.818359375, -0.495849609375, -0.478271484375, 0.2379150390625, 1.0947265625, -0.2191162109375, 0.415771484375, 0.34814453125, 0.322021484375, 0.007659912109375, 1.259765625, -0.77490234375, 0.69921875, 1.064453125, -0.76953125, 0.056182861328125, -0.630859375, 0.324951171875, 0.796875, -1.3056640625, -2.021484375, 0.395263671875, -0.568359375, 0.75732421875, 0.2193603515625, 0.81982421875, -0.67431640625, -0.452880859375, -0.46337890625, 0.693359375, -1.177734375, 0.79248046875, -0.654296875, -0.71826171875, 0.2890625, -0.59423828125, -0.29345703125, -0.8798828125, 0.671875, 0.9267578125, 0.84814453125, 1.1650390625, 0.1337890625, -1.3828125, 0.440673828125, 1.6591796875, 0.03460693359375, -0.873046875, 0.43212890625, 0.15087890625, -1.0869140625, -0.583984375, 0.350341796875, 1.3720703125, -0.98828125, 2.04296875, -1.078125, 0.357177734375, 0.97314453125, -1.939453125, -0.17578125, 0.31689453125, -0.391357421875, 1.2548828125, -1.373046875, 0.250732421875, 1.4013671875, -0.697265625, 1.1845703125, 0.96923828125, -0.41259765625, -1.873046875, 1.267578125, -1.1259765625, -0.37841796875, 1.3466796875, 0.342041015625, 0.1453857421875, -0.395263671875, 0.87255859375, -0.454833984375, 0.28076171875, 1.4892578125, -0.032196044921875, 0.04254150390625, 0.81103515625, 0.2430419921875, 0.318115234375, 0.59326171875, 0.416015625, 1.369140625, -1.068359375, -0.85400390625, 0.1376953125, -1.447265625, -0.17724609375, 0.408203125, -0.73388671875, -2.650390625, -1.0185546875, -1.318359375, -0.40673828125, 1.7900390625, 2.408203125, 0.203857421875, 0.6416015625, 0.336669921875, 0.03485107421875, -1.2294921875, 1.0390625, -0.9462890625, 0.2403564453125, 1.271484375, 0.348388671875, -0.84716796875, 2.0390625, -0.7451171875, 0.751953125, -0.7265625, 1.7587890625, 0.71875, 1.2265625, 0.4970703125, -0.91357421875, -2.607421875, 1.1005859375, -1.015625, 0.6025390625, 0.4501953125, -0.8330078125, -0.1983642578125, -1.0966796875, 0.71240234375, -0.93994140625, 0.2115478515625, -0.87353515625, 0.482666015625, 0.1409912109375, 0.435791015625, -1.486328125, -0.2239990234375, 0.0216522216796875, 1.701171875, -1.7958984375, 0.315673828125, -1.34375, 0.04510498046875, -0.1431884765625, 0.30712890625, -1.0791015625, 0.11187744140625, -0.51806640625, 0.2208251953125, -0.3154296875, 0.6611328125, -1.6650390625, 2.375, -0.505859375, 1.2373046875, 0.86767578125, -0.338623046875, 0.65869140625, -1.0078125, 0.4814453125, 1.736328125, -0.033416748046875, -2.73046875, 0.61865234375, 0.03790283203125, -1.0283203125, -1.017578125, 0.1217041015625, 2.48828125, 0.2147216796875, 0.74462890625, -1.765625, -0.0640869140625, 1.6513671875, -0.9326171875, -0.051116943359375, -0.4287109375, 0.95556640625, 0.1937255859375, -0.693359375, -0.876953125, -0.98388671875, 1.6318359375, -0.880859375, -0.1962890625, 2.068359375, -1.1044921875, -1.3369140625, 0.112060546875, -1.91796875, 0.0712890625, 0.37158203125, 1.5673828125, 1.8310546875, 0.39306640625, 1.1201171875, 0.54736328125, 0.74658203125, -1.0078125, 0.63427734375, 2.447265625, 0.89208984375, -0.1328125, 0.61962890625, -0.006847381591796875, -0.5771484375, 0.2486572265625, -1.138671875, -0.6640625, -1.08203125, 1.12109375, -1.45703125, 0.79248046875, 0.51611328125, -0.650390625, 0.33251953125, 0.3173828125, 1.658203125, 2.16796875, 0.6435546875, 0.8525390625, 0.276123046875, -0.85595703125, -0.1588134765625, -0.96630859375, -0.107666015625, -2.169921875, -0.035919189453125, 0.3779296875, -0.2052001953125, 1.2138671875, 0.90087890625, 0.53369140625, -1.1669921875, 0.490966796875, -1.193359375, 1.8955078125, 0.82666015625, 1.6220703125, 0.50390625, 1.5654296875, 0.845703125, -0.413818359375, -0.78076171875, 0.31298828125, -0.467041015625, -0.2354736328125, -0.3642578125, 0.52978515625, -1.8232421875, -0.1427001953125, 0.1748046875, 0.150146484375, -0.272216796875, -1.9599609375, 0.01467132568359375, 0.5126953125, 2.462890625, -1.8232421875, 0.162353515625, -0.66796875, -1.7216796875, -0.580078125, -0.6484375, 0.6611328125, 0.81591796875, 0.88916015625, -0.78955078125, -1.8125, 1.38671875, -0.31689453125, -1.568359375, -0.229736328125, 0.56494140625, 1.001953125, -0.67626953125, -1.30078125, -0.603515625, 0.6796875, -0.325439453125, 0.72021484375, 0.302734375, -0.499267578125, 0.292724609375, -0.56982421875, -0.245849609375, -0.59912109375, -1.775390625, -1.6748046875, 0.325927734375, -1.048828125, 0.09710693359375, 1.2900390625, 0.70849609375, -0.45947265625, -0.28857421875, -0.38720703125, 0.61669921875, -1.2412109375, 0.0411376953125, 0.62060546875, -0.70654296875, -0.97119140625, 0.09063720703125, -1.0322265625, 0.431640625, 0.20068359375, 0.2158203125, -0.08642578125, -0.60302734375, -0.83984375, -1.08984375, -1.4091796875, -1.8046875, -1.888671875, -0.050811767578125, 0.6533203125, -1.244140625, 1.1064453125, -0.99609375, -0.52001953125, -0.39111328125, 1.9599609375, 1.1494140625, 1.166015625, 0.6884765625, -0.736328125, 0.87255859375, 0.5703125, 0.2900390625, 1.1669921875, -1.193359375, -0.78369140625, 0.395263671875, -0.5751953125, 0.56396484375, -0.9833984375, -0.1866455078125, 0.89306640625, -0.72119140625, -0.282470703125, -1.3994140625, 0.052001953125, 1.15625, -2.275390625, -0.1859130859375, 0.16064453125, 1.0634765625, 1.1279296875, -0.2459716796875, -0.74560546875, 0.57373046875, -1.2734375, -0.75390625, 0.59716796875, -0.9375, 0.035186767578125, -0.380126953125, -1.67578125, -0.6455078125, 0.43701171875, -0.65380859375, 0.273193359375, -0.67578125, 0.2476806640625, -0.78759765625, 0.4638671875, -0.11865234375, 0.9658203125, 0.410400390625, -0.00290679931640625, 0.213134765625, -0.6171875, -1.0078125, 0.2880859375, 0.8271484375, 1.072265625, 0.69287109375, 0.2108154296875, 0.71923828125, 1.939453125, -0.06158447265625, -0.056488037109375, -0.18359375, 0.50927734375, 0.01120758056640625, -0.43505859375, 0.325439453125, 0.9345703125, 1.52734375, -0.18359375, -1.1396484375, -0.81982421875, -1.505859375, 0.01122283935546875, -0.724609375, -1.6630859375, -0.9072265625, 1.5078125, 2.197265625, 0.10467529296875, 0.51123046875, -0.493408203125, 1.107421875, 0.70458984375, -0.201904296875, -0.0703125, -0.63818359375, 0.242919921875, -0.054168701171875, 0.169189453125, -0.54345703125, -0.1031494140625, 0.7431640625, 1.974609375, -0.67236328125, 0.474853515625, 0.26318359375, -1.8193359375, -0.10430908203125, 0.432861328125, 0.2352294921875, -0.408447265625, -0.375, 0.002941131591796875, 1.1123046875, 0.57080078125, 0.480712890625, -0.86474609375, -0.468994140625, -1.06640625, -0.892578125, 1.3642578125, -0.373046875, 0.7099609375, -1.1337890625, 0.362060546875, 0.186279296875, -0.86669921875, -0.4677734375, 0.377197265625, -0.86572265625, -0.359375, 1.7919921875, 0.385009765625, 0.391357421875, 0.7939453125, 0.007556915283203125, -1.2177734375, -1.375, -1.2353515625, -1.3212890625, 0.75439453125, 1.0107421875, 0.54052734375, 0.05426025390625, 0.60986328125, 0.7861328125, -1.9580078125, 0.12347412109375, 0.57958984375, 0.10406494140625, 0.2376708984375, -0.099853515625, 1.08984375, 0.80517578125, 1.6083984375, 0.5458984375, -0.2421875, -0.7158203125, 0.10894775390625, 0.9580078125, 0.599609375, 0.490966796875, 1.2861328125, 1.111328125, -0.70458984375, 0.059539794921875, -0.339111328125, 1.662109375, -0.438232421875, -0.13623046875, 0.080322265625, -1.869140625, -0.94580078125, -1.4365234375, 0.0908203125, -0.60205078125, -0.29931640625, -2.3046875, -1.869140625, -0.25732421875, 1.6142578125, -0.447509765625, -1.3623046875, -1.7216796875, 1.984375, 0.343505859375, 0.03118896484375, 0.330810546875, 1.1572265625, 0.2105712890625, 1.025390625, -0.483642578125, 1.9677734375, 0.1676025390625, -0.83642578125, 1.482421875, -0.383544921875, -1.4404296875, 0.1585693359375, -1.4150390625, -0.4111328125, 1.2578125, 0.4951171875, -0.546875, 2.724609375, 0.70361328125, 1.19921875, 0.09185791015625, -0.33056640625, 1.2177734375, -0.478271484375, 0.919921875, -0.796875, -0.21826171875, -0.0947265625, 0.603515625, 1.3935546875, 0.6025390625, 0.6201171875, 2.369140625, -0.0911865234375, 0.2373046875, 0.29345703125, -0.1123046875, -2.193359375, -0.89990234375, -1.0869140625, 0.3466796875, 1.2529296875, 0.966796875, -0.72021484375, -0.330322265625, -0.412109375, -1.4189453125, 0.45654296875, 0.8623046875, -0.8173828125, 0.6865234375, -0.5830078125, -1.48828125, 0.720703125, -0.7568359375, -0.51416015625, 0.334716796875, -0.2071533203125, 1.6318359375, 2.078125, -0.25390625, -0.44091796875, -1.1982421875, 0.72021484375, -0.75244140625, 2.123046875, 0.2161865234375, 0.73974609375, -0.515625, -0.6806640625, -0.8896484375, -0.95068359375, -0.6494140625, -1.0712890625, -0.240478515625, 0.45556640625, -1.197265625, 0.91015625, 1.3232421875, 0.019989013671875, 0.9609375, 1.1240234375, 2.21875, 2.87890625, 2.193359375, 0.404296875, 0.59912109375, -1.19921875, -0.8857421875, 0.035247802734375, 1.498046875, -0.1676025390625, -1.1904296875, 0.1727294921875, -1.107421875, 0.87255859375, -0.3642578125, 2.857421875, 0.1722412109375], "dtype": "float16"}, {"__nd__": [1.43359375, -0.20751953125, -0.72412109375, 1.7666015625, 0.9990234375, 1.279296875, -1.3115234375, 1.0498046875, 1.9599609375, 1.0986328125, -1.376953125, 0.48095703125, 2.85546875, -0.873046875, 0.2069091796875, -0.1900634765625, 1.533203125, -0.5009765625, 0.1988525390625, -0.3759765625, 1.8994140625, -0.6416015625, 2.380859375, 1.9853515625, -0.625, -0.328125, -1.6884765625, -0.86279296875, 1.8193359375, -0.859375, 1.509765625, -1.765625, 0.119384765625, 0.32373046875, 0.1893310546875, 0.3837890625, 0.5234375, 0.2003173828125, -2.259765625, 0.9443359375, 0.53125, 0.75048828125, 0.0175323486328125, -0.0352783203125, 0.376708984375, 0.75390625, -2.060546875, 0.1412353515625, -0.4404296875, 1.8837890625, -1.0888671875, 0.2147216796875, 1.123046875, -0.123046875, -0.40234375, -0.5166015625, -1.294921875, 1.0205078125, -0.0153045654296875, 0.050811767578125, -0.162841796875, 0.8828125, -1.962890625, 1.9140625, 0.8017578125, -1.01953125, 2.1796875, -0.356201171875, 0.2452392578125, 0.93798828125, -0.9287109375, -1.4130859375, -0.61767578125, 0.7158203125, 1.0439453125, -2.30859375, -0.116455078125, -0.40478515625, -0.2015380859375, 0.33447265625, 0.60595703125, -1.435546875, 0.9501953125, 2.08203125, 0.96484375, -0.88623046875, 1.029296875, 1.5400390625, -1.5556640625, 0.25732421875, 0.1314697265625, -1.7626953125, -1.5732421875, -0.057891845703125, 0.63916015625, -0.16015625, 0.28564453125, 0.12939453125, -0.54736328125, -1.1904296875, 0.703125, -0.306640625, 1.3193359375, 0.318115234375, 0.74658203125, -0.42138671875, -2.041015625, -0.58154296875, -0.98388671875, -1.0380859375, 0.32958984375, -0.5791015625, -0.78662109375, 1.0166015625, 0.1947021484375, 0.90185546875, -0.33740234375, -0.544921875, -0.317626953125, -1.2529296875, 1.4375, -0.65673828125, -0.316650390625, 0.389892578125, -1.296875, -0.431640625, -0.344970703125, 1.408203125, 1.24609375, 0.045684814453125, -0.204833984375, -0.6943359375, 0.00507354736328125, 0.53564453125, -0.20263671875, -0.95166015625, -0.44677734375, 0.153564453125, -0.32275390625, -2.57421875, -0.303466796875, -1.728515625, -1.5419921875, -0.845703125, 0.51318359375, -0.60546875, 0.8623046875, -1.6181640625, -0.09271240234375, 2.337890625, -0.6552734375, -1.5302734375, -0.5712890625, -1.51171875, -0.06549072265625, 0.99560546875, 0.373046875, -0.580078125, -0.6044921875, -0.935546875, 1.3388671875, 0.7578125, 1.5625, 0.7060546875, -0.447998046875, -0.47607421875, 0.83740234375, -0.48779296875, -0.27685546875, 2.162109375, 0.01360321044921875, -0.05682373046875, 1.92578125, 0.7431640625, 0.273193359375, -1.7568359375, -0.8720703125, 1.55859375, 0.62939453125, 1.4482421875, -0.498291015625, -0.97265625, 0.41015625, 0.52783203125, -0.9521484375, -1.2568359375, -0.8544921875, 0.18603515625, -0.865234375, -0.837890625, -0.455078125, 1.7822265625, -0.9892578125, -1.052734375, 0.2471923828125, 1.1220703125, 0.63134765625, 1.451171875, -0.53759765625, -1.3466796875, -0.779296875, 0.10296630859375, -0.96923828125, -0.5244140625, -2.126953125, 1.529296875, -0.06719970703125, -1.6513671875, 1.6884765625, 1.6298828125, 0.8447265625, -1.017578125, 0.417236328125, -0.091064453125, 0.2666015625, 1.119140625, -1.0654296875, -0.91796875, 0.0030574798583984375, 0.4287109375, -0.8896484375, -0.94384765625, -0.12060546875, 2.1875, -0.059722900390625, 1.5302734375, 2.05859375, 0.2449951171875, -0.269775390625, -0.91552734375, -1.1982421875, 0.96923828125, -0.546875, 0.009674072265625, 0.59814453125, -1.1962890625, -1.4248046875, 0.95361328125, 1.0283203125, 0.9296875, 0.927734375, 1.05078125, 0.74560546875, 0.323974609375, 0.853515625, 0.42333984375, -0.78564453125, 1.3056640625, 0.33056640625, 1.2578125, 0.44384765625, -0.04083251953125, -0.33349609375, 0.274169921875, -1.2275390625, -0.482421875, -0.1260986328125, -1.3857421875, 0.658203125, -2.0390625, 0.451171875, -0.2117919921875, 0.6572265625, 0.2120361328125, -0.03289794921875, 1.3818359375, 0.392333984375, 0.484375, -0.95166015625, 1.2275390625, 1.091796875, -0.31494140625, -0.568359375, -1.7314453125, 1.3583984375, 0.58544921875, 0.2607421875, 0.91943359375, -0.60498046875, -1.3125, -0.6376953125, -0.5390625, 0.6279296875, 0.06439208984375, -0.77001953125, -1.0546875, -0.292724609375, 0.9501953125, 0.275634765625, -0.337158203125, -0.67431640625, 1.310546875, -1.669921875, 1.4287109375, -0.39794921875, 0.666015625, -0.9619140625, -0.0655517578125, -1.6533203125, 0.55615234375, 0.2000732421875, -0.98046875, 0.328857421875, 0.1488037109375, 0.52880859375, -0.2110595703125, 0.5400390625, 1.154296875, -1.5947265625, -0.6064453125, 0.84326171875, 0.521484375, -0.012054443359375, 0.019744873046875, 0.81201171875, 0.06982421875, 0.2132568359375, 0.1317138671875, -0.68310546875, -0.3427734375, -1.5517578125, 1.5234375, -0.1822509765625, -0.2061767578125, -0.4580078125, 1.083984375, -2.048828125, -0.0257415771484375, -1.9404296875, 1.0576171875, -0.455810546875, 0.50439453125, 0.7529296875, 0.2200927734375, 0.54345703125, -1.3427734375, -0.356201171875, 0.95361328125, 1.5673828125, 1.75390625, -0.31103515625, 0.2252197265625, 0.34326171875, 0.333740234375, -0.533203125, -0.1688232421875, 0.4033203125, 0.4052734375, 0.54638671875, 1.841796875, -1.78515625, 0.69482421875, 0.46728515625, 1.9794921875, 0.360595703125, -0.137451171875, -0.24169921875, -0.67431640625, 1.40234375, 0.69091796875, -1.68359375, 0.0303802490234375, -1.93359375, 0.51708984375, 0.8896484375, 1.37890625, 0.08526611328125, -0.403076171875, -0.04010009765625, -0.7744140625, -0.049652099609375, -0.08160400390625, -0.2020263671875, 0.57080078125, -1.1171875, -1.7197265625, -1.4658203125, 0.93994140625, 2.275390625, -0.2386474609375, 0.2056884765625, -0.402587890625, 2.1640625, -1.380859375, 0.130615234375, -0.39111328125, -0.417724609375, 1.4091796875, -0.64453125, -0.27685546875, -0.54443359375, 1.0302734375, 1.0908203125, -0.44189453125, 1.8544921875, -0.79443359375, -0.42333984375, -0.1124267578125, -0.6513671875, -0.5322265625, -0.73193359375, 1.0986328125, -0.09527587890625, 0.261474609375, -1.3779296875, 0.239990234375, -0.2032470703125, 1.099609375, -0.806640625, -0.453857421875, -0.55029296875, -0.94921875, 0.3251953125, 2.03515625, 0.47607421875, 1.2099609375, -0.6904296875, 0.810546875, -0.5732421875, 0.60888671875, -0.1962890625, -1.5869140625, 0.39599609375, -2.005859375, 0.326416015625, -2.130859375, -0.20849609375, 0.8837890625, 1.154296875, -0.344482421875, 0.0662841796875, -0.170654296875, -1.3359375, -1.9208984375, -0.99072265625, -1.541015625, 0.62451171875, 0.31396484375, -0.80322265625, -1.0048828125, 1.0771484375, 0.654296875, -1.0732421875, -0.69775390625, 1.314453125, 0.207763671875, 0.09246826171875, -1.35546875, 1.0087890625, 1.68359375, 0.1971435546875, 2.271484375, -0.458740234375, -1.41015625, 1.1103515625, -0.312744140625, 0.953125, -0.345947265625, -1.6337890625, -0.162353515625, 1.990234375, -0.5166015625, 0.439453125, -0.427001953125, -0.2340087890625, 0.177001953125, -0.496337890625, -1.314453125, -1.2080078125, 1.3203125, -0.96337890625, 0.6611328125, -0.7763671875, 0.73486328125, 1.2578125, 0.16015625, -1.3369140625, 0.166259765625, -0.27001953125, 0.7431640625, -0.6591796875, 0.25537109375, -0.181396484375, 0.65576171875, -0.6796875, -0.083740234375, -0.0232086181640625, 0.5947265625, -0.60791015625, -0.017608642578125, 0.254638671875, -0.1688232421875, -0.361083984375, 1.5146484375, -0.1998291015625, 1.5732421875, -1.162109375, 0.75830078125, -0.4443359375, 1.669921875, 0.226806640625, 0.18115234375, -1.2138671875, 0.144287109375, 0.231689453125, -0.3271484375, 0.55810546875, 1.18359375, -1.294921875, -0.98974609375, -0.042449951171875, 0.1568603515625, 0.98779296875, 0.640625, 1.095703125, -0.218017578125, -1.904296875, 0.58203125, 0.07373046875, 0.91845703125, 2.33984375, -0.412109375, -0.35791015625, 1.115234375, -0.051666259765625, 0.57861328125, -0.8017578125, -1.177734375, 0.5263671875, 0.35302734375, -0.32861328125, 0.12066650390625, -2.734375, -0.492431640625, 1.046875, -0.1781005859375, -1.2578125, 1.19921875, -0.9345703125, -0.40185546875, 0.53466796875, -0.689453125, 0.251953125, 0.8427734375, -0.5830078125, 1.6201171875, -0.68017578125, 0.07989501953125, -0.60986328125, 1.1494140625, -0.77001953125, -0.52294921875, 0.5361328125, 2.111328125, 0.1522216796875, 1.8388671875, 0.18310546875, 2.513671875, -1.5048828125, 0.7822265625, -0.10589599609375, -0.82275390625, -0.5087890625, -1.90234375, -0.77392578125, -1.359375, -2.51953125, -0.328125, 0.43212890625, -0.391357421875, 0.1917724609375, -2.224609375, 1.033203125, -0.51318359375, -0.470947265625, -0.41552734375, 0.0389404296875, -0.990234375, 1.935546875, 0.344970703125, 1.1025390625, -0.3623046875, -0.12744140625, -1.08203125, 1.2529296875, -0.376220703125, 0.57373046875, 0.84326171875, -0.0548095703125, -1.4443359375, 0.2413330078125, -1.328125, -0.78759765625, 1.0693359375, -1.2421875, -2.1328125, -0.101318359375, 1.7685546875, -0.441162109375, -0.271240234375, 1.78515625, -1.849609375, 0.7568359375, 0.318359375, 0.03607177734375, -0.138427734375, 1.6357421875, 1.333984375, -1.7392578125, -0.17919921875, 0.84228515625, -0.23486328125, 0.398193359375, -0.139404296875, 1.0556640625, -0.56201171875, 1.4169921875, 1.6923828125, 0.79052734375, 0.2078857421875, -0.339111328125, -0.2064208984375, -0.05535888671875, 0.294921875, 1.2294921875, -0.72802734375, -0.96875, 1.5537109375, -0.607421875, -0.57861328125, -0.76904296875, -0.0850830078125, 0.0003304481506347656, -0.124755859375, -1.3017578125, 0.8232421875, 0.438232421875, -2.435546875, 0.461669921875, -0.5439453125, -0.77587890625, -1.3486328125, 0.286865234375, 0.0889892578125, 0.4150390625, 0.38671875, -1.478515625, -0.60595703125, -0.390869140625, -2.90234375, -0.178955078125, -0.279296875, 0.467041015625, -0.435302734375, -2.248046875, 0.21923828125, 1.353515625, -0.5400390625, -0.7587890625, 1.5048828125, 1.263671875, -1.1064453125, 1.1357421875, 1.8525390625, -2.326171875, -0.4345703125, -0.1630859375, -0.86572265625, -0.677734375, 0.2939453125, 1.2333984375, -1.4306640625, -0.71044921875, -2.216796875, -1.125, 0.546875, -1.4892578125, 1.525390625, -0.454345703125, 0.31298828125, 1.7470703125, -0.0231170654296875, -1.2412109375, 0.47265625, 1.25, -1.2265625, -1.927734375, -0.84033203125, -0.966796875, -1.201171875, 0.464599609375, 0.407958984375, -0.18408203125, -0.7314453125, -0.72900390625, -2.103515625, -0.42138671875, -0.487548828125, 0.057037353515625, 0.321533203125, 0.71533203125, 0.5595703125, -0.258056640625, 0.47900390625, 0.1422119140625, 0.40625, -0.57373046875, 1.041015625, -0.26904296875, -0.611328125, -0.22265625, 0.024688720703125, -0.473388671875, 0.35400390625, 0.0767822265625, -0.84912109375, -0.427734375, -0.9150390625, -0.69287109375, 1.154296875, -0.165771484375, 0.277099609375, 1.2392578125, 0.521484375, 0.246826171875, -0.65625, 0.5751953125, 0.80126953125, 0.069580078125, -0.0009331703186035156, 1.1123046875, -0.6982421875, 0.90185546875, -0.810546875, -0.8056640625, -0.31689453125, -0.7177734375, -0.39013671875, 0.10302734375, -1.185546875, -0.8857421875, -0.90380859375, -0.11077880859375, 0.31298828125, -0.322265625, 1.7763671875, -0.70556640625, 0.67724609375, -0.378173828125, -0.9423828125, -0.3896484375, -0.395751953125, -0.4326171875, 0.0183868408203125, 1.818359375, 0.6630859375, -1.2177734375, -1.2919921875, -0.53125, -1.8984375, -1.62890625, -0.2445068359375, 1.87109375, -1.2626953125, -0.62451171875, -1.1826171875, -0.1357421875, 0.70703125, 0.9091796875, -1.1865234375, 0.68701171875, -0.74072265625, 0.281982421875, -1.103515625, 0.46875, 1.3876953125, 2.76171875, 1.2138671875, -2.150390625, -0.33056640625, -0.401123046875, -0.9609375, 1.857421875, -0.81298828125, 1.5185546875, 1.869140625, -1.0771484375, -0.58740234375, -0.3759765625, -0.505859375, -2.259765625, 0.3955078125, 0.30322265625, -0.5126953125, -0.994140625, -0.46533203125, 1.396484375, 0.94140625, 0.267333984375, 1.1884765625, -1.0927734375, -1.3310546875, 0.798828125, 0.379638671875, 0.1622314453125, -0.61474609375, 1.166015625, -1.4404296875, 1.521484375, -0.362548828125, -0.361083984375, -1.396484375, -0.14599609375, -0.6904296875, -0.486572265625, 1.49609375, 0.33154296875, -1.4736328125, -1.5810546875, 0.70263671875, 0.06695556640625, 0.7470703125, -0.701171875, -1.8935546875, 0.251708984375, 0.30712890625, -0.0797119140625, -0.85205078125, -0.22998046875, -1.48828125, -0.425048828125, 0.2474365234375, -0.5263671875, -0.837890625, 1.25390625, 0.90087890625, 0.04974365234375, -0.76025390625, -1.5771484375, 0.908203125, -0.82763671875, -1.2587890625, -1.2275390625, 1.712890625, 0.966796875, -1.1650390625, -1.3583984375, 0.406982421875, 0.28125, 0.4990234375, 0.6494140625, 0.2369384765625, 0.59765625, -2.3203125, -0.83447265625, 1.6298828125, 0.2447509765625, 0.14990234375, -0.55322265625, -0.458984375, 1.2412109375, -1.3203125, 0.67529296875, -0.034637451171875, -0.39306640625, -0.822265625, -2.203125, -0.61376953125, 0.198486328125, 0.287841796875, 2.265625, 0.36083984375, -0.2332763671875, 0.176513671875, -0.05560302734375, -0.2216796875, 0.2763671875, -1.595703125, 2.021484375, 1.10546875, 0.454833984375, -2.9375, -0.52490234375, -1.7646484375, 0.0989990234375, -1.96484375, 0.83154296875, -0.8681640625, -0.093017578125, -0.537109375, 1.4775390625, -0.0333251953125, 0.4775390625, -0.51513671875, -0.143798828125, -0.92626953125, 1.26171875, 0.421630859375, -0.63818359375, 1.5234375, 0.87841796875, -0.1866455078125, -0.495361328125, -0.6591796875, 0.56298828125, 1.078125, 0.0154876708984375, 0.8896484375, -0.94970703125, 1.4443359375, 0.38134765625, -0.8203125, -1.546875, 0.017547607421875, 0.12164306640625, -1.2177734375, 1.8369140625, -0.007030487060546875, -0.267822265625, -0.295654296875, 1.0869140625, 0.148193359375, -0.5224609375, -0.9111328125, 0.458984375, 1.7314453125, -1.9638671875, -0.50390625, 1.263671875, -0.82373046875, 0.26025390625, 0.76171875, 0.62109375, -0.38037109375, 0.484375, 0.96240234375, 0.8017578125, 0.8701171875, 1.7763671875, 0.1839599609375, -1.078125, 0.9091796875, 0.607421875, -1.427734375, 1.1708984375, -0.68701171875, -0.39892578125, 0.0171966552734375, 0.2357177734375, 0.990234375, -0.342529296875, -1.0107421875, 0.82275390625, 0.7822265625, 0.59228515625, 0.049346923828125, 0.49609375, -0.417724609375, -0.97216796875, 0.047515869140625, 1.1953125, 0.1783447265625, 0.1278076171875, 0.35888671875, 0.857421875, -1.3271484375, 0.06207275390625, 0.5224609375, -0.064697265625, -0.44091796875, -1.5634765625, 1.7607421875, -0.7841796875, -1.9130859375, -1.6728515625, 0.8154296875, 0.41015625, -0.1365966796875, 1.2275390625, 0.43017578125, 0.88232421875, -0.85888671875, -1.259765625, -0.2978515625, -0.0181427001953125, 0.251953125, 0.578125, -0.41162109375, 0.45458984375, 0.030792236328125, 1.76953125, -0.259765625, -0.130859375, 1.05859375, -0.97119140625, 0.51171875, -0.0222625732421875, 0.345458984375, 0.36865234375, 0.1846923828125, -0.383544921875, -0.282470703125, -0.5830078125, 0.775390625, -0.08660888671875, -0.755859375, -0.2325439453125, -0.280029296875, 1.7998046875, 2.310546875, -0.00047087669372558594, -2.212890625, 0.8505859375, 0.58251953125, -0.5859375, -0.58447265625, 2.482421875, 0.947265625, -1.1123046875, 0.82861328125, -0.5517578125, -0.263427734375, 0.603515625, 0.436279296875, -1.5634765625, 0.849609375, 0.17822265625, -0.837890625, 1.345703125, 0.091552734375, -1.1708984375, -1.0732421875, 0.419921875, 0.68505859375, -1.576171875, 0.86572265625, -0.93896484375, 0.48291015625, 0.369140625, 1.4482421875, 0.1331787109375, -0.359619140625, 0.5478515625, 0.1800537109375, 2.125, 0.7705078125, 0.7109375, -0.147216796875, 1.0537109375, 0.1749267578125, -1.513671875, 1.109375, -1.1630859375, -0.08465576171875, 0.62255859375, -1.4326171875, -0.309814453125, -0.387939453125, -0.2181396484375, 0.92041015625, 2.1015625, -0.666015625, -1.62890625, -0.52197265625, 0.2012939453125, -1.13671875, -2.234375, -1.5302734375, -1.1318359375, 0.08575439453125, -0.263671875, -0.57177734375, -0.1473388671875, -1.771484375, 1.9150390625, 1.6171875, 1.185546875, 0.3212890625, 1.0126953125, -0.352783203125, -0.365966796875, -0.74609375, 1.4404296875, 0.11468505859375, 1.87109375, -0.480712890625, 0.0745849609375, 0.361572265625, 0.73779296875, 1.94140625, -1.5009765625, 0.599609375, 0.01363372802734375, -0.6279296875, 1.0029296875, 1.9716796875, 0.99267578125, 0.467041015625, -1.837890625, 0.9072265625, -0.88720703125, 0.339599609375, -0.0684814453125, 0.72021484375, 0.138916015625, 0.76025390625, 0.95556640625, -0.482421875, 0.19677734375, 1.982421875, 1.3828125, -0.38232421875, 0.4296875, -1.8125, 0.8984375, -1.0966796875, 0.9970703125, 0.10101318359375, 0.1282958984375, -0.9384765625, 0.8330078125, -0.0297698974609375, 0.304443359375, 1.7490234375, -0.32861328125, -0.7373046875, -0.63037109375, -0.9033203125, 0.45458984375, 0.343505859375, 0.1964111328125, -1.0615234375, -1.4443359375, 0.31591796875, 0.450439453125, 0.394775390625, 2.0234375, 1.5556640625, -1.150390625, -0.6171875, 0.85009765625, 1.4150390625, -0.37109375, 0.1695556640625, 0.630859375, -1.328125, -1.177734375, -0.449951171875, -0.67431640625, 0.27783203125, -0.58984375, -0.33251953125, -1.8349609375, -0.7412109375, -0.08563232421875, 0.2457275390625, -1.298828125, -0.90087890625, 2.037109375, 2.951171875, 1.78125, 0.94091796875, 0.272216796875, -1.423828125, -1.1357421875, 0.07562255859375, -0.260498046875, -1.486328125, 0.309814453125, 0.0865478515625, 1.1767578125, 1.78515625, -0.3330078125, -0.8193359375, 0.173828125, 0.38720703125, 1.3134765625, -0.4443359375, 1.1796875, -0.35107421875, -0.1578369140625, -1.15234375, -0.160888671875, 2.228515625, 1.435546875, -0.369140625, 0.0399169921875, -0.02886962890625, 1.994140625, 0.12451171875, 2.53125, -0.876953125, -0.1114501953125, -1.927734375, 0.362060546875, 1.396484375, 2.564453125, 0.424072265625, -1.474609375, 1.115234375, 0.366455078125, 0.79345703125, -0.073974609375, -0.49560546875, -0.677734375, 0.156494140625, -1.00390625, -0.08477783203125, 0.07568359375, -1.19921875, -1.0732421875, 0.2298583984375, 0.61328125, 0.3447265625, -0.73583984375, -0.343017578125, 0.393798828125, -0.92431640625, 0.97412109375, 0.0987548828125, 2.5859375, -1.0224609375, -1.552734375, 0.52294921875, 0.261474609375, 2.486328125, 0.451904296875, 0.31201171875, 1.9189453125, -0.86328125, -0.11572265625, -1.3681640625, -0.032470703125, -0.44873046875, 0.169677734375, 0.60107421875, 0.388671875, -0.29296875, -0.5048828125, -0.6357421875, 0.61328125, 0.5830078125, 0.57275390625, -0.89453125, -1.15625, 0.09478759765625, 0.2548828125, 0.9111328125, -0.50439453125, 0.720703125, -1.3212890625, -1.0458984375, -0.0225067138671875, -0.326416015625, 0.0028228759765625, 1.6611328125, 0.304443359375, -0.2841796875, 1.1298828125, 1.3447265625, 1.9453125, 0.364501953125, 0.63037109375, -0.10272216796875, 0.66552734375, 1.091796875, -1.2060546875, -0.353759765625, -2.197265625, 0.0869140625, -1.6142578125, -0.42529296875, -0.47607421875, -1.2861328125, -0.650390625, 1.1337890625, -0.58642578125, -0.904296875, 0.329833984375, 0.1513671875, 0.70361328125, 0.3818359375, -0.90576171875, -1.8134765625, -0.2100830078125, -0.53173828125, 0.5078125, -1.625, -0.246337890625, 1.4814453125, 1.828125, 0.83447265625, 0.3642578125, -2.599609375, -1.6162109375, -0.2384033203125, -0.4521484375, -0.9990234375, 3.076171875, 0.771484375, 2.38671875, -0.666015625, -0.045196533203125, -0.90185546875, 2.232421875, 2.134765625, 0.97705078125, -0.3681640625, 1.08984375, -0.000766754150390625, -0.057830810546875, -1.0419921875, 3.005859375, 1.1865234375, -1.365234375, 0.515625, -0.31640625, -0.90283203125, 0.03790283203125, -0.9755859375, 0.99755859375, -0.21923828125, -1.84765625, -0.28271484375, -0.587890625, 1.76953125, 2.00390625, 0.67236328125, 0.4365234375, -0.109619140625, -0.478515625, -0.022613525390625, -0.59716796875, -1.6689453125, 0.86572265625, 0.95556640625, 0.46337890625, -0.224365234375, 1.33984375, 0.76953125, 0.95751953125, -2.384765625, 0.712890625, -1.9716796875, 1.00390625, 0.40771484375, 0.019012451171875, 1.4130859375, 0.1435546875, -0.37939453125, -0.1702880859375, -1.1044921875, -0.07781982421875, 0.47314453125, -0.94921875, 1.513671875, -1.1689453125, -0.03924560546875, -0.275634765625, -1.1513671875, -0.45361328125, 0.1439208984375, 1.0478515625, 0.49853515625, 0.467041015625, -0.73779296875, -1.28125, 0.2310791015625, 1.2314453125, 0.7080078125, -1.5322265625, 0.8896484375, -1.1484375, 0.08575439453125, -0.400146484375, -0.1561279296875, 1.734375, -2.01171875, -1.193359375, -0.81689453125, -0.82861328125, 1.4501953125, 1.23046875, 1.349609375, 0.3388671875, -0.16552734375, 2.779296875, -0.26708984375, 0.179931640625, 0.85205078125, -0.411376953125, 1.1025390625, -0.24560546875, 0.325927734375, 0.480712890625, -0.5390625, -0.23291015625, -1.1630859375, 0.85888671875, 0.188232421875, -0.2445068359375, 1.78515625, 2.35546875, 0.103271484375, -0.60595703125, -1.1943359375, -0.9599609375, -0.264404296875, -0.334716796875, -1.3740234375, 2.2265625, 0.80615234375, 1.1787109375, -0.84130859375, 1.0390625, -1.013671875, -0.7802734375, -1.5087890625, -0.3056640625, -0.1710205078125, 1.2060546875, -0.56689453125, 0.054107666015625, -0.578125, 0.07025146484375, 0.66064453125, -0.6796875, 0.89697265625, 0.06829833984375, 1.6689453125, -0.206298828125, 0.533203125, 1.0537109375, 1.67578125, 0.599609375, 0.50830078125, 0.77001953125, -0.8603515625, 0.58642578125, 0.97802734375, -0.3466796875, 0.444580078125, -3.033203125, -0.91455078125, 0.422119140625, -0.2188720703125, -1.1220703125, -1.0205078125, -0.42041015625, -0.90673828125, -0.43505859375, -0.78857421875, -0.38427734375, -2.24609375, -0.322998046875, 0.081787109375, -0.9267578125, -1.8623046875, -1.052734375, -0.9345703125, 0.62890625, 0.0872802734375, 0.06964111328125, 0.048492431640625, -0.2281494140625, -2.2734375, -1.7900390625, -0.7666015625, 1.037109375, 0.3671875, -1.0751953125, -1.0625, 1.25390625, -0.298095703125, -0.88330078125, -0.0755615234375, 1.0625, 0.170654296875, -0.01557159423828125, 1.6455078125, -0.681640625, 0.3515625, -0.51806640625, -0.93017578125, 1.765625, 0.266357421875, 1.287109375, 1.94140625, 0.4658203125, -0.052978515625, 1.2412109375, 0.2208251953125, 1.7109375, 1.734375, 0.71044921875, 4.297494888305664e-05, 0.64599609375, -0.0015974044799804688, -0.77392578125, 0.379150390625, -0.69921875, 0.6748046875, 0.07232666015625, -0.443603515625, -0.48046875, -1.7412109375, 1.1435546875, 0.2410888671875, -0.6640625, -0.36279296875, -1.5966796875, 1.3994140625, -0.00910186767578125, 0.272216796875, 0.390380859375, -1.0703125, 0.1619873046875, -1.1669921875, -0.7001953125, 1.791015625, 1.0595703125, -0.01507568359375, 1.02734375, 0.361572265625, 0.53759765625, -2.404296875, 2.142578125, -2.111328125, 0.6640625, -0.41162109375, 0.310791015625, -1.90234375, 0.0178680419921875, 0.477294921875, -0.544921875, 0.3349609375, 0.59423828125, -0.67578125, -0.40283203125, 0.70654296875, -0.0499267578125, 0.2142333984375, 0.01214599609375, 1.17578125, 0.0635986328125, 0.99169921875, 1.1591796875, -0.779296875, 0.716796875, -0.33349609375, -0.326416015625, -1.3154296875, -2.533203125, 0.65771484375, -1.337890625, 0.8876953125, -0.386962890625, -0.818359375, 0.89794921875, 2.615234375, -1.1640625, 0.705078125, -0.2210693359375, 0.2469482421875, -0.369873046875, -2.125, 0.89111328125, -1.2177734375, 1.1435546875, 1.6142578125, 1.384765625, 0.61767578125, -0.83056640625, 1.21875, -0.35986328125, 0.430908203125, -0.1861572265625, 1.130859375, -0.171630859375, 0.06561279296875, -1.5458984375, 0.49853515625, 0.73779296875, 0.08843994140625, -1.3369140625, -1.193359375, 0.7314453125, 0.87890625, 1.2451171875, -0.9873046875, -1.3486328125, -1.0224609375, 0.189453125, -0.40966796875, -0.144287109375, -0.3525390625, -0.08636474609375, 0.135009765625, 2.833984375, -0.40185546875, -0.179443359375, -1.2578125, -1.716796875, 0.28466796875, -0.160888671875, 1.76171875, 1.7109375, 1.064453125, -0.73388671875, 0.379150390625, -0.1663818359375, 0.366943359375, -0.640625, 0.224609375, 0.076904296875, -0.68603515625, 0.7138671875, 0.4931640625, 0.1488037109375, 0.54541015625, -1.36328125, 0.0455322265625, -0.92529296875, 0.1422119140625, 0.51953125, -0.95556640625, -0.52978515625, 1.376953125, 0.2166748046875, 0.30322265625, 0.8115234375, -0.06298828125, -1.271484375, -1.349609375, -0.0217437744140625, 0.27587890625, -0.047943115234375, -0.81005859375, 1.328125, 0.265380859375, 1.568359375, -1.431640625, 2.123046875, 0.69677734375, 0.54296875, -0.020660400390625, -1.7607421875, -0.5400390625, -0.371337890625, -0.90087890625, 0.60009765625, 2.240234375, -0.0225372314453125, -0.165771484375, 1.0244140625, 0.208984375, 0.52978515625, 1.798828125, -1.0078125, 0.141357421875, 0.276611328125, -0.305419921875, 1.208984375, -0.732421875, 0.2008056640625, 0.67724609375, 1.4873046875, 0.214599609375, -0.66845703125, 0.99462890625, -0.9296875, 0.3173828125, -0.66259765625, 0.67626953125, 1.18359375, -0.318115234375, 1.5400390625, -0.277587890625, 1.025390625, -0.85205078125, 0.139404296875, 1.3466796875, -0.70654296875, -1.1376953125, -2.451171875, -1.755859375, 0.439453125, 0.1279296875, -0.7763671875, 2.630859375, 1.40234375, 0.50048828125, 0.2900390625, -0.67578125, 0.2110595703125, 1.525390625, -0.7939453125, 0.88427734375, -0.46923828125, -0.640625, -0.26513671875, -1.017578125, 0.51953125, 1.7607421875, -0.25927734375, 0.95703125, -1.2119140625, -0.736328125, 1.8408203125, 0.59912109375, 0.52392578125, 0.373291015625, 0.6923828125, -0.572265625, 0.429443359375, 1.279296875, 0.79345703125, -0.4716796875, -1.2998046875, 0.5166015625, 0.32666015625, -2.369140625, 0.8798828125, -0.357666015625, -0.412841796875, -0.845703125, -2.2578125, 0.79443359375, -0.6279296875, 0.2366943359375, -0.28564453125, -0.1295166015625, -0.64306640625, -1.015625, -0.3505859375, -1.4423828125, 1.412109375, 3.607421875, -1.2138671875, 0.884765625, 0.5380859375, 0.88916015625, -1.095703125, -0.40576171875, 0.432861328125, 0.159912109375, -0.07415771484375, 0.6689453125, 0.09228515625, 2.15234375, 0.23828125, 0.4189453125, 0.84521484375, -1.5908203125, 1.3359375, 1.0380859375, 0.75537109375, -1.310546875, -1.8359375, -0.2783203125, -0.383056640625, 0.309814453125, 1.9970703125, -0.1959228515625, 1.623046875, 0.85205078125, 0.307861328125, -0.99658203125, 2.041015625, 1.763671875, 0.27294921875, -0.036590576171875, 0.90966796875, 0.47802734375, 1.564453125, 0.521484375, -1.9443359375, -0.408935546875, 1.8955078125, -0.93994140625, -0.67431640625, -0.7421875, -0.34814453125, 0.1300048828125, 0.2451171875, -0.82275390625, -0.279541015625, 0.59326171875, 0.353759765625, 0.8642578125, -0.640625, 0.1785888671875, 0.095458984375, -1.548828125, 1.3935546875, -0.2408447265625, -0.77392578125, -1.5419921875, 0.328369140625, -1.8115234375, -0.092041015625, 0.55419921875, 0.10614013671875, -1.76953125, 0.458740234375, -0.3125, 1.0576171875, -0.247802734375, 0.274658203125, 1.1474609375, 1.458984375, -0.88037109375, -0.53271484375, -1.8232421875, 0.06805419921875, 2.021484375, -0.4228515625, 0.040771484375, -1.1630859375, -0.64208984375, -1.125, 0.546875, 0.72412109375, 0.556640625, 0.0098876953125, 0.032806396484375, -0.986328125, 0.485107421875, 0.32373046875, 1.224609375, -0.9013671875, -0.857421875, -0.72314453125, 2.40625, -0.47802734375, -0.06524658203125, -0.43896484375, 2.0859375, 1.18359375, -1.1904296875, -1.6650390625, 0.96044921875, 2.212890625, 0.61279296875, 1.1787109375, 1.2841796875, -1.3203125, 1.1572265625, -0.73095703125, -1.107421875, -1.7294921875, -1.33984375, -0.288818359375, -1.228515625, 0.11346435546875, 0.498291015625, 0.09259033203125, 0.5224609375, -1.0869140625, 0.347900390625, -1.4140625, -0.1083984375, -0.04730224609375, 1.9228515625, -1.736328125, 0.521484375, 1.8974609375, -1.0703125, -0.85546875, 0.01788330078125, -0.0582275390625, -1.0126953125, 1.7783203125, 0.65869140625, 0.751953125, -0.421630859375, -0.461669921875, 0.1488037109375, -0.019775390625, -1.4580078125, -1.5087890625, -1.5048828125, -0.455322265625, -0.2529296875, -0.123046875, 0.384765625, 0.7001953125, 0.2861328125, -0.387939453125, 0.8759765625, 0.53759765625, 1.134765625, -2.107421875, 0.1790771484375, -0.6953125, 0.79248046875, -0.3310546875, 1.212890625, 0.8134765625, 0.8251953125, 0.05279541015625, 1.63671875, -1.33984375, 0.318359375, 1.6845703125, 1.23046875, -0.392578125, 0.4619140625, -2.00390625, 0.468505859375, -0.458740234375, -0.66845703125, 0.9453125, -0.85986328125, 0.01116180419921875, -0.357177734375, 1.5986328125, 0.37939453125, 0.9013671875, 0.458984375, 1.0927734375, 0.254638671875, 0.378173828125, 0.62841796875, -0.17431640625, -0.64599609375, -0.10992431640625, 0.52099609375, -1.1748046875, -1.01953125, 0.376220703125, -1.0419921875, 0.08123779296875, -0.088623046875, -0.323486328125, 0.323486328125, 0.394287109375, -1.5849609375, -2.1171875, -0.78759765625, -1.1171875, 0.74658203125, -1.4345703125, -1.1689453125, 1.865234375, 0.35986328125, 0.1395263671875, 0.477294921875, -0.40087890625, 0.17138671875, 0.1907958984375, -1.0009765625, 0.24560546875, -0.60595703125, 0.26806640625, -0.262939453125, 0.74169921875, -0.1922607421875, -0.705078125, 0.1767578125, 0.03619384765625, 0.443359375, -0.146240234375, -0.034088134765625, 1.4775390625, 0.29443359375, -0.96875, -0.31298828125, -0.2099609375, 0.0011587142944335938, -0.76318359375, 0.55712890625, -0.7333984375, 1.33984375, -0.75537109375, -0.0238037109375, -0.92041015625, -0.44921875, 1.05078125, 1.046875, -0.6123046875, -0.3310546875, -1.3408203125, 0.7431640625, 1.8193359375, 0.78515625, 1.9208984375, 1.7802734375, 1.201171875, -0.312255859375, 0.94580078125, -0.7412109375, 0.1817626953125, 1.630859375, -0.4677734375, -1.294921875, 0.017852783203125, -0.951171875, 0.2978515625, -1.4169921875, -1.07421875, 0.66162109375, -0.367431640625, -0.7607421875, -1.3916015625, -0.5361328125, 0.1689453125, -1.87109375, 1.5712890625, -0.490478515625, 0.24658203125, -0.568359375, 1.234375, -0.859375, -0.297607421875, 1.14453125, 1.021484375, 2.755859375, 0.78759765625, 1.3505859375, -0.1923828125, 0.3056640625, -0.3935546875, -0.263427734375, 1.5625, 1.212890625, -0.300048828125, -0.8193359375, 1.3798828125, -0.60205078125, 1.59765625, -0.23193359375, -1.255859375, -0.78271484375, 1.5146484375, -1.4765625, 0.57763671875, -0.307861328125, -0.76806640625, 1.3095703125, 0.4775390625, 0.048980712890625, -0.95361328125, -0.1563720703125, -2.009765625, 0.35791015625, -0.00846099853515625, -1.3017578125, 1.0517578125, -1.6748046875, 0.802734375, 1.158203125, 0.50146484375, -0.5107421875, -0.232177734375, -0.6123046875, -0.9248046875, 0.327880859375, -2.25, 0.603515625, -0.78173828125, 2.33203125, -1.1435546875, -0.079833984375, 0.2198486328125, 0.1900634765625, 1.650390625, -0.439697265625, -0.67822265625, 0.1309814453125, 0.86865234375, -0.576171875, -0.60546875, -1.552734375, -0.5029296875, 0.75048828125, -1.560546875, 1.8974609375, -2.41015625, 0.18359375, -0.135009765625, 0.650390625, 0.04595947265625, 1.3701171875, 3.158203125, 0.27099609375, -0.880859375, 0.7763671875, -1.095703125, -1.6904296875, 1.6630859375, -0.2169189453125, -0.261962890625, -0.153564453125, -0.88232421875, 0.53466796875, 0.5107421875, -1.94921875, 0.441650390625, -2.111328125, 0.51123046875, -0.5419921875, -0.69384765625, -0.73388671875, 1.205078125, 0.198974609375, -1.740234375, 0.313232421875, -0.72412109375, -0.73388671875, 1.0576171875, 0.0623779296875, -3.091796875, -0.60107421875, -2.736328125, 0.90771484375, -0.249755859375, 0.9560546875, -0.054779052734375, 0.52783203125, -0.93701171875, -0.94921875, -0.90625, 0.81640625, 0.15869140625, 0.32666015625, -1.376953125, -1.5126953125, 0.312255859375, -0.264404296875, 1.6630859375, -0.67626953125, 0.188232421875, -0.1717529296875, -0.46630859375, 1.4755859375, 1.1962890625, -0.35888671875, -0.281005859375, -1.2001953125, -1.0732421875, -0.52197265625, 1.978515625, 2.20703125, 0.66552734375, -0.85888671875, -0.25634765625, 0.1756591796875, 0.9033203125, 1.763671875, 0.5107421875, 0.396240234375, -0.4072265625, 0.82275390625, 0.2061767578125, -0.1715087890625, 1.220703125, -1.154296875, -2.78515625, -1.751953125, -0.405029296875, 0.239013671875, -0.6376953125, -0.3212890625, 0.6171875, -0.28271484375, 0.7080078125, 0.82568359375, -0.333984375, 1.3583984375, -2.19921875, 0.01401519775390625, -0.669921875, -0.51708984375, 0.859375, 0.0777587890625, 0.8837890625, 3.009765625, 0.40087890625, -0.39306640625, -0.93212890625, -0.2958984375, -0.767578125, -0.12249755859375, -0.08258056640625, -0.484130859375, -1.2568359375, -0.316162109375, 1.1708984375, -0.740234375, 0.8642578125, -0.2384033203125, 1.396484375, -0.44677734375, -1.298828125, -0.433349609375, 1.09375, -0.556640625, 0.990234375, -0.78076171875, -0.1644287109375, 0.301513671875, -1.8583984375, -1.30078125, -0.00632476806640625, 0.96435546875, -0.66259765625, -0.347900390625, 1.2109375, 0.86572265625, 1.1982421875, 0.370849609375, -0.0780029296875, -0.6640625, 0.67919921875, -0.14794921875, -0.650390625, 1.3779296875, 0.249755859375, -0.415771484375, -0.2109375, 1.41015625, 0.46240234375, -0.35205078125, -0.7744140625, 0.4111328125, 1.1943359375, -0.0183258056640625, -0.0948486328125, -0.95263671875, 1.61328125, -0.63525390625, 1.193359375, 0.034027099609375, 0.157470703125, -0.83056640625, 1.71875, 0.02105712890625, 0.12091064453125, -0.1566162109375, 1.115234375, -1.2705078125, 0.97900390625, 0.6162109375, -0.21435546875, -0.024200439453125, 0.65234375, 0.783203125, -0.1810302734375, -0.9384765625, -0.552734375, 0.1605224609375, 1.8447265625, -0.403076171875, -2.2578125, 0.36181640625, -0.75634765625, -1.4208984375, 0.059844970703125, 0.98095703125, -0.2188720703125, 2.5625, -2.072265625, -0.10064697265625, -1.3818359375, -0.476806640625, -2.59765625, -0.045074462890625, 0.03564453125, -0.869140625, -0.1942138671875, 0.1546630859375, -0.64794921875, -0.403564453125, -0.99853515625, -0.9560546875, -0.2568359375, 0.45654296875, 0.1409912109375, -1.59765625, 1.1025390625, 0.80029296875, 1.111328125, 1.41015625, -0.425048828125, -1.2998046875, -0.4833984375, -0.71484375, -1.5205078125, -1.9384765625, 0.80224609375, -0.97216796875, 1.3408203125, -1.4775390625, -0.4716796875, -0.0433349609375, 0.50830078125, -0.0911865234375, 0.96728515625, -0.60498046875, -1.2333984375, 0.58154296875, -0.744140625, -0.71923828125, 0.384033203125, 1.5009765625, -0.281005859375, -1.0244140625, -0.320068359375, -0.1796875, 1.224609375, 1.6240234375, 0.7392578125, 1.3583984375, 0.91162109375, 0.1939697265625, -0.49365234375, 0.293212890625, 1.16796875, 1.6357421875, 0.2783203125, 0.35546875, 0.03472900390625, -0.998046875, 0.0302581787109375, -0.9755859375, -0.8828125, 1.73046875, -1.017578125, 0.32958984375, 1.064453125, -0.75390625, 0.63623046875, -1.8681640625, -0.33056640625, -0.443359375, -1.5146484375, 2.26953125, -1.0830078125, 0.5283203125, -1.71484375, -0.005767822265625, -0.10394287109375, 0.9443359375, 0.52001953125, 1.53125, 0.1766357421875, -0.392578125, -0.418701171875, 0.7392578125, 0.98095703125, -2.501953125, -0.56005859375, 0.288330078125, -1.9921875, -0.98486328125, 0.302978515625, -1.1943359375, 0.122314453125, 0.418212890625, 0.724609375, -0.289306640625, 0.5986328125, 1.9921875, 0.26953125, 0.42724609375, -1.3857421875, 0.1900634765625, -0.2470703125, -0.505859375, -0.44970703125, 0.405029296875, -0.50830078125, -0.77001953125, -0.84619140625, -2.01171875, 0.93798828125, -0.0166015625, -0.10009765625, -1.6494140625, -0.10650634765625, -0.3046875, 1.0546875, 0.07598876953125, -0.9833984375, -0.120361328125, -1.8447265625, 0.81396484375, -2.0625, 1.1025390625, 0.595703125, 0.84619140625, 0.5595703125, 0.96533203125, 0.509765625, 0.76513671875, 2.654296875, -0.217041015625, -1.2275390625, -2.212890625, 0.6640625, -1.7060546875, -0.83251953125, 0.6943359375, 1.0927734375, -0.266357421875, 0.40478515625, 0.00797271728515625, 1.19921875, 0.09747314453125, -0.98291015625, 0.327880859375, -2.064453125, 0.1136474609375, -0.52099609375, -0.52392578125, 0.37890625, -1.23046875, 0.8916015625, -1.6357421875, -0.576171875, 1.224609375, -0.64111328125, -0.59375, -1.5458984375, 0.142578125, 0.4013671875, 1.484375, -0.1834716796875, 1.4228515625, 1.4384765625, 0.78076171875, -0.101318359375, 0.1180419921875, 1.2275390625, -1.3310546875, -0.479248046875, 1.1455078125, -1.20703125, -0.322265625, 0.59716796875, -0.59521484375, 0.08831787109375, 0.33349609375, 0.288818359375, -0.91845703125, -0.443115234375, 0.6220703125, 0.461669921875, -0.9091796875, -0.58984375, -0.400634765625, 1.21875, 0.235595703125, -1.5439453125, -0.3076171875, -0.7470703125, 1.0654296875, 0.1605224609375, 0.075439453125, 1.8046875, 0.76123046875, 0.79931640625, 0.2376708984375, -0.54150390625, 0.775390625, 0.9619140625, -1.765625, 1.6875, 0.433349609375, -1.1328125, 0.0035228729248046875, 0.40625, 0.48193359375, 0.62744140625, 0.53125, -1.5546875, -0.5068359375, 1.259765625, 0.0914306640625, -0.462890625, -1.2275390625, -1.7255859375, -0.78125, 0.61474609375, 1.0029296875, 1.427734375, -0.49560546875, -1.1142578125, -0.242431640625, -1.4189453125, 0.8662109375, 1.1572265625, 1.55078125, -1.216796875, -0.60107421875, 0.47509765625, -0.197509765625, 0.461669921875, -0.12396240234375, 0.828125, -0.685546875, -0.55712890625, -0.338134765625, -0.343505859375, 1.439453125, -0.214599609375, 1.0498046875, -2.4921875, 2.09375, -0.47509765625, -1.572265625, 0.6220703125, -1.6689453125, 0.3203125, 1.0458984375, -1.259765625, 2.111328125, -0.85595703125, -0.990234375, 1.3125, 0.78857421875, -1.2958984375, 0.72705078125, -1.5810546875, 0.94384765625, -0.955078125, 0.5771484375, 0.1441650390625, 0.78173828125, -0.9951171875, 0.2568359375, -1.4404296875, -1.46484375, 0.19580078125, -0.08160400390625, 1.3515625, -0.1416015625, -0.054779052734375, 1.27734375, -2.0625, 0.580078125, -0.59619140625, 2.08203125, -1.4384765625, -0.7119140625, 1.1044921875, -1.275390625, 0.7275390625, -2.07421875, -0.88525390625, 0.0760498046875, 0.61474609375, -2.369140625, -0.193603515625, 1.291015625, -0.7109375, 0.135498046875, -1.5673828125, -0.5498046875, -0.0015840530395507812, 1.2763671875, 0.875, -0.7841796875, -1.41015625, -1.0732421875, -0.068359375, 0.76611328125, 0.5458984375, -2.748046875, 0.56591796875, -0.52734375, 0.124755859375, 1.111328125, 0.9326171875, 1.228515625, 0.49755859375, -0.33056640625, -2.60546875, 0.09246826171875, 0.380859375, 0.5322265625, -0.1654052734375, 1.419921875, 0.394775390625, 0.697265625, 0.9365234375, -0.333251953125, 0.134765625, -1.0947265625, -0.226806640625, -1.814453125, -0.0467529296875, -0.287353515625, 1.048828125, 0.5576171875, -0.431396484375, 2.416015625, 1.880859375, 0.8525390625, 0.29052734375, 0.55224609375, 0.35693359375, -0.175048828125, 0.167236328125, -0.697265625, 0.8203125, 0.73681640625, -0.049163818359375, 1.439453125, -2.1875, 1.16015625, -1.1474609375, 0.2435302734375, -0.533203125, 0.58642578125, -0.76171875, -0.7158203125, 0.8486328125, -0.34228515625, -0.658203125, 1.0986328125, -0.53662109375, -1.6103515625, -0.06951904296875, -1.3349609375, 0.6513671875, 1.47265625, 1.525390625, -0.253662109375, -1.388671875, 0.243408203125, -0.364990234375, 1.3798828125, 2.0078125, 1.6875, -0.68603515625, 1.640625, 1.5263671875, 0.71826171875, -0.732421875, 0.016326904296875, 0.2254638671875, 1.05859375, -1.390625, -0.28955078125, 0.0281219482421875, 2.611328125, -1.1904296875, -0.935546875, 0.215087890625, 0.97119140625, -0.465576171875, 0.486328125, 0.77978515625, 0.07623291015625, -1.9423828125, -0.03826904296875, -1.7763671875, 1.216796875, -1.005859375, 0.703125, 1.11328125, -0.1429443359375, -0.0311126708984375, -0.36279296875, 0.42041015625, 0.044891357421875, -1.1064453125, 0.87939453125, 0.30859375, 0.587890625, -0.0108489990234375, -1.3779296875, -0.80078125, 0.2186279296875, -0.7666015625, -0.27685546875, -0.43701171875, 0.498291015625, 1.2421875, 1.65625, 0.45751953125, -0.058380126953125, -1.880859375, -1.3466796875, -0.6083984375, 0.494140625, -1.64453125, 2.0703125, -0.01177215576171875, 0.43310546875, -0.2607421875, -2.728515625, 1.31640625, 0.70263671875, 1.787109375, -0.4072265625, 0.286376953125, 0.5771484375, -1.328125, 0.4765625, 0.48876953125, -0.059783935546875, -0.0919189453125, -0.150390625, -0.1527099609375, 0.53515625, 0.67529296875, -1.123046875, 0.0233612060546875, -0.8955078125, 1.2236328125, -0.79443359375, -0.19091796875, 0.025146484375, -1.380859375, 0.40966796875, -1.7275390625, -0.112060546875, -0.6298828125, -1.236328125, -1.7490234375, 1.111328125, -1.1103515625, -1.12109375, 2.044921875, 1.5078125, 0.40673828125, 0.2081298828125, 0.51025390625, 1.5361328125, 1.564453125, 0.61376953125, 0.64306640625, -0.3310546875, -0.87939453125, 0.02239990234375, -0.421142578125, 0.25048828125, -0.498046875, 1.6689453125, -2.849609375, 0.8369140625, 0.42041015625, -0.09771728515625, -0.1734619140625, -2.888671875, 2.2890625, 0.2210693359375, 0.76220703125, 2.025390625, -0.08770751953125, -0.76171875, 1.59765625, 0.90380859375, -1.587890625, -0.49462890625, -1.03515625, -0.66357421875, -1.2158203125, -0.403076171875, 0.1729736328125, 0.1268310546875, 1.4169921875, 0.5966796875, 0.67333984375, 0.44384765625, 2.412109375, -1.0537109375, 1.0166015625, 0.81689453125, 2.1484375, 0.123779296875, 1.912109375, 0.44189453125, 1.7392578125, -0.7119140625, 1.3505859375, -0.129150390625, -1.052734375, -1.0419921875, 0.876953125, -0.64697265625, -1.830078125, 0.67822265625, -1.4365234375, 0.1798095703125, 2.21875, -1.416015625, -2.06640625, 0.1239013671875, -1.28515625, -1.9521484375, -1.263671875, -0.371826171875, -0.435791015625, -0.81640625, -0.77294921875, 0.724609375, 1.0615234375, 0.9072265625, 0.3818359375, -1.5771484375, 0.86376953125, -0.92333984375, 1.0498046875, -1.181640625, -0.45166015625, 0.431884765625, 0.08673095703125, 0.884765625, -1.177734375, -1.3447265625, 1.0283203125, -0.947265625, -0.2310791015625, -0.5712890625, -0.56591796875, 0.509765625, 1.1181640625, 0.1856689453125, -0.45703125, 1.2548828125, 1.677734375, -0.71630859375, -0.8046875, 0.17041015625, -1.3369140625, 0.322509765625, -1.349609375, 0.09417724609375, 0.408203125, 0.298828125, -0.081787109375, -1.330078125, 1.16796875, -2.234375, -0.2078857421875, 0.56494140625, -0.94140625, 0.06884765625, -0.161376953125, 0.962890625, 2.552734375, -1.0673828125, -0.281982421875, 1.294921875, 0.50439453125, -0.07684326171875, 0.052734375, -0.287353515625, 0.9384765625, -0.032989501953125, 0.3681640625, 0.12127685546875, 3.10546875, 0.05877685546875, -0.370361328125, 0.59423828125, 0.83740234375, 1.71484375, -0.4482421875, 0.1712646484375, -0.56005859375, 0.12274169921875, -1.017578125, -0.82666015625, 0.2095947265625, -1.39453125, -1.12109375, -0.359130859375, -0.79052734375, -1.2333984375, 0.65380859375, -0.5087890625, -1.16796875, 0.355712890625, -0.163330078125, 0.054931640625, -1.296875, 1.3154296875, -1.158203125, -0.0958251953125, -2.8515625, -0.366943359375, 1.298828125, -1.1611328125, -0.245849609375, 0.32275390625, -0.97119140625, 0.81103515625, -1.1923828125, 1.6845703125, -1.8583984375, 1.6396484375, -0.92529296875, -0.68701171875, -0.86962890625, -0.3017578125, -1.0390625, -0.90625, -0.470703125, 0.4072265625, 0.306884765625, -0.7197265625, 1.0283203125, -0.09375, -0.29248046875, -0.541015625, -0.332275390625, 0.619140625, -0.081787109375, -0.250732421875, -0.62548828125, -1.482421875, 0.72607421875, -0.1746826171875, 0.93994140625, -1.5556640625, -2.041015625, 0.1986083984375, -0.744140625, -0.14990234375, -0.65771484375, -0.0718994140625, 0.7587890625, -0.388916015625, 0.341064453125, 0.63525390625, -0.1754150390625, 1.744140625, 2.0, 0.05682373046875, 1.0537109375, -0.01172637939453125, -1.5009765625, 1.3125, 0.6025390625, 2.080078125, -1.666015625, -1.6728515625, 2.0859375, 2.123046875, -0.043853759765625, 0.54541015625, -1.478515625, -0.7255859375, -1.73046875, -0.65234375, 1.0361328125, -0.10650634765625, 1.4853515625, 0.71044921875, -2.55859375, 0.5859375, 0.2178955078125, -0.66259765625, -0.5126953125, -0.6552734375, 0.88916015625, 0.63037109375, 0.3564453125, 0.13427734375, -1.15234375, 0.5703125, -1.330078125, 1.2158203125, 1.3896484375, 0.291259765625, 1.103515625, 0.1019287109375, 1.4326171875, -0.666015625, 2.2578125, 0.1544189453125, 0.047088623046875, -0.54345703125, 0.947265625, 0.97900390625, -1.5458984375, 0.327392578125, -0.042694091796875, -0.4736328125, 0.478271484375, 1.34375, -2.931640625, 0.408203125, 1.1533203125, 0.219482421875, 1.4228515625, 0.6484375, 0.25830078125, 0.490478515625, -0.919921875, 0.0112457275390625, 0.919921875, -0.410888671875, 0.1885986328125, -0.275146484375, 1.87890625, -0.0740966796875, 0.6220703125, -1.4287109375, -0.5810546875, -0.59521484375, -0.297119140625, -2.09765625, 0.63818359375, -0.2122802734375, 1.470703125, -0.80908203125, -0.27197265625, 0.291259765625, 0.302490234375, 0.402099609375, -0.2445068359375, 2.287109375, 1.7353515625, -0.046112060546875, 0.90966796875, -0.0240325927734375, 0.9443359375, -1.1259765625, -1.271484375, 0.11273193359375, -1.4921875, 0.65185546875, -2.783203125, -0.6455078125, 0.94384765625, 0.18408203125, 1.451171875, 0.86376953125, -1.658203125, 0.1363525390625, -0.31005859375, -0.73876953125, 1.044921875, 0.52587890625, -1.314453125, 1.13671875, -1.4716796875, -0.083251953125, -1.0966796875, -2.677734375, 2.009765625, -0.64501953125, -1.2998046875, 0.2230224609375, -0.024932861328125, 0.7314453125, -0.6103515625, -0.07012939453125, -0.92724609375, -0.5810546875, -0.372314453125, 0.96533203125, 0.89501953125, 0.1793212890625, -1.890625, -0.53369140625, -0.59521484375, -1.2333984375, -2.123046875, 1.078125, 0.9169921875, 0.150634765625, -1.1845703125, -0.56689453125, -0.1629638671875, 0.2200927734375, -1.7236328125, -1.794921875, 1.123046875, -1.5849609375, 0.46044921875, -0.12200927734375, -0.110107421875, -0.24658203125, -0.41064453125, 1.7822265625, 0.734375, 0.309326171875, -0.10113525390625, -0.94482421875, 1.7431640625, -0.12005615234375, -0.60400390625, 1.3212890625, -1.05859375, -1.2841796875, -0.21044921875, -0.7763671875, -0.5458984375, -0.308837890625, -0.27392578125, -0.51123046875, 0.12115478515625, 0.66943359375, 0.0170745849609375, -0.1529541015625, 1.1806640625, 0.9296875, 0.2391357421875, -0.2666015625, 0.0941162109375, -1.16796875, 1.4697265625, 0.348388671875, 0.00949859619140625, -1.54296875, -1.326171875, 0.213134765625, -1.232421875, 0.58203125, -2.09765625, 0.19482421875, 0.80712890625, -0.201904296875, -1.1171875, -0.0009860992431640625, 0.296142578125, 1.1953125, -0.68603515625, -0.414794921875, -0.5888671875, 0.184814453125, -0.298095703125, 0.974609375, 1.240234375, -1.3701171875, -2.07421875, -0.740234375, -2.380859375, -1.0556640625, 0.6484375, 0.266845703125, 0.83544921875, 2.365234375, 2.0390625, -0.39306640625, 0.271728515625, 0.96728515625, -0.50439453125, -0.56982421875, 2.052734375, 0.241455078125, 0.591796875, -0.031341552734375, -0.8857421875, -1.5146484375, -0.01409912109375, 0.29052734375, 1.0205078125, 0.318359375, 1.0146484375, -0.232666015625, -0.62451171875, 1.1259765625, 0.46923828125, 0.04022216796875, 0.5673828125, -0.324462890625, -0.51806640625, -0.137939453125, -0.0009565353393554688, -1.0615234375, 0.1534423828125, -0.64208984375, 0.1737060546875, -0.14990234375, -0.8876953125, 1.2333984375, 0.474365234375, -0.162841796875, 1.4169921875, -0.666015625, 1.568359375, -0.62548828125, -0.10711669921875, 1.32421875, -0.73583984375, 0.2001953125, -0.0997314453125, 0.33544921875, 0.0014705657958984375, -0.81201171875, 0.12091064453125, -1.015625, -0.0222625732421875, -0.267822265625, -0.72119140625, 1.0947265625, 0.93017578125, -0.62158203125, -1.8544921875, -0.425048828125, -0.00948333740234375, -1.2041015625, 0.5810546875, 0.0484619140625, -2.412109375, 0.031280517578125, 2.369140625, 0.9169921875, 0.059112548828125, 1.7529296875, 0.55615234375, -0.60791015625, -0.6826171875, 0.128662109375, 0.399169921875, 0.69287109375, 0.1790771484375, -0.004634857177734375, -1.017578125, 0.1905517578125, 0.1798095703125, 0.873046875, 0.85205078125, -0.5380859375, -1.2431640625, -0.701171875, 0.5537109375, -0.477294921875, -0.49609375, 1.85546875, 1.2314453125, 0.90380859375, 0.1771240234375, 0.52685546875, 1.2255859375, -0.10882568359375, -0.033050537109375, -0.51171875, -0.384033203125, -0.92138671875, -0.203125, -0.5380859375, 1.1044921875, -1.1337890625, -0.72314453125, -0.30419921875, 0.72509765625, -0.212890625, -0.11541748046875, 0.55712890625, 0.09747314453125, 1.2431640625, 0.6396484375, -1.44921875, -0.1844482421875, -1.1396484375, 0.11151123046875, -0.35009765625, -1.3427734375, -0.039276123046875, 0.14697265625, 0.436279296875, 0.37548828125, 0.9189453125, -0.2347412109375, 0.93505859375, 0.4921875, -0.9365234375, -1.970703125, -1.8974609375, 0.7421875, 0.04473876953125, 2.91015625, -1.865234375, -0.25634765625, -0.83935546875, 1.447265625, -0.194091796875, -1.8857421875, 0.019744873046875, -1.15234375, -0.7900390625, 0.3779296875, -0.62890625, 0.1317138671875, 0.1558837890625, 0.34619140625, 0.1119384765625, -1.8466796875, 0.07794189453125, -1.119140625, -0.42578125, 1.123046875, -0.401123046875, -1.4033203125, 0.8232421875, 1.275390625, 0.638671875, 1.90625, 0.119873046875, -0.64453125, -1.7080078125, -1.06640625, 0.98095703125, -1.4990234375, -1.033203125, -1.6826171875, 0.744140625, -0.248046875, -0.47900390625, 1.462890625, 0.483642578125, -0.79345703125, 0.260009765625, 1.23828125, 0.1727294921875, -0.7861328125, -1.740234375, -0.5498046875, 0.0214385986328125, 0.71875, -0.69677734375, 0.78271484375, -1.1640625, -0.8310546875, -0.178955078125, -0.0595703125, -0.98583984375, 1.0849609375, -1.3388671875, -1.47265625, 0.2159423828125, 0.445068359375, 0.2183837890625, 0.6044921875, -1.3232421875, -1.2587890625, -1.16015625, -1.171875, -0.93896484375, 0.07373046875, 1.6171875, 0.7119140625, 0.1602783203125, -0.18505859375, -0.06182861328125, -1.0556640625, 0.888671875, 1.2802734375, 0.339111328125, 0.3505859375, 1.6240234375, 1.9638671875, -0.689453125, -0.6845703125, 0.54443359375, 0.478515625, -1.1162109375, -0.416748046875, -0.658203125, 0.01227569580078125, -1.12890625, 1.091796875, 1.31640625, 1.1904296875, -0.32666015625, 1.3310546875, -1.583984375, -0.07489013671875, 1.4697265625, 0.350341796875, 0.27587890625, 0.2078857421875, -0.155029296875, -0.662109375, 0.00011348724365234375, -1.24609375, -0.54736328125, 0.5107421875, -0.1644287109375, 0.243896484375, -1.5126953125, 0.15087890625, -0.444580078125, -0.55419921875, -0.00786590576171875, -1.556640625, -0.666015625, -1.140625, -0.337158203125, -0.34619140625, 0.28955078125, -1.712890625, -0.171630859375, -1.6162109375, -0.16455078125, 1.9482421875, -0.30615234375, 1.2021484375, 0.568359375, 0.1573486328125, -0.0083160400390625, -0.5771484375, -0.005001068115234375, 0.9228515625, -0.84765625, 0.8916015625, 1.0234375, -0.30126953125, 0.2484130859375, -0.288818359375, 0.1265869140625, 0.626953125, -0.43408203125, 0.62548828125, -0.39453125, -1.32421875, -0.74658203125, -0.96875, 0.2509765625, -1.4208984375, 0.59521484375, -0.36279296875, 0.1483154296875, -0.5439453125, -2.197265625, 0.8154296875, 0.52490234375, 0.0173492431640625, -0.1876220703125, -0.314697265625, -0.65576171875, 1.1171875, 1.8642578125, -0.97216796875, -0.08343505859375, -0.7734375, 0.2034912109375, -0.266357421875, -0.06365966796875, 0.25732421875, -0.373291015625, 0.435546875, -0.057525634765625, 1.2490234375, -0.2010498046875, -1.421875, 0.88671875, -0.155029296875, 0.85302734375, 1.619140625, 1.08203125, 0.370849609375, -0.09332275390625, 1.1669921875, -1.4443359375, -0.1966552734375, -0.779296875, 0.094482421875, -0.8720703125, -1.0244140625, 1.1513671875, -0.290283203125, 0.2423095703125, 0.57421875, -0.7900390625, 1.6328125, 0.39208984375, 0.2529296875, 0.09722900390625, -0.197021484375, 0.456787109375, 1.3095703125, 0.57666015625, 0.11370849609375, -0.90087890625, -0.14404296875, 1.1708984375, 1.5166015625, 2.421875, 1.32421875, -2.197265625, 0.1365966796875, -0.6220703125, 0.767578125, 0.220458984375, -0.890625, 1.1611328125, 0.2047119140625, -0.88232421875, -0.54150390625, -0.86181640625, 0.59619140625, -1.169921875, 1.015625, -1.9443359375, 0.420654296875, 0.41015625, -0.66748046875, 0.474365234375, 1.326171875, -0.92431640625, -1.103515625, 0.38330078125, 1.373046875, -0.283447265625, -0.8212890625, -0.474609375, -0.35009765625, 0.59765625, 0.374755859375, -1.4013671875, 0.70654296875, -0.60986328125, 0.03216552734375, 0.9970703125, -0.89404296875, 0.386962890625, 0.50927734375, 0.08392333984375, -0.1602783203125, -0.91748046875, 0.91650390625, 0.376220703125, -0.287109375, -1.458984375, 0.56640625, 0.93701171875, 0.180419921875, 0.06585693359375, -1.40625, -0.1455078125, -0.55810546875, -1.9716796875, -1.0478515625, 1.7861328125, -0.318603515625, 0.70849609375, 0.315673828125, 0.51318359375, 0.11407470703125, -1.361328125, -0.1324462890625, 1.0830078125, -0.76171875, -0.7578125, 0.64697265625, 0.5703125, -0.33349609375, -2.951171875, 0.2401123046875, 0.005443572998046875, -0.466552734375, 0.51611328125, 1.2490234375, 0.98291015625, -0.7353515625, 0.11834716796875, 0.546875, 1.287109375, -0.4560546875, -1.1015625, -0.6591796875, 0.42138671875, -0.91064453125, 0.2264404296875, -0.92138671875, -0.7060546875, 0.66552734375, 0.0792236328125, 1.4736328125, -1.2939453125, -0.8046875, -0.83642578125, 1.828125, 1.888671875, 0.0006103515625, 0.7490234375, -0.0118408203125, 1.4404296875, -0.21875, -0.367919921875, 0.83544921875, 0.6669921875, 0.499267578125, 1.0263671875, 0.51220703125, -1.17578125, -0.71337890625, 0.587890625, 1.2666015625, -0.94384765625, -1.0068359375, -0.89208984375, -0.02557373046875, -0.2459716796875, -0.01552581787109375, 1.5068359375, -0.796875, -1.1552734375, -0.08721923828125, 0.60302734375, -0.77197265625, 0.1070556640625, -0.1922607421875, -2.515625, 1.0673828125, -1.138671875, 0.79736328125, 0.81396484375, 0.630859375, 0.36767578125, 0.25146484375, -0.20361328125, 1.8525390625, 0.765625, -0.320068359375, 0.537109375, 1.40625, -0.71337890625, -0.66796875, 0.72021484375, -1.466796875, 0.52197265625, -1.556640625, 0.27685546875, -0.94189453125, 0.68994140625, 1.0810546875, -1.4375, -0.63232421875, -0.427001953125, -0.09539794921875, 2.828125, -0.375, 1.375, 0.798828125, 0.72705078125, 1.3466796875, 0.2447509765625, -0.437744140625, -0.1688232421875, -1.21875, -0.41162109375, 0.00951385498046875, -0.274169921875, 0.6943359375, -0.80029296875, 0.0257720947265625, -0.66796875, -0.5087890625, 1.4580078125, -2.03125, -1.4111328125, -0.6376953125, -2.69921875, -0.285888671875, -0.9677734375, -1.111328125, -0.277099609375, 0.323486328125, -1.7001953125, -0.405517578125, -0.58544921875, -0.18115234375, -0.8095703125, -0.958984375, 0.615234375, -1.03515625, -0.0374755859375, -0.0498046875, 0.642578125, 1.796875, 1.2685546875, -0.73583984375, 0.5126953125, -0.09808349609375, 0.53564453125, 0.0887451171875, -2.841796875, 0.0408935546875, 1.02734375, -0.29541015625, -1.9951171875, -0.42236328125, 1.01171875, -0.0927734375, 0.407470703125, 1.0283203125, 0.0007452964782714844, 1.4892578125, 0.92578125, 0.51708984375, 1.755859375, 2.30078125, -0.393798828125, -0.34814453125, -0.529296875, -0.407958984375, 0.662109375, -0.81201171875, -0.004817962646484375, -0.1451416015625, 0.08447265625, 0.377685546875, -1.66796875, 0.6416015625, 0.2457275390625, 0.017730712890625, -1.90234375, -0.37451171875, 1.875, -1.068359375, 2.0078125, -1.6015625, -0.5107421875, -0.80712890625, 0.58349609375, 0.137939453125, 0.74365234375, 0.744140625, -0.373046875, -0.794921875, -1.0390625, -0.220947265625, 0.6328125, -0.24462890625, 1.0576171875, 0.853515625, 0.11956787109375, -0.459228515625, -2.001953125, -0.76611328125, -0.93408203125, 0.24169921875, 0.0310211181640625, -0.425537109375, 0.8955078125, -0.407470703125, 0.53955078125, -0.006500244140625, -1.16796875, 1.4716796875, 0.053863525390625, -0.5908203125, -0.751953125, -0.5615234375, 1.0478515625, -0.005611419677734375, -1.0380859375, 0.305908203125, -0.84912109375, -1.564453125, -1.564453125, 0.35302734375, 0.09393310546875, -0.01837158203125, 0.0201263427734375, -0.11944580078125, 2.107421875, -0.64990234375, -0.046600341796875, 1.205078125, 0.50634765625, 0.3359375, 0.21044921875, -2.068359375, -0.76904296875, -0.485595703125, 0.365234375, 0.03564453125, 2.578125, -0.8857421875, 0.497314453125, -1.150390625, 0.572265625, -0.513671875, 0.001922607421875, -0.32666015625, -1.7421875, 1.0078125, 0.4599609375, -1.1025390625, 3.0859375, 0.693359375, 0.377685546875, 0.10040283203125, 0.06634521484375, -0.15478515625, 1.5087890625, 0.60107421875, 0.135986328125, -0.5576171875, 1.1376953125, -0.50732421875, -0.1773681640625, -0.46826171875, 1.77734375, 0.548828125, 0.81884765625, 0.68701171875, 0.03472900390625, -0.6123046875, 0.765625, 0.95361328125, -0.38818359375, -1.0771484375, 1.1279296875, -1.2265625, 0.77587890625, 0.11529541015625, 1.447265625, 0.5810546875, 0.8955078125, 0.1556396484375, 0.6083984375, -1.931640625, 0.67822265625, 0.2978515625, -0.9208984375, 0.26318359375, 0.2362060546875, -0.833984375, -0.71337890625, 0.2435302734375, 0.89990234375, 1.607421875, -0.5146484375, 1.6171875, 0.1009521484375, -1.9189453125, -1.0498046875, 1.1669921875, 1.7158203125, 0.677734375, 0.84130859375, 0.02606201171875, -2.244140625, 1.015625, 0.71923828125, 0.07916259765625, -0.2900390625, -1.36328125, 2.58984375, 0.7783203125, 1.833984375, -1.2294921875, -0.9404296875, -0.439453125, 0.30615234375, -0.669921875, -1.7666015625, -0.31005859375, -1.8056640625, 0.71240234375, -0.391845703125, 0.90966796875, 1.2099609375, 0.0946044921875, 0.447265625, 0.012908935546875, -0.87060546875, 0.2376708984375, 1.9599609375, -0.2491455078125, -1.4833984375, -1.2314453125, 0.68017578125, 0.62451171875, -0.56982421875, -0.387939453125, -0.4697265625, -0.82275390625, 0.326904296875, 1.5634765625, 0.97021484375, 1.27734375, -0.8720703125, -0.488525390625, -0.81396484375, -0.2646484375, -0.0513916015625, -0.10711669921875, 0.46240234375, 0.369873046875, 0.17724609375, 0.71484375, -0.52392578125, -0.372802734375, -0.36572265625, 0.000640869140625, -0.9619140625, 0.873046875, 3.150390625, 0.0097198486328125, -0.40087890625, -0.66455078125, 0.316650390625, -1.013671875, 0.66357421875, 0.59228515625, 1.6162109375, 0.0771484375, -0.40283203125, -0.07867431640625, 1.6796875, -0.9775390625, -0.54052734375, -0.947265625, 0.9794921875, 0.96826171875, 0.0268402099609375, -0.3154296875, -0.7001953125, -1.4931640625, -1.828125, -0.1783447265625, -0.4873046875, -0.03765869140625, 1.47265625, 0.32470703125, -0.46533203125, 0.828125, 0.159423828125, -0.5244140625, -0.37744140625, 0.15576171875, -0.6943359375, 0.13232421875, 0.86279296875, -2.2265625, -0.3330078125, -0.087646484375, -0.0408935546875, -0.0582275390625, 0.1025390625, 0.36083984375, 0.5009765625, 0.251953125, 1.4267578125, -1.6474609375, -0.47705078125, -0.1492919921875, 1.169921875, -0.020050048828125, -0.51171875, 0.2247314453125, -2.60546875, 0.802734375, 0.86474609375, 0.771484375, -0.25634765625, -0.036590576171875, 1.09765625, 0.49609375, -0.80029296875, -0.24462890625, -0.81787109375, 0.1385498046875, -1.408203125, -0.578125, 0.049224853515625, -0.1383056640625, -0.1380615234375, 0.1590576171875, 0.8232421875, 0.91259765625, 0.379638671875, -0.51611328125, -0.459716796875, -0.89599609375, -0.091064453125, 0.089599609375, -0.475830078125, -0.5751953125, -0.304443359375, 0.489990234375, 0.244384765625, -1.2431640625, 0.9677734375, 0.0361328125, 1.7158203125, 0.35888671875, -1.6220703125, 0.6416015625, -0.6337890625, -0.9853515625, -0.68603515625, 0.1492919921875, -1.33984375, -0.57568359375, -0.359375, 0.78466796875, -0.51416015625, 0.74853515625, -0.5078125, -1.91796875, 0.5244140625, 1.8779296875, 0.34375, -0.276123046875, 0.09814453125, 0.5380859375, -0.53076171875, -0.12225341796875, -1.6474609375, -1.1328125, -0.2421875, -1.291015625, -0.4931640625, -1.884765625, -0.0357666015625, 1.02734375, -0.84326171875, 0.3115234375, 0.451416015625, 0.4326171875, 0.94873046875, -0.9169921875, 0.634765625, -0.0711669921875, -0.47021484375, 0.08453369140625, -2.34765625, 1.3203125, 0.356201171875, -1.2236328125, -1.3857421875, 0.60791015625, -0.1082763671875, 0.440673828125, 1.7060546875, 0.039794921875, 0.76171875, -0.0012731552124023438, 1.8603515625, 0.5595703125, -1.265625, 0.45166015625, 0.0274200439453125, 0.300537109375, 1.6796875, -0.216796875, -0.41357421875, -2.84375, 0.72265625, -0.0975341796875, -0.4443359375, -0.572265625, -0.290283203125, 0.64599609375, 0.73291015625, -2.0078125, 0.79296875, 1.0615234375, -0.440185546875, 0.07232666015625, -0.689453125, 0.452392578125, 0.12646484375, 0.497314453125, 0.84814453125, -0.031005859375, -0.25830078125, -0.452392578125, 2.529296875, 0.2357177734375, -0.50732421875, 0.64599609375, -1.94140625, -1.50390625, -0.27880859375, -0.447998046875, -0.53173828125, 0.26611328125, 0.1396484375, 0.095703125, 0.062469482421875, 0.26611328125, -0.99365234375, 0.188720703125, -0.6796875, 0.405029296875, -0.1033935546875, -0.1693115234375, 0.035369873046875, 1.1640625, -1.01953125, 0.998046875, -2.595703125, 1.5048828125, -0.389892578125, -0.9462890625, -0.5439453125, 0.021697998046875, -1.1220703125, -1.3779296875, 0.84521484375, -1.9384765625, -0.62109375, 0.916015625, 0.474365234375, 1.212890625, 0.01702880859375, 0.62744140625, 0.90625, 0.365966796875, -0.0242462158203125, 0.94384765625, -0.267822265625, 0.308837890625, -1.1982421875, 0.83447265625, -2.439453125, 0.75439453125, 1.0634765625, -0.5087890625, -0.035797119140625, -1.5546875, -0.40673828125, -0.6015625, 0.34130859375, 0.76416015625, 0.1422119140625, 0.024139404296875, 1.0400390625, 0.87646484375, -0.36181640625, -0.213623046875, -1.9345703125, 0.11273193359375, 0.20068359375, -0.22509765625, 0.642578125, 0.5458984375, 1.083984375, -1.4833984375, -0.7919921875, -0.5888671875, 0.994140625, -0.300537109375, 1.0751953125, 0.80322265625, 1.5791015625, 0.465576171875, -1.3349609375, -0.443115234375, -0.01396942138671875, 1.0419921875, -2.271484375, 0.1766357421875, 0.1885986328125, 0.2320556640625, -0.1876220703125, 0.88916015625, -0.373291015625, -0.016998291015625, 0.127685546875, 0.57861328125, 1.0419921875, -0.85888671875, -0.35595703125, -0.65234375, -0.91796875, 0.10394287109375, 0.7958984375, -0.264892578125, -0.6982421875, -0.2427978515625, 0.445068359375, -0.91357421875, -0.83203125, 1.76953125, -1.544921875, -0.29833984375, 2.5234375, 0.06719970703125, -1.0849609375, -1.251953125, 0.371826171875, 1.0419921875, 0.10833740234375, 0.77099609375, -0.11639404296875, 0.7001953125, -0.309326171875, 0.1907958984375, 0.689453125, 0.322265625, 0.10992431640625, 1.6962890625, -0.076416015625, -1.0380859375, 0.74072265625, 0.931640625, 0.2666015625, -0.77490234375, -0.7705078125, -1.5185546875, 1.0087890625, 1.1728515625, 1.712890625, 0.495849609375, -0.99951171875, 1.744140625, -0.62890625, 0.68359375, 2.16015625, 0.390869140625, 0.78369140625, 0.115966796875, -0.392333984375, -0.9228515625, -0.77978515625, 0.8759765625, -0.966796875, 1.7978515625, -0.5693359375, 0.459716796875, 0.26220703125, -0.068603515625, 0.98193359375, 0.97802734375, -1.375, -1.298828125, 1.4765625, -0.04736328125, 1.5908203125, 0.330322265625, -0.1016845703125, -0.79345703125, -0.64453125, 0.6328125, 0.5390625, 1.515625, -0.0892333984375, 0.60400390625, 0.492431640625, 0.80029296875, -1.615234375, 2.900390625, 0.78125, 1.4765625, 1.0888671875, -0.29052734375, -0.84716796875, -1.1630859375, -1.453125, 0.09039306640625, -2.0078125, 0.427490234375, 0.341064453125, -0.3525390625, 0.57177734375, -0.09332275390625, -0.7080078125, -0.48046875, -0.27001953125, -0.2000732421875, 0.81396484375, -0.006866455078125, 0.0191192626953125, -0.65234375, 0.01157379150390625, 0.0592041015625, -1.0224609375, 0.95263671875, -0.236083984375, -1.1552734375, 0.65966796875, -0.5185546875, -0.0631103515625, -0.361572265625, -0.4111328125, 0.322998046875, 0.52490234375, 1.9130859375, -1.568359375, -1.4794921875, -0.342041015625, 0.047454833984375, -1.84375, 1.18359375, -1.181640625, -0.01497650146484375, 0.79638671875, 0.331787109375, -0.480712890625, 0.072998046875, -1.947265625, 0.45068359375, -0.0063629150390625, -0.1475830078125, -1.626953125, -1.4501953125, -1.111328125, 0.67529296875, 0.85986328125, -0.43701171875, -0.59033203125, 0.12493896484375, 0.326904296875, 0.5, 0.156982421875, 0.74951171875, 1.677734375, 0.72998046875, -0.023956298828125, -0.73291015625, -0.4736328125, 0.66845703125, -0.99560546875, -0.78369140625, -0.244873046875, 1.671875, 0.255126953125, -1.4833984375, -0.32373046875, -1.2138671875, -0.85498046875, 0.72607421875, -2.119140625, -0.5107421875, 0.246826171875, 0.20361328125, -0.4208984375, -1.4423828125, -1.66015625, -0.367431640625, 0.3740234375, -0.74951171875, 1.1748046875, 0.0190277099609375, -0.2052001953125, 0.62939453125, -1.51171875, 0.06585693359375, 0.65234375, -0.154296875, 0.2415771484375, 1.7158203125, 0.27734375, -0.7060546875, -1.9482421875, -0.7490234375, 1.05078125, -0.289794921875, 0.892578125, 0.7509765625, -0.333740234375, 0.70458984375, -0.023468017578125, -0.72412109375, -0.6865234375, 0.962890625, 0.00833892822265625, -1.0732421875, 0.8583984375, -1.7666015625, -0.833984375, 0.5146484375, -1.6396484375, -0.0126495361328125, 1.107421875, -0.485595703125, 0.52587890625, 1.9521484375, -0.9130859375, 0.12115478515625, -0.8876953125, -0.52392578125, 0.71044921875, 0.426513671875, -0.376220703125, -0.79638671875, 0.52294921875, -0.039825439453125, 0.54296875, 0.33203125, -1.673828125, -0.35546875, -1.072265625, -0.56982421875, -0.80126953125, -0.051055908203125, 0.10009765625, 0.440185546875, 1.4423828125, 1.3486328125, 1.185546875, -0.239013671875, 0.93212890625, 1.447265625, -1.3798828125, 0.87890625, -0.931640625, -0.9150390625, -2.591796875, -1.2890625, 0.296630859375, -1.318359375, -0.73779296875, -1.771484375, -0.18603515625, -0.9140625, 0.09149169921875, -1.216796875, 0.1490478515625, 0.88720703125, -0.328369140625, -0.207763671875, 0.1708984375, 0.53271484375, 0.08233642578125, -1.158203125, 0.24951171875, -1.0078125, -0.19091796875, 0.666015625, 1.623046875, 0.025421142578125, 2.015625, -0.001895904541015625, 2.212890625, -0.82177734375, 3.30859375, 2.00390625, -0.1365966796875, 0.38720703125, -0.1787109375, -0.5927734375, -2.279296875, -0.5322265625, 1.0615234375, -0.342041015625, 0.08636474609375, 1.7880859375, 0.69384765625, -0.2279052734375, -0.1500244140625, 0.318359375, 1.2021484375, -0.23193359375, -1.5302734375, 0.058837890625, 1.224609375, 1.8310546875, 0.51806640625, -0.76171875, -1.7626953125, -0.50146484375, 0.33984375, -0.9130859375, 1.482421875, 0.88525390625, 0.83935546875, 1.728515625, -0.02545166015625, 0.054656982421875, 0.70556640625, 1.1748046875, -0.7666015625, 1.212890625, 1.2822265625, -0.2412109375, -2.298828125, -0.673828125, 1.466796875, 0.05169677734375, 0.0303192138671875, 1.4560546875, 1.6630859375, 1.166015625, 1.08984375, 0.54345703125, 0.08941650390625, -0.407470703125, 0.306396484375, -1.2998046875, 0.88134765625, -1.22265625, -0.47802734375, 0.9384765625, -1.4599609375, 2.0625, -0.77880859375, 0.80029296875, 1.0400390625, -0.196533203125, 0.91162109375, -0.74462890625, -0.1868896484375, 0.40478515625, 0.9609375, 1.4404296875, 0.685546875, 0.7373046875, -1.1435546875, 0.434814453125, -0.9345703125, -1.3828125, -0.68115234375, -1.1494140625, 0.291259765625, -0.73828125, 0.34912109375, -1.666015625, -0.93896484375, -0.314697265625, -2.271484375, 0.7958984375, -1.2685546875, -1.39453125, 1.986328125, -0.12274169921875, -0.425048828125, -1.0556640625, -1.3125, 0.1279296875, -0.71484375, -1.0732421875, -1.6494140625, -0.171630859375, 0.76708984375, 1.8642578125, -0.033935546875, 2.3203125, 1.1826171875, -1.9599609375, 0.67724609375, 0.177490234375, 0.63232421875, 0.441162109375, -0.1270751953125, 1.091796875, -0.69580078125, -0.2264404296875, -1.533203125, 1.041015625, 0.486328125, -0.2529296875, 0.81982421875, -0.2330322265625, 0.43212890625, -0.0799560546875, 0.96044921875, 0.3642578125, -0.70166015625, -1.19921875, 0.7373046875, -1.166015625, 1.484375, -1.4384765625, -1.578125, -1.7099609375, 0.701171875, 0.3466796875, 2.05859375, 1.64453125, 0.0174407958984375, 0.3759765625, 0.153564453125, 0.75537109375, -0.27490234375, -1.0498046875, 0.02423095703125, -0.80078125, 1.265625, 0.34716796875, 1.15625, -0.47265625, 1.1845703125, 1.931640625, 0.6455078125, -1.15625, -0.59130859375, 1.3583984375, -0.1640625, 0.152099609375, -1.048828125, 0.024993896484375, 0.51171875, 0.7333984375, -2.267578125, -1.34375, 1.9306640625, -0.79248046875, 2.0859375, 0.4111328125, 2.357421875, 0.322021484375, 1.615234375, -0.294677734375, 0.3603515625, -1.4501953125, 0.9345703125, -0.9560546875, 0.1424560546875, 1.1767578125, -0.90576171875, 0.1517333984375, 1.23046875, 0.435791015625, 0.336181640625, 0.271484375, 0.85400390625, -0.492431640625, -0.8544921875, 0.33447265625, -0.541015625, -0.04248046875, -0.18798828125, 0.259765625, 0.447509765625, 1.515625, -2.126953125, -0.5927734375, 1.5302734375, -0.51513671875, -0.11700439453125, 0.90673828125, -0.1673583984375, -0.84814453125, 0.64892578125, 0.0926513671875, -0.98388671875, 1.8095703125, 0.65576171875, -0.80810546875, -0.80859375, 0.057159423828125, 0.2474365234375, 0.267822265625, 0.85302734375, -0.369140625, -0.1812744140625, -0.08746337890625, 1.0595703125, -0.0390625, 0.2476806640625, 0.55908203125, -0.15283203125, -0.7509765625, 0.83740234375, 0.55078125, -0.6416015625, 1.3544921875, 0.16162109375, -0.490234375, 1.369140625, -0.67236328125, 0.88525390625, 1.6201171875, 1.4931640625, -1.1572265625, 1.1162109375, -0.055816650390625, -1.3828125, -1.9384765625, -0.02685546875, -0.5517578125, -0.02593994140625, 0.73388671875, 1.0732421875, 0.0160675048828125, -1.15625, -0.049102783203125, -0.63134765625, -0.5087890625, -1.4775390625, 0.52197265625, 0.35400390625, -1.0419921875, -1.6455078125, 1.01953125, 0.63818359375, -0.9599609375, 1.22265625, 0.296875, -1.9775390625, -0.4990234375, -0.63427734375, 0.1414794921875, -1.33203125, -0.2205810546875, 0.385498046875, 0.251220703125, 0.19580078125, 0.038665771484375, -0.60302734375, 0.445068359375, -1.375, -0.5048828125, 0.079345703125, 0.62890625, -1.150390625, -1.251953125, -1.9443359375, 1.3369140625, 0.79248046875, 0.21923828125, -2.2890625, 1.3466796875, -0.440673828125, -0.034759521484375, -1.3857421875, 0.396728515625, -0.525390625, -0.05279541015625, -0.221923828125, 0.51513671875, -0.09307861328125, -1.173828125, -0.42333984375, -0.73291015625, 0.88427734375, -0.11968994140625, 0.6669921875, 0.802734375, 2.662109375, 0.3994140625, 0.73779296875, -0.98583984375, 0.337158203125, -0.24951171875, -1.189453125, -2.173828125, -0.93017578125, 0.025787353515625, 0.467529296875, -0.15625, 0.250732421875, 1.6318359375, 0.395751953125, -0.76318359375, 0.135986328125, 0.60546875, -0.2440185546875, -0.3984375, -0.483154296875, -0.09185791015625, -0.0214996337890625, -0.79736328125, 2.634765625, 0.673828125, 0.497802734375, -2.0390625, -0.1304931640625, 0.6982421875, -0.68994140625, 0.303466796875, 0.88134765625, -0.728515625, 0.84619140625, 0.1715087890625, 0.68896484375, 2.490234375, -0.34765625, -0.476806640625, -0.1507568359375, 1.32421875, 1.232421875, 0.32470703125, 1.849609375, 1.19140625, 0.4150390625, 0.982421875, 1.5166015625, -1.40234375, -0.6494140625, 1.140625, 0.2467041015625, -0.9052734375, 0.48828125, 0.43115234375, -0.5888671875, -0.1307373046875, 0.79736328125, -1.232421875, -0.146484375, -0.5224609375, 0.99853515625, -0.34130859375, 0.023529052734375, 1.0693359375, 0.432861328125, -0.60107421875, -0.64404296875, -0.57421875, 1.4453125, 1.4775390625, -0.427734375, 0.59375, 0.84716796875, 1.2890625, 1.5263671875, 1.0576171875, 0.1116943359375, 0.09332275390625, 0.60791015625, 0.845703125, 0.88720703125, -0.41845703125, 1.66015625, -0.36328125, -1.341796875, -0.062744140625, 0.9267578125, 1.19921875, 1.0712890625, -0.377197265625, 1.0146484375, -0.310546875, 0.1781005859375, 0.60400390625, -0.340087890625, 1.3642578125, 0.333740234375, -0.298583984375, 0.94873046875, 0.923828125, 1.18359375, 0.65966796875, -0.86572265625, -0.50537109375, 0.2164306640625, -0.493896484375, 0.42578125, -0.112548828125, 0.6005859375, -1.142578125, -0.57763671875, 1.2783203125, -0.26904296875, -0.07684326171875, -0.06243896484375, 0.407470703125, -0.25537109375, -0.314697265625, 0.60986328125, 0.417724609375, -0.313720703125, -1.27734375, 1.611328125, -0.84814453125, 2.33203125, -0.452392578125, 0.5078125, -0.375, 0.951171875, 0.79345703125, -0.88330078125, 1.8662109375, 1.4404296875, 0.4638671875, -0.7412109375, 0.1531982421875, -0.2459716796875, -1.880859375, -0.70458984375, -0.513671875, 0.54296875, 2.13671875, 0.94873046875, -1.232421875, -0.4228515625, -0.91259765625, -0.94140625, -0.90576171875, 1.314453125, 0.83447265625, 0.060882568359375, 0.4794921875, -0.07464599609375, -1.119140625, 1.3388671875, 0.26806640625, 1.662109375, -0.80908203125, -0.285400390625, -1.0224609375, -0.2724609375, -1.13671875, -0.87646484375, -1.0185546875, 0.84716796875, -0.99169921875, 0.3583984375, 0.84814453125, 0.041412353515625, 1.283203125, -1.380859375, -1.6298828125, 0.5986328125, 0.56591796875, -1.578125, -1.3076171875, -1.404296875, 0.69482421875, -0.833984375, -0.1871337890625, 0.86865234375, -0.7021484375, -0.093505859375, 1.7060546875, 1.7744140625, 0.482177734375, 0.04254150390625, 1.1650390625, 1.158203125, 0.8310546875, -0.392822265625, -0.16162109375, 1.2958984375, -2.671875, -0.09368896484375, 2.30859375, -0.84765625, -0.8310546875, -0.038177490234375, -0.89111328125, 1.0751953125, 1.34765625, -1.9130859375, 0.20947265625, -1.2724609375, -0.268310546875, -1.2177734375, -1.1064453125, -1.5546875, -0.1092529296875, 0.68603515625, -0.59912109375, -0.0305938720703125, 0.52490234375, -0.51025390625, -0.08428955078125, 2.044921875, 1.9912109375, 0.480224609375, -0.320068359375, 1.9677734375, 1.41015625, -3.5859375, 0.59033203125, -0.185546875, -1.94921875, -0.02947998046875, 1.048828125, 0.465087890625, -0.8623046875, -0.07080078125, -0.018951416015625, 0.0472412109375, 0.52880859375, -0.3017578125, 1.3046875, 1.822265625, 1.09765625, 0.3046875, 1.19140625, -0.037811279296875, 0.07696533203125, -0.552734375, 0.07147216796875, -0.1068115234375, -0.86181640625, 1.53515625, 0.281494140625, 0.20458984375, -0.0059967041015625, 0.74609375, 0.2734375, 2.4765625, -0.228759765625, 0.76904296875, -1.6767578125, -0.069091796875, 0.10772705078125, 0.685546875, -0.470947265625, -2.150390625, 0.8271484375, 0.474365234375, -0.0072784423828125, 1.0634765625, -0.876953125, 1.2822265625, 0.410400390625, -0.55615234375, 0.755859375, 0.387451171875, 0.619140625, -0.61572265625, 1.0615234375, 0.18310546875, 0.381591796875, 0.1346435546875, 1.58984375, 1.5341796875, -0.1732177734375, -0.08380126953125, -0.62353515625, 0.496337890625, 0.250732421875, 0.148193359375, 1.7626953125, -0.65185546875, 0.419921875, -0.3818359375, -1.7783203125, -1.19140625, 0.63671875, 0.861328125, 1.39453125, 0.273193359375, -0.2362060546875, -0.82373046875, 0.61474609375, 0.69921875, -1.021484375, -0.16845703125, 0.136474609375, 0.2125244140625, 0.2120361328125, -0.4326171875, -0.6484375, -1.3896484375, -0.416748046875, -0.46044921875, -0.266357421875, -0.400634765625, 0.07177734375, 0.8017578125, 0.552734375, 0.38818359375, 1.126953125, -0.755859375, 1.6875, 1.8310546875, -0.37744140625, -1.3916015625, 0.382080078125, 0.330810546875, 0.44580078125, 0.361328125, -0.9482421875, -2.0625, -0.55712890625, -1.625, -0.66162109375, -0.0196533203125, 0.392578125, -1.5048828125, -0.345458984375, -0.345703125, 0.56884765625, -0.724609375, 0.052825927734375, 1.177734375, -1.0244140625, 0.0160980224609375, -0.427978515625, -0.3818359375, -0.496826171875, 0.1290283203125, -0.87841796875, 0.3828125, -0.435546875, -0.70654296875, 0.2734375, 0.465087890625, -2.275390625, 0.232177734375, -0.67236328125, -1.865234375, 1.0087890625, 0.07415771484375, -0.1844482421875, 2.048828125, -1.0341796875, -1.65234375, 0.52685546875, -0.6904296875, 0.073974609375, -1.20703125, -0.4375, -0.91943359375, -0.71044921875, -0.1607666015625, -0.174072265625, 1.87890625, 0.9609375, 1.2451171875, 1.4267578125, 1.6796875, -1.7724609375, 0.72705078125, 0.0007047653198242188, 1.2724609375, 0.51953125, -2.3125, 1.517578125, -0.9091796875, 0.0116119384765625, -0.27197265625, -0.42041015625, -0.36572265625, 1.76953125, 0.6474609375, -0.544921875, 0.280517578125, 0.305908203125, 0.36962890625, 0.0643310546875, -0.85205078125, -0.2430419921875, -1.1884765625, 1.744140625, -0.65087890625, -1.0087890625, 0.0186309814453125, 0.76806640625, -1.146484375, -0.42041015625, -0.248779296875, 2.44921875, 1.0947265625, 0.75634765625, -1.9921875, 1.8056640625, -0.484375, -0.4248046875, 0.088623046875, -1.1474609375, -0.436767578125, 0.84228515625, 2.626953125, 0.058074951171875, -0.289794921875, -0.955078125, -0.2252197265625, -0.189208984375, 0.93798828125, 0.603515625, -0.7548828125, -0.383544921875, 0.050048828125, 1.197265625, 1.9267578125, -0.0249176025390625, -0.78662109375, -0.59716796875, -0.243896484375, 1.62109375, 2.396484375, 0.2052001953125, 1.619140625, 1.583984375, -0.92822265625, -0.5888671875, -0.95654296875, -0.314453125, -0.91357421875, 0.458984375, -1.2236328125, -0.439697265625, 0.434814453125, -1.57421875, 0.1429443359375, 0.8681640625, -0.36376953125, 0.06494140625, -0.7490234375, -0.329345703125, 1.2265625, -0.04425048828125, -1.615234375, -0.45068359375, -0.72802734375, 0.44482421875, -0.066650390625, -0.61181640625, 0.0308990478515625, 1.1591796875, -0.29345703125, 0.30078125, 0.317626953125, 1.2802734375, 0.15771484375, 0.3759765625, 1.00390625, 1.2353515625, -0.822265625, 1.623046875, 0.54638671875, -0.08209228515625, -0.07513427734375, 0.71484375, 0.400146484375, 0.90625, -2.26171875, -1.5380859375, 1.19140625, -0.047760009765625, 1.5224609375, -0.33935546875, 0.73681640625, 1.3828125, 1.5185546875, 0.25634765625, 0.26611328125, 0.43115234375, 1.6826171875, 0.456298828125, 0.0230712890625, -0.16845703125, -0.02642822265625, 0.0452880859375, 0.0167236328125, -1.4580078125, 0.40673828125, 1.306640625, 0.324462890625, -0.1773681640625, 0.630859375, -0.1607666015625, 1.134765625, 1.8232421875, -0.72998046875, 0.28515625, -0.466552734375, -1.064453125, -1.400390625, -0.4453125, 1.2275390625, 0.017822265625, 0.03369140625, 1.1025390625, -2.017578125, -1.107421875, 0.751953125, -0.763671875, -0.1505126953125, 0.57177734375, -0.359619140625, 0.74169921875, 1.6708984375, 0.9189453125, -0.65087890625, -0.249267578125, 0.33251953125, -0.59912109375, -2.234375, 0.32080078125, 0.347412109375, -0.1075439453125, -0.159912109375, 0.454833984375, 0.12335205078125, -0.402587890625, -0.67919921875, 0.64013671875, -0.12054443359375, -0.74755859375, 0.1949462890625, 0.326416015625, 0.5087890625, 0.1314697265625, 0.319091796875, -0.50341796875, 1.4296875, -0.343994140625, 0.73779296875, 0.501953125, 1.076171875, 1.9736328125, -0.5673828125, -0.9599609375, 0.1507568359375, -0.81591796875, -1.326171875, 1.337890625, 0.055877685546875, 0.434814453125, -0.416015625, -0.8623046875, -0.85498046875, 0.227294921875, 0.00791168212890625, -0.420166015625, -1.7373046875, 0.9326171875, -1.8544921875, -0.1571044921875, -0.35107421875, -1.18359375, 0.2685546875, -0.38671875, -0.1575927734375, 0.662109375, 0.0248565673828125, -0.279052734375, 0.2457275390625, 1.5205078125, -0.36279296875, 0.791015625, 2.91015625, -1.576171875, -0.72119140625, -0.828125, 0.61181640625, 1.181640625, -0.93896484375, -0.6806640625, -0.421142578125, 0.82861328125, 0.2236328125, -0.0296783447265625, -3.052734375, 0.65185546875, -1.212890625, -1.373046875, -0.7197265625, -0.58056640625, -0.312744140625, 1.0546875, 0.69775390625, 0.364013671875, 1.2275390625, 0.66650390625, 1.9677734375, -0.44970703125, -0.8671875, -0.295166015625, -0.72607421875, 1.666015625, 0.45458984375, 0.490234375, 1.64453125, 1.8828125, 1.2265625, -0.37353515625, 0.35986328125, -1.1240234375, 0.442626953125, -0.9521484375, 0.207763671875, 0.52734375, -3.048828125, -0.9501953125, -0.1766357421875, 1.5478515625, 0.94873046875, -0.37109375, 0.67626953125, 0.477783203125, 0.720703125, 0.08343505859375, -0.568359375, 1.9208984375, 0.98828125, 1.4814453125, 0.17333984375, 0.0168304443359375, 0.56640625, 0.767578125, 0.40966796875, -1.3740234375, 0.38720703125, -0.08074951171875, 0.10528564453125, -0.7451171875, -0.48828125, 0.302490234375, -0.1339111328125, 1.6513671875, -0.56103515625, -0.055084228515625, 0.07611083984375, 0.393798828125, 0.0294189453125, -0.1500244140625, 0.47216796875, -0.11566162109375, 1.1279296875, 2.36328125, 0.54931640625, -0.0140838623046875, 1.3427734375, 0.499755859375, -2.365234375, 2.607421875, 0.90087890625, -0.556640625, -1.38671875, -1.4375, 0.0850830078125, -0.80224609375, -0.5400390625, 0.18017578125, -0.1552734375, -0.5927734375, 0.8466796875, -1.037109375, -0.277099609375, -2.142578125, -1.076171875, 0.7412109375, -0.8134765625, 0.58740234375, 0.278076171875, 0.1527099609375, -1.01953125, -0.50732421875, -0.403076171875, 0.246826171875, -1.0537109375, -0.94921875, 1.0166015625, -0.393310546875, 0.07830810546875, 0.517578125, 0.35498046875, -0.9150390625, 0.1630859375, 1.8642578125, 1.5693359375, 0.296630859375, 1.07421875, 1.3837890625, -0.736328125, 0.4755859375, 0.2152099609375, 0.8232421875, 0.587890625, 0.408447265625, 0.255126953125, 0.56787109375, -0.1619873046875, -0.476806640625, 1.0107421875, -0.7802734375, -0.85400390625, -0.681640625, 0.3037109375, 0.0469970703125, 0.048431396484375, -0.454833984375, 0.09368896484375, 1.25, 0.285400390625, -0.3701171875, -1.125, 0.81787109375, 1.3291015625, 0.22802734375, 0.7548828125, -3.083984375, -1.1142578125, 1.072265625, 0.9599609375, 0.350341796875, -0.1605224609375, -0.3486328125, -0.5830078125, 2.8203125, -0.953125, -0.23681640625, 0.771484375, -0.71875, 0.59375, -0.0010852813720703125, 0.57421875, 1.7998046875, -0.09893798828125, 0.0576171875, -0.3173828125, -0.14111328125, -0.30029296875, -0.292724609375, -0.0859375, -0.154541015625, 0.2076416015625, -0.353271484375, 2.03515625, -0.1610107421875, 0.56201171875, -0.11517333984375, -0.04791259765625, 0.479248046875, -1.15625, 1.5556640625, 0.689453125, 0.051055908203125, 0.06781005859375, -1.4375, -0.8291015625, 0.44384765625, 1.3359375, -0.14306640625, -0.3544921875, -0.8818359375, 1.724609375, 1.4111328125, 0.148193359375, -1.994140625, -0.6474609375, -1.931640625, 0.027984619140625, -0.8251953125, 0.136474609375, 1.75, 2.0, 0.697265625, -0.810546875, 1.4091796875, 0.034454345703125, 1.0302734375, -0.300048828125, -0.150390625, 0.1783447265625, 0.1546630859375, -1.486328125, -1.591796875, -0.394287109375, -1.3603515625, 0.5107421875, 0.3603515625, -7.092952728271484e-06, 0.1552734375, 0.6298828125, 0.3388671875, 0.5166015625, -1.8544921875, -0.0567626953125, 0.60107421875, -0.414794921875, 1.3876953125, 0.58935546875, -0.018585205078125, 0.5732421875, -0.21533203125, -0.258544921875, -0.138671875, 1.1064453125, -0.32568359375, -1.7451171875, 0.240234375, 0.447509765625, -0.295166015625, 0.3203125, 0.414794921875, 1.314453125, -0.288818359375, 1.1064453125, -0.72021484375, -1.7900390625, -0.60302734375, 0.44580078125, -0.54736328125, -0.111083984375, -1.658203125, -0.47314453125, -0.91162109375, 0.67333984375, 0.5458984375, 0.0533447265625, -1.6826171875, -0.9345703125, 0.49462890625, 0.12646484375, -1.9638671875, -1.833984375, 1.07421875, 1.1357421875, -0.376953125, 0.90869140625, -2.419921875, -0.57568359375, -0.2822265625, 0.650390625, -0.2003173828125, -0.2188720703125, 0.290771484375, -1.5224609375, -0.39892578125, 0.8544921875, 2.3984375, -0.6259765625, -1.267578125, -0.73779296875, 0.322021484375, -0.81298828125, 0.630859375, 1.85546875, -0.1025390625, -1.23046875, -0.23046875, -1.646484375, -0.058319091796875, 0.07220458984375, 1.697265625, 0.69384765625, 0.732421875, -0.93115234375, -1.3984375, -1.19140625, -1.2763671875, 0.1331787109375, 0.16748046875, 0.79296875, -0.98828125, 1.0029296875, -1.318359375, -0.443115234375, 0.2166748046875, -0.450927734375, 0.9736328125, 1.87109375, 0.0040740966796875, -0.0298309326171875, -1.80078125, -0.384765625, -1.39453125, 1.3740234375, 0.0228271484375, 0.417724609375, 0.5732421875, 0.80908203125, -0.857421875, 0.58837890625, 1.3671875, -2.091796875, 0.67822265625, -1.1787109375, -0.2666015625, -1.9267578125, 1.2685546875, 1.166015625, 0.51318359375, -0.9345703125, -0.264404296875, 0.6171875, 0.81298828125, -0.2135009765625, -0.06072998046875, -0.47509765625, 0.044525146484375, 0.319580078125, 0.1396484375, -0.1387939453125, -0.343017578125, -0.11279296875, 0.40625, -1.935546875, 2.185546875, 1.8544921875, 2.0625, -0.449462890625, 0.0799560546875, -1.1875, -0.1959228515625, -0.71484375, 0.5400390625, -0.5029296875, -2.43359375, -0.423828125, -0.74609375, 1.6640625, 0.9091796875, -1.0068359375, -2.26953125, -0.08251953125, 0.330810546875, 1.9345703125, -2.671875, 0.354736328125, -1.0234375, -0.269775390625, -0.40576171875, -0.1695556640625, 0.93701171875, -0.10601806640625, 1.064453125, -0.287353515625, -0.12103271484375, 0.67529296875, 0.423583984375, 1.1611328125, -2.2109375, 1.0390625, 1.978515625, 0.1510009765625, -0.8193359375, -0.77294921875, 1.0244140625, -0.061492919921875, 0.6298828125, 1.751953125, -0.2666015625, 0.1728515625, 1.203125, -1.435546875, -1.0849609375, 0.5712890625, -1.1123046875, -0.60498046875, -2.45703125, -0.11663818359375, -0.669921875, 0.28955078125, -0.83154296875, 1.900390625, -0.34716796875, 0.2078857421875, -1.1171875, -0.93408203125, 1.353515625, 0.7626953125, -0.60009765625, -1.6337890625, 0.315185546875, -0.448486328125, -0.8095703125, 0.09234619140625, 1.0908203125, -0.93603515625, 0.9248046875, -0.08111572265625, 0.8955078125, -0.127685546875, 0.1866455078125, -0.47314453125, 0.3798828125, 1.2275390625, 0.293701171875, -1.548828125, 0.26611328125, 0.10015869140625, -0.1573486328125, 0.328369140625, -0.69189453125, -1.6748046875, -0.435791015625, 0.98974609375, 1.7021484375, -0.141845703125, 1.560546875, -0.3642578125, -0.7080078125, 0.1343994140625, -0.9794921875, 0.69287109375, 0.6669921875, 1.3505859375, -0.2822265625, 1.3837890625, 1.1279296875, 0.955078125, -2.703125, 0.87939453125, 0.17138671875, -1.1923828125, -0.070556640625, 0.16650390625, 0.5595703125, 1.0185546875, -0.320068359375, -0.08746337890625, -0.91259765625, 0.68359375, -0.36572265625, 0.40576171875, -1.962890625, -1.373046875, 1.5029296875, -1.142578125, -0.86767578125, -1.72265625, -2.09765625, 1.376953125, 0.22265625, -1.0322265625, 0.59130859375, -0.98046875, -0.662109375, -0.062225341796875, 1.1396484375, -1.091796875, 0.96484375, 0.74609375, 0.2265625, -0.34814453125, -0.1282958984375, 0.487548828125, 0.576171875, 0.279052734375, 0.1219482421875, 0.0491943359375, 1.1513671875, 0.03704833984375, 0.48681640625, 0.76708984375, 1.4013671875, -0.0462646484375, -0.724609375, -0.5146484375, -1.7138671875, 0.72412109375, -0.021575927734375, 0.9580078125, -1.8232421875, 0.5126953125, -0.7109375, -0.50341796875, -0.053924560546875, -1.0302734375, -0.85791015625, 1.1884765625, -0.38134765625, -0.09521484375, -1.9052734375, -1.908203125, 0.2071533203125, -0.38134765625, 0.7939453125, -0.58447265625, 1.6611328125, -1.568359375, -1.1806640625, -0.81689453125, 0.0911865234375, 1.1796875, 0.0887451171875, -0.1263427734375, 0.49560546875, -1.4208984375, -1.705078125, -1.2373046875, -1.810546875, 0.01849365234375, 0.57421875, 0.118408203125, 0.2000732421875, -0.96240234375, -0.35791015625, -0.47412109375, 0.181640625, 1.517578125, -0.1875, -1.7265625, 0.8544921875, -0.8798828125, -0.293701171875, 0.56298828125, 0.362548828125, 0.8271484375, -1.3203125, 0.5986328125, -1.5888671875, -1.1015625, 0.72216796875, 0.75439453125, 0.7080078125, -0.1671142578125, -2.185546875, 2.11328125, -0.58837890625, -0.31982421875, 0.61865234375, 0.849609375, 1.6650390625, 0.9111328125, -0.63525390625, -0.5595703125, 0.54248046875, 0.91015625, 1.2109375, -0.36083984375, 0.71533203125, -1.216796875, -0.88232421875, 0.0927734375, 1.423828125, 0.95361328125, -0.2010498046875, -0.1214599609375, 0.578125, -1.095703125, 0.6396484375, 2.1640625, -1.51953125, -0.7041015625, -0.09564208984375, -0.4111328125, -0.8623046875, 0.19873046875, -0.525390625, 0.88720703125, 1.0732421875, -0.466064453125, -0.3544921875, -0.94384765625, -1.1357421875, 0.3408203125, 0.84619140625, -0.56689453125, 1.53515625, 0.5126953125, 0.80322265625, 1.9267578125, -0.53515625, 0.51318359375, 0.923828125, -0.583984375, -1.0595703125, 0.376953125, 0.806640625, -0.12548828125, -0.59326171875, -1.2451171875, 0.76708984375, 0.1785888671875, -1.1279296875, -0.2386474609375, -1.5673828125, 1.2216796875, -0.346923828125, 0.2159423828125, -0.251708984375, 0.83544921875, -1.6279296875, -1.0048828125, -1.8203125, 1.9716796875, 0.62939453125, -1.0078125, 1.236328125, -1.041015625, 1.498046875, -1.8056640625, -0.212646484375, -2.0546875, -0.986328125, -0.177490234375, -0.5224609375, 1.8779296875, -0.41552734375, -2.240234375, -0.060516357421875, -0.96240234375, -0.9931640625, -0.57763671875, 2.05859375, -0.09405517578125, 0.01131439208984375, -0.791015625, 0.77685546875, -1.1767578125, 0.166748046875, -0.6318359375, -0.314697265625, -0.453857421875, -0.2132568359375, 1.0595703125, -0.6513671875, -2.390625, -1.8955078125, 1.03125, -1.65625, -0.40576171875, 0.498779296875, -0.184326171875, -0.40869140625, -1.3251953125, 0.68115234375, -0.478515625, 0.9345703125, -1.111328125, -0.255859375, 0.3798828125, -0.5849609375, -1.171875, 1.0712890625, -0.21044921875, -0.301025390625, 0.0684814453125, 1.6767578125, -1.109375, -1.0146484375, -1.3251953125, 0.2431640625, 1.646484375, -0.223876953125, -0.3486328125, 0.58154296875, -0.75732421875, 1.541015625, -0.466064453125, 0.76123046875, -0.98486328125, -0.4609375, 0.64697265625, -0.054534912109375, 1.7822265625, -0.802734375, -1.37890625, -0.16162109375, 0.68701171875, 2.107421875, -0.039031982421875, -0.73388671875, 1.0625, 0.55615234375, 0.5029296875, -1.2529296875, 0.28515625, 0.64892578125, 0.99755859375, 0.88671875, 0.1666259765625, -1.5810546875, 0.154296875, -1.4677734375, -0.472900390625, 0.2763671875, -1.7080078125, -0.1092529296875, -0.71728515625, -0.27587890625, 0.50146484375, 0.01751708984375, 0.072509765625, -0.0738525390625, -1.3056640625, -0.396728515625, 0.282958984375, 0.1732177734375, 0.64013671875, 0.27978515625, 0.62451171875, 1.1640625, -0.7099609375, 1.1259765625, 0.469482421875, -0.91357421875, 1.4580078125, -0.91064453125, -1.0869140625, 1.4580078125, 0.60693359375, -0.2091064453125, -0.62109375, -0.2509765625, -0.073486328125, 1.703125, -0.7119140625, -0.1741943359375, -0.3671875, 0.04962158203125, -0.9873046875, -0.56005859375, 0.07037353515625, -0.37109375, 0.689453125, -2.9921875, 1.3583984375, 0.3544921875, -0.798828125, 0.202392578125, 1.001953125, -0.199951171875, -1.986328125, -0.5263671875, -1.7197265625, -1.0458984375, -1.052734375, -0.7763671875, -0.638671875, -0.654296875, 0.1646728515625, -3.04296875, 0.40478515625, -1.2412109375, 0.4638671875, 0.9306640625, 0.379150390625, 2.107421875, -0.067626953125, 1.1181640625, -1.94921875, 1.6103515625, 1.1826171875, 0.95751953125, 0.62060546875, -0.44677734375, 0.441650390625, -0.0765380859375, 1.70703125, -0.365478515625, -1.5654296875, 2.033203125, -0.70751953125, -1.1708984375, 0.3828125, -0.84033203125, 0.69921875, 1.1923828125, 1.2216796875, -1.8251953125, 1.84765625, -0.5947265625, -1.08984375, 0.78515625, -0.58154296875, 1.5029296875, -1.39453125, 0.40869140625, 0.483154296875, 1.1083984375, -0.54443359375, -1.3623046875, 0.56640625, -0.88232421875, 2.146484375, -0.42431640625, -0.02935791015625, -1.1669921875, -0.8505859375, -0.0931396484375, -0.007152557373046875, -1.298828125, 0.2470703125, -1.763671875, -1.2080078125, -0.54052734375, 0.319580078125, 0.11700439453125, 0.5029296875, 0.98046875, -1.2109375, -0.6875, 1.939453125, 0.0229034423828125, 1.142578125, 0.282470703125, -0.998046875, 0.3798828125, -0.86279296875, -1.0693359375, -0.26953125, 0.8876953125, -1.7958984375, -0.5673828125, 1.3369140625, -0.0160980224609375, 0.654296875, -0.6328125, 0.75, -1.751953125, -1.41796875, 0.78076171875, 0.4345703125, 1.32421875, -0.92333984375, -0.41015625, -1.01171875, 1.5302734375, -2.14453125, -1.0234375, -0.007659912109375, 0.6513671875, -1.0107421875, -1.0888671875, -1.095703125, 0.125732421875, 0.69189453125, -1.0322265625, -1.439453125, -0.417236328125, -1.3310546875, -0.505859375, -0.90478515625, 0.54931640625, -0.62158203125, -1.58984375, -0.301513671875, -0.427490234375, 0.0421142578125, -1.1396484375, 1.791015625, -0.44189453125, 0.6953125, 0.04571533203125, -1.322265625, -0.80810546875, -0.9697265625, -0.399169921875, -2.212890625, 1.3740234375, -0.8701171875, 2.77734375, -0.07562255859375, 0.33154296875, -0.634765625, -0.384033203125, -0.7607421875, 0.9091796875, 1.13671875, -1.2841796875, 0.9912109375, 0.77294921875, 1.6279296875, -0.525390625, 0.06414794921875, -0.76123046875, 0.057586669921875, -0.1866455078125, 1.583984375, -2.900390625, -2.2265625, 0.0987548828125, -0.5107421875, 0.658203125, 1.990234375, 0.75634765625, 0.85546875, 0.50830078125, -2.716796875, -0.5205078125, 0.1888427734375, -1.68359375, -0.083984375, -0.1981201171875, 0.405029296875, -0.29296875, 0.39208984375, 1.0048828125, -2.958984375, 0.313232421875, -0.2232666015625, -1.0458984375, 0.98388671875, -1.6904296875, 1.296875, 0.67138671875, 0.53515625, 0.5048828125, 0.2308349609375, -0.0963134765625, 0.0225067138671875, -0.1324462890625, 1.0205078125, -0.2056884765625, 0.11505126953125, -0.5830078125, -0.8818359375, 0.61181640625, 0.2318115234375, -0.849609375, 0.1790771484375, -1.3564453125, -0.97265625, -0.88037109375, -0.5556640625, 0.032012939453125, 0.164794921875, -0.36279296875, 0.55712890625, 0.3203125, -1.3173828125, 1.84765625, -0.9814453125, -0.63720703125, -0.62353515625, -0.312255859375, 0.119384765625, -0.2244873046875, 0.1356201171875, -0.1221923828125, 0.46240234375, 0.310302734375, 0.11846923828125, -1.326171875, 0.5234375, 0.1927490234375, 0.08770751953125, 0.37158203125, 0.316162109375, -0.6357421875, 0.347900390625, 0.53125, -0.1605224609375, -0.341796875, 0.318115234375, 0.732421875, -0.63916015625, 3.005859375, 1.3876953125, 0.4765625, -0.81103515625, 0.87158203125, -0.65673828125, 1.265625, 0.09149169921875, -1.2275390625, 0.3427734375, -1.083984375, -0.062103271484375, 0.14990234375, 0.31494140625, -0.2587890625, 0.33984375, -0.58837890625, -0.54541015625, 0.1441650390625, 0.12176513671875, 0.17236328125, -1.462890625, 0.68115234375, -0.06988525390625, 0.9443359375, -0.80224609375, 1.27734375, -0.876953125, 0.50146484375, 1.228515625, -0.53515625, -1.23046875, -1.2861328125, -0.71533203125, -0.7177734375, -1.208984375, 1.6416015625, -0.923828125, -0.9052734375, 1.2158203125, -1.1904296875, -0.78271484375, 0.669921875, -0.53076171875, 0.8369140625, 0.321044921875, -0.29345703125, 0.5732421875, -1.5546875, 0.449462890625, 0.9951171875, 0.8408203125, -0.9833984375, -0.8095703125, 2.22265625, 2.40234375, -0.509765625, -1.447265625, -1.1552734375, -0.81884765625, 0.38330078125, 0.65087890625, 1.5078125, -1.2490234375, 0.442626953125, 0.89306640625, 0.767578125, 1.134765625, -2.279296875, 0.0138092041015625, -0.0242767333984375, -0.439208984375, 1.2333984375, -0.1805419921875, -0.03961181640625, -0.55810546875, 0.66162109375, 0.1265869140625, -0.91455078125, 0.76220703125, -0.86474609375, -0.1614990234375, -1.3720703125, -1.12890625, 0.426025390625, -1.3505859375, -1.0380859375, -1.1142578125, -0.69287109375, -0.63427734375, -0.439453125, -1.3515625, 0.86962890625, -0.4755859375, 0.486328125, 1.5751953125, 2.125, -0.7880859375, 0.39111328125, -1.23046875, 1.0146484375, -0.037109375, -0.032745361328125, -0.0701904296875, -0.095703125, -1.0703125, -1.51171875, 0.2227783203125, 1.26953125, 0.7578125, 2.5546875, 0.239501953125, 1.0205078125, -0.7294921875, 0.1385498046875, 0.163330078125, -0.80517578125, 0.86083984375, 0.185546875, -1.212890625, -0.57763671875, 0.395263671875, 0.42041015625, 0.7060546875, 0.890625, 1.6044921875, -0.8125, -0.2344970703125, -0.59619140625, 0.595703125, -0.078369140625, -1.0546875, -0.7412109375, 0.30322265625, -0.42529296875, 0.69580078125, 0.94189453125, -0.418701171875, 0.6982421875, 0.301513671875, -1.205078125, 0.3525390625, -0.6318359375, 0.4677734375, -1.47265625, 0.80078125, -0.33203125, 0.264404296875, 1.5166015625, 0.03509521484375, -0.1553955078125, 0.28759765625, -1.1748046875, -0.8046875, 0.6591796875, -0.58837890625, -0.0299530029296875, 0.55126953125, -1.2939453125, 1.1357421875, 0.8427734375, -0.31591796875, 0.03265380859375, 2.9453125, 0.2430419921875, 0.66552734375, -0.3291015625, 1.734375, 0.66357421875, -1.1474609375, -0.1295166015625, -0.374267578125, -0.1385498046875, 0.03131103515625, 0.6552734375, -0.6650390625, -0.56005859375, -0.666015625, -0.85791015625, -0.78857421875, 0.45703125, -0.2113037109375, 0.69580078125, -0.69921875, 0.71044921875, -0.6083984375, -1.1328125, 0.8310546875, -1.9951171875, -0.0258941650390625, -0.9658203125, 0.4765625, 1.017578125, -0.84619140625, -0.74560546875, -1.1845703125, 0.90625, 1.4130859375, 0.26611328125, -0.40625, 0.9169921875, 1.7333984375, 0.43798828125, 1.126953125, 0.73583984375, 0.056243896484375, -0.90673828125, 1.41796875, -0.68408203125, 0.0142364501953125, 0.158203125, 0.908203125, 0.170166015625, -0.7109375, 0.1163330078125, 1.3056640625, 0.01499176025390625, 1.150390625, 1.46484375, 0.53076171875, 0.123779296875, -0.59326171875, 0.470703125, 0.7080078125, 1.5185546875, -0.0275726318359375, 0.44677734375, 0.51708984375, -0.08087158203125, -0.273193359375, 1.3359375, -0.446044921875, 0.50732421875, -0.79296875, 0.0654296875, -0.2197265625, 1.2607421875, 0.09967041015625, 1.001953125, 1.349609375, 0.035308837890625, 1.337890625, -2.193359375, -1.3349609375, -0.810546875, -0.1064453125, 1.46484375, -0.1026611328125, 0.041412353515625, -1.765625, -1.048828125, -0.49267578125, 1.0361328125, -1.0400390625, -1.203125, 0.892578125, 0.2379150390625, -0.144287109375, -0.95947265625, -0.693359375, 1.142578125, 0.0322265625, 0.79931640625, 2.01953125, -1.158203125, -1.3955078125, 0.1807861328125, -0.08502197265625, -0.0050201416015625, 0.5791015625, -0.798828125, 2.24609375, -0.08746337890625, -1.7626953125, -1.103515625, 1.1181640625, -0.66943359375, -0.432861328125, -1.0234375, 0.053009033203125, 1.5869140625, 0.59716796875, 0.12347412109375, 0.8388671875, -1.0458984375, 1.5263671875, 0.06976318359375, -0.943359375, 1.560546875, 0.91796875, -1.6845703125, 1.9892578125, -0.1893310546875, 2.23046875, 0.89404296875, 0.299560546875, 0.2177734375, 2.181640625, 0.76708984375, -0.381103515625, -0.51953125, 0.0697021484375, 0.50048828125, -1.5390625, -0.75390625, -0.79931640625, 1.0595703125, 1.03125, -0.9580078125, -0.486328125, -0.0153656005859375, -1.1591796875, -0.2020263671875, -1.1796875, -0.343505859375, -1.810546875, -1.4716796875, -0.328857421875, -0.214599609375, -1.3427734375, -1.2177734375, -0.60546875, 1.771484375, -0.119873046875, -1.599609375, -0.68505859375, 0.1771240234375, 1.052734375, -0.59912109375, 0.221923828125, -0.59130859375, -0.1204833984375, 0.994140625, 0.5927734375, 1.849609375, 1.1220703125, -1.6455078125, 0.464599609375, 0.255126953125, 0.048065185546875, 1.0107421875, 0.09625244140625, 0.62255859375, -1.474609375, 0.307861328125, -0.85009765625, -0.19384765625, 0.496337890625, -0.08416748046875, -1.19140625, -1.1376953125, 0.427490234375, -1.658203125, 1.236328125, -0.8232421875, -1.2509765625, 0.95458984375, -0.56640625, -0.1376953125, 0.446533203125, 0.03863525390625, 0.28125, 1.1044921875, -1.58203125, -0.533203125, -0.85546875, -0.1383056640625, -1.6181640625, 0.07000732421875, 0.29296875, 0.2203369140625, 1.060546875, -1.767578125, -0.167236328125, 1.07421875, 0.174072265625, -1.10546875, -1.2822265625, 1.1123046875, 0.46630859375, 1.7216796875, 1.3095703125, 0.283203125, 2.080078125, 0.53173828125, -0.8134765625, -0.40625, -0.90771484375, 0.311767578125, 0.1622314453125, -0.95556640625, 0.00847625732421875, 1.55078125, 0.5966796875, -0.425048828125, 0.067626953125, -1.6865234375, 0.29052734375, -0.67919921875, 0.2607421875, -1.8369140625, -0.4638671875, 0.093994140625, 0.7509765625, 1.2724609375, 0.305908203125, 0.055023193359375, 0.30615234375, -0.309326171875, -1.3212890625, 0.188232421875, 1.49609375, 0.351318359375, 0.2298583984375, -0.830078125, -0.41259765625, 1.056640625, -1.3994140625, 1.0751953125, 1.2177734375, 1.3525390625, 1.4404296875, 0.09954833984375, -0.58935546875, -0.48046875, -1.0, 1.3515625, 0.375, 1.5498046875, -0.12249755859375, -0.306396484375, -0.92626953125, 0.041656494140625, -0.1900634765625, 0.249755859375, 0.69384765625, 0.2484130859375, -0.22509765625, 0.17431640625, -0.86181640625, 0.90625, 0.2242431640625, 1.7646484375, 0.52294921875, -1.65234375, 1.6455078125, 0.0293731689453125, 0.060638427734375, 1.4306640625, -0.43212890625, -0.3115234375, 0.72412109375, -0.02362060546875, -0.3154296875, -0.044281005859375, 1.3896484375, 0.498046875, -0.17626953125, -1.4873046875, 0.1856689453125, 0.296630859375, -0.59619140625, -0.8515625, -1.0029296875, 0.147705078125, 1.1201171875, 0.6220703125, 0.1502685546875, -2.802734375, 0.6494140625, -1.53125, -1.591796875, 0.087158203125, 0.473388671875, -1.1943359375, -0.591796875, -0.5029296875, -0.93359375, 0.1905517578125, -0.87548828125, -0.87353515625, -0.25634765625, -0.305908203125, -0.269287109375, -0.4931640625, 0.405517578125, -1.4453125, 0.7939453125, -0.96435546875, -0.73681640625, 1.8857421875, -0.291015625, -0.59814453125, -0.9345703125, 1.0556640625, 1.0625, -0.72607421875, 0.2196044921875, -0.1317138671875, -0.3955078125, -0.2296142578125, 1.3193359375, -1.806640625, -0.607421875, -0.04962158203125, 1.1025390625, -0.9921875, 0.09747314453125, -0.70947265625, 0.480224609375, -0.1021728515625, 0.261474609375, 0.140869140625, 1.8212890625, -1.1376953125, -0.0714111328125, 1.205078125, 0.10137939453125, -0.68212890625, -0.75390625, 0.497802734375, 1.1796875, 0.05078125, -0.5068359375, 0.78076171875, 1.1328125, -0.0714111328125, 2.703125, -0.137451171875, 0.80419921875, 0.67333984375, 0.4580078125, 0.94482421875, -1.197265625, -1.189453125, 2.388671875, 0.63818359375, 0.194580078125, 0.35009765625, -0.9365234375, 0.3154296875, 1.2314453125, 0.59228515625, -0.74853515625, 1.05859375, -1.0087890625, 0.385986328125, -0.433837890625, 0.259033203125, -0.72900390625, -1.0576171875, -0.2149658203125, 0.064697265625, 0.69873046875, -1.3681640625, 0.1485595703125, -1.166015625, -0.09033203125, 0.55517578125, 0.444091796875, -0.07867431640625, 0.038970947265625, -0.60595703125, 1.9423828125, 0.86767578125, 0.984375, 1.146484375, 0.40185546875, -0.0025959014892578125, 0.2149658203125, 0.363525390625, -0.8232421875, 0.8330078125, -0.83056640625, -0.87158203125, 0.662109375, 1.5107421875, -0.8818359375, 0.332763671875, 0.08880615234375, -0.1434326171875, 1.3076171875, -0.437255859375, 0.390625, 0.646484375, -0.6357421875, -0.430908203125, -0.11962890625, 1.7939453125, 1.0908203125, -0.296630859375, -0.8271484375, 0.4501953125, 0.12158203125, -0.212158203125, -1.9111328125, 0.81494140625, 1.2646484375, 1.0986328125, 0.638671875, -0.60986328125, -0.80810546875, -1.0166015625, -0.6298828125, -0.28369140625, 1.7080078125, -1.7294921875, 0.1387939453125, -0.6962890625, 1.765625, 0.60546875, 1.216796875, -1.4287109375, -0.60205078125, -0.1524658203125, 1.12109375, -0.149169921875, 0.546875, 0.72412109375, 0.1995849609375, 0.81982421875, -1.1357421875, 1.3671875, 0.381103515625, -0.120849609375, 0.09130859375, 0.1104736328125, -0.6796875, -0.62939453125, 0.4892578125, -1.2333984375, -0.268310546875, 2.26171875, -0.58642578125, 0.46240234375, 0.04425048828125, -1.33203125, 1.4248046875, -0.66943359375, -1.236328125, 0.2685546875, -2.154296875, 2.087890625, 0.2469482421875, 1.890625, -0.57763671875, 0.230712890625, -0.755859375, 0.5751953125, 0.67822265625, -0.98046875, 0.73876953125, -0.5185546875, 0.10614013671875, 0.249755859375, -0.6865234375, -0.1644287109375, 0.306884765625, -0.595703125, -1.51171875, 1.193359375, -0.36962890625, 0.241455078125, -0.457763671875, 1.5771484375, 0.007171630859375, -1.0615234375, 0.453369140625, 1.3408203125, 0.2418212890625, 0.8525390625, 0.09295654296875, -1.876953125, -0.51220703125, -0.052459716796875, -0.62939453125, 0.23828125, -0.85205078125, 0.86962890625, 0.0650634765625, -0.443115234375, 1.0927734375, -0.83740234375, 0.380126953125, -0.57373046875, 0.006565093994140625, 0.25830078125, 0.0751953125, 1.05078125, -0.1724853515625, -1.8115234375, 1.2568359375, 0.381591796875, -2.197265625, 1.2568359375, 0.4765625, -0.5341796875, 0.40087890625, -0.67919921875, -0.292236328125, 0.85986328125, 0.78955078125, 0.69189453125, -0.5263671875, 0.375732421875, -1.2958984375, -1.8798828125, 1.1357421875, 0.476806640625, 2.82421875, -0.2236328125, -0.42578125, 1.583984375, -0.75927734375, 0.465576171875, 1.8095703125, 0.1903076171875, -0.400634765625, -0.359375, -0.286376953125, -1.833984375, 1.27734375, 0.79443359375, -2.37109375, -0.6650390625, -1.3828125, -0.0003192424774169922, 0.11273193359375, -0.78173828125, -2.767578125, 0.7431640625, 1.2744140625, 0.388427734375, 0.57421875, 0.1785888671875, 1.388671875, 0.29150390625, -0.5322265625, 0.478515625, -0.299560546875, 0.546875, 0.708984375, -0.75341796875, -1.5966796875, 0.7783203125, 1.283203125, -0.464599609375, 0.26318359375, 1.111328125, -0.95068359375, 1.2958984375, 1.1376953125, -0.56591796875, -1.5517578125, 1.3203125, 1.3955078125, -0.93212890625, -0.29443359375, -1.693359375, 1.23828125, -2.5, 0.7041015625, -0.88037109375, 0.037750244140625, -1.0634765625, -0.01451873779296875, -0.7021484375, -0.2340087890625, -0.460693359375, -0.8408203125, -0.25732421875, 0.96435546875, -0.2325439453125, -0.429931640625, -0.6142578125, 1.2392578125, 0.826171875, -1.8330078125, 0.861328125, 0.9384765625, 0.28759765625, 1.4775390625, -0.52197265625, 0.2276611328125, 1.4560546875, -0.431884765625, 0.7626953125, 0.611328125, -0.9765625, -0.1856689453125, 1.638671875, -0.39111328125, 0.83837890625, 0.9423828125, -0.9541015625, -0.751953125, -2.783203125, -0.6650390625, -0.300048828125, -1.265625, -1.2587890625, 0.576171875, -0.4765625, 0.378662109375, -0.49853515625, 1.6728515625, -0.86962890625, 0.90869140625, 0.016937255859375, 0.50830078125, 0.040496826171875, -0.88330078125, 1.17578125, -0.254150390625, 0.355224609375, 1.0654296875, 0.27490234375, -0.2044677734375, 0.2403564453125, 2.576171875, 0.256103515625, -0.732421875, -0.234375, 0.587890625, -0.1221923828125, 0.89306640625, -1.369140625, -1.2158203125, -2.23046875, 1.388671875, 0.2430419921875, 0.53125, -0.6455078125, 0.35400390625, -1.046875, -0.68310546875, -2.3671875, 1.8115234375, 0.478515625, 0.0745849609375, -0.0193634033203125, -1.5947265625, 1.2216796875, 0.350341796875, 0.050384521484375, -0.27587890625, 0.7509765625, -0.9853515625, -0.5185546875, -0.89501953125, -0.796875, 1.5673828125, -0.2135009765625, -0.69873046875, -0.053741455078125, -0.5234375, 0.74072265625, 1.4619140625, -1.419921875, 0.64306640625, 0.87939453125, -0.27783203125, -0.439453125, -0.10888671875, 0.84130859375, -0.8203125, -1.2138671875, 0.423583984375, 0.053619384765625, 0.130615234375, 1.134765625, -0.136962890625, 1.4296875, 0.55029296875, -0.275634765625, -0.5517578125, -1.7373046875, -0.810546875, 2.44921875, -2.673828125, -0.464111328125, 0.6103515625, -0.1697998046875, -0.0926513671875, -1.3154296875, 1.1591796875, -0.159912109375, 0.6787109375, 0.7294921875, -1.7685546875, 0.0146636962890625, -0.81494140625, -0.439697265625, 0.63427734375, 0.44482421875, -0.36376953125, 0.41455078125, 0.59326171875, 1.388671875, 0.35595703125, -1.158203125, 0.350341796875, -0.407958984375, -0.323974609375, 0.603515625, -0.83544921875, 0.34228515625, 0.255126953125, -0.3828125, 0.59033203125, -0.384521484375, -0.1375732421875, 0.30517578125, -1.57421875, -0.451171875, -0.53369140625, 1.34765625, -1.3740234375, 0.280029296875, -0.9951171875, 0.6953125, 0.9892578125, -0.31298828125, 0.06195068359375, -0.474365234375, -0.12310791015625, -1.39453125, -0.50390625, -0.0501708984375, 1.275390625, 0.53173828125, -0.80517578125, 0.2252197265625, 0.20947265625, -1.451171875, 0.2178955078125, 0.0207366943359375, 0.73974609375, 2.623046875, -1.7470703125, 0.49609375, -0.06707763671875, -0.46533203125, 0.60791015625, -0.85791015625, 1.4033203125, -0.451416015625, 1.4365234375, -0.17822265625, 0.87548828125, -0.95751953125, 0.22119140625, -0.21044921875, 0.42431640625, 0.6640625, -0.85400390625, -0.37646484375, 0.06182861328125, 0.7080078125, 1.583984375, -2.861328125, -2.34765625, 0.78466796875, 1.107421875, 0.8037109375, 1.484375, 0.89892578125, -2.7890625, -0.298828125, -0.701171875, 1.7373046875, -0.130126953125, -0.953125, 1.4130859375, -1.2734375, 0.24462890625, -0.79443359375, 0.01409912109375, 0.56298828125, -0.337158203125, 1.0732421875, 1.75, -0.100341796875, 0.0970458984375, 0.2291259765625, 0.489013671875, -0.7060546875, -0.09765625, -0.023101806640625, -0.353759765625, 1.2431640625, -1.291015625, 0.52734375, 1.806640625, 1.43359375, 0.6103515625, -1.052734375, 1.03515625, -1.36328125, 0.453125, 0.86474609375, -0.03466796875, -0.16650390625, -0.96240234375, -0.06927490234375, -0.50830078125, 0.9765625, -2.1328125, 0.46630859375, -1.2294921875, 0.2440185546875, -0.42724609375, 0.442626953125, -1.7587890625, 1.6748046875, -0.370849609375, -0.09332275390625, -2.666015625, -0.154052734375, -0.471923828125, 1.5068359375, 0.603515625, 1.5947265625, -0.6796875, -1.1474609375, -0.7177734375, -0.458251953125, -1.3017578125, -0.06146240234375, -1.78515625, 0.263427734375, 0.2305908203125, -0.61572265625, -2.11328125, -2.19140625, 0.2164306640625, 0.77783203125, 1.068359375, -1.3095703125, 1.5400390625, 0.62890625, 0.869140625, 0.06915283203125, -1.08203125, -0.2303466796875, 0.048828125, 0.0200042724609375, -0.361328125, 2.15625, -1.580078125, -0.8701171875, -1.6884765625, -0.96875, 0.880859375, -0.94287109375, 1.6025390625, 1.080078125, -0.154541015625, 0.33984375, 1.26171875, -0.193359375, -1.2080078125, -0.388916015625, -0.77783203125, -1.0078125, 0.501953125, 0.302978515625, -2.78515625, 0.6181640625, -0.31787109375, 0.6298828125, 0.53515625, 0.69775390625, -0.4345703125, 0.994140625, -1.728515625, -0.6064453125, -0.59130859375, -0.0254974365234375, -0.759765625, -0.73681640625, -0.7802734375, 1.4560546875, 0.03179931640625, 0.49072265625, 0.5966796875, -1.677734375, -1.2392578125, -0.48291015625, 0.02325439453125, -1.0771484375, 0.716796875, 0.292236328125, -0.689453125, -0.389404296875, -0.11004638671875, -0.2099609375, -0.2457275390625, 0.37353515625, 1.5224609375, -0.19677734375, 0.09375, -1.3134765625, 0.2919921875, -1.6123046875, -0.53564453125, -0.38427734375, -1.2900390625, 0.55615234375, 0.79296875, 1.6484375, -0.8828125, -1.0361328125, 1.328125, 1.5546875, 0.7802734375, -0.97802734375, 0.327392578125, -1.111328125, -1.314453125, -0.61767578125, -0.56103515625, 0.289306640625, -0.25634765625, -0.3134765625, 0.70947265625, 0.24267578125, -2.146484375, -0.42431640625, 0.28271484375, 0.344482421875, 0.03009033203125, -0.034027099609375, -0.21630859375, 2.703125, -0.76513671875, 0.108642578125, -0.322265625, -1.919921875, -0.1551513671875, 0.916015625, -0.484619140625, 1.513671875, 0.3349609375, 1.2607421875, 0.33251953125, -1.232421875, 1.0419921875, -0.974609375, -1.5048828125, 1.76953125, -0.281005859375, 0.32958984375, 1.2802734375, -0.412841796875, -0.410400390625, -0.9951171875, 1.11328125, 0.96337890625, -2.13671875, -0.17236328125, -0.08038330078125, -0.0802001953125, 0.11279296875, -0.5791015625, -1.8310546875, -1.9638671875, -1.94921875, -0.461181640625, -1.125, 0.181396484375, -0.40087890625, -0.61376953125, -0.36572265625, 0.0902099609375, 0.173583984375, 2.189453125, 1.046875, -0.5087890625, -0.58056640625, 0.51806640625, 0.1712646484375, -0.30810546875, 0.08648681640625, 0.70947265625, 0.211669921875, -1.1201171875, 1.150390625, -0.0697021484375, 0.2161865234375, -0.9345703125, -0.371826171875, -0.1099853515625, -0.5546875, 0.208740234375, -0.720703125, 1.001953125, -1.1689453125, 0.03729248046875, -0.6416015625, -0.54638671875, 0.9697265625, 0.076416015625, -0.438720703125, 0.27392578125, 0.34326171875, -0.261474609375, -0.54833984375, -0.154296875, -0.262451171875, -0.224853515625, 1.4169921875, -0.260986328125, 0.1143798828125, -0.34619140625, 1.041015625, 0.45458984375, -1.34765625, -0.79638671875, 0.9794921875, 0.264892578125, -0.99169921875, -0.331787109375, -0.260498046875, 1.67578125, 0.19677734375, -0.272705078125, -0.0303955078125, 0.052947998046875, -0.12078857421875, -1.7119140625, -1.2744140625, -0.017852783203125, -0.51904296875, 1.767578125, 0.412353515625, -0.96875, 1.146484375, 0.3798828125, 0.155029296875, 0.39794921875, -1.986328125, 1.06640625, 1.75, 2.279296875, 0.7890625, 1.734375, 0.37353515625, -1.287109375, 3.041015625, 0.116455078125, 0.1610107421875, -0.52978515625, -0.84912109375, -1.5166015625, -0.11175537109375, -0.004985809326171875, 1.32421875, -0.86279296875, -2.201171875, -0.184814453125, -0.193115234375, -0.1475830078125, -0.374267578125, 1.919921875, -1.3203125, 0.12115478515625, 0.1148681640625, 0.09991455078125, 0.6357421875, -1.2744140625, -0.2008056640625, -1.16015625, -0.1197509765625, 0.034210205078125, -0.1722412109375, 0.5029296875, 0.216552734375, 0.156494140625, -1.8486328125, -0.7734375, -0.013519287109375, -1.6962890625, -0.3076171875, 0.2017822265625, 0.900390625, 0.5185546875, -0.04754638671875, 0.042205810546875, -1.7041015625, 0.2144775390625, 0.33349609375, 1.4775390625, -1.3271484375, 0.30322265625, 0.7412109375, 1.2744140625, -2.322265625, -0.619140625, -0.64208984375, -0.61669921875, 2.2734375, -0.47998046875, 1.01953125, 1.537109375, 0.411376953125, -0.61376953125, -1.177734375, 1.369140625, 2.96875, 0.123046875, -0.9716796875, 0.60009765625, -0.2437744140625, 0.828125, -0.70458984375, -1.5966796875, 0.10528564453125, -0.2152099609375, -0.376708984375, -0.67333984375, -0.11083984375, 0.99267578125, -1.154296875, 0.2919921875, 0.931640625, -0.59521484375, 0.2412109375, 2.455078125, -0.99609375, 0.31884765625, -0.252197265625, 2.314453125, 2.03125, 0.81884765625, 0.82763671875, -0.2218017578125, -2.14453125, -0.390869140625, 0.019317626953125, 1.2568359375, 0.189453125, -0.65087890625, 0.336669921875, -0.33544921875, -1.8466796875, -0.849609375, -1.318359375, 0.69091796875, -1.0498046875, 0.2376708984375, -0.0797119140625, 1.5205078125, 0.353759765625, 0.19384765625, 0.12200927734375, -0.152587890625, 0.5888671875, 0.75634765625, 0.19140625, 0.2100830078125, 0.2578125, -1.0244140625, 0.681640625, 0.276123046875, -0.2049560546875, 0.74560546875, -0.56103515625, -0.09356689453125, 0.10784912109375, -1.005859375, -1.2890625, 0.142822265625, 0.634765625, -0.46728515625, 0.61279296875, 0.54052734375, 1.392578125, 0.47607421875, 0.06488037109375, -0.73291015625, -0.724609375, 0.497314453125, 0.62158203125, 0.2052001953125, -0.37158203125, 1.5478515625, -0.69287109375, 1.1728515625, -0.339599609375, 1.4013671875, 0.153564453125, -0.96630859375, -1.626953125, -1.0390625, -0.08306884765625, 2.689453125, 1.1240234375, 0.9150390625, -0.54052734375, 0.900390625, 0.16064453125, 0.290771484375, -1.173828125, 0.931640625, -0.9267578125, -0.30810546875, -0.974609375, -0.82763671875, -0.76318359375, -0.7900390625, 0.447021484375, 1.259765625, 0.1517333984375, 0.50146484375, 0.61962890625, -0.55859375, -1.732421875, -1.2587890625, 1.6865234375, -0.10809326171875, -0.76513671875, -0.61962890625, 0.85595703125, 0.8447265625, -0.1268310546875, 0.233154296875, -1.0, -0.2064208984375, -0.061004638671875, 0.65625, -0.54296875, -0.303466796875, 0.64794921875, -0.74755859375, 0.33642578125, -0.02239990234375, 0.345703125, -0.900390625, -0.51171875, 2.220703125, 1.650390625, -0.05584716796875, 1.095703125, -1.107421875, 1.287109375, 0.5693359375, -1.5048828125, -0.65869140625, 0.465087890625, 0.76953125, 1.3232421875, 0.3564453125, -0.009368896484375, -1.060546875, 0.07745361328125, 1.6572265625, -1.560546875, 1.21484375, -0.468994140625, 0.908203125, -0.436767578125, 1.654296875, 0.44287109375, -0.2568359375, 1.2470703125, -2.29296875, 0.007762908935546875, -1.119140625, 0.6484375, 1.0751953125, 0.44384765625, 0.333984375, 0.10650634765625, -1.2724609375, 2.60546875, -0.615234375, 1.51953125, -0.70361328125, 1.015625, -0.5068359375, -1.3662109375, -0.350341796875, -1.1123046875, -0.30224609375, -1.3173828125, -0.1544189453125, 0.302001953125, 2.181640625, 0.9208984375, -0.7763671875, 1.1435546875, -0.1285400390625, -1.3544921875, -0.5078125, -0.56298828125, 0.5068359375, -0.5966796875, 0.276123046875, 1.556640625, 0.1142578125, 2.087890625, -0.28857421875, 0.1260986328125, -0.861328125, -1.349609375, -1.375, -0.271728515625, 0.0908203125, -0.4091796875, 1.6171875, -1.7099609375, 0.0044097900390625, -0.1348876953125, 1.1923828125, -1.5380859375, -0.4033203125, -1.4765625, 0.50537109375, 1.9697265625, 0.38671875, 0.5263671875, -1.2451171875, -1.259765625, 0.78564453125, -0.250244140625, -0.28076171875, -0.48779296875, -0.50341796875, 0.66064453125, -0.60791015625, 0.26513671875, 0.1297607421875, -0.316162109375, -0.7060546875, -0.1759033203125, -0.7490234375, 1.0244140625, -0.1485595703125, 0.4384765625, 0.673828125, 0.06134033203125, 0.67626953125, -0.8515625, 0.94921875, -0.40673828125, 0.360595703125, -0.67138671875, 0.91845703125, 0.96435546875, 0.5009765625, -0.0845947265625, 1.69140625, -0.794921875, -0.267822265625, 2.037109375, 0.21435546875, -0.005146026611328125, 0.4814453125, 0.66357421875, -2.521484375, -0.669921875, 0.818359375, -1.271484375, 0.205322265625, 0.0738525390625, 0.5693359375, -0.13037109375, -0.0882568359375, 2.076171875, -0.283935546875, -0.798828125, -0.02410888671875, 0.78271484375, 0.1300048828125, 0.41357421875, 1.71875, -0.52978515625, -1.36328125, -0.10418701171875, -0.7099609375, 1.7724609375, -0.6005859375, -0.049407958984375, 1.2802734375, 0.17138671875, -0.6083984375, -0.2017822265625, 0.471923828125, -0.2705078125, 0.70849609375, -0.1910400390625, 0.3125, -0.73193359375, -0.52294921875, -2.4140625, 0.21435546875, 1.2021484375, 0.7607421875, -0.314208984375, 1.458984375, 1.130859375, 0.07525634765625, -0.36962890625, -1.7001953125, -2.580078125, -0.87451171875, -0.81201171875, 0.98046875, 0.030731201171875, 1.8447265625, -0.87353515625, -1.259765625, -0.148193359375, 0.953125, 0.044097900390625, -0.9169921875, 1.1923828125, -1.341796875, 0.84228515625, 0.17431640625, -0.46142578125, -1.2744140625, -0.06781005859375, -0.463134765625, -0.24462890625, 1.3349609375, 0.489990234375, -1.255859375, 1.220703125, 1.177734375, -1.494140625, -0.4091796875, 3.236328125, 0.157470703125, -1.45703125, 0.765625, -0.31396484375, -0.222900390625, -1.1474609375, 0.2344970703125, 0.70654296875, -0.4375, 0.72998046875, -1.1162109375, 0.311279296875, 0.90185546875, -0.7890625, -1.0546875, -0.8740234375, 0.68310546875, 0.1002197265625, 0.658203125, -1.3037109375, -0.232177734375, -1.0712890625, -0.38623046875, -0.11676025390625, 0.970703125, 1.2802734375, 0.6396484375, -0.74951171875, -0.391357421875, -0.7783203125, -1.1640625, 0.10205078125, 1.0107421875, 0.476318359375, 0.2100830078125, -1.3046875, 0.06390380859375, -1.5673828125, -1.087890625, 0.994140625, -0.564453125, -0.276611328125, -0.49365234375, 0.06829833984375, 0.420654296875, 0.169921875, -0.43896484375, 0.021514892578125, -0.149169921875, -0.8798828125, -1.009765625, 0.468994140625, -0.68603515625, 0.082763671875, 0.291259765625, 0.027679443359375, 1.9443359375, -2.439453125, -0.62744140625, 0.5556640625, -0.71435546875, 0.1895751953125, 0.87646484375, 0.763671875, 0.58203125, 0.19775390625, 0.58447265625, 0.90966796875, -1.7158203125, 0.5634765625, -0.327880859375, -0.72802734375, 1.7958984375, 0.4931640625, 0.2646484375, 0.8466796875, 1.478515625, 0.4736328125, 0.08770751953125, -0.035858154296875, -0.7392578125, 0.2080078125, -1.00390625, 0.1763916015625, 0.145751953125, -0.242431640625, -0.273681640625, 0.293701171875, 1.04296875, -0.06585693359375, 1.412109375, 0.7890625, -1.5673828125, -0.7216796875, 1.794921875, -0.37109375, -2.00390625, 0.335205078125, -0.51904296875, 0.4833984375, -1.052734375, 1.2373046875, -0.56494140625, -2.498046875, -1.51171875, 0.4267578125, -0.23828125, -0.306640625, 0.37060546875, -1.259765625, -0.609375, -0.52978515625, 2.201171875, 1.1796875, 0.1102294921875, 0.1781005859375, 0.336181640625, 0.3525390625, -0.53564453125, 0.5087890625, 0.35107421875, -0.51416015625, 0.72802734375, -0.155517578125, 0.9658203125, 0.327880859375, -0.2430419921875, -0.280029296875, 0.04791259765625, -1.8486328125, 1.693359375, 1.1630859375, 1.369140625, -1.611328125, 1.7666015625, 0.284423828125, 1.0556640625, 0.0662841796875, -2.376953125, 1.779296875, -0.96484375, -0.86572265625, 1.251953125, -1.0947265625, 1.2080078125, -0.2413330078125, -1.3427734375, 1.9560546875, -0.72216796875, 0.9755859375, -1.2900390625, 0.1514892578125, 0.0947265625, 0.1768798828125, 0.75244140625, -0.5322265625, -0.401123046875, -0.12548828125, 0.1016845703125, 0.459228515625, 1.6796875, -0.15087890625, 0.66357421875, -0.6279296875, 0.52783203125, -0.1024169921875, -1.07421875, 0.11834716796875, -0.55224609375, 0.472412109375, -0.1422119140625, 2.390625, -0.0875244140625, 1.6259765625, 0.169189453125, -0.5537109375, 0.9482421875, -0.97314453125, -0.10736083984375, 0.057464599609375, 0.427490234375, -2.1875, -0.4775390625, -1.37109375, -0.53955078125, 0.04156494140625, -0.60400390625, -0.39501953125, -0.83056640625, -0.22998046875, -0.67138671875, 0.775390625, 0.81591796875, 1.28515625, 1.388671875, -0.429931640625, -0.77685546875, -1.0126953125, -0.1279296875, -0.09051513671875, 0.352294921875, 0.80029296875, -0.328857421875, 1.1279296875, 0.393798828125, -0.51513671875, -0.6708984375, -0.0633544921875, 0.73486328125, -0.27734375, -2.15234375, -0.99267578125, 1.603515625, -0.1668701171875, -0.86328125, -0.57958984375, -0.1265869140625, 0.10589599609375, -0.8056640625, -0.85302734375, 0.30908203125, 2.171875, 0.4990234375, -0.0826416015625, 1.2099609375, 0.233154296875, 0.1236572265625, 1.0361328125, 2.0703125, -0.56103515625, 0.244873046875, -0.6982421875, -0.548828125, 1.0087890625, -1.421875, 1.5400390625, 0.51123046875, 1.0693359375, -1.333984375, -0.544921875, -1.0029296875, -0.78515625, -1.2041015625, -1.126953125, -0.7890625, -0.736328125, -0.96826171875, 0.057647705078125, -0.5771484375, 1.314453125, -0.29248046875, 0.193603515625, 0.454833984375, -1.87890625, -1.7763671875, -0.385498046875, -0.87451171875, -0.96044921875, -0.29296875, 0.6787109375, -1.6494140625, 0.292236328125, 0.388427734375, -0.61474609375, -0.89599609375, 1.0859375, -0.75341796875], "dtype": "float16"}], "e": {"__nd__": [1.3134765625, 0.213134765625, 0.14794921875, 0.4189453125, 0.65673828125, 0.46044921875, -2.66796875, 1.7392578125, 1.494140625, 1.7939453125, -3.0078125, -1.607421875, 2.791015625, -0.9765625, 1.66015625, 0.1695556640625, 0.4716796875, 0.021484375, 0.4228515625, -1.439453125, 1.3486328125, -0.64111328125, 2.853515625, 0.568359375, 0.2275390625, 0.484375, -1.794921875, -2.546875, 3.47265625, -0.9072265625, 2.564453125, -1.380859375, -2.1875, 0.195556640625, 1.2734375, -1.0419921875, -0.1572265625, -0.0079345703125, -0.7568359375, -1.041015625, 0.005859375, -0.1318359375, -0.76904296875, -1.490234375, 0.5615234375, 1.8525390625, -1.6396484375, -0.80078125, -1.0625, 1.6845703125, -2.486328125, -1.15234375, 0.5859375, 0.255859375, -0.46923828125, 0.4677734375, -2.02734375, 0.42333984375, 1.443359375, 0.31640625, 1.78125, 2.00390625, -1.291015625, 1.3828125, -0.3173828125, -0.91943359375, 2.533203125, 0.5576171875, -1.4072265625, -0.81298828125, -0.2587890625, -1.5166015625, 0.73876953125, -1.5869140625, 2.953125, -2.486328125, -1.8740234375, -0.90576171875, -0.9677734375, -1.0703125, 0.190185546875, -2.802734375, 0.65625, 2.904296875, 0.0830078125, -0.84716796875, 0.92529296875, 1.0244140625, -0.96923828125, 1.056640625, 1.7890625, -2.58203125, -0.521484375, 0.99853515625, 1.3359375, -0.5390625, -1.228515625, 1.8046875, -3.576171875, -1.490234375, 1.166015625, 1.5712890625, -0.0693359375, -0.072509765625, -1.416015625, -0.78857421875, -1.201171875, -0.74560546875, -1.244140625, -0.72900390625, -2.3203125, 1.2568359375, -2.24609375, 2.080078125, 1.572265625, -0.33447265625, 0.30078125, -0.80078125, -0.9423828125, -1.2265625, 2.189453125, -0.99462890625, -0.32080078125, 1.5791015625, -0.2275390625, -0.96435546875, -1.3603515625, 1.1748046875, 0.1474609375, -0.467041015625, -0.107177734375, -0.91650390625, 0.3134765625, -0.1279296875, 0.89208984375, -0.41162109375, 0.61962890625, -1.1298828125, -1.419921875, -2.6484375, 1.154296875, -1.58203125, -0.5478515625, -1.537109375, -0.017578125, 0.4501953125, 1.345703125, -1.8466796875, -1.634765625, 0.61328125, -0.19775390625, -1.869140625, -1.0859375, -2.7265625, -0.900390625, 0.90380859375, 1.697265625, -0.412109375, -0.61279296875, -2.37890625, 1.5791015625, 1.052734375, 3.515625, 0.427490234375, -0.31884765625, -1.2158203125, 0.66357421875, 0.36474609375, 1.162109375, 2.671875, 1.7998046875, 0.8232421875, 2.53125, -0.8408203125, -1.05078125, -3.56640625, -1.0732421875, 1.728515625, 1.0302734375, 1.3447265625, -1.896484375, -0.4140625, 0.7880859375, 1.505859375, -0.06298828125, -1.115234375, -0.09033203125, -0.87939453125, -0.84814453125, -1.890625, -1.2021484375, 3.005859375, -3.83203125, -1.3857421875, 0.212646484375, 0.142578125, 0.50732421875, 1.267578125, -1.197265625, -0.091796875, 0.55859375, 0.857421875, 1.5, -0.27978515625, -2.41015625, 2.119140625, -1.15234375, -2.087890625, 3.62109375, 3.533203125, 1.353515625, -1.7080078125, 2.7109375, -0.353271484375, -0.6201171875, 0.15087890625, -1.998046875, -2.189453125, 1.34765625, 1.05859375, -1.765625, -0.603515625, -1.021484375, 3.537109375, -0.44140625, 3.64453125, 3.341796875, -0.66064453125, -1.00390625, -3.751953125, -1.5380859375, 1.1689453125, -0.6064453125, -1.2548828125, 0.101318359375, -0.982421875, -2.8515625, 1.15625, 2.203125, 0.08203125, 1.59765625, 0.11181640625, -0.99755859375, 1.505859375, -0.162109375, 0.8251953125, -0.78466796875, 1.8125, -0.185546875, 0.6748046875, 1.92578125, -0.6591796875, -0.483642578125, 0.285400390625, 0.39453125, -0.78125, -1.212890625, -0.6875, 1.3369140625, -0.4560546875, 0.042724609375, -0.66162109375, -0.4521484375, 3.64453125, -0.2021484375, 1.0283203125, -0.6630859375, 1.02734375, -0.56884765625, 0.076171875, 1.154296875, 1.0078125, -1.7783203125, -1.408203125, 2.50390625, 1.259765625, -1.6630859375, 0.646484375, 0.02734375, -1.1064453125, 0.5830078125, -1.1220703125, 0.252197265625, -0.7646484375, -0.99560546875, -0.5537109375, 0.003173828125, 0.22216796875, 0.69287109375, -0.048583984375, -2.421875, 1.5029296875, -2.701171875, 1.64453125, -1.3759765625, 0.712890625, -0.9892578125, 0.69580078125, -2.01953125, -0.64599609375, 0.5322265625, -0.4697265625, 0.974609375, -1.40234375, -1.12890625, -0.6650390625, 0.38916015625, 1.1025390625, -0.6806640625, -2.13671875, 0.85205078125, 1.607421875, 0.77880859375, 1.7587890625, 1.2509765625, 1.287109375, 0.0062255859375, 2.41015625, 0.18115234375, -0.603515625, 0.5009765625, 2.439453125, 0.6572265625, 1.595703125, -0.123779296875, 0.1103515625, -2.185546875, 1.0751953125, -2.15234375, 1.1513671875, 0.9013671875, 0.30859375, 2.5078125, 1.2265625, 2.673828125, -2.0703125, -1.2607421875, 0.888671875, 0.3193359375, 1.4658203125, -0.0955810546875, 0.5830078125, -1.720703125, 0.1129150390625, 0.4755859375, -0.0577392578125, -1.263671875, -0.26708984375, 0.145751953125, 0.2646484375, -2.205078125, 0.505859375, 0.59521484375, 1.009765625, 1.330078125, 0.339599609375, 0.41015625, -1.009765625, 0.0341796875, -0.39208984375, -2.115234375, -0.40234375, -1.498046875, 1.146484375, -1.015625, 2.66796875, 0.448486328125, -1.72265625, 0.383544921875, -1.076171875, -1.2197265625, 0.52197265625, -0.051513671875, 1.669921875, -1.0849609375, -0.4619140625, -2.068359375, 1.251953125, -0.443359375, 0.619140625, -0.658203125, 0.209228515625, 3.623046875, -0.8994140625, 0.04425048828125, -0.634765625, -0.089111328125, 0.99609375, 0.2861328125, 0.14111328125, -1.052734375, 1.1640625, -0.197265625, -0.30224609375, 2.05078125, -0.7705078125, -0.41162109375, 0.6103515625, -4.52734375, -1.8583984375, -0.115234375, 1.447265625, 0.86669921875, -1.0595703125, -0.345703125, 0.44580078125, -0.501953125, 0.8408203125, 0.02294921875, -0.027099609375, -1.4755859375, -0.90087890625, 0.63671875, 1.75, 0.32421875, 0.1025390625, 0.478515625, 0.966796875, -0.6279296875, 0.50537109375, 0.4443359375, -2.08984375, -0.3134765625, -1.2255859375, -1.0947265625, -2.796875, -0.0394287109375, 2.173828125, 0.7724609375, 0.199462890625, -0.2257080078125, -0.309326171875, -1.10546875, -0.517578125, -0.828125, -1.14453125, 0.978515625, -0.51904296875, -1.71875, 0.58203125, 0.9912109375, -2.18359375, -1.6533203125, -0.70361328125, 1.3994140625, -0.400634765625, 0.70556640625, -2.625, 0.63671875, 2.37109375, 0.218017578125, 3.107421875, -0.61572265625, 0.80078125, 1.5107421875, -0.401123046875, -1.126953125, 0.0087890625, -1.482421875, 0.424560546875, 1.0546875, 0.43115234375, -0.38818359375, -0.970703125, -0.151611328125, -0.429931640625, 0.356201171875, -2.87109375, -2.78125, 0.603515625, 0.41748046875, 1.21875, -1.6572265625, 0.939453125, 2.685546875, 0.634765625, -1.4267578125, 0.5810546875, -1.392578125, -1.111328125, -0.525390625, 0.489990234375, -0.5234375, 2.2734375, 1.13671875, 1.6513671875, 0.0782470703125, 1.767578125, 0.14208984375, -0.70263671875, 0.1962890625, -0.95263671875, 0.177001953125, 2.380859375, -2.244140625, 0.0869140625, 0.17578125, 1.607421875, 0.5537109375, -0.087890625, 0.9033203125, -1.064453125, -0.39208984375, 0.193603515625, 0.8603515625, 0.7236328125, -1.2890625, 1.8564453125, -2.72265625, -1.1435546875, -1.95703125, -0.40380859375, 0.8232421875, 0.204833984375, 1.0732421875, -0.9384765625, -3.0703125, 0.0673828125, 1.40625, 0.595703125, 2.67578125, -0.9560546875, -1.767578125, 0.98291015625, 0.408935546875, -0.86376953125, -2.328125, 0.2216796875, -0.0419921875, 0.72607421875, -0.517578125, 0.80810546875, -4.0078125, -0.297119140625, 1.9609375, -0.0225830078125, -0.20703125, 0.9384765625, -0.7314453125, -0.023193359375, -0.00537109375, 0.982421875, 0.8447265625, 1.4765625, -0.5703125, 1.0009765625, -1.447265625, 0.397705078125, -1.787109375, 2.12109375, 1.224609375, 0.13818359375, 1.4365234375, 1.880859375, 0.51953125, 0.8330078125, -0.213134765625, 1.1767578125, -2.23828125, 0.49755859375, -0.2705078125, -0.59033203125, -1.2587890625, -0.7919921875, -1.8046875, -0.54541015625, -2.845703125, -0.95703125, 2.044921875, 0.9658203125, 2.900390625, -0.986328125, 1.2998046875, -1.552734375, 2.142578125, -0.81201171875, -0.6494140625, -0.5009765625, 2.056640625, 0.43505859375, 2.5, -0.2164306640625, -0.5087890625, -3.314453125, 0.8642578125, -1.6298828125, 3.115234375, 0.62255859375, -0.41015625, -1.6259765625, 1.9560546875, -0.251953125, -1.64453125, 1.0888671875, -1.7275390625, -2.369140625, 1.0048828125, 1.9072265625, -1.78125, 0.447021484375, 2.5703125, -1.8857421875, 0.2724609375, -0.02587890625, 1.2177734375, 1.568359375, 2.08984375, 1.833984375, -1.23046875, 1.525390625, 2.466796875, -0.6015625, 0.71533203125, -0.9033203125, 0.44189453125, -0.02783203125, 2.572265625, 1.1796875, 0.69677734375, 2.5546875, 1.8486328125, 1.3046875, -1.673828125, 1.81640625, 1.4013671875, -0.75537109375, -1.4501953125, 1.6650390625, 0.29345703125, -1.23046875, 0.24365234375, 1.232421875, 0.32470703125, 1.25390625, -1.1689453125, 1.8486328125, -0.065673828125, -2.51953125, -1.275390625, 0.83984375, -0.479248046875, -1.328125, 0.101318359375, -0.78662109375, -0.07568359375, 1.12890625, -1.599609375, -1.64453125, -1.9287109375, -1.384765625, 0.1181640625, 1.376953125, 1.1005859375, -0.42236328125, -2.357421875, 1.60546875, -0.2685546875, 1.6728515625, -2.36328125, 0.3837890625, 1.3798828125, -1.560546875, 0.7294921875, 1.3955078125, -3.41796875, -1.00390625, -1.5771484375, -0.90283203125, -1.20703125, -0.341796875, 0.6923828125, -2.23828125, -1.359375, -1.74609375, 0.0087890625, 1.46875, -1.009765625, 1.9970703125, -0.6142578125, 1.611328125, 1.318359375, 0.313720703125, -0.1044921875, -1.693359375, 0.5302734375, -2.259765625, -2.05078125, -1.3984375, -0.6201171875, -0.7236328125, 0.413330078125, 0.8017578125, -1.064453125, -1.09765625, 0.50146484375, -0.4560546875, -0.451171875, -0.943359375, -1.0810546875, 0.473388671875, 0.44921875, 0.132080078125, 0.18896484375, -1.482421875, 0.37451171875, 2.529296875, -1.6015625, 1.302734375, -0.1292724609375, -0.44091796875, -1.0224609375, 0.25634765625, 0.149169921875, -0.419921875, 0.69580078125, -0.7998046875, 1.1826171875, -1.435546875, 0.32861328125, 2.33203125, 0.505859375, -1.7890625, 2.05078125, 0.477783203125, 0.732421875, -0.60986328125, -1.6533203125, 1.486328125, 0.9091796875, -0.3232421875, 0.31640625, -1.3447265625, 0.7138671875, -1.197265625, -0.77734375, -0.73046875, -2.21875, -1.7578125, -1.220703125, -2.296875, -1.1103515625, -1.291015625, 0.703125, 0.951171875, -0.8447265625, 2.072265625, -0.86669921875, -0.17822265625, -1.7783203125, 0.1259765625, -0.39111328125, 0.447998046875, -2.044921875, 1.0, 1.193359375, 0.9775390625, 0.701171875, -2.10546875, -2.1953125, -3.0, -2.09765625, -0.272705078125, 2.005859375, -0.8251953125, -0.498046875, -0.76513671875, 0.2197265625, 2.40234375, 0.6494140625, -1.515625, 0.39404296875, -1.0, 0.39697265625, -0.9814453125, 0.021484375, -0.38671875, 2.45703125, 1.52734375, -1.15625, -0.236572265625, -0.14404296875, -2.1953125, 3.19140625, -1.2451171875, 0.0068359375, 2.740234375, -2.380859375, 0.11767578125, 0.681640625, -2.0625, -3.6796875, 1.458984375, 0.1787109375, 0.01513671875, -1.607421875, -1.3896484375, 1.287109375, 2.47265625, 1.10546875, -2.015625, -2.6171875, -2.59765625, 0.232421875, 1.0556640625, -0.1680908203125, 0.12744140625, -0.130859375, -2.11328125, 2.25390625, -0.35986328125, -0.349853515625, -2.25, -0.9443359375, 0.609375, -1.03515625, 2.029296875, 0.7998046875, -1.111328125, -3.1328125, -1.19140625, 2.265625, 0.86962890625, -0.37158203125, -3.126953125, 0.0435791015625, 0.65673828125, -1.775390625, 0.83251953125, -0.4287109375, -3.158203125, -1.037109375, -1.2431640625, -0.415771484375, -0.9716796875, 0.2041015625, -0.31494140625, 1.2265625, -0.04541015625, 0.095703125, 1.3466796875, -0.9794921875, -0.982421875, -1.9375, 0.85498046875, 1.4931640625, -0.69677734375, -1.875, 1.46875, -1.501953125, 0.1689453125, -0.02880859375, 0.223876953125, 0.720703125, -1.525390625, 0.13232421875, 1.9912109375, 1.4306640625, -0.2392578125, 0.70361328125, 0.419921875, 1.8505859375, -0.08984375, 0.92236328125, 0.322021484375, -1.2734375, -1.9365234375, -1.1064453125, -1.830078125, 0.755859375, 2.302734375, 1.6728515625, -1.18359375, 1.16015625, -0.0504150390625, 0.4033203125, 0.059326171875, -1.3056640625, -1.3779296875, 3.0078125, 0.6767578125, -0.9296875, -3.953125, 0.59130859375, -1.20703125, 0.1385498046875, -1.7578125, 1.25390625, -1.6171875, 0.28173828125, -0.56982421875, 1.28125, -1.275390625, 0.3173828125, 1.2578125, 1.1484375, -1.228515625, 1.5546875, -0.958984375, -1.55859375, 1.8486328125, 1.1875, 0.4052734375, 0.472412109375, -0.87255859375, 1.26171875, 0.9208984375, -0.7509765625, 2.0625, -1.734375, 1.041015625, -0.44482421875, -1.19921875, -1.08984375, 0.291015625, -2.3359375, -1.1513671875, 0.6083984375, -1.302734375, 0.5224609375, 0.1533203125, -0.9091796875, 0.450927734375, 0.046875, 0.0439453125, 1.7197265625, 3.166015625, -1.47265625, -0.09814453125, 3.546875, 1.12109375, -0.6064453125, 0.69873046875, 0.10693359375, -0.9521484375, -0.6875, 0.4033203125, 0.19287109375, -0.9599609375, 1.4794921875, 0.623046875, -1.494140625, 1.66796875, 0.429443359375, -2.599609375, -0.126953125, -1.041015625, 2.732421875, 1.197265625, 0.71435546875, 0.90771484375, -1.376953125, -0.6005859375, 0.0185546875, -0.08642578125, 0.68798828125, 1.4072265625, 0.025390625, -1.9345703125, -3.017578125, -1.2646484375, 4.2109375, -0.83251953125, 1.4052734375, 0.3427734375, -0.09912109375, -1.7568359375, 0.252197265625, 1.236328125, 0.190185546875, -0.419921875, -2.958984375, 0.740234375, -0.39794921875, -3.125, -0.85400390625, -0.548828125, -0.40625, -0.3359375, 1.4287109375, 2.634765625, 1.4375, -0.21435546875, -0.1708984375, -0.69287109375, -0.94580078125, 1.0576171875, 1.787109375, -2.169921875, -0.083984375, -1.431640625, 1.884765625, -1.0419921875, 2.08203125, -0.185546875, -3.251953125, 0.163818359375, -0.292724609375, 0.166259765625, 0.34228515625, -0.1912841796875, -1.154296875, -0.01806640625, 1.275390625, -0.556640625, -0.01959228515625, -0.45361328125, 0.7275390625, -0.413818359375, 1.8623046875, 1.998046875, 0.9267578125, 0.001953125, 2.08203125, -1.466796875, -1.7431640625, -0.76611328125, 2.255859375, 1.359375, -0.220703125, -0.36572265625, -0.74072265625, 0.6826171875, -0.568359375, 2.73828125, -2.486328125, 0.6259765625, -0.70166015625, 0.5703125, -0.11328125, 1.462890625, -0.3623046875, -1.09765625, 1.015625, 0.845703125, -3.0, 1.05859375, -0.90625, 1.029296875, 0.6533203125, 2.169921875, 0.80615234375, -0.5537109375, 0.99755859375, 0.131103515625, 1.54296875, 0.44091796875, 0.9365234375, -1.4111328125, 2.64453125, 2.125, 0.09765625, 0.005859375, 1.1181640625, -0.2275390625, 0.264892578125, -0.4169921875, 0.249755859375, 0.5595703125, -1.3408203125, 0.439697265625, 2.85546875, -0.96337890625, -1.2744140625, -1.787109375, -0.0482177734375, -1.900390625, -1.734375, -1.4345703125, -3.5078125, 0.09136962890625, 0.8037109375, -0.5791015625, -1.548828125, -0.173828125, 2.1640625, 2.431640625, 0.0615234375, 1.697265625, 1.1298828125, 0.798828125, -0.8984375, 0.4970703125, 0.62646484375, 0.1817626953125, 1.4267578125, -1.322265625, 1.638671875, 0.173583984375, 1.087890625, 1.9150390625, -3.6875, 1.5068359375, -1.4140625, -1.578125, 2.677734375, 2.921875, -0.31494140625, -0.7099609375, -3.125, 0.7080078125, -1.01171875, 0.435791015625, -0.9140625, 0.1982421875, 0.896484375, 1.2841796875, -0.43017578125, -1.6142578125, -0.82568359375, 1.8974609375, 0.7880859375, -0.3251953125, -0.025390625, -1.431640625, 0.62890625, -1.833984375, 1.240234375, 0.51025390625, -1.5947265625, -1.552734375, -1.2392578125, -0.89794921875, -1.4130859375, 0.1396484375, -0.83544921875, -2.142578125, 1.181640625, -1.126953125, 0.5546875, 0.61328125, -0.1505126953125, 0.7255859375, -1.576171875, 0.053466796875, -0.258056640625, -0.256103515625, 1.166015625, 1.28125, -1.697265625, -0.6611328125, 1.443359375, 1.595703125, -0.869140625, 1.154296875, 1.025390625, -1.8037109375, -1.0390625, -0.509765625, 1.47265625, 0.85302734375, 0.7294921875, -2.078125, -0.0517578125, 0.3916015625, 0.307861328125, -0.2471923828125, -1.9345703125, -0.1806640625, -0.03515625, 3.203125, 1.6298828125, 0.70458984375, 2.57421875, 0.19921875, -0.46435546875, -0.4736328125, -1.2744140625, -1.5380859375, 0.28173828125, 0.404296875, 1.884765625, 3.12890625, -1.6171875, -1.212890625, 1.5712890625, 0.6728515625, 0.125, -1.701171875, -0.466796875, -1.408203125, -0.87939453125, -0.83251953125, 0.496826171875, 4.0234375, 2.390625, -1.068359375, 1.1171875, 1.31640625, 1.26953125, -0.35400390625, 1.9150390625, -0.8310546875, -0.603515625, -0.3076171875, -1.8388671875, 2.2265625, 3.388671875, 0.9091796875, 0.2099609375, 0.12890625, 0.8427734375, 0.05517578125, 1.0224609375, 0.0068359375, -0.06103515625, 0.08709716796875, 0.7626953125, 0.333740234375, -0.7958984375, -1.36328125, -3.390625, -0.470703125, 0.1474609375, 0.11767578125, -2.244140625, -0.68798828125, -0.7333984375, -0.05712890625, 1.65234375, 0.96484375, 3.388671875, -0.70458984375, -1.4287109375, -0.10986328125, 0.2265625, 1.9140625, -0.154541015625, -0.116455078125, 1.341796875, 1.1953125, 1.380859375, -2.33203125, -0.798828125, 0.1484375, -1.3681640625, 1.6953125, 0.09912109375, 0.91796875, -0.12744140625, -1.630859375, 0.06103515625, 0.72509765625, 1.751953125, -1.34765625, -2.18359375, -0.82666015625, 0.421142578125, 1.3759765625, 0.06103515625, 1.5869140625, -1.9921875, -1.14453125, -0.76025390625, -2.013671875, -0.1881103515625, 2.05078125, 0.8671875, 0.0654296875, -0.59375, 1.4736328125, 1.291015625, 0.9853515625, -0.0419921875, -0.65625, 0.296875, 0.951171875, -1.447265625, 1.818359375, -0.9755859375, -0.198486328125, -2.216796875, -1.025390625, -0.728515625, -2.06640625, 0.03271484375, 0.658203125, 0.67236328125, 0.298828125, -0.411376953125, -0.8271484375, 2.88671875, -0.52978515625, -1.720703125, -1.9453125, -0.8681640625, -1.8984375, 1.513671875, -1.171875, 0.1025390625, 0.86181640625, 2.853515625, 0.47412109375, 0.26123046875, -2.9765625, -0.79736328125, -2.52734375, 0.4482421875, -0.4853515625, 4.23046875, 0.720703125, 3.26171875, -1.1044921875, 0.0753173828125, -0.04833984375, 3.7890625, 1.04296875, -0.33056640625, -2.6953125, 0.375, 0.77685546875, -0.311279296875, -1.41015625, 3.00390625, 2.90625, -2.99609375, 0.4814453125, 0.33642578125, -0.00927734375, -1.044921875, -0.0009765625, -0.90087890625, 1.103515625, -2.763671875, -0.572265625, -0.77880859375, 0.716796875, 2.521484375, 1.4375, -0.8662109375, 0.87109375, 0.08447265625, 0.68017578125, -0.38134765625, -1.2197265625, 2.216796875, 1.501953125, 0.599609375, -0.830078125, 2.18359375, 0.5537109375, 0.1650390625, -3.0390625, -0.12255859375, -0.6748046875, 1.6494140625, 1.00390625, 1.8369140625, 2.83203125, -0.900390625, 0.6044921875, -1.30859375, -2.546875, -0.54345703125, 0.6533203125, -0.92236328125, 3.171875, -1.1962890625, -0.443359375, 0.13916015625, -1.5146484375, 0.030029296875, -1.0537109375, 1.3349609375, 0.140869140625, 0.83447265625, 0.39501953125, -1.5498046875, -1.3173828125, 1.16015625, 0.3349609375, -0.662109375, 0.93115234375, -2.47265625, 2.193359375, 0.266845703125, 0.0721435546875, 1.4658203125, -2.70703125, -2.720703125, -0.75537109375, -1.41796875, 1.947265625, 2.66015625, 1.609375, 0.07177734375, -0.114990234375, 4.0078125, -1.16796875, 0.5712890625, -1.435546875, -0.70751953125, 1.8681640625, -0.2021484375, 0.779296875, 0.5185546875, 1.091796875, 0.0283203125, -0.45654296875, 1.021484375, 0.5830078125, -0.421875, 2.796875, 3.484375, -0.135986328125, -1.02734375, -1.6005859375, -0.3310546875, 0.317138671875, -0.162841796875, -1.5009765625, 0.7490234375, -0.60791015625, -0.740234375, -0.62890625, 0.36474609375, 0.3076171875, -0.6279296875, -2.244140625, -0.91845703125, 0.6650390625, 0.9296875, -0.6796875, 1.37890625, -1.158203125, 1.3837890625, -0.44580078125, -0.59716796875, 1.53125, 0.6044921875, 0.97216796875, -0.5244140625, -0.408203125, 1.0048828125, 1.24609375, 0.474609375, 0.47265625, 2.03125, -1.8203125, 1.0986328125, 0.15185546875, 0.6748046875, 0.814453125, -2.58203125, -1.87890625, -0.041748046875, 1.2734375, -0.015625, -0.4296875, 1.287109375, -1.9140625, -0.90625, -0.58056640625, 0.112548828125, -2.619140625, -0.1138916015625, -1.4541015625, -0.84375, -2.712890625, -1.111328125, -1.326171875, 0.04931640625, -1.701171875, 0.623046875, 0.481689453125, -1.5927734375, -3.033203125, -1.275390625, -1.55078125, 1.767578125, -1.5732421875, -0.4443359375, 0.41796875, 0.638671875, -0.8310546875, -2.220703125, -1.1513671875, -0.5400390625, 0.10565185546875, 1.3046875, -0.9306640625, -1.47265625, 1.443359375, 0.10546875, -1.0830078125, 4.19921875, 0.000244140625, 2.04296875, 1.31640625, 0.890625, -0.876953125, -0.009765625, 0.3369140625, 1.271484375, 2.62109375, 1.322265625, 1.439453125, 1.765625, -1.119140625, -0.4482421875, 0.57177734375, -2.12109375, -1.4482421875, -0.22607421875, 0.450439453125, 0.1767578125, -3.0546875, 0.1298828125, -1.3505859375, 0.08056640625, -0.3828125, -1.720703125, 0.5390625, -0.64013671875, -0.236083984375, -1.875, -2.12890625, 1.6494140625, -0.28369140625, -1.4375, 1.330078125, 0.6669921875, -0.05145263671875, 1.154296875, -0.293212890625, 0.60595703125, -2.83984375, 2.578125, -2.95703125, 3.048828125, 0.52783203125, 1.9072265625, -1.49609375, -0.39013671875, -0.84765625, -1.27734375, -0.78515625, 0.9306640625, -0.1240234375, 0.88916015625, 0.875, 0.50244140625, -0.018798828125, 0.3349609375, -1.078125, 0.69140625, -1.126953125, 0.7490234375, -2.0546875, 0.591796875, -0.591796875, -0.7744140625, -0.517578125, -2.6953125, -0.2265625, -2.1171875, 1.658203125, -0.970703125, -1.1953125, 0.150390625, 2.197265625, -0.63671875, 0.5126953125, -1.447265625, -0.1995849609375, 0.7177734375, -2.8203125, 1.091796875, -0.61083984375, 1.416015625, 1.8935546875, 3.015625, 0.7275390625, -1.455078125, 0.0458984375, -0.99462890625, -1.560546875, 0.35205078125, 2.2890625, -0.08148193359375, -1.9140625, -0.2177734375, 0.126708984375, 1.17578125, 1.4404296875, -0.63671875, -0.9658203125, 1.7412109375, 1.6640625, 2.15625, -1.541015625, -1.8740234375, -0.8720703125, 2.88671875, 0.61865234375, 0.0126953125, 0.666015625, 0.12030029296875, 0.912109375, 2.751953125, 0.34375, -1.3876953125, -0.28369140625, -2.78515625, 0.17236328125, 1.3818359375, 2.67578125, 3.52734375, 0.5595703125, -1.58984375, 2.2734375, -0.0220947265625, -0.252685546875, -2.3515625, -0.093994140625, -0.333251953125, -0.17724609375, 0.440185546875, 0.8525390625, 0.58251953125, 0.64501953125, -0.0166015625, -0.7744140625, -1.2080078125, 3.021484375, -0.17529296875, 0.58740234375, -0.4873046875, 2.416015625, 0.1658935546875, 2.537109375, 1.083984375, -0.79736328125, -1.83984375, -0.2900390625, 0.83349609375, 1.04296875, -0.264404296875, 0.60205078125, 1.017578125, -0.5361328125, 2.1875, -2.2578125, 2.34375, 1.51171875, 0.5322265625, 0.5859375, -1.732421875, -1.880859375, -0.04296875, -1.1328125, 0.4150390625, 0.64453125, 0.1448974609375, -0.958984375, 0.07861328125, 1.12109375, -1.35546875, 1.8720703125, -0.61962890625, -0.75390625, 0.186767578125, -1.6630859375, 1.798828125, -0.40283203125, 0.0224609375, 1.51953125, 1.03515625, 0.271240234375, -1.97265625, 0.7392578125, 0.1826171875, 0.41455078125, -0.8408203125, 1.888671875, 2.109375, 0.454345703125, 1.0322265625, -0.25341796875, 0.1396484375, -0.8173828125, -0.7392578125, 0.3544921875, -1.732421875, -3.7109375, -2.701171875, -0.88916015625, -1.7890625, 0.25732421875, -1.9423828125, 5.01953125, 2.259765625, -0.50537109375, 0.8203125, -0.673828125, 0.84033203125, 0.1396484375, -1.3671875, 1.123046875, 0.50830078125, -4.0234375, -1.455078125, -0.6318359375, 0.4296875, 2.25390625, -0.6474609375, 0.492431640625, -1.5625, -0.1162109375, 1.9248046875, 1.71484375, 0.5146484375, 0.56005859375, -0.6259765625, 0.7578125, 3.103515625, 0.8837890625, 0.0986328125, 0.806640625, -0.171875, 1.6396484375, -0.44677734375, -1.662109375, 0.52392578125, -0.91015625, 0.214111328125, -1.8671875, -2.8125, 0.92333984375, -0.0263671875, 1.2666015625, -1.43359375, -2.220703125, -1.130859375, -0.5087890625, 0.3037109375, 0.427734375, 0.9951171875, 4.22265625, -0.66943359375, 0.365234375, -0.02197265625, -0.67138671875, -2.328125, -2.48828125, -0.124755859375, 0.837890625, 1.73046875, 1.63671875, 0.340576171875, 1.154296875, 1.8291015625, 0.095458984375, -0.07080078125, -0.62744140625, -0.4111328125, 1.767578125, 1.1435546875, -0.49365234375, -2.177734375, 0.190185546875, 0.054931640625, -1.3447265625, 0.5126953125, 0.447265625, 0.9921875, 3.0, 0.1005859375, -1.0673828125, 1.9248046875, 1.2861328125, -0.75537109375, -1.34765625, 0.76904296875, 1.384765625, 0.7685546875, 2.4453125, -0.5537109375, -0.8798828125, 3.8203125, -0.228515625, -1.822265625, -0.59716796875, -0.97412109375, 1.5341796875, -0.129150390625, -1.3515625, -0.0352783203125, 0.0830078125, 0.241943359375, 1.3515625, 1.501953125, 0.62255859375, 1.40234375, -1.095703125, 1.4619140625, -0.673828125, -0.26220703125, -1.798828125, 1.4833984375, -1.916015625, -0.9296875, -1.265625, 1.0166015625, -2.32421875, -0.312744140625, -0.76025390625, -0.9638671875, -1.251953125, 0.4853515625, 1.1337890625, 0.044921875, -1.419921875, 0.61181640625, -3.53515625, -0.436279296875, 2.353515625, -0.3876953125, -1.7744140625, -0.7998046875, 0.3017578125, -0.49365234375, -0.744140625, -0.72900390625, 1.546875, 0.294921875, 0.491455078125, -0.28955078125, -0.6982421875, 0.03857421875, 2.296875, -2.12109375, -1.9873046875, -0.11474609375, 3.62109375, 0.33984375, 0.189453125, -1.0595703125, 2.1171875, 1.615234375, -1.21484375, -1.9296875, 1.2685546875, 2.9140625, 2.16015625, 0.54052734375, 3.546875, -1.2568359375, 0.74072265625, 0.27783203125, -0.6708984375, 0.03125, -0.0966796875, -1.857421875, -1.181640625, -0.8916015625, 0.252197265625, -1.1689453125, 1.8544921875, -0.95751953125, -0.150146484375, -0.1396484375, -1.7578125, -1.5244140625, 1.873046875, -2.56640625, 0.515625, 2.064453125, -1.416015625, -0.6396484375, -1.5234375, -0.5947265625, -0.74609375, 2.212890625, -0.19384765625, 0.21435546875, -0.9990234375, -0.096435546875, -0.63623046875, -0.85546875, -1.3681640625, -1.80859375, -2.49609375, -0.54638671875, 0.48583984375, -1.1845703125, 1.1953125, 1.21484375, -1.224609375, -0.39306640625, 0.472900390625, -0.087890625, -0.5791015625, -0.5576171875, 0.016845703125, -0.52587890625, 1.244140625, -0.6337890625, 1.9970703125, 0.28125, 1.9169921875, 0.859375, 2.060546875, -1.505859375, 1.236328125, 3.404296875, 2.8359375, -0.734375, -1.140625, -0.2333984375, -0.84765625, -0.3310546875, -2.861328125, 1.583984375, -0.7373046875, 0.625, -0.403076171875, 1.3837890625, 0.387451171875, 0.29736328125, -0.40478515625, -0.13671875, 2.087890625, 0.496337890625, -0.54443359375, -1.2734375, -1.5517578125, 0.5947265625, -0.2509765625, -0.20703125, -2.03125, -0.6787109375, -1.6142578125, 1.65234375, 0.384521484375, -0.106689453125, 0.0782470703125, 0.94140625, -1.052734375, -1.3984375, -1.490234375, -1.2783203125, -0.67236328125, -3.59765625, -1.392578125, 2.802734375, 0.003173828125, -0.9638671875, -1.3056640625, -1.23046875, -0.47900390625, 2.435546875, 0.44140625, -0.082763671875, -0.9130859375, -0.4873046875, 0.427490234375, -0.60986328125, 0.37890625, -1.4697265625, -0.81787109375, -0.6748046875, 0.8857421875, 0.435791015625, -0.3037109375, 1.388671875, -0.0234375, -2.1015625, 0.79150390625, 0.4951171875, 1.11328125, -2.2109375, -0.26904296875, -0.8212890625, 0.037109375, -0.30322265625, 0.1939697265625, 0.56396484375, -2.19921875, -1.4609375, -0.7119140625, -0.136474609375, -0.5322265625, -0.68212890625, -0.4521484375, 2.4140625, 0.8037109375, 2.869140625, 2.138671875, 1.28125, -1.669921875, 2.25390625, -0.548828125, -0.2310791015625, 0.84375, 0.43310546875, -1.9541015625, 1.63671875, -0.28857421875, 0.0814208984375, -1.8046875, -0.5048828125, 0.8818359375, 1.28125, -0.052734375, -1.34375, 0.24755859375, -1.2548828125, -0.7373046875, 2.31640625, -0.73046875, -0.3095703125, 1.0107421875, -0.095703125, -1.1787109375, 0.01025390625, 2.59375, 0.97412109375, 1.5361328125, 1.04296875, 2.43359375, -1.3974609375, 0.93017578125, -2.52734375, -0.025634765625, 0.3134765625, 0.728515625, 0.7431640625, -0.11474609375, 1.4033203125, 0.61376953125, 3.529296875, -0.54248046875, -0.31982421875, -1.24609375, 2.185546875, -0.48681640625, 1.662109375, -1.1337890625, -0.94140625, 1.1923828125, 2.2734375, 0.1451416015625, -2.1875, 0.64990234375, -1.708984375, -0.23876953125, -1.3759765625, -2.494140625, 2.8125, -0.962890625, 0.80810546875, 2.306640625, -0.373046875, -0.34423828125, -2.115234375, -0.57470703125, -0.81787109375, 1.701171875, -1.3857421875, 1.1796875, -1.6552734375, 1.490234375, -1.0048828125, 0.34765625, -0.1148681640625, -0.0924072265625, 1.666015625, 1.8125, 0.72509765625, 1.3759765625, 2.33984375, 0.3857421875, -1.0107421875, -2.220703125, -1.931640625, 0.12255859375, -1.9443359375, 1.5498046875, -2.205078125, -0.697265625, 1.2998046875, 1.892578125, 0.058074951171875, -0.0888671875, 4.07421875, 1.177734375, -1.3447265625, 1.1357421875, -0.74853515625, -1.5087890625, 2.59375, -1.685546875, -0.9423828125, -1.7333984375, 0.0703125, 1.087890625, -1.3466796875, -1.84765625, 0.109619140625, -2.353515625, 0.059814453125, 0.11962890625, 0.146484375, -1.05859375, 2.55078125, -0.615234375, -2.3359375, -1.7490234375, -0.4384765625, -0.49462890625, 0.26611328125, 1.0625, -3.275390625, -2.029296875, -3.640625, 0.91259765625, 0.042724609375, 1.3125, -0.64599609375, 2.021484375, 0.90087890625, -0.10595703125, -2.14453125, 0.96826171875, 0.81689453125, -0.04443359375, -3.119140625, -3.38671875, 1.83984375, -1.7763671875, 2.064453125, -0.415283203125, 1.3984375, 0.64990234375, -0.405029296875, 2.80859375, 3.78515625, 0.71923828125, -0.7451171875, 0.0009765625, -0.98046875, -2.201171875, 2.50390625, 1.2265625, -0.38037109375, -0.7900390625, -0.7705078125, -1.0498046875, 1.216796875, 2.92578125, -1.0869140625, 1.1328125, 0.3232421875, 0.1875, -0.0753173828125, -0.828125, 0.7705078125, -1.3876953125, -3.138671875, -2.97265625, -2.611328125, 0.167724609375, -2.400390625, -2.8046875, 1.552734375, 0.21533203125, 0.62646484375, 0.44775390625, 0.4921875, 1.552734375, -2.423828125, 0.3876953125, 0.853515625, 0.66845703125, 2.251953125, -0.548828125, 1.271484375, 3.4921875, 0.849609375, -1.443359375, -0.11376953125, -0.395751953125, -0.341552734375, -1.4921875, -0.39111328125, -0.529296875, -0.4755859375, 0.8408203125, 1.25, -0.1103515625, 1.021484375, 0.74560546875, 1.330078125, -1.244140625, -1.9375, -0.15185546875, 0.45556640625, -1.28515625, 0.7705078125, -0.10009765625, -0.45751953125, 1.375, -0.025390625, -1.30859375, 1.2314453125, 0.6455078125, -1.09765625, -0.64697265625, 1.4501953125, 3.1015625, 3.15234375, 2.04296875, -0.233642578125, -0.03466796875, -0.2236328125, 0.115966796875, -0.8505859375, 2.685546875, 0.093994140625, 1.1015625, -0.5498046875, 2.306640625, 0.255615234375, -1.076171875, -0.9033203125, 3.03515625, 1.8662109375, 1.103515625, 0.5322265625, -0.4228515625, 1.1328125, -0.9638671875, 1.125, 1.9736328125, -1.1650390625, -1.8125, 0.93408203125, 0.6796875, -0.406982421875, -1.0224609375, -0.6435546875, 0.111328125, -0.67138671875, 1.0869140625, -1.66015625, -0.03167724609375, -0.5791015625, 0.334228515625, -0.92578125, -1.52734375, -1.236328125, 0.2269287109375, 0.84716796875, -0.50439453125, -4.03125, 0.2183837890625, -1.078125, -2.638671875, 0.96337890625, 0.9482421875, -0.4833984375, 1.3125, -1.3232421875, -0.5283203125, -1.220703125, -0.876953125, -2.1953125, -0.485595703125, -0.04986572265625, -0.470703125, 0.9580078125, -0.2427978515625, 1.076171875, -0.1947021484375, 0.76806640625, -0.974609375, -0.6669921875, -0.014404296875, -0.1312255859375, -0.96923828125, 1.65234375, -0.107421875, -0.5283203125, 0.65185546875, -0.57275390625, -2.21875, -0.880859375, -0.875, -2.009765625, -2.251953125, 1.625, -0.681640625, 2.037109375, -0.1845703125, 0.46875, -1.177734375, -1.28125, -0.64794921875, 0.388671875, -0.078125, -0.75439453125, 1.21484375, 0.7265625, -0.80517578125, 0.701171875, 2.3046875, -1.287109375, -0.5615234375, -2.576171875, -0.0206298828125, 1.857421875, 0.7119140625, 0.255126953125, 1.75, 0.68505859375, 1.2080078125, -0.7783203125, 0.410400390625, 0.8955078125, 2.130859375, -0.17822265625, -0.8310546875, 0.16943359375, -0.97412109375, 0.78662109375, 0.462890625, -1.13671875, 0.42578125, 0.1611328125, -1.654296875, 0.37646484375, -1.95703125, 1.505859375, -2.484375, 0.5380859375, -2.205078125, -0.86376953125, 1.0751953125, -0.38427734375, 0.8994140625, -1.7109375, 0.9287109375, 0.06524658203125, -0.0693359375, -1.13671875, 1.26171875, 0.86083984375, -0.36669921875, -0.413330078125, 1.1689453125, -0.21044921875, -1.091796875, -1.232421875, -0.357666015625, -2.806640625, 1.11328125, 0.935546875, -1.74609375, -1.0380859375, 0.544921875, 0.92236328125, 0.724609375, -0.125, 1.0078125, 0.69580078125, -1.10546875, -1.2021484375, 0.4296875, -2.18359375, -1.46875, -0.228271484375, 0.82275390625, -2.798828125, -0.85546875, -0.28125, -2.1796875, 1.3125, 0.5, 0.78466796875, -0.5556640625, 0.98046875, 1.3486328125, 0.87451171875, -0.57958984375, -2.01171875, 0.7060546875, -3.640625, 0.513671875, -1.95703125, 0.603515625, -1.251953125, -0.1416015625, -1.2255859375, 1.30078125, 2.34375, 1.345703125, 3.95703125, -0.287109375, -1.54296875, -2.1015625, 0.9296875, -1.5361328125, -0.11962890625, 1.1298828125, 0.5205078125, -0.0888671875, -0.79150390625, -0.452392578125, 3.171875, -0.432373046875, -1.3427734375, -0.289794921875, -2.251953125, -0.57177734375, -0.0537109375, 0.02001953125, 1.947265625, -1.654296875, 1.712890625, -1.8134765625, 0.03515625, 1.2421875, -0.02294921875, -0.60791015625, -1.4462890625, -1.47265625, -1.0517578125, 2.90234375, 0.1221923828125, 0.53271484375, 0.359375, 0.48388671875, 1.3681640625, 1.3896484375, 2.0546875, -0.70703125, -0.76513671875, 1.08203125, -0.177734375, -0.249755859375, -0.74658203125, -0.63427734375, 0.59716796875, -1.353515625, 1.5830078125, -1.7109375, -0.740234375, -0.818359375, -0.441162109375, -1.7451171875, -0.6669921875, -0.09765625, 1.138671875, 0.6435546875, -2.259765625, -1.8330078125, -0.477294921875, 1.734375, -0.1844482421875, 1.6513671875, 2.82421875, 1.51171875, 0.89990234375, 0.158203125, 1.193359375, -0.18017578125, 2.20703125, 0.275390625, 1.876953125, -1.796875, -0.60107421875, -0.0133056640625, -0.23193359375, 1.0849609375, 1.25, -0.361328125, -1.61328125, -1.5927734375, 2.00390625, -1.1337890625, -1.38671875, -3.1171875, -1.185546875, -1.5595703125, 0.67041015625, -0.7197265625, 1.990234375, 1.16015625, -1.3232421875, 0.248291015625, -3.01953125, 0.05419921875, 0.90966796875, 1.17578125, -1.744140625, 1.130859375, 0.177001953125, 1.5751953125, 0.5029296875, 0.133056640625, 0.23779296875, 0.5419921875, -0.68798828125, -1.1796875, 0.609375, 2.0546875, -0.296142578125, 0.7421875, -0.98046875, 2.41796875, -0.099853515625, -2.7109375, 1.734375, -2.21875, 1.224609375, 1.5576171875, -0.4814453125, 2.291015625, -0.1337890625, -0.3896484375, 0.67138671875, -0.36767578125, 0.005859375, -1.11328125, -2.578125, 1.404296875, -0.671875, 2.16015625, 0.2373046875, -0.21923828125, -0.6630859375, -0.102783203125, -1.189453125, -1.974609375, 0.9931640625, -0.7451171875, 1.3369140625, 0.56201171875, 1.365234375, 1.134765625, -0.537109375, 1.5263671875, -0.158203125, 1.4736328125, -0.94580078125, -2.5859375, -0.205078125, -2.041015625, -0.7919921875, 0.7890625, -1.716796875, 1.0048828125, 1.79296875, -2.8046875, -2.123046875, 1.3173828125, 1.291015625, -0.0435791015625, -1.029296875, -0.127685546875, -0.56884765625, -0.8232421875, 0.267578125, 0.5849609375, -0.12109375, -1.8173828125, 3.044921875, 1.396484375, 1.470703125, -2.89453125, 0.64794921875, -0.02392578125, 0.225341796875, 1.7890625, 1.01953125, 0.8349609375, -0.52294921875, -2.033203125, -4.35546875, 1.1552734375, 1.0205078125, 0.83154296875, -1.3251953125, 2.802734375, 0.291748046875, 2.41015625, 0.5595703125, -1.5302734375, 0.188720703125, 0.0361328125, -0.724609375, -3.25390625, -0.38232421875, 0.7587890625, -0.2041015625, -0.3154296875, 0.446044921875, 4.16015625, -0.228515625, 1.27734375, 0.8486328125, -0.04638671875, 1.853515625, -0.473388671875, -0.496826171875, -1.7275390625, -0.2138671875, -0.0205078125, 0.67578125, 3.02734375, -3.7734375, 1.28125, -0.833984375, 0.132568359375, 0.986328125, 1.265625, -1.361328125, -0.51806640625, 0.1181640625, -1.29296875, -0.19677734375, -0.5458984375, -0.25244140625, -2.8359375, 2.333984375, -0.45068359375, 0.5849609375, 3.2265625, 1.431640625, -1.1533203125, -0.0048828125, 0.6328125, 0.253662109375, -0.3974609375, 1.9658203125, 1.9072265625, 1.056640625, 2.330078125, 2.4609375, -0.83349609375, -0.35302734375, -0.277587890625, -0.2440185546875, 2.55859375, -0.232421875, -0.5869140625, 0.007049560546875, 3.828125, -1.2578125, -1.4208984375, 0.98046875, -0.93408203125, 0.148681640625, -1.115234375, 0.5859375, 1.46484375, -2.47265625, 0.8798828125, -3.09765625, 1.3798828125, -1.2119140625, -0.7958984375, 1.193359375, 0.080078125, -0.47216796875, -0.021240234375, 0.54248046875, 0.492431640625, -1.21875, 0.8798828125, 0.0166015625, 2.1953125, -0.1416015625, -1.38671875, -0.58740234375, 0.28271484375, 0.908203125, -1.671875, -1.0703125, -1.146484375, 1.3955078125, 0.740234375, -0.94677734375, 0.91845703125, -1.8583984375, -1.1845703125, -2.671875, -1.75390625, -0.85107421875, 1.3203125, 0.90283203125, 0.148681640625, -1.48046875, -3.98828125, 2.2109375, 1.91015625, 3.853515625, -0.375, -1.375, 0.0224609375, -2.7890625, -0.30859375, -0.2880859375, -0.833984375, -0.56591796875, -0.33349609375, -0.5048828125, 1.7724609375, -0.2001953125, -1.1376953125, -0.6943359375, -1.119140625, 0.744140625, -1.9140625, 0.39208984375, -0.8642578125, 0.1923828125, -0.69482421875, -2.171875, -0.1268310546875, 0.4287109375, -2.01171875, -0.2421875, 0.5283203125, -1.458984375, -1.9755859375, 1.9921875, 0.37890625, 0.37841796875, -0.62109375, -2.853515625, 1.0048828125, -0.220703125, 0.9462890625, 1.392578125, -0.9248046875, -0.76953125, 1.0732421875, -0.73046875, 0.63916015625, -1.4072265625, 2.6484375, -2.958984375, -0.185546875, 0.55322265625, 2.27734375, 0.1190185546875, -1.3857421875, 2.404296875, 0.1904296875, 0.55859375, 5.171875, -0.84521484375, -0.60546875, -1.009765625, -0.80517578125, -1.68359375, -1.216796875, -1.4619140625, 0.53369140625, -0.38427734375, -0.2288818359375, 0.46533203125, 0.95947265625, 1.5361328125, 1.208984375, 0.21435546875, 0.814453125, 2.1796875, 0.537109375, 1.5078125, 2.171875, -0.662109375, -0.6826171875, 1.5517578125, -0.013427734375, 2.36328125, -2.0234375, 1.3046875, 0.69140625, -0.6689453125, -2.318359375, 1.05859375, -0.1796875, -2.259765625, 0.304931640625, -0.5712890625, -0.028564453125, 1.7353515625, -0.3876953125, -2.45703125, 0.252685546875, -1.3388671875, -2.90625, -1.904296875, -0.8505859375, -1.677734375, -0.14501953125, -1.978515625, 0.27783203125, 0.5185546875, 1.30078125, 0.072998046875, -1.658203125, 0.541015625, -0.2978515625, 0.033203125, -0.49365234375, -0.93359375, 0.2880859375, 1.130859375, -0.5380859375, -1.4736328125, -1.2666015625, 0.7685546875, -0.72900390625, -1.501953125, -0.740234375, -1.109375, 1.5537109375, 2.90234375, -0.38818359375, -0.787109375, 1.4814453125, 1.642578125, -0.03369140625, -0.89599609375, -1.26171875, -1.6103515625, 1.6103515625, -3.0546875, 1.7099609375, 1.154296875, -1.4345703125, 0.58984375, -1.23046875, 1.671875, -2.96875, 0.1571044921875, 0.5234375, -1.001953125, -0.4140625, -0.317626953125, 0.33984375, 3.45703125, -1.439453125, 0.427490234375, 2.0078125, 0.43701171875, -0.19873046875, 0.147216796875, 0.385009765625, 0.35400390625, 0.1103515625, 0.7392578125, -0.05303955078125, 4.83203125, -0.05047607421875, -0.71435546875, 1.763671875, 1.0478515625, 1.6796875, 0.18603515625, -1.7177734375, -0.6044921875, 1.1875, -1.337890625, 0.76318359375, 0.75732421875, -1.2890625, -2.13671875, 0.7880859375, -0.41357421875, -0.4951171875, 1.53125, -2.91796875, -1.5263671875, 0.859375, -0.5361328125, 2.291015625, 0.1552734375, 0.55322265625, -0.42138671875, -0.0086669921875, -2.470703125, 0.188232421875, 1.29296875, 1.0712890625, 1.2314453125, 2.98828125, -1.998046875, 1.84765625, -1.46484375, -0.2431640625, -1.5087890625, 2.60546875, -0.708984375, -0.5517578125, -0.5537109375, 1.6220703125, -1.103515625, -0.18798828125, -1.04296875, 1.9365234375, 0.439697265625, 0.08837890625, 1.962890625, -2.154296875, 1.767578125, -0.481689453125, 0.02294921875, 0.16943359375, 0.8076171875, -0.5361328125, -1.390625, -3.158203125, -0.43505859375, -0.69677734375, 1.0927734375, -2.51171875, -1.587890625, 1.3134765625, -0.669921875, 0.304443359375, -1.533203125, -0.55908203125, 1.017578125, 0.309814453125, -0.004150390625, 1.888671875, -0.76708984375, 0.29296875, 1.427734375, 0.136474609375, 0.5830078125, 0.473388671875, -0.91796875, 0.8916015625, -0.12939453125, 3.36328125, -2.666015625, -2.61328125, 4.1875, 2.748046875, 0.312255859375, 0.60400390625, -1.13671875, -0.78271484375, -0.0927734375, -0.74462890625, 1.9033203125, -0.19873046875, 2.4375, -0.67626953125, -1.1689453125, -0.6259765625, 1.22265625, -2.267578125, 1.041015625, 0.25244140625, 1.6865234375, 0.908203125, -0.20654296875, 0.1259765625, -0.076171875, 0.148193359375, -0.80224609375, -0.7763671875, -0.58203125, 1.6484375, 0.93603515625, 1.138671875, 2.70703125, 1.208984375, 2.345703125, 0.58349609375, 0.59521484375, -0.7626953125, 2.34375, 1.4921875, -0.3291015625, 1.1669921875, 0.685546875, -0.8935546875, 1.6533203125, 2.396484375, -3.072265625, -1.62890625, 1.4091796875, 1.423828125, 1.4169921875, 0.2099609375, -0.328125, 0.178955078125, -2.1171875, 0.99755859375, 0.4736328125, -0.8525390625, -0.054443359375, -1.42578125, 3.64453125, -1.5234375, 1.96484375, -1.302734375, -0.76953125, -0.5341796875, 0.05517578125, -3.515625, 1.189453125, 0.76171875, 0.49560546875, -0.31201171875, 1.341796875, -0.9296875, -1.1650390625, 0.8935546875, -0.20556640625, 3.390625, 2.35546875, -1.2431640625, 0.2578125, -0.333251953125, -0.330078125, -0.1416015625, -1.4267578125, -0.82861328125, -2.50390625, 1.0, -3.953125, -1.572265625, 0.30615234375, 1.732421875, 0.51220703125, 0.50537109375, -0.302734375, -0.36572265625, -2.138671875, 0.54736328125, 0.001953125, -0.80029296875, -1.833984375, 2.046875, -1.068359375, 0.74609375, -0.9248046875, -2.669921875, 3.021484375, -1.763671875, -1.740234375, 0.58642578125, 0.016265869140625, 0.5048828125, -1.2470703125, 0.10662841796875, -1.498046875, -0.87939453125, -0.1654052734375, 2.650390625, 1.1787109375, 1.87109375, -2.130859375, -0.53564453125, 1.66796875, -1.1298828125, -3.216796875, 2.400390625, 0.6591796875, -0.791015625, -0.6806640625, -2.19921875, -0.6513671875, 1.7451171875, -0.3916015625, -1.5625, 2.1015625, -0.73193359375, 0.2481689453125, 0.04754638671875, 1.0927734375, 0.091552734375, -0.2471923828125, 2.197265625, 1.357421875, 0.6650390625, -0.267822265625, -1.015625, 1.453125, -1.173828125, 0.26318359375, 0.18359375, -2.1015625, -2.1796875, 0.117919921875, -0.59375, 1.5322265625, -2.63671875, 0.4609375, -0.016845703125, -0.147216796875, 0.5908203125, -0.80029296875, -1.787109375, -0.3876953125, 0.98583984375, 1.474609375, 0.40771484375, 0.84521484375, -0.7060546875, 3.703125, 0.7919921875, -0.03790283203125, -2.369140625, 0.279296875, 0.1656494140625, -0.9248046875, 0.17236328125, -0.583984375, -0.251708984375, 0.8056640625, 0.556640625, 0.4814453125, -0.0576171875, -0.53515625, 2.99609375, -0.47265625, -0.09814453125, -0.4287109375, -0.90234375, 0.1513671875, 0.04150390625, 2.79296875, -1.212890625, -1.310546875, 0.92578125, -2.35546875, 0.294921875, 2.00390625, 0.9677734375, 0.83349609375, 0.90625, 3.03125, 0.3271484375, 0.8828125, 1.0556640625, -0.974609375, 0.69091796875, 1.7822265625, 0.47314453125, -1.173828125, 0.40576171875, -0.3125, -1.73046875, 0.73779296875, 0.14697265625, 1.8583984375, -0.7421875, -0.806640625, 1.15234375, 0.49951171875, 0.84375, 0.86376953125, 0.66796875, 1.814453125, -0.165771484375, -0.55517578125, -0.3779296875, -2.869140625, -0.720703125, -0.9453125, -2.060546875, 0.185791015625, 0.21728515625, -1.955078125, 2.076171875, 0.3037109375, 1.2587890625, 2.099609375, -1.7158203125, 3.30078125, 1.673828125, 0.720703125, 0.107421875, -0.264892578125, -0.3369140625, -0.1112060546875, -0.88720703125, -0.30224609375, -0.45947265625, 1.2421875, -1.5439453125, -0.1319580078125, -1.3193359375, -0.308349609375, 0.73974609375, 1.81640625, -0.2587890625, -2.443359375, -0.1484375, 0.1812744140625, 0.9638671875, 1.466796875, -0.703125, -3.453125, -1.3212890625, 1.1083984375, 0.7412109375, 0.71875, 0.83154296875, -0.65576171875, -1.748046875, 0.169921875, 0.5341796875, -0.5166015625, 0.11474609375, 1.189453125, 1.0419921875, -0.88623046875, 0.05419921875, 1.287109375, 1.439453125, -0.98486328125, 1.02734375, -1.0693359375, -0.54443359375, 2.72265625, 0.028076171875, -0.7900390625, 1.25, 4.515625, 0.98291015625, -0.0899658203125, -1.705078125, 1.80859375, -0.0728759765625, -0.132080078125, 1.376953125, -0.080322265625, -2.1015625, 1.5146484375, -0.102294921875, 0.9716796875, -1.69140625, -1.6669921875, 1.052734375, 0.55810546875, 1.4130859375, 1.0576171875, -0.3125, -0.759765625, -0.1328125, 0.193359375, 0.02734375, -0.9248046875, -0.49462890625, 1.21484375, -1.39453125, -0.1201171875, -0.390869140625, -2.44921875, 1.4248046875, -0.5068359375, 1.4130859375, -0.95458984375, 1.947265625, -0.8515625, -0.4013671875, -2.9375, -4.0078125, 0.9638671875, -1.177734375, 3.1953125, -2.76953125, -0.97021484375, -2.23046875, 2.013671875, 0.6884765625, -2.64453125, 1.5703125, -3.046875, 0.6787109375, -2.078125, 0.07666015625, -0.0098876953125, 0.07244873046875, 1.470703125, 0.1578369140625, -0.6474609375, 0.53125, -2.517578125, -0.95458984375, 0.982421875, -0.142578125, -0.41162109375, 1.1669921875, 0.8662109375, -0.5634765625, 1.2685546875, 0.314208984375, -0.70068359375, -1.083984375, -0.0654296875, 1.751953125, -1.4931640625, -0.35009765625, -2.759765625, 1.1640625, -0.03564453125, -0.27587890625, 2.3828125, 1.7626953125, -1.609375, -0.07470703125, 1.654296875, -0.0379638671875, -0.469482421875, -0.6298828125, 0.3095703125, -1.1162109375, 0.40966796875, 1.666015625, 0.94287109375, -1.244140625, -0.7802734375, -0.22265625, 0.83544921875, -0.7548828125, 1.51953125, -2.552734375, -0.61962890625, -0.2403564453125, 0.9501953125, -0.0843505859375, 0.177978515625, -1.8662109375, -2.08203125, -3.74609375, -0.4931640625, -0.8681640625, -0.8251953125, 1.193359375, 0.0234375, 0.292236328125, -0.87060546875, -0.08172607421875, -1.458984375, 1.931640625, 2.46484375, 0.7021484375, 0.888671875, 0.267578125, 1.294921875, 0.3193359375, -1.6669921875, 0.4150390625, -0.107421875, -0.736328125, 0.6826171875, 0.5263671875, 0.74365234375, -0.9482421875, 1.6279296875, 1.23828125, 1.40625, -1.6953125, 0.279296875, 0.0107421875, -0.20166015625, 1.8896484375, -0.9287109375, 1.0517578125, -0.39306640625, -0.1796875, -0.46337890625, 0.74755859375, -1.76953125, 0.26513671875, 0.96044921875, -0.4814453125, -0.411865234375, -2.10546875, 0.97607421875, -0.72998046875, -0.03662109375, -0.216796875, -2.60546875, -0.194091796875, -0.42822265625, 1.359375, 0.050048828125, -1.240234375, -1.5302734375, -0.880859375, -1.990234375, -0.248046875, 1.6953125, 1.349609375, 0.57080078125, 1.4375, 0.93017578125, 0.369873046875, -0.9228515625, 1.32421875, 0.10595703125, -1.537109375, 4.1328125, 1.66796875, -0.29541015625, -0.88232421875, -0.56982421875, -0.89892578125, 0.233154296875, 1.18359375, 0.125, 0.020751953125, -1.4169921875, -1.3427734375, -0.34814453125, 1.1484375, -2.708984375, 1.689453125, -0.98974609375, -1.6318359375, -1.6298828125, -1.404296875, 1.431640625, 1.46484375, 1.5693359375, 1.0361328125, 1.2373046875, 0.12255859375, 1.47265625, 0.53125, -0.54296875, -1.6552734375, 0.26953125, -1.27734375, 1.7705078125, 1.248046875, 0.16162109375, -1.755859375, 0.29541015625, 0.57177734375, 2.21484375, -0.37451171875, -1.2333984375, 1.2685546875, -0.2008056640625, 1.7158203125, 3.111328125, 2.08203125, 1.2900390625, -0.93701171875, 2.546875, -1.0478515625, -0.34228515625, -0.9052734375, -0.318115234375, -0.95947265625, -0.85693359375, 2.51953125, -1.0869140625, -2.53125, -0.380859375, -0.9736328125, 1.267578125, 2.01171875, -0.015625, 1.9375, -0.55126953125, 0.78662109375, 0.83447265625, -0.60498046875, -1.517578125, -0.1884765625, 0.654296875, -0.1669921875, 2.375, 1.7294921875, 2.337890625, -3.39453125, 1.2265625, -0.3173828125, -0.7880859375, 0.654296875, 0.583984375, 1.5966796875, 0.25341796875, -0.4462890625, -1.0595703125, -3.67578125, -1.154296875, -1.970703125, 0.2509765625, -1.130859375, 0.65869140625, 0.62744140625, -1.1171875, -1.2060546875, 1.208984375, -1.201171875, -1.3935546875, 0.7333984375, 0.951171875, -1.3740234375, 0.14013671875, -1.28515625, -1.271484375, 1.1328125, 0.9052734375, -2.046875, -0.43603515625, -1.349609375, 0.58642578125, 0.91162109375, -1.0087890625, 0.32177734375, 0.8466796875, -0.12744140625, 0.1004638671875, 0.29833984375, -0.01611328125, -0.489013671875, 0.8837890625, -1.2080078125, 0.00830078125, 1.599609375, -0.365966796875, -0.98876953125, -1.611328125, 1.7978515625, 0.0224609375, -1.658203125, -1.1162109375, 1.3056640625, 0.6708984375, 1.638671875, -0.8505859375, 1.5, -0.50830078125, -0.65966796875, -0.0667724609375, 4.796875, -0.61572265625, -0.375244140625, 0.908203125, 0.337646484375, -0.445556640625, -3.474609375, -0.1275634765625, 0.044189453125, 0.5771484375, 0.65576171875, 0.69384765625, 1.4208984375, -1.0234375, 0.7353515625, 0.97705078125, 0.58642578125, 0.57421875, -1.25, -1.263671875, 1.2578125, -1.044921875, 1.0224609375, -0.73779296875, 0.17138671875, 1.12890625, 0.57177734375, 0.2548828125, 0.107421875, 0.6005859375, -0.56640625, 4.1640625, 1.6689453125, -0.3837890625, 1.890625, -0.78125, -0.7001953125, 1.2705078125, -0.8017578125, -0.0380859375, -1.1875, -0.8369140625, -0.6826171875, 0.57666015625, -3.046875, -0.67529296875, 1.615234375, 0.416015625, -0.93603515625, 0.369140625, -0.496337890625, -0.626953125, -0.227783203125, -1.001953125, 1.236328125, -0.8359375, -1.765625, 0.01513671875, 1.517578125, -0.7705078125, -0.40966796875, -0.53271484375, -2.646484375, 2.28125, 0.029296875, 0.83935546875, 0.39794921875, 0.0263671875, 0.099609375, -0.001220703125, -2.001953125, -0.03125, 2.322265625, 0.904296875, 0.0322265625, 0.89013671875, -1.06640625, -0.60791015625, -0.0693359375, -2.296875, -0.00244140625, -1.7880859375, 0.2296142578125, -0.048828125, 0.962890625, -0.2255859375, -0.58935546875, -0.06884765625, 0.399658203125, -0.493896484375, 2.671875, -0.41357421875, -0.3291015625, -0.07080078125, 1.021484375, 2.130859375, -0.2332763671875, -0.890625, 0.60546875, 0.0712890625, 0.3388671875, 0.8125, 0.11328125, 1.38671875, 1.603515625, -0.05596923828125, -0.7021484375, -2.29296875, 1.2890625, -2.47265625, -1.3388671875, 1.146484375, -3.26171875, -0.724609375, -1.8525390625, 0.7158203125, 0.076904296875, 0.6611328125, -1.8447265625, -0.853515625, 0.05859375, 2.515625, -2.056640625, 0.458984375, 1.697265625, -3.44921875, 2.107421875, -2.33984375, 0.3974609375, -0.537109375, 1.998046875, -1.587890625, 1.7568359375, -0.69580078125, 1.9140625, -0.2327880859375, -1.1865234375, 0.6708984375, 0.705078125, -1.109375, -3.060546875, -0.353515625, 1.9501953125, 0.4609375, -0.7138671875, -0.1435546875, 1.3125, 1.466796875, 2.46875, 0.251708984375, 2.06640625, 1.408203125, 0.52734375, 0.10546875, 0.06494140625, -1.8359375, 1.8603515625, -2.45703125, 0.88720703125, -0.134765625, 2.166015625, 1.9404296875, -0.82568359375, 1.9892578125, 0.29931640625, -1.3759765625, -2.48828125, -1.05859375, 3.880859375, -1.46875, 2.2421875, 0.1884765625, -0.9287109375, -1.078125, 0.269775390625, 0.953125, 1.88671875, 0.685546875, -0.348876953125, -2.337890625, -0.7705078125, 0.409912109375, -0.38671875, -0.39208984375, 1.833984375, 0.83935546875, 1.0419921875, -0.32568359375, -2.26171875, 0.36279296875, -0.86572265625, -0.176513671875, -0.331298828125, -0.61474609375, 2.142578125, 0.270263671875, 0.7294921875, 1.705078125, -0.64501953125, 0.2958984375, 0.7646484375, -1.2294921875, 0.3056640625, 0.625, 0.3369140625, -1.4736328125, -1.857421875, 0.61376953125, -1.978515625, -0.2939453125, -0.8505859375, 1.34765625, 0.45263671875, 0.4140625, -1.333984375, -1.8896484375, 2.3203125, -2.21875, -0.6708984375, -0.236328125, 1.234375, -0.35546875, -0.646484375, -1.6474609375, 0.1611328125, 0.7841796875, 1.130859375, -0.335205078125, 0.322265625, -0.36572265625, -0.105224609375, 0.6494140625, -0.7998046875, 0.8681640625, 0.0245819091796875, -1.353515625, -0.2626953125, 2.2109375, 0.5009765625, -1.6201171875, 2.8515625, 2.392578125, -1.2294921875, 1.224609375, -1.20703125, -1.419921875, 1.3173828125, -0.1494140625, 0.23193359375, -0.32373046875, 0.9599609375, -0.00634765625, 0.1622314453125, -0.420166015625, 2.01953125, 0.094482421875, -0.66943359375, -0.107421875, -1.2421875, -2.15234375, 1.455078125, 1.1748046875, 0.4921875, -0.5029296875, -0.13671875, 0.642578125, 1.2705078125, -0.130126953125, 1.373046875, 3.19140625, 1.5009765625, 1.0615234375, -2.24609375, -0.265625, 0.383544921875, 0.0850830078125, 1.0888671875, 0.9619140625, 0.77685546875, -0.53759765625, -2.6796875, 1.1396484375, -0.71142578125, 1.65625, -0.7021484375, 1.42578125, 1.736328125, -2.8984375, -0.9443359375, 1.5810546875, 0.5078125, 1.2255859375, 0.28662109375, -0.3310546875, -1.1240234375, 1.298828125, 1.025390625, -0.72607421875, 0.01513671875, -1.41015625, 1.283203125, 0.22607421875, 2.17578125, -0.79541015625, -0.00146484375, -0.7177734375, 0.095947265625, -1.767578125, -1.333984375, -0.572265625, -4.01953125, 0.7119140625, -0.340576171875, 1.375, 0.64697265625, -0.6279296875, 0.76123046875, 1.615234375, -1.27734375, 1.50390625, 2.8125, -0.61279296875, -2.02734375, -1.6591796875, 2.427734375, 2.66796875, -0.8271484375, 0.379638671875, -1.693359375, -1.58203125, 1.03125, 2.3359375, 1.6240234375, 1.3974609375, -0.10205078125, -0.402587890625, 0.1123046875, -1.6630859375, 0.474609375, -0.23193359375, 0.9384765625, 1.9990234375, 1.1875, -0.4384765625, -0.20849609375, -1.3203125, -1.373046875, -0.1572265625, -1.01953125, 1.318359375, 1.5595703125, -0.4482421875, 1.0390625, 0.61083984375, -1.83984375, -1.0478515625, -0.71435546875, 0.115478515625, 1.2734375, 1.6328125, -0.313232421875, -0.015869140625, -1.12890625, -2.98046875, -1.677734375, -1.26953125, -0.6025390625, -0.40283203125, 0.2093505859375, -0.94873046875, -1.048828125, -1.548828125, -1.8798828125, 1.5693359375, -0.114501953125, -1.1728515625, 1.8564453125, 0.8515625, 0.91455078125, 0.70947265625, -1.2177734375, -0.0703125, -0.826171875, -0.4501953125, 0.0595703125, -0.69921875, -0.81494140625, -3.703125, 1.08984375, -0.147705078125, 0.66259765625, -0.0623779296875, -0.9462890625, 1.09765625, 0.9072265625, -1.3330078125, 1.82421875, -1.2724609375, -1.4765625, -1.0380859375, 1.048828125, -0.0914306640625, -0.245849609375, 0.7900390625, -3.66796875, 1.212890625, 1.20703125, 1.1845703125, 1.005859375, 1.8603515625, 0.47265625, 0.8720703125, -1.61328125, -0.26806640625, -1.609375, 0.791015625, -1.4248046875, -1.09765625, 0.666015625, -2.451171875, -1.552734375, 0.162841796875, 0.509765625, 0.7431640625, 1.7587890625, -1.75390625, -1.8681640625, -0.82470703125, 0.0274658203125, 0.6064453125, -0.32568359375, -3.12890625, -1.1708984375, 1.3623046875, 1.6591796875, -1.578125, 1.0029296875, -2.07421875, 1.9951171875, 0.837890625, -2.75, 0.199951171875, 0.33544921875, 0.2666015625, 0.2138671875, -0.2113037109375, -3.828125, -0.6494140625, -0.1361083984375, -0.54736328125, -0.248291015625, 1.01953125, -1.154296875, -1.216796875, 0.92236328125, 1.5810546875, 0.3251953125, -0.5234375, -0.2666015625, 2.82421875, -2.119140625, -0.05120849609375, -2.796875, -1.3857421875, 0.6083984375, -0.3984375, -0.54638671875, -1.244140625, 0.021697998046875, 0.3056640625, -2.251953125, -0.072509765625, -0.220947265625, 0.86572265625, -0.46142578125, -1.0234375, 2.28125, 0.160400390625, -0.080078125, 1.583984375, -2.814453125, 2.05078125, 0.2978515625, 0.4423828125, -0.88525390625, 0.7841796875, 1.583984375, 1.029296875, 2.421875, 0.583984375, 1.1455078125, -0.133544921875, 2.755859375, 0.8583984375, -0.66796875, -0.56884765625, -0.2301025390625, -0.673828125, 0.501953125, -0.58740234375, 1.0234375, -1.482421875, 0.8994140625, 0.443359375, -0.90625, -0.130615234375, -0.7685546875, 0.130859375, -0.10595703125, -0.56640625, 0.310791015625, 3.15234375, 1.0009765625, -1.525390625, -0.79736328125, 0.60009765625, -0.79541015625, 0.3876953125, 2.0625, 0.4306640625, -0.0626220703125, -0.7783203125, 2.296875, 1.0810546875, -2.32421875, 0.89453125, -3.216796875, -2.109375, 0.5830078125, -0.8818359375, -1.857421875, -0.95947265625, 0.4130859375, -0.529296875, 0.464111328125, 0.96533203125, -0.36328125, -1.759765625, 0.1162109375, 0.6181640625, -1.5029296875, 1.9677734375, 0.0278167724609375, 1.537109375, -0.458984375, 0.18603515625, -3.81640625, 2.130859375, 1.4423828125, -2.3984375, 1.4697265625, 1.19140625, -2.71484375, -2.326171875, 2.408203125, -1.5693359375, -0.990234375, 0.921875, 0.196533203125, -1.05859375, -0.521484375, 0.64794921875, 1.361328125, 0.379638671875, 1.34375, 0.3681640625, -0.9912109375, 1.6689453125, 0.4326171875, 1.09375, -2.7109375, 1.376953125, 1.61328125, -0.6474609375, -1.2255859375, -1.2734375, -1.259765625, 0.171875, 0.7783203125, 1.8984375, -1.6044921875, -1.138671875, 2.267578125, -0.06591796875, 0.46337890625, -2.1328125, -2.70703125, -0.09088134765625, 1.21875, 1.6796875, 0.63916015625, 0.9765625, 0.71044921875, -0.80029296875, -0.88330078125, -0.02783203125, 2.125, -1.5595703125, 2.0859375, 1.140625, 1.6025390625, 0.318603515625, 0.4111328125, -1.0751953125, -1.8955078125, 1.599609375, -1.576171875, 1.990234375, -2.1171875, 1.958984375, -0.2169189453125, 1.462890625, -1.53515625, -0.82373046875, -0.66796875, 1.2900390625, 0.4609375, 0.67724609375, 0.4833984375, 0.984375, -0.638671875, -1.0693359375, 1.345703125, -2.744140625, -0.4111328125, -1.068359375, -0.289794921875, 0.91259765625, -0.52099609375, 1.7666015625, 0.859375, -2.26953125, 2.77734375, 1.2060546875, -2.3359375, -0.81201171875, -0.384521484375, 0.9990234375, -0.52734375, 1.884765625, -0.179931640625, 1.55078125, -0.059814453125, 0.4677734375, 1.12890625, -0.150390625, 2.30859375, 1.947265625, -0.60546875, -2.265625, -1.2578125, 1.0791015625, 0.982421875, 0.83837890625, -1.7763671875, -0.78076171875, 0.88720703125, 1.0966796875, 2.615234375, 1.666015625, -3.0234375, 3.24609375, -2.08984375, 0.3671875, 2.90234375, -0.0615234375, 1.931640625, -1.359375, -1.154296875, -0.91552734375, -1.1767578125, 1.587890625, -2.095703125, 3.25390625, 0.07568359375, 2.525390625, 0.042236328125, 1.171875, 2.357421875, 0.111328125, -2.34765625, -3.4140625, 1.818359375, 1.1953125, 1.94140625, 1.3408203125, 1.0966796875, -0.368896484375, -0.80029296875, -1.509765625, 0.8671875, 2.431640625, 0.00262451171875, -0.3896484375, -1.275390625, 0.87109375, -2.591796875, 3.3203125, 1.90234375, 2.4453125, 2.47265625, -0.81005859375, -0.224609375, -2.265625, -0.302734375, -0.8935546875, -2.69140625, 1.6591796875, 1.7998046875, 0.24072265625, 0.349609375, -1.3701171875, 0.130859375, -2.40234375, -2.0625, 0.29150390625, 0.496337890625, -0.89404296875, 0.349853515625, 0.08544921875, -2.435546875, -0.1065673828125, -1.1015625, -0.60302734375, -0.6669921875, -1.501953125, 1.87890625, 1.07421875, 0.3818359375, 0.000732421875, -0.62841796875, 2.37109375, -1.42578125, 1.421875, -1.333984375, -1.279296875, 0.91796875, -0.2091064453125, -3.00390625, 0.67138671875, -1.2900390625, -0.66015625, 1.6640625, 1.591796875, -0.9208984375, 0.591796875, -2.486328125, -0.79541015625, 0.409912109375, -0.533203125, -0.1806640625, -1.7080078125, -1.1552734375, 0.888671875, 0.142578125, 0.4462890625, -1.9765625, -0.11541748046875, 0.74609375, -0.13818359375, 0.292724609375, -1.12890625, 0.99560546875, 1.74609375, 0.66357421875, 0.51318359375, 0.8203125, 1.15625, -2.37890625, -1.7802734375, -0.0302734375, 2.94921875, -0.429443359375, -0.3876953125, 1.943359375, -2.46875, 0.18115234375, -0.35400390625, -2.857421875, -2.28515625, 0.7998046875, -0.45849609375, 0.55419921875, -0.9130859375, -2.857421875, -0.857421875, -0.06884765625, -0.9580078125, 1.486328125, -0.0484619140625, 0.869140625, 1.75, -0.71484375, 2.314453125, 0.5478515625, 1.3427734375, 0.845703125, 2.1015625, -0.384765625, -0.99609375, -2.330078125, -1.537109375, -0.765625, -1.166015625, 1.6484375, 0.4345703125, 0.262451171875, 1.0791015625, -0.1336669921875, -2.62109375, -0.67236328125, 0.99755859375, -0.974609375, 0.5380859375, 0.416015625, -2.314453125, -2.58984375, -1.8642578125, -3.259765625, 0.42919921875, 1.9609375, 0.9931640625, 0.78515625, 2.5625, -0.35986328125, -1.1552734375, -1.25, -1.728515625, 1.3896484375, 0.56884765625, -1.974609375, -1.205078125, -1.41796875, 0.70556640625, 0.78466796875, 0.361083984375, -2.40625, -0.1588134765625, -0.63232421875, -1.095703125, -1.37109375, -1.9736328125, 1.0009765625, 1.2197265625, 1.2578125, 2.40234375, 0.521484375, -3.017578125, 2.966796875, 1.15625, -0.0908203125, 1.77734375, -1.0439453125, -0.3427734375, -1.734375, -1.1943359375, -0.7265625, -1.654296875, -1.197265625, -1.572265625, -1.078125, -1.2724609375, 0.428466796875, -0.4462890625, 0.63916015625, 0.25634765625, 0.330322265625, -0.9248046875, -0.053955078125, 0.385986328125, -0.418701171875, -0.447265625, 0.01123046875, -0.82421875, 0.64013671875, 1.9736328125, 3.14453125, 0.3134765625, 1.5673828125, 0.9228515625, 3.65234375, -0.1982421875, 2.857421875, 2.841796875, 1.25390625, 0.3798828125, -0.93017578125, -0.716796875, -1.8271484375, -0.7998046875, 1.6748046875, -0.626953125, -0.35107421875, 2.375, 0.35302734375, -0.226806640625, 1.3134765625, -0.328125, -0.146484375, 1.4921875, -0.4658203125, -0.7490234375, 0.03125, 2.1953125, -0.1708984375, -0.263427734375, -2.400390625, -0.136474609375, -0.8212890625, -0.705078125, 1.1025390625, -0.93212890625, 2.22265625, 2.609375, -0.5537109375, -0.4609375, 0.88720703125, 0.68212890625, -2.109375, 1.2158203125, 0.75048828125, 1.16796875, -2.9453125, -1.462890625, 0.5224609375, 0.430908203125, 0.08734130859375, 0.2109375, 0.94580078125, 1.9140625, 1.83203125, 0.06005859375, -0.18603515625, 1.48046875, 0.630859375, 0.69140625, 1.041015625, -2.30078125, 0.2470703125, 1.763671875, -2.52734375, 3.8984375, -2.060546875, 1.0234375, 1.20703125, 0.6337890625, 0.53125, 0.23486328125, 0.041015625, 1.2978515625, 1.470703125, 2.294921875, 0.27294921875, 0.21923828125, -1.490234375, 0.48828125, -0.27880859375, -0.67041015625, -0.341552734375, -0.2626953125, -0.485595703125, -2.140625, 1.68359375, -2.21484375, -0.68505859375, 0.720703125, -0.88671875, 0.98828125, -2.109375, -1.427734375, 2.4140625, -1.6015625, -0.5849609375, -1.4296875, -1.798828125, -1.8505859375, 0.8681640625, -1.08203125, -1.970703125, -1.2294921875, 1.80859375, 2.208984375, -0.10089111328125, 3.05078125, 1.365234375, -3.57421875, 0.69775390625, 0.6708984375, -0.1484375, -0.5986328125, 0.4609375, 1.44921875, -1.001953125, 0.66455078125, -2.29296875, 2.2734375, 1.9130859375, 0.263671875, 1.4296875, -0.181640625, -0.75830078125, 0.0726318359375, 1.927734375, -2.359375, -1.44140625, -1.8310546875, 0.1708984375, -0.1953125, 0.3720703125, -0.84912109375, -1.1796875, -1.66015625, 0.4345703125, -0.6474609375, 4.30078125, 0.83740234375, -0.2476806640625, -0.097900390625, 0.08697509765625, 0.68408203125, -1.734375, -0.77685546875, 1.4794921875, -2.0390625, 1.0859375, 0.11376953125, 1.134765625, -0.077392578125, 1.509765625, 1.87109375, 1.318359375, -2.849609375, -0.876953125, 2.09765625, -0.64208984375, -0.464599609375, -1.13671875, 0.32470703125, 0.8203125, -0.322265625, -1.890625, -0.44921875, 3.34765625, -1.2275390625, 1.51171875, 2.5234375, 2.6171875, 0.159912109375, 1.3037109375, 0.195556640625, 0.05029296875, -1.01953125, -0.375, -3.4921875, 1.3681640625, 0.18994140625, -0.8125, 0.93115234375, 1.62890625, 0.9970703125, -0.221923828125, -0.267578125, -0.86181640625, -2.796875, 0.44921875, 1.880859375, -0.78662109375, 0.6669921875, -1.705078125, 0.81005859375, 0.8671875, 1.1484375, -1.689453125, -1.37890625, 4.2578125, -1.015625, 0.5556640625, 0.22509765625, -1.2919921875, -2.400390625, 0.368408203125, -0.36181640625, -1.1103515625, 1.2314453125, 0.281005859375, -1.24609375, -0.10888671875, -0.81787109375, 0.63232421875, 0.630859375, 1.884765625, -2.376953125, -2.162109375, -1.3427734375, 0.30908203125, -0.54248046875, 0.49560546875, -1.4921875, 0.5068359375, -0.8720703125, 0.82275390625, 0.908203125, -0.51953125, 0.97412109375, -0.18310546875, 0.33544921875, 3.0625, -2.546875, -0.26416015625, 0.78515625, 1.154296875, -1.8173828125, 0.50537109375, 0.80078125, -2.5390625, -0.587890625, -1.31640625, -0.373046875, -1.1396484375, 0.64111328125, 2.767578125, -1.0595703125, -0.50537109375, -0.35888671875, -2.14453125, -0.093994140625, -1.138671875, -0.33447265625, 0.49951171875, -2.08203125, -0.857421875, -0.2470703125, 1.16796875, -1.126953125, 1.787109375, -1.2216796875, -0.5703125, -0.18310546875, -1.140625, -0.4755859375, -0.8994140625, -0.4306640625, -0.95703125, 0.45361328125, -1.263671875, 0.418701171875, -1.0849609375, 1.515625, 0.4638671875, -1.3486328125, 1.234375, -1.083984375, -3.298828125, -2.265625, -2.419921875, -1.3740234375, 1.369140625, 1.69921875, -2.57421875, 0.2958984375, -0.8642578125, 0.859375, -1.30078125, -0.9853515625, -1.634765625, -1.169921875, -1.080078125, -1.3828125, -1.044921875, -0.8369140625, 0.70068359375, -0.14404296875, 0.671875, -0.62744140625, 2.71875, 0.1015625, 2.345703125, -0.26220703125, 0.673828125, -0.052734375, 1.873046875, 0.00732421875, -1.55078125, -3.75390625, -0.07666015625, 3.134765625, 2.474609375, -1.5849609375, 0.0145263671875, 0.587890625, -1.2841796875, -0.4287109375, -1.3134765625, 1.9208984375, -0.102783203125, 0.29931640625, 0.8564453125, -0.1572265625, -1.0322265625, -0.43994140625, 2.111328125, -1.568359375, -0.429443359375, -0.5029296875, 0.2462158203125, -1.177734375, -1.154296875, -1.8662109375, -0.46337890625, -1.248046875, -0.38427734375, -0.53369140625, 0.21923828125, 1.548828125, -1.1826171875, -0.115234375, -0.0584716796875, 0.84375, 1.16015625, 1.599609375, 1.5078125, 1.3974609375, 1.0888671875, 3.0390625, 1.1455078125, -2.310546875, -0.86572265625, 0.7421875, 1.142578125, -0.005859375, 0.564453125, 0.057373046875, 1.017578125, -0.0980224609375, 1.458984375, -1.876953125, 0.8642578125, -0.076416015625, 1.197265625, -0.72412109375, -2.255859375, 0.697265625, 1.5771484375, -2.00390625, -1.25390625, 0.484375, 4.78125, 1.1142578125, -0.5595703125, 1.17578125, 2.712890625, -0.2509765625, 2.296875, -0.0927734375, 1.474609375, -1.8095703125, 0.306640625, 1.98046875, 0.35302734375, 0.4267578125, 2.88671875, 0.39892578125, -0.6611328125, 0.431640625, 1.1357421875, 0.4912109375, 0.0166015625, -0.9619140625, 0.96142578125, -1.8525390625, -2.029296875, 1.0517578125, -0.2939453125, 1.53515625, 1.1533203125, 0.890625, -0.29248046875, 2.16796875, 1.2802734375, 0.7568359375, -0.87646484375, -0.70947265625, -1.341796875, -1.2421875, -0.16259765625, 0.058349609375, -0.31884765625, 0.1044921875, -0.44970703125, 1.166015625, -1.392578125, 0.29541015625, 0.433349609375, 0.412109375, -0.170654296875, 0.7041015625, 1.8671875, 2.716796875, -0.53662109375, -1.8154296875, 2.669921875, -0.7236328125, 0.9755859375, -0.80078125, -1.978515625, -0.032958984375, 0.88525390625, 1.078125, -1.00390625, 0.2177734375, 3.06640625, 1.5146484375, -1.810546875, -0.073974609375, 0.5380859375, -1.009765625, -1.041015625, 0.13720703125, 0.049560546875, 1.8427734375, 0.26953125, -1.5751953125, -0.2203369140625, -0.5732421875, -0.17333984375, -3.0, 0.884765625, 1.2490234375, 0.18408203125, 1.318359375, 0.07855224609375, -1.982421875, 2.140625, 0.98193359375, 2.15234375, -0.465576171875, -0.828125, -1.6435546875, -0.576171875, -2.1484375, -1.0498046875, -2.15234375, 0.59765625, -0.91064453125, 0.73291015625, -0.1279296875, 0.45361328125, 0.40283203125, 0.64453125, 0.1279296875, 1.5380859375, 0.092041015625, -0.4306640625, -0.71630859375, -1.4111328125, 0.1083984375, 0.14013671875, 0.76416015625, 1.373046875, 0.0400390625, -1.115234375, 1.8984375, 1.951171875, 1.9404296875, 0.037200927734375, 3.60546875, 1.326171875, 2.21484375, -1.7900390625, 0.708984375, 0.08203125, -4.6875, -0.032928466796875, 1.421875, -2.265625, -1.345703125, 1.0517578125, -0.376953125, 1.94921875, 1.578125, -2.08203125, -1.65625, -1.125, 1.3603515625, -1.76953125, -2.17578125, -1.61328125, 1.234375, -0.86962890625, -0.427490234375, -1.158203125, 0.74462890625, 0.3896484375, -0.5283203125, 1.787109375, 3.82421875, 0.6279296875, -1.0673828125, 2.619140625, 2.05859375, -3.662109375, 1.78125, -2.1171875, -1.8564453125, -0.2451171875, 1.623046875, 0.086181640625, -1.46875, -1.330078125, 0.5234375, 0.95263671875, 2.560546875, 0.3115234375, -0.0244140625, 1.943359375, 1.45703125, -0.8671875, 1.87109375, -0.040924072265625, 0.5849609375, 0.2490234375, 1.0009765625, -1.0341796875, 1.9765625, 1.5810546875, 0.53173828125, 0.96826171875, 1.4873046875, 0.53076171875, -0.222412109375, 1.28515625, -1.544921875, -0.31689453125, -2.75, 0.2646484375, -0.388916015625, 2.359375, -1.1044921875, -2.8359375, 1.298828125, 0.115478515625, -0.394775390625, 2.2734375, -0.28662109375, 1.9765625, 0.6787109375, -1.3359375, 1.8203125, -0.9306640625, 3.0703125, -2.126953125, 1.3232421875, -0.0245361328125, -0.232666015625, 0.1328125, 1.41796875, 1.501953125, 0.53076171875, 0.1533203125, 0.00048828125, -0.5166015625, 0.5078125, 0.39501953125, 0.751953125, -0.55419921875, 1.73828125, 0.38134765625, -3.12109375, -0.78125, -1.83203125, 1.884765625, 0.91015625, -0.234130859375, -0.8017578125, -1.6875, 0.81591796875, 3.0625, -2.05078125, -1.513671875, -1.931640625, 0.12939453125, -0.0799560546875, -2.146484375, -0.08349609375, -1.0859375, 2.021484375, -0.8291015625, -0.826171875, -1.3408203125, 1.43359375, 1.9150390625, -0.75, -1.009765625, 0.662109375, -2.44140625, 1.07421875, 2.3828125, -0.828125, -0.42626953125, -0.8310546875, 1.0380859375, 1.841796875, 0.8935546875, -2.326171875, -4.265625, -0.05322265625, -1.39453125, -0.5439453125, -0.52490234375, 0.025390625, -1.8720703125, 1.01171875, 0.140380859375, 0.1328125, -1.3505859375, -0.642578125, 1.85546875, 0.515625, 0.8642578125, -0.5029296875, -0.833984375, -0.64697265625, -0.38720703125, -2.18359375, 0.77734375, -1.06640625, 0.80322265625, 1.6171875, 1.2783203125, -1.2724609375, 0.849609375, -0.16943359375, -2.3359375, 2.02734375, -0.3935546875, 0.34423828125, 0.7060546875, 0.54296875, -1.25390625, -0.03759765625, -1.6416015625, 0.9873046875, -2.85546875, 0.5966796875, -1.0166015625, -0.223388671875, 1.6474609375, 1.4697265625, 1.05078125, 0.28515625, 1.51171875, 0.796875, 0.4208984375, -2.193359375, 1.9296875, 1.93359375, 0.75830078125, 1.857421875, -2.466796875, 1.537109375, -0.37353515625, -1.0546875, 0.0654296875, 0.36767578125, -1.0869140625, 2.58203125, 1.03125, -0.861328125, 0.463623046875, 0.5380859375, -1.365234375, -0.1878662109375, -2.212890625, -2.384765625, -1.3564453125, 0.9462890625, -2.314453125, -0.3271484375, -0.97900390625, 3.435546875, -0.28076171875, 0.1767578125, -0.857421875, 3.568359375, -0.1513671875, -0.25830078125, -1.9150390625, 1.2431640625, 0.07763671875, -1.6376953125, -0.8037109375, -1.5419921875, 0.334716796875, 1.703125, 1.4736328125, 0.51708984375, 1.572265625, 0.01220703125, 0.40576171875, -0.9775390625, -0.91259765625, 0.80029296875, 0.525390625, -0.5078125, 0.640625, -0.0224609375, 0.955078125, 0.845703125, 0.0029296875, 0.74462890625, -0.0142822265625, 1.92578125, 2.400390625, 0.91455078125, 2.81640625, 0.90673828125, -1.98046875, -1.6611328125, -0.4423828125, -1.7490234375, -2.1328125, -0.2568359375, -0.93896484375, -0.828125, 0.578125, -3.8515625, 2.4765625, 0.09814453125, -0.74609375, 0.366943359375, -0.458984375, -0.62890625, 1.181640625, -0.6943359375, -1.89453125, -0.3525390625, -1.1494140625, 0.67333984375, 1.2001953125, -1.314453125, -1.2958984375, 0.44384765625, -2.224609375, 1.8408203125, 0.335693359375, 1.31640625, 0.75927734375, -0.4462890625, 1.8486328125, 1.2314453125, -0.33642578125, 0.87939453125, 0.42431640625, -0.79931640625, 1.2666015625, -0.4462890625, 0.775390625, 2.30078125, -2.0859375, 0.103515625, -0.88671875, -1.671875, -1.0205078125, -0.814453125, 0.681640625, 2.095703125, 1.7626953125, 2.109375, 0.490234375, -0.62548828125, 2.482421875, 0.409912109375, -0.06060791015625, -0.332275390625, -1.3369140625, -0.86376953125, -0.345703125, -2.2265625, 1.0966796875, 1.052734375, 0.474853515625, -0.34716796875, 2.6796875, -1.28515625, 1.505859375, 1.6796875, 0.47998046875, -0.164306640625, -0.5400390625, -0.32763671875, 0.005859375, 0.048583984375, 0.958984375, -0.755859375, 0.0667724609375, 2.630859375, -1.16796875, -1.935546875, -1.115234375, -0.197265625, 0.2939453125, -0.56005859375, -0.463134765625, -0.01318359375, 3.96875, 0.158203125, -2.166015625, -0.7978515625, 2.7265625, -1.37109375, -2.79296875, 2.103515625, -1.5302734375, -0.90087890625, 2.29296875, 0.073486328125, -0.86669921875, -1.392578125, 0.03857421875, 1.48046875, -2.193359375, -1.53515625, 1.00390625, -0.048095703125, 0.748046875, 0.05499267578125, 0.67138671875, -1.4609375, 1.0283203125, -0.7822265625, 1.5439453125, 0.0869140625, 0.31787109375, 2.51953125, -1.029296875, -0.61962890625, 0.54638671875, 0.33740234375, -1.80859375, 1.2216796875, 0.70556640625, 0.90234375, 1.2861328125, -1.875, -1.388671875, -0.5908203125, 0.50439453125, -0.9228515625, -2.625, 1.28125, -2.009765625, 0.61572265625, -1.111328125, -3.234375, -1.234375, -0.080322265625, -1.9130859375, 1.900390625, -0.390380859375, -1.3115234375, 0.299560546875, 1.3369140625, -0.1044921875, 1.73828125, 3.693359375, -1.08203125, 0.29345703125, -1.392578125, 1.767578125, -0.5029296875, -2.423828125, -1.5810546875, -0.431640625, -1.240234375, 0.1754150390625, 0.84033203125, -1.8916015625, 0.7666015625, -1.775390625, -0.86376953125, -1.28515625, 0.97509765625, 1.845703125, 3.103515625, -0.09619140625, 2.810546875, 2.154296875, -0.88232421875, 2.373046875, -3.19140625, 1.013671875, 0.8681640625, -1.5830078125, 1.7958984375, -0.88720703125, 0.39990234375, 0.1064453125, 1.6181640625, 1.0302734375, 0.591796875, 0.9619140625, -0.73388671875, 2.263671875, -1.11328125, 0.9169921875, -0.0458984375, -3.5234375, -2.265625, -1.7734375, 2.01171875, 2.154296875, -0.40185546875, 0.6123046875, -0.5615234375, -0.3466796875, 0.35986328125, 0.38671875, 2.15625, 0.3564453125, 1.1943359375, 0.7119140625, -0.90478515625, 0.099365234375, 0.6318359375, -0.14111328125, -0.623046875, 0.96044921875, -0.7216796875, 0.95654296875, 0.3623046875, -0.64501953125, 1.2080078125, -0.99169921875, 2.357421875, 0.93310546875, 0.87109375, -0.65625, 0.071044921875, -0.609375, -0.6142578125, 0.853515625, -2.109375, 0.94482421875, 2.27734375, 0.12109375, 0.72900390625, 0.8955078125, -0.254150390625, -1.5478515625, 2.171875, 4.17578125, -0.45263671875, -3.294921875, -0.3046875, 0.392578125, -1.658203125, -2.83203125, 0.033447265625, -1.044921875, -1.068359375, 1.60546875, -1.2001953125, -1.404296875, -2.90234375, -1.5908203125, 0.47314453125, -1.37109375, -0.2421875, 0.9453125, 0.97607421875, -2.97265625, -1.09765625, -1.6572265625, 0.5751953125, -0.6689453125, -0.453857421875, 1.0986328125, -0.078857421875, 0.00311279296875, -0.19873046875, -0.79541015625, -0.83935546875, 1.2333984375, 2.265625, 1.595703125, 0.9189453125, 1.4921875, 0.751953125, -0.06982421875, 0.615234375, 0.85009765625, -0.533203125, 0.455078125, 1.080078125, 0.787109375, 0.87939453125, 0.1597900390625, -0.7255859375, 0.1728515625, -0.2490234375, -0.6865234375, -0.9443359375, 0.607421875, -0.6689453125, -0.2239990234375, -1.3447265625, -0.04412841796875, 1.2197265625, 0.46923828125, -3.3203125, -0.9345703125, 0.6201171875, 2.044921875, -0.107177734375, 0.2490234375, -2.38671875, -1.16015625, 1.0400390625, 0.9970703125, -0.224853515625, 0.0150146484375, -0.6572265625, 0.6572265625, 3.43359375, 1.232421875, -1.775390625, 0.9248046875, 0.3662109375, 2.0859375, 0.35400390625, 1.291015625, 2.73828125, -0.56689453125, -0.388427734375, 0.150146484375, 0.348388671875, 0.435546875, -1.7431640625, -1.65234375, -0.097900390625, 1.1962890625, -0.064208984375, 3.24609375, 0.810546875, 1.048828125, -0.0992431640625, 0.304443359375, -0.84375, -0.60693359375, 2.46875, 1.7041015625, -0.97705078125, 0.7646484375, -2.302734375, 0.4560546875, -1.154296875, 2.451171875, 0.44873046875, -1.3310546875, -1.0078125, 1.19140625, 0.84423828125, 1.9521484375, -3.6015625, -0.9541015625, -2.6171875, -0.373779296875, -1.9013671875, 0.9267578125, 0.931640625, 3.666015625, 0.8349609375, -0.93408203125, 0.40625, -1.4921875, -0.3671875, -0.833984375, -0.818359375, -0.93017578125, -0.2479248046875, -1.046875, -1.4482421875, -0.406982421875, -0.83203125, 1.298828125, 1.064453125, -0.1541748046875, -1.119140625, 0.204345703125, 1.447265625, 0.85546875, -0.900390625, -0.47314453125, 1.095703125, 0.225830078125, 1.77734375, 1.228515625, 0.0088653564453125, -0.591796875, -1.45703125, -0.65966796875, 2.498046875, -0.591796875, -0.2049560546875, -0.0732421875, 1.580078125, 1.095703125, 1.0283203125, 0.58544921875, -1.1044921875, 1.42578125, -0.266845703125, 1.0087890625, -1.625, -1.5234375, -0.37158203125, 2.109375, 0.16552734375, -1.533203125, -1.4462890625, -2.408203125, -1.02734375, -0.46533203125, 0.223876953125, 0.53173828125, -2.123046875, 0.4111328125, 0.8369140625, 0.787109375, -3.392578125, -2.3515625, 0.34619140625, -0.0322265625, -0.1181640625, 0.96875, -1.9716796875, -0.51025390625, -0.49755859375, 0.12890625, -0.0430908203125, 0.1500244140625, 0.07080078125, -0.99560546875, 0.28955078125, 2.126953125, 3.87890625, -0.3388671875, -2.04296875, -0.0693359375, 1.267578125, -1.185546875, -0.5263671875, 1.205078125, 0.61962890625, -0.0078125, 0.046875, -2.953125, 0.56982421875, -1.1318359375, 2.177734375, 0.646484375, 1.23828125, -0.7412109375, -2.1484375, -1.11328125, -1.697265625, 0.67626953125, 0.927734375, -1.224609375, -1.55078125, 0.9755859375, 1.15625, -1.0888671875, 1.5068359375, -0.480712890625, -0.7890625, 2.15234375, -0.0595703125, -1.09375, -0.9521484375, 0.55419921875, -3.455078125, 1.3486328125, 0.8427734375, 1.6064453125, 1.8271484375, 0.10400390625, -0.23046875, 0.407470703125, 1.7822265625, -3.025390625, -0.31005859375, -2.859375, 0.115234375, -1.099609375, -0.0537109375, 1.1611328125, 1.517578125, 0.06201171875, -0.9599609375, -0.158203125, 0.6982421875, 0.85400390625, 0.07318115234375, -0.39501953125, 0.40673828125, -0.422607421875, 0.222900390625, 0.494140625, -1.9423828125, 1.208984375, -0.32421875, -1.150390625, 2.34375, 3.646484375, 1.8681640625, -2.716796875, 0.5048828125, -0.96630859375, -2.076171875, -1.427734375, 0.9873046875, -0.21435546875, -1.64453125, -1.734375, -1.34375, -0.1484375, 2.20703125, -0.654296875, -3.255859375, 0.66064453125, 0.017578125, 1.02734375, -2.6640625, 0.067626953125, -0.751953125, -0.47900390625, -1.23046875, -3.064453125, 0.18408203125, -1.287109375, 2.68359375, -0.615234375, 0.44384765625, -0.56201171875, 1.1357421875, 1.736328125, -1.0322265625, 0.1611328125, 2.515625, -0.1351318359375, -0.00341796875, 0.70458984375, 0.99462890625, -1.8779296875, 0.274169921875, 1.9140625, -0.5712890625, -0.8662109375, 0.33203125, -2.216796875, 0.4072265625, 2.28515625, 0.9072265625, -1.0263671875, -2.3125, -0.3857421875, -1.8974609375, 0.6728515625, -2.3046875, 3.47265625, -1.0615234375, -1.5439453125, -2.052734375, -0.97216796875, 1.6767578125, 3.109375, -1.0712890625, -0.62109375, 0.9990234375, 1.16796875, 1.3349609375, -0.662109375, 0.953125, -2.466796875, 0.39404296875, 0.38427734375, -0.4228515625, -0.374267578125, -0.59521484375, -0.4228515625, 2.0390625, 1.7705078125, -1.005859375, -0.5244140625, -0.244140625, 1.6640625, -1.673828125, -0.9892578125, 0.66259765625, -1.4970703125, -1.9296875, 0.87646484375, 1.01171875, -3.39453125, 2.19921875, 1.0986328125, -1.0029296875, 1.3828125, -2.373046875, 0.83056640625, 0.63671875, 0.41552734375, -0.70654296875, 1.892578125, 1.1103515625, 0.97607421875, -3.134765625, 0.103515625, 0.6201171875, 0.5830078125, 0.23388671875, -0.5703125, 0.595703125, 0.9208984375, -0.07861328125, -0.204345703125, -0.685546875, -0.2421875, -0.448974609375, 1.818359375, -1.21484375, -1.22265625, -1.5087890625, -0.849609375, -2.92578125, -0.1494140625, -1.783203125, 2.29296875, 0.5908203125, -0.48046875, 0.55322265625, -0.07421875, 0.8046875, -0.002197265625, 0.92919921875, 0.0263671875, 1.962890625, 0.72705078125, 0.0218505859375, 0.88232421875, -0.96240234375, 1.021484375, 1.294921875, 0.796875, -0.3310546875, 1.2001953125, 2.904296875, 0.325927734375, 1.529296875, -0.74365234375, 0.384765625, 0.49755859375, -1.474609375, -0.70361328125, -0.892578125, -0.33447265625, 1.6591796875, -0.4375, -2.267578125, 0.94091796875, -1.2587890625, 0.38916015625, -0.404541015625, -2.07421875, -1.4892578125, 0.3251953125, -0.412353515625, 0.646484375, -1.150390625, -0.44921875, 1.703125, -0.52734375, 2.4609375, -1.556640625, 2.05078125, 0.4306640625, -0.1162109375, -1.2705078125, 0.1568603515625, -0.2802734375, 0.5263671875, 0.6767578125, -1.1484375, -1.103515625, -0.83154296875, -1.1123046875, -1.453125, 0.72998046875, 1.7236328125, 0.09661865234375, 0.64111328125, -0.47509765625, -0.497802734375, -0.308837890625, -1.3076171875, 2.8828125, 0.46923828125, -2.1796875, 0.66357421875, -1.3876953125, -0.888671875, -0.19384765625, 1.5244140625, 0.099609375, -1.5517578125, 1.283203125, -2.990234375, -2.205078125, 1.9609375, 1.169921875, 0.85546875, -2.71484375, -2.455078125, 2.31640625, -0.7880859375, -0.302490234375, 0.7265625, -0.17578125, 2.248046875, 1.244140625, 0.17578125, -2.88671875, 0.8583984375, -0.06591796875, 2.65234375, 0.63134765625, 2.228515625, -2.193359375, -1.3203125, -1.90234375, 2.54296875, 1.2109375, -1.55078125, 0.1866455078125, -0.71875, 0.3828125, 1.1767578125, 1.8154296875, -0.271484375, 0.5185546875, 0.56201171875, -0.238037109375, -0.67041015625, 0.0411376953125, -0.966796875, 0.341796875, 2.380859375, -0.62890625, -1.296875, 0.67333984375, -1.357421875, 2.3203125, 0.16162109375, -0.9541015625, 1.2734375, 0.7216796875, 1.392578125, -0.6962890625, 0.876953125, 2.00390625, 1.013671875, -0.05078125, -1.244140625, 0.7568359375, 1.701171875, -0.357177734375, 1.166015625, -2.69921875, 2.064453125, 0.34130859375, -2.701171875, 0.91357421875, -0.12109375, 0.408203125, -0.385986328125, -0.60546875, 0.490478515625, 1.1767578125, -0.939453125, -1.7783203125, -0.9892578125, 2.9140625, -0.00341796875, -1.82421875, 2.765625, -0.10302734375, 2.58203125, 0.5400390625, -0.1929931640625, -2.17578125, -3.111328125, 0.290771484375, -1.01171875, 2.724609375, 1.427734375, -2.34375, 1.1318359375, -1.896484375, -0.82666015625, 1.123046875, 2.88671875, -0.32568359375, 0.16845703125, -0.9755859375, 1.0908203125, -3.82421875, 1.634765625, 0.431640625, -0.334228515625, 2.08203125, 0.5791015625, -0.0068359375, -0.5849609375, -2.88671875, -1.185546875, -0.0224609375, -2.248046875, 0.64306640625, -0.030029296875, -0.984375, 0.93017578125, -1.404296875, 1.216796875, -0.64892578125, -0.54296875, -1.57421875, -0.3720703125, 2.44140625, -1.185546875, -1.439453125, 2.30078125, 0.075439453125, -1.130859375, -1.7470703125, 1.306640625, 0.3896484375, -1.890625, -1.84375, -0.5576171875, 2.0390625, 0.169677734375, -0.7138671875, -0.28271484375, 0.1884765625, 2.017578125, 0.6181640625, 1.79296875, -0.67138671875, -1.1796875, 1.525390625, -1.224609375, 2.36328125, -0.15673828125, -1.4384765625, -0.1239013671875, -0.69677734375, 1.08203125, 0.9970703125, 0.158203125, 0.50830078125, -1.005859375, -1.08984375, -0.568359375, 0.0938720703125, -0.076171875, 0.9404296875, 1.7265625, 0.0675048828125, -0.4423828125, -1.2685546875, -0.005859375, 0.02685546875, -0.46923828125, -2.080078125, -0.070556640625, -1.453125, -0.176513671875, 1.013671875, -0.81298828125, 0.68359375, 1.150390625, -1.123046875, -0.6328125, -0.069580078125, 0.48681640625, -0.52783203125, 0.63671875, 0.288818359375, 1.546875, -1.361328125, 0.0263671875, 2.671875, 0.21142578125, 0.7841796875, -0.1728515625, -0.60498046875, 1.3056640625, 0.09765625, 0.0352783203125, -0.65576171875, -0.64892578125, 0.439697265625, 2.322265625, -1.1240234375, -0.5712890625, -0.41064453125, 1.3134765625, -1.326171875, 1.185546875, 0.0782470703125, 0.25732421875, -1.357421875, -3.208984375, 1.59375, 0.49365234375, -2.39453125, 0.685546875, 1.283203125, 0.67578125, -1.728515625, 0.2294921875, -1.5009765625, -0.38037109375, -2.55078125, -1.130859375, -0.0341796875, -1.3896484375, 1.142578125, -4.03125, 0.491943359375, -1.6904296875, -0.994140625, -1.3154296875, 1.162109375, 3.16796875, -0.1317138671875, 0.89404296875, -3.21484375, 2.44921875, 1.259765625, 1.3115234375, -0.17919921875, 0.55322265625, 1.2744140625, -0.1234130859375, 1.92578125, -1.333984375, -2.42578125, 2.20703125, 0.99658203125, -1.181640625, -1.384765625, -0.220703125, 1.9736328125, 1.205078125, 2.80859375, -2.583984375, 2.416015625, 1.185546875, -0.76806640625, 3.529296875, -0.087158203125, 0.7060546875, -1.421875, 0.444091796875, 0.6259765625, 0.9921875, -1.62890625, -0.587890625, 1.638671875, -0.03271484375, 3.080078125, -0.095947265625, 2.263671875, -0.08203125, -2.54296875, 1.3232421875, 0.000438690185546875, -0.943359375, -1.3310546875, -3.072265625, -2.05859375, 0.09423828125, -0.671875, -0.74560546875, 0.2998046875, 2.162109375, -1.6279296875, -1.880859375, 2.3125, -0.60791015625, -0.328125, -2.712890625, -2.35546875, -0.7470703125, -0.0517578125, -0.234375, 1.4814453125, 1.53125, -1.6376953125, 0.4384765625, 0.76904296875, -0.99267578125, -0.431640625, 0.322265625, -0.4638671875, -1.884765625, -1.76953125, -0.0966796875, 0.304931640625, 0.9208984375, -1.623046875, 0.94921875, 0.23828125, 0.7470703125, -2.80859375, -1.6337890625, -1.4609375, -0.462890625, 0.7529296875, -0.93505859375, -0.234375, -2.408203125, 1.51953125, -2.0390625, -0.0830078125, -1.033203125, 0.3447265625, -0.302490234375, -0.1083984375, -0.47607421875, -0.265380859375, -2.609375, 0.7255859375, 0.775390625, 0.4755859375, -1.1884765625, 0.93310546875, 0.61474609375, 0.5126953125, -0.775390625, -1.7470703125, 0.1884765625, 0.6845703125, -0.9375, -3.21484375, 1.6845703125, -0.90185546875, 4.12890625, -0.82275390625, -0.085693359375, -0.6513671875, -0.87646484375, -0.7724609375, 1.515625, 0.837890625, -1.4892578125, 0.5146484375, 0.13720703125, 0.625, 0.3232421875, 0.2392578125, -0.6845703125, 1.4169921875, 0.8740234375, 3.078125, -3.82421875, -2.974609375, 0.80224609375, -0.86865234375, 0.2080078125, 1.5556640625, 1.541015625, 2.0546875, -0.1376953125, -2.53515625, 0.1689453125, 0.90283203125, -0.5439453125, -0.29248046875, -1.8759765625, 0.447021484375, -0.0849609375, 0.1881103515625, 1.283203125, -1.1220703125, -0.302001953125, -1.01171875, -0.54052734375, -0.15966796875, -2.80859375, 1.951171875, -1.03125, -1.44921875, -1.111328125, -0.310546875, -0.3115234375, 0.5537109375, -0.408203125, 1.7763671875, -1.48046875, -0.50830078125, -0.83837890625, -1.38671875, 1.681640625, 1.8896484375, -0.99462890625, 0.44482421875, 0.2666015625, -0.8955078125, -1.69140625, -0.625, -0.52783203125, -0.350830078125, -0.78125, 0.5703125, 3.3125, -3.328125, 1.734375, -1.48828125, 0.43505859375, -1.306640625, -1.931640625, -0.978515625, 0.31787109375, -1.0556640625, 0.26416015625, -1.150390625, -1.4462890625, 0.04833984375, -1.765625, -0.6806640625, 0.248046875, 0.8623046875, -2.013671875, 0.7216796875, -3.625, 1.8466796875, -1.703125, -0.43994140625, 0.20458984375, -0.5166015625, 1.265625, 0.1484375, 3.119140625, 0.541015625, 0.3232421875, -0.8134765625, 1.572265625, -0.7958984375, 1.654296875, -0.82421875, -1.3828125, 0.853515625, -1.087890625, -1.9599609375, -0.50830078125, -0.1767578125, -0.8212890625, 1.1328125, -1.220703125, -1.390625, 1.375, -1.021484375, -0.0299072265625, -0.5439453125, 1.7109375, 0.5, 1.962890625, -1.103515625, -0.5205078125, -1.0986328125, 0.4931640625, 2.169921875, -1.12109375, -1.52734375, -1.0771484375, -1.4521484375, -0.59375, -0.55126953125, 2.16015625, -2.625, -0.5791015625, 0.39697265625, -0.626953125, -0.30859375, 0.51171875, 0.90576171875, 1.3984375, 1.3828125, -0.49169921875, -0.734375, -2.13671875, -0.056884765625, 1.845703125, 0.17236328125, -0.08154296875, -1.5068359375, 1.77734375, 3.81640625, -0.51904296875, -2.06640625, -1.7470703125, -1.87109375, 1.455078125, 2.12890625, 0.95263671875, -0.556640625, 2.453125, 1.1123046875, 0.197265625, 1.79296875, -3.033203125, -1.748046875, 0.1444091796875, 0.302490234375, 1.0400390625, 0.65087890625, 0.258544921875, 1.32421875, 1.07421875, -0.38818359375, -0.654296875, 1.806640625, -0.9169921875, -0.947265625, -0.1630859375, -0.763671875, 0.2366943359375, -0.2958984375, -0.61962890625, -1.9287109375, -2.013671875, 0.43505859375, -0.043701171875, 0.1416015625, 0.6083984375, 0.3935546875, 1.951171875, 1.849609375, 2.4375, -0.1923828125, 1.3779296875, -2.416015625, -0.126953125, 0.04754638671875, -0.5439453125, 0.064697265625, 0.155029296875, -1.8486328125, -1.51171875, 1.2724609375, 1.642578125, 1.5390625, 2.625, 1.005859375, 0.92041015625, -0.3408203125, 0.7236328125, -0.794921875, 0.1455078125, 0.3916015625, -0.7607421875, -1.1357421875, -0.90625, 1.509765625, 0.96240234375, 0.92919921875, 0.97314453125, 3.078125, 0.232421875, -1.1728515625, -0.00537109375, 1.42578125, -0.60546875, -0.841796875, -1.36328125, 0.9326171875, -1.7109375, 0.72607421875, -0.31787109375, -0.151123046875, 1.6572265625, 0.859375, -0.4541015625, 0.319091796875, -2.189453125, 0.97021484375, -1.94140625, 0.07421875, 1.2685546875, 1.06640625, 3.09765625, 0.5224609375, -0.255859375, -1.05859375, -1.7255859375, -1.2421875, 0.9033203125, -1.5751953125, 0.61083984375, 1.4892578125, -1.57421875, -0.427734375, 1.55859375, 0.83251953125, 1.322265625, 3.75390625, 1.50390625, -0.46337890625, -0.6259765625, 1.64453125, 1.0791015625, 0.0458984375, 0.46826171875, -2.216796875, -1.2666015625, 1.5, -0.19775390625, -0.72900390625, -0.47412109375, 0.37890625, 0.70849609375, -1.341796875, 0.3115234375, -0.6787109375, -2.013671875, -0.425048828125, 0.8896484375, -1.00390625, 0.1337890625, 0.1083984375, -3.18359375, -0.64794921875, -2.41796875, 1.2578125, 1.140625, -1.650390625, -1.0927734375, -0.0703125, 1.6328125, 3.04296875, -0.56787109375, -0.451416015625, 0.5048828125, 1.19140625, 1.39453125, 2.18359375, -0.88916015625, -1.75, 0.88134765625, 1.3193359375, 1.470703125, -0.64453125, 1.3994140625, 0.62109375, 0.0135498046875, -1.666015625, 0.67431640625, -0.4833984375, -1.4150390625, 0.54931640625, 1.90234375, 1.625, 1.6865234375, -0.8857421875, 1.0751953125, 0.1552734375, 0.77978515625, 0.9814453125, 0.443603515625, 0.3125, 0.25, -0.751953125, 0.2138671875, -0.6845703125, -1.107421875, -0.43701171875, -0.379150390625, 1.6435546875, 1.4970703125, 0.74951171875, 0.98779296875, 1.615234375, 0.347412109375, -0.046875, -2.720703125, -0.3984375, -1.361328125, -0.3984375, 2.15625, -1.30859375, 0.2069091796875, -2.296875, 0.130859375, 1.16796875, 1.341796875, -0.634765625, -0.197265625, 1.5478515625, 0.4482421875, 0.64453125, 0.34033203125, -2.037109375, 1.615234375, 0.623046875, 0.02880859375, 2.720703125, -2.45703125, -1.48046875, -0.8603515625, 0.71044921875, 0.4462890625, -1.4833984375, -2.015625, 2.22265625, 0.38916015625, -2.236328125, -1.857421875, 1.75, -1.583984375, -1.5439453125, -1.08984375, 1.0048828125, 1.5654296875, 0.6484375, 0.089599609375, -0.04541015625, -0.6708984375, 0.451171875, -0.466796875, -2.2578125, 1.5009765625, 0.19921875, -2.486328125, 2.55859375, -0.4921875, 3.251953125, 0.12744140625, -1.5361328125, -0.35791015625, 4.921875, 0.15234375, -1.80078125, -0.9345703125, 0.1744384765625, 1.953125, -0.64990234375, -0.8046875, -1.169921875, 2.06640625, 0.76953125, -1.1630859375, 0.8779296875, 0.1668701171875, -1.5576171875, 0.1947021484375, -1.33203125, -0.474609375, -1.666015625, -0.916015625, -0.42529296875, 0.5068359375, -0.5869140625, -2.1328125, 0.19384765625, 1.4873046875, 0.478271484375, -1.392578125, 0.2626953125, -0.40869140625, 0.29638671875, -2.216796875, 0.859375, -0.54052734375, -0.56005859375, -0.7001953125, 0.9287109375, 2.17578125, 0.5, -2.421875, 0.476318359375, -0.325927734375, -1.240234375, 1.15234375, -0.2939453125, 0.178466796875, -0.4677734375, 2.072265625, -0.115234375, 0.84130859375, -1.052734375, 0.59814453125, -0.9169921875, -0.02734375, 0.79248046875, -1.4345703125, 1.626953125, 0.3232421875, -1.0966796875, 1.48828125, -1.5380859375, 0.276611328125, -0.411376953125, -1.0283203125, 1.4677734375, 3.9765625, -2.9375, -0.76904296875, 1.0537109375, -0.9326171875, -1.75, 0.9658203125, -0.0546875, 0.70751953125, 0.5908203125, -2.779296875, 0.521484375, 1.431640625, 0.763671875, -1.8505859375, -1.376953125, 0.88818359375, 0.119873046875, 1.947265625, 2.2421875, 1.53515625, 1.287109375, -0.21044921875, 1.00390625, 0.15380859375, -2.025390625, 1.365234375, 0.25341796875, -0.76318359375, -1.48828125, 2.548828125, 1.72265625, -0.9560546875, -0.461669921875, -1.654296875, 0.93994140625, -0.07470703125, -0.50390625, -1.794921875, -0.978515625, 0.349365234375, 2.9609375, -0.1220703125, -0.5771484375, 2.05859375, 0.292724609375, 0.071533203125, -1.923828125, -1.482421875, 1.0791015625, 0.56396484375, 0.385009765625, -0.484619140625, -0.435791015625, 0.1123046875, -1.6845703125, 0.51513671875, 2.03515625, 2.2890625, 1.3388671875, 1.455078125, -0.57958984375, -0.7470703125, 0.765625, 1.470703125, 1.12109375, 2.58203125, 1.9853515625, 0.196044921875, 0.72216796875, 1.3720703125, -1.00390625, 1.78515625, 0.00244140625, 0.68408203125, 0.380859375, -1.009765625, -0.322265625, 1.7060546875, -0.87353515625, 0.1787109375, 0.884765625, -1.6884765625, 1.6025390625, 1.7705078125, -1.5048828125, 2.11328125, -1.55859375, 0.132568359375, 1.333984375, 1.8369140625, 0.662109375, -0.5078125, 0.8955078125, 0.208251953125, -0.9580078125, -1.9814453125, 0.63916015625, -0.202880859375, -0.76806640625, -0.9951171875, -1.060546875, -0.219482421875, 1.9814453125, 0.498046875, 1.6669921875, -2.755859375, 0.779296875, -2.2265625, -1.03515625, 0.08673095703125, 2.90625, -1.4501953125, 0.689453125, 0.208984375, -0.97607421875, 1.1044921875, -1.650390625, -1.91015625, -1.650390625, 0.0732421875, 1.091796875, 0.45166015625, 0.2861328125, -1.5498046875, 0.03271484375, -1.4921875, -3.32421875, 2.45703125, -0.248779296875, -0.994140625, -2.4609375, 1.2685546875, 1.361328125, 0.91455078125, 0.1397705078125, -0.2548828125, -1.275390625, 1.548828125, 1.482421875, -2.29296875, -1.3623046875, 2.173828125, 3.2578125, -1.716796875, 0.38671875, -1.220703125, 1.642578125, -0.70751953125, 0.982421875, -0.199951171875, 1.9453125, -1.5478515625, -0.888671875, 1.1669921875, 2.2109375, -1.0576171875, -0.47900390625, -0.095947265625, 1.0625, -0.6142578125, -0.953125, 0.3095703125, -0.251953125, -0.6396484375, 2.39453125, 0.78125, 0.0693359375, 1.27734375, -0.48681640625, 3.017578125, -0.9931640625, -2.09765625, 2.109375, 0.33740234375, -0.345458984375, 0.29638671875, -0.072265625, 0.5458984375, 0.95361328125, 0.74267578125, -1.8203125, 1.59375, 0.322265625, -0.587890625, -2.826171875, 0.2252197265625, -1.703125, -1.33203125, -0.0587158203125, -0.216552734375, 1.283203125, -0.44287109375, -1.41796875, 0.544921875, -1.44140625, 2.841796875, 1.5009765625, -0.98974609375, 0.55126953125, -1.43359375, 3.671875, 1.12890625, 2.03515625, 2.875, -0.67822265625, -0.53515625, -0.7626953125, 1.822265625, -0.9453125, -0.2919921875, -1.6240234375, -0.83740234375, -1.466796875, 2.037109375, -0.892578125, 0.0994873046875, -0.07989501953125, -0.4697265625, 1.1142578125, -1.697265625, 2.427734375, 2.931640625, 0.0703125, -2.83984375, 0.849609375, 1.04296875, 0.8125, -0.6123046875, -1.2509765625, 0.8466796875, -0.71875, -1.728515625, -1.255859375, 2.19921875, 1.6005859375, 0.7197265625, -0.7578125, -2.205078125, -1.1181640625, -0.62890625, -1.09765625, 1.53515625, 0.7587890625, -0.220703125, 0.66064453125, -1.5830078125, 0.724609375, 0.9873046875, 2.251953125, 0.4658203125, -1.3984375, 0.3916015625, 1.2392578125, -0.50341796875, 1.8115234375, 0.48193359375, 2.0390625, -0.89013671875, 1.0380859375, 2.51171875, 0.654296875, 0.669921875, 1.3515625, -1.9189453125, -0.60302734375, -0.599609375, -0.6875, -0.1376953125, 0.9814453125, 1.779296875, -2.619140625, 0.87744140625, -0.444580078125, -1.5078125, -0.16015625, -1.03515625, -1.2744140625, 1.9970703125, -0.6064453125, 2.109375, 0.9482421875, 1.9638671875, -0.276123046875, -0.5517578125, 0.376953125, -1.5107421875, -1.826171875, -0.591796875, 0.416259765625, -0.419921875, 0.1666259765625, -2.1484375, -1.98046875, -1.220703125, 0.34326171875, -0.6650390625, -1.5673828125, 3.529296875, -0.2900390625, -0.169677734375, 0.8359375, 1.97265625, 0.76220703125, 0.046875, 0.2646484375, 1.6796875, 1.8134765625, 0.2001953125, -1.375, -1.197265625, -2.044921875, -1.98046875, -0.491455078125, -0.49951171875, -2.525390625, 1.724609375, -0.96826171875, -0.186279296875, 1.052734375, -0.5478515625, 1.8828125, 0.43603515625, 0.7841796875, 1.177734375, 0.5576171875, 2.62109375, -0.183349609375, -1.169921875, -1.2548828125, -0.865234375, -3.90234375, 2.46875, -1.890625, -0.332275390625, 0.9521484375, 0.65087890625, -0.41748046875, 2.28125, 0.82470703125, -0.216796875, -0.7724609375, 0.285888671875, -0.5, -2.91796875, 0.81591796875, -0.953125, 3.982421875, 0.0238037109375, 1.3115234375, 2.72265625, -1.205078125, -0.410400390625, 0.6142578125, -0.5546875, -0.8896484375, -1.119140625, -0.0941162109375, -2.734375, 1.83984375, 2.083984375, -3.470703125, -1.416015625, -0.97802734375, -0.251708984375, 0.0992431640625, 0.26513671875, -3.076171875, 1.3642578125, 0.20703125, -1.3701171875, -0.6162109375, 1.5087890625, 1.7744140625, -2.1796875, 0.6767578125, -0.28564453125, -0.348876953125, 2.09765625, 1.09765625, -1.1728515625, -1.1826171875, 2.07421875, 0.60693359375, 0.051513671875, 0.01318359375, 2.009765625, -0.845703125, 0.634765625, 0.39794921875, -0.1923828125, -1.1943359375, 4.8203125, 1.0224609375, -0.34228515625, 0.43115234375, -2.39453125, 1.685546875, -1.3271484375, 0.1904296875, -0.826171875, 1.701171875, -0.32080078125, -1.1875, -0.021484375, 0.0245361328125, -1.6455078125, 0.15283203125, -1.1904296875, -1.029296875, 1.25, 1.00390625, -1.2578125, 1.111328125, 0.2353515625, -2.96875, 1.9345703125, 1.1552734375, -0.71240234375, 2.515625, -0.54150390625, 0.76953125, 2.875, 0.051513671875, 0.619140625, 1.201171875, 1.4921875, -0.4384765625, 2.197265625, 1.06640625, 1.958984375, 1.271484375, -2.74609375, -2.37109375, -3.20703125, -0.7978515625, -2.310546875, -1.103515625, -1.6845703125, 0.92919921875, -0.496826171875, 0.044189453125, 0.77490234375, 0.88134765625, -2.533203125, 2.09765625, 2.087890625, 1.3095703125, -0.71923828125, -1.93359375, 0.0703125, 0.338623046875, 1.146484375, 0.85693359375, 1.12890625, -1.54296875, 0.85302734375, 1.4921875, 1.2119140625, 0.01513671875, -0.6103515625, 2.193359375, -0.626953125, 1.0615234375, 0.908203125, -0.40087890625, -2.91015625, 2.21875, -0.1209716796875, 1.359375, -1.2216796875, 0.1365966796875, -2.220703125, -2.0234375, -2.755859375, 1.51171875, 0.1298828125, -1.1240234375, 0.5771484375, -1.3798828125, 1.853515625, -1.443359375, -0.85009765625, -0.5947265625, 0.35546875, -0.78271484375, 1.8271484375, -0.65380859375, -1.619140625, 0.7783203125, -0.176513671875, -0.03662109375, -2.171875, -1.458984375, 2.06640625, 0.263671875, -2.3125, 1.0078125, 1.900390625, -0.90576171875, -0.117431640625, 1.6640625, 1.32421875, -0.321044921875, -1.3095703125, 1.1337890625, -0.270751953125, -1.2109375, 0.59814453125, -1.0537109375, 1.6591796875, 0.4423828125, -1.6767578125, 1.5888671875, -1.0361328125, -2.08203125, 1.87109375, -2.80859375, -1.7958984375, -0.626953125, -2.431640625, -0.31689453125, -0.7763671875, 2.66796875, 0.163330078125, 0.6806640625, 1.1083984375, -2.529296875, 0.18115234375, -2.15625, -0.37158203125, 0.366455078125, 0.716796875, -0.41552734375, 1.2236328125, -0.19140625, 2.09375, 1.205078125, -0.63720703125, 0.4482421875, -0.21337890625, 1.349609375, 0.53515625, 0.56103515625, 0.55712890625, -1.333984375, -0.82568359375, 1.3115234375, -1.7578125, -0.84375, -0.41552734375, -2.8671875, 1.41015625, -1.58203125, 1.27734375, -0.908203125, -0.939453125, -2.921875, 0.9443359375, 0.625, -1.880859375, 0.158935546875, -1.8349609375, 0.56494140625, -0.2890625, 1.185546875, 0.0906982421875, 1.80859375, 2.1875, -0.68212890625, 1.8564453125, 2.375, -1.19140625, -1.052734375, 0.94921875, 0.82666015625, 4.0234375, -3.275390625, -0.8466796875, 0.67431640625, -1.44921875, 2.134765625, -1.23046875, 1.970703125, 0.432373046875, 0.80908203125, -2.47265625, 1.80078125, 0.29736328125, 1.427734375, -0.35791015625, 0.468017578125, 0.8037109375, 0.93115234375, -0.496826171875, -0.92822265625, 1.9970703125, 1.884765625, -2.814453125, -0.080078125, 1.0263671875, 0.5234375, 1.501953125, 1.869140625, 0.1884765625, -4.73828125, 0.07080078125, 0.1357421875, 2.65625, 0.7841796875, -0.8876953125, 0.09375, -1.1611328125, 1.224609375, -0.9365234375, 1.203125, -0.107421875, -0.64013671875, 1.177734375, -0.294921875, 0.173583984375, 0.943359375, -0.1829833984375, 2.6015625, -0.41650390625, -1.2939453125, 1.7158203125, -0.57177734375, 2.701171875, -0.76513671875, 0.8935546875, 2.404296875, 1.0078125, 0.159912109375, -1.18359375, 2.509765625, -1.564453125, -0.4951171875, 0.39599609375, 0.81201171875, 0.50146484375, -1.908203125, -0.90087890625, -0.76318359375, -0.015625, -1.8603515625, 0.50146484375, -1.2158203125, 0.767578125, -1.091796875, 1.0693359375, -1.322265625, 2.66796875, 0.517578125, -0.8896484375, -1.568359375, 0.191162109375, -1.625, 1.81640625, 0.7294921875, 3.3125, -1.61328125, -0.67041015625, -0.5634765625, -1.83984375, -1.515625, 0.54150390625, -2.94140625, 1.2724609375, -2.45703125, -0.62451171875, -2.673828125, -2.259765625, -1.814453125, 0.7041015625, -1.37890625, -0.97265625, 1.712890625, -1.1201171875, 0.6298828125, -0.24853515625, -0.2646484375, -1.248046875, -1.224609375, -0.468505859375, 0.16650390625, 1.6875, -2.0859375, -1.5625, -2.41015625, -2.2578125, 2.27734375, -0.03515625, 1.3076171875, 0.3046875, -0.48681640625, 0.87646484375, 0.29833984375, -0.8359375, -0.38427734375, -0.1751708984375, -0.9853515625, -1.4033203125, 1.609375, -0.376708984375, -1.7421875, 0.244140625, -1.2744140625, 1.12109375, 1.96484375, 0.08544921875, 1.8056640625, -0.8671875, -3.12890625, -0.9638671875, -0.6005859375, 1.0146484375, -0.66455078125, 0.255859375, 0.078125, 0.7890625, 0.5947265625, -0.50244140625, 0.857421875, -0.7841796875, -1.9453125, 0.56396484375, -0.10980224609375, -0.919921875, 1.216796875, -1.6640625, -1.08203125, 0.084716796875, 1.76171875, -0.21484375, -1.595703125, 1.560546875, 0.94970703125, 2.359375, 1.4091796875, -1.2275390625, 2.025390625, 0.4482421875, -1.208984375, -1.84375, -3.4140625, 0.9130859375, 2.171875, -0.5234375, 0.4931640625, -3.07421875, 0.91845703125, 1.755859375, 0.93701171875, -2.69921875, 0.406494140625, -2.060546875, -1.2919921875, -0.7548828125, 0.09521484375, 0.91796875, -1.25, -0.53662109375, -0.55224609375, 0.095947265625, -2.3984375, -0.8828125, -0.2880859375, 0.75830078125, -0.26708984375, 1.8125, 1.619140625, 2.38671875, -1.5234375, -0.5361328125, -0.1365966796875, -0.8388671875, -0.6728515625, 3.2109375, -2.08984375, 1.177734375, 0.1380615234375, 0.701171875, 0.13671875, -0.5810546875, 1.2646484375, -0.3525390625, -0.91943359375, 3.4140625, -1.87890625, 0.767578125, 2.158203125, 0.171142578125, 0.92578125, -0.7890625, 0.671875, 2.2734375, -2.35546875, 1.08203125, -1.3935546875, 0.0479736328125, 0.93115234375, 0.12158203125, -1.3212890625, -2.30859375, -1.4208984375, 1.1318359375, -1.470703125, 1.935546875, 0.010498046875, -0.4775390625, -0.7275390625, 0.03692626953125, 1.201171875, 3.9296875, 1.4599609375, 0.353515625, 0.3193359375, 0.931640625, -0.2357177734375, -2.728515625, 0.415771484375, 1.76953125, 0.66162109375, -1.0244140625, 3.279296875, 1.462890625, -1.322265625, -0.1826171875, 0.02734375, 1.072265625, -0.7939453125, -2.126953125, -1.017578125, 0.46826171875, -0.2275390625, 0.222900390625, 0.33056640625, 1.013671875, 1.294921875, 1.423828125, -0.546875, -0.213134765625, 0.541015625, -0.436279296875, 0.70458984375, -0.04632568359375, -0.11962890625, 1.451171875, -0.439453125, 0.438720703125, 0.054534912109375, 0.84326171875, -0.37109375, 1.068359375, -0.990234375, 0.65478515625, 1.4638671875, -1.0703125, 0.38427734375, -0.1258544921875, -0.40478515625, 1.77734375, -0.69873046875, -0.6025390625, -0.71875, 2.0703125, 0.11419677734375, -3.244140625, -1.640625, -1.3125, -0.439208984375, 0.5537109375, 0.77587890625, -0.69140625, 2.2421875, 0.755859375, 0.3330078125, 2.748046875, -2.921875, 1.646484375, 1.712890625, 1.83984375, 1.53515625, 1.775390625, -0.88134765625, -2.46484375, 3.8046875, -1.376953125, -1.7275390625, -1.2763671875, -1.4296875, -1.0341796875, 0.7724609375, 1.767578125, 0.60205078125, -2.85546875, -0.994140625, -0.2308349609375, -1.0537109375, -0.615234375, 0.220458984375, 1.33984375, -2.37890625, -0.4365234375, 0.2398681640625, -0.02581787109375, 1.078125, -0.875, -1.060546875, -0.3076171875, 0.73876953125, 0.043914794921875, 0.5576171875, 1.083984375, -0.5263671875, 0.40185546875, -2.423828125, -5.515625, -0.8623046875, -2.05078125, 0.36962890625, -0.66259765625, 0.3916015625, 0.3203125, 0.365966796875, 0.478759765625, -2.595703125, -0.84326171875, 1.845703125, 2.419921875, -1.099609375, 0.018798828125, 1.5263671875, -0.1630859375, -2.416015625, -0.198486328125, 0.17626953125, -1.1123046875, 1.794921875, -0.2420654296875, 2.11328125, 1.318359375, 0.8271484375, -0.265625, -0.85546875, 1.376953125, 4.2265625, -0.65185546875, -0.2724609375, 1.6640625, -1.013671875, 0.88427734375, -1.3359375, -1.271484375, 0.90234375, -1.5205078125, -2.3984375, -0.278076171875, -0.67919921875, 1.75, -0.93505859375, 1.111328125, 0.25732421875, -1.0478515625, -0.22216796875, 3.1484375, -2.173828125, 1.111328125, -0.90625, 1.595703125, 2.3203125, 0.224609375, 0.5341796875, -1.1015625, -1.47265625, 0.5361328125, 0.86767578125, 2.421875, 0.3232421875, -2.033203125, 0.77734375, 1.32421875, -1.8125, -1.72265625, -0.88623046875, 0.841796875, -2.13671875, -0.34619140625, 0.2705078125, 2.892578125, -0.634765625, 2.236328125, -0.9560546875, 0.20458984375, 1.5625, -1.18359375, 0.015625, 0.52685546875, -0.133544921875, 0.23046875, -0.69140625, 0.52685546875, 1.1962890625, 0.04833984375, 0.62353515625, 0.87548828125, -0.3046875, -2.87890625, -0.021484375, -0.9833984375, 0.25634765625, 0.87939453125, 0.955078125, 0.68603515625, 0.9970703125, 1.3486328125, -0.389892578125, -0.4521484375, 0.7646484375, 0.465087890625, 0.6640625, 1.0166015625, -0.1285400390625, 1.8662109375, -0.099609375, 1.5888671875, 1.029296875, 0.3330078125, -0.7001953125, -0.82861328125, -3.07421875, -1.216796875, 0.3251953125, 1.955078125, -1.5263671875, -0.103515625, -1.859375, 0.49365234375, 1.951171875, 2.69921875, -0.9697265625, 1.5732421875, -0.58984375, -0.273193359375, -2.203125, 0.21142578125, -1.708984375, -0.5498046875, 1.71875, 1.6083984375, -0.6953125, 2.541015625, -0.12548828125, 0.193359375, -2.458984375, 0.5, 2.40625, 1.1181640625, -0.26806640625, -1.533203125, -1.751953125, 1.9453125, -1.142578125, 0.8359375, -0.5498046875, -1.0390625, -0.25927734375, -0.4404296875, 0.16943359375, -1.2431640625, 0.859375, -1.62109375, 0.8193359375, 0.11859130859375, 0.78125, -2.38671875, -0.73583984375, 2.2421875, 3.3515625, -1.8515625, 1.4111328125, -2.451171875, 1.33203125, 0.42626953125, -1.197265625, -1.73828125, 0.5771484375, 0.25146484375, 1.5439453125, 0.041015625, 0.65185546875, -2.7265625, 2.453125, 1.1513671875, -0.3232421875, 2.08203125, -0.8076171875, 1.56640625, -1.4443359375, 2.13671875, 2.1796875, -0.290283203125, -1.4833984375, -1.673828125, 0.045654296875, -2.1484375, -0.369140625, 1.197265625, 2.931640625, 0.548828125, 0.85107421875, -3.0390625, 2.541015625, 1.0361328125, 0.5869140625, -0.7548828125, 0.5869140625, 0.44873046875, -1.1728515625, -1.0439453125, -1.9892578125, -1.2861328125, 0.314453125, -1.03515625, 0.105712890625, 4.25, -0.18359375, -2.11328125, 1.255859375, -2.046875, -1.283203125, -0.13623046875, 1.00390625, 2.337890625, -0.20361328125, 1.396484375, 2.103515625, 0.86083984375, 1.080078125, 0.345703125, 2.57421875, 0.03076171875, -1.482421875, -0.75537109375, -0.278564453125, -0.486328125, -0.1605224609375, 0.478515625, -2.375, -1.0771484375, 0.986328125, -0.2646484375, -0.74560546875, 0.11279296875, -2.126953125, 0.837890625, 2.287109375, 2.044921875, 2.6953125, -0.6015625, -0.4072265625, 1.0615234375, -1.1064453125, -0.439453125, -1.4541015625, -0.611328125, -1.509765625, -0.64404296875, 0.64306640625, -0.075439453125, 0.8974609375, 0.19482421875, 0.35791015625, -1.916015625, 1.515625, -1.341796875, 2.333984375, 1.5, 1.68359375, 1.1796875, 0.7138671875, 1.794921875, -0.8203125, -0.420166015625, -0.3583984375, 0.451416015625, 0.72900390625, 0.13671875, 0.4453125, -0.1318359375, -0.9375, -0.093017578125, 2.1875, -0.057861328125, -1.96484375, 0.49609375, 1.17578125, -0.05859375, -2.4921875, 0.98046875, -1.939453125, -1.5166015625, -0.50634765625, -0.0791015625, 0.53076171875, 0.7275390625, 2.96484375, -1.0732421875, -2.611328125, 1.3623046875, 0.4658203125, -1.4384765625, 0.183837890625, 2.283203125, 0.47216796875, -2.0390625, -1.4052734375, -1.3134765625, 2.453125, -0.92578125, 0.6708984375, 1.5830078125, -0.327880859375, -0.315673828125, -0.771484375, 0.22607421875, -0.86962890625, -1.06640625, -1.8662109375, 0.638671875, -1.78125, -0.42578125, -1.1240234375, 0.9228515625, 0.74267578125, 0.47216796875, -0.701171875, 2.076171875, -0.1103515625, 0.11639404296875, 0.2509765625, -2.40625, -3.55078125, -0.78369140625, -1.84375, 1.412109375, 0.2314453125, 2.060546875, -0.9599609375, -1.86328125, -0.98828125, -0.13671875, -1.365234375, -2.72265625, -0.6962890625, -1.392578125, 1.49609375, -1.0703125, 0.64501953125, -2.26953125, -0.587890625, -0.8544921875, 1.71484375, 2.484375, 1.65625, -0.5673828125, 0.484375, 2.05078125, -0.923828125, -0.119140625, 4.40234375, -1.0361328125, -2.240234375, 1.1611328125, -0.88916015625, 0.341064453125, -2.130859375, 0.0478515625, 1.599609375, -1.158203125, 0.447509765625, -2.515625, 0.36328125, 2.05859375, -3.064453125, -1.240234375, -0.71337890625, 1.74609375, 1.228515625, 0.412109375, -2.048828125, 0.341552734375, -2.34375, -1.140625, 0.48046875, 0.033203125, 1.3154296875, 0.259521484375, -2.42578125, -1.037109375, -0.34130859375, -1.818359375, 0.375244140625, 0.3349609375, 0.72412109375, -0.57763671875, -0.8408203125, -0.05474853515625, -0.6015625, -0.677734375, 0.9912109375, -0.351318359375, -0.8935546875, -1.501953125, 0.3564453125, 1.248046875, 1.2421875, 0.25390625, 0.2322998046875, 0.5703125, 1.0595703125, -1.0712890625, 0.41259765625, -0.86962890625, 0.591796875, 0.302490234375, -0.407470703125, 2.26953125, -1.5048828125, 0.89990234375, 0.3720703125, -1.853515625, -0.63037109375, -0.62939453125, 0.77490234375, -0.142578125, -1.46484375, -0.32275390625, 2.41796875, 0.4814453125, 0.66796875, 0.183349609375, -1.2216796875, 2.90234375, 1.197265625, 0.062744140625, 0.7763671875, 0.84033203125, 0.716796875, 0.033538818359375, 0.13330078125, -1.283203125, 0.1048583984375, -0.2607421875, 2.150390625, -0.5263671875, 0.232421875, -0.010498046875, -1.525390625, 0.9384765625, 0.366943359375, 1.6474609375, 0.380615234375, -1.9423828125, -0.71875, 2.90625, 0.19970703125, -1.5234375, -0.529296875, -0.98828125, -0.5830078125, -1.9453125, 2.6015625, -0.93798828125, -1.7880859375, -2.64453125, 0.7890625, -0.052001953125, -1.173828125, -0.09716796875, -0.8828125, -1.474609375, -0.88916015625, 3.9921875, 1.564453125, 0.50146484375, 0.97216796875, 0.34375, -0.865234375, -1.91015625, -0.7265625, -0.97021484375, 0.240234375, 1.73828125, 0.385009765625, 1.0205078125, 0.9375, 0.54296875, -2.23828125, 0.17138671875, -1.26953125, 1.7978515625, 1.400390625, 1.26953125, -0.521484375, 2.572265625, 1.892578125, 1.6015625, -0.1759033203125, -3.09375, 1.888671875, -0.0068359375, -0.26611328125, 1.7431640625, 0.19140625, 2.3203125, -0.94580078125, -1.283203125, 1.6171875, 0.93994140625, 0.537109375, -1.42578125, 0.2318115234375, -1.7744140625, -0.76904296875, -0.68408203125, -0.44140625, -1.0029296875, -0.4248046875, -2.203125, -1.41015625, 1.421875, 1.462890625, 0.216064453125, -1.990234375, -1.193359375, 1.8818359375, -0.73046875, 0.1495361328125, -0.221435546875, 1.6298828125, 0.068359375, 3.416015625, -0.5712890625, 3.59375, 0.3369140625, -1.390625, 2.4296875, -1.3564453125, -1.5478515625, 0.216064453125, -0.9873046875, -2.59765625, 0.7802734375, -0.8759765625, -1.0859375, 2.765625, 0.099609375, 0.80419921875, -0.73876953125, -0.560546875, 0.54638671875, 0.297119140625, 1.736328125, 0.48828125, 1.169921875, -0.5244140625, -0.17333984375, 0.380859375, 0.474609375, 0.52978515625, 2.720703125, 0.708984375, -0.091552734375, 1.421875, 0.281494140625, -2.708984375, -1.5703125, -1.150390625, 1.08203125, 0.9755859375, -1.185546875, -1.712890625, 1.2734375, -0.5791015625, -2.28125, -0.123046875, 0.73583984375, -0.71142578125, -0.119140625, -1.435546875, -1.1796875, 2.892578125, -0.2578125, -0.5966796875, 1.544921875, 0.0260009765625, 1.755859375, 3.11328125, 1.81640625, -1.001953125, -0.953125, 0.02197265625, -1.30078125, 3.1328125, -1.2060546875, 2.279296875, -0.00439453125, 0.388671875, -2.22265625, -1.49609375, -1.65234375, -1.8564453125, -1.4443359375, -0.67138671875, -1.986328125, 0.173828125, 0.35498046875, 0.07763671875, 0.3837890625, 2.4375, 1.92578125, 3.072265625, 2.6484375, -1.474609375, -1.177734375, -1.5849609375, -1.759765625, -0.92529296875, 1.205078125, 0.51123046875, -2.83984375, 0.46484375, -0.71875, 0.2578125, -1.259765625, 3.943359375, -0.5810546875], "dtype": "float16"}}, {"a": [{"__nd__": [-0.43765613436698914, 0.53459233045578, -1.545465111732483, -2.1885623931884766, 0.23326174914836884, 1.884547233581543, 0.33218279480934143, -0.4464961290359497, -0.6955038905143738, 1.4180816411972046, 0.24660077691078186, 1.341305136680603, 0.48913905024528503, 0.4626809358596802, -0.4094502627849579, -0.5119193196296692, 1.271378993988037, -0.5694510340690613, -0.5562089085578918, 0.3638042211532593, 0.026427509263157845, 0.6019096970558167, 1.0300546884536743, -0.6986364126205444, 0.6247507333755493, -0.18093927204608917, 0.8901083469390869, 0.46835145354270935, -0.13361889123916626, 0.4696112871170044, -0.5131105780601501, -0.22383712232112885, 1.6050498485565186, -0.7543278336524963, 0.5103232264518738, 0.14584015309810638, -2.2096245288848877, -1.7563941478729248, -0.2390700727701187, -0.1909196525812149, 0.03850416839122772, 1.0369956493377686, -0.769800066947937, -1.3527089357376099, -0.6764004230499268, 0.7664797306060791, 0.3202661871910095, -0.8553302884101868, 0.7725240588188171, -1.207782506942749, -1.0645886659622192, 0.07147569954395294, 1.4206243753433228, -1.2229187488555908, 0.33381152153015137, 1.3059298992156982, 0.8797905445098877, 0.7839398980140686, 0.7090185880661011, 0.6592782735824585, 0.08378206938505173, -1.3200205564498901, 0.7919129133224487, -0.31031879782676697, 1.3896335363388062, 1.4235610961914062, -1.195897102355957, -0.19324032962322235, 1.4748165607452393, 0.9175082445144653, -0.2755499482154846, 0.27084508538246155, 0.4903281331062317, -0.6868623495101929, 0.5676612854003906, -0.17240014672279358, 0.3039526343345642, -1.9823261499404907, 1.4105961322784424, 1.2247475385665894, 1.4567196369171143, 0.9115380048751831, 0.1632424145936966, -0.2916245460510254, -0.7359356880187988, -0.5005773901939392, -0.44426217675209045, -0.5311362147331238, -0.8625249862670898, -0.9878543615341187, 0.4402318298816681, -1.0791497230529785, -1.5117154121398926, 0.6814757585525513, 2.6326851844787598, -0.374757319688797, 0.7724671363830566, 1.7289011478424072, -0.3523041307926178, 0.9157520532608032, 0.671943187713623, -0.1759808212518692, -2.061678171157837, -0.8608563542366028, 0.5123628377914429, 2.1616251468658447, 2.4329330921173096, 0.12435246258974075, -0.7250327467918396, -1.4239951372146606, -1.1211227178573608, 0.447288453578949, 1.4620168209075928, -1.620141625404358, -0.7980644106864929, 1.1618294715881348, 1.233016848564148, -0.1290755569934845, 0.08472440391778946, -0.7191439867019653, -2.2503392696380615, 0.6184743642807007, 0.1726863831281662, 0.8515044450759888, 0.3314477801322937, -1.3604148626327515, -0.9390583038330078, 2.1186821460723877, -1.3568434715270996, 0.6055176258087158, 1.3824595212936401, -1.3524434566497803, 1.0095503330230713, 0.2136225402355194, -1.135817050933838, -0.9456440806388855, 0.7765902280807495, -0.8754352927207947, 1.5447688102722168, -0.022996189072728157, -0.1456853151321411, -0.31685489416122437, 1.3864293098449707, -0.10582667589187622, 0.07009156048297882, 1.3468923568725586, 0.37528523802757263, 0.3915928304195404, -1.7808310985565186, 2.3230936527252197, -0.651432991027832, 1.2035417556762695, -2.105473041534424, -0.6039894819259644, -1.8111404180526733, 1.3987414836883545, 0.5235859751701355, -0.5967880487442017, -0.45035600662231445, -0.3401910960674286, 1.4351675510406494, -1.1942390203475952, 1.8153117895126343, 1.2940229177474976, 0.9758567214012146, -0.07668225467205048, 0.601118266582489, 0.6360336542129517, -0.3170388638973236, 2.5331056118011475, -0.28403598070144653, 0.5556104779243469, 2.554494857788086, 0.953244149684906, 0.7092507481575012, -0.7796356081962585, 0.822937548160553, -0.029231950640678406, 0.0891852155327797, -0.02665664255619049, -0.24661462008953094, -1.2170032262802124, 0.47412335872650146, 1.6524145603179932, -1.0811687707901, 0.7512228488922119, -0.399353951215744, -1.1011916399002075, -0.2907993495464325, 1.342225193977356, 0.32145363092422485, 0.945402979850769, -0.9882616996765137, -0.9970155358314514, 0.09135482460260391, -1.0025429725646973, 0.5956771969795227, 1.6734930276870728, -0.05426385626196861, -0.26532042026519775, 0.6981300115585327, 0.5553001165390015, -1.4208753108978271, -1.4657708406448364, 1.1120129823684692, 0.25036898255348206, -1.2987470626831055, -0.22314892709255219, 0.0016776619013398886, -0.21158207952976227, 0.9268776178359985, -0.4556657075881958, 0.7067751884460449, 1.2530882358551025, -1.035157561302185, -1.183853030204773, -0.4912927746772766, 0.04150949791073799, 0.11230550706386566, -0.23878136277198792, -0.42538636922836304, 1.2542043924331665, -2.5104148387908936, -0.19432339072227478, 0.8369331359863281, -1.0456559658050537, 1.4140363931655884, 0.7140517234802246, 0.8423051834106445, -0.4439270794391632, -0.05258724465966225, -1.0475441217422485, -0.027402639389038086, 0.3886079490184784, -0.4419308006763458, -0.4607561528682709, -1.8479106426239014, -0.6004160642623901, 0.3117450475692749, -1.7419161796569824, -0.05024335905909538, 0.010615103878080845, -1.4566386938095093, -1.1145961284637451, -0.7153669595718384, 0.7726214528083801, 0.18476811051368713, 0.6194764971733093, -2.0057930946350098, -1.8785122632980347, 0.5796017050743103, 0.7296137809753418, 0.8831273317337036, -0.974544107913971, -0.2380523979663849, -0.8836239576339722, 2.3049144744873047, 0.8099979162216187, -1.0491762161254883, 0.4912512004375458, -0.5808548331260681, 0.3594844937324524, -0.873279869556427, -1.2473289966583252, -0.6316194534301758, -1.6565078496932983, 0.9759195446968079, 1.0015407800674438, 0.14783942699432373, 1.9887243509292603, -0.7280283570289612, -1.789634346961975, -0.6611068844795227, 0.3102003335952759, 1.1066639423370361, -2.828324794769287, -2.448119878768921, -0.6849069595336914, -1.648053526878357, 0.32046034932136536, 0.1355258673429489, 0.7489475607872009, -0.43554675579071045, 1.1605689525604248, 0.18257537484169006, -0.5482358336448669, -0.40805602073669434, -0.3817463517189026, 1.3008724451065063, 0.261555939912796, 0.4276938736438751, 0.1688108593225479, -0.9271926879882812, -0.9396837949752808, -1.3406097888946533, 1.1895502805709839, -0.3379809260368347, -0.032451361417770386, -0.06749311834573746, -0.6647447347640991, -0.546707034111023, -0.2397955358028412, 0.6657328605651855, -0.408245325088501, 1.0655020475387573, -0.19611501693725586, 1.0528093576431274, -0.14016586542129517, -0.19650085270404816, -0.2153683453798294, 0.14652112126350403, 0.5120909810066223, -1.926866054534912, 0.8890362977981567, -0.16967573761940002, 1.295458436012268, -0.9520187377929688, 0.8116533160209656, 0.5325064659118652, 1.1017160415649414, -1.483237624168396, -1.3091689348220825, -0.08573686331510544, 0.47943052649497986, 0.3175938129425049, -1.0006184577941895, 0.545142650604248, 1.6421527862548828, 0.10922929644584656, 0.9542324542999268, 0.685627281665802, -1.0279021263122559, -0.029209649190306664, -1.0810046195983887, 0.7196481227874756, -2.0851211547851562, -0.9257332682609558, -0.054774463176727295, 0.9594678282737732, 0.29981181025505066, 0.32971400022506714, -0.6848090291023254, -0.5103275179862976, 0.008014305494725704, 1.401424765586853, -0.4342014789581299, 0.19104169309139252, 0.8357760310173035, -0.6423519849777222, 0.6102279424667358, 0.4537335932254791, -1.5310128927230835, -0.40022653341293335, -0.14067037403583527, 1.0972397327423096, -1.2228645086288452, 0.6336866617202759, 0.5177457332611084, 0.2207963466644287, -3.3901782035827637, 0.2702347934246063, -0.4526909291744232, -0.12838710844516754, 0.5491189956665039, -0.9688854813575745, 0.8814938068389893, -0.2723681628704071, -1.2720180749893188, -1.5296010971069336, 0.15614326298236847, -0.47156938910484314, 0.311393678188324, -1.534766674041748, 0.4513314962387085, 0.4124961197376251, 0.6401373147964478, 0.30742189288139343, -2.1199123859405518, 1.1597070693969727, -1.466741919517517, 0.0183703675866127, -0.48834922909736633, 0.6439473628997803, 1.5132976770401, 2.0729901790618896, 1.8976399898529053, -1.2018253803253174, 1.2458324432373047, -0.32725879549980164, -0.25054091215133667, -0.4262721836566925, 0.8796132802963257, -1.3183104991912842, 0.11230083554983139, 1.0116225481033325, 1.1824653148651123, 2.66945743560791, 0.41955119371414185, 0.489610493183136, 0.15768711268901825, 1.0261539220809937, -1.0436960458755493, 0.15352579951286316, -1.2725262641906738, 0.3002987205982208, 0.1660517156124115, -0.4893394708633423, -0.969524621963501, -0.16691839694976807, -0.8458313941955566, 2.215059518814087, -0.6827386021614075, -0.1270902156829834, -0.21476349234580994, 0.07797734439373016, -1.322263240814209, -0.3464585244655609, -0.4075987637042999, -0.22253037989139557, 1.9689431190490723, -0.8238358497619629, 1.810968041419983, 1.2756261825561523, 1.3472602367401123, -0.39958247542381287, 0.19332554936408997, -0.45464178919792175, -0.5434123873710632, -1.0992517471313477, -0.36639708280563354, -0.5598204731941223, 0.804904580116272, -0.055839903652668, -0.6575095057487488, -0.048885542899370193, 1.4277758598327637, 1.0414018630981445, 1.1103132963180542, 0.24506776034832, 0.9579217433929443, -0.4648655354976654, -0.8143501281738281, 0.3743564486503601, -0.42536333203315735, -0.18609891831874847, 1.1820544004440308, -0.25132444500923157, -0.4423329532146454, -0.5616948008537292, -0.7699851989746094, 0.4330909848213196, -0.6504247188568115, 0.1266702115535736, 2.166367769241333, 2.3266043663024902, 0.23918335139751434, -1.3406751155853271, -0.2149093598127365, 0.21618254482746124, -0.7794533967971802, -1.5125006437301636, -1.0174930095672607, 0.3096974790096283, 0.20932507514953613, -0.17379416525363922, -0.8178531527519226, 0.796838641166687, -0.7211321592330933, 0.6798459887504578, -0.5777726769447327, 1.037092685699463, -1.8553354740142822, -1.2140692472457886, 2.242298126220703, -1.1550512313842773, -0.5314176678657532, 0.3044421076774597, -0.22350022196769714, 0.44209912419319153, 0.6564834713935852, -1.2855042219161987, -0.9567081928253174, -0.88026362657547, 1.9147869348526, 1.1907057762145996, 0.49795687198638916, 9.507462527835742e-05, -0.6035178303718567, -0.36231860518455505, -0.32669997215270996, -0.9899267554283142, -1.1585161685943604, -0.6773158311843872, -1.3248196840286255, 0.35212838649749756, -0.03315068408846855, 0.8322081565856934, 0.07717183232307434, 0.8218950629234314, -0.018605496734380722, -0.0802118256688118, -0.3028774559497833, -1.0374218225479126, 0.7643122673034668, -0.02243713103234768, 1.6157728433609009, 0.891140878200531, -1.3007088899612427, -0.5851123332977295, 0.5210089087486267, -1.8658462762832642, -2.3008830547332764, 1.0381433963775635, -0.25025057792663574, -0.2978336215019226, -1.352513074874878, 0.4964195787906647, 0.44367361068725586, 0.38652658462524414, -0.685113787651062, 0.5375611186027527, 0.8902108073234558, -0.3116128742694855, -0.7230555415153503, -0.6816496849060059, -0.4996667504310608, 0.2807195484638214, -1.3686984777450562, -1.861228346824646, -1.3909780979156494, 0.3478609323501587, 0.2709079384803772, 0.4795905649662018, 0.997850775718689, -0.10460195690393448, 0.5983151197433472, 0.9071372151374817, 0.5624397397041321, -1.381100058555603, 0.7483146786689758, -1.3775789737701416, 0.6772246360778809, -0.3651596009731293, 0.9862369894981384, 0.5595583319664001, -1.075282096862793, 0.4133557081222534, 0.625084638595581, -1.4557857513427734, -0.7130443453788757, -0.6839855909347534, -0.5871244072914124, -1.1867812871932983, -1.0277327299118042, -0.9790070056915283, 0.7120649218559265, -0.83962482213974, -1.5734965801239014, 0.3369522988796234, 0.17591740190982819, 0.3942260146141052, -0.2448720633983612, -1.3938907384872437, 0.418341726064682, 0.5839117169380188, 0.7641373872756958, 2.011596441268921, 1.204182744026184, -0.33531850576400757, 1.557309627532959, 0.5549849271774292, 0.3792228102684021, -0.026997728273272514, 0.4674111008644104, 1.1050492525100708, -0.5452100038528442, 1.5111173391342163, -0.9868372082710266, 0.18167270720005035, -1.1331313848495483, 0.12710510194301605, 0.7869697213172913, 1.2894030809402466, -0.7461887001991272, -0.3412792384624481, 0.10530965775251389, -0.45830419659614563, 0.7207285761833191, -1.2383817434310913, -0.014404075220227242, -1.4486724138259888, -0.7852996587753296, 1.8195295333862305, 0.028659982606768608, 0.6562861800193787, 1.1204936504364014, -0.478273868560791, -1.5969328880310059, -0.06000574305653572, 0.4686456024646759, 0.3034253418445587, 0.0683279037475586, 1.2813588380813599, -0.2546265423297882, 0.2531573176383972, 0.8318573832511902, 0.16734831035137177, -1.0624533891677856, -0.5421788096427917, -0.1710631251335144, -0.9123961329460144, 0.5129261016845703, 0.12889401614665985, -1.1153172254562378, -0.4750291407108307, -0.9846305847167969, 0.6299195289611816, -0.9094771146774292, 0.08319664746522903, -0.4147413671016693, -0.33961519598960876, 1.1383033990859985, -0.3267176151275635, 1.0134892463684082, 0.013181272894144058, 0.46996113657951355, -0.8464650511741638, 0.37920382618904114, 0.9533764719963074, 1.2222105264663696, -1.0134189128875732, 0.5230858325958252, 1.4818228483200073, 0.009272361174225807, 1.0154787302017212, -0.6044986248016357, 0.7541020512580872, 0.20415085554122925, -1.758378028869629, -0.7716847658157349, -0.8531798720359802, -0.846735954284668, 0.33939942717552185, 0.8192301988601685, -0.35744646191596985, 0.489838182926178, 1.0797770023345947, 0.4874829053878784, -1.9147080183029175, -0.4789107143878937, -1.1869597434997559, 0.356483519077301, 0.2877252995967865, -0.21948185563087463, -0.32371455430984497, 1.3894835710525513, 0.9791508913040161, -0.8319600820541382, -1.8343334197998047, 1.0447521209716797, 0.22185386717319489, -0.019604429602622986, 0.058703333139419556, 1.6611195802688599, -0.38380780816078186, 0.22086822986602783, -0.8815888166427612, 0.07391772419214249, -0.6860283613204956, 0.12141963094472885, 1.3331987857818604, -0.8957470059394836, -2.1382579803466797, -1.0441513061523438, -0.09825585037469864, -0.3611421585083008, 0.8897730112075806, -1.0590871572494507, -1.5417163372039795, -0.4526195526123047, -0.05239629000425339, -0.3429163992404938, 0.3131817877292633, 1.0117874145507812, -2.269336223602295, -0.32486584782600403, -0.684292197227478, -2.1485443115234375, 0.2947203814983368, 1.16427743434906, -0.036863185465335846, 0.9649288654327393, 1.2091407775878906, -0.1140885278582573, 0.2600688934326172, -1.0203083753585815, 1.9028624296188354, 0.16549931466579437, -0.7648479342460632, -1.2358263731002808, -0.12003009021282196, -2.3148956298828125, -0.013725870288908482, 0.696960985660553, 1.4238693714141846, -1.911672830581665, -0.4156520366668701, -0.826279878616333, 0.4791772961616516, -1.3915345668792725, 1.2553761005401611, -0.07763760536909103, -1.006807565689087, 1.3363882303237915, -0.368998259305954, 0.8910132050514221, 0.6017724275588989, -0.018447043374180794, -1.026350975036621, 0.9585354924201965, 0.6644884347915649, -1.7245032787322998, -1.9095925092697144, 1.119448184967041, -1.4556666612625122, 1.510349988937378, -0.6299636363983154, 0.6299099326133728, 1.3729634284973145, -0.9511213302612305, 0.49956443905830383, 0.7998775839805603, -0.6158682703971863, -0.25408074259757996, 2.2681195735931396, -0.4867759644985199, 0.5045683979988098, -0.6819688677787781, 1.15632963180542, 0.033799607306718826, -2.2504255771636963, -0.7700615525245667, 0.8313615918159485, 1.8924392461776733, 0.08141545206308365, -0.2966565489768982, 1.4819597005844116, 0.7360701560974121, -2.8444125652313232, -0.6280178427696228, -2.5022878646850586, -1.2728979587554932, 0.1945393979549408, 0.7415598034858704, 1.2472381591796875, -0.26387494802474976, 0.8843260407447815, -1.1568206548690796, 0.7740632891654968, 0.016130764037370682, -0.10623108595609665, -0.5057459473609924, -1.0790694952011108, -0.573310375213623, -1.057437539100647, 1.5846335887908936, -0.9598525762557983, 0.5595740675926208, 0.4898189902305603, -0.20733113586902618, 1.8176265954971313, 1.0757253170013428, -0.9382382035255432, 0.29307398200035095, 2.2072997093200684, 0.503329873085022, 1.5870436429977417, -0.693178653717041, 0.38058048486709595, -0.5979458689689636, -0.932345986366272, 0.44205012917518616, 1.2290306091308594, 0.23005393147468567, -1.3183138370513916, 1.2548575401306152, 2.3542168140411377, 1.4112293720245361, 1.4330979585647583, 2.0988643169403076, -1.2122178077697754, -0.5525711178779602, -0.6083630919456482, 1.3726980686187744, -1.210343837738037, 0.19593289494514465, -0.7866541147232056, 0.3320201635360718, -0.2535628080368042, -0.6254843473434448, -0.07609032094478607, -0.00831920001655817, -0.22988249361515045, 1.2190899848937988, -0.15667150914669037, 0.7435086965560913, 2.33404541015625, -1.1603879928588867, 0.822293758392334, -0.16847027838230133, -1.4674471616744995, -1.135148048400879, 0.473638653755188, -0.2051149606704712, 0.8174933791160583, 0.12601380050182343, 0.36576446890830994, 0.0476251058280468, 0.3022766411304474, 1.9250353574752808, -0.45154300332069397, -0.4100823402404785, 1.1775680780410767, -1.641493320465088, 0.4659223258495331, -0.47885653376579285, 0.8786201477050781, -1.2072163820266724, -0.1424219161272049, -0.1326688677072525, -1.113532543182373, 0.43411022424697876, -0.7204089760780334, -0.2611047923564911, 1.0498967170715332, -2.2122466564178467, -1.4809070825576782, 0.7126416563987732, 0.1806803196668625, 0.43097570538520813, 0.14871418476104736, -0.7732794880867004, 1.191969871520996, 1.2437642812728882, 0.3578806519508362, 0.2903595268726349, -0.22893202304840088, -1.3225868940353394, 1.0666189193725586, -0.6003713607788086, 2.1393136978149414, -0.6441309452056885, 2.340116024017334, 1.5410369634628296, 1.645438313484192, -0.6820710897445679, 0.8480907678604126, 1.470757246017456, 0.8329437375068665, -0.7493895888328552, -1.9366403818130493, -2.2751035690307617, 0.07187869399785995, 0.22868578135967255, -0.020475825294852257, 1.4971469640731812, 0.863201379776001, 2.8787267208099365, -0.28671085834503174, -0.1011803150177002, 0.034514594823122025, -0.6773085594177246, 1.4335328340530396, -0.2745939791202545, -0.03391548991203308, -1.1029653549194336, -0.40629303455352783, -0.41491058468818665, 1.572907567024231, 1.7551183700561523, 1.310017466545105, -0.30215224623680115, -0.18606428802013397, 0.4492427110671997, 0.05911925435066223, -0.5812149047851562, -0.5407753586769104, -0.5762096047401428, -0.36933279037475586, 1.424765706062317, -0.27276843786239624, -0.15669794380664825, 1.5507242679595947, 0.3390509784221649, -0.6613114476203918, 0.8249058127403259, 0.5703022480010986, -0.21940846741199493, 0.5745712518692017, 0.4424428641796112, 0.7524848580360413, 2.1918318271636963, 0.9892317652702332, -0.35892149806022644, -0.2900320887565613, -1.5229606628417969, -0.7034825682640076, -0.7837270498275757, 0.5503122210502625, -0.016272714361548424, -0.4486871659755707, -1.331449031829834, 1.2454960346221924, 0.30922743678092957, 0.12166813015937805, 1.7799100875854492, 1.4548547267913818, 0.9426736235618591, 1.5585112571716309, 0.4986925423145294, -0.30264177918434143, -0.38897329568862915, -0.9395151734352112, -0.3977753221988678, -0.6930821537971497, -0.6195765137672424, 0.41201090812683105, 0.38468649983406067, -0.025218835100531578, 0.2897784113883972, 0.021238453686237335, -0.7092177271842957, 0.6167230010032654, 0.6598345041275024, 1.2155306339263916, -0.22145602107048035, -0.11082799732685089, -2.326382637023926, 0.21427801251411438, -0.2868580222129822, -0.7919878959655762, -0.45780420303344727, -0.13499069213867188, -0.024688469246029854, 1.4930940866470337, -0.33168718218803406, 0.8414179086685181, -1.1756678819656372, 0.7804045081138611, 0.7704479694366455, 0.7446048259735107, -0.01854274608194828, -0.9313939213752747, 1.0117664337158203, -0.7132748365402222, -0.7419438362121582, 0.6656699776649475, -0.2078917771577835, -1.1109929084777832, 1.9786230325698853, 1.9726682901382446, 2.140608310699463, 1.7343248128890991, 0.04179823771119118, 0.21188685297966003, -0.8060389161109924, -1.2876346111297607, -1.6300668716430664, -0.6670555472373962, 0.1639195829629898, 0.2961462438106537, 0.44997817277908325, -0.3755939304828644, 1.4880307912826538, 1.2543679475784302, -0.8120098114013672, 0.9615228772163391, 0.16948957741260529, 1.076048493385315, -0.625922679901123, 1.1388182640075684, -0.64769047498703, -1.6628928184509277, 0.3091799318790436, -1.0284271240234375, -0.013654379174113274, 1.5030934810638428, -1.209761619567871, -0.08131570369005203, -1.0343736410140991, 0.5056210160255432, 0.19860003888607025, 0.7017651796340942, -1.2964012622833252, -1.0286492109298706, 0.24226805567741394, -1.129199504852295, -2.2797698974609375, 1.2389222383499146, -0.06649015098810196, 0.7808326482772827, -2.0086278915405273, 0.32776546478271484, -0.35027769207954407, -0.5283442735671997, -0.057477712631225586, -0.40378302335739136, -0.20297257602214813, -1.3117412328720093, -0.09016350656747818, -0.6736175417900085, -0.2685055434703827, -0.7442923188209534, -0.039465416222810745, -1.3651858568191528, 0.17040342092514038, 0.14533494412899017, 0.09533727169036865, -0.7226067781448364, 1.0035102367401123, 0.250971257686615, 0.3030427098274231, -0.45244312286376953, -0.32531219720840454, 1.1228777170181274, 3.245091676712036, -0.9044551849365234, 1.0792125463485718, -0.31791529059410095, -0.5939377546310425, 1.7406336069107056, -0.3555358946323395, 1.80131995677948, 1.395002007484436, -1.471204400062561, -0.8139606714248657, -0.45947375893592834, 1.4513410329818726, 0.8842176795005798, -1.3983862400054932, 0.45642104744911194, -0.21807287633419037, -0.4846697747707367, -0.5171878337860107, 0.2518460750579834, -0.05171988531947136, 1.2036212682724, -1.2691246271133423, 0.3307677209377289, -0.7351031303405762, -0.9809619188308716, 1.7661707401275635, 0.03846464678645134, -0.36348697543144226, 0.8922025561332703, -0.9028948545455933, 0.5520720481872559, 0.35537248849868774, 0.4684109091758728, 0.5327378511428833, -0.07260699570178986, -0.5141249895095825, 2.046609401702881, -0.1523558348417282, -1.2163960933685303, 0.9481764435768127, 0.7611791491508484, -1.6088358163833618, -0.31742095947265625, 1.1942061185836792, 1.1528072357177734, 2.1866161823272705, 0.7022526264190674, 2.221217155456543, 1.8711187839508057, -0.24723978340625763, -1.662862777709961, -0.9911191463470459, 2.0370888710021973, -0.3660951256752014, -0.24407164752483368, -0.8323014378547668, 0.5517569780349731, -1.653072714805603, 1.8462038040161133, -1.8609076738357544, -1.472048282623291, 0.9587563276290894, -0.7230517864227295, 0.02383369766175747, 1.1149362325668335, -0.6575551629066467, -0.3113723397254944, 0.5472745299339294, 0.6064728498458862, -0.5879026055335999, -0.9226323366165161, -2.3323416709899902, 0.8758760094642639, -2.5250492095947266, 1.2160946130752563, 1.2464364767074585, 1.4765076637268066, -0.5008557438850403, -0.6357945799827576, 1.085789680480957, 0.8403980731964111, -0.35325077176094055, 1.6649200916290283, -0.8807535171508789, 0.5029191970825195, 2.3257813453674316, 0.959364116191864, -0.8399603962898254, -0.8931240439414978, 0.37135952711105347, -0.05793668329715729, -0.5133399367332458, -0.9810076951980591, -0.3808496594429016, -1.379294514656067, 0.14900369942188263, -0.5672020316123962, 0.29191452264785767, 0.11527381092309952, -0.19424480199813843, -1.2156236171722412, 1.4240883588790894, -1.4483572244644165, -1.476380705833435, 0.8689501881599426, -1.5566293001174927, -1.0947505235671997, -1.6637682914733887, -0.7148928642272949, -0.8721149563789368, -1.3845192193984985, -0.27552106976509094, 0.005614025052636862, -0.054123442620038986, 0.09962733834981918, 0.6625126600265503, 1.1337050199508667, 0.02200539968907833, 0.8284053206443787, -0.7954614162445068, -1.456568717956543, -1.066780686378479, 1.0479826927185059, -1.289138913154602, 0.18459436297416687, -0.6842527389526367, -0.12249447405338287, -0.17767149209976196, -0.2719085216522217, 0.6425358057022095, -0.742911696434021, 3.13936185836792, 2.3809382915496826, -0.5154361724853516, 0.5209844708442688, -0.16404028236865997, -0.8961922526359558, 1.5670260190963745, 1.1346120834350586, -0.984644889831543, -1.30072021484375, -0.02335100620985031, 0.33849895000457764, -0.41808295249938965, 1.025243878364563, -0.6065728664398193, 0.7004866600036621, 0.7724853157997131, 0.05999968945980072, -0.38126909732818604, 0.9287316203117371, -2.318203926086426, -0.7665119171142578, -1.8642942905426025, -0.8120651841163635, 1.4382790327072144, -0.10837780684232712, 0.48299160599708557, 0.6497774124145508, 1.1308603286743164, 2.149585485458374, -0.5038214921951294, 0.9468952417373657, -0.9893319606781006, 0.5627405643463135, -0.15342026948928833, 0.08511428534984589, 0.6106746792793274, -0.6815357208251953, 0.8681068420410156, 0.38861605525016785, -0.41147083044052124, -1.8701882362365723, 0.5772879719734192, 0.5506977438926697, -0.8179663419723511, 0.7935748100280762, 0.679429292678833, -0.7193748950958252, -0.6030463576316833, 1.3555442094802856, 1.384609580039978, -0.4642490744590759, 0.2808276414871216, -0.10020781308412552, -1.2006664276123047, -0.8233212232589722, -0.062052007764577866, -0.585462749004364, 0.3398907780647278, -0.49132925271987915, -3.113725423812866, -0.12768222391605377, -1.7401725053787231, -0.1660100221633911, 2.1795401573181152, -0.5699175596237183, 0.6805115342140198, 0.3683960437774658, -0.1608894169330597, 1.2004295587539673, 0.14835719764232635, -0.22861148416996002, 3.704685688018799, -1.1403720378875732, 0.5503893494606018, 0.13616317510604858, 2.9968550205230713, 1.751015305519104, -2.2092065811157227, -2.5831682682037354, -0.6360037922859192, 1.0750075578689575, 0.1675461232662201, -1.1320563554763794, 1.2797305583953857, -0.18739567697048187, -0.08315731585025787, -0.34729939699172974, 1.6165472269058228, -0.5230294466018677, -0.05095789209008217, 0.8598352670669556, -0.1598222851753235, 0.2728356122970581, -0.012656082399189472, -1.2996776103973389, 0.492740660905838, 1.0701076984405518, -0.9537243843078613, -0.29144278168678284, -1.2586338520050049, 2.1372039318084717, -0.27291399240493774, -0.2283007949590683, -0.9853574633598328, -0.07418126612901688, -0.941810131072998, -0.6171046495437622, 0.21496684849262238, 0.5618563890457153, 0.7801268696784973, -1.404511570930481, -1.014208436012268, -1.2867063283920288, 0.4415449798107147, 0.024765774607658386, 0.6244544386863708, 0.6416902542114258, -1.472032070159912, 1.7030810117721558, 0.9087383151054382, -0.7077207565307617, 0.6379583477973938, 0.0017607561312615871, -0.19899004697799683, 1.0391701459884644, 1.2922325134277344, 1.2721216678619385, -0.7017688751220703, -0.35410210490226746, 0.5762742161750793, 1.021852970123291, 1.5693985223770142, -1.0567002296447754, -1.4434971809387207, -0.9500192999839783, 0.8077664375305176, -0.2687819302082062, -0.12717734277248383, -0.4675804674625397, -1.698744773864746, -0.8683153390884399, -0.2998906970024109, 0.314280241727829, 2.633192300796509, -1.83012056350708, -0.9705582857131958, 0.15021666884422302, 0.6064671874046326, 0.08533880859613419, 1.082214117050171, 1.4259997606277466, -0.764511227607727, 1.5375328063964844, -0.18548113107681274, -0.8605585098266602, -0.27336356043815613, -0.39165404438972473, -1.1923640966415405, -2.079439878463745, -0.5136595964431763, 1.2578397989273071, 0.8004876375198364, -0.8476502895355225, -1.8697912693023682, -1.861133098602295, -0.15865476429462433, 0.8543845415115356, 1.1209397315979004, 0.39107510447502136, -1.8773152828216553, -1.506710410118103, -0.5727005004882812, -1.4240466356277466, 0.5174856185913086, -0.9676479697227478, -1.3016029596328735, -0.1037520319223404, 2.337235689163208, -0.7397310137748718, -0.1832813024520874, 2.4287571907043457, -1.620821475982666, 1.0146915912628174, -0.46934717893600464, -0.20692212879657745, -1.2969250679016113, -1.458959937095642, -1.0204640626907349, 0.7838316559791565, -1.1080597639083862, 0.48969581723213196, -0.6253018975257874, -0.48696956038475037, 0.8766745924949646, -0.9483655095100403, -1.081498146057129, 0.06949298828840256, -0.889620304107666, 0.3272566497325897, -0.643011212348938, -1.5232571363449097, -0.2239886075258255, 0.7794683575630188, -1.6084495782852173, -1.0888062715530396, 0.9434300065040588, 0.48386532068252563, -0.7143629193305969, 0.5041026473045349, -0.3650788962841034, -0.1090066060423851, -0.024271683767437935, 1.2864437103271484, -0.8614622950553894, 1.2524640560150146, 0.4253327548503876, 1.2790861129760742, -1.79362154006958, -0.38747596740722656, 0.8679563999176025, 0.46393659710884094, 0.6318038105964661, -1.3039679527282715, -0.26311299204826355, 0.5853800177574158, -0.09994624555110931, 0.1712864190340042, 1.6169732809066772, 0.10255858302116394, -0.07772848755121231, 0.37955114245414734, 1.5074049234390259, -0.34731748700141907, 0.2060147374868393, 1.24876070022583, 0.28068795800209045, -1.3365981578826904, 0.3554714322090149, 2.1003501415252686, 0.8612102270126343, -0.026113487780094147, 0.6194883584976196, 1.6453428268432617, 0.07361377030611038, -1.5893250703811646, 0.2279493808746338, 0.5933079123497009, 0.18540215492248535, -0.8378502726554871, -0.4085191488265991, -0.732418954372406, 1.3857150077819824, 1.1474255323410034, -2.0923104286193848, -0.19755536317825317, -0.026576600968837738, -0.41486218571662903, -2.1437602043151855, 0.7623329758644104, -0.6399492025375366, -0.3506506085395813, -1.698591947555542, 0.1399642825126648, -0.7200853824615479, 1.3525242805480957, -0.8541873693466187, -0.022569017484784126, 0.6211159229278564, -1.4121710062026978, -0.7751140594482422, -1.2772679328918457, -0.11623437702655792, 0.21087732911109924, 0.38459163904190063, -1.5560343265533447, 0.4875292181968689, -1.024320363998413, -1.6235963106155396, 1.5223020315170288, 1.2672579288482666, 0.24927206337451935, 0.39034658670425415, 0.8013822436332703, -0.3484154939651489, -3.5715487003326416, 1.2136969566345215, -0.03758605569601059, -0.15239453315734863, 0.19885556399822235, 0.007190517149865627, -0.49382489919662476, -1.7428115606307983, -0.8108400702476501, -0.4661538004875183, -0.22180651128292084, 2.3917691707611084, -1.1944185495376587, 0.17267872393131256, -0.4078240990638733, -0.18376626074314117, -0.3763163089752197, -0.5544529557228088, 2.141254425048828, -0.9935875535011292, -0.8720887303352356, 2.1449756622314453, 0.5272153615951538, -1.3604891300201416, -1.6847881078720093, 0.9429691433906555, 0.1853713095188141, 0.2225395143032074, -0.040035929530858994, 1.5517284870147705, 0.5435680747032166, 0.44858407974243164, -0.44030851125717163, 0.17384295165538788, -1.1210553646087646, 1.088840126991272, 0.8454999923706055, 0.8348931074142456, 1.513025164604187, -1.4807134866714478, 1.556928038597107, 0.7324522137641907, -0.21627245843410492, 0.40369462966918945, 3.1328296661376953, -1.3656935691833496, 0.6144918203353882, 1.389631748199463, 0.4409942030906677, -0.388205349445343, -0.8374349474906921, -0.6904003024101257, -0.2263157069683075, -0.36290663480758667, -2.8468239307403564, -1.3731402158737183, 0.2978137731552124, 1.0605394840240479, -0.32245728373527527, -0.6417452096939087, -1.1113934516906738, -0.911076009273529, 1.3389747142791748, 0.23302477598190308, 0.5865638256072998, -0.3798765242099762, -0.8402196168899536, -0.27537405490875244, -0.8146679401397705, 2.691580057144165, 0.5257277488708496, -0.5773918032646179, -1.0340656042099, 0.18487444519996643, 1.8058435916900635, -0.5420200228691101, 0.5242063999176025, -0.08235933631658554, 1.5817010402679443, 0.1763976812362671, -1.9717961549758911, -0.3999563157558441, 0.02671818993985653, 0.6300995349884033, -0.47773852944374084, -0.6996147036552429, -1.5202596187591553, -0.5280248522758484, -1.3600064516067505, -0.3298115134239197, -0.021262051537632942, -0.40741485357284546, -0.6439306735992432, -1.2010579109191895, 1.817983865737915, -0.8537155389785767, -0.0254498478025198, 0.11407151818275452, -1.3400943279266357, -0.29341399669647217, -0.49607324600219727, -0.4673779606819153, 2.5900378227233887, -1.461176872253418, -0.7494668364524841, 0.32068830728530884, 1.374759316444397, -1.9880023002624512, -0.0962694063782692, 1.1197068691253662, 0.6116508841514587, -2.6591434478759766, 0.4041173756122589, 1.2278969287872314, 1.8435300588607788, -0.2604275345802307, -0.3413456976413727, -2.3210177421569824, 0.04480687528848648, -0.23775990307331085, -0.3873404264450073, -0.45285338163375854, 0.09972544014453888, 0.5377384424209595, -0.23648382723331451, -0.5298081040382385, -0.13298924267292023, 0.46086353063583374, 0.46761757135391235, -0.31316351890563965, 0.11793962866067886, -0.32319268584251404, 1.8492498397827148, 0.24066711962223053, -0.35793155431747437, 0.6767739057540894, 0.5723845958709717, 0.8566408753395081, -1.4103364944458008, 2.6290361881256104, 0.04026149958372116, 2.584280014038086, 0.759770393371582, 1.3362889289855957, 1.0794594287872314, 1.1519527435302734, -0.84073406457901, 0.8358440399169922, -1.8671493530273438, -0.36511969566345215, 0.10530845820903778, -0.22274118661880493, 1.1292064189910889, 0.4148014783859253, 2.316406488418579, 0.3315337002277374, 0.19198961555957794, -0.7458952069282532, -0.1779254823923111, 0.5862852931022644, -0.8390750885009766, -0.12708733975887299, 0.05736899748444557, 0.02343934401869774, -0.048608191311359406, 0.24473385512828827, -1.4287755489349365, 0.5730981826782227, -1.1268260478973389, -1.2882705926895142, -0.4578043222427368, -0.2931683659553528, -0.25330832600593567, 0.7595281600952148, 0.8517423868179321, 1.4483301639556885, 0.6889738440513611, -0.7198154330253601, 0.4149111807346344, 0.4425944685935974, -1.0206661224365234, -0.09827196598052979, 0.5350626111030579, -2.008211374282837, -1.9736368656158447, -0.2547387480735779, 1.2415709495544434, 0.3951311707496643, -0.8223555684089661, 1.4376300573349, 0.5247650146484375, -0.0995844155550003, -2.9503109455108643, 1.360222339630127, -0.557285726070404, 0.39014938473701477, -0.3469773530960083, -0.06765919923782349, -0.041563812643289566, -0.994935154914856, -0.09618085622787476, -0.6500808596611023, -0.19534720480442047, 0.685113251209259, 1.4374933242797852, -1.3981637954711914, 0.08980687707662582, -1.3963526487350464, -1.7180335521697998, 0.08143027871847153, 0.33304381370544434, 0.36301857233047485, 0.9659481048583984, -0.06929624080657959, 0.10222425311803818, -0.5078101754188538, 0.32816576957702637, -0.7618916034698486, -1.3718347549438477, 0.2325335144996643, 0.3700042963027954, -1.5467356443405151, 0.4484182596206665, 0.20513461530208588, 0.8506169319152832, 1.4987633228302002, -1.5502800941467285, -2.922524929046631, -0.10883583128452301, -0.6657160520553589, -0.9544132947921753, -2.450146198272705, -0.02962454780936241, -1.2339904308319092, 1.20417058467865, -1.1685380935668945, 1.1464266777038574, 0.029197752475738525, -0.5445529222488403, -1.0812668800354004, 1.1766422986984253, 2.08943510055542, -0.1613183170557022, 0.4194556772708893, 0.5628886818885803, 0.3803137540817261, 0.8253099322319031, 0.554450511932373, -1.1430031061172485, 0.38478681445121765, 0.15028540790081024, 1.0290541648864746, -2.311265468597412, 0.34308430552482605, 0.134177103638649, -0.6287847757339478, -0.42391613125801086, -2.3585944175720215, -0.545052707195282, 2.293989419937134, -0.5177903771400452, -0.32315245270729065, -0.3719938397407532, -0.9421472549438477, -0.6952936053276062, 0.0413803793489933, 2.1141018867492676, 0.09719637036323547, -1.0505423545837402, -0.21361762285232544, 0.10547306388616562, 0.7837664484977722, -0.1957419365644455, 0.2266080379486084, 0.5471958518028259, 0.742800235748291, -1.137740969657898, -0.6075173616409302, -0.30114316940307617, 0.5972141623497009, 0.006680660881102085, -0.6995404958724976, 0.1484701782464981, -0.36054399609565735, 0.7351883053779602, -1.5391008853912354, 0.7407315373420715, 0.8907574415206909, 1.0466557741165161, 0.4894016981124878, 1.3064249753952026, -0.06727488338947296, 1.3200305700302124, 0.31231361627578735, 0.15606917440891266, 1.4809250831604004, 1.3708113431930542, 0.4004082977771759, -1.4442616701126099, -1.686989665031433, -1.6838219165802002, -0.3553025722503662, 0.43953901529312134, -0.6069043278694153, 0.23775815963745117, -0.591103732585907, 0.4899216294288635, -0.9502538442611694, -0.4717334806919098, 0.31737783551216125, 0.32067930698394775, 1.2149680852890015, 0.1599096804857254, -1.1493377685546875, -0.7665749788284302, -0.6131986379623413, 0.6252108812332153, 1.263584852218628, 1.4558137655258179, 0.5889500975608826, 1.0192450284957886, -0.647517740726471, -0.5034161806106567, -1.9795416593551636, 0.73979651927948, 0.8525886535644531, -1.3609856367111206, 1.3781640529632568, 0.8104198575019836, -0.3383413553237915, 0.5317466259002686, -1.3307600021362305, 0.029947444796562195, 0.6963412165641785, -0.6124127507209778, 0.3040183186531067, 0.7805927395820618, 0.8125497102737427, 2.387786865234375, 2.0976948738098145, 0.9502488374710083, -0.05538520589470863, 0.7585557699203491, -0.28476712107658386, 0.9678547382354736, -0.9971223473548889, -0.1313813030719757, -1.4279357194900513, 1.0352863073349, 0.6394214630126953, 0.15799137949943542, 0.5691531300544739, -0.9365978837013245, 0.9975312352180481, -1.405039668083191, 0.3841857612133026, 0.652587354183197, -0.20682960748672485, 0.862897515296936, -0.023942971602082253, 1.3814767599105835, 0.9724466800689697, -1.5360299348831177, 1.9639873504638672, -0.06849212199449539, 1.8181220293045044, 0.05594811588525772, -1.0111279487609863, 0.6254182457923889, 0.21435686945915222, -0.8479500412940979, -0.3519575893878937, 0.4561420679092407, -0.43953999876976013, 0.11117871850728989, 0.13136623799800873, 0.06694265455007553, 0.43263211846351624, -1.144219994544983, 0.6421756148338318, -0.6279433369636536, 0.0838596373796463, -1.8267720937728882, 0.27430692315101624, -0.8774109482765198, 0.2585856020450592, -0.15504008531570435, 0.9328920841217041, -1.217511534690857, 0.7353417873382568, 1.4569453001022339, 0.18987023830413818, 0.15962201356887817, -0.25252437591552734, 0.48025256395339966, -0.22755146026611328, 1.4081544876098633, 1.2013232707977295, 1.4710125923156738, -0.42657509446144104, 1.0382864475250244, 0.9739373326301575, 1.3596745729446411, -0.6447174549102783, 0.6093763709068298, -1.4812527894973755, 0.42643022537231445, -0.5465510487556458, -0.5978410840034485, -0.5310865044593811, 0.9281403422355652, -0.5457528233528137, -0.12208875268697739, -0.375544011592865, -1.2825318574905396, -0.46858081221580505, 0.12798304855823517, -0.041378479450941086, -3.1455962657928467, -0.5933001041412354, 1.269012689590454, -0.8007458448410034, -0.7917683720588684, -0.383012592792511, -1.3599495887756348, 2.2887444496154785, 1.5825470685958862, -0.47639045119285583, -2.046492099761963, -0.5132752060890198, 0.9454156756401062, -0.8781844973564148, 0.8925877809524536, -0.3323117196559906, -0.633026123046875, -2.7322604656219482, 1.4671162366867065, 0.2730260491371155, 1.4715831279754639, -0.1465083360671997, -0.5498006939888, -0.2312474101781845, 0.8760385513305664, 1.1708354949951172, -1.9186367988586426, -1.0295637845993042, -0.6645183563232422, -0.3433438241481781, 0.33484289050102234, -0.32037514448165894, 0.019596178084611893, -1.0823551416397095, -0.6761202216148376, 0.6920049786567688, -0.5530219674110413, -0.3719814717769623, -1.1563245058059692, -0.8356079459190369, 0.41002506017684937, -1.5148630142211914, 2.740004301071167, -0.0051192231476306915, 0.46977436542510986, 0.6301651000976562, -0.6450506448745728, 0.01542279589921236, 2.523895740509033, 0.9579649567604065, -1.0675253868103027, 0.7299880385398865, -1.0174484252929688, 0.4538813531398773, -0.20242442190647125, 1.5048542022705078, 0.13320156931877136, 1.0453933477401733, -2.4987733364105225, -0.5530493855476379, -0.8106728196144104, -0.5818579792976379, 1.0609074831008911, -1.3301185369491577, -0.04774961248040199, 0.602407693862915, -0.9918546080589294, -1.3443429470062256, 1.3292889595031738, 0.8155524730682373, 1.5999075174331665, -1.0731139183044434, -0.9671412706375122, 1.335935354232788, -0.8617002367973328, 0.45611846446990967, -1.4699407815933228, -0.49423182010650635, -0.3448008894920349, -0.13091880083084106, -0.03748532757163048, -0.4036894738674164, 0.39354395866394043, -0.5200835466384888, -0.8973191380500793, 0.10221679508686066, 0.5943399667739868, 1.2009223699569702, 0.20017416775226593, -1.5743849277496338, 0.8015591502189636, -1.4368703365325928, 1.581329107284546, -0.8986143469810486, 0.5553800463676453, -2.463073492050171, -0.9337173104286194, -0.8450084328651428, -1.8094391822814941, -0.23642030358314514, 1.4317457675933838, 3.4834868907928467, 0.4794190526008606, -0.7172079086303711, -1.5664663314819336, 0.3701862096786499, -0.31744733452796936, -1.0474882125854492, -0.4786036014556885, 0.8585776686668396, -0.029782610014081, -1.3559757471084595, 0.18256951868534088, -1.016294002532959, -2.2501113414764404, -1.1121841669082642, -0.6799651384353638, -0.801773726940155, -1.7691541910171509, -0.8552932739257812, 1.2213574647903442, -0.12288041412830353, -1.9063876867294312, 1.3700220584869385, 1.5403927564620972, -0.09580878913402557, 0.03128060698509216, -0.4142875075340271, -0.4989149868488312, 0.8667466044425964, -0.2760058641433716, 0.3497603237628937, -1.7810873985290527, -0.3766951858997345, 0.3220924139022827, 0.8486979603767395, -0.4909181296825409, -0.04166770726442337, -1.818045973777771, 1.070412039756775, -0.44704216718673706, 0.6627452373504639, 0.48389312624931335, -1.1997102499008179, 0.25303906202316284, 0.7230273485183716, -0.9414187669754028, 0.514471709728241, -0.16070207953453064, 1.3321326971054077, -0.24485865235328674, -0.8153741359710693, 0.8691291213035583, 0.5985668301582336, -0.41544273495674133, -0.6822076439857483, 0.15881049633026123, -1.9847079515457153, 0.5279217958450317, 1.470975399017334, 0.5672870874404907, 0.6768125295639038, 0.6468496322631836, -1.2110522985458374, 0.5202338695526123, 1.897342562675476, -1.9673316478729248, -1.952354073524475, 0.154080331325531, -0.3994666337966919, -2.307274341583252, 0.29401451349258423, -0.15598240494728088, 0.5909031629562378, 1.3625500202178955, 0.4605855941772461, 0.9069156646728516, -0.9371345639228821, 0.2921529710292816, -0.5923978090286255, -0.8594011068344116, 2.129413604736328, -1.0681142807006836, -0.5987129807472229, 0.08846145123243332, -1.4164998531341553, 1.2297561168670654, 1.4401808977127075, -0.3475956320762634, -0.6693263649940491, 0.681485652923584, 2.222116470336914, -0.12668780982494354, -0.7257294654846191, -0.13502287864685059, 0.6045257449150085, -1.9293262958526611, 0.3958514928817749, 1.5355280637741089, 0.015009546652436256, -0.3442420959472656, 0.5922293663024902, 0.931915819644928, 0.04973604530096054, 1.976362705230713, 0.4494142234325409, -1.6020519733428955, -0.27924036979675293, -0.28918081521987915, -0.18037135899066925, -0.6952582597732544, 0.18375390768051147, -0.20977044105529785, -1.5522031784057617, -0.7563302516937256, 0.9468480348587036, -1.3441343307495117, 0.5817029476165771, 0.7706260681152344, -0.09254904091358185, 0.8576749563217163, 2.2983243465423584, -0.40018564462661743, 0.8452988266944885, -1.5425140857696533, -0.2569168508052826, 1.7379062175750732, 0.20622724294662476, -1.055120825767517, -0.9687944054603577, 0.09946998208761215, -0.01125814113765955, -0.3693724274635315, 0.3752528131008148, -1.3335424661636353, -0.834994375705719, 0.6325401663780212, -0.0928989127278328, 0.372085303068161, 1.503166675567627, -0.7202169299125671, -0.05807288736104965, 0.9963270425796509, 0.692604660987854, -1.0118557214736938, 0.8802501559257507, -0.16389887034893036, -0.9176366329193115, 0.310675710439682, 0.4003981947898865, -1.6247491836547852, -1.3281797170639038, 0.35251694917678833, 1.328942060470581, -0.4977284371852875, 0.8870654106140137, -1.0567612648010254, -0.4036587178707123, 0.7058652639389038, 0.6084821820259094, -0.026336312294006348, 1.128060221672058, -0.3832497000694275, -0.16974946856498718, 0.6265882253646851, -0.9535720348358154, -1.31333327293396, -0.7312455177307129, 0.5322709083557129, -0.4342986047267914, 0.32520437240600586, 0.6910902261734009, 0.10638577491044998, 2.050349712371826, -0.13235945999622345, -1.3331032991409302, 0.19555996358394623, -0.8903144001960754, 0.20258663594722748, 0.4102790057659149, 0.4002790153026581, -0.6165481209754944, 1.273474931716919, 0.6011510491371155, 1.5853506326675415, 2.119063377380371, -0.7592307329177856, 0.3859235942363739, 0.021201318129897118, -0.4026646912097931, -0.5249566435813904, -0.8319610357284546, -1.4123146533966064, 0.3844280242919922, -0.9010348916053772, 0.6498562097549438, -0.2941376268863678, -0.08371489495038986, 2.215489625930786, -0.2828826904296875, 0.08228377997875214, -0.13627943396568298, -0.821636974811554, -0.0022595482878386974, -0.028927048668265343, 0.45614200830459595, 0.6904359459877014, 0.38349199295043945, 0.14817659556865692, -1.4339706897735596, -1.8195984363555908, 0.3656507134437561, -0.2684458792209625, -0.013475049287080765, 2.159133195877075, -1.248460292816162, -0.6820152997970581, -0.19996477663516998, 0.7592127323150635, 1.0204941034317017, 0.9648365378379822, -0.5627323389053345, 0.6830123066902161, -1.3408459424972534, 0.8737863898277283, 0.12559720873832703, 0.9906700253486633, -0.7205533385276794, -0.06218806281685829, 1.436096429824829, 0.09041634202003479, 0.37758100032806396, 1.3473247289657593, -0.9142882227897644, 1.1118144989013672, -0.7358976602554321, 0.2599460780620575, -0.6199042201042175, 0.12052925676107407, -1.2127045392990112, 0.5736391544342041, -0.2425595074892044, -0.4355985224246979, 0.06502096354961395, 1.1652582883834839, 1.9617059230804443, -1.0925202369689941, 0.9314808249473572, 0.7630256414413452, -0.9819591045379639, 0.4966058135032654, -1.0054888725280762, -0.5441628098487854, 2.2508957386016846, 2.2317144870758057, 0.053379256278276443, -0.09532427787780762, -0.014701028354465961, 0.6243757605552673, -1.2057065963745117, 0.5562793612480164, 0.3759314715862274, 1.4184669256210327, 1.3333812952041626, -0.7321636080741882, -1.1028586626052856, 2.0026369094848633, 1.2479678392410278, -0.8249266147613525, 0.3924664258956909, -1.3823192119598389, -1.314252257347107, 0.38876864314079285, -0.6088100671768188, -0.6684678792953491, 0.9044709205627441, 0.44967415928840637, -0.15706458687782288, -0.650131106376648, 0.2344518005847931, 0.1568417102098465, -1.2457836866378784, 0.0657103881239891, -0.0740707591176033, -1.0361758470535278, 1.4574577808380127, -1.2229517698287964, -1.0515592098236084, -0.4111458659172058, 0.03858339786529541, -1.2265268564224243, -0.7867063879966736, 1.2480539083480835, -1.1397147178649902, 0.05070066452026367, 0.1020241305232048, 1.5792864561080933, -0.8115089535713196, -0.15310460329055786, 1.4783638715744019, 1.6649945974349976, 1.5410847663879395, 0.17737023532390594, -1.3711473941802979, -0.06909801065921783, 1.1471827030181885, -1.646248698234558, 0.9701076745986938, -1.8319112062454224, -0.3256014287471771, 0.07799327373504639, 0.3852297067642212, 0.951403796672821, 1.2437256574630737, 0.6526747345924377, 1.3604328632354736, -1.4013324975967407, 1.12056303024292, -1.2965691089630127, -0.613370954990387, 0.8572989106178284, 0.07237741351127625, -0.5383710265159607, -0.47718971967697144, 1.27798593044281, 0.2810552716255188, -0.4619283080101013, 0.4660879671573639, -1.7905333042144775, 1.0957432985305786, -0.770612895488739, 1.8914411067962646, -0.12637858092784882, 0.40428027510643005, 0.06622562557458878, 0.3687314987182617, -1.3908129930496216, -0.8533968925476074, -0.14176616072654724, -0.6310998797416687, -1.3850399255752563, -0.3055393695831299, -0.9190400242805481, -1.9100247621536255, -0.21736223995685577, 1.6766294240951538, -1.4191795587539673, 0.811438798904419, -0.36214619874954224, 2.268636465072632, -0.21652504801750183, 0.6255521178245544, -0.5970253348350525, 2.392643690109253, 0.27788326144218445, 0.11682312190532684, 0.3159106373786926, -1.5265682935714722, -1.5614718198776245, -0.1376856118440628, 0.8712792992591858, 0.1850191354751587, 0.8660727739334106, 1.027146577835083, -0.31301167607307434, 0.767929196357727, 0.2431412637233734, -1.1041516065597534, -1.0713695287704468, 0.734626829624176, -0.3647783100605011, -0.28715837001800537, 1.2651350498199463, 0.879564642906189, 1.0705050230026245, -0.27138474583625793, 2.713914394378662, 0.344844251871109, 1.447328805923462, 0.0794840157032013, -0.9740209579467773, 0.9034350514411926, -0.0031692120246589184, 0.9040378332138062, -0.26010921597480774, -2.459784507751465, -1.1566976308822632, -0.13416261970996857, -1.0817906856536865, 1.3931028842926025, -0.599216639995575, -1.409950852394104, 0.6776590347290039, -0.8684872388839722, -0.08800625056028366, -1.9851034879684448, -0.8591405749320984, -1.4541141986846924, 0.5130705237388611, -0.7924891710281372, -0.46935367584228516, -0.2991367280483246, 0.5948898196220398, -0.7922906279563904, 0.9699430465698242, -1.9519299268722534, -0.08922211080789566, -1.3969972133636475, 0.6756006479263306, -0.10696454346179962, -1.0603684186935425, -0.1604735404253006, 0.23959816992282867, 0.08532977104187012, 0.2617044150829315, 0.225805401802063, -0.8560182452201843, 0.018291529268026352, -0.7569213509559631, -3.160196304321289, 0.011988724581897259, -0.0780908539891243, 1.6857646703720093, 1.5714069604873657, -0.5979201793670654, -0.3453986644744873, -2.0735559463500977, 1.5523974895477295, -1.172795295715332, -0.7989389300346375, 1.141921877861023, -0.11211852729320526, -0.12948347628116608, -0.14535564184188843, -1.1261247396469116, -0.8987070918083191, 0.5257666707038879, -1.392519474029541, -0.9655547738075256, -0.3571856617927551, -1.285251259803772, -0.4252980351448059, 0.6839274168014526, -1.3693116903305054, 0.663263201713562, 1.0003628730773926, 0.09393217414617538, 0.8134886026382446, 0.4803529381752014, -0.22175711393356323, -0.47511765360832214, 0.562842607498169, 1.2403528690338135, -0.6552725434303284, -0.8679159879684448, -0.8796493411064148, 1.4216374158859253, -0.35010504722595215, -0.9436953663825989, -1.1209681034088135, -2.9099011421203613, 0.38713207840919495, 0.9426136612892151, 0.6213026642799377, 1.02593994140625, 0.06182852387428284, 0.43246492743492126, -0.11824733763933182, 0.4286937415599823, -2.1517257690429688, -0.8751274943351746, 1.1707417964935303, -0.5593363046646118, 0.9087103605270386, -0.8088372945785522, -0.017370428889989853, -1.7461552619934082, -0.9776421785354614, -1.4103411436080933, 2.178609609603882, 0.36548545956611633, 1.3017692565917969, -0.8041091561317444, 0.7362666130065918, -0.0261875931173563, 0.7356333136558533, 0.27186068892478943, -0.15373384952545166, -0.8741275072097778, -0.4497032165527344, -1.60822331905365, 2.4538559913635254, 0.13666194677352905, -0.5824859142303467, 0.6702260375022888, 1.317320704460144, -1.3076070547103882, -1.2989729642868042, 0.8782042860984802, 0.9105308651924133, -0.16097943484783173, -0.27950531244277954, -1.003944993019104, 0.7500951886177063, -0.5314279794692993, 0.543205201625824, -1.5556823015213013, 0.3043915629386902, -0.8780491948127747, 1.1228843927383423, 2.0178990364074707, -0.6564433574676514, -0.8710094094276428, 1.7864668369293213, -0.23268970847129822, 0.20608755946159363, 0.662619948387146, -1.199157953262329, -0.8754876255989075, 1.622987151145935, -0.40188029408454895, -0.15537655353546143, -0.6603933572769165, -0.8187037110328674, 0.5553725361824036, 1.045949935913086, -0.9629175066947937, 0.36959731578826904, -0.29541686177253723, -0.41710150241851807, 0.586536705493927, 1.4297637939453125, -0.625292956829071, -1.5541281700134277, 0.6919813752174377, 0.47731029987335205, -0.09623810648918152, -1.3627569675445557, 0.7304812073707581, 0.9628459811210632, 0.41371408104896545, 0.0064746723510324955, -2.5301353931427, 0.8430628776550293, 0.03240012004971504, 0.19074328243732452, -0.30980944633483887, 0.7884541749954224, -0.16748033463954926, 0.2963760197162628, 0.6367378830909729, 1.3671637773513794, 0.25555503368377686, 1.9034290313720703, -0.7414703965187073, -0.0560183972120285, 0.06687379628419876, -0.5606350898742676, 2.3688910007476807, 0.3721449375152588, -0.30094754695892334, -0.23729336261749268, -0.9276466965675354, 0.9504925608634949, 0.19690686464309692, 0.7076674699783325, -1.0463933944702148, 1.4937355518341064, 0.2194078266620636, 0.8585553765296936, 1.6280293464660645, 0.437181293964386, -0.08066672831773758, -0.5274937152862549, 0.028207853436470032, -0.3620939254760742, 0.8192434906959534, 0.19999796152114868, -0.6272355914115906, -0.3722754120826721, 0.4666858911514282, 0.18062949180603027, -0.6085788607597351, 0.2100822925567627, -0.146020770072937, 0.34850168228149414, -0.16285914182662964, 0.434531569480896, 0.48155853152275085, -0.5449544191360474, 3.1430094242095947, 0.8847946524620056, -1.6818532943725586, 1.1762983798980713, 0.08399222046136856, -0.21676760911941528, -0.6089422106742859, 0.8498678207397461, 0.10592422634363174, 1.236629843711853, 0.1402444988489151, -0.21647199988365173, -0.35781681537628174, 1.5550636053085327, -1.1000690460205078, -1.1667194366455078, 0.8249025344848633, -0.1628728210926056, 1.2760727405548096, 0.003999846056103706, 1.5939338207244873, 0.5200300812721252, -0.2776930034160614, 0.7977904081344604, -1.7352263927459717, -0.7291494011878967, 0.6611125469207764, 0.8095293641090393, -2.9343435764312744, -1.9783920049667358, 0.10069236904382706, 1.5428814888000488, -0.16325847804546356, -0.2483988255262375, -1.455578327178955, -0.6772837042808533, 0.09590661525726318, 1.9280396699905396, -1.1066433191299438, -1.2968065738677979, 0.174860879778862, -1.383172869682312, 0.24549448490142822, -1.4471207857131958, 2.64520525932312, 0.7470552921295166, 0.47478020191192627, -0.9248292446136475, 0.05813183635473251, -0.9824337363243103, -2.0299668312072754, -0.297426700592041, -2.038295269012451, 0.21101157367229462, -0.8607844710350037, 0.5526278018951416, -1.613637924194336, -1.9616401195526123, 0.5973787903785706, -0.6322038769721985, -1.6968930959701538, 0.75832599401474, -0.09544157981872559, 0.6210211515426636, -1.4261043071746826, -0.22580917179584503, -0.040017399936914444, 0.8702735304832458, -0.7566590309143066, 0.0208504106849432, -0.7789016366004944, -1.1104695796966553, 0.16135002672672272, -0.7497004270553589, 1.9025119543075562, -0.6254437565803528, -0.11920779943466187, -0.5352415442466736, 1.8451021909713745, -0.06109823286533356, -0.42876362800598145, 1.2889844179153442, 2.0223329067230225, -0.45701244473457336, -0.09402604401111603, -1.855931282043457, -0.6389753222465515, -0.18240872025489807, -0.7671298384666443, 1.4380441904067993, -0.06437867134809494, -1.634374976158142, 0.7748736143112183, -1.4976519346237183, -1.4170457124710083, -0.006431623362004757, -0.9372912645339966, 1.0909022092819214, 1.38102126121521, -0.8597933053970337, -1.0570465326309204, -0.17321155965328217, 0.2300361841917038, -0.7120823264122009, -0.4860166311264038, -0.3408028781414032, -1.028236985206604, 0.8274960517883301, -1.130233883857727, -0.8387225270271301, -0.902225136756897, 1.3313302993774414, 1.6736332178115845, -2.5516114234924316, 2.6516284942626953, -0.6933006644248962, -1.878648281097412, -1.7177714109420776, -0.9025192260742188, 0.4698233902454376, -1.2248369455337524, -0.3628453016281128, 0.5657936930656433, 1.4918352365493774, 2.482693672180176, -0.1756608933210373, 0.7783483862876892, -1.1094770431518555, 0.8778937458992004, 0.14697198569774628, 1.3926537036895752, -0.4220361113548279, 0.8306674361228943, -0.13851122558116913, -0.3863447308540344, -0.3133520781993866, -1.4246160984039307, -1.0611103773117065, -0.7925165891647339, -1.5076977014541626, 3.1217641830444336, 1.0587847232818604, 0.7467495203018188, 0.44441163539886475, -0.16948281228542328, -1.6083430051803589, -0.15120887756347656, -0.21837803721427917, -0.2761000394821167, -0.05125980079174042, -0.9586578011512756, 0.20628266036510468, 0.9779478907585144, -1.2631158828735352, -1.6098301410675049, -1.4550248384475708, 0.21335722506046295, -0.982559323310852, 1.2750191688537598, 0.25362759828567505, 0.763168454170227, -0.5887988209724426, -0.6967865228652954, 1.0154142379760742, 1.3333595991134644, -0.09782084822654724, 0.0798768699169159, -0.5272701978683472, 1.8824952840805054, 1.0411579608917236, 1.7518852949142456, 0.2432437241077423, -1.063062071800232, -0.4210675358772278, -0.8387790322303772, 0.11214426159858704, -0.37124621868133545, 0.9451204538345337, 0.004179901909083128, -1.4760105609893799, 0.7664361000061035, -0.42125725746154785, 0.07954535633325577, -2.6238465309143066, -1.7195667028427124, 0.4584454298019409, -1.1463663578033447, 1.636050820350647, 0.5152267813682556, -1.081532597541809, 0.23208126425743103, 0.3390170633792877, 0.5330401659011841, -1.0395660400390625, -0.4966548681259155, 1.5448415279388428, -0.552315354347229, -2.298880100250244, 0.40508294105529785, -0.6094405651092529, 0.09711504727602005, -0.14182209968566895, -0.405672162771225, 1.2868378162384033, -1.0918210744857788, -0.45013511180877686, 0.4722932279109955, 0.37593889236450195, 0.8772670030593872, 0.6079259514808655, 0.4774395823478699, -1.4609320163726807, -0.3827251195907593, 0.9438074231147766, 1.6881533861160278, -1.8877464532852173, -1.4794750213623047, -1.6077548265457153, 0.2917935252189636, -2.4972879886627197, 1.3132420778274536, -1.4338738918304443, -1.2489888668060303, 0.38614434003829956, 1.36327064037323, 0.7589061260223389, -0.19579079747200012, -0.8519868850708008, 0.9263811707496643, -0.3558737635612488, 1.103372573852539, -0.4722552001476288, -0.5587588548660278, -0.4193172752857208, -1.0143500566482544, -0.7318897247314453, 1.607639193534851, -1.2402909994125366, 1.3272801637649536, -0.8342787027359009, 0.5715728998184204, 0.424161821603775, -0.7097800374031067, 0.33671626448631287, 1.1900100708007812, -0.39467132091522217, -0.011490389704704285, 0.7003685235977173, 1.0317941904067993, -2.9529526233673096, -0.09269317984580994, 1.009397268295288, 1.164009690284729, 1.0627530813217163, 1.0782959461212158, 1.1281341314315796, -1.9491527080535889, -0.6432419419288635, 1.312326431274414, -0.9536319375038147, 0.781997561454773, -1.1900759935379028, 1.1319583654403687, 1.2786535024642944, 0.17497102916240692, 0.9198427796363831, -0.7356226444244385, -0.445374459028244, 1.2475807666778564, -0.8410043716430664, 0.3370898962020874, -1.3238160610198975, -1.0286269187927246, -1.0699275732040405, -1.1685832738876343, 0.591277003288269, 0.737946629524231, 0.1771339774131775, -0.09173508733510971, 0.056963831186294556, -0.2690444886684418, 1.784781813621521, 0.4524540603160858, -0.6378757953643799, -0.43203502893447876, 0.6970692873001099, -1.3821641206741333, -0.34715569019317627, -1.0695000886917114, 0.3351868987083435, -0.49722179770469666, -0.9391639828681946, 1.2048996686935425, -1.0387768745422363, -1.7133147716522217, 0.5360369086265564, -0.8248493671417236, -0.5799347162246704, 1.0217182636260986, -0.7515723705291748, -0.3720698356628418, 0.8470661640167236, -0.647297739982605, -1.3611637353897095, -0.9828301668167114, -2.462740182876587, -0.7555541396141052, -0.9919296503067017, -0.5951197743415833, -0.8946316838264465, -1.3245166540145874, 1.311238408088684, -1.3696200847625732, -0.9202864766120911, 0.48116180300712585, -0.32780301570892334, -0.17731469869613647, -0.9508506059646606, -0.030718257650732994, 1.0904759168624878, -1.0132123231887817, 0.13119296729564667, -2.311108112335205, -0.2301853895187378, -0.3016001284122467, -1.267459750175476, 0.4825165271759033, -0.5355419516563416, -0.38337886333465576, 0.37320300936698914, -0.06674336642026901, 1.159152626991272, 1.5388931035995483, -0.4129011631011963, 0.022844435647130013, -0.5817838907241821, 1.8953540325164795, -1.284786581993103, -0.08730798959732056, 0.5601375102996826, 1.44181227684021, 1.5204893350601196, 0.5135757923126221, -1.5513238906860352, 0.08165886998176575, -0.8922078609466553, -2.9862401485443115, 0.31488779187202454, -3.0606331825256348, 0.7399737238883972, 0.33752942085266113, 0.5978130102157593, 0.00944619718939066, 1.759780764579773, 0.017494525760412216, 0.08945410698652267, -0.9263713359832764, -0.9898330569267273, -0.6292685270309448, -0.6227421760559082, 1.4811800718307495, 0.015927625820040703, 2.15742564201355, -1.0878362655639648, -0.6456555724143982, 1.880281925201416, -0.8661993741989136, -0.9051626920700073, -0.6867521405220032, -0.5822042226791382, 2.094437599182129, -0.5217621326446533, -0.11109402775764465, -0.022589009255170822, -0.6195423007011414, -0.9896035194396973, 0.24596045911312103, 0.0030241874046623707, -1.0658226013183594, -0.2396382987499237, 1.1301847696304321, -0.13075973093509674, 1.1954344511032104, 0.21140067279338837, -0.4274304509162903, 0.09557194262742996, -0.6228388547897339, 2.001899480819702, 1.2703748941421509, 0.4566381573677063, 1.0694819688796997, 0.978023886680603, 0.31306397914886475, -0.7646446228027344, -0.1678684949874878, -0.20338045060634613, 0.39714542031288147, -0.4760161340236664, -0.2551960349082947, 2.0919528007507324, -1.012748122215271, 0.8389367461204529, -0.08739591389894485, -1.3530645370483398, 0.24104614555835724, -1.3334908485412598, 1.258604884147644, -0.07549667358398438, 0.24182240664958954, -1.1026389598846436, 0.08809391409158707, -0.42695093154907227, -1.3640546798706055, 0.6381880044937134, 1.6073764562606812, 0.6655265092849731, 0.6210328936576843, -0.6961541175842285, -0.3686155378818512, 0.9472144842147827, 0.025471946224570274, 0.8974223136901855, 0.9874384999275208, -0.995482861995697, 0.3045988976955414, -0.17436975240707397, 0.8739110827445984, -0.6694526672363281, -1.759477138519287, 0.480877548456192, -0.5403515100479126, 0.5504010319709778, 0.4998825490474701, -1.116142988204956, -0.44260072708129883, -0.41735708713531494, 0.8662000894546509, -1.2579467296600342, 0.16935759782791138, 0.3221849501132965, 0.374418169260025, -0.32067006826400757, -1.431289792060852, 0.733945906162262, 0.626201331615448, -1.5897164344787598, -0.8467234969139099, -1.3818782567977905, 0.7114222049713135, 0.944021463394165, -0.5196000337600708, -0.15966741740703583, -2.1537437438964844, 0.03904452174901962, 0.5342734456062317, 0.009367178194224834, 1.7438597679138184, 1.747503399848938, -1.150766134262085, 1.0481023788452148, -2.325965404510498, 0.08364725112915039, -0.666587769985199, 0.016462257131934166, 0.3463592529296875, 0.8027510046958923, 0.251600980758667, -1.3304424285888672, -1.9125536680221558, -0.3213053047657013, 0.3744480013847351, 0.33847102522850037, 0.7740463018417358, -0.4144276976585388, 0.19118604063987732, 0.9404303431510925, 0.1288415640592575, -1.0796198844909668, -0.4336969554424286, 0.43718984723091125, -0.3908010423183441, 0.7996132969856262, 1.008391261100769, -0.34548819065093994, -0.5719612240791321, -0.22930015623569489, 1.3384448289871216, 0.8925557136535645, 1.7558709383010864, -0.6163074374198914, -0.1570136696100235, 1.1199111938476562, 0.44227659702301025, 0.6003465056419373, -0.4715738892555237, 0.6185547113418579, 1.2081178426742554, 0.059670478105545044, -0.2952626347541809, 0.5922288298606873, -0.017677022144198418, -0.8600045442581177, -1.2868645191192627, 0.3886620104312897, 0.6066957712173462, 0.9745467901229858, -0.08672041445970535, 0.417367547750473, 0.050985340029001236, 0.40006959438323975, 2.101290702819824, 1.137946367263794, 2.5215883255004883, -0.5882688164710999, 1.4692986011505127, 0.30761682987213135, 1.087863802909851, -0.30528271198272705, 1.1184083223342896, -0.6617004871368408, 0.30327922105789185, -0.45371928811073303, 0.43684202432632446, -0.402219295501709, -1.2836438417434692, -0.9963666200637817, 1.2296643257141113, -2.6260464191436768, -0.6213219165802002, -0.37987813353538513, 1.22586190700531, 1.4661835432052612, 0.046186164021492004, 0.24955663084983826, -0.8895830512046814, 0.481790691614151, 0.4152173697948456, 0.5472902655601501, 1.0377107858657837, 0.2660977840423584, -1.8898464441299438, -0.5465649962425232, 1.7923580408096313, 0.7922393679618835, 0.09466753900051117, -1.5185421705245972, 0.10501787811517715, -1.37357759475708, 1.6916158199310303, 1.8553956747055054, -0.2930896580219269, 1.2239688634872437, -0.8950181007385254, -0.5066931843757629, -0.32748299837112427, -1.3067775964736938, -0.6482178568840027, -1.6374380588531494, -1.108148455619812, 2.761857509613037, 0.8427960872650146, 0.9867222905158997, -0.4458712935447693, -1.7351274490356445, 0.8995985388755798, 1.3671878576278687, -1.254934310913086, -0.5638230443000793, -0.14807265996932983, 0.7987604141235352, 0.835221529006958, -0.15762373805046082, 0.3572704792022705, 0.8493913412094116, 0.12994498014450073, 0.5213212370872498, 1.0826424360275269, -0.19363053143024445, -0.49736034870147705, 1.48048996925354, -1.3570977449417114, -0.23739740252494812, -2.8087549209594727, 0.11149322986602783, 0.295483261346817, 0.5344749093055725, -0.19604730606079102, -0.019215678796172142, -0.28118473291397095, -1.9543532133102417, 0.8641201853752136, -0.29363951086997986, 0.25978899002075195, -1.4780738353729248, -1.1593999862670898, 0.23676423728466034, 0.6060585975646973, -1.2219460010528564, -2.326535224914551, 2.0222437381744385, -0.5939280390739441, 0.7180585265159607, 0.979240894317627, 1.3262742757797241, -0.9985949993133545, 0.631138265132904, -1.6146774291992188, 1.6503478288650513, -0.7368175983428955, -0.9602761268615723, -0.7176263928413391, -0.912093460559845, 1.1850107908248901, 1.2459120750427246, -0.3673580586910248, 1.0573819875717163, 1.5745056867599487, 1.0339362621307373, -0.8949428796768188, -0.5661167502403259, -0.41073697805404663, 0.49584949016571045, 0.8294032216072083, -0.46540379524230957, 0.16635197401046753, -0.9026193022727966, -0.025931788608431816, -0.6590822339057922, 1.7306113243103027, -1.0673578977584839, 0.3623902201652527, -0.2510085999965668, 0.0067110140807926655, -0.7631130218505859, 0.5708533525466919, 1.131795883178711, 1.0756690502166748, 0.5072861313819885, -0.6432810425758362, -0.5276290774345398, 0.642497181892395, -0.4009486138820648, -0.8634362816810608, -0.07878748327493668, -0.6613430976867676, 1.1971750259399414, -1.023687720298767, 0.8251251578330994, 0.13985085487365723, 0.8320724964141846, 0.2102142870426178, -2.0410523414611816, -1.903559684753418, 1.663844347000122, 1.4700253009796143, -0.3534936308860779, -1.1856193542480469, 0.5389024019241333, 1.0147210359573364, -0.42438316345214844, -2.4918746948242188, -1.0240799188613892, 0.5377069711685181, 0.6373869180679321, -0.484256386756897, -0.7867656350135803, -1.5280202627182007, -1.8791217803955078, 0.4679166376590729, -1.4171411991119385, -0.3455619812011719, -0.9837735891342163, 0.7950772643089294, -0.5647978782653809, -0.44241389632225037, -0.5894994735717773, 0.7553831934928894, -1.8947715759277344, 0.4538644254207611, 0.6533447504043579, 0.016675731167197227, 1.3205249309539795, -0.6935170292854309, 0.3388908803462982, -2.6653835773468018, 0.9021696448326111, 0.5475662350654602, 1.7773391008377075, -0.5700154900550842, -0.1816258579492569, 0.9681591987609863, 1.5645020008087158, -0.8679681420326233, -0.1013210266828537, -0.3104408383369446, 1.6105035543441772, -1.7747715711593628, -2.438624620437622, -0.10338487476110458, 0.28435227274894714, -0.7512211203575134, 0.7717677354812622, -2.008436441421509, 1.37041175365448, -1.4378489255905151, 0.043139200657606125, 1.314681887626648, 0.8932709097862244, -1.1001484394073486, -0.9297056794166565, -0.7411154508590698, 0.7474740743637085, -0.3629680275917053, 0.7723488211631775, 1.5258841514587402, 1.3004130125045776, 1.0118268728256226, -0.3156896233558655, 0.734539270401001, 1.0508549213409424, -1.052001953125, 2.6009418964385986, 0.15254777669906616, -0.6350967288017273, -0.13073362410068512, -0.0015462326118722558, 0.35113880038261414, -0.5566281080245972, -0.1781649887561798, 0.7306315302848816, 0.055397115647792816, -1.114902138710022, 0.10289483517408371, 0.532541811466217, 0.9000912308692932, 0.4835357069969177, -1.291715145111084, 0.5742776989936829, 0.5564633011817932, 0.5498759746551514, -0.24461527168750763, -0.0006942124455235898, -1.5616099834442139, -0.7186847925186157, -0.11235550045967102, -0.12548935413360596, 1.0239142179489136, -0.09650236368179321, 0.015307750552892685, 0.40885069966316223, 0.32671287655830383, 1.9446325302124023, 0.39331671595573425, -1.2383543252944946, 1.7266643047332764, 2.376274347305298, -0.522739052772522, 0.08390479534864426, 0.20215345919132233, -0.35053735971450806, -0.7664377689361572, 1.8756438493728638, -0.7505953311920166, -0.7645903825759888, 1.776025414466858, -1.7086243629455566, 0.5488060712814331, 0.43664121627807617, 2.47123122215271, 0.4080808460712433, 0.32008764147758484, 1.6233216524124146, -0.30578574538230896, -2.1425364017486572, 1.627984642982483, 0.8342512249946594, -0.06685230880975723, 0.18155238032341003, -1.8756870031356812, -0.3772463798522949, -1.4858068227767944, 1.2730958461761475, 1.1531033515930176, -1.1386879682540894, 3.632589817047119, 0.8282855153083801, -1.061075210571289, -0.9479851126670837, 0.7352325320243835, -1.2797982692718506, -0.5426375865936279, -0.3068177402019501, 0.5340854525566101, 1.280875563621521, 0.32951462268829346, 0.05952751263976097, 0.5511642098426819, -0.14775994420051575, -2.9909656047821045, -0.1706707924604416, 0.11735077202320099, -1.147669792175293, 0.5655124187469482, 0.5933840274810791, 0.1114894300699234, -1.1783561706542969, -0.27336177229881287, 1.4886306524276733, 0.45188698172569275, 0.21095608174800873, 0.295190691947937, 0.1793699711561203, -0.44630616903305054, -2.0687596797943115, 0.15327946841716766, -0.21780478954315186, -0.2296137660741806, -0.49995139241218567, -0.7422336339950562, -0.8448633551597595, -0.6955024600028992, 0.5264443755149841, -0.3701736032962799, 0.11317180097103119, -0.6314586400985718, 0.9664755463600159, 0.06532852351665497, -0.6886309385299683, 0.534542441368103, -0.40199053287506104, -0.5539277791976929, 1.4379031658172607, 1.2141432762145996, -0.1252938210964203, -1.7463033199310303, -0.799453854560852, 0.7212893962860107, -2.1497299671173096, -0.19594533741474152, -0.729637622833252, -1.2245943546295166, 1.1203540563583374, 0.6516322493553162, -0.5574829578399658, 0.33062663674354553, -0.7184547781944275, 0.6233211755752563, 0.6799286007881165, -0.32293128967285156, -1.6747596263885498, -1.172128677368164, -0.6513116359710693, -0.1402401179075241, 0.35438984632492065, 0.003750189207494259, -0.2001061737537384, 0.303561270236969, 2.6943342685699463, -1.4400392770767212, 0.12041091173887253, -0.3669312000274658, 1.2384244203567505, -0.6514298915863037, -0.954497754573822, 1.8019249439239502, 0.28032711148262024, -0.10114865750074387, 0.811703085899353, -0.6565941572189331, -0.148015096783638, 0.7662144899368286, -0.5152779817581177, 0.7191609740257263, 0.19501325488090515, 0.46727102994918823, -0.3893054723739624, 0.18594971299171448, -1.2065486907958984, 0.24710655212402344, -0.2524150609970093, -0.90751051902771, -0.711053729057312, 0.6148009896278381, -1.080868124961853, -0.18478083610534668, -0.9506866335868835, 1.5693422555923462, 1.796302080154419, -3.010258197784424, 0.6068685054779053, -1.606878638267517, 0.7972248196601868, -0.9243202209472656, -0.6013180017471313, -0.771862804889679, 0.2027946561574936, 0.2571325898170471, 0.03608248382806778, -0.8658818006515503, 1.291706919670105, 0.2268219292163849, 0.38148489594459534, -1.1725525856018066, -0.014683197252452374, 1.4245152473449707, -0.208715558052063, -0.6578449010848999, -2.0076286792755127, -1.159409761428833, 0.8395845890045166, -0.46041807532310486, 0.8776208758354187, -0.8892388343811035, 0.5415654182434082, -0.19986693561077118, -0.5293108224868774, -0.789413332939148, 0.5324558019638062, -1.2069694995880127, -1.1984775066375732, -0.3506465554237366, -0.18047784268856049, 1.1701538562774658, 0.7864140868186951, 0.8062877058982849, -0.5622341632843018, -0.9514818787574768, 0.36294472217559814, -0.8051255345344543, 1.5059679746627808, -0.34970951080322266, -0.2593004107475281, 0.8609259724617004, -0.8604555726051331, -0.2078588306903839, 0.13842622935771942, 0.5590737462043762, -0.43549439311027527, 0.276373952627182, -0.35926780104637146, 0.4106213450431824, 0.4602804481983185, -0.6486212015151978, 1.0162380933761597, 1.034847378730774, 0.7206778526306152, -0.2690161466598511, 2.3717973232269287, 0.007168550509959459, 0.5704048275947571, -0.6468119025230408, -0.5260350108146667, -0.9798714518547058, 1.5706450939178467, 0.7659916877746582, 1.2745810747146606, -1.2606658935546875, 1.7547860145568848, 0.18049809336662292, -0.9143466353416443, -0.14263123273849487, -0.04721362143754959, -0.38030144572257996, 1.4835200309753418, -0.705055832862854, -0.864587128162384, 0.31508076190948486, -0.016802530735731125, -0.3677625060081482, 0.026382677257061005, 0.6432411074638367, -0.14636999368667603, -0.4034338891506195, 0.1941755712032318, 0.6988069415092468, -2.15714693069458, 0.7607280611991882, -0.6557712554931641, -0.14829784631729126, -1.0617525577545166, 0.3017260730266571, -0.9559146165847778, -0.28660818934440613, -1.2407459020614624, -0.056496020406484604, 2.7952001094818115, 0.501535177230835, -0.5125217437744141, 0.9172121286392212, -0.29680660367012024, -1.4454951286315918, -0.6889428496360779, -1.2755227088928223, 0.8046473264694214, -2.073129653930664, 0.31629282236099243, -0.6201831698417664, -0.3748002052307129, 0.8415667414665222, -1.812444806098938, 0.4682067334651947, 1.699778437614441, 1.1511285305023193, 0.20446845889091492, 0.3593454658985138, 0.27093106508255005, -0.3950965106487274, 0.5119495391845703, -0.8012248277664185, 1.0137834548950195, -0.5944980978965759, 0.32601144909858704, -1.4948983192443848, 0.38199105858802795, 0.33417046070098877, 1.1725983619689941, -0.1726675182580948, -0.04934632033109665, -0.15907855331897736, 0.740292489528656, 0.5291399955749512, 1.4681779146194458, -0.8542562127113342, 1.1315629482269287, 1.1153523921966553, 0.9364225268363953, -0.5350883603096008, -1.0302984714508057, -0.17564630508422852, 0.28048548102378845, 1.0071295499801636, -0.45212894678115845, 0.05854606628417969, -0.29489219188690186, -1.007068157196045, -0.8547609448432922, -0.1193593218922615, 0.9014391899108887, 0.772940456867218, -2.4974136352539062, 2.356123685836792, 1.0449700355529785, 0.3436870276927948, -1.330759048461914, 0.4554707407951355, 2.043424129486084, -2.4307544231414795, 0.4824409782886505, -1.2257416248321533, 1.4214959144592285, 0.07225733995437622, -0.7490785121917725, 0.41082337498664856, -0.8521481156349182, 1.7737421989440918, 0.45948973298072815, 0.44078803062438965, 1.7013485431671143, -1.6792720556259155, -0.9409774541854858, -0.9459351897239685, -1.2783511877059937, 1.8756834268569946, -1.0336782932281494, -0.12588812410831451, -1.9637032747268677, 0.00880955345928669, -0.0278916172683239, -0.17499126493930817, -0.15804453194141388, -0.8504467010498047, -0.2893678843975067, 0.11342750489711761, -1.0133459568023682, 1.3255677223205566, -0.9436661601066589, 1.288703441619873, 0.6582211852073669, -2.2515029907226562, -0.7338763475418091, 0.4359167218208313, 0.004272698890417814, 0.7863158583641052, -1.3971538543701172, 0.48161616921424866, -0.3060183823108673, -1.837018370628357, 0.4558349847793579, 0.15990354120731354, -0.4954644441604614, -0.09418314695358276, 0.6616818308830261, 1.0283526182174683, -1.1511646509170532, -0.6282058954238892, 0.5384896397590637, -1.2576332092285156, 1.1408331394195557, 0.8429173827171326, 0.5513843894004822, 1.1398777961730957, 0.9427474737167358, -1.3018953800201416, 0.4277843236923218, -1.5526036024093628, -0.4835037291049957, -1.4376766681671143, 1.7641528844833374, 1.1433368921279907, 1.1063086986541748, -0.2758130133152008, -0.11003924161195755, 0.27168604731559753, 0.6369374394416809, -0.9743306040763855, -1.209952712059021, -0.1567143052816391, 0.7125149369239807, -0.5716062784194946, -0.09387104958295822, 1.0651164054870605, -0.049645569175481796, -1.304382562637329, 0.5413956642150879, 1.8356305360794067, 1.3542754650115967, 1.9137251377105713, 0.25556784868240356, -0.6306852102279663, 1.251511573791504, 0.32397422194480896, 0.28496992588043213, -1.9635220766067505, 0.4303097426891327, 0.5699795484542847, 0.6857393383979797, -0.334346204996109, -0.8529989123344421, -0.0965835452079773, 0.5765201449394226, -1.019968867301941, 1.6706833839416504, -1.3142776489257812, 0.6570243835449219, -0.9880065321922302, -0.3894651532173157, 0.18951617181301117, -0.29591894149780273, 1.3939555883407593, 2.0600476264953613, 1.0770422220230103, -0.9409566521644592, -1.3255866765975952, -0.19369371235370636, -0.9447429776191711, -0.08874291181564331, -0.5369901657104492, 1.1816397905349731, -0.20959176123142242, 0.5704859495162964, -0.39239373803138733, -1.3948243856430054, 1.4421323537826538, -1.9637893438339233, 0.14948219060897827, -0.08027906715869904, -0.6873076558113098, 0.25620049238204956, 0.15265892446041107, -1.061660647392273, -0.5252600908279419, 0.5504735112190247, 0.8690480589866638, 1.2098499536514282, -0.6570871472358704, -0.1140056699514389, -0.14867503941059113, -2.0028305053710938, -0.8524307012557983, 1.3316895961761475, 0.8069303631782532, -2.1197516918182373, 0.6338009238243103, 1.6952142715454102, -0.18212628364562988, 0.056860409677028656, -1.540279746055603, 0.15573057532310486, 1.3224191665649414, 0.35579827427864075, -0.5455564260482788, 1.0059926509857178, 0.9276881217956543, -0.12746091187000275, -1.5494078397750854, 1.119011402130127, 1.0785375833511353, 1.1236164569854736, 1.8397715091705322, -2.202998399734497, 0.44556352496147156, 0.21456144750118256, -0.4615182876586914, -0.8251000046730042, 2.2161312103271484, -1.136759877204895, 0.9808852672576904, -0.9865857362747192, -1.9882830381393433, -1.3041154146194458, 0.901011049747467, 0.5925784111022949, 2.2305915355682373, 0.6766839623451233, -0.5318397879600525, -0.5217832922935486, 1.8339871168136597, -0.0058877780102193356, 2.4925143718719482, 1.2748839855194092, -0.12266673892736435, -0.3574833869934082, 0.17474952340126038, 0.6507642269134521, 0.07054413855075836, 0.07166396826505661, -0.7620202302932739, -0.7535543441772461, -0.451175332069397, 0.3062732517719269, -0.18256472051143646, 0.9435778856277466, -0.95195472240448, 0.816426694393158, -2.128298282623291, -0.33540865778923035, 1.1983110904693604, -0.08025608956813812, -1.0811502933502197, -1.2172017097473145, -0.14833462238311768, 1.132658839225769, 0.3339137136936188, 1.6156330108642578, -1.713511347770691, -0.6519201993942261, 0.04387909173965454, 0.7021879553794861, -1.9152286052703857, -2.1012916564941406, 0.5807645916938782, -2.104656219482422, -1.418899655342102, -1.89893639087677, 1.2363688945770264, 0.21232102811336517, -0.30789193511009216, 1.2182530164718628, -0.26030251383781433, 0.29017552733421326, -1.3026920557022095, -0.6779950857162476, 1.2058964967727661, -0.7191917896270752, 0.4944078028202057, 1.5689280033111572, -0.06299963593482971, 0.8227973580360413, 0.4990355372428894, -0.2638188898563385, 0.3549025356769562, 1.1074248552322388, 0.9787319898605347, -1.0565654039382935, -1.6194144487380981, -1.0134199857711792, 0.5129020810127258, -0.35308316349983215, -1.5715978145599365, -1.0557725429534912, 0.13477110862731934, -1.3417896032333374, -0.1658032387495041, -1.0497299432754517, -0.7381606101989746, 1.3442500829696655, -0.4730132520198822, 0.34113582968711853, -1.0741262435913086, 0.6660265922546387, 0.36540526151657104, 0.38735878467559814, -0.8844829797744751, -0.5722142457962036, 0.3641316294670105, 2.244202136993408, -1.4124622344970703, 0.6101077795028687, 0.9638198614120483, -0.7733331918716431, -0.41417503356933594, 0.6431077718734741, 0.372749924659729, 0.6065793037414551, 1.0550559759140015, 1.9218854904174805, 0.2674652934074402, -0.6613089442253113, 0.7495644688606262, -0.18609251081943512, 0.49572843313217163, 0.002197840018197894, 0.8496010899543762, 1.310323715209961, -1.1995981931686401, -1.8473844528198242, 0.17740614712238312, -1.101854681968689, -1.6077101230621338, -0.6170130372047424, 0.028141457587480545, -0.8717246055603027, 1.2533077001571655, -0.598090648651123, 0.142997145652771, -1.3197296857833862, 0.8810359835624695, 1.2644116878509521, -0.2750924229621887, -0.416974812746048, 0.20813606679439545, 2.096400022506714, 0.43167802691459656, -0.4211939573287964, 0.4133344888687134, -0.639261782169342, -0.24026057124137878, 1.2146657705307007, 0.8877068758010864, 0.30693021416664124, 0.1820152848958969, -1.0968551635742188, 0.19873370230197906, 1.080169439315796, -0.5903436541557312, 0.8499307036399841, 0.016527272760868073, 1.9169559478759766, -0.9127244353294373, -0.7615117430686951, -0.6472998261451721, -1.6855477094650269, 0.6960619688034058, 0.8236701488494873, 1.8973431587219238, 2.2550249099731445, 0.4214492440223694, -0.12863227725028992, -0.15942130982875824, -0.7015755772590637, 0.8616718053817749, 2.720226764678955, -0.17583051323890686, 1.4939687252044678, 0.1591910421848297, -1.944927453994751, 0.6285760998725891, -0.5597412586212158, 0.059246938675642014, 0.5293723344802856, -0.2481546252965927, -1.5006492137908936, 0.12557660043239594, -0.655656099319458, -1.2189511060714722, 0.9002718925476074, -0.9711839556694031, 0.4933651387691498, -0.26932063698768616, -0.005977033171802759, 0.12908077239990234, -0.8165882229804993, 0.8259217739105225, -1.3099861145019531, -1.301630973815918, 0.309332013130188, 0.6095259785652161, 0.8255380988121033, -1.6195100545883179, 0.2742500603199005, 0.7454448938369751, -0.2331269234418869, 1.6781202554702759, 0.69807368516922, -0.8701804280281067, -2.2425243854522705, -0.6804775595664978, -0.5238458514213562, -2.3007137775421143, -1.2506850957870483, -0.5168816447257996, -1.101123332977295, 0.2745298743247986, -0.19675520062446594, 0.27795490622520447, -0.3207722008228302, 0.4150422513484955, -0.5612577199935913, 0.10820002853870392, 0.9455236792564392, -0.8776155114173889, -1.0956604480743408, -0.5759074091911316, 0.4646778404712677, -0.052572645246982574, -0.5343493223190308, -1.38540780544281, -0.5254904627799988, -0.7412154674530029, -0.132569819688797, -0.1574888676404953, -0.8387921452522278, 0.819557249546051, 1.22284996509552, -0.1859603077173233, 0.2460019588470459, 1.09541654586792, -0.49669501185417175, 1.3068976402282715, 0.747748613357544, 2.7572271823883057, -1.1715465784072876, -0.7329992651939392, -0.6511369347572327, 0.37952011823654175, 0.4744725823402405, 0.4085450768470764, -1.6631197929382324, -0.5300307273864746, 0.8529825210571289, 0.1596761792898178, -1.0435460805892944, -0.6962502598762512, -1.2136270999908447, 0.5024955868721008, -0.12488295882940292, -0.08680939674377441, 1.2806572914123535, -0.30289778113365173, -0.9176290035247803, 2.213919162750244, 0.45371025800704956, 0.09392368048429489, 0.003925927449017763, -0.39355310797691345, -0.31084632873535156, -0.5426431894302368, 0.9691672325134277, -0.31810933351516724, 1.453967809677124, -0.9379947781562805, -1.34330153465271, 0.7513729333877563, 0.6705268025398254, -2.005012273788452, -1.0811889171600342, -0.8065493106842041, 1.1813830137252808, 0.5443437099456787, -1.1064784526824951, 0.5490415096282959, 0.429838627576828, -0.48998838663101196, 0.14652119576931, 0.6525201797485352, 0.33875054121017456, 0.556509792804718, 0.21965156495571136, -0.048207033425569534, 1.5746283531188965, -0.7308838367462158, 0.46017271280288696, -1.7975209951400757, -0.5067631006240845, 1.6491081714630127, -0.4586080014705658, -0.8536746501922607, -0.3784913420677185, 0.8400874733924866, -0.44913262128829956, 2.1440389156341553, -1.6147631406784058, -1.0073612928390503, -0.1866844743490219, 0.6244720220565796, 1.5307307243347168, -0.45953986048698425, 0.7798233032226562, -0.9290891289710999, 1.415171504020691, -0.5764344334602356, 1.7518752813339233, 2.0209343433380127, -0.4303268492221832, -0.6586072444915771, -1.2271305322647095, 0.6136413216590881, 1.1738754510879517, 2.0239784717559814, -1.1373573541641235, -0.380653977394104, 0.3040103316307068, -1.662658452987671, -0.7989736199378967, -0.6277495622634888, 0.008269701153039932, -0.49467790126800537, 0.25269705057144165, 0.7837258577346802, -1.3590675592422485, 0.521451473236084, 0.8205761313438416, -0.11611975729465485, 0.8979417085647583, -0.944494366645813, 0.574018120765686, -0.20361585915088654, 2.1374382972717285, -0.5565314888954163, 1.9635893106460571, -0.03652970865368843, -0.5756325125694275, 0.6894035935401917, 0.873683750629425, 0.7455007433891296, -1.301106333732605, -0.891743004322052, -0.0806770995259285, 0.30848273634910583, -1.0512315034866333, 1.0620431900024414, -1.2713719606399536, 1.0585236549377441, 0.18289592862129211, 1.8351110219955444, -0.7387392520904541, -0.4517675042152405, -1.7606593370437622, 0.85175621509552, 0.20870637893676758, -2.55794620513916, 0.771535336971283, -2.089693546295166, -2.002164602279663, -0.19868896901607513, 0.49510839581489563, -1.0849229097366333, 2.741780996322632, -0.5640632510185242, 0.2063082456588745, -1.1763943433761597, -0.555779218673706, 0.27207380533218384, -0.865651547908783, -1.9777590036392212, 0.7864454984664917, -0.5831882953643799, 1.3132121562957764, 2.120478391647339, 0.08954916149377823, 0.34182223677635193, -0.12364627420902252, -1.2271031141281128, -0.4882602095603943, -0.9132757782936096, 2.319409132003784, -0.28225183486938477, -0.5774189829826355, -2.1040420532226562, -2.7004451751708984, -0.3578735589981079, -1.060126781463623, 0.3445998728275299, -0.6734104752540588, -0.07911717891693115, -0.07524977624416351, -0.4988565444946289, -0.2939780056476593, 0.939148485660553, -1.4390554428100586, 1.04139244556427, 1.8315423727035522, -0.8558319211006165, -1.3403195142745972, -0.31818869709968567, 0.9163594245910645, -0.5869101285934448, -0.24669213593006134, -0.6562049984931946, 0.12298501282930374, -0.2435269057750702, 0.7667203545570374, -0.7887301445007324, 0.7213473916053772, 0.6326209902763367, 0.8394735455513, 2.0613443851470947, -0.017391305416822433, 0.19256152212619781, 0.4921684265136719, -1.3867511749267578, -1.1133443117141724, 0.5779586434364319, 0.8865733742713928, 0.4620262682437897, 0.1078563928604126, -0.5937896370887756, 0.8604262471199036, 1.2379978895187378, 0.3798671364784241, -2.25045108795166, -0.4405534267425537, 1.0095304250717163, -0.12255705893039703, 0.2207527458667755, 0.23202763497829437, 1.914166808128357, -0.19440561532974243, 0.7065308690071106, -0.25145024061203003, 1.1058770418167114, 2.241642713546753, 0.46209338307380676, -0.460401713848114, -0.29379957914352417, -0.7464027404785156, -1.5725713968276978, 1.1015206575393677, 1.6022963523864746, 1.4094078540802002, 1.6530728340148926, -0.020124465227127075, 0.9308682084083557, -0.700856626033783, 0.3663627803325653, -0.34043020009994507, 0.8265874981880188, -0.0035068511497229338, 0.5600448250770569, 0.6493639945983887, -0.02664276957511902, 0.19579832255840302, -0.4192214012145996, -1.4539082050323486, 0.6096042394638062, 0.044594090431928635, -1.2331948280334473, 1.8604317903518677, -0.00927717611193657, -0.8664944767951965, 0.31881415843963623, 0.3140982389450073, -0.8778078556060791, 0.5418881177902222, 0.5309426784515381, -0.4993499517440796, -0.25480857491493225, 0.04628516361117363, 2.508317470550537, 0.19848226010799408, 0.12590527534484863, -0.6705732941627502, -0.495441198348999, -0.7110998630523682, -0.6983548402786255, 0.2691349983215332, 0.3194728195667267, -0.200112983584404, 0.41339218616485596, -0.7551698684692383, -1.2397574186325073, -0.13169479370117188, 0.6663769483566284, -1.3665430545806885, -0.38634011149406433, 2.142181873321533, 0.10971485078334808, -0.7420057654380798, 1.1933027505874634, -0.06026633828878403, 1.3749216794967651, 0.41143110394477844, 1.519844651222229, 1.6468305587768555, 0.6074537038803101, 0.8458653688430786, 1.7862484455108643, -0.9426368474960327, 0.43298739194869995, -0.08798912167549133, 0.6674562096595764, -0.037155844271183014, -1.5498238801956177, 0.9390581250190735, -0.1394205391407013, -0.26207414269447327, -2.5658516883850098, -1.5724844932556152, -0.7726128101348877, -0.2966267466545105, 1.3585710525512695, -0.6554873585700989, 1.4002532958984375, 1.5862938165664673, 0.978668749332428, 0.003328063990920782, 0.25307878851890564, 1.171301245689392, 1.1889891624450684, 0.1887713074684143, 0.28296393156051636, -0.5404981970787048, 1.1221758127212524, -2.1626076698303223, -0.41931620240211487, 0.16494031250476837, 1.9383078813552856, -1.0431796312332153, -1.9661767482757568, -0.8174232244491577, 0.5168899893760681, -0.9655247926712036, 0.5815626978874207, -1.064019799232483, 1.5516360998153687, -1.0420113801956177, 0.3991064429283142, -1.507222056388855, 1.2953877449035645, -1.376752495765686, 0.5382517576217651, 1.2579160928726196, -0.38264742493629456, -1.4800862073898315, 0.6561093330383301, 0.4564953148365021, -1.3882859945297241, -0.0011883582919836044, 0.38182899355888367, -0.2474738210439682, -1.0882110595703125, 0.9587072134017944, -0.4664931893348694, -0.7822992205619812, -0.07644928246736526, 0.16883297264575958, 0.8329822421073914, -0.09383533149957657, 1.0448381900787354, -1.7204021215438843, -1.2661434412002563, 0.8025143146514893, 0.3611436188220978, 1.4594486951828003, 1.3338849544525146, 1.8714107275009155, 2.173689842224121, -0.6671565175056458, -0.3141418993473053, -0.5169064402580261, -0.2863856256008148, 0.12363633513450623, -0.8160837888717651, -0.8589990735054016, -0.41639581322669983, 1.1066571474075317, 0.29211607575416565, 0.3426567018032074, 0.2813912332057953, -1.3096904754638672, 1.5506267547607422, -0.1794082373380661, 0.7920937538146973, 0.3735819160938263, -0.6950609683990479, 0.2539101541042328, -1.1670491695404053, 1.7775721549987793, 0.5767677426338196, -1.1592891216278076, -0.09152856469154358, -0.3814410865306854, -0.4125889837741852, -0.25265979766845703, 0.37927451729774475, 0.5470957159996033, -0.005166937131434679, -0.8820973634719849, -1.839164137840271, -0.4885726869106293, -0.2218378633260727, -0.46125876903533936, 0.8338752388954163, 0.32923805713653564, -0.5155105590820312, -1.218257188796997, -0.9935042858123779, -0.007979893125593662, -1.2456375360488892, 1.0353777408599854, 0.30206289887428284, 0.20786216855049133, 0.4084939956665039, 0.1032179743051529, 1.0454009771347046, -1.191286563873291, 0.46602749824523926, -0.5939791798591614, -1.1084213256835938, -0.4468534588813782, -1.396684169769287, 0.7059646248817444, -0.8828475475311279, -0.2869029939174652, 1.064253330230713, -0.9914763569831848, 1.2062318325042725, 0.6791681051254272, -0.9575685858726501, -0.42916339635849, -2.218533754348755, -0.9090805649757385, 0.5049846768379211, -0.49703100323677063, 0.12559786438941956, -0.2763383388519287, 0.8438339233398438, -1.5509223937988281, -0.5143042802810669, -0.8570623993873596, 0.4787677526473999, 0.14619873464107513, -0.5734663605690002, 0.07373631745576859, -1.0086302757263184, 0.6153616309165955, 0.2747558057308197, 1.3434675931930542, 0.7456327676773071, 0.44274428486824036, -1.4867345094680786, -0.05419360101222992, 1.3809893131256104, 1.047720193862915, -1.592788815498352, 0.46511051058769226, 0.18581140041351318, -0.014953380450606346, -0.5385745763778687, -1.4234284162521362, 0.17507119476795197, 0.4370662569999695, 1.8372812271118164, -0.5113674998283386, 0.31929001212120056, -1.3575360774993896, 0.6244230270385742, 0.43422970175743103, 0.9103871583938599, -0.08088309317827225, 0.10574125498533249, -0.7381845712661743, -0.000873828015755862, -0.8669769763946533, 0.27128851413726807, 1.1623032093048096, 2.8230702877044678, 0.1280512511730194, 0.6578044891357422, -0.1433602273464203, 0.6924753189086914, -0.7407722473144531, 1.0227867364883423, 0.637977659702301, 0.4566255509853363, -0.03342313691973686, 0.7405174374580383, 0.17415790259838104, -0.6892041563987732, -0.6551396250724792, -0.5685489773750305, 0.07657764852046967, 0.8085375428199768, 1.0909538269042969, 0.8286435604095459, -0.36597394943237305, 0.8441762328147888, 0.25626039505004883, -0.07853670418262482, 0.20755422115325928, 0.30083608627319336, 0.31763455271720886, -1.1276999711990356, -0.7167667746543884, -0.9465426206588745, 0.5613378882408142, -0.683528482913971, -2.35135555267334, -0.7376161217689514, -1.8258646726608276, -1.850757122039795, -0.3077487349510193, -0.6532624959945679, -0.3602195382118225, 1.1902819871902466, 0.04043256491422653, 0.37269988656044006, 1.1420261859893799, -0.8175245523452759, -0.07575145363807678, -1.6379140615463257, -0.8410117030143738, -1.2616633176803589, 0.43912410736083984, 1.0897207260131836, -0.4542418122291565, 0.5531725883483887, -0.4603448808193207, -0.028126776218414307, -0.356204628944397, 0.9835590124130249, -1.0999505519866943, -1.1818110942840576, -0.2225068062543869, -0.20873109996318817, 0.7092612385749817, 1.496634840965271, 0.7063280344009399, -0.3825623691082001, 1.2257885932922363, 0.2768822908401489, 0.19422636926174164, 0.27445361018180847, -1.9012335538864136, -0.6265247464179993, 1.009691596031189, -0.9848384857177734, 1.1118944883346558, -0.7697039842605591, 0.6421511173248291, 0.021159591153264046, -0.6690762042999268, 0.1982784867286682, -1.5041441917419434, 1.99862802028656, -0.79300457239151, -1.2936065196990967, 0.9462534785270691, -0.9389546513557434, 0.36492234468460083, 2.508345603942871, -0.46782469749450684, 0.6677505373954773, 0.6135671734809875, 1.245417833328247, 0.995434582233429, -0.09670964628458023, -0.6253919005393982, -0.14530810713768005, -0.8984506130218506, -2.161083459854126, -0.08743315190076828, 1.052473545074463, 0.5012340545654297, -0.8059099912643433, 0.9085175395011902, 0.9110235571861267, 0.009958059526979923, 0.2585964500904083, -0.8273272514343262, -0.3013301193714142, -0.24583281576633453, 0.4801032543182373, 0.12722443044185638, 0.3491259217262268, -0.8765830397605896, -0.8863816261291504, -0.34522151947021484, -0.8336463570594788, 0.8982146382331848, -1.1357570886611938, -0.3931823670864105, 0.4330284893512726, -1.4134702682495117, 1.9325454235076904, 0.6947839856147766, -1.795284390449524, -1.335495948791504, 0.9845931529998779, 0.41484588384628296, -0.058663509786129, -0.5952751040458679, 0.8993213176727295, 0.2577800154685974, -0.711851179599762, -1.2116769552230835, 1.374162197113037, 0.9940730929374695, 0.9764865636825562, 2.2437798976898193, -2.199049472808838, -1.1076405048370361, 0.3613535165786743, 0.29058656096458435, 0.9775984287261963, -0.6200795769691467, 0.015322006307542324, -0.5228198170661926, -0.03632880002260208, 1.882156252861023, -1.3617444038391113, 0.8762726783752441, 0.40006133913993835, -0.3231727182865143, -1.8252413272857666, 0.45423099398612976, -0.4817746579647064, -0.6076038479804993, 0.843681812286377, -0.009730069898068905, 0.936246931552887, -1.3741005659103394, 1.3714810609817505, 1.3625400066375732, -1.7251112461090088, -1.3940870761871338, 0.5638011693954468, -0.7050907015800476, -0.5961185693740845, 0.32661300897598267, 0.9585010409355164, 2.1998989582061768, 0.40427690744400024, 1.5405489206314087, 0.44510260224342346, 1.0797947645187378, -0.8766165971755981, 0.4982812702655792, -0.5780104398727417, 0.007198765873908997, 0.8576842546463013, 1.6610394716262817, -0.9395379424095154, -1.4578120708465576, -0.34555232524871826, -0.20873886346817017, -0.002361532999202609, -0.7441540956497192, 1.1091221570968628, -0.5273888111114502, 0.9555454850196838, -0.22984452545642853, -0.3936685025691986, -0.3046077787876129, 1.7928729057312012, 1.4644231796264648, 0.2881145477294922, -0.2613776922225952, 0.7549581527709961, 1.364558219909668, -0.8439763784408569, 0.22567275166511536, -0.22236159443855286, 0.723172128200531, -1.6160730123519897, -0.13624724745750427, 0.8474322557449341, -0.6713812947273254, -0.5775588154792786, -1.0669996738433838, 0.6197482943534851, -0.23850305378437042, -0.9053307175636292, -0.0852133259177208, 0.02581380493938923, -0.9736021757125854, -0.8931939005851746, 0.7568669319152832, -1.3540536165237427, -0.10767189413309097, -0.8567076325416565, 1.7145732641220093, -0.7414206862449646, -2.1956839561462402, 0.4301071763038635, -0.8871715664863586, -0.11479885876178741, 0.7160158753395081, 1.1462278366088867, -0.634937047958374, 0.6258040070533752, -0.612235426902771, 0.7319864630699158, 0.3359698951244354, 0.765699028968811, 0.35692426562309265, 0.20494918525218964, -1.4793949127197266, 2.027083158493042, -1.4226760864257812, -0.4346325993537903, 0.38137298822402954, -0.3537745177745819, -0.805766224861145, -1.2644423246383667, 0.882887601852417, -1.4040471315383911, -0.9663584232330322, 0.6242533326148987, -0.4703490734100342, -1.201952338218689, -0.3845245838165283, -1.093782901763916, 0.3029956519603729, -0.13277746737003326, -1.241512417793274, 2.1313517093658447, -0.3556218147277832, 0.2197193056344986, -0.427687406539917, -0.4008385241031647, 0.14398568868637085, 2.0372114181518555, 0.6732779145240784, -0.21382875740528107, -0.8053898811340332, 0.2938518226146698, -1.4889177083969116, 1.0760835409164429, 0.29958054423332214, -1.5463380813598633, 1.554097056388855, 1.491628646850586, 0.6330892443656921, 0.879119336605072, 0.23800954222679138, 0.024054573848843575, -2.0628888607025146, -0.1944057047367096, -0.508208155632019, 0.07737961411476135, -1.7259570360183716, -1.3998401165008545, -1.1860692501068115, -0.7841585278511047, -0.07103234529495239, 0.2171449065208435, -1.5837533473968506, 1.208937168121338, 0.32431039214134216, -0.515934944152832, -0.8272039294242859, -1.4540201425552368, -0.18574126064777374, -0.3394559323787689, -1.0786406993865967, 0.758598804473877, 1.3340275287628174, 0.9346449971199036, -1.9108819961547852, 0.4111451804637909, -0.8805949091911316, -0.021096348762512207, -0.7853350639343262, 0.19873984158039093, -1.0963999032974243, -1.0149165391921997, 1.1339164972305298, -0.48073074221611023, 0.6777816414833069, 0.12338535487651825, 0.8495883941650391, -2.052537441253662, 2.159973621368408, 0.07120241969823837, 0.2551403343677521, -0.09320276975631714, 0.9366675019264221, 0.6354069113731384, 2.5557897090911865, -0.9687662124633789, 0.8391069769859314, -0.7802276015281677, 0.7176607251167297, 0.7637536525726318, 0.09679509699344635, 0.09092115610837936, 2.289306163787842, -0.007075615227222443, 0.416953980922699, -1.403098464012146, -0.5306718945503235, 1.5492655038833618, 0.21404121816158295, -1.2923463582992554, 1.5232467651367188, -0.9890463352203369, 0.4731394648551941, -0.8076984286308289, 0.5730442404747009, -0.7091361880302429, -0.0781317725777626, 1.4994298219680786, -0.1794329434633255, -0.1416233777999878, 1.0779931545257568, -0.18334084749221802, 0.9198294281959534, 0.05881866067647934, 0.483843594789505, -0.5856428146362305, -2.1410000324249268, 0.3011959493160248, -0.7013450860977173, 0.5081564784049988, 1.6083076000213623, 0.31154385209083557, -0.5216386914253235, 0.44183167815208435, -0.08651407808065414, 0.20764464139938354, -1.082161545753479, -0.21396216750144958, -1.0487691164016724, 0.422018826007843, -1.1167800426483154, 1.204788327217102, 0.7191941142082214, 1.011724591255188, -0.7907848954200745, 0.12010347098112106, 0.40584489703178406, -0.012783167883753777, 0.6865575313568115, 0.9382490515708923, -0.6988790035247803, -0.33194461464881897, 1.0735552310943604, -0.9563829302787781, 0.5097500085830688, -0.3069863021373749, 0.9809583425521851, 0.18941453099250793, -0.14693395793437958, 0.14834238588809967, -0.5866771340370178, 0.31984174251556396, 0.34311026334762573, 0.4639993906021118, -0.26899784803390503, 0.20752492547035217, 0.8186060786247253, -0.07057245820760727, 0.4652497470378876, 1.782790184020996, 1.1910817623138428, 0.4879412353038788, 1.6605173349380493, 1.1465954780578613, -0.24464330077171326, 0.5616587400436401, 1.3081077337265015, -0.5917279720306396, 0.006489210296422243, 0.4381689131259918, 0.9159616827964783, 0.382477730512619, -0.7172165513038635, -0.6493756771087646, -0.5044869780540466, 0.14800222218036652, 0.5689697861671448, -0.012303488329052925, 1.3726909160614014, -0.7293801307678223, -0.2898492217063904, -0.6240013241767883, -1.4088184833526611, -0.8254708647727966, 0.4733516573905945, 0.4906683564186096, -1.2107402086257935, 0.12939192354679108, -0.5184797048568726, -0.10155456513166428, 0.48625648021698, -0.32824504375457764, -1.564895749092102, -0.39838162064552307, 1.0228010416030884, -0.6070638298988342, -0.5114426612854004, 0.49806761741638184, 0.2964623272418976, 0.4881211221218109, -0.2535049617290497, -1.5791126489639282, -0.08516262471675873, -0.8420619368553162, 0.6637925505638123, 0.7903534173965454, -0.13484840095043182, -0.23409537971019745, -0.8970027565956116, -0.5076290965080261, -1.981823205947876, 1.7896181344985962, -1.0671948194503784, 0.36411064863204956, -1.084972620010376, 0.6280678510665894, 0.4020213782787323, -0.5297192335128784, -0.11596771329641342, 0.20292328298091888, -1.7583292722702026, 0.27103981375694275, 1.331335186958313, -0.19911135733127594, 1.330571174621582, 0.8048697113990784, -0.0015149314422160387, 0.9086540341377258, 0.4888920187950134, -0.5631319880485535, 1.267400860786438, 0.4786897897720337, -0.14627015590667725, -0.9064050316810608, -0.5237167477607727, -0.39140239357948303, 1.3628178834915161, 1.8008449077606201, 0.24032698571681976, 1.3638386726379395, -0.49127551913261414, 0.014640817418694496, -1.6032062768936157, 0.49993279576301575, 0.37130916118621826, -0.5284377336502075, 1.155193567276001, -1.266893744468689, -0.03957946226000786, -1.0664572715759277, -0.33791249990463257, 1.1625343561172485, 1.224894404411316, -0.04626988247036934, -0.3463283181190491, 2.1501545906066895, -0.4767743945121765, 1.2129321098327637, -0.4790745973587036, -1.6102995872497559, -0.030777059495449066, -1.4136933088302612, -0.33083683252334595, 0.30975356698036194, 0.4841669797897339, -1.7276644706726074, -0.46626120805740356, 0.8331655263900757, 1.4899914264678955, 1.797005534172058, -1.0779244899749756, 1.4052950143814087, -1.571913242340088, -0.3995382785797119, -0.14345556497573853, -1.8680802583694458, 0.9953337907791138, 0.36221960186958313, -0.9847177863121033, 0.3945929706096649, -0.09006144106388092, 1.5779314041137695, -0.46876055002212524, 0.07241414487361908, 0.596039354801178, 0.5132291913032532, 0.9894844889640808, -1.2971112728118896, -0.9482176899909973, 1.3401679992675781, 0.6920057535171509, -1.2173945903778076, -0.8635516166687012, -0.6705290675163269, 0.12757481634616852, 1.3511959314346313, -0.7387478947639465, 0.4683653712272644, -0.96021568775177, -0.0776316449046135, 0.026599405333399773, -2.03953218460083, -0.4053070545196533, -0.41484716534614563, -1.3013710975646973, 0.9500086307525635, -1.0955067873001099, 1.3938482999801636, -0.0657404437661171, -1.0667496919631958, -0.26185664534568787, -2.984971046447754, -1.6051771640777588, 1.4531770944595337, -1.4391285181045532, 0.08742671459913254, 1.826284646987915, 0.33450204133987427, -0.9864741563796997, 1.458664894104004, 1.8717055320739746, 0.12949275970458984, 0.08725695312023163, 1.7486708164215088, -0.4198409914970398, 1.2257354259490967, 0.2011290043592453, 0.5299769043922424, -0.7080602049827576, 1.3208860158920288, 0.7125998735427856, 0.5585564374923706, -1.3742141723632812, -0.8384637832641602, -0.41973841190338135, -1.5037033557891846, -0.8070881366729736, -0.7535741925239563, 0.7615915536880493, 0.5437606573104858, -0.07589947432279587, 0.7569248080253601, -0.9551278948783875, -1.407650113105774, -0.020418453961610794, -0.2434651106595993, 0.11907290667295456, -0.0835791602730751, 1.0133039951324463, -0.27756956219673157, 0.4713882803916931, -0.5228054523468018, 0.6532077789306641, 1.196702480316162, 1.3487526178359985, 2.0333092212677, -0.34802383184432983, 0.14087262749671936, -0.20741736888885498, -0.7974205613136292, 0.4150516390800476, -0.24086980521678925, 1.0613594055175781, -1.3947163820266724, -0.7110433578491211, 0.6512750387191772, 0.2611771821975708, -0.1791333109140396, -0.829215943813324, 1.1763131618499756, -1.1524168252944946, 0.5870460867881775, 0.07262671738862991, 0.781481921672821, 0.4688999652862549, 0.42030978202819824, -0.3305412232875824, -0.14753642678260803, -1.4917091131210327, 1.377270221710205, 0.04024694114923477, -1.207632303237915, -0.3024682104587555, -0.949105441570282, -0.08527600765228271, -1.3224847316741943, 1.3769229650497437, 1.3182121515274048, 0.36525198817253113, 0.21268628537654877, 0.5802934765815735, 0.4221276640892029, 1.21510910987854, -0.11774150282144547, -1.163532018661499, -0.510349690914154, -0.28356292843818665, -0.1070464625954628, -0.7610429525375366, -2.0058767795562744, 1.7236371040344238, 0.3600330650806427, 1.1931979656219482, -1.6837375164031982, -1.5670007467269897, 2.4121453762054443, -0.005770571995526552, -0.7326512932777405, 0.17725908756256104, -0.5366490483283997, -1.3802160024642944, 0.7211218476295471, -0.40356266498565674, -0.5723326802253723, -0.9123920798301697, -0.7436488270759583, 0.6381847262382507, 0.8988760113716125, 0.5937564969062805, 0.18117989599704742, 1.2503646612167358, -2.3904812335968018, 1.1106356382369995, -0.8881859183311462, 2.3884243965148926, 0.7701784372329712, -1.5132490396499634, -0.2976042330265045, -0.4438839256763458, 1.7468006610870361, -0.027382833883166313, -2.2110564708709717, 0.7199731469154358, 0.8662155866622925, 0.3508622348308563, 1.4937411546707153, -0.42961251735687256, 1.148556113243103, 0.20127242803573608, 1.0777132511138916, 0.7517138123512268, 1.2951050996780396, 1.5892380475997925, 0.6808647513389587, -0.6715143918991089, 1.4545310735702515, -0.47900864481925964, -0.10986430943012238, -0.4109202027320862, 0.24053619801998138, -0.9292087554931641, 0.37756243348121643, -0.48243674635887146, -0.085160031914711, 0.39211204648017883, 0.5652381777763367, 0.4106922149658203, -1.3275498151779175, -1.8664695024490356, 0.6568406224250793, 0.6382436752319336, 0.902743935585022, 1.303930401802063, -1.2051008939743042, 0.8817108273506165, 1.0761239528656006, 1.3244472742080688, -0.5847799181938171, 0.7028570175170898, 1.734877109527588, -0.06494538486003876, -1.483924388885498, 0.6449146866798401, 2.023831367492676, 0.907136857509613, -0.10533197969198227, -0.043724920600652695, -0.6403043866157532, 0.73728346824646, -1.7321672439575195, -0.2726808488368988, 2.189507007598877, -0.2431752234697342, 0.6915829181671143, 1.757987380027771, -0.5383248329162598, -1.0921101570129395, -1.0964857339859009, -0.14092111587524414, -0.7940107583999634, -0.8889736533164978, 0.09957762807607651, -0.31420934200286865, 2.081888198852539, 0.316709965467453, -0.34307917952537537, 0.8672043085098267, 0.2586839497089386, 0.1561601459980011, 0.8413789868354797, -1.46757173538208, -1.4799883365631104, 0.7655859589576721, -2.471745252609253, 0.19680027663707733, -0.2733556628227234, 2.144704580307007, 0.9753903746604919, -3.0214357376098633, 1.0073612928390503, -0.566293478012085, -0.6491168737411499, -0.7750537395477295, -1.2428879737854004, -1.3584411144256592, -1.6522748470306396, -0.6571181416511536, 0.3100685775279999, -0.2681502103805542, 0.05307254195213318, 1.869690179824829, 0.01804235577583313, -0.3925642967224121, 1.2862608432769775, 0.10334600508213043, 1.8748605251312256, 0.6196819543838501, -2.2587685585021973, 1.515989065170288, -0.9829999804496765, -1.3455331325531006, -1.5237369537353516, 0.4418705701828003, -1.7915538549423218, 0.3855069875717163, -0.4001295268535614, 1.0559144020080566, -0.8898886442184448, 0.35819143056869507, 0.23599645495414734, 0.01444544643163681, 0.30845338106155396, -0.4830339550971985, -1.3515326976776123, -1.8571877479553223, -1.9825773239135742, -0.5118070840835571, -0.6085137724876404, -0.0145100774243474, 0.05105502903461456, 1.388358235359192, 2.319888114929199, 0.586423397064209, 1.9285269975662231, 0.7527485489845276, -0.06719134002923965, 0.9128967523574829, 0.9360845685005188, -0.5276052355766296, 0.7332324385643005, 0.6729833483695984, -1.0139521360397339, -1.360088586807251, 0.45531150698661804, -0.6174036860466003, -0.3943403959274292, 2.47699236869812, 2.51541805267334, -1.1704281568527222, -0.9142793416976929, -0.34816625714302063, 1.1049379110336304, -1.6839139461517334, 0.7354421019554138, -0.33724549412727356, -0.905015766620636, -0.4501310884952545, -0.018972141668200493, -0.1490863859653473, 0.3013787865638733, -0.10767418146133423, -0.08595732599496841, 0.37117865681648254, 1.106459617614746, 0.014012167230248451, -0.010917018167674541, 0.06167268753051758, -0.35652926564216614, 0.46275514364242554, -0.826666533946991, 0.3308066427707672, 1.073776364326477, 0.21592992544174194, 2.5583138465881348, -0.5363108515739441, 0.05322954058647156, 1.6849445104599, -0.06601621210575104, 0.33573850989341736, -0.18772141635417938, -0.9987261891365051, 1.6347472667694092, 1.6482244729995728, -1.2912886142730713, -0.261679083108902, -0.4583456814289093, -1.8059395551681519, -0.3445342183113098, 1.0182043313980103, 1.2854701280593872, -0.9429996013641357, 2.575836658477783, 0.7092648148536682, -0.686883807182312, -0.2703220546245575, -0.43510884046554565, 0.09304086118936539, 0.15428270399570465, -0.6491584181785583, 0.6475375890731812, 0.6437357068061829, 1.8510559797286987, 0.028598761186003685, 0.5985658168792725, -1.1308597326278687, 0.876730740070343, 1.86668860912323, -0.6996549963951111, -2.8619604110717773, 2.2595324516296387, -1.1909667253494263, -0.7464213371276855, -1.577082633972168, 1.7153459787368774, 1.476631760597229, -0.3440170884132385, 1.4109325408935547, 0.13948044180870056, -0.5902087092399597, 0.8288211822509766, 0.6222580075263977, 0.2145553082227707, 0.8109312653541565, 0.25073620676994324, -0.4179486632347107, -0.18728166818618774, -2.3839149475097656, 0.11305689066648483, -0.7231617569923401, -0.8494063019752502, 0.022685594856739044, -0.08504275977611542, -0.36595991253852844, -0.5648849606513977, 1.5448639392852783, 0.28174009919166565, 0.7908962368965149, 1.5867578983306885, -2.5125720500946045, 0.8578973412513733, 0.9664521217346191, 0.2104613035917282, 0.8916915059089661, 0.3201972246170044, -0.8268106579780579, 0.7997979521751404, 0.6990584135055542, -0.4928848147392273, -1.295170783996582, 0.3229238986968994, 0.431424081325531, 1.1770422458648682, -0.6365188360214233, 0.5739865303039551, 0.7629243731498718, -0.3903922438621521, 0.8039366602897644, -1.1251626014709473, -0.14956244826316833, 0.524840772151947, -0.663569450378418, 0.2109350860118866, -0.02905377931892872, 0.39455607533454895, -0.24041970074176788, 0.634842038154602, -0.14967051148414612, -0.22210845351219177, -1.874947190284729, 1.3142342567443848, 1.5600045919418335, -2.17769718170166, -0.777356743812561, -1.0685361623764038, -1.1682547330856323, -0.6659524440765381, 1.76408851146698, -1.4144984483718872, 0.022685782983899117, 1.5146808624267578, -1.0723825693130493, 0.7369181513786316, 1.4168604612350464, 0.7029531002044678, -0.7648796439170837, -0.8573172688484192, 1.2861148118972778, -1.3526819944381714, 0.3474540114402771, -0.16890907287597656, -0.16272756457328796, -1.4276005029678345, -1.8546525239944458, -0.4456164240837097, 1.1085284948349, -0.41575729846954346, -0.20406189560890198, 1.1081608533859253, 0.43807950615882874, -0.9130480289459229, 0.37150752544403076, -0.3518708050251007, 0.8124309778213501, -0.15137693285942078, -0.773732602596283, -0.37296855449676514, 0.19951418042182922, -0.31212833523750305, 0.10252555459737778, -0.3492045998573303, -2.8397457599639893, 0.6770656704902649, -1.766752004623413, -0.730900228023529, -0.7477331757545471, 1.2567944526672363, 1.0671112537384033, 1.498970627784729, -0.016782965511083603, 0.4696401357650757, 0.9273537397384644, -0.2056698501110077, 1.0902684926986694, 0.5835305452346802, 0.026382317766547203, 0.0020787548273801804, 1.8940610885620117, -2.0140960216522217, -1.489781141281128, -0.09431297332048416, 0.25873950123786926, 1.4674091339111328, -0.7870136499404907, 0.341012179851532, 0.6526122689247131, 0.02927965484559536, -0.440137654542923, 0.757914125919342, -0.7664117813110352, -0.8722240328788757, -0.5804342031478882, -0.9773160219192505, -1.6936880350112915, 0.14340147376060486, 1.8713418245315552, -0.3942449390888214, -1.4101916551589966, 0.7298257350921631, -0.10881655663251877, -2.2470829486846924, -2.4198644161224365, 0.08231816440820694, 1.9859205484390259, -0.20398297905921936, 0.13685743510723114, 0.3285342752933502, 0.32637548446655273, 0.6240800619125366, -0.8282873630523682, 1.4988552331924438, 1.8728704452514648, -0.4336242079734802, 0.5649253129959106, -1.0875377655029297, 0.7014822959899902, -0.7321524024009705, -0.3740369975566864, 0.1069200411438942, 0.8503943085670471, -0.8125468492507935, -0.8139180541038513, 0.5436638593673706, -0.8235061168670654, -0.4787576496601105, -0.5366357564926147, -0.8331652283668518, -1.2556438446044922, 0.6749211549758911, 0.24053163826465607, -2.6836283206939697, 0.8705641031265259, 0.04988586902618408, -0.1145072728395462, -0.3819439709186554, 1.954923391342163, 0.5234178304672241, -0.05150235816836357, -0.6930367946624756, -0.41721242666244507, 0.49333655834198, 1.4670872688293457, -1.1385180950164795, 0.12894921004772186, -1.4789588451385498, -2.0692238807678223, -0.6489008665084839, -0.2730584740638733, 0.6450523138046265, -1.323171854019165, 0.8150993585586548, -0.3244129419326782, 0.0685495138168335, 1.2336450815200806, 0.36406680941581726, -0.39932316541671753, -0.6398113965988159, -0.2609950602054596, -0.5204212665557861, 0.04191236570477486, -0.38097018003463745, -1.2193522453308105, -0.23312556743621826, -0.15120790898799896, -0.631557822227478, -2.3233633041381836, -0.3031432032585144, -0.1757659614086151, 1.4498757123947144, 0.42328616976737976, 2.3937978744506836, 0.5800063014030457, -0.07856327295303345, -0.4693447947502136, 0.7831286787986755, 0.24112211167812347, 1.8657243251800537, -1.3607722520828247, -1.5310221910476685, -1.5722321271896362, 1.535304069519043, -0.07215674221515656, -0.26605477929115295, 0.5667423009872437, 0.49823930859565735, -2.1894099712371826, 0.16900072991847992, 0.37324634194374084, 1.7784470319747925, 0.826893150806427, 1.3621593713760376, 0.7405069470405579, 0.1943202018737793, 0.8846462368965149, 2.307004928588867, -2.3433003425598145, -0.11114849150180817, -0.6939780116081238, 0.2420605719089508, -1.2073743343353271, 0.6744793653488159, -0.11187371611595154, 1.226550817489624, 0.06968166679143906, -0.40226051211357117, 2.956568717956543, -0.027481921017169952, 1.5235671997070312, 0.13060292601585388, 0.5605790615081787, -0.16205675899982452, -0.11361977458000183, -0.1472572684288025, 0.42470410466194153, 2.537834405899048, 0.7219730615615845, 1.1100660562515259, -1.219839096069336, 1.3251655101776123, -0.2934821546077728, 0.9836246967315674, -0.5235424637794495, -0.916923463344574, -0.661183774471283, 1.2553012371063232, 0.5164153575897217, -0.8159982562065125, 0.06381785869598389, 0.6251601576805115, -0.03664971515536308, -0.005009530112147331, 1.1023175716400146, 0.5035138726234436, 0.6297337412834167, 0.7928259968757629, 0.660415530204773, -0.0007149386219680309, 0.3359642028808594, 1.4168020486831665, 0.754466712474823, -1.5376864671707153, -0.08086942881345749, 0.8504412770271301, 0.5048105716705322, 0.5596067309379578, 0.891423761844635, -1.2909705638885498, 1.2766093015670776, 0.09145338833332062, -1.5496594905853271, 1.1490190029144287, 0.38811036944389343, 0.5101231932640076, 0.4892450273036957, 0.4548674523830414, 0.02572237327694893, -0.19928056001663208, -1.153794765472412, 0.30884766578674316, 0.7270200848579407, 0.294016569852829, 0.13933131098747253, 2.224193811416626, 1.4871940612792969, 0.27692335844039917, 1.1232507228851318, -1.6345360279083252, -0.6687573194503784, 0.18771493434906006, 0.468449205160141, 1.3012454509735107, -0.06489809602499008, -0.20949697494506836, -1.4889096021652222, -0.37679848074913025, -0.2073788046836853, -2.52296781539917, 0.990759015083313, -0.2512495815753937, 0.4258996546268463, 1.0137873888015747, 0.7086674571037292, -0.41579824686050415, 0.4514772295951843, 0.5578732490539551, -0.8256699442863464, 0.01714758388698101, -1.1689085960388184, -0.3064541518688202, -1.01291823387146, 0.42582857608795166, 0.6880851984024048, -0.43655139207839966, 0.7622976303100586, -2.7711048126220703, 0.04122886806726456, 2.323314905166626, -0.3102816641330719, 1.3487170934677124, -0.0035535164643079042, -0.6774783730506897, -1.5710043907165527, 1.0674477815628052, 0.23385147750377655, 0.30199116468429565, -1.37515389919281, 0.05408356338739395, -0.9031537771224976, -0.780780553817749, 1.864210844039917, 1.7082440853118896, -1.2869086265563965, 0.5648482441902161, -0.1474546194076538, -1.042584776878357, 0.5524885058403015, 0.8748272061347961, -0.5159449577331543, -0.2996143698692322, -0.660881519317627, -0.7045758366584778, -1.8976292610168457, 1.9845631122589111, -0.535747230052948, -0.7371474504470825, -0.6863900423049927, 0.2852703928947449, 1.9810516834259033, 1.2153087854385376, -0.6832306981086731, -0.05791810527443886, -0.10468604415655136, -1.0715649127960205, 0.4905422031879425, -0.34973305463790894, -0.4649372100830078, -0.014588738791644573, -0.06026272848248482, 0.34740710258483887, 0.6212157607078552, -1.775926113128662, 0.6975163221359253, -0.0709359273314476, 0.0726911723613739, -1.154792308807373, 0.8644309639930725, 1.972448706626892, -1.7218998670578003, -0.021557843312621117, 0.13157232105731964, 1.4305894374847412, -1.92716383934021, -0.8752931356430054, 1.3673516511917114, 0.7354910969734192, -0.6333063840866089, -0.4707001745700836, 0.374726265668869, -1.4148908853530884, -0.21043312549591064, -1.5168731212615967, 1.7748509645462036, -2.4631893634796143, -0.26743096113204956, 0.4722317159175873, -0.9276403188705444, -0.5039647817611694, 0.25670522451400757, 0.8238661885261536, 1.5362211465835571, 1.0465055704116821, -0.11386474967002869, 0.36187806725502014, 0.6524274945259094, -0.2630157172679901, 1.1359295845031738, 0.12199092656373978, 1.6812324523925781, 1.8532003164291382, -0.6356552243232727, 0.14212463796138763, -1.1328023672103882, 0.024696988984942436, -2.5157909393310547, -0.7096874713897705, 0.04852903261780739, 1.2026872634887695, 0.7028531432151794, -0.9102402329444885, 1.219305396080017, -0.36981117725372314, 0.7148123979568481, -1.2777910232543945, -0.14994724094867706, -2.51627254486084, -0.10310825705528259, -0.3714183568954468, -0.12077057361602783, -1.2437409162521362, -0.9776654243469238, 0.05348223075270653, -0.7051038146018982, -1.5550649166107178, -1.0180354118347168, 2.319894790649414, 0.17622056603431702, 1.095292329788208, 2.110578775405884, -0.6148803234100342, 0.6296666860580444, -0.22807012498378754, -0.025036178529262543, -0.494045615196228, 0.32579922676086426, -0.6078048944473267, -1.449108600616455, -0.3277375102043152, 1.4914988279342651, -2.0975568294525146, -0.4414169490337372, 1.0503655672073364, -0.39277660846710205, -2.065986156463623, -0.12925519049167633, -0.12644848227500916, -1.9848250150680542, -0.5148403644561768, 1.1772685050964355, 0.4521690011024475, -1.5781304836273193, -0.17126810550689697, 0.5678942203521729, 0.6976426243782043, 0.928947925567627, 0.5065500736236572, -0.0847587063908577, 1.459960699081421, -0.33668819069862366, -0.23316256701946259, -0.19164450466632843, -1.7105227708816528, -0.422558069229126, -0.21568208932876587, -0.2211378812789917, -0.9016913175582886, -1.4056594371795654, 1.3010773658752441, -1.202022671699524, -0.6199455857276917, -1.3121994733810425, 0.046278972178697586, -2.0152740478515625, -2.1627564430236816, -1.0098589658737183, -1.1704134941101074, 0.9481313228607178, -0.2648075819015503, 1.8287010192871094, 0.540118396282196, -0.19672490656375885, -0.6448181867599487, 0.21564005315303802, 1.6003247499465942, -0.7324057817459106, -0.47427070140838623, -0.2608044445514679, -1.6575698852539062, -0.05835758522152901, -2.0532965660095215, -0.1624225676059723, 1.3773123025894165, 0.6344675421714783, 0.650108814239502, -0.17020581662654877, -0.16657200455665588, -1.567862868309021, 0.08313598483800888, -2.2980434894561768, 0.8649973273277283, -0.09053168445825577, -0.9162153601646423, 0.36883530020713806, -0.09006348252296448, -1.4776597023010254, -1.056308627128601, 0.6370274424552917, 0.21366754174232483, 1.0009633302688599, -0.08458118885755539, 0.3799974322319031, 0.20802345871925354, -0.20492789149284363, -0.12602312862873077, -1.4332349300384521, 0.18541157245635986, 0.6113089919090271, -0.23124049603939056, 0.3484936058521271, 0.1758253574371338, -1.192065715789795, -1.9167054891586304, -0.6177098155021667, -0.45831459760665894, 0.7633076310157776, 0.8746339678764343, 1.7539516687393188, 1.9087496995925903, -0.3045634925365448, -0.7460250854492188, 0.602337658405304, -2.335897445678711, 1.3938556909561157, 0.07709165662527084, 0.1287865787744522, -0.23328818380832672, 0.30647462606430054, 1.1439818143844604, 0.3681947588920593, -0.22803708910942078, 0.15578411519527435, 0.8148835301399231, 1.674515962600708, -0.9809933304786682, -0.07761392742395401, 0.3582065999507904, -0.7764893174171448, -0.1365104466676712, -0.24247491359710693, 0.552448034286499, 2.8241443634033203, 2.6987266540527344, 1.5379377603530884, -0.21910125017166138, 1.4201565980911255, -0.6206439137458801, 1.1927353143692017, -0.38281169533729553, -1.0857222080230713, 0.6549741625785828, 0.000253294943831861, -0.48824208974838257, 0.04945286735892296, -0.9687535762786865, 0.25255873799324036, -0.1295226514339447, 0.7330438494682312, 0.9227823615074158, 0.6894649267196655, 0.4189941883087158, -2.2496402263641357, -0.7895052433013916, 0.9020137190818787, 1.8232522010803223, -0.502383828163147, 0.2955154776573181, 0.5425963997840881, -0.24535442888736725, 0.3537026345729828, 1.6721560955047607, -0.10708378255367279, -0.4052683115005493, 1.0811612606048584, -0.053463079035282135, 0.7936869859695435, -1.1612588167190552, 0.5331123471260071, -0.48539528250694275, -1.18636953830719, 1.598940372467041, -0.20753151178359985, 0.5608283877372742, -0.3680129945278168, 1.4804612398147583, 0.10555259138345718, -1.1445789337158203, -0.6801295280456543, 0.5220412611961365, -2.182720422744751, -0.2532239556312561, 2.381082773208618, 0.7225592136383057, 0.3143676519393921, 0.3059503436088562, -0.5341058373451233, 2.4114015102386475, 0.5065677762031555, 0.604016125202179, -1.0159664154052734, -0.8321080803871155, -0.6050559878349304, 1.131312370300293, -1.0812910795211792, -1.6860803365707397, -0.5777829885482788, 0.00959201343357563, 0.49632400274276733, -0.12677788734436035, 1.7410310506820679, 0.41155025362968445, 1.270273208618164, -0.28196555376052856, -1.447014570236206, -2.697571277618408, 2.2086222171783447, -0.7410117983818054, 1.2376679182052612, -2.4157893657684326, 0.6167784929275513, -0.09391295164823532, -0.37585940957069397, 0.8809348940849304, -0.25955188274383545, -0.8725294470787048, -0.17880409955978394, 0.6985837817192078, 0.509328305721283, -0.25214388966560364, 0.4742383658885956, -0.3183222711086273, -0.8551719784736633, 0.5180589556694031, 1.3517857789993286, 0.5943161249160767, 0.27224522829055786, -0.018624532967805862, -0.7821558117866516, 0.1629858911037445, -0.4564357101917267, 1.2557331323623657, -2.953752040863037, 1.8913365602493286, 0.5885975956916809, -0.544605553150177, 1.4084950685501099, 0.8383526802062988, 0.38268861174583435, 0.32124894857406616, -0.04168025404214859, 0.6163336634635925, -0.3851703107357025, -0.44686606526374817, 0.3796193599700928, 1.0145490169525146, 1.1216050386428833, 0.9928398132324219, -0.5796305537223816, -0.701606035232544, -1.7701694965362549, -0.08402889221906662, 0.11972574889659882, -0.03964487835764885, -0.7871252298355103, 0.2631992697715759, -0.45282283425331116, 0.6161689162254333, -1.3140218257904053, 1.4487756490707397, 0.0382855050265789, -0.835374116897583, 0.2170950025320053, 1.0088880062103271, -0.5794716477394104, 0.5956586003303528, 1.1152368783950806, -3.069401979446411, 0.42847809195518494, 1.7884966135025024, -1.449733018875122, -1.4266778230667114, 1.5039681196212769, 0.5684241652488708, -1.0303056240081787, -0.33727195858955383, 0.4964047372341156, 1.6441919803619385, 0.44367918372154236, -0.23829719424247742, -0.7501466870307922, 1.049660563468933, -0.42102527618408203, 1.1809961795806885, 0.9931139945983887, 0.5071569681167603, -0.7856137156486511, -1.5753976106643677, -0.7488747835159302, -0.7586873769760132, -0.15968579053878784, -0.4993129074573517, 0.9564887881278992, 0.8066837191581726, -0.13083793222904205, 0.12271779775619507, 0.47928106784820557, 0.6379165053367615, -0.5304909944534302, -0.34630489349365234, -0.5106595158576965, -0.6589052677154541, 0.45890897512435913, -0.07749912887811661, -0.6073734164237976, -0.10736940056085587, 0.0031688050366938114, 0.6006662845611572, 1.1286369562149048, 1.5019245147705078, 0.5366590619087219, 1.3820751905441284, 0.21985268592834473, -1.2492307424545288, -1.132817268371582, -0.36376041173934937, -0.3206310272216797, 1.8173224925994873, 0.03417712822556496, 0.40362444519996643, -0.7512410879135132, 0.7011451721191406, -0.9026849865913391, -0.31962111592292786, 2.1855223178863525, 0.46868470311164856, -1.0559704303741455, 1.1054999828338623, -0.43028298020362854, -0.629443347454071, -1.3695026636123657, 0.4937373101711273, 0.6269723176956177, -0.08444689959287643, -0.564385712146759, -0.6825722455978394, 0.13818351924419403, 1.0370131731033325, -0.36071300506591797, 0.2667410969734192, -1.8148729801177979, 1.0012028217315674, -0.5276565551757812, -0.4336104989051819, -0.5082113742828369, -0.42447322607040405, -1.0529119968414307, -1.0925066471099854, 1.1414289474487305, 1.338642954826355, -0.5073851346969604, 1.7834429740905762, -1.3635635375976562, -0.3105255961418152, -0.1291731297969818, 2.5109968185424805, 0.4243728518486023, 0.9612115025520325, 0.016411514952778816, -1.4391319751739502, 0.3802059292793274, 0.8846855163574219, 0.18229883909225464, 0.27518606185913086, -0.8871667385101318, 1.4815019369125366, -1.4350135326385498, 0.9670374989509583, -0.1046236902475357, 0.22435909509658813, -1.6807458400726318, -0.4501831829547882, -1.0420953035354614, 0.04897807911038399, 0.21409334242343903, -0.34407514333724976, -0.2616262435913086, 0.32674676179885864, -0.6296198964118958, -0.5251358151435852, 0.1799275279045105, 0.1584436446428299, -2.0209579467773438, -0.6165483593940735, -0.2500794231891632, 0.38902154564857483, 0.592566967010498, 0.2116483449935913, -1.087791919708252, -0.5664721727371216, 0.4985831081867218, -0.529522180557251, -0.7399424910545349, 1.8517204523086548, 0.13750115036964417, -0.37018904089927673, -0.7220857739448547, -0.05769145488739014, 0.06093098968267441, 0.9192833304405212, 0.1904495358467102, -2.138684034347534, -1.3556352853775024, -1.2896854877471924, 0.14097677171230316, -0.24220964312553406, 0.5676320195198059, 0.47560739517211914, 0.3170749545097351, 0.4120960831642151, -0.16499929130077362, -0.3319292366504669, -0.34816864132881165, -0.37135204672813416, 0.006902690045535564, -1.7352938652038574, -1.05807363986969, 0.34370216727256775, -0.9976282715797424, -0.9285658597946167, 0.350191593170166, -0.5915853381156921, -0.5793145895004272, 0.8954792022705078, -0.02727719210088253, 0.1282053142786026, -0.5466738939285278, 0.6906489133834839, 0.1674460768699646, 0.5369788408279419, -1.0782594680786133, 1.8379666805267334, 0.927380383014679, -0.3772204518318176, 0.7165631055831909, -1.3374420404434204, 0.7424089312553406, 0.12426163256168365, 0.8436741828918457, -1.6568337678909302, 0.33851107954978943, 0.7653517723083496, -1.4718745946884155, -0.9112651348114014, 1.1009738445281982, 0.22080253064632416, 1.2409201860427856, 0.17514558136463165, 0.6060121059417725, 0.8115768432617188, -0.4355735778808594, 0.06422293931245804, -0.6754264831542969, 0.32951655983924866, 1.1799293756484985, -2.278902530670166, -0.8196870684623718, 0.15279193222522736, -0.4006781280040741, -0.12691734731197357, -0.9202356338500977, -1.1971114873886108, 0.4246084988117218, -0.9069486260414124, 1.1367197036743164, 0.6312170028686523, -0.9971827268600464, 0.4671090543270111, 0.548258900642395, -1.6993858814239502, -0.20247432589530945, -0.20697902143001556, 1.4471272230148315, -0.7692095041275024, 0.5931155681610107, 0.27656424045562744, 0.08983239531517029, 0.140338733792305, 1.256239414215088, 0.5291410088539124, -0.6620996594429016, 0.9083278179168701, -0.5224093794822693, 1.602472186088562, -0.8118414282798767, -1.1040529012680054, 0.023262998089194298, 0.9100222587585449, -1.3060517311096191, -1.3550634384155273, -1.8130614757537842, 0.6832825541496277, 0.04958641156554222, 0.1400996595621109, 0.12191074341535568, 1.2392219305038452, -0.8369826674461365, 1.6275988817214966, -2.1016714572906494, 0.0953313410282135, -0.010933557525277138, -0.1525152325630188, -1.0896955728530884, -0.7647467255592346, -2.507654905319214, 0.022664863616228104, 0.2714245021343231, 1.1538338661193848, 1.5518220663070679, 1.4291750192642212, -0.5278754234313965, 0.2463122308254242, 1.7552080154418945, -1.8475898504257202, 1.306726098060608, -0.49258705973625183, -2.3620517253875732, -0.13986194133758545, -0.21370844542980194, 0.9210599660873413, -0.5755603909492493, -0.3677922785282135, 0.7875442504882812, 1.7566797733306885, -0.15495896339416504, 1.0145913362503052, -0.5821287035942078, -0.7820307612419128, 0.33108821511268616, -1.0725715160369873, -0.284859299659729, -0.6257142424583435, 0.17444300651550293, -0.4916355609893799, -0.6557037234306335, -1.1704485416412354, 0.68400639295578, -0.7320482134819031, 0.6360054612159729, 0.3926958441734314, -1.0409412384033203, 0.7999345660209656, 1.2049744129180908, 0.3966366946697235, 0.6859045624732971, -0.2515298128128052, 1.389608383178711, -1.287939429283142, 0.9504473209381104, -0.52361661195755, 0.7918896675109863, -0.3578077256679535, 0.3964097797870636, -1.4930832386016846, -1.2738744020462036, 0.04518002271652222, 0.2987208962440491, -2.682952642440796, 0.004373864736407995, -0.7879880666732788, -0.5791723728179932, -0.7474793791770935, -1.3870723247528076, -0.16019132733345032, 0.7967712879180908, 1.6710642576217651, -0.39647915959358215, -0.24412663280963898, 0.05996694788336754, -0.9743463397026062, 1.4137401580810547, 0.5233154892921448, 1.2008311748504639, -0.4448026120662689, 0.1879487782716751, 0.24214869737625122, 0.1903085857629776, 0.1849210113286972, 0.44820788502693176, -1.0786856412887573, -0.3369981050491333, -0.9359675049781799, -0.16577406227588654, 0.2111595869064331, 1.9733521938323975, 1.2603297233581543, -0.5520117282867432, 0.5171294212341309, 1.261038899421692, -0.079811692237854, 1.3863625526428223, 1.1175668239593506, -1.274113416671753, 0.009321513585746288, -0.1572531759738922, 1.1730412244796753, 0.3233796954154968, -2.4990432262420654, 0.537189781665802, -0.046532608568668365, -0.009937507100403309, 0.4739121198654175, -0.4241836965084076, -0.7905681133270264, -1.6231229305267334, -0.4902939200401306, 1.2010350227355957, 0.4344104528427124, -0.5357147455215454, -1.1446120738983154, 0.11647724360227585, -1.2374976873397827, -0.18495962023735046, -0.33244597911834717, 0.9572915434837341, -1.1876360177993774, 0.7289303541183472, 0.7565169930458069, -0.7441131472587585, -1.56425142288208, -0.7350308895111084, 0.39197689294815063, -0.2372848093509674, 0.5755568742752075, -0.5235379934310913, -0.4121014177799225, -0.8563682436943054, -1.1005809307098389, 1.9287779331207275, 1.1422175168991089, 2.446070671081543, 0.3076069951057434, 0.154320627450943, -0.45057213306427, -0.10398082435131073, -0.31058692932128906, 0.12537816166877747, -2.422475814819336, -0.17480921745300293, 0.19823023676872253, 1.8448656797409058, -0.1968183070421219, 0.36607640981674194, 0.07143424451351166, 0.607120931148529, 1.7528761625289917, -0.364454060792923, 1.6195107698440552, 1.2926470041275024, 1.4331024885177612, -1.1850751638412476, -0.43181079626083374, -0.5553908944129944, -0.19708387553691864, 0.6335619688034058, -0.6869332194328308, -1.8017891645431519, 0.36372387409210205, -1.3373916149139404, -0.10316216200590134, 0.8713862299919128, 0.873157262802124, -1.1144611835479736, -0.09931168705224991, -0.38540491461753845, -1.350799560546875, 1.0320408344268799, -1.0159571170806885, 1.3485839366912842, 0.6378297805786133, 0.3090445399284363, -0.12401442974805832, 0.3472610414028168, 1.567499041557312, 1.5682475566864014, 1.808421015739441, 1.3485348224639893, -0.6811996102333069, 1.1748124361038208, -0.856079638004303, -1.2166258096694946, 0.5538467764854431, 1.3931583166122437, 0.23773571848869324, 0.38045039772987366, 0.6248703002929688, 0.9599748849868774, -1.109408974647522, -1.7045788764953613, -0.08207432180643082, 1.1258575916290283, -0.4384276270866394, 0.345363587141037, -0.6592825055122375, 0.3791065514087677, 0.13076657056808472, 1.120195746421814, -0.23776483535766602, -0.054846566170454025, -0.12119569629430771, -0.5291659235954285, -1.1674001216888428, 0.6914782524108887, -0.18131420016288757, -1.0093923807144165, 0.38503071665763855, 0.753078818321228, -0.43208709359169006, -0.39955684542655945, -1.8584274053573608, 0.16428633034229279, -0.6921301484107971, 0.007463503163307905, -0.7739496231079102, 1.0414257049560547, -0.5835824012756348, -0.6450056433677673, -1.3833569288253784, -1.8380277156829834, 0.13864625990390778, -1.0293148756027222, 1.1619974374771118, -0.040924299508333206, 0.8075012564659119, -1.6749759912490845, -0.6382131576538086, 0.48904481530189514, 1.3240479230880737, -0.10013022273778915, -1.116903305053711, -0.04410959780216217, -2.0252761840820312, -0.8831348419189453, -1.1128331422805786, 0.052018389105796814, -2.475874662399292, 0.05873093754053116, 0.8803179860115051, -0.1397089809179306, 0.27701330184936523, -1.0225402116775513, 0.17301298677921295, 1.5427193641662598, -0.7259885668754578, -0.48731568455696106, -0.5133137106895447, 0.21082431077957153, -1.0537874698638916, 0.1066921278834343, 0.24811220169067383, -1.419725775718689, 0.1107981950044632, 1.1025280952453613, 0.7295016050338745, -0.6262980103492737, 1.008981466293335, -0.8458484411239624, 1.922453761100769, 0.1027960479259491, -0.8293689489364624, -0.3081299066543579, -1.6878594160079956, -1.6051135063171387, -1.1992779970169067, -0.9131108522415161, -0.31477588415145874, -0.9245480298995972, 0.04048897325992584, -0.04891595616936684, 1.6741396188735962, 0.32029521465301514, -0.45312830805778503, -0.3673906922340393, 0.3814500570297241, 0.36511820554733276, 1.0737873315811157, 1.373842716217041, 0.5054117441177368, 0.8814681172370911, -0.49393078684806824, -0.7266155481338501, 0.10938471555709839, 0.902306079864502, 1.1544318199157715, -0.7255542874336243, -0.5023462772369385, 0.775107204914093, 0.730521023273468, 0.7613359093666077, -0.43558579683303833, 1.8884856700897217, 1.195870280265808, 1.394752025604248, -1.6612550020217896, -1.581661343574524, 0.11606854200363159, -1.0948126316070557, 1.686350703239441, -1.5230039358139038, -0.12000425904989243, -0.624504029750824, -1.2056808471679688, 1.0718210935592651, -0.49236828088760376, -0.05400007590651512, -0.5957462787628174, 0.2048719972372055, -0.44495511054992676, -0.2891100347042084, 0.331386536359787, -0.04988083615899086, -0.39685797691345215, 1.0565954446792603, -0.15579450130462646, -1.8700730800628662, 1.0589323043823242, -0.20432810485363007, 0.198138028383255, 1.135411262512207, 0.2481621652841568, 0.2972710430622101, -1.016688346862793, 0.7717811465263367, 1.975870966911316, 0.5821298956871033, 1.0907384157180786, -1.2982674837112427, 1.7369364500045776, -1.522853136062622, 0.40285524725914, 0.3739364445209503, 0.7712813019752502, 0.14145955443382263, 1.0564342737197876, -0.03915100544691086, -0.556967556476593, 0.2853940725326538, 1.2171542644500732, -1.0648472309112549, -0.13330374658107758, 0.8815720677375793, -0.7562534213066101, 1.0644150972366333, -0.6112189292907715, -0.15055276453495026, 0.37217825651168823, 1.7672423124313354, 0.43085962533950806, 1.0588022470474243, 0.16126538813114166, 1.2638118267059326, -1.1648675203323364, 0.3146478831768036, -0.7115697264671326, 0.5713827610015869, -1.094779372215271, 1.842367172241211, -0.11958695948123932, -0.10150633007287979, 0.38996589183807373, 0.3641479015350342, -0.26267901062965393, 0.33366310596466064, -0.8709266781806946, 1.2296031713485718, -0.08121612668037415, -0.6208316087722778, 1.488443374633789, -0.5422729253768921, 0.007502829656004906, 1.3826402425765991, -0.9821147322654724, -0.3397767245769501, -1.4622527360916138, 0.8381235599517822, 1.3400537967681885, 1.1852569580078125, 2.6745994091033936, -1.4497218132019043, -1.1766655445098877, -1.3202849626541138, 0.09922889620065689, 0.6339874863624573, -0.5480798482894897, -0.5147047638893127, -2.226938486099243, -0.353696346282959, -0.7525473833084106, 1.0293360948562622, 0.6139800548553467, 0.8344084024429321, -0.4903425872325897, 0.8352358341217041, -0.21301347017288208, -0.15703439712524414, -0.44850850105285645, 0.4357243776321411, -0.7450107932090759, -0.9151127934455872, -0.47512149810791016, -0.3833579123020172, 1.8014336824417114, -0.4994639456272125, -1.0284749269485474, 0.9786405563354492, 0.9320492744445801, -0.6930062174797058, 1.7700190544128418, -1.1057825088500977, -0.25199830532073975, -0.26430293917655945, 0.5293354392051697, 1.1660386323928833, 0.953612208366394, 3.391775369644165, -1.1449291706085205, -0.2084289938211441, 1.7525962591171265, -0.15387852489948273, -0.6242179870605469, -1.2003456354141235, -0.05364036187529564, 1.0374423265457153, 0.11287801712751389, 0.05970295891165733, 0.8228785991668701, 0.2806829810142517, -0.054814573377370834, -0.7612093091011047, -0.03677196055650711, 0.803346574306488, 0.5208328366279602, -0.08472547680139542, -1.522257924079895, -0.820425808429718, 1.9551974534988403, 0.7375629544258118, 0.8869643211364746, 0.04411713778972626, 1.0107793807983398, 0.77348393201828, -0.4146817922592163, -1.4786382913589478, 1.551940679550171, -0.7042979001998901, 0.5248919129371643, -0.0857621356844902, 0.02379705011844635, 0.3170180022716522, -0.6724960803985596, 0.8841759562492371, 0.4305615723133087, -0.0994923934340477, 0.4665270447731018, -0.6296955347061157, -0.7269760966300964, -0.03799857199192047, 0.7997146248817444, 0.27138832211494446, -0.22452697157859802, 0.6175974011421204, 0.7056066989898682, -0.5212861895561218, -0.4606669247150421, -0.5293090343475342, -3.0230603218078613, 1.9841241836547852, -0.7289892435073853, 0.1361454725265503, 1.23753023147583, 1.1465635299682617, 1.2753212451934814, 1.1611826419830322, 0.02951839566230774, -0.8972394466400146, 0.2561601996421814, 1.115399718284607, 0.15311285853385925, 0.011355609633028507, -0.20393981039524078, -0.012180931866168976, -1.159222960472107, -1.0645416975021362, 2.463460683822632, -0.7188136577606201, 1.7719624042510986, 0.16788455843925476, -0.6444736123085022, 0.7865812182426453, 2.2099645137786865, -0.260079026222229, 0.6809799075126648, 0.9052380323410034, -0.297319233417511, -1.5478655099868774, 0.26187875866889954, 0.8316611647605896, -0.19888901710510254, 0.9914422631263733, -0.6635469794273376, -0.28724804520606995, 0.2632651925086975, -1.7555075883865356, -0.9762658476829529, 0.20754949748516083, -0.3462602198123932, -1.7853959798812866, 0.21984121203422546, 0.04942460358142853, 0.3768690526485443, -0.4489901661872864, 0.35933661460876465, -1.621878743171692, -0.36449167132377625, 0.5124305486679077, 0.013505927287042141, 0.8026195168495178, -0.027058787643909454, -0.8703367710113525, -0.41846364736557007, 1.5274029970169067, 1.302209496498108, -0.0030374755151569843, 1.9405100345611572, -0.6946489810943604, 0.800145149230957, -0.6202703714370728, 2.208841323852539, -0.2694603502750397, -0.8849987983703613, 1.3786430358886719, 0.5546872615814209, 1.5370934009552002, 0.6246998310089111, 0.9565262198448181, -1.5014256238937378, -0.7946776151657104, -0.3986509144306183, 0.8062278628349304, -0.4606429934501648, -1.0880863666534424, -0.0544947125017643, -0.6079258918762207, -0.8794628977775574, -1.1942503452301025, -1.310528039932251, -0.29370006918907166, 0.7362440824508667, -0.393168568611145, -1.353488802909851, -0.1952219158411026, -1.5488742589950562, -0.41674551367759705, -0.36556127667427063, 2.403838634490967, -1.341092824935913, 0.022236213088035583, -1.6530749797821045, 2.0339059829711914, 0.5252272486686707, -1.2902171611785889, 1.1849141120910645, 0.3731613755226135, -1.859024167060852, -0.9746551513671875, 0.24923528730869293, -1.172494649887085, -1.2763948440551758, 0.17032817006111145, 0.620428740978241, 0.3947222828865051, 0.8022491931915283, -0.45295658707618713, 0.9333993196487427, -0.10624827444553375, -1.283324122428894, -0.6236450672149658, -1.634973406791687, -2.4931070804595947, 1.8022974729537964, -1.0559724569320679, 1.480061411857605, -0.6892474889755249, -0.2472243309020996, -1.2028813362121582, -1.2859406471252441, 0.9074125289916992, 0.4949149787425995, 0.3315749168395996, 0.8687831163406372, 0.5772879719734192, -0.5591533184051514, 0.7738443613052368, -1.2669159173965454, -1.3674266338348389, 0.6023313999176025, 1.688097596168518, 0.604860246181488, -1.1107442378997803, -0.5067787170410156, 1.6183137893676758, 1.3963261842727661, -1.3896363973617554, 0.26127469539642334, -0.3879520297050476, -0.45411011576652527, 0.5614182949066162, 0.4659713804721832, 1.5023165941238403, 1.6660394668579102, -0.001151059870608151, -0.9685674905776978, -1.054356575012207, 0.5321325659751892, 0.9260002970695496, 1.674945592880249, -0.4573870897293091, 1.0061678886413574, -0.5498908758163452, 0.37079185247421265, 2.5838119983673096, 1.1617703437805176, -0.4653237462043762, -1.0001777410507202, -0.15697301924228668, 0.02293587662279606, -0.14899691939353943, -1.3988478183746338, 0.7810871005058289, -0.7797527313232422, -0.7863129377365112, -1.3244984149932861, -0.6191969513893127, -0.27416497468948364, -0.9432414174079895, -0.5825443863868713, -1.19080650806427, -2.235058307647705, 1.7297189235687256, 0.9909180402755737, -1.3560693264007568, -0.23760147392749786, 0.2638004720211029, 0.05595512315630913, 0.08511751890182495, 0.6675667762756348, -0.6946628093719482, 1.3150612115859985, -0.011108179576694965, 2.138690233230591, 0.5995221734046936, 0.747870922088623, 0.3021697998046875, -1.6815520524978638, -0.46107396483421326, 1.2185332775115967, 0.027319511398673058, 0.7375302910804749, -0.5840802192687988, 1.3365658521652222, 0.9878274202346802, -0.9404782056808472, 0.036154065281152725, -0.9727250933647156, 1.0134470462799072, -0.07333395630121231, 0.541528582572937, -0.9244714379310608, -0.5712818503379822, 0.7730675339698792, 0.0819443017244339, 1.3809080123901367, 1.9908242225646973, 0.7732024788856506, 0.23718181252479553, 1.0935344696044922, 0.6636783480644226, 0.6026782989501953, 0.7261626124382019, 0.2051272690296173, 2.0611541271209717, 0.07213044911623001, -0.9718297123908997, -0.6640127301216125, 1.1941665410995483, 1.2294944524765015, 1.3339500427246094, 0.213514044880867, 0.7220755219459534, -1.0151323080062866, -0.9044592976570129, 1.2153751850128174, -1.1577575206756592, 0.7605658769607544, -0.015263856388628483, -0.2661583125591278, -0.9717049598693848, -0.41741350293159485, -0.425768107175827, 0.0773775577545166, 0.2578900158405304, 0.6448076963424683, -0.10824886709451675, 0.5938843488693237, -1.6012896299362183, -0.003718459513038397, 0.9236196279525757, -0.3274175226688385, 1.8969357013702393, 1.0788320302963257, 0.3270503282546997, -0.0484941266477108, 1.1648900508880615, 1.6699029207229614, -1.7461400032043457, 0.05053268000483513, 0.18559791147708893, -0.9146298766136169, 1.400666356086731, 0.5895651578903198, -0.7707106471061707, -1.3618392944335938, -1.0699554681777954, 0.0858171358704567, -0.8138145208358765, 0.9460745453834534, 0.13443820178508759, 0.24341179430484772, 1.8984777927398682, 1.7256821393966675, -0.7732226848602295, -0.18912850320339203, -1.6421864032745361, 0.28026118874549866, -0.7570759654045105, 1.056662678718567, 0.1447855830192566, -0.46883007884025574, 0.41152364015579224, 0.19011595845222473, -1.4804490804672241, -0.10359939932823181, -0.9311673045158386, -2.393758535385132, -0.41131189465522766, 0.7278342843055725, 0.36017248034477234, -0.42115268111228943, -0.14211304485797882, 0.8778426051139832, -1.7065322399139404, -0.6471508741378784, 0.14060714840888977, -0.19549570977687836, 1.514967918395996, -0.5797417759895325, -0.48157188296318054, -0.38441476225852966, 0.5698916912078857, -1.223127007484436, -0.24611182510852814, 0.19165998697280884, -0.34792977571487427, 1.3202550411224365, 0.058951471000909805, -0.5123866200447083, 0.7948204278945923, 0.7929988503456116, 0.4632405936717987, 1.8839001655578613, -0.16958566009998322, -1.0689375400543213, 0.21463491022586823, 1.7933770418167114, -0.7748667597770691, 2.495396614074707, -0.7674910426139832, -0.47861239314079285, 0.12063482403755188, -0.7160218954086304, -2.4656457901000977, -1.7016887664794922, 0.96054607629776, -0.16160576045513153, 0.16220636665821075, 1.0627877712249756, -0.433767706155777, 0.057415325194597244, 0.8804079294204712, -0.9560567140579224, -0.02072753570973873, -1.1275464296340942, -0.347234308719635, 0.5705029964447021, -1.3163496255874634, 0.7248058319091797, -0.17644073069095612, 1.04885995388031, -0.5682656764984131, 0.7446900606155396, 0.46000924706459045, 1.024816632270813, 1.0274816751480103, -1.1464558839797974, -0.05178559944033623, -0.7547540664672852, -0.9358862638473511, -0.01621256209909916, -0.7613691091537476, -0.9383366107940674, 0.9914253354072571, 1.0845216512680054, 1.6493133306503296, 0.21834425628185272, 0.513087809085846, -1.5330018997192383, 1.3570938110351562, 1.625048041343689, -1.8433250188827515, 1.0393539667129517, 2.0971806049346924, -0.811163067817688, -0.2759760618209839, 1.5242769718170166, 0.5434905290603638, -0.20700252056121826, 0.21122942864894867, -1.328706979751587, -1.0387811660766602, 0.09676823019981384, -0.16626855731010437, -0.9398111701011658, 0.4803095757961273, -0.938473641872406, 2.568608283996582, 1.7016427516937256, -1.7660019397735596, -0.20557422935962677, 0.21833999454975128, -1.0251580476760864, -0.9143797755241394, -0.7098525166511536, -0.4056048095226288, -0.26442229747772217, -1.067549467086792, 0.8960137963294983, 0.7023988962173462, 0.2804914116859436, -0.4000259041786194, 0.24343355000019073, -0.2656056880950928, -0.19677521288394928, 0.31249889731407166, 0.25776681303977966, -1.9231460094451904, -1.2508355379104614, 0.2914942502975464, -0.6959949135780334, -0.08133944123983383, 0.38702037930488586, -0.19883675873279572, -0.4343111217021942, 0.030240999534726143, -0.022757677361369133, -1.6717126369476318, 0.7487539052963257, -0.9837291240692139, 1.7916446924209595, 0.5288121700286865, 0.4857504367828369, -0.6987978219985962, -0.9428771138191223, 0.6337109804153442, 1.6167287826538086, -0.32961758971214294, -1.4636977910995483, -0.40169307589530945, 0.365142822265625, 0.6608561277389526, 0.799554169178009, 0.09068452566862106, 0.019734453409910202, 0.4404712915420532, 0.4032288193702698, 0.07994206249713898, -2.258026123046875, -0.7624943256378174, -1.2030893564224243, -0.11489516496658325, 0.5137746334075928, 0.5721482634544373, -0.3259908854961395, -0.730987012386322, -0.40200275182724, -1.0965125560760498, -1.917367696762085, 1.0677422285079956, -1.3904452323913574, 0.72160804271698, -1.2971969842910767, -0.12601831555366516, 1.5767548084259033, 1.6214213371276855, 0.12378207594156265, -0.5096482038497925, 1.0678998231887817, -0.4164484441280365, 1.2942596673965454, 0.0917166918516159, -0.5456117391586304, 1.036715030670166, -1.017983317375183, -0.47590431571006775, 1.5421196222305298, 0.09812035411596298, -2.0139076709747314, 0.28127792477607727, -0.1769685000181198, 0.8409833908081055, -0.8263069987297058, 1.7990508079528809, 1.5049508810043335, 0.09651810675859451, 0.16038642823696136, -1.3775639533996582, 0.4205580949783325, 1.2650846242904663, -0.048991866409778595, -0.427755743265152, -0.8844629526138306, -0.6646645665168762, -1.2655876874923706, 0.46237170696258545, -0.7975781559944153, 0.11299759894609451, -0.3961782157421112, 1.096329927444458, -0.8315724730491638, -0.769302487373352, 1.0352783203125, 0.28443148732185364, -1.1435136795043945, 1.0798896551132202, -1.0247055292129517, -0.9543228149414062, -0.24803656339645386, -0.040379598736763, -1.272946834564209, -0.23235662281513214, -1.3322392702102661, -1.2405096292495728, -0.5190617442131042, 1.3515821695327759, 0.47948136925697327, -0.4588824510574341, 1.7076116800308228, -0.012720927596092224, -0.02478373423218727, -0.6443191766738892, -0.6173092722892761, 2.793606996536255, -0.7472675442695618, 0.08138663321733475, 1.6482373476028442, 0.09355567395687103, 0.49021103978157043, 0.022630680352449417, 1.1228644847869873, -2.621389150619507, 0.9670373201370239, -0.24883617460727692, 0.42294028401374817, -0.2538462281227112, 0.12771078944206238, 0.14333009719848633, 3.564194917678833, -0.6051949858665466, 0.8555683493614197, -0.33551710844039917, 0.5427936315536499, 0.5389624834060669, -0.6067723035812378, -0.11786482483148575, 1.6545593738555908, -0.27168771624565125, 0.8770701289176941, -1.0619996786117554, -1.2277164459228516, 1.1772592067718506, 0.13827703893184662, 1.19643235206604, 0.9378439784049988, 0.5688419342041016, 0.9160647392272949, 0.2073219269514084, -0.11838630586862564, 0.3906347155570984, -0.1518758088350296, 0.2890954911708832, 1.2151837348937988, -0.09888026118278503, -0.9228775501251221, -0.5620975494384766, -0.3384750187397003, 0.09679573774337769, -1.946001410484314, -1.2566410303115845, 1.1325311660766602, -0.1400037407875061, -1.2128068208694458, 0.8118781447410583, 0.040621791034936905, 1.130345344543457, 0.9996266961097717, -1.9846630096435547, -0.32711324095726013, 0.09647758305072784, -0.26613423228263855, 1.9328179359436035, 0.24598480761051178, -1.1092408895492554, -0.3329966068267822, 0.4398142993450165, -1.2380033731460571, 0.5400400161743164, 0.41472941637039185, 0.4912702739238739, -1.740371584892273, -0.5793794393539429, 1.4002673625946045, -1.4525864124298096, -0.17795106768608093, -1.222298264503479, 0.09132494777441025, 0.38614052534103394, 0.6311215758323669, 0.46004578471183777, -0.267336905002594, 0.07428465038537979, -0.11709866672754288, 0.7190872430801392, 0.013826563954353333, -0.36934414505958557, 0.018952466547489166, 1.515150785446167, -0.5098127722740173, 0.4391366243362427, -1.5007562637329102, 0.295166552066803, 0.08534198999404907, -0.38526540994644165, 1.8712148666381836, -0.11790668219327927, 0.06407685577869415, -1.235121726989746, 0.6728106141090393, 0.15316970646381378, 0.7904720306396484, -0.7011744976043701, 1.1994184255599976, -1.2331551313400269, -1.1362367868423462, -1.5598974227905273, 0.10856407880783081, 0.9467034935951233, 0.18454903364181519, 0.8599156737327576, 0.27823778986930847, -0.7415958046913147, -0.8215388655662537, 0.9234492182731628, -0.36679530143737793, -0.2617073357105255, 1.2132388353347778, 0.12801116704940796, -0.2459316998720169, 0.5727939605712891, -0.648835301399231, 0.8427727818489075, 0.5428735017776489, -0.3004632890224457, -0.04316069558262825, 0.36508211493492126, -0.13245443999767303, -0.13287189602851868, 0.8530449867248535, -0.08950593322515488, 0.5073989033699036, -0.8221209049224854, -0.042351242154836655, 1.2632505893707275, 0.6314153671264648, 0.7109546661376953, 0.5427819490432739, 0.4403379261493683, -1.261788249015808, -0.48293012380599976, -0.943285346031189, 0.10829012840986252, 0.69814532995224, 0.04047568514943123, 0.9701961874961853, 0.7934674620628357, 1.2759268283843994, 1.7302602529525757, 1.789618968963623, -1.8837192058563232, 0.21149814128875732, 0.043054599314928055, -0.8031424880027771, -0.6439469456672668, -1.211950659751892, 0.022004352882504463, 0.39358824491500854, -0.4729149341583252, -0.4012138843536377, -2.7398521900177, -0.5828568339347839, 1.0713481903076172, -0.841802179813385, -0.5186384916305542, -0.9374642968177795, 0.30079108476638794, 0.0643012598156929, 0.3661690354347229, -1.7470735311508179, 0.9795650839805603, 0.27213531732559204, 0.6922581195831299, -0.30905431509017944, 0.6713986992835999, -0.05168566107749939, -1.2048430442810059, -1.5069786310195923, 0.3767012059688568, 1.2366644144058228, -0.37377044558525085, 1.7981202602386475, 0.2968815267086029, 1.6483614444732666, -0.4290473461151123, 1.0594743490219116, -0.025051290169358253, -0.5142287611961365, 2.139644145965576, 1.055773377418518, -1.2909029722213745, -0.7555361986160278, -0.644278347492218, 1.350607991218567, -0.3174387216567993, 0.7976219058036804, -0.18873915076255798, 1.1626899242401123, -1.0811108350753784, 0.8386462330818176, -1.0812828540802002, -0.1094391867518425, -2.8702805042266846, -1.0777380466461182, -0.5356518626213074, -0.34407126903533936, -1.9039615392684937, 0.012927033938467503, -0.5839917659759521, -0.75290846824646, -1.052249789237976, -0.8427892923355103, -2.1561107635498047, -0.9845935106277466, -0.9207154512405396, 0.8674110770225525, 0.3936648666858673, -0.3740043044090271, 0.7982719540596008, -1.948858618736267, -1.7631902694702148, -1.024661660194397, 0.6059408187866211, 1.3538668155670166, 1.6227529048919678, -0.12944194674491882, -2.165557384490967, 1.1788970232009888, 0.5607960224151611, 0.48947209119796753, -0.008866374380886555, -0.25663474202156067, 2.00759220123291, -1.4892487525939941, -0.2211497277021408, 1.4162667989730835, -0.10901086032390594, 1.7487852573394775, 0.18665382266044617, -0.5236127972602844, -2.385629653930664, 0.03878363221883774, 0.9761353135108948, -0.9821894764900208, -1.9432471990585327, 1.033108115196228, 0.675927996635437, 0.8166386485099792, 0.47087061405181885, 0.021371519193053246, 1.6326098442077637, -0.6656020283699036, 2.584433078765869, 1.2824652194976807, -0.8491219878196716, -1.3652130365371704, 2.5995595455169678, 0.12747611105442047, 0.20662716031074524, 0.41664355993270874, 0.36115342378616333, 0.5742025971412659, -1.8395389318466187, 1.555209994316101, 1.3647774457931519, -0.39114898443222046, -0.14743360877037048, -0.9660479426383972, -1.3960039615631104, 1.1240538358688354, 0.1486794352531433, -0.6484509706497192, 0.13704216480255127, -0.09118498861789703, 1.320054054260254, -0.09039221704006195, 0.3655630350112915, -0.756336510181427, 1.0915108919143677, 0.3060562312602997, -2.8834176063537598, 0.2234722375869751, 1.243039608001709, 0.05127473920583725, 1.0151797533035278, -0.22714227437973022, -0.8183810710906982, -1.582950234413147, -0.6568064093589783, 0.6544004678726196, 1.1709117889404297, 0.9994657635688782, -0.5892540216445923, -0.9333941340446472, 0.2421228289604187, -0.5890969038009644, -0.05451524257659912, 0.7730996012687683, -1.1623210906982422, 3.8230652809143066, -0.25294655561447144, -1.527607798576355, 2.513465404510498, 1.0106279850006104, 0.6842570304870605, 0.10685563832521439, 2.2938594818115234, 2.0116045475006104, -1.0485105514526367, -0.046348392963409424, -0.5545994639396667, -0.7236698865890503, 0.5380083918571472, -0.18724046647548676, 0.7669386267662048, -0.21751999855041504, 0.5328977704048157, -2.1530261039733887, 0.5335366725921631, 1.9576416015625, 0.1565619260072708, 1.0561680793762207, -1.2780312299728394, -0.9822970032691956, 1.4664198160171509, -0.7008392214775085, -0.1274409145116806, 0.8872973322868347, -1.9003034830093384, -0.36463046073913574, 0.49895530939102173, 0.321878045797348, 0.5609608292579651, -0.9974476099014282, -0.2074722945690155, -1.9721506834030151, -0.8432598114013672, -0.666683554649353, 0.3242902457714081, 0.648989737033844, 0.4118265211582184, 1.791929841041565, -0.26261213421821594, 0.03380914404988289, -1.2183490991592407, 0.9891250729560852, -0.41882628202438354, 0.7429686188697815, 0.15420082211494446, -0.8453541398048401, 0.14782269299030304, 0.8219884037971497, 1.3495615720748901, -0.37237054109573364, -0.8333043456077576, -2.3253824710845947, -0.40370291471481323, -1.2603204250335693, 1.2967398166656494, 1.5944818258285522, 1.035759449005127, 2.4389195442199707, 0.4143473207950592, -0.3512239456176758, -0.21127808094024658, -0.5137486457824707, -0.14915716648101807, 1.2535218000411987, 0.1913779079914093, 1.3509708642959595, -1.243727684020996, -0.5054916739463806, -1.4917168617248535, -1.0897024869918823, -0.9616164565086365, -0.29939648509025574, 0.5242601037025452, -0.2581769526004791, 0.5948544144630432, 0.8251470923423767, 0.1674358993768692, 0.1630425751209259, -0.41488954424858093, -0.5516742467880249, -0.14356239140033722, -2.062931537628174, -0.8196604251861572, 1.2140311002731323, 0.06570373475551605, 1.154512882232666, 1.7250142097473145, -0.857376217842102, -1.2746142148971558, -0.9191210865974426, -1.0710502862930298, -1.4638099670410156, 0.8805021047592163, 0.43226420879364014, -0.7066336870193481, -2.466488838195801, 1.436944842338562, -0.09746228903532028, -1.6272425651550293, -1.021030306816101, -0.38758543133735657, -0.9964174032211304, -1.6445257663726807, 1.5300432443618774, -0.3208611011505127, 1.2099159955978394, 0.06078748404979706, -0.033714089542627335, 0.15954717993736267, 1.6221156120300293, 1.7453032732009888, 2.502781867980957, 0.24309955537319183, -0.15848872065544128, -1.2694097757339478, -0.06939463317394257, 0.013755462132394314, 1.1016689538955688, 0.7063388824462891, -0.7578834891319275, -0.39534059166908264, -1.1152677536010742, 0.2500174343585968, 0.1446307897567749, 0.4246966540813446, 0.7572200894355774, 1.5476937294006348, -1.3286631107330322, 0.7242560386657715, 0.21237078309059143, 1.0966180562973022, -0.6914928555488586, -0.5487145781517029, 1.0669548511505127, -1.0856363773345947, -0.4267060160636902, -0.15563756227493286, -1.239445686340332, -1.9615604877471924, 1.318367600440979, -0.261793315410614, 0.5021653771400452, 0.122355155646801, 0.719693124294281, 0.6874342560768127, 0.12686777114868164, 0.9152026772499084, 1.0740208625793457, -0.04971858859062195, 0.4448496103286743, -0.6241806149482727, -0.23349781334400177, 1.2690985202789307, 0.11297664046287537, -0.12905222177505493, 1.62252938747406, -0.4088042974472046, 0.14727641642093658, 0.7939527630805969, -0.39981207251548767, 1.0851659774780273, 0.3089396357536316, -2.143657922744751, -0.6097801923751831, -0.09021786600351334, 0.5696172118186951, -1.5854792594909668, 0.10885041952133179, 0.7853928804397583, 0.46319741010665894, 1.089483618736267, -1.970557689666748, 0.7604169845581055, 0.3568514287471771, -0.029292745515704155, 1.1078121662139893, 0.29145562648773193, -0.3464110195636749, 0.25008630752563477, 1.5841635465621948, -2.8662590980529785, 1.4647595882415771, 0.8888611793518066, -0.8780717849731445, -1.469481348991394, -1.0409494638442993, 0.9382891654968262, 0.8092389702796936, -0.43177297711372375, 1.0488966703414917, -1.1447818279266357, 0.01308669988065958, -0.7175822854042053, 0.7129535675048828, -0.9639022350311279, 0.6700403690338135, -0.9046282172203064, -1.0727945566177368, -0.7306654453277588, 0.4727904200553894, 0.251880943775177, 0.58011794090271, -0.05424286797642708, 0.5597071051597595, 1.1880757808685303, 0.7942363023757935, 0.03820717707276344, 0.5556102991104126, -1.5504108667373657, -1.2318445444107056, 1.6974610090255737, 3.2341649532318115, 0.01931224949657917, -0.1473984569311142, -0.8974992632865906, -2.10906982421875, -1.8338425159454346, 1.31912100315094, -0.31907615065574646, -0.4731895923614502, 0.5808500647544861, 0.9935720562934875, 0.3261970579624176, 0.3686237037181854, 0.5356090664863586, -0.43941518664360046, 0.02093186415731907, -0.9425578117370605, 1.0811971426010132, -1.2273191213607788, -0.8750393390655518, 0.3587273061275482, -0.3144415318965912, -0.40676724910736084, 0.29695820808410645, 1.1926987171173096, 0.02502007782459259, 1.3708031177520752, 0.04364621639251709, -0.38632985949516296, 0.28241389989852905, -2.513368606567383, -0.08095110952854156, -0.925512969493866, 0.04871616140007973, 0.8386657238006592, -0.6274718642234802, 0.7832716703414917, 2.46520733833313, 1.3822723627090454, 2.864424467086792, 0.7436208128929138, 0.41446882486343384, 0.45941153168678284, -1.0161930322647095, 0.4444328844547272, 0.7867094278335571, -0.015638023614883423, 0.16716834902763367, -0.08143220096826553, -0.251374751329422, 0.16899409890174866, 1.5209095478057861, -2.0385560989379883, 0.7655647993087769, 1.3626154661178589, -0.4103021025657654, -1.7845813035964966, 0.9763367176055908, -0.2231198400259018, -0.9863165020942688, -0.08014227449893951, -0.7530622482299805, 0.32571354508399963, 0.4881821572780609, 0.3147129714488983, 0.8700783848762512, -0.16175569593906403, -1.9722718000411987, -0.23672446608543396, -0.15707872807979584, 0.08629608899354935, 0.20089340209960938, -0.8730481266975403, -0.10826757550239563, 0.24235445261001587, 1.1398500204086304, -1.1880285739898682, -0.8471697568893433, 2.3742456436157227, -0.4482510983943939, -1.3836551904678345, -1.4494613409042358, -1.636502981185913, -0.6469355821609497, -1.9631962776184082, -0.1305106282234192, -0.09235229343175888, -0.13451136648654938, 1.1330443620681763, 0.19388289749622345, -0.06381964683532715, 1.3090763092041016, 0.43146008253097534, 0.3537987172603607, 0.2035049945116043, -0.4722561538219452, -1.0969983339309692, -0.42964068055152893, 0.5740801692008972, 0.06837447732686996, 0.7780191898345947, 0.33693310618400574, 0.5699726343154907, 0.3527236878871918, 0.1285746991634369, 0.589349091053009, 1.1031609773635864, -0.13807906210422516, 1.9686256647109985, -0.44241151213645935, -0.964053213596344, 1.0336596965789795, 1.6051963567733765, -0.05009688436985016, 1.4917024374008179, 0.039600808173418045, 1.032821774482727, -0.06528577208518982, -1.2554752826690674, 0.6096264719963074, -0.04921015352010727, -1.2780210971832275, 0.5544257164001465, 0.2608974874019623, 0.8404532074928284, -1.5875418186187744, -1.3235830068588257, -0.7308880686759949, 1.5389562845230103, 0.30675652623176575, -1.0981084108352661, 0.5062496066093445, 0.7191768288612366, -0.41471847891807556, 0.08639594912528992, -0.3026881217956543, -1.7641263008117676, 0.06799983233213425, 0.5220256447792053, -1.4935917854309082, -0.10458812862634659, 0.3919682800769806, -0.6928613185882568, -0.506260097026825, -1.129356026649475, 0.6109247803688049, -1.9569910764694214, -1.081148386001587, 0.05751882120966911, -0.08493661880493164, 0.2776264250278473, 2.009664297103882, 2.0433075428009033, -0.4469633400440216, 0.11831767112016678, -1.0179997682571411, 0.7633516788482666, 0.9512775540351868, 0.5826646685600281, -0.4082719087600708, 0.041584525257349014, -2.8167290687561035, -0.13787776231765747, -0.9572725892066956, 0.35824576020240784, -0.10683124512434006, 1.3229204416275024, 0.7762157917022705, -1.4093788862228394, -0.9268112778663635, 1.5280404090881348, 1.7360426187515259, 0.8608590364456177, 0.38871079683303833, -1.0445497035980225, 0.9720619320869446, 1.1447113752365112, -1.0166044235229492, 1.053928256034851, -0.14678077399730682, 1.0782442092895508, 0.14007210731506348, -1.369985580444336, 0.3101576864719391, 0.8060327172279358, 0.05550536513328552, 0.6520513296127319, -0.7510954737663269, -0.6950057148933411, 0.5730419754981995, 0.38979342579841614, -0.27476227283477783, -0.19678497314453125, -0.5574294924736023, -1.499187707901001, 1.4513952732086182, 0.40461891889572144, -0.541866660118103, 0.07381931692361832, -1.0465551614761353, 0.6208801865577698, 0.9864063858985901, 1.4117960929870605, 0.6060264110565186, 0.05076727271080017, 1.3030292987823486, 1.4325929880142212, -0.030232595279812813, 0.10876395553350449, -0.9730930924415588, 0.23151427507400513, -1.2521458864212036, -1.851707100868225, -0.9766380190849304, -1.2357450723648071, 0.2025483101606369, -0.6797640919685364, -0.1169767677783966, 1.152441382408142, -0.17646382749080658, 0.8088982701301575, -0.5992868542671204, -0.5308037400245667, 1.8700741529464722, 0.2005067765712738, 0.8649397492408752, 0.17601078748703003, 0.13988856971263885, 0.8928499817848206, -0.3020569980144501, -1.4431586265563965, 0.8118476867675781, 0.7883645296096802, 0.2974163889884949, 0.7474234700202942, -0.5563045144081116, 0.3960719108581543, 1.5382778644561768, -0.7243412137031555, -0.674729585647583, -0.08092435449361801, -0.8021513819694519, 1.0082167387008667, -0.5257556438446045, 0.5187225937843323, 0.5781580805778503, -1.2007347345352173, 0.6869480609893799, 0.4641014039516449, -0.059005897492170334, 1.0552629232406616, 0.7322824597358704, -0.006214037537574768, -0.023700952529907227, -0.5601831078529358, -0.3449273705482483, 1.3818292617797852, -0.22272992134094238, -1.1600306034088135, -0.889680027961731, 1.9761600494384766, -1.3215224742889404, -0.7494691014289856, 1.0765302181243896, -0.3632948696613312, 0.19015255570411682, -0.20150518417358398, -0.26525187492370605, 1.553346872329712, 1.0585545301437378, 1.4092059135437012, 0.4581981301307678, -0.8940116167068481, -0.04513847455382347, 0.05053278058767319, 0.38912421464920044, 0.586134672164917, -0.41939592361450195, 0.5469664335250854, 2.114989995956421, -1.0592812299728394, 0.2265402376651764, -0.29456913471221924, -0.19937750697135925, 1.6192760467529297, -1.4410029649734497, 0.5315816402435303, -1.1075128316879272, 1.8842097520828247, -0.6763803362846375, -1.17270028591156, 1.4236406087875366, -0.45921871066093445, -0.6574387550354004, -1.4995431900024414, 0.1483384221792221, -0.46254777908325195, 1.4751205444335938, 0.012462522834539413, 0.4604681432247162, 0.9361767768859863, 0.5383114814758301, 0.07533825933933258, 0.423278272151947, 0.7355888485908508, 0.33632805943489075, -0.5621197819709778, -0.3756454885005951, 1.8985950946807861, -1.2135238647460938, -0.6587297320365906, 0.8415320515632629, -1.6965266466140747, -1.2634903192520142, -1.0909513235092163, -0.3859040141105652, -0.9898486733436584, 1.1167761087417603, 1.7943270206451416, -1.7207765579223633, -1.0682530403137207, -0.00125860795378685, 1.4066660404205322, -0.48011544346809387, 0.03046553209424019, -0.8590770363807678, 0.7489240765571594, -1.9889885187149048, 0.4197329878807068, -0.3810354769229889, -0.4317929744720459, -0.5609592199325562, 1.3168575763702393, 0.2163858860731125, 0.5366351008415222, 2.49118709564209, 1.1372227668762207, 0.7469204068183899, -0.08020152896642685, 0.9255300760269165, 0.04114285111427307, 0.9694393277168274, 0.1047462597489357, -1.5949151515960693, -0.2691778242588043, -0.8135291337966919, 1.3342479467391968, -0.2338283658027649, -1.7625582218170166, -0.5701966881752014, 0.6805450916290283, 0.2631409168243408, -1.458669900894165, -0.6186251640319824, -0.32026374340057373, 0.39753827452659607, 0.9576802849769592, 1.8592251539230347, -0.6766132712364197, -0.4220580458641052, 2.823810338973999, 0.3686075210571289, -1.3826704025268555, -0.08418503403663635, 0.6940430998802185, 0.010941819287836552, 0.9858538508415222, 0.08862735331058502, 0.07382117211818695, 0.8672065138816833, 0.48434334993362427, 0.2961478531360626, -0.2226303368806839, -0.6134256720542908, -0.2846164107322693, 2.0740573406219482, 0.03272900730371475, 1.1106621026992798, -0.839714527130127, 1.6008328199386597, -1.1126497983932495, 0.08455789089202881, -0.618740439414978, -0.8134670853614807, -0.41270744800567627, 0.30145490169525146, -2.0103251934051514, 0.245746448636055, 0.40833091735839844, 0.6695321202278137, 1.1607848405838013, -0.40951108932495117, -0.008019929751753807, -0.3476339280605316, 2.4909024238586426, 1.0287848711013794, 1.0058664083480835, 1.2864413261413574, 0.5742388367652893, -1.1034843921661377, 1.3498433828353882, 2.797105312347412, 0.3380662202835083, -0.30658799409866333, -0.8895679712295532, -0.8634422421455383, -0.11113113909959793, 1.020844578742981, -0.15223932266235352, 0.12694022059440613, 2.0399272441864014, 0.9841060638427734, -1.0224087238311768, 0.9792248010635376, -2.0690932273864746, -0.6197286248207092, 0.5595743656158447, -1.2821366786956787, -0.9817200303077698, -0.5026364326477051, 0.03912064805626869, 0.4124615490436554, 1.415939450263977, 0.9482094645500183, 0.4726025462150574, 0.9337958097457886, 0.22740444540977478, -0.19502200186252594, -0.3768033981323242, -1.377183198928833, 0.7683242559432983, 0.14815154671669006, -1.562785267829895, -1.528604507446289, -0.5080649852752686, 0.9182047843933105, 1.098718523979187, 0.14745059609413147, 1.403072714805603, 0.8259071111679077, -0.5866933465003967, -0.6795116662979126, -1.0087001323699951, 0.8405574560165405, -0.3987094759941101, -0.382421612739563, 1.4239839315414429, -0.49866563081741333, -0.5137432217597961, -0.13263727724552155, -0.3033384382724762, -0.6803925633430481, -1.104583978652954, -0.5880128145217896, -2.2565200328826904, -0.5549351572990417, -2.3614580631256104, -1.3202629089355469, 0.25668370723724365, 1.3626399040222168, 1.129219889640808, 0.4769630432128906, -0.6680260300636292, 0.2957184314727783, 0.48284754157066345, 1.1121543645858765, 1.0084024667739868, 0.07039597630500793, 1.9261367321014404, 1.3324549198150635, -1.9149670600891113, -0.7993984222412109, 0.6404500603675842, -0.8690544962882996, -1.1075439453125, 0.5660272836685181, -0.26717430353164673, -1.3521767854690552, 2.354652166366577, -0.28337642550468445, -0.20313774049282074, -0.5925624966621399, 1.1089489459991455, 0.36968880891799927, 1.2484034299850464, -0.5609948635101318, -0.12330403923988342, 1.970771074295044, 0.9083781242370605, -1.0500211715698242, 1.2702898979187012, 0.29258111119270325, -0.09276704490184784, 0.8648084402084351, -0.09801532328128815, -1.7397711277008057, 0.11143758147954941, 0.03307903930544853, -0.6440354585647583, 0.05687928944826126, 0.37677937746047974, -0.2954236567020416, 0.30570149421691895, 1.3592584133148193, 0.9419342875480652, 0.14353500306606293, 1.412826657295227, -0.27655500173568726, -2.6041040420532227, -0.3283097743988037, -0.9944407343864441, -1.273027777671814, 1.1681616306304932, 0.2993316650390625, -1.4618035554885864, -0.31965020298957825, -1.4465901851654053, -0.18085075914859772, 0.9421365857124329, 1.6754399538040161, 0.571528971195221, 0.7682086229324341, 0.26539671421051025, -0.34989598393440247, 0.6441894769668579, -0.9487100839614868, 2.3241124153137207, 0.7958227396011353, -0.3694663643836975, 0.7940031290054321, 0.4121536314487457, -0.789297342300415, -1.1958140134811401, 0.3240686058998108, 0.11694159358739853, -0.08337731659412384, 0.6555731296539307, 0.4932190477848053, 1.716512680053711, -0.9134288430213928, -0.609478235244751, 0.518346905708313, 2.7911858558654785, 0.7284432649612427, -1.3211755752563477, -0.46655917167663574, 0.17599934339523315, 1.1782344579696655, 0.10778197646141052, -0.8740437626838684, 0.7593340277671814, -0.8841902613639832, -0.9353903532028198, -0.356207013130188, 0.22029335796833038, 0.35792186856269836, 1.6340147256851196, -1.365738034248352, 1.8790180683135986, -0.6621030569076538, 0.7899366021156311, -0.29164737462997437, -0.5723187327384949, 2.011960506439209, -0.6887073516845703, 1.1171329021453857, -1.159153699874878, 0.026617296040058136, -1.0249053239822388, 0.5692169666290283, -0.9565072059631348, -0.24862222373485565, 0.31073272228240967, -0.405830055475235, -0.5866674184799194, 0.2743876874446869, -0.7575670480728149, 0.0358949713408947, -0.6885514855384827, 0.020211635157465935, -0.05111400783061981, 0.3255946934223175, 0.6653968691825867, -0.676361083984375, 1.9080504179000854, -1.554372787475586, 0.6402056813240051, 2.071809768676758, -1.1609468460083008, -1.4248672723770142, -0.7442999482154846, 0.45462507009506226, 0.5464452505111694, 0.6226205229759216, -1.6924407482147217, -0.5544718503952026, -0.22861672937870026, 1.1101456880569458, 0.736863374710083, 1.1423498392105103, 0.6412051916122437, -2.4118733406066895, 1.396116018295288, -0.8196430802345276, 0.1107965037226677, 0.08386395871639252, -0.32295095920562744, -0.8409484028816223, 1.9767628908157349, 2.470113754272461, 1.3270530700683594, 0.1284964233636856, -0.0499652661383152, -0.42433273792266846, 1.5354398488998413, -0.7656943798065186, -0.2842023968696594, 1.4987571239471436, -1.538616418838501, -0.3345116376876831, 1.1767722368240356, -0.03177560120820999, 1.0727314949035645, 1.348442554473877, 0.9904143810272217, -1.1710611581802368, -0.12220954149961472, -0.4051806926727295, -0.13835735619068146, 0.024535764008760452, -0.8799209594726562, -0.03672359511256218, -0.5329460501670837, 1.8997162580490112, -0.04788193479180336, 0.23444679379463196, 0.4280661344528198, -1.6466917991638184, 1.0401626825332642, -0.7758561968803406, 0.3557300567626953, 1.3469053506851196, -1.1236895322799683, -1.5742334127426147, -1.0381072759628296, -0.22745145857334137, 0.057820066809654236, 0.36991041898727417, 0.3831097185611725, -0.24501504004001617, 0.12320160865783691, -0.24996495246887207, -0.6968651413917542, 1.3569961786270142, 0.4703247547149658, 0.7497409582138062, -1.1573457717895508, -0.7698994874954224, -0.6744290590286255, 1.5632480382919312, 0.059687063097953796, -1.0451273918151855, 0.08091024309396744, -1.257601261138916, -0.6118071675300598, -0.1677345186471939, 1.4037903547286987, 1.8432679176330566, 2.567686080932617, 0.05396155267953873, 0.6790746450424194, 0.16077692806720734, 0.12201080471277237, -0.4896152913570404, -1.0858057737350464, 0.06512988358736038, 1.6036428213119507, 0.210640549659729, 0.657169759273529, -1.4344147443771362, 0.6507155299186707, 0.13257677853107452, -0.9023420214653015, 0.16846413910388947, 0.6789045929908752, 1.1423418521881104, -0.5136069059371948, -0.314130038022995, 1.7404370307922363, 0.3736472725868225, 0.22872552275657654, 0.30405163764953613, 0.11015615612268448, 0.5788077116012573, -0.26063063740730286, 0.6224122643470764, 1.5182784795761108, 0.7501611113548279, 1.0975465774536133, 1.726758360862732, 0.2814487814903259, -0.1328781247138977, -1.35435950756073, -0.8767338395118713, -0.6005656123161316, 1.2069501876831055, 0.678085446357727, -0.571215033531189, 0.9403595328330994, 1.5265530347824097, -0.32992789149284363, 1.5020818710327148, 0.023855043575167656, 1.1379293203353882, -0.02000625804066658, -0.07181939482688904, 0.07111693173646927, 0.6643213629722595, 0.926723062992096, -0.00797546748071909, 0.01890316605567932, -1.3486839532852173, 0.4581184387207031, 0.21960684657096863, 1.4746716022491455, 0.9461629986763, -0.7925689816474915, 0.5459293723106384, -0.7582436800003052, -0.07075007259845734, -1.720415472984314, 0.24268673360347748, 1.5885847806930542, 0.4411991834640503, -0.20330405235290527, -2.459484338760376, -1.048209309577942, -0.011209933087229729, 0.08211341500282288, 1.8450230360031128, 1.589514136314392, -0.5560581684112549, -0.8516233563423157, -0.6919251680374146, -1.6569486856460571, -0.6251344680786133, 1.0816960334777832, -0.7238463759422302, -0.8531782031059265, -0.5681279301643372, 0.04658439755439758, 1.4805397987365723, 1.061382532119751, -1.284134030342102, 1.7527952194213867, -0.38380149006843567, -0.5048292279243469, -0.6695920825004578, 0.08532620221376419, -0.3442942500114441, 2.189267873764038, 0.1153641790151596, 0.4401759207248688, -1.1955385208129883, 1.4693751335144043, 0.9404181241989136, -0.0836850106716156, -0.48556724190711975, -0.6114433407783508, 2.0051777362823486, -1.5643119812011719, -0.9159809947013855, 0.4505768418312073, 0.1629553884267807, -0.20215104520320892, 0.7744563817977905, -2.2464046478271484, 0.8189089894294739, -0.09067293256521225, -0.18998470902442932, 1.0647114515304565, 0.6528805494308472, -0.8957136869430542, -1.7131576538085938, -0.8459955453872681, 0.4109204411506653, 1.4476851224899292, 0.7468999028205872, 0.04281739518046379, -0.8354110717773438, -1.2048192024230957, 0.7782808542251587, 0.16225220263004303, -0.7847067713737488, -1.5123478174209595, 0.38894182443618774, 0.44137051701545715, 0.9972236156463623, -0.5045416355133057, -0.6300069689750671, -1.3912802934646606, 1.5678235292434692, -1.717159628868103, 0.5338494181632996, 0.9231619238853455, -1.676951289176941, -0.9791758060455322, -1.937544345855713, -2.461005449295044, -0.22888164222240448, -1.3762184381484985, 1.4076977968215942, -0.5405733585357666, -0.648946225643158, -1.303398847579956, 0.9878618717193604, 1.0316438674926758, -2.766526937484741, 0.4640398621559143, 0.6612130403518677, -1.2728170156478882, 0.17997099459171295, -0.34012725949287415, -0.2204475998878479, -0.2655642330646515, -0.4988618493080139, -1.571813702583313, -0.5798143148422241, -1.8992012739181519, -0.9771349430084229, -0.4790443181991577, 0.1112477257847786, -2.2435359954833984, 0.36428558826446533, -0.03944644704461098, 0.7235095500946045, 0.006197873968631029, 1.580626130104065, 0.5721137523651123, 0.1488010287284851, 0.664378821849823, -0.48186078667640686, 0.09197887778282166, 0.9205203056335449, -0.35251080989837646, -0.18496891856193542, -0.1319996863603592, -1.3540303707122803, -1.2993171215057373, 0.3419124484062195, 0.6861401796340942, -1.217162013053894, 0.40248510241508484, -1.9007270336151123, 0.26769787073135376, 0.5876412987709045, 0.5660893321037292, 1.9609988927841187, 1.2015477418899536, -0.8876548409461975, 0.3399747908115387, 1.4788649082183838, 0.5753791332244873, -0.10656410455703735, 1.6454421281814575, -0.3820294141769409, 1.5952112674713135, 0.4503709077835083, 0.004939105361700058, 0.35797786712646484, -0.3070247173309326, -0.6855644583702087, -1.2196638584136963, -0.45520707964897156, -0.37015384435653687, 1.5870827436447144, -0.5017355680465698, -0.46836936473846436, -0.5778620839118958, 0.09858124703168869, -0.4226939380168915, 0.9591418504714966, 0.10412309318780899, -0.5460835695266724, -0.07878400385379791, -0.9740148186683655, 1.4706089496612549, 0.7265498638153076, 0.5510488152503967, -0.5825364589691162, 0.4735460877418518, -0.32745596766471863, 0.5103070139884949, -0.09487538784742355, -0.7770867943763733, -0.27153563499450684, -0.2541367709636688, 2.2824106216430664, 0.49811604619026184, 0.9652605652809143, 0.4699923098087311, -1.1332447528839111, -0.4237287640571594, -0.8540067672729492, 1.0163377523422241, 0.9840151071548462, -0.6219698190689087, 0.6116461753845215, -0.1374376267194748, -0.5813822150230408, 1.3445274829864502, -0.42772772908210754, 0.8556832075119019, 0.5470829010009766, 0.1949937641620636, -1.4672964811325073, -0.3996507227420807, 0.15463435649871826, -0.4576803743839264, 0.5885030627250671, -2.2349936962127686, 0.8657900094985962, 0.6351709365844727, -1.0457944869995117, 1.1472150087356567, -1.7466832399368286, 0.37788495421409607, 1.164936900138855, -0.5817500352859497, 0.11088046431541443, -1.5244832038879395, -1.2522634267807007, -0.49119308590888977, -1.037292718887329, -0.6854097843170166, -0.3914239704608917, -1.2004386186599731, -1.707226037979126, -0.30102604627609253, 0.8488807678222656, 0.6420236825942993, -0.3694077730178833, -1.0298336744308472, -2.1819007396698, 1.132326602935791, -1.1926566362380981, -0.3202568590641022, -0.18553133308887482, 1.2525566816329956, 1.5603053569793701, 0.9332139492034912, -1.010164499282837, 1.1789597272872925, -1.4320179224014282, -1.0910279750823975, 1.1764260530471802, -1.6059551239013672, -0.8729509711265564, 0.48400622606277466, -0.07679510861635208, 0.17363615334033966, 0.16319140791893005, 0.20588171482086182, 1.1255598068237305, -0.2695859968662262, -0.10025015473365784, -0.17618592083454132, 0.04899413511157036, 0.45450863242149353, 0.4741511344909668, 1.5803472995758057, -0.5090525150299072, -2.3018035888671875, 0.16500768065452576, 0.3019583523273468, 0.4469515383243561, 0.35339489579200745, 1.0681710243225098, 0.807854175567627, -0.0794391930103302, 0.015089076943695545, -0.47624093294143677, -0.6205606460571289, 0.6024045944213867, 1.1090267896652222, -0.841020941734314, 0.27676424384117126, -2.1420767307281494, 0.8627906441688538, -0.5397734045982361, 1.737941861152649, 0.3297997713088989, -0.4537223279476166, -0.18226566910743713, -0.9505640864372253, -0.6935275793075562, -0.7319592833518982, -2.875270366668701, -0.36796051263809204, 0.13604366779327393, 0.050816480070352554, -0.33321115374565125, 0.00844060443341732, -1.3832745552062988, 1.2378729581832886, -0.7434046268463135, -0.7645550966262817, 1.3264566659927368, -0.5505379438400269, -1.404517412185669, 1.2661150693893433, 1.911433219909668, -0.35732850432395935, 0.7202581763267517, -0.11043676733970642, -1.2229397296905518, 0.5929151177406311, 1.0781241655349731, 0.8330463767051697, -0.29536449909210205, -1.2966187000274658, 1.2919453382492065, 1.1506562232971191, 0.06223386153578758, 0.5338473916053772, -1.1244734525680542, 0.15295740962028503, 0.1433323621749878, 1.0992166996002197, 0.8740388751029968, -0.6869557499885559, 1.7299175262451172, -0.5670834183692932, 1.8415709733963013, 1.188293695449829, -1.025314450263977, 1.0921560525894165, 1.533350944519043, 0.5693281888961792, -1.8904889822006226, 0.04275571182370186, -0.30702629685401917, 1.0261467695236206, 0.7796684503555298, 0.7311495542526245, -0.5509795546531677, 0.536873459815979, -1.6891233921051025, -0.6354403495788574, 0.319472074508667, -0.8794212937355042, -0.3037111461162567, 1.3917526006698608, -1.7708697319030762, -0.5722261667251587, 1.2167994976043701, -0.06046901270747185, -0.5880447030067444, -0.28260767459869385, 0.6775118708610535, 1.5658174753189087, -0.2360505908727646, 1.0063856840133667, 2.890587091445923, -0.1237853392958641, 0.8688939213752747, -0.009314391762018204, -0.9392431974411011, 0.9312099814414978, -0.5911226868629456, -1.6880571842193604, -1.2457648515701294, 1.5887211561203003, -2.0567965507507324, 0.18048055469989777, -0.5097475051879883, -0.16093307733535767, -1.0162353515625, -0.41193777322769165, 0.7713006138801575, -1.4807413816452026, -0.4276970326900482, 1.266402006149292, 0.30357304215431213, -0.515057384967804, 0.8995236158370972, 0.9274154305458069, 1.0351998805999756, 2.5173075199127197, 0.8834332823753357, 0.29156768321990967, 0.0376737043261528, -0.7842275500297546, 0.12074745446443558, 1.0298289060592651, -1.4376283884048462, -0.6505727767944336, 0.8232172131538391, -0.6422522664070129, 1.2281156778335571, -0.22898292541503906, -0.13319131731987, 0.9175527095794678, 0.7105757594108582, -0.9897248148918152, 0.8618370890617371, -0.6485443115234375, 1.2907264232635498, -0.5814052820205688, 0.1822277307510376, -1.3633100986480713, 0.999645471572876, 0.3596426844596863, 0.2160232961177826, -0.6705122590065002, 1.1623069047927856, 0.819460391998291, 0.5844368934631348, 1.2177716493606567, -1.7342090606689453, 0.5989090800285339, 0.28875601291656494, 0.3913966715335846, -0.17327016592025757, 1.7005515098571777, 0.5220088362693787, -0.028884289786219597, 0.5511272549629211, 0.07017995417118073, -0.017001330852508545, 1.3739912509918213, 0.1955704391002655, -0.1105683445930481, 1.59213387966156, 0.07021044939756393, 1.0124748945236206, 1.6719093322753906, -0.2397347092628479, 0.7332859039306641, 1.2694731950759888, -0.14825065433979034, 1.8330276012420654, -1.0914374589920044, 0.27713796496391296, -0.649734616279602, 0.4696609377861023, 0.33695051074028015, -1.2718766927719116, 0.21914571523666382, -1.5459543466567993, 0.6486092209815979, 1.1619946956634521, -2.378964424133301, -0.6098499298095703, -0.0013225340517237782, -0.47263094782829285, -0.5431808829307556, 0.18280990421772003, -0.26527076959609985, -0.4196396768093109, -1.2138967514038086, -0.45178401470184326, 1.1957545280456543, -0.6807944178581238, 0.3617818355560303, -0.13174858689308167, 0.9050842523574829, 1.5634360313415527, -0.19048352539539337, -0.28106701374053955, 0.5946237444877625, 1.9266304969787598, -0.5050642490386963, 1.6156432628631592, 0.5213075280189514, 1.0840396881103516, 1.4387339353561401, -0.536937952041626, 0.5948866605758667, 0.5233603715896606, 0.2832811176776886, 1.1178780794143677, 0.12137003242969513, 0.016532421112060547, -0.8009434938430786, -1.2014087438583374, -0.9415460228919983, -1.262373447418213, -0.5188887715339661, 1.1577497720718384, 0.6508870124816895, 0.4047919809818268, 0.5277202725410461, 0.4124523103237152, 0.14619311690330505, 0.06310191005468369, 0.1924849897623062, 0.9158732891082764, -1.206373929977417, -0.16562853753566742, -0.0031751906499266624, -3.050956964492798, 1.1123957633972168, 0.9707323312759399, -1.0719233751296997, 0.2709895074367523, 0.679750919342041, -1.5331637859344482, -0.4148433208465576, -1.2761174440383911, 0.5076019167900085, -0.6806893348693848, -0.10647004097700119, -0.21149155497550964, 0.6398895382881165, -1.0593150854110718, 0.6393618583679199, 2.6319386959075928, 0.08906187117099762, 1.2958364486694336, -0.46670621633529663, -1.2701574563980103, 1.0372531414031982, 0.2527015805244446, 0.8254101276397705, -0.1277332305908203, 0.6166746020317078, 0.34766483306884766, 0.3925335109233856, -0.9342828392982483, 0.059572312980890274, -1.917094111442566, 0.443163126707077, 2.2619035243988037, -1.5772342681884766, -0.03995102271437645, 0.5113078951835632, 0.23919963836669922, -0.6019901633262634, -0.8531404137611389, -0.10104826092720032, 0.46309414505958557, -0.0998968705534935, -0.4791436493396759, -0.0875513032078743, 0.7353466749191284, 1.6621599197387695, -0.557563304901123, 0.680267870426178, -0.6241593360900879, -1.5304261445999146, -0.452452689409256, 0.9085820913314819, 0.7988507151603699, 1.9557322263717651, 0.8478405475616455, -0.8805211186408997, 1.1113238334655762, 2.771214723587036, -0.54718416929245, 0.37288936972618103, 0.49757975339889526, -0.8126987814903259, -0.9130460619926453, -0.052297912538051605, 0.05338472127914429, 1.1681379079818726, -0.6699094176292419, -1.2427036762237549, -1.4232995510101318, 0.5802181363105774, 0.4883461892604828, 1.0501070022583008, 1.308441162109375, 0.03691744804382324, -1.4003255367279053, -0.06909666955471039, -0.03743704780936241, -1.7235692739486694, 0.59831702709198, 2.3610031604766846, 0.9373816847801208, -0.5547031760215759, -0.9033388495445251, -0.3212847411632538, -1.2600148916244507, 0.3986276686191559, 0.6158369779586792, -0.48919978737831116, 1.4020164012908936, -0.1662014275789261, 0.7491440176963806, 0.6057266592979431, 0.5109907984733582, -0.8899944424629211, 0.08709380775690079, -0.2586635947227478, -0.36240246891975403, 0.7337353229522705, 0.2803550958633423, 0.33854973316192627, 1.4174858331680298, 0.568600058555603, -0.8837053179740906, -2.2261369228363037, -0.3702239692211151, -0.08445814996957779, -0.25815245509147644, 1.598703145980835, 0.177572101354599, -0.8122385740280151, 0.6545767188072205, -0.6159388422966003, -0.860555112361908, 0.8318512439727783, -1.1305092573165894, -0.5201166272163391, 1.4293198585510254, -0.19006198644638062, -0.515075147151947, -0.6506522297859192, 2.135791301727295, 2.2374765872955322, 0.11876782029867172, 1.7498629093170166, -0.6199191808700562, -1.3720207214355469, -1.2314478158950806, 1.0529206991195679, 0.7628799676895142, 0.17420610785484314, 0.20176084339618683, 0.1037374958395958, -0.6477089524269104, 0.5363097786903381, -0.7144883275032043, -1.6273062229156494, 0.6243748068809509, -0.8766186237335205, 1.2714002132415771, -0.41730400919914246, -0.7682960629463196, 0.8598355650901794, 0.8005945682525635, 1.9048371315002441, 0.6120222806930542, 0.054518841207027435, 0.8040328025817871, 1.2022786140441895, 0.36793166399002075, 0.34892696142196655, -0.0007938878843560815, -0.17306075990200043, -1.3272998332977295, -0.9423931837081909, 0.5773909687995911, -0.9258643984794617, 1.2730071544647217, -2.2884199619293213, -1.0040309429168701, -1.0404576063156128, 0.0659661665558815, -0.10698999464511871, -1.1317108869552612, -1.4879565238952637, -1.296921968460083, 0.039687927812337875, -0.2061060518026352, 0.9878579378128052, 0.04844898357987404, -0.39293843507766724, -1.7263789176940918, 0.1043192520737648, -0.1882563680410385, -0.17677269876003265, 0.9420843124389648, 0.5392868518829346, 0.2684895992279053, -0.06401674449443817, 0.6576750874519348, -2.485351085662842, -1.1698923110961914, 1.1342939138412476, 0.9016340374946594, -0.6671633720397949, 0.430669367313385, -0.14874885976314545, 2.6298863887786865, -0.7222122550010681, 0.3958357870578766, -0.4709875285625458, -0.09032817929983139, -0.5145200490951538, -0.9637873768806458, 1.0240005254745483, -0.4454215466976166, 3.3589224815368652, 1.262107014656067, 1.4234328269958496, -0.9998622536659241, -1.2108155488967896, -1.8648412227630615, -1.6227598190307617, -1.707764744758606, -1.444087028503418, -0.16719795763492584, -1.4592854976654053, 1.0013536214828491, -1.1043736934661865, -1.034488320350647, 0.020993784070014954, 0.15004441142082214, -0.7255656719207764, -0.9162054061889648, 0.2217898964881897, -0.6262701749801636, 0.4168833792209625, 1.4208054542541504, 0.09220325201749802, -1.5445624589920044, 0.17199108004570007, -1.2726298570632935, 0.4296287000179291, -1.625369906425476, -0.9118110537528992, 0.19541415572166443, 2.3724770545959473, 2.108208179473877, -0.06784138828516006, 1.4434552192687988, -0.49843084812164307, 1.3129242658615112, -0.6165682077407837, 0.3319661617279053, 0.09062575548887253, 0.4178556799888611, -1.0221396684646606, -0.5293115973472595, -1.0888476371765137, 0.7451474666595459, -1.3811125755310059, -1.1086994409561157, 0.7000687122344971, -0.638623058795929, -0.4691486060619354, -0.5878876447677612, 1.8014129400253296, -0.4824659526348114, -0.1888132244348526, -2.164358615875244, -1.0110466480255127, -0.7926065325737, -0.1668112426996231, 0.5385501384735107, -0.9524250030517578, 1.894675612449646, 0.4680550694465637, -0.01698555424809456, -0.536697268486023, -1.8803844451904297, 0.007895702496170998, -0.7091663479804993, 0.1457778364419937, -0.3857608139514923, -2.1070258617401123, -0.07223580777645111, 0.81137615442276, 0.26388755440711975, 0.4852621853351593, -0.37126997113227844, -1.4575022459030151, -1.7473442554473877, -1.0660295486450195, 0.17799371480941772, 1.7821799516677856, -0.5666010975837708, 0.07178667932748795, 1.4878343343734741, -2.388610363006592, 1.0216643810272217, 0.10772693902254105, 0.5732903480529785, 0.12066280096769333, 0.09435029327869415, 1.0570487976074219, -0.3191593885421753, -1.9668598175048828, -1.2546379566192627, -1.4777592420578003, 0.17911121249198914, -0.6535289287567139, -1.4131885766983032, -1.3835684061050415, -1.9689533710479736, 1.1591835021972656, 1.2210184335708618, 2.8185019493103027, 1.4954907894134521, 0.36757534742355347, 0.17112010717391968, -1.3398418426513672, 0.879410982131958, 0.9715651273727417, -0.9066985249519348, -0.30308228731155396, 1.207381248474121, -1.2154672145843506, -0.191992849111557, -0.7131253480911255, -1.1931525468826294, 0.22751028835773468, 0.0247048269957304, 1.749855875968933, -0.013776729814708233, -0.3438705503940582, 0.45922836661338806, -0.04835943505167961, -0.36679065227508545, 0.9292201399803162, 0.9197882413864136, 0.2643466889858246, 0.15931221842765808, 0.023831360042095184, -2.3461475372314453, -0.08937214314937592, -0.2428535521030426, -0.180295929312706, -0.09887869656085968, -1.0386866331100464, -1.7705119848251343, -1.4718592166900635, 1.3004989624023438, 0.08229079842567444, -0.4996975362300873, 2.3526296615600586, 0.8356854319572449, -0.6773326396942139, -0.6550083756446838, -1.9057680368423462, -1.4954230785369873, 0.9183016419410706, 0.9429183602333069, 1.805853247642517, -0.4576466679573059, -0.22437813878059387, 0.5288441181182861, -0.18230149149894714, 0.8850656151771545, -0.41292402148246765, 0.9111687541007996, -1.2310338020324707, 0.5248355269432068, -0.18790845572948456, -0.37964579463005066, -0.959135115146637, 1.571434736251831, 1.685868263244629, 0.8464688062667847, -0.21398845314979553, 0.4696902632713318, -0.7020891904830933, -1.3822646141052246, 1.1495239734649658, -0.4118395149707794, 0.843795895576477, -0.9927191138267517, 0.6259946227073669, 0.14402805268764496, -0.09385617822408676, -0.1890084147453308, -0.5825456380844116, -0.11308571696281433, -0.6686331033706665, -0.3480835556983948, -1.2434864044189453, -1.333065152168274, 0.11082851886749268, -1.2552176713943481, -0.40180134773254395, 0.30873292684555054, -1.5009045600891113, -0.2898894250392914, -0.24739253520965576, -1.4408141374588013, -1.147249698638916, 0.39352166652679443, 1.7323898077011108, -0.30937057733535767, 0.29574212431907654, 1.454756498336792, 0.21660993993282318, -1.2972122430801392, 1.5416613817214966, -0.5545743107795715, 0.0748017355799675, 0.4494786560535431, 0.31381338834762573, 0.1868966966867447, 2.2479023933410645, -1.4521516561508179, 1.0327118635177612, -0.37983012199401855, 1.7125099897384644, -1.0463104248046875, -0.01647964119911194, 1.2119114398956299, -1.499707579612732, -1.1048763990402222, 0.013685030862689018, -1.1747664213180542, 0.8149900436401367, 0.1235756203532219, 1.3773130178451538, 0.6384586095809937, 0.3529594838619232, 1.0514612197875977, 0.20229746401309967, -1.4234830141067505, 0.16574494540691376, -0.9917578101158142, 1.3096951246261597, 1.563176155090332, -0.8677763938903809, 1.1305596828460693, 1.6871912479400635, 0.5302435159683228, 0.08982709795236588, -0.5612765550613403, 0.5294508934020996, 0.6170132160186768, 0.2549276053905487, -0.4757145941257477, -2.5049846172332764, -0.26683923602104187, 0.47482919692993164, -0.001538446405902505, -1.4715617895126343, 0.037053175270557404, 0.2908550798892975, -0.7335461974143982, -0.7442249059677124, 1.250093936920166, -0.18291901051998138, 1.0740705728530884, 0.9189435243606567, -0.9289937019348145, -0.8976274132728577, 0.7251251935958862, -1.1135988235473633, -2.898503065109253, -1.0959323644638062, -0.2505432665348053, 0.7690156698226929, 1.4667807817459106, -0.48935070633888245, -0.9829414486885071, 1.248702049255371, -0.9095540046691895, 1.8687459230422974, -1.6423439979553223, 1.1558877229690552, -1.308897852897644, -0.05173703655600548, -0.6002361178398132, 1.5400381088256836, -1.806725025177002, -0.82221519947052, -0.15094682574272156, -0.1784869134426117, 0.8488380312919617, 0.6056347489356995, 1.0368762016296387, 1.5379297733306885, -0.24295777082443237, 0.7231109738349915, -1.121065616607666, 0.7354719638824463, -0.916534423828125, 0.2728676199913025, -0.10457666218280792, 0.9905548691749573, 0.23892095685005188, 0.5914276838302612, -1.74131441116333, -0.5599988698959351, -0.501049816608429, 0.8338526487350464, 0.5866628289222717, 0.2798481285572052, 1.8818867206573486, 0.52953040599823, 0.8495050668716431, -0.27156636118888855, 1.0430002212524414, 0.7831799387931824, -0.8750179409980774, 1.7780015468597412, 0.39911726117134094, 1.4441523551940918, -0.9612338542938232, 0.562307596206665, 0.3005950152873993, -2.2789711952209473, 0.47049862146377563, -1.9984142780303955, -1.331498384475708, 0.11234509199857712, 1.4494037628173828, -1.7609636783599854, -0.12499771267175674, 1.125234603881836, -0.582206666469574, 0.42166319489479065, -0.6300133466720581, 0.5190266966819763, -0.30185404419898987, -0.1911984086036682, -0.4600730836391449, 0.2706931233406067, -1.4759622812271118, 1.0105702877044678, -1.263306975364685, -0.2386338859796524, 1.0022751092910767, -0.0067432839423418045, 0.1704321801662445, -0.05583381652832031, 0.23632493615150452, -1.3914934396743774, 0.37435322999954224, -0.3419084846973419, 2.1249608993530273, 0.3781079947948456, -1.6466561555862427, -2.6217827796936035, 1.7432754039764404, -1.098479986190796, 0.7688190937042236, -0.9963581562042236, 0.21584074199199677, 0.04804228991270065, -0.45814114809036255, 0.2724322974681854, -0.1813647598028183, 0.5106143951416016, 1.0331709384918213, 0.3741779923439026, -1.0728622674942017, -0.2303556501865387, 0.9974189400672913, 1.372605562210083, 1.3163228034973145, -1.6014857292175293, -0.6286336183547974, 0.21302847564220428, -0.7988501191139221, 1.1014257669448853, 0.10882757604122162, 0.6035473346710205, 1.194074273109436, 1.257627010345459, 0.8597365021705627, 0.9348639249801636, -0.736703634262085, -0.8571603298187256, -1.7484601736068726, -1.324371337890625, -1.2880703210830688, -0.8298258781433105, 1.3494341373443604, 0.46347612142562866, -1.1919856071472168, 1.2180674076080322, -0.9514843821525574, -1.2251286506652832, 0.13753868639469147, -0.6306621432304382, 0.9569286108016968, -0.22946161031723022, -0.30960431694984436, 0.9661593437194824, -1.2376983165740967, 0.6484017968177795, -0.07579783350229263, -0.9643086194992065, 0.005108669400215149, 1.4111504554748535, 0.09115038812160492, 0.4582671821117401, -0.017505815252661705, 0.5598896145820618, 1.278328776359558, 0.41341909766197205, 0.6409271955490112, 1.2282317876815796, -0.035102978348731995, -0.6834433674812317, -0.18021099269390106, 0.6815842390060425, -1.279491662979126, 0.10691792517900467, -1.662201166152954, -1.4142515659332275, -0.020978938788175583, -0.3998521566390991, 1.0631771087646484, -0.28365179896354675, -0.18033093214035034, -2.031385898590088, 1.5353189706802368, -0.15312930941581726, 1.608020305633545, -1.0318204164505005, 2.246006727218628, -1.6585235595703125, -1.9343105554580688, 1.162029504776001, 0.9379253387451172, 0.6766945719718933, -0.24719849228858948, -0.5440332889556885, 0.3209707736968994, -0.781624972820282, -1.4456305503845215, 0.5910508036613464, -1.2145006656646729, 0.4998749792575836, -1.4013938903808594, -0.20397241413593292, -0.9961748123168945, 0.7379400134086609, -1.2233684062957764, -0.8230211138725281, 1.220197081565857, 0.06014782935380936, -1.1792118549346924, 0.06572696566581726, -0.09914425760507584, 1.2011622190475464, -1.5678884983062744, 0.16346696019172668, -0.4360974133014679, 0.8789793252944946, -0.3519238531589508, -0.18348267674446106, -0.4559611976146698, 1.6339091062545776, 0.40771833062171936, -0.06608321517705917, 1.0750685930252075, 0.3907485008239746, 0.32832270860671997, -2.1321489810943604, -0.16000214219093323, -1.4904578924179077, 1.1697025299072266, -0.09698248654603958, 1.20497727394104, 0.5428312420845032, 0.7458246350288391, -0.2708355188369751, 0.8881698846817017, -0.07448791712522507, 0.47162744402885437, -0.02950538508594036, -2.1155736446380615, -0.554053008556366, 0.31117624044418335, 0.4191437065601349, 0.22559569776058197, -0.34206804633140564, -1.0553549528121948, 1.153409719467163, 0.2735050916671753, 0.5762127637863159, 1.388316035270691, 1.164623498916626, -1.2610924243927002, -2.3929569721221924, -0.7584720253944397, 1.5889043807983398, -1.3209805488586426, -0.3919130861759186, -0.7252758145332336, -0.22667694091796875, 1.5887856483459473, 1.147692084312439, 0.47394391894340515, -0.7832009196281433, -0.7071146368980408, 0.393385648727417, 1.4948625564575195, -0.005472155287861824, -0.11012819409370422, 1.075179100036621, -0.018056092783808708, 1.4517606496810913, -0.8570992946624756, 0.6709335446357727, 0.9479600787162781, 0.5478214025497437, 0.7343597412109375, -0.9026279449462891, 0.3653360605239868, 0.6719386577606201, 0.7015081644058228, 0.25234025716781616, 1.863088846206665, -0.4151866137981415, 1.8748180866241455, -0.3054507374763489, 0.2982083559036255, 2.383383274078369, 2.465808153152466, 1.891965389251709, -0.4345826208591461, -0.11878263205289841, -2.2347187995910645, 1.9404337406158447, 0.5346408486366272, -1.0394742488861084, 0.9809909462928772, -1.2792352437973022, 0.03405838459730148, -0.03865910694003105, 0.02245541848242283, -1.1722650527954102, -0.5056126117706299, -0.7253618240356445, -0.3826448321342468, -2.949065685272217, 0.047838181257247925, 0.7325573563575745, 0.49925756454467773, -0.22220833599567413, 0.01881149597465992, -1.0515847206115723, -0.08811827003955841, -1.4055736064910889, -1.7141648530960083, -0.6604012846946716, 1.8337335586547852, -0.3339298963546753, 0.5186715722084045, -2.548247814178467, -0.3803149461746216, -0.41855376958847046, 0.560646116733551, -0.3520151674747467, -0.6007819771766663, 2.35575532913208, -0.3772777318954468, -0.6340063810348511, 0.6628686189651489, 0.0007957842317409813, 0.3617171347141266, -0.043700650334358215, -0.23799097537994385, 0.1638251394033432, -0.34969362616539, 1.0892481803894043, 0.8491886258125305, -0.30113112926483154, 1.6578879356384277, -0.6423513293266296, -0.44436007738113403, 0.3759249150753021, 1.269841194152832, 0.6234681606292725, -0.3795012831687927, -0.8794938325881958, 1.3460357189178467, 1.3661847114562988, 0.796556293964386, -0.863705039024353, -0.05417148396372795, -0.33034101128578186, -0.27858084440231323, -1.172613263130188, -0.9629945755004883, 1.7111947536468506, 0.5124805569648743, -1.1667832136154175, 1.002427101135254, 1.2882421016693115, 2.693357229232788, -0.7057627439498901, -0.5323449373245239, 1.4853968620300293, -1.2640421390533447, -0.8134822249412537, -1.4218536615371704, -1.0599645376205444, 1.2976908683776855, 1.2773083448410034, 0.11466667801141739, -0.22228959202766418, 0.13107073307037354, -0.827911376953125, -1.3508397340774536, -0.3455471396446228, 0.4237767159938812, 0.3544265329837799, -0.09184569120407104, 2.1224825382232666, 1.616126537322998, -1.7801380157470703, 0.2829134166240692, 0.10551932454109192, 1.0064929723739624, -1.6886299848556519, 0.10271374881267548, -0.5875135064125061, 0.018780259415507317, -1.05196213722229, 0.817944347858429, 0.5786489248275757, 2.2034006118774414, 0.4417465925216675, 0.5523250699043274, 0.6777493953704834, -0.27893897891044617, 0.7581221461296082, 1.7681310176849365, -2.036661148071289, -0.5512054562568665, 0.25391948223114014, 0.13254566490650177, 0.5104960799217224, 0.17242653667926788, -0.2680816352367401, -0.9709219932556152, 0.010909977369010448, 1.413254737854004, -0.1766408532857895, -1.5561280250549316, -1.1973628997802734, -0.7873796820640564, -0.8003644347190857, 0.30634450912475586, 0.29543596506118774, -3.2072343826293945, -0.44111713767051697, -0.06558957695960999, -0.5424308180809021, 1.1828824281692505, -0.5343607664108276, -0.538334310054779, -0.3299182057380676, 1.0026392936706543, 0.680230975151062, 0.5143145322799683, 1.5974931716918945, 0.027721375226974487, -1.8912193775177002, -2.1209182739257812, -1.2730568647384644, -1.312934160232544, 0.7237784266471863, 1.668962836265564, -1.1428145170211792, -0.1611326038837433, -0.8537631034851074, 0.6175858378410339, -0.3518194556236267, -1.5233521461486816, -0.53852778673172, 0.8327167630195618, 0.04938148707151413, 1.7157841920852661, 1.7432516813278198, -0.8770208358764648, 0.5202935934066772, -0.4758184254169464, -0.0911974161863327, -1.6235429048538208, -1.9002794027328491, -0.2998288869857788, -0.7159229516983032, 0.8629506826400757, -1.7432420253753662, 0.20040780305862427, -1.0727816820144653, 0.5839844942092896, 0.10791472345590591, 1.3086339235305786, -0.5290069580078125, 0.6084880232810974, 1.5334935188293457, 0.0228272695094347, 0.20016446709632874, 0.012144616805016994, 0.7306630611419678, 0.4200141429901123, 0.5421652793884277, -0.449334055185318, 0.7459540963172913, 1.7680264711380005, -0.05899420380592346, 0.7601291537284851, -1.977652907371521, 1.2896006107330322, 0.30254775285720825, -0.7341101765632629, -0.4136104881763458, 0.26320332288742065, 0.24277032911777496, 0.18735450506210327, 1.2134053707122803, -0.05192957818508148, -0.26028159260749817, 2.686897039413452, -1.0847978591918945, 0.9042449593544006, 0.29937562346458435, -1.3459243774414062, -0.5091967582702637, -1.2426261901855469, 0.7461885809898376, 1.4713671207427979, 0.762100100517273, 0.6147345900535583, 0.7558276057243347, 0.02916189283132553, 0.809027373790741, -1.3356211185455322, 1.471940517425537, -0.5757684111595154, 0.9140912294387817, -0.4436546564102173, -0.06218663603067398, -0.4903770685195923, -0.5165223479270935, 0.5590125322341919, 1.5836371183395386, 0.6500030755996704, -1.7182910442352295, -0.053916528820991516, 0.40369078516960144, 1.7012763023376465, 1.0040489435195923, -0.2755620777606964, 1.5002753734588623, -0.23245732486248016, 0.9416648745536804, -0.474442720413208, -0.5746128559112549, -1.0172291994094849, 0.722305178642273, -0.8126304149627686, -0.12759929895401, 0.08558949828147888, 2.847954750061035, -1.0028847455978394, 0.05917101353406906, -0.6341409683227539, -0.5655857920646667, 1.826336145401001, -0.32523033022880554, 0.6033847332000732, 0.33348825573921204, 0.4623543620109558, 0.6419694423675537, 0.49549025297164917, 0.08865521848201752, 0.8094846606254578, 0.2926851212978363, -0.6602833867073059, 1.7324159145355225, 0.1165682002902031, -2.041956663131714, -1.5952214002609253, -2.437878370285034, -0.4781472086906433, -0.47816431522369385, 1.3445265293121338, -0.368543416261673, 1.8051320314407349, -0.7608892321586609, -2.3840723037719727, -0.9338608384132385, -0.22080554068088531, 1.868072509765625, -0.2276218682527542, -0.1872619092464447, 0.4403112530708313, -1.1297807693481445, 0.12417588382959366, 0.9761276841163635, 0.18222028017044067, 0.9834956526756287, 1.0236315727233887, 0.11170033365488052, 0.512174665927887, 0.3243108093738556, -0.012013283558189869, -0.4311507046222687, 0.07332711666822433, 0.37495312094688416, -0.07552938163280487, -1.1464073657989502, 1.6975452899932861, -0.3927444517612457, -0.4263361394405365, 0.609244167804718, -0.3953016996383667, -0.34183353185653687, 0.3929564356803894, 0.2795643210411072, -1.7204504013061523, 0.7336145639419556, 0.40480536222457886, -2.582430362701416, -0.7384687066078186, -0.23070429265499115, 1.0438261032104492, 1.2333886623382568, -1.0856820344924927, -0.20244480669498444, 0.08218121528625488, 0.2081371396780014, -0.8593199849128723, -0.600242018699646, -0.08032896369695663, -1.1299724578857422, -0.840798020362854, -0.8250700831413269, -0.34039270877838135, 2.452350378036499, -0.019506728276610374, -1.9802414178848267, 0.2288457453250885, -0.8905921578407288, -0.6120980978012085, -0.26069122552871704, -1.1756961345672607, 0.46495112776756287, 0.10934708267450333, -0.8198633193969727, 0.9348672032356262, -3.1884102821350098, 2.1090407371520996, 0.24132812023162842, 1.160739779472351, 0.17717689275741577, -0.9965425133705139, 1.0410083532333374, -0.08167597651481628, 0.2565411627292633, 0.8741801977157593, -0.8053944110870361, 0.4157010316848755, 1.0287288427352905, 1.7906479835510254, 0.39221787452697754, 0.21490700542926788, 0.6577698588371277, -0.8518460988998413, 0.07819042354822159, -1.1686774492263794, 0.019078098237514496, 0.2233276516199112, -0.10682987421751022, 2.0573174953460693, 1.2136534452438354, 2.1952178478240967, -0.0814218744635582, 0.5263664126396179, -0.24258245527744293, 0.23833516240119934, 0.5174808502197266, 1.1250402927398682, -0.5059722065925598, -0.32620832324028015, 0.5696147680282593, -1.5143219232559204, 0.454179972410202, 0.029636487364768982, 0.6794599890708923, -0.37463492155075073, 0.8805505633354187, -1.9918814897537231, 0.7338330745697021, 1.8503166437149048, 0.05540546774864197, 0.11581845581531525, -0.4009366035461426, -1.3038147687911987, -0.5598581433296204, -0.8613735437393188, -0.7757596969604492, 1.3132522106170654, -0.4441249370574951, -0.4477955102920532, -0.9544793367385864, -1.4065356254577637, 0.7195684313774109, 1.131259799003601, -1.4850620031356812, -1.0941874980926514, -0.1770883947610855, -1.4956858158111572, -0.4292241334915161, 0.39093008637428284, -1.4608700275421143, -0.3948725461959839, -0.5154408812522888, 0.39632025361061096, -0.10920842736959457, 0.1966973841190338, 1.5251528024673462, 1.061166763305664, -2.0330708026885986, 1.3792924880981445, -0.3530080318450928, 1.5820177793502808, 0.903096616268158, -0.5112083554267883, 0.04009191319346428, -0.500501275062561, 0.6008800864219666, -0.9176449775695801, 0.24589507281780243, -0.618348240852356, -1.4321776628494263, 1.4362372159957886, -0.7735602259635925, -0.12512266635894775, -0.8438405990600586, -0.7263627648353577, 1.338580846786499, 0.6130810976028442, -0.49127310514450073, -1.5279780626296997, 1.019459843635559, 1.7082659006118774, 0.16897858679294586, -0.8244025111198425, 0.12552015483379364, 1.030438780784607, 0.8610892295837402, -1.5090858936309814, 1.289884328842163, -0.7104990482330322, -0.8854636549949646, 0.4257853925228119, 0.508094310760498, 0.6378130316734314, -1.3107389211654663, 0.8742777705192566, -0.3957538306713104, 1.356938123703003, -1.2660173177719116, 1.660935878753662, 2.1242899894714355, 0.6292107701301575, 2.011545181274414, 0.13069862127304077, 0.936915934085846, -0.026498815044760704, 1.2425727844238281, -0.32590728998184204, -0.36339646577835083, 1.7870045900344849, -0.7188898921012878, 0.028506677597761154, 0.5360442996025085, 0.629481852054596, -0.24481205642223358, -0.6896515488624573, -1.0438225269317627, -0.3405304253101349, 0.6701284646987915, -0.906564474105835, 1.0440914630889893, 0.16318511962890625, 0.40945538878440857, 0.27749061584472656, 0.4607418477535248, -0.5982133746147156, 0.4855753779411316, 0.15911883115768433, 1.3508350849151611, -2.0586884021759033, -0.11091636121273041, 0.02789555862545967, 0.07857558131217957, -0.37209993600845337, 1.2791342735290527, 0.7232822775840759, 0.47004613280296326, 0.08410486578941345, -1.7969142198562622, -0.20849762856960297, -0.09593652933835983, 1.2692790031433105, -0.05037882924079895, -1.1810054779052734, 1.6034071445465088, -0.810124397277832, -0.5623733401298523, 0.4435376822948456, 0.22817543148994446, 0.15524882078170776, 1.3725751638412476, 0.11372845619916916, -0.5887156128883362, -0.5078167915344238, 1.2036014795303345, -1.4188450574874878, 1.0097662210464478, 0.8944252729415894, -1.9758367538452148, -1.0227335691452026, 0.04778359830379486, -0.3892786502838135, 1.266284704208374, 0.040516894310712814, -0.7431278228759766, -1.4190316200256348, 0.6398575305938721, -1.1869183778762817, 0.8583124876022339, 0.8350624442100525, -0.0869499072432518, 0.8790743350982666, 0.5610503554344177, -0.763422429561615, 0.310962975025177, -0.5529798269271851, 1.173098087310791, 0.0789235308766365, 0.5788870453834534, -1.3012173175811768, 0.13279098272323608, -1.0938442945480347, 0.03948165848851204, 0.28440743684768677, -0.27750521898269653, 0.2960238456726074, -0.5374257564544678, 0.8144336342811584, -0.5527196526527405, 1.4345965385437012, -0.651016891002655, 0.3306805491447449, 0.5514202117919922, 0.3651534914970398, 0.6243667006492615, 1.0811444520950317, 0.2313338965177536, -0.5262514352798462, 0.9226393103599548, -0.661202609539032, -1.0442887544631958, 0.24053825438022614, 0.7595919370651245, 1.2280770540237427, -1.755439043045044, 0.06604096293449402, 0.7253744602203369, 0.8346980214118958, 1.411243200302124, 1.2047467231750488, 1.8197215795516968, -0.5335526466369629, -0.05253898724913597, 0.32980969548225403, -0.7848581075668335, -0.651375949382782, -0.048765093088150024, -1.0563907623291016, 1.497525930404663, 0.467510461807251, -0.33159545063972473, 1.4283987283706665, -1.9621683359146118, 0.8587828278541565, -1.289269208908081, -1.8559852838516235, -0.7179338335990906, 1.8351424932479858, -0.24336323142051697, -1.5586479902267456, -0.9116456508636475, -0.6148039102554321, 1.4310938119888306, 0.09129183739423752, 1.3665157556533813, 1.562042236328125, -1.0732247829437256, 0.15747280418872833, -1.4857490062713623, -1.9830330610275269, 0.5961843729019165, -0.9041299223899841, -0.42212432622909546, 0.03861158713698387, -0.7430020570755005, -1.4873653650283813, 0.458061546087265, 0.2562367916107178, 1.4053436517715454, 1.0397944450378418, -0.15740688145160675, -0.634930431842804, 1.341964840888977, 0.4766789674758911, 0.6013387441635132, 0.09670930355787277, 1.247721552848816, 0.48909011483192444, 0.37745076417922974, 0.44439610838890076, -0.2825600504875183, -0.915821373462677, 0.22894501686096191, 0.3176787197589874, 0.03715020418167114, -1.5859203338623047, 0.9053625464439392, 0.4851658344268799, 0.20942391455173492, 1.2289648056030273, -0.3822374939918518, -0.3645208179950714, -2.0994338989257812, -0.5006208419799805, 0.25042134523391724, -0.0007108810823410749, -1.1680313348770142, 0.9257221221923828, -0.17274168133735657, -1.9461644887924194, 0.399829626083374, 1.517581820487976, 0.49168524146080017, 1.1054785251617432, 0.820176899433136, 0.2020748257637024, -0.24443306028842926, 0.30191126465797424, 0.621188759803772, -0.5449562072753906, 1.759932518005371, -1.2703312635421753, -0.5224642157554626, -1.6927348375320435, 0.8688353300094604, 0.6378787159919739, -0.6551573872566223, 1.527209758758545, 0.1588735580444336, -0.3231879770755768, -0.5210309028625488, 1.8088433742523193, 0.43621161580085754, -1.0114326477050781, -0.27170881628990173, -0.7637826800346375, -1.4501283168792725, -0.4078676998615265, -1.681631088256836, 1.0070891380310059, 1.5625532865524292, -0.7700400352478027, 0.2932353913784027, -0.5281410813331604, -0.008315454237163067, 0.050547558814287186, 1.125842809677124, 0.2833024561405182, -0.3798162639141083, 2.065704584121704, 0.5678950548171997, -1.5446419715881348, 1.1036845445632935, 0.610395073890686, 0.24040032923221588, -1.1143299341201782, -1.7316699028015137, -0.6439273357391357, -0.2832329273223877, 0.75065016746521, 0.5200009346008301, 0.43710392713546753, -0.4537316560745239, -0.7653972506523132, 0.6599642634391785, 1.5540835857391357, 0.3865174651145935, -0.5683915615081787, 0.04821500554680824, 0.060658250004053116, -0.7895758748054504, -0.5974701642990112, 0.20541460812091827, 0.6140111088752747, 0.8692243099212646, 1.9348984956741333, 0.6851744651794434, 0.46198463439941406, 0.2042127102613449, 1.3427128791809082, -0.22615604102611542, -0.8546890616416931, -2.881631851196289, 0.17674750089645386, -0.5186342000961304, 0.2937682271003723, -0.370064914226532, -0.5762705206871033, 0.5998658537864685, -1.0411142110824585, 1.1835209131240845, 0.02902795374393463, -0.2172033190727234, -0.16263674199581146, -0.832223117351532, -0.31574198603630066, 0.2344856560230255, -1.7375911474227905, -0.5840421319007874, 2.2038846015930176, 1.7182016372680664, -0.376433789730072, -0.48221778869628906, 0.9478113651275635, 1.6832714080810547, 1.644512414932251, 1.3930224180221558, -0.1869887411594391, -1.48862886428833, -1.2136483192443848, -0.7681359052658081, 0.7865219116210938, -1.001965045928955, 0.5854288935661316, 1.7013509273529053, -0.15683318674564362, -0.34367838501930237, -0.09626767784357071, -1.3904796838760376, 0.6668334603309631, 0.1160861998796463, -0.8422808051109314, -1.6410566568374634, -0.16756248474121094, -0.4253515601158142, -0.483866423368454, -1.1374895572662354, 1.0451167821884155, 1.2390767335891724, -0.6148767471313477, 0.31319963932037354, 0.865666389465332, -0.06837902963161469, -0.6138891577720642, -0.6274146437644958, 1.0336802005767822, 1.4197821617126465, -1.370431661605835, -0.5657542943954468, 0.39677196741104126, 0.4486483335494995, -0.9308874011039734, 0.03128644824028015, -0.4582372307777405, -1.698325276374817, 2.0179028511047363, -0.5664661526679993, -0.3629296123981476, 0.5302441120147705, 0.3861360251903534, 0.05773821473121643, 1.1415778398513794, -0.5774630308151245, 1.2024500370025635, -1.6923792362213135, -0.3613942861557007, -0.6435724496841431, -0.13277223706245422, -1.6852753162384033, 0.5774363279342651, 1.681119680404663, 1.1538289785385132, 1.8469383716583252, 0.596077024936676, -0.4686894118785858, 1.0342659950256348, 1.3122037649154663, -1.9182095527648926, 1.221237301826477, -0.2820209860801697, -2.096421241760254, 0.07616786658763885, -0.21806304156780243, -0.3148632347583771, -0.02494603395462036, -1.462764024734497, -0.08848516643047333, -2.396106243133545, -0.2582005560398102, 1.357221245765686, 1.4521968364715576, 1.809109091758728, -1.2759000062942505, 0.5704934000968933, 1.0320491790771484, -0.6133802533149719, -1.931154727935791, 1.0165661573410034, 1.1600911617279053, -0.6771959662437439, 0.45605888962745667, 0.11791215091943741, -0.3537452518939972, 2.3640129566192627, 0.035666413605213165, 1.303700566291809, -0.8537935018539429, -0.9005178809165955, 0.6931065320968628, 0.6217871308326721, -1.2937463521957397, -0.04024500772356987, -0.44815680384635925, 0.29968464374542236, 0.021606003865599632, -1.9846456050872803, 0.73378986120224, 0.6728439927101135, 2.4111814498901367, -0.18512952327728271, -0.4081946611404419, -0.5292571783065796, -1.462913155555725, -0.07709209620952606, -2.3450469970703125, -0.10831334441900253, 0.11336356401443481, -0.610547661781311, 1.6153054237365723, -0.5844125747680664, 2.304579496383667, -0.8165586590766907, -0.32903340458869934, -0.027367230504751205, 1.1628141403198242, -0.022962676361203194, -1.6887376308441162, 1.0654478073120117, -0.8810209631919861, 0.9206545352935791, 0.5966126322746277, 0.6022366285324097, -0.06844637542963028, 0.6635385751724243, -0.5181010365486145, -0.41860339045524597, -0.5566074252128601, 1.7458699941635132, -1.0884617567062378, -1.1233131885528564, 0.2980518937110901, 0.4640110433101654, -0.06509450823068619, 0.8690668940544128, 0.36593353748321533, -0.3858432471752167, 2.047588586807251, 0.17432060837745667, 0.6554626226425171, -0.9304220676422119, -0.7548520565032959, 0.5458073616027832, 0.5046381950378418, 0.5799627304077148, 0.08818651735782623, -0.10450837761163712, -0.39531415700912476, 0.6826512813568115, -1.8659769296646118, -0.5822076201438904, -0.9337923526763916, -0.7296261191368103, -1.1613643169403076, -1.4099044799804688, 0.10883956402540207, 0.2724742591381073, -0.040104374289512634, -1.3414725065231323, -0.267351895570755, -1.935245156288147, 0.12364193052053452, -0.19458812475204468, 0.2864178419113159, 1.0833046436309814, 1.0983177423477173, 1.2851316928863525, -0.03985535725951195, -0.3417658805847168, 1.7470197677612305, 0.788676917552948, -0.36763113737106323, 0.4310952425003052, 0.2259715497493744, 0.6173793077468872, 0.3498823344707489, -1.4092904329299927, -0.9834162592887878, -1.2576836347579956, 0.31753063201904297, 0.09306307882070541, -0.7498536109924316, 0.576396644115448, 0.851262092590332, 0.5438981056213379, -1.0135905742645264, -0.5732459425926208, -0.395551472902298, -0.6531376242637634, 0.5171675682067871, -1.0495949983596802, -1.19703209400177, -1.4704501628875732, -0.6145152449607849, -0.5694929361343384, 0.5959787964820862, 0.2708699405193329, -0.755478024482727, 0.7419189810752869, 0.3650433421134949, 0.14359696209430695, -0.4728408753871918, 0.15646348893642426, -0.8419444561004639, -0.09885945916175842, 0.04555891826748848, 1.6123278141021729, -0.2056771069765091, -0.2148604691028595, 0.3849632740020752, -2.4618728160858154, 0.11935427784919739, 0.27511313557624817, 0.8227148652076721, -1.9276906251907349, -0.593780517578125, 0.49439334869384766, -0.22900240123271942, -1.552585244178772, -1.3879389762878418, 0.4635169208049774, -1.7548409700393677, 3.6037003993988037, 0.7602612376213074, 1.8754260540008545, -1.9988967180252075, 0.03405589610338211, 0.9358490705490112, 0.8391902446746826, -0.7394324541091919, -0.4180341958999634, -0.9667438864707947, -0.46959298849105835, 1.115237832069397, -1.0931181907653809, -0.8557299375534058, -0.7092531323432922, -0.6637997031211853, 0.8033101558685303, -1.8710012435913086, 0.7881931066513062, -0.08157321065664291, -0.30449560284614563, 1.8525969982147217, -0.5640661120414734, -1.504270076751709, -1.5180250406265259, 0.07967819273471832, -0.10666868835687637, 0.9892688393592834, 0.19662703573703766, -0.14359211921691895, 0.6598082184791565, -0.38564494252204895, -2.2917237281799316, -1.4822064638137817, -1.445132851600647, 0.22286871075630188, 1.1897016763687134, -0.9246755242347717, 0.5953744053840637, -0.12414742261171341, 1.008225679397583, -1.5372645854949951, -1.1746493577957153, 0.41876932978630066, 1.3937569856643677, -0.7567248344421387, 0.7536948919296265, -0.29042309522628784, -0.15139931440353394, -0.04766473174095154, 0.07998299598693848, 1.4674183130264282, 1.873655915260315, -0.8005788922309875, 0.2017628252506256, 0.9517813324928284, 2.728874683380127, 0.8666114807128906, 0.9916590452194214, 2.2831380367279053, 1.492650032043457, 0.37218642234802246, 0.4902510643005371, -0.24727468192577362, -1.070190668106079, 1.0598580837249756, -1.6273493766784668, -2.6847174167633057, 0.46564507484436035, -1.151102900505066, -0.6341063380241394, 1.470788598060608, 0.4941502511501312, -0.2200394868850708, 2.03586745262146, -0.32843127846717834, 1.1148396730422974, 1.009296178817749, -0.3776487410068512, -1.0365880727767944, 0.3420228958129883, -0.47172537446022034, -0.7819844484329224, 0.8276936411857605, -0.5512349605560303, 0.3243197500705719, 0.8521578907966614, -0.4089778661727905, -1.0086594820022583, -1.5356119871139526, -0.9704799056053162, -0.9094374775886536, 0.0633062869310379, -0.9270328879356384, 0.1590455025434494, -1.1794875860214233, -0.672413170337677, -0.09654057770967484, -0.4526121914386749, -0.6511983871459961, -0.6443228721618652, -0.037334103137254715, 0.41679656505584717, -0.4723462760448456, 2.557774543762207, -0.3139035999774933, 0.1561768352985382, -0.784238338470459, -0.3988259732723236, -0.4569963812828064, -0.43375441431999207, 1.2937803268432617, 0.4793410301208496, -0.9577934145927429, 0.3796745538711548, -0.28365057706832886, 0.9692870378494263, 0.011731253005564213, 0.4945101737976074, 1.5402002334594727, 0.04225265234708786, -0.1274537891149521, -0.6895053386688232, 0.8605215549468994, 1.0384052991867065, -2.02241849899292, -0.6923565864562988, 1.0699503421783447, 0.44921669363975525, -0.20605024695396423, 0.673732340335846, 0.5853238701820374, -0.3271239101886749, -0.0036891105119138956, 0.389265239238739, -0.9336476922035217, -0.6205609440803528, 0.02835916168987751, 0.21822287142276764, 0.10202739387750626, -2.1589746475219727, 0.6618968844413757, -0.810841977596283, -0.6855458617210388, 1.6063833236694336, -1.5946846008300781, 0.027200620621442795, -0.4016377031803131, 1.4398804903030396, -0.3579910397529602, -0.17028628289699554, -0.3219677209854126, 0.1686716228723526, 0.20911246538162231, -0.052286114543676376, -0.5165697336196899, -0.0987928956747055, -0.6706159114837646, -0.526496946811676, 0.5717626810073853, 0.7289348244667053, -0.24390657246112823, -0.81502765417099, -2.0287911891937256, -1.8462690114974976, -1.2619413137435913, -0.7870051264762878, 0.24185487627983093, -0.10898548364639282, 0.08377489447593689, 0.3071330189704895, 0.5062556266784668, -0.5624843239784241, -1.177614688873291, 0.9171357154846191, -0.20433133840560913, 0.07854964584112167, 0.1445903480052948, 1.9564276933670044, 0.36279597878456116, -0.59367436170578, -1.0034301280975342, -0.6650295257568359, 0.32387858629226685, -2.0158681869506836, -1.6143898963928223, 0.12907342612743378, 2.240673065185547, -0.6653743982315063, -1.2667851448059082, -0.33188381791114807, 0.1670733243227005, 0.8859686851501465, -0.7000932097434998, -0.2145444005727768, -0.23622608184814453, 0.979824423789978, -0.5298280119895935, 0.979590654373169, 1.4443252086639404, -0.11090895533561707, -0.8857079148292542, 0.40813612937927246, 1.1996217966079712, 0.16586677730083466, 2.016571044921875, 0.8274314403533936, -0.8297067880630493, -0.4368928372859955, -1.3466359376907349, -0.16331663727760315, 0.4906507432460785, -0.0070785218849778175, 0.45193687081336975, 0.12103495001792908, -0.5632739663124084, -0.9929040670394897, -0.5110270380973816, 1.3202509880065918, 0.42489057779312134, -0.2835918664932251, -1.096584677696228, -0.7254788875579834, 0.8225085139274597, 0.9073862433433533, 0.43664565682411194, 1.5356600284576416, 1.084212064743042, -0.3315739631652832, -1.7563918828964233, -0.5129110813140869, -0.9793463945388794, 1.013043761253357, -1.4061824083328247, 0.08270849287509918, -1.8693023920059204, 0.40578678250312805, -0.0695195347070694, -1.0542248487472534, -0.10902808606624603, 0.3092564046382904, 0.1451951563358307, 0.7841023802757263, -0.824125349521637, -0.2195267528295517, 0.17303937673568726, 0.2832285463809967, 0.5382106304168701, -0.24986986815929413, 0.22752052545547485, -1.5534660816192627, -0.4074953496456146, -0.6712233424186707, -0.2402913123369217, -0.7068111896514893, 0.17028400301933289, -1.7908889055252075, 0.6035832166671753, 0.19065359234809875, 1.078956961631775, 0.4264559745788574, -0.26446181535720825, -1.0512222051620483, 2.110098123550415, -0.18826404213905334, 0.9343748092651367, 0.792089581489563, 1.0307393074035645, 0.14317122101783752, 0.5302091836929321, -0.19719509780406952, -0.767013669013977, -0.3383820354938507, -1.243024230003357, -0.08496686071157455, 2.0371198654174805, 0.4175552427768707, 0.8982236385345459, 2.1575160026550293, -0.613556444644928, 1.2825802564620972, 0.46854734420776367, 0.5823017358779907, 1.1773582696914673, -0.30412521958351135, 0.4011257290840149, 0.9796785712242126, 0.09915746748447418, 0.6439676284790039, 1.076198697090149, 0.6644765734672546, -0.3348589241504669, 0.07765641808509827, -0.5444947481155396, -2.0351932048797607, -0.8032561540603638, 0.758270263671875, 0.34403082728385925, 1.2388709783554077, 0.34831783175468445, 0.180084228515625, 0.4283154010772705, -0.5732725858688354, -1.9337319135665894, -1.187312364578247, -1.3053773641586304, -0.772051990032196, -0.8199266791343689, 0.8626893758773804, -0.2043485939502716, -2.0936450958251953, 0.6760516166687012, 1.966225266456604, 0.9121196269989014, -1.4136682748794556, -0.41391468048095703, -0.44179248809814453, -0.5607996582984924, 0.3938977122306824, -0.25193601846694946, 0.886266827583313, 0.2743511497974396, 1.0584115982055664, -1.231227159500122, 1.5667710304260254, 0.14836713671684265, -0.1636931449174881, -1.00006103515625, 0.11266662925481796, 0.22407610714435577, 0.4046511650085449, -1.9700900316238403, -0.46097034215927124, 0.07100225985050201, 1.0423489809036255, -1.0831897258758545, 2.008194923400879, -0.09728005528450012, -0.07333477586507797, 0.0632835254073143, -0.32932522892951965, 0.6660673022270203, 0.9180894494056702, 0.5628504157066345, 1.8178088665008545, 1.5184437036514282, 0.8784754276275635, 1.6640629768371582, -2.43314528465271, -0.0659852847456932, 1.052522897720337, -0.14776477217674255, 1.8865596055984497, -0.31099754571914673, -1.180185079574585, 1.1696431636810303, 1.9804625511169434, -0.42959481477737427, -0.46978235244750977, 0.38506656885147095, -1.3901832103729248, 0.6967148780822754, 0.19193100929260254, -0.19012337923049927, -0.7177989482879639, -0.5477904081344604, 1.19674551486969, -0.4321860373020172, -0.07811769843101501, -0.14689795672893524, -0.908351480960846, -0.5440270304679871, 0.09660749137401581, -0.1794021725654602, 0.20865005254745483, 0.008885673247277737, -1.1300915479660034, 2.096130847930908, 2.1406702995300293, -1.1071676015853882, -0.5203605890274048, -1.4365603923797607, 1.4712449312210083, -0.36239171028137207, -0.01842224784195423, -0.3235859274864197, 1.7958768606185913, -1.1852988004684448, 1.2954298257827759, -0.33383795619010925, -0.08145032078027725, -1.1762968301773071, -0.6796716451644897, -0.1521705985069275, -0.34094375371932983, 0.861853837966919, 0.5318018198013306, 1.1488063335418701, 2.2715351581573486, -0.8717528581619263, -1.794500470161438, 0.16346475481987, 0.26062363386154175, -0.9996628761291504, -0.2998031973838806, 1.7716602087020874, -1.8781863451004028, -1.574149250984192, 1.052478313446045, -0.2378843128681183, -0.32927167415618896, 0.546649694442749, 0.5926879048347473, 0.6207659244537354, 0.1207251250743866, -2.5317885875701904, 0.3006880581378937, 0.8261130452156067, -0.819168746471405, -0.2058282196521759, -0.925384521484375, 0.05742063373327255, 0.9370914697647095, -2.4721481800079346, 1.734013557434082, 0.5064142346382141, 1.1441413164138794, 0.562880277633667, 0.1263333261013031, 1.3005900382995605, 0.15946999192237854, -0.6387823224067688, -0.9514536261558533, -0.7626498341560364, 2.639011859893799, -0.36972662806510925, 1.0602484941482544, 0.19383181631565094, -0.5950835943222046, -1.7187496423721313, -1.0616759061813354, -0.47876423597335815, 0.6958358883857727, -0.7014448046684265, 0.19584551453590393, -0.5753350257873535, 1.2496906518936157, -0.5049862265586853, 0.9669234752655029, -0.1801103800535202, -0.1380837857723236, -0.48570650815963745, -1.4638277292251587, 1.207064151763916, -2.0261712074279785, -0.15895147621631622, 0.026464950293302536, -0.6131293773651123, -0.3379843831062317, -0.4435966908931732, -0.6684799194335938, -0.4906432628631592, -0.39381882548332214, 1.8848820924758911, 0.3228016793727875, 0.9718836545944214, 0.45377907156944275, -0.27938374876976013, 1.5167574882507324, -1.3873482942581177, 1.0280755758285522, -1.2253509759902954, 0.2682541310787201, 0.3533630967140198, -1.0708122253417969, -0.7894548177719116, 1.2681694030761719, -1.0562013387680054, 0.4143453538417816, -0.4078798294067383, 0.1775241494178772, -0.1325329691171646, 0.07536723464727402, -0.9626878499984741, 0.46965673565864563, 0.1875569373369217, 0.07666086405515671, -1.7496711015701294, -0.7758317589759827, -0.3299129903316498, -2.6517412662506104, 0.25469517707824707, -0.36208638548851013, 0.20122160017490387, -0.1437440663576126, 0.05277872458100319, 1.5436286926269531, -0.149533212184906, -0.2999865710735321, 0.4223316013813019, 3.2008495330810547, -0.04504851996898651, 0.19941657781600952, -0.36244550347328186, 0.779935896396637, -0.13059088587760925, -0.22047832608222961, -1.1050052642822266, 0.5385841131210327, -0.5141186118125916, 1.6683908700942993, 1.4972686767578125, 0.996990978717804, -0.13166968524456024, 2.1165597438812256, 0.24886298179626465, -0.7661645412445068, 0.7716413736343384, -0.044534701853990555, -0.194540873169899, -1.2009778022766113, 0.2295239418745041, -1.2709366083145142, 0.4555057883262634, 1.905716061592102, -0.11836409568786621, -0.12812232971191406, 0.8695576190948486, 0.5970327258110046, 0.6167372465133667, -0.34170669317245483, 1.5544027090072632, -0.6633684635162354, 1.629016399383545, -1.1602704524993896, 0.44837477803230286, 0.34217119216918945, 0.3540113568305969, 0.9403516054153442, -0.2747516334056854, -1.792982816696167, 0.7250703573226929, -0.8157718181610107, 0.5081781148910522, 1.2220263481140137, -2.07038950920105, -0.5416350960731506, 0.014067822135984898, -0.7504247426986694, 0.03599315509200096, -1.306829571723938, -1.7504842281341553, 0.020693736150860786, -0.6962251663208008, -0.06707555055618286, 0.47278302907943726, 1.2304152250289917, 1.823029637336731, -1.078928828239441, 0.5451156497001648, -1.3184734582901, -0.006593095604330301, -0.7342960834503174, 0.05585349723696709, -1.0300366878509521, 0.2925189733505249, -1.7242445945739746, 0.42627185583114624, -0.19109147787094116, 1.1908949613571167, -1.9890799522399902, -1.9474979639053345, -1.453384280204773, 0.8214506506919861, 0.6513237953186035, -0.2400127351284027, -0.34432145953178406, -1.3196609020233154, 0.21231190860271454, -4.214058876037598, -0.7371386885643005, 0.46338599920272827, 0.5760376453399658, 1.5178440809249878, -1.4882333278656006, 2.5943877696990967, 0.6931799650192261, 1.236557126045227, 0.8808774948120117, -0.14837919175624847, 0.4557359516620636, -0.014941510744392872, 1.9578670263290405, 0.43945157527923584, -1.1036179065704346, -0.5619589686393738, -1.7554290294647217, -0.9292685985565186, 0.4613848328590393, 1.8038740158081055, 2.232778310775757, 0.08256103098392487, -0.767268717288971, 1.0078041553497314, 0.41209810972213745, 0.866102397441864, -0.01182229071855545, -1.8968114852905273, 0.6870132088661194, -0.5501249432563782, 0.1512708067893982, -0.08445774018764496, 1.2124372720718384, 1.3686480522155762, 1.2306650876998901, 0.5595741868019104, 2.2809722423553467, 0.87413090467453, -1.385892629623413, 0.15619608759880066, 0.09154767543077469, 0.2989528775215149, -0.6604908108711243, 0.8585828542709351, 0.19483351707458496, -0.9250713586807251, -0.8720125555992126, -0.49134624004364014, -0.8283329010009766, 0.038600362837314606, -1.092577576637268, -1.342879295349121, 1.1248193979263306, -2.5512588024139404, -1.2759379148483276, 1.989389419555664, -2.085657835006714, -1.1231330633163452, 0.287722110748291, 1.4843733310699463, 1.7443692684173584, -0.5301228761672974, -2.4062938690185547, -0.7706833481788635, -1.3253146409988403, 0.015876643359661102, -0.08990678191184998, -1.9035155773162842, 0.6354537606239319, 1.9209098815917969, -0.24636545777320862, -1.5459061861038208, -1.0311635732650757, -0.06210694462060928, 0.9912192821502686, -0.6319124102592468, -0.39345744252204895, 1.6728309392929077, -0.530446469783783, -0.7038407921791077, 0.9582999348640442, -1.7264838218688965, -0.6924387216567993, -0.1472085863351822, 1.2737399339675903, 1.3555251359939575, -0.7924667000770569, -0.39813482761383057, -0.5436312556266785, -0.1354544758796692, -1.5112369060516357, 0.08520113676786423, -0.44521352648735046, 0.6848576664924622, 0.5297509431838989, -0.4122108817100525, -0.38588836789131165, -2.7444868087768555, 0.34952375292778015, -0.7197136282920837, -0.9683828949928284, 0.5424102544784546, 0.3203967213630676, 0.5270146727561951, -2.097343921661377, 0.27658241987228394, -1.8558388948440552, -0.6591501235961914, 0.11452150344848633, -0.03293468430638313, -0.8550084233283997, 0.6218117475509644, -0.03001697175204754, 0.6820632815361023, 0.5621654987335205, -1.557602047920227, -1.13909113407135, -0.1860252171754837, -0.6615400314331055, -0.40529865026474, -0.053943995386362076, 0.35780829191207886, -0.027426837012171745, 0.515116274356842, 2.1911158561706543, -0.21866962313652039, -1.6152350902557373, -0.5914691686630249, 0.3823213577270508, -0.28836435079574585, 1.8027842044830322, -0.6802977323532104, 1.0589505434036255, 0.15123093128204346, 1.311684250831604, 0.32969117164611816, -0.02148052677512169, -2.0393459796905518, 1.3646825551986694, 0.9191474318504333, -0.23287834227085114, 0.37309205532073975, 0.12805211544036865, -1.2738447189331055, 0.07787521183490753, -1.0892678499221802, 0.854439914226532, -2.2604470252990723, 0.26964133977890015, -1.1738665103912354, 0.024438131600618362, 0.03603924438357353, 1.0977085828781128, 0.20977537333965302, 0.06381696462631226, -1.2598286867141724, -0.4392535388469696, -1.2547234296798706, 0.8454225063323975, 0.9144868850708008, 2.428837299346924, 1.4470654726028442, 0.09857476502656937, 1.102578043937683, 0.10087361931800842, -0.2490694224834442, -2.6731197834014893, -0.24354247748851776, -0.5973694920539856, -1.0971494913101196, -0.8191664814949036, 0.49850767850875854, 1.485325813293457, 0.9580692648887634, -0.6899161338806152, 0.4492016136646271, -0.5304681062698364, 2.3532633781433105, -0.7667486667633057, -1.2964600324630737, -1.2950552701950073, -1.1776028871536255, 0.21010828018188477, -0.03827723488211632, 1.8983889818191528, -0.0752977654337883, 1.0489661693572998, -0.013780406676232815, -0.2805160880088806, -1.2559804916381836, 1.44230318069458, 1.5047940015792847, -1.8647171258926392, -1.1014384031295776, -0.8131411671638489, -0.7948930859565735, 0.9988957047462463, -0.32536348700523376, 0.27551937103271484, 0.4958428144454956, 2.5077626705169678, 0.33733177185058594, -1.3000966310501099, -1.0918166637420654, 1.4036920070648193, -0.20066601037979126, 0.503700315952301, 0.8496111631393433, 1.5711748600006104, 0.4486379623413086, 0.30080851912498474, -1.5229676961898804, -0.5459258556365967, -0.3975927233695984, -2.066986083984375, 0.6141693592071533, -0.979059100151062, 0.1478651463985443, 2.050499200820923, 0.026811448857188225, 0.27621060609817505, 1.9373469352722168, 0.17548498511314392, -0.7517939805984497, 0.15030698478221893, 1.1656315326690674, -0.4535423219203949, 0.4832211136817932, 1.237082839012146, -1.4038307666778564, -0.09296828508377075, -0.018449043855071068, 2.334585428237915, 0.604260265827179, -0.5211274027824402, 0.4058319330215454, -1.0117361545562744, 0.5012102723121643, 0.08697264641523361, -0.885044515132904, 1.0684059858322144, -0.18903696537017822, 1.7243424654006958, -0.5347381830215454, -1.0613548755645752, -0.2620874047279358, -1.5449399948120117, -0.3506535291671753, 0.2781544327735901, 1.7285124063491821, -0.4521619975566864, 0.342924565076828, -2.7861740589141846, -0.5330536365509033, 0.5295718312263489, 0.770032525062561, 0.18727827072143555, 0.26546812057495117, 1.048309326171875, -0.2012050300836563, 0.801746666431427, -0.43361589312553406, -0.17396551370620728, -0.42257556319236755, -1.6787925958633423, -0.9164077043533325, 0.6936461329460144, -0.7523991465568542, 0.009088258258998394, 0.6620193719863892, 0.26123109459877014, 0.2715153098106384, -1.1768832206726074, -0.2756345272064209, 0.5967627763748169, -0.600642740726471, 0.244384303689003, 0.012933398596942425, -0.26335230469703674, 0.28636375069618225, 1.3063725233078003, 0.8540448546409607, 1.1892658472061157, -1.742240071296692, -0.29416877031326294, -0.9409452080726624, -0.011672292836010456, -0.39554691314697266, -0.3355584740638733, 0.19293764233589172, 1.5161422491073608, 0.4291210472583771, 2.3099095821380615, 0.2953934371471405, -0.2166445255279541, 0.6626864075660706, -0.017672013491392136, 0.8256428241729736, 0.44684335589408875, 0.089580237865448, 0.505966067314148, -0.054462842643260956, -1.1008375883102417, -0.980768084526062, -1.537613868713379, 0.6769573092460632, 0.41856664419174194, 0.10608331114053726, -0.1445741504430771, -0.17089465260505676, -0.7371159195899963, 0.4479281008243561, 1.1369730234146118, -0.27329665422439575, 0.6906489133834839, -0.3237365782260895, 2.068645715713501, -0.824539065361023, -2.255364179611206, 0.6040100455284119, 0.3588998019695282, -0.8609398007392883, -0.21008840203285217, -0.42705464363098145, 0.22361595928668976, 0.5375651121139526, 0.25814878940582275, -0.6504709720611572, -0.5469346642494202, -0.06690862774848938, -0.792519748210907, -1.3209595680236816, 0.5992034077644348, 1.0952892303466797, 0.0452972836792469, -0.02480531856417656, -0.28402915596961975, -0.10006222873926163, -1.354848027229309, 1.6422735452651978, -1.9470762014389038, 0.07181286811828613, -0.9559578895568848, 1.9409259557724, 0.9264704585075378, 1.8286926746368408, -0.18574681878089905, -0.27822840213775635, 0.14960461854934692, 0.5969794988632202, -0.5007948279380798, 0.012614013627171516, 1.4350436925888062, 0.13580287992954254, 0.922034502029419, 0.9923340082168579, -1.3286117315292358, -0.05781187862157822, -2.4285008907318115, -0.6510170698165894, 0.29236143827438354, -0.09730462729930878, -0.434535413980484, 3.0621278285980225, -0.25767529010772705, -1.5693374872207642, -0.8000612854957581, 0.41114571690559387, -0.8269656300544739, 2.188631057739258, 0.4502303898334503, 1.4731638431549072, -0.5304417014122009, -0.6949130296707153, 0.09869752824306488, 0.4143182337284088, 1.2421468496322632, -1.6503819227218628, -1.0090553760528564, -1.1113760471343994, -0.22336392104625702, -1.6850967407226562, 0.06578603386878967, 2.020994186401367, -0.6552827954292297, -0.5593260526657104, -0.27739691734313965, -0.25333160161972046, 0.9405157566070557, 0.6560801267623901, -0.7077839970588684, -1.2919435501098633, -0.2536775469779968, -0.567764163017273, -0.8112522959709167, -0.10775184631347656, 1.752030611038208, 2.446458339691162, 0.9496327638626099, 0.14935269951820374, 0.06457681208848953, 1.6069644689559937, -0.17801755666732788, -2.1931867599487305, -0.3234603703022003, -2.867163896560669, -0.5122120976448059, 0.3386404812335968, -0.9469164609909058, 0.27119389176368713, -1.0488733053207397, 0.8035596013069153, -0.5919633507728577, 0.2864087224006653, 1.1039397716522217, 0.5168696641921997, -1.3085592985153198, 0.252885639667511, 0.6647542715072632, 0.40976935625076294, -0.8973983526229858, -0.34468427300453186, -0.2221095710992813, 1.0953787565231323, 2.061418056488037, 0.02022227831184864, 0.3726733922958374, 0.21344642341136932, 0.23081088066101074, 0.16986672580242157, 0.5889222025871277, -0.8240694403648376, -1.375948429107666, -1.3077709674835205, -0.3149057924747467, 0.4901428818702698, -0.043155550956726074, 0.04730452597141266, 0.13481135666370392, -1.3094910383224487, 0.6050373315811157, 0.35594266653060913, -0.724273145198822, -0.21921183168888092, -2.0491104125976562, -0.12558427453041077, -0.06697700917720795, -0.43570560216903687, -0.5807288289070129, 0.3317822515964508, 1.7984001636505127, 1.2070084810256958, -1.1618393659591675, 0.8040167689323425, -1.991277813911438, 0.3356687128543854, -0.4235256314277649, -1.628829002380371, 1.6282610893249512, -2.3375775814056396, -0.5596615672111511, -1.3573024272918701, -1.006638526916504, -1.4454771280288696, -0.6117588877677917, 0.3448071777820587, 1.2222421169281006, -2.115198850631714, 1.22258460521698, 0.8472903966903687, 1.2362608909606934, 1.6670626401901245, 0.4173818528652191, -0.9929125905036926, -0.6835086345672607, 0.45495858788490295, -0.6700553297996521, 0.9381978511810303, 0.3235994279384613, 0.8237623572349548, 0.3360273540019989, -0.5153819918632507, -0.24295304715633392, -0.47628054022789, -0.9609717130661011, 0.6153106689453125, 0.2573223412036896, -0.6984397172927856, 0.6038100123405457, -0.06805229187011719, 1.6320512294769287, 0.5667524337768555, -0.8168374300003052, 2.2539992332458496, -0.6868306994438171, -3.1482949256896973, -0.9143469333648682, 0.5934759974479675, -0.6935515403747559, -0.1268092542886734, -2.9378206729888916, -0.8114407062530518, 0.4968632459640503, -1.7534635066986084, 0.3333776891231537, 0.7836824655532837, 0.1257941573858261, 0.2545500695705414, -0.7206286787986755, -2.0292651653289795, -0.27335038781166077, 2.0691049098968506, -0.5959241986274719, -0.7700862288475037, -0.25711503624916077, -0.14338813722133636, -2.299934148788452, 1.0198609828948975, 0.8043452501296997, -0.47093889117240906, 0.07800102978944778, 1.2443015575408936, -0.3786795735359192, 0.09082306921482086, -0.8584669232368469, 0.6772124767303467, -0.6642974615097046, -0.49381694197654724, -0.8854454755783081, -1.1757975816726685, 1.6775786876678467, 1.043015718460083, -0.18783017992973328, 0.47936004400253296, -0.493908554315567, -0.4754285216331482, -0.3643433451652527, -0.04630549997091293, -0.05705997720360756, -0.045308876782655716, -1.5207728147506714, 1.0902434587478638, 0.13731060922145844, 0.9183802604675293, 1.216157078742981, 0.01912689581513405, 0.25863105058670044, -1.1152139902114868, 0.2026132494211197, -1.4935230016708374, 0.5970312356948853, 0.33786848187446594, 0.5844952464103699, -0.034584611654281616, -1.3333470821380615, -0.41035911440849304, 0.546567976474762, 1.3192131519317627, -0.33009207248687744, 1.1021792888641357, 1.1002596616744995, 1.7782528400421143, 1.011291742324829, 0.7242541909217834, -0.8176988363265991, 1.3478018045425415, -0.5343787670135498, -1.1661434173583984, -0.49701884388923645, 1.149244785308838, -0.8696820139884949, -1.1775939464569092, -0.33782514929771423, -0.20235960185527802, -0.22186721861362457, 0.3440580666065216, -1.3065357208251953, -1.3763759136199951, 0.7605965733528137, 0.4429658353328705, -0.8362745642662048, -1.8919252157211304, -1.2117745876312256, -0.6902692317962646, 0.6376187801361084, 0.2221643626689911, -0.1557057946920395, -0.10575319081544876, 0.5079097747802734, -0.01748429425060749, 0.13531823456287384, 1.8403469324111938, 1.6058176755905151, -0.0019716050010174513, 0.651999294757843, -0.6069837808609009, -0.14307476580142975, 1.0646153688430786, -1.7934091091156006, 0.3786521852016449, -0.6282939910888672, -0.05231894180178642, 0.4071735143661499, -0.7722715139389038, 0.9262596368789673, -0.4988540709018707, -0.09395918250083923, 1.4029408693313599, 0.21209359169006348, 0.2717020809650421, 1.9380346536636353, -0.6028880476951599, -0.321493536233902, -0.0663968026638031, -1.963560938835144, -1.0420576333999634, 0.7967925071716309, 1.3953498601913452, 0.1424383521080017, -0.024844719097018242, 2.433184862136841, -0.9378220438957214, -0.8837152123451233, -2.149414300918579, 0.09797894209623337, -0.3841262459754944, 0.16876128315925598, -1.316556692123413, 1.6077840328216553, 0.21067693829536438, -1.115006685256958, 0.7851304411888123, -0.9131887555122375, -0.04737946391105652, 0.5560754537582397, 0.29008617997169495, -0.2618975043296814, 0.18053539097309113, -0.06330946832895279, 0.2189394235610962, -0.8981901407241821, 0.5165191292762756, 0.13037735223770142, 0.8452532887458801, -0.3386339843273163, 0.7608904838562012, -0.14678488671779633, -1.7012560367584229, -1.9346588850021362, -0.5429157018661499, 0.6361735463142395, 0.8889638185501099, 0.1250361055135727, -0.14481280744075775, -0.9648606777191162, -0.31615668535232544, -0.7704096436500549, -0.1961139589548111, 0.36203059554100037, 1.1099716424942017, 1.5290390253067017, -0.7916374802589417, 0.3896469473838806, 1.5706599950790405, 1.1952269077301025, -0.5614621639251709, -0.30311331152915955, 1.5864564180374146, -1.3966621160507202, 0.7424609065055847, 1.0287187099456787, 2.1037843227386475, -0.12390408664941788, -1.0208227634429932, -1.0848126411437988, 1.4075231552124023, -1.7051846981048584, 0.8199348449707031, 0.0033286744728684425, -0.059270229190588, 0.43423137068748474, 1.6969372034072876, -0.9426281452178955, -0.11703471839427948, -0.3941991925239563, -0.4766744077205658, 0.6467182636260986, 1.3663197755813599, -1.5426157712936401, 1.08858060836792, -1.4260157346725464, 0.5563920736312866, 0.7640573978424072, -1.4158332347869873, 1.2238309383392334, 0.7894692420959473, 1.1011431217193604, -1.1906590461730957, -1.75613534450531, -1.2359275817871094, -1.2687997817993164, -1.0313206911087036, -0.9344723224639893, -0.6213753819465637, -0.7922335267066956, -0.11689695715904236, -0.8781324028968811, -0.09471505135297775, -1.5489513874053955, 0.9010398387908936, -0.28948432207107544, 0.5512657761573792, -0.580508291721344, 1.3628355264663696, 0.8735319375991821, -1.448108434677124, 1.264685034751892, 0.3944704830646515, -0.5997252464294434, -1.0560542345046997, 0.3808070123195648, -0.9939911365509033, 0.8378689885139465, -0.3062628507614136, -0.7386215925216675, 0.23696525394916534, 0.3113807737827301, 0.9683762192726135, 1.2150734663009644, -0.6501636505126953, 1.4588265419006348, -0.27049288153648376, 1.2117655277252197, 1.5060737133026123, 0.17134785652160645, -0.05760952830314636, -0.6091586947441101, -1.219500184059143, 0.6112545132637024, 0.7455009818077087, -0.27891823649406433, -0.11166177690029144, -1.0427788496017456, 0.0967441201210022, -0.3035705089569092, -1.198354721069336, -0.8314942717552185, 1.6679149866104126, 2.1779630184173584, 0.4330788850784302, -0.4392241835594177, 0.3304674029350281, -0.1416015774011612, -0.7075150012969971, -1.0345687866210938, -1.5914872884750366, -0.1220698431134224, -0.4990406334400177, -1.1882822513580322, 1.7999402284622192, -0.3900066614151001, -0.33099570870399475, -0.6082242131233215, 1.23383629322052, -0.4439513385295868, -1.641640543937683, 1.326029658317566, 0.9712633490562439, -0.8984384536743164, 0.03607737272977829, -0.3697109818458557, -0.33070826530456543, -1.6672608852386475, 0.47439149022102356, -0.5651641488075256, 0.008595244958996773, 1.4810446500778198, 0.09299300611019135, -0.40768077969551086, -1.7082005739212036, 0.1880086064338684, -1.977702021598816, 0.10418349504470825, -0.16395610570907593, 0.7194721698760986, 1.2002134323120117, 0.48836180567741394, -0.7588129639625549, -1.3499542474746704, -1.6763501167297363, -0.5685210227966309, -0.6157998442649841, 0.0653303861618042, 0.4080539345741272, 0.6543494462966919, -0.68560391664505, -1.0241464376449585, -0.9657260179519653, 1.6578571796417236, 0.5078693628311157, 1.6196643114089966, 0.2360590249300003, -0.8968397378921509, -0.5240932106971741, 0.48722562193870544, -1.5802475214004517, 1.6359143257141113, 1.1554200649261475, 0.9636852145195007, -1.218839406967163, 0.6302892565727234, -0.3403375446796417, 1.0152850151062012, 1.303823471069336, 0.017972376197576523, 0.22260157763957977, 0.6128767132759094, -1.4650731086730957, -0.28833210468292236, -0.612839937210083, 0.1484277844429016, 0.07302568107843399, 0.9759854078292847, -1.5571959018707275, 0.8443328142166138, 1.420904278755188, -0.5634482502937317, 0.21047446131706238, 2.127728223800659, -0.356731653213501, -1.895351529121399, 0.42964062094688416, -0.20350773632526398, -1.3801947832107544, 1.1229640245437622, -0.6884490847587585, 0.7803174257278442, -1.0411345958709717, -0.5120969414710999, -0.10263942182064056, 0.2291337102651596, -1.4524946212768555, 0.9429178237915039, 0.62664395570755, -0.7874175310134888, 0.9264218807220459, -2.8736138343811035, -0.20483559370040894, 0.9552323222160339, 2.199617862701416, -0.2980218529701233, -0.8074236512184143, -0.28193727135658264, 0.302225798368454, 0.2521805167198181, -0.35892149806022644, 1.0665267705917358, -0.797785758972168, 1.7178912162780762, 0.7770669460296631, -0.977776825428009, 0.01853230781853199, 0.2183229923248291, -0.8958799839019775, -0.6204580068588257, -0.6522687673568726, 0.8192477226257324, -0.1988358199596405, -0.6827109456062317, 0.5901821255683899, 1.8651366233825684, 0.10941369831562042, 2.3298299312591553, -0.32783254981040955, 1.2637521028518677, 1.8678351640701294, -0.08750413358211517, 0.4674305021762848, 0.08603233844041824, 0.3322398066520691, 1.7408111095428467, -2.7127914428710938, -1.0189284086227417, -0.05443481355905533, -0.6711390018463135, -0.010496487841010094, -1.4367685317993164, 1.7703708410263062, -0.7496644258499146, 0.9785250425338745, 0.09607356786727905, 1.1923073530197144, -0.15364107489585876, -2.2885987758636475, -1.746849775314331, -0.8955101370811462, 0.9044367671012878, -1.379258394241333, -0.07082216441631317, -0.7240394949913025, -1.0347508192062378, -0.9417286515235901, 0.7705959677696228, 0.851662814617157, -0.550685703754425, -2.2752139568328857, 0.03153935819864273, -0.20168370008468628, 0.2006201595067978, -0.00944663304835558, 0.3718368709087372, 1.8797972202301025, 0.8704605102539062, -0.8501507043838501, 0.03012668341398239, -0.5040968656539917, 0.007687120698392391, -2.2557597160339355, 2.307699203491211, -0.4009098708629608, -1.290543794631958, -0.785966157913208, 1.6999152898788452, 0.19701026380062103, 1.1675432920455933, -0.04807407781481743, -1.3450016975402832, 0.8789845705032349, 0.5855732560157776, -1.6696481704711914, -1.5610724687576294, -0.5862939357757568, -0.5812153816223145, -0.6007332801818848, -0.6498336791992188, -1.305071234703064, -0.6591305732727051, -1.1272339820861816, -0.516594409942627, -0.799679696559906, 0.0327409990131855, 0.8348785042762756, 1.4920207262039185, 0.13786755502223969, 0.35472097992897034, -0.017606589943170547, 0.23817496001720428, 0.4118777811527252, 0.8603617548942566, -0.250461608171463, 0.9920846223831177, 1.7119238376617432, 0.06744416058063507, 0.8814325928688049, -1.0795865058898926, 0.9372348189353943, -1.7551724910736084, 0.8182592391967773, -0.05528995767235756, -0.14065347611904144, -0.8737888932228088, -0.07281767576932907, -0.8053960800170898, -2.3018453121185303, 0.4889346957206726, 2.4874043464660645, 1.668765664100647, 0.5470367074012756, -0.23458483815193176, -1.0909698009490967, 1.0007168054580688, -0.08892545104026794, 1.2522573471069336, 0.3970169425010681, -0.1076875850558281, -0.878653883934021, 0.2952635884284973, -1.7242919206619263, -0.36923453211784363, -0.7453804612159729, 1.4702280759811401, -0.4591851234436035, -0.22225292026996613, 0.19550947844982147, -0.7375741004943848, -0.1847112476825714, 1.2281821966171265, 1.2563637495040894, -0.0507500059902668, 1.5422775745391846, -0.8010053038597107, 0.6396839022636414, -0.6815686821937561, 0.5006393790245056, -2.5952768325805664, 0.5534502267837524, -1.008428692817688, -0.548141598701477, -0.8026824593544006, -0.6681697368621826, -0.7098414897918701, 0.2341640442609787, 1.2124837636947632, -0.4056857228279114, -0.264079749584198, 1.2853052616119385, -0.5405197739601135, -1.2404769659042358, -1.365319013595581, 1.9078483581542969, -2.072570562362671, -0.028086669743061066, 0.6573078036308289, 0.09549503028392792, -2.061126947402954, 1.5341585874557495, -1.0827230215072632, -0.4340663254261017, -0.6975995302200317, -1.8020542860031128, 0.5763965845108032, 1.0934219360351562, 1.374741792678833, -0.8938193321228027, -1.101269245147705, -0.37482786178588867, 0.3694538176059723, 1.0880038738250732, 0.41103848814964294, 2.064786672592163, -0.08340958505868912, 1.064070463180542, 0.0013661296106874943, -0.22236734628677368, -1.7083102464675903, -1.924218773841858, 0.27839237451553345, 1.3133902549743652, 0.08723798394203186, -0.6158065795898438, 0.7030020356178284, -0.5878636837005615, -0.6315499544143677, 1.471138596534729, 0.7846769094467163, 0.3691488206386566, -0.5570494532585144, -1.1454558372497559, 0.5961948037147522, -1.1086539030075073, -0.3500075340270996, 0.9177365899085999, -0.10151200741529465, 1.2167139053344727, 0.4666546583175659, 0.15867246687412262, 0.7019476294517517, 0.3615148961544037, 0.9617247581481934, -0.16622722148895264, -0.3140794336795807, 0.28710266947746277, -1.3246207237243652, -0.09178008884191513, -0.01711552031338215, -0.05768347904086113, -0.15488861501216888, -0.8015120625495911, 1.8877514600753784, -0.17495518922805786, 0.47193413972854614, -0.6376532912254333, 0.7392318248748779, 1.51665461063385, -0.9517862200737, 0.6172797679901123, 0.6565510630607605, -1.311582088470459, -0.5543832778930664, 0.8181604743003845, 0.7561757564544678, 0.46579670906066895, 0.20421579480171204, -1.7360960245132446, -1.1062008142471313, -0.02934301272034645, 0.8154147863388062, -1.1884485483169556, 0.6285721063613892, -0.47364747524261475, 1.4196040630340576, -0.057608041912317276, 0.2532234191894531, 0.6435343623161316, -1.373842477798462, -0.1108621284365654, -0.9508399963378906, -1.3739205598831177, -1.3821353912353516, 0.8146981000900269, 0.45922717452049255, 0.8431566953659058, -1.0860021114349365, -0.23656709492206573, -1.5486310720443726, -0.9535736441612244, 0.16611887514591217, -0.0939205214381218, -1.5749006271362305, 0.04251161962747574, -1.3056963682174683, -2.3533480167388916, 1.0656983852386475, -1.1153802871704102, 1.0136181116104126, -0.9025693535804749, 0.5632177591323853, -1.352165699005127, -1.1380362510681152, 1.724923014640808, 0.7835344672203064, 1.5715750455856323, -0.9081547856330872, -0.4753324091434479, 0.18462684750556946, 0.1408023238182068, 0.19290407001972198, -0.42534586787223816, -1.350063443183899, 0.311636745929718, -0.04215645417571068, 1.6135565042495728, 0.07737411558628082, 1.5792696475982666, 0.2457454800605774, -0.7665004134178162, -0.1832367181777954, -0.1259181648492813, -0.7072534561157227, -1.9729102849960327, -0.5211690068244934, -0.8511909246444702, -0.885139524936676, 0.9336802959442139, -0.6552273035049438, 1.0360244512557983, 1.1964352130889893, -0.7964582443237305, -0.2094779759645462, 0.05981118604540825, -0.10510170459747314, -1.907292366027832, 0.334869921207428, -1.291007399559021, -0.710597038269043, 0.8062471747398376, -0.16111691296100616, -0.47143954038619995, -1.1342175006866455, 0.0032513493206351995, 1.452675223350525], "dtype": "float32"}, {"__nd__": [1.1776317358016968, 0.9271461963653564, 0.3674967885017395, 0.8716446161270142, -1.360854983329773, 2.097766876220703, -0.8397868275642395, -0.7233887910842896, -0.005808103829622269, -0.5699368119239807, -1.541334867477417, -1.3366039991378784, 0.6122663617134094, 1.0557104349136353, -0.676864504814148, 0.7662214636802673, 0.4671439528465271, 1.6318873167037964, 0.1583709418773651, 0.5899075865745544, -1.1850311756134033, 1.8710776567459106, 0.8109210133552551, 0.2099158763885498, 0.16393843293190002, 1.7521294355392456, 0.39135754108428955, 1.534854769706726, -1.1866250038146973, -0.5235992074012756, -0.5281406044960022, -0.11228812485933304, 2.456768751144409, 0.048848479986190796, -0.6611418128013611, -0.23836876451969147, -0.5430973172187805, 0.336122989654541, -0.8585789799690247, -0.4629316031932831, 0.4725230634212494, -0.9393762350082397, -0.4041508436203003, 0.3719960153102875, 0.19575025141239166, -0.14743822813034058, -0.45689186453819275, -0.03167982026934624, -1.7398549318313599, 1.2523438930511475, -1.3909187316894531, -0.6669422388076782, -0.8623569011688232, -0.7308446168899536, 0.06293012946844101, -0.9424426555633545, -0.014488446526229382, 2.246860980987549, 0.8625010251998901, -0.30238133668899536, -1.1519465446472168, -0.5518172979354858, 0.2648596465587616, -0.17015545070171356, -0.4307067394256592, 0.022449299693107605, 0.3248070776462555, 0.9194625616073608, 1.1740615367889404, 1.312720537185669, 1.2369871139526367, -0.855219841003418, 0.5231726169586182, -0.18193979561328888, 0.005461135879158974, 0.09289733320474625, -0.8447777628898621, 1.4898682832717896, 0.8766921162605286, -1.482971429824829, 1.5563160181045532, -0.07165376096963882, -2.565617799758911, -0.4542572498321533, -0.0423414446413517, -0.6571974158287048, -1.1439405679702759, -0.2184096723794937, -1.053928017616272, 0.5189551711082458, 0.8651009798049927, 1.328413724899292, 1.5795323848724365, -0.43691590428352356, -0.23642390966415405, 1.9274731874465942, -0.410856157541275, 0.32208719849586487, 0.556861162185669, 0.3533525764942169, 0.5387137532234192, 0.7664722800254822, -0.17123441398143768, -0.052872687578201294, 0.7329831719398499, -0.15942411124706268, -0.5022345185279846, 2.2255876064300537, 0.5282422304153442, 1.3733279705047607, 0.8977782130241394, -1.0221049785614014, 0.42764338850975037, -0.09702098369598389, -1.1010507345199585, 0.7550948858261108, -2.244107723236084, -1.6303493976593018, 0.29470163583755493, -1.5049844980239868, 2.4664061069488525, 0.5761626362800598, 0.10251300781965256, -1.2148686647415161, 0.5422061681747437, -0.950667142868042, -1.6991230249404907, 0.7558512687683105, 0.32087334990501404, -0.21956738829612732, -0.4665612280368805, 0.5150647759437561, -0.5411325693130493, 1.2863656282424927, -0.5073213577270508, 1.0869354009628296, 0.5534781813621521, 0.48246636986732483, -0.6625530123710632, -0.6981427073478699, -0.7569625377655029, -0.3784931004047394, 0.3916146159172058, -1.17423677444458, 1.6450541019439697, -1.6794686317443848, 0.052778374403715134, -1.0765399932861328, -0.23093266785144806, 0.49457162618637085, 0.9248014688491821, -0.0875416025519371, 0.4577940106391907, -0.3396487832069397, -0.8446800708770752, -1.6267857551574707, 2.0713789463043213, -0.4345310628414154, 0.11744952946901321, -0.4536578059196472, -0.20724846422672272, -0.39938461780548096, -0.1815655678510666, 0.9918044209480286, 1.1356184482574463, 2.011622667312622, -0.46553298830986023, 1.3302124738693237, -0.22365528345108032, 1.1273584365844727, -0.21592451632022858, -0.0003246732521802187, 0.18466773629188538, 0.3527070879936218, -0.39760056138038635, 1.537947654724121, -2.048248052597046, -1.1166658401489258, 0.5107352137565613, -0.01538858562707901, -1.1481820344924927, -0.27597954869270325, 0.6985230445861816, 0.5603681802749634, 0.0900023952126503, -0.013770939782261848, 1.2928961515426636, 0.4777393341064453, 0.8697395324707031, 0.5349969267845154, -0.7592486143112183, 1.184043526649475, 2.323176145553589, -0.3023938238620758, 1.162448763847351, 1.5698754787445068, -1.006440281867981, 0.08839147537946701, 1.0957762002944946, -0.20721369981765747, 0.7811959385871887, 0.26106053590774536, -1.953784465789795, 1.1452839374542236, 0.8201039433479309, -0.8972948789596558, -0.5299260020256042, 0.5094078183174133, 2.049453020095825, -3.4983389377593994, 0.22876237332820892, -0.2768746018409729, 2.20649790763855, -0.356858491897583, 1.4121774435043335, -0.007097797002643347, -1.0928281545639038, -1.5338706970214844, 0.8673304915428162, -2.1142776012420654, 0.6062033176422119, -0.5360360741615295, 0.6403022408485413, 0.08555790781974792, 0.539458692073822, -0.16349564492702484, 0.500015377998352, -0.19919362664222717, -0.10188493877649307, -0.15778830647468567, -2.262579917907715, -1.0274604558944702, -0.15409880876541138, 0.24624525010585785, -0.4513264298439026, -0.21448834240436554, -0.030421409755945206, -0.0770670622587204, 1.2028963565826416, 0.7619999051094055, 0.8127363920211792, 0.3586723208427429, 0.47511059045791626, 0.4086320102214813, -0.7781591415405273, -1.0134224891662598, -0.9843223690986633, 0.21793778240680695, 0.35908088088035583, 0.8759347796440125, -0.4052439033985138, -0.8389734029769897, 1.258521318435669, -1.4293265342712402, 0.4346983730792999, 0.29534465074539185, -1.4362733364105225, 0.09024199098348618, 1.0828735828399658, -1.273100733757019, 0.6076141595840454, 1.0621790885925293, 0.42472216486930847, 1.5400612354278564, 1.196637511253357, -0.8308337926864624, -1.059112548828125, -1.0581295490264893, 0.13169100880622864, 0.8632415533065796, -0.7585163712501526, -0.6763418912887573, 0.43678194284439087, 0.2615239918231964, 0.6582964062690735, 0.6001613736152649, 0.19373802840709686, 2.1276772022247314, 0.38973405957221985, -1.3619352579116821, -2.208559513092041, 0.7325326800346375, -0.5883240103721619, -1.652523398399353, -0.5682832598686218, 0.49835658073425293, -0.8885543346405029, 0.9643542766571045, 0.36702534556388855, -1.237598180770874, 1.501084327697754, -1.41180419921875, 1.0424658060073853, 0.19124969840049744, -1.207672357559204, -0.7874425649642944, -0.6332393884658813, -1.0412098169326782, -0.918149471282959, -0.16647867858409882, 0.5392796397209167, -1.6356676816940308, -0.1880556344985962, -0.07460547983646393, 0.02407703548669815, 0.28828245401382446, -1.5916200876235962, 0.7088401913642883, -0.16394099593162537, 0.46316036581993103, -0.6765769124031067, 1.1563358306884766, -0.4198234975337982, 1.5483921766281128, -0.8470950722694397, 1.4869061708450317, -0.6270074844360352, 0.1569964587688446, 1.3039486408233643, 0.7774395942687988, 0.2670229971408844, -1.3500628471374512, 1.0161042213439941, 0.3531838357448578, 0.14551424980163574, -0.6574238538742065, -1.4495609998703003, -0.32042503356933594, 0.6316981911659241, 1.6515586376190186, 0.21958859264850616, 0.45587703585624695, -0.5162140727043152, 1.1483502388000488, -0.3596014976501465, 0.06859802454710007, 0.0021855616942048073, 0.20508189499378204, -0.218771293759346, 0.30835607647895813, -0.560815691947937, 2.452502489089966, -0.7954347133636475, -0.9503177404403687, 0.8674082159996033, 1.0741885900497437, 0.126596137881279, 2.1068272590637207, 1.0144712924957275, 1.1957730054855347, 0.23475584387779236, -0.008997258730232716, 1.0762617588043213, -0.9133040904998779, -1.1142338514328003, 0.6072245836257935, -0.5819002985954285, 0.1184970885515213, 0.41685372591018677, -0.7118777632713318, -0.6780773401260376, 1.074088215827942, -1.8964675664901733, -0.12237183749675751, 0.929577112197876, 1.3368695974349976, 0.5076615214347839, 1.4686565399169922, 1.6373445987701416, 0.38061848282814026, 1.0824809074401855, 1.1546293497085571, -0.1939307600259781, 1.560020089149475, -0.49740108847618103, -0.07637391984462738, 1.1446104049682617, 0.2834557592868805, -1.6201235055923462, 0.25868457555770874, -0.9047476649284363, 0.3325521945953369, -1.7394107580184937, 0.6598637700080872, -0.0715453177690506, -0.3733094036579132, -0.37859055399894714, -0.5401408076286316, 0.6292641162872314, 0.12445229291915894, 0.8067529797554016, -1.7400360107421875, 0.002536119893193245, -0.9841147661209106, 0.421517550945282, -0.9869899749755859, 1.4050464630126953, -0.5513980388641357, 2.079139232635498, -0.7978759407997131, -0.5688446760177612, -0.42762190103530884, 0.4301256239414215, -0.48596295714378357, -2.059394121170044, -0.5300534963607788, -0.39951542019844055, 1.3803147077560425, 1.0287134647369385, 1.0745433568954468, 0.6045916676521301, 1.047498106956482, 0.8940876722335815, 2.270876884460449, -0.22909711301326752, -0.7127147316932678, 0.2557056248188019, 1.312064528465271, -0.5921998620033264, 0.45805469155311584, 0.7316074371337891, 0.17861522734165192, -0.17546384036540985, -0.8779889941215515, -0.6630063056945801, 0.20279137790203094, -0.1669771522283554, -0.7902054190635681, 0.25693413615226746, 0.42979320883750916, -0.9312272071838379, 0.23588630557060242, 0.8757694363594055, 2.4449737071990967, 1.4982482194900513, 0.9404119253158569, -0.8374667167663574, -0.02481575310230255, -0.29095959663391113, 0.5588961839675903, 0.053532592952251434, -0.7423748970031738, -0.2937832772731781, -0.8893499374389648, 0.373375803232193, 0.12233345210552216, 1.6835490465164185, -1.3334019184112549, -0.21845048666000366, -0.7007483839988708, 0.31583479046821594, 1.838991641998291, 0.5979288816452026, -0.1293763965368271, -1.1630789041519165, -0.6650773286819458, 2.5790324211120605, 1.1717383861541748, 0.2055349498987198, 1.0896403789520264, -0.22994565963745117, 0.3601123094558716, 1.0495198965072632, 1.1089023351669312, 0.5322801470756531, -1.331674337387085, -1.328922152519226, 0.31810328364372253, 0.6802451610565186, 0.3778744041919708, -0.6591200232505798, 0.9401506781578064, -0.20144635438919067, 0.29267048835754395, 0.7784500122070312, 0.43321681022644043, -0.038575462996959686, 0.007339654490351677, -1.2789480686187744, 0.16758646070957184, -1.3484374284744263, -0.8788100481033325, -1.3668878078460693, 2.00494122505188, -0.12359926104545593, -0.6647803783416748, 0.35272106528282166, 1.1020026206970215, -0.7676886916160583, -1.1956628561019897, 0.0770130529999733, 0.9717223644256592, -1.2232922315597534, 0.26394933462142944, 2.002882480621338, 0.9144688844680786, 0.12520626187324524, -0.5164461135864258, -1.373131275177002, -1.0288751125335693, -0.26404571533203125, -0.2713422179222107, -0.07032309472560883, 0.23068253695964813, 0.09868423640727997, 1.2111151218414307, -0.06147724390029907, -0.2576408386230469, 0.47224727272987366, -0.5082963705062866, 0.5870551466941833, 1.0419354438781738, 0.2250458300113678, -0.110145203769207, -0.669693112373352, 0.4929574131965637, 0.7687423229217529, 0.2576165795326233, 0.8187087178230286, -0.9103118777275085, -0.1404104381799698, 1.8621008396148682, 1.1059054136276245, -0.8146718740463257, 0.5708881616592407, -1.0969094038009644, 1.473799705505371, -0.8011477589607239, -0.4863302409648895, -1.0928829908370972, 0.09245262295007706, -0.3179115355014801, 1.0033797025680542, 0.009201055392622948, 0.09603047370910645, -0.1298133283853531, 0.5048453211784363, -1.7757288217544556, 0.41126537322998047, 0.8766130805015564, 0.417318731546402, -0.9007230401039124, 0.5448129177093506, -2.1196131706237793, -0.23477086424827576, 1.0888904333114624, -0.9361912608146667, -0.08844739198684692, 0.3242945671081543, -0.803434431552887, 0.17329755425453186, -0.7909761071205139, -0.32860812544822693, -1.6432428359985352, -0.8002630472183228, 0.6398617625236511, -0.41413092613220215, 1.2117961645126343, -0.35129570960998535, 0.9487559199333191, -1.1521166563034058, -1.5707069635391235, -0.4783939719200134, 0.4856664836406708, 0.4893069565296173, 0.9416770339012146, 0.06278318911790848, 0.34347832202911377, -0.8784974217414856, 0.06802794337272644, 0.7428314685821533, 0.5216914415359497, -0.33969980478286743, -0.5258896946907043, 1.1245999336242676, 0.31879106163978577, 0.6614553332328796, 1.8340750932693481, -0.24709779024124146, -0.761336624622345, 0.02017850987613201, -0.5862278938293457, -1.5460704565048218, 1.628602147102356, 0.6149107813835144, -0.4497934579849243, 0.9436677098274231, -0.5831751823425293, 0.7758297324180603, 0.1471770703792572, 0.3049161434173584, -0.3129410147666931, 1.6557198762893677, -1.0160276889801025, -0.8872999548912048, -0.7664933800697327, 1.2340590953826904, -0.679095447063446, 0.3452998995780945, -1.0590087175369263, -0.2265823483467102, 0.028986884281039238, -0.3428991436958313, -1.2649054527282715, 0.19912077486515045, -1.51218581199646, -2.1041932106018066, -0.4325345456600189, -0.9689907431602478, -1.488881230354309, 0.9701994061470032, -1.3367877006530762, -0.03525800630450249, -2.6071293354034424, -0.37002748250961304, -0.37618714570999146, 0.9978847503662109, 1.2643051147460938, -1.9365061521530151, -0.8706855773925781, -0.9062780737876892, 0.8918402194976807, -1.5801879167556763, -1.2377684116363525, 0.6400208473205566, -1.4275226593017578, -0.6587166786193848, -0.6719010472297668, -0.16216935217380524, 1.5002336502075195, 0.3231569528579712, 1.1607908010482788, -0.8362227082252502, 1.0668617486953735, 1.6882623434066772, 1.1717907190322876, -0.9535499811172485, -2.037144422531128, 1.8532114028930664, 0.050186388194561005, -0.7922364473342896, 0.6688010692596436, 0.33586061000823975, 2.3135390281677246, -0.9149444699287415, 0.8035051226615906, 1.0864371061325073, -0.8450383543968201, -0.2143353521823883, -0.4799388647079468, -0.5343483686447144, 0.09172512590885162, 0.35938626527786255, 0.03285568952560425, 0.7042865753173828, -0.9119839668273926, 0.14615121483802795, 0.25326934456825256, 0.409517765045166, -1.0345330238342285, 0.3935661315917969, 0.22091206908226013, -2.3145253658294678, 1.3493412733078003, 0.9300114512443542, 0.08981230854988098, -0.4056464433670044, -2.1251487731933594, 1.7903892993927002, -0.3820112645626068, -1.0277905464172363, -0.0014851734740659595, -1.0566632747650146, -0.5822728276252747, -0.21586304903030396, -1.6232692003250122, -0.149924173951149, 0.2999183237552643, -1.3635773658752441, 0.3743160665035248, -0.8668453693389893, 0.48149847984313965, -0.4623883068561554, 1.573891520500183, 0.5334711670875549, -0.8205147981643677, -0.7834996581077576, 0.4463268220424652, 0.58986896276474, -1.0419176816940308, -0.054788194596767426, 0.545838475227356, 2.211691379547119, 1.2175325155258179, 1.0502023696899414, -2.837564706802368, -1.646645426750183, -1.981972575187683, -0.16707830131053925, -1.3873645067214966, -1.0169203281402588, -0.5405764579772949, 0.5148352384567261, 0.01994071714580059, 0.4421491324901581, -1.1003026962280273, 0.8399858474731445, -2.747952699661255, 1.4115748405456543, -0.026121271774172783, -1.0167663097381592, 0.21506857872009277, 0.4093272089958191, -0.6082751154899597, -0.08532564342021942, 1.1123450994491577, 0.5553429126739502, 0.32687607407569885, -0.6664565205574036, 1.0421663522720337, -0.844312846660614, -1.5053520202636719, 1.8611037731170654, 1.2211552858352661, -0.13672155141830444, -0.1534639596939087, -1.0116560459136963, 2.494553565979004, 0.0433661974966526, -1.2530663013458252, 0.6877882480621338, -0.4935548007488251, 0.4337019920349121, 0.748977780342102, 1.524421215057373, 2.825330972671509, -0.3187161087989807, -0.38056766986846924, -1.3692600727081299, -1.7525593042373657, -0.26082032918930054, 1.7028324604034424, 0.9686443209648132, 0.9013912677764893, -0.8932897448539734, 1.4921742677688599, 0.26195666193962097, 1.6946539878845215, 0.7389876842498779, 0.387602835893631, 1.5269299745559692, 2.211679458618164, 1.0795469284057617, 0.20803605020046234, -0.15422092378139496, -0.19126658141613007, 0.416036993265152, -0.3311178386211395, -0.6833739280700684, 0.8983619809150696, -0.5694888830184937, 0.572594165802002, -0.9814701080322266, 0.5547195076942444, -1.2007229328155518, -0.9866605401039124, -0.3138394057750702, 0.351992666721344, 1.7071266174316406, -1.0076894760131836, 0.2828497886657715, 0.001805064151994884, -1.4239768981933594, 0.25219443440437317, 0.6530877351760864, -0.02715785801410675, 1.2524219751358032, 0.6445118188858032, 0.47905233502388, 1.048231601715088, 2.1625025272369385, 1.5732792615890503, -0.5169107913970947, -0.21973884105682373, -0.6024013161659241, 1.054608941078186, -0.38048577308654785, 0.9367408752441406, -0.5014093518257141, -0.45593494176864624, 0.8804013729095459, -2.196892499923706, -0.6798810958862305, -0.7144155502319336, -1.134505271911621, 1.115314245223999, 1.3925496339797974, 0.5332363843917847, -0.5699626207351685, 0.35883429646492004, -0.08984040468931198, -0.9491648077964783, 1.0189087390899658, -0.4928086996078491, -0.19418691098690033, 0.3044659197330475, -0.10340908169746399, -1.4758456945419312, -1.4236140251159668, 0.4602154493331909, 1.7477028369903564, -2.6488709449768066, 0.3987056612968445, -0.203839972615242, 1.2093898057937622, -0.9417773485183716, 1.171604037284851, -1.6950289011001587, -0.46594953536987305, -1.001430630683899, 0.8089878559112549, 1.3775278329849243, 0.15529195964336395, -0.8857744336128235, -1.5345309972763062, 0.02889513410627842, 0.6229549646377563, -0.10893727838993073, 1.0047392845153809, -0.06340446323156357, 0.838981568813324, -0.20568127930164337, 0.711517333984375, -0.01040232926607132, 0.47681546211242676, -0.9622510075569153, 1.0841130018234253, 1.0941609144210815, 0.21475768089294434, 0.41473889350891113, -0.2633143365383148, -0.8250938057899475, 0.1529342532157898, 1.3826044797897339, -0.5346832275390625, -1.806447148323059, -0.654135525226593, -0.13107401132583618, 0.48380205035209656, 0.9947063326835632, 0.25335922837257385, 0.1210758313536644, 0.882714569568634, 0.5884186625480652, 0.6598954796791077, 1.08902108669281, 0.7802237868309021, -0.12948626279830933, -1.52078378200531, 0.13263525068759918, 0.12785936892032623, 1.2896934747695923, -0.7245630025863647, 1.595345139503479, 0.1583983153104782, 1.0186190605163574, 0.02899266965687275, 1.0171828269958496, 0.9476276636123657, 1.2523550987243652, 1.9306195974349976, -3.0962493419647217, 0.8824542760848999, -0.15398390591144562, -0.03615059331059456, -0.9259481430053711, 0.12544305622577667, 1.5301874876022339, 0.02969355508685112, 1.0261706113815308, -1.7731410264968872, -0.1254485547542572, -1.8068186044692993, 2.0177083015441895, -1.2095032930374146, 0.45530104637145996, -0.14586521685123444, -1.751201868057251, -0.7343931794166565, -0.4421667158603668, -2.4726240634918213, -0.3475556969642639, 1.328774094581604, -0.5979765057563782, 0.16692191362380981, -0.7060264348983765, -0.4458406865596771, -1.9663125276565552, -0.0185070913285017, -0.036787427961826324, 1.1693997383117676, -0.34654074907302856, 0.5206640362739563, 0.6053281426429749, 1.0893580913543701, 0.8387923240661621, -0.07453752309083939, 0.22383750975131989, 1.2568109035491943, 0.4902088940143585, -0.2624426484107971, 0.22773392498493195, 0.00585306528955698, -0.041880689561367035, -0.4760240912437439, 0.777892529964447, 0.2149847149848938, 1.1827102899551392, -1.9926782846450806, -1.4994785785675049, -0.3432813286781311, 0.8880417346954346, 0.7172378897666931, -0.6773533225059509, -1.3314731121063232, -1.0617376565933228, 1.8174985647201538, 2.084768056869507, 0.5991223454475403, -0.8573225140571594, 0.59398353099823, 0.5560035705566406, -0.24257253110408783, -0.148223415017128, -0.2764754593372345, 0.5383635759353638, -1.4313069581985474, 0.9749071002006531, -0.16507361829280853, -0.3978341221809387, -0.6966403722763062, 0.5069388151168823, -1.2162472009658813, -0.7368841767311096, 0.9344690442085266, -1.0386875867843628, -0.22934207320213318, 0.29471367597579956, 0.8617880344390869, 0.6734330654144287, 0.8041715025901794, -0.9010822176933289, -0.974870502948761, -2.265615940093994, -1.1206369400024414, -2.1009469032287598, -1.0949302911758423, -0.06116490066051483, -0.4385746717453003, 1.0359675884246826, -0.21979622542858124, 0.30704769492149353, 0.6287662386894226, 2.2471039295196533, -1.546040654182434, 0.5318451523780823, 0.3386152982711792, -1.7438023090362549, 0.3450578451156616, 0.5337939262390137, -0.9807609915733337, 1.3497036695480347, 0.811211347579956, 0.4590567648410797, -0.06976575404405594, 1.335909128189087, -0.8079127669334412, 0.5937566161155701, 0.23923969268798828, 0.514598548412323, 0.45051470398902893, -0.040497999638319016, -0.762592077255249, 1.5980371236801147, -0.13008952140808105, 0.2076263278722763, -0.9514068365097046, -0.3828822374343872, 2.199692726135254, 0.07792960107326508, 1.0133957862854004, -0.4996596872806549, 0.4664057791233063, 0.8017095923423767, -1.9719630479812622, -1.6061676740646362, -1.3072917461395264, -0.3167226314544678, 0.6409467458724976, -0.501055896282196, 0.22310252487659454, 0.36768627166748047, 1.2319849729537964, 0.5264793038368225, -0.3676134943962097, -0.40679502487182617, -0.645693302154541, 0.35771122574806213, 0.8862319588661194, 2.3332090377807617, -0.45326027274131775, 0.13602252304553986, 1.2340587377548218, -0.5933602452278137, -0.5350658297538757, 0.4299013316631317, -0.26935988664627075, -1.7756258249282837, -0.15074928104877472, -0.39182695746421814, -0.7667880654335022, -1.429438591003418, 0.847000777721405, 0.6310386657714844, -1.233845591545105, 1.7487103939056396, 0.9288911819458008, -0.32551366090774536, -0.49783602356910706, 0.4568706154823303, 0.832643449306488, -1.0783929824829102, -0.584748387336731, -2.9288036823272705, 0.6316649913787842, 0.21609847247600555, -1.9892311096191406, -0.4473429322242737, -1.013792872428894, -0.5055328011512756, -0.030252179130911827, 0.9099743366241455, 1.2569245100021362, 1.5416181087493896, 1.2837120294570923, 2.0576863288879395, -0.5519557595252991, -1.5751266479492188, 0.6454282999038696, -0.33719369769096375, -0.39323508739471436, -0.5154480338096619, 0.03397432342171669, -0.11406006664037704, 0.9842211008071899, 0.6810645461082458, -0.780600905418396, -0.5325322151184082, -0.6284888982772827, -0.12781548500061035, -2.2501776218414307, -1.353927493095398, -1.6430631875991821, 0.4904368221759796, -0.12273969501256943, -0.9688069820404053, -0.6681846976280212, -0.08748344331979752, -3.1339330673217773, 0.3918047845363617, 2.158545732498169, -0.34733831882476807, -0.09344596415758133, -0.6282467842102051, -0.11794744431972504, 1.4650788307189941, 0.3565984070301056, -2.260094165802002, 1.4705216884613037, -0.5385221838951111, 1.0474002361297607, -1.5607577562332153, -0.8880713582038879, -0.1393561214208603, -0.5610764622688293, -0.7095320224761963, 0.2998459041118622, 0.27335724234580994, 0.006067429203540087, 1.2987638711929321, -0.4369889795780182, 0.5007613897323608, 0.642088770866394, 1.3256635665893555, 0.5991691946983337, 0.6955961585044861, -0.020592469722032547, -1.1184649467468262, -0.6899108290672302, -1.4716436862945557, 0.343657523393631, 0.5897439122200012, 0.3704518675804138, -1.450229287147522, 0.7309527397155762, -0.5503799319267273, 0.0021622171625494957, -1.5666271448135376, -1.6899515390396118, -1.6337438821792603, 0.6150599122047424, 0.6112197041511536, -0.4011097848415375, 1.2051520347595215, -2.2754673957824707, -0.2517526149749756, 0.4072470963001251, -1.3083655834197998, -0.006300946697592735, -0.7817633748054504, 2.5333855152130127, -0.92519611120224, 0.14913423359394073, 1.438787579536438, 0.5232700109481812, -1.5973085165023804, -0.15319335460662842, 0.13340774178504944, -0.7408484816551208, 0.18425370752811432, -0.6582068800926208, 0.3905966281890869, -1.6313201189041138, -0.26848989725112915, 0.9080119132995605, -0.5230274796485901, 0.43473851680755615, -1.0718404054641724, 0.22789354622364044, -0.5114859342575073, -0.3120783865451813, -1.0405858755111694, 1.250544548034668, -1.4852405786514282, -0.3763996362686157, 0.6766244173049927, 0.3082100749015808, -2.7674076557159424, 2.817678213119507, 0.6082546710968018, -0.48721978068351746, 1.5840870141983032, -0.4004962146282196, -0.32035115361213684, 0.05366853252053261, -1.8346874713897705, -0.36782732605934143, 1.2922827005386353, 1.6138160228729248, -0.7741177678108215, 1.0435959100723267, 0.7448493242263794, -1.7699052095413208, 0.7645697593688965, -0.860889732837677, -1.056755781173706, 0.4297010004520416, 0.4694940745830536, 0.10823346674442291, 0.15876245498657227, 2.679641008377075, 0.31810683012008667, 0.7737162113189697, 0.4528175890445709, 0.5357601046562195, -0.4416312277317047, 0.4636511206626892, 0.6088457107543945, -1.521440863609314, 0.7207943201065063, 2.0658750534057617, 1.2494778633117676, 0.7755376696586609, 1.7030209302902222, 0.11988606303930283, 0.5033828020095825, 1.184828519821167, 1.6291489601135254, 0.5182105898857117, 0.24074549973011017, 0.8304450511932373, -1.0470225811004639, 1.933393120765686, -0.23112817108631134, -0.41363874077796936, 0.7968804240226746, -0.659311056137085, 0.40604081749916077, -0.8348888158798218, 0.2481040358543396, -0.48798269033432007, 2.125016212463379, -1.2485466003417969, -1.2875725030899048, 1.1685094833374023, -0.1853741556406021, 0.7837291955947876, 0.8358694314956665, -2.134411573410034, 0.9229757785797119, 0.34678229689598083, -0.6637129187583923, 0.11404132843017578, -0.9823988080024719, 0.3915664553642273, -0.3283052146434784, -1.2360374927520752, -0.24457775056362152, 1.1322485208511353, -0.5224472880363464, 0.5381815433502197, 0.4850359261035919, -0.20847931504249573, -0.5586981177330017, -0.9940305352210999, -0.24700360000133514, -1.3637397289276123, -1.8060449361801147, -0.6591537594795227, -1.1720702648162842, 1.2112141847610474, -0.6513063907623291, 1.4333789348602295, 0.15885484218597412, 0.20036128163337708, 1.2322567701339722, 0.40967801213264465, -1.0316084623336792, 0.4049338698387146, 1.150122880935669, 0.3980860114097595, 0.3149482309818268, 0.7764095067977905, -0.3523385226726532, -0.2340022474527359, -0.10596534609794617, 1.8697236776351929, -0.9141387939453125, 0.817356526851654, 0.7929684519767761, 1.336031436920166, -1.339512825012207, 1.3900381326675415, 0.6826637387275696, -0.3695213198661804, -0.2701318562030792, -1.5226753950119019, 0.16594703495502472, -0.5161409378051758, 0.19574293494224548, 1.3743807077407837, 1.577580213546753, -0.41451412439346313, 0.9558413624763489, -0.40298891067504883, 1.1031385660171509, -1.1272735595703125, -2.8978896141052246, 0.4051501452922821, 0.6134243011474609, 0.971245527267456, 0.22661936283111572, 0.0874532163143158, -1.6769520044326782, 0.7566744089126587, -0.5825030207633972, 0.9906932711601257, 2.780825614929199, -0.4757487177848816, -0.11850317567586899, -1.186479091644287, -1.3620896339416504, -0.45416149497032166, -1.2993032932281494, 0.4172939956188202, 1.7519241571426392, 1.0567415952682495, 1.3151137828826904, -0.5676203966140747, 2.039106845855713, -0.11079571396112442, -0.5800852179527283, -0.9161956310272217, 0.3170589506626129, -0.8475753664970398, -0.32122325897216797, 0.6101610064506531, 1.0852245092391968, 0.16301211714744568, -0.08045248687267303, -1.7645742893218994, -1.3190844058990479, 0.8920676112174988, 0.6116931438446045, 0.7366215586662292, 0.7031360864639282, 0.48459911346435547, 1.2950745820999146, -0.8977124094963074, -0.21857912838459015, 0.484820157289505, -0.9339475631713867, -0.6696221232414246, -0.6004933714866638, -0.2595537304878235, -0.21110719442367554, -1.0756462812423706, -0.6197690367698669, -1.586138367652893, 0.7117911577224731, 0.1188860684633255, -0.25752463936805725, -0.1991473287343979, 0.19986671209335327, 0.5550133585929871, -0.021238181740045547, 0.6768128275871277, 1.2049262523651123, 0.590705931186676, 1.0823938846588135, 0.1629406362771988, 0.24496980011463165, -0.17183031141757965, -2.316072463989258, 1.7414814233779907, 1.4218674898147583, 0.13305239379405975, -0.5513865351676941, 0.46641233563423157, 1.3849372863769531, 0.6842920184135437, -0.10905683785676956, -0.5809310078620911, 1.0649951696395874, 0.34283319115638733, -0.19467978179454803, -0.3783371150493622, -0.30214282870292664, 1.3037985563278198, 1.4830492734909058, 0.22852209210395813, -0.34506750106811523, -0.7971377372741699, -1.3421032428741455, 0.49005556106567383, 0.6923019289970398, 0.26646074652671814, -0.22274045646190643, 0.5503622889518738, -1.1112748384475708, 1.6588150262832642, -2.193134069442749, 1.981484293937683, 1.8840186595916748, -0.6472804546356201, 0.597139835357666, 0.6911805272102356, 0.39458999037742615, 0.777466893196106, -0.5355626940727234, 0.9776908755302429, -1.2790801525115967, -0.3499611020088196, 0.5431165099143982, -1.336151361465454, -1.1573249101638794, -0.1624271124601364, -0.8544777631759644, -1.0095425844192505, 1.8572055101394653, -0.6735510230064392, -0.011996709741652012, 0.8923444747924805, -1.1840168237686157, -1.2329164743423462, -0.4742859899997711, -0.3888343274593353, -1.0608055591583252, 1.3540539741516113, -0.19865263998508453, 0.5898866057395935, 1.2037911415100098, 2.1186435222625732, -0.5739064812660217, -0.5309483408927917, 0.09408488124608994, -0.39857617020606995, 0.8164481520652771, -0.3660069704055786, -0.21399354934692383, -0.1782749444246292, 0.10050911456346512, 0.386578768491745, 0.738158106803894, -0.5625985264778137, 1.8355792760849, -0.4606797695159912, -0.6359170079231262, 0.1290537565946579, -0.6990293264389038, 0.03629153594374657, -0.3124682605266571, -0.8119183778762817, -0.6888270974159241, -0.5581969618797302, -0.07631269097328186, 0.3271726071834564, 0.9577277302742004, -1.4871734380722046, 0.37757736444473267, -0.8926607966423035, -1.20082688331604, -0.16745075583457947, 1.190908670425415, -0.9748832583427429, -0.3143874704837799, -1.9092508554458618, 0.3214940130710602, -0.3768603503704071, -0.3576918840408325, -0.3727271556854248, -1.712346076965332, 0.3641606569290161, -0.6536546945571899, -0.1903810203075409, 1.0459574460983276, 0.8580016493797302, 0.6038005352020264, 0.3378189206123352, -0.0036310874857008457, -0.7957101464271545, 2.2711009979248047, -0.3996620178222656, -1.3329964876174927, 0.037693094462156296, 0.6875927448272705, 0.8153398036956787, 2.3070573806762695, 0.3646784722805023, 1.8579398393630981, -0.77342689037323, -0.4199681580066681, 0.24773986637592316, -0.18778179585933685, 1.9937844276428223, -0.07506819069385529, 0.37915459275245667, 0.8233805298805237, 0.5264004468917847, 0.7334828972816467, -0.576883852481842, -1.63861882686615, -1.9605512619018555, 0.22059616446495056, 0.6076205968856812, -0.4838666021823883, 0.5607022643089294, 0.3006759285926819, 1.0232104063034058, 0.936601996421814, -1.1096454858779907, 0.6633532643318176, 0.29110798239707947, -0.30816271901130676, -1.577873706817627, -1.4826635122299194, 0.4503462612628937, -0.3821662664413452, 0.2118426412343979, 0.32283613085746765, 0.8504485487937927, 0.41107359528541565, -0.19170978665351868, 0.8213368654251099, -1.1732304096221924, 0.3746601343154907, -0.18058249354362488, 0.09434836357831955, 0.36341509222984314, 0.9054723978042603, -1.4856020212173462, 1.7358739376068115, 0.6973767876625061, 0.6840339303016663, 0.8193628191947937, 1.1625796556472778, 0.3143859803676605, -0.9494116306304932, -0.3770204782485962, -1.532044529914856, -1.4776995182037354, -1.6143323183059692, 0.8873451948165894, 0.3467678129673004, 1.2241178750991821, 0.3161129653453827, 0.4431026577949524, -0.5418084859848022, 0.1718798130750656, -0.29597699642181396, 0.438429594039917, 0.6167997717857361, 0.8833491206169128, -0.22925330698490143, -1.6014331579208374, -0.05177553743124008, 0.11994966864585876, 0.04679347574710846, -0.06535481661558151, 0.9740018248558044, -1.1936843395233154, -1.7242748737335205, -0.4856187701225281, -0.3719162046909332, -2.207764148712158, -0.6408036351203918, -1.623733401298523, -0.37580588459968567, -2.0420167446136475, 0.8340034484863281, 0.7276239991188049, -0.6182031631469727, 0.5184181928634644, -0.31487539410591125, 0.14930331707000732, 1.8675954341888428, 0.22826044261455536, -0.829526424407959, -0.7260246276855469, -0.18713228404521942, 1.0471307039260864, -0.18046534061431885, -0.28496208786964417, 0.1113092228770256, 0.5323523879051208, 2.055911064147949, -0.3139991760253906, 0.09251812100410461, -0.9298912286758423, 1.5070751905441284, 0.3016509413719177, 1.7833504676818848, 1.609967827796936, 0.7541093230247498, -0.8575959205627441, 0.37220510840415955, 1.113462209701538, -0.19823071360588074, -1.4973326921463013, 0.20533616840839386, 0.5524109601974487, -1.1748292446136475, -0.6808244585990906, 1.4827266931533813, -0.9462012052536011, 1.6006674766540527, -0.7440690398216248, -2.0688791275024414, 0.6528491973876953, -1.138338565826416, -0.6116318106651306, -1.5373483896255493, -0.6897526383399963, 2.1807024478912354, 1.596709132194519, 0.7907606363296509, -0.46404170989990234, 0.02578856237232685, 0.17851616442203522, 1.937333106994629, -1.1960928440093994, -0.4833468794822693, 0.006070687435567379, -1.4448554515838623, 0.5424063205718994, -2.2044601440429688, 1.257271409034729, 0.07602597773075104, 0.2592468559741974, 1.3542964458465576, -0.7248310446739197, -1.6848024129867554, 1.9057514667510986, -0.3883296549320221, 1.0762900114059448, -0.423881858587265, 0.3226114809513092, 0.1543576419353485, 0.28853023052215576, 0.611213207244873, -1.45682954788208, 0.13931620121002197, 0.06440065801143646, -1.4545478820800781, -1.2435827255249023, 1.652565598487854, 0.4918573498725891, 1.3730003833770752, 0.3556203246116638, -0.6393259763717651, -0.4412959814071655, -0.12571050226688385, 2.408262252807617, -0.01406580489128828, -0.8772852420806885, -0.6437082886695862, 0.8358871340751648, -0.3690565526485443, 0.7402163147926331, -0.5322443246841431, -0.03777138516306877, 1.529273509979248, -0.546404242515564, 0.21533213555812836, 0.46020156145095825, 0.5058501362800598, -0.5153183341026306, -0.3905915915966034, 0.7327878475189209, 0.15141889452934265, 2.0062079429626465, -0.36067995429039, 0.5612841248512268, -0.17838746309280396, 0.9096456170082092, -1.065337896347046, 1.714372158050537, 0.5533361434936523, -1.8217194080352783, 0.9929704070091248, -0.5806280374526978, 1.834778070449829, 0.10338135808706284, 0.5593316555023193, 0.6354652047157288, 0.05195413529872894, 0.2612168490886688, 1.7823851108551025, -0.6326553821563721, 0.6127346754074097, -0.36416977643966675, -0.9226469397544861, -1.330320119857788, 0.9985747337341309, -0.7817102670669556, 0.557037889957428, 0.174571692943573, 0.5988132357597351, -0.9074640274047852, 0.47032856941223145, -0.6759219765663147, 0.45854702591896057, 1.5808197259902954, 0.14752709865570068, 1.2006289958953857, -0.4913879334926605, -0.25807034969329834, -2.348008155822754, 1.1577098369598389, -0.4311968684196472, 0.27894675731658936, 0.25675663352012634, -0.5915201902389526, 0.40550026297569275, -0.16179810464382172, -0.05807270109653473, -0.6775546669960022, 0.7018608450889587, 0.75246262550354, -2.644293785095215, 0.1831331104040146, 0.16630347073078156, 0.7581855058670044, -1.084087610244751, 1.8548033237457275, -0.7551952600479126, -1.6117024421691895, -0.7043037414550781, 1.3184694051742554, 0.7336571216583252, -0.06786813586950302, -0.5199799537658691, -0.11859314143657684, -1.8572975397109985, 1.3129267692565918, 0.3115485608577728, -2.074887752532959, 0.12576685845851898, 0.48380470275878906, 1.2910783290863037, 1.5507092475891113, -0.9769980907440186, -0.33684155344963074, 2.533179521560669, 1.2702709436416626, 0.21827557682991028, 0.2179204523563385, -0.3285541832447052, 0.11530233919620514, 1.2737864255905151, -0.4271186888217926, -0.9895097017288208, 1.069496989250183, -0.3301709294319153, -0.7335332036018372, 0.24631601572036743, 0.1852324903011322, 0.3883616328239441, 0.7504522204399109, -0.036994557827711105, 2.107952833175659, -0.33617788553237915, -0.006872242316603661, 0.16412118077278137, 0.6467875242233276, 0.5688087344169617, 1.4177093505859375, 0.18862725794315338, 0.38671278953552246, 0.9065002799034119, -0.9356193542480469, 0.8931587338447571, -0.053430940955877304, -2.421262264251709, 0.7962898015975952, -0.6614492535591125, -0.9746267795562744, 0.06174946576356888, 1.76857328414917, 1.3418972492218018, -0.23385083675384521, 0.004103748593479395, -1.0948139429092407, 1.2555286884307861, -1.530104398727417, 1.2040464878082275, -0.9339425563812256, 0.30054327845573425, 0.4626658856868744, -1.2469825744628906, 0.7062844634056091, 1.3001346588134766, -0.031777288764715195, 0.12096519023180008, -0.7736221551895142, -0.8404421806335449, 0.9307291507720947, 0.46854856610298157, 0.2762199342250824, -0.7363266944885254, 0.19168823957443237, 0.8332287669181824, 0.32903414964675903, -0.2806403636932373, 1.7838138341903687, 1.9153804779052734, 1.4554532766342163, 0.21862199902534485, 0.9327147006988525, 0.704369843006134, -0.10491917282342911, 1.1747212409973145, 0.7314966917037964, 0.404891699552536, -0.5304020643234253, -1.0196343660354614, 0.9050025343894958, -0.3768240511417389, 0.8430884480476379, -0.3932412266731262, 0.6514221429824829, -0.16349156200885773, -0.6546517610549927, -0.1838361918926239, 1.154079556465149, -0.8194758892059326, 0.38002198934555054, -0.0009093214757740498, 0.4736967384815216, 0.8325907588005066, 1.5617797374725342, 1.3857495784759521, -0.1863536834716797, 0.498972624540329, 1.3300747871398926, -0.5657910704612732, 0.39591196179389954, 2.3861353397369385, -1.2417020797729492, -0.2984575629234314, 1.0975062847137451, 0.031468652188777924, 2.0202059745788574, 1.2372065782546997, -0.10647071152925491, 1.575263500213623, 0.23959463834762573, -1.3018046617507935, -0.04475221782922745, -1.4455463886260986, -0.6233525276184082, 1.1279572248458862, -1.5029394626617432, 0.23002339899539948, 1.6419703960418701, 0.1533844918012619, 0.6066826581954956, -0.2734052836894989, 1.7016676664352417, 0.6891295313835144, 0.4705880880355835, 0.31421327590942383, -0.43095019459724426, -0.16402606666088104, 0.6556081175804138, -0.18091131746768951, -0.07091470807790756, -0.8899625539779663, -0.5282186269760132, 0.05140000954270363, 0.8229787945747375, -0.5878590941429138, 1.0799942016601562, -0.242076113820076, -0.7339114546775818, -0.9197221994400024, -1.1003172397613525, 2.9248807430267334, 0.7593127489089966, -0.5076088309288025, 1.1607182025909424, -1.896551251411438, 0.5432485938072205, -1.6091748476028442, 1.723160743713379, -0.8846153020858765, -1.366618037223816, 1.4221938848495483, -0.23092961311340332, -0.6402871608734131, -1.706859827041626, 0.6950969696044922, 0.05894400551915169, 1.4100050926208496, 2.7881276607513428, 0.09755420684814453, -0.3667283058166504, -0.49972692131996155, -0.7933334708213806, 0.23365570604801178, 0.8889302611351013, 1.5243172645568848, 0.030469100922346115, -0.143732950091362, -1.1218312978744507, 0.18824416399002075, 0.5098314881324768, -1.4285752773284912, 1.2204006910324097, -1.4336159229278564, 0.4656377136707306, 0.9615718126296997, -0.4612220823764801, 0.5778574347496033, -1.1723793745040894, -0.34871751070022583, 1.1542444229125977, -0.44112569093704224, -1.0151209831237793, 1.2514084577560425, -0.5686374306678772, -1.697633981704712, 1.9547157287597656, 0.8995860815048218, 0.3260260224342346, -0.7315930724143982, 2.22623610496521, 0.8944331407546997, 0.3038786053657532, 0.4099765121936798, -0.9942905902862549, -0.7312554717063904, -1.791870355606079, -1.9898033142089844, 0.27283063530921936, 0.1433403342962265, -1.21909499168396, -0.34777942299842834, -1.5086842775344849, -0.3702298104763031, 0.44710347056388855, 1.5604784488677979, -0.03490763530135155, -0.5845515727996826, 0.5409446954727173, 0.03490171581506729, 1.1978516578674316, 0.7457648515701294, -2.157273530960083, -1.4792773723602295, 0.15322352945804596, -0.4184875786304474, -0.22957196831703186, -0.8283790349960327, 2.3417229652404785, 0.005909713450819254, 0.47031596302986145, 1.6002953052520752, 0.2633894681930542, -0.2901003956794739, -0.5687792897224426, -0.02624097280204296, 2.070782423019409, 1.1377038955688477, -0.334236204624176, -1.0050177574157715, -0.9012634754180908, 1.4283881187438965, 0.09448269754648209, -0.7165743708610535, 1.0182781219482422, 0.4359499216079712, 0.08686301857233047, 0.8715388178825378, 2.082576274871826, -0.15788400173187256, -0.4431312382221222, 0.6581861972808838, -1.0724444389343262, 1.43651282787323, -1.5315437316894531, -1.9907433986663818, 0.12418801337480545, 0.3229368329048157, -0.9611643552780151, 0.9174521565437317, -1.569684624671936, -0.310985267162323, -0.9350001811981201, -0.2538260221481323, 0.4459264874458313, -1.0102697610855103, 0.16592195630073547, -1.2241237163543701, 0.9136245250701904, -0.5885728001594543, 0.22863352298736572, -1.0757243633270264, 0.9790681600570679, -0.9697378277778625, -1.4456497430801392, 0.31245145201683044, 0.8880040049552917, 0.7579644918441772, -0.6327692866325378, 0.04982709884643555, -0.5120753049850464, 0.5008294582366943, 0.15054261684417725, 1.199182152748108, 0.3967037498950958, 0.8860999941825867, 0.22327201068401337, 0.36480048298835754, 0.9501891136169434, 0.5562515258789062, 0.8257026672363281, 0.5245139002799988, 0.5021271109580994, 0.4616136848926544, -1.7248954772949219, 0.16145607829093933, -0.0834345817565918, 0.05639861524105072, -0.053209323436021805, 0.4299014210700989, 0.1803475171327591, 0.1316053420305252, -0.3785579800605774, -1.674320936203003, -0.09912163764238358, 1.197041630744934, -0.3147798478603363, -1.3859041929244995, -0.47208407521247864, 0.2030605673789978, 0.2240634709596634, -0.05644962936639786, 1.0588407516479492, 1.293079137802124, -0.7753243446350098, 0.06734764575958252, 2.7440013885498047, -0.45460039377212524, -0.7064154744148254, -0.29355666041374207, 0.2857283353805542, 0.05584219843149185, -0.4522343575954437, 0.7222884893417358, 0.2485860288143158, 1.8957761526107788, -1.1625233888626099, 0.9309301376342773, 0.35910114645957947, -0.43277376890182495, -0.9673122763633728, -0.32361072301864624, 0.24176907539367676, 1.4193170070648193, 0.9279928803443909, -0.651473343372345, 0.32528045773506165, 1.239270806312561, -1.3678171634674072, 0.5155346989631653, -0.894133985042572, -1.5892539024353027, -0.13084037601947784, -0.9926984310150146, 0.9249727129936218, 1.1286587715148926, 0.34025514125823975, 1.8227319717407227, 0.42846187949180603, 0.5676196217536926, -0.21988411247730255, 0.6702940464019775, 1.193381667137146, -0.3914846181869507, -0.8627922534942627, -0.4418826401233673, 0.966759979724884, 0.4491484761238098, -0.1684822291135788, 0.7729319334030151, 0.45247989892959595, 0.4099246859550476, -1.6534695625305176, -1.4288041591644287, -0.2674616277217865, -1.1018781661987305, -0.5390472412109375, 0.4477570950984955, -0.1877644807100296, 0.020574066787958145, 0.3607816994190216, 0.24728865921497345, -0.8730212450027466, -2.297555685043335, 0.13164183497428894, 1.8322644233703613, -0.6999134421348572, -1.3449329137802124, -0.22661039233207703, -1.1405911445617676, 1.2360228300094604, -1.3801933526992798, -0.05169101804494858, -0.5316183567047119, 0.790928065776825, 0.06289564073085785, 0.17133909463882446, -0.6572208404541016, 1.2835633754730225, 1.2868789434432983, 0.4443082809448242, -1.859565258026123, -2.533663511276245, -2.279052972793579, -0.517737865447998, -0.28206247091293335, 0.119229756295681, -0.9009061455726624, -0.040108513087034225, 0.4814184904098511, 0.10627948492765427, -0.782107412815094, -1.569594144821167, -0.8856583833694458, 1.3362202644348145, -0.9559180736541748, 1.2006738185882568, 0.8585473299026489, -0.7700703740119934, -1.5811176300048828, -0.9466641545295715, 0.18449927866458893, -0.8305577635765076, -0.214091956615448, 1.492254376411438, -0.3032630383968353, -2.4973857402801514, -0.8049229979515076, 0.7783078551292419, 0.1817883551120758, 0.5717763900756836, -0.74867182970047, -1.1754883527755737, 1.885970115661621, -0.6036356091499329, 0.37579697370529175, 1.5092129707336426, -0.4450817406177521, 0.1807485669851303, 0.3222813308238983, -0.16869904100894928, 0.45120489597320557, -0.1989283561706543, -1.7454464435577393, -1.0654181241989136, 0.34402382373809814, 0.06590741127729416, -0.6431761980056763, -0.04998612031340599, 0.7298797965049744, 1.7960103750228882, 0.7529141902923584, -0.849134087562561, -1.616668462753296, -0.07730379700660706, 0.5099554657936096, 1.1380112171173096, -1.9714101552963257, 0.30241963267326355, 1.4246041774749756, 1.112323522567749, -1.8237707614898682, -0.5001763105392456, -0.03701753169298172, 0.7538673877716064, 0.34536194801330566, -0.6522831916809082, -0.18379096686840057, -0.05976615101099014, -2.419635057449341, 0.9137123227119446, 1.316736102104187, 0.4222249984741211, 1.8436577320098877, -0.012143369764089584, -0.9287250638008118, -0.277434378862381, 0.37193819880485535, 0.3830935060977936, 2.444047689437866, -0.5121940970420837, 0.5145220756530762, -0.46843674778938293, -1.5364569425582886, -0.8050934076309204, -0.8907255530357361, -0.5085228681564331, -0.44026777148246765, 0.18992429971694946, 1.0854182243347168, -0.9591260552406311, 1.2955338954925537, 0.382272869348526, -0.6296915411949158, 0.2596298158168793, -2.3096723556518555, 0.7593284845352173, 0.4256761372089386, -0.09637412428855896, 0.3556787967681885, 0.36906513571739197, 0.4597862958908081, -1.0823918581008911, 0.510887086391449, -1.7163572311401367, 1.4086657762527466, -0.9822536110877991, 0.36266401410102844, 0.2542296350002289, -0.0739021748304367, 0.03374353051185608, 0.1438741832971573, 0.5998683571815491, 1.2790029048919678, 0.6513908505439758, 0.5136916637420654, 0.10116036981344223, -1.263114094734192, 0.7065029740333557, -1.1583362817764282, 0.9882947206497192, -0.9699745774269104, -1.0912261009216309, -1.3073744773864746, 0.8571984767913818, 0.6937869787216187, -0.8735334873199463, -0.34841033816337585, 0.09073598682880402, -1.107497215270996, 0.86024010181427, -0.5352698564529419, -0.3187907636165619, 1.6879298686981201, -1.071757435798645, -0.9225983023643494, -0.7603821158409119, 0.46865561604499817, 1.0038111209869385, -0.4184713363647461, 0.5392209887504578, 0.0663595199584961, -0.4469471573829651, 0.5108164548873901, -0.3952580988407135, 0.8836550116539001, 1.4508517980575562, 2.6188302040100098, 0.08051303029060364, 0.6831613779067993, -0.5794587135314941, -0.3476993441581726, 0.12504273653030396, 0.7832809090614319, 0.7919312715530396, 1.0542912483215332, 0.43976524472236633, -0.6178348064422607, -0.5240859389305115, 0.826621413230896, -1.1617227792739868, -1.2280956506729126, -0.8867942690849304, 1.497010588645935, -0.06254090368747711, 1.298589825630188, 0.7017765045166016, -0.2391694337129593, 0.48425599932670593, 1.101050615310669, -0.6576858758926392, -0.49323052167892456, -0.539530873298645, -1.2500758171081543, 0.9274849891662598, -0.0015699958894401789, 0.3519015312194824, -0.5665712952613831, -0.8870726823806763, -0.0501219667494297, -0.07331793010234833, 0.5335183143615723, 2.2651243209838867, -0.04737716168165207, -2.0831871032714844, -0.31475427746772766, 0.7456598877906799, -2.2616512775421143, -0.2690652012825012, 0.9221111536026001, 1.6260384321212769, -0.1736142337322235, 0.4156320095062256, 0.1949922740459442, -0.7259188294410706, 0.14422178268432617, -0.8690965175628662, 0.6334943771362305, 1.0417388677597046, 0.9421160817146301, 0.9116653800010681, 0.8427503108978271, 0.04385954886674881, 1.4683011770248413, 0.09428195655345917, -0.7267112135887146, -0.5136533975601196, -0.9697613716125488, 1.1297826766967773, -0.9927712678909302, 0.36735406517982483, -2.7087655067443848, -1.4643232822418213, -1.4396960735321045, 1.9012722969055176, 0.8106239438056946, 0.9542829990386963, 1.2874473333358765, -1.4137762784957886, -1.22447669506073, 1.4682689905166626, 0.13473699986934662, 0.9370341300964355, -0.005200810730457306, 0.3219549059867859, 0.2939088046550751, -2.592278003692627, 0.7950312495231628, 0.8263270854949951, 1.2771360874176025, -0.6159765124320984, 0.4479588270187378, -0.4765285551548004, -0.1927366852760315, 0.2816897928714752, -0.6786518096923828, -0.42905452847480774, -0.6123262643814087, -0.837908923625946, -0.44962725043296814, 0.34862273931503296, 0.345027893781662, 0.03642328828573227, 0.27800580859184265, 1.0300178527832031, -0.007237700279802084, 1.8373138904571533, -0.9686111211776733, 0.46345433592796326, -1.591918706893921, 1.2856334447860718, -0.4555495083332062, -0.21924258768558502, 1.534753680229187, -0.061104074120521545, -0.941238284111023, 0.004260870162397623, 0.2589400112628937, 1.3470324277877808, 0.5050329566001892, 1.2959787845611572, -1.1311646699905396, -0.011328458786010742, -1.3257958889007568, -1.7328438758850098, -2.079688549041748, -0.36248302459716797, -1.2726553678512573, 0.6022204160690308, -0.6438072919845581, -0.1792106032371521, -0.3597051501274109, 0.5278380513191223, 1.242431879043579, 0.258198618888855, -1.101946234703064, 1.0788650512695312, 0.09708882123231888, 1.0427021980285645, -0.2921029031276703, -1.8593096733093262, 1.1571017503738403, 1.0079829692840576, -0.4348549544811249, -0.19123193621635437, 0.49562111496925354, -1.3331259489059448, 0.33565661311149597, -0.02237013354897499, -1.1198041439056396, -2.4986140727996826, 0.9185237884521484, -0.1505274474620819, -1.1543446779251099, 1.1559244394302368, -0.22482921183109283, 0.8034346103668213, -1.1577880382537842, 0.9987953305244446, -0.009806125424802303, -0.06394291669130325, -1.4920194149017334, -1.5771723985671997, -0.7409864664077759, 1.1249135732650757, -1.1621131896972656, 1.490694522857666, 0.8811497092247009, -0.9210647940635681, -0.36035245656967163, 1.741632342338562, 0.6660293936729431, -0.7779335379600525, -0.9642776250839233, 0.4773580729961395, 0.05950116738677025, -1.7079464197158813, -1.4875752925872803, -1.1283103227615356, -0.08877406269311905, -1.2438997030258179, -0.028196362778544426, 1.2298481464385986, 0.5971827507019043, -2.158961772918701, 0.8070045113563538, 1.6313508749008179, -1.5019757747650146, 1.7034912109375, -0.17919130623340607, 1.9111640453338623, 0.2822361886501312, 0.6428961753845215, -0.07995838671922684, -1.5463552474975586, -0.6315100193023682, -0.07006040960550308, 0.33435583114624023, -0.08334878832101822, -0.5769106149673462, -0.2751202881336212, -0.6839024424552917, 1.1380088329315186, 1.1153260469436646, -0.6367290616035461, 1.2359917163848877, 0.3501288890838623, -1.0799962282180786, 2.0075466632843018, -0.864730179309845, -0.47776374220848083, -0.750907838344574, 0.20765537023544312, -0.7964508533477783, -0.7264281511306763, 0.45020103454589844, -1.1925047636032104, -0.4543500244617462, 0.29428842663764954, 1.2624714374542236, -0.8701286315917969, -1.1331173181533813, -1.3433117866516113, -0.14179320633411407, -0.9430662989616394, 1.7655527591705322, -0.7611019611358643, 0.214908167719841, -2.955152988433838, -0.04056169465184212, -0.26529574394226074, -0.22901490330696106, -1.8395304679870605, 0.11374112218618393, 1.0561949014663696, 0.045598555356264114, -0.19573189318180084, -0.5344285368919373, 1.86347234249115, -0.3886912167072296, 2.8412041664123535, -1.7227145433425903, -1.0195634365081787, -0.6516352295875549, 0.8203321695327759, 1.0827000141143799, 0.06798902899026871, -0.911705493927002, -0.03371414169669151, 1.4962588548660278, 0.4963945746421814, -0.7789154648780823, -0.9654781222343445, -0.4325674772262573, 1.8804837465286255, 0.45803773403167725, -1.327480673789978, 0.3839111626148224, -0.11749622970819473, -0.6435763239860535, 0.22030431032180786, 0.0913393571972847, -0.5780985355377197, -0.3331524431705475, -0.13039490580558777, 1.3342247009277344, -1.4728080034255981, 0.6000468730926514, -1.737240195274353, -1.6499003171920776, 0.292524129152298, -0.6588519215583801, 0.05077040195465088, -1.0560160875320435, -0.15704718232154846, -0.40286681056022644, -0.7085074782371521, 0.14733724296092987, -1.4060044288635254, -0.4567084014415741, 0.11680062860250473, 0.07893173396587372, -0.019684210419654846, -0.6733061671257019, -2.297307014465332, -1.3517595529556274, 2.7048373222351074, -0.47127246856689453, -0.04793572798371315, -1.2211300134658813, 0.060263752937316895, 1.282596230506897, 1.3394461870193481, -0.960294246673584, -0.9623906016349792, -0.9578359127044678, -0.5068005323410034, 0.37999504804611206, -1.922471046447754, -0.6964963674545288, 1.1643515825271606, 0.5438210964202881, -1.2783063650131226, -0.9124985337257385, -0.40630045533180237, -1.0150120258331299, 0.06720678508281708, -0.29208704829216003, 1.2912229299545288, -0.06785470247268677, 0.796951413154602, 0.7537355422973633, 1.1603972911834717, -0.8287763595581055, 0.18742133677005768, 0.9035577774047852, -0.04607034847140312, 0.7381923198699951, 2.8539834022521973, 0.09627602994441986, -1.2460527420043945, 1.032822847366333, -1.8523657321929932, 0.8732582926750183, 0.17623412609100342, -0.8379901051521301, -0.03486614301800728, 0.8767430782318115, 0.49571141600608826, -1.6959049701690674, -0.21066917479038239, -1.5357811450958252, -0.45282241702079773, -0.36652326583862305, -0.5447336435317993, 0.858417272567749, -0.25404196977615356, 0.05119919776916504, 1.2500394582748413, -0.4922957718372345, -0.3739570677280426, -1.7708337306976318, -0.2190171182155609, 0.23287248611450195, 0.07665401697158813, -0.8378704786300659, -1.174723744392395, 0.23795083165168762, 0.1953253597021103, -0.5788202881813049, -0.6918864846229553, 0.5023963451385498, 0.3126760721206665, 0.1882922649383545, 0.5740747451782227, 0.5458557605743408, 0.4060251712799072, -1.4100556373596191, -1.0572826862335205, -0.18679362535476685, -0.8162448406219482, -0.020394541323184967, 1.2983750104904175, -0.3737228810787201, -0.5313106179237366, 0.6024513840675354, 0.16412733495235443, 0.013251804746687412, -2.1622257232666016, -1.3622022867202759, 1.1835774183273315, 0.016853777691721916, -1.4044365882873535, -0.03825198486447334, -0.09142494201660156, 0.9805426001548767, -0.718744695186615, -0.28411605954170227, -0.8839261531829834, 1.0396937131881714, -0.29405322670936584, 0.8792980909347534, 1.712863802909851, 0.5576134920120239, -1.4992308616638184, -0.5808854699134827, -0.7117598652839661, 0.9963369369506836, -0.13399681448936462, -2.8044848442077637, 0.48907947540283203, -0.04381901025772095, -0.056085579097270966, 0.31877243518829346, 0.36491817235946655, 0.4874630272388458, 0.9121775031089783, 2.6377928256988525, 0.12127517908811569, -0.36536672711372375, 0.911965012550354, -2.0986649990081787, -0.7607593536376953, 0.8255294561386108, -0.008067200891673565, -2.080510139465332, -1.0427697896957397, -1.7474961280822754, -0.3818056285381317, -1.593252182006836, 0.2726130187511444, -0.7169522643089294, 0.1477975845336914, 0.6293161511421204, -1.1455974578857422, -1.6297768354415894, 0.3903236389160156, 1.0041636228561401, 0.08343120664358139, 0.4576870799064636, 0.005814594216644764, 0.05773374065756798, 0.10122233629226685, 0.701591968536377, -2.04034686088562, -0.12115250527858734, 0.3690630793571472, -0.9560480713844299, -2.006512403488159, 1.4001598358154297, -0.2301718294620514, 1.0766828060150146, -0.3909705877304077, -0.9230095744132996, 0.05509160831570625, -0.7301497459411621, -0.19257085025310516, 1.528710126876831, 1.1848020553588867, 1.8527098894119263, 0.5127982497215271, 0.04674099385738373, -1.4947212934494019, -0.5209236145019531, 0.6687670946121216, 0.7514887452125549, 0.32222193479537964, 1.4575361013412476, -0.09905348718166351, 0.7980494499206543, -0.5173065662384033, -0.9081584811210632, -0.6239761710166931, 1.5739951133728027, -0.7320421934127808, -0.13645786046981812, -1.3264942169189453, -0.45617949962615967, -0.16480903327465057, -1.7672418355941772, 0.10044650733470917, -0.35380828380584717, -0.43700075149536133, -1.3757741451263428, -0.7045530676841736, -0.3141283094882965, 0.9061629176139832, -0.9362956881523132, -1.3740829229354858, -1.143760085105896, 0.6025569438934326, 0.41374629735946655, -2.1180126667022705, 0.5635497570037842, 1.2118616104125977, -0.21279489994049072, -0.1950172334909439, 0.9456514120101929, -0.41334784030914307, 0.867452085018158, 0.9680746793746948, -1.3639342784881592, -1.9451234340667725, 0.797466516494751, -3.6050257682800293, 1.7491482496261597, 0.16600321233272552, 0.7052409052848816, 0.40777313709259033, -1.9649310111999512, 0.4218951165676117, 0.8306079506874084, -0.5455706119537354, -0.6774984002113342, -1.6001046895980835, 0.2979709208011627, -0.15650299191474915, 1.157559871673584, 1.747517466545105, 0.22653521597385406, 0.616977334022522, 1.2672094106674194, 0.5277664065361023, -0.34372973442077637, -0.4674485921859741, 0.6910893321037292, -1.4793075323104858, -1.3754595518112183, -0.37590116262435913, -0.7036861181259155, 0.1759364753961563, -0.3332538604736328, 1.1152046918869019, 0.3578732907772064, 0.7434346675872803, 0.8552640676498413, 0.7243843078613281, 1.4795596599578857, 0.820827841758728, -1.7348662614822388, 0.769045352935791, -1.2609277963638306, 1.3887418508529663, -0.028455371037125587, -1.187790870666504, 0.4544035792350769, -1.9426079988479614, -0.7286720275878906, -1.9038548469543457, -0.4908887445926666, -0.5183810591697693, 0.9038714170455933, 0.20836418867111206, 1.0681952238082886, 1.1645188331604004, -0.06163971871137619, -1.051113486289978, -0.41319015622138977, 0.26099106669425964, -0.9916722178459167, -0.45018190145492554, 0.58714359998703, 0.10190846771001816, 1.5228614807128906, 0.6955147981643677, 0.3541276454925537, -2.62253737449646, 2.1779234409332275, -0.26853689551353455, 1.1845715045928955, 1.062600016593933, 0.13255198299884796, -0.12378337234258652, -0.14442114531993866, 0.3755668103694916, -0.012207976542413235, 1.0760656595230103, -0.4507424235343933, -0.49217158555984497, 0.011041335761547089, 0.4207727909088135, 1.0862922668457031, -1.2458552122116089, 1.1602402925491333, -0.8982157707214355, 0.8562173843383789, 1.3785510063171387, -0.16445553302764893, -1.3791760206222534, 0.4848516583442688, 1.1240652799606323, -0.38205721974372864, 0.7657086849212646, 1.645449161529541, 1.2799785137176514, 0.6780264377593994, -0.5727615356445312, 0.19223719835281372, 0.827437162399292, -0.3770453631877899, -2.2660181522369385, -0.044101156294345856, -1.955673098564148, -2.0954205989837646, 0.655921995639801, -0.4847368597984314, 0.08307967334985733, -0.15014004707336426, 0.5790199041366577, -0.18945208191871643, -0.21725252270698547, 0.5655168294906616, -0.371957391500473, 1.5814391374588013, -0.907433807849884, -0.04524017497897148, 0.3063531219959259, -2.123277187347412, -0.9219202995300293, 0.05527245253324509, -0.28661051392555237, -0.008343681693077087, 0.7276906371116638, -0.5306196212768555, -0.8202002644538879, 1.0033042430877686, 0.6380980014801025, 1.4198248386383057, -0.3263999819755554, -0.73885178565979, 0.10202133655548096, -0.728073239326477, -1.820447564125061, 0.8298434019088745, 0.13129623234272003, -2.351912260055542, -0.5337193012237549, 0.7280135750770569, -1.7569386959075928, 0.2606801688671112, 0.7305018305778503, 0.816794753074646, 1.7197262048721313, -1.2315254211425781, -0.027196122333407402, -2.5522735118865967, -1.138458490371704, 1.697575330734253, 0.12355658411979675, 1.3413586616516113, 0.39967167377471924, -1.007149577140808, 1.0910433530807495, -1.8298091888427734, -2.3611161708831787, -1.49140465259552, -0.7598448395729065, 0.8183822631835938, 1.5444549322128296, -1.5719735622406006, 1.2671977281570435, -1.3187278509140015, 1.2049084901809692, 0.026161104440689087, 0.016673428937792778, 1.6467022895812988, -1.9436006546020508, 0.15032801032066345, 1.205522894859314, 1.2341235876083374, 0.49701589345932007, 0.25225579738616943, 1.1672691106796265, 0.7776245474815369, -0.012246903963387012, 2.242932081222534, -0.4748115837574005, -0.024880532175302505, -0.7294716238975525, -0.07826469838619232, 0.7043412923812866, -1.7384412288665771, 0.12477727979421616, 0.9081504344940186, -1.0393012762069702, -0.6163778305053711, -1.004095196723938, -0.13032333552837372, 0.9779929518699646, -2.098647117614746, -0.6259951591491699, 1.2115833759307861, 0.47066330909729004, 0.2559477686882019, -0.4850853681564331, -0.03730727359652519, 0.4708649516105652, -1.4508436918258667, 0.011437135748565197, -0.40816566348075867, -1.0336029529571533, -0.6337153315544128, 0.010143596678972244, -0.79051274061203, 0.18794980645179749, -0.7256420254707336, 0.7101186513900757, 0.17862580716609955, -1.5522433519363403, 1.9389047622680664, 0.7225255370140076, -0.5178965330123901, 0.7142224311828613, -0.10820204764604568, 0.7700311541557312, -1.323006272315979, -1.0255824327468872, 0.9919298887252808, -0.4445996582508087, 1.2125461101531982, -1.1847902536392212, -0.33848655223846436, -1.328648567199707, 0.799993634223938, -1.3392125368118286, 0.8125675320625305, 2.391529083251953, 0.6343094706535339, 0.1638937145471573, -0.44352301955223083, -1.1019173860549927, 0.15830855071544647, 0.3609362244606018, -0.7457343339920044, 0.2571450173854828, 0.6528922915458679, -0.22542959451675415, 0.603097140789032, -0.49941444396972656, -0.533808708190918, -1.3102375268936157, -0.08659101277589798, -1.1233316659927368, -0.7135087251663208, -0.6755967140197754, 0.1573830544948578, -0.5930871963500977, -0.5712994933128357, -0.2723904848098755, -0.8095344305038452, 1.0805505514144897, 0.2293524295091629, -0.6223368048667908, -0.34900444746017456, -0.20061588287353516, 0.6928750276565552, -1.2319514751434326, 1.1845905780792236, -0.9231524467468262, 0.5884446501731873, -0.45183074474334717, -0.12732936441898346, -1.149108648300171, -1.6896891593933105, 1.0780977010726929, 2.54469633102417, -0.14530697464942932, -0.287760466337204, 1.1514335870742798, -1.1830294132232666, -0.64075767993927, -1.432827115058899, -0.07154230028390884, 0.17716698348522186, -0.2765266001224518, -0.03241759538650513, -0.10216136276721954, -0.03806133195757866, -0.17290650308132172, -0.8832789659500122, 0.0787142962217331, -1.8998464345932007, 0.4026024043560028, 1.034903645515442, 0.9232161641120911, 0.8010949492454529, -1.3808479309082031, 1.0238573551177979, 0.18332281708717346, 0.8711326122283936, 1.260554552078247, -0.17418819665908813, 0.7480757832527161, 3.338923931121826, 0.5573837757110596, 0.587913990020752, 0.34007519483566284, 3.968355417251587, -0.20206201076507568, -0.151131272315979, -0.4754685163497925, -0.0832594558596611, 0.7065419554710388, 0.9611454010009766, -0.4259316325187683, -0.4650235176086426, 0.18852511048316956, 0.1577892154455185, 1.0117586851119995, 0.13573598861694336, 0.029228009283542633, -0.06677618622779846, 0.270364910364151, 0.6428700089454651, -0.7642784118652344, -1.0437496900558472, -0.7009496688842773, 0.8687663078308105, -0.1649598926305771, 1.9494973421096802, -0.9984679222106934, 0.8735995888710022, 0.32446467876434326, 1.0679689645767212, -0.29198122024536133, 0.4943341910839081, -0.7009904384613037, 0.4074971675872803, -0.7718216776847839, -0.21034470200538635, 0.13812346756458282, 0.708451509475708, 0.917224109172821, 0.3891808092594147, -1.0216047763824463, 0.2871192693710327, 1.8700538873672485, 1.03787100315094, 0.31492605805397034, -2.305647850036621, -0.835473358631134, 0.40200477838516235, 1.009490966796875, 0.3602944016456604, -0.439531147480011, 0.9515970945358276, 0.4854256808757782, 0.573765754699707, 1.0581732988357544, 0.1646382063627243, -0.5502644181251526, 1.0707154273986816, -0.41585904359817505, 0.06690645217895508, -0.03438575193285942, -0.2770811915397644, 0.8568987846374512, 0.5553997755050659, -1.125033974647522, 0.7388162612915039, 0.36802536249160767, 0.4637252986431122, 0.4639287292957306, -0.48023512959480286, 2.0393083095550537, 0.2429109811782837, 0.1437492072582245, 1.5084707736968994, 1.611984372138977, 1.1799174547195435, 0.6958394050598145, -1.260289192199707, 0.25596827268600464, 0.4937916398048401, -0.1835627406835556, -0.2920816242694855, -0.41754859685897827, 1.5286844968795776, -1.0609335899353027, -1.2114413976669312, 0.580157995223999, 1.772514820098877, 0.7484773993492126, 1.7669559717178345, -0.048412613570690155, -0.6299747824668884, -0.09052132815122604, -0.6831052899360657, 1.115893840789795, 0.33105728030204773, -0.8297509551048279, -0.21170784533023834, -0.9007200598716736, 0.24400024116039276, -0.088206447660923, -1.1049147844314575, -0.5593584179878235, 0.7297354936599731, 0.471049040555954, -0.12789592146873474, 1.6892958879470825, -2.642669439315796, -0.7815786600112915, 0.40137019753456116, -0.42286217212677, 1.2291218042373657, 0.2095564305782318, -0.20490868389606476, 0.8866291046142578, -0.24262234568595886, -0.0562652051448822, -1.4326967000961304, -1.885623574256897, -0.048398833721876144, -0.6029481887817383, 0.2054186761379242, -1.008498191833496, -0.2970976233482361, 0.13276374340057373, -2.0885324478149414, -0.11773043125867844, -0.6859342455863953, 0.0025329943746328354, 0.24747434258460999, -0.5311120748519897, 0.25802716612815857, 0.8294476866722107, 0.5094332695007324, -0.5289991497993469, 1.076048493385315, 1.248603343963623, -0.9243123531341553, -0.5063446760177612, 0.40162304043769836, 0.6461257934570312, -0.17643308639526367, -0.5075081586837769, 0.15106360614299774, 0.33822059631347656, -0.7049597501754761, -1.2421149015426636, 0.6919867992401123, -0.2572546601295471, 1.276816725730896, -0.06766310334205627, -0.8239405751228333, -0.21692274510860443, -0.3077172338962555, 0.9263436794281006, 0.8140562772750854, -0.8720558881759644, 0.7448770999908447, 0.7843313217163086, 0.43487972021102905, 0.8036200404167175, 0.6745510697364807, -0.7404943108558655, -0.08806933462619781, -1.0598697662353516, -0.9922084212303162, -0.25011250376701355, 0.8775065541267395, -0.13898904621601105, 0.3063294291496277, 1.8610118627548218, -0.954357385635376, -1.3872449398040771, 0.4841466546058655, 1.990976333618164, 1.079311728477478, 0.121919184923172, 0.6863111853599548, 1.6013392210006714, 0.05772240459918976, 1.793607473373413, -0.9881116151809692, -0.232183575630188, -3.0524749755859375, -1.1646977663040161, 0.4692525863647461, -1.6925415992736816, 0.2870123088359833, 1.7723956108093262, -1.0351310968399048, -1.9944217205047607, -1.4218260049819946, -0.6965137720108032, 0.40690234303474426, -0.6286516785621643, -0.19489574432373047, 0.8085729479789734, -0.001125218695960939, 1.3095773458480835, -0.783482551574707, -1.3754589557647705, 1.0767693519592285, 0.5424501895904541, 1.0540870428085327, 0.11995000392198563, 1.5549437999725342, -0.39519932866096497, 0.4011303186416626, 0.9372122287750244, 0.5597001910209656, -0.3620506227016449, -1.7079838514328003, 2.6945266723632812, 0.767169177532196, 1.4991998672485352, -1.4639253616333008, 0.1543462723493576, -0.11713618040084839, 1.824951410293579, 1.1655850410461426, -0.4496564567089081, 0.40586671233177185, -0.5080492496490479, 1.9193652868270874, -0.3777678608894348, -0.26714423298835754, 0.7764056324958801, 2.3431527614593506, -1.0652414560317993, -2.2613842487335205, -0.4328010678291321, 1.8904887437820435, 0.3387109041213989, 0.8712911009788513, 1.4527372121810913, -0.3976190984249115, -0.1953316181898117, -1.1960042715072632, -0.7176258563995361, 0.016521062701940536, 1.4321308135986328, -0.4881911277770996, -1.006055235862732, 0.22619858384132385, 0.11499269306659698, -0.5172092318534851, -0.4090580642223358, 0.8477521538734436, -0.5296114087104797, -0.059694088995456696, -0.7042004466056824, -0.2408059686422348, -0.605685830116272, 1.478848934173584, 1.7076610326766968, 0.32323387265205383, 1.2114702463150024, 0.5349903702735901, -1.1805412769317627, -0.5216652154922485, -1.2873224020004272, 0.15992726385593414, 0.9269995093345642, 0.5851327776908875, 0.6413149833679199, 0.6415017247200012, 0.6434437036514282, -0.10553814470767975, -0.7867740988731384, -0.3400789797306061, -0.1148720532655716, -0.9467941522598267, -1.602152943611145, 0.8324711918830872, 1.2940980195999146, 0.44361305236816406, -0.2569657862186432, 0.4308435618877411, 0.4169357419013977, -1.506758689880371, 0.6135990023612976, -0.3196931481361389, 0.3106381297111511, -0.04643451049923897, 0.6089039444923401, 0.3802172839641571, -0.37354132533073425, 0.25638532638549805, -0.4991321861743927, -0.45023149251937866, 1.690482497215271, 0.20968833565711975, 0.962535560131073, 0.6831072568893433, -0.22057963907718658, -0.19042035937309265, -0.31123965978622437, -0.006862556096166372, 0.9343039989471436, 0.5727782845497131, 0.14118921756744385, 0.7389572262763977, 0.6245574951171875, 1.3699973821640015, 1.3502178192138672, -0.4803457260131836, -0.315911203622818, -0.7717182636260986, -1.3177850246429443, -1.072615146636963, -0.9826127886772156, 2.1582610607147217, -0.8623334169387817, -0.065397247672081, -0.8790097236633301, -1.0796160697937012, 1.4088871479034424, -1.6300102472305298, -0.23428218066692352, -1.0578022003173828, 0.07850488275289536, 0.3695477545261383, -0.5903096199035645, -0.9785766005516052, 1.2701562643051147, 0.7032850384712219, 0.18384107947349548, -0.003311139764264226, 0.11244644969701767, 0.19300317764282227, 0.2678792476654053, 0.5469319224357605, -0.7673959732055664, 0.811886727809906, -0.4273639917373657, 0.5758634805679321, -0.5912985801696777, -0.3495570123195648, 0.7634575963020325, -0.7983676195144653, 0.22321635484695435, -0.3191712200641632, 0.4026392996311188, -0.04457587003707886, -1.2296031713485718, 0.014862368814647198, -0.9647474884986877, -1.0727038383483887, -1.0431488752365112, 1.0737119913101196, -0.3100941777229309, 0.7913033962249756, 0.061015043407678604, 2.045785665512085, -1.1607153415679932, 1.0145983695983887, -0.5432142615318298, 0.9549553394317627, -2.2420012950897217, 0.07042276859283447, -2.5665175914764404, 0.5366579294204712, -0.18178395926952362, -0.04919830337166786, 1.309038758277893, -0.20129089057445526, -0.6511684060096741, 0.7336066961288452, 0.023899702355265617, -1.0986342430114746, 0.9631993174552917, 0.3583814203739166, -0.019712021574378014, 0.2312752604484558, -0.7814778089523315, -0.6450731754302979, -0.5366869568824768, 1.0682108402252197, 3.2797040939331055, 0.20058029890060425, 0.7327060103416443, -2.8278889656066895, 1.0837911367416382, -0.342345654964447, 0.8475401401519775, -0.9306915998458862, 1.067720651626587, 1.1072909832000732, -0.04184143617749214, -0.6919196844100952, -1.7212750911712646, 0.27442359924316406, -1.2424629926681519, 0.20853601396083832, -0.12390408664941788, 0.49139150977134705, -1.5239200592041016, -1.1340197324752808, -0.29797545075416565, -0.7156555652618408, -0.7499483227729797, -0.6530573964118958, -0.3453516364097595, 1.1709071397781372, -0.4299684464931488, -0.170381098985672, -0.08681806921958923, 0.4252432882785797, 1.1439005136489868, -0.6313614845275879, -0.345430463552475, 0.8752110004425049, -0.44616177678108215, -0.9839706420898438, -0.529697060585022, 1.0064753293991089, 0.3687518537044525, 0.9442113041877747, 0.7943130135536194, -0.49333205819129944, -0.19596047699451447, -0.08197371661663055, 0.9771077036857605, -0.6544579267501831, -0.3424241244792938, 0.820277988910675, 0.40197816491127014, 0.058991383761167526, 1.140883207321167, -1.2636966705322266, 0.33701375126838684, 0.2672714591026306, -0.768169641494751, 0.6540659070014954, 1.0419716835021973, 0.8173383474349976, 0.7495734691619873, 1.9691892862319946, -0.5067136883735657, 0.7572440505027771, -0.8143046498298645, 0.4570324420928955, 0.7221284508705139, -0.7824488878250122, -0.10118725150823593, 2.4352972507476807, 0.12731391191482544, -1.6415777206420898, 1.0443699359893799, 0.55255126953125, -0.04771984741091728, -0.16448257863521576, 0.7407671809196472, -0.4074711203575134, 1.5437796115875244, 2.103825569152832, 1.784839391708374, 0.3514816462993622, -1.180670976638794, -0.9617123007774353, -0.2782384157180786, 0.33712539076805115, -0.8546244502067566, 0.7304152846336365, -0.09370580315589905, -1.8869324922561646, -0.44827592372894287, -0.8790234327316284, -1.1808185577392578, 1.2207833528518677, -0.825072705745697, 0.5848903656005859, -1.6277655363082886, -0.3072262704372406, 1.9847465753555298, 0.17349569499492645, 1.6142430305480957, -0.377235472202301, 1.6799066066741943, 1.9216982126235962, -0.99268639087677, -0.6586913466453552, 0.06507616490125656, 0.2313365489244461, -0.8973067998886108, 0.45200225710868835, 1.1233344078063965, -0.01887449249625206, -0.25579577684402466, 0.43596795201301575, 0.631637692451477, -0.19943757355213165, 0.00122295250184834, -1.9355499744415283, -0.8219223022460938, 0.22322797775268555, -0.6889876127243042, 0.8208670616149902, -1.7803962230682373, -0.8074460625648499, 0.5689757466316223, -0.9038845300674438, 0.1169118732213974, -0.06941718608140945, -0.22977009415626526, 0.6289264559745789, -2.004575490951538, 0.4430508613586426, -0.9795153141021729, 0.3227458894252777, 1.2148267030715942, -0.4886944890022278, 0.5091307163238525, -0.22709804773330688, -0.10020250082015991, 2.0309526920318604, -0.566428542137146, 1.6831599473953247, -1.2324172258377075, -0.10217246413230896, 0.040327854454517365, 1.3155643939971924, 2.057283639907837, -1.4533873796463013, 1.3582403659820557, 0.7844721674919128, 0.8616398572921753, -0.3814275860786438, -0.8526825904846191, -0.9070976376533508, -0.9732467532157898, -1.3209640979766846, 1.1161811351776123, -0.08556026965379715, -0.17005763947963715, 0.20660556852817535, -0.3623247742652893, -0.08308231085538864, 0.2192145586013794, -0.12915141880512238, 0.36278006434440613, 0.215720072388649, -1.3946200609207153, 0.349965900182724, -0.16868631541728973, 0.6781805157661438, -1.3094217777252197, -1.1627272367477417, -0.9745100736618042, -0.8864337801933289, 0.324837327003479, 0.4015347361564636, -0.12529802322387695, -1.7137490510940552, -0.6091347932815552, 1.7235007286071777, -0.13651689887046814, 0.6285420656204224, 0.6888338923454285, -2.0380470752716064, -2.094309091567993, 0.8683907985687256, 0.4847133159637451, 1.7156637907028198, -1.4514044523239136, 0.026322979480028152, 2.455070972442627, 1.1624324321746826, -1.4709926843643188, 0.6353763341903687, 2.0172815322875977, -1.1422239542007446, -0.4927757680416107, 0.2002250999212265, -2.0334720611572266, -0.754154622554779, -0.12756231427192688, -1.5649828910827637, 0.30351579189300537, -1.475495457649231, 1.1365349292755127, -0.09677095711231232, 1.1024413108825684, 2.7994439601898193, 1.5375618934631348, -0.5315484404563904, 0.2244689166545868, -1.0699259042739868, -0.2883666157722473, -0.6889399290084839, 0.8491356372833252, 0.5050027966499329, -0.019842609763145447, 0.5661510229110718, -0.4726688265800476, -0.12254145741462708, 1.7318681478500366, -0.7519740462303162, 0.4805692732334137, -0.11082794517278671, 0.5983771681785583, 0.48099181056022644, 0.15595592558383942, 1.4688366651535034, -0.7586562633514404, 0.17372490465641022, -0.8475700616836548, -0.7399791479110718, -1.7081420421600342, 1.8148680925369263, 0.04445866867899895, 1.241249918937683, 0.9700528979301453, 0.22139045596122742, -1.5964491367340088, -0.724746823310852, -0.1473066508769989, -1.4198863506317139, 0.3725168704986572, -0.9088645577430725, -1.000920057296753, 0.43498852849006653, -0.057025521993637085, 0.4187013804912567, -1.8971840143203735, -2.168363094329834, 0.9689478278160095, -0.624723494052887, -0.5645477771759033, -0.012862764298915863, 1.4236233234405518, -0.9874209761619568, -0.2259119600057602, 0.30740052461624146, 0.5732948184013367, -1.5300042629241943, 0.08697900921106339, 1.0999352931976318, 2.001767158508301, 0.9815470576286316, -0.5546460151672363, -0.30270111560821533, -0.3121224641799927, -1.0403215885162354, -0.2697193920612335, -1.307534098625183, -1.2190501689910889, 0.5812657475471497, -1.4341957569122314, 0.9571683406829834, 1.2714635133743286, 1.2441208362579346, 0.8476791977882385, -0.017832394689321518, -2.9590415954589844, -0.671723484992981, 0.6740065813064575, 1.3140652179718018, 1.2628999948501587, 0.6209943294525146, -0.08329392969608307, 0.8038002252578735, -1.029392957687378, 0.2677907943725586, -0.06073731556534767, -0.40448811650276184, -0.6217718720436096, -1.4013909101486206, 0.1481074094772339, 0.21090848743915558, -0.8362239003181458, -0.9806524515151978, 1.407887578010559, 0.15333129465579987, 0.27585935592651367, 1.0280500650405884, 0.7976564168930054, 1.0726362466812134, -0.034370243549346924, -0.47164520621299744, -0.5918029546737671, -1.9622195959091187, -0.11081336438655853, 0.48016127943992615, 0.14643193781375885, -0.5830733180046082, 0.9501447677612305, 0.17075392603874207, -0.10549289733171463, -0.5777236819267273, -1.62468683719635, 0.3151370882987976, -1.7995485067367554, -0.264800101518631, -0.45141106843948364, -2.023188829421997, 0.182276651263237, 0.606062650680542, -0.5124270915985107, 0.7877570986747742, 1.2869244813919067, -0.10688920319080353, -0.08141153305768967, 0.7601593732833862, -1.2941396236419678, 0.6579061150550842, 0.8990773558616638, -2.376685857772827, -0.07690703868865967, -0.2200731784105301, 0.5347830653190613, -1.1504830121994019, 1.2434154748916626, -1.1202632188796997, 0.6356456279754639, -0.8996978402137756, 0.9822855591773987, 0.6017348766326904, 0.9445416927337646, 1.0105383396148682, -0.4259563982486725, 0.23298870027065277, 1.4245244264602661, -1.717612862586975, 0.34712663292884827, 1.6013071537017822, 0.6440114974975586, -0.9743542075157166, 1.607682466506958, 0.26388421654701233, 2.090944766998291, 0.5852770209312439, 1.5930612087249756, -1.3198586702346802, -1.0603584051132202, 1.5068508386611938, 0.9991334676742554, -0.7894426584243774, 0.2862178683280945, -0.47787266969680786, 0.616550624370575, 0.9298707246780396, -1.9138877391815186, -0.868536651134491, 0.7945266366004944, -1.4518202543258667, -0.6714974045753479, -1.2924896478652954, 0.4052174687385559, 1.9762269258499146, 1.2308509349822998, 0.5361970067024231, 1.7574551105499268, -1.3249613046646118, 0.5542101263999939, 0.6586819291114807, 2.2825229167938232, 0.02547958865761757, -1.1039289236068726, 0.17053233087062836, 0.031080998480319977, -1.466976523399353, -2.293001413345337, 0.35803523659706116, -0.21494755148887634, 0.387154221534729, 0.3355955183506012, -0.34648165106773376, 0.2541329264640808, -1.5028393268585205, -2.189970016479492, 0.5126978158950806, -1.868079423904419, -0.27747172117233276, -0.8379257321357727, -0.5637223720550537, -1.7375391721725464, 0.7026305794715881, -0.002591405063867569, 0.3448127508163452, 0.6079362034797668, 1.9775073528289795, 1.1483381986618042, -0.6001973152160645, -0.07652720808982849, -2.4710445404052734, 0.7682441473007202, 1.5993775129318237, 0.7439259886741638, -1.323167085647583, -0.8781642317771912, 0.1283496618270874, -0.6815202832221985, -1.3378033638000488, -0.1971287727355957, 0.16676759719848633, -0.9483603239059448, 0.632114589214325, -0.26809847354888916, -0.9164320826530457, 1.5121734142303467, -1.3124759197235107, -0.6894324421882629, 0.19817818701267242, 0.8437450528144836, 0.33406078815460205, -0.32090041041374207, 1.032457947731018, 1.7927778959274292, 0.7327605485916138, -0.7274113893508911, -1.3492239713668823, -0.7204010486602783, 0.6054316163063049, 0.7500150799751282, 0.887811005115509, -0.8662751317024231, 0.5214477181434631, -0.6803848743438721, 1.4002068042755127, 1.0621023178100586, 0.5690821409225464, 0.782281756401062, -0.629951536655426, -0.19483834505081177, 1.2488325834274292, -1.5344237089157104, -0.253844290971756, 0.5739775896072388, -0.17274457216262817, -0.9504216313362122, 1.353297472000122, -0.26097580790519714, 1.6747345924377441, -0.0945754125714302, 1.1415998935699463, -0.7169307470321655, -0.900067150592804, 1.902328372001648, 1.001172661781311, 0.7016537189483643, 0.17540709674358368, 0.1431368887424469, 0.0182062815874815, -2.3327815532684326, -1.176677942276001, 0.7351120114326477, 0.11324544250965118, 0.873855471611023, 0.2747901976108551, 0.5321460366249084, -0.20291107892990112, 0.5433539748191833, -1.2435498237609863, -0.16240786015987396, -2.427109479904175, 0.9041116833686829, 0.3824578821659088, -0.7597759962081909, -1.0723766088485718, -1.369458556175232, 1.8119876384735107, 0.2828678488731384, 1.4884300231933594, -0.19764196872711182, -0.09302181750535965, 0.8787795901298523, -1.3197969198226929, -0.636076807975769, 0.5588521361351013, 1.6802046298980713, 0.826624870300293, 0.8027576804161072, 0.11147490888834, 0.5688650608062744, -0.40440869331359863, 0.7292875647544861, -0.7923184633255005, 0.5716069340705872, -1.7503405809402466, -0.8560893535614014, 0.7932873368263245, 0.5697179436683655, 0.04232161492109299, 0.5378924012184143, 1.1765490770339966, -0.0010148909641429782, -2.315795660018921, -1.2011425495147705, -1.0700427293777466, 0.20092074573040009, -0.12273856997489929, -0.7490494847297668, 0.9371001124382019, 1.0880025625228882, 1.0964957475662231, -2.4483540058135986, -0.3233603537082672, 1.2644634246826172, -0.3272688686847687, -0.21572880446910858, -0.2159436047077179, -1.2813695669174194, -0.015076212584972382, 0.0011511873453855515, -0.2793358862400055, 0.2929436266422272, -0.5294026732444763, -1.9727988243103027, -0.05643844231963158, 0.6720333695411682, 0.09959498047828674, -0.9328195452690125, -0.1611444652080536, 1.0192660093307495, 0.1177566796541214, -0.46575692296028137, 0.2638891339302063, 0.9232906103134155, -0.05316217243671417, 2.0844831466674805, -0.014831784181296825, 0.08811995387077332, 2.428959608078003, -1.3752490282058716, 0.8818254470825195, 1.3141767978668213, 0.621250331401825, -1.315800666809082, 0.16717272996902466, -1.6326271295547485, -3.2613775730133057, -0.7938256859779358, -0.5404395461082458, -0.370893657207489, 0.7990038990974426, -0.42252448201179504, -1.2337985038757324, -0.21089553833007812, 0.11110686510801315, 0.10803485661745071, 0.6499485969543457, -0.72720867395401, 0.5152420401573181, -0.29425129294395447, 0.609601616859436, -0.22983966767787933, -0.3566787540912628, 0.4582440257072449, 1.2831531763076782, 0.5317773818969727, 1.0673096179962158, -0.7754794359207153, 1.65492844581604, 0.7341654300689697, 0.2524334192276001, -0.14239680767059326, 0.16464735567569733, 0.39467713236808777, 0.7569975852966309, -0.26525628566741943, -1.4006919860839844, 1.2438913583755493, 0.9840598106384277, -0.11895566433668137, -0.7576308846473694, 0.018687985837459564, -0.12030427902936935, 1.007419228553772, -0.8212857842445374, -1.312558889389038, 0.043532174080610275, 0.5823488831520081, -1.6479274034500122, 0.4576399028301239, -2.0299742221832275, -0.8501215577125549, 0.5822603702545166, 1.0052851438522339, -2.1782116889953613, -0.7371753454208374, -2.0153868198394775, 1.5050181150436401, 0.37388885021209717, 1.187487006187439, 1.117445945739746, -0.6975427865982056, -0.27955785393714905, -2.0557570457458496, -0.49358081817626953, 1.5656154155731201, -0.3188321888446808, -0.24623443186283112, -0.5118533968925476, -2.363903284072876, 0.6473823189735413, 2.101731538772583, -0.39810624718666077, 0.07907058298587799, 0.1819547861814499, -1.1323236227035522, 0.9753500819206238, -0.5588471293449402, 0.29907798767089844, -0.2750406265258789, -0.4348832368850708, -0.03901226446032524, 0.5178610682487488, -0.48491448163986206, -0.5803884863853455, 1.5692061185836792, -1.3500254154205322, 0.06381426006555557, -0.17710715532302856, 2.0769152641296387, 0.16248458623886108, 0.1882999837398529, -0.22347372770309448, -1.2509949207305908, 1.3998183012008667, -1.416473388671875, -0.2748509347438812, -0.552596926689148, -0.4170476496219635, -0.9265692234039307, 0.20065081119537354, -1.8470823764801025, 1.617943286895752, 0.2590806484222412, 0.7305474281311035, -1.3457164764404297, 0.5201886892318726, -0.3075107932090759, 0.6215985417366028, -1.0052738189697266, -0.9214184880256653, 0.5969310402870178, 0.7502391934394836, -1.6672604084014893, -2.655388116836548, 0.1793457567691803, 0.7574930191040039, -0.6764103770256042, 0.8986802101135254, -1.0602630376815796, 1.746802806854248, 0.27392858266830444, -2.3213186264038086, 0.625266969203949, -1.300521731376648, -1.378818154335022, -1.3823944330215454, -0.6332265138626099, -0.028018424287438393, -1.6336021423339844, 1.0034631490707397, 0.5191741585731506, -0.1920480728149414, -0.23800966143608093, -2.305270195007324, -0.7869473695755005, 1.3569713830947876, -0.7785452604293823, 1.9442812204360962, -1.6868189573287964, 0.27425411343574524, 0.6847130060195923, 1.1987651586532593, 2.1234419345855713, 0.044090427458286285, 0.5982144474983215, 1.152522325515747, -0.6874628067016602, -0.19377242028713226, 0.835629940032959, 0.10847004503011703, -0.588936984539032, 1.9611502885818481, 0.4360007047653198, 0.21367183327674866, -0.9204978346824646, 1.1564972400665283, -0.6309748888015747, 0.3870927095413208, -0.7907886505126953, 0.5922726392745972, 1.6251367330551147, 1.356383204460144, -0.9034298062324524, 0.386855810880661, -0.29442569613456726, -1.5296943187713623, 1.0534615516662598, 0.48154744505882263, -0.8941742181777954, 1.4113409519195557, -0.8799054622650146, -0.32715505361557007, -1.2543044090270996, 0.775127112865448, 1.5970696210861206, 0.49146321415901184, 0.47060009837150574, -0.5391975045204163, -1.2618311643600464, -1.0355085134506226, 1.6140773296356201, -1.6898612976074219, 1.9706737995147705, -0.1873549520969391, 1.3352603912353516, 0.8656368255615234, -0.4377709925174713, -2.008056640625, -0.4823508858680725, 2.250659942626953, 0.6041452884674072, -0.15759487450122833, 0.7998496294021606, 1.8479169607162476, 0.0812356024980545, -0.6262407302856445, -1.125168800354004, -0.44282156229019165, 1.1187145709991455, 0.14473745226860046, -0.6055017113685608, -0.6112403273582458, 0.42813149094581604, 3.121732473373413, -1.5894672870635986, -0.1796342134475708, -0.3669181764125824, 0.16212496161460876, -1.4689674377441406, -0.7388921976089478, -1.9204211235046387, -0.5660420656204224, 1.3124408721923828, -0.5700520277023315, 0.004782041534781456, -0.2228938490152359, 0.4929945170879364, -0.8211942315101624, 1.2299379110336304, -0.6596518158912659, 2.2589011192321777, -1.2347294092178345, 0.7813394665718079, -0.327450156211853, -0.5824837684631348, -0.910079300403595, 0.9362829923629761, 0.4337374269962311, -1.3591989278793335, 0.9669026732444763, 0.5766222476959229, 0.5026208162307739, 0.9552997946739197, 1.2146791219711304, -0.1652366667985916, 0.10810136795043945, -0.635154664516449, 1.1262023448944092, 1.5836286544799805, 1.8955122232437134, 0.40532442927360535, 1.5562025308609009, 0.3986945152282715, 0.11077739298343658, -0.6308097243309021, 0.11247356235980988, 0.023486105725169182, 0.1403803825378418, -2.1260852813720703, -2.370112180709839, 0.5194841623306274, 2.6877493858337402, 2.8419814109802246, -1.684626579284668, 0.43844175338745117, 1.0469346046447754, -0.9296715259552002, -0.06251134723424911, -0.26995447278022766, -1.5976182222366333, 0.27471303939819336, -0.40824711322784424, 0.2946595847606659, -1.5356073379516602, -0.07257755100727081, -0.11092844605445862, 0.017676325514912605, 0.09901576489210129, -1.1552011966705322, 1.3578565120697021, -0.9376163482666016, 2.063524007797241, -0.8604516386985779, -0.22682450711727142, 0.18031743168830872, 0.32534268498420715, 1.1168639659881592, 0.3426973819732666, -0.17545238137245178, -0.06609293818473816, -0.7398141622543335, -0.6333916187286377, -1.622327446937561, -0.7409038543701172, -0.5598217248916626, -1.28070068359375, -2.010213851928711, 0.02246655896306038, -0.07651858776807785, 1.2270803451538086, 0.3175814747810364, -0.5652050375938416, 0.266467422246933, 0.24794118106365204, 0.10766711086034775, 0.7763711810112, -0.15115953981876373, 0.0917995348572731, -0.08969176560640335, 0.4907897114753723, 0.6861802935600281, 0.008252265863120556, 0.14089925587177277, 0.9580357670783997, -1.0215176343917847, -0.9910447597503662, 0.1985705941915512, -0.2575240135192871, 0.9681404829025269, 0.46367213129997253, -1.2491791248321533, -1.8059462308883667, 1.026120901107788, -0.1862444281578064, -1.402442455291748, -0.9642890691757202, 1.0323169231414795, 1.0363705158233643, -0.3916040062904358, -0.30488866567611694, -0.4837529957294464, 0.6672155261039734, 0.18718969821929932, 0.1441587209701538, 0.35338208079338074, -0.7743950486183167, -0.8324325680732727, 0.5164290070533752, -2.148516893386841, -0.6084567904472351, 1.1181124448776245, -0.10490190982818604, 1.4044562578201294, -0.8679647445678711, 0.9506420493125916, -0.014295170083642006, 0.33137965202331543, -1.4779703617095947, -0.8196865916252136, 0.48203375935554504, -0.02473323978483677, 0.7839673757553101, 0.7951988577842712, 0.39135777950286865, -0.11325912922620773, -0.42444437742233276, 0.6787440180778503, -0.9025138020515442, 2.564908266067505, -1.5197014808654785, -0.16979682445526123, -0.6805376410484314, 0.8020190000534058, 0.13371723890304565, -0.6075284481048584, 0.4303421676158905, -1.0978041887283325, 0.5353342294692993, -2.593599557876587, 2.29733943939209, 0.8744508028030396, -0.8485503196716309, -1.768342137336731, -0.5441586971282959, -0.8446369171142578, -0.09786166995763779, 1.1625070571899414, -0.0035058322828263044, 0.35640886425971985, -0.624090850353241, 1.6341698169708252, -0.6421337723731995, 1.1866785287857056, 1.662915587425232, 0.4777289927005768, 0.2446504384279251, -1.7240402698516846, 1.2386451959609985, 0.27996018528938293, 0.9171829223632812, -1.5918350219726562, 0.6508023142814636, 0.5850561857223511, -1.0302941799163818, -0.4127773940563202, 0.008136844262480736, -0.6195080876350403, 0.9460256695747375, -0.22958460450172424, 0.008463200181722641, -0.6040632128715515, -0.23469677567481995, 0.5852044820785522, -0.5903377532958984, 0.6546043157577515, -0.5494821667671204, -1.1374613046646118, -0.24844548106193542, -0.14865373075008392, -2.890397548675537, -0.1145833432674408, 1.086977481842041, -0.529904305934906, 1.2138569355010986, 0.8330361247062683, 1.6868071556091309, -0.9476982355117798, 1.2464061975479126, -1.086045265197754, -0.020142052322626114, 0.016185833141207695, -0.5852130055427551, -0.15222693979740143, 0.7737491726875305, -0.6491885185241699, -1.288839340209961, 1.9846376180648804, 0.07054536044597626, -1.3079861402511597, -0.2549234628677368, -1.3815234899520874, -0.7883042693138123, 0.6475961208343506, 0.096869558095932, 1.8015745878219604, 0.477655827999115, 0.23890259861946106, 1.3133057355880737, 0.0246325321495533, -1.3032588958740234, 0.1996825486421585, 0.3199993669986725, 0.3897337019443512, 0.052256688475608826, -1.2201093435287476, -0.7815815806388855, 0.7248221635818481, -0.106415756046772, 0.9945313334465027, 0.8365854024887085, 0.4924701452255249, 1.1827343702316284, -0.8214060068130493, -0.08705005794763565, 1.3328529596328735, -0.4454673230648041, 1.2279530763626099, -1.3304874897003174, 1.6792601346969604, 1.456243634223938, -1.1029129028320312, 1.404071569442749, -0.21293871104717255, 0.7383221983909607, 0.12993557751178741, 2.3116302490234375, 0.5399674773216248, -0.5574231743812561, -0.708613932132721, -0.6250520348548889, -1.3741768598556519, -2.578547716140747, -1.4603989124298096, -1.0787099599838257, -0.2951805293560028, 1.246797800064087, -0.9456823468208313, -1.103900671005249, 2.100594997406006, -0.8578387498855591, -0.08249308913946152, -1.7467243671417236, -1.6240988969802856, -0.3663533926010132, -0.5845776796340942, -0.27765873074531555, -2.0481300354003906, -0.11576109379529953, 0.6655721068382263, 0.8985653519630432, 1.491139531135559, 1.4161311388015747, -1.5441843271255493, -0.8641128540039062, 0.10919062048196793, -0.7598637342453003, 0.03292660787701607, 0.9923547506332397, 0.7630006670951843, 1.3339731693267822, -0.11391238868236542, -1.7592041492462158, -1.4861781597137451, 0.5464528799057007, -0.7489145994186401, -0.8939939737319946, -1.0960088968276978, -1.6756877899169922, 1.1113662719726562, 0.2320259064435959, 0.5350175499916077, 0.6137889623641968, -1.5972360372543335, 2.051276445388794, 1.36905038356781, -0.3710257112979889, 0.295344740152359, 0.1836964190006256, 0.05489916354417801, 0.24001209437847137, 0.7437126040458679, 1.2245724201202393, -0.1194414496421814, 1.253326416015625, -0.38361257314682007, 1.253082275390625, -1.1837244033813477, -1.0190621614456177, -3.0913138389587402, 0.4818291664123535, -0.22316895425319672, 0.8901706337928772, -0.1899159550666809, -1.206030011177063, -0.06796387583017349, -1.583367109298706, -0.6423448920249939, 1.006026029586792, -0.5477665066719055, 0.0012059964938089252, 1.2366085052490234, -0.5687872767448425, 0.05517221614718437, 0.255051851272583, -0.2438354790210724, -0.772251546382904, 1.3014718294143677, -2.830717086791992, -0.7854228615760803, 1.233168125152588, -1.8938274383544922, -0.08549151569604874, 0.681951105594635, -0.8275243639945984, 0.18056529760360718, -0.8162112236022949, 0.4929847717285156, 0.433836966753006, 0.25365325808525085, -0.6355605125427246, 2.2319447994232178, 1.2564928531646729, 0.5027861595153809, -0.5190286636352539, -1.2715423107147217, -0.962536633014679, 0.9918716549873352, -0.8114931583404541, 0.8354703187942505, 0.08569815009832382, 1.1832901239395142, 1.1116304397583008, -0.5438968539237976, 0.23186081647872925, 1.6102354526519775, -0.8273636102676392, 0.7030254602432251, -0.4781527817249298, -0.005683588795363903, -1.999759316444397, -1.1248923540115356, -0.9699147939682007, 1.26707923412323, -0.23952676355838776, -0.37457579374313354, -0.01578185148537159, -0.2895926535129547, 0.14408843219280243, -0.03586437180638313, 0.9329873323440552, 0.4522206783294678, -0.601874053478241, -0.12852831184864044, 0.6133999228477478, 0.74126797914505, -0.8063650727272034, 0.5476571321487427, -0.08270213007926941, -0.6123700141906738, 0.38255244493484497, 1.8892375230789185, 0.6392987966537476, -0.3588578701019287, -0.7624133825302124, 1.0863866806030273, -0.3697616755962372, -0.1991749256849289, -1.2157683372497559, -0.5045129060745239, 1.1783263683319092, 1.7942886352539062, -0.3852619528770447, -0.548993706703186, 0.8888499140739441, -0.30942583084106445, 0.5454062819480896, -0.023594412952661514, -0.510267972946167, -0.07831712067127228, -1.004608154296875, 0.3386620581150055, 1.0228087902069092, -1.891558051109314, 0.46339285373687744, -1.3841650485992432, 0.10454058647155762, 0.15745508670806885, -0.49677643179893494, 0.4869283139705658, -0.46236467361450195, 0.5006838440895081, -0.8997403383255005, 1.0171327590942383, 1.788312554359436, 0.514200747013092, -0.4872550368309021, 1.357666254043579, 0.9374786615371704, -0.19146904349327087, -0.3705475330352783, -0.08299736678600311, 2.1446409225463867, 0.9516466856002808, 0.37204915285110474, 1.2049885988235474, -1.1671546697616577, 0.45659613609313965, -0.015897711738944054, 1.5907357931137085, -0.3755455017089844, 0.35904499888420105, 1.5023232698440552, -1.0559645891189575, -0.9214328527450562, -0.4217607080936432, 0.4380275011062622, -0.9428448677062988, -0.3352392613887787, 1.5633150339126587, 0.5324977040290833, -0.3059001564979553, 1.0461325645446777, 0.018669631332159042, 0.7508304119110107, -0.6801793575286865, -0.760521411895752, -0.44349226355552673, -0.8530339598655701, 1.3982124328613281, -2.1529436111450195, 0.12248755991458893, 0.3793586790561676, 1.8661749362945557, -0.8814176321029663, -0.6414692401885986, 0.38171711564064026, -0.2528949975967407, -0.769538164138794, -0.8256615996360779, 0.5371773838996887, 0.8868690729141235, 0.2887333929538727, 0.1959410011768341, 0.8290331959724426, -0.9340318441390991, -0.006429822649806738, 0.6655682325363159, -1.0109448432922363, 1.9723310470581055, 0.20152339339256287, 0.6115499138832092, -0.4788563549518585, -1.3349032402038574, 0.7385423183441162, -1.1086722612380981, -1.2730388641357422, -0.3661770522594452, 1.2476418018341064, -0.7168964147567749, -0.19259637594223022, 0.9199299216270447, -0.5239428281784058, -0.6031925678253174, -1.0569804906845093, 0.5181869268417358, 0.5886720418930054, 1.019381046295166, -0.27677980065345764, 0.14722387492656708, 0.7170937061309814, 0.10729629546403885, -0.7190189957618713, 0.7883904576301575, 1.9840201139450073, -0.22506555914878845, -2.020775556564331, 0.1143372654914856, 0.4713665544986725, -0.32970038056373596, 1.183433175086975, -0.7732138633728027, 1.9610741138458252, -0.8431677222251892, -0.44813627004623413, 1.0650173425674438, 0.6638339161872864, 0.40984228253364563, 0.6758763194084167, 0.5609323978424072, -0.6014642119407654, -0.11654024571180344, -1.305765151977539, 1.3428325653076172, 1.0682028532028198, 0.2646908164024353, 0.25602203607559204, 0.34450390934944153, -0.7290108799934387, -1.9931434392929077, -0.10220605880022049, 1.5540051460266113, -0.5998938679695129, -0.8486805558204651, -1.336037039756775, -1.7985498905181885, 1.057479739189148, 0.43555283546447754, -1.6982653141021729, -0.5154027938842773, -0.06480266898870468, -0.4279606342315674, -2.267320394515991, -0.9782005548477173, 1.1731140613555908, -0.8905277252197266, 0.5789186954498291, 0.21424055099487305, -0.6690061092376709, -0.6203683614730835, 0.14219845831394196, 0.48673731088638306, -0.3820957839488983, 0.9468930959701538, 1.670820713043213, 0.8121734261512756, 2.172184705734253, -1.0951716899871826, 0.8381436467170715, -1.4736255407333374, -1.6576756238937378, -0.15969136357307434, 0.5431715250015259, -0.3774612843990326, -0.6221935749053955, 1.176038146018982, -0.23307426273822784, -0.1781710684299469, -1.8327558040618896, 0.22531336545944214, 0.12610363960266113, -1.4338312149047852, -2.6728007793426514, 1.043678879737854, -0.21591448783874512, 0.2579711973667145, 1.747753620147705, -0.030567936599254608, -0.7027407288551331, -0.26369795203208923, -1.1351250410079956, -0.25493142008781433, 0.5825997591018677, -0.8947262167930603, -0.8053326606750488, 0.6540488004684448, 1.1084418296813965, -0.5488654971122742, 1.2483456134796143, -0.21414610743522644, 0.8203731179237366, -1.559280514717102, -1.1002819538116455, -0.39076876640319824, 0.8070275783538818, -0.5103780627250671, -1.4129414558410645, -0.318205863237381, -0.44920310378074646, -1.7937674522399902, -0.5845900177955627, -1.022027850151062, 1.7680734395980835, -0.29510805010795593, 1.5122277736663818, 0.9667645692825317, 1.7822980880737305, -1.5567195415496826, -0.57822585105896, -1.5062745809555054, 0.008165515027940273, -0.48972439765930176, -1.290583848953247, 1.1816457509994507, 0.4475654065608978, -0.8042647242546082, 0.005779607221484184, -0.24260155856609344, 0.09391794353723526, 0.08239028602838516, 0.15636709332466125, -1.0675886869430542, 0.13515521585941315, 1.309234380722046, -0.11706554144620895, 1.8891938924789429, -2.6110289096832275, 0.5486263632774353, -0.08401095122098923, -3.079913377761841, -1.455203890800476, -0.5336378812789917, -0.7430489659309387, -0.24329598248004913, 0.792078971862793, -1.674591302871704, -0.49334484338760376, -1.695332407951355, -0.8572982549667358, -1.3914800882339478, 0.49225184321403503, 0.11202937364578247, -0.7927318215370178, -0.06772097945213318, -0.01052831020206213, 0.1810547262430191, -0.5589006543159485, 0.2589550018310547, 0.7644621729850769, -0.26440665125846863, 0.47677847743034363, -0.07493859529495239, -0.7812068462371826, 0.10792399942874908, -0.6454929709434509, -0.894219160079956, -0.3570888638496399, -0.009337281808257103, -0.5077452063560486, 0.6035627722740173, -1.2130277156829834, -1.6997390985488892, 0.5896403789520264, -1.1629643440246582, 0.7991390228271484, 0.019925978034734726, 0.04845130816102028, 0.1519126296043396, -0.5474016070365906, 2.0877771377563477, -1.172476053237915, -0.05022257938981056, -0.3508926331996918, -0.5379534959793091, -0.8693082928657532, -0.0701952576637268, 0.6703788638114929, 1.3367522954940796, -1.654674768447876, -0.5296976566314697, -0.09035684913396835, 0.9396262168884277, -1.4350544214248657, 1.101501703262329, -1.9044067859649658, -0.03894471749663353, -2.1505699157714844, 0.540985107421875, -0.23660294711589813, -0.18339693546295166, -0.6010487675666809, -1.0569745302200317, -0.1597680151462555, 1.3550255298614502, 0.5824626088142395, -0.5176498293876648, 0.36319220066070557, 0.0038157186936587095, 0.39196860790252686, -0.9214725494384766, 0.18913650512695312, 0.6478493213653564, 0.14231795072555542, 0.4575745463371277, 0.5793912410736084, -1.11089026927948, 0.7625377178192139, 0.6279976963996887, 0.9717028737068176, -0.021429432556033134, 0.5051012635231018, 0.9934834837913513, -0.6343998908996582, 0.886570394039154, -0.08880459517240524, 1.1835263967514038, -1.969404697418213, -0.11469007283449173, -0.5073617696762085, 0.46889129281044006, -0.042176250368356705, -1.2725707292556763, 0.90333092212677, -0.07002247869968414, 0.27211523056030273, -0.9195689558982849, -0.6883973479270935, 1.812842607498169, -0.14127889275550842, -0.36632782220840454, 0.23729871213436127, -0.20343294739723206, -0.47896772623062134, -0.43558454513549805, -0.4861184358596802, 1.3376810550689697, -0.7989139556884766, -1.451249122619629, -0.5586928725242615, 1.1683744192123413, 0.449823796749115, 0.8774809241294861, -0.005935260560363531, -0.20942077040672302, 0.8075161576271057, -0.9195869565010071, 0.9455304145812988, 0.9773892760276794, -1.1949794292449951, 0.8415360450744629, 1.2015317678451538, -1.2791340351104736, -0.9090704321861267, 1.12373948097229, -0.729665219783783, 0.930965781211853, -2.6015374660491943, 0.5663741230964661, -0.26798415184020996, 0.16700959205627441, 0.8065645098686218, -2.499213218688965, 0.26910337805747986, -0.36083051562309265, 0.174778550863266, -0.7787288427352905, 0.6561144590377808, 0.7365620136260986, 0.07333391159772873, -0.3068908751010895, -0.45271849632263184, 1.3186391592025757, -0.013167080469429493, -0.03125092014670372, -0.25290247797966003, -1.6842193603515625, -0.04222090542316437, 0.9046729803085327, 0.31064194440841675, -0.7044114470481873, -1.0473703145980835, -1.6060351133346558, 0.0588284395635128, 0.12598916888237, 0.542081356048584, -0.1023169457912445, 0.2637576758861542, -0.6565066576004028, 1.9682031869888306, -0.6159555315971375, -1.705885887145996, -0.6599951982498169, -0.4876280725002289, 1.288048267364502, -1.4200330972671509, -1.0440372228622437, -3.1678974628448486, -0.40935665369033813, -1.0263841152191162, -0.5107645988464355, -0.7473321557044983, -1.407476782798767, 0.2770008444786072, 0.8653993606567383, -2.632272243499756, 0.5073310136795044, 0.1356886327266693, -0.6764644384384155, 1.0524520874023438, 0.7654574513435364, -0.24403336644172668, 1.5584419965744019, 0.7618950009346008, -0.00258813239634037, 1.4224613904953003, 0.056797515600919724, 0.5738354325294495, -0.9688180088996887, 0.24398888647556305, -0.8349165320396423, 0.30695703625679016, -1.2310822010040283, -0.0033933657687157393, 0.11053254455327988, -0.6027900576591492, -0.09062743186950684, 0.07377466559410095, -1.0477150678634644, -0.1984153538942337, -0.8557286262512207, 0.02361290715634823, 0.4017462730407715, 0.5241077542304993, -0.17585435509681702, -0.44019293785095215, 0.13694921135902405, 0.9408561587333679, 0.8434756398200989, -1.1951173543930054, -1.5204674005508423, -0.359515905380249, -0.7210492491722107, -0.7617126107215881, 1.678707242012024, -0.7478018403053284, -0.6482737064361572, 1.2883579730987549, -0.8555505275726318, -0.00502419238910079, 0.31316009163856506, 0.45280060172080994, 0.9831821918487549, 0.12516319751739502, -0.40773239731788635, -1.6049177646636963, 1.872849464416504, 0.5963509678840637, -1.126911997795105, 0.4935140013694763, 0.2890374958515167, -1.131799340248108, 1.4013444185256958, -1.340898871421814, -1.3154942989349365, 0.11634912341833115, 0.8515617251396179, -0.9168370366096497, 1.8039385080337524, -0.18863511085510254, 0.5412075519561768, -0.3900856673717499, -0.33602404594421387, 1.553997278213501, -0.34649792313575745, 1.6432321071624756, 0.044969864189624786, -0.15888598561286926, 2.4560883045196533, 0.7408604025840759, -0.8770323395729065, -0.895652174949646, -0.2663123905658722, -0.15273815393447876, -0.9075477123260498, -0.9079189896583557, 0.5987240076065063, -0.17883813381195068, 2.853274345397949, -0.3038754463195801, 1.6294679641723633, -0.2804906964302063, -0.9970897436141968, -0.355386346578598, 0.024548817425966263, -0.44263705611228943, -0.783969521522522, -0.8718516826629639, 1.6563324928283691, 2.1903345584869385, 0.2135351002216339, 0.6092021465301514, -0.03167830407619476, -1.2949159145355225, 1.336809754371643, -0.6415702700614929, -1.2328795194625854, -0.15883959829807281, 0.0848206952214241, 0.19395282864570618, -0.297807514667511, -2.770730972290039, 0.024036003276705742, 0.02309105172753334, -1.117561936378479, -1.2669968605041504, 0.26897332072257996, 2.0066723823547363, 0.28879380226135254, 0.6452033519744873, 0.275852769613266, 1.9170677661895752, 1.0110667943954468, -0.827625572681427, -2.1366426944732666, 1.076915979385376, -1.3680616617202759, -0.2245766818523407, -0.11885018646717072, 1.2036126852035522, -0.22227449715137482, -1.648354172706604, -1.2123630046844482, 0.750758171081543, 0.42781347036361694, 1.089202880859375, 1.4853532314300537, -0.940621554851532, 1.4862457513809204, 0.9951905608177185, 1.4248974323272705, -0.14543354511260986, 1.3774189949035645, -0.4959052801132202, -0.2008201628923416, -1.3921579122543335, 0.892842173576355, -0.7024228572845459, 1.6873236894607544, 0.017004095017910004, -0.3283783197402954, 0.7952925562858582, 0.04221673682332039, -0.18073639273643494, 0.014104530215263367, -0.5813292264938354, 0.09411374479532242, -0.28734782338142395, 0.76883864402771, -2.4634618759155273, 1.366302490234375, -0.1121644452214241, 0.09979023784399033, 0.766443133354187, 0.591166079044342, 0.14035455882549286, -1.2600053548812866, -0.6332710981369019, 0.004041575361043215, 1.552551031112671, 0.022102495655417442, 0.2832123935222626, 0.1845414638519287, -0.2348603457212448, 0.7543237805366516, 1.0103654861450195, 1.0864627361297607, -0.06464460492134094, 1.5208103656768799, 1.9090052843093872, 0.7754433155059814, 0.39142993092536926, -0.7675127983093262, -0.07964271306991577, -0.19644521176815033, 0.4366168677806854, 0.746385931968689, 1.1832923889160156, 0.3324931263923645, 0.1606750190258026, -0.08034782856702805, 0.029001642018556595, 0.16834568977355957, -1.9228477478027344, 0.25130781531333923, 0.876606822013855, -0.7478929162025452, -0.4790749251842499, -0.20447658002376556, 0.009723186492919922, -0.24256248772144318, -1.5620734691619873, -0.6320995688438416, -0.6616069674491882, 0.5563861131668091, -0.2639584243297577, -2.059807300567627, 0.35157594084739685, 1.4278415441513062, -0.27029454708099365, 1.879374623298645, 1.0475062131881714, -0.14873705804347992, 1.9017304182052612, 1.3970181941986084, -0.09377091377973557, -1.062282681465149, 0.3861772418022156, -0.9274810552597046, -0.9795989990234375, -0.0318509116768837, -1.541542887687683, 0.6039756536483765, 0.3804788887500763, -0.3085024058818817, 2.1172685623168945, -1.3709214925765991, -2.2702877521514893, 0.8443990349769592, 0.31516698002815247, -1.018530011177063, -1.1122331619262695, -1.609133243560791, -0.19976617395877838, -0.39731502532958984, 0.5162984132766724, -2.1865768432617188, -1.1502100229263306, 1.0070648193359375, -1.6088969707489014, -0.489353209733963, -0.6478291153907776, 1.2611501216888428, -0.942499041557312, 0.5999093055725098, -1.0672168731689453, -0.3282450735569, -1.8710277080535889, -0.4406587481498718, -0.33557337522506714, -0.667773425579071, 0.8244002461433411, -1.2091453075408936, -0.26475849747657776, -1.3421725034713745, 1.887259840965271, -1.1941107511520386, -0.4210350811481476, 0.8493853211402893, -0.49467840790748596, 0.7861480712890625, 1.0257130861282349, 0.19333387911319733, 0.6545820236206055, 0.41739311814308167, -0.6238142251968384, 1.5298162698745728, 1.0916305780410767, -0.14165569841861725, -0.24034900963306427, -0.006194129586219788, -0.0025352470111101866, 1.7956053018569946, -0.8307186365127563, -0.004014621954411268, -0.4498838186264038, -0.21320997178554535, -0.895770788192749, 0.3027476966381073, -0.012955810874700546, 0.49905893206596375, -0.9390093088150024, -2.694542407989502, -1.1083248853683472, 1.1201002597808838, 0.11385054141283035, 2.095428705215454, 0.6848816871643066, 1.09329092502594, 0.5809831619262695, -1.0862617492675781, -1.2272684574127197, 0.5923802852630615, -0.5249350666999817, -0.09628300368785858, 1.4723007678985596, 1.2682979106903076, 1.9678599834442139, -0.32895806431770325, -2.1835641860961914, -0.011578752659261227, -1.2917670011520386, -0.1518498659133911, -0.1533280611038208, 0.4444447159767151, 1.1688436269760132, 0.4653698801994324, -0.4380643665790558, -0.7389364242553711, -0.6413564682006836, 1.330062985420227, -0.9025546312332153, -0.11379145830869675, 0.24789093434810638, 0.520376980304718, 0.516435980796814, 0.5799641013145447, -2.0498762130737305, -0.7935284972190857, -0.1789878010749817, 0.07041329145431519, 0.1986784040927887, -1.3488328456878662, -0.7685458064079285, -0.386800080537796, 0.17576752603054047, 0.6205486059188843, -0.3509061634540558, 1.319385051727295, 0.522339940071106, 2.496306896209717, 0.08352302014827728, -0.7874383330345154, 0.012256959453225136, -1.961999773979187, -0.2290840595960617, -0.6431304216384888, -1.7538516521453857, -0.1906682699918747, 0.8818906545639038, 0.5657707452774048, 0.5475291013717651, -2.3392717838287354, 1.0187736749649048, 1.0184270143508911, 1.416496753692627, -1.5205910205841064, 0.029239991679787636, 1.069555401802063, -2.3906431198120117, -0.10975360125303268, -0.6325308084487915, 0.2927168309688568, -0.11867030709981918, -0.2194531112909317, 0.15793396532535553, -0.815820574760437, -0.5337425470352173, -0.4065745174884796, 0.8030962944030762, 0.26334184408187866, -0.3174705505371094, 0.16084663569927216, 0.07259673625230789, 0.6807261109352112, -0.06843262910842896, -1.1413676738739014, 0.0773705318570137, -0.014796790666878223, 1.2385810613632202, 0.21205031871795654, -0.8971434235572815, 1.138844609260559, 0.8793699145317078, 0.7617824077606201, 0.8942208290100098, -1.1820192337036133, -0.46621087193489075, 1.9481768608093262, -0.2560521066188812, -1.2181137800216675, 0.11054466664791107, -0.9214627742767334, 0.9482030272483826, -0.2496502548456192, 0.5338661074638367, -0.8144497275352478, 1.7616724967956543, 0.48464667797088623, -0.151763454079628, 0.7843396067619324, -0.9887377023696899, -1.3581544160842896, -0.4921830892562866, -1.7549240589141846, 0.20020437240600586, -1.1572926044464111, -1.1950438022613525, -0.36903131008148193, -0.29697948694229126, 0.9346374869346619, -1.1080505847930908, 1.0680700540542603, -1.5071756839752197, 0.8622761964797974, 0.875413179397583, -0.9928208589553833, 0.7159809470176697, -0.04222416505217552, -1.2575734853744507, -0.5759883522987366, 0.16776327788829803, 1.7107219696044922, 0.9881000518798828, 0.7545240521430969, 0.63676917552948, 1.391632318496704, 0.9277265667915344, 0.7843615412712097, 0.6811904907226562, -0.23102401196956635, 0.01309147384017706, 0.5180726051330566, -0.4731089174747467, -0.24493582546710968, -0.7229241132736206, 0.07527761906385422, -0.1892320066690445, -0.3329640328884125, -0.40931370854377747, -0.18174239993095398, -0.1533259153366089, -0.21978017687797546, -0.6607004404067993, -1.236064076423645, -0.720339298248291, 0.47482457756996155, -0.9647467136383057, 0.7605941891670227, -0.24541142582893372, 0.2298835664987564, 0.02163444086909294, -0.6775336265563965, -1.2595624923706055, 0.7340914607048035, -0.307603120803833, 1.062280297279358, -0.19559024274349213, 0.792895495891571, -0.8345073461532593, -1.5961874723434448, -0.5865097045898438, -0.05604228004813194, 0.5009267926216125, -0.6228557229042053, -0.038512762635946274, -1.0750442743301392, 0.3783474564552307, -1.6140053272247314, 0.7404040098190308, 0.4381008744239807, 1.150099277496338, -0.21674662828445435, -0.5351305603981018, 0.015080397948622704, -0.09760730713605881, -0.027879053726792336, -0.6037784218788147, -0.5117021203041077, -0.9770296812057495, -0.8608875870704651, -1.6829689741134644, 0.8492016196250916, -2.310579538345337, -0.822186291217804, 1.4289534091949463, 0.0670592337846756, -0.33892160654067993, -1.1190431118011475, 0.20580820739269257, 0.8347192406654358, 0.7613006830215454, 1.413202166557312, -0.13144566118717194, 1.3602700233459473, -0.36555299162864685, 0.4968283474445343, 1.7038735151290894, -0.6945415139198303, -1.0029348134994507, 1.0900139808654785, 1.4125639200210571, -0.6379899978637695, -1.6931699514389038, 0.2952564060688019, 1.8901606798171997, -2.05210280418396, 0.04750923439860344, -0.9097530245780945, 0.10062211751937866, 0.5004685521125793, -0.765614926815033, 0.41777241230010986, -0.9361383318901062, -0.22625094652175903, -1.5622488260269165, -1.0111018419265747, -0.5426348447799683, 1.8997024297714233, 1.6356106996536255, 0.2365477830171585, -0.5302388668060303, 0.4576178193092346, 0.7451126575469971, -0.11344125866889954, 0.2935735881328583, -0.4935181736946106, 0.4726945757865906, 0.8004926443099976, 0.6734036803245544, 0.4861334562301636, 0.12829482555389404, 0.1545920968055725, 0.20491667091846466, -0.17110706865787506, 0.30726996064186096, 0.4574124813079834, -0.6472389101982117, -0.7048925757408142, -1.5075565576553345, 0.21076154708862305, -0.42573273181915283, -1.107529640197754, 0.04583129286766052, 1.2961817979812622, 0.653947651386261, -0.6891338229179382, 1.144411563873291, 1.1925472021102905, 0.575718879699707, 0.8444190621376038, -0.8710480332374573, 0.8176111578941345, 0.2742660641670227, 0.3648289442062378, -0.13847285509109497, -0.24812085926532745, -0.06257808953523636, 0.44005417823791504, -1.6323161125183105, -0.6156505346298218, 0.8742731213569641, 1.9502625465393066, -0.34755218029022217, -1.339556336402893, 0.061205215752124786, 0.5840893387794495, 0.4267662465572357, -0.5481656193733215, -1.187179446220398, -0.5425081849098206, 2.0926859378814697, -1.4991555213928223, -0.6720066666603088, 0.2747083306312561, 0.40692025423049927, 0.9482353329658508, -0.08408983051776886, 1.2469351291656494, -0.6960108876228333, 0.5206215977668762, 1.407346487045288, -1.4960232973098755, 0.03559271618723869, -0.6535313725471497, -0.7622105479240417, -0.019588813185691833, 0.359418123960495, -0.9966276288032532, 1.7690573930740356, -0.13633129000663757, -0.9009571075439453, 0.2736152112483978, 0.8130456805229187, 0.17019738256931305, 1.2639379501342773, 0.606185793876648, -1.2148706912994385, 1.339159369468689, -1.1793845891952515, -0.625876247882843, 0.5265516042709351, 1.0280262231826782, 0.10671240091323853, 0.7871735095977783, 1.1184614896774292, -0.9979576468467712, 1.8796875476837158, 0.9574897885322571, 0.349297434091568, -0.22617746889591217, 0.24770556390285492, -0.6135981678962708, 1.2960164546966553, -1.3575174808502197, -0.5606952905654907, -0.817395031452179, -1.7062636613845825, 0.15059271454811096, 0.1028687059879303, 2.1141886711120605, 1.3649042844772339, 0.16286084055900574, -1.8007234334945679, -0.9900827407836914, -1.6458147764205933, 1.9022835493087769, 0.6225744485855103, 0.44212237000465393, -0.2227238118648529, 1.0226237773895264, -0.018037015572190285, -0.8103636503219604, 0.8686009049415588, -0.0007450946141034365, 0.06952011585235596, 0.2833482325077057, 0.8045563697814941, 1.5837868452072144, -1.1435997486114502, -0.8468029499053955, 0.8456140756607056, -0.5261764526367188, -1.1986706256866455, 0.24989619851112366, -0.19017012417316437, -1.487684726715088, 1.139174461364746, -0.4485057294368744, 1.0244109630584717, 2.614361524581909, -0.2148720920085907, 0.18821901082992554, -0.24891994893550873, -0.6411156058311462, 0.6062781810760498, -0.5704357028007507, -0.6590476632118225, 0.10221261531114578, -1.2034976482391357, 0.16223947703838348, -0.11557605862617493, 1.2173691987991333, -0.27667897939682007, 0.49754104018211365, 2.0753848552703857, -3.3663547039031982, -1.7331217527389526, -0.42174291610717773, -1.1461306810379028, -0.9830725789070129, -0.9398438930511475, 0.5053279399871826, -1.1738554239273071, 0.26710087060928345, 1.0000851154327393, -0.37258750200271606, -0.08675479143857956, -0.2767517566680908, -0.19238263368606567, 0.29706907272338867, 0.21604086458683014, -0.5876720547676086, 0.43579378724098206, 0.6015986800193787, -1.3902291059494019, -1.0204272270202637, -0.88577800989151, 0.022263063117861748, 0.9445488452911377, -0.9757452607154846, 0.9355621933937073, 0.875744104385376, -0.5831137299537659, -0.7659018039703369, 0.6318545937538147, 0.1996258795261383, -1.1074110269546509, -0.9058157205581665, -0.6405662298202515, -1.7541314363479614, -0.35204508900642395, 1.496773600578308, 0.05332975834608078, 0.3418080508708954, -0.8762514591217041, 0.21773870289325714, -0.15833720564842224, -0.32043060660362244, 0.3175489604473114, -0.7438439726829529, -0.13619759678840637, 0.7400519847869873, 0.42003220319747925, 1.5224913358688354, 0.21618393063545227, 0.799299418926239, 0.13383762538433075, -1.0731016397476196, 0.5324189066886902, -0.4938550889492035, -0.5930507183074951, 1.9067591428756714, -0.4310857653617859, -0.7561801671981812, -2.9565796852111816, -1.0571762323379517, -0.10676246136426926, 0.3588173985481262, -1.2646745443344116, -0.5344650149345398, 0.7473710775375366, 0.8344837427139282, -0.09118498116731644, 0.1580980271100998, -0.7312481999397278, 0.5677710771560669, 0.23066526651382446, 0.3699679374694824, -0.5292277336120605, 0.1569264829158783, 0.05488644167780876, -0.3571038246154785, 0.8496644496917725, 0.16875946521759033, 1.4091942310333252, -0.8972926139831543, -0.7730966806411743, 1.3190078735351562, 0.46814045310020447, 0.9599160552024841, -0.2724950611591339, 0.24336709082126617, 0.7955002188682556, 0.4694565534591675, 0.9415769577026367, 0.3096858561038971, 0.7150387167930603, -2.5965569019317627, -0.3200156092643738, 0.4643797278404236, 2.751861810684204, -0.7842614054679871, 0.7312628030776978, -0.06044450402259827, 0.39358463883399963, 1.2769453525543213, -1.318601131439209, 0.7049953937530518, -0.08118907362222672, -0.4425651431083679, -1.2194560766220093, -0.44890737533569336, -1.600675344467163, -0.3017626106739044, -0.020877396687865257, -0.5721572041511536, 1.4504975080490112, -1.1833117008209229, 0.742554247379303, -1.1868603229522705, -1.5449753999710083, 1.7887380123138428, -0.3254680931568146, -1.1460000276565552, -0.9478231072425842, -1.5598275661468506, 0.19031494855880737, -0.052719082683324814, 0.4201590120792389, -0.5352123975753784, -1.762487769126892, -1.4357867240905762, -0.3341396450996399, -2.156956672668457, -0.51701420545578, -0.2903587222099304, -0.6443034410476685, -0.8648573160171509, -0.32618579268455505, 1.3772501945495605, 0.6092959046363831, -0.965508759021759, -1.0481257438659668, -1.1135081052780151, -0.7547087669372559, -2.051478862762451, -0.2847025394439697, -0.27660709619522095, -1.5560615062713623, -1.1621383428573608, 0.6500535607337952, -0.6580736637115479, -1.4911599159240723, -1.1653109788894653, 1.2916358709335327, -0.35522323846817017, -0.2564876079559326, 0.4406706690788269, -0.20458367466926575, -1.2969064712524414, 1.058766484260559, 0.7689977884292603, 0.5168071985244751, -1.5755562782287598, -0.6494528651237488, -1.3147140741348267, 0.6236757040023804, 1.5468826293945312, -1.3857454061508179, -1.2752656936645508, -0.5877575874328613, -0.48366084694862366, 0.6098278760910034, -1.7313847541809082, -0.042127739638090134, -1.1191853284835815, -1.5202285051345825, 2.0599985122680664, 0.06024501845240593, 0.6918360590934753, -0.7391180396080017, -1.3403488397598267, 0.3534964919090271, 0.32558175921440125, -0.6945925951004028, 1.1076663732528687, 0.19650301337242126, 0.6356062889099121, 0.43363961577415466, -0.6506436467170715, 1.6426721811294556, 1.0742393732070923, -0.7634096145629883, -0.2687337100505829, 0.9609935879707336, 0.4763033092021942, -0.42271456122398376, 0.23157408833503723, -0.7964531183242798, -1.299401879310608, 0.28496253490448, 0.18046019971370697, 0.21687273681163788, -0.7915514707565308, -0.3195847272872925, 1.056786060333252, 2.1386733055114746, 0.5068359375, -1.1999399662017822, 1.6411899328231812, -0.3550167679786682, 0.29252487421035767, 0.1778622269630432, 0.5435245633125305, 0.8012537956237793, -0.47962450981140137, -0.9754716157913208, 0.2682296633720398, -0.5505325198173523, -0.23603981733322144, -1.021036148071289, -1.2937901020050049, -1.1697028875350952, -0.6853368878364563, -0.5971401929855347, 0.8203661441802979, 0.4757237136363983, 0.4613199830055237, 0.03799960017204285, 0.7420192360877991, -0.5883263349533081, -0.3720564842224121, -0.5784662365913391, -1.1271283626556396, -0.33569246530532837, 1.0447531938552856, -1.399945855140686, 0.6777795553207397, -0.8412216901779175, -0.6733298897743225, -0.4302119314670563, -0.9316530823707581, 0.06705639511346817, 0.8402262926101685, 0.3622449040412903, -0.37142637372016907, -1.32695472240448, -0.3408947288990021, -0.8766276836395264, 0.31405600905418396, -0.20397132635116577, -0.11254090815782547, 0.937269389629364, 1.304005742073059, 0.7598857283592224, -0.15319161117076874, 0.18436938524246216, 1.1123507022857666, -1.2817730903625488, -0.8294597268104553, -0.621232271194458, 1.3508960008621216, 0.17668087780475616, -1.7122091054916382, 0.5698243975639343, 1.733265995979309, -2.246394157409668, -1.061939001083374, 1.6256319284439087, 0.491732120513916, 0.15779954195022583, -0.8915470838546753, 1.8602452278137207, -0.37836235761642456, 1.3596166372299194, 0.6825093626976013, 0.8046894669532776, 0.4972708821296692, 0.6807729601860046, 0.7811638116836548, -1.564630150794983, -0.11247306317090988, 0.17127247154712677, 0.41080501675605774, -0.22078800201416016, -0.9591120481491089, 0.0547696091234684, 0.5996931195259094, -0.6193704009056091, -0.49743208289146423, 1.6461966037750244, -0.9425182342529297, 0.17244061827659607, -1.034279465675354, -0.3326619267463684, 0.6669487357139587, -1.7929848432540894, 1.0326957702636719, -0.5310832858085632, -1.0067737102508545, -0.23385973274707794, -0.32263317704200745, -0.011553621850907803, 0.8079516291618347, 1.3905593156814575, 0.30193954706192017, -0.7583644390106201, -0.726094663143158, 0.8112095594406128, 0.09902781248092651, 0.3624657988548279, 0.7904722094535828, -0.6191355586051941, 0.1233135238289833, -0.5699644088745117, -0.04139082506299019, -1.6909584999084473, 0.816317617893219, 0.26784589886665344, 0.7149950265884399, -1.95424222946167, -0.022398579865694046, 1.2065953016281128, -0.8583611249923706, 0.4640154540538788, 1.7029218673706055, 1.1263012886047363, -0.3090285360813141, -0.570364236831665, 0.856674075126648, -0.537749171257019, 0.5178570747375488, 0.4940437972545624, -0.42696988582611084, -1.795899748802185, 0.7669233679771423, -0.20987270772457123, 1.2049459218978882, -0.8688101768493652, -0.7143320441246033, -0.2122763693332672, -0.009564553387463093, -1.032044529914856, 1.5939226150512695, -0.3670605719089508, -0.00989783089607954, -1.0305120944976807, 1.019350290298462, -0.7036665081977844, -0.22633697092533112, -0.46127039194107056, -0.5708566308021545, -1.6718451976776123, -0.6716816425323486, -0.13904568552970886, -0.02515295147895813, -1.4071139097213745, 0.32519879937171936, -0.10396598279476166, -0.12223155796527863, -1.8857462406158447, 0.1672568917274475, 1.3994652032852173, 0.43075406551361084, -0.7162208557128906, -0.4315739572048187, -0.48298946022987366, -0.684781014919281, -1.1211626529693604, -0.1469685137271881, -0.15556393563747406, -1.23976731300354, 0.049480557441711426, -1.327293038368225, -0.8110404014587402, -0.26450374722480774, -0.27809712290763855, -1.416412591934204, 0.18767152726650238, 1.429414987564087, -1.6692389249801636, 0.37559014558792114, -1.001527190208435, -0.06426762789487839, -0.38119038939476013, -0.42394301295280457, -0.24620020389556885, -1.4832907915115356, 0.48689571022987366, -0.3863103985786438, 1.8881044387817383, -2.3001394271850586, -0.2998563349246979, 1.5195974111557007, 0.7177363038063049, 0.5234919190406799, -1.0143675804138184, 0.5313606858253479, 0.6658872961997986, 1.3370682001113892, -0.5643101334571838, 0.8231770396232605, -1.0616137981414795, -0.20640026032924652, -0.012526283040642738, 0.9270102977752686, 0.04701821878552437, -0.4093545079231262, -1.3668125867843628, 1.713127851486206, -0.04248558357357979, 0.41680648922920227, -2.0133469104766846, 1.0076031684875488, 1.9059491157531738, 0.5567435622215271, -1.1828142404556274, -0.007546942215412855, -0.6830282807350159, 1.0230792760849, -1.1664007902145386, -0.8256306052207947, -0.05092482268810272, -0.49301695823669434, -1.167118787765503, 0.7532530426979065, -0.4972081780433655, 0.34746670722961426, -0.19549590349197388, -0.901685893535614, 1.4782038927078247, 0.5886308550834656, -0.3888450860977173, -0.47188109159469604, 2.892932415008545, 0.47125139832496643, 1.4146873950958252, -0.05455821007490158, -0.07782906293869019, -1.0824217796325684, 1.1451010704040527, -1.2496031522750854, -0.019475100561976433, 1.3550995588302612, 0.6182348728179932, -2.452436923980713, -0.05461369827389717, 0.33689796924591064, 0.7006044387817383, -1.2690136432647705, 1.5399110317230225, -0.5446563959121704, -2.1451034545898438, 0.5221491456031799, 0.36102572083473206, 0.16907720267772675, -1.5664781332015991, 0.14995597302913666, -1.9621503353118896, -1.5208128690719604, 0.6168946623802185, -1.8762046098709106, -0.29252883791923523, 2.096036195755005, 0.014809560030698776, -0.10003935545682907, -0.34665268659591675, 0.7811188101768494, 0.6336908340454102, 0.13670016825199127, 0.8085616230964661, -0.3068530857563019, 0.7037329077720642, 1.5197783708572388, -0.7466809153556824, 0.8486701250076294, -0.7625875473022461, -1.3952056169509888, 0.7502077221870422, -0.12725970149040222, 0.07481859624385834, 0.8306999206542969, -1.2030436992645264, 0.22475019097328186, -0.11598263680934906, 0.6377649307250977, -1.1744239330291748, -1.3809106349945068, -0.4627768397331238, 0.7712835073471069, -0.9794324636459351, -0.2987064719200134, -1.2755119800567627, 0.06396446377038956, 0.6209070682525635, 1.1592128276824951, 0.11630360782146454, 0.9245477318763733, 0.3768676817417145, 0.8308307528495789, -1.0492262840270996, 0.11263562738895416, -0.795050859451294, -1.204707384109497, 1.2516608238220215, -0.18928921222686768, 0.7271796464920044, 0.46737906336784363, -1.3119409084320068, -0.9639652371406555, 0.39225438237190247, 0.6598964333534241, -0.8031622767448425, 1.1610314846038818, 0.15896481275558472, 0.6968579888343811, -0.9818538427352905, -0.13204888999462128, -0.11591430008411407, -0.08780393749475479, -0.5193028450012207, 0.0369618684053421, 1.9909908771514893, -1.2059125900268555, -0.30792975425720215, -1.1473352909088135, 0.057924676686525345, -2.801635503768921, 0.13188068568706512, -0.037571799010038376, 0.31000107526779175, -1.9897527694702148, -1.9733529090881348, -0.9584813714027405, -0.06409867107868195, -0.905514121055603, -1.527565598487854, -0.7059162855148315, 0.8020776510238647, -0.5857701301574707, 0.5597180724143982, -0.4250900447368622, 2.141584873199463, -0.9086589217185974, -0.2642439603805542, -0.6952480673789978, 1.6649833917617798, 0.029625654220581055, 0.2953738272190094, -2.155815362930298, -0.9377657175064087, -0.8252328634262085, 0.4566069543361664, 1.8575439453125, -0.8071556091308594, -2.344182014465332, -0.35773083567619324, -1.5507538318634033, -0.9580926299095154, -0.37875884771347046, 0.1289093792438507, -1.1267662048339844, -0.07751970738172531, 0.1057935282588005, -0.8389270901679993, 0.764025092124939, 0.6480878591537476, 0.77516108751297, -0.7780196666717529, -2.0104784965515137, -0.9366089105606079, 0.3636070489883423, -2.42802357673645, -0.2842225432395935, -0.26786142587661743, 1.0647488832473755, -0.9959300756454468, -1.7288461923599243, -1.8509318828582764, -1.339371919631958, -0.3762207329273224, 0.980592668056488, 1.0117559432983398, 0.0006922826287336648, -0.14432381093502045, 0.6581240296363831, 0.045694056898355484, -0.8210513591766357, 0.1897197663784027, 0.8575002551078796, 0.8353901505470276, -0.9822056889533997, 0.25104907155036926, -0.6856046319007874, -0.7212896347045898, 1.29415762424469, 0.4629480838775635, 1.0083506107330322, -0.11365292221307755, 1.1928324699401855, -0.7753325700759888, 0.960297167301178, -1.187314510345459, -0.03819875046610832, -0.33090940117836, 0.8322732448577881, -0.16992735862731934, 1.833925724029541, 0.7849726676940918, -0.19127444922924042, -0.07842019200325012, 0.12707658112049103, 0.6652936935424805, -0.31880825757980347, -0.9360514879226685, 0.014480366371572018, 0.7184725999832153, 0.5583511590957642, -0.5627946853637695, -0.3684956431388855, -1.4502952098846436, -0.25147247314453125, 2.2189035415649414, -0.6266021132469177, 0.7959756255149841, -0.7911992073059082, -1.0189238786697388, -0.8384646773338318, -0.49932828545570374, 0.4032013416290283, 0.24499456584453583, -1.330268383026123, 1.2288625240325928, 1.8908476829528809, -0.8674653172492981, 1.7805362939834595, -0.5391842722892761, 0.1772388219833374, -0.29036587476730347, 1.1825200319290161, -1.0597360134124756, 0.1154058575630188, -0.5047712922096252, -0.34876471757888794, -0.7735207676887512, 1.2366397380828857, -2.2435107231140137, 0.16795510053634644, 0.8845862746238708, 0.47938039898872375, -0.48912203311920166, -0.7007378935813904, 0.07462980598211288, 2.0027825832366943, 0.7365748882293701, 0.22725318372249603, 1.8760631084442139, -0.7243016362190247, 1.0044028759002686, 0.29223859310150146, 1.2281662225723267, 0.6285223364830017, 0.44298115372657776, 1.1845985651016235, -0.18711286783218384, -0.5977303385734558, 0.741472065448761, 0.14646410942077637, 0.8121103048324585, 1.8737519979476929, -1.6903090476989746, -1.3819366693496704, 0.299450159072876, 0.378665953874588, 1.5882123708724976, 1.648350477218628, 0.28812411427497864, 0.02876419574022293, -1.1199284791946411, -0.660062849521637, -2.1692659854888916, 1.3619846105575562, 0.08033782243728638, 0.12033259868621826, 0.5607703328132629, 0.12395045906305313, 0.5922789573669434, -0.2480608969926834, -1.086429238319397, -0.16295860707759857, 0.961107611656189, 1.0332449674606323, -0.9517083168029785, 1.8570046424865723, -1.044599175453186, -0.8097285628318787, -1.1558231115341187, -0.5213570594787598, -0.5739675164222717, 1.070937156677246, 0.7299107909202576, 0.29103031754493713, 0.204628586769104, -0.5200536847114563, 0.4067886769771576, 0.45528703927993774, -0.1432441622018814, -0.8073722124099731, -0.5222260355949402, -0.8789324760437012, -0.4240555763244629, 0.41277384757995605, -1.1193792819976807, -0.44871705770492554, -0.17966210842132568, -0.6570737361907959, -0.32162171602249146, 0.7245906591415405, 1.4665201902389526, 0.27242329716682434, 0.20511089265346527, -0.017833849415183067, -1.758888840675354, -1.0077229738235474, 0.7879362106323242, 0.3722447454929352, 0.09570876508951187, -0.16277600824832916, -1.217983365058899, 0.667909562587738, -0.5143198370933533, 1.4428672790527344, 1.164551019668579, -0.20666228234767914, 0.326817125082016, 1.6087646484375, -0.38827642798423767, 0.32021620869636536, 0.9703415632247925, -0.8288153409957886, 0.3947871923446655, 1.5629781484603882, -0.774336576461792, -0.6126314997673035, 0.22540464997291565, -1.4229308366775513, 0.75447678565979, 0.9859309196472168, 1.2721418142318726, -0.26147136092185974, -0.5836693644523621, -1.5327730178833008, 0.15300889313220978, -0.5470571517944336, -1.2755523920059204, -0.64288330078125, -1.3684613704681396, 0.6121016144752502, 0.8058515787124634, -0.6418498158454895, 0.137212336063385, -0.4728713035583496, 1.6747801303863525, 0.3722350299358368, 1.789892315864563, -1.0831875801086426, -0.63588547706604, -0.799221932888031, 0.6537555456161499, 0.2346070259809494, 0.4778565466403961, -1.6349127292633057, -1.835811734199524, -0.836042582988739, 0.4598960876464844, 1.2060511112213135, -0.516654372215271, -0.7908745408058167, 1.1553024053573608, 0.43741974234580994, -0.904680073261261, 0.31522324681282043, -0.7712198495864868, 0.0825846865773201, -0.38536563515663147, 1.5070794820785522, 0.5644931197166443, -0.7323351502418518, -0.5253177881240845, -0.4722006022930145, 0.4929671287536621, -2.126739978790283, -0.07243265956640244, -1.4683927297592163, -0.8544430136680603, 1.6010830402374268, 0.057731639593839645, 0.3708585500717163, -1.6815515756607056, 0.25788748264312744, 0.04713042825460434, -2.2442853450775146, 0.19533702731132507, -0.2580417990684509, 0.27126026153564453, 0.19748914241790771, 0.3754071295261383, -0.2879917621612549, -2.3490822315216064, -0.7785977125167847, 0.8559421300888062, 0.4662300646305084, 0.09670543670654297, 1.230342149734497, 0.49653732776641846, -0.09018387645483017, -0.6878337264060974, -0.26860031485557556, 0.6983139514923096, 1.5721628665924072, 1.3824841976165771, 0.3612021803855896, -0.13389292359352112, 0.9701693654060364, 0.2443852424621582, 0.9242801666259766, -0.8566049933433533, 0.8865405917167664, -0.0034855403937399387, -0.045633234083652496, 1.1902780532836914, 0.36314085125923157, 0.41524064540863037, 0.4938162863254547, -0.48519474267959595, 0.4816296398639679, 0.3487128019332886, -2.1296894550323486, -0.8789682388305664, 1.3438693284988403, -1.2970788478851318, 2.167534351348877, -0.18011446297168732, 0.9553452134132385, 2.1617507934570312, -0.36878475546836853, -1.2013134956359863, -1.3445314168930054, 0.02322138100862503, -0.9251885414123535, 0.04253871738910675, -0.8450689911842346, -0.5362585783004761, -0.5961989760398865, -0.33920592069625854, 1.2617522478103638, -1.2536760568618774, -0.43877559900283813, -0.0748976469039917, -0.6023392081260681, -2.13023042678833, 0.7233343720436096, -0.5601518154144287, -0.23359496891498566, -1.3015414476394653, -1.034409761428833, 1.8724972009658813, -0.7960149049758911, -0.4762178957462311, 1.1791809797286987, 1.2449997663497925, 0.4880351126194, 1.1197439432144165, 0.3475404381752014, -1.7611188888549805, -1.0414224863052368, 0.4185379147529602, 1.1354538202285767, 1.4401319026947021, -1.3162050247192383, -0.5242876410484314, 0.6192322373390198, -0.42203885316848755, 0.7357361316680908, 1.5438750982284546, 0.05470210313796997, 0.7575961351394653, -0.8285187482833862, -0.27361002564430237, 0.5178809762001038, 0.09636906534433365, -1.6071940660476685, -0.4009724259376526, -0.5336857438087463, -1.194459080696106, 1.4704536199569702, 0.10728513449430466, -0.9520448446273804, -0.6564537286758423, -0.5416259169578552, 0.5751428604125977, -0.9695922136306763, 0.06411051750183105, 0.9301759004592896, -0.043259527534246445, -0.38513997197151184, -0.06948074698448181, 0.400258868932724, -0.1948270946741104, 0.3523347079753876, 0.1355365663766861, -0.4634738266468048, -0.5057125091552734, -0.14056086540222168, -0.27785804867744446, -1.1065536737442017, 0.23245353996753693, 0.7143751382827759, 0.21313801407814026, -0.306174635887146, 0.1180582121014595, 1.2470574378967285, -0.05723773315548897, -0.920954167842865, -0.4204982817173004, 1.0242125988006592, -1.157575011253357, 1.2273285388946533, 1.3943918943405151, 1.3112667798995972, 0.9455575346946716, 0.7826191186904907, 0.7115337252616882, 0.6731376647949219, -1.3687225580215454, -0.8105520606040955, -1.2688589096069336, 0.6067093014717102, -1.224516749382019, 0.08766869455575943, 1.337511658668518, 1.429740309715271, 1.4763838052749634, 0.7292838096618652, 0.9572030305862427, -1.0393950939178467, -0.6116028428077698, 0.06342733651399612, -1.9391542673110962, -1.059966802597046, 1.0783358812332153, 0.07875800877809525, 0.5370988249778748, -0.40820595622062683, -1.0311758518218994, -1.14547860622406, -0.16404671967029572, -0.5778558850288391, -0.31637051701545715, 0.13594743609428406, 0.6091393232345581, 1.3580025434494019, 0.27748870849609375, 2.0201871395111084, -1.6396853923797607, -1.4569169282913208, -1.6759711503982544, 0.1997489184141159, -0.11795990914106369, 0.4372544586658478, -0.3972287178039551, 0.10969505459070206, 0.10784532129764557, 0.12247265875339508, -0.957062840461731, 0.6724048852920532, 1.617551565170288, 1.1185680627822876, -0.35120320320129395, -1.2250127792358398, -0.961962878704071, -1.667277216911316, -0.8697540760040283, 1.0812286138534546, 0.5609643459320068, 0.2048376351594925, -2.002007246017456, 0.22487632930278778, 2.0131444931030273, 0.783068060874939, -0.06805802881717682, 0.3402262330055237, 0.9764112830162048, 0.0913674458861351, 0.024936024099588394, -0.15929017961025238, -1.2942843437194824, -0.6605194807052612, 1.147736668586731, 0.6146761178970337, 0.1153797060251236, -0.36039111018180847, -0.15931159257888794, -0.6761303544044495, -0.02784583531320095, 0.2648501992225647, -1.0765684843063354, 0.39201804995536804, -0.8904592990875244, -0.16969887912273407, 0.5576087236404419, -1.3227696418762207, -1.704057216644287, -0.4934689998626709, 0.656170666217804, -0.4234142601490021, 0.25846609473228455, 1.0608519315719604, -0.4243728220462799, 1.2998526096343994, -0.5251958966255188, -1.0793133974075317, -0.2558494210243225, -0.6708002686500549, -0.30650898814201355, 0.743169903755188, -0.8017240166664124, 1.116540789604187, 0.16265274584293365, 1.3272675275802612, 1.0300096273422241, -0.24148595333099365, -0.1759137660264969, -1.0685253143310547, -0.7639209032058716, -0.6426324248313904, -0.5864206552505493, -0.09495244920253754, 0.7934805154800415, -0.07493191957473755, -0.08357346057891846, -0.626646101474762, -1.201851487159729, -0.8899998664855957, 0.04958711192011833, 0.6366426348686218, -1.0019158124923706, -0.5889161825180054, 1.3715239763259888, 0.4053918719291687, 1.0253597497940063, -0.526286780834198, 0.4943532943725586, -0.7866954803466797, -0.9009199738502502, -0.8229469060897827, -0.34605205059051514, -0.690615177154541, -0.6089890599250793, 0.4430212378501892, 0.12222404032945633, -0.5749925374984741, -0.328840047121048, -1.2425475120544434, -0.05332930386066437, 0.44406142830848694, 1.3615097999572754, -1.508164644241333, -1.478044867515564, -0.6491687297821045, -0.6304970979690552, 0.3414532542228699, 0.519092321395874, 0.5019968748092651, 0.08335992693901062, -1.4922826290130615, -0.3383078873157501, 0.15111121535301208, -0.15738725662231445, -1.4926997423171997, -2.234971523284912, 1.0387171506881714, 1.562071681022644, 0.8497707843780518, 0.5790883302688599, 1.0826756954193115, -0.0515226349234581, 0.13071952760219574, 0.7112939953804016, -1.596313238143921, 0.22722318768501282, -0.024256441742181778, 1.2596971988677979, -0.49157989025115967, -0.38605624437332153, -1.1259281635284424, -0.12690559029579163, 0.5929310917854309, 0.2181810885667801, 0.27622050046920776, -1.0027486085891724, 0.3241107761859894, 0.2755936086177826, -0.5470184683799744, 0.18312685191631317, -0.0973903238773346, 1.8225747346878052, 1.108325481414795, -1.4147218465805054, -2.121915578842163, -2.2234623432159424, 0.0003936065186280757, -0.7258619666099548, -1.11565363407135, 0.626686692237854, -0.11367660015821457, 0.1348913013935089, -0.0037070601247251034, -0.332478404045105, 2.001673460006714, 0.3274138867855072, -1.4811116456985474, -1.2482719421386719, 0.37909549474716187, 0.3136025071144104, 2.1812121868133545, 1.4009082317352295, -0.696302592754364, -0.03741896152496338, -0.8803135752677917, 1.5558146238327026, -0.3986493945121765, -0.6942732334136963, 1.345704436302185, 1.0403861999511719, 0.6074899435043335, 0.3069908618927002, 0.4950576424598694, -0.1184576004743576, 0.3128105103969574, 0.25863218307495117, 0.604088306427002, -0.9088655710220337, -1.1082909107208252, 0.382087767124176, 0.514252245426178, 2.0385360717773438, 0.04148406907916069, 1.5527640581130981, -0.28301557898521423, -0.7466068863868713, -1.143388271331787, -0.5608386993408203, -0.18952029943466187, 2.055324077606201, 0.6766137480735779, -0.5177773237228394, -0.47696152329444885, 1.6861474514007568, -0.9244339466094971, 0.18008087575435638, -0.45598819851875305, -0.15456463396549225, 0.8682348132133484, -0.16424834728240967, -0.773433268070221, -0.02388075739145279, -1.0607057809829712, -0.26321321725845337, 3.228543519973755, -1.3499056100845337, 0.15875737369060516, -1.4073376655578613, 1.194578766822815, -1.4417258501052856, 0.324394553899765, -0.754994809627533, -0.9278503656387329, 0.8390418291091919, -0.6618452668190002, 2.4380104541778564, 0.2464817315340042, 0.8330339789390564, 1.531487226486206, -0.007780163548886776, -0.8857309222221375, 0.1758318990468979, 3.2293312549591064, -0.37625613808631897, 0.3365178108215332, 1.2315384149551392, -0.4042111039161682, 0.4078321158885956, -0.8801930546760559, -0.36915943026542664, -1.1205663681030273, 0.8557001352310181, 0.3853617310523987, 0.2951575517654419, -2.079709529876709, -2.0194945335388184, -1.1493533849716187, -0.8258323073387146, 1.5679945945739746, -1.6292699575424194, 0.2528534531593323, -0.015449024736881256, -0.33258000016212463, 1.2599722146987915, -1.0068070888519287, -2.4830970764160156, -0.20288260281085968, 1.425527572631836, 0.31736281514167786, -0.9027296304702759, 2.799940586090088, -0.8309564590454102, -0.11187935620546341, -0.4869872033596039, 0.7313623428344727, -2.3426589965820312, 1.3045408725738525, 1.1557568311691284, 2.119858503341675, -0.9343499541282654, -0.6627804636955261, -1.3248825073242188, -1.554844617843628, -0.9006933569908142, 0.33036625385284424, 0.17029902338981628, 0.3364892601966858, -0.3795297145843506, 1.142755150794983, 0.3387681841850281, 0.07486855983734131, -0.5999887585639954, -0.3952176868915558, 0.09754525125026703, 1.9087061882019043, 0.7572422623634338, -0.9649601578712463, 0.48133933544158936, 1.594632625579834, 0.4167027771472931, -0.2333638072013855, 0.17999596893787384, 0.733936607837677, 0.30223777890205383, 0.5535827279090881, -0.9914188385009766, 1.0551732778549194, -2.35125732421875, -1.8997019529342651, 0.7196050882339478, -0.4047420620918274, 0.21980468928813934, -0.35325464606285095, 0.7846032977104187, -1.1436563730239868, 0.8808733820915222, 0.21153537929058075, 0.17087028920650482, 0.6702755093574524, 0.6299453973770142, 0.8868609070777893, -0.8719899654388428, 0.33534935116767883, -0.4999676048755646, -0.6841757893562317, 2.2628798484802246, -0.10578779131174088, -0.7583551406860352, -0.6748687028884888, -0.3236024081707001, 3.0099995136260986, -0.40704330801963806, -2.5389623641967773, -0.6986663341522217, 0.9225330948829651, -1.2941079139709473, -1.4397424459457397, -1.138526201248169, -1.7078617811203003, 0.5827255845069885, 0.26221442222595215, -0.6439616084098816, -0.1724129617214203, 1.9012057781219482, -0.8347561955451965, -0.12848375737667084, -1.1616650819778442, -1.7367604970932007, 0.07380274683237076, 1.0336240530014038, -2.286067008972168, -1.1295965909957886, -0.5580500960350037, -0.4611435532569885, 0.582783579826355, 0.7199530005455017, -1.3705905675888062, -0.8199424147605896, 0.8672292232513428, 0.8476735353469849, -0.3024599552154541, 0.5737324357032776, 0.5181020498275757, 0.15860119462013245, 0.35858383774757385, 0.49480774998664856, -0.4621334373950958, 0.7953224182128906, 0.8179300427436829, -0.7336096167564392, 1.4711472988128662, -1.8074417114257812, 0.49552789330482483, 1.108214020729065, -0.48351985216140747, 0.7149069905281067, 1.3325783014297485, -0.1445847600698471, -1.2420637607574463, 1.2482105493545532, -0.15431202948093414, -0.24906976521015167, 0.6685494184494019, -1.3594329357147217, -1.4307633638381958, -0.8128769993782043, -0.5304649472236633, 0.02435750514268875, -0.3665580451488495, -0.6936466097831726, -0.9062595367431641, 0.44697093963623047, 0.2871852517127991, -0.8900142908096313, 1.8374501466751099, 1.5408002138137817, -0.18767628073692322, -0.3486323654651642, -1.6919670104980469, -0.31705614924430847, 0.6750995516777039, -0.3236399292945862, -0.9099204540252686, 0.3740460276603699, 1.6045434474945068, 1.174436092376709, 1.0033150911331177, 0.6425269246101379, -0.7716918587684631, 0.564754843711853, 0.4484487771987915, -1.2210688591003418, -0.05384698882699013, 1.8734030723571777, 0.5905092358589172, -0.6492644548416138, -0.568516194820404, -0.6329686641693115, 0.7845409512519836, -1.5583761930465698, -0.8506929278373718, 1.166671872138977, 0.42613479495048523, -1.2788848876953125, -0.23271934688091278, -0.3317098617553711, 0.44058507680892944, -0.20962458848953247, 0.5144694447517395, 0.8319979310035706, 2.2324914932250977, 1.6833930015563965, 0.22892138361930847, 0.7772163152694702, 1.2296156883239746, 2.041104793548584, 0.671130895614624, -1.512871503829956, 0.5272930264472961, -0.0993187353014946, -0.35970598459243774, -0.6319133043289185, -1.3097965717315674, -0.3941504955291748, 1.7959412336349487, 1.7095720767974854, -0.4292951822280884, -0.3624739348888397, 0.1290391981601715, 1.2237526178359985, -0.30144545435905457, -0.32982221245765686, -1.2092105150222778, -0.5977453589439392, 0.6396845579147339, 1.379677653312683, 0.1989961713552475, -0.9332203269004822, -2.0298500061035156, 0.09196201711893082, 0.7485989928245544, -0.04610402137041092, -0.7875066995620728, 0.7400210499763489, -0.5464248657226562, -2.106003761291504, 2.192491292953491, 0.337310254573822, -0.36031201481819153, -1.155800461769104, 0.2098889797925949, -0.9886868000030518, 0.2529679238796234, -1.9587926864624023, -0.09454610198736191, 0.06275244057178497, -0.2122606784105301, 0.7925456166267395, -1.6271932125091553, 1.2482646703720093, 1.4108160734176636, -0.4437984228134155, -0.09013668447732925, 0.4104502201080322, -0.4346269965171814, -1.6834224462509155, 2.208730936050415, 2.653225898742676, -0.6494159698486328, 0.23611964285373688, -0.869462788105011, -0.7770438194274902, -0.8732362985610962, 1.0800049304962158, 0.2145436555147171, -0.21410301327705383, -1.6882357597351074, -1.8069241046905518, -0.5937614440917969, 0.10834548622369766, -1.0760908126831055, 1.4063432216644287, 0.019520778208971024, -0.6140173673629761, -0.8236308097839355, -1.321584939956665, 1.157590627670288, 0.5774659514427185, 1.0599302053451538, 0.25822848081588745, -0.28347164392471313, -1.204727053642273, 1.0898090600967407, 1.3155370950698853, 0.14038977026939392, -1.472107172012329, -0.6649370193481445, -0.9340991973876953, -0.9549726247787476, 0.714497447013855, 0.4380398094654083, -0.7534487247467041, 0.6242377758026123, 0.8569694757461548, -0.2332182228565216, -0.8009580373764038, 1.186505913734436, -1.5957647562026978, -2.6149983406066895, 1.5789512395858765, -1.0011391639709473, -0.4270896315574646, 1.1578031778335571, -0.7728221416473389, 0.2713097035884857, -0.1777690052986145, -0.7348995804786682, 1.43783700466156, -1.0432523488998413, -0.23483338952064514, -1.058267593383789, -0.746665358543396, -1.4014331102371216, -1.1051362752914429, -0.2352161705493927, 0.702073335647583, -0.7270933985710144, -1.0122270584106445, 0.30552780628204346, -0.7016685009002686, -2.3703510761260986, 0.852958083152771, -0.6522437334060669, -0.7491982579231262, -0.14414972066879272, -0.6115169525146484, -0.21399633586406708, -1.4026151895523071, 0.04072151333093643, 0.8303053975105286, -0.8073359131813049, -1.0913711786270142, -0.002310836687684059, -0.44831252098083496, -1.418282389640808, -0.5621272921562195, 0.444398432970047, 0.038037486374378204, 1.2775592803955078, 0.6637772917747498, -0.8417673110961914, -0.5893366932868958, 0.8806342482566833, 0.5474004745483398, -1.0726242065429688, -1.2171831130981445, -0.08412226289510727, -1.4100672006607056, -0.31454920768737793, -0.5977087020874023, -0.35344332456588745, -1.2070033550262451, 0.604493260383606, 0.16445407271385193, -0.2149014174938202, -2.019721746444702, -1.308367371559143, -0.3647487759590149, -0.2816782593727112, 2.206902503967285, -0.1255064308643341, 1.6105588674545288, 0.8561927676200867, -0.6415802240371704, -0.5200788974761963, -0.356049120426178, -1.4494370222091675, 0.7707042098045349, 0.3662729859352112, 2.003885507583618, -1.015129804611206, 1.0562142133712769, 1.3883572816848755, 1.8674383163452148, 0.16998283565044403, 0.4789107143878937, -1.757373332977295, -0.05945715680718422, -0.03383232280611992, 1.0065237283706665, 0.921420693397522, 1.1667218208312988, -1.367929220199585, 0.46790117025375366, -0.40318524837493896, 0.8551903963088989, -0.4954962730407715, -1.7410056591033936, 0.9599215388298035, -1.2361559867858887, -0.7111725807189941, -1.259419560432434, 0.09342775493860245, 2.565753698348999, 0.20255859196186066, 0.30013421177864075, -0.5726140141487122, -1.925240397453308, -0.6027930974960327, 0.8972190618515015, 0.5915580987930298, 1.0450464487075806, 0.5508894324302673, 1.9433192014694214, 0.7972091436386108, 0.35746121406555176, 1.0623533725738525, -0.5124432444572449, 0.12818309664726257, -1.3458609580993652, -0.2136048674583435, 0.015104547142982483, 2.527599811553955, 0.4167637825012207, 0.5742066502571106, -0.2917144000530243, -0.08598190546035767, 1.6593570709228516, -0.10968903452157974, -1.356704831123352, 0.004105947446078062, 1.0960451364517212, 0.8893415331840515, -1.2153428792953491, 0.6938517093658447, 1.6778960227966309, -0.24877935647964478, 0.98475581407547, -0.11245088279247284, -0.03342510759830475, -0.09984976053237915, 0.6288005709648132, -1.5165395736694336, -1.5624210834503174, -0.36996814608573914, 0.5614922642707825, 0.41432276368141174, 1.2903716564178467, -0.14599113166332245, 0.07506249099969864, 0.3812972605228424, 1.2657569646835327, 0.5024800896644592, -1.7339708805084229, -1.0738335847854614, -1.6920212507247925, -0.5947580337524414, 1.1940268278121948, 2.4907097816467285, -0.7993053793907166, 0.9390601515769958, 0.7215223908424377, -0.5697020888328552, 0.5820100903511047, 0.5064589977264404, 0.7219064831733704, -1.1986167430877686, -0.31072404980659485, -0.010954956524074078, -0.3289388418197632, 0.32029157876968384, 0.6156783699989319, 0.9341433048248291, -0.42716118693351746, 1.903348684310913, 1.0031425952911377, 0.05529339984059334, -1.4362603425979614, 0.24569915235042572, -0.09983693063259125, 0.18754945695400238, 1.2780671119689941, -0.38306447863578796, 0.2679971158504486, -2.2179625034332275, -0.7384338974952698, -0.32979676127433777, -2.161223888397217, 0.37533634901046753, 0.6245958805084229, 0.1358243077993393, -0.7890057563781738, 1.249617576599121, 2.0578174591064453, 0.36757156252861023, -0.8522510528564453, 0.7016347646713257, 1.2451319694519043, 0.25543972849845886, -0.6972241401672363, -1.459831953048706, -0.04190054535865784, -1.6240307092666626, 0.3670307397842407, -1.7478102445602417, 0.3372901380062103, 1.208937406539917, 0.9016940593719482, 0.3188225030899048, 0.1292741596698761, -0.918773353099823, 0.1923145204782486, -1.4949878454208374, -0.885363757610321, -0.8143430948257446, -0.284917950630188, 0.5866751670837402, -0.2004280537366867, 1.2089606523513794, -1.044711709022522, 0.9490599632263184, -0.0320226289331913, -0.6902804374694824, -0.30797865986824036, -0.6153804659843445, 0.440289705991745, -1.4860446453094482, 1.5759462118148804, -1.3736977577209473, 0.20376254618167877, 1.1392892599105835, -0.016098065301775932, 0.8965520262718201, 0.5972204804420471, -1.2481391429901123, 0.847864031791687, 0.507093071937561, 0.004574816208332777, 0.43622204661369324, 0.20153014361858368, 0.23575609922409058, -0.15653333067893982, 1.4350674152374268, 1.69570791721344, -0.27175015211105347, 0.8932508230209351, 0.32999885082244873, -0.8299933075904846, -1.6172704696655273, 0.38054952025413513, 1.0762057304382324, -0.26020339131355286, -2.537686824798584, -0.32086801528930664, 1.169877052307129, -1.7515699863433838, -1.295980453491211, 0.4527696669101715, -0.274565726518631, 0.47609004378318787, 0.1561158448457718, -1.2257264852523804, 0.7913513779640198, -2.016502618789673, -0.09547408670186996, 1.039725422859192, -0.0018053579842671752, 1.0892096757888794, -1.1684648990631104, 0.1768995225429535, 0.8500595688819885, -2.0690555572509766, 0.04501093924045563, -0.36410775780677795, -0.09569225460290909, 1.0516935586929321, -0.5976543426513672, 0.48624399304389954, -0.6219214797019958, 0.9555802941322327, -0.10221625119447708, 0.35761335492134094, 0.40175095200538635, 0.08399287611246109, -1.4404507875442505, -0.5282246470451355, -0.8177395462989807, -0.24691367149353027, -1.8727706670761108, 0.3570689857006073, -0.09593354910612106, -0.16876699030399323, -0.2883129119873047, 1.9293431043624878, 0.031995028257369995, 0.3353049159049988, -1.4038589000701904, 0.4313630759716034, 0.09056846052408218, 0.8241365551948547, -0.2976261079311371, -0.8641293048858643, -1.989861249923706, 0.04086582735180855, 0.051334403455257416, -0.1890556365251541, 0.901584804058075, -0.6371227502822876, -0.06626386195421219, 0.47960785031318665, 1.1145561933517456, 0.8154749870300293, -0.6759610772132874, -0.6667470335960388, 1.7647807598114014, 2.0181691646575928, -1.7710825204849243, -0.45362234115600586, 1.1700191497802734, -0.05280255526304245, 0.630989134311676, 0.7166973948478699, -0.955930769443512, -1.340077519416809, -0.6774839758872986, 0.007600185927003622, -1.5905994176864624, -1.500842809677124, -3.002199649810791, 0.6869736313819885, 0.23745441436767578, -1.6139229536056519, 0.3025754690170288, 0.1657802015542984, -0.008740529417991638, 0.778451144695282, -1.642290711402893, -0.4229128658771515, -1.1338613033294678, 0.4619981348514557, 0.0035256652627140284, -0.6586306691169739, 0.09306084364652634, 0.2474280297756195, 0.9814445376396179, -0.23268871009349823, -0.7592658996582031, 0.8919497132301331, 1.3677842617034912, -0.15516313910484314, -0.15917016565799713, -0.9412505030632019, 1.5610958337783813, -1.0396407842636108, 0.2748725414276123, 1.1486777067184448, 1.1386561393737793, -0.3538164794445038, 1.9107459783554077, 2.1722025871276855, -1.8018560409545898, -0.0984460860490799, 0.036655448377132416, 1.6873794794082642, -0.9389690160751343, 0.8236597180366516, -1.1403368711471558, -1.7098137140274048, 2.3009860515594482, -0.12072526663541794, -0.4513310194015503, 0.7239876389503479, -0.30821824073791504, -1.0920330286026, -0.44419804215431213, -1.2396637201309204, -1.4285305738449097, -0.18012653291225433, 0.0884627178311348, 0.5245686769485474, 1.0670194625854492, 0.010701705701649189, -1.766917109489441, -2.3336236476898193, 0.41130778193473816, 1.8998339176177979, 0.5088104009628296, -0.15795236825942993, 0.27364686131477356, -1.6614444255828857, -0.377606600522995, 1.825447916984558, -1.0574921369552612, 0.5357695817947388, 1.2258225679397583, 0.4971548616886139, -1.066679835319519, 1.3436638116836548, 0.4908067286014557, 0.19916418194770813, -0.8960634469985962, 0.023046908900141716, 0.22419488430023193, -0.6960682272911072, -0.9401822686195374, -0.12815889716148376, 1.4037179946899414, 1.5395410060882568, 0.9486067891120911, -0.28894951939582825, 1.4129678010940552, 0.8933916687965393, 1.5823967456817627, -1.4124423265457153, -0.8219270706176758, 0.5586780309677124, -0.6713026165962219, 0.8909149765968323, 0.2933392822742462, 1.0649725198745728, 0.7684191465377808, 1.3414287567138672, 0.4705768823623657, -1.1906120777130127, 2.246572732925415, -1.048828363418579, 0.009692623279988766, -0.7389037013053894, -0.43882399797439575, -1.9343684911727905, -0.19431182742118835, 1.4680507183074951, -0.7099863290786743, 0.10234977304935455, -0.16144207119941711, -0.058352116495370865, -0.8223282098770142, 1.8559577465057373, -0.7699140906333923, -0.34462735056877136, 0.9313482046127319, 0.24990248680114746, -0.060692835599184036, 1.6442950963974, -1.298634648323059, 0.5256156921386719, 0.09986117482185364, 0.15878237783908844, 0.35176512598991394, 1.6310410499572754, 0.48134922981262207, -0.659546434879303, -0.3364173173904419, 0.25870025157928467, 0.5094437003135681, 1.2817109823226929, -2.634082317352295, -0.4982872009277344, 0.16314725577831268, 1.7570583820343018, 0.6751593351364136, -1.5580503940582275, -0.04594743996858597, 0.25067877769470215, -0.8214784860610962, 0.33805978298187256, -1.5957056283950806, -1.154863953590393, -0.478890985250473, 0.7776909470558167, 0.8185914754867554, -0.29445043206214905, -1.4021632671356201, -2.244861125946045, 1.051709532737732, -1.1511625051498413, 0.05002979934215546, -0.23931509256362915, -0.10230784118175507, 0.7062855362892151, 0.2691737115383148, 0.7774552702903748, 0.041421983391046524, -0.8548024892807007, 1.2016476392745972, -0.05804119259119034, 0.5403071641921997, 1.7361130714416504, 0.5747821927070618, -0.12052807956933975, -1.33688223361969, -0.031917642802000046, -0.8576122522354126, -0.9056530594825745, -1.6404279470443726, -0.43815523386001587, 1.271417498588562, 1.8795166015625, 1.0437440872192383, -1.9685040712356567, -0.15008166432380676, -0.6705852746963501, -0.2953629195690155, 1.2564997673034668, -0.06220570206642151, 1.267244577407837, -0.17161038517951965, -0.11103358864784241, -0.040029000490903854, -0.6330991983413696, 0.1966765820980072, 0.5473304986953735, 1.2151914834976196, -0.902387261390686, 0.34654220938682556, 1.3887715339660645, 0.716229259967804, -0.45004233717918396, -0.3074371814727783, -0.531911313533783, -0.5152235627174377, -0.9800752401351929, -0.9501180648803711, 1.1166906356811523, -2.3157458305358887, -0.2582016587257385, 0.5585215091705322, -2.7674858570098877, -1.2807005643844604, -2.762018918991089, 1.8375811576843262, 0.7684749364852905, 1.2129054069519043, 0.035786062479019165, -0.14337466657161713, -1.0786207914352417, 0.46673569083213806, 1.5447100400924683, 0.039076417684555054, -0.823155403137207, -2.149456739425659, -0.1919407695531845, -0.6273325085639954, 0.3140566051006317, -0.1864229142665863, 0.06920933723449707, -1.1815714836120605, -0.9733943939208984, 0.5056257247924805, -0.5590766072273254, 1.0061980485916138, 0.20889294147491455, 0.6488471627235413, -2.1373562812805176, 0.28890636563301086, 0.7758796811103821, -1.2341370582580566, -0.6083175539970398, 0.6095730662345886, -0.9110975861549377, -1.1200593709945679, -1.7042253017425537, 0.8336054086685181, -0.14624260365962982, -0.31419554352760315, 0.6850029826164246, -1.1722664833068848, 0.9972949028015137, 0.5529854893684387, 0.20231151580810547, 1.657266616821289, 1.0576726198196411, 0.34006860852241516, -0.7999026775360107, -1.8513845205307007, 0.4252966344356537, -0.10294458270072937, 1.6460611820220947, -0.436185359954834, -0.4968474209308624, 1.2541415691375732, -1.060692548751831, -1.493060827255249, 0.16676850616931915, -1.0992836952209473, -1.4357305765151978, -0.12065533548593521, 1.157849907875061, -1.5587385892868042, -0.409560889005661, 0.9275953769683838, 0.18606817722320557, -0.33690115809440613, -0.17872175574302673, -1.0870559215545654, 0.4449760615825653, 1.7573490142822266, -0.06249551102519035, 0.45483940839767456, 1.2756823301315308, 1.4123115539550781, 0.8135010004043579, 0.8236956000328064, 1.2222943305969238, 0.3799934685230255, 0.7823076248168945, -0.5520236492156982, -0.7488607168197632, 1.1679362058639526, -0.3645915687084198, -0.8735913634300232, -0.1333976835012436, 0.2176683247089386, 0.7902646064758301, -1.172768235206604, -0.7065898776054382, -0.565460205078125, 0.3530113697052002, 0.19316671788692474, 0.18655788898468018, -1.0029491186141968, -1.4359872341156006, 1.080255389213562, 0.9776252508163452, 0.2832484245300293, 0.37915998697280884, 1.2262464761734009, 3.359560251235962, -0.7276012301445007, 1.0709894895553589, 0.02258571796119213, -0.8830220103263855, 1.5004855394363403, 0.2716045677661896, 0.9268195033073425, -0.24050623178482056, -0.8177423477172852, 0.8479368686676025, 0.21011783182621002, 0.29533469676971436, -0.6475628018379211, 1.2459410429000854, 1.124037504196167, 1.691141128540039, 0.5874925851821899, -0.23706848919391632, 0.3924259841442108, 1.7157918214797974, 0.7409162521362305, -0.683082640171051, 0.964490532875061, -1.241579532623291, -0.34457525610923767, -0.15838903188705444, 0.5096966624259949, -0.6617922782897949, -1.325006127357483, 0.3814921975135803, 0.891848623752594, 1.301338791847229, -1.4517734050750732, 0.2149825394153595, -0.6394604444503784, -0.5203507542610168, -1.026824712753296, 0.17456644773483276, -0.8631824254989624, -0.8851416707038879, 1.4037305116653442, 0.28066912293434143, 0.3963836133480072, 0.10287480056285858, 0.34565073251724243, -0.7417229413986206, 0.37753400206565857, -0.21765734255313873, 0.20710043609142303, 0.17649418115615845, 0.9895998239517212, 0.9333333373069763, 0.9971179366111755, -0.382110595703125, -0.576208770275116, 0.45314326882362366, 0.2077132761478424, 0.03239365667104721, 1.1319235563278198, 0.8005774617195129, -0.46034350991249084, 1.258739709854126, -0.9269810318946838, 0.5329424142837524, -1.3906954526901245, 0.4090029001235962, 0.23488256335258484, -0.6262454986572266, 0.6171785593032837, 0.9776162505149841, 0.8078371286392212, -1.0022897720336914, -1.239546537399292, -1.2401808500289917, -0.5834358334541321, 0.7653177380561829, 0.12071768939495087, -1.1207982301712036, 0.024753250181674957, 1.0415537357330322, -0.016883883625268936, 1.1260477304458618, 0.3765961825847626, 2.143871545791626, -1.5592668056488037, -1.1781913042068481, 1.1631333827972412, 0.261922150850296, -0.47727152705192566, 0.7130506038665771, 0.34716320037841797, -0.2335527241230011, 2.462507963180542, 0.4617755711078644, -0.45172134041786194, -1.3854738473892212, 0.5822807550430298, 1.7001667022705078, -0.5039845108985901, 0.11082646995782852, 1.1079695224761963, 0.18031229078769684, 0.17825530469417572, 0.24127709865570068, 0.5010289549827576, 0.9766380786895752, -0.8958346843719482, -1.1332876682281494, 0.5440294742584229, -1.4255614280700684, 1.8086024522781372, -1.6524406671524048, -2.0234668254852295, 0.12981955707073212, -0.26118704676628113, -1.0403646230697632, -0.09211572259664536, 0.898001492023468, 0.9604272842407227, -0.5929422378540039, 0.7325257658958435, -0.17879949510097504, -0.4463299810886383, 0.2838827073574066, -0.28413650393486023, 0.13726791739463806, 0.4648890495300293, 1.1253938674926758, 0.5822845697402954, 0.8214088678359985, -0.2879987359046936, -0.6896700263023376, -0.7314088344573975, -0.26499080657958984, 1.428226351737976, 0.02140023000538349, 1.012557029724121, -1.0466668605804443, -1.9501194953918457, -0.4359605312347412, -0.06177348643541336, 0.2974265217781067, -0.6586852073669434, 0.38113197684288025, -0.15004155039787292, 0.08787693083286285, -1.1557122468948364, -1.9386513233184814, -0.00015967532817739993, -1.2746572494506836, 0.9402351379394531, -0.16447444260120392, 1.1111425161361694, 1.1788930892944336, -1.463965892791748, -0.9919939041137695, -2.3249385356903076, 0.19762593507766724, 0.8111611008644104, -0.9611514210700989, -0.22367754578590393, -0.14831459522247314, 1.957632303237915, 2.5588550567626953, 0.856214165687561, -0.750924289226532, -0.7020028829574585, -0.9133106470108032, -0.524141252040863, 0.5282023549079895, 0.2302878201007843, 1.5637896060943604, -0.8552875518798828, -0.49564316868782043, 0.513670027256012, -0.19159157574176788, 0.11043108999729156, -1.1370675563812256, -0.6086580157279968, 0.520796537399292, 1.000862956047058, 0.7390629053115845, -0.9872462749481201, 0.7234092354774475, 0.3119712471961975, 0.4654558002948761, 0.8864513039588928, 0.4448601305484772, -1.057531714439392, -1.5122584104537964, 0.2505292594432831, 0.7059178948402405, -0.4696640074253082, -0.10273589938879013, 1.3718239068984985, 1.3913713693618774, -1.1481587886810303, 0.39273983240127563, 1.5709664821624756, -0.6541017293930054, -0.7511018514633179, -0.4502876102924347, -0.8199556469917297, -1.1411250829696655, 0.9906128644943237, -0.7869009375572205, 0.09413956105709076, -0.6833510398864746, 0.6965147256851196, 2.6921372413635254, -0.7310948371887207, -0.5934433937072754, 1.4212069511413574, 1.2186459302902222, -0.2779286503791809, 0.9263746738433838, 0.51613849401474, 1.01667058467865, -0.6126538515090942, -0.13283349573612213, 0.12148554623126984, 1.805517315864563, -1.1757512092590332, 0.024086682125926018, -1.2160248756408691, -2.1160669326782227, 0.40555205941200256, 0.050066541880369186, 1.0164666175842285, -0.18841908872127533, 0.15409207344055176, 0.41876116394996643, 0.03446429595351219, -1.756857991218567, 0.06540276110172272, -0.2795528173446655, 0.19494961202144623, 0.2697261571884155, 0.02481650747358799, 0.698285698890686, 0.4593113958835602, -0.9147065281867981, 0.9834845662117004, 2.359156608581543, -0.5105223655700684, 0.679591715335846, 0.7328019142150879, -0.5793259739875793, 0.6348965167999268, -1.6138495206832886, -0.7782665491104126, -0.616104781627655, -0.7367480993270874, -0.6455368995666504, -0.9014098644256592, 0.49199771881103516, -0.8815677165985107, 1.7860288619995117, -1.0920116901397705, -0.5786010026931763, 0.8707794547080994, -0.7183435559272766, -0.6278300881385803, 0.5939534306526184, -0.009925173595547676, -1.8749475479125977, 0.5560495257377625, -0.10616941004991531, 0.10108854621648788, 0.47003573179244995, -0.9589958190917969, 0.20276939868927002, -0.3674057126045227, 0.5922255516052246, -1.408115029335022, 0.20607030391693115, -2.5638325214385986, 1.215911626815796, -0.5491800308227539, 0.1399294137954712, -2.8400986194610596, -0.06173955276608467, -0.7326983213424683, -1.0026980638504028, 1.3095511198043823, -0.6056228280067444, -0.8121423721313477, 0.6410218477249146, -0.7027285099029541, -0.7540442943572998, 1.5477434396743774, 0.2721392512321472, 1.5442278385162354, -0.03556198254227638, 0.6486923098564148, 0.06622973084449768, -0.9124636054039001, 0.21650366485118866, 1.1946280002593994, -1.587971806526184, -1.8187716007232666, -1.3168350458145142, -1.4184290170669556, 1.6448872089385986, 0.10281272977590561, 0.316102534532547, 0.4040720760822296, 0.08228995651006699, -0.9483690857887268, 0.10335992276668549, -1.4430886507034302, -0.971853494644165, -0.37838077545166016, -1.1127526760101318, 1.6752079725265503, -0.6884396076202393, 0.7128461003303528, 1.109796404838562, 0.4442775845527649, 0.11880246549844742, 0.5283495783805847, -0.16595962643623352, 0.5758660435676575, 0.5777161717414856, 0.7134074568748474, -0.2284938097000122, 0.5382705330848694, -0.14595139026641846, 0.8582187294960022, -0.35967493057250977, 0.40939897298812866, 0.6676429510116577, 1.0429915189743042, 1.7760881185531616, -0.18541960418224335, 1.6338341236114502, 0.6423907279968262, -2.1561357975006104, 0.4959787428379059, -1.2413772344589233, -0.5619189739227295, 0.3517179787158966, 0.36385366320610046, -1.8750685453414917, -1.2259728908538818, 1.0638740062713623, 0.290790855884552, 1.3042899370193481, 0.7265223860740662, -0.3202974200248718, -0.16426849365234375, -0.7623098492622375, -0.23489274084568024, -0.43778181076049805, 0.3344995677471161, -0.041814401745796204, 0.3950318992137909, -1.701963186264038, 2.163285732269287, 0.19242477416992188, -0.7274135947227478, 0.6228283643722534, -0.03182763606309891, -0.9787357449531555, 0.15487773716449738, -2.361238956451416, 1.5885236263275146, -0.7669068574905396, 0.2452811449766159, 0.5393256545066833, 0.8961138129234314, -0.42831411957740784, -2.036466360092163, 0.5916923880577087, -0.42489901185035706, -0.5805202722549438, -0.870010256767273, 1.1890101432800293, 1.0272541046142578, -0.02070721611380577, -1.7771462202072144, -1.564596176147461, 0.5973892211914062, -0.017067326232790947, 1.2849923372268677, 2.0909416675567627, -1.163489580154419, -1.2922227382659912, -1.4824934005737305, -0.5285163521766663, -0.3693801462650299, -2.6982390880584717, 0.6525295972824097, -1.0846734046936035, -1.3291667699813843, -0.628230631351471, 1.3671002388000488, 0.44759801030158997, 1.1176255941390991, 0.47455817461013794, 1.0445348024368286, 0.4809991121292114, -0.24230559170246124, -0.5553304553031921, 1.0153727531433105, 1.3059760332107544, -1.0710867643356323, -0.8671207427978516, 0.2508418560028076, 2.2012040615081787, -0.26332971453666687, 0.59937584400177, 0.2328159660100937, 0.14038598537445068, -0.4632577896118164, -1.2564665079116821, 2.178417921066284, 1.5803202390670776, -0.9954674243927002, -0.6687318682670593, -0.31831300258636475, -0.3256939947605133, -0.5402020812034607, -2.0049045085906982, -2.109830617904663, 1.0694737434387207, -2.2307827472686768, 0.7467479109764099, -0.7774708271026611, 0.3532690405845642, -0.1288551539182663, -0.9526813626289368, 0.9254611730575562, -0.4359113574028015, -0.9465050101280212, -0.21940912306308746, 0.28898295760154724, 0.15597416460514069, -0.6103930473327637, -0.9817754030227661, 0.09759725630283356, -0.6335011720657349, 1.3356194496154785, -0.17976704239845276, 0.711273193359375, 1.3627961874008179, -0.10261223465204239, 0.10470498353242874, 1.0964542627334595, 1.0810954570770264, 0.5404115319252014, 1.012840747833252, 0.20461948215961456, 0.9877016544342041, -0.9669076800346375, -0.8985689878463745, -1.4284387826919556, 0.1457185447216034, -1.4537333250045776, -0.8696498274803162, 0.8905627727508545, -0.16828563809394836, -2.2383759021759033, -0.7783002257347107, 0.36963263154029846, -0.6392905116081238, 2.1431565284729004, 0.6811811923980713, 1.1187595129013062, 0.05099422112107277, 0.08476118743419647, 0.03367258235812187, -0.1281299740076065, 0.4470687210559845, 0.02800876274704933, -0.5380110740661621, -0.8974734544754028, 2.3820064067840576, 0.13411466777324677, 0.7821530103683472, -0.30569103360176086, -1.1288846731185913, 0.2953788936138153, -0.26778748631477356, -1.2905436754226685, 1.4989114999771118, -1.0546576976776123, 1.2514723539352417, -0.09111876785755157, -0.71580970287323, -1.234808325767517, -0.07956036925315857, 0.23224592208862305, 0.8707152605056763, 0.10038288682699203, -0.4354560971260071, 0.3001505434513092, 0.20210778713226318, 0.24828054010868073, 0.09491096436977386, -0.697160542011261, -0.004981425125151873, 0.7016068696975708, -0.6409175992012024, -0.7521800398826599, -0.8412142992019653, 0.8522907495498657, 0.41756242513656616, -0.16777299344539642, -0.5532241463661194, 1.3489792346954346, -0.5619823932647705, -0.9100655913352966, -0.3008497655391693, -0.021512174978852272, -0.8703351616859436, 0.29100584983825684, -1.0235066413879395, 1.7746678590774536, 0.5202534794807434, 1.4587639570236206, 0.23859824240207672, 0.6843814849853516, -0.7130135297775269, -1.1109521389007568, 0.34125855565071106, 0.9753087162971497, -0.9749577641487122, -0.8232674598693848, -0.11889780312776566, -0.9867536425590515, -0.5538971424102783, 0.17818580567836761, 1.0453267097473145, -0.8772104382514954, -1.1363050937652588, 0.8679503202438354, 0.01217023003846407, 0.463624507188797, 0.7550807595252991, -0.16076095402240753, -0.14699453115463257, -0.6245474815368652, -0.5971857309341431, -1.7711023092269897, -0.6972783207893372, -0.7435272932052612, 0.07666053622961044, 0.6344372034072876, -0.5403833985328674, 0.5359576344490051, -0.15449003875255585, 1.093173623085022, -1.7107986211776733, 0.7189428806304932, 0.3941377103328705, -2.426830768585205, 0.8093389272689819, -0.8910380005836487, -0.6781073212623596, -0.0800648108124733, -1.523280143737793, 1.3162847757339478, -0.061901845037937164, 0.39769884943962097, -0.7539719343185425, -1.8062455654144287, -1.046547293663025, -0.5449837446212769, -0.2919315695762634, -1.8198583126068115, 0.4132809638977051, 0.8148183822631836, 1.8281090259552002, 0.6837531924247742, 0.33767902851104736, -0.996096134185791, -1.2448078393936157, 0.0038379912730306387, 0.570871114730835, -0.761396050453186, 1.5430448055267334, -0.4391028881072998, 0.03573012724518776, -0.25099486112594604, -1.3052163124084473, -2.2527503967285156, -1.4765053987503052, 0.3984096646308899, -1.0585334300994873, 0.9083380699157715, -0.5426583290100098, -0.14274142682552338, -0.44636479020118713, 0.07973667979240417, -0.1838381588459015, 0.15205435454845428, 1.6557286977767944, 1.317256212234497, -0.3184155821800232, -0.38112980127334595, 1.9183669090270996, 1.2896385192871094, -1.3504184484481812, 0.10392899811267853, 0.06274321675300598, -1.1813219785690308, -1.758017659187317, -1.9098238945007324, 0.830424427986145, -1.5015780925750732, -1.066583275794983, -1.5996342897415161, -1.1393133401870728, 0.6178146004676819, 0.050661828368902206, 1.8185302019119263, 1.0251375436782837, -2.0022366046905518, 1.450249195098877, -0.8075181245803833, 1.1373732089996338, -1.0035878419876099, -0.25407665967941284, -1.7824269533157349, 0.4043385684490204, -1.5394129753112793, -0.5356598496437073, -0.03997942805290222, -0.37778183817863464, 0.5325892567634583, 0.001794707728549838, 1.8237305879592896, -0.847777783870697, -0.1426570564508438, -0.9084696769714355, -0.3859122693538666, 0.03920483961701393, 0.3615468740463257, 1.8657419681549072, -1.1164299249649048, -0.2940022349357605, -0.785771906375885, -0.3871087431907654, 0.07237089425325394, 0.7550081610679626, -0.27342915534973145, -1.3861221075057983, -1.8869637250900269, 0.4138050377368927, 0.7711696028709412, 0.5043236613273621, 0.9436148405075073, -1.3274885416030884, 0.28371769189834595, 0.4777013659477234, 0.36037442088127136, -0.005495603196322918, 0.4097369313240051, 2.1814076900482178, -2.2606682777404785, -0.012094025500118732, 0.8086610436439514, -0.3512220084667206, 1.3871334791183472, -0.7765453457832336, 0.9863468408584595, -0.8714328408241272, -0.3305581510066986, -1.8053683042526245, -1.2002333402633667, -2.280977725982666, 0.34259623289108276, -0.10207173973321915, -0.4230941832065582, -0.1579803079366684, -0.3775686025619507, -1.1593176126480103, 1.3254737854003906, -0.5825044512748718, -1.0808513164520264, 0.8337609767913818, -0.9420248866081238, -0.9053475260734558, 2.8581013679504395, -0.7069310545921326, -1.6811792850494385, 0.24422171711921692, -1.5118815898895264, 1.3367114067077637, 1.895815134048462, 0.08002526313066483, 1.5861223936080933, 0.32014456391334534, 0.9005822539329529, 1.5656038522720337, -1.3652112483978271, 0.32816630601882935, -0.7268204092979431, 0.06492400914430618, 0.656083345413208, 0.4150015711784363, -0.714597761631012, -0.9877383708953857, 0.8148292303085327, -0.5971531867980957, 0.16584375500679016, -0.027143215760588646, 1.1054259538650513, -0.08524824678897858, -0.6085793375968933, -2.0025346279144287, -0.5063125491142273, 0.7903473377227783, 0.07044210284948349, -0.29401910305023193, -0.5337202548980713, 0.18834039568901062, 0.5688636302947998, -1.5018961429595947, 1.037068247795105, 2.00862717628479, 0.6307168006896973, -0.530035138130188, 0.3283521831035614, -1.0139788389205933, -0.06328380107879639, 0.8156867623329163, 0.07981189340353012, -0.5957709550857544, -0.6727636456489563, 0.07920826971530914, 0.6782810688018799, 0.7570944428443909, -0.32867205142974854, 0.9168692827224731, -0.34766772389411926, 0.34583044052124023, 2.673323154449463, -1.0550929307937622, 1.3272128105163574, 0.260245680809021, -0.332759827375412, -1.2646054029464722, 0.18659082055091858, 1.1016688346862793, -1.009414792060852, 1.3091304302215576, -0.2816413640975952, -1.2876399755477905, 0.446384072303772, -0.9420422315597534, 1.2122762203216553, 1.6317174434661865, 0.6272619962692261, -0.053617268800735474, 0.674187958240509, -0.3540658950805664, -0.6979893445968628, -0.4235849678516388, 0.4081355929374695, 0.1718054711818695, -1.058276891708374, -0.5241135954856873, -1.2691456079483032, -0.13008424639701843, 0.017347263172268867, 1.072060465812683, -0.9163997769355774, -2.610062599182129, -0.624467670917511, -0.2577208876609802, -0.006616835482418537, 0.23407666385173798, -1.3031493425369263, 1.5939804315567017, 0.09676945209503174, 0.09038909524679184, 0.14897756278514862, 2.1775968074798584, 0.5467332005500793, -0.41368019580841064, 0.35691365599632263, -0.14950942993164062, 1.370558738708496, -0.5653912425041199, -0.07102754712104797, 1.0500051975250244, 1.5755048990249634, 0.1418997198343277, -0.3692878782749176, 0.8901623487472534, -0.016917049884796143, -0.177687406539917, -1.0644588470458984, -2.2090182304382324, 1.3802627325057983, 0.03735761344432831, 0.8506166934967041, 1.1280356645584106, -0.0763578936457634, -0.40222012996673584, -0.7050634622573853, -0.058193642646074295, 1.0681530237197876, -0.40721067786216736, -0.1124878078699112, 1.0564138889312744, -0.5487819314002991, -0.809691309928894, 1.2823681831359863, 0.4613783359527588, -0.4846721291542053, -0.7013352513313293, -0.16717690229415894, 1.1922074556350708, 1.1417160034179688, 0.032200928777456284, -1.00557541847229, 0.8815135955810547, -0.24719227850437164, -0.03901618719100952, -0.43634483218193054, -0.3919353783130646, 1.0173007249832153, -1.0612514019012451, 1.1914435625076294, 0.5554001331329346, 0.2278003841638565, 0.2464480698108673, 0.37707096338272095, 0.9923967719078064, 0.1152675524353981, -0.35105666518211365, -1.981080174446106, 0.051007021218538284, -0.9054762721061707, -1.4761728048324585, 0.8367070555686951, 1.5495054721832275, 2.8346972465515137, -0.23850400745868683, -2.0897271633148193, -0.10385230928659439, -0.686254620552063, 1.252413272857666, 0.5227578282356262, -0.5335485935211182, -0.3629585802555084, -0.8644883632659912, -0.6516571044921875, 0.37311452627182007, -0.49966755509376526, -0.21205595135688782, 0.02607034519314766, -0.4138663113117218, -0.16875013709068298, -1.3997602462768555, -0.6606732606887817, 0.588502824306488, -0.9322479367256165, 0.7239364385604858, 0.8207906484603882, -1.4285160303115845, -0.18521156907081604, 1.0027908086776733, 0.15678973495960236, 1.5080015659332275, 1.211391568183899, 0.8807414770126343, -0.9439582228660583, 0.6322152614593506, -0.3441959023475647, -0.6179279088973999, 0.002617017598822713, 0.19232116639614105, 1.320928692817688, 1.2652615308761597, 1.0418992042541504, -1.4049787521362305, -0.6381327509880066, 1.6064057350158691, 0.6397135853767395, 0.3456389009952545, 1.2989370822906494, -1.2696232795715332, -2.534909725189209, -0.42926883697509766, 0.16691268980503082, 1.0086699724197388, -0.5755780935287476, 0.9131196737289429, -0.9020318388938904, 1.788948893547058, -1.1105228662490845, 2.2723288536071777, 0.30778902769088745, -0.11203005164861679, 0.12357088923454285, 0.3415277600288391, 0.5234102010726929, 0.24505497515201569, 1.6467692852020264, -0.8357674479484558, 0.9539092779159546, -1.004725456237793, -0.34722375869750977, -1.5138624906539917, 1.178752064704895, -1.2031760215759277, 1.049098253250122, -0.17276421189308167, 0.916617214679718, 0.2533617615699768, 1.1480169296264648, 1.3781605958938599, 1.1233301162719727, -0.4294503331184387, -0.605792224407196, -0.029586931690573692, -1.1591033935546875, 0.2066049873828888, -1.399491786956787, -1.051032543182373, 1.2117102146148682, -0.8604816198348999, 1.5326085090637207, 0.1595052033662796, 0.6144322752952576, -1.4223896265029907, -1.7304010391235352, 0.05551653727889061, -0.20085327327251434, -0.13447318971157074, 1.0713517665863037, -1.2293628454208374, -0.1363113820552826, -0.24579043686389923, 1.119248867034912, -1.271493673324585, -1.3601821660995483, 0.16475103795528412, -1.0582612752914429, -1.4815505743026733, -0.27559494972229004, -0.28255975246429443, 0.9260219931602478, 2.1564066410064697, 1.3348699808120728, -0.8087329268455505, -1.3206983804702759, -0.16096718609333038, -1.5003366470336914, 3.6424548625946045, 1.3010526895523071, 0.6102164387702942, 1.2987347841262817, -0.9279198050498962, -1.8199456930160522, 2.8030197620391846, 0.7383705973625183, -1.3784620761871338, -0.11379104107618332, -0.4785395860671997, -0.4611830413341522, -0.9991621375083923, 0.7931641936302185, -0.5433307886123657, 1.0705466270446777, -0.30035752058029175, -0.10603407770395279, -0.581914484500885, -0.9595350623130798, -1.1075000762939453, 0.12242570519447327, 0.20785243809223175, 1.7348653078079224, -0.5704652070999146, -1.1263413429260254, -0.8444893956184387, 0.7240009903907776, -1.3078813552856445, 1.4089066982269287, 1.0765609741210938, -1.6265008449554443, 1.3271303176879883, -0.45300331711769104, -0.5750643610954285, 0.634796679019928, -0.16326187551021576, -1.1943984031677246, -0.7377910017967224, 0.1598386913537979, -1.158626914024353, 0.9102749824523926, 1.2123993635177612, 0.4003804624080658, 1.4313300848007202, 1.5781904458999634, 1.2242000102996826, 0.04739898070693016, 0.9744261503219604, -0.4583286643028259, 0.4202832281589508, 2.749218225479126, 0.653908371925354, -0.5136207938194275, 0.1532870978116989, -1.0116461515426636, 1.0433776378631592, -0.5973499417304993, 0.5726271271705627, -0.9972819685935974, -1.0303517580032349, -0.4292997717857361, 0.5647669434547424, 1.0804295539855957, 0.9166126251220703, 2.023862838745117, 0.6429977416992188, 1.857563853263855, -0.15971355140209198, 1.4329975843429565, -2.0913238525390625, 0.14252988994121552, -0.9889230132102966, 1.4336726665496826, 0.027351774275302887, 0.8322768211364746, 0.9903276562690735, -0.8632816076278687, -0.2541970908641815, -0.24472089111804962, -1.5885138511657715, 0.9970486164093018, 1.11134934425354, -0.887789785861969, -0.7127209901809692, 0.12881051003932953, 0.24171455204486847, 2.219749689102173, 0.44845008850097656, -0.3998327851295471, -0.6935982704162598, 0.42038556933403015, 0.8802998065948486, 1.5264177322387695, -1.2657568454742432, -1.7630702257156372, 0.008415630087256432, -0.6930082440376282, -1.3592950105667114, -0.6000601649284363, -0.34787797927856445, 0.31011489033699036, -1.3650158643722534, 0.8557608127593994, 0.5375962257385254, -1.9348915815353394, -0.20285089313983917, -1.2594517469406128, 0.2978666424751282, 1.292820930480957, 0.909295916557312, -0.6495066285133362, -0.48098739981651306, -1.8320376873016357, -0.29614734649658203, -0.9381012320518494, 0.19514106214046478, -1.5995218753814697, 0.28872978687286377, -1.5084538459777832, 0.1848360151052475, -0.5013142824172974, -1.7439460754394531, 0.9729804992675781, 0.004335097968578339, 0.23159915208816528, -0.32663658261299133, 0.7082340717315674, 0.5515954494476318, 0.20728006958961487, 1.191161036491394, -0.3290769159793854, 0.6979172229766846, -0.026775488629937172, -1.5261023044586182, 0.707150399684906, 3.0704498291015625, 0.5438535809516907, 0.41782817244529724, -0.19351568818092346, 1.41628098487854, 0.9637697339057922, -0.4606635272502899, 0.43602192401885986, 0.8282772898674011, 1.482751727104187, -0.07963433116674423, -0.6860815286636353, -0.31912124156951904, -0.03772209212183952, -0.08405639976263046, 1.740006446838379, -0.9837371110916138, 0.2824088931083679, -0.7813530564308167, 0.26823121309280396, -2.17521333694458, -1.5133275985717773, -1.7787461280822754, -0.4234842360019684, -0.2469015121459961, 0.47446373105049133, 1.3301231861114502, -0.23000353574752808, -0.8395584225654602, -1.0521347522735596, -0.30632439255714417, -1.5220659971237183, 0.10279440134763718, -0.7827377319335938, -1.205212950706482, 0.18349836766719818, -0.9580545425415039, -0.7356235384941101, 1.6351536512374878, -0.22864188253879547, 1.5738918781280518, 1.5810774564743042, -3.064661979675293, 0.6009278893470764, -0.4664711654186249, 1.000853180885315, 0.21021027863025665, -0.36678051948547363, 1.2978099584579468, -0.19579002261161804, -0.45668160915374756, 0.26418766379356384, -1.0135860443115234, 0.15522168576717377, 0.5127749443054199, -0.9266191720962524, 0.40615159273147583, -0.1602058857679367, -0.006895955186337233, -0.45754119753837585, -1.3812037706375122, 0.5932785272598267, 0.7948755621910095, 1.0344960689544678, 1.0889614820480347, 0.1358744502067566, 1.6811714172363281, 0.010192678309977055, -0.03381785377860069, 1.2092037200927734, -0.32811832427978516, 1.58463454246521, -0.5514552593231201, -0.5436336994171143, -0.1578323096036911, 0.19741250574588776, -0.18344064056873322, 0.5006260275840759, 0.33037856221199036, 1.8546195030212402, 0.3443124294281006, 0.5871171355247498, 0.28569847345352173, -0.0777873620390892, -0.8985700607299805, -0.46679702401161194, 0.2934540808200836, 0.37868064641952515, 0.38647639751434326, 0.2998493015766144, 0.5080679655075073, -1.803100347518921, -1.195557951927185, -0.4360758066177368, 0.35816434025764465, -0.5750501751899719, 0.5369077324867249, 0.7119899392127991, 2.2827041149139404, 1.0956131219863892, 0.3488242030143738, 0.9755170345306396, -0.40883681178092957, 0.7655999660491943, 0.19704800844192505, -0.9628446698188782, -2.5225589275360107, 1.1868258714675903, 0.012942757457494736, 1.0044312477111816, 0.02299664355814457, 0.6905634999275208, -0.6715558767318726, -0.39479827880859375, -1.6451025009155273, 0.13808293640613556, -0.5658388137817383, -1.1421962976455688, -0.5719417333602905, -1.722902536392212, -1.1523668766021729, -0.5598150491714478, -0.9997580051422119, -2.119671106338501, 1.0287429094314575, -1.243393063545227, 1.6405056715011597, -0.3084242343902588, 0.9865890145301819, -0.08219582587480545, 0.6155139803886414, -1.352016568183899, 0.5404770374298096, -1.070096731185913, -0.10172349959611893, -0.3478488326072693, -1.619864583015442, 1.9343847036361694, 0.7882799506187439, 0.4218275845050812, -0.05906856432557106, 1.530916690826416, 0.44909486174583435, -0.3148914873600006, -0.8176770210266113, 0.5582621693611145, -1.0928637981414795, 0.21676868200302124, 0.8540431261062622, -0.4188459813594818, -0.07004285603761673, 0.47388437390327454, 0.9692321419715881, -0.37842029333114624, 0.005000499542802572, -0.5869782567024231, -0.6182237267494202, -0.30415335297584534, 0.3934626281261444, -0.029380058869719505, -1.5451456308364868, 0.444970965385437, 2.0384697914123535, -1.2708714008331299, -0.01277379970997572, 1.5102068185806274, 0.4794989824295044, 0.8541991114616394, 1.8390744924545288, -1.63916015625, -0.5717869997024536, 1.1194766759872437, 0.293803870677948, -1.2114579677581787, 1.1982122659683228, 0.1837008148431778, -1.0231151580810547, 0.11880753189325333, -1.8784147500991821, -1.1296778917312622, -1.5841127634048462, 0.4435405731201172, -0.6735369563102722, 0.24895307421684265, -1.3116934299468994, -1.6590769290924072, -0.716549277305603, 0.7507262825965881, -0.708905816078186, -0.7442852854728699, -1.0288641452789307, -0.1431969255208969, -0.3458869755268097, 0.758761465549469, -1.1876252889633179, 0.45528918504714966, 0.9586418867111206, 1.647570252418518, 0.8794518113136292, -1.4959241151809692, -0.30155354738235474, 0.9846367239952087, 0.011358903720974922, 0.7367664575576782, -1.173187017440796, -3.013183832168579, 0.3105575442314148, 0.33943846821784973, -0.7145607471466064, -2.274118423461914, -0.31898343563079834, 1.4358153343200684, -1.1435664892196655, -0.2194914072751999, -0.9392945170402527, 0.35198482871055603, 0.7316421866416931, -1.6718199253082275, -1.3712012767791748, 0.2893295884132385, 0.271902471780777, -0.47891274094581604, 1.3875051736831665, 0.7972767949104309, -0.5295795202255249, 1.1009471416473389, -2.4576663970947266, 0.266974538564682, 0.4125409722328186, -0.1639098823070526, 0.13533835113048553, 0.25397148728370667, 1.5447378158569336, -0.8261030912399292, 0.18791672587394714, -0.1878901869058609, -0.6962593793869019, 0.19840466976165771, -1.2423932552337646, 0.9416316151618958, -1.148492455482483, -1.2782516479492188, -0.04619783163070679, 1.2148855924606323, -1.215449333190918, 0.5974659323692322, -0.017430249601602554, -0.9600613117218018, 1.1107661724090576, 1.1308810710906982, 0.6345459818840027, -1.3197202682495117, 0.7335459589958191, -0.37762558460235596, -2.587258815765381, 0.006896595004945993, 0.9250770211219788, -0.4362098276615143, 0.2070080041885376, -1.2506266832351685, -1.0349934101104736, 0.3866380453109741, 0.7389402389526367, 0.4929719865322113, -1.194307565689087, -0.7822006344795227, -0.6946715116500854, -0.5037420988082886, 0.7724021673202515, -1.131152629852295, -1.3330676555633545, -0.7813221216201782, 1.8044856786727905, -1.1935968399047852, -1.4059323072433472, 0.8491751551628113, 0.7353292107582092, 0.6933942437171936, -0.09775301814079285, -1.5677381753921509, 1.0364774465560913, 1.1897610425949097, 0.26403743028640747, 1.4255732297897339, 1.0208112001419067, -0.43304771184921265, 0.7103679776191711, 0.11235737800598145, 1.1241769790649414, 1.0774534940719604, -0.6478458642959595, 0.9314941167831421, -1.521813988685608, 0.9207153916358948, 0.3516112267971039, 0.3762446343898773, 0.5822646021842957, 0.8731104135513306, -2.057333469390869, -0.9951698184013367, 0.1555175632238388, 0.40097764134407043, 0.585077166557312, -1.9203375577926636, 0.39029666781425476, -0.5862732529640198, -1.6530418395996094, 0.09354358911514282, 0.3227975368499756, -0.11080572754144669, 0.3471839129924774, -1.7874577045440674, -0.9934240579605103, -2.147852659225464, -0.87031090259552, 0.8681884407997131, -0.995093584060669, 0.2298102229833603, 0.23649050295352936, -0.4030381739139557, -0.35950568318367004, -0.437534898519516, 1.4719194173812866, 1.0655547380447388, 0.12968680262565613, -1.0849525928497314, 0.1639757752418518, -0.8243657946586609, 1.0220775604248047, -1.1210124492645264, -0.12400331348180771, -0.03864593803882599, -1.403907060623169, 0.13637836277484894, 0.12714534997940063, 0.44546544551849365, -1.2042293548583984, 0.18818677961826324, -1.4069082736968994, 0.9352275133132935, 0.5612426400184631, 0.2320445328950882, 0.4835570752620697, 0.5145813822746277, -1.5659795999526978, -0.13136400282382965, -1.9114751815795898, 0.0013800529995933175, 0.7571092247962952, 1.8951135873794556, 0.25744643807411194, -0.12284187227487564, 1.0199083089828491, -0.6944482922554016, 0.562954306602478, -0.36361628770828247, -0.25826242566108704, 1.6191600561141968, -1.2396966218948364, -0.4052458107471466, 0.11543948203325272, 0.5533826351165771, -0.11567230522632599, 0.9748139977455139, -1.0720057487487793, -0.15752965211868286, -0.8667445182800293, -0.03940471261739731, -0.8763309121131897, -0.3358305096626282, -0.3326266407966614, -0.16098865866661072, -0.5392288565635681, 0.22758030891418457, -0.7314132452011108, -0.5326951742172241, -0.048699162900447845, -0.5158448815345764, -1.0483691692352295, 0.5556336641311646, 0.25140663981437683, -1.067905306816101, 0.9130576252937317, -0.021525049582123756, 0.5967035889625549, 0.30641576647758484, 0.8645513653755188, 0.7945800423622131, -0.5165680646896362, -0.19210274517536163, 0.6558607220649719, -1.3299953937530518, -1.117109775543213, 0.5189467668533325, 1.2841241359710693, -0.008339553140103817, -1.249954104423523, 0.10016309469938278, -1.1643668413162231, 1.0609309673309326, -0.3776860535144806, -0.22442111372947693, -0.4237842261791229, -0.7856366038322449, 1.707154631614685, -0.061595622450113297, 0.2686266303062439, -0.10465610772371292, 1.4288382530212402, 0.5389992594718933, 1.002288818359375, 0.7685781121253967, 0.18842940032482147, 1.1253958940505981, 0.5282694101333618, -1.2169902324676514, -3.2485263347625732, -0.0870003029704094, -0.46581628918647766, -0.6542818546295166, 2.519803524017334, -0.4877564013004303, 0.7146628499031067, -0.13920338451862335, 0.9607893824577332, 0.3350616991519928, 1.0812371969223022, -1.062502145767212, -0.5767626166343689, 0.12931524217128754, -0.6357663869857788, -0.0837864950299263, 1.0814313888549805, -0.4652165472507477, -0.35000142455101013, -0.934695303440094, 0.2768386900424957, -0.6904381513595581, 0.07607810199260712, -0.5806077122688293, 1.0409084558486938, -0.8204215168952942, -0.48891669511795044, -0.8187669515609741, 0.824316680431366, 0.32989922165870667, 0.3758109211921692, 2.917072057723999, 0.3719472885131836, -0.2550843060016632, 1.3779767751693726, -0.6513530015945435, -0.011180337518453598, 0.02526097372174263, 0.8057059049606323, 0.34682267904281616, 0.3140783905982971, 0.37110182642936707, -1.3377195596694946, -0.11465364694595337, -1.405012607574463, -2.321916103363037, -1.9073225259780884, -2.5127577781677246, -1.3222254514694214, -0.7081748247146606, 0.5846440196037292, -0.04575558751821518, -0.5322269201278687, 0.12304875254631042, 1.9597668647766113, 1.4493200778961182, -2.5369136333465576, 0.4421691596508026, 0.7575538754463196, -0.6359634399414062, -0.764424204826355, -0.6236506700515747, 1.4517115354537964, -0.9151829481124878, 1.241899847984314, -1.1259701251983643, -0.6111545562744141, 2.13746976852417, -0.10962000489234924, -0.6699244379997253, -0.27835553884506226, -0.13443559408187866, -0.3534170091152191, 0.7267733216285706, 0.7063466906547546, 0.3623225688934326, 0.8894447684288025, 0.8839290142059326, -0.496263325214386, 1.62325119972229, -0.10862112045288086, 0.7455429434776306, 0.902586042881012, -2.7867395877838135, 0.15554894506931305, 0.16685284674167633, -1.3394962549209595, -0.6606829166412354, -0.40919485688209534, 1.926558017730713, 2.4976649284362793, -0.08133003860712051, 0.12402091920375824, 1.3557860851287842, -1.2430458068847656, -0.6146764159202576, 0.284400999546051, 0.17052413523197174, 0.11190451681613922, -0.10880986601114273, 1.1853588819503784, -1.0069993734359741, 1.55690336227417, -0.05899198353290558, -1.9477394819259644, -0.1904267519712448, -0.15324196219444275, 0.6236527562141418, -0.5291157960891724, -0.4029289484024048, 0.2545299530029297, -1.1757018566131592, 2.0514633655548096, -0.24263614416122437, -0.9531151652336121, -0.503254771232605, 1.2019948959350586, 0.5496848225593567, 0.08700632303953171, -1.084083914756775, 0.01799992099404335, -1.9507499933242798, 1.67237389087677, -1.2894630432128906, -0.36729276180267334, 0.3046599328517914, 1.8485769033432007, -1.6740028858184814, 0.20317204296588898, -0.3806867003440857, 0.7158147692680359, -0.6733423471450806, 0.05696019157767296, -0.3925838768482208, 0.7899630069732666, 1.177918553352356, -1.1034936904907227, 0.9277605414390564, -0.1569753736257553, 1.2080490589141846, 1.7385371923446655, -0.7456719279289246, -0.8699244260787964, 0.029663823544979095, 0.37243351340293884, 1.0831799507141113, 0.0735504999756813, 0.10954312235116959, -1.4894448518753052, 0.35411766171455383, 1.752727746963501, -0.10734734684228897, -1.2406842708587646, -0.2882797122001648, -0.013199305161833763, 0.24303005635738373, -1.305517315864563, -1.6828279495239258, 2.0492570400238037, -0.6411474943161011, -0.18085914850234985, -1.4988466501235962, -0.2727476954460144, -0.10079061985015869, -0.5592004060745239, -1.6317590475082397, 0.3474947214126587, 0.8181631565093994, -0.11757471412420273, 1.541205644607544, 0.09483733773231506, 0.5120677351951599, 0.8666847348213196, -0.8038086295127869, 0.1620197892189026, 1.3873525857925415, -1.053239107131958, -0.5781195759773254, -0.8668099641799927, 0.6426898837089539, 0.6507236361503601, 0.06823238730430603, 0.9701617956161499, 0.681355357170105, 1.147173285484314, 0.9742891788482666, 0.6470397114753723, -0.09526056051254272, -1.5956776142120361, -0.7863656878471375, 2.938845634460449, 0.3989027738571167, -0.6171290874481201, 0.08676758408546448, -0.15023161470890045, -0.42349478602409363, -0.5886402726173401, -0.43938565254211426, 0.2697690725326538, -1.6027019023895264, -1.2053253650665283, 0.5308001041412354, -0.7919812202453613, 1.0114537477493286, -0.28830936551094055, 0.37650108337402344, -0.35264506936073303, 0.7472343444824219, 0.8233978748321533, -1.0423663854599, 0.7916758060455322, -1.0170812606811523, 0.3195737302303314, -1.1867549419403076, 1.642199158668518, 0.049420248717069626, -0.13745218515396118, -1.375320315361023, 0.537452220916748, 0.8618809580802917, 0.48828378319740295, -1.5004607439041138, 1.4321660995483398, -1.009026050567627, 0.8485191464424133, 1.8723353147506714, -0.925910234451294, -1.7124474048614502, -0.09732652455568314, -0.2210940271615982, -1.9452637434005737, -0.5662177205085754, 0.9921172857284546, 0.6990959048271179, -1.7850784063339233, -0.7136884927749634, -0.3821714520454407, -0.10300976783037186, -1.4997236728668213, 0.039568789303302765, -0.8135359883308411, -1.0604867935180664, -0.29172173142433167, 0.42402639985084534, -0.25402936339378357, -0.23945322632789612, 1.4666281938552856, -2.130101442337036, -1.498203158378601, 0.4118172824382782, 0.6860083937644958, 1.697601318359375, -1.7220078706741333, -1.8508020639419556, -0.6136706471443176, 0.3487322926521301, 0.26875993609428406, 0.15702028572559357, 0.7791802287101746, 0.4839441478252411, 0.8055497407913208, 0.5229613780975342, -0.09006302803754807, 1.06990647315979, 0.11984093487262726, 1.9975178241729736, -0.08594969660043716, 0.06884591281414032, 0.244233638048172, -1.1579397916793823, -1.608738660812378, 2.115795135498047, -0.14837560057640076, -0.02842123433947563, -0.23774947226047516, 1.7981562614440918, -1.0469335317611694, -1.003075122833252, 0.03361193835735321, -0.689392626285553, -0.7618901133537292, 0.598850667476654, 1.387140154838562, -2.4220659732818604, -0.5157282948493958, 1.0246756076812744, 0.41198042035102844, 0.3813769519329071, -0.4950620234012604, -0.20210878551006317, 0.8162340521812439, -0.9579386115074158, -2.1513073444366455, -1.8503304719924927, 0.563853919506073, -0.9855864644050598, -1.5933562517166138, -1.731979489326477, 0.09657435864210129, 1.8336424827575684, -0.12103576958179474, 1.0639680624008179, 0.39229100942611694, -1.8405786752700806, -0.6714897155761719, 1.0983481407165527, 0.2628282308578491, -0.502311110496521, 0.4758684039115906, -1.270214319229126, -1.859052062034607, 0.6705991625785828, 2.48608136177063, 0.29321327805519104, 0.5894172191619873, -2.295165777206421, 1.5347756147384644, -0.20550289750099182, 0.5829036831855774, 1.2116142511367798, 0.11999329924583435, -0.8812522292137146, 1.6895544528961182, -0.7558090090751648, -0.2657216787338257, -0.5557785630226135, 0.1228046864271164, -0.2133404016494751, 1.331593632698059, -0.5346238017082214, 1.0061168670654297, -0.7380856275558472, -1.27731454372406, -1.5819995403289795, -0.4670175015926361, 1.5604842901229858, -1.958924412727356, 1.8546195030212402, -0.032802514731884, -0.36597609519958496, 1.190545916557312, 0.19208264350891113, -0.5126247406005859, -0.5479683876037598, 0.6236314177513123, 0.10095328092575073, -0.3911600112915039, 0.8488565683364868, 1.041217565536499, -0.7308899760246277, -0.1001472920179367, 0.4596650302410126, 0.6494235992431641, -1.061566710472107, -0.19598057866096497, -0.6141240000724792, 0.6733943223953247, -1.0891870260238647, -0.5181008577346802, 0.71955806016922, -1.9624032974243164, 1.8511704206466675, -1.390230655670166, -1.4128371477127075, 1.1364909410476685, -0.27326542139053345, -1.6612544059753418, 2.1686837673187256, -1.1048798561096191, -0.01658231019973755, 0.3455444276332855, -1.8892393112182617, -0.41155511140823364, 0.5436609983444214, 1.4889819622039795, 0.007176890969276428, 0.8019177317619324, 1.021033525466919, 0.6933429837226868, -0.28850409388542175, 0.42607352137565613, 1.603650689125061, 0.7017726302146912, 0.44762957096099854, -0.48322466015815735, 0.9248427152633667, 1.0880839824676514, -0.8631784915924072, -1.046518087387085, -0.7232410907745361, -0.027053102850914, 0.029740506783127785, -0.5770693421363831, 0.3042224049568176, -2.198815107345581, -0.2680307924747467, 0.5324507355690002, -2.8171982765197754, -0.9737361073493958, -0.28828200697898865, -0.2357376664876938, -0.2112365961074829, -0.6262953281402588, -1.2076923847198486, 1.9522743225097656, -0.9538019299507141, 1.0268727540969849, 0.5031518340110779, 0.36289623379707336, 0.34452304244041443, -0.6872808337211609, -1.447674036026001, 3.6865315437316895, -1.4279195070266724, 0.23356027901172638, 0.08401830494403839, -0.518683135509491, -1.1747299432754517, 1.6755621433258057, 1.1683603525161743, 0.5962164402008057, 1.0244810581207275, 0.017810683697462082, -0.3468065857887268, -0.09764047712087631, -0.2879815697669983, -0.9985748529434204, -0.5389600396156311, -0.16392040252685547, -1.0425466299057007, 0.6950169205665588, -0.38003456592559814, -0.5170297026634216, -1.2078070640563965, -0.7723953127861023, -0.1212463304400444, -0.5451490879058838, -0.5011244416236877, 0.5749286413192749, -1.1249585151672363, 1.37479567527771, -0.6999739408493042, 0.0861331969499588, -2.886324405670166, 1.6535354852676392, 1.7190555334091187, 0.3020961880683899, 0.23182371258735657, 0.4488999545574188, 0.5107740759849548, -0.7414835691452026, -0.6512377858161926, -0.2614063024520874, -0.8051538467407227, -0.5177227854728699, -0.46290862560272217, -1.3875812292099, 0.2384788542985916, -1.8613781929016113, -3.5076699256896973, 0.3441823422908783, 0.46112343668937683, 0.8105028867721558, 0.578461766242981, 0.5170620083808899, 0.8799706697463989, -1.292524814605713, -0.26107311248779297, -0.22905586659908295, -0.21933354437351227, -0.16666682064533234, 0.5715160369873047, 0.6666756272315979, -0.6051871180534363, -0.862385630607605, 0.43029239773750305, 1.0626016855239868, -1.1281626224517822, 2.107494354248047, -0.37283602356910706, -1.5015130043029785, 0.41614341735839844, -0.8533828854560852, 0.690575361251831, 0.21063221991062164, -0.7961894869804382, -2.181060314178467, -1.7073745727539062, 0.6210035681724548, 0.055118151009082794, -1.1299769878387451, -0.3133121728897095, 0.6615561246871948, 1.3278626203536987, 0.3939959406852722, -0.6609218716621399, 1.6418706178665161, -0.42234674096107483, 0.5172250270843506, 1.0873093605041504, 1.7590076923370361, 0.5808536410331726, 0.9374048709869385, -0.7962926626205444, -0.41437414288520813, 0.40597009658813477, 1.9041662216186523, -0.12554311752319336, -1.0121264457702637, 0.025195052847266197, 0.07434599846601486, -1.116302251815796, -0.8305575251579285, -0.0671192854642868, 1.0077391862869263, 1.1327093839645386, -0.809749186038971, -1.0098809003829956, 0.9230622053146362, -0.945986807346344, -1.2851512432098389, -1.7529200315475464, 0.8781949281692505, 1.5283730030059814, -0.05819902941584587, -1.6539534330368042, -0.5041276216506958, 0.1664162129163742, -0.6470085382461548, -0.8082243800163269, -0.6471567153930664, -0.7625563740730286, 1.5168781280517578, 1.1002553701400757, 1.089512586593628, -1.189597725868225, -0.9154521226882935, 0.7290955185890198, -1.9203578233718872, 1.5617631673812866, -0.06359576433897018, -0.14865368604660034, -3.130887985229492, 1.2622947692871094, -0.8979732990264893, 0.7712108492851257, -0.9475952386856079, 0.252377450466156, -0.5227583646774292, 0.48188266158103943, 1.7799689769744873, 0.1664181500673294, 0.14327476918697357, 0.5253404974937439, 2.7042179107666016, 0.31436532735824585, -1.1675182580947876, -1.9392046928405762, -1.3251264095306396, 0.638155996799469, 0.6725070476531982, 1.3192445039749146, 1.3966126441955566, 1.3728041648864746, -0.35984107851982117, 0.7162116169929504, 1.4140923023223877, 1.3257653713226318, -0.5726029276847839, -0.1660853624343872, 1.1239509582519531, 0.08238408714532852, -0.08367522060871124, 1.0603657960891724, 0.5748077630996704, 0.24321337044239044, 1.3470462560653687, -0.17145739495754242, -0.13629412651062012, -1.4164891242980957, 0.8067951798439026, -0.9757772088050842, -0.3953785300254822, 1.9870400428771973, -0.6457170248031616, 0.6262128949165344, -0.7384659051895142, 0.9058211445808411, 0.4723774492740631, -0.8431708812713623, 0.14942263066768646, -0.16879218816757202, -0.5289855003356934, -0.3755068778991699, -0.41312339901924133, -0.20321975648403168, -1.1558561325073242, 1.9218299388885498, 0.09700613468885422, 0.8037464618682861, -0.5999879240989685, 0.32885828614234924, -1.6581242084503174, -0.8390186429023743, 0.45211392641067505, 0.5166051983833313, -3.1458423137664795, 0.7852436304092407, -0.3787114918231964, -1.3872772455215454, -2.865715980529785, 0.5762993693351746, 0.013183084316551685, 0.9836961030960083, 0.15165570378303528, 0.6790720820426941, 1.413561224937439, -0.491038978099823, 0.5864144563674927, -0.1502392739057541, 0.34923335909843445, 0.3343496024608612, 1.1960867643356323, -0.7090110778808594, -0.19483135640621185, 0.6414300203323364, -0.9880110025405884, -0.8407948613166809, -0.5145735144615173, -0.428151398897171, 0.16597393155097961, -0.48156312108039856, 0.5547310709953308, -0.1202293187379837, -0.1418113112449646, -0.45586737990379333, 2.845804452896118, 2.284085750579834, 0.4116448163986206, -0.594367504119873, -2.2525269985198975, -0.9909735321998596, 0.6515266299247742, -0.6210430264472961, 1.2939382791519165, -1.4636868238449097, 0.260768324136734, -0.9297308921813965, 1.1080780029296875, -0.14775949716567993, 0.5743481516838074, 1.4275519847869873, -0.11801628768444061, -0.5593070983886719, -1.6401914358139038, 0.3300834000110626, 0.28476613759994507, -0.09446880966424942, 0.06845640391111374, -0.2773934304714203, -0.63385409116745, -0.021636366844177246, -0.682260274887085, 1.1520497798919678, 1.7011619806289673, -0.5335030555725098, 0.19106842577457428, 0.6668680310249329, 0.8384318351745605, -0.44581207633018494, 1.1828784942626953, 0.08936712890863419, 0.28888562321662903, 0.18154621124267578, -0.03827032446861267, -0.17198914289474487, -0.2318507879972458, -1.2794772386550903, 0.599184513092041, -0.8894529938697815, 0.5956588983535767, 0.6308321356773376, -0.2069048285484314, -1.9272950887680054, 0.8289852142333984, 1.3252196311950684, 0.28069007396698, -0.7396036982536316, 1.3354301452636719, 0.9746121764183044, 0.13595061004161835, 1.1698975563049316, -1.0158506631851196, 0.4900183379650116, 0.9389299750328064, -1.3905078172683716, 0.1671905219554901, 0.46288001537323, -0.8734930157661438, -0.6586945652961731, -1.339980125427246, 1.2338415384292603, -1.5748465061187744, 1.0503745079040527, -1.3392107486724854, -0.38867828249931335, 1.3231384754180908, 0.8559814691543579, -0.5453649163246155, 2.337104320526123, -0.07632588595151901, -1.147587776184082, 0.28021061420440674, -1.193603277206421, 1.1926900148391724, 0.16881127655506134, -0.8360546231269836, -0.699570894241333, -0.14721716940402985, -2.5058813095092773, 1.5875602960586548, -0.5387703776359558, 0.39909112453460693, 0.34018224477767944, -0.6036391258239746, -0.933292806148529, -1.5034345388412476, 0.2924146354198456, 1.10544753074646, 1.4668750762939453, 0.7717623114585876, 0.6132255792617798, 2.1469268798828125, 0.2449401468038559, 0.06494512408971786, 1.2468273639678955, -0.02378726378083229, -0.40337514877319336, 1.1762816905975342, 0.5275013446807861, -0.07915869355201721, 0.4974568784236908, 0.7500869631767273, 1.1722780466079712, -0.9126627445220947, 0.6507654190063477, -2.1757187843322754, -0.8036806583404541, 1.2970056533813477, 1.3597277402877808, -0.66376131772995, -0.010015570558607578, -0.1610529124736786, -0.9454764723777771, 0.042460016906261444, -0.4406662583351135, 1.1846215724945068, 0.28439652919769287, -1.9565213918685913, 0.05295669287443161, -0.9465641975402832, 1.4566458463668823, -2.090846300125122, 1.272948980331421, -0.09938155859708786, 0.5024403929710388, 0.3094116449356079, 0.5258036255836487, 1.6621986627578735, -2.274160861968994, 0.15278364717960358, -1.2331368923187256, -0.08833828568458557, -0.8685584664344788, -0.9860897660255432, 0.12728941440582275, -0.557458758354187, -1.680261492729187, -0.40133005380630493, 1.068084716796875, 0.11503308266401291, -1.7267847061157227, -0.07273365557193756, 0.12762518227100372, 1.6728657484054565, -0.2929489314556122, 0.4938000440597534, 0.05800625681877136, 0.5538638830184937, -0.7072408199310303, 1.5680609941482544, -0.3890039622783661, -0.6424838900566101, 0.3453206419944763, 0.5904644131660461, 3.3341970443725586, 1.5467357635498047, 0.864503800868988, -0.5352947115898132, -0.24033738672733307, -0.6392762660980225, 1.4517768621444702, 0.10296885669231415, -0.7195207476615906, -1.084033727645874, 2.215733528137207, 0.4432648718357086, 0.48365482687950134, -0.28582507371902466, -0.7923532128334045, 1.0821716785430908, 0.4074137806892395, 0.6560512185096741, 0.25471630692481995, -1.3011460304260254, 0.28832799196243286, 0.9126886129379272, -0.640819787979126, -0.853140652179718, 2.225762367248535, 0.5518441200256348, 0.24654395878314972, 1.5146783590316772, 0.3630954623222351, 0.01344047486782074, -0.5321886539459229, 0.5378980040550232, 1.4981471300125122, -0.43899890780448914, -0.04492887109518051, 1.648505687713623, 0.8806628584861755, -0.6513200998306274, -0.3355390727519989, 1.4563758373260498, 1.3413212299346924, -0.09197980910539627, -0.6882984638214111, -1.8135952949523926, -0.36844363808631897, 0.6053960919380188, 0.13770535588264465, -0.49556541442871094, 0.03296110779047012, -0.7302011251449585, 2.1812756061553955, -0.22639350593090057, 2.0442280769348145, 0.7633560299873352, -0.5701399445533752, -0.607629656791687, 0.5182074308395386, 1.1683125495910645, -0.10129905492067337, 0.4718805253505707, -0.23276275396347046, 0.018981903791427612, -0.9984362721443176, 0.6305881142616272, -0.714368999004364, -2.3909049034118652, -0.8524336218833923, -2.621741533279419, -0.878312349319458, 0.6738202571868896, 1.7145742177963257, -0.629161536693573, 0.24820061028003693, -0.21656300127506256, -0.7137151956558228, 2.827951192855835, 0.08650831133127213, -0.36522331833839417, 2.0178263187408447, -1.0650269985198975, -1.6273738145828247, -0.3321532607078552, 0.6750684380531311, 1.0703952312469482, 0.6922971606254578, 1.473009705543518, 0.07633468508720398, -0.9410504102706909, -0.8058258295059204, -0.24740852415561676, -0.41054999828338623, -2.169623374938965, -2.239619016647339, -0.5721018314361572, -1.2151422500610352, 2.1861517429351807, 0.0016953860176727176, 0.9468184113502502, -0.03090805560350418, -0.132024347782135, -0.2602662444114685, 0.7527007460594177, 1.3324615955352783, 0.5581853985786438, -0.35981956124305725, -0.18091726303100586, -0.982013463973999, -0.7552759647369385, 0.4901377856731415, -0.8626261949539185, 0.4981723725795746, 0.3423417806625366, 1.66147780418396, 0.19697678089141846, 0.9953653216362, 0.2647577226161957, 1.0489367246627808, -1.1652560234069824, 0.3547298014163971, 1.0750961303710938, 0.635389506816864, -2.0829925537109375, 2.4072234630584717, -0.6132247447967529, 0.5153628587722778, -1.1915782690048218, 0.37497013807296753, -0.8804157972335815, -2.5360896587371826, -1.6103030443191528, 0.26666948199272156, -0.3697859048843384, -1.675764799118042, -0.844612717628479, 0.2347647249698639, -0.8014359474182129, 0.2086694985628128, 0.3794742226600647, -0.0927983894944191, 0.2550137937068939, 0.001894308254122734, 1.0721811056137085, 0.45836037397384644, -0.6571946144104004, -2.605905294418335, 0.040803585201501846, 0.1474989354610443, -1.4600516557693481, -0.5345980525016785, -0.5754739046096802, -0.5342724919319153, -1.184522032737732, 0.13900555670261383, -0.40755611658096313, -1.5012537240982056, -0.03553004190325737, -0.11874935030937195, 1.4154082536697388, 0.8119937181472778, 1.1455589532852173, 0.4070907235145569, -0.3520292043685913, -0.14365404844284058, 2.3470406532287598, 0.8259756565093994, -0.6006326079368591, -1.220334768295288, -0.340717077255249, 1.496106505393982, -0.4904536008834839, 0.06799501925706863, -0.8738090991973877, -0.6903920769691467, 2.8338122367858887, 0.33772844076156616, 0.5948044657707214, 0.6224287152290344, -0.27948376536369324, 0.3135179281234741, -0.14315137267112732, -0.25352293252944946, -1.1928892135620117, 0.7836269736289978, 1.8880865573883057, 0.10902107506990433, 0.1461009532213211, -0.6754093766212463, -1.341681718826294, -0.5337284803390503, -0.5553493499755859, 0.5665505528450012, 0.5361208915710449, 1.8324964046478271, -1.0721797943115234, -0.8998529314994812, -1.322827696800232, 0.5325862765312195, 2.223445177078247, 1.1126011610031128, 2.2109878063201904, -0.44281503558158875, 0.12470436096191406, 0.36761733889579773, -2.038851022720337, 0.06385162472724915, 1.616956353187561, 0.7932447195053101, -0.483249306678772, 0.3873027563095093, 0.31161659955978394, -0.43751057982444763, 0.40065106749534607, -0.447473406791687, 1.0670995712280273, 0.6557616591453552, -0.9033223390579224, -0.45537641644477844, -0.19991160929203033, -1.8017175197601318, -0.29721906781196594, -2.198965549468994, 0.8064161539077759, 0.28060293197631836, -1.246985912322998, -0.004352331627160311, -0.9619377851486206, -0.059616461396217346, -0.20631979405879974, -0.32649102807044983, -0.18285208940505981, 0.20807291567325592, -0.15938521921634674, -0.4411519765853882, 0.9695756435394287, 0.20371407270431519, 1.656163215637207, 0.0859876349568367, -0.12301028519868851, -1.7959414720535278, 2.3159897327423096, -0.4802476167678833, -0.001450867741368711, -0.3482419550418854, 0.5979153513908386, -1.2113336324691772, 0.5244327783584595, 0.7724542021751404, 1.9742547273635864, 0.696372926235199, 0.19278691709041595, -0.6791558265686035, 0.5365119576454163, -1.0964646339416504, -1.04489004611969, -1.7199395895004272, -0.07400975376367569, -0.23863153159618378, -0.2591070234775543, -0.6577568054199219, -0.6530903577804565, 0.378767192363739, 0.18384301662445068, -0.030340557917952538, -0.7046939730644226, 1.48481285572052, -0.666832685470581, -0.6803189516067505, 1.7073938846588135, -0.39861616492271423, -0.22696615755558014, 1.3109906911849976, 0.24182048439979553, 0.2876988351345062, 0.32002243399620056, -0.631762683391571, -0.7400872111320496, -2.285881996154785, -0.8954518437385559, -0.47590500116348267, -0.03757299482822418, -0.650262176990509, -0.9790533185005188, 1.3606326580047607, -0.23472750186920166, 1.0865870714187622, 1.364494800567627, 0.3409527540206909, 2.055561065673828, -1.8488391637802124, -0.07274129986763, 0.7377097010612488, 0.6666973233222961, -1.8865878582000732, 1.8016380071640015, -0.32804980874061584, -1.0933164358139038, 0.4891504645347595, 0.33347567915916443, -0.011680332943797112, -0.6272891759872437, 1.154709815979004, -0.5025713443756104, -0.8026683926582336, 0.30671611428260803, 1.1086853742599487, 0.27581846714019775, 0.6441696286201477, 0.6880692839622498, 0.3512881398200989, -1.233250617980957, 0.135190948843956, -1.082815408706665, 0.7848356366157532, 0.34153619408607483, 0.5394949316978455, 0.739443838596344, -0.6080516576766968, -0.27653738856315613, -0.056126877665519714, -0.1932029128074646, -0.07919567823410034, 1.4880143404006958, -0.327997088432312, -1.575600266456604, 1.367906093597412, -1.8368622064590454, -0.24328266084194183, 0.4037112593650818, 1.1287237405776978, -0.22196845710277557, -0.6839355230331421, 0.3589200973510742, -1.6311089992523193, 1.3986842632293701, 0.5348720550537109, -1.4748362302780151, 2.029982089996338, -0.16027610003948212, 0.0731932669878006, -0.432597815990448, 1.7381305694580078, -0.45975080132484436, 0.9042611122131348, -0.9375408291816711, 0.6253005862236023, 1.4234882593154907, 0.44597211480140686, 1.976172685623169, -0.38229110836982727, 0.12949593365192413, 1.1567368507385254, -1.279199242591858, 0.8685367703437805, -0.8839782476425171, 0.7680873870849609, -0.261563241481781, 1.6420820951461792, 0.5205555558204651, 2.125600814819336, -0.6605948805809021, -0.10368437319993973, -0.28603899478912354, 0.6390534043312073, -1.5808509588241577, 1.5248655080795288, -1.9265758991241455, 0.5154655575752258, -0.3561539053916931, -0.464215487241745, -0.8742078542709351, 0.004728599451482296, -1.5140125751495361, 0.24883681535720825, 0.056636992841959, -0.13230662047863007, 0.15853658318519592, -0.2594320476055145, -0.9042009115219116, 0.5668847560882568, 1.614688754081726, 0.9818562865257263, -1.6397703886032104, -0.1074238121509552, 0.027873406186699867, -0.13712649047374725, -0.7289045453071594, 1.0249379873275757, 0.9742380380630493, -1.8136560916900635, -0.2661837637424469, -0.3099821209907532, 0.711089551448822, 1.6493464708328247, -0.08901065587997437, -0.32351940870285034, -0.3368076980113983, -1.4636462926864624, -0.8877317905426025, 0.5659229159355164, 1.828599214553833, 1.116804599761963, -1.4859681129455566, -1.936344027519226, -1.6100116968154907, 1.0919098854064941, -1.1485238075256348, -0.6083968877792358, 0.44850659370422363, -1.3703320026397705, 1.0068556070327759, -0.7493387460708618, -0.4129104018211365, -2.1134085655212402, -0.2250121831893921, -0.3672555685043335, -0.4523633122444153, 1.442661166191101, 0.8923242092132568, -0.9132274985313416, -0.07861568033695221, 0.5423933267593384, 0.7093415260314941, -1.0106388330459595, 1.0906468629837036, 0.25895556807518005, 0.270355224609375, 1.278041958808899, 1.5749839544296265, -0.7677162289619446, 0.3324439227581024, 1.2455284595489502, 1.3296722173690796, -0.33750686049461365, -0.22719751298427582, -0.25154754519462585, 0.7457111477851868, -1.0748003721237183, 0.532582700252533, -0.6879597306251526, 2.7122132778167725, 0.42003223299980164, -2.029608726501465, -1.307205080986023, 0.10524453967809677, 0.6148560643196106, 0.533160388469696, -0.5185514092445374, -0.7097548246383667, -0.9013041257858276, -0.7552818059921265, -1.302618384361267, -1.3052626848220825, -0.8625735640525818, -0.063022181391716, 0.11756492406129837, 0.6307797431945801, 0.7126668095588684, 0.05444743484258652, -0.92976975440979, 1.292286992073059, 0.1107625961303711, 0.3629249334335327, -0.585940957069397, 1.622383713722229, 3.008617877960205, 1.1691040992736816, 0.2136746346950531, 1.3497340679168701, 0.1123327687382698, -1.0913448333740234, -1.259246826171875, 0.8689740896224976, -1.1221435070037842, 0.4346228241920471, -0.37779614329338074, -0.05729183927178383, 0.8596257567405701, -1.6250685453414917, -2.479790449142456, -2.510375499725342, -0.22271433472633362, -0.3318392038345337, -0.17610830068588257, -0.4379465579986572, 1.7785210609436035, -0.4888158440589905, 0.2506773769855499, -0.5120648741722107, 0.3375592827796936, -0.5961090922355652, 0.8080031871795654, 0.10027564316987991, -0.6486932635307312, -0.7184761762619019, -0.871160089969635, -0.008688623085618019, 0.620027482509613, -0.9481027126312256, 2.577031373977661, 0.23065078258514404, -0.05802370235323906, 1.6625601053237915, 2.29480242729187, 0.21763776242733002, 0.12035021930932999, -1.5834126472473145, -0.19000184535980225, -0.3694506287574768, -0.07652352750301361, -1.438957929611206, 0.1369415819644928, 1.1341073513031006, 0.7369357943534851, -0.31833186745643616, -1.315412998199463, -0.6766411662101746, -0.7991650104522705, 0.8203791379928589, 1.5520594120025635, -0.25993531942367554, -1.072059154510498, -1.389946699142456, 0.8635819554328918, -0.2136354297399521, -1.0039242506027222, 0.61549973487854, -2.494785785675049, 0.7610997557640076, -1.1158920526504517, 1.573703408241272, 1.147983431816101, -0.9383805990219116, -0.2409193366765976, -0.18280918896198273, -0.8752179741859436, -0.7830648422241211, 0.4450157582759857, -0.2757117748260498, -0.9507712125778198, -0.29824355244636536, 0.8629390001296997, 1.929297685623169, -0.9935008883476257, 0.8698145151138306, 1.084249496459961, 0.029651610180735588, 0.2241239696741104, 0.46648088097572327, -1.1540554761886597, -0.9424186944961548, 0.7324690818786621, -0.6248751282691956, -0.4962659478187561, 0.8774887323379517, 1.738268256187439, -0.014621644280850887, -1.7250043153762817, 1.3466070890426636, 1.4791182279586792, 1.2373038530349731, -0.38962745666503906, 0.12073703110218048, 1.1241669654846191, 1.0082982778549194, -0.08728498220443726, 0.8177837133407593, 0.1490376889705658, -2.418363094329834, -1.479730248451233, -1.3017774820327759, 3.1587581634521484, -1.5348294973373413, 0.6922240257263184, 0.23228032886981964, 1.9440940618515015, 1.6302340030670166, 0.2930324375629425, -0.31692472100257874, -0.6052204370498657, -0.45622363686561584, -0.7618132829666138, -1.087440013885498, 1.8603276014328003, -0.9899507164955139, 0.5123045444488525, 0.4365595281124115, 0.6500529646873474, 1.05813467502594, 1.0621452331542969, 0.12918721139431, 0.16000831127166748, -1.6306250095367432, -1.2808259725570679, 0.5166833996772766, 1.2991491556167603, 0.3415330946445465, -0.3458418548107147, 0.4480631351470947, -0.24744243919849396, 0.8507736325263977, 0.821031391620636, -0.4585106074810028, 0.3538690209388733, 0.32295680046081543, -0.3291781544685364, -0.02243228442966938, -0.027782529592514038, -0.9869202375411987, 0.867434024810791, 0.8040466904640198, -1.0760631561279297, 0.6247777342796326, -1.675467848777771, -0.3151482939720154, 1.4041930437088013, 0.7328568696975708, 2.0578222274780273, -1.932423710823059, 1.133894920349121, -1.3193962574005127, -1.2760988473892212, -0.05462281033396721, 1.655869960784912, -0.6032260656356812, 0.3414618670940399, -0.18774567544460297, 0.10094911605119705, -0.7127882242202759, -0.1527617871761322, -2.275620937347412, 0.2978091239929199, -0.9368204474449158, -0.5939904451370239, -0.7720437049865723, 1.7146774530410767, 0.4154433310031891, -0.2276603877544403, -0.8346467614173889, 0.742424726486206, 0.1922302544116974, 0.6856054067611694, 1.3272545337677002, 1.400246262550354, -1.2929742336273193, 0.6525391340255737, -0.3501113951206207, 0.4762658476829529, -0.7338272929191589, -0.2347487360239029, 0.14174579083919525, -0.473209947347641, -1.251396656036377, 0.7712851762771606, -0.886928915977478, 0.2329045683145523, -0.45485955476760864, -1.3024934530258179, -1.5299668312072754, 1.1775147914886475, 0.384922057390213, 0.6583778858184814, -0.9284487962722778, -1.2284084558486938, 0.8153086304664612, -1.0106281042099, 0.5591596364974976, 1.3520464897155762, -0.28403472900390625, -0.5097128748893738, 0.5843046307563782, -0.5297946333885193, 0.774446427822113, -1.711092233657837, -0.609599769115448, -0.9177559018135071, 1.2590728998184204, -1.2449754476547241, -0.30752500891685486, -0.520439088344574, 0.6903651356697083, 0.48103177547454834, -0.32252100110054016, -1.334921956062317, 1.5119324922561646, 0.10975757986307144, 1.4603562355041504, 1.0377507209777832, 0.6533686518669128, -0.6417636871337891, 0.7870779633522034, 0.0900968536734581, -0.8694083094596863, -0.5308473110198975, 0.34966039657592773, -0.328437864780426, -0.4373607933521271, -1.7604761123657227, 1.3263094425201416, 0.027995845302939415, -0.6202494502067566, -0.3517687916755676, -0.7782341837882996, 0.46329769492149353, 0.3649727702140808, 0.5138291120529175, 1.02542245388031, -1.1516863107681274, -0.48681262135505676, 2.3417649269104004, 0.9253799319267273, -0.4888134002685547, -3.0588552951812744, -1.0636537075042725, -0.5555278658866882, -0.03549235686659813, 0.16040587425231934, -0.7313949465751648, 2.0992660522460938, 0.24394051730632782, 0.7262036204338074, 0.4149860441684723, -0.06984248757362366, -1.367777705192566, 1.0969384908676147, -0.16030921041965485, -0.461681991815567, -1.0171617269515991, 0.7394106388092041, 0.40105322003364563, -0.20199795067310333, -0.2310170829296112, 0.03450697660446167, -0.5184789299964905, 0.5848835110664368, 0.4309479892253876, 0.8497320413589478, -0.2506335377693176, 0.036015186458826065, 2.195568561553955, -2.4907262325286865, -0.4076005220413208, 1.0027167797088623, 1.3541247844696045, -1.9331923723220825, -0.27028366923332214, 0.20979897677898407, -0.038829561322927475, -0.17802700400352478, -0.2892792820930481, 0.43666568398475647, 0.9313461780548096, 0.9350510239601135, -1.170210599899292, -0.11488577723503113, 1.033971905708313, 0.31090855598449707, 1.3225007057189941, -0.4570511281490326, 0.4755980372428894, 0.293926477432251, 0.13341212272644043, -1.8644447326660156, -0.3590327799320221, 3.0930421352386475, 0.7168539762496948, -0.5917468070983887, 0.35032039880752563, 1.1978466510772705, 0.9935147762298584, 0.10995499789714813, 0.062319543212652206, 0.4814751148223877, 1.7282474040985107, 0.5877996683120728, 0.6845528483390808, 0.29247549176216125, -0.12237998843193054, -0.6858030557632446, -0.7759613394737244, 0.634800374507904, 0.7678470015525818, -1.1261014938354492, 1.2798901796340942, 0.39579981565475464, -0.1363653838634491, -1.7324554920196533, -0.6409804821014404, 0.3912859261035919, 0.08539370447397232, 0.9159817099571228, 0.5709931254386902, -0.10807238519191742, 1.2270634174346924, -2.100001811981201, 0.320624440908432, 0.5619447827339172, -0.5992296934127808, 0.0045732902362942696, 0.6589248180389404, 1.045942783355713, -0.05705069378018379, -1.3330011367797852, -0.2371797114610672, -1.5455806255340576, 1.6618585586547852, 0.7622827887535095, -0.38945895433425903, -0.28254643082618713, -0.6315612196922302, 0.7923595905303955, -0.3023279309272766, -0.8570579886436462, 0.16641353070735931, 0.2935689687728882, -0.1094575971364975, 2.0800771713256836, -0.11127433180809021, -1.020259141921997, 1.113357663154602, -1.1846729516983032, -0.9760693311691284, -0.5553072690963745, -0.4236855208873749, 1.5227957963943481, 1.5650540590286255, 0.6027005910873413, 0.3759860396385193, -1.3154594898223877, -1.6918128728866577, -1.4775872230529785, -1.9963717460632324, 0.9246145486831665, -0.9143122434616089, -1.0332465171813965, -2.1645803451538086, 0.23106080293655396, -0.15107889473438263, -0.2768694758415222, -0.42805013060569763, -0.9264600276947021, -1.4875653982162476, 1.3387881517410278, 0.37814292311668396, -1.0445470809936523, -0.9374422430992126, 0.25664377212524414, -0.77113938331604, -0.06051745265722275, -0.6771529316902161, 0.2753339409828186, -0.25979849696159363, -0.49520519375801086, 0.27137452363967896, -1.5471694469451904, -0.7626011967658997, -0.8230100870132446, 0.16818979382514954, 0.8869447112083435, 0.6522580981254578, -0.703840434551239, 0.49955499172210693, 0.16382569074630737, -0.01591959409415722, 0.5524055361747742, 0.1975642889738083, -0.530252993106842, -1.0524989366531372, -1.0248504877090454, -0.9809367060661316, -1.5355794429779053, -1.437964677810669, 0.20500291883945465, 1.1550301313400269, -0.6213876605033875, -0.323168009519577, 0.01016993261873722, 0.772007942199707, 0.2531127333641052, -1.7284468412399292, -0.8994318842887878, 0.4303038418292999, 1.1541813611984253, -0.20222602784633636, -0.4155537784099579, -1.5904991626739502, 0.5997889637947083, -0.11575600504875183, -0.4621294140815735, 1.6862740516662598, -0.40917089581489563, 0.6186898946762085, 1.881626009941101, -0.08536713570356369, -1.854085087776184, -1.1096280813217163, 0.9332578182220459, 0.9102218151092529, 2.060411214828491, -1.925167202949524, -0.2437754124403, -1.219274640083313, -2.2031993865966797, 0.8621693849563599, -0.594981849193573, -0.6269819140434265, -0.526746928691864, 0.5550432801246643, 0.10305467247962952, 0.3916931450366974, 0.7637318968772888, -0.5759942531585693, -0.14005856215953827, -0.9072303771972656, 2.5147366523742676, 0.07170596718788147, 0.48761796951293945, -1.8288012742996216, -0.5043879747390747, 0.505643904209137, -0.5174733996391296, -0.6100224852561951, -0.4742734134197235, 0.031065845862030983, 0.3280884027481079, -0.7634961009025574, -0.16182738542556763, -0.6803111433982849, 0.7920780777931213, -0.2426416128873825, -0.2959132194519043, 0.9399005174636841, -1.2653251886367798, -0.11845274269580841, 1.2296851873397827, 1.1370198726654053, -0.038925107568502426, 0.4080245792865753, -0.6336985230445862, 0.012022285722196102, 0.9599096179008484, -1.1883289813995361, 1.5317310094833374, 1.3851251602172852, -1.1350301504135132, -0.26211804151535034, -0.648471474647522, -0.6638643741607666, 0.4641684591770172, 1.1444740295410156, -0.6539828777313232, -0.5372544527053833, -0.09973682463169098, 0.30556923151016235, 0.8776546716690063, 1.4369500875473022, 0.8922469019889832, 0.3799472749233246, 1.836167335510254, 1.3786187171936035, -0.2716355323791504, 0.9479407072067261, 1.0374128818511963, -1.6118274927139282, 0.1858261525630951, -0.7146262526512146, 0.042942263185977936, -1.2460672855377197, -0.7318557500839233, -1.7844527959823608, 0.7784685492515564, 1.5398544073104858, -1.2340843677520752, 0.6308915615081787, -1.5354501008987427, 0.5854771137237549, 0.173879012465477, -0.7850233316421509, -0.9035729765892029, 0.044917549937963486, 0.6013592481613159, -0.3767455518245697, -0.29209524393081665, 0.38975951075553894, 0.3034132719039917, -1.9896190166473389, -1.0669853687286377, -0.9385326504707336, 2.2793774604797363, -2.096903085708618, -0.5531253218650818, -0.1653224378824234, 0.1585386097431183, -0.23409803211688995, -0.34875425696372986, 0.6997905969619751, -0.7297613024711609, -0.13479194045066833, -0.27494579553604126, -1.174556016921997, -0.7799862027168274, 0.5536746978759766, 0.34835633635520935, 1.596304178237915, 0.11480681598186493, -1.321709156036377, 0.8113465905189514, -0.6551440358161926, -0.15100400149822235, -1.023780345916748, 0.1751229465007782, 2.836155891418457, -1.1099897623062134, 0.8553276658058167, 0.5692185163497925, -1.7686970233917236, 0.09852098673582077, 0.9963368773460388, 0.5656797885894775, 0.6158421039581299, 0.2947794497013092, 0.37890225648880005, -1.1026231050491333, 1.0864462852478027, 0.05217691883444786, 1.466519832611084, -1.2563793659210205, -0.8843863606452942, -1.1420947313308716, 0.739109218120575, -1.6016697883605957, 0.48943087458610535, -0.8687398433685303, 0.531444787979126, 0.21333613991737366, 1.6943200826644897, -0.14187976717948914, 0.6724315881729126, -0.7428616285324097, -1.2298656702041626, -1.1995292901992798, -0.5331155061721802, -0.26870208978652954, -0.27016159892082214, 1.683300256729126, 0.3766474723815918, -0.5277913808822632, 0.43082237243652344, 0.7626121640205383, -0.4913504421710968, 0.5834485292434692, -2.2281181812286377, -1.0698455572128296, 0.0029718957375735044, -0.19520647823810577, -2.976484537124634, -0.026937153190374374, -0.00776510639116168, 1.613505482673645, -0.3148125410079956, -0.18956324458122253, 0.26885467767715454, -0.16967009007930756, -1.6675437688827515, 0.6372860670089722, 0.354072630405426, 0.44943615794181824, -0.5909140110015869, 0.6501075625419617, 0.49890121817588806, -0.037022415548563004, -0.0519585907459259, -1.3618264198303223, -1.0329838991165161, -2.385829210281372, -1.0406689643859863, -0.4210774898529053, -1.6507169008255005, -0.5475963354110718, -0.9297109842300415, -0.47120004892349243, 1.323730707168579, 0.1476120948791504, -0.7306769490242004, 0.1427493691444397, 0.6722355484962463, 1.3577170372009277, 1.4827702045440674, -0.23802989721298218, -1.062059760093689, -1.0099852085113525, -1.3363007307052612, -0.7572134733200073, -0.7692742943763733, -1.6072211265563965, 1.6697486639022827, 0.10784507542848587, -1.4422032833099365, -0.3365267515182495, 0.16349570453166962, 0.14193210005760193, -0.025907954201102257, -2.336726665496826, -0.7761089205741882, 0.36298850178718567, 0.17997083067893982, 0.5323466658592224, 0.3305204212665558, -1.5157525539398193, -0.20468834042549133, -1.1409534215927124, -0.638771116733551, 0.7354702949523926, 0.7793030738830566, -0.931485116481781, -0.08849307149648666, 1.5354044437408447, 1.3598741292953491, -0.5426109433174133, -0.4233439266681671, 0.0451621375977993, 0.4857334494590759, 1.3018202781677246, 1.9690357446670532, 0.30714234709739685, -1.0407201051712036, -1.1693495512008667, 1.2599648237228394, 0.9759014844894409, -0.1773059219121933, 1.3154031038284302, -0.44714444875717163, -1.158503532409668, 1.7455759048461914, -0.6338410973548889, 0.9605703949928284, -0.6491344571113586, -0.2692433297634125, 0.2624994218349457, -0.13584931194782257, 0.9923366904258728, 1.180177092552185, -1.9477887153625488, 0.33539867401123047, -0.2321261614561081, 1.2802850008010864, 0.3991447687149048, 1.9387668371200562, -1.5817630290985107, 0.2274613082408905, 0.7876457571983337, -0.880609393119812, 0.6254080533981323, -0.435965895652771, 1.4874924421310425, 0.8993337154388428, -0.5001946687698364, -0.3171001076698303, 0.7169138193130493, 1.0566480159759521, 0.7172646522521973, 0.6128543615341187, -2.276322603225708, 1.661546230316162, 0.6340672969818115, 0.17102059721946716, -0.35712575912475586, 1.2280555963516235, -0.72411048412323, 0.2576846182346344, 0.8954914808273315, 0.5296626091003418, 0.6791973114013672, -0.5367609262466431, 0.11925993114709854, -0.3031008839607239, 0.4407728314399719, -1.3905686140060425, 0.528781533241272, -0.17222601175308228, 1.4984934329986572, 1.270995855331421, 0.37493765354156494, -0.774036705493927, -0.09967845678329468, -0.8916111588478088, 0.2413434237241745, -0.8022172451019287, -0.3329319953918457, -0.4949900209903717, 1.1404914855957031, 0.5420559644699097, -2.2284274101257324, 0.6377602815628052, 2.157956123352051, 0.06855349242687225, -0.8077433705329895, 0.2516688406467438, -1.6786015033721924, -0.5691075325012207, 1.436715006828308, -1.5582207441329956, -0.0939013808965683, -1.4163672924041748, -1.2319916486740112, -1.0632284879684448, -1.3005942106246948, 0.3120323419570923, 0.7079296112060547, -0.08395738154649734, -2.8471150398254395, 0.4182143211364746, -0.8261104822158813, 1.8358389139175415, -2.1462438106536865, -1.3219133615493774, -1.3613080978393555, -0.9311452507972717, -0.9114643335342407, 0.45318228006362915, -0.0925385057926178, -0.9053825736045837, 0.6422215104103088, -2.1874427795410156, 0.08616994321346283, 1.2008700370788574, 0.30255600810050964, -0.19412019848823547, -0.920576274394989, -1.5668562650680542, 2.1222829818725586, -0.6782300472259521, -0.5546113848686218, -0.892753005027771, 0.550625741481781, -1.0788629055023193, -2.026953935623169, -0.9455394744873047, -0.17258523404598236, -0.04418550431728363, -1.2168328762054443, 2.053955554962158, 0.5567009449005127, 0.11658229678869247, 0.08716164529323578, -2.0062339305877686, -0.5264531970024109, 1.1346323490142822, 0.11076036840677261, 1.991403341293335, -1.5982519388198853, 1.068463921546936, -0.3149510622024536, 0.12153173983097076, 0.1940423995256424, -0.8088948726654053, -0.00505338329821825, -1.1187649965286255, 1.826317548751831, -0.16587671637535095, -0.7630221247673035, 2.3046059608459473, -0.9035597443580627, 1.8662902116775513, -0.1630139946937561, 0.08798536658287048, 0.6083871126174927, -0.14592915773391724, -0.8285170197486877, 0.1969647854566574, 0.11998501420021057, 0.4282379150390625, -0.5685103535652161, 0.6406188011169434, 1.1726763248443604, -0.4303589165210724, -0.7968334555625916, -0.5854701399803162, 0.1328599900007248, -0.8151394724845886, -1.2394739389419556, 0.7557257413864136, 1.517749309539795, 0.12475187331438065, 0.6715661287307739, -0.26614800095558167, -0.1795928180217743, 0.170768141746521, -0.3053199052810669, -0.10959305614233017, 0.9743998050689697, -0.35392677783966064, -0.9055720567703247, -0.91826331615448, 0.48930153250694275, -0.23952658474445343, -1.5302517414093018, -0.35534563660621643, -0.25437217950820923, 0.2513715326786041, 1.8424839973449707, 0.8180699348449707, 0.7124415040016174, -0.0069037675857543945, -0.45001980662345886, -0.3541118800640106, -1.077110767364502, -1.1307406425476074, -1.2326178550720215, -0.6435231566429138, -0.37462499737739563, -0.7936817407608032, 0.6168209314346313, -0.34473559260368347, -1.0029325485229492, 1.179559588432312, 0.6074508428573608, 0.17626577615737915, -0.04530813917517662, -1.1751512289047241, -1.4448468685150146, -1.9427106380462646, -2.041966199874878, -0.9425653219223022, -0.5197626948356628, -0.19871322810649872, 0.4667230248451233, 0.3378528952598572, -1.1550124883651733, 0.38214796781539917, -1.1456834077835083, 0.17138168215751648, -2.665285110473633, -0.11957819014787674, 0.20428666472434998, 0.3094279170036316, 0.8228138089179993, 1.8352380990982056, -1.042523741722107, -0.25625550746917725, 1.3774303197860718, 0.8580901026725769, -0.06107379123568535, 0.1442599892616272, -0.10744193196296692, 1.5065923929214478, -0.31148767471313477, -0.14912056922912598, -1.2307207584381104, -0.15652409195899963, -0.2979198694229126, 0.7221486568450928, 0.003484624670818448, 0.9916599988937378, 0.17079976201057434, -0.6587612628936768, -1.1923999786376953, -0.6327927112579346, -0.46895429491996765, -0.3094836473464966, 0.38971441984176636, -1.2914024591445923, 0.17275919020175934, -0.8469426035881042, -1.3768855333328247, -0.08265931159257889, 1.9056124687194824, -1.17438542842865, 1.1504381895065308, 0.5831080675125122, 0.051501911133527756, -0.818077802658081, -0.9695180654525757, -2.7622246742248535, 1.021770715713501, -1.3948577642440796, -0.5576181411743164, -0.47790849208831787, 0.7223490476608276, -0.3756149411201477, 0.7969803214073181, 1.0190826654434204, -0.09379322826862335, -0.44764086604118347, -1.2122383117675781, -0.9661146998405457, -0.3433602452278137, 0.7048308253288269, -0.3863767087459564, -0.04130448028445244, 1.2556232213974, -0.17812009155750275, 1.766497254371643, -0.38898956775665283, 0.7092459201812744, -0.003670153208076954, 1.0932490825653076, 1.615523338317871, -1.3016977310180664, -0.1669163554906845, -0.7002896070480347, -0.8203260898590088, 0.7960874438285828, -1.008120059967041, 0.07678545266389847, 0.08162057399749756, -1.1745600700378418, -0.603583812713623, 0.5984915494918823, 0.6194553375244141, -0.4359062612056732, -0.7366276979446411, 0.9526920318603516, -0.6961441040039062, 0.19733281433582306, 1.4024322032928467, 0.47956931591033936, 1.0118281841278076, 0.695607602596283, 0.4258067011833191, 2.187795877456665, -0.3463020324707031, 1.6718693971633911, 1.4776016473770142, 0.03370318561792374, 0.036397773772478104, -0.6260321736335754, 0.6019358038902283, -0.11641886830329895, -0.8119983673095703, 0.8941293954849243, -0.16427287459373474, -0.7069909572601318, -0.36270156502723694, 0.7779567241668701, 0.2705768942832947, 0.31288206577301025, -0.9644491076469421, -0.1691908836364746, 0.09120398014783859, 0.8019922971725464, 0.4666440188884735, 0.10796947032213211, -1.1050065755844116, -0.984963595867157, 0.5454317331314087, -0.9480834007263184, 0.8525081872940063, 0.5850980281829834, 0.41039204597473145, -1.1013760566711426, -0.8278017044067383, 1.6203798055648804, 0.1459413319826126, -0.11409855633974075, 1.1725151538848877, -1.599561095237732, -0.22877514362335205, -0.9722550511360168, 0.590802013874054, -0.462674617767334, -1.278138279914856, 1.2774553298950195, 0.3522516191005707, -1.5559731721878052, -0.8204270005226135, -0.3238607943058014, 1.9560459852218628, -0.41408678889274597, -0.5078029632568359, -0.09555897116661072, -0.5414717197418213, 0.9879216551780701, -1.9300895929336548, 0.10159221291542053, 0.9724889993667603, -0.5388787388801575, -1.2944386005401611, -0.5776256918907166, -0.5067673921585083, 0.7079285383224487, 1.3652966022491455, -0.3891354501247406, -0.8823814988136292, -0.09425968676805496, 0.7043753266334534, 1.0304272174835205, 0.0013188126031309366, -1.1340571641921997, -0.19358041882514954, -0.06871913373470306, -0.67137610912323, -0.2925373911857605, 0.6834495663642883, -0.1926531046628952, -0.713016152381897, 0.0128774493932724, -0.1574765294790268, -0.33987516164779663, -0.5828380584716797, 1.335338830947876, 1.2470927238464355, -0.048711977899074554, -0.5246909260749817, -0.08722450584173203, -0.5010907649993896, -0.9077363610267639, 0.7725462913513184, -0.24217502772808075, 0.2603665590286255, -2.3810572624206543, 0.159632608294487, -0.5672497749328613, 0.4323212504386902, 0.44019585847854614, -0.810302734375, 2.8526833057403564, 0.8783932328224182, 0.18426953256130219, -1.896767258644104, 0.9879146218299866, 0.19861960411071777, 1.0922843217849731, 0.1344127655029297, -1.791346549987793, -1.0940390825271606, 0.05447601526975632, 0.5426482558250427, -1.3621546030044556, -1.7347031831741333, -0.6456295847892761, 1.395871639251709, -0.31723156571388245, 0.7250651121139526, -0.38134273886680603, -0.15678022801876068, -2.101191997528076, -1.2243194580078125, -0.40812090039253235, -0.6640462875366211, 0.10249743610620499, 0.7745878100395203, 0.5708438754081726, -1.5075840950012207, -0.46681755781173706, -0.22027912735939026, -0.27332761883735657, 0.3839954137802124, 0.5469444990158081, 0.14060911536216736, 0.4821837842464447, 1.8702350854873657, -0.4739486277103424, 2.2436201572418213, -0.5348546504974365, -0.735867977142334, -0.6357553601264954, -1.5882556438446045, 1.053217887878418, -0.5057398080825806, -1.2245844602584839, 0.6954426169395447, 0.5074949860572815, -0.019453899934887886, -0.41156670451164246, -0.24330948293209076, 1.1125633716583252, -0.38154396414756775, 1.4180245399475098, 0.20396463572978973, -1.2498779296875, -0.13880454003810883, -0.434882789850235, 0.8939793705940247, -0.08637447655200958, 0.3725709915161133, -1.4930628538131714, 0.8886396884918213, -0.49195125699043274, -0.9196929335594177, 0.3802592158317566, 0.8804900646209717, 0.9104376435279846, -1.1957393884658813, 1.6570037603378296, 1.8787699937820435, 1.048490047454834, -2.294905424118042, 2.1389517784118652, 1.3647633790969849, -0.11711788922548294, -0.233490452170372, 0.3360314965248108, 0.6549826860427856, 0.4372551143169403, -0.24386000633239746, -1.593595266342163, 0.9416658282279968, -2.0047483444213867, 0.43043944239616394, 1.1123771667480469, 2.367225170135498, 0.397365927696228, 1.5543603897094727, -0.6285923719406128, -0.8232825994491577, 0.6210039854049683, -1.3181719779968262, 0.06756051629781723, 0.4382551312446594, -1.3269153833389282, 0.15748994052410126, 1.605699062347412, -0.7714826464653015, -0.8361475467681885, 0.633970320224762, -0.30523690581321716, -0.8725202083587646, -0.04153824970126152, -0.35585522651672363, -0.1925184428691864, -0.8665565848350525, -0.9011098742485046, -1.981131672859192, 1.362259864807129, 0.4606325626373291, 0.8392154574394226, -2.168146848678589, -0.35270002484321594, 2.632056951522827, -0.15011873841285706, -0.32921069860458374, -1.142056941986084, -0.11906495690345764, 0.8543348908424377, 1.1633553504943848, -0.5728827118873596, 0.9397478103637695, 0.8759083151817322, 1.227670669555664, 1.1488600969314575, -1.5255277156829834, -0.6344361305236816, -0.21978597342967987, 0.6113981008529663, 0.30227911472320557, 1.587082028388977, -0.8747997283935547, 0.45647379755973816, -1.5173159837722778, 0.048563238233327866, 1.7703322172164917, 0.32042616605758667, 1.518233060836792, -1.1247838735580444, 1.1581156253814697, -1.1752601861953735, -1.8877815008163452, 0.0029624910093843937, 0.0691014900803566, -0.1498185694217682, 0.299653023481369, -0.8355230689048767, 1.6707987785339355, 1.3782342672348022, -0.2499532848596573, 0.7264426350593567, -1.109056830406189, -0.3595631420612335, -0.04797575995326042, 0.007244414649903774, -1.7074023485183716, 0.3928765654563904, -0.30073997378349304, 0.4083828926086426, 0.775065541267395, -1.3305832147598267, -0.7715211510658264, 0.2997158467769623, -0.3586157262325287, -0.018039366230368614, -1.03572678565979, 0.8955907821655273, 0.7640419006347656, 1.0543384552001953, 0.7535622119903564, 1.8149094581604004, -0.06648554652929306, -0.21711666882038116, 1.0371589660644531, 0.5731872916221619, 0.20485511422157288, 0.035380154848098755, -1.240083932876587, 0.12844473123550415, -0.26742634177207947, -0.32762596011161804, 0.3553943634033203, -0.9589695930480957, -0.00346521963365376, -0.2858489453792572, -0.16771040856838226, 0.9472461342811584, -0.3078666031360626, 0.07076134532690048, -1.2586581707000732, -0.043697845190763474, 1.481429100036621, -0.10164307057857513, 0.9853697419166565, 0.9125719666481018, 2.244610071182251, 1.5649583339691162, 0.6710681915283203, -0.7131856083869934, -0.33987459540367126, -0.5278841257095337, 0.1533581018447876, -0.18973372876644135, -0.17301315069198608, -0.25639626383781433, -3.67250657081604, -0.22033904492855072, -0.5588268041610718, -0.2753492593765259, -0.36016348004341125, -0.2139422744512558, -2.9572205543518066, -1.53909432888031, -0.8759173154830933, 1.122619867324829, -0.7245186567306519, 0.25332263112068176, -0.27967387437820435, -0.113151915371418, 1.0692542791366577, -0.533655047416687, -0.5868144631385803, 0.32846054434776306, 0.7011094093322754, 1.7336409091949463, 1.3073310852050781, 0.306639164686203, -0.7510542273521423, -1.6494300365447998, 1.0486042499542236, 0.5178058743476868, -0.05439971014857292, -0.2455536127090454, -0.05941171199083328, 1.6077446937561035, -1.2309699058532715, 0.26400884985923767, -0.33256861567497253, 0.6148179769515991, -0.5648135542869568, 0.59734708070755, -0.05201350152492523, 1.4095953702926636, 1.5788764953613281, -0.6824925541877747, -0.43355318903923035, 0.8529454469680786, 2.0186197757720947, 0.02234559692442417, -0.7288607358932495, -0.8752377033233643, 0.26408350467681885, -1.206790566444397, -2.990856885910034, 0.8055418133735657, -0.0572303868830204, 0.7072910666465759, -0.004447111859917641, -0.13580051064491272, -0.9887473583221436, 0.09237950295209885, -0.23139385879039764, -0.36124420166015625, -1.1443288326263428, 0.42945098876953125, 1.2020459175109863, -0.3546464741230011, -2.1870412826538086, 1.2244555950164795, -0.2513261139392853, -1.7860835790634155, 0.2186143547296524, 0.8613893389701843, -0.24901553988456726, -0.07861791551113129, -0.1301928460597992, 0.6530150175094604, -0.018321730196475983, 1.5224716663360596, 0.8945043683052063, -0.7665376663208008, -0.9063097238540649, 0.20755364000797272, -0.8409823179244995, 0.43239155411720276, -1.6845242977142334, -1.2446446418762207, 1.4445538520812988, -0.18945063650608063, 0.21321094036102295, -0.6378380060195923, -1.49025559425354, -0.0762614905834198, 2.191486120223999, -1.520086646080017, -1.068873643875122, 0.693752110004425, -0.7210651636123657, -0.6610397100448608, -0.4109041690826416, 0.6021130681037903, 0.6993412971496582, 1.0134367942810059, 1.1491528749465942, 0.31284675002098083, -0.8792689442634583, 0.9765804409980774, 1.1756597757339478, 2.507781982421875, -0.4366410970687866, 2.235079765319824, 1.1235930919647217, 0.5051885843276978, 0.25964590907096863, 0.720423698425293, 0.6612776517868042, 2.4927244186401367, 0.3096332550048828, 0.2602565586566925, -0.14216206967830658, 0.02804507687687874, 1.4498428106307983, -1.414625883102417, 0.6112625002861023, 0.1402846723794937, 0.41213417053222656, 1.3648276329040527, -0.3708437979221344, 1.458450198173523, 0.9430597424507141, -0.24385283887386322, -0.3995514512062073, 1.4359359741210938, -0.42374128103256226, 0.47225865721702576, 0.03574015200138092, 1.5292781591415405, -0.11081737279891968, -0.563879132270813, -0.9943720102310181, -0.724271833896637, -0.3235090374946594, -0.5683141946792603, 0.8226650357246399, -0.4457279443740845, 0.11988849192857742, -0.5195375680923462, 0.9640503525733948, -1.595960259437561, 0.3476172685623169, -0.8549569249153137, 1.4182260036468506, 2.1357665061950684, 0.9606629014015198, 0.32544100284576416, -1.725447654724121, -1.8395041227340698, -1.2337863445281982, -0.40435609221458435, 0.1077699065208435, -0.31947529315948486, -1.0646047592163086, -0.42729315161705017, 1.4781944751739502, -0.977441132068634, 0.7446755170822144, 0.2712072730064392, 1.0959385633468628, -0.19624105095863342, -0.027249254286289215, 1.17837655544281, -0.14489664137363434, -0.37764084339141846, 0.15850897133350372, 0.5116009712219238, -1.3775849342346191, 1.4791786670684814, 1.1999479532241821, 0.7568690180778503, 0.5709541440010071, 0.0981607660651207, -0.61202073097229, 1.6667155027389526, -0.9975594282150269, -1.0605013370513916, -0.34518393874168396, -0.8599303960800171, -0.027262602001428604, 1.2665551900863647, 0.32938268780708313, -0.863486111164093, -0.6514979004859924, 2.3365535736083984, 1.0200059413909912, -0.5275698304176331, 1.5465854406356812, 0.5664387941360474, 2.3739266395568848, 0.5109099745750427, 0.09662161022424698, -0.9780731201171875, 0.5576881170272827, 0.8868231773376465, 0.14349642395973206, 1.1326820850372314, 1.35171377658844, 0.016897546127438545, 0.36509183049201965, -0.2795190215110779, 0.9384624361991882, 0.5287980437278748, -2.453688621520996, -0.10240813344717026, 0.8203919529914856, -1.0948082208633423, -1.828985333442688, 0.6346979737281799, -1.6832870244979858, -0.9828308820724487, -0.2756158709526062, 0.43720543384552, 0.792273759841919, -0.704444944858551, 0.3885559141635895, 0.8114213347434998, 0.3724817931652069, 1.5622583627700806, 0.6360266804695129, 0.07732760161161423, -1.550425410270691, -0.7469684481620789, 1.5415295362472534, -1.0090115070343018, 0.6270670890808105, -0.563478946685791, -0.4725185036659241, 0.07143577188253403, -0.14757274091243744, 0.3172599673271179, -0.3456558585166931, -0.4461439251899719, -1.0687555074691772, 0.46884211897850037, -1.5432907342910767, 1.2573823928833008, -0.2889834940433502, 1.7524980306625366, -0.2564036250114441, -0.8044641613960266, 0.5489760637283325, 0.05992582067847252, 0.2560068666934967, 0.6408201456069946, 1.6030149459838867, 0.40665262937545776, -0.7017143368721008, -0.8282278180122375, -1.7140371799468994, 0.7402671575546265, 0.6356555819511414, 1.3862360715866089, 0.078684002161026, 0.7649151086807251, -0.6497332453727722, 0.351309597492218, -0.7445922493934631, 1.6901293992996216, -1.2627421617507935, -0.4123401641845703, -0.3391832709312439, 0.43551012873649597, 0.5526120066642761, -0.20737139880657196, -0.4595472514629364, 0.28761276602745056, 1.1972029209136963, 0.9700392484664917, 2.0495686531066895, -2.227816104888916, -1.0531809329986572, -0.04651057347655296, 1.106454610824585, -0.668579638004303, -0.024532655254006386, -0.3088000416755676, 0.8234910368919373, -0.7971434593200684, -0.3027283251285553, -0.855635941028595, -0.5976558327674866, 0.3119680881500244, 2.0655629634857178, 0.7602488398551941, -0.4067875146865845, -1.3988714218139648, 0.42770010232925415, -0.009753595106303692, -0.8875934481620789, 0.8128857612609863, 1.1836727857589722, -0.5074002146720886, -1.741994857788086, 1.5190670490264893, 1.9187616109848022, 0.941623330116272, 0.5486029386520386, 0.6044500470161438, -1.5169565677642822, -0.33154910802841187, -0.9931930899620056, 1.5238795280456543, -1.0400067567825317, -1.3396505117416382, -0.12391646951436996, -0.6432925462722778, -1.1971547603607178, -0.08956822752952576, 0.9709565043449402, 0.8305425643920898, -0.5934468507766724, -0.7584119439125061, 0.23497174680233002, -0.9489633440971375, -0.44251763820648193, 0.6820704340934753, 0.4770042896270752, 1.0063871145248413, -0.41199472546577454, 1.5712988376617432, -0.29126226902008057, 1.918549656867981, -0.058972667902708054, -0.08786021918058395, 0.27886509895324707, -1.4214130640029907, -0.6394892334938049, 0.2512003779411316, 0.3684086799621582, -0.5834792256355286, 1.5196739435195923, -0.08106377720832825, 0.8357558846473694, 1.8847129344940186, -0.08954518288373947, -0.8868095278739929, 0.19879640638828278, -0.7831124663352966, -1.9353415966033936, -2.610811233520508, 0.4208534359931946, 0.1474677324295044, -0.7161421775817871, -0.22202251851558685, 0.32359424233436584, 0.9917635321617126, -0.36903414130210876, -0.3275059461593628, 0.46958351135253906, 0.0007838679011911154, -1.0314593315124512, 1.021993637084961, -2.1680893898010254, -0.11044489592313766, -0.11661844700574875, -1.2781881093978882, 0.20964482426643372, -1.2290178537368774, 0.0817747488617897, 0.8697235584259033, -0.3083747923374176, -0.9150058627128601, 0.03830140084028244, -1.0946582555770874, -0.10858321934938431, -0.709122896194458, -1.0359550714492798, 0.2144204080104828, 0.08551172912120819, 2.571364641189575, 1.9842660427093506, -1.44434654712677, 1.8710386753082275, 1.0023999214172363, 0.8202452063560486, -0.29812246561050415, 1.048105001449585, 0.5090648531913757, 0.3175787627696991, 1.0209085941314697, 1.2031551599502563, -1.1676768064498901, -0.24069085717201233, -0.17653615772724152, 0.10551191121339798, 1.3539128303527832, -0.8253751397132874, -0.215590700507164, -0.34443891048431396, -0.41715314984321594, -1.0515937805175781, -0.9491361379623413, 0.8391674160957336, 0.49808749556541443, 0.027424780651926994, -0.5587741136550903, -1.1557024717330933, 0.02137083001434803, 0.46664974093437195, -1.0018819570541382, -0.5638256072998047, 0.7275140881538391, -0.31907615065574646, -0.39684244990348816, -0.10945802181959152, 1.681594729423523, -0.2738499045372009, 0.46600964665412903, -1.567427635192871, -1.4915270805358887, 1.810334324836731, 0.28703510761260986, -0.2525333762168884, -2.9825704097747803, 0.07997538894414902, 0.04867815971374512, -0.21484313905239105, -0.20267252624034882, 1.9932929277420044, 0.018529321998357773, 0.4668803811073303, -1.2929013967514038, 0.37739819288253784, -0.5004240870475769, -0.6397111415863037, -0.36373311281204224, 0.6840829849243164, 0.33431872725486755, -1.2597054243087769, 0.11077644675970078, 0.8136891722679138, 1.2650614976882935, -0.29358020424842834, -0.6379281282424927, 1.0732388496398926, 1.6603909730911255, 0.3234965205192566, -0.8228617310523987, -0.8717918395996094, 1.1135417222976685, -0.43101248145103455, -0.40552690625190735, -0.5741708278656006, -0.08492347598075867, -0.23606844246387482, 0.9751302003860474, 2.3591043949127197, 3.036755084991455, -0.344245582818985, -1.0528929233551025, 0.2972259521484375, -0.05938878282904625, 0.05994785577058792, -1.2591570615768433, -0.8256004452705383, -1.592207670211792, -1.108899712562561, -0.2518543303012848, 0.8108488917350769, -0.8933414220809937, -0.6060351729393005, -0.20330651104450226, -0.22734174132347107, 0.5700231194496155, 0.28618496656417847, 0.2103160321712494, 0.9914898872375488, 0.6279207468032837, -0.2964398264884949, 1.6988170146942139, -0.8220037221908569, 0.5977570414543152, 0.027188098058104515, -1.3006811141967773, -0.3830687701702118, -1.6673930883407593, 0.15123622119426727, -1.037769079208374, 0.3776911199092865, -1.19986891746521, 0.0070976573042571545, -0.11433752626180649, -1.5131343603134155, -1.6106253862380981, 0.963813066482544, 1.5605337619781494, -0.551100492477417, -0.38014930486679077, 1.0620890855789185, 0.15681688487529755, -0.27696067094802856, -0.1878150850534439, -0.029370054602622986, 1.06419038772583, 0.5272030234336853, -0.7213869690895081, -0.7191617488861084, 0.412562757730484, -0.20427460968494415, 0.752848744392395, 0.3071966767311096, -0.1808694750070572, 0.3709481358528137, -0.6875409483909607, 0.3647040128707886, -0.6214687824249268, 1.1665703058242798, -1.3434715270996094, -0.47714391350746155, 1.8689773082733154, 2.0099058151245117, 0.6885480880737305, -0.6490544676780701, 1.0290955305099487, -0.165235698223114, -0.23708271980285645, -1.0159780979156494, -1.85332453250885, -1.5585814714431763, -0.9081173539161682, -0.6437493562698364, 0.6310738325119019, 1.5529941320419312, 0.5168304443359375, -0.15471695363521576, 0.7375167608261108, 0.4566791355609894, 0.41230282187461853, 1.7044360637664795, -0.6300679445266724, 0.7112630605697632, -1.0224941968917847, -0.7266305685043335, 0.35707566142082214, 0.0634293183684349, -1.5000675916671753, -0.07598880678415298, 0.1043955460190773, -0.1085679829120636, -1.684395670890808, 1.2693027257919312, 0.09406818449497223, 0.1512007713317871, 0.4621269702911377, -0.05383094772696495, 0.024658959358930588, 0.7014480233192444, -0.9483810067176819, 0.4007440507411957, -2.0404436588287354, 1.1622220277786255, -1.317493200302124, 1.0013046264648438, 0.7631338238716125, -0.20225101709365845, 0.4404723346233368, -1.2488503456115723, 0.36314859986305237, -0.39388537406921387, -1.7624002695083618, 0.0364670529961586, 0.17760588228702545, 1.2194947004318237, 1.065380573272705, 2.4469668865203857, -0.45982643961906433, 0.01717805303633213, 1.1258718967437744, 0.11300744116306305, -1.9318926334381104, 1.1989021301269531, -0.13128292560577393, -0.009799120016396046, -0.4623315930366516, -0.39714178442955017, -0.524064302444458, -0.9562866687774658, 1.116225242614746, 0.5118919610977173, -0.1490560621023178, -0.674014687538147, -0.4079437255859375, 1.111753225326538, -0.5852643847465515, -1.3223716020584106, 0.19748784601688385, 0.965238630771637, 1.6636085510253906, -0.25505006313323975, -0.5634949207305908, 0.41574233770370483, 1.1172462701797485, -0.34927451610565186, -0.6904754042625427, 1.2332295179367065, 0.3426336646080017, 2.911902904510498, 0.5259605646133423, -0.6281629204750061, 0.48849114775657654, 0.4157146215438843, -0.06989386677742004, 0.19500596821308136, -1.653016448020935, -0.0071472362615168095, 0.596380352973938], "dtype": "float32"}], "e": {"__nd__": [0.7399755716323853, 1.4617385864257812, -1.1779682636260986, -1.3169177770614624, -1.1275932788848877, 3.982314109802246, -0.5076040029525757, -1.1698849201202393, -0.7013120055198669, 0.8481448292732239, -1.2947341203689575, 0.004701137542724609, 1.101405382156372, 1.5183913707733154, -1.0863147974014282, 0.25430214405059814, 1.738523006439209, 1.0624363422393799, -0.39783796668052673, 0.9537118077278137, -1.1586036682128906, 2.472987413406372, 1.8409757614135742, -0.48872053623199463, 0.788689136505127, 1.5711901187896729, 1.2814658880233765, 2.003206253051758, -1.3202438354492188, -0.05398792028427124, -1.0412511825561523, -0.3361252546310425, 4.061818599700928, -0.7054793834686279, -0.1508185863494873, -0.09252861142158508, -2.7527217864990234, -1.4202711582183838, -1.097649097442627, -0.6538512706756592, 0.5110272169113159, 0.09761941432952881, -1.1739509105682373, -0.980712890625, -0.4806501865386963, 0.6190415024757385, -0.13662567734718323, -0.8870100975036621, -0.9673308730125427, 0.04456138610839844, -2.455507278442383, -0.5954665541648865, 0.5582674741744995, -1.9537633657455444, 0.396741658449173, 0.36348724365234375, 0.8653020858764648, 3.0308008193969727, 1.5715196132659912, 0.35689693689346313, -1.0681644678115845, -1.871837854385376, 1.0567725896835327, -0.48047423362731934, 0.958926796913147, 1.4460103511810303, -0.8710900545120239, 0.7262222170829773, 2.6488780975341797, 2.230228900909424, 0.9614371657371521, -0.5843747854232788, 1.013500690460205, -0.8688021302223206, 0.5731224417686462, -0.07950281351804733, -0.5408251285552979, -0.49245786666870117, 2.287288188934326, -0.25822389125823975, 3.013035774230957, 0.8398842215538025, -2.4023754596710205, -0.7458817958831787, -0.7782771587371826, -1.157774806022644, -1.588202714920044, -0.7495458722114563, -1.9164530038833618, -0.4688991904258728, 1.3053327798843384, 0.24926400184631348, 0.06781697273254395, 0.2445598542690277, 2.396261215209961, 1.5527158975601196, 0.3616109788417816, 2.0509884357452393, 0.20455703139305115, 1.2691045999526978, 1.2106568813323975, 0.5904914736747742, -2.232912540435791, -0.9137290716171265, 1.2453460693359375, 2.0022010803222656, 1.9306986331939697, 2.349940061569214, -0.19679051637649536, -0.0506671667098999, -0.22334450483322144, -0.5748165249824524, 1.8896602392196655, -1.7171626091003418, -1.8991150856018066, 1.9169243574142456, -1.011090874671936, -1.7594249248504639, 0.3794260323047638, -2.224128484725952, 0.21606683731079102, 1.1946370601654053, 0.27519938349723816, -0.36336421966552734, 0.8736539483070374, -2.311081886291504, -2.638181209564209, 2.8745334148406982, -1.0359700918197632, 0.3859502375125885, 0.915898323059082, -0.8373786807060242, 0.468417763710022, 1.4999881982803345, -1.6431384086608887, 0.1412913203239441, 1.3300683498382568, -0.39296892285346985, 0.8822157979011536, -0.7211388945579529, -0.902647852897644, -0.6953480243682861, 1.7780439853668213, -1.2800633907318115, 1.7151457071304321, -0.33257627487182617, 0.42806360125541687, -0.68494713306427, -2.01176381111145, 2.8176653385162354, 0.2733684778213501, 1.1160001754760742, -1.647679090499878, -0.943638265132904, -2.655820369720459, -0.2280442714691162, 2.5949649810791016, -1.0313191413879395, -0.33290648460388184, -0.7938488721847534, 1.227919101715088, -1.5936236381530762, 1.6337462663650513, 2.285827398300171, 2.1114752292633057, 1.9349404573440552, 0.13558527827262878, 1.9662461280822754, -0.5406941175460815, 3.66046404838562, -0.4999604821205139, 0.5552858114242554, 2.7391626834869385, 1.3059512376785278, 0.31165018677711487, 0.7583120465278625, -1.2253105640411377, -1.1458977460861206, 0.5999204516410828, -0.0420452281832695, -1.39479660987854, -1.4929827451705933, 1.172646403312683, 2.212782859802246, -0.991166353225708, 0.7374519109725952, 0.8935421705245972, -0.6234523057937622, 0.5789401531219482, 1.8772220611572266, -0.4377949833869934, 2.129446506500244, 1.3349144458770752, -1.2994093894958496, 1.2538036108016968, 0.5673325061798096, -0.41076308488845825, 1.7618844509124756, 1.041512370109558, -0.4725341200828552, 1.4793260097503662, 0.8163606524467468, -3.374659776687622, -0.3204869031906128, 1.932116985321045, -0.6469259262084961, -1.8286731243133545, 0.28625887632369995, 2.051130771636963, -3.70992112159729, 1.1556400060653687, -0.7325403094291687, 2.9132730960845947, 0.8962297439575195, 0.37701988220214844, -1.190950870513916, -1.5841209888458252, -1.4923611879348755, 0.979636013507843, -2.3530590534210205, 0.18081694841384888, 0.718168318271637, -1.870112657546997, -0.10876548290252686, 1.376391887664795, -1.2091516256332397, 1.9140517711639404, 0.5148581266403198, 0.7404202222824097, -0.6017153859138489, -2.315167188644409, -2.0750045776367188, -0.18150144815444946, 0.634853184223175, -0.8932572603225708, -0.6752445101737976, -1.8783320188522339, -0.6774831414222717, 1.5146414041519165, -0.9799162745475769, 0.7624930143356323, 0.3692874312400818, -0.981528103351593, -0.7059640884399414, -1.4935261011123657, -0.24080103635787964, -0.7995542287826538, 0.8374142646789551, -1.6467121839523315, -1.002577543258667, 0.1743578016757965, -0.10935962200164795, 2.141648769378662, -2.4038705825805664, 0.19664597511291504, -0.5882793068885803, 0.8686411380767822, 0.900239884853363, 0.03369736671447754, -0.7818495035171509, 0.026759326457977295, 1.421663522720337, -0.44855770468711853, 0.29273223876953125, 0.5650180578231812, -2.4873416423797607, -0.08319300413131714, -0.05658876895904541, 0.27953043580055237, 2.85196590423584, -1.4865447282791138, -2.4659762382507324, -0.22432494163513184, 0.5717242956161499, 1.7649602890014648, -2.228163480758667, -2.2543818950653076, 1.44277024269104, -1.2583194971084595, -1.0414749383926392, -2.073033571243286, 1.4814802408218384, -1.0238707065582275, -0.4919544458389282, -0.38570788502693176, -0.049879252910614014, -1.2966103553771973, 0.5826079249382019, 1.6678978204727173, -0.9760422706604004, 1.9287781715393066, -1.2429933547973633, 0.115273118019104, -0.7484340667724609, -2.5482821464538574, 0.40210771560668945, -0.9712203145027161, -1.073661208152771, -0.9856426119804382, -0.8312234282493591, -0.007427394390106201, -1.8754632472991943, 0.47767722606658936, -0.4828507900238037, 1.0895791053771973, 0.0921674370765686, -0.5388107299804688, 0.5686743259429932, -0.3604418635368347, 0.24779202044010162, -0.5300557613372803, 1.668426752090454, -2.346689462661743, 2.4374284744262695, -1.016770839691162, 2.7823646068573, -1.579026222229004, 0.9686497449874878, 1.8364551067352295, 1.8791556358337402, -1.216214656829834, -2.659231662750244, 0.9303673505783081, 0.8326143622398376, 0.4631080627441406, -1.658042311668396, -0.9044183492660522, 1.3217277526855469, 0.7409274578094482, 2.6057910919189453, 0.905215859413147, -0.5720250606536865, -0.5454237461090088, 0.06734561920166016, 0.3600466251373291, -2.0165231227874756, -0.9235476851463318, 0.15030743181705475, 0.7406965494155884, 0.6081678867340088, -0.23110169172286987, 1.7676935195922852, -1.3057622909545898, -0.9423034191131592, 2.2688329219818115, 0.6399871110916138, 0.3176378309726715, 2.942603349685669, 0.37211930751800537, 1.8060009479522705, 0.6884894371032715, -1.5400100946426392, 0.6760352253913879, -1.0539745092391968, -0.016994118690490723, -0.6156399250030518, 0.05178636312484741, 0.6362428069114685, 0.6376500725746155, -4.10205602645874, -0.4078425467014313, 0.6213972568511963, -2.0248546600341797, 0.4267471432685852, -0.039308369159698486, 2.2183632850646973, 0.23529335856437683, 0.19663846492767334, 0.10774350166320801, 0.5367617607116699, 0.61091148853302, 1.4660229682922363, -1.728697419166565, 2.0113515853881836, -0.08490496873855591, 0.5637633800506592, 1.4520323276519775, -1.8364566564559937, -0.46041643619537354, -1.2080574035644531, -0.8863772749900818, -0.15579703450202942, -1.0954633951187134, 2.173161506652832, 2.0014448165893555, 1.5243306159973145, -1.580415964126587, 0.7056916356086731, 0.3020053207874298, -0.12608861923217773, 0.3804807960987091, -0.8604227304458618, -1.3157743215560913, -0.871813952922821, 1.4331400394439697, 0.19547533988952637, 4.0745038986206055, -0.1318468451499939, 2.5687496662139893, -0.6401888132095337, 0.4573092460632324, -1.471318006515503, 0.5836514234542847, -1.7584892511367798, -1.7590954303741455, -0.3640017807483673, -0.8888548612594604, 0.4107900857925415, 0.8617950677871704, 0.22871196269989014, 2.8196511268615723, 0.36475950479507446, 0.7669974565505981, 2.0561134815216064, -0.15111976861953735, -2.034977912902832, -0.09075289964675903, 0.9044657945632935, -0.8147302269935608, 2.4269979000091553, -0.09222841262817383, 1.9895832538604736, 1.100162386894226, 0.4692712426185608, -1.0625888109207153, 0.3961169123649597, -0.621618926525116, -1.3336178064346313, -0.8423175811767578, 0.06339612603187561, -1.4910476207733154, 1.0407909154891968, 0.8199295401573181, 1.7874641418457031, 1.4493626356124878, 2.36818790435791, 0.2039351463317871, 1.085497498512268, -0.045891836285591125, 1.5168179273605347, -0.4113329350948334, -1.556725025177002, 0.080573171377182, -1.3147132396697998, 0.18727688491344452, 1.3043878078460693, 1.4322246313095093, -1.7757349014282227, -0.7801452875137329, -1.470733642578125, 0.7489258050918579, 1.1885669231414795, 0.7245991230010986, 2.0369913578033447, 1.1635254621505737, -0.42589396238327026, 1.2383573055267334, 0.9568290114402771, 0.42171749472618103, 0.3101869821548462, -1.7424463033676147, -0.6573807001113892, 1.3592174053192139, 1.3182274103164673, 0.35848599672317505, -2.1495275497436523, -0.5320835113525391, -0.4030288755893707, 1.360091209411621, -0.19989827275276184, 0.37797266244888306, -0.9151847958564758, -1.415515661239624, 2.534968614578247, -0.3766012191772461, -0.09820085763931274, 0.26586663722991943, -0.21616056561470032, -0.8368489742279053, 0.8240699172019958, -2.633941650390625, -1.83551824092865, -2.2471513748168945, 3.9197282791137695, 1.0671064853668213, -0.16682350635528564, 0.3528161346912384, 0.4984847903251648, -1.130007266998291, -1.5223628282546997, -0.9129136800765991, -0.18679380416870117, -1.9006080627441406, -1.0608704090118408, 2.355010986328125, 0.881318211555481, 0.9574143886566162, -0.43927428126335144, -0.5512362122535706, -1.047480583190918, -0.34425753355026245, -0.5742197036743164, -1.1077449321746826, 0.9949948191642761, 0.07624710351228714, 2.826888084411621, 0.8296636343002319, -1.5583497285842896, -0.11286506056785583, 0.012712538242340088, -1.2787911891937256, -1.2589476108551025, 1.2631891965866089, -0.36039578914642334, -0.9675267338752747, -0.8595556616783142, 1.2651618719100952, 0.7012901902198792, 1.205235242843628, -1.5954256057739258, 0.3971506953239441, 2.7523117065429688, 0.7942925691604614, -1.5377273559570312, -0.11076152324676514, -1.59657621383667, 1.7545192241668701, -2.169846296310425, -2.3475584983825684, -2.483860969543457, 0.44031354784965515, -0.047003597021102905, 1.4829702377319336, 1.0070518255233765, -0.008571483194828033, 0.4685018062591553, 1.411982536315918, -1.2132890224456787, -0.9698346853256226, 1.6249277591705322, -0.960260272026062, -0.2234984040260315, 0.1796533167362213, -1.133376121520996, 0.3247874677181244, 0.013608336448669434, -0.5228355526924133, 0.5366372466087341, -1.1314911842346191, -1.5164787769317627, -0.510688066482544, -1.3781005144119263, -1.5153894424438477, -2.670975685119629, -1.779270052909851, 1.3519266843795776, -1.253755807876587, -0.3617004156112671, -0.014343410730361938, 1.1246733665466309, -0.7578906416893005, -1.8155790567398071, -1.8722846508026123, 0.9040082097053528, 1.0732187032699585, 1.7058143615722656, 2.0743796825408936, 1.5476610660552979, -1.2138159275054932, 1.6253376007080078, 1.2978163957595825, 0.9009142518043518, -0.3666975200176239, -0.058478593826293945, 2.229649066925049, -0.22641894221305847, 2.172572612762451, 0.8472378849983215, -0.0654250830411911, -1.894468069076538, 0.1472836136817932, 0.20074182748794556, -0.2566673755645752, 0.8824134469032288, 0.2736315429210663, -0.34448379278182983, 0.48536351323127747, 0.1375533938407898, -0.462552011013031, 0.1327729970216751, -1.1437562704086304, -1.098240613937378, 3.4752492904663086, -0.9873676896095276, -0.23101377487182617, 0.3540002703666687, 0.7557852268218994, -2.2760283946990967, 0.28529414534568787, -0.5903631448745728, 0.07684299349784851, 0.09731478989124298, 0.9384596943855286, -1.5195319652557373, 0.4522780776023865, -0.6803284287452698, -1.9368449449539185, -1.494987964630127, -1.5111695528030396, -1.6599442958831787, 0.05780327320098877, -0.8238615989685059, 0.09363600611686707, -3.7224464416503906, -0.8450566530227661, -1.3608176708221436, 1.6278042793273926, 0.35482800006866455, -1.8533095121383667, -1.2854269742965698, -1.2458932399749756, 2.0301437377929688, -1.9069055318832397, -0.22427916526794434, 0.6532021164894104, -0.9575614929199219, -1.5051817893981934, -0.2926972210407257, 0.7912071347236633, 2.7224440574645996, -0.690261960029602, 1.683876633644104, 0.6456001400947571, 1.0761340856552124, 2.7037410736083984, 0.5672920942306519, -0.19944792985916138, -1.832993507385254, 0.0948333740234375, -0.7214983701705933, -1.645416259765625, -0.17793488502502441, 0.675260066986084, 3.1327691078186035, -1.2723909616470337, 1.2933433055877686, 2.1662139892578125, -0.35755544900894165, -2.1290433406829834, -0.9588495492935181, -1.7213081121444702, 0.44820863008499146, 0.6471115350723267, -0.18662616610527039, 0.38057202100753784, 0.4774996042251587, 1.1253020763397217, -0.578690767288208, -1.4248156547546387, 0.010219097137451172, 0.6154199838638306, 0.20130763947963715, -2.255821943283081, 3.01046085357666, 0.54620361328125, 0.3106805384159088, -1.2872352600097656, -2.0512311458587646, 1.1043609380722046, -0.26059162616729736, 0.305408239364624, -0.897232174873352, -3.1949212551116943, -1.6264240741729736, -0.314118891954422, -1.984411358833313, 0.7398488521575928, -0.7591688632965088, -2.9052937030792236, -0.07830348610877991, -0.9192416667938232, 0.13858208060264587, -0.1492065191268921, 2.585679054260254, -1.7358651161193848, -1.1453806161880493, -1.4677917957305908, -1.70221745967865, 0.8845893144607544, 0.1223597526550293, -0.09165138006210327, 1.5107673406600952, 3.4208321571350098, 1.10344398021698, 1.3102712631225586, -3.85787296295166, 0.25621700286865234, -1.8164732456207275, -0.9319262504577637, -2.6231908798217773, -1.1369503736495972, -2.8554720878601074, 0.5011093616485596, 0.7169017195701599, 1.866018533706665, -3.0119755268096924, 0.4243338108062744, -3.574232578277588, 1.8907520771026611, -1.4176558256149292, 0.23860979080200195, 0.13743096590042114, -0.5974803566932678, 0.7281131148338318, -0.4543238878250122, 2.0033583641052246, 1.1571153402328491, 0.3084290325641632, -1.6928074359893799, 2.000701904296875, -0.17982441186904907, -3.2298552989959717, -0.048488736152648926, 2.3406033515930176, -1.5923881530761719, 1.3568860292434692, -1.6416196823120117, 3.1244635581970215, 1.4163296222686768, -2.2041876316070557, 1.1873526573181152, 0.30632278323173523, -0.18216627836227417, 0.4948970377445221, 3.7925407886505127, 2.338555097579956, 0.1858522891998291, -1.0625364780426025, -0.21293044090270996, -1.7187596559524536, -2.5112459659576416, 0.9327709078788757, 1.8000059127807617, 2.793830394744873, -0.811874270439148, 1.1955177783966064, 1.743916392326355, 2.4307241439819336, -2.1054248809814453, -0.24041500687599182, -0.9753578901290894, 0.9387814998626709, 1.274086356163025, 0.9495958685874939, 1.0930172204971313, -0.455141544342041, 1.3003630638122559, -1.4879385232925415, 0.09068936109542847, 0.9144927263259888, -0.6757199764251709, 0.06684821844100952, -2.060539722442627, -0.018590867519378662, -2.2581605911254883, 0.5979730486869812, -1.273692011833191, 0.9115667343139648, 2.1969456672668457, -1.2150206565856934, 2.1004762649536133, 1.0775303840637207, -2.362215042114258, 0.5452684164047241, 2.8603873252868652, 0.4761720299720764, 2.839465618133545, -0.04866683483123779, 0.8596328496932983, 0.45028573274612427, 1.2301565408706665, 2.015329360961914, 0.7121198177337646, 0.010315090417861938, -1.920715093612671, 2.3094663619995117, 1.9737310409545898, 2.3479702472686768, 0.9316886067390442, 1.6429293155670166, -0.3318164348602295, -2.7494635581970215, -1.2882442474365234, 0.6582825183868408, -2.344849109649658, 1.3112471103668213, 0.6058955192565918, 0.8652565479278564, -0.8235254287719727, -0.2666500508785248, -0.16593071818351746, -0.9574840068817139, 0.7890262603759766, 0.7262812852859497, -0.3508584201335907, 1.0479745864868164, 2.2306363582611084, -2.6362338066101074, -0.6013202667236328, 0.2917451858520508, 0.28025567531585693, -3.7840189933776855, 0.8723443150520325, -0.408954918384552, 2.026883125305176, -0.815763533115387, 1.5373685359954834, -1.6474038362503052, -0.16367289423942566, 0.9236047267913818, 0.3574448525905609, 0.9674454927444458, 1.332859992980957, -2.5272676944732666, -1.0686086416244507, -0.449961394071579, 1.5015751123428345, -1.316153645515442, 0.8623173832893372, -0.19607332348823547, -0.2745509743690491, 0.2284289449453354, -0.008891642093658447, -0.2715071141719818, 1.52671217918396, -3.174497604370117, -0.39679408073425293, 1.80680251121521, 0.395438015460968, 0.8457145690917969, -0.11460015177726746, -1.598373293876648, 1.3449041843414307, 2.626368761062622, -0.17680257558822632, -1.5160876512527466, -0.8830675482749939, -1.4536609649658203, 1.5504209995269775, 0.39433497190475464, 2.3926730155944824, -0.5230551362037659, 3.2228305339813232, 2.12945556640625, 2.3053338527679443, 0.4069499969482422, 1.62831449508667, 1.341270923614502, -0.6878400444984436, -0.6167543530464172, -1.8087810277938843, -0.9854100942611694, -0.6526843309402466, 1.824030876159668, 0.1379224956035614, 2.515766143798828, 0.8921940326690674, 3.895909547805786, 0.660916805267334, 1.151174783706665, 1.9651341438293457, -3.7735579013824463, 2.3159871101379395, -0.42857789993286133, -0.07006607949733734, -2.0289134979248047, -0.28084999322891235, 1.1152769327163696, 1.602601170539856, 2.7812891006469727, -0.4631235599517822, -0.42760080099105835, -1.9928828477859497, 2.4669508934020996, -1.1503840684890747, -0.1259138584136963, -0.6866405606269836, -2.327411413192749, -1.1037259101867676, 0.9825990200042725, -2.7453925609588623, -0.504253625869751, 2.8794984817504883, -0.25892552733421326, -0.49438953399658203, 0.11887937784194946, 0.12446156144142151, -2.185720920562744, 0.5560641884803772, 0.4056554436683655, 1.921884536743164, 1.8452911376953125, 1.5098958015441895, 0.2464066445827484, 0.7993260025978088, -0.6841683387756348, -0.7780200839042664, -0.559889554977417, 1.8071231842041016, 0.47393617033958435, -0.7111297845840454, -1.1037150621414185, 1.2513490915298462, 0.26734673976898193, -0.35435596108436584, 2.557802677154541, 1.6698393821716309, 2.1253838539123535, -0.4341670274734497, -1.0007860660552979, -0.6459231376647949, 0.4990684390068054, -0.22227728366851807, -1.0751286745071411, -2.024555206298828, -1.68131422996521, 2.2295093536376953, 2.469454526901245, 0.573903501033783, -0.5675441026687622, 0.6152219772338867, -0.15321415662765503, 0.37415045499801636, 0.5116111040115356, 0.9390552043914795, 0.3169075548648834, -1.5421350002288818, -1.351475477218628, 0.04920439422130585, -0.6846921443939209, -1.4886282682418823, 0.04913461208343506, -1.3512378931045532, -0.7615726590156555, 2.427563190460205, -1.3703747987747192, 0.6120758056640625, -0.8809542059898376, 1.6421926021575928, 1.4438810348510742, 1.548776388168335, -0.9196249842643738, -1.9062644243240356, -1.2538495063781738, -1.8339117765426636, -2.842890739440918, -0.4292603135108948, -0.26905667781829834, -1.5495675802230835, 3.0145907402038574, 1.752872109413147, 2.4476559162139893, 2.363090991973877, 2.2889022827148438, -1.3341537714004517, -0.27419376373291016, -0.9490193128585815, -3.3738691806793213, -0.3219977021217346, 0.6977134943008423, -0.6846147775650024, 1.7996819019317627, 0.4356174170970917, 1.9470875263214111, 1.18460214138031, 0.5238993167877197, 0.15361011028289795, 0.7632461786270142, 1.3152881860733032, -0.11132413148880005, 1.589332938194275, -0.6881884932518005, -2.4254848957061768, 1.907217025756836, -1.1585166454315186, 0.19397194683551788, 0.5516866445541382, -1.5926438570022583, 2.1183769702911377, -0.9564440250396729, 1.5190167427062988, -0.30105966329574585, 1.1681709289550781, -0.4946916699409485, -3.000612258911133, -1.3638995885849, -2.4364912509918213, -2.5964925289154053, 1.879868984222412, -0.5675460696220398, 1.0039352178573608, -1.6409416198730469, 1.5597504377365112, 0.17620161175727844, -0.8959577679634094, -0.46427273750305176, -1.0494763851165771, 0.154738649725914, -0.4255092740058899, 2.2430455684661865, -1.126877784729004, -0.13248302042484283, 0.4897664189338684, -0.6328256726264954, -1.9002516269683838, 0.6003047227859497, -0.12402494251728058, -1.680288553237915, -0.87335604429245, 0.6116832494735718, -0.5158168077468872, -1.1263959407806396, 0.3945576548576355, 0.30572646856307983, -0.11096787452697754, 4.993802070617676, 0.024435997009277344, 0.7536988854408264, -0.815751314163208, -0.13706713914871216, 2.573276996612549, -1.4339288473129272, 1.216571569442749, -1.5338016748428345, -0.8395394086837769, -0.597862184047699, -2.448704957962036, 1.003998041152954, -0.1295751929283142, -1.903918981552124, 0.42616885900497437, 0.691901445388794, 0.7722547054290771, 1.024430274963379, 1.5355581045150757, 2.0059664249420166, 0.6516655087471008, -2.8442511558532715, 0.9761960506439209, -1.0722968578338623, -1.374197006225586, 1.2507226467132568, 0.07243897020816803, -0.4775470495223999, 1.8764235973358154, -0.2218303084373474, -0.22852885723114014, -0.17715972661972046, -0.1600779891014099, 0.40492236614227295, -2.322784662246704, -1.8680524826049805, 0.40354621410369873, 0.3380810022354126, -1.339135766029358, -0.02063053846359253, 0.09299445152282715, -1.6963192224502563, -3.4513540267944336, 1.5860109329223633, 3.3113529682159424, 1.8392778635025024, 0.6088066697120667, 1.592970371246338, 1.7531713247299194, 1.217839002609253, -1.3062644004821777, -3.251213312149048, 3.507610559463501, -0.9046173095703125, 0.8033285737037659, -2.393059253692627, -0.3363143801689148, -1.7924288511276245, 1.2851274013519287, -2.5704398155212402, -1.1722023487091064, 1.2321135997772217, -0.7169843316078186, 1.322597622871399, 0.6779472827911377, -0.1567937731742859, 0.33071643114089966, 1.8729381561279297, 1.2056419849395752, 0.10769355297088623, -0.9432247877120972, -3.4508066177368164, 0.1859651803970337, -3.9966928958892822, 1.559752106666565, 1.8361804485321045, 1.8469595909118652, -1.951085090637207, 0.0951581597328186, 0.5354097485542297, 0.8425602912902832, -1.9198778867721558, -0.025031447410583496, -2.5144972801208496, 1.1179790496826172, 2.9370009899139404, 0.5582543611526489, 0.36519163846969604, -3.1685914993286133, 0.11960691213607788, 0.34931039810180664, -1.8217055797576904, -0.9873086214065552, -1.162613034248352, 1.1540910005569458, -0.7761924266815186, -0.4180678129196167, 1.7307021617889404, 0.6385438442230225, -1.791553258895874, -1.3688169717788696, 1.5574960708618164, -2.1892056465148926, -1.292127013206482, 0.21074330806732178, -1.1660326719284058, -2.7260706424713135, -1.932258129119873, 0.19311904907226562, -1.3951424360275269, -0.9497807025909424, -1.347361445426941, 0.23350757360458374, -0.5656093955039978, -0.2124510407447815, -0.37807321548461914, 2.384249687194824, -1.4632351398468018, 0.45200568437576294, -0.11883699893951416, -1.1483585834503174, -3.834188461303711, 3.8656609058380127, -0.6808842420578003, -0.3026254177093506, 0.8998342752456665, -0.5229907035827637, -0.4980226457118988, -0.21823999285697937, -1.192151665687561, -1.11073899269104, 4.431644439697266, 3.9947543144226074, -1.2895538806915283, 1.5645804405212402, 0.5808090567588806, -2.666097402572632, 2.3315958976745605, 0.2737223505973816, -2.041400671005249, -0.8710192441940308, 0.4461430609226227, 0.44673240184783936, -0.2593204975128174, 3.7048850059509277, -0.28846603631973267, 1.4742028713226318, 1.2253029346466064, 0.5957598090171814, -0.8229002952575684, 1.3923827409744263, -1.7093582153320312, -2.2879528999328613, -1.1434999704360962, 1.253809928894043, 2.6877570152282715, 0.6671598553657532, 2.1860125064849854, 0.7696634531021118, 1.634243130683899, 3.334414005279541, 1.125327467918396, 1.4651057720184326, -0.7485864758491516, 1.3931856155395508, -1.2004427909851074, 2.018507480621338, 0.37954652309417725, -1.0951744318008423, 1.664987325668335, -0.2706950008869171, -0.005430012941360474, -2.7050771713256836, 0.8253920078277588, 0.06271505355834961, 1.3070498704910278, -0.4549717903137207, -0.6081432104110718, 0.44913458824157715, -0.7884204983711243, 2.1392734050750732, 2.2204790115356445, -2.598660707473755, 1.2038034200668335, 0.2465744912624359, -1.8643794059753418, -0.7092798948287964, -1.0444507598876953, -0.19389629364013672, 0.01158556342124939, -1.7273666858673096, -3.3583030700683594, 1.0045663118362427, -2.262619733810425, 0.3721715211868286, 2.6645760536193848, -0.7783968448638916, 0.12181341648101807, -0.625634491443634, -0.40789300203323364, -0.16331017017364502, -1.657687783241272, -0.8877652287483215, 2.5326154232025146, 0.07084214687347412, -0.1009170413017273, 1.5695421695709229, 3.155709743499756, 1.9513765573501587, -0.9769498109817505, -2.173490285873413, -1.6676123142242432, 1.4799413681030273, 1.3176690340042114, -0.7339703440666199, 1.5946787595748901, 0.5890138149261475, -0.43549585342407227, -0.5813016295433044, 1.5105818510055542, 1.3466942310333252, -0.9650967121124268, 1.6771917343139648, 0.6331461668014526, 1.6088670492172241, -1.3521689176559448, 0.09036052227020264, 1.17540442943573, 0.7005863785743713, -1.2238562107086182, -1.8141181468963623, -1.0926867723464966, 1.621062994003296, -0.07717105746269226, 1.1460798978805542, 0.5922227501869202, -0.4886953830718994, 0.01403123140335083, -1.020093560218811, 1.3181054592132568, -0.5654171705245972, -2.117762804031372, -0.9993613958358765, -0.40078413486480713, -0.31546080112457275, 0.6681643724441528, 0.11221899092197418, -1.0524976253509521, 1.3983646631240845, -2.054535150527954, 2.6937742233276367, 3.6895639896392822, -1.183469533920288, 0.5194551944732666, -1.184718370437622, -1.561079740524292, 0.5850086212158203, -0.007070779800415039, 1.689415693283081, 1.0501552820205688, 0.7026394605636597, 1.891387939453125, 0.4542325735092163, 3.6085052490234375, -1.1674959659576416, -2.0235824584960938, -1.8662149906158447, 1.124825358390808, -1.1163573265075684, -0.448400616645813, 0.1425805389881134, -0.6135202646255493, -0.7053031921386719, -0.3803431987762451, -1.450294017791748, 1.314107894897461, -0.9380529522895813, -0.3588651418685913, 0.8868381977081299, 1.309603214263916, 0.5699379444122314, 2.377288818359375, 0.5282873511314392, -0.983090341091156, 2.022352933883667, -1.1194286346435547, -1.5301806926727295, -0.8738569021224976, -0.6512078046798706, -1.4034712314605713, -3.155086040496826, -1.1334285736083984, -0.32829856872558594, 1.5122787952423096, -0.7287642359733582, -2.1273159980773926, -2.0602803230285645, 0.04121194779872894, 1.409397840499878, 1.0997015237808228, 1.0678879022598267, -0.672389030456543, -0.916004478931427, 0.5096933841705322, -1.261106014251709, 0.762455403804779, -1.139478325843811, -3.617675304412842, 1.6377294063568115, 3.759103298187256, -0.6066786050796509, -0.7346678376197815, 2.895169496536255, -0.2358841896057129, 1.6989836692810059, -0.5784040093421936, -0.7878531217575073, -0.23192989826202393, -1.1161267757415771, -1.2151437997817993, 0.4054945409297943, -1.4102026224136353, 1.7934943437576294, 0.8577473759651184, -0.25844746828079224, 0.5316070914268494, -1.7455031871795654, -2.4236013889312744, 0.559548556804657, -0.19731837511062622, 0.5937173962593079, -0.8657516837120056, -0.9728948473930359, -1.3352634906768799, 2.4382834434509277, -3.801583766937256, 0.8926780223846436, 2.827448606491089, -0.16341513395309448, -0.11722308397293091, 1.1952831745147705, 0.029511094093322754, 0.6684603095054626, -0.559834361076355, 2.264134645462036, -2.140542507171631, 0.9025029540061951, 0.9684492349624634, -0.05706524848937988, -2.95094633102417, -0.5499030947685242, 0.013478636741638184, -0.5456060171127319, 2.489009380340576, -1.9775190353393555, -0.2751097083091736, 1.477724552154541, -1.2839630842208862, -1.0616300106048584, 1.1426873207092285, -0.2862757444381714, -1.1385340690612793, 1.733605146408081, 1.3087522983551025, 0.24256911873817444, 1.4098058938980103, 3.3674042224884033, -0.2932185232639313, -1.867546558380127, 0.4495563209056854, 1.701774001121521, 1.6776583194732666, -0.39212045073509216, 0.4054948091506958, 1.467067837715149, 0.1741228848695755, -1.2027462720870972, 0.9661074876785278, 0.030709385871887207, 2.0209813117980957, -1.298530101776123, -1.0444362163543701, -0.6033651828765869, 0.6866856813430786, 1.1837170124053955, -2.4047787189483643, -1.0094738006591797, -0.71540367603302, -0.9730591773986816, -2.2200729846954346, 1.0895055532455444, 0.3177785277366638, -1.8378241062164307, -1.321014642715454, -0.7526965141296387, -1.920912265777588, 1.1850734949111938, 0.3367213010787964, -0.9974522590637207, 0.30672845244407654, -3.3214218616485596, -0.453620046377182, -1.6541283130645752, -0.47392624616622925, -0.16184982657432556, -1.3277544975280762, -1.1918736696243286, -0.16612547636032104, -1.2147014141082764, -0.5776388645172119, 2.3803036212921143, 1.871058464050293, 0.5870909690856934, 0.38671550154685974, 0.005672097206115723, 1.9226855039596558, -3.9712107181549072, -0.11929953098297119, 0.00010703876614570618, 0.5351982116699219, 1.0141953229904175, 2.3142478466033936, -0.12914642691612244, 0.1151282787322998, -1.5842669010162354, -0.8861219882965088, 0.02593335509300232, 2.2039873600006104, 0.7993658781051636, 0.09761053323745728, -0.028669506311416626, 0.6396142840385437, 0.15008413791656494, 0.1790299415588379, 1.5643706321716309, -2.632206439971924, -2.8326399326324463, 2.3655717372894287, 1.134835958480835, -1.8443557024002075, -1.1240859031677246, 1.2436450719833374, 1.2085816860198975, 1.1591415405273438, -1.1496814489364624, 2.2150816917419434, 0.8346760272979736, 0.14042136073112488, -2.0181822776794434, -1.3088206052780151, -0.6707091331481934, 0.7066738605499268, 1.0573426485061646, 1.1577292680740356, 2.363473653793335, -1.0696399211883545, 1.3652182817459106, 1.5537891387939453, -1.3895028829574585, 0.7783547639846802, 2.952247142791748, -1.2713452577590942, 0.9779069423675537, 2.2951040267944336, -1.0446078777313232, 1.3476686477661133, -0.14005815982818604, -0.006366372108459473, 0.5930471420288086, 0.7996730208396912, -2.532438039779663, -2.322551727294922, -0.07920670509338379, -0.4715050458908081, -1.800156831741333, -2.256077527999878, -0.22404825687408447, -0.5643081665039062, 2.5630927085876465, 0.5491377115249634, 1.0296664237976074, -0.921684980392456, -0.6683397889137268, -0.5713510513305664, -0.3762383460998535, 3.308379888534546, 1.4090769290924072, -0.8066450953483582, -2.6354987621307373, 0.13309890031814575, 1.9257932901382446, -0.49522656202316284, 0.45885157585144043, 0.8916425108909607, 0.3880167007446289, -1.5478771924972534, -2.4574148654937744, -0.7718725204467773, -2.1810460090637207, -0.010704100131988525, -2.1014719009399414, -1.075420618057251, -3.5622763633728027, 0.30597859621047974, -0.6323824524879456, -0.9480146765708923, 0.49715614318847656, -0.7222902774810791, -0.49462735652923584, 0.6665375232696533, 2.0462443828582764, -1.6832419633865356, -0.7514744997024536, -0.0730607658624649, -0.2929636240005493, -0.473879337310791, -0.781035304069519, -0.3560687303543091, 3.1223902702331543, 0.5947341918945312, -1.0634660720825195, 0.41320642828941345, 0.4448680877685547, -0.48092710971832275, 0.20538154244422913, 2.903057336807251, 2.22161865234375, -1.905034065246582, -0.45347854495048523, 1.6001020669937134, 2.9569921493530273, -0.45865824818611145, -1.8386783599853516, -2.1156816482543945, 0.597217857837677, -1.412589192390442, -1.0681648254394531, 1.0298733711242676, -0.8464757800102234, 2.1384057998657227, -0.9805528521537781, -2.598687171936035, 0.5198599696159363, -0.6774750351905823, -0.14401423931121826, -1.850511908531189, -0.5718129873275757, 1.857509732246399, 3.4459590911865234, 1.0314277410507202, -0.8219732642173767, 0.7025624513626099, 0.7509007453918457, 2.793973922729492, -2.6064293384552, 2.1456892490386963, 0.046332187950611115, 1.1394245624542236, 1.3021767139434814, -0.868171215057373, 2.33673095703125, 1.2279787063598633, -0.5814871788024902, 2.19014048576355, -2.591980457305908, -2.049921989440918, 2.0110599994659424, -0.6110708713531494, 2.205496311187744, -0.009080380201339722, 2.6390180587768555, 0.48589134216308594, 0.4805198311805725, -0.13468199968338013, -1.63475501537323, 0.7256014943122864, -0.7746744155883789, -1.5816352367401123, -1.186213731765747, 1.6760048866271973, 0.4432491660118103, 1.6177341938018799, -1.073155164718628, -0.06622779369354248, -1.5681220293045044, -1.4139810800552368, 1.9504579305648804, -0.30723416805267334, -1.1305935382843018, 0.11581987142562866, 1.6876294612884521, 1.0792735815048218, 1.4291901588439941, -1.2520596981048584, 0.3771398067474365, 1.9718680381774902, -1.5670703649520874, 0.11706016957759857, 0.9952641725540161, -1.5023612976074219, -2.48895525932312, -0.6453303098678589, 1.9743587970733643, 0.5465500354766846, 1.1838524341583252, 1.0769500732421875, 1.0860490798950195, -0.27797186374664307, -2.0406653881073, 0.29488444328308105, 1.1570863723754883, 0.9434854984283447, -2.168696880340576, 0.9253112077713013, -0.622191846370697, 0.8398429155349731, 0.00720050185918808, -0.09074920415878296, 0.4401180148124695, 0.7370674014091492, 1.6987102031707764, 0.38422131538391113, -0.542848527431488, -0.7836179733276367, -2.0822033882141113, -0.8412166833877563, -0.9972763061523438, 1.361593246459961, 0.18423783779144287, 0.4877416491508484, 0.2767959535121918, 0.09100306034088135, -0.5792982578277588, -0.2915630340576172, -2.0477566719055176, 0.6910805702209473, 1.9508240222930908, -1.3992085456848145, 1.6490472555160522, -0.28625333309173584, 0.5925465822219849, -0.8492448329925537, -0.39257025718688965, -3.353721857070923, 0.17011092603206635, -0.40895941853523254, -1.545933485031128, -2.0446460247039795, -0.19142265617847443, -1.2920631170272827, 0.5266159176826477, -0.4666772484779358, 1.8988893032073975, -2.615096092224121, -0.36141979694366455, -0.91496342420578, 1.9348278045654297, 1.005347490310669, 1.6934850215911865, -0.3357395827770233, -1.048813819885254, -0.32398998737335205, 2.1437792778015137, 1.2881076335906982, -1.2108712196350098, -0.1351931393146515, 0.0316922664642334, -0.8282433748245239, -0.9983386993408203, 0.6546328663825989, -1.9407106637954712, -0.5030179023742676, 0.0598885715007782, -1.0675160884857178, 1.0056564807891846, 1.3169913291931152, -0.8546319007873535, 2.210026979446411, 0.8982771039009094, -0.7238717079162598, -0.4773731529712677, -0.2871738076210022, 2.2294042110443115, 1.3709827661514282, -1.4776610136032104, -1.203127384185791, 1.174970030784607, 0.45359551906585693, -0.9292751550674438, 0.47292405366897583, 0.7324283123016357, 1.1311619281768799, -0.38728874921798706, -0.6445119380950928, 1.806809663772583, 0.2610362768173218, -0.00019158143550157547, -0.5354193449020386, 0.7952576875686646, 0.20826473832130432, 2.152897596359253, -1.3504736423492432, 1.1274442672729492, 1.797257661819458, 0.11103641986846924, 1.3825604915618896, 1.2529940605163574, -2.488537073135376, 2.1163203716278076, -0.3491356372833252, -0.818557620048523, 1.5426745414733887, 3.1393847465515137, 1.7423055171966553, -1.678112506866455, -1.6828858852386475, -2.7786359786987305, 0.9002261161804199, -1.0905654430389404, 0.5971421599388123, -0.6961843967437744, -0.29056045413017273, 0.9525874853134155, -2.1972365379333496, 0.23455098271369934, 1.6175124645233154, 0.28890201449394226, 1.3359333276748657, -0.61371248960495, -1.9897799491882324, 0.16415417194366455, -0.14465007185935974, 0.9014308452606201, 0.5272581577301025, 1.6475019454956055, 1.422178864479065, 1.3482792377471924, -0.9281581044197083, 1.280397653579712, -0.06416118144989014, 2.1952497959136963, 1.0712106227874756, -0.42827093601226807, 2.082533836364746, 0.7055006623268127, 0.836379885673523, 1.263243317604065, -0.9258682727813721, -0.5004546046257019, -0.32329314947128296, 0.29258978366851807, -0.0728057324886322, 1.6236811876296997, 0.41930848360061646, 3.0392088890075684, 1.9342032670974731, 0.2955970764160156, -0.23922139406204224, 1.912635326385498, -1.1042430400848389, 1.347876787185669, -0.9980316758155823, 0.3423154354095459, -0.5953449606895447, 2.5970659255981445, 2.0251710414886475, -0.028362303972244263, 1.0681257247924805, 0.3934769034385681, 0.4317401647567749, -1.0091277360916138, 2.7703211307525635, -0.5891147255897522, -0.5052871704101562, 1.9604038000106812, 0.007525680586695671, 3.4016828536987305, 2.209653377532959, -1.642500638961792, 3.5392508506774902, 0.17110252380371094, 0.5163173675537109, 0.011195898056030273, -2.456674337387085, 0.002065718173980713, 1.3423141241073608, -2.3508894443511963, -0.1219341903924942, 2.0981125831604004, -0.2861555218696594, 0.7178613543510437, -0.14203904569149017, 1.7686103582382202, 1.121761679649353, -0.6736319065093994, 0.9563888907432556, -1.0588935613632202, -0.08016642928123474, -1.1711640357971191, 0.09339560568332672, -0.9483256340026855, -0.6313769817352295, -0.6832587122917175, 0.9842920899391174, -0.3945327401161194, 0.14748269319534302, 2.5369396209716797, -0.052205875515937805, -0.5742894411087036, -1.1722465753555298, -0.6200646758079529, 2.69732928276062, 2.1674671173095703, 0.693714439868927, 2.631730794906616, -2.3231263160705566, 1.5815351009368896, -0.6352375149726868, 3.0828351974487305, -1.5293327569961548, -0.7572416663169861, -0.05905890464782715, 0.19550061225891113, -1.186838150024414, -2.3047008514404297, 0.16401046514511108, 0.9870843291282654, 0.8642522692680359, 2.666038990020752, -0.27798980474472046, -1.64926016330719, -0.9683077335357666, -0.6653504371643066, 0.1922772228717804, -2.2566659450531006, 0.9310171604156494, 1.299481749534607, -0.9444788098335266, -1.9135997295379639, -0.19476842880249023, -0.850118100643158, 0.8601691722869873, 2.802947759628296, -1.9100064039230347, -1.5808544158935547, 0.44829660654067993, 0.4841935932636261, -0.3003270626068115, -0.27979159355163574, -0.681029200553894, 0.5212182998657227, -3.1733860969543457, 0.45199525356292725, 1.5244345664978027, 0.9029456973075867, -1.8441423177719116, 1.4049150943756104, 0.6683386564254761, 1.2020645141601562, 0.439242422580719, 0.3075993061065674, -0.1351306438446045, -0.360639750957489, 0.06663268804550171, -0.6594476699829102, -1.0516306161880493, -1.772274136543274, -3.0721583366394043, -0.4032895863056183, 0.8353453278541565, -1.7721168994903564, -0.7197608947753906, -2.665008783340454, -1.2058377265930176, 0.8571285009384155, 0.045615434646606445, 2.70509672164917, -0.5896707773208618, 1.0107190608978271, 0.6650668382644653, 0.5528010129928589, 0.7611876726150513, 0.3666222095489502, -0.521312415599823, -0.914301872253418, 0.3115004599094391, -1.2470203638076782, -0.3744976818561554, 2.139298439025879, 1.5107638835906982, 0.6035175323486328, 2.645688533782959, -2.235383987426758, -0.8431497812271118, -1.379452109336853, -0.6080989241600037, 3.13169002532959, -0.19241464138031006, -0.3819858133792877, -0.40261006355285645, -1.893118143081665, 0.0840451717376709, 1.423771619796753, 0.09897810220718384, 2.618185520172119, -0.6371639966964722, -0.8802782297134399, 2.2074742317199707, 1.2208759784698486, 0.2982344627380371, -1.9130719900131226, 0.16395437717437744, -1.4172453880310059, 1.3055939674377441, -1.5690290927886963, -2.394432783126831, 0.5177319645881653, -0.1971467137336731, -1.8584835529327393, 1.0196689367294312, -0.9753446578979492, 0.8899371027946472, -0.7348260283470154, -1.8282109498977661, 1.247485637664795, -2.4471402168273926, 1.747251033782959, -2.1227381229400635, 1.4690046310424805, -3.0516462326049805, -0.7050837874412537, -1.9207327365875244, -0.8303710222244263, -1.20615816116333, -0.013903975486755371, 3.79593825340271, 1.3674230575561523, 0.04075658321380615, -2.199235677719116, 0.42001330852508545, -0.8295226097106934, -0.5466587543487549, -0.32806098461151123, 2.0577597618103027, 0.3669211268424988, -0.4698757529258728, 0.40584152936935425, -0.6514935493469238, -1.299922227859497, -0.5559326410293579, 0.14573752880096436, -0.27725982666015625, -1.2670271396636963, -0.39367958903312683, -0.5035380125045776, 0.0385756641626358, -1.989822268486023, 1.4264206886291504, 1.4871834516525269, 0.3340926170349121, 0.21162812411785126, -0.2826821804046631, -0.877472996711731, -0.8075743317604065, -0.37512749433517456, 1.5468019247055054, -2.095867156982422, -1.7625993490219116, -0.14999166131019592, 1.0517585277557373, -0.2668546438217163, -0.09811733663082123, -0.7592052221298218, 2.3634910583496094, -1.2223665714263916, 0.7300928831100464, 3.2278945446014404, -1.654310703277588, -0.4533764123916626, 0.4294706881046295, -0.6556904315948486, 0.5703139305114746, -0.6129364371299744, 2.0544211864471436, 0.0037273764610290527, 1.0804020166397095, -0.2933942675590515, 1.5294969081878662, -0.056341588497161865, -1.1149814128875732, -0.8085017800331116, -2.308318614959717, 0.7696908712387085, 2.8902924060821533, 1.4952800273895264, 0.025339186191558838, 0.9721300601959229, 0.028218507766723633, -0.8475832939147949, 2.412877321243286, -2.8614656925201416, -3.5416078567504883, 0.02323995530605316, -1.3921650648117065, -1.3823015689849854, 1.422673225402832, 0.18427273631095886, 2.41363525390625, 1.791011929512024, 1.028205156326294, 0.6870315670967102, -0.26684051752090454, 1.48553466796875, -0.9838824272155762, -1.7221933603286743, 1.6875309944152832, -0.10135430097579956, -0.14956450462341309, -0.08002077788114548, -0.6435679197311401, 1.6822359561920166, 1.8501055240631104, -2.001065254211426, -2.098130464553833, 0.4140240252017975, 1.1202383041381836, -0.6657350659370422, -0.27797237038612366, -0.322787344455719, 0.6250998377799988, -1.568544626235962, 0.6431401371955872, 0.6625068187713623, -2.282546043395996, -0.21260026097297668, 2.4244937896728516, 0.2320023775100708, -1.2951968908309937, 1.7497522830963135, -0.6911768913269043, -0.36602914333343506, -1.6594337224960327, -0.3408718407154083, -0.71198970079422, 0.09566980600357056, 0.24664954841136932, -0.03843134641647339, -2.2094240188598633, 0.5272331237792969, 2.233726978302002, -0.8998260498046875, -1.277862310409546, -1.7630374431610107, -2.3716020584106445, 0.33993709087371826, 2.0162618160247803, -0.28095588088035583, -0.05560731887817383, -1.5826226472854614, 0.22450163960456848, 1.844185709953308, -0.5758801698684692, -2.6247148513793945, -1.8544528484344482, 1.4356902837753296, -0.9671761989593506, 0.8313013911247253, 1.2338001728057861, -2.1036128997802734, -2.416111946105957, -0.3141239881515503, 0.09160036593675613, -0.45847246050834656, 1.2890746593475342, 0.7720374464988708, -0.3613359332084656, -1.5010586977005005, -0.11231833696365356, -0.2335478663444519, 1.062038540840149, 0.40787750482559204, -1.6663084030151367, -0.8648126125335693, 2.2863683700561523, -2.2283847332000732, -0.9523827433586121, 1.8617298603057861, 0.8838603496551514, -0.31697988510131836, 1.2093467712402344, -1.2254602909088135, 0.047546178102493286, 0.5069369077682495, -1.1369643211364746, -1.09175443649292, 1.4720840454101562, -0.31734228134155273, -0.8129256963729858, 0.5766021013259888, -0.22369223833084106, 0.4826771020889282, 0.021668672561645508, -0.31686317920684814, -2.05096697807312, 0.2479005753993988, 1.2010457515716553, 1.2443970441818237, 0.07893955707550049, 0.1700601726770401, 0.09150087833404541, 1.3078835010528564, -2.714085102081299, -0.29758965969085693, 0.3732614815235138, 1.154146432876587, -0.2711861729621887, 0.6211917400360107, 0.4173600673675537, 1.5255844593048096, -0.3005716800689697, 0.15448158979415894, 1.7026597261428833, 0.44342631101608276, 1.440993070602417, -0.5371000170707703, -1.7606861591339111, -1.689749002456665, 0.7563662528991699, -0.5179413557052612, 3.0939040184020996, -0.8063317537307739, 0.4308071732521057, 1.7470529079437256, -1.819339632987976, -0.7228096127510071, -1.0270049571990967, -1.3301599025726318, -0.44252732396125793, 0.16099725663661957, 1.541560173034668, -0.2686901092529297, 1.6790258884429932, 0.5304494500160217, -2.06366229057312, -1.5599685907363892, -1.9440217018127441, 0.49088260531425476, 0.41220107674598694, 2.0627591609954834, -0.8927814960479736, -0.31295016407966614, 0.2598215341567993, -0.32317912578582764, 1.5313811302185059, -0.7515206933021545, 0.8459334373474121, -0.299241304397583, -0.9781819581985474, 1.1280159950256348, 0.05169503390789032, 1.0244135856628418, -0.5766791701316833, 0.5376802682876587, 2.715099334716797, 0.741807222366333, 0.8912726640701294, 1.4484851360321045, -2.1774022579193115, 1.8183174133300781, -1.8942339420318604, 1.2482408285140991, -1.589878797531128, -0.9706968665122986, -2.5200791358947754, 1.430837631225586, 0.45122748613357544, -1.3091319799423218, -0.2833893895149231, 1.2559943199157715, 0.8542087078094482, -0.23228013515472412, 0.3962109684944153, 0.4442348778247833, 0.7059707641601562, -0.5751516222953796, -1.9280872344970703, -1.3045449256896973, 2.7195513248443604, 3.235525608062744, -0.36509206891059875, 0.44389671087265015, 0.05165849253535271, 0.17742860317230225, -0.6948901414871216, 0.16102126240730286, 1.2595864534378052, 2.869318723678589, 3.952211380004883, -0.6516505479812622, -0.41969728469848633, 1.4231781959533691, 0.9002684950828552, -0.6998838782310486, 1.1757473945617676, -0.5903879404067993, -0.25996100902557373, 0.8285338878631592, -1.2266448736190796, -1.1925537586212158, 1.7310923337936401, -0.7120486497879028, -1.385160207748413, -1.5369253158569336, 1.7314623594284058, 0.09430080652236938, 0.05280613899230957, 0.7674868702888489, -0.3132402002811432, -0.5519198179244995, 2.5585083961486816, -1.8806376457214355, -1.5447897911071777, -0.9506767392158508, -1.2114924192428589, -0.29904186725616455, -0.7882763743400574, 1.599955439567566, -1.7062859535217285, -0.8363720178604126, 0.0519021637737751, 1.5059685707092285, -0.2779906392097473, 2.1120197772979736, 1.430986762046814, -0.4181925058364868, 1.2263305187225342, 0.9230301380157471, -3.632798671722412, -0.33816319704055786, 2.069293975830078, -0.02021026611328125, 0.796493411064148, -1.4162791967391968, -0.1306091547012329, -0.6479255557060242, 0.5294514894485474, 0.08230727910995483, 1.8772200345993042, 1.694413661956787, 2.302548885345459, -0.4896671175956726, 1.963313341140747, -1.2527095079421997, 0.8549302220344543, 0.9515808820724487, -0.6543338298797607, -1.0520243644714355, -1.446951150894165, 2.407768726348877, -0.7117159962654114, -0.09457424283027649, -2.2426774501800537, -3.254856586456299, -0.3439527750015259, 1.1306593418121338, 2.7020649909973145, 0.8279044032096863, 1.691727638244629, -1.347550630569458, -0.8557451963424683, 0.07745599746704102, -0.7186598777770996, 0.7952679395675659, -0.6363006830215454, -1.0630850791931152, -0.01163056492805481, -3.5113179683685303, -1.1149935722351074, 0.6089648604393005, 2.953765392303467, -2.035156011581421, 1.2593976259231567, -0.838674783706665, 2.075899839401245, 0.06516474485397339, -0.05309969186782837, -1.0260798931121826, 1.7803174257278442, -0.560025691986084, -0.3328041434288025, 0.6645333766937256, -1.1815403699874878, -1.5250484943389893, 0.14032019674777985, 1.9012970924377441, 0.17778143286705017, 2.7033867835998535, 0.05853545665740967, 0.15044265985488892, -0.8239895105361938, 1.5287747383117676, -1.5597010850906372, -1.2906121015548706, 2.269380569458008, -0.42588239908218384, -1.2283966541290283, 1.2693959474563599, 1.1385046243667603, 2.4175374507904053, 0.23364821076393127, 4.009893417358398, -0.7863204479217529, 1.4360003471374512, -1.246311902999878, -2.706864833831787, -1.1762535572052002, -0.3656522333621979, -0.36861753463745117, 0.342111200094223, -3.1035919189453125, -1.3359081745147705, -0.49386775493621826, -0.5539526343345642, 2.6355347633361816, -0.34101802110671997, -2.511897087097168, 1.7565240859985352, -0.7713984251022339, 0.9546959400177002, -2.2772064208984375, -2.7184503078460693, -0.29701244831085205, 1.5210535526275635, -1.2273441553115845, -0.6605856418609619, 0.19648438692092896, -0.738236129283905, -0.4566340148448944, 0.9475728869438171, -3.0717339515686035, -2.587836265563965, -0.478473424911499, 0.5250731706619263, -1.261309266090393, 0.09555602073669434, -0.38530275225639343, 1.0430327653884888, -1.072458267211914, 1.2604997158050537, 0.2159992754459381, -0.919961154460907, -1.4737279415130615, -2.3340938091278076, -3.9011826515197754, 1.1369023323059082, -1.240204095840454, 3.176459312438965, 2.452556610107422, -1.5189850330352783, -0.7057511210441589, -0.33192360401153564, 2.2184269428253174, -1.9507288932800293, -1.763216495513916, 1.6192799806594849, -0.05261735990643501, -1.8374298810958862, -1.6329309940338135, -2.2544350624084473, -0.9874811768531799, -0.7181330323219299, -1.4207158088684082, 0.264293372631073, 0.23999708890914917, -3.4442129135131836, 0.38170647621154785, 2.3152782917022705, -2.8712873458862305, 2.3667545318603516, 0.8211715817451477, 2.005096197128296, 1.0957248210906982, 1.1232490539550781, -0.3017154932022095, -2.021472930908203, -0.06866741180419922, 1.1702924966812134, -0.32091671228408813, -0.9512647986412048, -1.4565598964691162, 1.1465171575546265, -1.0340075492858887, 0.19431346654891968, -0.005642056465148926, -3.5466301441192627, 1.6231237649917603, 1.2927424907684326, -0.45869356393814087, 3.0334866046905518, -0.8029016256332397, -0.04529881477355957, -0.8691551685333252, 0.636349081993103, -2.948176622390747, -1.601555585861206, 1.6209428310394287, -1.7518410682678223, 0.45436033606529236, -0.5145488977432251, 1.245100975036621, -2.616283893585205, -2.1107594966888428, -2.753653049468994, 2.036816358566284, -0.5775808095932007, 3.067322015762329, -1.5652110576629639, 0.951174795627594, -2.9813406467437744, 0.6950716376304626, 0.0065649449825286865, -0.3827487528324127, -2.713657855987549, -0.33596208691596985, -0.5520284175872803, 2.4994544982910156, -0.05906994640827179, -1.1169145107269287, 2.533698320388794, 0.9286295175552368, 1.5335971117019653, -3.0216875076293945, -0.1413591504096985, 0.2588956356048584, 0.659352719783783, 0.8031947016716003, -0.9359559416770935, -0.16161030530929565, -0.5651420950889587, 2.039463996887207, -1.0592877864837646, -0.4745239019393921, -1.8435273170471191, 0.690316915512085, 3.8983826637268066, -0.19840562343597412, -2.1984901428222656, 2.1703779697418213, -0.35018593072891235, -0.43748876452445984, 0.8829242587089539, -1.107818603515625, -1.4535861015319824, 1.28983473777771, -0.5322751998901367, 1.178848147392273, -2.1332013607025146, -0.21865683794021606, -1.1818675994873047, -0.6039503812789917, -0.6703933477401733, -0.2892546057701111, -0.24464645981788635, -1.4731175899505615, 0.42948952317237854, 1.0268969535827637, -1.3338004350662231, -1.4067909717559814, -0.7140230536460876, 0.020601898431777954, 0.02056252211332321, -1.2838252782821655, 0.7107970118522644, 0.28953981399536133, -1.883592963218689, -1.3452849388122559, 0.17470192909240723, 0.37179040908813477, -0.015535607933998108, -1.0303866863250732, -0.24954569339752197, 2.0710504055023193, 1.1719658374786377, -0.6639182567596436, -0.32565271854400635, 0.4093278646469116, -0.25124549865722656, 2.283424139022827, -2.6639413833618164, -0.7525147795677185, 1.2312253713607788, -0.016813993453979492, 1.090584635734558, -0.5403535962104797, -0.7072479724884033, -1.2523053884506226, -0.8604398965835571, 0.6584055423736572, 1.4881298542022705, 0.6398127675056458, -0.2494419813156128, 2.2474710941314697, 1.379805088043213, 0.029779016971588135, 1.815450668334961, 1.3407390117645264, -0.1267370730638504, 0.21069860458374023, 2.8821911811828613, -0.26581788063049316, -0.42680925130844116, 1.232820749282837, -2.4796013832092285, 0.5009828805923462, 0.6429200172424316, -0.6573606133460999, -0.6434450149536133, 1.0868253707885742, 0.34969064593315125, -1.3474032878875732, -0.37352830171585083, -1.1012495756149292, 0.028736114501953125, -0.9114776849746704, 2.598275661468506, 1.7432119846343994, -1.9358952045440674, 1.2274975776672363, 1.3340317010879517, -0.7090634107589722, -0.9828993082046509, -0.9209659099578857, -0.11309289187192917, 1.469502329826355, 0.21689851582050323, -1.05434250831604, -1.5325405597686768, 1.793014407157898, -0.9047436714172363, -1.745539665222168, 0.13301604986190796, 0.3395235240459442, 1.588748812675476, 0.19229210913181305, 2.16800856590271, 1.0658857822418213, 0.12833216786384583, -0.6122652292251587, -2.792509078979492, -0.9159430265426636, -0.15513229370117188, 0.7891348004341125, -1.635968565940857, -2.3521149158477783, -0.4306182563304901, 2.1453328132629395, 0.0008688569068908691, -0.23514701426029205, -3.6178040504455566, -2.0394859313964844, 1.2794840335845947, 1.944893479347229, -2.511079788208008, -1.335058569908142, 0.08343593776226044, -0.4026302695274353, -0.47325021028518677, -1.7312368154525757, 1.7612791061401367, 1.786749005317688, 0.18072697520256042, -0.04553115367889404, 1.7709956169128418, -0.4248202443122864, -3.5291976928710938, -0.8783121705055237, -2.7500550746917725, 1.2073484659194946, -0.9947812557220459, -2.251857042312622, -1.124558448791504, -2.0054590702056885, 0.541293203830719, -0.31343144178390503, -1.331974983215332, 1.2457890510559082, 0.8167359232902527, 3.2588138580322266, -1.3048291206359863, -0.59117591381073, 0.8719475865364075, -1.228391408920288, -1.517418384552002, 0.8463798761367798, -0.7869688272476196, -3.1909797191619873, -0.8814197778701782, -2.497196674346924, 1.520706295967102, -2.218695878982544, 0.15340521931648254, -1.252193808555603, 1.992899775505066, 0.568217933177948, -1.5743610858917236, -0.3407924175262451, 2.412656545639038, 0.5471512079238892, -0.010594837367534637, -1.3982441425323486, -0.6331607103347778, -0.1246749758720398, -0.6659075021743774, 2.1396360397338867, -2.1047255992889404, -1.7555274963378906, 1.1439366340637207, -2.453700065612793, -3.423558235168457, 1.393728256225586, -1.1674630641937256, 2.1675848960876465, 0.9900506734848022, -1.7828028202056885, -1.0019549131393433, -0.9033613204956055, 0.03746533393859863, 0.8166278004646301, 0.6987854242324829, 1.5119069814682007, -0.5154387354850769, 0.874237060546875, -2.624955177307129, -1.3596460819244385, -0.2334580421447754, 2.0828189849853516, 1.9958550930023193, -1.094075322151184, 2.55257511138916, 0.10474878549575806, -2.3959548473358154, -2.625929832458496, -1.5264954566955566, 2.043818473815918, -1.9568791389465332, -0.4993031620979309, -0.760700523853302, 1.0356557369232178, 2.317884683609009, -1.942902684211731, 0.8787949085235596, -1.4632853269577026, 0.4408929944038391, -1.22880220413208, 0.6881006360054016, -0.7361644506454468, 1.7368303537368774, -1.0748069286346436, -1.760427713394165, -1.457112193107605, -0.822059154510498, -0.64736407995224, -2.910529136657715, -0.9441479444503784, 4.333625793457031, 0.8459898233413696, 0.5517323017120361, 1.3900630474090576, -0.5828306674957275, -0.7408909201622009, 0.8168658018112183, -1.5823123455047607, -2.2212233543395996, 0.7462067008018494, -4.56368350982666, 1.9554308652877808, 1.1439510583877563, -0.5578749775886536, -1.2020570039749146, -3.4199557304382324, 0.6352523565292358, -0.1519513726234436, 0.7294485569000244, -0.4238708019256592, -0.8369362354278564, -0.2908279001712799, -0.8532894849777222, 2.172974109649658, 3.0808770656585693, 0.12871436774730682, 0.6968542337417603, 0.7399392127990723, 2.410261631011963, 0.6974282264709473, 1.2844367027282715, 0.934333086013794, -2.5423696041107178, -1.7965271472930908, -1.2146801948547363, -0.5915418863296509, -0.19530974328517914, 0.6118665933609009, 1.1193846464157104, -1.118137240409851, 1.5098707675933838, 0.43400681018829346, 0.8039296865463257, -1.144286870956421, -0.8987388610839844, -1.2764208316802979, -0.3773210048675537, 0.3751230239868164, 1.9039685726165771, -1.1099879741668701, -0.9557095766067505, 0.793420672416687, -1.4095678329467773, -1.7682380676269531, -2.400509834289551, 1.0539528131484985, -1.0706963539123535, -1.3950086832046509, 0.6134471297264099, 0.45875465869903564, 1.2616338729858398, -0.20346182584762573, -1.4567856788635254, 0.8736476898193359, -0.8308299779891968, -1.4418072700500488, 0.022111326456069946, 0.963082492351532, 0.9791754484176636, 2.1307873725891113, 1.1729543209075928, -1.106804370880127, -3.0052623748779297, 3.1217308044433594, 1.419616460800171, -0.7031749486923218, -0.4168750047683716, -1.4752027988433838, 0.1680101454257965, -2.641709089279175, 1.6888089179992676, -1.4460818767547607, -0.17292320728302002, -0.06459808349609375, 0.871099054813385, 0.7699474692344666, 0.22498199343681335, 0.23430538177490234, -0.3194740414619446, 0.8043665289878845, 0.20515680313110352, 0.3839621841907501, 0.8197921514511108, -0.5837727785110474, -2.393526077270508, -0.2470380663871765, 2.7317044734954834, -1.6223481893539429, 2.092988967895508, 0.8111704587936401, 1.8515514135360718, 1.102188229560852, -1.2825415134429932, 0.5289534330368042, 2.0174472332000732, -0.7717167139053345, -2.277508497238159, 0.6562673449516296, -0.9238789081573486, -5.048373222351074, 0.5632288455963135, 0.5246604084968567, 1.2470893859863281, 0.912613034248352, 1.6573158502578735, 0.9386820793151855, -2.166405200958252, -0.0777251124382019, 0.9403690099716187, 0.6278071999549866, -0.12543624639511108, -1.2353161573410034, 1.4383114576339722, -0.8446236848831177, -0.7469492554664612, 0.9751152396202087, -1.0222331285476685, -0.4537181258201599, 1.975271463394165, -1.3716239929199219, -0.48311036825180054, -0.3205118179321289, -0.39052891731262207, 0.34989726543426514, -1.494983196258545, -0.147574782371521, 0.8399679660797119, -0.5509392619132996, -1.9121826887130737, 0.8868072032928467, -0.13774825632572174, -0.567130446434021, -0.08126524090766907, 0.090137779712677, -2.1889736652374268, 0.9577494859695435, -0.651662290096283, 0.4696390628814697, 0.6502261161804199, -0.8963385224342346, -0.5244179368019104, -3.4914374351501465, 0.06644117832183838, 0.6587984561920166, -1.5897581577301025, 1.8773956298828125, -0.4251776933670044, -1.5870842933654785, 2.1127614974975586, -2.5813815593719482, -2.7331860065460205, -0.6443384885787964, -1.4071426391601562, -0.5427814722061157, 0.5616247653961182, -4.0347137451171875, 0.5116435885429382, -2.310657501220703, 0.609788715839386, -0.8684705495834351, -1.3078432083129883, 2.9579405784606934, -3.313220739364624, -0.76995849609375, 1.6866847276687622, 0.9063205718994141, 0.3197011947631836, -0.6985948085784912, 1.1365509033203125, 1.8681004047393799, -1.0254591703414917, 2.3741250038146973, -2.785919666290283, -0.25506591796875, -1.0310717821121216, -1.3457244634628296, 1.18685781955719, -2.2739832401275635, -0.258601576089859, 1.28135347366333, -1.1060446500778198, 0.5427747964859009, 0.5347979068756104, -0.5432245135307312, 1.0008374452590942, -2.6804308891296387, 1.2693588733673096, -0.0732032060623169, 0.3833553194999695, 0.8160852789878845, 0.9567269086837769, 1.4831820726394653, 0.9844407439231873, -3.0021677017211914, 0.09309600293636322, -1.3003735542297363, -4.019843101501465, -0.3188275396823883, -3.050489664077759, -0.05053901672363281, 0.5254791975021362, -0.12782901525497437, 0.7195648550987244, 1.9384065866470337, -1.5347487926483154, 2.0283589363098145, -0.2038457989692688, -1.5077295303344727, 0.0849539041519165, -0.7309442162513733, 2.251211166381836, -1.3070785999298096, 1.1318432092666626, -0.09590637683868408, -1.0902552604675293, 3.0928280353546143, -2.0509896278381348, -1.2436492443084717, -2.0154006481170654, 0.2177894115447998, 0.7552250623703003, 0.2908053994178772, 2.280435085296631, 0.6117204427719116, -0.45564860105514526, -1.4331265687942505, -0.8559569120407104, 0.16133274137973785, -0.7048863768577576, -0.9853726625442505, 1.3873298168182373, 0.5221325755119324, 0.9700048565864563, 0.8144978284835815, -0.9268448948860168, -0.4382367730140686, -1.9330763816833496, 1.9153084754943848, 0.14704322814941406, -0.2568705677986145, 0.3938852548599243, 1.1354069709777832, -0.2800232172012329, -1.3359441757202148, -0.4402589797973633, -1.0129148960113525, 1.4776959419250488, -0.24666370451450348, -0.8775328397750854, 1.742948293685913, -1.2133640050888062, 1.5318117141723633, -1.3193473815917969, -0.1684739589691162, -0.6821063160896301, -0.7450461983680725, 0.8067741394042969, -0.20282603800296783, -0.9072862267494202, -2.792328119277954, 1.166191577911377, 2.1177453994750977, -1.5093616247177124, 0.3504275381565094, 2.758810043334961, -0.5175029039382935, -0.019724786281585693, -2.128981113433838, -0.44015783071517944, 1.124381422996521, -0.25105464458465576, 0.8650047183036804, 0.8852771520614624, -1.0335441827774048, 0.13169239461421967, -1.0576486587524414, 0.9526253938674927, -2.5692992210388184, -1.356874704360962, 1.5157811641693115, 0.38286465406417847, 1.3514959812164307, -0.8809653520584106, -0.0922856330871582, -0.25927790999412537, 0.4537755250930786, 2.1267547607421875, -1.4321348667144775, 0.9174333810806274, 3.66110897064209, 0.9318019151687622, 0.2672439217567444, -1.091214656829834, 4.702301502227783, 0.4241393208503723, -1.7408477067947388, -1.3221919536590576, -1.4651377201080322, 1.417964220046997, 1.9051668643951416, -0.9455316662788391, -0.6246909499168396, -1.9652186632156372, 0.19683372974395752, 1.546032190322876, 0.14510317146778107, 1.773087739944458, 1.680727243423462, -0.8804012537002563, 1.6909723281860352, -3.0902438163757324, -0.9601024389266968, -1.367537498474121, 0.8852285742759705, 0.1813993602991104, 2.7522482872009277, -0.7468669414520264, -0.456842839717865, -1.5880889892578125, 0.7466636896133423, 0.08246678113937378, 0.8328052163124084, 0.07305586338043213, -0.006930530071258545, -0.580635666847229, 0.7300856113433838, 0.26696503162384033, -0.3711683750152588, 0.48352715373039246, 0.8263706564903259, -1.4124058485031128, 1.0867326259613037, 2.8784451484680176, 0.6923828125, -0.25703516602516174, -2.5349481105804443, 0.5029714703559875, 1.294560432434082, 2.765361785888672, -0.25601303577423096, -0.5965448021888733, 2.0715084075927734, 0.9277023077011108, 1.174112319946289, 0.5865994095802307, 0.7831929326057434, 0.6578534245491028, 1.1303858757019043, -0.711121678352356, 0.6591352820396423, -0.05206277221441269, -1.1370856761932373, -0.4299657344818115, 0.9440617561340332, -0.5183382034301758, 1.7133630514144897, 0.2813049554824829, 0.8810928463935852, 0.5149140954017639, -0.08016553521156311, 4.140599250793457, 1.3808573484420776, 2.665337562561035, 0.9202019572257996, 3.0812830924987793, 1.4875342845916748, 1.7837032079696655, -1.565571904182434, 1.3743765354156494, -0.16790884733200073, 0.11971648037433624, -0.7458009123802185, 0.01929342746734619, 1.1264652013778687, -2.3445773124694824, -2.207808017730713, 1.8098223209381104, -0.8535315990447998, 0.12715548276901245, 1.387077808380127, 1.177449345588684, 0.8362087607383728, -0.04433516412973404, -0.4335486590862274, 0.22631078958511353, 0.8128479719161987, -0.4145335853099823, 0.335582435131073, 0.1369907259941101, 0.5100980401039124, -1.9780528545379639, -1.651479721069336, 1.232999563217163, 1.521974802017212, 0.565716564655304, -1.6464381217956543, 1.7943137884140015, -4.016246795654297, 0.9100371599197388, 2.256765842437744, -0.7159518003463745, 2.4530906677246094, -0.6854616403579712, -0.7116018533706665, 0.5591461062431335, -1.549399971961975, -0.7044830322265625, -3.0701346397399902, -2.993772029876709, 2.713458776473999, 0.23984789848327637, 1.1921409368515015, -1.4543695449829102, -2.0322251319885254, 1.0323622226715088, -0.7213445901870728, -1.3726646900177002, -1.2497572898864746, -0.14553967118263245, 1.0462347269058228, 0.30410945415496826, 0.10040342807769775, 1.186718225479126, 1.358824610710144, -0.3990541696548462, 1.59736967086792, 2.3312458992004395, -1.1179429292678833, -1.0037050247192383, 1.882112979888916, -0.7109719514846802, -0.4138304889202118, -3.316263198852539, 0.26255685091018677, 0.6337038278579712, -0.17048484086990356, -1.4381622076034546, 0.6727710962295532, -0.5384393930435181, -0.6775364875793457, 0.796457052230835, -1.1175800561904907, 0.04286624491214752, -1.785791039466858, -0.23305630683898926, 1.0508204698562622, -0.2659972906112671, -0.4770689010620117, -1.5422039031982422, 2.4571235179901123, 0.20969200134277344, 1.3926095962524414, 0.23874658346176147, 1.2382049560546875, -2.058464765548706, -0.3610701560974121, -1.8647899627685547, 2.5278544425964355, -0.8758066296577454, -0.6539466977119446, 1.143385410308838, -1.8664507865905762, -0.202234148979187, 1.7300586700439453, 1.623618245124817, 2.1366937160491943, 1.6964248418807983, 1.720247507095337, 0.7063963413238525, -0.5083943605422974, 1.3828704357147217, -0.4922621250152588, 0.5972196459770203, -3.517878770828247, -0.9983457922935486, -0.43336671590805054, -1.7184734344482422, -0.37206992506980896, 3.503006935119629, -2.1024889945983887, -1.6320314407348633, -1.6728346347808838, -0.6898027658462524, -0.3562106788158417, -0.05779832601547241, 0.9369001388549805, 1.884242057800293, 0.5061609148979187, 0.6662963032722473, -1.3111116886138916, -0.7329617738723755, 0.6758207082748413, -0.3209860920906067, 0.9752995371818542, -0.5413931012153625, 2.7521188259124756, -1.4188870191574097, 1.2262554168701172, 1.0770630836486816, 1.391772747039795, -0.1518363356590271, -3.7490363121032715, 0.7909669876098633, 2.431013584136963, 2.9692251682281494, -1.8174190521240234, -1.0312731266021729, 0.4217662215232849, 2.839672565460205, 0.7412018775939941, -2.941531181335449, -0.6182131767272949, 0.029657721519470215, 2.5567522048950195, -0.8620242476463318, -1.0539098978042603, -0.7516146302223206, 0.4640309810638428, -0.5973248481750488, -3.678525447845459, -0.778363049030304, 0.9067151546478271, 1.1337881088256836, 0.30649322271347046, 1.0103232860565186, -0.9871186017990112, 0.5600515604019165, -3.090775966644287, -0.263761430978775, 0.6698657870292664, 1.4488065242767334, 0.8323338031768799, -1.6995723247528076, 0.5650894641876221, -2.5503909587860107, 0.384960412979126, 0.1385081708431244, 2.625091314315796, -1.099626898765564, -0.2413199543952942, 0.26395875215530396, 1.3236960172653198, -1.47365403175354, 1.3775279521942139, 1.3972201347351074, 1.9337373971939087, -0.5633013248443604, -1.9036343097686768, -1.2839261293411255, -0.2373129427433014, -2.038543462753296, 0.9316949844360352, -1.0814368724822998, 1.9555444717407227, -0.7965339422225952, 0.6846409440040588, 1.9581255912780762, 0.7877327799797058, -1.8869225978851318, -1.269784688949585, -0.8559874892234802, -0.19932007789611816, -1.9651210308074951, 1.6048200130462646, 2.8199820518493652, 1.7440260648727417, 0.7548611164093018, 0.11515393853187561, 1.151474952697754, -0.4559037685394287, -0.4384029507637024, 2.2812488079071045, 0.4631859064102173, -0.6815312504768372, 0.47817033529281616, 0.3786710500717163, -0.022402524948120117, -0.3002427816390991, -0.6772971749305725, 0.28040003776550293, 1.7458796501159668, -0.9052138328552246, 1.0654304027557373, 1.215649127960205, 0.6795116066932678, 0.2931153476238251, -1.6029548645019531, 0.5674151182174683, 1.490767240524292, 1.1226541996002197, -0.10342605412006378, 0.7382630109786987, -0.9370524883270264, 0.6513125896453857, 1.2378623485565186, -0.6058350801467896, 0.708003044128418, -0.8682206273078918, -1.302477240562439, -0.663764476776123, -0.6558998823165894, 4.102893829345703, -0.4690167009830475, -1.3037515878677368, 0.8476545810699463, 1.2966582775115967, 0.8861480951309204, -1.5461055040359497, -0.032128721475601196, -1.408339500427246, -0.6879329085350037, 2.2451915740966797, -1.340904951095581, -1.7431669235229492, 3.0461816787719727, -1.0053393840789795, 0.732647180557251, 0.43333008885383606, 2.5836777687072754, 0.6010839939117432, 0.5879669189453125, 2.1702535152435303, -1.0731817483901978, -1.3306496143341064, 1.2006206512451172, 1.4101147651672363, -0.6581509113311768, -0.16800463199615479, -1.112229347229004, -1.1756139993667603, -1.2625904083251953, 0.9539246559143066, 1.555742621421814, -1.1832637786865234, 2.402986526489258, 0.843147873878479, -2.025822639465332, -2.020689010620117, -0.3079163432121277, -0.20608627796173096, -0.8527317643165588, 0.4844856560230255, 0.5951005220413208, 3.3266611099243164, -0.8312007188796997, 1.07412588596344, 0.00794994831085205, 0.8071954250335693, -5.232966899871826, -0.10024802386760712, -2.449166774749756, -0.6110118627548218, 0.3837284445762634, 0.5441856980323792, 1.4205281734466553, -1.3796470165252686, -0.9245301485061646, 2.2222373485565186, 0.4757866859436035, -0.8876781463623047, 1.258389949798584, 0.5377513766288757, -0.4660181999206543, -1.837484359741211, -0.6281983256340027, -0.8628779649734497, -0.7663007378578186, 0.5682594776153564, 2.5374703407287598, -0.6442830562591553, 0.03720355033874512, -2.3014445304870605, 0.7136175632476807, -0.22917385399341583, 0.21608150005340576, 0.03578394651412964, 1.1330491304397583, 0.418660044670105, 0.49270099401474, -1.0939102172851562, -2.275202751159668, 1.7123267650604248, -0.028319716453552246, 0.08324219286441803, -1.87020742893219, -0.308062344789505, -0.8026306629180908, -3.283749580383301, -0.49392080307006836, -1.4452931880950928, -1.9745426177978516, 0.46729665994644165, 0.30628061294555664, 0.6134241819381714, -0.09934180974960327, -0.8888359069824219, 0.5365030765533447, 1.1051719188690186, 0.8209692239761353, -2.3061211109161377, -1.5175591707229614, 0.22389936447143555, -0.5864018797874451, -0.6295807957649231, -0.525946855545044, 0.8063691854476929, 0.6723130941390991, 3.638545513153076, -0.6457262635231018, -0.3729211390018463, -0.5628916621208191, 1.1564507484436035, 0.3256778120994568, -1.6089556217193604, 1.459500789642334, 1.1006051301956177, 0.3008294999599457, 0.8706944584846497, 0.4842890501022339, -1.4117118120193481, 1.103228211402893, -0.24800652265548706, -0.04900866746902466, 0.8490791320800781, 1.5092427730560303, 0.42803287506103516, 0.9355231523513794, 0.7626405954360962, -0.25960713624954224, 0.5048289895057678, -1.7218151092529297, -0.2540212869644165, 1.336929440498352, -1.8633170127868652, -0.2859680950641632, 1.4846105575561523, 1.6966562271118164, 0.1547243595123291, -1.965888261795044, 1.1594197750091553, -1.6545984745025635, 0.6327422261238098, -0.1835530400276184, -1.0087890625, 0.7719168066978455, 2.3066201210021973, 2.0419719219207764, 0.38756412267684937, -2.0465526580810547, 0.3299946188926697, -0.051416486501693726, 0.7186102867126465, -2.027177095413208, 0.7157320976257324, 1.330809473991394, -2.0956480503082275, -1.1061208248138428, -2.8866519927978516, -2.340228319168091, 2.060368061065674, -1.2854907512664795, 1.4625113010406494, -2.5170044898986816, 0.2343391478061676, 1.7848796844482422, -0.3558151125907898, 0.8248296976089478, 0.15522032976150513, 0.47293710708618164, 0.723220705986023, -1.3433330059051514, -0.8391692042350769, 1.2352299690246582, 1.01775062084198, -0.09101909399032593, -0.1102319061756134, 0.17185252904891968, 0.3440702259540558, -1.060921311378479, 1.9419358968734741, 0.2819281816482544, -0.45873796939849854, 0.8621489405632019, -2.7960054874420166, -1.0297811031341553, 0.36165422201156616, -0.12991386651992798, 0.38537266850471497, -1.504022240638733, -1.166713833808899, 0.9795970916748047, -0.44360408186912537, -0.5317093133926392, 0.9468209147453308, 0.805077314376831, 1.3496043682098389, -2.2735915184020996, 2.8148481845855713, -0.9723467826843262, 0.8931506872177124, 0.5680148005485535, -1.0147294998168945, -0.47074073553085327, 1.3435471057891846, 0.6657891869544983, 3.3055338859558105, -1.8270944356918335, 3.43794584274292, -1.0519191026687622, -1.0165190696716309, -0.10230337828397751, 1.2683507204055786, 1.6769821643829346, 0.030132651329040527, 0.6531845331192017, -0.08011496067047119, 1.1767206192016602, -0.39823010563850403, -1.220445156097412, -0.8807149529457092, -0.3300056457519531, -1.4673340320587158, 0.7127472162246704, 0.10861530154943466, 0.5287492871284485, -1.950541377067566, 0.3984032869338989, -0.7388535737991333, 0.07091671228408813, -1.1909040212631226, 0.6645061373710632, -0.74019455909729, -1.6812282800674438, -0.890779972076416, -0.22518233954906464, 3.4733805656433105, -0.8078866004943848, -1.6752489805221558, -0.05729794502258301, -1.1832404136657715, -1.1206578016281128, -0.28740811347961426, -1.4008207321166992, -0.9091017246246338, -2.6822643280029297, 2.0397934913635254, -0.7567000389099121, 0.2537418603897095, 1.5304006338119507, -3.850492000579834, -1.626102328300476, 2.568169116973877, 1.6358418464660645, 1.9201322793960571, -1.0920590162277222, 0.2972540557384491, 2.059974431991577, 1.674381971359253, -2.2722175121307373, 1.6491597890853882, 1.422783374786377, -0.81621253490448, -1.9876741170883179, 0.5822161436080933, -1.6993016004562378, 0.4184437394142151, -0.30022984743118286, -1.614329218864441, 0.14443723857402802, -0.735202968120575, 1.6656749248504639, 1.37140691280365, 0.24818509817123413, 3.931006908416748, 2.65291428565979, 0.4048740863800049, -0.31061944365501404, -2.100224494934082, -0.46401292085647583, -0.40845444798469543, 1.8562651872634888, 0.052873849868774414, 0.03870345652103424, 0.2712588310241699, -1.4797370433807373, -0.9773024320602417, 1.612508773803711, 0.1494651436805725, 1.253509759902954, -2.608241558074951, 2.954500913619995, 1.5259618759155273, 0.4996429681777954, 0.13807761669158936, -0.30318552255630493, 2.217149019241333, -3.278324604034424, -0.25753816962242126, -2.9338836669921875, 3.2363638877868652, 0.11671601235866547, 0.49217140674591064, 1.3808763027191162, -0.6307576894760132, 0.177293062210083, -0.2652570903301239, 0.29348137974739075, 0.2814621925354004, -1.3067551851272583, -1.8498420715332031, -1.9468553066253662, -0.8433626890182495, 1.8186578750610352, -0.6149768829345703, -2.0230722427368164, -4.132066249847412, 0.9777573943138123, -0.6526151299476624, -0.7395390272140503, -0.17090728878974915, 0.5731766223907471, -1.2767888307571411, -0.11248445510864258, -0.7059454321861267, 1.898862600326538, -2.473670482635498, 1.3756824731826782, 1.7581565380096436, -0.24973583221435547, 0.2476707100868225, -0.11872929334640503, -0.29842841625213623, 0.47419339418411255, -2.4374754428863525, 0.21189677715301514, -1.613552451133728, -3.0560684204101562, 1.0371007919311523, -1.274292230606079, 0.461703896522522, 1.1772804260253906, 1.9058027267456055, 1.8760318756103516, -1.168997049331665, -3.587247371673584, -0.13323384523391724, -0.5836266279220581, 2.4548983573913574, 2.1058173179626465, 1.1723787784576416, 1.0565838813781738, 1.7465476989746094, -2.3312883377075195, 0.6955751180648804, -1.613340973854065, -0.8879918456077576, -2.059448480606079, 0.3627619743347168, 1.2914443016052246, 1.317217230796814, -1.112036943435669, -1.0906916856765747, 1.679573655128479, 0.7902687191963196, -0.6984712481498718, -0.18190264701843262, 0.6409420967102051, 1.7851512432098389, -0.6059765219688416, -0.5655162334442139, 0.47331345081329346, -2.0118651390075684, -1.4151959419250488, 1.0215569734573364, 1.9820624589920044, 0.7712021470069885, 2.8638699054718018, 0.42632177472114563, -0.7361781001091003, 0.6737878918647766, -1.3007125854492188, 0.6001070141792297, -3.763070583343506, 0.1655096411705017, 0.11856848001480103, -1.337449550628662, -0.152069553732872, -0.24693626165390015, -0.609010636806488, 1.3642772436141968, 0.2669556140899658, 1.5637941360473633, -1.3956891298294067, 1.417183756828308, -2.2821462154388428, 0.26844096183776855, 1.0885934829711914, -2.67260479927063, 1.3170485496520996, 1.8399744033813477, 1.6118252277374268, -2.091439723968506, -0.08217120170593262, -1.3139569759368896, -0.3090973496437073, -0.988440752029419, 0.44529539346694946, 1.7833746671676636, 0.7349499464035034, 1.5810242891311646, -0.8183501362800598, -1.1618356704711914, 2.86665678024292, -3.6814022064208984, 0.49660882353782654, 1.5210280418395996, -0.04329615831375122, -0.718153715133667, 1.7603414058685303, -0.797776460647583, 1.5656846761703491, 1.1357505321502686, 2.462109327316284, -0.11000871658325195, -1.7174456119537354, 1.3928451538085938, 0.8504584431648254, -2.7922730445861816, -0.5662128329277039, 0.8538169264793396, 1.4234809875488281, -1.1898809671401978, -1.2800867557525635, 0.8266776204109192, 0.6124003529548645, -1.394959807395935, -2.2117772102355957, -1.1367590427398682, 1.7276365756988525, 2.3320252895355225, 0.685294508934021, 1.542189598083496, 2.685143232345581, -1.4524222612380981, -0.9951977133750916, 1.777693271636963, 3.361060619354248, 1.1490960121154785, 0.7358425855636597, -2.032466173171997, 0.47664451599121094, -1.2524150609970093, -2.7545197010040283, -0.467064768075943, 2.0011837482452393, -0.749605655670166, 1.3164807558059692, -1.3330674171447754, -1.7341501712799072, -2.806954860687256, -1.28895902633667, 1.1052762269973755, 0.36251211166381836, 0.3992122411727905, -1.3697655200958252, -1.085505723953247, 0.09644794464111328, 0.6967427730560303, 2.4899230003356934, 1.6196967363357544, 0.4852694571018219, 1.6200239658355713, 1.3230876922607422, 0.050566911697387695, -0.005983069539070129, -2.399380683898926, 0.006223917007446289, 0.8458231687545776, 0.29275065660476685, -1.0168938636779785, -1.060728907585144, 1.071927547454834, -1.6334750652313232, -0.5213766694068909, -2.3254270553588867, -0.16864106059074402, 0.24995076656341553, 0.5518584847450256, -1.3492487668991089, -2.133633852005005, 1.363838791847229, -0.1798170804977417, -0.35551872849464417, 1.813811182975769, -0.8697662949562073, -0.317859411239624, -0.2770213186740875, 1.7346458435058594, -0.12245070934295654, -1.3685311079025269, -0.14664679765701294, -3.4538803100585938, -2.13930082321167, -1.2935047149658203, 1.9863839149475098, 1.1001319885253906, -1.1741670370101929, 1.7397007942199707, -0.9406874179840088, 1.6903823614120483, -0.24058973789215088, -0.10891294479370117, 1.9881782531738281, -1.3491432666778564, 0.2995694577693939, 2.817760467529297, -1.5974233150482178, 0.5689530372619629, 1.0730130672454834, -0.4365634620189667, -0.5955190658569336, 2.4607224464416504, 0.7177561521530151, 0.6181691884994507, -1.7139898538589478, 0.1281799077987671, -0.2040286660194397, -1.2531503438949585, 0.3307305574417114, -0.054599881172180176, 0.8364248275756836, -1.1663825511932373, -0.02266635000705719, -1.0315237045288086, -3.0709421634674072, 0.16757214069366455, 0.2620987594127655, 0.4543812870979309, -0.20027077198028564, 0.9408167600631714, 0.8975512981414795, 0.18444770574569702, -0.34112900495529175, -1.81576406955719, 0.20172376930713654, -0.1829073429107666, -0.5083505511283875, 0.9925656318664551, 0.20404386520385742, -1.8457098007202148, -1.7836335897445679, 2.4550952911376953, 0.6556177735328674, 2.0950093269348145, 0.8574140071868896, 1.8288636207580566, 1.1462448835372925, -1.9811058044433594, 0.11348766088485718, 0.3727596402168274, 2.1759331226348877, 0.8288227319717407, 1.6523587703704834, 1.421798586845398, -0.6307331323623657, -2.251793146133423, 0.906693696975708, -1.8941731452941895, -1.0361032485961914, -2.367353677749634, -0.8279479146003723, -0.07843726873397827, 1.8230257034301758, -0.5557690262794495, 0.6808895468711853, -0.14318060874938965, 0.8800210952758789, -1.0513839721679688, -1.4762349128723145, -1.4870175123214722, 0.40905681252479553, 1.9736614227294922, -0.3173714578151703, 0.5159061551094055, 1.5013370513916016, 0.4572339653968811, -2.6886146068573, 0.8913054466247559, 2.152170181274414, -0.02033865451812744, -0.03371351957321167, -1.3127987384796143, -1.0826358795166016, 1.0650932788848877, -0.5891924500465393, 0.5705947875976562, 0.30947089195251465, 1.3875532150268555, -2.8855233192443848, -0.8179501891136169, 0.024733543395996094, -1.5859527587890625, -0.2367575764656067, 0.6625256538391113, 2.916609287261963, 2.37278151512146, -0.04430767893791199, 0.13525685667991638, 0.7638692855834961, -0.7547377347946167, 2.946155071258545, 2.705394983291626, -0.08771055936813354, 3.9229283332824707, -1.2160580158233643, -1.0631020069122314, 1.9427528381347656, 0.06150907278060913, -1.2565537691116333, 0.6965450644493103, -1.8807817697525024, -4.762026786804199, -0.6682490706443787, -1.1960957050323486, -1.5898447036743164, 1.6992757320404053, -1.3937084674835205, -0.7404333353042603, -0.4802161753177643, 0.10512983053922653, 0.23711562156677246, -0.16663962602615356, 0.09871309995651245, -0.794744074344635, -1.5958822965621948, 0.918933629989624, 0.37968629598617554, 0.46885934472084045, -1.1612660884857178, 1.5574032068252563, 1.2772222757339478, 0.8341826796531677, 0.9026408195495605, 2.3530020713806152, -0.13601499795913696, -1.9900909662246704, -0.8228743672370911, -0.35919851064682007, -1.906036615371704, -0.4936875104904175, -0.782137930393219, -2.5018153190612793, 1.5184211730957031, 0.7873046398162842, 0.1589992344379425, -1.078403115272522, 0.43373024463653564, -0.6815620064735413, 1.1156193017959595, 0.12423789501190186, -2.1901743412017822, -1.0521283149719238, 0.006441473960876465, -1.183249592781067, 0.4050672650337219, -2.5643234252929688, -2.2355294227600098, 0.05676990747451782, 0.26406967639923096, -2.310781478881836, -0.8946642279624939, -2.8541789054870605, 2.324575424194336, 1.5967388153076172, 1.0015267133712769, 1.363447904586792, 0.39787375926971436, -0.7762528657913208, -0.7488594055175781, 0.2541677951812744, 4.322842597961426, -1.490378737449646, -0.9792336821556091, -1.1629903316497803, -1.9843831062316895, 1.1218549013137817, 2.5102765560150146, -2.0612261295318604, -0.4509601593017578, 1.0349372625350952, -0.9726474285125732, -0.06819599866867065, -1.2550973892211914, -0.9145491123199463, 0.22745496034622192, -0.5597661733627319, -0.12582166492938995, 1.798518419265747, -0.7878122329711914, -1.4980175495147705, 3.783125400543213, -0.8963151574134827, 0.15773794054985046, -0.1731812208890915, 1.6833621263504028, -0.14836174249649048, -0.3543432056903839, 0.7456935048103333, -1.5691041946411133, 2.853785991668701, -2.3544681072235107, -1.6181524991989136, 0.1987760066986084, 0.25347915291786194, -2.931581497192383, -0.8805381059646606, -2.6536316871643066, 2.7993264198303223, 0.8034243583679199, -0.3759310245513916, -0.7966749668121338, 0.950027346611023, -0.7974991798400879, 0.768119752407074, -0.3527536392211914, -0.5826679468154907, 1.1534408330917358, 0.9698907732963562, -1.7154674530029297, -1.0807597637176514, -0.5515381097793579, 1.217665672302246, -2.473931312561035, 0.3919171094894409, 0.5888451337814331, 1.2881947755813599, -0.5797460675239563, -2.699810028076172, 1.4653544425964355, -1.7496542930603027, 0.7652207612991333, -2.997157573699951, -1.6405878067016602, -0.21470290422439575, -1.0091301202774048, 2.534193992614746, 0.05963429808616638, 0.5877752304077148, -1.1670987606048584, -0.8900986909866333, -1.3633818626403809, 3.108846664428711, 1.2423890829086304, 1.5139544010162354, -2.345426082611084, -0.9528764486312866, 1.2983543872833252, 2.372640609741211, 4.147420406341553, -1.0932669639587402, 0.21756047010421753, 1.4565327167510986, -2.350121259689331, -0.9927460551261902, 0.20788037776947021, 0.11673974990844727, -1.0836148262023926, 2.2138473987579346, 1.2197265625, -1.1453957557678223, -0.3990463614463806, 1.9770734310150146, -0.7470946311950684, 1.285034418106079, -1.7352830171585083, 1.1662907600402832, 1.4215208292007446, 3.493821620941162, -1.4599612951278687, 2.350445032119751, -0.3309554159641266, -2.1053268909454346, 1.7428650856018066, 1.3552311658859253, -0.14867347478866577, 0.11023461818695068, -1.7716484069824219, -0.40783214569091797, -0.9458216428756714, -0.2761043906211853, 2.6591129302978516, -0.7799087762832642, 1.5291237831115723, -0.35630157589912415, 0.573279857635498, -1.7742477655410767, 1.1623098850250244, -3.4505205154418945, 2.82243013381958, 0.02135142683982849, -1.2226858139038086, 1.6371722221374512, -2.5274646282196045, -4.010221481323242, -0.6810398697853088, 2.7457683086395264, -0.4807776212692261, 2.58418607711792, 0.23578637838363647, 2.054225206375122, -1.0951586961746216, -1.1820199489593506, -0.8530949950218201, -1.3084731101989746, -0.8590444326400757, 0.9311829805374146, -1.188689947128296, 0.7019718289375305, 2.548609972000122, 3.2112815380096436, -1.2476450204849243, -0.30328047275543213, -1.5940213203430176, -0.3261352479457855, -2.3822431564331055, 1.5805169343948364, -2.2026729583740234, -1.143460988998413, -0.7916011810302734, -3.2704973220825195, -0.3530915081501007, -1.2830206155776978, 0.8375943899154663, -1.4946047067642212, 1.1508207321166992, -0.7349016070365906, 1.7600445747375488, -1.5287073850631714, 1.7204879522323608, -1.7665055990219116, 0.45890867710113525, 0.9214630722999573, 0.08045107126235962, -0.9065821170806885, -1.6773875951766968, 1.8832621574401855, -0.010287880897521973, 0.2559286952018738, 0.2990947961807251, 1.3376641273498535, -0.408763587474823, 0.8748217225074768, -1.4238848686218262, 1.8475496768951416, 2.216249704360962, 2.734985828399658, 2.4666688442230225, 1.538811206817627, 0.5912560224533081, 0.6029458045959473, -2.0175609588623047, -1.000870704650879, 0.6014447212219238, 1.0269536972045898, -1.664059042930603, -2.2622556686401367, -0.0743054747581482, 3.548175573348999, 4.079979419708252, -1.3047595024108887, -1.812009334564209, 0.6063811779022217, 0.07985889911651611, -0.18506839871406555, -0.04920172691345215, -1.3655905723571777, 2.18887996673584, -0.6026527285575867, 1.001190423965454, -1.787057638168335, 1.033299446105957, 2.130714178085327, 0.4797697067260742, -0.36138594150543213, -1.4490008354187012, 0.6114537715911865, -2.5101876258850098, 3.1650447845458984, 0.7418447136878967, 1.1825833320617676, 1.833390235900879, 0.3052182197570801, 2.04773211479187, -0.35815924406051636, 0.19091039896011353, -0.4065231382846832, 0.0867733359336853, -0.6368984580039978, -1.0622825622558594, -0.09153985977172852, -0.586464524269104, -1.0849024057388306, -2.4294352531433105, -1.4314416646957397, 0.5330856442451477, 1.271674394607544, -0.9156133532524109, 1.295226812362671, 0.2571902573108673, -0.6185532808303833, 0.4264812767505646, 1.0904693603515625, -1.0289673805236816, 0.6336876749992371, 0.44125092029571533, -0.008560240268707275, 0.4313717186450958, 0.05453743040561676, 2.649216651916504, 1.1565179824829102, -0.895612359046936, -1.6616179943084717, -0.2968705892562866, -0.9686238765716553, 0.26978564262390137, 0.7328071594238281, -0.929706335067749, -2.006059169769287, 1.439513087272644, -0.9414142966270447, -2.642199993133545, -1.095983862876892, 1.698693871498108, -0.3301725387573242, -0.7779440879821777, 1.8372931480407715, -0.3740381598472595, -0.07479023933410645, 1.3804924488067627, 0.08389238268136978, 1.7283037900924683, -0.3629639446735382, 0.6874120831489563, 2.163259506225586, -1.5410631895065308, 0.2374085783958435, 2.904360771179199, -1.0475387573242188, 1.8374435901641846, -0.95595383644104, 1.618098258972168, -0.05145101249217987, -1.2184442281723022, -0.5389122366905212, -0.9591071605682373, 0.21995961666107178, -2.5905849933624268, -0.7885171175003052, 0.022586047649383545, 0.09473103284835815, 1.245311975479126, -1.0799317359924316, 2.0789973735809326, 0.6837800145149231, 3.543576955795288, -1.5163733959197998, 0.08328196406364441, 0.4907636046409607, 1.9910081624984741, 0.32248854637145996, -0.32456451654434204, -0.11015602946281433, 0.024371623992919922, -1.627273440361023, -3.012915849685669, 2.462279796600342, 2.812758684158325, -1.8917299509048462, -3.7345190048217773, -1.3615819215774536, -0.3277469277381897, -1.0633864402770996, 1.7440698146820068, -1.0675256252288818, 1.9080449342727661, -1.6661021709442139, 2.033276319503784, -2.149355888366699, 2.4820661544799805, 0.2861630916595459, 1.0159807205200195, 1.5025665760040283, -2.1066877841949463, -0.241441011428833, 0.9360694885253906, 1.373678207397461, -2.98012113571167, 0.6496139764785767, 0.9668852090835571, -1.2777680158615112, -1.500988483428955, 0.9668440818786621, -1.0860012769699097, 0.16372644901275635, -0.3060338795185089, 0.17729617655277252, 0.22891902923583984, -0.3285320997238159, 1.6300426721572876, -2.3107399940490723, -0.6115391254425049, 0.2530321478843689, -0.7763177156448364, 1.2110031843185425, 1.1852312088012695, -1.0189868211746216, 2.0591065883636475, 0.41982096433639526, -0.8440462350845337, 0.6969504952430725, 0.5466505289077759, 1.8104435205459595, -1.763782024383545, 0.387407124042511, -1.5024410486221313, 1.0865150690078735, 0.3083018958568573, -0.24255630373954773, 0.12916429340839386, -0.5359413027763367, 0.9014382362365723, -1.4682475328445435, 2.776731491088867, 0.44412726163864136, -2.003046989440918, -0.0010133087635040283, -2.548572540283203, 0.989267885684967, 1.2243638038635254, -1.0624195337295532, 1.7100460529327393, 0.09621474146842957, -0.17368638515472412, 1.0606459379196167, 0.40390706062316895, -0.7561631798744202, 0.19451561570167542, -0.5620980262756348, -1.4494304656982422, -0.43631601333618164, -1.4419472217559814, -1.24284029006958, 1.5586974620819092, 0.22282230854034424, 0.47902077436447144, -0.3816717863082886, -0.501034140586853, 1.1747545003890991, -2.0670435428619385, 0.9483276605606079, 1.634915828704834, -0.23760515451431274, 1.6364470720291138, -1.2272695302963257, 2.724661111831665, 0.264957070350647, -0.636885404586792, 0.8100923895835876, -1.3213599920272827, 0.2914687395095825, -1.2667485475540161, 3.017594814300537, -0.3428800702095032, -0.8443261384963989, 0.35563939809799194, -1.6165283918380737, -0.1679450273513794, -1.8993796110153198, -2.4179675579071045, -1.507873296737671, -2.51371431350708, 0.3377172350883484, -0.44069766998291016, -1.6009316444396973, 2.2261929512023926, -1.1341770887374878, 0.761340856552124, -3.2976467609405518, -2.1384031772613525, -1.2234158515930176, -0.10580992698669434, -0.13145999610424042, -2.621596336364746, -0.042024776339530945, -0.34305816888809204, 1.5139269828796387, 1.7658953666687012, 2.759598731994629, -0.7985515594482422, -0.4213685691356659, -1.3775439262390137, -0.8140573501586914, 1.4139158725738525, 2.0400748252868652, -0.8297881484031677, 1.7990837097167969, 0.07189901173114777, -1.7741575241088867, -2.024752616882324, -0.8769755363464355, -0.5738434195518494, -0.45692771673202515, 0.7412723302841187, -2.1870553493499756, 1.4306563138961792, -1.1255102157592773, 1.159440517425537, 1.0480186939239502, -0.6868488788604736, 1.9703933000564575, 1.4747916460037231, -1.1092102527618408, 0.2944709062576294, -0.6832805871963501, 0.3261876702308655, 1.4023152589797974, 3.5667829513549805, 1.352623701095581, 0.5383630394935608, 1.1099661588668823, 0.30886274576187134, 0.5123100280761719, -0.16093766689300537, -0.38108450174331665, -2.634688377380371, 0.44840604066848755, 0.5173484683036804, 1.0643285512924194, -0.8791201114654541, -1.8611695766448975, -0.6365128755569458, -1.5067894458770752, 0.1661926507949829, 2.096979856491089, 0.2808770537376404, -0.3647679388523102, 2.080784797668457, -0.3125268816947937, -0.023364488035440445, 0.4626060724258423, 0.05700060725212097, -0.4546169936656952, 0.17377185821533203, -3.5474839210510254, -1.7319655418395996, 1.7945060729980469, -2.5773558616638184, -2.436846971511841, -0.055665016174316406, -2.6533889770507812, -1.670191764831543, -1.123960018157959, -0.16027772426605225, 0.07361742854118347, 1.4439352750778198, -0.5951279401779175, 2.604644775390625, 2.3985190391540527, -0.314738392829895, -0.5947800874710083, -2.909456253051758, -1.8035483360290527, -0.2697916626930237, -0.37236905097961426, 1.925191044807434, -0.3685436546802521, 1.7364627122879028, 0.6512855291366577, -0.5720236301422119, -0.12434381246566772, 2.593794345855713, -1.9273141622543335, -0.4787856340408325, -0.7006595730781555, -0.21441468596458435, -1.2904980182647705, 0.37174248695373535, -0.26358675956726074, 0.8845168352127075, 0.9862618446350098, -0.09769350290298462, 0.1784445196390152, -0.01513904333114624, -1.7571451663970947, -0.6623890995979309, 1.9426789283752441, -0.5326178073883057, 0.5100204348564148, -0.8982322812080383, 1.2555510997772217, 0.762427568435669, -1.4754412174224854, 0.7459356188774109, -1.5868463516235352, 1.3862580060958862, -0.41045212745666504, 0.5956310033798218, 1.5855522155761719, -1.2978124618530273, -0.3974910378456116, 3.5947322845458984, -0.8375864028930664, 0.4685755968093872, -0.6022011637687683, 0.7409049272537231, 2.1737608909606934, 1.697579026222229, -1.0106538534164429, -0.6943018436431885, -0.009600698947906494, -2.4705092906951904, 0.4579731225967407, 1.028879165649414, -0.009033918380737305, -0.8842270970344543, -0.09609061479568481, 1.2496856451034546, 1.0327668190002441, -1.632961630821228, -0.36393439769744873, -1.685495138168335, -0.14129222929477692, 0.6375583410263062, -0.36955201625823975, 0.8360542058944702, -1.3389477729797363, -0.38569778203964233, -1.2449618577957153, 0.18348640203475952, 2.6865272521972656, -0.6215563416481018, -0.8804373741149902, 1.7906947135925293, -0.4759916067123413, 1.7410763502120972, 0.3242364525794983, -1.8782817125320435, 0.8091449737548828, 1.9362398386001587, 0.7868950366973877, 1.1463251113891602, -1.7624297142028809, 1.3559174537658691, 0.2418823093175888, 0.8788846135139465, -1.5872224569320679, 1.7332072257995605, 2.49639630317688, -0.07947802543640137, 1.3223470449447632, -2.6208102703094482, -0.6696130037307739, -0.5814913511276245, -0.044652700424194336, 2.5409135818481445, -0.08758187294006348, -0.290578156709671, 0.5233127474784851, -0.01765916869044304, 2.632986545562744, -2.041923761367798, 0.11575126647949219, -0.04343092441558838, -1.1762067079544067, -0.4270288944244385, -1.6987125873565674, -0.3592870831489563, -0.22824516892433167, 2.7098567485809326, -0.8911476731300354, 0.29477769136428833, -0.9923834800720215, 1.1185860633850098, 0.5930018424987793, -2.5507729053497314, -0.8569096922874451, 1.4506702423095703, -0.4163573086261749, -0.40017756819725037, 1.1556462049484253, 0.024469196796417236, 2.1934690475463867, 1.069845199584961, 0.5296040773391724, 2.417433738708496, 1.281318187713623, -0.2650666832923889, 0.019424915313720703, -1.9129136800765991, 0.745741069316864, -0.2509880065917969, 0.38800060749053955, -1.3057149648666382, -0.21017026901245117, -1.0624487400054932, -0.4013352394104004, 0.9175683856010437, -1.268096923828125, 0.5059295892715454, -1.5843693017959595, 1.4737324714660645, 0.35882753133773804, 0.625712513923645, -0.5813875794410706, 1.9400967359542847, 2.1815168857574463, 0.39541083574295044, -0.9803966879844666, 1.5433485507965088, 3.348578453063965, -1.0690419673919678, -1.795102834701538, -0.10802432894706726, 1.1945387125015259, -1.9457733631134033, 1.0471858978271484, 0.07421839237213135, 1.2896928787231445, -1.4207265377044678, -1.5151360034942627, 1.6847655773162842, 0.42533087730407715, -0.4954884350299835, 0.5906630158424377, 0.5867462158203125, -1.575066328048706, -1.0097341537475586, -0.5488982200622559, -0.011221051216125488, 0.9605309367179871, -0.5920168161392212, 1.970595359802246, -0.39691677689552307, -2.924694776535034, -1.5630362033843994, -0.9893776178359985, 1.4392062425613403, 0.11612200736999512, 0.29754728078842163, -1.970974087715149, -1.172745943069458, 0.44524431228637695, 1.167539358139038, -1.362295389175415, 0.2502962350845337, 0.2921215891838074, -0.22301144897937775, -3.7467153072357178, 1.0488826036453247, -0.24956202507019043, -1.325160264968872, 0.9602916836738586, -0.13953396677970886, -1.474772334098816, -1.8848106861114502, 1.0250860452651978, -0.9173098206520081, -1.348454236984253, 1.5711464881896973, 1.2004716396331787, -0.38977891206741333, 1.7876601219177246, -2.1889545917510986, 1.141139268875122, -1.6064029932022095, -2.8991880416870117, 1.9716603755950928, 0.18754971027374268, -0.157741978764534, -1.0498809814453125, 0.7751996517181396, -0.089088574051857, 1.859040379524231, -1.159477949142456, 0.011484608054161072, -0.6792862415313721, -1.139979362487793, -4.161718368530273, 2.119762420654297, 0.08366605639457703, -1.2883669137954712, 3.3018507957458496, 1.4610607624053955, -0.06965148448944092, 0.6154214143753052, -0.8971154689788818, -0.2308768481016159, -1.480289101600647, -1.0891319513320923, -1.3135408163070679, 0.7314283847808838, -0.6175152063369751, -1.9487056732177734, 0.062276363372802734, -0.9983046054840088, 0.7493407726287842, -1.3421356678009033, -2.684035301208496, 0.8181684017181396, 1.1313380002975464, -1.026313066482544, -2.240145444869995, -1.7722259759902954, -0.6349443793296814, -2.133223295211792, -1.6632306575775146, -0.26342904567718506, 3.1021008491516113, 0.63953697681427, -0.3986542224884033, 1.377909779548645, 0.9017031788825989, -1.5778158903121948, -1.3635609149932861, -1.3075346946716309, -1.0882344245910645, -1.5046409368515015, -0.15666735172271729, 0.7009149789810181, 1.1253470182418823, -0.6808793544769287, 0.8553680181503296, -2.2951390743255615, 2.2538914680480957, 0.15359270572662354, 0.41150742769241333, -1.1607913970947266, 1.0718227624893188, 1.944641351699829, 2.4387242794036865, 0.920427680015564, -1.7719218730926514, -0.23160123825073242, 0.6336497664451599, -2.316159725189209, -1.358408808708191, -0.44271671772003174, 1.5462572574615479, -0.2503716051578522, 1.2090330123901367, -3.0776896476745605, -1.0240167379379272, -0.14606690406799316, -0.6432570219039917, -2.683826446533203, 2.015498638153076, -0.8770169615745544, -0.31959235668182373, -0.8754193782806396, 0.562515914440155, -0.528081476688385, -0.6370324492454529, 1.7583848237991333, 0.5850292444229126, -0.4060300290584564, 1.5547716617584229, -0.2582794427871704, 0.13862258195877075, 0.16674265265464783, -0.16164937615394592, -1.4798619747161865, -2.498088836669922, 0.2918586730957031, -1.209090232849121, 1.1117192506790161, 0.3952798843383789, -1.388195276260376, 0.06800168752670288, -0.7211326360702515, 0.7126249670982361, 0.22757062315940857, -1.033710241317749, -0.062049537897109985, -1.5961706638336182, 2.509795904159546, -2.2892560958862305, 1.154565691947937, 0.36830148100852966, 0.4737710952758789, -1.6600931882858276, 0.04990821331739426, 1.0762237310409546, 1.3239691257476807, -0.9681172370910645, 0.4085513949394226, -0.7892358303070068, 0.6076816320419312, -0.36149919033050537, 0.14511877298355103, -1.394656777381897, -0.3459310233592987, -1.1696115732192993, 0.7303996086120605, -0.3835369050502777, -0.03505454957485199, -1.1877259016036987, -0.7371327877044678, 0.18334224820137024, 1.819024920463562, 0.3134647607803345, -0.3101249039173126, 1.1817982196807861, -0.06675674021244049, 0.8572183847427368, 0.8613176345825195, 1.380218267440796, 1.1357905864715576, 1.80283522605896, 1.6041700839996338, 0.33474794030189514, -0.5492315292358398, 2.070645332336426, 0.03626972436904907, 0.9781920909881592, 0.41673949360847473, 1.42106294631958, 1.375961184501648, -1.351616382598877, 0.2371947169303894, -0.5932915806770325, 1.331528663635254, -1.400434970855713, -0.1269935667514801, 0.8653291463851929, -0.2604888379573822, -0.3320254683494568, -1.8965721130371094, -0.5054875612258911, -0.8954933285713196, 0.7454668879508972, -0.4289005994796753, -1.8991374969482422, 1.9422345161437988, -0.6597585678100586, -0.4678823947906494, 0.7235552072525024, -0.5316779613494873, -2.043863534927368, -0.8339661359786987, 0.5366826057434082, 0.7306172251701355, -1.310356616973877, -0.9531815052032471, -0.2622305452823639, 1.6564955711364746, 0.1963188350200653, -0.7016317248344421, -0.0910978838801384, -1.0514826774597168, 1.471308708190918, -0.12923353910446167, 0.8106819987297058, 0.7432938814163208, -2.091982126235962, 0.33390694856643677, -0.7802914381027222, 0.5104840993881226, -1.9762651920318604, 1.4878501892089844, -1.8146378993988037, 1.5590336322784424, -2.1995160579681396, 0.036654889583587646, -0.383951872587204, 0.3699328899383545, -0.9517647624015808, -2.2281734943389893, 1.6004385948181152, -0.5599418878555298, 1.5053497552871704, 0.026140868663787842, 0.6545995473861694, 1.6452159881591797, 0.5622259378433228, -0.8700228929519653, 0.8146823644638062, 1.7973289489746094, -0.15943723917007446, -0.9376559257507324, -0.7766191959381104, -2.0756218433380127, 1.3205969333648682, 2.7055177688598633, 0.5509689450263977, 0.6594272255897522, -1.53864586353302, -1.591394305229187, -1.5443778038024902, 0.6259219646453857, 0.9133905172348022, -0.6307547092437744, 1.4189512729644775, -1.9234004020690918, 1.9286236763000488, -1.68241286277771, -2.0437984466552734, 0.5025391578674316, 0.7372663021087646, 1.2417783737182617, -1.7663614749908447, 1.1061173677444458, -3.64467191696167, 0.8035754561424255, -1.5054587125778198, -2.1210641860961914, -0.7781091928482056, -2.8211700916290283, -0.05383598804473877, 1.1751528978347778, -2.1481051445007324, -1.220333456993103, -0.33057257533073425, 0.15670108795166016, 2.5424435138702393, 2.5624630451202393, -1.3219578266143799, 2.9637370109558105, -0.8100182414054871, -0.40212640166282654, 1.279005765914917, -1.811282753944397, 1.569169282913208, -0.6065983772277832, -0.7407289147377014, -0.4403235614299774, 0.21689559519290924, 0.3468492031097412, -0.472153902053833, 0.18294668197631836, -0.006750702857971191, 0.42260175943374634, 1.0632591247558594, -2.3448262214660645, -1.1466330289840698, 0.4844393730163574, 0.7156186699867249, -0.8156483173370361, -0.3394438624382019, -0.8463834524154663, -0.3126181364059448, 1.488145112991333, 0.2021082639694214, 1.3118410110473633, -2.1553330421447754, -1.5980989933013916, -0.3329164981842041, -2.7605814933776855, -1.1670196056365967, 1.2638601064682007, -2.049172878265381, 0.30173492431640625, 0.19285118579864502, 0.5382977724075317, -0.07076463848352432, -0.7535896301269531, 0.19094395637512207, -2.001788854598999, -1.4800139665603638, 1.0454447269439697, -3.044046401977539, 1.9602761268615723, 2.422635555267334, -0.7924099564552307, -0.4929601550102234, 1.7477023601531982, 0.7399061918258667, 1.5308371782302856, -1.253641963005066, 0.43317651748657227, -0.30349186062812805, 2.0772972106933594, -0.7157080173492432, 2.3339154720306396, -0.8966953158378601, 1.8620935678482056, 0.32251420617103577, 0.22253239154815674, 0.17978310585021973, -1.1849616765975952, 1.2234936952590942, -1.4587334394454956, -0.9659740924835205, 1.7025141716003418, 1.5024518966674805, -0.33327168226242065, -0.9715516567230225, 0.4906124174594879, -1.1078660488128662, -2.3151979446411133, -0.9283374547958374, 0.35525888204574585, -0.059765227138996124, 2.769695281982422, 0.7094285488128662, 1.351898431777954, 0.19089758396148682, -1.5198951959609985, 0.29782143235206604, 1.2212512493133545, 0.9061155319213867, 1.2493396997451782, -1.2198755741119385, 1.7972050905227661, 1.9829171895980835, -0.5838854312896729, 1.0242538452148438, -0.272548109292984, -0.23355650901794434, -0.0579066276550293, -1.3526136875152588, -0.5816044807434082, 0.10233758389949799, -0.09431261569261551, -0.6352630853652954, 0.8785056471824646, -3.923147678375244, 0.6110820770263672, 0.09571777284145355, -0.33608001470565796, -0.7980968952178955, 0.6892831325531006, 1.6761311292648315, 0.1412573754787445, -0.8465057611465454, 1.6531230211257935, 1.9573147296905518, -0.19656550884246826, -1.1300938129425049, -3.0857481956481934, 0.9916399717330933, -2.6905465126037598, 1.1523462533950806, 1.1993619203567505, 1.5688647031784058, -0.00958821177482605, -1.0680606365203857, -0.7902353405952454, 1.965867280960083, 0.3100719749927521, -0.07432913780212402, 0.9750035405158997, -1.224184513092041, 1.3791992664337158, 0.23414760828018188, -0.5809793472290039, 1.578203558921814, 1.7374520301818848, 0.697292685508728, -1.8845577239990234, -2.9591586589813232, 3.3049874305725098, -0.7081934213638306, 0.9546723961830139, 0.19426319003105164, -0.8650273680686951, -0.5849234461784363, 0.763338565826416, -0.5842990875244141, -0.5582281351089478, -1.4937212467193604, -0.649535059928894, 0.3508369028568268, 1.6677145957946777, -1.8697054386138916, 1.5474823713302612, 1.1382001638412476, -2.2906908988952637, 1.8770787715911865, -0.2970198392868042, 2.5287790298461914, -0.48982691764831543, -2.1465201377868652, -0.293562650680542, 1.1086671352386475, 1.768903136253357, 0.2558295726776123, -2.026515007019043, 0.4851127862930298, 1.6205394268035889, 1.3612277507781982, 2.5802040100097656, -0.4942571222782135, 2.6693663597106934, 2.1102776527404785, 1.853156566619873, 1.1431437730789185, 0.5275923013687134, 1.5095953941345215, 0.4844195246696472, -0.23489752411842346, 2.2009170055389404, 0.7042837142944336, 0.22262881696224213, -0.25024518370628357, 0.16018837690353394, -0.9002071022987366, 0.5459080934524536, -2.4052844047546387, 0.16614778339862823, 1.2687188386917114, -0.1826547384262085, -0.06838271021842957, -1.5320264101028442, -1.8567463159561157, 0.41427814960479736, -0.9238297939300537, 0.2706443667411804, 0.6423234343528748, -0.6487147808074951, 0.6177524328231812, -0.9836833477020264, 1.676023244857788, 0.843061625957489, 0.4325624704360962, 3.6142516136169434, 0.9825608134269714, -1.6326614618301392, 2.546645164489746, 3.420849561691284, 0.8133659362792969, -1.1676146984100342, 0.3424523174762726, -1.5677855014801025, -0.24231553077697754, -1.764018177986145, -1.8142237663269043, 2.793482780456543, 0.1373036652803421, 0.38308051228523254, 3.875256061553955, -1.9092463254928589, -3.3623979091644287, -0.25208669900894165, 0.17424586415290833, -1.8125407695770264, -2.001206874847412, -1.5095555782318115, -0.5139755010604858, 1.6845731735229492, 0.8330084085464478, -2.529655933380127, -0.2830057144165039, 1.2657487392425537, -1.4527368545532227, 0.3520257771015167, -2.115400791168213, -0.21883821487426758, -0.1769130825996399, -1.8718359470367432, -0.8704165816307068, -0.6016007661819458, 0.27367687225341797, 0.5347316265106201, -3.357009172439575, 0.33958786725997925, 0.2581067681312561, -1.8582621812820435, -1.0398122072219849, -2.5850605964660645, 0.5288187265396118, -2.8463854789733887, -1.0781532526016235, 1.1594538688659668, -0.7628285884857178, 0.8392206430435181, 2.8954033851623535, 0.21137623488903046, 0.26201772689819336, 1.7036539316177368, -0.5204682350158691, 3.404676914215088, 1.7113125324249268, -2.4004242420196533, 1.2756400108337402, -0.9891940951347351, -1.348068356513977, 0.27186834812164307, -0.38884806632995605, -1.7955684661865234, -0.0643768310546875, -0.6133394837379456, 0.16014361381530762, -0.5871409177780151, 0.3452356159687042, 0.7350553870201111, -0.9245638847351074, -2.3860890865325928, -1.5913589000701904, -0.23143243789672852, -1.7433371543884277, 0.11285138130187988, 0.1730746030807495, 0.48477715253829956, 0.5664730668067932, -1.03520667552948, 0.16108977794647217, 2.9122684001922607, 0.061488330364227295, 1.8322440385818481, 2.2250492572784424, 1.2011065483093262, 2.8807568550109863, 0.6071264743804932, -2.711169481277466, 0.7216536998748779, -0.6187836527824402, -1.165802001953125, -1.5134166479110718, 0.8997561931610107, 0.5514399409294128, 0.07102948427200317, 2.0389280319213867, 1.7764816284179688, -1.8117846250534058, 0.4157836437225342, -1.2507208585739136, 0.991146445274353, -1.4360229969024658, 1.2558190822601318, 0.1791904866695404, -0.3250516653060913, -2.500007390975952, -0.8125006556510925, -0.328074187040329, 0.3717920780181885, 0.09100422263145447, -1.4347901344299316, -0.3973671495914459, 0.7196595668792725, 0.18977969884872437, 0.6096315979957581, -0.2892334759235382, 0.9628558158874512, 0.9850950837135315, 1.669640302658081, 0.4143296480178833, 0.28633803129196167, 0.22818689048290253, 0.5963140726089478, -0.765394926071167, -0.5899008512496948, -0.06890714168548584, -0.25668448209762573, 1.2176291942596436, 0.3780493140220642, -0.45119708776474, -0.7045245170593262, 2.6669981479644775, -0.2728615999221802, 1.1548177003860474, -1.9789366722106934, -1.7766995429992676, 0.7250211834907532, -1.3724387884140015, 1.175716519355774, -1.5755304098129272, 2.868553400039673, 0.5905945301055908, -0.9063369035720825, -0.11238808929920197, -1.250929355621338, -0.4407016932964325, -0.25229179859161377, 0.15393787622451782, 0.9108794331550598, 0.3262651562690735, 2.0119025707244873, 0.10119549930095673, 1.2792918682098389, -1.1992924213409424, -0.26463693380355835, 1.944059133529663, -0.714451789855957, -1.6233793497085571, 2.4715828895568848, -2.0881102085113525, 0.39242327213287354, -0.6977127194404602, 2.477128505706787, 2.370852470397949, -1.526036262512207, 0.9447216987609863, 2.0876572132110596, -0.8462607860565186, -0.3892925977706909, 0.73280268907547, -0.7069074511528015, 1.759134292602539, 0.0010859519243240356, 0.11591744422912598, -1.0017313957214355, -0.6222424507141113, 0.5977035760879517, -0.8749251961708069, -0.06506669521331787, -0.9660521149635315, -1.4431971311569214, -0.8581429719924927, -2.3198089599609375, 1.7450683116912842, -0.8755525350570679, -0.40414756536483765, 1.2177265882492065, -2.809551477432251, 1.7925348281860352, -0.14159846305847168, 1.2785313129425049, -0.6154841780662537, 1.1824734210968018, 0.048602521419525146, -0.19302290678024292, 1.415039300918579, -0.5351089835166931, -2.5527443885803223, -0.25306445360183716, 0.5991873741149902, 2.8877642154693604, 0.3515812158584595, 1.3285105228424072, 1.399693489074707, 1.0012400150299072, 1.7316632270812988, -0.34080106019973755, 0.5316280126571655, 0.2938167452812195, -0.6504780054092407, 0.7290077209472656, -0.5021626949310303, 0.14962024986743927, -0.9633437991142273, 0.7101196646690369, -0.3389025330543518, -0.5550724864006042, -2.2842609882354736, 1.1324918270111084, 1.4066786766052246, -2.397477388381958, -1.4380571842193604, -2.304600238800049, -1.8885940313339233, -0.19112786650657654, 0.7993417978286743, -0.6539042592048645, -0.22272564470767975, 1.744564414024353, -1.0507481098175049, 0.05938452482223511, 0.15729796886444092, 1.437044620513916, -1.0724828243255615, 0.20496302843093872, 1.0905245542526245, -0.5597864985466003, -0.4870533347129822, -1.7650965452194214, -0.7492372989654541, -1.483642816543579, -1.3537256717681885, -1.068472146987915, 1.070015788078308, -1.4908015727996826, 0.17428556084632874, -0.5058444738388062, 1.178483486175537, -0.47494715452194214, 1.5216068029403687, -0.5686174631118774, 0.2773004174232483, -0.13629654049873352, -0.8713399171829224, -0.4008476138114929, -0.4042642414569855, -0.8238304853439331, -0.8745041489601135, -1.2100921869277954, -4.522714614868164, 1.5262672901153564, -4.07733154296875, -1.553086519241333, 0.6812202334403992, 1.3238537311553955, 0.7281896471977234, 0.37992751598358154, 0.18902523815631866, 1.3043594360351562, 1.6886544227600098, 1.207532286643982, 0.9588228464126587, 1.9438005685806274, -0.3391706645488739, 0.49890708923339844, 3.5979347229003906, -2.7086374759674072, -2.492715835571289, 0.995701014995575, 1.671303391456604, 0.8294191360473633, -2.4801836013793945, 0.6362686157226562, 2.5427730083465576, -2.0228230953216553, -0.39262843132019043, -0.15183889865875244, -0.6657896637916565, -0.3717554807662964, -1.3460490703582764, -0.5595436096191406, -2.629826307296753, -0.08284947276115417, 0.30909299850463867, -1.4053467512130737, -1.9528264999389648, 2.629528045654297, 1.526794195175171, -2.01053524017334, -2.950103282928467, 0.5399360060691833, 2.7310333251953125, -0.3174242377281189, 0.4304310083389282, -0.16498389840126038, 0.7990700602531433, 1.4245727062225342, -0.15488368272781372, 1.9849886894226074, 2.0011653900146484, -0.2790321111679077, 0.7698419690132141, -1.2586448192596436, 1.0087522268295288, -0.27473992109298706, -1.0212758779525757, -0.5979725122451782, -0.6571622490882874, -0.6017853021621704, -1.2396507263183594, -0.5638657808303833, -0.7776747941970825, 0.8174241781234741, 0.11731189489364624, -1.52229905128479, -0.11123228073120117, 1.8674683570861816, 0.8162505030632019, -1.8392093181610107, -0.0004839301109313965, 0.8674970269203186, 0.1597587913274765, -0.017115026712417603, 1.816450595855713, 0.27529698610305786, -0.11408044397830963, -0.25298261642456055, -2.0495285987854004, -0.1223139762878418, 2.341360330581665, 0.8117444515228271, -0.2186029702425003, -2.8185153007507324, -2.008018732070923, -0.06481152772903442, 0.15370777249336243, 0.09688669443130493, -2.5103511810302734, 0.27259117364883423, 1.7682729959487915, -1.4306060075759888, 0.5616384148597717, 0.638775110244751, 0.007597088813781738, 0.3084239363670349, -0.34508490562438965, 0.7265138626098633, -0.6540985107421875, 0.13965141773223877, 0.18799424171447754, -1.7291488647460938, -0.11561518907546997, -1.2850892543792725, -3.08557391166687, -0.32273203134536743, 0.18365216255187988, 0.4532480835914612, 2.1923434734344482, 2.2574665546417236, -0.32095080614089966, 0.19505193829536438, 0.3437008857727051, 0.9533260464668274, 1.505060076713562, 2.471909999847412, -2.5756430625915527, -0.1918628215789795, -2.7516167163848877, 0.9094278216362, 0.4543948769569397, 0.7619714736938477, 0.6734547019004822, 1.2854127883911133, -1.0709484815597534, -0.8289569020271301, 2.252933979034424, 2.7359368801116943, 1.1761906147003174, 1.1359819173812866, 0.988212525844574, -0.41927796602249146, 2.1806626319885254, 0.9494874477386475, -2.9039955139160156, -0.9285435080528259, -2.4002416133880615, 0.39265328645706177, -1.1045056581497192, 2.788668155670166, 1.25303053855896, 1.3894116878509521, -1.7310417890548706, -1.392343282699585, 1.3107539415359497, 1.8748016357421875, 2.146141529083252, 0.5727252960205078, 0.3378552496433258, 0.860567033290863, -0.13165679574012756, -0.9576209187507629, 1.2933050394058228, 2.5370893478393555, 0.7914931774139404, 1.3934142589569092, -0.4152827262878418, 2.908952236175537, -1.4370819330215454, 0.13682174682617188, 0.3220716118812561, -1.4430999755859375, -1.8598544597625732, 1.5051974058151245, 0.3262452483177185, -2.303683042526245, 1.20299232006073, 0.17665442824363708, 0.9877612590789795, 2.6093521118164062, 0.8874454498291016, 0.6917328834533691, 0.3808137774467468, 0.1517103910446167, 1.2666937112808228, -0.5711506605148315, -0.32308346033096313, 1.5190147161483765, -0.44903093576431274, -1.3754470348358154, -0.196445494890213, 2.067810535430908, 0.22813159227371216, 1.057147741317749, 2.966808557510376, -4.657325267791748, -0.456512451171875, -0.3302895426750183, -2.6957902908325195, 0.16594642400741577, -0.5517334938049316, 1.015451192855835, -0.6846103668212891, 0.7219682931900024, 1.025807499885559, -0.5718680620193481, -1.2405495643615723, 0.032095909118652344, 0.534637451171875, 0.59108567237854, 0.35537219047546387, 1.636521816253662, 1.9229878187179565, 0.8785220384597778, -0.26697838306427, -2.654963254928589, -1.5545353889465332, 0.20997799932956696, 1.412998080253601, 0.3255001902580261, 0.870664119720459, 0.6662471294403076, -2.072023391723633, -1.1427003145217896, 0.4244757890701294, -2.3233418464660645, -0.11665201187133789, -1.1570652723312378, -0.21466657519340515, -0.7403440475463867, 0.3566223680973053, 1.0809752941131592, 0.5048069953918457, 0.8996813297271729, -1.7019214630126953, 0.2348862886428833, -1.327245831489563, -0.6268847584724426, -0.6953692436218262, -0.3180153965950012, 0.5518876314163208, 0.30350059270858765, 1.1823298931121826, -1.2486134767532349, 0.25741279125213623, 3.1226143836975098, -0.17644403874874115, 0.2756154537200928, 0.528865396976471, -1.1713334321975708, -2.164055109024048, 2.9742069244384766, -0.19723428785800934, -0.4541890025138855, -4.331733703613281, -1.003092646598816, -1.0099161863327026, -0.4219631552696228, 0.5995362997055054, 1.173779010772705, -0.5395375490188599, 1.399332046508789, -0.23863959312438965, -0.884486734867096, -0.17875969409942627, 1.4425983428955078, -0.28527969121932983, 0.07035356760025024, -1.1901092529296875, -0.5476493835449219, -1.8427428007125854, 1.6274592876434326, 0.31391721963882446, -0.5683879852294922, 0.7228041887283325, -0.6120222210884094, 1.207955002784729, 2.5343165397644043, -0.21509024500846863, 0.9019979238510132, -0.37718111276626587, -0.8281978368759155, 1.2860424518585205, 0.11972349882125854, 0.4766397476196289, 0.29509711265563965, 0.6547759771347046, -2.249149799346924, 0.30120015144348145, -1.3115463256835938, 3.44937801361084, -0.8551973104476929, 0.803954005241394, -1.215236783027649, 1.2580156326293945, 3.249393939971924, -3.040501117706299, 0.6834375262260437, 0.050383247435092926, 0.9880242943763733, -3.1466197967529297, -1.3242005109786987, -0.23332369327545166, 0.43372848629951477, -0.6541838049888611, -1.0428574085235596, 1.8252238035202026, -2.598202705383301, 0.5321211218833923, -2.703733444213867, 0.2298755645751953, -0.6744513511657715, -0.5928990840911865, -0.6737682819366455, -1.8754634857177734, -2.0637922286987305, 0.44702017307281494, 0.7711471319198608, 1.9563801288604736, 0.5112931728363037, -1.8763525485992432, -1.0739086866378784, 0.31828784942626953, -2.4199724197387695, 0.6189153790473938, -0.16836780309677124, 1.0369290113449097, 0.9883430004119873, -0.9618409872055054, 1.5193748474121094, -0.5235064625740051, -0.9408117532730103, -3.5639166831970215, -1.8231955766677856, -0.7061797380447388, -0.8487915992736816, 0.4181506037712097, -1.1868473291397095, -0.3367561101913452, -1.531949520111084, 1.364866018295288, -1.9358646869659424, -1.641107201576233, -3.6815834045410156, 1.1885275840759277, -0.7266415953636169, -0.37725818157196045, -0.8030702471733093, -1.1822490692138672, -1.2434242963790894, 0.3536626696586609, -0.7860671281814575, -0.5012282133102417, 0.7443385124206543, -0.47323229908943176, -0.21942174434661865, 2.7342543601989746, 0.9320023059844971, -0.7560787200927734, -1.5033358335494995, -0.6127937436103821, -0.9777064323425293, 0.9356271028518677, -2.3391895294189453, -1.4912363290786743, -1.446922779083252, -0.028729677200317383, -0.03755831718444824, -0.38117194175720215, 1.742201566696167, -1.131894588470459, -3.40633487701416, 0.22424130141735077, 0.1991332769393921, -2.679417610168457, 0.5928260087966919, 1.3737715482711792, 1.0877752304077148, -1.1444908380508423, -0.8219117522239685, 2.210566520690918, 1.7718820571899414, 0.16553831100463867, 0.23781636357307434, 0.8762348890304565, 1.9362640380859375, -0.7594027519226074, -0.001588478684425354, -0.988097608089447, -3.0099246501922607, -0.137595534324646, -0.0352218896150589, -0.004265144467353821, -1.6932427883148193, -1.725244164466858, 2.357863426208496, 0.9366506338119507, -0.11310964822769165, -2.512139320373535, 1.6874688863754272, -2.370290756225586, -1.8702316284179688, -0.831996738910675, -0.6268889307975769, 1.749385118484497, -0.7444320917129517, 0.8532294034957886, 0.8083480596542358, -0.74725741147995, -0.8808580040931702, -0.8053960800170898, 0.30653464794158936, -1.9021086692810059, -1.1596076488494873, -0.8579446077346802, -0.8372037410736084, 0.4173661172389984, -1.5919766426086426, -0.12442296743392944, 2.1193315982818604, 0.046141207218170166, 0.27805233001708984, -0.7486720681190491, -1.2937003374099731, -1.9035553932189941, 1.1278891563415527, -3.6979894638061523, 1.5427768230438232, -0.931753396987915, -1.5895452499389648, -0.06137663125991821, -1.021716594696045, -1.4106032848358154, -0.21608233451843262, 0.999272346496582, -0.15775883197784424, -0.3259913921356201, -0.42547592520713806, -0.4966302514076233, 0.5220794677734375, -0.4088992178440094, -0.23856404423713684, -0.49596554040908813, 1.489417314529419, 1.3711947202682495, -0.3844321072101593, 0.5328630208969116, 1.2881760597229004, -2.4738388061523438, -2.7461652755737305, -1.2389421463012695, 0.8925814032554626, 0.9399884939193726, -0.8375751376152039, 2.3237760066986084, 3.6420156955718994, -2.550957679748535, -1.8079640865325928, 2.2279696464538574, -1.844165325164795, 1.5516552925109863, -0.8144554495811462, 1.9890317916870117, -0.6116505265235901, 1.6660912036895752, 1.826491117477417, 1.172884225845337, 0.2692337930202484, 0.8365570902824402, 1.5960474014282227, 0.1098858118057251, -1.0934664011001587, 0.09365854412317276, 0.7690116167068481, -0.9972773194313049, -1.0956225395202637, -0.18770530819892883, 1.1521410942077637, 2.2047739028930664, 2.2012946605682373, 3.1841344833374023, -1.1616194248199463, 1.592597246170044, -1.654923439025879, 0.8600733876228333, 0.2841370403766632, -2.878706932067871, 1.6876699924468994, -0.5308299660682678, -1.4950158596038818, -0.18440686166286469, -1.2913867235183716, 0.24100512266159058, 0.6784290075302124, 2.123603105545044, 1.224721908569336, -0.06889951229095459, -0.30710047483444214, -1.438430666923523, -0.6904774308204651, 1.2644795179367065, 2.61372447013855, -1.1215193271636963, 0.4188289940357208, -0.027368009090423584, -0.28674525022506714, -1.337255835533142, 2.488473653793335, 0.16076211631298065, 0.3097267150878906, -0.8730809688568115, -0.07586166262626648, 2.0002822875976562, -2.019619941711426, 0.9971277713775635, 1.2175265550613403, -0.06006824970245361, 1.2899118661880493, -0.7778957486152649, 1.4175024032592773, -0.9057621955871582, 1.9983183145523071, 0.599596381187439, -1.5715488195419312, -2.476029396057129, 1.2889646291732788, -2.3925931453704834, 0.9517219662666321, 1.512272596359253, 0.008227169513702393, 0.10209128260612488, 0.296385794878006, -1.566150426864624, 4.005324363708496, 0.1395072042942047, 0.5941182971000671, -2.046478509902954, 0.18724220991134644, -1.3087224960327148, 0.904975414276123, -1.5425615310668945, -2.256937026977539, -2.2496280670166016, -0.6620896458625793, 0.35727831721305847, -0.15193083882331848, 0.33391714096069336, 0.7367490530014038, 1.1663072109222412, -0.404197096824646, -3.332760810852051, -2.5303144454956055, 3.6080875396728516, -0.3102577328681946, 0.5214470624923706, -2.847363233566284, 0.1337890326976776, -0.7786939740180969, -1.497022032737732, 0.7339663505554199, -0.4151158332824707, -2.1122968196868896, -0.1293235421180725, -0.6287092566490173, -0.3017120957374573, -0.5166476368904114, 0.19614124298095703, -1.7347348928451538, -0.6675004363059998, 1.9474740028381348, -0.31745314598083496, 0.9699062705039978, -0.7292819619178772, -0.08289216458797455, -1.1633461713790894, -0.26095712184906006, -0.7026358842849731, -0.22755765914916992, -2.4668562412261963, 1.50502610206604, 2.4767019748687744, -2.844744920730591, 1.1086387634277344, 2.357950210571289, 1.100424885749817, 0.8447408676147461, -1.056047797203064, 1.1476943492889404, 0.28071698546409607, 0.8902021646499634, -0.18469077348709106, 1.83772611618042, 0.05999124050140381, 0.7864395380020142, -0.5921568274497986, 0.2254042625427246, -1.7231513261795044, -0.49338340759277344, -1.2470868825912476, 1.6734830141067505, -0.8296108245849609, 0.6800057888031006, -2.466169834136963, 1.623772144317627, 0.5919272899627686, 2.005519151687622, -1.1445287466049194, -0.8429210782051086, -0.46593326330184937, 2.0319671630859375, -1.7458724975585938, -0.2299720048904419, 1.0643121004104614, -3.5624189376831055, -0.7386406660079956, 2.5417497158050537, -1.9469411373138428, -1.0792111158370972, 1.3084721565246582, -0.33326172828674316, 0.447898268699646, 0.25135889649391174, 0.10755965113639832, 1.1723108291625977, 3.33661150932312, 0.232954204082489, 0.664540708065033, 0.9951023459434509, -0.4988543391227722, 0.09857439994812012, 2.1382150650024414, -0.7424461841583252, -0.8050888180732727, -0.22029805183410645, -0.130639910697937, -3.2111244201660156, -0.21429948508739471, -0.16241493821144104, 1.6570932865142822, -0.4623299241065979, 1.4090731143951416, -0.42193859815597534, -1.6658223867416382, 1.1600656509399414, -0.16946527361869812, -0.1772276908159256, -2.0771377086639404, -0.5089492797851562, -1.5032413005828857, -1.5983120203018188, 0.009521245956420898, -1.9835740327835083, -0.28936004638671875, 2.696702480316162, 1.1434465646743774, 1.4018851518630981, 0.19000637531280518, 2.163193941116333, 0.8535435199737549, -1.1125305891036987, -0.32425564527511597, -0.6706135272979736, 0.3831018805503845, 3.3371009826660156, -0.7125037908554077, 1.2522945404052734, -1.5138286352157593, -0.6940604448318481, -0.15247726440429688, -0.4468808174133301, 2.260340929031372, 1.299384593963623, -2.259014129638672, 1.3302501440048218, -0.5462656021118164, 0.008321583271026611, -2.54392671585083, -0.8871732950210571, 0.1641954779624939, 0.6868366003036499, -1.5438182353973389, -0.9812787175178528, -1.1373285055160522, 1.1009776592254639, 0.2601940631866455, 1.4259538650512695, -1.698569416999817, 1.925750494003296, -0.15078887343406677, 0.397220253944397, -1.5574376583099365, -0.3118376135826111, -1.8479628562927246, -2.2972140312194824, 2.393089771270752, 1.1493537425994873, 0.21979451179504395, 2.250822067260742, -2.675504446029663, -1.2744908332824707, 0.26308125257492065, 3.1708931922912598, -0.37878942489624023, 2.1222429275512695, 0.17537632584571838, -0.7422739863395691, -0.6016479134559631, 0.7526366114616394, 0.06638453900814056, 0.18738213181495667, -1.4064695835113525, 1.5184638500213623, 0.5559773445129395, -0.23887509107589722, -0.41255342960357666, -0.9229761958122253, -1.6228212118148804, -3.2518186569213867, -0.9102146029472351, 0.011406280100345612, 0.5240944027900696, -2.3338279724121094, -2.2349791526794434, -0.6317346096038818, -0.6937185525894165, -1.430649995803833, -1.3476381301879883, -0.5474726557731628, -1.218880295753479, -1.2023184299468994, 0.309638649225235, -0.036068499088287354, 2.734151840209961, -0.6970105767250061, -1.3520358800888062, -1.2617201805114746, 2.1635665893554688, -0.4998965263366699, -0.4445686638355255, -0.30409491062164307, -0.8002645969390869, -1.1954219341278076, -0.26547881960868835, 1.7998524904251099, -0.7462246417999268, -1.424898624420166, -0.16728129982948303, -3.6894378662109375, -2.313727855682373, -1.6684443950653076, 0.2698861360549927, -1.3689758777618408, 0.4901123046875, 0.5814009308815002, -0.5218521356582642, 1.1761212348937988, 0.48308855295181274, 0.44323185086250305, -1.1261882781982422, -2.3818304538726807, -0.9297062158584595, -1.3716868162155151, -3.4860973358154297, 0.05947962403297424, -1.2654896974563599, 0.1361830234527588, -0.6457384824752808, -2.3204314708709717, -2.430246353149414, -0.4438927173614502, -0.40349793434143066, 1.1087980270385742, 0.465082049369812, 0.69134122133255, 0.023122265934944153, 1.1951029300689697, -1.0325653553009033, 1.0169153213500977, 1.1171001195907593, 0.480279803276062, 1.5519533157348633, -2.319647789001465, 0.9934580326080322, -0.5613430142402649, 0.12238454818725586, -0.36267614364624023, 0.8014591932296753, 1.7737023830413818, -1.5855275392532349, 0.2815673351287842, 0.3256412744522095, 1.1810996532440186, 0.05360567569732666, 0.13694682717323303, 0.2751027047634125, 1.6438500881195068, -0.6055009365081787, 1.8981486558914185, 0.10954618453979492, 0.13824211061000824, 1.1015092134475708, -2.1518259048461914, -0.15439337491989136, -0.1660163253545761, -1.336729645729065, -0.11243698000907898, -0.20176303386688232, -0.6387603282928467, -0.13818618655204773, -1.2754442691802979, -0.31357550621032715, 0.3797445297241211, 1.221720814704895, -0.15949305891990662, 1.3442344665527344, -2.4905850887298584, -1.2213982343673706, -1.0454436540603638, 0.9477989673614502, -0.3660081624984741, 0.8381101489067078, -1.0537041425704956, 1.3186949491500854, 2.03118634223938, 0.3887740969657898, 2.3096773624420166, -1.2012839317321777, 1.0855666399002075, -0.8127752542495728, 2.784992218017578, -1.871577501296997, -0.9886470437049866, -0.4815082848072052, 0.561257541179657, -2.0795724391937256, -0.1184237003326416, -4.056571960449219, 0.8512376546859741, 0.9341726899147034, 0.6194800734519958, -0.3672112822532654, 0.5384840369224548, -0.7623528838157654, 3.6303815841674805, -1.3650965690612793, 0.32258450984954834, 1.8651295900344849, -0.8768168687820435, -0.08529269695281982, -0.47250813245773315, -1.2794886827468872, 0.6511871814727783, 0.7144056558609009, 2.3384323120117188, 1.3647091388702393, 0.8314446806907654, 0.2135966420173645, 0.39277634024620056, 2.5673184394836426, 0.026162147521972656, -0.3835829496383667, -1.8745237588882446, -2.0626015663146973, 0.23880401253700256, 1.3745039701461792, 2.5694103240966797, -0.28743627667427063, -0.33902809023857117, -0.33238422870635986, 1.0966169834136963, -2.3242249488830566, 2.3765759468078613, -0.5017908811569214, -0.6616981625556946, 0.8918585777282715, -0.9486210346221924, 0.30741965770721436, -0.8737751245498657, -0.911986231803894, -0.6545941829681396, 0.3054038882255554, -0.13720357418060303, -0.2677019238471985, 1.1249563694000244, -0.40859371423721313, -0.41703271865844727, -2.1967644691467285, 0.2785775065422058, 0.6310068964958191, 1.467573881149292, 1.4158153533935547, 0.03950050473213196, 1.594236969947815, -1.8079931735992432, 1.3572360277175903, -0.0683295726776123, 0.6486455202102661, -1.165179967880249, -0.1258162558078766, -2.3720157146453857, -1.6979299783706665, 0.45795387029647827, -0.8206583857536316, -3.131669759750366, -0.17528824508190155, -1.4450618028640747, -0.9007940888404846, -0.02288872003555298, 0.07944786548614502, 0.11223196983337402, 1.0018821954727173, 1.6532304286956787, -2.1553680896759033, -1.25184965133667, 0.8479031324386597, -0.6021015644073486, 1.5094488859176636, 0.3605394959449768, -0.01715219020843506, 0.22310695052146912, -0.32637107372283936, 1.6850159168243408, 1.3548595905303955, -0.021741271018981934, 0.7750250101089478, 0.5300790071487427, -0.7252745628356934, -0.6157512664794922, 0.8045675158500671, -0.6176557540893555, 2.3681392669677734, 2.823307991027832, -1.3263483047485352, -0.09550207853317261, 1.4864435195922852, -1.5027425289154053, 2.1408393383026123, 2.1034977436065674, -0.001971602439880371, -0.2521498501300812, -0.7409225702285767, -0.3597317934036255, 0.4763885736465454, -3.046100378036499, -0.7383626103401184, -0.6894159317016602, -1.3783988952636719, 1.0860137939453125, 0.3816678822040558, -1.432417869567871, -1.4859106540679932, -0.9631652235984802, 2.8758151531219482, 0.8066455125808716, 1.2541775703430176, -2.227799654006958, -0.5194082260131836, -2.036719560623169, 0.46879592537879944, -0.09783895313739777, 1.4351481199264526, -2.8225488662719727, -1.1068813800811768, -0.07952558994293213, -0.28421705961227417, -0.3582003116607666, -1.2516852617263794, -0.398897647857666, 0.9180176258087158, 1.0129766464233398, -1.428218126296997, -0.09687817096710205, -1.6275880336761475, -1.0179961919784546, 1.5434123277664185, 2.649296998977661, 3.010563850402832, -0.4247281551361084, -0.3709971606731415, -0.9227727651596069, 0.3889862895011902, -2.4373269081115723, 0.05294550210237503, -3.890868663787842, -1.029252290725708, 1.7993133068084717, 1.9025973081588745, 0.17404024302959442, -1.3154752254486084, 0.3293217420578003, 0.6542513370513916, -0.49140918254852295, -0.1691170334815979, 1.361469030380249, 1.563907265663147, 1.630591630935669, -0.8096680641174316, -0.7198025584220886, -2.904473066329956, -0.9756816029548645, 1.489504098892212, -0.2207031548023224, -1.7050837278366089, 1.5940660238265991, -0.840854287147522, -0.1933460384607315, 0.18355250358581543, 0.6045569181442261, -0.41614723205566406, 1.4728511571884155, 0.9970792531967163, -0.9895973801612854, 0.8981479406356812, -0.0457877516746521, 1.5929691791534424, 1.5621099472045898, -0.547560453414917, 0.7625261545181274, 0.34377551078796387, 1.5218658447265625, 2.7585256099700928, 2.1715619564056396, 1.7637754678726196, -0.18738332390785217, 0.6896176934242249, -0.3744499981403351, -0.867913007736206, -1.5758426189422607, 0.5141900777816772, 1.581605076789856, -0.9166284799575806, 2.7924046516418457, 0.7798604369163513, -0.15406376123428345, 0.4571719169616699, -0.45085906982421875, -0.07545590400695801, -1.782958984375, 0.3685849606990814, -1.5844709873199463, 0.42164528369903564, -0.7143024206161499, 0.5839371681213379, -0.8339638113975525, -0.3940524756908417, 1.1405565738677979, -1.7828419208526611, -1.6061756610870361, 0.616580605506897, -0.7836533784866333, -3.139622688293457, 1.1083650588989258, 0.19292700290679932, -0.6656820774078369, -1.7010983228683472, -2.8928370475769043, 2.036783456802368, -1.488145112991333, -0.468754380941391, 0.4052313566207886, 2.2864255905151367, -0.09554728865623474, 0.47473829984664917, -1.0358164310455322, -3.599146604537964, -0.9027762413024902, -0.610776960849762, 2.2974512577056885, 1.399207592010498, -0.5087037682533264, -2.199263572692871, -0.01898092031478882, 0.06700596213340759, 2.059783935546875, 1.4437448978424072, -1.0622012615203857, 0.7134865522384644, -2.853795051574707, -1.1567448377609253, -0.5949521660804749, 0.14838746190071106, -4.08306884765625, -0.342241495847702, 0.3466322422027588, -1.3341680765151978, 1.7474669218063354, -0.915255069732666, -0.7790318727493286, 0.8862656354904175, -1.267614483833313, 0.0878271758556366, -1.4829058647155762, 0.2749348282814026, -0.12361156940460205, 0.06343260407447815, -0.137027770280838, -1.4892065525054932, 0.5110570788383484, 0.9077010154724121, 1.0818363428115845, -0.4907614588737488, 0.5455076694488525, -1.3515609502792358, 1.7818928956985474, -0.17506200075149536, -1.935922622680664, -0.07567636668682098, -0.9734842777252197, -1.3919755220413208, -1.5054526329040527, -0.7950526475906372, 0.9322815537452698, -0.981785774230957, -0.8804652094841003, -0.46941423416137695, 2.698352336883545, -0.8372797966003418, 0.7742002010345459, 1.027001142501831, 1.6927168369293213, 1.3106757402420044, 1.8564064502716064, 2.085376501083374, 1.1785494089126587, -0.48725444078445435, -1.3044828176498413, -1.9954744577407837, 0.7160940170288086, -0.3222106695175171, 1.242100477218628, 0.6119573712348938, 0.9273940324783325, 2.251491069793701, 1.4598047733306885, 1.7185389995574951, -1.4749808311462402, 1.2768828868865967, 1.2592976093292236, -0.5444022417068481, -2.721221923828125, -0.5033254623413086, 0.19482654333114624, -0.5577138066291809, 1.2781447172164917, -2.5541796684265137, -1.2654829025268555, -0.7885507345199585, -1.783536672592163, 0.7554506063461304, -0.3564208447933197, 0.5551392436027527, 0.7622562646865845, 0.48236072063446045, 1.5752320289611816, -1.9287954568862915, -1.1255303621292114, -1.7258520126342773, -0.19710905849933624, 0.938635528087616, 0.2814599573612213, -2.2673017978668213, 1.168627381324768, -0.0964827835559845, 0.3206107020378113, 0.17834842205047607, 0.9205670356750488, 1.9148225784301758, 0.10187971591949463, 0.4205779433250427, 0.7508581876754761, -0.3798329830169678, -0.5765388011932373, -2.1680216789245605, 2.8181650638580322, -0.9618887901306152, 0.6076928973197937, -1.6280708312988281, 0.9961576461791992, 2.154603958129883, 1.8395023345947266, -0.10720903426408768, -0.21674132347106934, 1.2618052959442139, 1.3085217475891113, -1.039911150932312, -0.29259392619132996, -0.4127122759819031, -1.4167728424072266, 2.2121517658233643, 0.003457188606262207, -0.03517305850982666, 0.01178714632987976, 1.6079306602478027, -0.2452707290649414, 1.0309563875198364, 0.42611557245254517, 0.18724334239959717, -0.772849440574646, -0.5758113861083984, -0.8812686204910278, 1.1289914846420288, -2.4175491333007812, 0.13830995559692383, -0.613055944442749, 0.5546643137931824, -0.033448368310928345, 0.6226140260696411, 0.7981729507446289, -0.09070971608161926, 0.42892593145370483, 0.704407274723053, -1.1605294942855835, -0.8766810297966003, 0.8176431059837341, -0.848781943321228, 0.7506727576255798, 0.5809162259101868, 0.1344260573387146, -0.17712397873401642, -0.13498520851135254, 1.8681331872940063, 1.0985678434371948, 1.009343147277832, 1.6060740947723389, -2.2136425971984863, -1.8192980289459229, -1.906705617904663, 0.004276446998119354, 1.4274680614471436, -0.6230117678642273, -0.5982782244682312, -2.8535845279693604, -1.555547833442688, -1.6425472497940063, 1.078923225402832, 1.2506227493286133, -0.16750741004943848, -1.0792587995529175, 2.2067599296569824, 0.19237840175628662, 0.8683253526687622, -0.9747952818870544, 0.9300776720046997, -1.5317063331604004, -1.8160327672958374, -1.2980684041976929, -0.72940993309021, 1.1108185052871704, -1.1084530353546143, -0.5854536890983582, 1.1008646488189697, 0.35705673694610596, -1.0218462944030762, 0.5274715423583984, -1.1591118574142456, 0.1920631229877472, 1.0972068309783936, -0.9788292050361633, -0.31200623512268066, 0.30444347858428955, 2.7612781524658203, -0.8034759163856506, 0.3106633424758911, 2.2545931339263916, -0.0705185979604721, -2.1165006160736084, -1.5386534929275513, 0.09747084975242615, 0.8800550699234009, -1.37982177734375, -2.1752686500549316, 1.8615957498550415, 1.842754602432251, 0.7949562072753906, -0.18212097883224487, 1.0459036827087402, 0.7518239617347717, 0.6515523791313171, 0.6265684962272644, -3.1185712814331055, -0.5932025909423828, 1.9309409856796265, 1.9972600936889648, 0.39538443088531494, -0.3419390916824341, -0.11514878273010254, 0.646578311920166, 0.1782492995262146, -1.260457158088684, 1.8281612396240234, -1.7070465087890625, 0.8490027189254761, 0.1898314654827118, -0.5232214331626892, 0.5001448392868042, -0.7698863744735718, 2.7067506313323975, 1.5388870239257812, -1.514214277267456, -1.655388593673706, -2.8531579971313477, -0.7265824675559998, -0.7638605237007141, -0.3159390091896057, 0.8980749845504761, -0.3382035791873932, 0.7524887323379517, 0.7018996477127075, -0.8537645936012268, 1.5410065650939941, -0.20189514756202698, -4.504171848297119, 0.7358522415161133, -0.3498937487602234, 0.4497479796409607, 3.4187424182891846, 2.547471761703491, 0.5790186524391174, 1.1237636804580688, -0.8507951498031616, 0.658575177192688, -0.14248919486999512, 0.42112648487091064, 1.4988173246383667, 1.0517418384552002, 0.4035501480102539, 0.2948099374771118, -0.6641653180122375, -1.1829992532730103, 2.776271104812622, -0.46018147468566895, 2.3760507106781006, -0.7409809827804565, -1.7527644634246826, 1.1686689853668213, 2.7242166996002197, 1.7784570455551147, 0.7224639654159546, 2.4580020904541016, -0.5803347826004028, -2.2944724559783936, -0.88150954246521, 0.2708224654197693, -0.3884093165397644, 3.0467662811279297, 0.013066768646240234, -0.8050253391265869, -0.21369633078575134, -0.06936013698577881, -1.9006998538970947, 0.3876303732395172, -0.8022484183311462, -1.9399605989456177, 1.0880759954452515, -0.11482374370098114, -0.39656421542167664, -0.47287091612815857, -0.7013691663742065, -1.88509202003479, 2.8640518188476562, -0.837475061416626, 0.17226329445838928, -0.6047181487083435, 1.1675199270248413, -2.3120627403259277, -0.09406909346580505, 0.7724081873893738, 0.374359130859375, 0.836004376411438, 1.2786648273468018, 1.743361473083496, 1.0466269254684448, 0.21276360750198364, 3.740328550338745, -0.277240514755249, -1.7707297801971436, 1.554474949836731, 3.7840185165405273, 1.1608372926712036, 0.9612176418304443, 2.1880645751953125, -1.9056367874145508, -0.38684549927711487, -1.2788439989089966, 0.4370684325695038, -1.581209421157837, -0.23238623142242432, 0.3308670222759247, -0.3127683401107788, -2.959172487258911, -3.213744878768921, -2.45988130569458, -1.1195323467254639, 2.304238796234131, -2.0224385261535645, -1.100635290145874, -0.21067094802856445, -1.8814542293548584, 0.8432266712188721, -1.372368335723877, -0.07925844192504883, -1.5439754724502563, 1.4477638006210327, -1.335712194442749, 1.1311763525009155, 3.3251678943634033, -2.121173620223999, 1.0730347633361816, -0.11382582783699036, -1.1276618242263794, -3.3173141479492188, 1.5537761449813843, -0.016737818717956543, 0.843463659286499, -0.7640217542648315, -0.042351722717285156, -0.9301602244377136, -0.7525954246520996, -1.3536499738693237, 1.263765573501587, 0.06405074894428253, -0.9468348622322083, -1.0031747817993164, -0.4922182559967041, -2.154338836669922, 1.8771660327911377, -1.655961275100708, 1.0848437547683716, -0.5917022228240967, 1.6614818572998047, -0.44563907384872437, -2.2509007453918457, 1.3887518644332886, 2.089547634124756, 0.7482776641845703, 0.6354193091392517, 0.7572839260101318, 0.17478328943252563, 1.0760821104049683, -0.7133331894874573, -2.3588454723358154, 1.657504677772522, -0.6631597280502319, -1.2948417663574219, -0.3911391496658325, -0.911520779132843, 1.8381184339523315, 1.0430715084075928, -0.6050330996513367, -0.8823816776275635, 0.4929213523864746, -0.24257473647594452, 0.7322885990142822, 1.136246919631958, 2.1322619915008545, 2.5529003143310547, -0.8731410503387451, -0.6332181692123413, -1.5543241500854492, -0.15204322338104248, 3.188880205154419, 1.5691578388214111, -1.2157422304153442, 0.33129918575286865, -0.8734933137893677, 3.380791425704956, 2.1767687797546387, -1.3771920204162598, -1.1639900207519531, -0.07764464616775513, -1.4510809183120728, -1.4168065786361694, -1.2875231504440308, -3.1067094802856445, 1.3638126850128174, -0.51753830909729, -1.430274486541748, -1.4969114065170288, 1.2820088863372803, -1.1089211702346802, -1.0717251300811768, -1.7442095279693604, -2.9275670051574707, -2.1612555980682373, 2.76334285736084, -1.2951489686965942, -2.485665798187256, -0.7956515550613403, -0.19734308123588562, 0.6387386918067932, 0.8050705194473267, -0.7030237913131714, -1.5146052837371826, 2.182290554046631, 0.836565375328064, 1.8362302780151367, 1.1732546091079712, 1.2659729719161987, 0.46077099442481995, -1.3229682445526123, 0.0337337851524353, 0.7563998699188232, 0.822641909122467, 1.5554603338241577, -1.3176898956298828, 2.807713031768799, -0.8196142911911011, -0.44495031237602234, 1.144368052482605, -1.456244945526123, 1.7283539772033691, 1.2592443227767944, 0.3969438076019287, -2.1665351390838623, 0.676928699016571, 0.6187555193901062, -0.16712546348571777, 2.049457550048828, 0.6313912868499756, -0.6575608849525452, -0.5756951570510864, 0.5630695223808289, 0.6880358457565308, 0.23612025380134583, 0.0325160026550293, -0.7011322975158691, 2.508125066757202, 0.3593156933784485, -1.8618440628051758, 1.1734373569488525, 2.73496675491333, 1.0418181419372559, 0.9853177070617676, -1.4784529209136963, 0.4050193727016449, -0.34003275632858276, -1.2280992269515991, 0.30545473098754883, -0.7837114930152893, 2.365109443664551, 1.1591721773147583, 0.7371567487716675, -0.3291780352592468, -1.1891053915023804, 0.138986736536026, 0.5258263349533081, -0.9631788730621338, 0.590960681438446, 1.765154242515564, 1.1843936443328857, -2.250554084777832, -0.5722346305847168, 0.29065096378326416, 0.45712342858314514, 0.33855950832366943, 0.22813910245895386, 1.4937222003936768, 0.37764066457748413, -0.11399483680725098, 1.4371836185455322, -2.077849864959717, 0.4911177456378937, -0.024026677012443542, -0.40016043186187744, 2.2326643466949463, 2.822056770324707, 0.9126823544502258, -1.132917881011963, -0.2927391529083252, 1.3154327869415283, 1.2272902727127075, 1.6172053813934326, -1.378433346748352, 0.7707048058509827, 1.799159049987793, 1.365976095199585, -1.405135989189148, -1.4989250898361206, -2.036336898803711, 2.076202392578125, 0.9524961113929749, 0.6273674964904785, -0.21768835186958313, -0.33979088068008423, 1.6352763175964355, -0.11132949590682983, -1.8102712631225586, -1.312809944152832, -1.5289126634597778, -1.754073977470398, 0.9683657884597778, 0.9268304705619812, -0.5730478763580322, -2.451002597808838, -0.050151027739048004, 1.6264415979385376, -1.7526363134384155, -1.4346575736999512, 0.880628228187561, -0.7419205904006958, -0.5910358428955078, 1.6127495765686035, -0.14426162838935852, -0.7447267770767212, -0.5859087705612183, -1.0132380723953247, -1.2347986698150635, 0.44462791085243225, -2.306722402572632, 1.2257089614868164, 0.12170390784740448, -0.7246472835540771, 1.5873661041259766, -0.8341943621635437, 1.7115052938461304, 3.2947163581848145, -0.6133840680122375, -1.1590741872787476, 0.6250851154327393, 1.3587501049041748, -2.45828914642334, 4.704127311706543, 1.8857347965240479, -1.128028392791748, 0.35675448179244995, -1.5854847431182861, -3.242689609527588, -2.574924945831299, 2.040550947189331, 0.05293789505958557, -0.05189664661884308, -0.6254479885101318, -2.240691900253296, -0.5363461375236511, 0.9887534379959106, -2.0321474075317383, 1.3856157064437866, -1.1080256700515747, -0.9612516760826111, -0.2531278133392334, -2.637934684753418, 1.8823964595794678, 0.4010252356529236, 2.108790159225464, -0.31003719568252563, 0.4612184166908264, -0.7447178363800049, 2.1146256923675537, 2.3430187702178955, -1.006066083908081, -1.5238927602767944, -1.4196910858154297, -1.8699854612350464, -0.9711852073669434, -0.04687166213989258, -0.5002968311309814, 0.23797661066055298, 1.7087594270706177, 2.5062828063964844, -0.014873966574668884, -0.28787022829055786, -0.34649598598480225, -0.2386709451675415, -0.9899502992630005, -0.264373779296875, 0.038214802742004395, 1.670090913772583, 0.34664011001586914, -1.0487982034683228, 1.7955867052078247, 0.36572152376174927, -0.9419021010398865, 1.64906644821167, -2.3719592094421387, -1.273614525794983, -0.9614993333816528, -0.9129339456558228, -2.3412442207336426, -0.6248266696929932, -1.173689842224121, 3.270681619644165, 0.9745493531227112, -2.778228998184204, 0.09995357692241669, -0.48332852125167847, -3.3955092430114746, -0.06142169237136841, -1.3620963096618652, -1.1548030376434326, -0.4085720181465149, -1.6790664196014404, 0.68201744556427, -0.7002162933349609, 0.32121291756629944, 0.4302794933319092, -0.5639023780822754, -1.356976866722107, -0.1990860551595688, -0.1358136236667633, -1.160515546798706, -2.4852733612060547, -0.8064371347427368, 0.329531729221344, 0.5815643668174744, 0.5824378728866577, -0.45474693179130554, -0.7881734371185303, 0.44632312655448914, 0.577641487121582, -1.0953818559646606, -2.8888957500457764, 0.6646316647529602, -2.393796443939209, 1.4770954847335815, -0.06889653205871582, 0.13230711221694946, -1.9058011770248413, -0.33838385343551636, 0.7981650829315186, 1.401827335357666, -2.349339246749878, -2.7720651626586914, -0.766441822052002, 0.08346456289291382, 2.8677587509155273, 0.6740477085113525, 1.7012434005737305, 0.875927209854126, -0.2011089324951172, -0.11685007810592651, -0.2761070728302002, -3.707463264465332, 0.00820988416671753, -0.8368163704872131, 1.8889904022216797, -0.5013551712036133, 1.6283624172210693, 1.0623663663864136, 1.136451244354248, -0.23201991617679596, -0.6176018714904785, -3.67474102973938, 1.0082850456237793, -1.4242775440216064, 1.7281317710876465, -0.3757762908935547, 1.040703535079956, 0.20882558822631836, 2.089322566986084, -0.2794031798839569, 0.34554219245910645, 0.5724035501480103, -2.157454013824463, 2.254181146621704, -1.1444393396377563, -1.2567843198776245, -0.22270452976226807, -0.9245555400848389, 2.0898494720458984, 1.744678258895874, 0.3982545733451843, -2.586521625518799, -1.6439625024795532, -0.7797616124153137, 1.738202452659607, -0.23474889993667603, 2.844097137451172, 2.055840253829956, 2.03983736038208, 0.9575955867767334, -1.0201027393341064, 1.482911467552185, 0.7526413798332214, 0.07919123023748398, -1.7736166715621948, -1.0980677604675293, -0.6495600342750549, 1.2620121240615845, 0.8791354894638062, -0.2233715057373047, -0.17871680855751038, -0.48216012120246887, 2.7556869983673096, -0.9412615299224854, -2.126007318496704, 1.0393842458724976, 1.3804765939712524, -0.254172146320343, -0.1354532241821289, -0.33085381984710693, 0.7235732078552246, -0.49681591987609863, 0.9443762302398682, -1.3853976726531982, -0.2657817304134369, -1.43208909034729, -0.6117090582847595, -2.0356013774871826, -0.2108389139175415, 0.10951322317123413, 0.10260981321334839, 2.121934413909912, 1.2776507139205933, -0.17077486217021942, -0.5692566633224487, -0.23601201176643372, 4.059363842010498, -0.24478745460510254, -1.652584195137024, 0.5744037628173828, -1.5984655618667603, -0.10454699397087097, 1.2166575193405151, 3.613574266433716, -3.420694589614868, 1.906097412109375, 0.472686231136322, -0.14676180481910706, 0.32816386222839355, 0.6341698169708252, 0.8652365803718567, 2.3655781745910645, -0.9159190654754639, 0.8446133732795715, -0.6644559502601624, 0.8630852103233337, 1.1546409130096436, 0.3273710012435913, -0.5450260043144226, 3.557908058166504, 0.7314548492431641, 0.9323635101318359, -2.498260021209717, -0.9820172786712646, 1.0774222612380981, 0.325826495885849, 2.474499464035034, 0.5547795295715332, 0.8368390798568726, -1.3018977642059326, -0.5311119556427002, -0.4481830596923828, -1.7705891132354736, 0.22346054017543793, 0.9136914014816284, 1.3510080575942993, -0.8878860473632812, 0.326740026473999, 1.4957199096679688, 0.029096543788909912, -0.7554553151130676, -1.2443666458129883, -0.011509060859680176, 1.3879709243774414, -0.8372278809547424, -2.6726388931274414, 0.7699775695800781, -1.5834089517593384, 1.4973760843276978, -0.74818354845047, -1.647372841835022, 0.8818241357803345, 0.9981716275215149, 0.052688270807266235, 2.0620920658111572, -0.6727885603904724, -0.916926383972168, -1.8279844522476196, -0.44554945826530457, -2.0523464679718018, 0.2551220655441284, 1.0014045238494873, 0.2908422350883484, -0.5314109325408936, -1.6240911483764648, 2.349327325820923, -1.4846090078353882, -0.8682315349578857, -1.530276894569397, -0.524055540561676, 0.8264302015304565, -0.8549230694770813, 2.035991907119751, -1.6410346031188965, 0.27804720401763916, 1.0221905708312988, 0.7029891610145569, 0.9103785753250122, 0.22787633538246155, -1.2291866540908813, 2.3630146980285645, -0.002719700336456299, 0.44371142983436584, -1.0645341873168945, 0.49669671058654785, 0.32109808921813965, -0.5417987108230591, 3.3062822818756104, 1.57780122756958, -0.20767329633235931, -0.34187090396881104, 1.0028095245361328, -0.676823616027832, -0.8267984390258789, -0.320624977350235, 2.2756242752075195, -1.4933584928512573, -3.6739234924316406, -1.880765438079834, 1.2784411907196045, -0.8048664927482605, -1.111431360244751, 1.3126853704452515, 0.0036720633506774902, -0.26550576090812683, -0.6654230356216431, -0.30227726697921753, 0.42455607652664185, -2.278209924697876, 1.1177647113800049, 1.167736530303955, -0.2477370649576187, 1.6620036363601685, -1.8173002004623413, 1.0196722745895386, 1.3929331302642822, -2.369518756866455, 0.0018502436578273773, 0.0009743571281433105, -0.22814670205116272, 0.9188216924667358, 0.25539064407348633, 0.39673805236816406, -0.11452257633209229, 0.13345938920974731, -0.14456748962402344, 1.620863914489746, 1.0331662893295288, 0.7949475646018982, -0.8976688385009766, -0.08788672089576721, -2.0795278549194336, -0.72984379529953, -2.8160560131073, 0.4653591215610504, 0.6022117733955383, -0.1282913088798523, 0.6818832755088806, 2.7228105068206787, 1.3079218864440918, 2.0655651092529297, 0.3857600688934326, -1.4523561000823975, 0.3020665943622589, 0.8671911358833313, -1.1007685661315918, -1.5080761909484863, -3.2018117904663086, 0.06287018209695816, 0.44492265582084656, -0.6619705557823181, 0.5003709197044373, -3.3769750595092773, -0.6491206884384155, 1.5509560108184814, 0.2727540135383606, 0.2968364953994751, -1.613425374031067, -0.3659559488296509, 1.8290820121765137, 2.384338140487671, -3.518156051635742, 0.5259427428245544, 1.4421544075012207, 0.6394555568695068, 0.3219348192214966, 1.3880960941314697, -1.007616400718689, -2.5449204444885254, -2.184462547302246, 0.38430139422416687, -0.35393500328063965, -1.8746132850646973, -1.2040793895721436, 0.983855128288269, 1.8858158588409424, -2.0429701805114746, 1.3620498180389404, 0.1407289057970047, -0.5229693055152893, 2.918095350265503, -0.586517333984375, -1.7138158082962036, -1.8893975019454956, -0.18228021264076233, 1.3541336059570312, -0.9760693907737732, 0.8906827569007874, 0.05868887901306152, 2.144134521484375, -1.313799500465393, 0.0793803334236145, -0.18933314085006714, 1.258345127105713, -3.0254435539245605, -1.236908197402954, -1.4769023656845093, 1.217024564743042, -2.9436023235321045, 0.28779956698417664, 0.5646859407424927, 0.38574767112731934, -1.4060662984848022, 1.0679566860198975, 0.01609182357788086, -2.786449432373047, -1.019161581993103, 0.9040665030479431, 2.0810444355010986, -1.3129732608795166, 1.6219316720962524, -3.089195489883423, -3.47300386428833, 1.2763243913650513, 0.48521554470062256, 0.9025357961654663, 2.346740484237671, -0.43766018748283386, -3.2575902938842773, 0.734699010848999, -0.6788676977157593, -0.9390584826469421, -0.18899290263652802, -0.16817203164100647, 2.532160758972168, -0.4222292900085449, -0.2104480266571045, -0.3506503105163574, -2.4426345825195312, 2.160093069076538, 2.0864877700805664, -0.014802396297454834, -2.543581962585449, 0.3124305009841919, -0.685309112071991, -1.3597960472106934, -0.11779928207397461, -0.024384021759033203, 1.2116975784301758, 2.0424611568450928, 0.9680254459381104, -1.0453083515167236, 2.976273536682129, -0.17479529976844788, 2.783597230911255, 0.3864017724990845, -0.8260750770568848, -1.1410181522369385, 1.9034912586212158, -0.8127061724662781, 0.07846826314926147, 1.820361614227295, 1.9006943702697754, 1.522809386253357, -2.128488540649414, 2.9681777954101562, 2.258169174194336, 1.1912477016448975, -1.5598759651184082, -1.7879750728607178, -0.837325930595398, 0.4527512192726135, 1.0395944118499756, -0.355111688375473, 1.202014684677124, 0.6772341728210449, 2.661482810974121, 0.3801846504211426, -0.8250490427017212, 1.4902362823486328, 0.042682528495788574, 0.31574884057044983, -3.622321367263794, -0.21535176038742065, -0.6913288831710815, -0.1430370807647705, 2.4832305908203125, -0.9371286034584045, -0.7160313129425049, -1.7443922758102417, -0.7151585221290588, -0.16792774200439453, 3.026869535446167, 0.22955167293548584, -0.933881402015686, -0.002045929431915283, 0.49202531576156616, -0.6497897505760193, 1.5897798538208008, -0.5255350470542908, -0.6367053985595703, 3.922926425933838, -0.094164177775383, -1.1758426427841187, 4.144506454467773, 1.4919772148132324, 0.02471059560775757, -0.2295616865158081, 2.5525598526000977, 2.5210483074188232, 0.23320043087005615, -2.6804306507110596, -1.052886724472046, -0.5605226159095764, 2.2950668334960938, 0.4879188537597656, -0.7911117672920227, -0.2634674310684204, 0.7835765480995178, -2.9745044708251953, 0.8715964555740356, 0.36193597316741943, -0.9983020424842834, 0.5772770643234253, -0.5003402829170227, -0.16370552778244019, 1.1719694137573242, -2.1030025482177734, -2.3723020553588867, 1.9390068054199219, -3.0514659881591797, -0.3146006464958191, 0.2596402168273926, 0.21957020461559296, 1.2672463655471802, -0.728273868560791, 0.5699830055236816, -1.9307286739349365, -1.6980623006820679, 0.5349640846252441, 0.26624906063079834, 1.1892969608306885, 2.147939682006836, 2.3667120933532715, -0.3831402063369751, -1.3030730485916138, -1.2502667903900146, 0.1315128207206726, -1.324479341506958, -0.8974593281745911, -0.2839544117450714, 0.4260633587837219, 2.027339220046997, 1.8657324314117432, -0.6189424991607666, -0.5224522352218628, -1.503889560699463, -2.6207454204559326, 0.8527968525886536, -1.3225260972976685, 2.5639843940734863, 1.422871470451355, 0.9247258901596069, 2.398890495300293, -0.21875187754631042, -0.15454736351966858, 0.33605241775512695, 0.7014428377151489, -1.051544427871704, 1.6000640392303467, 1.5801494121551514, 2.067200183868408, -1.6937700510025024, -0.8129288554191589, -2.0236282348632812, -1.6049261093139648, -1.9416916370391846, -1.2495145797729492, 1.6409506797790527, -2.573922872543335, 0.3366527557373047, 1.3836686611175537, -2.6000499725341797, -1.117658019065857, -3.176908493041992, 1.2859069108963013, 0.6249125599861145, -0.8500261306762695, -0.7838743925094604, 1.070656418800354, -1.0129170417785645, 1.6212486028671265, 3.2697243690490723, -0.8182997703552246, -2.0977697372436523, -3.068577766418457, -1.2629910707473755, -2.091142416000366, 1.1945587396621704, 0.24584129452705383, -0.6374243497848511, -3.6480603218078613, 0.4635504484176636, 0.4081634283065796, -2.18631911277771, -0.014832258224487305, -0.17869248986244202, -0.3475702404975891, -3.7818820476531982, 1.818949580192566, 0.4550185799598694, -0.024221062660217285, -0.5475300550460815, 0.5758589506149292, -0.7515504360198975, 0.5020562410354614, 0.04107797145843506, 3.3363871574401855, 0.09685695171356201, -0.47268426418304443, -0.5844067931175232, -1.2416610717773438, 1.0110503435134888, 1.6546545028686523, 0.9086503982543945, 0.8993831276893616, 0.6623320579528809, -0.7751991748809814, -0.5498852729797363, -1.7067537307739258, 0.8499932885169983, 0.6542755365371704, 3.1937549114227295, -1.7648484706878662, 0.22740861773490906, 1.4665123224258423, 0.03592550754547119, -2.184553623199463, -0.3819460868835449, -0.03232884407043457, -2.521367073059082, -0.5473613739013672, 1.0022122859954834, -2.798184394836426, -2.371121406555176, 2.2459630966186523, -0.07572513818740845, 0.16526421904563904, -0.05636660009622574, -0.3673627972602844, 1.1324102878570557, 1.8842167854309082, 0.8527071475982666, 1.528860330581665, 1.2259637117385864, 1.8571611642837524, 0.1893203854560852, 0.5901978015899658, 2.4913928508758545, 0.4929701089859009, 0.6532554030418396, 1.0705057382583618, -1.1576650142669678, 1.315212607383728, 0.4293611943721771, -1.2734034061431885, 0.9517682790756226, 0.5266079902648926, -1.353393316268921, -1.782548427581787, -0.7968077659606934, 0.004157006740570068, -1.2324678897857666, 0.3020171523094177, 0.9719507694244385, -0.5397517085075378, -0.3465036153793335, -0.890302300453186, 1.7380422353744507, 0.6400998830795288, 0.3498672544956207, 2.3340587615966797, 3.6510157585144043, -1.074012279510498, 1.3210757970809937, 1.6067492961883545, -3.749281167984009, 2.965245246887207, 1.1604657173156738, 0.048747718334198, -1.7099876403808594, -1.8586918115615845, 1.7862260341644287, 1.0193568468093872, -0.1364382803440094, 0.40133386850357056, 0.10115921497344971, 1.1371241807937622, 0.9735588431358337, 1.3004461526870728, -1.2009707689285278, 1.0624663829803467, 0.811163604259491, -0.33187830448150635, -1.413748025894165, 1.4372808933258057, -0.989698588848114, 0.2355426847934723, -0.21263189613819122, 1.0694037675857544, 0.5262835025787354, -0.5307698249816895, 0.41969937086105347, 1.4474589824676514, -0.24907207489013672, -2.6836180686950684, 1.9124435186386108, 2.5947046279907227, -0.5010384917259216, -1.1742231845855713, -0.7229328155517578, -2.972252368927002, -2.7189841270446777, 2.722851514816284, -0.03840702772140503, -0.076805979013443, 0.6837248802185059, 1.33922278881073, -0.415525883436203, 0.746157705783844, 0.3179517388343811, -0.23231475055217743, 0.19742605090141296, 0.047042012214660645, 2.0145304203033447, -0.23020118474960327, -1.2571499347686768, -0.21748146414756775, 0.13870173692703247, -0.19905397295951843, 0.32935187220573425, 2.32462215423584, 0.8255975246429443, 0.9104596376419067, 1.302385926246643, -1.3133108615875244, 0.8153563141822815, -3.904064178466797, 0.3280518054962158, -0.6906304359436035, -0.5775293111801147, 1.4558442831039429, 0.3501443862915039, 1.591108798980713, 1.4629175662994385, 0.14272582530975342, 1.6242436170578003, 0.16018497943878174, 1.1797865629196167, 0.5801292061805725, -2.136991262435913, 0.46918612718582153, 1.8282631635665894, -0.03252190724015236, 1.2932161092758179, 0.29516398906707764, 1.8924968242645264, -1.3902727365493774, 0.342718243598938, -0.8754227161407471, 1.0274869203567505, 0.8853439092636108, 0.30274850130081177, -1.4374181032180786, 0.7427840232849121, 2.2393882274627686, -0.524540901184082, -0.5318636298179626, -2.138535976409912, 0.907994270324707, 2.1883487701416016, -0.18927153944969177, 0.9809048771858215, 0.9462138414382935, -1.791959524154663, -0.05846916139125824, 0.08419837057590485, 0.5873250365257263, 1.1775314807891846, -1.7688827514648438, -1.2415552139282227, 0.7863839268684387, -0.285711407661438, 0.620573878288269, -2.499610424041748, 0.35077881813049316, -0.318431556224823, -1.644842267036438, -2.489825963973999, -1.7286187410354614, 0.2510659098625183, -1.0027689933776855, -0.7234528660774231, 0.6401734948158264, -0.3133108615875244, 0.6867144107818604, 0.47776561975479126, -0.3479561507701874, 1.446344256401062, 0.8963491320610046, 1.4791926145553589, 0.7857895493507385, 0.34915271401405334, -1.3849971294403076, -1.119310736656189, -0.15732866525650024, -0.19661632180213928, 2.2062454223632812, 0.3583333492279053, 1.5825296640396118, -0.6939431428909302, -1.8215447664260864, 0.15338855981826782, 1.0413874387741089, 0.15934745967388153, 1.3099404573440552, -0.0612795352935791, -1.1140947341918945, 1.1215366125106812, 0.44948410987854004, -1.9887481927871704, 1.4915428161621094, -1.2350564002990723, 1.9730569124221802, -0.22976021468639374, -0.14433276653289795, 1.7885196208953857, -1.5131760835647583, -2.270015001296997, -1.7705128192901611, 0.4585234224796295, 1.6516143083572388, -2.5486931800842285, -1.5472605228424072, -0.879202663898468, 3.496588706970215, 2.8656115531921387, -0.24189424514770508, -0.2446746826171875, 0.017173945903778076, -1.3280291557312012, -0.4377453029155731, 0.2255142331123352, -1.5338385105133057, 1.6317894458770752, -0.3332619071006775, -1.9892349243164062, 0.40908190608024597, 0.2003767043352127, -0.5824302434921265, -1.6433277130126953, -1.7380139827728271, 1.1317212581634521, -0.9561281204223633, -0.34208548069000244, -0.9297274351119995, 0.6384726166725159, 0.5895977020263672, 2.4751200675964355, 2.9297587871551514, -0.0021032094955444336, -0.9392140507698059, -2.5302581787109375, 1.013880968093872, 1.6571954488754272, 0.11300066113471985, -0.5110077857971191, 1.4134083986282349, -1.425357699394226, -1.286036491394043, -0.5645327568054199, 1.929212212562561, -0.760932981967926, 0.5718185901641846, 0.3259281814098358, -2.229334592819214, -2.067936420440674, 2.518653392791748, 0.9491416811943054, 0.9549986124038696, -0.2946402430534363, -0.34803497791290283, 3.664199113845825, 0.4136165380477905, -1.6100478172302246, 2.475135326385498, 1.071865200996399, 0.8003155589103699, 1.0664467811584473, -0.853847086429596, 1.3268282413482666, 0.19337886571884155, -0.07732813060283661, 0.7735368609428406, 1.0544219017028809, -1.8707568645477295, 0.5971286296844482, -0.8262314796447754, -2.390829086303711, 0.2087670862674713, -0.507362961769104, -0.48272109031677246, 1.2629761695861816, 0.5587109923362732, -0.1231054961681366, 0.10828360915184021, -2.803413152694702, 0.6862829327583313, 0.7068535685539246, 1.606745719909668, 0.8757525682449341, 0.07558377832174301, 2.001315116882324, 1.891904354095459, -0.944939136505127, 1.092248558998108, 1.386063575744629, -0.27900809049606323, -0.5725541710853577, -1.1189051866531372, -1.5559639930725098, -0.6008485555648804, -1.4113012552261353, -1.4580307006835938, -0.733081579208374, 0.4156932830810547, -0.8220007419586182, -0.09251159429550171, -0.1072891354560852, -1.4123713970184326, 3.6561031341552734, -0.8915048837661743, 0.286338746547699, 1.0467902421951294, -0.5784549713134766, 0.26501989364624023, 0.29189643263816833, -1.4530837535858154, -1.0630998611450195, 1.3444139957427979, 0.19124698638916016, 0.8485119938850403, -0.08626878261566162, -0.5629239082336426, 1.7410472631454468, -1.0917469263076782, -0.0825040340423584, -1.489039421081543, -0.5960810780525208, -1.555615782737732, 0.6901559829711914, -0.03045743703842163, 0.7180874943733215, -4.040833473205566, 0.6252084970474243, -0.26859691739082336, -1.0617039203643799, 2.364814043045044, 0.12665963172912598, -0.8183563947677612, 0.6173208951950073, -1.2629115581512451, -1.0989716053009033, 2.929572582244873, 0.049409329891204834, 0.3841972351074219, -0.925242006778717, 2.624852418899536, -1.2552927732467651, -1.6619327068328857, 1.2930338382720947, 0.8313331604003906, -1.3978192806243896, -2.0202767848968506, -1.5820869207382202, 0.13491785526275635, 2.703441619873047, 1.5120186805725098, 0.7743006944656372, -0.48993954062461853, 0.037151481956243515, -0.8978363275527954, 0.4924841523170471, -0.8569539785385132, -1.391249418258667, 0.1685856580734253, 1.002237319946289, 0.6159267425537109, -0.46189936995506287, 0.41827696561813354, 0.9104188680648804, 2.06355357170105, -1.3222005367279053, 1.0599312782287598, -1.2734724283218384, 2.460075855255127, -0.09866416454315186, -0.45929282903671265, 1.1951467990875244, 0.07905182242393494, -0.8033901453018188, -0.6413244605064392, -0.21133650839328766, -0.05314880609512329, 2.142763614654541, 1.0554540157318115, 2.2365562915802, 0.7507571578025818, 2.1721456050872803, 0.7177289724349976, -1.7328574657440186, 1.231567621231079, -0.905049204826355, -1.1240386962890625, -0.023927509784698486, 2.262448787689209, -3.088592529296875, -1.8847026824951172, 1.9054059982299805, -1.405735731124878, 0.040799617767333984, -0.36442893743515015, -0.706201434135437, -1.1541171073913574, 0.3544662594795227, 1.5594342947006226, -2.1585583686828613, -0.7337534427642822, -0.043073009699583054, 1.8016979694366455, -2.1820785999298096, 2.193751335144043, -0.666652262210846, 0.02151048183441162, -1.3661601543426514, 0.3879053592681885, -1.3597712516784668, -0.2769152522087097, -2.9221982955932617, 2.905381202697754, -0.5505209565162659, 0.7819162607192993, 3.030512809753418, 2.033336639404297, 0.31860628724098206, -2.1166679859161377, 1.5172224044799805, -0.383756160736084, 0.38891905546188354, -0.7652639746665955, -0.40590500831604004, 0.7580763101577759, -0.8342363238334656, -0.4428982734680176, -1.798424482345581, -1.1651690006256104, -0.5872640013694763, 1.965537428855896, 2.3540825843811035, -2.622159481048584, -1.9108479022979736, -1.8027571439743042, -0.1309780776500702, 0.5883001089096069, -0.839013934135437, -0.02408367395401001, -1.5067315101623535, 1.4946435689926147, -0.25962311029434204, -0.01557016372680664, 0.3634129762649536, 1.8116686344146729, 0.4855000078678131, 2.030388593673706, 0.5696264505386353, -0.1684844195842743, 0.3118760585784912, 1.49971604347229, 1.6021238565444946, -1.2937171459197998, -1.480546474456787, -0.03377455472946167, 4.275261402130127, -0.23060071468353271, 1.7100379467010498, -0.6068985462188721, 1.7412188053131104, -1.575907588005066, -1.1719086170196533, 1.5596774816513062, 0.7668531537055969, -1.4081748723983765, -0.36727696657180786, -2.3286380767822266, -0.07994754612445831, -0.13187116384506226, -1.3353724479675293, -0.9490457773208618, 0.6599626541137695, -2.238802671432495, 0.3991139829158783, 1.7134315967559814, 1.3820538520812988, 0.877011239528656, 0.33375996351242065, 1.4997000694274902, -1.539395809173584, 0.40333837270736694, 2.5776960849761963, 0.6270492076873779, -0.15061382949352264, -1.499961018562317, -1.8452177047729492, -0.013533882796764374, 0.3873434066772461, 1.183380126953125, -0.05282682180404663, 2.7512004375457764, 2.346902370452881, -1.125020980834961, 1.0839297771453857, -0.9726389646530151, 0.46136683225631714, 1.0999858379364014, -0.26929593086242676, -0.7771005630493164, 0.485065221786499, -0.9277870059013367, -0.4861074388027191, -0.012499332427978516, 1.0939279794692993, -0.9811307787895203, 0.06414598226547241, 1.1179672479629517, -0.3633076548576355, -2.6151793003082275, -2.1554834842681885, 1.1379568576812744, -0.4911389648914337, 0.5803712606430054, -0.8474233150482178, 0.6106945276260376, 0.969199001789093, 1.1834796667099, 0.18112318217754364, 1.2749427556991577, 1.2729758024215698, -0.5586845874786377, -1.2175227403640747, -1.906173586845398, 3.2225637435913086, -0.26459479331970215, 0.3997313976287842, 1.1182929277420044, -1.6275503635406494, -0.21836432814598083, -0.4004247784614563, -1.5938820838928223, 0.8185189366340637, -2.1592416763305664, 0.6634595394134521, -2.3476388454437256, -1.270744800567627, -3.596266269683838, -1.3998233079910278, 0.4889296293258667, 2.2333550453186035, 1.2296028137207031, 0.041506946086883545, -0.36787548661231995, 0.4978262186050415, 0.7311280965805054, 1.2070653438568115, 0.31124192476272583, 0.06541454792022705, 2.627743721008301, 0.6915373206138611, -2.667147159576416, -1.6406127214431763, 1.4927408695220947, -0.4514920711517334, -1.2753169536590576, 0.012803137302398682, 1.0818049907684326, -1.9141591787338257, 1.4445865154266357, -0.5842261910438538, -0.22464992105960846, -1.4628976583480835, 1.3999547958374023, -0.6538178324699402, 3.0230712890625, -0.04074138402938843, 1.3354599475860596, 2.209369421005249, 1.592759609222412, -1.763034701347351, 0.15933775901794434, 0.6338396668434143, 0.8825416564941406, -0.1101493239402771, -0.9212827682495117, -1.8586689233779907, -0.8753160834312439, -0.5208181142807007, -0.4658496379852295, 1.1022059917449951, -0.5004310607910156, -1.431728720664978, 1.1736518144607544, 1.3714286088943481, 1.4055588245391846, 0.8986157774925232, 1.252065658569336, -0.4235495328903198, -3.228651523590088, -0.9254955053329468, -2.765542984008789, -1.970306158065796, 0.42463433742523193, 0.37599220871925354, -0.8273663520812988, -0.8600336313247681, -0.9106325507164001, -0.33534079790115356, 2.0353102684020996, -0.03535866737365723, 1.2904717922210693, 1.162346363067627, -2.1614341735839844, 0.45944294333457947, -0.24684852361679077, -1.6268174648284912, 2.2440476417541504, -0.7274574041366577, 0.9468184113502502, 0.7321012616157532, 0.8098524808883667, -1.5432692766189575, -3.0020594596862793, -0.7224786877632141, -0.42804214358329773, -0.3753088712692261, -1.1642851829528809, 0.906499981880188, 2.5313310623168945, 0.9146801829338074, 0.0742749571800232, 0.8560259342193604, 1.7950897216796875, -0.516364574432373, -1.3173376321792603, 0.10431194305419922, -0.5853967070579529, 2.7212791442871094, -0.3313209116458893, -0.838313639163971, 0.5083391666412354, -2.189406633377075, -3.188140869140625, -1.8327124118804932, 0.6187030076980591, -0.7006115913391113, 2.5423526763916016, -1.9083963632583618, 1.736276626586914, -1.1084678173065186, 0.8696732521057129, -0.47548553347587585, -0.4202643632888794, 3.667689323425293, 0.6285488605499268, 0.7987173199653625, -1.540283441543579, 1.9449841976165771, 0.2647331953048706, -0.7812014818191528, -0.8525782227516174, -0.18587900698184967, -0.8705892562866211, -2.1638476848602295, -2.4964914321899414, 1.1048121452331543, -2.2591452598571777, -1.0306882858276367, -2.2881858348846436, -1.1191017627716064, 0.5667005777359009, 0.37625652551651, 2.483927011489868, 0.3487764596939087, -0.09418618679046631, -0.10412359237670898, -0.16731244325637817, 3.2091829776763916, -2.164534568786621, -1.6789438724517822, -2.5267269611358643, 0.8589636087417603, -0.9929677248001099, 0.08696067333221436, -1.7324202060699463, -0.9322537183761597, 0.3039725422859192, 1.1119403839111328, 2.560594081878662, 0.29457205533981323, 0.49854815006256104, -3.320343017578125, 1.0102037191390991, -0.780438244342804, 0.472343385219574, 1.949605941772461, -1.4393808841705322, -1.1349506378173828, 1.190990924835205, 2.083004951477051, 1.3994239568710327, 0.8835045695304871, -0.32339441776275635, -1.8104548454284668, -0.35152387619018555, -0.35188934206962585, 0.48696720600128174, 2.0030808448791504, -0.5950015783309937, -1.6620001792907715, 1.4604899883270264, 0.445925772190094, 1.4331059455871582, 1.342947006225586, 1.400151252746582, 1.010346531867981, -2.382877826690674, -0.41727471351623535, 0.6703037023544312, -0.32668623328208923, 0.5072125196456909, -0.8132689595222473, 0.45340079069137573, 1.0282833576202393, -0.37844008207321167, -1.570921540260315, -0.7721672058105469, -3.9276695251464844, 1.3827588558197021, -0.8779279589653015, -0.06736412644386292, 1.18892502784729, -1.501258134841919, -2.733551025390625, 0.28736650943756104, -0.809955894947052, -1.023031234741211, 1.2036714553833008, -0.5589151382446289, -1.1503626108169556, 2.9813029766082764, -0.9568960070610046, -2.378044366836548, 1.6012178659439087, -1.0415568351745605, 2.0864524841308594, 0.7384693622589111, -0.6898742318153381, 0.9116933345794678, 1.883392572402954, 0.9602693319320679, 0.5204764604568481, -1.2843010425567627, -0.9294349551200867, -1.338627576828003, -0.10281050950288773, 2.059873580932617, 2.2582695484161377, 1.85308837890625, -0.9337767958641052, 1.4939038753509521, -0.43637627363204956, 0.28785455226898193, -0.5167585015296936, 0.019620180130004883, -0.020118363201618195, 0.9950634837150574, -1.7918940782546997, 0.15085721015930176, -0.6440674066543579, 0.7211576104164124, -0.1614423245191574, -1.4360623359680176, 0.3568045496940613, 1.2477681636810303, -0.3595542907714844, 0.5234613418579102, 1.6944971084594727, 2.3711538314819336, -0.15638786554336548, 0.5570777058601379, -0.7099272012710571, 0.04687235504388809, 1.3944945335388184, -0.18081873655319214, 0.02664130926132202, 0.8455148339271545, 0.8293693661689758, 1.7758276462554932, 2.4838528633117676, -0.04722326993942261, 0.7839911580085754, -1.7020272016525269, -0.5309033989906311, 2.0727574825286865, 0.15185725688934326, 2.005298137664795, -0.31096935272216797, 0.6075997352600098, 0.2619476318359375, -0.14333707094192505, 2.603750705718994, -0.9855597615242004, 2.4470596313476562, -0.3016476333141327, -1.359459400177002, 0.5175009965896606, -0.2777208685874939, 2.1389992237091064, 1.6237419843673706, 0.646165132522583, -1.4023011922836304, 1.1323063373565674, -0.13445904850959778, 0.7766822576522827, 0.5225780010223389, -0.384433388710022, 0.7177348136901855, -1.8165205717086792, -0.5948636531829834, -2.989561080932617, 0.11260248720645905, 1.6059319972991943, 1.5132596492767334, -1.119703769683838, -5.069546699523926, -1.6726770401000977, -0.2689308226108551, 0.07549657672643661, 2.079099655151367, 0.2863647937774658, 1.0379222631454468, -0.7548539042472839, -0.6015360951423645, -1.507971167564392, 1.5524623394012451, 1.6284291744232178, -1.137526512145996, -0.4962645471096039, -0.7176373600959778, 1.4171431064605713, 0.9151485562324524, 0.9903550148010254, -0.23412883281707764, 3.3282999992370605, -0.24190177023410797, -0.8741171360015869, 0.22057026624679565, 0.06840915232896805, -0.5219816565513611, 1.1248090267181396, -2.093654155731201, 1.8204386234283447, -1.1581809520721436, 2.3199918270111084, 2.068453788757324, -0.1600429117679596, -0.8877873420715332, -1.3165068626403809, 1.946984052658081, -0.4961589574813843, -1.3231916427612305, 0.3380890488624573, 1.2193692922592163, -0.7509329915046692, -0.035234928131103516, -0.9640364646911621, 1.280287265777588, -0.5753450393676758, -0.891319990158081, 0.8975345492362976, 1.845088005065918, 0.24600231647491455, -1.6809567213058472, -1.851570963859558, 1.2924339771270752, 1.2004928588867188, 0.7078837156295776, -0.39352744817733765, -1.227346420288086, -0.18751847743988037, -0.2829705476760864, 1.3536957502365112, -0.2293066382408142, -1.2845474481582642, 0.6353899240493774, 0.8184415102005005, 1.9896204471588135, -0.38927409052848816, -0.9810636043548584, -3.3723604679107666, 1.6188305616378784, -2.622635841369629, -0.9423233866691589, 1.7598689794540405, -0.12744581699371338, 1.8555214405059814, -2.1760482788085938, -4.550732612609863, -0.33273395895957947, -2.0624730587005615, 2.6601109504699707, -0.01781553030014038, -1.182494878768921, -1.666357398033142, 0.12337350845336914, 0.3799867630004883, -2.3934123516082764, -0.03562769293785095, 0.44915708899497986, -1.2467466592788696, -0.23389531672000885, -0.5088773965835571, -1.6202077865600586, -0.9262374639511108, 0.08964097499847412, -2.504061698913574, 0.14412212371826172, -1.0784106254577637, -2.405651092529297, -0.6642558574676514, 1.1140385866165161, -2.0867462158203125, 1.8722871541976929, 1.1719450950622559, 1.6042510271072388, -0.9377603530883789, 2.212841510772705, 0.2279178500175476, -0.4691268801689148, 0.6669958233833313, -0.289539635181427, 1.412907600402832, 2.185781955718994, 0.6893883943557739, -1.5899477005004883, -0.7701324224472046, 0.25237536430358887, -0.6596035361289978, 0.6875513792037964, 1.9850772619247437, -2.486785411834717, -2.1324245929718018, -2.32999587059021, 0.4346105456352234, 1.596311330795288, -0.00948876142501831, 2.8741185665130615, 0.29951590299606323, 0.9012940526008606, -0.7705481052398682, 3.7511937618255615, 0.8831681609153748, -0.21859416365623474, 1.7690130472183228, -0.04050165414810181, 2.118621349334717, 0.6954258680343628, 1.6517083644866943, -0.47778958082199097, 0.646884560585022, -1.6902899742126465, -1.566887617111206, -1.9690696001052856, 0.8085982203483582, 0.3839067220687866, 0.5473626852035522, -0.6411335468292236, 0.33875513076782227, 0.3519430160522461, 0.725322961807251, 2.3373024463653564, 1.2274532318115234, -0.9755339026451111, -0.6845762133598328, -1.0036017894744873, 0.3115055561065674, 0.933154821395874, -0.8484429717063904, -1.6335690021514893, 1.6852562427520752, -1.187937617301941, 2.0429155826568604, 0.06462981551885605, -0.16265451908111572, -1.6939252614974976, -1.9845378398895264, 2.3379271030426025, 0.2972627878189087, 0.8307873606681824, 1.5413440465927124, -2.362607479095459, -0.5600401163101196, -1.099797248840332, 2.135586738586426, -0.28747856616973877, -1.982151985168457, 0.7763972282409668, -1.195698857307434, -2.0629327297210693, 1.0689325332641602, -0.7102874517440796, 1.7817051410675049, 2.7034895420074463, 1.529863715171814, -2.276029348373413, -1.7203490734100342, -0.006332829594612122, -1.9580169916152954, 4.230957984924316, -0.9339410066604614, 1.4760065078735352, 1.9339057207107544, -1.9737143516540527, -0.6727306842803955, 1.056336522102356, 1.116255521774292, -0.2135251760482788, -0.6955410838127136, -0.3676591217517853, -1.985666275024414, -2.2514255046844482, 0.30197110772132874, -1.5806235074996948, 0.38513684272766113, -0.6917815208435059, -1.306472659111023, -2.289140462875366, -1.2605611085891724, -0.2586193084716797, 0.7644493579864502, -0.16155533492565155, 0.7050316333770752, -2.752366065979004, 0.005985260009765625, -2.0371460914611816, 0.4037441313266754, -1.493412733078003, 2.6614632606506348, 2.636866331100464, -0.6932868957519531, 0.31696581840515137, 0.7259564399719238, -2.007082223892212, -0.4562312960624695, 1.0131641626358032, -2.800353527069092, -1.6107419729232788, 0.6438449025154114, -1.2354220151901245, 1.0839111804962158, 1.3755908012390137, 0.60626220703125, 2.5568900108337402, 1.3086044788360596, 1.1239498853683472, -0.12878693640232086, 1.0234203338623047, -0.0038200318813323975, 0.8944343328475952, 4.329565525054932, 0.14485585689544678, -2.8154244422912598, 0.3182947635650635, -0.7096878290176392, 1.4903291463851929, -0.24395504593849182, 1.6407980918884277, -0.18942779302597046, -1.1097909212112427, -0.414210706949234, 0.08852601051330566, 0.4598689079284668, 1.519017219543457, 3.132889747619629, -0.19802320003509521, 2.1343281269073486, -2.301790237426758, 2.295788288116455, -2.6310973167419434, 1.8804717063903809, -0.6591232419013977, 0.9799503087997437, -0.15491390228271484, -0.11828726530075073, 0.29680007696151733, -1.595240831375122, -3.129467487335205, -0.6126813888549805, -1.4524701833724976, 1.0478651523590088, 0.7781381607055664, -0.8793491721153259, -2.0959954261779785, 1.3666834831237793, -0.5016900897026062, 1.4551945924758911, 1.7749067544937134, -0.950370728969574, -2.0981156826019287, 1.6865006685256958, 2.7917330265045166, 1.1690891981124878, -0.5454986691474915, -1.873507022857666, -1.2145241498947144, -0.10009312629699707, -0.2811708450317383, 0.2329862117767334, -0.6432424783706665, -0.9865038394927979, -0.07307052612304688, 2.0064170360565186, 0.5998300909996033, -1.4010441303253174, -1.327324390411377, -1.1064943075180054, 0.44119900465011597, 2.3920376300811768, 1.783334732055664, -1.336462378501892, 1.2489300966262817, -2.399121046066284, 1.5454236268997192, 0.25019246339797974, -0.8301733732223511, -0.5073658227920532, 1.8220807313919067, -0.939125657081604, -1.7056529521942139, -0.4585585594177246, -2.0509724617004395, 1.9991272687911987, 0.7840035557746887, 0.9627487063407898, -0.8776161670684814, 1.2451075315475464, -1.1375279426574707, -0.42816027998924255, 1.510633111000061, -1.208498239517212, 0.39420607686042786, 1.3649771213531494, -3.2969720363616943, 0.13492423295974731, 4.287249565124512, 0.4833845794200897, -0.17021653056144714, -0.4761233627796173, 2.0937929153442383, 2.5295872688293457, -0.6967141032218933, 1.4424076080322266, 3.7188644409179688, 1.35896635055542, 0.7892596125602722, -0.6953959465026855, -1.2583644390106201, 0.8934878706932068, -0.6751790642738342, 0.051949262619018555, -2.229501962661743, 1.8711299896240234, -2.8381495475769043, 0.4487117528915405, -2.6849608421325684, -1.6742606163024902, -2.7949814796447754, -0.8354220390319824, 0.5243991017341614, -1.0062776803970337, 0.9024261236190796, 1.0363984107971191, -0.5359853506088257, -1.5671920776367188, 0.5931992530822754, -0.5946505665779114, 1.1379942893981934, 1.734569787979126, -0.32177966833114624, 0.47506606578826904, -0.9203808307647705, -1.5198510885238647, 1.7559010982513428, 0.8011870384216309, 0.13626348972320557, 0.9305046796798706, -2.2414448261260986, -0.04132437705993652, 0.7616444826126099, 0.7718702554702759, 0.07701896131038666, 0.5507721900939941, 2.00838565826416, -1.1855148077011108, 0.4051554799079895, -0.38435664772987366, 0.27714037895202637, -0.4261835813522339, 0.6950026750564575, -2.2899293899536133, 1.405797004699707, 0.19943679869174957, 0.20912733674049377, -1.1280534267425537, -0.21889686584472656, 1.4127389192581177, 1.379312515258789, 2.252267837524414, -0.6452475786209106, 0.7347835302352905, 1.969927430152893, 0.40158936381340027, -0.20708802342414856, 2.909755229949951, 0.1938905119895935, 1.5557502508163452, -0.0003280043601989746, -0.4734537601470947, -0.17483364045619965, 1.5714037418365479, 0.012129798531532288, 0.39005768299102783, 1.922512412071228, 1.9248299598693848, 1.3567873239517212, 2.259026527404785, 0.04596376419067383, 0.6554985642433167, 0.3709031343460083, -0.6150476932525635, 2.126481771469116, -0.7127568125724792, 0.6636143922805786, -0.34988531470298767, 0.9777289032936096, -1.4661498069763184, -2.4674346446990967, -0.216930091381073, -1.187790036201477, 0.07355904579162598, 1.6989023685455322, -1.6669745445251465, 1.6728541851043701, 1.0942906141281128, -0.12380674481391907, 0.43233615159988403, -0.22602690756320953, 0.5003291964530945, -0.22259166836738586, -2.176741361618042, -2.9743428230285645, 2.382580280303955, -0.6678516864776611, 1.366213083267212, -0.10875194519758224, 1.5956478118896484, 0.8918801546096802, -0.5852817893028259, -1.926169514656067, 0.7327066659927368, 1.3607916831970215, -1.6472605466842651, 1.0437015295028687, -1.2015950679779053, -0.06832718849182129, 0.8789188861846924, -1.536695957183838, -1.5247844457626343, 1.5521032810211182, -0.9601119756698608, 2.7583837509155273, -0.18705420196056366, 1.0031213760375977, -0.8831393122673035, -0.585894763469696, -2.293562650680542, -0.7218964099884033, -1.5889854431152344, 1.0560262203216553, 0.30303817987442017, -1.2150726318359375, 2.4621050357818604, 1.2007322311401367, 0.5680207014083862, 0.004033345729112625, 1.723401665687561, 1.364968180656433, -1.5212653875350952, -0.9833055734634399, 0.5550869703292847, -4.143820762634277, 1.3291645050048828, 1.8247754573822021, -1.490769386291504, 0.2009466588497162, 1.1536352634429932, -0.5639316439628601, -0.7932636141777039, -1.2711169719696045, -0.07937633991241455, -1.2989130020141602, -0.4106234014034271, 0.18197107315063477, 0.61050945520401, -2.6044607162475586, 1.084332823753357, 4.670408248901367, -1.1818095445632935, 1.2830626964569092, 1.0435006618499756, -0.7906584739685059, 1.8914523124694824, 2.091776132583618, -0.8137500286102295, -0.6995202302932739, 1.7361512184143066, 0.6414687037467957, -0.8189244270324707, 0.26392942667007446, 0.24327312409877777, -2.94020938873291, 0.5619706511497498, 0.3834887742996216, -2.706912040710449, -1.6240637302398682, 0.9548484683036804, -0.434337317943573, -0.3530370891094208, -2.1648337841033936, -1.7601251602172852, -0.25345513224601746, 0.6508294343948364, -1.1880494356155396, -0.8318365812301636, -0.29351747035980225, 1.5189629793167114, -0.9034502506256104, 1.439029335975647, -1.8117846250534058, -1.0751368999481201, 0.506189227104187, 2.55615234375, 1.678302526473999, 0.4598081111907959, 0.5462870001792908, 0.10411560535430908, 1.1226826906204224, 3.507981300354004, -1.7203712463378906, -2.6402945518493652, 0.8081372976303101, -0.4732603132724762, -1.6276068687438965, -2.326416254043579, -0.26559871435165405, 2.6039533615112305, -1.8134758472442627, -1.4621950387954712, -2.3625941276550293, 0.932202935218811, 1.2199883460998535, -0.6217129230499268, -0.0627601146697998, 0.32624703645706177, -1.1284230947494507, -0.5480093955993652, 1.3500680923461914, -0.9262924790382385, 0.06873750686645508, 3.4619503021240234, -1.520284652709961, -0.2877286374568939, -0.49079787731170654, -0.4851946234703064, -1.1246765851974487, 0.6525991559028625, 2.1605749130249023, -1.315302848815918, 1.589933156967163, -0.354091614484787, 0.05288463830947876, 0.8041313290596008, -0.7314024567604065, 0.05163717269897461, -1.0613986253738403, -1.5369153022766113, -0.4086003005504608, 1.9486209154129028, -0.9350942373275757, 0.9360156655311584, 1.4000555276870728, -0.39146125316619873, 0.22706085443496704, -1.0952558517456055, 0.2643220126628876, -1.4041783809661865, 0.47539350390434265, 1.221077561378479, -2.409686803817749, -0.8053419589996338, 1.5796537399291992, -1.052148699760437, -0.6535471081733704, -0.41877543926239014, -2.1655025482177734, -0.133478581905365, 2.168260097503662, 0.3029100000858307, -1.7093827724456787, -1.432852864265442, 1.4411197900772095, 1.7337344884872437, 0.8911699652671814, 0.6187102794647217, -1.9529868364334106, -2.1533427238464355, 0.57303786277771, -0.14067614078521729, -0.643052339553833, 1.023381233215332, 0.9370900392532349, 0.7971317172050476, -0.7454619407653809, -1.031428337097168, 0.32198911905288696, -0.43754518032073975, 0.8884122371673584, 0.5489546060562134, 2.2922115325927734, -0.8503516912460327, -0.05792808532714844, 0.9721929430961609, 1.9247715473175049, 2.982290744781494, -0.03582358360290527, 0.9860129356384277, -0.7177811861038208, 2.1229939460754395, 0.7195428609848022, 0.7251715660095215, 0.5814707279205322, 0.700049638748169, -3.3846333026885986, -1.9375629425048828, 0.7329085469245911, -0.5248867273330688, 1.8580843210220337, -4.208757400512695, -0.613734245300293, -1.6267309188842773, -1.5870757102966309, -0.013446405529975891, -0.8089133501052856, -1.5987622737884521, -0.9497380256652832, -1.747769832611084, -1.1995301246643066, -1.1599947214126587, -0.8218619227409363, 0.4752500057220459, -2.7214725017547607, 0.3341294825077057, 0.048234134912490845, -0.5798108577728271, 0.5825786590576172, 0.10175195336341858, 1.740409016609192, 1.0015380382537842, 0.7873618602752686, -3.5703036785125732, -1.0059165954589844, 0.30992811918258667, 1.9237115383148193, -1.7881758213043213, 0.3066660463809967, -0.18739479780197144, 1.2259793281555176, -0.5858339071273804, 0.5229811668395996, -0.025522083044052124, -1.2945575714111328, -0.32633328437805176, -2.3706955909729004, 1.9592280387878418, 0.11582109332084656, 3.5909669399261475, 1.745664119720459, 1.938014268875122, -2.5658419132232666, -1.342179536819458, -3.7763164043426514, -1.6213797330856323, -0.9506555199623108, 0.4510265588760376, 0.0902484804391861, -1.582127332687378, 2.0212619304656982, -1.7988219261169434, -0.47153401374816895, -0.3426225185394287, -0.10821801424026489, 0.8935943841934204, -2.1559019088745117, -0.1834559142589569, -0.5108307003974915, 0.9702659845352173, 1.3051331043243408, 1.0670171976089478, -2.616568088531494, 0.014461427927017212, -2.139374256134033, 0.39022397994995117, -2.5017008781433105, -1.2476415634155273, -0.13721248507499695, 2.2114884853363037, 1.568979263305664, 0.1597389280796051, 0.712041974067688, -1.0311260223388672, 1.2642251253128052, -1.1324131488800049, -0.7164030075073242, 0.6462594270706177, 0.6692622900009155, -2.0900449752807617, 0.38374602794647217, -1.1103726625442505, 1.341850996017456, -1.0746967792510986, -0.24414807558059692, 1.4946486949920654, -1.15519118309021, -0.6612513661384583, 0.0679730772972107, 0.47141754627227783, -1.5995757579803467, 0.3301335573196411, -0.8802344799041748, -1.0193861722946167, -2.042560577392578, -0.06664814800024033, -0.6258167028427124, 0.1085059642791748, 1.5169895887374878, 0.2436339557170868, -0.4407697916030884, -1.322333812713623, -0.17322981357574463, -0.05369991809129715, -0.44053971767425537, 0.04112172871828079, 1.0430774688720703, -1.5680265426635742, 0.9300529956817627, 1.5799542665481567, 0.45231693983078003, 1.610658049583435, 0.15699943900108337, -2.674492359161377, -4.995870590209961, -1.1530297994613647, -0.28782257437705994, 1.127898097038269, 1.953202486038208, -0.41596972942352295, 2.2024972438812256, -2.527813673019409, 1.9824538230895996, 0.44278863072395325, 1.6545275449752808, -0.941839337348938, -0.48241233825683594, 1.1863640546798706, -0.9549257755279541, -2.0506463050842285, -0.17320656776428223, -1.9429757595062256, -0.170890212059021, -1.588224172592163, -1.1363499164581299, -2.0740065574645996, -1.89287531375885, 0.5785757899284363, 2.2619268894195557, 1.9980804920196533, 1.0065741539001465, -0.45119160413742065, 0.9954367876052856, -1.009942650794983, 1.2552218437194824, 3.888637065887451, -0.5347512364387512, -0.5581666231155396, 2.585358142852783, -1.866820216178894, -0.2031731903553009, -0.687864363193512, -0.38744664192199707, 0.5743329524993896, 0.3387832045555115, 2.120957612991333, -1.3514963388442993, -0.4585241973400116, -0.9457842111587524, -2.3702754974365234, -2.274113178253174, -1.5835375785827637, -0.4024372100830078, -0.44382813572883606, 0.7439562082290649, -0.021924227476119995, -2.8783745765686035, 0.03367660939693451, 1.7169133424758911, 1.269024133682251, -2.6357922554016113, -0.5965174436569214, -1.01295804977417, -2.1078226566314697, 0.5360747575759888, -0.5413599014282227, 0.9520139694213867, 1.4374467134475708, 2.077585220336914, -1.8033027648925781, -1.2661628723144531, 0.23170173168182373, -1.6050430536270142, 0.24837720394134521, 0.6645628213882446, 1.6714177131652832, -0.8110636472702026, 0.5023951530456543, 1.2351908683776855, 0.18002107739448547, 1.774510383605957, 0.47100499272346497, 0.4149054288864136, 0.39221739768981934, 0.4162144064903259, 0.5576344728469849, 0.5229402780532837, -3.7458746433258057, 1.726983666419983, 1.852721095085144, -0.4930274486541748, -0.8746713399887085, 0.06049540638923645, 1.2244688272476196, 1.1154003143310547, 1.0681939125061035, -0.2878186106681824, 2.199582099914551, -2.235764980316162, 0.011318206787109375, 0.4284290671348572, 0.07666795700788498, -0.07710389792919159, -0.6913555264472961, 1.0722731351852417, -1.6756324768066406, 1.20881986618042, -1.3024784326553345, -3.2808046340942383, -0.07959823310375214, -1.4084596633911133, 0.2218514084815979, -0.22038286924362183, -1.9038335084915161, -0.035359472036361694, -1.423094391822815, 0.6106492280960083, -1.3898859024047852, -0.5595934987068176, 1.2291350364685059, 0.8926243185997009, 0.8454269170761108, 1.5417628288269043, -0.8674739599227905, -1.2792123556137085, -0.4090886116027832, 1.1177995204925537, -1.2146613597869873, 0.08218589425086975, 0.6184732913970947, 2.035473585128784, 0.573899507522583, -1.2489795684814453, 0.6520251631736755, 0.33598464727401733, 1.0391676425933838, -0.9893502593040466, -0.40906351804733276, 2.0018744468688965, -0.321789026260376, -2.2083702087402344, 0.9414455890655518, -1.3317418098449707, 2.0230391025543213, 1.8621127605438232, 0.6316410899162292, -0.23146581649780273, 0.3826233148574829, 1.4238947629928589, 1.2854773998260498, -1.3499325513839722, 0.27528807520866394, -2.4812026023864746, 1.6638127565383911, 3.315903902053833, -0.9751237630844116, -0.11012458801269531, 1.398911476135254, 0.517044186592102, 0.3328571617603302, -1.8667938709259033, -1.1533770561218262, 2.6662702560424805, -0.38621988892555237, -0.6565737724304199, -4.003831386566162, -0.5395869016647339, 0.37403857707977295, -0.5607388615608215, -3.103320837020874, 0.3845478892326355, 1.1090182065963745, -0.8511208891868591, 0.7969807386398315, 1.3449312448501587, 0.32914870977401733, 1.9407553672790527, 0.11513489484786987, -0.7669739127159119, 0.48972517251968384, -0.3281139135360718, -1.691718339920044, -3.765313148498535, -0.4532424807548523, 0.4001803696155548, 0.8372480869293213, 2.4369425773620605, 0.19200465083122253, 0.16423183679580688, 2.2229912281036377, -0.26251429319381714, 1.7734854221343994, -3.2380216121673584, 0.3695220351219177, 1.6299477815628052, 0.3471657335758209, -1.2173652648925781, 1.6268056631088257, -1.9569566249847412, -1.245710015296936, -0.7395870685577393, -0.6178725957870483, 1.1186070442199707, -0.9970671534538269, -0.16844916343688965, 2.068729877471924, -1.0349390506744385, 1.7345647811889648, -1.4093749523162842, 1.1119730472564697, -1.2691794633865356, 1.0201020240783691, 0.7188212275505066, -0.05181151628494263, 1.0305967330932617, -0.4256535768508911, -1.4217406511306763, -1.7467538118362427, 1.1411492824554443, 0.8832728862762451, 0.44921064376831055, -1.0954722166061401, 2.4193389415740967, 1.391411304473877, 1.3377888202667236, -1.7720271348953247, 2.4751663208007812, -0.22584611177444458, -0.026498794555664062, 3.650336742401123, -0.5267930030822754, -0.2682950496673584, -1.0585603713989258, 0.341213583946228, -1.644668698310852, -2.845188856124878, 1.462615966796875, -1.2993183135986328, -3.116576671600342, -0.6013433933258057, 1.067232370376587, -1.8639734983444214, -1.6247214078903198, 1.164803385734558, -1.395742654800415, -0.6388236284255981, -0.9217350482940674, 0.943053126335144, -0.5558834075927734, -0.43065163493156433, 1.0065550804138184, -1.8594083786010742, -2.974165439605713, 1.4223875999450684, -0.5772985816001892, 1.4589674472808838, -0.7197327613830566, -1.8575453758239746, -0.4432384669780731, 0.2928984761238098, 0.5050848722457886, -1.2344731092453003, 1.1535334587097168, 0.14203566312789917, 2.9305105209350586, 0.9010694026947021, -1.7367191314697266, -1.5518763065338135, 1.8631163835525513, 0.8990378379821777, 0.6828693747520447, -0.9275122284889221, 0.46007436513900757, -1.109897494316101, -2.0668797492980957, 2.3882274627685547, -0.32974034547805786, 0.4821931719779968, 0.7954214811325073, 2.1723341941833496, -2.119795799255371, -1.2334307432174683, 1.0310308933258057, 0.68321293592453, 0.5544326901435852, -1.0026350021362305, 0.7585065364837646, -2.2090375423431396, -1.3145784139633179, 2.126101493835449, 0.5208079814910889, 0.98492431640625, 0.699012279510498, 1.0555182695388794, 1.6759705543518066, -0.023074686527252197, -2.8880109786987305, -2.707490921020508, -1.1846063137054443, -2.30995774269104, -2.8814265727996826, -2.561805248260498, 1.4460084438323975, 2.297118663787842, -1.3130214214324951, 2.2820353507995605, -0.5591933727264404, -3.065707206726074, -0.5339510440826416, 0.4676859974861145, 1.219756841659546, -0.7317727208137512, 0.16626408696174622, -0.30405497550964355, -3.096750259399414, 1.3190009593963623, 2.4102835655212402, -0.6710953712463379, 0.5945258736610413, -0.8840153217315674, 1.6259260177612305, 0.2527642846107483, 0.5653978586196899, 1.7715039253234863, 1.3983221054077148, -0.46783313155174255, 2.33048152923584, 0.4724227786064148, -0.3008246421813965, -1.2392219305038452, -0.05740630626678467, 0.4682438373565674, 0.052101969718933105, -0.42770588397979736, -0.6560842990875244, -2.152337074279785, -1.2982934713363647, -1.9818516969680786, 0.5961595773696899, 1.2768324613571167, -2.1392552852630615, -0.17676639556884766, 1.502516508102417, -0.5191054344177246, 2.7985663414001465, -0.8397377729415894, 1.733381986618042, -2.2064919471740723, -1.3106791973114014, 1.2629828453063965, 0.5467653274536133, 1.5255510807037354, 0.7940191030502319, -1.274923324584961, 0.2208234816789627, -0.3219599425792694, -0.7962069511413574, -0.4705159068107605, -1.4104812145233154, -0.11424902081489563, -0.7279995679855347, -1.2931594848632812, -1.5142756700515747, 1.4574980735778809, -3.1857717037200928, 1.0281493663787842, -0.17003357410430908, -1.352689266204834, -0.042720913887023926, -0.2075384557247162, -1.7603986263275146, 3.3698458671569824, -2.6727683544158936, 0.14688464999198914, -0.09055298566818237, -1.010259985923767, -0.7634789943695068, 0.3601783215999603, 1.0330207347869873, 1.6410859823226929, 1.2096360921859741, 0.9549503326416016, 1.768411636352539, 0.10224440693855286, 0.7543962001800537, -0.5284982919692993, 0.5417704582214355, -1.0428283214569092, 0.6864778995513916, 0.8278602361679077, 2.2930612564086914, -0.32034724950790405, -0.30069345235824585, -0.9940766096115112, 0.8611167669296265, -0.04474741220474243, -0.10544189810752869, 0.2747170329093933, -4.314388751983643, -0.8220838308334351, 0.8436269760131836, -2.398054599761963, -0.7481403946876526, -0.6303500533103943, -1.2910926342010498, 0.9421731233596802, -0.3527902364730835, -0.6314796209335327, 3.340590476989746, 0.21082156896591187, -0.23421967029571533, -1.8898050785064697, -0.39557579159736633, 1.9334274530410767, -2.0082614421844482, -1.8395870923995972, 2.9612557888031006, -1.6545964479446411, 1.8223459720611572, 1.231710433959961, -0.044739216566085815, -1.9579308032989502, 0.9684475064277649, 1.5617460012435913, 2.091078996658325, 1.0190088748931885, -0.09231750667095184, 0.7283725142478943, -0.11569657176733017, 1.1637790203094482, -1.855674147605896, 0.1319735050201416, 0.7840396761894226, -0.49472522735595703, 1.4293766021728516, -1.2826625108718872, -0.15169364213943481, -0.5358684062957764, -0.07088714838027954, 0.13109391927719116, 1.3179397583007812, -0.9163110256195068, 2.449746608734131, -1.4304091930389404, 1.6730040311813354, 1.683409333229065, 2.551941394805908, -0.994359016418457, 1.2189528942108154, 1.6002728939056396, -1.9326226711273193, 2.172257423400879, 0.9835407733917236, -0.5287001729011536, 0.23950737714767456, -1.9304730892181396, -0.22734791040420532, -0.8438129425048828, -0.4952673614025116, -1.6351736783981323, -1.8931938409805298, -0.4868829846382141, -2.244023084640503, -6.456735610961914, 0.3920205235481262, 1.193680763244629, 1.3097604513168335, 0.356253445148468, 0.535873532295227, -0.17161405086517334, -1.3806431293487549, -1.6666467189788818, -1.9432207345962524, -0.8797348141670227, 1.6670666933059692, 0.2375861406326294, 1.1853471994400024, -3.153434991836548, -1.2427005767822266, 0.011738628149032593, 1.6232478618621826, -1.4801777601242065, 1.5067124366760254, 1.9829193353652954, -1.8787907361984253, -0.21786296367645264, -0.19051426649093628, 0.6913711428642273, 0.5723493695259094, -0.8398901224136353, -2.419051170349121, -1.5435494184494019, 0.2713099420070648, 1.1443663835525513, -0.2807883620262146, -0.614443302154541, 2.319444179534912, 0.6855112910270691, -0.050364136695861816, -0.28499695658683777, 2.9117116928100586, 0.20112141966819763, 0.13772374391555786, 0.2078155279159546, 3.105043411254883, 1.9470384120941162, 1.7339611053466797, -1.6599977016448975, -0.4685456156730652, 0.0756290853023529, 1.6255853176116943, -1.2981563806533813, -1.975121021270752, 1.7363897562026978, 0.5868265628814697, -2.283085346221924, 0.17186957597732544, 1.2211228609085083, 3.701096534729004, 0.42694664001464844, -1.3420941829681396, 0.4755159616470337, -0.3409799337387085, -1.7594690322875977, -2.707005023956299, -2.812884569168091, 2.1758856773376465, 2.8056812286376953, 0.05646764859557152, -1.876242995262146, -0.37305688858032227, -0.6614951491355896, -1.9978482723236084, -1.1537715196609497, -0.22337999939918518, -0.40812984108924866, 1.425032377243042, 3.2227377891540527, 2.705639123916626, -2.969735622406006, -0.6325386762619019, 0.8346148729324341, -0.9138648509979248, -0.12686681747436523, 0.03911798447370529, -0.7361671924591064, -3.112107753753662, 0.21033263206481934, -0.0800289511680603, 1.3498597145080566, 1.2558053731918335, 0.6941240429878235, 0.029566705226898193, 1.1596320867538452, 1.5010299682617188, 0.9245402812957764, 1.9114058017730713, -1.5113205909729004, 2.15301251411438, 0.568284809589386, -1.0349725484848022, -1.428708553314209, -1.1526998281478882, 0.3700743615627289, -0.298414945602417, 1.330154538154602, 2.8098673820495605, 1.196163296699524, -1.9159691333770752, -0.481151282787323, 0.6267126202583313, 0.5254009366035461, -0.2662584185600281, 0.12935060262680054, -2.0832834243774414, -0.35873305797576904, -0.14926479756832123, 0.5179349780082703, 1.757690191268921, -0.2911474108695984, 0.8087119460105896, -0.5013756155967712, 0.8663451671600342, -0.7362581491470337, 1.3211097717285156, 0.6217159628868103, -0.3676571547985077, 0.09582066535949707, -2.7666354179382324, -0.6468439698219299, -2.0514001846313477, 1.6295995712280273, 2.1413402557373047, -1.9859853982925415, -0.011709973216056824, -1.0225553512573242, 0.08860033750534058, -0.7273263335227966, -1.9364755153656006, -0.7417475581169128, -0.32313936948776245, 1.9712114334106445, 1.8127902746200562, 2.5469980239868164, -1.4770088195800781, 0.8491518497467041, -2.1339426040649414, -0.9302160739898682, -1.171428918838501, -1.383674144744873, -3.4456710815429688, 0.0693206787109375, 0.4842391908168793, -3.130519390106201, -2.6653082370758057, -0.49648231267929077, 0.5971675515174866, 1.0916107892990112, 1.4602895975112915, 0.1500651240348816, 2.0220491886138916, 1.042454481124878, 0.6092417240142822, 0.049925193190574646, 0.3613779842853546, 1.0650126934051514, 1.6161009073257446, -0.16684579849243164, -0.6441653966903687, 1.3873841762542725, 0.7800154685974121, -0.8997890949249268, 0.24555563926696777, -2.405804395675659, 1.4555745124816895, -0.1790153682231903, -0.17937910556793213, -0.5338398218154907, 0.12139201164245605, -0.21309705078601837, 3.033159017562866, 3.4974911212921143, 0.35971522331237793, -0.8546490669250488, 0.4343700408935547, -2.0757713317871094, 1.5557715892791748, -0.3216674029827118, -0.051986098289489746, -1.9728835821151733, -0.9818578958511353, -0.18354231119155884, 2.5794451236724854, 0.614340603351593, 1.1890827417373657, 2.183379650115967, -0.08885439485311508, 0.2497202754020691, -2.9758124351501465, 1.8020238876342773, -0.2910022735595703, 0.8196223974227905, -0.37519824504852295, -0.33958005905151367, -1.1242311000823975, -0.5381587147712708, -0.12324774265289307, 2.735686779022217, 2.3511650562286377, -2.2517940998077393, 0.13715189695358276, 1.070558786392212, 2.539708137512207, 0.558236837387085, 0.9073164463043213, 1.5896425247192383, 0.056428298354148865, 1.123211145401001, -0.5127130746841431, -0.7466019988059998, -1.249079942703247, -0.5571720600128174, -0.21344590187072754, -1.0170522928237915, 0.6812484264373779, 3.4787869453430176, -1.209789514541626, -1.8681241273880005, 0.19484424591064453, 0.7596338391304016, 2.1070261001586914, -1.0648339986801147, 1.9388148784637451, 1.3081004619598389, 0.5983049869537354, 1.8118669986724854, -0.5203604102134705, 0.5786735415458679, 1.7484146356582642, -1.097822666168213, -0.4930928647518158, 2.195295810699463, -0.7569248080253601, -2.700651168823242, -2.935201644897461, -1.204036831855774, -2.0529937744140625, 0.5722101926803589, 0.0053157806396484375, -0.7572216987609863, 3.1282706260681152, 0.09509223699569702, -2.9294371604919434, 1.4032435417175293, -0.29713141918182373, 0.720484733581543, 0.05258874595165253, -1.380865216255188, 1.6330013275146484, -0.9609695076942444, -0.7118787169456482, 0.2765567898750305, 0.03500311076641083, -1.522385597229004, 2.611191749572754, -0.4270700514316559, 0.9112657904624939, 0.6644930839538574, -0.6156523823738098, -1.3644435405731201, -1.4301074743270874, 0.6673677563667297, 1.0299181938171387, 0.3204677104949951, 2.4693076610565186, 0.22048112750053406, 1.7205907106399536, 0.8541843295097351, -0.33035656809806824, 0.9049938321113586, 0.3691691756248474, -0.12381082773208618, -0.5441687107086182, 1.2611159086227417, 0.32564666867256165, -2.0849735736846924, 0.011618256568908691, 0.9415737390518188, 0.1311633586883545, 1.8841540813446045, -3.2614006996154785, -1.0061254501342773, 1.3791868686676025, 1.5678648948669434, -1.5230813026428223, -0.6102575659751892, -0.24138188362121582, -2.075448989868164, -0.798337996006012, -1.2657363414764404, 0.8442288637161255, 2.7367467880249023, -1.9760280847549438, -1.9272847175598145, -0.7177184820175171, 0.5660536885261536, -2.702944278717041, 1.0122578144073486, -1.2750777006149292, 0.9673914909362793, 0.41875872015953064, -0.294059693813324, 2.5970659255981445, -5.462571144104004, 2.261824369430542, -0.9918087720870972, 1.072401523590088, -0.691381573677063, -1.9826322793960571, 1.1682977676391602, -0.6391347646713257, -1.423720359802246, 0.47285014390945435, 0.26269030570983887, 0.530734121799469, -0.6980558633804321, 1.717914342880249, 0.5198430418968201, 1.887772798538208, 0.3648209273815155, -0.3580460548400879, 0.13619667291641235, -0.6148135662078857, -0.6881627440452576, 1.7913886308670044, -0.4958338439464569, 1.4148335456848145, 1.558974027633667, 2.785682201385498, 3.252775192260742, 2.0731022357940674, 0.6219213604927063, -0.2969595491886139, 0.2771434783935547, 0.4857640266418457, 0.9458046555519104, -0.223239466547966, -0.1499059796333313, -2.598355770111084, 2.6699135303497314, 0.4729013442993164, 1.1631147861480713, -0.6604599952697754, 0.08819735050201416, -0.9097098112106323, 1.1412467956542969, 2.5063679218292236, 0.3101217746734619, -1.1853275299072266, -0.11260861158370972, -0.3911261558532715, -1.2006778717041016, -1.7145142555236816, 1.450002670288086, 1.8650963306427002, -0.1975809782743454, 1.066882848739624, -0.5913838744163513, -1.3930951356887817, 0.18737977743148804, 1.6691577434539795, 0.013085126876831055, -1.533186435699463, -0.22201725840568542, 0.15281987190246582, 0.4514387249946594, -0.2603900134563446, -1.7964091300964355, 1.061503291130066, 0.8258803486824036, 0.3043404519557953, -0.7975068688392639, -1.6168979406356812, 1.1567091941833496, 1.666562795639038, -1.8953654766082764, 0.8837270736694336, -0.32004693150520325, 0.8518166542053223, 3.0843722820281982, -0.7376018762588501, 2.084320068359375, 0.26285475492477417, 0.03074014186859131, -1.525274634361267, 0.7641025185585022, 0.5499643087387085, -1.533476710319519, 1.9081177711486816, -1.006322979927063, -0.10614076256752014, -1.8422768115997314, -0.09577465057373047, 0.624211847782135, -1.777823805809021, -1.343706727027893, -4.149719715118408, 0.14114749431610107, 2.3820862770080566, 1.8835527896881104, -1.4535640478134155, 0.37372076511383057, 0.8138757944107056, 0.14737403392791748, 1.3188652992248535, 1.3763926029205322, -1.075722336769104, 1.1323626041412354, -0.6392415761947632, -1.1192795038223267, 0.30565977096557617, -0.6356704831123352, 1.9446730613708496, 0.29654332995414734, 2.8299479484558105, -1.1896826028823853, 0.7198854684829712, 1.3184641599655151, 0.3818022608757019, 1.6009951829910278, -2.0389246940612793, -1.3027031421661377, -0.5986006259918213, 0.02743053436279297, 1.8602445125579834, -0.3617010712623596, 2.73382306098938, -0.7497979402542114, -0.10351766645908356, 0.27577805519104004, 1.3821825981140137, 1.0876495838165283, -0.13146615028381348, -1.4036420583724976, -0.5214476585388184, -0.3118849992752075, -1.6618404388427734, 1.5342292785644531, -0.6994410753250122, 0.9076277613639832, 0.6198323965072632, 2.1222195625305176, -0.4012365937232971, 1.4809406995773315, 0.42387655377388, 2.3997716903686523, -3.2239444255828857, 0.2438134402036667, 1.1029917001724243, 0.7139650583267212, -2.455092430114746, 3.6863577365875244, 0.110057532787323, 0.9854090213775635, -1.107473373413086, -1.4219441413879395, -1.0889134407043457, -2.632026195526123, -0.3410240411758423, 0.2162906527519226, -1.5507913827896118, -0.0723576545715332, -1.654737114906311, -0.3276086151599884, -0.3578982651233673, 0.43684494495391846, 0.5347230434417725, 1.2797768115997314, 0.3687422573566437, -0.5868213176727295, 0.5643643140792847, 1.6619617938995361, -2.0760397911071777, -1.5961390733718872, 0.9352288842201233, -1.8283377885818481, -2.482785224914551, -0.4868144392967224, -0.9647525548934937, 0.7320122122764587, -1.1440051794052124, -0.6041222810745239, -1.8265876770019531, -0.8613961935043335, -1.222448468208313, 0.7395631074905396, 2.2504706382751465, 0.7250438332557678, 2.0246334075927734, 0.9681410789489746, -1.1154515743255615, 0.16730892658233643, 1.7940608263015747, 1.9990737438201904, -0.5217090845108032, -0.6414477229118347, -1.6419343948364258, 1.6288974285125732, -1.5842978954315186, 0.10747668147087097, -0.5894016623497009, -0.9678972959518433, 3.129836082458496, -0.1996973156929016, 1.4092381000518799, 0.06970906257629395, 1.1551127433776855, -0.3374989628791809, 0.18752917647361755, 0.2978972792625427, -0.8277357220649719, 1.4079936742782593, 2.969231128692627, 0.3403549790382385, -0.3801504969596863, 0.2472299337387085, -2.0028843879699707, -1.578017234802246, -0.314811110496521, 1.3261425495147705, 1.7641979455947876, 0.0770573616027832, -1.006138801574707, -0.1744784712791443, -0.4881296753883362, 1.9438295364379883, 3.428191900253296, 2.9323227405548096, 1.6774351596832275, -0.495354026556015, 0.4545140564441681, -0.41724076867103577, -2.6902270317077637, 0.015086531639099121, 0.5605655908584595, 2.2907705307006836, -0.015738844871520996, 0.055707305669784546, 1.7400152683258057, -2.399678945541382, 1.2594338655471802, -1.736742615699768, -0.7888857126235962, -0.06217217445373535, 0.9318201541900635, -0.6987396478652954, -1.7585595846176147, -2.7133631706237793, -0.9120229482650757, -0.7678717374801636, 0.897707998752594, 1.6471186876296997, 0.31505632400512695, -1.0775771141052246, -0.8044649958610535, -1.5453654527664185, -2.1893527507781982, 0.2696933448314667, -1.086982011795044, -0.21405141055583954, -0.12077362835407257, -1.1841540336608887, -0.5177897214889526, 0.6617755889892578, 1.9124000072479248, 1.4913313388824463, 0.9167841672897339, -1.9533483982086182, 1.6810593605041504, 0.8617172241210938, 0.47522810101509094, 0.2530967891216278, 0.694624662399292, 0.03638792037963867, 1.0135228633880615, 1.1499049663543701, 2.4186508655548096, 0.41381287574768066, -0.7230344414710999, -0.4502108097076416, 0.8541907072067261, -1.059314489364624, -2.630810260772705, -0.814577043056488, 0.4111560881137848, -0.029207617044448853, 0.9698578119277954, -1.039994239807129, -1.0176111459732056, -1.7206666469573975, -0.3167778253555298, 0.22008079290390015, -0.7054048776626587, 0.31678152084350586, 0.25888943672180176, -0.8530606031417847, -0.23877060413360596, 0.00121346116065979, 1.2906156778335571, 1.8026759624481201, 1.3472989797592163, 1.1078757047653198, 0.5220972299575806, -0.8761957287788391, -0.4381759464740753, -1.6646932363510132, -1.4404079914093018, 1.2840275764465332, -1.3079042434692383, -1.1727263927459717, -2.671788215637207, 2.2294678688049316, 0.4031512141227722, 0.4314296841621399, 2.891704559326172, 0.4998263120651245, 1.7323731184005737, -2.369870185852051, 1.7361021041870117, 1.1739213466644287, -0.344735324382782, -2.158296585083008, 1.0378553867340088, -1.778178095817566, -1.501184105873108, -1.1924805641174316, 1.3405648469924927, 1.5508729219436646, -1.3973292112350464, 1.447945237159729, -1.030712366104126, -0.8109838366508484, 0.3572636842727661, 2.234528064727783, 0.5591208934783936, 0.26435336470603943, 2.7537739276885986, 0.9191831946372986, -2.777892589569092, 1.2388755083084106, -0.472420334815979, 1.0252360105514526, -0.7727937698364258, -1.1921749114990234, 0.09551650285720825, -0.8912845849990845, 0.47411277890205383, 0.46387404203414917, 0.24390101432800293, -0.5329273343086243, 0.7226170897483826, 0.33196717500686646, -0.02151668071746826, 1.7544236183166504, -2.4052538871765137, -0.1950676590204239, 0.4643695056438446, 0.3391478657722473, -0.819438636302948, -0.478520929813385, 0.9729312062263489, -0.7618846893310547, 3.333582878112793, 1.2200465202331543, -1.012851595878601, 2.234194755554199, 1.1824368238449097, -0.15296277403831482, -1.2872868776321411, -1.1435012817382812, -0.2830033004283905, 0.3856269121170044, -0.6437726020812988, 0.2552356719970703, 0.8472177386283875, 1.0458379983901978, 0.9350584745407104, 0.8012298345565796, 0.15852388739585876, 0.939533531665802, -1.4418359994888306, 0.036313652992248535, -1.1997202634811401, 1.002573013305664, -1.9991543292999268, 1.058039903640747, 2.724440097808838, 3.8438024520874023, -1.0370286703109741, -0.5859021544456482, 0.6617723703384399, 2.322324752807617, 0.06366145610809326, 2.9178879261016846, -2.1135647296905518, -0.9731633067131042, -1.5698022842407227, -1.2323514223098755, -0.08768594264984131, -0.997236430644989, -0.9285836815834045, 1.9501876831054688, -0.10019619762897491, -0.47598499059677124, 0.062268905341625214, -1.6499117612838745, -0.2373674511909485, 0.6829709410667419, 0.7724079489707947, -0.6592003703117371, -1.8073328733444214, -0.5327754020690918, -0.45599302649497986, -1.274616003036499, 0.3162122368812561, 2.264014720916748, 0.35936129093170166, -1.50045645236969, 0.5994826555252075, -0.37836116552352905, 0.09720039367675781, 1.0219318866729736, 0.9446695446968079, 1.0962626934051514, -1.7072393894195557, -2.029400587081909, -0.4909598231315613, 1.014571189880371, 0.8977118134498596, 1.1480910778045654, -1.9442052841186523, -3.634669303894043, 0.4078911542892456, 0.5254437327384949, -1.51145339012146, -0.07815277576446533, 0.8346426486968994, -1.3125938177108765, 2.1484334468841553, -1.3268017768859863, 0.789539635181427, -3.8057878017425537, -0.5864064693450928, -1.0108280181884766, -0.5851355791091919, -0.24261415004730225, 1.469760537147522, 0.7678921818733215, 1.0752133131027222, 2.389331817626953, 1.3054184913635254, -1.4793282747268677, 2.124912738800049, 1.5711593627929688, -1.6478543281555176, 2.499279260635376, 1.2929630279541016, -2.8641374111175537, 0.4086117744445801, 1.0274654626846313, 1.014809012413025, -0.362452894449234, -1.689961552619934, -0.340032696723938, -1.650395154953003, -1.333000898361206, 1.8898038864135742, 0.764237105846405, 4.521322250366211, -0.8558677434921265, -1.4591152667999268, -0.2751559019088745, -0.5081357359886169, -1.3162987232208252, 1.5497264862060547, 0.6415397524833679, -1.3869507312774658, -0.44524523615837097, -0.6373696327209473, -1.656363606452942, 1.0587502717971802, -0.8269071578979492, 1.2406784296035767, -0.7362285852432251, -0.2697381377220154, 1.405773401260376, 0.6762345433235168, -2.2235159873962402, 1.2520419359207153, -0.33739420771598816, 0.6626095771789551, -0.5643349289894104, -0.36226189136505127, 3.74240779876709, 1.8419480323791504, 2.6248559951782227, 1.1646045446395874, -0.2958618998527527, -1.620602011680603, -2.7221598625183105, 0.7918819785118103, -3.4671905040740967, 0.326309472322464, -0.2644325792789459, -0.667839527130127, 2.474931240081787, -2.2094812393188477, -0.17521095275878906, -3.3269340991973877, -0.551747739315033, -0.3592064380645752, 0.9867058396339417, -0.46090924739837646, 0.0897834300994873, 0.5766319632530212, -0.6303435564041138, 0.4085896611213684, 0.9341719150543213, 0.006127536296844482, 0.7395567893981934, 0.7638142108917236, -1.1667943000793457, -1.1370795965194702, -1.4277675151824951, 1.7371814250946045, -0.46843427419662476, -2.071415901184082, 2.8750832080841064, 0.6946618556976318, -0.12311820685863495, 2.5316269397735596, 2.660736083984375, -0.16820548474788666, 2.167938709259033, -1.4090920686721802, 0.46546077728271484, -1.299872636795044, -0.8313755989074707, -0.8931505680084229, 0.6415797472000122, 1.7140700817108154, 0.8251222968101501, -0.4228402376174927, -1.7107272148132324, 0.006010115146636963, -2.665142059326172, 0.2381715178489685, 0.6182670593261719, -0.9895614385604858, -2.2334234714508057, -2.799851179122925, 0.9724215269088745, 0.05883882939815521, -1.044028639793396, -0.7259727716445923, -2.7621376514434814, -1.1741454601287842, -0.9922501444816589, 1.379115343093872, 1.434401273727417, 0.14492404460906982, 0.8573983907699585, 1.1023224592208862, -0.9150733351707458, -1.124830722808838, 2.192035436630249, 0.5129651427268982, -1.3184022903442383, 0.13285169005393982, 1.0889105796813965, 2.5466771125793457, -0.6436185836791992, -0.5394759178161621, 0.1008332371711731, -1.2280319929122925, 0.5416545867919922, 0.5595439672470093, -1.9039090871810913, -0.3660220503807068, 1.5837311744689941, -0.08097702264785767, -1.5098564624786377, 0.3042427897453308, 1.3427168130874634, -0.6677592396736145, -1.2078367471694946, 0.2970120906829834, 0.2820861339569092, -0.2331463098526001, -1.0041427612304688, -0.4487559199333191, 1.7201457023620605, 1.2791682481765747, -0.8427630066871643, 1.5597026348114014, 0.5140810012817383, -2.274766206741333, -1.952571153640747, -1.1453139781951904, 2.3168137073516846, -1.6336889266967773, 0.7377829551696777, 1.844608187675476, 1.7384169101715088, 1.4153735637664795, 0.6779956817626953, -2.7787976264953613, -0.48586615920066833, -0.18111050128936768, 0.06090158224105835, -3.0151305198669434, 1.2665470838546753, -0.49555736780166626, 0.2833021283149719, -1.116025686264038, -0.7378860116004944, 1.5216516256332397, -0.6926957368850708, 3.7328875064849854, 0.9202695488929749, 0.24480104446411133, -3.2797226905822754, 0.5507392883300781, 2.2349982261657715, 1.1807233095169067, -1.085274338722229, 0.030028939247131348, -1.2141863107681274, 0.38118064403533936, 1.9362692832946777, -1.551628828048706, -0.5018609166145325, -0.3862963318824768, -0.9929778575897217, 0.7808778882026672, -1.898783802986145, -0.19872713088989258, 0.7858608365058899, 0.49955108761787415, 0.776533842086792, 0.06071162223815918, -3.1797380447387695, -1.8331732749938965, 1.4838712215423584, 0.6261881589889526, 3.047091007232666, -1.7357966899871826, 0.9903028011322021, -0.6595880389213562, -1.6617437601089478, -2.346346616744995, 0.17366349697113037, -2.048358917236328, 0.5643305778503418, 1.0019559860229492, -0.8237264156341553, -0.11741381883621216, -0.276909202337265, -1.267395257949829, -1.2394554615020752, -2.1114697456359863, -0.17522111535072327, 0.6217132806777954, 0.957952618598938, 1.1691381931304932, -0.5180834531784058, -0.9860460758209229, 0.6947599649429321, 0.27221325039863586, 2.1530237197875977, 3.2009105682373047, 0.5996673703193665, -1.0912114381790161, 1.6043205261230469, 2.378763198852539, 1.3428773880004883, 0.25783175230026245, 2.048389196395874, 1.6343958377838135, -0.10102352499961853, -0.7611455917358398, 0.5240104794502258, -1.9571195840835571, 1.2927626371383667, -2.0822088718414307, -3.987210750579834, -1.064321756362915, 0.026411890983581543, -0.2491842806339264, 2.129166603088379, -0.4342985451221466, -1.4484479427337646, 2.8511760234832764, -1.3390593528747559, 1.673999309539795, 2.361342668533325, -0.6616834402084351, -1.5463008880615234, 0.9263275265693665, -1.001520037651062, -0.007538020610809326, -0.8833985924720764, -1.160834789276123, -0.5934361219406128, 2.1112308502197266, -1.6539533138275146, -1.3161845207214355, -2.056051015853882, -0.2801147699356079, -0.4284057021141052, -0.25921469926834106, -2.2619547843933105, 1.6709779500961304, -1.0697300434112549, 0.7879430651664734, 0.941210150718689, 0.20075646042823792, -1.2929620742797852, 0.14275509119033813, 0.052762750536203384, -0.4526117444038391, -1.0031936168670654, 2.9074349403381348, -0.6423414945602417, -0.28118395805358887, -2.5447144508361816, 0.9274834394454956, -0.42900052666664124, -1.0540038347244263, 0.9420115351676941, -0.29889315366744995, -0.4944957196712494, 0.7446473240852356, 0.23017853498458862, 1.9947094917297363, -1.1399550437927246, 0.007697552442550659, 3.881965160369873, 0.9676325917243958, -0.616267204284668, -3.7483606338500977, -0.20313215255737305, 0.4828774333000183, -2.057910919189453, -0.5319507122039795, 0.33855539560317993, 2.548482656478882, 0.037890270352363586, 1.3999359607696533, 1.000309944152832, -0.3969663977622986, -1.371466875076294, 1.486203670501709, -1.0939569473266602, -1.0822429656982422, -0.9888025522232056, 0.9576334953308105, 0.5030806064605713, -2.3609726428985596, 0.4308798015117645, -0.7763350009918213, -1.2040247917175293, 2.1912667751312256, -1.1637365818023682, 0.876932680606842, -0.6522712707519531, 1.475895643234253, 1.8375775814056396, -2.6610124111175537, -0.7295682430267334, 1.1713883876800537, 1.563237190246582, -1.9854785203933716, -0.7868534326553345, 0.11100608110427856, -0.7094454765319824, -0.7045239210128784, 0.28248339891433716, 1.1656005382537842, 0.6874396204948425, 0.12002336978912354, -3.1990017890930176, -1.961154818534851, -0.22796940803527832, -0.47609657049179077, 1.5643556118011475, -0.566036581993103, 0.5593729019165039, 0.6010594964027405, 0.6396677494049072, -2.426928997039795, -1.5366474390029907, 4.0101776123046875, 0.5125226378440857, -0.5131971836090088, 0.49491074681282043, 3.1542744636535645, 1.3563107252120972, -0.4837193489074707, -0.9411106109619141, -0.18355441093444824, 2.052125930786133, -1.4280685186386108, -0.9298370480537415, 0.42154890298843384, 2.118293046951294, -1.351177453994751, -2.0427465438842773, 0.302916556596756, 0.9349203109741211, -0.24013280868530273, 0.5797969698905945, 0.18125541508197784, -0.37259146571159363, -0.7526310682296753, -1.1708085536956787, 1.3708765506744385, 1.5297188758850098, 0.8050727844238281, -0.31471478939056396, 0.30006372928619385, 2.426685333251953, -1.93413507938385, 2.33719539642334, 1.389376163482666, -1.42893648147583, -0.4323195517063141, -0.6877111196517944, 0.8826261758804321, 0.4336000382900238, -1.340079665184021, 0.21475715935230255, -1.4245456457138062, 1.0985846519470215, -0.23062127828598022, -0.9004859924316406, 1.037704586982727, -0.2066706418991089, 0.5087677240371704, -1.3989126682281494, -1.5825369358062744, 0.9889220595359802, 1.2009551525115967, 0.32718807458877563, 3.615737199783325, 0.9729377031326294, -1.3518331050872803, -0.6430342197418213, -1.6975840330123901, -1.9554157257080078, 0.4577364921569824, -1.829867959022522, 1.6055042743682861, -0.3042483329772949, 1.008487343788147, 0.3064665198326111, -2.3696842193603516, -1.800840973854065, -1.1683307886123657, -1.8511766195297241, 1.708716869354248, -1.7384376525878906, -1.2527732849121094, -1.9915409088134766, 0.514289379119873, 0.3871317505836487, -0.5267393589019775, -0.20052960515022278, -2.479926109313965, -1.8950607776641846, 0.6675648093223572, 0.13785161077976227, -1.7513582706451416, -0.7671582698822021, -1.5342451333999634, -0.16755616664886475, 0.1301361322402954, 0.40180402994155884, 0.701789915561676, -0.5242602825164795, -1.5464273691177368, 2.381472587585449, -1.7354334592819214, 0.17177361249923706, -0.03092050552368164, 1.1989290714263916, 1.0301159620285034, 1.1824672222137451, -0.9010355472564697, -0.2674586772918701, -0.17455634474754333, -1.258943796157837, 0.467438668012619, 2.2346842288970947, -0.11269775032997131, -0.1542752981185913, 1.1326655149459839, -1.5944931507110596, -0.2529991865158081, -0.9694173336029053, 0.7873046398162842, 2.332388401031494, -0.9255129098892212, 0.07795771956443787, 0.9898484945297241, 0.87116539478302, 0.8970803618431091, -0.6522481441497803, -0.2349553108215332, 0.09544491767883301, 1.2318377494812012, -0.7467207908630371, -2.450747013092041, -2.3937554359436035, 1.3580591678619385, 0.22827482223510742, 0.7767415642738342, 2.0345919132232666, -0.22908666729927063, 1.047005295753479, 1.3083534240722656, -2.019098997116089, -3.0413975715637207, -2.4150054454803467, 0.16120582818984985, 0.09029513597488403, 2.923100471496582, -2.1295158863067627, -2.3374204635620117, -0.5432230234146118, -0.23697412014007568, 1.7742890119552612, -2.008650064468384, -1.0408966541290283, -0.9685394167900085, -0.005756378173828125, 0.4969523847103119, 0.13975712656974792, 1.649998664855957, -0.30164310336112976, 0.9183530211448669, -2.1384575366973877, 4.081507682800293, 0.22007310390472412, 0.32392483949661255, -2.828862190246582, -0.39172133803367615, 0.7297199964523315, -0.11282223463058472, -2.5801124572753906, -0.9352437257766724, 0.10206810384988785, 1.3704373836517334, -1.8466858863830566, 1.846367597579956, -0.7775912284851074, 0.7187433242797852, -0.1793580949306488, -0.6252384185791016, 1.6059677600860596, -0.3472357392311096, 0.4443976879119873, 3.0474939346313477, 2.655463695526123, 0.8395503163337708, 2.072087526321411, -3.0668437480926514, -0.05396299809217453, 2.01243257522583, -1.336093783378601, 3.418290615081787, 1.0741276741027832, -2.3152151107788086, 0.9075251221656799, 1.3319910764694214, -1.093459129333496, -0.005613893270492554, 1.5295405387878418, -2.044166088104248, 0.1594604253768921, 0.09219418466091156, 0.11544585227966309, 0.15985572338104248, 0.8891596794128418, 2.0889923572540283, -0.05223876237869263, 1.7580496072769165, 1.2317208051681519, -1.1799869537353516, 0.403913676738739, 1.1340203285217285, -1.7912297248840332, 0.3944762051105499, -0.7057405710220337, -1.0871492624282837, 0.8500635623931885, 1.408814549446106, -2.891620397567749, 0.2581079602241516, 0.1032940149307251, 0.2371605634689331, 0.26849985122680664, -1.5538723468780518, 0.2618911862373352, 1.9697558879852295, -1.9703221321105957, 0.391856849193573, -0.28892040252685547, 0.5199089050292969, -1.5530424118041992, -0.9717668890953064, 0.23758891224861145, -0.037530481815338135, -1.12776517868042, -0.5351835489273071, 0.21027368307113647, 4.550912857055664, -2.968656063079834, -2.347625732421875, -0.0018576830625534058, 0.41916224360466003, -1.233760952949524, -0.6485574245452881, 2.4714508056640625, -2.607947587966919, -1.7089412212371826, 0.7775325179100037, -1.412440299987793, -1.1092579364776611, 1.1003243923187256, 0.9410442113876343, 2.2170701026916504, 0.23553194105625153, -3.8534977436065674, 1.1120346784591675, 0.17096900939941406, -0.9701727628707886, -1.2296085357666016, -0.7502615451812744, 2.8935766220092773, -0.1728982925415039, -1.6168205738067627, 2.303232192993164, -1.2622828483581543, 1.2426623106002808, 1.5592172145843506, 0.692013144493103, 1.9164321422576904, 0.45424944162368774, -0.25988006591796875, -2.054076671600342, 0.32379645109176636, 2.6911888122558594, 1.0967931747436523, -0.1961308717727661, -0.6905545592308044, -1.7371783256530762, -0.9796404242515564, -2.6633458137512207, 0.010666638612747192, -0.17290395498275757, -0.17000001668930054, 0.4091816544532776, 1.1189850568771362, 1.1078108549118042, 0.1674453616142273, 0.22406184673309326, -1.4099760055541992, -1.3376131057739258, -1.0188219547271729, -1.732529878616333, 0.9369025230407715, -0.34287095069885254, 0.21769599616527557, -0.5013264417648315, -0.18230700492858887, 0.42462778091430664, -0.93494713306427, -0.08503139019012451, -2.718761444091797, -1.4636644124984741, 1.887853980064392, 0.1275952011346817, -2.004601001739502, 0.4268419146537781, -0.2871488630771637, 3.130262851715088, -1.7021608352661133, 0.8385123014450073, -0.9564962983131409, 0.09858404099941254, -1.314180612564087, -0.4335261583328247, -0.4353821873664856, 1.7176055908203125, -1.6471153497695923, 1.064452886581421, 0.09102138876914978, 0.1405017375946045, -0.18449155986309052, -1.28645920753479, -1.9956717491149902, -1.9161725044250488, -0.8531120419502258, -0.34441661834716797, -3.40038800239563, -1.3234281539916992, -1.2596240043640137, -3.122941255569458, 1.5784258842468262, -0.21447429060935974, -0.5294553637504578, -0.0009946972131729126, 0.7250142693519592, 2.901345729827881, 1.3332369327545166, -0.5380164384841919, -0.6397281885147095, 2.190864324569702, -1.3813492059707642, -0.5577968955039978, -1.1317198276519775, -0.8272852301597595, 1.539157748222351, -0.11263325065374374, -2.547208547592163, 0.2020573616027832, -0.35062289237976074, 1.8103229999542236, 1.471360683441162, -1.339735746383667, -0.9077786207199097, 2.479548215866089, 0.42883381247520447, -0.23381787538528442, 1.1021617650985718, -1.5602872371673584, -0.3992292284965515, -2.3419313430786133, -0.40924715995788574, -0.5354663133621216, 1.2348089218139648, 0.974230945110321, -0.20685717463493347, 1.4072821140289307, 2.229431629180908, 0.05442178249359131, 0.19339331984519958, -0.29654455184936523, 2.0401360988616943, 0.6384518146514893, 3.5980520248413086, -0.8531280755996704, -0.5923453569412231, -0.8271783590316772, 1.613976240158081, 1.9162530899047852, -0.45205754041671753, -0.4775797128677368, 0.27792590856552124, -1.9742753505706787, 2.253754138946533, 0.5881852507591248, -1.1098191738128662, -1.1907695531845093, -0.25517550110816956, -0.48792532086372375, -0.09985615313053131, -0.3144928812980652, -0.5703071355819702, -1.9270949363708496, -0.3608264923095703, -0.29920172691345215, 1.753067970275879, 1.6295599937438965, 3.761796474456787, -2.660691738128662, 0.7725769281387329, -0.5308277010917664, -0.8872025012969971, -0.10888803005218506, -0.3801124095916748, 0.45745575428009033, 1.1918526887893677, -2.2244391441345215, 0.10917174816131592, 0.5258223414421082, 2.2475428581237793, -1.271815299987793, -1.3346436023712158, -3.7297067642211914, 2.482996940612793, 1.285391092300415, -0.06899213790893555, -0.7014472484588623, -0.0916053056716919, -0.5117985606193542, -3.956374168395996, 0.158352792263031, 0.9930486083030701, 1.255234956741333, 0.9810831546783447, -1.3689733743667603, 2.2912869453430176, 1.1339528560638428, -0.15401148796081543, 1.4096590280532837, -0.32060521841049194, 1.9542293548583984, 1.256054401397705, 2.3328046798706055, -0.33458513021469116, -1.203296422958374, -1.4535701274871826, -1.5140856504440308, -1.7314858436584473, 0.1284528374671936, 1.3088840246200562, 3.37326979637146, 0.6246169805526733, -2.9956960678100586, 1.6455644369125366, 2.570054292678833, 0.9346559047698975, -0.8195656538009644, -1.645142674446106, -0.991588294506073, -1.119232416152954, 1.5879857540130615, -1.6426784992218018, 1.1185358762741089, -0.04771924018859863, -0.0013265609741210938, -0.5036543011665344, 0.9803780317306519, 1.1861631870269775, -0.6779630184173584, 0.07223870605230331, -2.7555673122406006, 0.7171671986579895, -1.4866013526916504, 2.6944217681884766, -1.9514102935791016, -2.2469847202301025, -2.233320713043213, -1.4224915504455566, -1.7397972345352173, 0.49178263545036316, -1.1851160526275635, -2.2482619285583496, 1.7670409679412842, -4.738701820373535, -1.1897679567337036, 3.1902594566345215, -1.7831017971038818, -1.3172532320022583, -0.632854163646698, -0.08248293399810791, 3.866652250289917, -1.2083529233932495, -2.9609053134918213, -1.6634364128112793, -0.7746888995170593, -1.0629862546920776, -2.1168606281280518, -2.849055051803589, 0.46286851167678833, 1.876724362373352, -1.4631983041763306, 0.5080493688583374, -0.474462628364563, 0.05447535216808319, 1.0783809423446655, -2.63814640045166, -0.9199106693267822, 2.8074631690979004, -0.41968610882759094, 1.287562608718872, -0.6399520039558411, -0.6580199003219604, -1.007389783859253, -0.025676846504211426, 1.4677823781967163, 0.5466302633285522, -0.797520101070404, -1.516899824142456, 1.2826862335205078, -0.30133119225502014, -2.274259090423584, 2.3898069858551025, -1.3487732410430908, 2.551147937774658, 0.3667369484901428, -0.324225515127182, 0.22249874472618103, -2.890415906906128, -0.4789932668209076, -0.5227488279342651, -0.8483978509902954, 0.9706481695175171, -0.24811363220214844, 1.1676335334777832, -0.9246675968170166, -0.15377649664878845, -2.652672290802002, -1.2446203231811523, 0.24738149344921112, -0.8480741381645203, -2.094482421875, 1.377537488937378, 1.4877322912216187, 0.8068151473999023, 1.2337316274642944, -1.8237500190734863, -1.3186839818954468, -0.015257075428962708, -0.9668599367141724, -0.5148916840553284, 0.920455813407898, 0.003881514072418213, -0.9329988956451416, -0.40314704179763794, 2.68041729927063, -0.458196222782135, -3.145486831665039, -0.946814775466919, 0.12794917821884155, -0.036992818117141724, 3.645268201828003, 0.13777220249176025, 1.7713921070098877, 0.14432716369628906, 0.8616644144058228, -0.024420708417892456, -1.0985913276672363, -3.170086622238159, 0.13206470012664795, 0.27562427520751953, -0.607503354549408, -0.4205896854400635, 0.744873046875, -1.6185803413391113, -0.9250573515892029, 0.09029173851013184, 1.461890697479248, -2.084181308746338, 0.22433319687843323, -2.34901762008667, -1.4204087257385254, -1.906671404838562, -0.9442576169967651, -0.732789933681488, -0.4559457302093506, -1.4585418701171875, 0.027469485998153687, -0.9168705344200134, -0.3095899820327759, 1.2966349124908447, 1.2831538915634155, 1.618447184562683, -2.5667102336883545, 0.982999861240387, 0.3051602840423584, 0.06035849452018738, -1.8503060340881348, 1.5916956663131714, -1.6398932933807373, -1.3534049987792969, 0.5582638382911682, 1.3565977811813354, 1.4242520332336426, 1.1023292541503906, -0.7973580360412598, 1.9557939767837524, -0.8419557809829712, 2.2041428089141846, -1.997469425201416, -1.452984094619751, -1.59297513961792, -0.4554542303085327, 0.21359290182590485, 0.9533827900886536, 2.0691888332366943, -0.7340590357780457, -0.1434338092803955, -0.6465731263160706, -0.7494703531265259, -1.5654641389846802, 1.8320176601409912, 0.21339154243469238, -1.691957950592041, -1.948380947113037, -2.1900267601013184, -0.8775523900985718, 2.904508113861084, -1.499748945236206, 1.4259575605392456, 1.0789508819580078, 2.559264659881592, -0.4807460308074951, -2.2696146965026855, -3.854041337966919, 2.4254627227783203, -1.5955238342285156, -0.05391782522201538, 0.3717026710510254, 2.2935237884521484, 0.07302302122116089, 1.0977888107299805, -0.50388503074646, -0.6397190690040588, -0.8452335596084595, -3.279224395751953, -0.35194534063339233, -1.3224194049835205, 0.8526959419250488, 1.664122462272644, -0.014493031427264214, 1.5318338871002197, 1.7592267990112305, 1.9419822692871094, -1.1407835483551025, 0.8595529198646545, 1.1619614362716675, 0.6397067308425903, 2.0987443923950195, -0.06461489200592041, -1.5707471370697021, -0.7932578921318054, -0.8387751579284668, 3.1306729316711426, -0.40385979413986206, -0.4443419575691223, 0.48745250701904297, -2.186296224594116, -0.10237354040145874, 0.6854642033576965, -0.26558917760849, 0.6324996948242188, -0.9256646633148193, 2.677034378051758, -1.2308822870254517, -0.8640220761299133, 1.1403448581695557, -1.0653706789016724, 0.6611746549606323, 0.973762035369873, 2.1543190479278564, 1.7356338500976562, -0.003377467393875122, -1.1143046617507935, 0.9445480108261108, 0.5632750391960144, 0.8064302802085876, -0.4387539029121399, 0.8674039244651794, 0.9318904876708984, -1.0132033824920654, 1.695876121520996, -0.5978887677192688, -0.8809564709663391, -0.7852771282196045, -0.9008358716964722, -0.6458308100700378, 1.0065281391143799, -1.7168482542037964, -0.16010262072086334, 0.7532233595848083, 1.0632233619689941, 0.7381592988967896, -1.0689136981964111, -1.3806411027908325, -0.3882008194923401, -0.055211007595062256, -0.7036991119384766, 0.8654415607452393, 0.32174572348594666, 0.6967557668685913, 0.20499646663665771, 0.026243150234222412, 2.809645652770996, -1.5962986946105957, -0.4082673192024231, 0.23156994581222534, -1.6112333536148071, -0.6243220567703247, -1.3078135251998901, 0.7837396860122681, 1.0534676313400269, -0.8490172624588013, 3.587364912033081, 0.6476450562477112, -1.7726176977157593, -0.15774059295654297, -0.34153279662132263, 2.781688690185547, 0.03275656700134277, -0.41822272539138794, 0.41040709614753723, -0.5959345698356628, -0.11291593313217163, -2.910857677459717, -1.4360216856002808, 1.6494462490081787, -0.12031209468841553, -1.1883553266525269, -0.7221998572349548, -0.6776620149612427, -0.029187381267547607, 1.8132246732711792, 0.7478375434875488, -1.155678153038025, 0.5963892340660095, 0.3806387484073639, 3.0990729331970215, -0.8232202529907227, -3.3894214630126953, 0.41042962670326233, 0.29018068313598633, -1.532315969467163, -0.5026258230209351, 0.2563949227333069, 0.030962854623794556, -0.17545104026794434, 0.27102625370025635, -0.8079475164413452, -0.8868098258972168, -0.6497466564178467, 0.542819082736969, -0.0738668441772461, 0.550491452217102, 0.570598304271698, -0.04192722216248512, -0.5258960723876953, -1.191765546798706, 0.6724840402603149, -1.5970230102539062, 1.9026401042938232, -4.328133583068848, 0.23144547641277313, -1.523207664489746, 2.3732471466064453, 1.366666316986084, 1.0183899402618408, 2.6669363975524902, 0.6001648306846619, 0.3338741660118103, -1.2997877597808838, 0.48711979389190674, 0.21123361587524414, 2.5273280143737793, 0.27021563053131104, -0.869312047958374, -0.10170507431030273, -1.274135708808899, 0.4848363697528839, -3.7906556129455566, -2.3857202529907227, -0.3532681465148926, 1.2985670566558838, -0.7517669796943665, 3.7871928215026855, -0.6390180587768555, -1.726117730140686, -2.9012532234191895, -0.813173770904541, -1.2350865602493286, 1.5245847702026367, 0.5527278184890747, 2.2477517127990723, 0.04040217399597168, -2.2024970054626465, -0.368120014667511, 0.19403910636901855, 0.968819260597229, -1.2663865089416504, -0.46211087703704834, -0.9707669019699097, 0.2588198781013489, 0.18513834476470947, -0.40816259384155273, 4.264614105224609, -1.1901373863220215, -1.2951940298080444, -0.913152277469635, -1.8415873050689697, 1.9937336444854736, 0.15034031867980957, -1.932368516921997, -0.5965009331703186, 0.25381743907928467, -0.5872180461883545, -1.2228189706802368, -0.3510613441467285, 2.864593982696533, 2.0649144649505615, 2.36765718460083, 0.3533173203468323, -1.1853010654449463, 1.4681599140167236, -0.6129003763198853, -1.2992074489593506, -0.4098348617553711, -2.4945929050445557, -2.005275011062622, 1.2272801399230957, -1.4388676881790161, -0.6484990119934082, -0.6686140894889832, 1.6840496063232422, 0.31847429275512695, -0.9093306660652161, 2.7609434127807617, 2.395639657974243, -0.26006925106048584, -2.042019844055176, 2.803706169128418, 1.7745327949523926, -1.0145162343978882, -0.5781747102737427, 0.11392192542552948, 1.750361442565918, 2.4986732006073, -0.22363772988319397, -1.2209218740463257, 1.1551122665405273, -1.773937463760376, 0.6003061532974243, 1.7012994289398193, 1.5431556701660156, -0.978582501411438, 0.24658942222595215, -0.9434981346130371, -0.33313971757888794, 0.5778484344482422, -1.2708674669265747, 0.20237186551094055, -0.8712359070777893, -0.7218780517578125, 0.5134326219558716, 0.8814259171485901, -0.9906944632530212, -2.8852579593658447, 0.5083860158920288, -0.3722139000892639, -1.3082258701324463, -0.6222670674324036, -0.024072974920272827, 1.605881690979004, 0.3404518961906433, -2.0629491806030273, -1.1771149635314941, -0.6290179491043091, 0.7963012456893921, 0.4156898260116577, -3.79697585105896, 1.2755610942840576, 0.2944793701171875, -0.7097803354263306, -1.6865131855010986, -2.148695468902588, -1.5645420551300049, 0.242576003074646, 1.508162498474121, 0.649359405040741, -1.1754510402679443, 2.0984928607940674, 2.0749611854553223, 2.3851208686828613, 0.1415349245071411, -0.21705427765846252, -1.2126985788345337, -0.07211053371429443, 0.7572376728057861, 0.917026698589325, 0.06339812278747559, 0.7800732254981995, -0.693553626537323, 0.38459059596061707, 1.2549502849578857, 0.07747311890125275, 1.0419524908065796, -2.0857555866241455, 1.7734262943267822, -0.9179378747940063, -2.586221218109131, 0.6067724823951721, 0.0010491982102394104, 1.482232689857483, 0.8664054870605469, -1.652360439300537, 3.924798011779785, 0.6914035677909851, -3.3982481956481934, -0.18790429830551147, -0.5155808329582214, -1.053114652633667, -0.17478501796722412, -2.9305763244628906, -2.518843173980713, 0.8897398114204407, -2.054203510284424, 0.7417606115341187, 1.5587480068206787, -1.2047890424728394, -0.5169711112976074, -0.42091283202171326, -2.387880802154541, -0.2913897633552551, 1.0333781242370605, 0.2996665835380554, -0.006044328212738037, 0.7972234487533569, 0.6101740598678589, -0.48502469062805176, 0.9533754587173462, 0.5872285962104797, 0.5662200450897217, 0.651188313961029, 1.449156641960144, -0.34329941868782043, -1.1492608785629272, -0.7300221920013428, 0.4097861349582672, -0.991923451423645, -0.13842257857322693, -1.8444150686264038, -1.1792627573013306, 1.391729712486267, 0.8753052949905396, 0.7594159841537476, 0.17149344086647034, -0.42314720153808594, -1.7340867519378662, -0.40804117918014526, 1.4351235628128052, -0.1587030440568924, 0.9400608539581299, -0.6082008481025696, 3.3348536491394043, 1.7022689580917358, 1.5894484519958496, 0.5029714703559875, -0.3207477033138275, -0.26925307512283325, -0.9618558883666992, 0.012879520654678345, -1.6665360927581787, 0.3406349718570709, -3.3346381187438965, 0.36415618658065796, -0.5934114456176758, -1.6086963415145874, -0.7705225944519043, 0.33262568712234497, -1.638007402420044, -1.8691864013671875, 0.22626197338104248, 2.222879409790039, 1.0537341833114624, 1.2646143436431885, 0.4445803165435791, -0.9308507442474365, 2.417056083679199, -1.0680338144302368, -1.752957820892334, -0.1685582995414734, 1.8503541946411133, 0.8639588952064514, 0.12973713874816895, -0.03118598461151123, -0.9534138441085815, -1.8712972402572632, 1.3926622867584229, -0.7887298464775085, -1.4307756423950195, 0.5150429606437683, 0.3835541307926178, 0.7714701294898987, -3.1228952407836914, -0.9477657079696655, -1.0228378772735596, 1.2524367570877075, -0.3426491916179657, 0.44164127111434937, -0.1577666997909546, 1.917505145072937, 1.5613921880722046, -0.547174334526062, 1.4067937135696411, 2.4587631225585938, 2.016648054122925, 0.6743448972702026, -1.3358445167541504, -1.0183124542236328, 1.3286988735198975, -3.000199794769287, -2.6122047901153564, 0.1772478222846985, -0.10954932868480682, 1.114464521408081, -0.7767186164855957, 0.790459156036377, -1.487601399421692, -0.001579679548740387, 1.1715470552444458, -0.14915060997009277, -0.872626781463623, 2.367485523223877, 0.5991578698158264, -0.6761400103569031, -2.2534379959106445, -0.7391053438186646, -1.2933837175369263, -0.9892910718917847, 1.6139642000198364, 1.003827691078186, -0.27386024594306946, 2.354567050933838, -1.0680148601531982, -0.23070019483566284, -2.167736053466797, 1.6204506158828735, 0.5103781223297119, -0.5977764129638672, -2.2228665351867676, 1.8153376579284668, -0.6303054094314575, -0.6826151609420776, -0.8993938565254211, -2.1578333377838135, 1.39717435836792, 0.3666248321533203, 0.5032970905303955, -0.8997355103492737, -1.3097201585769653, -0.139570951461792, 2.4104256629943848, -2.418276786804199, -0.5523545145988464, 0.8241294622421265, 0.1241881251335144, -0.9996737241744995, 0.34998631477355957, 0.45532816648483276, -1.0019147396087646, -0.9212220907211304, 0.6062371730804443, 0.949020266532898, 0.009694874286651611, 1.1016165018081665, 1.0308469533920288, 1.5429213047027588, -0.7527977824211121, 1.464670181274414, 0.9274791479110718, 0.8672192096710205, 1.3696175813674927, 2.249462604522705, -0.13035982847213745, 2.882371425628662, 1.8802932500839233, 1.4554834365844727, -0.7036242485046387, -0.2750682234764099, 3.036299228668213, -2.8112878799438477, 1.353723406791687, 1.1690033674240112, 2.515918493270874, 1.240923523902893, -1.3916665315628052, 0.3736375570297241, 2.3505828380584717, -1.9490375518798828, 0.42038339376449585, 1.4392646551132202, -0.48301151394844055, 0.9064900279045105, 1.7326773405075073, 0.586650013923645, -0.22785209119319916, -0.9580783247947693, -1.4710464477539062, -0.07755357027053833, 1.0428106784820557, -2.1109299659729004, 1.911245584487915, -1.8717436790466309, 0.6762805581092834, 0.24451982975006104, -0.45178288221359253, -0.37212932109832764, 1.1370865106582642, 0.24618619680404663, 0.22756695747375488, 0.3796311616897583, -0.2752646803855896, -0.9433587789535522, -2.756768226623535, -2.7739763259887695, -1.8551616668701172, -1.1965895891189575, -0.009127050638198853, -1.1976077556610107, -1.1593197584152222, -1.976244568824768, 2.3792343139648438, -1.2669254541397095, 1.2959413528442383, -0.3093010187149048, 2.4587740898132324, 0.6772909164428711, -1.4753576517105103, 2.443061590194702, 0.24957384169101715, -0.9773660898208618, -0.8975452780723572, 0.892408013343811, -2.3715760707855225, 2.317047595977783, 0.8936851024627686, 0.01824742555618286, 0.8079193830490112, 0.4095415472984314, 0.3563554883003235, 2.881788969039917, -1.6477230787277222, 0.39832520484924316, -0.6156768202781677, 0.35183513164520264, 1.4788111448287964, 1.4379030466079712, 0.27177315950393677, -1.4726448059082031, -1.8709981441497803, 2.947808027267456, 1.7655069828033447, -0.806488037109375, 1.4349236488342285, -0.47634005546569824, 2.470670700073242, 0.20733946561813354, -1.1017330884933472, -1.8095674514770508, 2.2256031036376953, 3.064786195755005, 0.5765752792358398, 0.6934579014778137, 1.6821811199188232, -0.1247040331363678, -0.3424231708049774, -1.3140878677368164, -0.6530248522758484, 0.40672820806503296, -2.9527292251586914, -1.2906904220581055, 2.6203322410583496, -1.4848148822784424, -2.1599810123443604, 0.0264737606048584, -0.4494507312774658, -1.426782250404358, -1.9172563552856445, 1.763235092163086, 1.7635371685028076, -1.6028833389282227, 0.42463329434394836, 0.44171035289764404, 0.04177352786064148, -0.1050025224685669, 1.1104182004928589, -0.4878365397453308, -1.5418301820755005, 0.734076201915741, 1.634522557258606, -1.4166922569274902, -1.081133484840393, -0.3754703402519226, -2.4502205848693848, 0.17561927437782288, -0.31152886152267456, 1.0367321968078613, 0.8545575737953186, 0.04221788048744202, -1.827568531036377, -0.8811120986938477, -3.2196407318115234, 0.6888613700866699, -0.9047833681106567, 1.8178284168243408, 0.1516503095626831, -0.15011471509933472, -0.13662785291671753, -0.9642206430435181, -0.709719181060791, 2.298677444458008, 2.110884189605713, 2.0263168811798096, -0.4656553268432617, -1.7250676155090332, -2.2381303310394287, 1.2274928092956543, -0.9445919394493103, 3.0221505165100098, 1.234104037284851, 1.728600263595581, -1.86857271194458, 0.9815988540649414, -1.0849298238754272, 2.705414295196533, 0.04108130931854248, -0.3943677842617035, -0.11658169329166412, 1.048386812210083, -0.9124611020088196, -0.4957035183906555, -1.0723872184753418, 0.4360405504703522, 1.270228624343872, 1.9460246562957764, 0.4923727512359619, -1.3834832906723022, 0.36772334575653076, -0.609958827495575, 1.3169291019439697, 1.459148645401001, -0.3812642991542816, -2.2041516304016113, 1.253131628036499, -1.000651240348816, -1.6829230785369873, 0.26732808351516724, -1.2861049175262451, 1.0922855138778687, 1.024428367614746, 0.24815189838409424, -0.5094269514083862, -1.169737696647644, -1.024794578552246, 0.9331642389297485, -0.2609494924545288, 0.02546823024749756, 2.1100945472717285, -3.381014108657837, -1.9468305110931396, 2.474299430847168, 4.118379592895508, 0.6436014771461487, -0.25882071256637573, 0.32251277565956116, -1.2147307395935059, -0.07936859130859375, -1.3521145582199097, 2.5904064178466797, -1.8377925157546997, 0.378240704536438, 0.6531504988670349, -1.6210694313049316, -1.1786224842071533, 0.12875476479530334, 0.07507652044296265, 0.21008455753326416, -1.245715618133545, 0.06083577871322632, 0.036135926842689514, -1.6316742897033691, 0.14766448736190796, 2.5472071170806885, 0.5864179730415344, 3.336216926574707, -0.7398272752761841, 2.8350510597229004, 1.5765728950500488, 1.8310455083847046, 0.40845784544944763, -0.0018278807401657104, 0.6111049056053162, 0.31939804553985596, -3.352280616760254, -0.7677280306816101, 0.31397387385368347, -1.2546181678771973, 1.5091774463653564, -1.5178322792053223, 2.6061267852783203, 1.135048508644104, 0.8889798521995544, -0.7907359600067139, 1.391103744506836, -0.936753511428833, -4.223940372467041, -4.357661247253418, -0.47465670108795166, 1.0519044399261475, -2.09540057182312, -0.2928446829319, -0.40044525265693665, -0.042987287044525146, -1.3107627630233765, 0.44309002161026, 1.3212463855743408, -0.5499018430709839, -3.306673288345337, 1.0535329580307007, -2.3697731494903564, 0.09017526358366013, -0.12606507539749146, -0.9063512086868286, 2.089442014694214, -0.3585573434829712, -0.7683759331703186, 0.8998502492904663, -0.8124716281890869, -0.9073187708854675, -2.2174582481384277, 1.2130409479141235, -0.5094931125640869, -1.999666690826416, -1.8219212293624878, 1.9143357276916504, 0.2825219929218292, 3.738907814025879, 1.9361919164657593, -2.7893481254577637, 2.750023365020752, 1.5879731178283691, -0.8494029641151428, -1.8591949939727783, 0.4618110656738281, -0.07215052843093872, -0.28315451741218567, 0.371074914932251, -0.10191607475280762, -1.8268073797225952, -1.3679248094558716, -0.6931305527687073, -0.6941677927970886, 1.3866537809371948, 0.009503364562988281, 1.2764300107955933, -0.20657135546207428, -0.062432169914245605, -1.0692003965377808, -0.7109611630439758, 1.2510452270507812, 1.3584492206573486, -0.22303682565689087, 0.43331050872802734, 0.5562213659286499, 0.08881498873233795, 1.3480823040008545, -2.0814685821533203, 0.3734092116355896, -1.027658462524414, 0.4991830885410309, -0.4521324038505554, -0.25011149048805237, 0.8078058362007141, -0.3466675877571106, -0.3393864333629608, -3.8692729473114014, -1.0025923252105713, 4.297738552093506, 1.9558007717132568, 0.2945033311843872, -3.2171552181243896, -1.0109944343566895, 1.049394965171814, -0.3037685751914978, 1.0495848655700684, 2.3903098106384277, -0.08915826678276062, -0.4117735028266907, -0.9976378083229065, -1.3468937873840332, -0.8696585893630981, -1.3850915431976318, 1.1064949035644531, 0.2248978614807129, 0.11206580698490143, -1.064195990562439, -0.6267976760864258, 0.62897789478302, 2.49324369430542, 0.9627835750579834, -0.6886781454086304, 2.615516424179077, 0.8593856692314148, 0.963180422782898, -1.5044304132461548, -0.37115246057510376, -1.481735110282898, 0.1224377453327179, -1.413955569267273, -1.1223124265670776, -0.8876059055328369, -0.9042381644248962, 0.26528871059417725, 2.593268394470215, 4.249238967895508, -0.7499313354492188, -1.3169727325439453, 1.582531213760376, -0.5999085307121277, -1.1805291175842285, -2.6244759559631348, 1.0822479724884033, -3.664778232574463, -1.1369863748550415, 0.40545347332954407, 0.906343936920166, -2.954468250274658, 0.928123414516449, -1.286029577255249, -0.6614080667495728, -0.12757641077041626, -1.515869379043579, 0.786712646484375, 2.084911823272705, 2.0026626586914062, -1.1902592182159424, 0.5975477695465088, -1.1968315839767456, 0.9672108888626099, 1.11519193649292, -0.889642596244812, 1.681717872619629, -1.7508026361465454, 1.2153066396713257, -1.036402940750122, 0.15532377362251282, -2.90817928314209, -1.917121171951294, 0.16405484080314636, -0.1997441053390503, -1.5233874320983887, 0.3480064868927002, 2.263535737991333, -1.1389641761779785, -1.0116991996765137, 2.5332276821136475, 0.941493809223175, 0.09218814969062805, -0.7448645234107971, -1.17482590675354, 1.6603851318359375, -0.581450879573822, -1.071394443511963, 0.19857484102249146, 0.31105074286460876, 1.012439250946045, 1.219503402709961, 0.46586912870407104, 0.5210781693458557, 0.732463002204895, 0.27418380975723267, 0.19847679138183594, -0.9355481863021851, 1.453673005104065, -2.6680922508239746, -0.5689240097999573, 1.851861834526062, 1.9522223472595215, 0.5336594581604004, -1.4505665302276611, 2.916846990585327, -0.3401908874511719, 0.2348514199256897, -1.6536314487457275, -1.1140927076339722, -0.04192686080932617, -1.8599035739898682, -0.02646958827972412, 1.2876248359680176, 0.24141204357147217, -0.037552833557128906, 0.6634435057640076, 1.4936925172805786, 0.9224758148193359, 0.6165186166763306, -0.03165996074676514, -1.7362687587738037, 0.681920051574707, -0.20707941055297852, -1.915079116821289, 0.9856477975845337, -0.41021814942359924, -0.08046352863311768, -0.13359685242176056, 0.35761895775794983, 0.5349663496017456, -3.0582380294799805, 1.1584405899047852, -0.8567718267440796, -1.2227197885513306, -0.9200084209442139, 0.760867178440094, 0.48388612270355225, 1.544604778289795, -2.0343830585479736, 0.16417695581912994, -3.5890746116638184, 0.20864838361740112, -1.151374340057373, 0.9073840975761414, -0.8117668032646179, -0.1597394049167633, -0.8652240037918091, -3.602198362350464, 1.4288469552993774, -1.509265661239624, -0.7487821578979492, -0.8661022782325745, 0.7408236265182495, -0.13267099857330322, -0.07265567779541016, 4.171889781951904, 0.32370802760124207, 1.5887531042099, 0.21771711111068726, -0.36232495307922363, -1.7472658157348633, 1.3397045135498047, 0.06162114441394806, -0.43514499068260193, -1.8123950958251953, -0.08550503849983215, -0.566220760345459, 0.6572698354721069, 1.1935993432998657, 2.0911617279052734, 0.09668941795825958, -1.4405150413513184, -0.5911804437637329, 0.985835075378418, -1.292517900466919, -3.2952818870544434, -0.32368117570877075, 0.11404770612716675, 0.7784690260887146, 0.6786302328109741, -1.2187222242355347, 1.4517667293548584, 2.3136816024780273, -1.1457327604293823, -0.8999533653259277, 1.2930407524108887, 0.23753196001052856, 1.004610538482666, 0.8608304858207703, -1.9191703796386719, -0.22210589051246643, 1.2219617366790771, -0.2310107797384262, -0.2764335870742798, -2.787233829498291, -0.00389588694088161, 2.049055576324463], "dtype": "float32"}}, {"a": [{"__nd__": [[0.10626717656850815, 1.2792949676513672, -0.8432371616363525, 0.5471110939979553, 0.16479383409023285, 0.6476709246635437, -0.17275261878967285, 0.2666794955730438, 0.4237259030342102, 0.9851970076560974, -0.9598482251167297, -0.13781200349330902, 0.45508307218551636, 0.8514658212661743, 0.39224478602409363, 0.44025862216949463, 1.8084229230880737, -1.026099443435669, 0.8299751877784729, 0.3386266529560089, 0.1659504473209381, -0.5158142447471619, 0.07170572131872177, 1.200886845588684, -1.004461407661438, 1.2777174711227417, -0.3652086853981018, 0.13831447064876556, -2.178251266479492, -0.6271129846572876, -1.005407452583313, 0.9318586587905884, 0.9773702025413513, 0.12871776521205902, 0.9062482118606567, 0.6804015636444092, -0.22021736204624176, 0.22045214474201202, 0.47323867678642273, -0.2632599174976349, -1.279166340827942, -1.4372667074203491, -1.2406121492385864, -0.8396475911140442, -1.7462990283966064, -0.48248621821403503, -0.4677542448043823, 1.2772958278656006, -0.2684173882007599, 0.6711426377296448, -0.612667977809906, -1.8181943893432617, -0.6794741749763489, 0.1600341796875, 0.07640433311462402, 2.71005916595459, 1.0712162256240845, -1.8529640436172485, -0.2128961831331253, -1.6213126182556152, 0.500145673751831, 0.5172940492630005, -2.4787869453430176, -0.3620752692222595, 1.0260800123214722, -1.5573430061340332, -0.1470620483160019, -0.7041381597518921, -0.2505780756473541, 0.4713764488697052, 0.2877253293991089, -0.05314209312200546, 1.1614844799041748, 0.6294060945510864, 0.546046793460846, -1.958626389503479, 1.263878583908081, 0.7632832527160645, 1.7284389734268188, 1.2975847721099854, 0.8995665311813354, 0.6464781165122986, 1.4180221557617188, -2.1064507961273193, 0.5357361435890198, -0.39338359236717224, 1.5216964483261108, 0.9215404987335205, -1.4775769710540771, -0.3633747100830078, -0.011245089583098888, 1.282286524772644, 1.473354697227478, -0.700324296951294, 0.4734048843383789, 0.2966901957988739, 0.493096262216568, 0.8261318802833557, 0.2559225559234619, -0.19300270080566406, -1.287061333656311, 2.0057530403137207, -1.7561278343200684, -1.6890625953674316, -0.009677288122475147, -1.1740326881408691, -0.5573903322219849, -0.4530752897262573, 0.4675023853778839, 1.023533821105957, 0.6471467018127441, 1.3090487718582153, -1.272611379623413, 0.28725919127464294, 0.849998950958252, 0.7263913750648499, -0.956540048122406, 0.596028208732605, -0.07769183069467545, -2.132066249847412, -0.2743658423423767, -0.7484250664710999, 0.5803720355033875, -1.4040368795394897, 0.3454490005970001, 0.2717324495315552, 0.8918389678001404, 1.9385217428207397, 0.02797318622469902, -0.2135712206363678, -0.3648509979248047, 1.18708336353302, -1.8548961877822876, 1.4944801330566406, -0.1718641221523285, -1.3193295001983643, 1.3398433923721313, -1.0984641313552856, -0.31881049275398254, -1.0343339443206787, -1.033605694770813, 1.5052719116210938, -0.553128182888031, -0.5274238586425781, 0.3754523992538452, 0.5496132373809814, -0.10032600909471512, 1.0796762704849243, 1.1155812740325928, 0.6477580666542053, 1.1774238348007202, -0.34762686491012573, 0.3238488733768463, 0.15766339004039764, -0.7405911087989807, 0.5763903260231018, -1.075353741645813, -0.20380675792694092, -0.07982311397790909, 0.10214346647262573, -0.40573999285697937, 0.3351246416568756, 0.124901682138443, -1.3104108572006226, 1.7855794429779053, -0.7542805075645447, -0.0223927553743124, -0.4315963387489319, -1.9305717945098877, 0.6333808898925781, -0.31388941407203674, -1.5214369297027588, -0.15399104356765747, 0.846397876739502, -0.41459670662879944, -1.7658298015594482, -1.045036792755127, 1.0782228708267212, 0.4234529733657837, -0.3678973913192749, 0.6358452439308167, 0.08784981071949005, -1.4305663108825684, 1.7360129356384277, -0.7815329432487488, -0.4504646956920624, 0.34614989161491394, 1.4497010707855225, -1.2235616445541382, -0.3556223511695862, 0.5333914756774902, -0.2560914158821106, 0.5485160946846008, -0.6082568764686584, -1.5514464378356934, 1.4926782846450806, 2.1462161540985107, -2.840163469314575, 0.18090718984603882, -0.5205444097518921, 0.5875999927520752, 1.379281759262085, -0.8741253018379211, -1.861939549446106, -1.780692458152771, -1.2503561973571777, -1.1024746894836426, 0.05361626297235489, 1.462559461593628, -0.4368610680103302, 0.9770646691322327, -0.16726918518543243, 0.17692144215106964, -0.2556110620498657, 1.0578701496124268, 1.888669490814209, 0.13267205655574799, -1.9719750881195068, -0.7779510617256165, -0.22739870846271515, -0.8475432991981506, -0.8002709746360779, -0.6232893466949463, 0.2535761892795563, -1.1759798526763916, 0.7726287841796875, -0.9177384376525879, -0.525671124458313, 1.4958462715148926, 0.2689039409160614, -0.831578254699707, -2.215461015701294, -1.2140532732009888, -0.17817790806293488, 0.8484733700752258, 0.3212014436721802, 0.3454620838165283, 0.890963077545166, 1.9438869953155518, 0.21219904720783234, 0.6788995265960693, 0.8363362550735474, 0.5697809457778931, 0.06922060251235962, 0.8499875664710999, -0.08463623374700546, -1.7298859357833862, -0.24067015945911407, -0.7592663168907166, 0.38570699095726013, 0.06283602863550186, 0.6831603646278381, -0.5148865580558777, -0.5602656006813049, -0.09477293491363525, -0.7391011714935303, 0.5958400964736938, 0.3075113296508789, -0.32926198840141296, -0.9007723927497864, 1.5208888053894043, 0.6578512191772461, 0.2619239389896393, 0.12571075558662415, 0.888784646987915, 1.523053526878357, -0.0317181795835495, -1.2418159246444702, 0.23223553597927094, -0.1371351033449173, -0.9918850064277649, 0.76368647813797, -3.818227529525757, 0.4420648515224457, -1.493923544883728, 0.43943652510643005, -1.2008835077285767, -0.13486629724502563, -1.742517352104187, -0.3986152708530426, 0.40989938378334045, -1.5770899057388306, -0.28922733664512634, -0.32412856817245483, 0.019088691100478172, 1.1943873167037964, -1.7624762058258057, -0.1228778138756752, 1.1304857730865479, -0.9518100619316101, 0.21319915354251862, -0.6122761964797974, 1.4265273809432983, -1.3054215908050537, -1.6469224691390991, -0.8125040531158447, 1.2825238704681396, -1.1867361068725586, -0.29046353697776794, -0.010263017378747463, -0.4487095773220062, 0.6474564075469971, -1.4986389875411987, -0.18052542209625244, -1.1950814723968506, 0.022977754473686218, -0.2644413113594055, 1.2872380018234253, -0.5522946715354919, 0.02283789962530136, -0.4791135787963867, 0.06224915757775307, -0.5215770602226257, 0.22232753038406372, 0.4878804683685303, 1.4383094310760498, -1.611038088798523, 1.3072158098220825, -1.809876799583435, 1.4357938766479492, 0.12762515246868134, -1.6691564321517944, 2.3015944957733154, 1.100448489189148, 0.00238055526278913, -1.7124441862106323, 0.4384956955909729, 0.26798686385154724, -0.33213531970977783, -0.486005961894989, -0.4046686887741089, 2.516225576400757, 0.9039369225502014, 1.2323015928268433, -0.46148237586021423, -0.11509136110544205, -0.6770459413528442, -1.9207916259765625, -2.1891207695007324, -1.0482393503189087, 0.6810075640678406, 1.2446318864822388, 0.06290016323328018, 0.5680041313171387, 1.042794108390808, -1.3397380113601685, -0.5300703048706055, -0.5690032243728638, -0.7941035628318787, 1.6784625053405762, -0.5520136952400208, 0.07834720611572266, 0.047964565455913544, -0.21596752107143402, -0.9991625547409058, 2.0226128101348877, 1.5175751447677612, 0.3802390396595001, -0.8409329652786255, 0.806819498538971, -1.569858193397522, -0.7484334707260132, -0.274221271276474, 0.20074647665023804, -0.3354777991771698, -0.1521921157836914, -0.6428512930870056, 0.6600147485733032, 1.2300069332122803, -0.21140766143798828, -0.9594210386276245, -0.95453941822052, 0.2961048483848572, -0.4405558705329895, -0.2314506471157074, -0.7079276442527771, -0.6313511729240417, 0.07242926210165024, -1.7793076038360596, 0.8382129669189453, 0.4555879533290863, -0.6050190925598145, -1.237317442893982, -0.4676586389541626, -0.16621917486190796, 0.6243241429328918, 0.01607516035437584, 0.03582711145281792, -1.0622868537902832, 0.7344405651092529, -1.2915709018707275, -1.2362209558486938, 0.7505379319190979, 0.48629137873649597, 0.1563860923051834, -1.9559623003005981, -0.43672606348991394, 0.6100752949714661, -0.8639984726905823, 0.5809658169746399, -1.0780738592147827, -0.47995904088020325, -2.5495312213897705, -3.4893243312835693, -0.5418992042541504, -0.8291623592376709, 0.5644356608390808, -0.09966253489255905, -0.7935000658035278, 0.10870429873466492, -0.14860138297080994, 1.1563279628753662, 0.13205091655254364, 0.7297511100769043, -0.04063571244478226, -0.41910532116889954, 0.5904622077941895, 1.2332847118377686, -0.38757050037384033, 1.7659035921096802, -0.6987889409065247, 1.1600068807601929, -1.4421945810317993, -0.5871357321739197, 0.3123891353607178, 0.6559138298034668, -1.4068461656570435, 0.9647672176361084, -0.1405738890171051, 0.6453947424888611, 0.4142949879169464, -0.5761517882347107, -0.2330312430858612, 1.2433160543441772, -0.22720372676849365, -0.10800455510616302, 2.0418589115142822, -1.1439666748046875, 0.4578538239002228, -0.7383628487586975, -1.5610307455062866, 0.600217342376709, 0.8432058095932007, 0.2913128435611725, 0.5317047238349915, -0.08631496131420135, 0.35181811451911926, 1.0200947523117065, 1.4965604543685913, 0.2587766945362091, 1.5321117639541626, 1.4151029586791992, 0.4239751696586609, -1.3345344066619873, 0.10026151686906815, -0.26395779848098755, 0.2610774338245392, -0.1038745567202568, -0.9230356216430664, -1.3077467679977417, 0.02650325372815132, -0.3973013162612915, -1.4701279401779175, 0.7579357624053955, 1.0336735248565674, -0.279676616191864, 0.293020635843277, -3.2199552059173584, 0.6239537596702576, -2.336606979370117, 0.23623386025428772, 0.77848219871521, -0.3262122571468353, -0.3839268088340759, -0.2585753798484802, 1.0180556774139404, 0.5507001280784607, -0.8260690569877625, 0.20580293238162994, 0.9673849940299988, 0.08143758773803711, -0.4396238625049591, 0.6603392958641052, 2.0652105808258057, 1.2076971530914307, -0.5835016369819641, -0.5775975584983826, 1.3442864418029785, 0.7822033166885376, 0.32164087891578674, -0.16479818522930145, 2.0153884887695312, -0.04882332310080528, -0.3108236789703369, -1.6292670965194702, -0.7614820599555969, -1.4578418731689453, -1.1287411451339722, 1.4902734756469727, 0.11615720391273499, 0.8775964379310608, -1.6161106824874878, -0.2656456232070923, -0.0027520754374563694, 1.4059581756591797, -0.4259720742702484, 0.7607157826423645, -1.0354732275009155, -0.37152254581451416, -0.3881131708621979, 0.8796262741088867, -0.005690887104719877, 0.43796175718307495, 0.157809779047966, -0.42142006754875183, -1.2228379249572754, -0.7655558586120605, 1.4557563066482544, 0.8352389335632324, -0.2792004346847534, 0.07385388761758804, 0.6857317090034485, -0.12104490399360657, 0.7214325070381165, -0.8289151191711426, -1.3757519721984863, 0.15787628293037415, 1.5113656520843506, -0.6866942048072815, 0.7319290041923523, -0.11105382442474365, 1.565997838973999, 0.7502902150154114, 1.2886009216308594, -1.3604227304458618, 0.21639928221702576, -0.8863576054573059, -0.24909266829490662, -0.1369769126176834, -0.10969636589288712, -0.6261117458343506, -1.1765252351760864, 0.012224291451275349, -1.248740315437317, -0.3870566487312317, 2.6377649307250977, 0.5206450819969177, -0.59060138463974, -0.4588978588581085, 1.9734828472137451, -0.0700877457857132, 0.16346409916877747, 0.8549841046333313, 0.814879298210144, -1.0916035175323486, -0.7963101863861084, -1.258359432220459, 1.1968632936477661, -1.2326393127441406, -0.8906993865966797, -0.006727402098476887, 1.343286156654358, 0.0354059562087059, 0.6303653717041016, 0.14648130536079407, -1.301141381263733, -0.9664554595947266, 1.6930625438690186, 0.45384296774864197, -2.1037685871124268, 1.1078258752822876, 0.5774103403091431, -1.6113110780715942, -0.8602334856987, 0.7139551043510437, 0.004551432561129332, 0.8307347297668457, -0.9230518341064453, -1.2359050512313843, -0.8542970418930054, -1.0002493858337402, -0.5016128420829773, 1.187016248703003, 1.46969735622406, 0.9113892316818237, 0.08194705098867416, -2.104419469833374, -0.20744480192661285, -0.29784175753593445, 1.0418212413787842, -1.1311039924621582, 0.14026890695095062, 0.6113056540489197, -2.4117958545684814, 1.0225716829299927, -0.20781800150871277, -0.31087806820869446, -0.18994377553462982, 1.4151991605758667, -0.6273785829544067, 0.2753572463989258, 0.514467179775238, 0.42887160181999207, 1.1392228603363037, -0.9437035918235779, 1.2563133239746094, -0.26378241181373596, 0.5706722736358643, 1.1102620363235474, 0.24105055630207062, 2.117447853088379, -1.0737900733947754, 1.9560638666152954, 0.429313987493515, -1.052773356437683, -0.6741341948509216, -1.8300129175186157, 0.003339508781209588, -1.9449355602264404, -1.0436190366744995, 0.8817536234855652, 0.24387796223163605, 2.295114040374756, -0.9691854119300842, 0.16299471259117126, -0.7628956437110901, 0.871502161026001, 0.6887145638465881, -0.4171142578125, -0.20260591804981232, -1.0829217433929443, 0.3364090621471405, -0.5142936110496521, 1.519787073135376, -0.6091339588165283, -0.38876819610595703, 0.0013772242236882448, -0.3588356077671051, 1.525752305984497, -0.4777885377407074, -2.1902072429656982, -0.19228127598762512, 0.4688236713409424, 0.4970133900642395, 0.41615110635757446, -1.6933624744415283, -0.18446969985961914, 0.6158924698829651, -0.9504620432853699, 0.5609886646270752, 0.6745485663414001, 0.4485938847064972, -0.2497880458831787, -0.3684001564979553, -1.1913573741912842, -0.7866283655166626, -0.4545040726661682, 0.049366239458322525, 0.5704173445701599, -0.9967281818389893, 0.48923930525779724, 0.09370376914739609, -0.789289116859436, -1.2283494472503662, -0.8481146693229675, -1.1186983585357666, -1.1521931886672974, 0.2854660749435425, -1.7721632719039917, -0.3961089551448822, -0.43229880928993225, 0.6200626492500305, 0.45851001143455505, -0.8701716065406799, -0.6158291101455688, 0.3413253724575043, 1.0987458229064941, -0.36189574003219604, -0.3487774729728699, 0.7391786575317383, 0.32511791586875916, 0.08627431094646454, 0.8405270576477051, 0.03125595301389694, -0.021803276613354683, 0.04709435999393463, -0.054001811891794205, -1.9847779273986816, -0.5059455037117004, 0.2843281924724579, -0.7690712809562683, -0.1387011557817459, 0.31884443759918213, 1.1175826787948608, 0.08908472955226898, 0.7125126123428345, 0.1511354148387909, -0.25574934482574463, 0.7376613616943359, 0.5583757758140564, -0.16054300963878632, 0.7481858134269714, -0.2783045172691345, 0.37847140431404114, 0.4098116159439087, 0.1392183005809784, 0.012912203557789326, 0.6262338757514954, -3.302121877670288, 1.1282074451446533, 0.2911452054977417, 0.9524783492088318, -1.021560788154602, -0.37555602192878723, -2.4226479530334473, -0.16004204750061035, -1.0862970352172852, -0.1247989684343338, 0.5405951142311096, -1.386273980140686, 0.8781676292419434, -0.5589461922645569, -0.9396657943725586, -0.11631875485181808, 2.0004773139953613, 0.7369781136512756, -0.41676265001296997, -0.4520871341228485, 0.5021586418151855, 0.9503116011619568, -0.7025932669639587, -1.2097082138061523, 0.699737012386322, -0.48465603590011597, -0.8685116767883301, 0.7628063559532166, 1.7166166305541992, 0.07596275955438614, -0.23119717836380005, -0.09223771095275879, -0.5364028811454773, 0.9960364103317261, 1.0556650161743164, -0.24682043492794037, 0.20010273158550262, 0.7329738140106201, -0.58382648229599, -0.6778948903083801, -1.5219545364379883, -0.674369752407074, -1.8483805656433105, -0.33472636342048645, -1.247833251953125, -0.5975298285484314, 0.5173822045326233, 2.9685585498809814, 0.6798480749130249, -0.1780276745557785, 0.7536721229553223, -0.39652568101882935, 1.876713514328003, 0.5340633392333984, 0.6111360192298889, 0.9289978742599487, 0.37951451539993286, 0.09740965068340302, -1.1077985763549805, 0.11809135228395462, 0.44247111678123474, -0.09658654034137726, -1.1924378871917725, -0.6669938564300537, -0.33786889910697937, -0.5322823524475098, -1.2508432865142822, -0.39564427733421326, -0.6154916286468506, -0.008712214417755604, 1.28977632522583, -0.18340660631656647, 1.117889404296875, -0.9157536625862122, 0.6234884262084961, 0.352612167596817, -1.0280038118362427, -1.2194368839263916, -0.9026867151260376, 1.6390016078948975, 0.3289884924888611, -0.43202078342437744, -0.25928765535354614, 0.3348795473575592, -0.8362287282943726, -0.4273761212825775, -1.2214964628219604, 0.806194007396698, 0.7647033333778381, -2.04537034034729, 0.30021005868911743, 0.08610398322343826, 0.4713987708091736, -1.0553927421569824, -1.0694470405578613, -1.1105722188949585, -0.25028952956199646, 0.9052841663360596, -0.9281092882156372, -0.6671974062919617, 0.8424358367919922, 2.5216662883758545, -0.8996856808662415, 0.8009786009788513, -1.2639952898025513, 0.7405681610107422, -0.6522684097290039, -0.06726094335317612, 1.514950156211853, 0.2685820758342743, -0.6785098314285278, 0.347282350063324, -0.3151547312736511, -0.4470621943473816, -0.09324150532484055, 1.3654905557632446, 1.0254820585250854, -2.462771415710449, 1.3041056394577026, 1.0838476419448853, 1.4584932327270508, 0.21422024071216583, -1.0919110774993896, -1.1222494840621948, 0.2461177110671997, 1.2165052890777588, 0.7364955544471741, -0.5170437693595886, -0.8457851409912109, 1.2245732545852661, 1.6057565212249756, -0.43597912788391113, 1.5395625829696655, 0.944192111492157, -0.13841965794563293, -0.8898860812187195, 0.6523417830467224, -1.7685836553573608, 1.1070935726165771, -0.0684611052274704, -0.2963339388370514, 0.10989399999380112, 0.04834762588143349, 0.07131559401750565, -0.016455022618174553, 0.865018367767334, -0.328227162361145, 0.4522591233253479, -0.6479153037071228, 0.21344329416751862, -0.6941941976547241, -0.4903642237186432, -1.340671420097351, 1.764735460281372, -0.44798821210861206, 1.2130141258239746, 0.07829546928405762, -0.8807539939880371, -0.4079520106315613, -1.3467400074005127, 0.4944736361503601, 0.8768532872200012, -0.02699682116508484, -0.6644343733787537, -1.4360251426696777, 0.40870344638824463, -0.5438187718391418, -0.847669243812561, 0.16480956971645355, -1.0968163013458252, 1.485917568206787, 0.20352251827716827, -0.5170263648033142, 1.0223544836044312, -1.5395739078521729, 1.95443856716156, 0.2521931827068329, 0.7056382894515991, -0.9601872563362122, -0.4710480868816376, 0.08540551364421844, -0.451448917388916, -1.0216723680496216, 0.5535990595817566, 0.25011318922042847, 0.5764914155006409, 1.1302977800369263, 1.3809008598327637, -0.29835599660873413, 0.04507043957710266, 0.3958854377269745, 0.2671476900577545, -1.1421998739242554, 0.008497156202793121, 0.7756159901618958, 0.00045934997615404427, -0.8344583511352539, -1.0749170780181885, 0.1583261638879776, -1.1782379150390625, 2.659080743789673, -1.2289786338806152, 0.5536338090896606, -0.6726040244102478, -0.3413085341453552, 0.39972150325775146, -0.29745450615882874, 0.8178991675376892, 1.8355803489685059, -1.1055641174316406, -0.40441396832466125, -1.6895530223846436, 0.872791051864624, -0.7144861817359924, -1.4082869291305542, -0.4578615128993988, 0.2828010320663452, -0.1496206521987915, 1.0453611612319946, 0.1267833709716797, -0.5515549778938293, -0.48578962683677673, -1.0343292951583862, -1.750518560409546, 1.5001963376998901, -0.5366340279579163, 1.310990571975708, -0.9967414736747742, 0.12168748676776886, 0.8430612683296204, 1.2543178796768188, 0.3471275269985199, 0.4248947203159332, -0.9952700734138489, 1.0372450351715088, 0.12522143125534058, -0.3105594515800476, 1.873707890510559, -1.056644320487976, -0.5127726197242737, 0.8632939457893372, 0.8941678404808044, 0.49063578248023987, 0.9434035420417786, -0.6559981107711792, 0.9037331938743591, 0.6563552021980286, -1.7830110788345337, 0.8229187726974487, 0.09801424294710159, -0.06792944669723511, 1.9835437536239624, 1.7446305751800537, 0.8084818124771118, 0.23377405107021332, 0.5293409824371338, -0.6880690455436707, 0.7388811111450195, -0.9442137479782104, -0.7730311155319214, -1.0596164464950562, -0.8737848401069641, 0.5868802666664124, -0.10211224853992462, -0.04593219608068466, 0.2559845447540283, 0.17773878574371338, -1.2824558019638062, -0.7390560507774353, 0.14398398995399475, -0.748811662197113, -0.376015305519104, -1.4793144464492798, 0.33457043766975403, -0.6572544574737549, -0.19065405428409576, 0.6189701557159424, 0.07010947167873383, -1.5657143592834473, 0.016166776418685913, 0.5369189381599426, 0.6134403944015503, -0.7135782837867737, 0.14751671254634857, 1.0439388751983643, -0.1488671600818634, 0.9217827916145325, -0.8033159375190735, -1.0149054527282715, 0.4949493408203125, 0.5956212282180786, -1.689132571220398, 0.6937698721885681, 0.9549640417098999, -0.17149296402931213, 1.1231187582015991, 0.38973018527030945, -0.7474474906921387, -0.6422476768493652, 0.8172405362129211, -2.9427103996276855, -0.3309880197048187, 0.8864386081695557, 0.5567861199378967, -0.21118797361850739, 0.47854357957839966, 0.08355794847011566, -0.2010030895471573, -1.1869046688079834, -1.37962806224823, 0.37126263976097107, 0.9828483462333679, -1.100815773010254, 0.9525412321090698, 1.7856358289718628, 0.5967345833778381, -1.614482045173645, 0.23030905425548553, -0.3636024296283722, 0.5656431913375854, -0.5725180506706238, 0.08327382802963257, 1.1476373672485352, 0.32823923230171204, 0.2008724957704544, -0.04423993080854416, -1.8160853385925293, 1.9628822803497314, -0.0932704284787178, -0.27925094962120056, 0.2703363001346588, 0.6863425970077515, -0.17109525203704834, -0.27218854427337646, 0.9268754124641418, -0.07674097269773483, -1.2465437650680542, -0.7820402383804321, 0.9889901876449585, -1.0071461200714111, -0.24487422406673431, 0.060200415551662445, -1.1951141357421875, 1.0816453695297241, 0.2707279324531555, -0.9280776381492615, 0.3087179958820343, -0.6444831490516663, 0.2282862812280655, 2.345921516418457, 2.080967903137207, -0.20748302340507507, 0.5402897000312805, -0.6653008460998535, -0.5546238422393799, -0.2728580832481384, -1.3899738788604736, 0.28786659240722656, 0.09070786833763123, 0.3713832199573517, -0.9753890633583069, 0.8483800292015076, -1.4690396785736084, 0.7687921524047852, 0.013713402673602104, 0.2169896811246872, -0.464267373085022, -0.9361892342567444, 0.4476635158061981, 1.1430810689926147, 0.6886779069900513, -3.0233914852142334, 0.5201163291931152, 0.9729226231575012, 0.9253660440444946, -1.562913417816162, 2.90034556388855, 0.8658181428909302, 0.1811646968126297, 0.6495434045791626, -1.1796579360961914, 1.1586142778396606, -0.10765901952981949, 0.6049025654792786, 2.814953565597534, 0.2864372730255127, 1.855276107788086, -0.6131475567817688, 1.7753654718399048, 0.6538972854614258, -2.2523181438446045, -0.4559118449687958, -0.7864721417427063, -2.0011372566223145, -0.7099351286888123, -0.5512375831604004, 0.5178165435791016, -0.9113705158233643, 0.6182663440704346, -0.8323790431022644, 0.10599985718727112, 0.48168307542800903, 2.257568120956421, 0.09783215820789337, -1.365654706954956, 1.064300775527954, 0.7293431758880615, -0.8170416355133057, 0.36464497447013855, -1.4256564378738403, 2.1703460216522217, 0.2699885964393616, 0.1640608161687851, -0.41817206144332886, -0.866071879863739, -1.402867317199707, -0.4346626102924347, -0.09500041604042053, -2.0139381885528564, -0.28093811869621277, -0.1887853890657425, 0.6691115498542786, 0.4412221610546112, 0.8001927137374878, 0.68561190366745, 1.3563737869262695, -0.15150709450244904, -0.5427432060241699, 1.3679249286651611, -1.0534827709197998, 0.8739145398139954, 0.9863240122795105, 1.272506833076477, 1.277932047843933, -0.13535049557685852, 1.2391778230667114, -0.12986598908901215, 0.5114820003509521, 0.2530131936073303, -0.10952062159776688, -0.21844226121902466, 1.3745125532150269, -1.0611722469329834, -1.194646954536438, 0.09231605380773544, 0.36680781841278076, -0.6628989577293396, -0.5195540189743042, 0.7316895723342896, 0.08335953205823898, -1.2286337614059448, -1.478376030921936, 0.3897683918476105, -0.482856810092926, -0.6832780241966248, -1.1742078065872192, -1.064045786857605, -0.10083802044391632, -1.2992771863937378, -0.5212548971176147, -0.08583217114210129, -1.5953011512756348, 0.8814712762832642, -1.1479815244674683, -0.6406238079071045, 1.1431365013122559, 1.1139252185821533, 1.3195918798446655, -0.2416079193353653, 1.2055882215499878, 1.000839352607727, 0.0011229651281610131, 0.662051260471344, -0.007438917178660631, 0.7596855759620667, 2.2323825359344482, -0.1216696947813034, 0.42907243967056274, 0.03767557814717293, 2.8994431495666504, 0.7394824028015137, -1.3990727663040161, -1.3573851585388184, -0.007438441272825003, -0.8007088303565979, -1.662007451057434, -0.4464683532714844, -0.7540085911750793, 0.3660620152950287, 1.2872869968414307, 1.4809672832489014, 0.9532468318939209, -0.7810035347938538, -0.25959697365760803, 0.22278478741645813, 1.0613420009613037, -0.18096975982189178, -1.210030198097229, 1.2557295560836792, 0.5866877436637878, -0.02449299581348896, -0.15762825310230255, -0.759219765663147, -1.14323091506958, 0.32313641905784607, 0.7730247974395752, -0.6011708378791809, -1.3044331073760986, -1.3079397678375244, -0.5100059509277344, 1.7393717765808105, 0.3267902731895447, -3.2030258178710938, -0.10638594627380371, -0.9066933393478394, 0.4992576539516449, -1.7772215604782104, 1.0958375930786133, 1.0516806840896606, 1.3423875570297241, -0.23256240785121918, -0.9677107930183411, -0.2648254930973053, 0.3460802435874939, -1.2492151260375977, -0.7461053729057312, -0.10754909366369247, -0.6929076910018921, -1.7049520015716553, -1.6992871761322021, -1.7708892822265625, -1.9645922183990479, -0.09077544510364532, 0.08006452023983002, -0.07137155532836914, 0.8213077783584595, -0.6785699129104614, -2.066708564758301, -0.6728013753890991, -1.289380669593811, 1.3467998504638672, -0.9138545393943787, 0.13913454115390778, -0.1640031933784485, 1.0726304054260254, 0.1304703652858734, 1.6341232061386108, -0.5048233866691589, 0.04951002076268196, -0.3740195631980896, 0.22951333224773407, 0.5680785775184631, 0.05065092071890831, -1.2064024209976196, 0.6005776524543762, 0.09444595128297806, 1.1212431192398071, -0.7266241312026978, 0.731573224067688, 1.6274120807647705, 0.37437522411346436, -0.8823179602622986, -1.052438497543335, -0.54213947057724, -0.6828690767288208, 1.447653889656067, -0.4728521704673767, -0.234233558177948, 0.39545738697052, 0.2659919261932373, -0.9045827388763428, -1.6662784814834595, -0.07525581866502762, 1.2225168943405151, -1.510094404220581, -0.15813983976840973, 0.9638411402702332, 0.4658263921737671, 0.44285130500793457, -0.5058610439300537, -1.053094744682312, -0.4681314527988434, 0.5769825577735901, 1.0755118131637573, -2.002133846282959, -0.5673362016677856, 0.31390079855918884, 0.2776978313922882, 1.1647632122039795, 0.183858722448349, -1.3418116569519043, 2.2588515281677246, -0.9197337627410889, 0.7116552591323853, -0.15442533791065216, 0.9753761887550354, 0.739456057548523, 0.881464958190918, -0.5843885540962219, 1.4380757808685303, 0.12380681186914444, -1.1640375852584839, -1.8131530284881592, -0.08700208365917206, -1.0691999197006226, 0.3952089846134186, -0.66374272108078, 0.17341963946819305, 0.22257651388645172, 0.5313846468925476, 1.788957118988037, 0.4997076094150543, -0.5168264508247375, -0.6359522342681885, -0.7437974214553833, 0.4577799439430237, 0.7063556909561157, 0.26505717635154724, 0.472144216299057, 0.6038675904273987, 2.2003493309020996, -0.6439828872680664, 0.6258329153060913, 0.4476417899131775, 2.2207086086273193, -0.3577989935874939, -1.5364093780517578, -2.0292701721191406, -1.4889825582504272, -0.5953917503356934, -1.0871583223342896, -0.020001566037535667, -0.5828786492347717, 0.5794724822044373, 1.9127496480941772, 0.1744474470615387, 0.7248179912567139, -1.1707029342651367, -0.3143209218978882, -1.714266061782837, -1.879944920539856, -0.7475272417068481, -0.9350038766860962, 0.4684951901435852, 1.3007025718688965, -0.019758909940719604, -0.941575288772583, 0.26466140151023865, -0.006263392977416515, 1.619022250175476, 1.6027436256408691, -0.16771197319030762, 1.9784198999404907, 0.5444777011871338, 0.7244871258735657, -0.1473567932844162, 1.2762659788131714, -0.1856544315814972, 1.0162255764007568, -0.15423168241977692, -1.5463298559188843, -0.8732138276100159, 0.04442392662167549, -1.3004175424575806, 1.1853398084640503, -0.5514052510261536, -1.2844029664993286, -1.1087673902511597, -0.12792323529720306, 0.22060009837150574, -0.11337850987911224, -0.7823576927185059, -0.5418914556503296, -0.036258187144994736, 0.11948257684707642, 1.0515855550765991, -0.31648969650268555, -1.769318699836731, -0.9102494120597839, 1.3431403636932373, 0.0032009549904614687, -1.2332565784454346, -0.21616917848587036, -0.7930428385734558, -0.5942596197128296, -1.8142095804214478, -0.452212393283844, 0.8511805534362793, -1.0190867185592651, -1.0362902879714966, -0.3288533687591553, 1.5719298124313354, 1.0818759202957153, 1.336946725845337, -1.051353096961975, 1.6235755681991577, 0.7832404971122742, -1.6944992542266846, -0.7548761367797852, 2.0813488960266113, -1.7106329202651978, -0.5769515633583069, 0.004237919580191374, 0.7705899477005005, 0.592552900314331, -2.5183768272399902, 0.3575977385044098, 1.2246838808059692, -2.09509015083313, 2.9202516078948975, -0.13936415314674377, 0.8646676540374756, -0.7669698596000671, -0.23081092536449432, 0.8248317241668701, 0.1955013871192932, 1.105439305305481, 0.6038531064987183, 1.9408376216888428, -0.3143891990184784, -0.5797612071037292, -0.7990403771400452, 1.2167274951934814, 0.2864023447036743, -0.8750869631767273, 0.9182074666023254, 0.5184387564659119, 0.19409863650798798, 0.831511378288269, -0.3762069046497345, -0.7533491253852844, -0.05844232067465782, -0.7043699026107788, -2.031993865966797, -1.61663019657135, 0.14155830442905426, -0.6625578999519348, 1.323727011680603, -0.6543663144111633, -0.40133288502693176, -1.8847551345825195, -0.5857959389686584, -1.377381443977356, -0.6387884616851807, -0.5803098082542419, 0.43949609994888306, 1.2311512231826782, -1.0146777629852295, -0.37208452820777893, -1.0401837825775146, -0.9835254549980164, 0.1630006730556488, -0.41611427068710327, 0.23290418088436127, -0.09259749203920364, 0.6475772857666016, 0.8937618136405945, -0.01930512674152851, 0.8895531892776489, -0.09815943986177444, 0.7042637467384338, 0.3096241354942322, 0.10476790368556976, -1.6643604040145874, -0.7295630574226379, -1.5176589488983154, 1.439872145652771, -0.8583559393882751, 0.715965986251831, -0.906536877155304, -1.2973536252975464, 1.2750977277755737, -0.7701430916786194, 0.8848106861114502, 0.8254607319831848, -0.6795915961265564, -1.3122328519821167, 0.13293492794036865, 1.0032227039337158, -0.897071123123169, 0.5058799386024475, 2.8261327743530273, -0.7964411377906799, 0.2696426212787628, -0.2830551266670227, -1.9198956489562988, -0.5337244272232056, 0.5900159478187561, -0.5271416902542114, 1.924914002418518, -0.6381815671920776, 0.25486519932746887], [-0.09197954088449478, 1.548754334449768, 0.9771835803985596, 0.1167319118976593, 1.1149243116378784, 0.22179998457431793, -1.0238343477249146, 0.0025670749600976706, 0.4431115686893463, 0.8025240898132324, 0.29504281282424927, -0.7166745066642761, -0.6817870140075684, -0.8963952660560608, -0.6853624582290649, 0.014154970645904541, 0.8998894691467285, 0.23208454251289368, 0.5588172674179077, -0.9471086263656616, 1.6525720357894897, 0.4211159348487854, -0.3776118755340576, 0.8754380345344543, 0.38350725173950195, 0.8066457509994507, -0.3775724768638611, 0.7696487307548523, 0.6076975464820862, 0.565774142742157, 0.1660795956850052, 0.9144629240036011, 0.8609915375709534, -0.07480346411466599, -0.8857663869857788, 0.027302686125040054, 0.2882212698459625, -0.2836596965789795, 0.3623189926147461, -1.5105490684509277, -2.213315010070801, 0.10517922788858414, -0.9674957990646362, 0.6665129661560059, -0.3338490128517151, 0.666408360004425, -0.9802273511886597, -0.049150578677654266, 1.420853853225708, -0.6557679772377014, 0.4073215126991272, 1.6676816940307617, -1.2380934953689575, 0.7734701037406921, 0.3156574070453644, 0.5759086012840271, 0.5064064860343933, -0.07892583310604095, 1.4007363319396973, 0.5307328701019287, -0.265036016702652, 0.8432504534721375, 0.6803247928619385, -0.7292153835296631, 0.9692328572273254, -0.4244820475578308, -0.7598552107810974, 0.7637813687324524, 1.6646974086761475, -0.21920053660869598, 0.35331061482429504, -0.1304975301027298, -0.7687095403671265, 0.025265229865908623, -1.853472113609314, -1.269356608390808, -0.4475646913051605, -0.05692390352487564, 1.0580418109893799, 0.09337314963340759, -1.5777182579040527, 0.35227730870246887, -0.9176411032676697, -0.2536160945892334, -1.5201107263565063, -0.7370873689651489, -1.0823841094970703, -0.4855935275554657, 0.171885147690773, -1.343767523765564, -0.28251582384109497, 0.38759279251098633, -0.7496361136436462, -0.501984179019928, 1.6974947452545166, -1.0570249557495117, 0.7410402297973633, -0.8594262599945068, 0.0893998071551323, -0.5376036167144775, 0.20213459432125092, 0.7725371718406677, 0.7058078646659851, -0.10628670454025269, 0.21143898367881775, -0.8168293833732605, -0.6115673780441284, -1.1645925045013428, -0.0988878458738327, -0.21048784255981445, -1.203774094581604, 0.12759234011173248, -0.08320558816194534, 0.6530029773712158, -0.32062363624572754, -1.4872944355010986, -0.7156357765197754, -1.5375473499298096, -1.2461087703704834, -0.48366779088974, -0.36793768405914307, -0.36639872193336487, -1.3673361539840698, 1.4113669395446777, 0.8456017971038818, -0.09011854231357574, 1.1078559160232544, -0.6335797905921936, -0.05608328804373741, -0.13121968507766724, 0.8154922723770142, -0.012514744885265827, -0.19423475861549377, 0.45216190814971924, 0.28689634799957275, -1.6980654001235962, -0.39957359433174133, -1.7774242162704468, -1.581059217453003, -1.1782252788543701, 0.6099914312362671, 1.1054599285125732, 0.9911484122276306, -0.08700674027204514, -1.2308794260025024, 0.8809064626693726, -0.40087348222732544, 0.8154459595680237, 1.523821234703064, 0.25405141711235046, -1.0557056665420532, 0.5508083701133728, -1.21830415725708, -0.690085232257843, -0.6844912171363831, 0.18674927949905396, -1.047519326210022, 0.18121610581874847, -0.4672533869743347, 1.2485510110855103, -0.817997395992279, -0.7516544461250305, -1.0401924848556519, 0.2710118293762207, -1.0019935369491577, -1.3655476570129395, 0.23649299144744873, 1.3953771591186523, -0.7228115200996399, -1.535845160484314, -1.2839473485946655, 1.2772620916366577, -0.6493673324584961, 0.7514732480049133, -1.1429985761642456, -1.0396878719329834, -0.9829580187797546, -0.50328528881073, 1.593347191810608, 1.3976351022720337, 0.9023016691207886, -0.44166600704193115, -1.5869745016098022, 1.6374603509902954, 0.9646040797233582, 0.6312767863273621, 0.6915220618247986, 0.13618174195289612, -1.3837050199508667, 1.0201789140701294, 0.12649191915988922, -0.3635413944721222, -0.16268667578697205, 0.045934926718473434, -0.5729370713233948, -0.32193154096603394, -0.14020518958568573, -0.5779296159744263, 0.7688979506492615, -0.9776196479797363, -0.252224326133728, -1.1702207326889038, -2.1465611457824707, 1.0548330545425415, -0.11728981137275696, -0.7043696641921997, -1.718543291091919, -0.6954430341720581, -0.6088486909866333, 0.020265890285372734, -0.4447536766529083, 0.8955366015434265, -1.4043101072311401, 0.044527288526296616, 1.2149983644485474, 0.06994035094976425, 0.6225295662879944, 0.6272217631340027, 0.35771405696868896, 0.40642377734184265, -1.1070706844329834, -1.6997489929199219, 0.01823382079601288, -0.6235368847846985, 0.34726762771606445, -0.5835778713226318, -0.6290651559829712, 0.029741179198026657, 0.3800111711025238, 0.10619423538446426, -1.9192906618118286, 1.0437318086624146, 0.9235702753067017, -0.19640126824378967, -0.03430938720703125, 0.0615113191306591, -0.09406478703022003, -0.38736769556999207, 0.2599787712097168, 1.1357746124267578, -1.0582411289215088, 1.6733640432357788, 0.2809978127479553, -0.165040522813797, 0.8764658570289612, 0.4423666000366211, -0.26995164155960083, 1.01609468460083, -0.16789788007736206, -0.006776327732950449, 0.2846188545227051, -0.4049493670463562, 1.1073613166809082, -0.2319130152463913, -0.4589923620223999, -0.722599983215332, -0.4264940619468689, 0.4604828655719757, -0.9190095663070679, -1.3504729270935059, 1.2687593698501587, 2.913182497024536, -0.26656532287597656, 1.0005881786346436, -0.16960661113262177, 2.013270139694214, 0.8830000758171082, 0.748383104801178, -0.007233987096697092, 0.23718860745429993, 0.35037824511528015, 0.3722997009754181, -0.8141306042671204, -0.10456693172454834, 1.5016403198242188, 0.235054150223732, -1.6355383396148682, -0.29866325855255127, -0.543438732624054, -0.7287760972976685, -1.795655369758606, -0.5914396047592163, -0.21359306573867798, 0.7872204184532166, -1.6239150762557983, 0.3102201819419861, -0.76688551902771, -0.28966301679611206, 0.7716389894485474, 1.6838945150375366, -0.6075021028518677, 1.0836840867996216, -0.06691430509090424, 0.12760022282600403, 1.8094360828399658, -0.07555797696113586, 0.03195704147219658, 0.9168789982795715, -0.5153473019599915, -1.8898284435272217, -0.855510413646698, 0.9397560358047485, 0.052013564854860306, -1.1510303020477295, -0.914678156375885, -0.3719761371612549, -2.619983673095703, -1.4610793590545654, 0.6242300271987915, 1.9069936275482178, -0.5884448289871216, 0.0056559424847364426, 1.1138077974319458, 0.11619783192873001, -0.6434529423713684, 1.001522183418274, -0.0002751844876911491, -0.5271030068397522, 2.156627655029297, -0.37574252486228943, 0.24507540464401245, 1.300045132637024, 0.9051212668418884, -0.24614398181438446, -0.6627622246742249, -0.2518848478794098, 0.41434329748153687, 0.11596024036407471, -0.5718147158622742, -0.17146095633506775, 2.31894850730896, 0.8877043128013611, -0.2710855305194855, 0.25291532278060913, 0.782374382019043, -0.6121050119400024, 0.6995575428009033, -1.5799201726913452, -0.46360376477241516, -0.8216090798377991, -0.962862491607666, 0.4963134229183197, -1.465911865234375, -1.5838929414749146, -1.114315390586853, -0.7961732149124146, -0.876292884349823, 0.4805600941181183, 1.231272578239441, 2.3620145320892334, 0.25380632281303406, -0.8474288582801819, 0.7728337049484253, -0.9486239552497864, -0.7441027164459229, -1.2032326459884644, 0.009505365043878555, -1.4606965780258179, 0.12330854684114456, 1.0917781591415405, -1.4572089910507202, -1.3187122344970703, -0.3151969015598297, 0.6574606895446777, 1.0932203531265259, 0.928602397441864, -0.47107046842575073, 0.9911189079284668, -1.1117080450057983, 1.4509131908416748, -0.5008916258811951, -0.9666702151298523, 1.0395092964172363, 0.19162073731422424, -1.657256841659546, 0.9338251948356628, -0.09828025847673416, -0.31872445344924927, 0.48620739579200745, 0.20447446405887604, -1.031014084815979, -1.9812712669372559, -0.40727949142456055, 1.4984793663024902, 0.1475706249475479, -1.1410306692123413, -1.3762109279632568, 0.9630428552627563, -0.12974578142166138, -0.14794597029685974, -0.47483715415000916, 1.075966238975525, -0.5516186952590942, -1.8497722148895264, 0.6070080995559692, 1.2387069463729858, -1.0341273546218872, 3.2542240619659424, -1.1888660192489624, -0.34322628378868103, 0.9705236554145813, 0.10860785841941833, 0.3318434953689575, 0.010197409428656101, -0.31315499544143677, -0.250552237033844, -1.6456340551376343, 0.8215214610099792, 0.42281487584114075, -0.32516956329345703, -0.7372300028800964, 0.17950044572353363, -1.227649211883545, -0.17143474519252777, 0.13249875605106354, -1.4048467874526978, -2.170043468475342, -0.30975762009620667, 0.0884048342704773, 0.789636492729187, -1.0492582321166992, -0.45113736391067505, -0.4967076778411865, -1.599634051322937, 0.4404524266719818, -0.8046773076057434, -1.1667474508285522, -1.154541254043579, 0.5475190281867981, 0.1181088387966156, -1.9707111120224, -1.4815980195999146, -0.6531388163566589, -0.5390260815620422, 0.019110530614852905, -1.8029398918151855, 0.287284255027771, -0.29342615604400635, -1.2485909461975098, 1.2694634199142456, 0.5828037858009338, 1.3411040306091309, 0.07032085955142975, 1.2565324306488037, 0.17119094729423523, -0.14723078906536102, -0.865943431854248, -1.185329556465149, -0.47867459058761597, 0.7675994634628296, 1.7919706106185913, -1.1417921781539917, 0.6171316504478455, 0.63653963804245, 0.5190598964691162, 1.1583199501037598, -1.6134732961654663, -0.03893428295850754, -0.1651909053325653, 1.3038196563720703, -0.21641512215137482, -0.08182758837938309, 1.0110934972763062, 1.440456509590149, -0.01874135434627533, -0.8318093419075012, 0.0640522837638855, -0.21224810183048248, -1.5014764070510864, 1.6111133098602295, -1.3952324390411377, 0.7753510475158691, 0.502586305141449, 1.4063682556152344, -0.011826795525848866, -1.3276852369308472, -0.2720952033996582, 1.5252299308776855, 0.8311395645141602, -0.8879871964454651, 0.2930550277233124, -0.25946688652038574, -0.4718797504901886, -2.1280157566070557, -1.2808656692504883, -0.5700953006744385, -0.31206634640693665, -0.7025543451309204, -0.19609801471233368, -0.4474601149559021, -0.017055463045835495, 0.644997775554657, 0.8546508550643921, 0.6714890599250793, -0.12710945308208466, -0.5336552858352661, 1.2799744606018066, 0.9532868266105652, 0.26040059328079224, 1.4669326543807983, -1.6383075714111328, 1.3813996315002441, 1.0201351642608643, 0.4943488538265228, -0.32194530963897705, -0.4033942222595215, -0.18745122849941254, 0.3971211314201355, -0.2585109770298004, 1.0656590461730957, 0.3648228049278259, -0.6788039207458496, 0.014375776052474976, 0.9134191870689392, 1.256662368774414, 2.572197675704956, 1.0671170949935913, -0.5910331010818481, 1.8862351179122925, 0.20149961113929749, -0.32614627480506897, 0.47391828894615173, -1.3785381317138672, 0.3953646719455719, -0.07527723163366318, 0.6687609553337097, 0.8922878503799438, 0.0665711909532547, -0.18301096558570862, 0.8628617525100708, -0.3826364576816559, 0.5742012858390808, 0.6970048546791077, 0.0986349880695343, -0.5451979041099548, 0.4458523690700531, 1.0651882886886597, -0.014107153750956059, -0.2287117838859558, 0.5271090865135193, 0.9600326418876648, -0.7717476487159729, 0.5652420520782471, -1.0906893014907837, -0.24493353068828583, -1.1333063840866089, 1.5795114040374756, 0.3508947193622589, -0.378823459148407, 1.2292184829711914, 0.2988223135471344, 0.4450734853744507, -0.3369002342224121, -0.9526840448379517, -0.25538498163223267, -0.04384085536003113, 1.3603534698486328, -0.6234083771705627, 0.6846524477005005, -1.297524333000183, -1.4388160705566406, -0.5370689630508423, 0.748132050037384, 0.7084143161773682, -1.3544368743896484, -0.11922731250524521, -0.7851177453994751, 2.851550817489624, -0.8278117179870605, 0.9309861063957214, 0.461758553981781, 0.01771535538136959, -0.6210812330245972, -1.1763319969177246, 0.665318489074707, 0.23605976998806, -0.6706646084785461, -0.30178821086883545, 1.6507635116577148, 0.6227458119392395, -0.2673227787017822, -0.5406274199485779, 0.5908700227737427, -1.8320413827896118, 1.4149227142333984, -1.3934248685836792, 0.3342726230621338, 0.6848856210708618, 0.7948859333992004, -0.7452711462974548, 0.6678869724273682, -0.41892340779304504, 1.2267630100250244, 0.675176203250885, 0.08120012283325195, 0.18129688501358032, -0.1790871024131775, 0.13596120476722717, 0.04203541576862335, 0.5483857989311218, 1.289345383644104, 0.3916786313056946, -0.04704741761088371, 1.270015001296997, 1.3097156286239624, -0.8259764909744263, 0.38271504640579224, -0.18882593512535095, -0.47442781925201416, 1.5427802801132202, 1.2312538623809814, 1.9822502136230469, 0.5603798627853394, -1.3614898920059204, -0.2026749700307846, -0.04042142629623413, 0.6645458936691284, -0.45344817638397217, -0.20657584071159363, -0.06401047855615616, -1.2729148864746094, 1.2018351554870605, 2.5400757789611816, -0.45839571952819824, -0.3122853636741638, 0.0465962290763855, 0.20620854198932648, -0.25692903995513916, 0.8475567102432251, 0.07530800998210907, -0.1945955902338028, -1.1313077211380005, -1.4589111804962158, 0.22545821964740753, -0.38050875067710876, -0.6196080446243286, -0.9556311368942261, -2.1190483570098877, 0.3453180193901062, 0.8210001587867737, 2.2372896671295166, 0.0002665282809175551, -0.6988382339477539, -0.2941948175430298, 0.8651607036590576, -1.6444650888442993, 0.8189254999160767, 0.14624010026454926, 0.6940029859542847, 0.37623992562294006, 0.3879374861717224, -0.09539850801229477, -2.743558168411255, 0.3434181809425354, 0.029087040573358536, 0.5612866878509521, -0.519027054309845, 0.5251033306121826, 0.7365147471427917, 1.0838901996612549, -1.3265713453292847, 0.41957610845565796, 0.7293158769607544, 0.16174066066741943, 0.3100533187389374, 0.44859999418258667, 2.4025771617889404, 1.1114252805709839, -0.4524126350879669, 1.7749887704849243, -1.14300537109375, -1.0298372507095337, 0.7415269613265991, 0.023311099037528038, -0.21022020280361176, -0.7060803771018982, 0.03680715709924698, -0.2021060585975647, -0.08261929452419281, 0.24771040678024292, 0.6221095323562622, -1.2175397872924805, 0.5442546606063843, -0.5368449687957764, 0.6983458399772644, -0.0023137324023991823, -1.7615920305252075, -0.16505762934684753, -0.1444053053855896, 1.6404999494552612, -0.3548034429550171, 0.7487850785255432, 2.321491003036499, 1.1265525817871094, 0.20584416389465332, 0.22606009244918823, -0.24791906774044037, 1.4231499433517456, 0.5212437510490417, -0.2874011695384979, 1.1073476076126099, 0.6239124536514282, 0.47329428791999817, -0.4114686846733093, -1.4855881929397583, -0.3769540786743164, 0.8460796475410461, -0.10496446490287781, -2.257075548171997, -2.3416950702667236, 0.8738386034965515, -0.4671301543712616, 0.2375895380973816, 0.11177507042884827, -0.42583781480789185, -1.3613941669464111, 2.4302978515625, -1.5850316286087036, 0.07842154800891876, 0.5922843813896179, 0.9916039109230042, 0.008362812921404839, -2.004055976867676, -0.7653332352638245, -1.5982576608657837, 2.363100290298462, 0.3427247405052185, -0.8154739141464233, -0.9027173519134521, 0.10744831711053848, -0.32156869769096375, -0.43840286135673523, 0.2575854957103729, -0.48863816261291504, 0.918454110622406, 1.1574327945709229, 1.0820599794387817, -0.2528745234012604, -1.763770580291748, 0.8641836047172546, 0.15635035932064056, -0.6790114045143127, 1.3081645965576172, -0.1364976465702057, 0.6023095846176147, -1.5194191932678223, -2.7657392024993896, 0.20053446292877197, -1.9657710790634155, -0.4303273558616638, -0.04078061878681183, -0.2580261528491974, -1.8653368949890137, 0.8299694657325745, -0.8301735520362854, -1.8621972799301147, -0.0654173195362091, 0.7004090547561646, -0.11007117480039597, -0.9050339460372925, -0.34428203105926514, -0.1510615050792694, -0.22170689702033997, -0.8897293210029602, -0.02302820235490799, -0.7072193622589111, -0.37099531292915344, -0.12301618605852127, 0.7178099751472473, -1.7453762292861938, -0.01512399222701788, -1.0164870023727417, 0.5086143612861633, 0.9426534175872803, -0.2154514640569687, 1.2794796228408813, 0.5239250659942627, 1.4986310005187988, -1.5860211849212646, -2.0831360816955566, 0.7601078748703003, 0.8547809720039368, 0.584953248500824, 0.19463799893856049, 0.7219797968864441, -1.1908575296401978, 1.5997686386108398, 0.003810296067968011, -1.184271216392517, 0.7059723734855652, 1.4212327003479004, -0.3927115201950073, -0.782256543636322, 0.6924805045127869, 1.2682116031646729, -0.4302130341529846, -2.0236942768096924, 0.21212103962898254, 1.7822526693344116, -1.3904225826263428, -0.3522370457649231, -0.9145415425300598, -1.3221683502197266, -1.0424071550369263, 1.0345810651779175, 2.6040170192718506, -0.506149172782898, 0.8317529559135437, -0.827407956123352, -1.271992564201355, 0.8132001161575317, 0.3512036204338074, 1.7200254201889038, 0.5325546264648438, -0.28051987290382385, -0.6682387590408325, -0.26681792736053467, -0.03446205332875252, 0.38078945875167847, 0.3005874752998352, 0.27146539092063904, -0.5186654925346375, -0.5309877991676331, -0.8249924778938293, 0.22443416714668274, -0.06032430753111839, 0.201994389295578, 2.0035431385040283, 1.1085035800933838, 0.8963592052459717, 0.2706848084926605, -0.3051500916481018, -1.029464602470398, -0.40945005416870117, -0.6382527947425842, 1.1789699792861938, -0.05204092338681221, 1.0862408876419067, -0.329193115234375, -0.38253679871559143, -0.20336535573005676, 0.36974361538887024, 0.6495239734649658, -0.10427858680486679, -0.8429880142211914, 1.1467108726501465, -0.6977609395980835, 0.39136552810668945, -0.2185998409986496, 0.9724939465522766, 0.5792624354362488, -0.20964443683624268, -0.4529876410961151, -1.058966875076294, -1.4074832201004028, -1.3894391059875488, -0.01641298085451126, 0.10739481449127197, -1.1531009674072266, -0.804159939289093, 0.43290436267852783, 1.3611953258514404, 0.02070697769522667, 0.01697719842195511, 0.30128392577171326, 0.452212929725647, 1.40935480594635, 1.2984247207641602, -1.5084736347198486, 1.0023994445800781, 0.05312998592853546, -0.8802887201309204, 1.862282395362854, 0.3070632815361023, 0.1648605912923813, -0.13276407122612, 0.374270498752594, -1.0272796154022217, -0.766946017742157, -0.7887374758720398, -2.4332892894744873, 0.07170778512954712, -0.3226187825202942, 0.22399108111858368, -0.8391913771629333, 1.0920976400375366, 1.3225674629211426, 0.6256450414657593, -0.1685977578163147, -1.207317590713501, -1.5882242918014526, -0.1883820742368698, -0.4790818393230438, -0.801692545413971, 0.9623836278915405, 0.6287181377410889, -0.2980723977088928, -0.8903425931930542, 1.6011812686920166, -1.243330478668213, 0.29707297682762146, 1.754156231880188, 0.15766441822052002, -2.3287644386291504, 0.17952796816825867, 0.9291699528694153, 0.07504045963287354, -1.0735652446746826, 1.7958539724349976, -0.3659266233444214, 1.803658127784729, 1.7049384117126465, -1.6235299110412598, -0.6257889866828918, 0.8094397187232971, -0.1318342685699463, -0.8125858306884766, 0.10506349056959152, 1.3081605434417725, -0.5174100399017334, 1.2047406435012817, 1.1149218082427979, 0.5472325682640076, -0.5776233673095703, 0.2499629557132721, 0.9612787365913391, 1.7566072940826416, -0.6295698285102844, 1.3498929738998413, 1.7202779054641724, -0.3743247985839844, 0.9824493527412415, 0.07068419456481934, 1.8006937503814697, 0.6677976846694946, -0.19480624794960022, -0.08644454181194305, 0.4491412937641144, -1.551754355430603, -1.2894136905670166, -0.13650567829608917, -2.8655409812927246, -0.35254281759262085, 0.5332863926887512, -0.8442590236663818, 0.5320616364479065, 1.3352270126342773, -0.010068099945783615, -0.9592459797859192, -0.08605290949344635, 0.9110057353973389, -0.7354316115379333, 0.009819908998906612, 1.7826581001281738, -0.44840380549430847, 0.3381003439426422, 1.4198123216629028, 1.6856685876846313, -0.6825463175773621, 0.023932430893182755, 0.9860084652900696, 1.0836575031280518, 0.10917532444000244, 0.05440591648221016, -0.7329919338226318, -1.5651485919952393, 0.24263031780719757, 1.411012887954712, 0.7540016174316406, -1.4116026163101196, 0.3543994724750519, 0.21896983683109283, 0.4526590406894684, -1.4120619297027588, 0.08199828118085861, -0.41024601459503174, 0.8095876574516296, 1.0743423700332642, -0.5280486941337585, -0.5276036262512207, 0.7248638272285461, -0.01899106800556183, -0.3346169888973236, 0.7842546701431274, -2.0056984424591064, -0.1710885912179947, 0.9122495651245117, 2.9716103076934814, 1.419946551322937, -1.1864889860153198, -1.065350890159607, 0.4278342127799988, -0.9023011922836304, -0.326498419046402, 1.0888386964797974, 0.661468505859375, 0.034376468509435654, 1.4938091039657593, -0.7000775337219238, -0.9337141513824463, -0.3101184666156769, 0.2195758819580078, -0.008250799030065536, -1.5239919424057007, 0.941154956817627, -0.39858144521713257, 1.3995041847229004, -1.0506843328475952, -1.1231558322906494, -0.03827401623129845, 1.2799279689788818, 0.4672101140022278, 0.7128696441650391, -0.7753663659095764, 0.49950626492500305, -1.2281882762908936, 0.6828967928886414, 1.2318602800369263, -0.35608476400375366, -1.2330517768859863, -1.313896656036377, 1.1716041564941406, -1.0921486616134644, 0.6733582019805908, 0.07412905991077423, 0.7326030731201172, -1.1642091274261475, 0.46240973472595215, -1.1295734643936157, -0.39213821291923523, -1.0489226579666138, -0.11256687343120575, 0.6017061471939087, 0.3013255000114441, 0.5354135036468506, -0.2020329236984253, 0.7868624925613403, 0.6683822870254517, -0.8157632350921631, -0.4675435721874237, -0.7896380424499512, -0.3321073055267334, -1.1735169887542725, -1.707193374633789, 1.9182835817337036, 0.7181256413459778, -0.24847382307052612, 0.18083423376083374, 2.179115056991577, 0.9868936538696289, 1.1897976398468018, 1.7799588441848755, -1.0166913270950317, 1.2537107467651367, -0.2789187729358673, 1.4275074005126953, 0.0032991066109389067, 0.3571762442588806, -1.7814463376998901, -1.398895025253296, -0.9945363998413086, -1.2562495470046997, 1.6095917224884033, 1.0052826404571533, -1.2541083097457886, -1.3333704471588135, 0.6781489849090576, -0.28442347049713135, 0.24344585835933685, -1.1169366836547852, -0.08268018811941147, 2.1349990367889404, 0.7255300879478455, -1.093788743019104, -1.1970409154891968, 1.9547260999679565, -0.5393626093864441, 0.021289294585585594, 0.38166937232017517, -0.10578861087560654, 0.1052379459142685, -0.1682143360376358, -0.12692491710186005, -0.9924042820930481, 0.7944977879524231, -0.8624933958053589, -0.5627502202987671, -1.1031886339187622, -2.0742225646972656, 0.08854617178440094, 0.6090989708900452, 0.09767389297485352, 0.4579218626022339, 1.5380280017852783, -1.3792345523834229, -1.3170785903930664, -1.648775577545166, 0.16594284772872925, 0.6932474970817566, -0.13001985847949982, 0.027950076386332512, -0.2756352424621582, -1.2110216617584229, -0.2235764116048813, -0.28349223732948303, -0.15254484117031097, -0.1122722327709198, 0.5753322243690491, 0.26452553272247314, 0.73435378074646, -0.7020225524902344, 0.10894585400819778, 0.8481559753417969, -0.8816524744033813, 2.0353386402130127, 1.0193114280700684, -0.797400712966919, -0.22127065062522888, -1.4965450763702393, -0.13162371516227722, 0.4632498621940613, -2.5154213905334473, 0.8313331007957458, -0.8203108310699463, -0.10892051458358765, 1.8530176877975464, 0.8690377473831177, 1.4158962965011597, -0.6486450433731079, 0.21255657076835632, -0.41844168305397034, -0.3973761200904846, -0.3341776132583618, -0.19425837695598602, -0.37588736414909363, -0.3862217366695404, 1.335527777671814, 0.3883293569087982, 0.8147781491279602, -1.0283554792404175, 0.025150611996650696, 0.5277451276779175, -2.9307119846343994, -0.2654651999473572, 1.4012508392333984, -0.6821659207344055, -0.10779624432325363, 0.2279268503189087, 0.5974599719047546, -0.09131626784801483, -1.4775269031524658, 1.399975061416626, -0.7719155550003052, 1.8639670610427856, 0.16567668318748474, 0.176502987742424, 0.24284055829048157, 0.5701370239257812, -0.3215990662574768, 0.6594790816307068, -2.205955982208252, -0.1521691232919693, -0.18945689499378204, 0.35930678248405457, 0.7283253073692322, 1.0010476112365723, 1.4503906965255737, 0.10896782577037811, -0.8209877014160156, -0.3672579228878021, -0.2601620852947235, 0.6278820633888245, 1.496695637702942, -0.6357642412185669, -0.18139447271823883, 0.19619899988174438, 1.4140498638153076, -0.4447883367538452, 2.413785219192505, -0.025319259613752365, 0.15270622074604034, 0.8190096616744995, 0.34457334876060486, -0.8928245306015015, 2.0049614906311035, 0.4177279472351074, -0.6947375535964966, -0.4413762390613556, -0.41484639048576355, -1.0600255727767944, 2.3443658351898193, -0.46188074350357056, 2.2811312675476074, 1.1982508897781372, 0.055733390152454376, -0.3957067131996155, 2.4664437770843506, 2.1684279441833496, -1.052232265472412, 0.5402716398239136, 0.6573243141174316, 0.2838377356529236, -0.7017356157302856, -0.07402580231428146, -0.7355154752731323, -0.4927487075328827, 0.764193058013916, 0.5039140582084656, -1.0601061582565308, 0.5571368932723999, -2.145357370376587, -0.35149967670440674, -0.469563752412796, -0.6810320615768433, -1.642557978630066, 0.7173455953598022, -1.2741482257843018, 0.4115888476371765, -0.8885087370872498, 0.6412850618362427, -0.4051266312599182, 0.6654484868049622, 0.2344447821378708, 1.3598319292068481, 1.7415196895599365, -0.6702995300292969, 0.4338390827178955, 0.2921830713748932, 0.9752880334854126, 0.6830292344093323, -0.7660090327262878, 0.4089406430721283, 0.4816911220550537, 1.4801045656204224, 1.7283594608306885, 1.676482081413269, -1.4023452997207642, 0.048434630036354065, 0.7322461009025574, 1.5020103454589844, -0.5042634606361389, 0.5032032132148743, 1.1633763313293457, 1.290449619293213, 2.0937020778656006, 0.5943396687507629, -0.26483801007270813, 0.41818517446517944, 0.12643097341060638, -0.2998993992805481, -0.7537457346916199, 0.5018181800842285, -0.0023174607194960117, 0.22519192099571228, 1.239816665649414, 0.6072580814361572, -0.2655714452266693, 0.5464531183242798, -0.6237821578979492, -2.723207950592041, -0.06433261930942535, -1.0177967548370361, -0.30053600668907166, -0.12340627610683441, 1.3875621557235718, -2.582869529724121, 1.4624170064926147, -0.07543204724788666, -0.18379412591457367, 0.10374071449041367, -0.3465515673160553, 0.9929558634757996, -0.4328242540359497, 0.785426676273346, -2.43699049949646, 1.0542993545532227, -0.3680911958217621, -1.6047776937484741, 1.364663004875183, -0.4676862061023712, 1.2285608053207397, -0.36669981479644775, 1.2687549591064453, -1.013727068901062, 2.277125835418701, 0.5174626111984253, 1.0320688486099243, -0.03922674432396889, -1.1030653715133667, -0.2825236916542053, 0.17557968199253082, -0.29824283719062805, -0.2278062254190445, -0.8008370995521545, -1.3660385608673096, 0.6240223050117493, -1.139876127243042, -0.3389948308467865, -0.010807720944285393, -1.3573623895645142, 0.8362635970115662, -0.752566933631897, 0.7401790618896484, -0.37557053565979004, 0.9507321119308472, 1.0427149534225464, -0.39492127299308777, 1.36275053024292, -0.41770389676094055, 0.32621634006500244, -0.6918814778327942, -1.131468653678894, 0.7988460659980774, -0.29032307863235474, 0.41090404987335205, 0.44710326194763184, -1.361749529838562, 0.0758097842335701, 1.1214046478271484, -0.24067474901676178, -1.0128012895584106, -1.4200313091278076, -1.305649995803833, -1.4001836776733398, 0.7086504101753235, -0.556683361530304, -3.0501978397369385, 0.5963959693908691, -0.412321537733078, -1.9825642108917236, 0.5513598918914795, 0.41994354128837585, 0.9677339792251587, -1.1144042015075684, 0.026427753269672394, -0.6987825036048889, 0.14360903203487396, -0.3088356554508209, 1.6532363891601562, -0.7898001670837402, 1.3027641773223877, -1.1129761934280396, 1.0832464694976807, -1.397051215171814, -0.5623762607574463, -1.4820294380187988, 0.2754278779029846, 0.11885193735361099, -0.18056558072566986, -0.4196247458457947, -0.3359660506248474, 0.6214364767074585, 1.061286211013794, 0.4362618625164032, 1.7633610963821411, 0.3142937123775482, -0.28208625316619873, 0.21190014481544495, 0.8329393863677979, -0.30253997445106506, -2.4027445316314697, 0.12335176765918732, 1.2988923788070679, 0.7261784076690674, 0.27636486291885376, -1.0581185817718506, 0.6030104160308838, 0.7770214080810547, -0.9921923875808716, 0.06449910998344421, 0.4842126965522766, 1.4072449207305908, -0.8894100785255432, 1.576688289642334, 1.118966817855835, -0.8654078841209412, 0.21068167686462402, 1.2674540281295776, 0.0794290229678154, 1.2202305793762207, -2.0122666358947754, 1.5293501615524292, -1.0863852500915527, -0.5440517663955688, 0.03100457228720188, 0.07619722187519073, 0.2635478377342224, -0.7096325159072876, -1.3792558908462524, -0.6350041031837463, -0.5501572489738464, -0.5891180634498596, 1.7111330032348633, 1.2158933877944946, -0.2071758657693863, -0.6653124690055847, 1.1602104902267456, 0.2667609751224518, 0.8798596858978271, 0.3942421078681946, 1.4688432216644287, -0.44436895847320557, 2.942301034927368, -0.5961104035377502, 0.40593838691711426, -0.12211005389690399, 1.6076105833053589, 0.9584293961524963, 0.6969491839408875, -0.0771961659193039, 1.1919807195663452, -0.011637005023658276, -1.2506866455078125, 1.0903098583221436, -0.03839575871825218, 0.7221133708953857, 0.4805295169353485, -1.199267864227295, 0.2201397716999054, -0.6877667903900146, 0.5129207968711853, -1.3204309940338135, 0.8731904029846191, 0.002045529428869486, -1.1570732593536377, 0.2816372811794281, 0.7438685297966003, -0.7366208434104919, -0.9766008853912354, -0.8244941830635071, -0.6874064207077026, 1.7624307870864868, 0.7250121831893921, -0.35462677478790283, -0.550381600856781, 0.17121873795986176, 0.06972957402467728, 0.08706188201904297, -0.04429563507437706, 0.7487043738365173, 0.9813224077224731, 2.277777671813965, -1.3354755640029907, 1.3850743770599365, -0.8071373701095581, 0.891260027885437, 0.8023907542228699, 0.5769832134246826, -0.22465907037258148, -1.2478655576705933, -0.04035814106464386, 1.7261711359024048, -0.8969074487686157, 0.674248993396759, 2.1397206783294678, 0.8128506541252136, -0.761829674243927, -0.7148268222808838, -1.0276678800582886, -0.8266112208366394, 0.7999526262283325, -1.135994791984558, 1.2940293550491333, 0.7775113582611084, 1.2685558795928955, 0.5820168852806091, -0.14977535605430603, -0.5583489537239075, -0.8874158263206482, -1.3014001846313477, 0.276529997587204, 0.10070362687110901, 0.10600437968969345, -0.36763811111450195, 1.4984533786773682, 0.10610933601856232, -1.613076090812683, -0.3810789883136749, 0.32749348878860474, -1.1883995532989502, 1.9042198657989502, 0.817491352558136, -0.20395705103874207, 1.5637155771255493, -1.0471173524856567, -0.2746294140815735, -0.20729024708271027, -0.4661983847618103, 0.9308415055274963, 0.42489728331565857, 0.37133732438087463, -0.46810582280158997, 0.21834778785705566, -0.42129838466644287, 0.594463050365448, 0.3517318367958069, 0.13696642220020294, -0.0003063144104089588, -1.0540673732757568, 0.8187845945358276], [0.21674473583698273, 0.44588103890419006, -0.5359947681427002, -1.3991931676864624, -1.0859187841415405, -0.09304501861333847, 0.7090255618095398, 0.9414485096931458, 1.0609642267227173, -0.41995999217033386, 1.9774584770202637, -1.2687487602233887, -0.8675147294998169, 0.1363316774368286, 1.0244983434677124, 0.674117386341095, 1.1720741987228394, -0.031045738607645035, -0.26389777660369873, -0.3832664489746094, -0.1613563597202301, -0.7870774865150452, 0.713887095451355, -0.6152560114860535, -0.5528698563575745, 0.035738505423069, 0.16055403649806976, 0.7970654368400574, 0.951585590839386, 0.05371743068099022, 0.2578814923763275, 0.23759563267230988, -0.24044270813465118, 0.9771838188171387, -0.4688195586204529, 0.7705168724060059, 1.5874176025390625, 1.6405402421951294, -0.832148551940918, 1.1639848947525024, -0.4629746079444885, 0.9154617190361023, 0.2587627172470093, 0.15243859589099884, 1.029030203819275, 0.712799608707428, 0.16105996072292328, 0.9030385613441467, -1.8758490085601807, -2.324723482131958, -1.1012933254241943, -0.43179062008857727, 0.4970993995666504, -2.658444404602051, 0.12437517195940018, -0.4667636454105377, 0.5426616668701172, -0.8675835728645325, -0.11624368280172348, -0.3931841254234314, 0.5852475762367249, -0.09409067779779434, -1.2832283973693848, -1.575292944908142, -1.7085663080215454, -0.36053648591041565, -1.3140671253204346, -0.9186357259750366, 0.0527857206761837, -1.1887445449829102, -0.9514361619949341, -0.4505818486213684, -2.2867441177368164, -0.7481082677841187, 1.3634968996047974, 0.8372698426246643, -0.23549067974090576, -1.3702762126922607, -1.5064665079116821, 0.19869160652160645, -0.5895925760269165, -0.4551265835762024, -0.35505253076553345, -0.24998553097248077, 0.5628777742385864, -0.44054508209228516, -0.05108371376991272, 0.28490179777145386, 0.4680021405220032, 0.8340269327163696, -0.7869041562080383, 0.07067947089672089, -0.37484660744667053, -0.06480517983436584, 2.3614983558654785, 0.30078887939453125, -0.3672989308834076, 0.3462587893009186, 1.690967082977295, 1.621971607208252, 0.30840542912483215, 1.3614975214004517, -2.5341427326202393, 0.44136717915534973, 0.41341015696525574, -0.12517985701560974, 0.3975173234939575, -1.7953226566314697, -0.16872163116931915, 0.2423713207244873, 1.2410175800323486, -0.3372887969017029, 0.22596131265163422, -0.8100064396858215, -1.0409795045852661, -0.4951138198375702, -0.07426045089960098, 0.2934553623199463, -1.559067964553833, 1.0514507293701172, -0.6559199094772339, -0.5310345888137817, 1.1721785068511963, -0.026315081864595413, -1.2797422409057617, 0.35844886302948, 0.6731523871421814, -1.5774966478347778, 2.2593936920166016, 0.436635822057724, 2.4534618854522705, -0.573685884475708, -1.603164792060852, -1.2344653606414795, -0.8473445773124695, -0.7890160083770752, 0.6110519170761108, -0.09433143585920334, 0.24857567250728607, -0.5999424457550049, 0.7244619131088257, -1.1133644580841064, 0.7311235666275024, -0.2822718918323517, -0.647922694683075, 0.1960202306509018, 1.2315897941589355, 1.1560686826705933, -0.41096383333206177, -0.11817539483308792, 0.6889739036560059, 0.5693540573120117, -0.8345791697502136, 0.6981144547462463, -0.9391143918037415, 1.427510142326355, -0.2515568435192108, 0.7976008653640747, 1.7986969947814941, -0.12246710807085037, -1.1084202527999878, 0.6968382596969604, -0.5425453186035156, -1.1420953273773193, 0.5391749739646912, -1.1893314123153687, 1.891072392463684, -0.06617814302444458, 0.9380131363868713, 0.1053786426782608, -0.04611073434352875, -0.5689522624015808, -0.6904933452606201, -1.162758708000183, 0.3535018563270569, 1.056904911994934, -0.3575853407382965, 0.9710895419120789, 1.4591974020004272, 0.02588917873799801, -0.06613175570964813, 1.8653277158737183, -0.6897264719009399, 2.121415138244629, 1.019378900527954, 0.5095754265785217, -0.7130721807479858, -1.5147912502288818, 1.8192546367645264, 0.6811764240264893, 0.31292930245399475, -0.6405429244041443, 0.20293891429901123, -1.0688869953155518, -0.30890268087387085, 0.8514830470085144, 0.7832978367805481, -0.12039903551340103, 0.8023868203163147, 0.20405440032482147, 1.4525140523910522, 1.3355034589767456, -1.9261622428894043, -0.058346256613731384, 0.8037241697311401, 0.39548730850219727, 0.4196804463863373, -0.6345471739768982, -0.49459776282310486, 1.3062337636947632, 0.8917595148086548, 0.27401870489120483, -1.2206194400787354, -1.2385612726211548, 1.285940170288086, 0.8726008534431458, 0.4125165641307831, -1.8944753408432007, -0.1613270491361618, 0.07038170099258423, 0.18279272317886353, -1.375691294670105, -0.452545166015625, -0.6237974166870117, 0.25824230909347534, -0.2764471769332886, 0.6315315365791321, -0.5917866826057434, -0.13389502465724945, -0.9619095325469971, 0.028516491875052452, 0.23613379895687103, -0.5024468898773193, 1.4404959678649902, -1.7880980968475342, 0.9982229471206665, 0.179303377866745, -0.32341402769088745, -2.135016679763794, -0.27147144079208374, 0.7171877026557922, -0.48397842049598694, 0.40558815002441406, 1.7185006141662598, -1.7898612022399902, 0.4689961373806, 1.8272889852523804, 0.9785413146018982, 0.8898391127586365, 0.8951249122619629, 0.5206609964370728, -0.21838675439357758, 0.6503622531890869, 1.4262796640396118, -1.2963199615478516, -1.7210842370986938, 0.27137991786003113, -0.6448392271995544, -0.06761299073696136, -1.0564548969268799, 0.5967866778373718, -1.2227407693862915, -0.14033129811286926, 0.007225886918604374, 0.046719904989004135, -0.7791769504547119, -1.3338416814804077, 0.08983425796031952, -1.1301357746124268, 0.4994823932647705, 0.8515066504478455, -0.0039464011788368225, 0.38629084825515747, 0.021144840866327286, -0.2697705626487732, 0.24545764923095703, 1.3200173377990723, -1.0655879974365234, -0.5791211724281311, -0.14471662044525146, -0.2361249476671219, -0.2692680060863495, -0.7451266646385193, 0.5404936671257019, -1.4112238883972168, 0.4990101456642151, 1.3386050462722778, -0.6436879634857178, -1.1075745820999146, -0.3795265555381775, -0.17371609807014465, -0.5758817791938782, -0.0667010098695755, 0.15232478082180023, 0.7945835590362549, 0.923809289932251, 0.01902749389410019, 0.3889086842536926, -0.8940099477767944, 1.7746555805206299, -0.36183279752731323, 0.6305533051490784, 0.023877514526247978, -2.6071856021881104, -1.1807574033737183, -0.7827678322792053, 0.44576746225357056, -0.4470914602279663, -0.9768090844154358, 1.359009861946106, -0.09599623829126358, 0.43691274523735046, 0.10992827266454697, 0.7201054692268372, -1.0915790796279907, -0.21040867269039154, -0.6217637062072754, 0.2945583164691925, 1.9016368389129639, 1.6237865686416626, -0.17573785781860352, -0.23620489239692688, -0.24421390891075134, 0.7139213681221008, -0.6274808645248413, 0.5278035998344421, -0.1655224859714508, 1.5661072731018066, 1.2067577838897705, -1.3966325521469116, 0.7282875180244446, 0.5671179294586182, 0.31415078043937683, 0.9780799150466919, -0.7652907967567444, 0.554470956325531, -1.1309577226638794, -0.11108075082302094, -0.9492518901824951, -0.40308210253715515, 0.47505441308021545, -0.9921524524688721, 0.8401958346366882, -0.855914294719696, -0.9784832000732422, -0.5192679762840271, -0.1932956874370575, -0.534100353717804, -0.9915271401405334, -1.0136412382125854, -0.18010148406028748, -0.05901925638318062, 0.07675369083881378, -0.027664432302117348, 0.1030966117978096, 0.7750024199485779, 0.956811249256134, -0.2873450517654419, -0.439655601978302, 0.9699118137359619, -0.8161934018135071, -0.8975762724876404, 0.8732171654701233, -0.5644896030426025, 0.15119564533233643, -0.03769448772072792, -0.3602266311645508, -0.7192646265029907, -0.7913787364959717, -0.18727761507034302, -0.27484330534935, -0.03138340637087822, -0.18095673620700836, -0.2743258476257324, 2.7454583644866943, -0.7305244207382202, -2.488905429840088, -0.668147623538971, -1.8821923732757568, -0.21762911975383759, -1.6999503374099731, -1.140901803970337, -0.6468985676765442, 0.021450506523251534, -0.06667360663414001, 2.2313783168792725, -0.943473756313324, -0.8246272802352905, -1.4963006973266602, 0.1100117638707161, -2.7355430126190186, -0.9425756335258484, -1.6218879222869873, 0.7258954644203186, 0.5742448568344116, 1.8638935089111328, -0.08821386098861694, -1.186812162399292, 0.9325907230377197, -0.09098268300294876, 0.8574669361114502, -2.2225563526153564, 1.0005168914794922, 0.37366777658462524, 0.82894366979599, -0.9819158315658569, -1.2920255661010742, 0.3695087134838104, -0.5236127972602844, 0.49859803915023804, 0.28540220856666565, 1.903724193572998, 0.7437262535095215, -2.215681314468384, -1.3154265880584717, 0.14934951066970825, -0.04839504137635231, -0.041562046855688095, -0.14894691109657288, 1.1204488277435303, -1.3707983493804932, -1.1268128156661987, 0.2186083197593689, 0.684467077255249, 0.27933356165885925, 0.4760771691799164, -0.3457169830799103, 0.605843186378479, -0.028713971376419067, 0.17664633691310883, 0.32547780871391296, 0.9260392189025879, -0.4169602692127228, 1.0810831785202026, 0.0168145839124918, -0.13081207871437073, -2.060511350631714, -0.3460751473903656, 1.1209312677383423, -1.0155935287475586, -2.393620252609253, -0.4464246928691864, -0.40883779525756836, -0.5568914413452148, 0.7006173133850098, -0.010011715814471245, -1.9932034015655518, 0.17333072423934937, 0.6466047763824463, 1.2315337657928467, 0.22167393565177917, -1.3483755588531494, 0.3662477433681488, 0.5030592083930969, -0.026206303387880325, 1.8590011596679688, 1.1470110416412354, 0.05857099965214729, -1.0973849296569824, 0.25803884863853455, -1.1095380783081055, 0.09628021717071533, -0.41595810651779175, -1.061268925666809, -0.3688976764678955, -0.47078022360801697, 1.2621456384658813, -0.058683644980192184, -0.12069404125213623, 0.2930699288845062, -1.394741415977478, -0.37476611137390137, -0.5934003591537476, -0.08467463403940201, 0.004554694518446922, -0.17805053293704987, 0.27292507886886597, -0.6454682946205139, -0.6235780715942383, -0.2931308448314667, 0.4922783374786377, -0.9316254258155823, 0.1320728361606598, 1.8699769973754883, -1.8705075979232788, -0.5809714198112488, -0.6195831298828125, 1.245928406715393, 0.7134419679641724, 2.367202043533325, -0.6101685166358948, -2.279799222946167, 0.032976966351270676, -0.17708361148834229, -1.1155824661254883, 1.6447933912277222, -2.055192470550537, 0.3701028525829315, 1.2053427696228027, -0.22054925560951233, 0.6393871307373047, 0.3414994180202484, -1.233187198638916, -1.4122058153152466, 0.45297300815582275, -1.050484299659729, -1.367121934890747, -0.0050268154591321945, 1.0759800672531128, -0.10590249300003052, 0.3653084635734558, 1.7133502960205078, -1.279336929321289, 2.096839666366577, 0.8569834232330322, 0.06921207159757614, 2.260181427001953, 1.6764602661132812, 0.456038236618042, 0.7978550791740417, -0.15936046838760376, 0.011553243733942509, -0.7478751540184021, 0.48212918639183044, -1.2153990268707275, 0.20773985981941223, 0.5038975477218628, -0.3261930048465729, -0.9037883877754211, 2.1397600173950195, 0.1855580359697342, 1.8058279752731323, 0.19249510765075684, -0.12318700551986694, -1.384212851524353, -1.4184942245483398, 1.4873661994934082, -1.3643532991409302, 1.8112223148345947, 1.7757097482681274, 0.5618061423301697, -0.08026853203773499, -0.9639173150062561, 1.3421847820281982, 2.086012363433838, -0.36980023980140686, 0.4494619071483612, 0.7751199007034302, -0.13711069524288177, -0.7429307699203491, 0.8635112643241882, 2.311357021331787, 0.569017767906189, -1.3572696447372437, 0.7217739224433899, 0.6908419132232666, 0.30625149607658386, -0.24702101945877075, 0.9853938817977905, 0.7756673097610474, 0.2936418950557709, -1.25699782371521, 0.22329357266426086, -1.3367024660110474, -0.6295734643936157, -1.3040533065795898, 1.6980668306350708, -0.5042580962181091, -0.8620185852050781, -0.9339473247528076, -0.27109062671661377, -0.76853346824646, -0.32772958278656006, 0.1981016844511032, 0.21557897329330444, -0.007084231823682785, 0.8199017643928528, -2.354823112487793, -0.890733003616333, 0.9423920512199402, -1.0518542528152466, -1.2172572612762451, 0.0350111685693264, -0.21407611668109894, -1.1324340105056763, -0.4786834120750427, 0.2477256953716278, -0.938453733921051, -0.13464809954166412, -0.4759694039821625, -0.8866782784461975, -1.4295680522918701, 1.287906527519226, 0.49398690462112427, 0.6339229941368103, 0.18119484186172485, -0.7344135046005249, 0.7474495768547058, 1.4401339292526245, -1.1550859212875366, 1.1747853755950928, 0.9427322745323181, -1.312139868736267, -1.185139775276184, -0.7984764575958252, 1.0360599756240845, -0.80414879322052, 0.9189364314079285, -0.4110761284828186, 0.35270532965660095, 0.23031258583068848, -0.5613661408424377, 1.2909594774246216, -1.1796176433563232, 0.78028404712677, -0.14832979440689087, 0.5389719605445862, -0.5944476127624512, -0.17230172455310822, -0.7344225645065308, -1.2668039798736572, 0.8838791251182556, -1.0298007726669312, -0.21261119842529297, -0.8201115131378174, -0.22949473559856415, 1.0719436407089233, -0.5590851306915283, 0.2266094982624054, 0.5411601662635803, -1.3664318323135376, -1.0333900451660156, -0.1488478183746338, 1.2597813606262207, -0.7313815355300903, 0.958901584148407, 2.163700580596924, -1.0093408823013306, 0.2258782982826233, -0.2602209150791168, 0.12519142031669617, 2.0349271297454834, 0.22013741731643677, -1.3449026346206665, 2.581723928451538, 2.737528085708618, 3.1044161319732666, 0.6076576709747314, -1.4879084825515747, -0.6896689534187317, 1.8143693208694458, 1.1596077680587769, -0.5055103898048401, -0.6049858331680298, 1.542466640472412, -1.5365369319915771, -0.5152620673179626, 1.1887181997299194, 0.7107207179069519, 1.1490477323532104, -0.5528267621994019, 1.0296298265457153, 1.1073322296142578, 1.4737002849578857, -0.5962504148483276, -0.7471255660057068, 1.4132611751556396, 0.6198568344116211, -0.8143967986106873, -0.2820393741130829, -1.3206640481948853, 1.0769627094268799, -2.373427391052246, 0.4888698160648346, 0.20112018287181854, -0.09713783860206604, 0.2309146672487259, -1.2857600450515747, -0.7000715732574463, 0.08132010698318481, 0.9941375851631165, -1.4660618305206299, 1.0172535181045532, 0.8950583934783936, 2.065826892852783, 1.1600873470306396, 1.6689871549606323, -0.9060389399528503, -1.2476699352264404, 0.9457097053527832, 0.35157671570777893, 1.3629299402236938, 0.20138299465179443, 1.0409295558929443, -0.6392085552215576, 2.195439338684082, 0.6177188158035278, 0.9639777541160583, -1.043529987335205, -2.5286879539489746, 0.06117057800292969, -1.0173170566558838, -2.5626111030578613, -1.2054708003997803, -0.3883468508720398, 0.10472140461206436, 0.23883531987667084, 1.2380781173706055, 1.978967308998108, -0.4917333722114563, -0.3726372718811035, -0.659241795539856, -1.248998999595642, 1.2823842763900757, -0.33241546154022217, -0.10292630642652512, 0.5650701522827148, -0.867611825466156, -1.9533560276031494, -0.7054517269134521, 1.2124698162078857, 0.6790078282356262, -0.3111550509929657, -1.1443932056427002, 0.08363961428403854, -0.6210595965385437, 0.13190707564353943, -1.087310552597046, -0.5923333168029785, -0.04979102686047554, -0.5952780842781067, 1.217106580734253, 0.9083977341651917, -2.0391881465911865, -0.4012661576271057, 1.8872992992401123, -2.2829742431640625, -0.2745327651500702, 0.2192869633436203, 3.1040451526641846, -1.7137062549591064, 0.37601497769355774, -0.3853505551815033, 0.7772409319877625, -0.6135649085044861, 0.039636995643377304, -0.07913115620613098, 1.2644033432006836, -1.8654133081436157, -0.220391646027565, 1.1062297821044922, -0.9078492522239685, -1.5695385932922363, 1.47218918800354, 1.2219305038452148, -0.30802053213119507, 0.9375002980232239, 1.056157112121582, 0.33394286036491394, -1.7723262310028076, -0.32357895374298096, -1.195802092552185, 1.2597814798355103, 0.38971102237701416, -0.18215253949165344, 0.8015440702438354, 0.5971020460128784, 0.477688729763031, 0.8654130697250366, -0.30209800601005554, -0.9265614748001099, 0.033233605325222015, 0.565483808517456, 0.9058681130409241, -0.8407809138298035, 0.45639538764953613, -1.3618481159210205, 0.7431653738021851, 1.103895664215088, 2.412954092025757, 2.2901530265808105, 1.7439298629760742, -1.5864859819412231, -0.6669954061508179, -0.31816011667251587, 0.30099019408226013, 0.7051520347595215, -2.1583569049835205, -0.6202975511550903, -0.3192594647407532, 0.7001573443412781, -1.6251497268676758, -1.006171464920044, -0.08907265961170197, -0.6481260061264038, -0.9342055916786194, -0.9638771414756775, 1.4079115390777588, 0.7221205234527588, -0.12236637622117996, 0.03989914804697037, 0.6978760957717896, 1.3795822858810425, 0.5962901711463928, 1.0794364213943481, 0.16749395430088043, -1.801400065422058, 0.8895158171653748, -0.17076046764850616, -1.183471441268921, 0.2677608132362366, 2.4380295276641846, -0.6662078499794006, -0.16404706239700317, -0.5867472290992737, 0.05573241040110588, 1.643865704536438, -0.2314717173576355, 0.6195712685585022, -0.9968258142471313, -1.08693528175354, 0.4088492691516876, 1.227156400680542, 1.2213321924209595, -0.6425519585609436, 1.4659706354141235, 0.3306295573711395, -1.028136134147644, -0.6149401664733887, -0.6754441261291504, -0.594910204410553, 0.12305616587400436, 1.5088096857070923, 1.6074779033660889, -0.722247838973999, -1.4889802932739258, 1.8051198720932007, -0.3908498287200928, 0.9203446507453918, -1.1594620943069458, -0.14025545120239258, 2.0507614612579346, -0.001978810178115964, -0.8000662922859192, 0.08132223784923553, 0.0245636235922575, 0.2702215909957886, -2.14918851852417, 0.008564336225390434, -0.3444264531135559, 0.9835443496704102, -0.13513123989105225, -0.10730097442865372, -0.3741433024406433, 1.509261131286621, -0.47442832589149475, -1.0006486177444458, 0.36933642625808716, 0.12090383470058441, -0.7926568984985352, -0.11548926681280136, -0.5008946657180786, -0.939780056476593, -0.13047371804714203, -1.3503155708312988, -0.4247238039970398, 0.00979729276150465, 0.7667993307113647, 0.7979283332824707, -0.060847923159599304, 0.13401317596435547, -0.5102715492248535, -1.331816554069519, 0.5781269669532776, 0.06470829248428345, -0.3497099280357361, 0.8927814960479736, 1.4460920095443726, 0.31698527932167053, -1.1979243755340576, -0.8488799333572388, 0.36476558446884155, -1.8803465366363525, -0.9827376008033752, -1.5959422588348389, 1.1043757200241089, 1.3031178712844849, 1.1325148344039917, -0.5403473973274231, -1.5706454515457153, 0.1379309743642807, 0.5719395279884338, 0.7623347043991089, -0.6408815979957581, 0.5760471224784851, 0.3873436152935028, 0.7940157651901245, 1.835886836051941, -1.8716546297073364, 0.04415624961256981, -0.40585610270500183, 0.9226315021514893, -0.8436295986175537, -0.899612307548523, 0.2853425145149231, -0.5931951999664307, 0.01026507094502449, -0.3226480484008789, 0.1966463178396225, -0.43279415369033813, -1.7074085474014282, 0.8558153510093689, -0.10055868327617645, -0.5140382647514343, -1.912995457649231, 0.196772962808609, 1.3565312623977661, 0.21283376216888428, 1.801214337348938, -1.4539679288864136, -0.18702435493469238, 1.245105504989624, 0.21142075955867767, 1.325019121170044, 0.8234809637069702, 1.5484849214553833, 2.7144460678100586, -0.06015949696302414, -0.2672553062438965, -1.280075192451477, 0.2764815092086792, -0.49875542521476746, 2.6192805767059326, 0.5625749826431274, 0.2123371660709381, -0.9323609471321106, -1.6819297075271606, 1.8092949390411377, -0.6748046875, -0.28609776496887207, -0.5189617276191711, -0.4389233887195587, -0.5201411247253418, 0.870938241481781, -0.004729640670120716, 0.2772664725780487, 0.5390587449073792, 1.1696516275405884, -0.9038895964622498, -0.14102324843406677, 0.2330196648836136, 0.3823837637901306, -1.6792017221450806, -0.7142555713653564, 1.5411059856414795, -0.292013019323349, -1.338485598564148, -0.5966762900352478, 0.6439633369445801, -0.5112010836601257, -0.5549111366271973, 0.0810508131980896, 1.0602315664291382, -1.6326218843460083, 1.5789904594421387, -0.3157917857170105, 0.7983688712120056, 0.8989936113357544, -0.34862929582595825, 1.6686984300613403, 0.8216909766197205, -0.5235039591789246, -0.5842736959457397, 0.534407913684845, 0.5562582015991211, 1.7182890176773071, 0.32169049978256226, 2.6541292667388916, -0.6147158145904541, 0.29419565200805664, -0.24291202425956726, -0.2186507135629654, 0.15920031070709229, 0.45246025919914246, -0.6996955275535583, -0.5344406366348267, 1.2819613218307495, 0.20104935765266418, -1.683916687965393, 0.13151827454566956, 0.6067817807197571, -0.24074405431747437, -1.2333065271377563, -1.0385411977767944, -0.1099199503660202, -0.6506041884422302, -1.3983293771743774, 0.5286368727684021, 1.0423439741134644, 0.007725903764367104, -0.37290486693382263, -0.27553030848503113, -0.0004055649333167821, 0.2729712724685669, 0.5893378257751465, -0.7173309922218323, -1.0653666257858276, -0.4368683695793152, -1.9984935522079468, 0.5928775072097778, -1.4830451011657715, 0.46953222155570984, -0.7099226117134094, 0.3031731843948364, 0.1009194552898407, -0.17420653998851776, -0.4805012345314026, 0.3152812719345093, 0.8734904527664185, 1.2244510650634766, -1.1952564716339111, 1.507107138633728, -0.7403964996337891, -0.26655885577201843, 0.6558352708816528, 0.6495283842086792, 1.9581965208053589, -0.7169851064682007, -0.7316440343856812, 1.4483004808425903, -0.5011920928955078, -1.560530662536621, 0.17721930146217346, 0.0017496770014986396, -2.1113243103027344, 0.17901167273521423, -0.31275486946105957, 1.6156822443008423, 1.7835477590560913, -0.12799619138240814, -1.0294383764266968, -0.1977909952402115, 1.1495487689971924, 1.064711093902588, 1.922431468963623, 0.41751667857170105, -0.28568586707115173, 0.6904202699661255, -1.2169009447097778, -0.8046988844871521, 2.4911181926727295, -0.9240716695785522, -0.5366299152374268, -1.532193660736084, 0.1877918243408203, 1.7294117212295532, -0.2665390968322754, -0.8051546812057495, -0.610701322555542, 0.14418776333332062, -0.37483271956443787, 0.40106716752052307, -1.271928310394287, 1.5263311862945557, -3.473975419998169, -0.22036485373973846, 0.08432916551828384, 1.1352185010910034, 0.18331897258758545, -0.42262598872184753, 0.5484547019004822, -0.7741320729255676, -0.7849283218383789, -0.009457969106733799, 0.7721409201622009, 0.28644344210624695, -0.7515823841094971, 0.017094600945711136, 1.383243203163147, -0.26713863015174866, -2.6729941368103027, 1.2606810331344604, -0.008572014980018139, 2.203958749771118, -0.053077582269907, 0.09890647977590561, -0.00047173662460409105, 1.1317758560180664, 0.1557014435529709, -0.30048292875289917, 0.7865138053894043, 0.15072667598724365, 1.1473792791366577, -0.9554219245910645, 1.8208914995193481, 0.09460276365280151, -0.6756417155265808, -0.5282086730003357, -1.0561469793319702, -0.020390627905726433, 0.009072408080101013, -0.06032291054725647, -0.9504573345184326, -1.7666465044021606, -0.05430584400892258, 0.6108041405677795, 0.6844364404678345, -0.869718074798584, -0.6386948823928833, 1.585052728652954, 1.8747185468673706, 0.05147591978311539, -1.0894532203674316, 0.6560441255569458, 1.00473952293396, 0.23177435994148254, 0.2006552517414093, -0.4172725975513458, 0.8396169543266296, 3.2248222827911377, -0.592981219291687, 0.565879225730896, -0.8361834287643433, 0.8304339647293091, -0.556467592716217, -0.8961192965507507, 1.342182993888855, 0.1730605959892273, 1.8635448217391968, -1.8455930948257446, -1.1862642765045166, -2.0845251083374023, 0.8640410900115967, -0.7138299345970154, 0.15526829659938812, 0.4513605535030365, 0.005192107055336237, 0.22050005197525024, 0.9379822611808777, -0.8730254769325256, -0.365856796503067, 0.4576798677444458, -0.27975934743881226, 0.03146379813551903, 0.7298225164413452, -0.8869450092315674, 0.6859350204467773, -0.23483778536319733, -3.0161783695220947, -1.6849229335784912, -1.0431808233261108, -1.634527325630188, -0.3147306442260742, 0.40836700797080994, 0.5853959918022156, -1.3990249633789062, 0.2867315709590912, -0.09747602790594101, 0.5175046920776367, 0.07737161964178085, -1.0491825342178345, 0.9610245823860168, 2.1397390365600586, -0.29931148886680603, 1.1608184576034546, 1.2464051246643066, 1.0257015228271484, -0.3545016646385193, -0.32458505034446716, -0.2668558657169342, 0.20920243859291077, -1.7063705921173096, -1.5727254152297974, 1.3101269006729126, 1.8036068677902222, 1.274852991104126, 0.06653505563735962, 1.6584597826004028, -0.27076172828674316, -1.480950951576233, 0.024736624211072922, 0.19743849337100983, 0.7361709475517273, -0.38636767864227295, -1.6171190738677979, 0.9181320071220398, 0.5731039047241211, -1.266829013824463, 0.236228808760643, 0.7513402700424194, -1.1819374561309814, -0.6396927237510681, -0.5749281048774719, -2.110980749130249, 0.17442266643047333, 0.43254128098487854, 2.486095905303955, -0.1924837827682495, -1.888676404953003, 0.17030297219753265, 0.37159040570259094, -0.1808491200208664, -1.7588701248168945, 1.6769835948944092, -0.5438945293426514, -0.9719906449317932, 1.6315404176712036, 0.5727240443229675, -1.2022576332092285, -0.5514870285987854, 0.2936544418334961, -1.0993813276290894, 0.9896069765090942, 0.0010050218552350998, 0.8186262845993042, 0.44545605778694153, 1.1074366569519043, -0.6743497252464294, -0.14705969393253326, 2.403177499771118, 0.4344244599342346, -1.2149773836135864, -0.3146204948425293, 2.3254103660583496, 1.5171751976013184, 0.8046993613243103, -0.6173886656761169, -0.8698896765708923, 0.9199411869049072, -1.0109554529190063, -0.42804697155952454, -0.1399611383676529, -0.3692256212234497, -1.6899770498275757, 0.5603630542755127, 0.9609043002128601, 1.0018091201782227, 2.590869903564453, 0.5972350239753723, 0.12404561042785645, 0.011486169882118702, -0.004444449208676815, -1.2936007976531982, 0.22323808073997498, 0.3688770532608032, -1.5178413391113281, 0.3353804051876068, -0.6814647316932678, 0.8084112405776978, -0.23721764981746674, -1.4440215826034546, -0.8902754187583923, 0.13812381029129028, -0.6679469347000122, -0.24903780221939087, 0.08128325641155243, 0.16063372790813446, -0.9520595669746399, -0.16490285098552704, 0.08390554040670395, 1.4164063930511475, 0.5133054852485657, -0.7931250929832458, 1.5203293561935425, 1.279303789138794, -1.1477199792861938, 1.2252795696258545, 1.1454750299453735, 0.24954482913017273, -0.14799584448337555, 0.6770129203796387, 1.1505836248397827, -0.3990941345691681, 1.0269981622695923, 1.60734224319458, -0.20550888776779175, 1.1280900239944458, -0.31996509432792664, 0.4103788733482361, 0.6146843433380127, -0.07470197230577469, -0.6561673283576965, -0.020332783460617065, -0.16440939903259277, -0.6669594049453735, -1.7055320739746094, 0.63951575756073, 0.5941497683525085, 0.13190476596355438, -1.6672476530075073, -0.6604187488555908, 0.9832446575164795, -0.5417721271514893, 0.9297032356262207, 0.89326411485672, 0.8697856664657593, -0.9033763408660889, -0.18549419939517975, -0.06160926818847656, 1.2982572317123413, -1.3357558250427246, -0.1378861516714096, -1.3371886014938354, 1.5006145238876343, -0.07992176711559296, -0.6361882090568542, 0.18268221616744995, -0.8918976187705994, 2.6793503761291504, -1.2981189489364624, 0.2677133083343506, 0.3543618619441986, 0.4823574125766754, 1.4118280410766602, -1.503893256187439, -1.0322844982147217, 1.6222821474075317, -2.4878346920013428, -0.528753936290741, -1.8187637329101562, 1.6082175970077515, 0.4598921537399292, -1.389829397201538, -0.8108697533607483, 1.5710821151733398, -1.1758642196655273, 0.9189061522483826, -0.33164599537849426, 0.4441881477832794, -1.7310518026351929, -0.8389888405799866, 0.1610625684261322, 1.0373079776763916, 0.770647406578064, 1.1150038242340088, 1.3238109350204468, 0.22091923654079437, 0.3973405659198761, -0.8391712307929993, -0.6283044219017029, 2.1346678733825684, 1.736870288848877, 0.25903716683387756, -0.12853193283081055, -0.14208100736141205, -1.415091872215271, 0.7684490084648132, 0.5297216773033142, -0.35287120938301086, 1.3883075714111328, -1.1336935758590698, -1.0317996740341187, 0.04112780839204788, -1.0043829679489136, 1.0512306690216064, 0.773315966129303, -0.4680294096469879, -0.589693546295166, 2.8673410415649414, -1.6089879274368286, 1.3766913414001465, -0.44045329093933105, -0.34471768140792847, 1.5716356039047241, -0.9019584655761719, -1.1051372289657593, -1.7737393379211426, -0.5537033677101135, 1.236430048942566, -1.345901370048523, -1.5622178316116333, -0.6108649373054504, -0.09967727959156036, 1.179114818572998, -0.6602069735527039, -1.0153157711029053, -0.3219565153121948, 1.0988624095916748, -0.6014476418495178, -0.8918402194976807, -0.4371832609176636, -0.5739122629165649, 0.37937167286872864, -1.8328077793121338, -0.25041213631629944, -0.18607276678085327, -0.6977360248565674, 0.07455545663833618, 0.09173854440450668, -0.35360875725746155, 0.27000415325164795, -1.1289522647857666, -0.5746327638626099, 0.3312990963459015, -1.2167176008224487, -0.7732797861099243, -0.9385477900505066, -0.17159484326839447, -0.006943186745047569, -0.6300033926963806, -0.7731152176856995, -0.28200504183769226, -0.8583011627197266, -1.6818163394927979, 1.9784706830978394, 1.709185004234314, -1.1069010496139526, -0.488038033246994, -0.6451058387756348, 0.8155644536018372, 0.10428578406572342, -0.5202129483222961, -2.833646535873413, -0.9420839548110962, 0.8329464197158813, -0.4115142822265625, -0.2761385142803192, -0.4567711353302002, -0.30507004261016846, 1.86416494846344, 0.37182366847991943, -0.5974554419517517, 1.7049219608306885, -2.9674532413482666, 0.4840683341026306, -0.18672685325145721, -1.3864549398422241, -1.6125071048736572, 1.3217482566833496, -0.6676169037818909, 2.05926251411438, -0.9302712678909302, 1.4541300535202026, -0.6307321190834045, -0.5889565944671631, 1.21095609664917, 0.6623157262802124, -0.7430729269981384, 0.8267398476600647, -0.06225163862109184, -0.9706289172172546, 1.0767803192138672, 0.5897777676582336, -0.30424606800079346, 2.107952833175659, -0.8947738409042358, 1.391871452331543, -0.8380489945411682, 0.18856598436832428, -1.7826162576675415, 2.370490550994873, 0.98878014087677, -0.4877128303050995, 1.0945169925689697, 2.1557319164276123, 0.11755776405334473, -1.7262136936187744, 0.5444088578224182, -0.050774119794368744, 1.0200310945510864, 0.8083030581474304, -1.2891647815704346, -0.7307665944099426, 0.6680259108543396, -0.20599929988384247, 0.6215862035751343, -0.07456178963184357, -0.23224779963493347, -0.3989656865596771, -1.483139991760254, 0.17597566545009613, 0.06535178422927856, -1.2478240728378296, 0.3817530572414398, -0.4092526435852051, 0.7778706550598145, -1.8970032930374146, 0.2521209716796875, -1.3976048231124878, 0.029115328565239906, 0.4791097342967987, 0.40367263555526733, 0.8882002830505371, -0.9547823071479797, 1.513696551322937, -0.24145567417144775, 0.31426340341567993, -1.8210264444351196, -0.7176573872566223, 0.35561349987983704, -0.8014528155326843, -0.9532344937324524, 1.017361044883728, 1.2137998342514038, -1.3250665664672852, 1.0602251291275024, -0.058870043605566025, -1.014521598815918, 1.2161318063735962, -0.6828305721282959]], "dtype": "float32"}, {"__nd__": [[-0.4100029170513153, -1.3830629587173462, 1.2831332683563232, 0.7483634352684021, -0.9925236105918884, -0.7691053748130798, -2.07328724861145, -0.993564248085022, 0.5725814700126648, 0.412048876285553, 0.27567917108535767, -0.28500497341156006, 0.9811275005340576, -0.26926669478416443, -0.9442029595375061, -0.5035977363586426, 0.06275065988302231, 1.2343692779541016, -0.3816240131855011, 1.038254976272583, 0.29769375920295715, -1.3027197122573853, 0.8941482305526733, -1.338822841644287, 1.1828465461730957, 0.5080280900001526, 1.938968539237976, 0.42613768577575684, 0.5724906921386719, -1.2159137725830078, -0.04940206557512283, -0.030965065583586693, 1.111023187637329, -0.3600037693977356, 1.0427013635635376, -1.0373470783233643, -1.284070611000061, -1.0018316507339478, 0.3098984658718109, -0.4391205608844757, -0.27273890376091003, 1.0856647491455078, -0.7423362731933594, 1.49153470993042, -1.1732484102249146, -0.3809926211833954, 0.9458492398262024, 0.928054690361023, -0.20914986729621887, 0.389999121427536, -0.9528998136520386, -2.4791884422302246, 0.7299318909645081, -0.6168593764305115, 0.5355652570724487, 1.5107665061950684, 0.2985471785068512, -0.11133791506290436, 2.004427909851074, 1.6319124698638916, -1.7598456144332886, -1.2883763313293457, -0.8666828870773315, 0.562014102935791, 1.1678524017333984, 0.6400011777877808, -1.9165161848068237, 1.055370807647705, -1.4564452171325684, 1.9026427268981934, 0.6548498868942261, -0.566748857498169, -0.19504576921463013, -0.9240915775299072, 0.3132375478744507, -0.22993266582489014, -0.338844358921051, -0.9068628549575806, 0.06955946236848831, -1.295832633972168, 0.2980833947658539, 0.04951310157775879, 1.1692038774490356, 0.2563626766204834, -0.1647941768169403, -0.08476334810256958, -0.7807460427284241, 0.31566351652145386, 1.2008538246154785, 0.9157135486602783, 0.7667067050933838, -0.2711317241191864, 0.7362672686576843, 0.10321176052093506, -0.07902225852012634, 0.1258428990840912, 1.612147569656372, 0.08943784236907959, -0.3238582909107208, -0.1448327898979187, -0.3569501042366028, -0.9651156067848206, 2.469235420227051, -0.7515851855278015, -0.30902183055877686, 0.23956705629825592, -1.3655462265014648, -0.4686402976512909, -1.1338227987289429, -0.7847965955734253, 0.8853562474250793, 0.6531942486763, -1.1965056657791138, 1.513895034790039, -0.33063575625419617, 0.7346076965332031, 0.2291155755519867, -0.8703304529190063, 1.684054970741272, 1.3204879760742188, 1.7508563995361328, -0.8285390734672546, 0.6456645727157593, 1.115182876586914, 0.028521670028567314, 0.9266544580459595, -0.46632933616638184, 0.39959201216697693, 1.5042787790298462, -0.7231664657592773, -0.19597850739955902, 0.7079499959945679, 0.019308170303702354, -0.5023342370986938, -2.9557301998138428, 1.1513922214508057, 1.6208418607711792, -1.939396619796753, 1.9480547904968262, -1.295709490776062, 1.3988300561904907, 0.6334228515625, -0.06121303141117096, -2.79736065864563, -0.4313685894012451, 0.829769492149353, -0.7509351968765259, -1.5291699171066284, 0.42457225918769836, 1.8233842849731445, 0.34751078486442566, 1.7376458644866943, -0.4108835756778717, 1.7690811157226562, 0.6033875942230225, -1.2970974445343018, 1.0825973749160767, 0.6937891244888306, 0.12117324769496918, 1.2718299627304077, 0.6098632216453552, -1.549782395362854, 1.0515881776809692, 0.12149707973003387, 2.0941390991210938, 1.1230298280715942, -0.9652172327041626, 0.8606904149055481, 0.44991591572761536, -0.4604615271091461, 0.9296147227287292, -0.4374235272407532, 1.104610800743103, 0.5719495415687561, 0.169725701212883, 0.0222526453435421, 0.7158534526824951, 1.204392433166504, 1.3245255947113037, -0.05500544235110283, -0.8669452667236328, -1.6448193788528442, 0.5687662959098816, -1.3042030334472656, -0.06083379685878754, 2.194898843765259, 0.5780139565467834, -0.9992921948432922, 1.9613192081451416, -0.6775735020637512, 0.06046057865023613, -0.06107542663812637, -0.7449896931648254, -0.28011617064476013, -0.6060788035392761, 1.5310730934143066, 1.1438658237457275, -1.6498050689697266, -0.009832457639276981, 0.5459234118461609, -0.8401243686676025, 1.8088687658309937, -0.43858277797698975, -0.06845418363809586, 0.18279774487018585, -0.02051280252635479, -0.7353262305259705, -1.2726560831069946, -0.9281354546546936, 0.22465860843658447, -1.468729853630066, 0.027031512930989265, 2.050396680831909, -0.031809668987989426, -0.07219942659139633, -0.2226509302854538, -0.17934897541999817, -0.22615346312522888, 0.010234723798930645, -1.071597695350647, -2.668330192565918, 2.684257984161377, -1.4704490900039673, 1.2478657960891724, 0.1909889578819275, -1.2782939672470093, -0.42806386947631836, 1.3033018112182617, -0.3335818946361542, 0.3431010842323303, -0.09649154543876648, 0.12165162712335587, 0.11686330288648605, -0.9676716327667236, 0.9717875719070435, -1.1430147886276245, -0.4916647672653198, 2.131176471710205, 1.9793946743011475, 0.3435644805431366, -0.3045330345630646, 0.234327495098114, 0.5959423780441284, -0.041971370577812195, 0.7209726572036743, 0.21110844612121582, 0.9780057072639465, 0.34839585423469543, -1.0772864818572998, 1.0627267360687256, 0.5885249972343445, -1.0211687088012695, 0.22576643526554108, -0.23747745156288147, 0.10149339586496353, 1.0958677530288696, 0.37517470121383667, -0.262633740901947, -0.43989333510398865, 1.1400713920593262, -1.1751192808151245, 0.2994591295719147, -0.7003044486045837, 0.4258616268634796, -0.9008322358131409, -0.9927846193313599, 0.6910515427589417, -0.7503518462181091, 0.6262415051460266, 0.6903533935546875, -0.8020954132080078, -0.656449019908905, -0.4800393283367157, -0.08386904746294022, 0.16805294156074524, -1.2407416105270386, -1.651408076286316, 1.0114789009094238, -0.6711154580116272, -0.07489410787820816, 0.6043667793273926, -1.5661592483520508, 1.2695765495300293, -0.39523500204086304, 0.44189155101776123, 0.17387260496616364, 0.7975496649742126, -0.5031651854515076, 0.4207281172275543, 0.1768043339252472, -1.4100719690322876, 0.3581639230251312, -0.25076985359191895, 0.7497268319129944, 1.202772617340088, -0.11148285865783691, -0.1363881230354309, 1.200302004814148, 0.40944766998291016, -0.21653176844120026, -0.8709728121757507, 0.3284221589565277, 0.5570005178451538, -3.2582592964172363, -0.23184834420681, 0.09183277934789658, -1.9904731512069702, -0.6096763014793396, -0.160419300198555, 0.8289751410484314, 0.17065660655498505, 0.16726447641849518, 0.5495380163192749, 0.959562361240387, -0.48251163959503174, -1.1655774116516113, 1.3599056005477905, 0.5276347994804382, -0.7529907822608948, -0.5867131352424622, -0.714809238910675, -1.0419113636016846, 1.3956612348556519, 0.8307186961174011, -1.1505643129348755, 0.07775068283081055, -0.8319966197013855, -1.1348930597305298, -0.2184789627790451, -1.3157010078430176, -0.7910167574882507, 0.3712848126888275, 1.6396996974945068, 1.464781403541565, 0.12239538133144379, -1.181881070137024, -0.3866913914680481, -0.13382872939109802, 0.4147171378135681, 0.714046835899353, -0.5582987666130066, 1.2760272026062012, 0.7822598814964294, -0.2894199788570404, 0.28087833523750305, 0.838397741317749, 0.15479601919651031, 0.32562142610549927, -0.7172626256942749, 1.5596668720245361, -0.5172980427742004, -1.9351942539215088, -0.6015499830245972, 0.6290730834007263, -0.7233260273933411, 1.2685362100601196, 0.9941056370735168, 2.241792917251587, -1.6080571413040161, -1.5567567348480225, -0.7950112819671631, -0.3043675422668457, 0.11228734254837036, -0.9867692589759827, -0.9107747673988342, 1.0101884603500366, 0.23248516023159027, -0.6257951259613037, 0.3433820903301239, -1.4867098331451416, 1.115859031677246, 0.6338764429092407, -0.10625291615724564, 0.15346217155456543, 0.36445945501327515, -0.8825762271881104, 0.3939899802207947, -0.4466751217842102, 0.7643786668777466, 1.254268765449524, -1.330501914024353, -1.6263395547866821, 0.8744736313819885, -1.4824453592300415, 0.7732480764389038, 0.8503637909889221, -0.6515524387359619, 0.7931093573570251, 0.4040429890155792, -0.33727723360061646, 1.1853078603744507, 1.1335837841033936, -0.3061046302318573, 1.186643123626709, 0.05254625901579857, -0.17777341604232788, -1.0190132856369019, -1.0369319915771484, -0.3126583397388458, -1.133453607559204, -0.2627185583114624, -0.8236045837402344, -0.05240460857748985, -1.414454460144043, 0.4654035270214081, -1.120863676071167, 0.586931049823761, 1.0898659229278564, -1.017366886138916, 1.810617208480835, 0.25583937764167786, 0.44516459107398987, 0.5979803204536438, -0.1426258236169815, 0.2553141415119171, 0.701719343662262, 1.6894129514694214, -0.2601774334907532, 0.3922700583934784, -0.27527356147766113, -1.2714111804962158, 1.2914650440216064, -0.4738682508468628, 1.01789128780365, 1.2231191396713257, -0.4031451344490051, 1.2708592414855957, 2.312059164047241, 0.8424427509307861, -2.1115050315856934, 0.4943515360355377, 0.06437808275222778, -0.27987927198410034, 0.7104749083518982, -0.10157764703035355, 0.08601213991641998, 0.3084706664085388, -0.7419228553771973, 1.003835678100586, -0.2876616418361664, 0.49158138036727905, 0.07026343792676926, 0.580420196056366, 1.7282638549804688, 0.2619380056858063, 0.48719075322151184, 0.4433521032333374, 1.1365776062011719, -0.12704679369926453, -2.016962766647339, -1.0831636190414429, -1.2947512865066528, 0.178288996219635, 1.96746826171875, -0.35660475492477417, -0.6216597557067871, -0.588959276676178, 0.6766323447227478, -0.6491612792015076, 0.33199578523635864, -0.36510929465293884, 1.0919184684753418, -0.7370684146881104, 1.1457470655441284, -0.10105390846729279, 0.8159282803535461, 0.21592067182064056, 0.694020688533783, 0.8762723803520203, -0.24131712317466736, -0.575613796710968, -0.7365649938583374, 0.1938461810350418, -1.189272165298462, -2.9806597232818604, 0.48859143257141113, 0.7272087335586548, -0.35504812002182007, -1.698552131652832, -0.4590296149253845, 0.18557707965373993, 0.3224879205226898, 2.1007463932037354, -0.41090577840805054, 0.6741743683815002, -0.5725526809692383, -0.10418087244033813, 0.48908793926239014, -0.45691239833831787, 0.011670171283185482, 0.5471576452255249, 0.2134341150522232, 1.7338649034500122, -1.0757025480270386, -0.9455924034118652, -1.110460877418518, -0.6568410992622375, -0.35138529539108276, -0.1692172735929489, -0.7836906313896179, -0.2790163457393646, -0.697983980178833, 1.7598460912704468, -0.8603751063346863, -2.2073111534118652, 0.09833454340696335, 0.5921791195869446, 0.524056077003479, 1.1573415994644165, 1.5393701791763306, -0.03531445562839508, -2.236478328704834, 0.24117033183574677, -0.19556933641433716, -1.204477071762085, 0.8152246475219727, 1.1416832208633423, -1.3324575424194336, -1.4539004564285278, 0.5360881686210632, 0.1339690387248993, 1.6015198230743408, 0.1495445966720581, 1.0429835319519043, -0.3224327862262726, -0.6449253559112549, 1.6929435729980469, 0.6716281175613403, -0.7858695387840271, 0.5099518299102783, -0.6718641519546509, 1.1959123611450195, -0.5711579918861389, -2.0482306480407715, 1.154222011566162, -0.6725040674209595, 0.3006480932235718, -0.1422799974679947, 0.008020730689167976, -1.0009870529174805, -1.0749845504760742, 0.5501304864883423, 0.2230716347694397, 0.11348559707403183, 0.4868219792842865, -0.4063529372215271, -1.086484670639038, 0.5738742351531982, -0.20159436762332916, -0.6061453819274902, 1.2679508924484253, -0.08477133512496948, 0.4297283887863159, 1.936639428138733, -0.21015511453151703, -1.3300422430038452, 0.6215608716011047, 0.04963988438248634, 0.19471220672130585, 0.6647853255271912, 0.1625571995973587, -0.21753956377506256, -1.3826069831848145, -0.43149757385253906, -1.0922722816467285, 0.02216937579214573, 0.1853834092617035, -1.2496966123580933, 0.10333680361509323, -0.6194174289703369, 0.6795147061347961, -0.2996268570423126, 0.9224360585212708, 1.0302239656448364, 0.3835931420326233, -1.3698127269744873, -0.0364808551967144, 0.31040188670158386, 1.6639107465744019, -0.23847170174121857, -0.256123811006546, -0.9698873162269592, -0.06034175679087639, -0.6191806793212891, -1.8774199485778809, -0.7277392745018005, -0.006113007664680481, 2.311708450317383, -0.07922280579805374, 0.20108377933502197, -0.7557519674301147, -1.5020357370376587, 0.6857817769050598, 1.2715009450912476, -0.9507946372032166, -0.3693704605102539, -0.656287670135498, -0.6400775909423828, -1.3793278932571411, -0.8178300857543945, -0.7601073384284973, 0.35919418931007385, -0.37528151273727417, -0.8713737726211548, -2.2579336166381836, -0.13106481730937958, 0.15034504234790802, 1.0160188674926758, -2.3189456462860107, -0.7134091258049011, 0.04843511804938316, 0.34339502453804016, -0.21630461513996124, 0.26114681363105774, 1.9186151027679443, 2.0285122394561768, 0.9242550134658813, -0.17314974963665009, 1.4717689752578735, -1.052854061126709, -2.0424959659576416, 0.3991284668445587, -0.7513561248779297, 0.27037060260772705, 0.6860030293464661, 1.0810270309448242, -1.301422119140625, -0.9400957226753235, -0.3825618326663971, 0.06377028673887253, -1.2867978811264038, -0.479230135679245, -0.4851125478744507, 2.1064674854278564, 0.18047522008419037, -0.08340384811162949, -0.11469908803701401, 0.5749295949935913, 0.6913807392120361, -0.8921648859977722, 1.025421142578125, 0.38470104336738586, 0.16376754641532898, 0.20728163421154022, -0.48600050806999207, -1.2073512077331543, 0.5694933533668518, 0.5722495913505554, -1.780031442642212, 0.8920067548751831, -1.152189016342163, 0.28458917140960693, -1.7796515226364136, -1.2825438976287842, -0.4564923346042633, 0.4024522304534912, -0.5128818154335022, 1.8888205289840698, -0.8636251091957092, 0.03522942215204239, -0.5378754734992981, 1.076310396194458, -1.187410593032837, 0.022501615807414055, 0.751108705997467, -1.105058193206787, 0.18455107510089874, 2.219965696334839, 0.18663424253463745, 0.744318962097168, -2.0546462535858154, 1.6783870458602905, 0.8001434803009033, -2.2195894718170166, -0.08152151107788086, 0.9114164710044861, 0.1909649819135666, -1.5983644723892212, 0.34091660380363464, -0.039571214467287064, 0.4878554344177246, 1.2478631734848022, -0.9131133556365967, -2.1572043895721436, -0.9402198195457458, -1.1179569959640503, -1.7357277870178223, -1.034553050994873, 0.5669935345649719, 2.6373038291931152, 0.7178611159324646, 0.8109816312789917, -0.5785036683082581, 0.03932522237300873, -0.05708754435181618, 1.5877232551574707, -1.8009048700332642, 0.7018247246742249, 0.5120380520820618, 1.062064528465271, 1.6987364292144775, -0.8274521827697754, -0.0776696428656578, -0.1595957726240158, 1.0734710693359375, 1.4575090408325195, -0.4683281183242798, 1.0509847402572632, 1.455725073814392, -0.1803785115480423, -0.035060036927461624, -0.7641839385032654, -1.6552826166152954, -0.1910136193037033, -0.045682214200496674, 2.683396339416504, 0.7610316872596741, 0.45547646284103394, 0.3085522949695587, -0.8782064318656921, -0.35279831290245056, -0.04251943528652191, 1.6411054134368896, 0.2898433208465576, 0.08015912026166916, -1.008110761642456, 0.4864872694015503, -0.21501249074935913, 0.9419054388999939, 0.9870902895927429, -0.1617235392332077, -0.7662661671638489, 1.0278517007827759, 1.0557074546813965, -0.2768729627132416, 0.7743033766746521, 0.9669885635375977, 0.8797451257705688, 1.0720956325531006, 0.8251060247421265, 0.1810595691204071, 0.47597312927246094, -1.1576100587844849, 0.6454319953918457, 0.03636080399155617, -0.84776371717453, 0.4154411852359772, 0.8068433403968811, 0.06756050884723663, 0.3555437922477722, -0.16089223325252533, -0.47905439138412476, -0.5228437185287476, -1.0167406797409058, -1.5630769729614258, -0.5821992754936218, 1.183267593383789, 0.560538113117218, 2.395770788192749, 1.781597375869751, -0.2741153836250305, -0.9379964470863342, -0.3571108877658844, 0.18521976470947266, -0.8347612023353577, -1.461213231086731, -1.142865538597107, -1.7857524156570435, 1.1801307201385498, -1.0268834829330444, 0.8926315307617188, 1.1801295280456543, 0.671890914440155, -1.9765552282333374, -0.7832902669906616, 0.25355881452560425, -0.25575950741767883, -1.4707332849502563, 0.11230561882257462, -0.6224551796913147, -0.6273710131645203, 0.9978095889091492, 1.8087667226791382, 1.9255589246749878, 1.3250752687454224, 1.550196886062622, 2.0475993156433105, -0.12363128364086151, -1.4798847436904907, -1.01313316822052, 0.3723086714744568, -0.40654581785202026, 1.0290868282318115, -1.401282787322998, -0.7878832221031189, -0.3120497763156891, -1.8090187311172485, -0.8948155641555786, -2.2418200969696045, -2.3904643058776855, 0.17468662559986115, -0.056751854717731476, -0.1485222429037094, 0.6830670833587646, 0.3529975712299347, -0.6811665296554565, -0.7401383519172668, -0.0687146857380867, -0.2857128977775574, -1.0652292966842651, 0.373219758272171, -0.34553059935569763, 0.6898743510246277, 0.6515756845474243, -1.9001744985580444, 1.5810192823410034, 0.881733775138855, -0.2583107650279999, -0.835392415523529, -1.2117736339569092, 1.9783893823623657, -1.9704630374908447, 0.07025733590126038, 0.34514492750167847, 1.4891642332077026, 0.09352762997150421, -0.7338232398033142, -1.031736969947815, 1.312890648841858, 0.6222030520439148, -0.707126796245575, 1.0887200832366943, 0.5715855956077576, 1.6883022785186768, 0.41610774397850037, 0.3055325746536255, 0.7997552156448364, -0.3134723901748657, 2.0099265575408936, -1.298738956451416, 1.0457687377929688, -1.388300895690918, -1.4579496383666992, 1.2608082294464111, 1.005967378616333, -1.5869545936584473, -0.6369664669036865, 1.0586079359054565, 0.053657591342926025, -0.014743984676897526, -2.353282928466797, -0.4990209639072418, 0.38370853662490845, -0.8537003397941589, 0.3759101629257202, 0.23092873394489288, 1.6509162187576294, -0.3417638838291168, -1.5439084768295288, -0.5270875692367554, -2.8247859477996826, -1.6259853839874268, -1.351556420326233, 0.3993922770023346, -0.2934718430042267, 0.7435856461524963, 0.550834059715271, 0.18924696743488312, -0.5682466626167297, -1.083727240562439, 0.7478841543197632, 0.15876822173595428, 0.024111857637763023, 0.49182942509651184, 1.126239538192749, 0.48963654041290283, -0.006092735566198826, 0.1516617089509964, 0.4271703362464905, -0.323928564786911, -1.2891743183135986, -1.7080656290054321, -1.2001621723175049, 0.5564308166503906, -1.0833008289337158, -0.011193458922207355, 1.252209186553955, 0.5852654576301575, -0.8454264402389526, 0.3359469175338745, -0.9138516187667847, -0.1902887225151062, 0.1794312596321106, 1.4699053764343262, 0.7865097522735596, 0.5935646295547485, -1.0067814588546753, -1.307544231414795, 1.73832368850708, -1.1387598514556885, -0.09557074308395386, -0.9441592693328857, 0.8089867830276489, 0.29032397270202637, -1.1048682928085327, 0.2704387307167053, 0.7101747989654541, -1.3051424026489258, 0.5335185527801514, -1.743503451347351, -2.1641061305999756, 0.5082527995109558, -1.8769270181655884, 0.5746274590492249, 0.7524514198303223, 1.5153934955596924, -0.6139907836914062, -0.13730774819850922, -1.0034929513931274, -1.4561821222305298, 0.8242427706718445, -0.9181323051452637, 0.15182363986968994, -0.8058249950408936, 0.23453330993652344, 0.22887685894966125, -1.453358769416809, -0.448691725730896, -1.1961246728897095, -0.04624064266681671, 0.40171751379966736, 1.685910940170288, 0.249258354306221, 0.9727206230163574, -2.1063082218170166, -0.5895294547080994, 0.6483404040336609, -1.0038573741912842, 0.08957477658987045, 0.02684367261826992, -1.6187833547592163, -1.715831995010376, 0.7645169496536255, -0.9973220825195312, 0.335155725479126, -0.9068602323532104, -1.465354084968567, 0.7359110116958618, -0.16463905572891235, -0.5638152956962585, -0.6863475441932678, 1.6500773429870605, 1.274595856666565, -0.5072701573371887, 0.7026576995849609, -0.6078531742095947, 1.3549838066101074, -1.9055659770965576, 0.1584802269935608, -1.001582145690918, 1.0747506618499756, -0.2281801849603653, 0.7549261450767517, 0.6295449137687683, 0.30736103653907776, 0.5220767855644226, 2.202871084213257, 0.07584170252084732, 0.3438011407852173, -1.266992449760437, -1.1318145990371704, 1.6100283861160278, 3.410274803172797e-05, -0.8091771602630615, -0.8232491612434387, 0.8646352291107178, 0.054042570292949677, -1.0906941890716553, -0.5597724318504333, -0.8285424709320068, -0.5819718241691589, 2.749619483947754, 0.0685785636305809, 1.3139140605926514, 0.09518203884363174, -0.41974323987960815, -0.08668389171361923, -0.6088848114013672, -0.056956902146339417, 0.22946742177009583, 1.8220326900482178, -0.17486101388931274, 2.1416985988616943, -0.6036662459373474, 0.4461025595664978, -1.572852373123169, 0.20978394150733948, -0.36945173144340515, -0.44929465651512146, 1.1778998374938965, -1.579715609550476, -0.16712160408496857, 0.4771691560745239, -0.9416770339012146, -1.0642744302749634, 0.08627574890851974, 0.6680254936218262, -0.03916379064321518, 0.15991732478141785, 1.1862152814865112, 0.2022092640399933, -1.9674556255340576, 1.0317106246948242, 0.6773139238357544, 0.7061405777931213, 0.2871132791042328, -0.33664610981941223, 0.7145975232124329, -1.3595590591430664, 1.0233343839645386, 0.9667876362800598, 0.18161876499652863, -0.27485811710357666, 1.0874273777008057, -1.3066517114639282, 0.293674498796463, -0.5017642378807068, -0.7379947304725647, -0.5659300088882446, -0.012824487872421741, 0.3625706434249878, 1.0687713623046875, 0.9009869694709778, 0.5059850811958313, 0.1316750943660736, -0.060755327343940735, 0.35513991117477417, 0.19093826413154602, -0.1561361849308014, -0.18672464787960052, 2.3519952297210693, 0.7043662667274475, 2.4070286750793457, 1.222244381904602, 0.4612930715084076, -0.8832954168319702, -2.128556728363037, 0.17601729929447174, -1.0489352941513062, 0.6045911312103271, 0.02288527972996235, -1.8596800565719604, -0.6711538434028625, -0.28855839371681213, 0.14128632843494415, 0.27786868810653687, 0.24935485422611237, 0.3486844003200531, -0.04229690879583359, 1.4581830501556396, 1.054363489151001, -1.661541223526001, -0.3294420540332794, -0.4700310230255127, -2.014211654663086, 0.5059278607368469, 0.7830531597137451, -0.9729948043823242, -1.0171362161636353, 0.31626009941101074, 0.31687504053115845, -0.18501825630664825, 1.4116134643554688, -0.13343414664268494, 0.6988284587860107, -0.012162357568740845, 1.5267492532730103, -0.30417588353157043, 0.45641812682151794, -0.2906567454338074, -0.11920759081840515, 1.4743858575820923, 0.017611507326364517, -0.1893455535173416, -1.0923645496368408, 1.0075682401657104, 1.1103801727294922, 0.6036507487297058, 0.6492258310317993, 0.22572633624076843, 0.36091575026512146, 0.014195403084158897, 0.5160101652145386, 0.7485755681991577, 1.2565295696258545, 0.7301623225212097, 0.19631950557231903, 0.9758612513542175, -0.5531896352767944, 1.2361756563186646, 1.1753267049789429, -0.4334639608860016, -0.4970651865005493, -0.3512413501739502, 0.7355329394340515, 0.36716294288635254, 0.27389395236968994, 2.3950674533843994, -0.2664661705493927, -1.41001296043396, 1.8563048839569092, -0.3612249195575714, 1.0599309206008911, -0.6165531277656555, 2.1528542041778564, 0.33288154006004333, 1.776397705078125, 1.3039088249206543, -0.9038424491882324, -1.6306260824203491, -0.5788384675979614, 1.1593561172485352, 0.3169211149215698, 0.533258318901062, 0.8572620749473572, -0.33377987146377563, -0.9157407879829407, -1.2532552480697632, 0.5752850770950317, 0.20420454442501068, 0.12890569865703583, -0.44890278577804565, 0.15066884458065033, -0.5648927688598633, 0.05812474340200424, 0.2610883414745331, 0.6391299962997437, -0.6602891683578491, -1.6571366786956787, 0.49850714206695557, -1.0009411573410034, 0.5509421229362488, 0.6494975090026855, -1.8855427503585815, -1.0164376497268677, 1.318140983581543, -0.28366196155548096, 3.202009677886963, -1.7102293968200684, -1.737245798110962, 0.054661113768815994, 0.7830498218536377, 0.07362598180770874, 1.7421411275863647, -0.08556529134511948, 0.7104083895683289, -1.2804908752441406, -0.8452332615852356, -2.308790445327759, -0.34609779715538025, -0.13001272082328796, 0.23763713240623474, -0.5996912121772766, 2.1194908618927, 0.5509862303733826, -0.7512744069099426, 0.017215091735124588, 0.9790804982185364, 0.3587454855442047, 0.1983264833688736, -0.7410995364189148, -0.49663245677948, 0.42017892003059387, 0.7411302328109741, 1.1070024967193604, 0.4832967519760132, 0.3209642469882965, 1.0102133750915527, -0.751825749874115, 0.07705549895763397, -0.7691095471382141, 0.6993424296379089, 0.9461283087730408, -0.8080131411552429, -1.41568922996521, -0.5445903539657593, -0.8792690634727478, -0.5039183497428894, 0.17893363535404205, 0.07709123194217682, 0.3719497323036194, -0.0787171870470047, -1.8535537719726562, 0.18641656637191772, 2.8135693073272705, -0.5676316022872925, -0.05057138949632645, 0.7451101541519165, -0.9348320364952087, -0.2689037322998047, 0.4996337592601776, 0.7254014015197754, -2.1333019733428955, -1.5924608707427979, -0.09493347257375717, 0.40887153148651123, -0.13439126312732697, -0.7645459771156311, 2.1059961318969727, 0.869096577167511, -0.4366244673728943, -0.30886271595954895, -0.5089044570922852, -0.19005140662193298, -0.06252143532037735, -0.36949360370635986, 0.211784228682518, 0.6475712060928345, -1.1758655309677124, -0.013467726297676563, -0.19725070893764496, 1.2912710905075073, 1.524551272392273, 0.13769321143627167, -0.9129898548126221, 1.2117834091186523, -0.060107674449682236, -0.46338167786598206, 0.5968598127365112, -1.2308292388916016, 1.6604957580566406, 0.09758219122886658, 0.4130040109157562, -0.09829308837652206, -0.9975883960723877, 0.5355921387672424, -0.6888027191162109, 0.002255570376291871, 0.6543301939964294, 1.0483065843582153, -0.2607089877128601, -0.1493370085954666, 1.2556846141815186, -0.1492275446653366, -1.4081891775131226, 1.9712727069854736, 0.5159708261489868, -0.1722320318222046, 1.5335932970046997, 0.7003313302993774, 0.6133540272712708, -1.0604948997497559, 0.7755973935127258, -0.29891103506088257, 0.7880574464797974, 0.009503711014986038, 1.789926290512085, -1.0254615545272827, 0.6404075026512146, 0.8513939380645752, 1.9874807596206665, 0.48450860381126404, -0.5797175765037537, -0.604695737361908, -1.5734868049621582, 0.6524225473403931, 0.07806017249822617, 0.35348090529441833, 1.0828663110733032, -0.42315050959587097, 2.122920513153076, -0.6590149998664856, 1.6207330226898193, 0.6429103016853333, -0.4635784327983856, 0.14483293890953064, -0.10983005911111832, -1.8692777156829834, 0.14560428261756897, -0.9938685297966003, 0.13996227085590363, -0.36781075596809387, -0.042982544749975204, 0.3532211482524872, -0.2783990800380707, -0.12258855998516083, -0.24386198818683624, -1.375617504119873, -1.2671886682510376, 1.5154402256011963, 0.820662796497345, -1.2635374069213867, -0.6123720407485962, 3.0104188919067383, -0.8884347081184387, -1.724449634552002, -1.1230987310409546, -0.8389860987663269, -0.4554985761642456, -0.24715067446231842, 1.070183515548706, 1.722242832183838, 1.8023370504379272, 0.5041171908378601, -0.5305933952331543, 0.3430219888687134, -1.4706220626831055, -1.5649659633636475, -0.35125768184661865, 1.7847403287887573, 0.5806713104248047, 1.911467432975769, 2.144623279571533, -0.649466872215271, 0.3624137341976166, -1.0828717947006226, -0.302315890789032, 0.6276124715805054, -0.5334581136703491, 2.726316452026367, -0.554243803024292, 0.6678184270858765, 1.6045011281967163, 1.3864396810531616, 0.1504531055688858, 0.8968571424484253, 0.569165825843811, -0.09908190369606018, 0.9337154030799866, -1.1175843477249146, 1.984572410583496, 1.9652156829833984, -0.6994562745094299, 0.6235668659210205, -0.16927459836006165, 0.08962449431419373, -0.2608325183391571, -0.7026759386062622, -0.5709423422813416, 0.023076936602592468, -0.28622961044311523, 1.4196635484695435, -0.02294285222887993, 0.9447048306465149, 0.15554232895374298, -0.15821143984794617, -0.21898259222507477, -0.0318388007581234, 0.5166965126991272, 0.658632218837738, -0.41933882236480713, 1.367139220237732, -0.38885989785194397, -1.7791091203689575, -0.12353542447090149, 1.5929034948349, 0.7795411944389343, -0.8134156465530396, 0.9964950680732727, -0.5907142758369446, 0.1618272066116333, 0.729523241519928, -0.8201582431793213, 0.49259963631629944, -1.1911860704421997, -0.12239307165145874, 0.6346461772918701, 0.9381410479545593, -0.49698466062545776, -0.2936009168624878, 0.8786906003952026, 1.4425774812698364, 0.34305450320243835, -0.9002432227134705, -0.07443375885486603, -0.6070316433906555, -1.3521372079849243, -1.7420059442520142, 0.3744189143180847, -0.07885629683732986, -0.18228136003017426, 0.35094717144966125, 0.271653413772583, -0.24977914988994598, -1.1321966648101807, 2.147942543029785, -1.5318865776062012, -1.786521315574646, 1.2651877403259277, -0.21324387192726135, -0.007523919455707073, -0.7574209570884705, 0.658717155456543, 1.0199942588806152, 0.8190635442733765, 0.7688548564910889, -0.21467246115207672, 1.0741429328918457, 0.10377506911754608, 1.590479850769043, -1.0678973197937012, -2.4419403076171875, 0.15045970678329468, -0.6703928709030151, -0.30913567543029785, 0.6424722671508789, -1.6107730865478516, 0.284780889749527, -0.8070489764213562, -0.8759937882423401, -1.0325442552566528, 1.1072732210159302, -0.09681219607591629, -1.753522515296936, 1.3475013971328735, -0.6612116098403931, -0.8202189803123474, -0.2471751570701599, 1.356758713722229, 0.6778479814529419, 1.1083192825317383, 0.11006028950214386, -0.0666331797838211, 1.3084852695465088, -0.268221378326416, 0.2854076027870178, 1.4085183143615723, 0.6044171452522278, -0.013485358096659184, 1.5167996883392334, -1.3701308965682983, -0.7284499406814575, 1.6222273111343384, -0.23214320838451385, -0.8995662331581116, 0.20105354487895966, -0.044502850621938705, -0.04437152296304703, 1.7386419773101807, 0.15180309116840363, 1.374656081199646, -0.8765894770622253, 0.19838012754917145, 1.5659884214401245, 1.053621530532837, 0.04321017116308212, 1.751360535621643, 0.6302708387374878, -1.1422984600067139, 0.6780471801757812, 1.3482693433761597, 1.7809983491897583, -1.2978949546813965, -0.6595455408096313, -0.8423717617988586, 0.948819100856781, 0.030018392950296402, 0.25002321600914, 0.04649883136153221, 0.17143502831459045, -0.2039192020893097, -0.5852020382881165, -0.7240932583808899, 1.7943320274353027, -0.6947451233863831, 0.36703699827194214, 1.8999154567718506, -0.1969652622938156, 1.792970895767212, 0.4911884069442749, 0.4819223880767822, 0.6618232131004333, 0.02567385882139206, -1.402684211730957, -0.6017785668373108, -0.7455958127975464, -0.643028736114502, -0.05845166742801666, -0.26276907324790955, 2.3223531246185303, 0.38531294465065, -0.965969443321228, -0.34703075885772705, 0.19254592061042786, -1.932550311088562, 0.09991837292909622, -0.476165235042572, 1.320103406906128, 0.4865848422050476, -1.078694224357605, -0.9226717948913574, 2.0653042793273926, -0.08739878237247467, 0.5445189476013184, -0.8949984312057495, 1.9557100534439087, 0.7285201549530029, 1.0336692333221436, -1.0332111120224, 0.5815182328224182], [1.11125910282135, -1.0798619985580444, 0.9917196035385132, -0.5017690062522888, -0.5594943165779114, -0.17045249044895172, 1.650270700454712, 0.9385486841201782, -0.9265055060386658, 0.1602834016084671, 0.5811394453048706, 1.1552965641021729, 0.641637921333313, -1.5477880239486694, 0.5802392363548279, -0.8525307774543762, 1.0163264274597168, 0.0365227572619915, -0.3801129460334778, -0.3838059902191162, -1.5805050134658813, 0.48137277364730835, 2.137615203857422, 0.4868144094944, -1.217828631401062, 1.5623785257339478, 1.58668851852417, -0.5863220691680908, 0.3906302750110626, 1.0889167785644531, -0.10262985527515411, -0.08289853483438492, 0.6185364723205566, 0.9647009372711182, 0.9914990067481995, 0.9611239433288574, -0.11960025131702423, -1.322257161140442, 0.07629990577697754, 0.29481321573257446, 0.2548488676548004, 0.6118259429931641, -1.1733884811401367, 1.2392985820770264, -1.561214804649353, 0.08347144722938538, 0.09187063574790955, 1.041225552558899, -1.3570231199264526, 0.8221877813339233, -0.12262070178985596, -0.18381790816783905, -1.0977815389633179, 1.218492865562439, -1.0388413667678833, 0.26074740290641785, 0.1607159674167633, -0.030917908996343613, 0.2399446964263916, 0.6577595472335815, -1.2870450019836426, 0.4968067407608032, 1.054284691810608, -1.3432921171188354, -1.2938902378082275, 2.394467353820801, 0.04103652387857437, 0.515229344367981, 0.2913559675216675, 2.0791189670562744, -0.701619565486908, 1.6594946384429932, 0.8863129615783691, -0.8082846999168396, 0.19081859290599823, 0.33418503403663635, 0.4397469162940979, 0.2782588005065918, 0.8702758550643921, 1.3637523651123047, 1.5916661024093628, -0.3285454213619232, -0.4387474060058594, 0.12298616766929626, 0.4662424325942993, -0.9071386456489563, -0.32732751965522766, -1.3262990713119507, -0.4775083363056183, -1.4573814868927002, 1.2940514087677002, -0.9207049608230591, -0.3196185529232025, 0.927388608455658, -0.009178172796964645, 0.8518038392066956, 0.8047870397567749, 1.661946177482605, -0.9858939051628113, -0.28228291869163513, -0.3649495840072632, -0.17332226037979126, 1.8080297708511353, 1.0284311771392822, 0.6831780672073364, -0.06922809779644012, 0.3328370153903961, 0.22153787314891815, -0.9592098593711853, -0.22375045716762543, -0.3026076853275299, -0.5729148387908936, 2.0171775817871094, -0.9451087713241577, 0.6411675214767456, -0.9361047744750977, 0.20168456435203552, 2.612102746963501, -0.6778935790061951, -1.4801260232925415, -0.10334484279155731, 1.4067431688308716, -0.5160983800888062, 1.0413159132003784, -0.7258486151695251, -1.0064350366592407, -1.4474972486495972, 0.8557247519493103, 1.5211784839630127, -0.1938912272453308, -1.2510638236999512, 0.9758825898170471, -0.763848602771759, 0.30060631036758423, -0.399535208940506, 1.415679931640625, -1.3651540279388428, -0.6286129951477051, -0.2970835864543915, 0.1403750628232956, -0.2623400092124939, -0.349528431892395, 1.0230364799499512, 0.3707016706466675, -0.14051908254623413, 1.6735819578170776, 0.036577481776475906, -0.6984319090843201, 0.00799806322902441, -0.4849078357219696, -0.38764533400535583, -0.2416819930076599, 0.08969797939062119, 0.040830548852682114, -0.4583989679813385, -0.3035500943660736, -0.689034104347229, -0.711868941783905, 1.3864541053771973, -1.2246137857437134, 1.1921744346618652, 0.34826377034187317, 0.38436099886894226, -0.5894032716751099, 1.724495768547058, -1.6796971559524536, 0.38752683997154236, -0.30634450912475586, 0.0648203119635582, -0.6297933459281921, -1.4871715307235718, -0.48526275157928467, 1.7708337306976318, 0.3184215724468231, -0.5183253884315491, 0.5599843263626099, -0.5529625415802002, -0.792364776134491, 0.04266098886728287, 1.115559697151184, -0.27236008644104004, -0.2777835726737976, -0.0836336612701416, -0.009889923967421055, -0.8910710215568542, -0.027553733438253403, -0.007478110957890749, 0.22789007425308228, 1.6483312845230103, -0.4272517263889313, 0.9900424480438232, 0.6400688290596008, -0.16996002197265625, -0.884534478187561, -0.9977012872695923, -0.2929162085056305, -0.3082153797149658, -1.2717270851135254, -0.6938529014587402, -0.37000733613967896, 1.0657705068588257, 0.8099170327186584, -1.0255968570709229, 1.15631902217865, -0.04733751341700554, 0.6378936767578125, -3.087541341781616, 1.136051058769226, -1.5031124353408813, 1.0630934238433838, -0.021061327308416367, 1.7215505838394165, -0.5462564826011658, 1.4031577110290527, -1.6126004457473755, 0.30908092856407166, -1.097806692123413, 0.20206917822360992, 0.1344638168811798, -1.5532667636871338, 0.8039976358413696, -2.3477768898010254, -1.2530730962753296, -1.3433557748794556, 0.5441113114356995, -0.09459061920642853, -0.09519948810338974, -0.10176809877157211, 0.715243935585022, 0.8029209971427917, -0.22903868556022644, 0.03768746554851532, -0.5610586404800415, 0.09229651093482971, -0.7897917032241821, -0.4147210121154785, -0.11235951632261276, -0.9787519574165344, 0.010146848857402802, 0.5105433464050293, 0.33186766505241394, 0.6672773957252502, 0.36701154708862305, 0.3055851459503174, -0.6339925527572632, -0.2962820827960968, 0.6517873406410217, -0.12204837054014206, -1.0525468587875366, 0.07702425867319107, 2.4507763385772705, -0.7309800386428833, 1.0177059173583984, 0.5544480681419373, -0.09698275476694107, 0.1366441696882248, 0.19749394059181213, 1.8906711339950562, -0.1949807107448578, 0.7802814245223999, 1.241721510887146, 0.8609161972999573, 1.2890065908432007, -0.4712204337120056, 0.7067062258720398, -1.054159164428711, 2.2731688022613525, -0.7342060208320618, 1.0710164308547974, -0.9575654864311218, 0.6759045124053955, 0.5463942885398865, -0.43349552154541016, 0.4419299364089966, -1.668665885925293, -1.3628122806549072, 0.2741532325744629, 0.06030883267521858, -0.03412909805774689, 0.39202776551246643, 0.5021315217018127, -0.8608651161193848, 0.1952127069234848, 0.10237521678209305, 0.9727709889411926, 0.17060232162475586, 1.3619725704193115, -1.517728328704834, 1.7440650463104248, 0.009587455540895462, 0.017315402626991272, 0.3240715563297272, 0.4159277677536011, 0.004235611762851477, 0.1957739144563675, 0.35651853680610657, -1.5470479726791382, 0.5461186170578003, -0.5091609954833984, 0.4823629856109619, 0.034460850059986115, -2.9750325679779053, -1.9149878025054932, -0.799649178981781, -1.379410743713379, 2.4231972694396973, -1.5079877376556396, 0.4639313519001007, 2.019120216369629, 0.4895069897174835, -0.272411972284317, 1.4677245616912842, -1.666930913925171, 1.5895001888275146, -1.1535836458206177, 0.7345266342163086, -0.7919539213180542, -0.2656964361667633, 1.6655004024505615, -0.17962022125720978, -0.016813693568110466, -0.40289488434791565, -0.5685054659843445, 1.7848005294799805, -2.661933660507202, 0.2863563001155853, -0.24628041684627533, -0.44107604026794434, 1.6121139526367188, -0.6435015797615051, -2.2343077659606934, -1.4060924053192139, 0.9843968749046326, -0.4713563323020935, -1.762729287147522, 0.4474802315235138, -0.4740225076675415, 1.2171859741210938, -1.491146206855774, 0.389544278383255, -0.7944040298461914, -0.1724717915058136, -0.5566670894622803, -1.0733611583709717, 0.012356899678707123, -1.4309357404708862, -1.188607931137085, 2.0312247276306152, -0.7839646935462952, -0.511011004447937, -1.5673516988754272, -0.10013137757778168, 1.0963475704193115, 0.4153352677822113, 0.7847737073898315, 0.2600879669189453, -0.8467198014259338, -0.27412551641464233, 2.4504172801971436, -0.6943226456642151, -0.4265994131565094, -1.1401981115341187, 1.0719281435012817, 0.963890790939331, 1.6679165363311768, -1.1269176006317139, -1.0321069955825806, -0.5621095895767212, 1.1510063409805298, 1.9891977310180664, -0.3806370496749878, 1.1082171201705933, 1.1338058710098267, -0.4469604790210724, -0.7094902396202087, -0.765443742275238, 0.09899357706308365, 0.7134730219841003, 0.5956149101257324, 3.039292335510254, 0.4025920033454895, 0.004971714224666357, 1.3556513786315918, -0.16755034029483795, -1.2023524045944214, -0.11664875596761703, -1.9104256629943848, -0.5347079634666443, -0.9670445919036865, 1.2189041376113892, -2.9156360626220703, -0.8602803945541382, 1.2146782875061035, -1.2564160823822021, -1.256166934967041, -0.16370326280593872, -0.8473892211914062, 1.4434759616851807, -1.7647091150283813, 0.3888331353664398, -1.4207708835601807, 0.7652764916419983, -0.8127434849739075, -0.8641346096992493, -0.43681061267852783, 0.07732751220464706, 1.1203056573867798, -0.6998937726020813, 1.3312357664108276, 0.10890935361385345, -0.21731141209602356, 0.015519518405199051, 0.8525631427764893, -2.137657403945923, 0.3479306101799011, -1.3116729259490967, 0.32527831196784973, -0.974279522895813, -0.3810828924179077, 0.5256906747817993, 0.47311931848526, -0.399071604013443, -0.392583966255188, -0.6186831593513489, 0.982014000415802, 0.6203734278678894, -1.153450608253479, -0.8531596064567566, 0.27634018659591675, 1.9464614391326904, 0.009126278571784496, -0.5320669412612915, -0.5938345789909363, 1.528764247894287, 1.6528010368347168, -0.33985647559165955, 0.6547355651855469, 1.1490532159805298, 0.22014635801315308, 0.48539602756500244, -1.1376157999038696, -0.3851723372936249, -0.3111060559749603, 2.9107563495635986, 0.618587851524353, -0.5491310954093933, 1.0718982219696045, -0.5483062863349915, -0.7253700494766235, -0.6945855021476746, -1.4609352350234985, 0.5162695050239563, -0.7795255780220032, 0.4832007586956024, 1.0092954635620117, -1.2696181535720825, 1.4834375381469727, 1.7828319072723389, -2.028684377670288, 0.359637051820755, 0.3036434054374695, 1.0382835865020752, -2.4350295066833496, 1.3672587871551514, 0.4601888358592987, -0.36789020895957947, 0.5409701466560364, -0.06784098595380783, -2.3971266746520996, 0.10253442078828812, -0.16395442187786102, -1.6209360361099243, -0.7551672458648682, 0.5586546659469604, 0.23566262423992157, 0.5927411317825317, 0.3358796536922455, 1.7129353284835815, 0.05980200693011284, 1.260149359703064, -0.9908931851387024, -0.08940815925598145, 1.1756459474563599, 0.7443610429763794, 1.1271623373031616, 0.0922086238861084, -0.617520809173584, 1.147044062614441, 0.8616748452186584, -0.9393100738525391, 1.325265884399414, -0.35098057985305786, -0.15508179366588593, -1.4582206010818481, 1.1059398651123047, 1.4827625751495361, 0.05982661619782448, -1.533328890800476, -0.3608773648738861, -0.17417335510253906, -0.09713834524154663, -1.5640840530395508, 0.11487001180648804, -0.7658911943435669, -0.11946781724691391, -0.9069361686706543, -0.5512952208518982, -0.06597428768873215, -0.5636330246925354, 0.4072863757610321, -1.8428113460540771, 0.3742854595184326, 0.39422354102134705, 1.5074557065963745, -0.24276508390903473, 0.4000832736492157, -0.69240403175354, 0.23942318558692932, -0.00630924804136157, 0.05717717483639717, 0.598456859588623, 3.915088176727295, -0.9371919631958008, -0.7769051790237427, 0.36125192046165466, -1.144775390625, 0.2387070655822754, 1.60824716091156, -1.1096599102020264, 0.9749760031700134, -1.2002835273742676, -0.6328902244567871, -0.8230597376823425, -0.9477189183235168, -0.5594399571418762, -1.293317198753357, 1.0674976110458374, 0.03777853399515152, -0.0522671639919281, -0.18303395807743073, 0.7907649278640747, -0.4752230644226074, -0.7799988985061646, 0.6898303627967834, 0.6893605589866638, 0.010826299898326397, -0.2624985873699188, -0.193589448928833, -0.6415342092514038, 0.7801554203033447, -0.44214028120040894, 1.5992224216461182, -0.07271497696638107, -0.5295101404190063, 0.20971858501434326, 0.3774645924568176, 0.20849297940731049, -1.238904595375061, -0.5943489074707031, 0.786080539226532, 2.0501692295074463, -0.28647634387016296, -1.3632299900054932, 0.8556448817253113, 0.3568663001060486, 1.2195359468460083, 1.0585554838180542, 0.48225006461143494, 0.038469795137643814, -0.4910374879837036, 0.00991254672408104, -1.7172539234161377, 0.03093799389898777, 1.7035057544708252, 1.805079460144043, 1.546412706375122, 2.7590134143829346, 1.0917078256607056, -0.26830023527145386, -0.5161985158920288, -0.3150838017463684, 0.7245841026306152, -0.9611476063728333, -1.4727307558059692, -0.34696662425994873, 0.4813626706600189, 1.1681691408157349, 2.0895133018493652, -0.1590038239955902, 0.9379284381866455, 0.39350971579551697, -0.05298779904842377, -2.3409156799316406, 0.7476544380187988, 0.20309613645076752, -0.1450633704662323, 1.0303364992141724, -0.6007535457611084, 0.2323664128780365, 1.887871265411377, -0.2988126873970032, -2.043700933456421, -0.8933704495429993, -0.8769398927688599, -0.875696063041687, -0.054395027458667755, 1.0180490016937256, 0.3510574996471405, 1.643333077430725, 0.2684713304042816, -0.010434824042022228, 0.8587925434112549, -0.8149250745773315, -1.2789465188980103, -0.6198849081993103, -0.8961567282676697, -0.48601940274238586, 0.36495348811149597, -0.09422095119953156, -0.6192765235900879, 0.5053489208221436, -0.6141182780265808, 0.6706427335739136, -1.4610309600830078, -1.1678849458694458, -2.3975672721862793, -0.6434186100959778, -0.4933050870895386, 0.26587769389152527, -0.7813608646392822, 1.0356751680374146, -0.04737323150038719, 0.8286687731742859, -0.7344639897346497, -1.3644359111785889, -0.2906608581542969, -1.2451927661895752, 1.1140103340148926, -0.9820582270622253, -0.6173385381698608, 0.37075549364089966, -1.4988200664520264, -0.01190872397273779, 0.6155073642730713, 0.36838528513908386, 1.600365400314331, 0.1856769472360611, -0.9780107736587524, 0.8660126328468323, -0.0604407899081707, -0.40515246987342834, 0.0012676578480750322, 1.232334017753601, 0.10200075060129166, -1.1292498111724854, 0.20376567542552948, 0.9452360272407532, 0.6640990972518921, 0.605033278465271, -0.6770334243774414, -1.9691864252090454, -0.44154658913612366, 0.7193023562431335, -1.535800576210022, -0.9657467007637024, -2.012314796447754, -0.07005786150693893, -0.14973975718021393, -0.12506169080734253, 0.4213584065437317, -0.5979064702987671, 0.5734129548072815, 1.4815768003463745, 0.032609038054943085, -0.18522854149341583, -0.40502116084098816, 1.7553836107254028, 0.8765381574630737, 0.2733931243419647, -1.6368403434753418, 0.7903666496276855, -1.170371174812317, 0.519563615322113, -2.6061623096466064, 0.3680609166622162, -0.5715616345405579, 1.4414559602737427, -1.3394311666488647, -1.120133399963379, 0.7636361718177795, 1.6252106428146362, -0.367198646068573, -0.8368903398513794, -0.18822415173053741, -0.35820531845092773, 1.5037846565246582, -1.2118542194366455, -0.9624781012535095, 0.5822470188140869, -0.777863085269928, 0.8249338269233704, 0.46817612648010254, -0.16470441222190857, 0.7287614941596985, 3.799342393875122, -0.20437094569206238, 0.9028573036193848, -1.9694254398345947, -0.002258890075609088, -0.41358545422554016, -0.14597782492637634, 1.0342546701431274, -1.367799997329712, 0.19462484121322632, -1.3261646032333374, -0.9876253008842468, -0.11439383774995804, -0.9665663838386536, -0.06380987912416458, -1.0156296491622925, 1.6186410188674927, -0.7863871455192566, 0.5344865918159485, -1.4401874542236328, 0.17988525331020355, -0.8775489926338196, 1.050808310508728, 0.40711966156959534, -0.43979501724243164, 0.7014339566230774, -1.1568852663040161, -3.1229093074798584, 0.4269537329673767, -0.5529458522796631, -0.9068339467048645, 0.9097758531570435, 1.6561518907546997, -0.9134861826896667, 0.7248145937919617, -0.74976646900177, -1.4684932231903076, 0.18743857741355896, -1.282002568244934, 0.7276372909545898, 0.4987585246562958, -0.4748545289039612, -0.20563188195228577, 0.763166606426239, -2.0175185203552246, 0.07432372868061066, -0.13990429043769836, 0.9614346027374268, -1.5182290077209473, 0.44967910647392273, 0.27026432752609253, -0.34052789211273193, 0.3621532618999481, 0.26673054695129395, -1.153209924697876, 0.565865695476532, 2.4790399074554443, 0.21116024255752563, 0.2275346964597702, -1.410068392753601, 0.1483064740896225, -1.8626419305801392, 0.08797548711299896, -1.0905001163482666, 0.9461173415184021, 1.1428403854370117, -0.07300082594156265, -0.7671194076538086, 0.632218599319458, 0.4792937636375427, -0.3765462338924408, 0.051655758172273636, -0.951060950756073, 0.486490935087204, -0.9168615341186523, -0.2552037537097931, -0.3863604962825775, -2.2162413597106934, 1.1348508596420288, -0.5590065121650696, 0.7512210011482239, -0.06246509030461311, -0.05432983860373497, 0.7831265926361084, -0.20862442255020142, -0.2622440755367279, 0.966895580291748, -0.30040284991264343, 0.5120207071304321, -0.7133198976516724, -0.28612563014030457, 1.0762879848480225, -0.7701629400253296, -0.7023245692253113, -0.1517305225133896, -0.49212950468063354, -1.5379970073699951, -0.596973180770874, -0.12534156441688538, 0.7480555176734924, 0.8802980780601501, -1.1734260320663452, 0.40743890404701233, 0.25155335664749146, 0.25125741958618164, -1.0007314682006836, 0.7207816243171692, -0.8308921456336975, -0.5179654955863953, 0.16440331935882568, -0.3921187222003937, -1.2975293397903442, 0.3701633810997009, 1.5236566066741943, -0.13898785412311554, -0.5007705092430115, 1.2559641599655151, 0.08447134494781494, -0.6186022758483887, -0.9182301163673401, 0.8486760258674622, 0.2991778552532196, -2.3330044746398926, 1.5421663522720337, 0.15253783762454987, 0.11940158903598785, -1.2467948198318481, -0.4952595829963684, -0.022828644141554832, 1.4432439804077148, -1.6468697786331177, 0.4050496816635132, -0.7153829336166382, 0.06331408768892288, -0.19921894371509552, -1.2357285022735596, 0.6594358086585999, -0.9463573098182678, 0.28348129987716675, -1.116113543510437, -0.629604697227478, -0.13995173573493958, 1.4641343355178833, 0.6955620050430298, 0.6977605819702148, -0.7592908143997192, 1.512557029724121, -0.23283928632736206, 0.7095998525619507, 1.5789316892623901, -0.3841153681278229, -0.47053712606430054, 1.115640640258789, -0.770221471786499, 0.01779734529554844, -2.8735158443450928, -0.9053876996040344, 0.1577039659023285, 0.339339941740036, -0.11296790838241577, 0.29683616757392883, 1.7538862228393555, 0.35731643438339233, 0.5826629996299744, 1.4239954948425293, 1.2548975944519043, 0.736478865146637, 1.3914381265640259, 1.0594508647918701, 1.9997590780258179, 0.3067798912525177, -0.509158194065094, -0.35149645805358887, 1.2639224529266357, 0.2328263819217682, 0.9728251695632935, 0.8578081727027893, 1.1639888286590576, 1.168087363243103, 1.0922902822494507, -0.16517725586891174, 1.5587267875671387, -1.9502280950546265, 0.4598364531993866, 0.21888281404972076, 1.1830753087997437, -0.9118900299072266, -0.7499266862869263, 0.03061399981379509, 1.1111419200897217, 0.6036825776100159, 0.7593805193901062, -2.1947743892669678, -0.1765553504228592, -0.2122924029827118, -2.04587984085083, 1.87520432472229, -0.6718432903289795, -0.5517841577529907, -1.3016539812088013, -0.23103030025959015, -0.5756585597991943, -0.016475969925522804, 0.3249377906322479, -0.32532039284706116, 1.3480454683303833, -1.248215675354004, 0.3872300386428833, -1.500693678855896, -0.6211389899253845, 0.38953137397766113, 1.168923020362854, 0.11173789948225021, -1.0260671377182007, -1.1903647184371948, 1.3139960765838623, 0.793148934841156, 0.573655366897583, 1.8618015050888062, 0.29192811250686646, 0.4263631999492645, -1.0599530935287476, 0.2905574142932892, 0.7433613538742065, -0.5142567157745361, 0.6387597918510437, -0.3331252634525299, -0.5645941495895386, 0.00927149411290884, -0.6511011719703674, 0.23948915302753448, -0.17443940043449402, 0.6948868036270142, -1.615997314453125, 3.134279727935791, -0.49460867047309875, 1.06352961063385, 2.2349557876586914, -1.249474287033081, -0.9644705653190613, -0.07943780720233917, 0.023308314383029938, -0.07112307846546173, 0.06628681719303131, -0.41091737151145935, -0.23234815895557404, 1.0433790683746338, 0.7737411260604858, 0.12352171540260315, -0.157782182097435, -0.44475802779197693, -0.20904481410980225, 1.8130899667739868, -0.7191997170448303, 0.08746617287397385, -0.8178333044052124, 0.0847337394952774, -0.7359780073165894, 0.17000973224639893, 0.6791908740997314, 0.010754825547337532, 0.9858732223510742, -0.7119511961936951, 1.1198184490203857, 1.4777199029922485, -3.0084497928619385, 0.6661496162414551, 0.815410315990448, 0.6679545640945435, -1.3131378889083862, 1.0501925945281982, -1.1383360624313354, 1.5527673959732056, -0.7531001567840576, 0.4405066668987274, 0.5396204590797424, 1.4453731775283813, -0.9351391792297363, -0.7193866968154907, -0.5065340399742126, 0.33692339062690735, 0.8031637668609619, -0.0022691304329782724, -0.4541926383972168, 1.902864933013916, -0.9056730270385742, 0.030895667150616646, -0.7091310024261475, -0.08243873715400696, 0.9392620325088501, 0.5512979030609131, -0.07977530360221863, 1.2332110404968262, -1.4202880859375, 0.6410032510757446, -0.14148220419883728, 0.5724732875823975, -0.10702550411224365, 0.19278885424137115, 0.38682103157043457, 0.2362469583749771, 0.6118260622024536, 0.4058643877506256, 1.3681477308273315, -0.22659029066562653, -0.1758086234331131, -1.0971850156784058, 0.104523666203022, 1.4765219688415527, -1.7458961009979248, -0.32267600297927856, 0.6308770179748535, 0.1271853893995285, -1.2628164291381836, 0.4929547905921936, 1.183957576751709, 0.5626934170722961, -2.203495502471924, -0.1549176722764969, -0.7290830612182617, 0.002357668476179242, -0.34462520480155945, 0.6758986115455627, 1.005576252937317, 0.4784270226955414, -0.6909182667732239, -1.9024710655212402, 0.4090110659599304, -0.5230823755264282, -0.4483577311038971, 0.8401327729225159, -0.7408725619316101, 0.8134726881980896, 0.21730616688728333, -0.04723947495222092, 1.3924814462661743, 0.9310241937637329, -1.043554425239563, 0.9080982208251953, -1.7285683155059814, 0.3469989597797394, 0.32196044921875, -0.9472343921661377, -0.26289132237434387, 0.14462924003601074, -0.8337744474411011, -0.9337775111198425, -1.1292229890823364, -1.183579921722412, -0.006145301274955273, 2.011446237564087, 0.7391869425773621, 0.24278083443641663, 1.490858793258667, 0.7862298488616943, 0.482658714056015, 0.2553321421146393, -1.01389479637146, -0.432136207818985, -0.7581987977027893, 1.0255814790725708, 0.86250901222229, 0.3710463047027588, -2.242347240447998, 0.12225987762212753, 0.05659295618534088, -0.3071073591709137, 0.6110531687736511, -0.21867355704307556, 0.7474160194396973, -1.2718353271484375, 2.0938796997070312, -1.4514288902282715, 0.06224522739648819, 1.042803406715393, -0.8419526815414429, 0.08748574554920197, -0.6983837485313416, 0.06234697625041008, -1.4823490381240845, 0.05568496137857437, -2.4586236476898193, -0.594815731048584, -0.044448330998420715, 0.8829501867294312, 0.631088376045227, 0.13692384958267212, 1.7503994703292847, -0.11445794999599457, -0.9642813801765442, -0.14005209505558014, -0.03709757328033447, -0.21774131059646606, 0.9593687057495117, -0.1231461688876152, -0.09326746314764023, -0.6456884145736694, 0.3899136781692505, -0.034115709364414215, 0.360312819480896, -0.24627630412578583, 1.4825263023376465, 0.037449996918439865, 0.11200657486915588, -0.05663595348596573, 0.6273289322853088, -1.1294021606445312, 0.38046255707740784, 1.395895004272461, 1.3458298444747925, -0.55576491355896, 0.4236927032470703, 1.9426863193511963, -0.12744395434856415, -0.13358311355113983, -0.25097718834877014, 0.9339279532432556, 0.9641011357307434, -0.3047262728214264, 0.252432256937027, -0.23701703548431396, 0.3829565942287445, -0.045053720474243164, -1.013437032699585, 0.4367332458496094, 0.4171721041202545, -0.7261602878570557, -0.3274669647216797, 0.4979010820388794, 1.7487062215805054, -0.8312462568283081, 1.65627920627594, -0.46149009466171265, 0.05362556502223015, 0.3656158745288849, 0.3478104770183563, 1.2325862646102905, 0.5857746005058289, 0.1638445258140564, 0.30041074752807617, 1.3135099411010742, 0.5359313488006592, 1.7492413520812988, -0.4639909863471985, -0.2539607286453247, -0.1383771002292633, 0.7868923544883728, -0.7390356063842773, -0.7080623507499695, -0.49818411469459534, -0.29822564125061035, 0.5739648938179016, 0.8842484951019287, -2.068998336791992, 0.14579728245735168, -1.180050253868103, -0.582594096660614, 0.7046804428100586, 0.32816019654273987, 0.6097383499145508, 0.3955579102039337, -0.5223520398139954, 0.6705193519592285, -0.7109400629997253, -0.8091840147972107, -0.47475898265838623, -0.2925690710544586, 0.5646092295646667, 0.5710680484771729, -1.4178333282470703, 1.420805811882019, 1.0882291793823242, 0.8927525877952576, 0.021215325221419334, 2.1261324882507324, 1.710700511932373, -2.2491486072540283, -0.8599629998207092, -1.3925975561141968, -0.17285451292991638, 0.4119105637073517, -0.0691135972738266, 0.877483069896698, -1.1081544160842896, -0.9851570725440979, 0.9948915839195251, 1.0467383861541748, 0.1620238721370697, 1.5201791524887085, -0.6899480223655701, 0.16792573034763336, -1.114844799041748, -0.20810216665267944, 0.7545529007911682, 0.6961208581924438, 0.21630129218101501, 0.29467645287513733, -0.46910804510116577, 0.6087960004806519, -0.9702779650688171, -0.48632580041885376, 1.272453784942627, 0.6049465537071228, 0.22667564451694489, -1.148608922958374, -0.10094443708658218, 1.673342227935791, 1.9141846895217896, 1.7579114437103271, -0.4131709933280945, -2.460343360900879, -0.0725722461938858, -0.6015750169754028, -0.9984400272369385, 0.7614615559577942, -0.13831746578216553, 1.1610314846038818, 0.41732123494148254, 1.7471392154693604, -0.06963364779949188, 2.1531314849853516, -0.287394642829895, -0.0830102190375328, 0.4532002806663513, 0.31879374384880066, -0.7149928212165833, 0.07415959239006042, 1.6355016231536865, 0.8453992009162903, 1.9639551639556885, -0.6106502413749695, -1.0443869829177856, -1.335998773574829, 1.0826517343521118, 1.1863377094268799, 1.1730998754501343, 1.8935604095458984, 0.7487928867340088, -0.8574307560920715, -0.27872931957244873, 0.5738263130187988, -0.28547585010528564, -0.11163091659545898, -2.082113027572632, 0.6788467764854431, 0.37420719861984253, 1.3431000709533691, 0.33986422419548035, 1.3626980781555176, 0.017211349681019783, -1.192771315574646, 1.4543126821517944, 0.4298989772796631, -1.1554269790649414, 0.8486207723617554, -0.921874463558197, 0.0787043571472168, 0.9185577630996704, 1.3947765827178955, -0.8320640921592712, 1.6036409139633179, -0.12090000510215759, 0.3562440574169159, 0.13193866610527039, 0.5637742280960083, 0.4589732885360718, 0.42982926964759827, 0.835760235786438, -3.4439520835876465, -1.0270256996154785, 0.3452695310115814, -0.21985137462615967, 0.32211723923683167, -0.6538613438606262, -0.2311096042394638, 0.38842305541038513, 0.5576921105384827, 0.4567638635635376, 1.407492995262146, -0.6518839597702026, 1.950548768043518, -0.461554616689682, -0.04505866765975952, 0.14315877854824066, -1.0563337802886963, -0.20591741800308228, -2.034423828125, -0.13488492369651794, -0.0508757010102272, 0.049232278019189835, 0.34622207283973694, -0.2588978707790375, -1.8132190704345703, -1.9328194856643677, -1.3538563251495361, -1.7841715812683105, -0.875419557094574, -0.9209040403366089, 0.24839311838150024, 1.4782600402832031, -0.4248816668987274, -1.8716540336608887, 0.6762053370475769, 0.0312351007014513, 0.44257351756095886, -0.1795070320367813, -2.433499813079834, -0.2808070778846741, 1.2086598873138428, 0.5894544124603271, 0.6302732825279236, 1.4766266345977783, -1.1409670114517212, -0.8930710554122925, -1.2073214054107666, 1.404856562614441, 0.9125550389289856, -0.6514694094657898, -0.8429638147354126, -0.3065119683742523, -0.18529058992862701, 1.103975534439087, 1.477489709854126, 0.6099002957344055, -0.03265118598937988, 0.45027589797973633, 0.516798198223114, -1.4829260110855103, 0.6259980797767639, -2.411902904510498, -0.7385749220848083, 0.9128895401954651, 0.8142134547233582, -0.06258506327867508, -1.4385836124420166, 1.117740511894226, -0.3619115948677063, -0.137173593044281, -0.7725969552993774, -0.6986818313598633, 1.4138103723526, -1.0243273973464966, -0.7587370872497559, -0.5476953387260437, 0.45528459548950195, 1.4549378156661987, -0.0598578117787838, -0.911685585975647, -1.3835545778274536, 0.4386734366416931, 0.14747409522533417, 0.1745533049106598, 2.3400421142578125, 0.6045953035354614, -0.46506205201148987, -1.6841260194778442, 1.2404582500457764, -0.45735564827919006, -1.4924805164337158, -0.5103808641433716, 0.8807282447814941, 0.05959663540124893, -0.8887012004852295, 0.05926141515374184, 1.0863373279571533, -0.006403209641575813, 1.3630229234695435, -0.8058286309242249, 0.08297719061374664, -0.6725166440010071, 0.4166431128978729, -1.5236972570419312, 1.1493078470230103, -1.0358980894088745, -1.2193244695663452, -0.5609800219535828, -0.92058265209198, -2.255866050720215, -2.549452304840088, -0.2682417333126068, 1.4790594577789307, 0.4235752820968628, 2.3694419860839844, 0.30829355120658875, -1.8881211280822754, -0.4652583599090576, 1.2223817110061646, 1.2797731161117554, -1.1078643798828125, -0.6183483600616455, 1.9347889423370361, 0.25203970074653625, -0.006840872578322887, 0.11606209725141525, 0.06967151910066605, -0.3498225212097168, 0.8900466561317444, 0.023574506863951683, 0.3833496868610382, 2.751525402069092, -0.7671172022819519, 0.39536789059638977, 1.4499012231826782, -0.7911302447319031, -0.7603453993797302, 0.5035615563392639, 1.1302790641784668, 1.0323668718338013, -0.6489163041114807, 1.4679380655288696, -0.49019861221313477, -1.644018292427063, 0.8751993179321289, -1.4074562788009644, 1.2560375928878784, 2.195300817489624, -1.1124217510223389, -2.1634292602539062, -0.6970506310462952, 0.5611889362335205, 0.20687440037727356, -1.5432887077331543, -0.29289671778678894, -1.5505505800247192, -0.47754064202308655, 2.343078374862671, 1.3459417819976807, 0.9930845499038696, -2.220881700515747, 0.5046408176422119, -0.8651726245880127, -3.3993217945098877, -0.5975590348243713, 0.536406397819519, 0.06564170122146606, 0.006060847081243992, 0.0014566569589078426, 0.6863496899604797, 0.20152343809604645, 0.17774128913879395, -0.5641042590141296, 1.5250605344772339, 1.4135489463806152, -0.9558128118515015, -0.48438259959220886, 0.13289077579975128, -0.278473436832428, -0.006115038879215717, 0.13778303563594818, -0.35121142864227295, 0.9262335300445557, 0.6543933749198914, -1.6087156534194946, -1.2710307836532593, -1.1697475910186768, 0.17684561014175415, 0.034543201327323914, 1.1251511573791504, 0.3060029447078705, 0.473518043756485, -0.04737148433923721, -1.3484395742416382, 1.5478320121765137, -0.05135994777083397, 0.602840781211853, -1.7301148176193237, 0.8558871746063232, -0.4857873320579529, 1.129563808441162, -0.29542645812034607, -0.43402841687202454, -0.8750261664390564, -0.03745761513710022, -0.22300025820732117, 0.1522761881351471, -0.038911379873752594, 0.8862207531929016, 0.7351979613304138, -1.000922679901123, 0.7839276790618896, 2.7047119140625, -1.3315482139587402, -0.7224228978157043, 0.13554945588111877, 0.7688589096069336, -0.6574130058288574, 0.7842831015586853, 0.9023828506469727, 1.8866780996322632, -0.7676097750663757, 0.1689697802066803, 0.6764985918998718, 0.09334223717451096, 0.16341464221477509, -0.07745511084794998, 0.8728029131889343], [-2.357983112335205, -0.1361236423254013, 1.3180841207504272, -0.6605616807937622, -1.0957739353179932, -1.0398097038269043, 1.34518563747406, 1.1238727569580078, 1.5045421123504639, 0.9739223718643188, -0.15557348728179932, -0.06683805584907532, 0.7300785183906555, -0.6839550733566284, -0.28836488723754883, 0.6946050524711609, 0.39272192120552063, -0.43800705671310425, -0.675166666507721, 1.3087337017059326, 0.5295761227607727, -1.3871647119522095, -0.8004405498504639, -0.07530266791582108, 0.199929341673851, 0.266187459230423, -0.9750139117240906, -0.2896624207496643, 0.9059774875640869, -0.693791389465332, 2.10317063331604, -0.9207683205604553, 0.20644333958625793, -0.6774090528488159, 1.0459543466567993, -0.35915178060531616, 0.6590732932090759, 0.40654832124710083, 0.5198859572410583, 0.6905845999717712, 1.1586920022964478, -1.3834770917892456, -0.05500820651650429, 0.4125855565071106, -0.054607439786195755, -0.9947008490562439, 1.0812013149261475, -0.11588984727859497, 0.637543797492981, 1.2261005640029907, 1.4461363554000854, -0.2781898081302643, -1.2196320295333862, -1.2788066864013672, 1.2422696352005005, 0.047878511250019073, 0.18966300785541534, 1.4076740741729736, 0.23007644712924957, 0.7953271865844727, 0.06651671975851059, 1.1807206869125366, -1.1112877130508423, -2.0512514114379883, -0.23838573694229126, 0.7598744630813599, -0.3643094003200531, -0.28643539547920227, -1.9031610488891602, 0.13517628610134125, -0.0426500029861927, -0.9128066897392273, -0.4048783481121063, 0.2925151586532593, -1.5320090055465698, 1.377858281135559, -0.780716598033905, -0.07802124321460724, -1.0915741920471191, -1.699048638343811, -0.5897713303565979, -0.033913783729076385, 0.06660380959510803, 1.2850558757781982, -1.5436131954193115, 0.1521911472082138, -0.9380262494087219, -0.7190484404563904, -1.1937103271484375, -0.5901510715484619, -1.471096158027649, 0.6251531839370728, 0.423544704914093, -0.08795557916164398, -0.5464038848876953, 0.2783862352371216, -0.09088768064975739, 0.789154052734375, 0.5447728037834167, 1.3751434087753296, -0.6096683144569397, -1.5867336988449097, -0.23215392231941223, 0.48774152994155884, 0.4491613209247589, -0.1654888242483139, 0.4696463942527771, -0.8452000617980957, -0.46464982628822327, 1.2602509260177612, 0.08977442234754562, -1.0113664865493774, -0.19624902307987213, 1.2242591381072998, -1.6450380086898804, 0.1819959133863449, -0.41521623730659485, 1.2038377523422241, 0.41030266880989075, 0.6168861389160156, -1.933568000793457, 0.3665686249732971, 2.2623133659362793, -0.1997692734003067, -0.597727358341217, 0.861552894115448, 0.7546471357345581, 0.7780429124832153, -0.482928603887558, -1.303242802619934, 0.19529414176940918, 0.9415209293365479, 1.4189765453338623, 1.0388617515563965, 0.7199172377586365, -0.7927458882331848, 1.6310826539993286, -1.024019479751587, -1.3685036897659302, -0.3602006435394287, 0.19485360383987427, 1.3083643913269043, -0.7129215598106384, 0.15562957525253296, -0.4453023672103882, 0.7791202664375305, 1.6095367670059204, -2.036372423171997, -2.432050943374634, -1.4911539554595947, -0.5495587587356567, 0.8772700428962708, 1.2985888719558716, 0.17778658866882324, -0.3398323655128479, -0.02366141974925995, 0.321403443813324, 1.4082585573196411, -0.745614767074585, -0.9844930768013, 0.17517851293087006, -1.463853359222412, 0.1826964020729065, -1.2084039449691772, 0.1728297621011734, -1.4372566938400269, 0.009844760410487652, -1.397504448890686, 0.4449199140071869, 1.249247431755066, 1.1788161993026733, 0.597776472568512, 0.766861081123352, -0.48425793647766113, -0.19195781648159027, -0.3451782763004303, 1.326417088508606, -1.6169384717941284, 0.6758003234863281, -2.0311977863311768, -0.42209580540657043, 0.2291429489850998, -1.6136746406555176, 0.5776764750480652, 0.09513664245605469, 1.06655752658844, -1.3749366998672485, 0.3391999900341034, 0.49062031507492065, -0.22259104251861572, 1.4640886783599854, -0.47581925988197327, 0.36511555314064026, 0.32865673303604126, -0.22749480605125427, 0.7657067179679871, 0.7638888359069824, 1.6360433101654053, -0.7255516052246094, 1.2935185432434082, -1.5383996963500977, 1.1979224681854248, 0.23085911571979523, -0.47072312235832214, 1.6117578744888306, -1.9253629446029663, -1.4215573072433472, -1.2202805280685425, -0.6893492341041565, 1.0920109748840332, 0.08623164147138596, 0.13069698214530945, -0.17334763705730438, 0.890819787979126, 0.8689828515052795, -0.4740370512008667, -1.461550235748291, 0.10321071743965149, -0.631530225276947, 0.5016710758209229, -2.526174783706665, 3.3965301513671875, 0.8800457119941711, -0.9920720458030701, 0.837634265422821, 0.6199265718460083, 0.9687615036964417, -0.5045877695083618, 0.802547812461853, 0.7251496315002441, 0.792811930179596, 0.2965518832206726, -0.5013545155525208, -0.5022164583206177, 0.4578065872192383, -1.9606995582580566, 0.5862689018249512, 0.9878870248794556, 0.1786964386701584, -0.24758677184581757, 0.005963880103081465, -0.16562193632125854, -1.1423062086105347, 1.5230381488800049, -1.1541639566421509, 0.887246310710907, 1.0610918998718262, 1.3912677764892578, 0.48924073576927185, 0.896713137626648, -0.830518901348114, -0.43140673637390137, 0.674490213394165, 1.1461442708969116, 0.37494218349456787, -0.6894994974136353, -1.819586992263794, -2.379936695098877, -1.3114439249038696, -1.373439908027649, -0.7565420269966125, -0.49824994802474976, 0.9182319641113281, 0.8012149333953857, -1.1343623399734497, -0.07860822230577469, 0.5453539490699768, 0.7396347522735596, -0.2042112648487091, -0.7707995176315308, -0.5611905455589294, -0.6335973739624023, 0.7942102551460266, 0.9185999631881714, 1.204306721687317, -1.8166645765304565, 0.017757456749677658, 0.12642137706279755, -0.10618511587381363, 1.6769179105758667, -0.14702972769737244, 0.27912643551826477, -2.2295544147491455, -1.382353663444519, -0.05943247303366661, -0.16445119678974152, -0.2762252688407898, 2.2301218509674072, -0.2195184826850891, -0.9105164408683777, 2.0740489959716797, 0.2957167327404022, 0.6909732818603516, -1.3672831058502197, 0.3149327039718628, 1.9258430004119873, -0.26362311840057373, 0.2424638718366623, 2.047828435897827, -0.3118956983089447, 0.30161625146865845, 0.19922098517417908, 1.2702701091766357, 0.5227877497673035, -0.9306088089942932, -1.9981937408447266, 0.8854534029960632, 0.910472571849823, -0.6821193099021912, -0.9919857382774353, -0.11575035750865936, -0.37572312355041504, 0.8824527859687805, 1.0596812963485718, 0.4087623059749603, -2.7603180408477783, -0.43919989466667175, -0.4686482846736908, 0.10025665909051895, -0.26332640647888184, 0.36215996742248535, 1.245416522026062, -1.5135397911071777, -0.6490954160690308, -1.2517753839492798, 2.0454883575439453, 0.14640536904335022, -1.4861185550689697, 0.8993446826934814, -0.5853855013847351, -0.31943371891975403, 0.32726189494132996, 0.2211606651544571, -0.43130287528038025, -1.0453131198883057, -1.159127950668335, -0.28988656401634216, -1.000930905342102, -0.24678455293178558, -0.36771461367607117, -0.46672406792640686, 2.6885483264923096, 2.5626819133758545, -0.6680116653442383, -0.9579342007637024, -0.11984851956367493, -0.19604036211967468, -1.300268292427063, -0.011946695856750011, 0.27988865971565247, -1.7229057550430298, -0.5613290667533875, -0.8637412190437317, -0.3644033670425415, 0.4429355263710022, -1.2333711385726929, 0.20693212747573853, 0.41814345121383667, 0.2958453297615051, -0.15573091804981232, -1.9432893991470337, -0.9340033531188965, 1.3525224924087524, -0.2722283899784088, 0.372963547706604, -0.19653292000293732, -0.8531057238578796, 0.07233487814664841, 0.9152824878692627, 0.533790647983551, 1.0708988904953003, -0.058033768087625504, 1.0526045560836792, -0.12926903367042542, -1.5935471057891846, 0.9381253123283386, 1.090598702430725, -0.621024489402771, -0.277519553899765, 1.3858779668807983, -2.1250271797180176, -2.4470267295837402, 0.9638696908950806, -0.11329640448093414, 1.4582936763763428, -0.8042996525764465, 0.2576451599597931, 1.0866140127182007, -0.806807279586792, 1.6933742761611938, -0.15029025077819824, 0.13329072296619415, -0.46754661202430725, -1.4521961212158203, -0.001349473837763071, 0.6168315410614014, 1.3895272016525269, -0.12221751362085342, -1.9849953651428223, -0.5381397604942322, -0.5648890137672424, 0.5919291377067566, -0.9242083430290222, -0.6545810103416443, 0.6707070469856262, -0.41341230273246765, 2.2228963375091553, -1.095531940460205, 0.435616672039032, 1.91898775100708, 1.1443687677383423, 2.279024600982666, 0.22634312510490417, 0.44159284234046936, -0.986277163028717, 0.35628628730773926, -0.46022099256515503, -1.1286886930465698, 0.050029490143060684, 0.5594053268432617, 0.3735450506210327, 0.813115119934082, 0.5656477808952332, -0.3715182840824127, -0.8505326509475708, 1.3503276109695435, 0.4455454349517822, 0.8869274258613586, -0.4997398257255554, 0.13380318880081177, -0.29195931553840637, 0.942565381526947, 0.4844350516796112, 1.778359293937683, 2.3514907360076904, -0.5414086580276489, -0.33047279715538025, 0.18326912820339203, -0.01702250726521015, -0.11207140982151031, -0.7336077690124512, 0.19990037381649017, 0.14958681166172028, -0.4884015917778015, 0.9184457063674927, 0.9508568644523621, -0.6072126626968384, 0.45708829164505005, 1.8843669891357422, 0.47174072265625, 1.7987059354782104, 0.5981513857841492, -0.8646901249885559, -0.6761137843132019, -0.1786811351776123, 1.2285865545272827, 1.8139840364456177, -0.28036248683929443, 0.5564520359039307, -0.45302584767341614, 0.4084866940975189, -0.7394722700119019, 0.1390790045261383, -0.4397425353527069, -0.34128671884536743, -2.993285655975342, -0.05164158344268799, -0.661136269569397, -0.6598916053771973, 1.6626125574111938, -1.0879764556884766, 0.9864208102226257, 1.3521373271942139, 0.7669036984443665, 0.46581438183784485, -1.2432775497436523, -0.037547558546066284, -1.673765778541565, 0.8626757264137268, 0.11211777478456497, 0.13798610866069794, -0.5666264295578003, -1.80181086063385, -0.09795812517404556, -1.104562759399414, -1.3849563598632812, -0.4016023576259613, 0.011222940869629383, 0.7166677713394165, 0.46266505122184753, -0.1280568689107895, -1.5458730459213257, -1.052505373954773, 0.8015726804733276, 1.1555064916610718, -0.2668163776397705, 0.7394418120384216, -0.006650315131992102, 1.3912650346755981, 1.203032374382019, -1.4550877809524536, -1.5235447883605957, 0.5504269599914551, -1.5827747583389282, -1.4650765657424927, 0.4204947352409363, -0.9894441962242126, 1.5438693761825562, 0.45457005500793457, -0.12017089873552322, 1.656654715538025, 1.0339537858963013, 1.8162665367126465, -0.46181702613830566, 0.2644187808036804, 0.6394661664962769, -0.5557448267936707, -0.13730588555335999, 1.1692173480987549, 0.33894744515419006, -0.008919326588511467, -0.8644681572914124, -0.9029281735420227, 0.5858450531959534, -1.7477405071258545, 1.025538444519043, -0.9970014691352844, -0.3048606812953949, 0.2968321740627289, -0.5722038149833679, 1.8620117902755737, -1.6144874095916748, 0.3458499014377594, 0.6018120050430298, -0.9828548431396484, 1.073371171951294, -0.998770534992218, -0.004296376835554838, -0.7279154062271118, 0.6212508082389832, 0.5346977114677429, 0.2388744056224823, 0.47389882802963257, 1.1641336679458618, 0.0475032776594162, 1.6236504316329956, -1.4937094449996948, -0.7451900243759155, 0.47157371044158936, 0.4813767671585083, -1.5872957706451416, -0.038041189312934875, 1.7247141599655151, -0.542227029800415, 0.7645301222801208, -0.4800277352333069, -2.2281606197357178, 1.564855694770813, 0.25335580110549927, -0.6542780995368958, -0.705963134765625, 1.9990769624710083, -1.0638742446899414, 1.0915602445602417, 0.633966326713562, 0.732616126537323, 0.08509817719459534, -1.36408269405365, 0.07431071996688843, -0.7318512797355652, -1.2561432123184204, -2.509303331375122, -1.1144955158233643, -0.8339661359786987, 0.24072696268558502, -0.08663659542798996, 1.2704356908798218, -0.4886958599090576, 0.28898853063583374, -1.4370858669281006, 1.0927014350891113, 0.14372314512729645, 2.169050455093384, 0.28846418857574463, 1.036713719367981, 0.3313792049884796, 0.21219028532505035, 0.6347483396530151, -0.5910259485244751, 1.2247676849365234, 0.4015022814273834, 0.4546988010406494, -0.5990275740623474, 0.11024098843336105, 1.618565559387207, 0.26341038942337036, -0.38646426796913147, 0.23933929204940796, -1.4206757545471191, 0.37375468015670776, 0.9868077635765076, -1.2298693656921387, 0.09596844762563705, 1.109728455543518, -0.35893216729164124, 0.30691710114479065, 0.4368842542171478, 0.30225372314453125, 0.3443496823310852, 0.06536220014095306, 1.4701281785964966, -0.2703806161880493, 0.4418087601661682, 0.839282214641571, 0.5567528009414673, 1.5799384117126465, 0.934655487537384, -0.5890548229217529, -1.7793062925338745, -1.6802815198898315, -0.29381921887397766, 0.43426769971847534, 0.6879226565361023, 0.2675686776638031, -0.34726300835609436, -0.36227285861968994, 0.1454015076160431, -0.15172211825847626, 1.6050776243209839, 0.22036410868167877, -0.045469868928194046, -0.8522756099700928, -0.36636364459991455, -0.07846097648143768, 1.6400004625320435, 0.21283093094825745, 0.8492310047149658, -0.5832085609436035, -0.4596349596977234, -0.9122676253318787, 2.0799720287323, -0.9973311424255371, 0.9979372024536133, -0.34898364543914795, 0.6823638081550598, 0.6661408543586731, -2.21506667137146, -1.224604845046997, 1.0099347829818726, -1.8721110820770264, -0.8413603901863098, 2.0971555709838867, 2.1761715412139893, 0.29566967487335205, 0.6855831742286682, -0.5479993224143982, 0.22666189074516296, 0.04300495609641075, 1.1824458837509155, 0.42201098799705505, -0.7145202159881592, -0.13214150071144104, -0.4802742004394531, 0.7847350239753723, -0.6890767216682434, 0.2750292420387268, -1.4875999689102173, -1.2263044118881226, -0.03180227056145668, 0.33683493733406067, 0.32250288128852844, 2.147263526916504, -0.7603656649589539, 0.26827695965766907, 0.4556995928287506, -0.5120667815208435, -1.123284935951233, 0.742587149143219, 1.8569021224975586, -0.08350182324647903, -0.8987504243850708, 0.8363010883331299, -0.732861340045929, 0.19755835831165314, 0.34163570404052734, -0.5485129356384277, -0.004761618562042713, 1.5645195245742798, -0.09341181069612503, 1.8465598821640015, 0.8229213356971741, 0.352735310792923, 0.8171403408050537, 0.027893085032701492, 0.797110378742218, 0.12714168429374695, -0.035805441439151764, 0.6311870217323303, 1.8033568859100342, -0.7512949705123901, -0.4244399964809418, 0.9159256815910339, 0.18618111312389374, -0.5239577293395996, -0.011070826090872288, 1.0794838666915894, -2.155001163482666, -0.8063563108444214, -0.2524002492427826, -0.2060578465461731, -1.700594186782837, -0.7287719249725342, 0.25422579050064087, 0.53482586145401, -1.2336640357971191, -0.727170467376709, 0.39722299575805664, -0.27704769372940063, -0.6223806142807007, -1.2927247285842896, -1.5163178443908691, 0.33757907152175903, -0.8297910690307617, -0.5316411852836609, 0.9036241769790649, -0.4696437418460846, -0.2617957890033722, 1.1758471727371216, -1.456295132637024, 0.7158010601997375, -0.7643557786941528, -0.11304095387458801, -0.3423548936843872, -0.3448251485824585, 0.13375535607337952, 0.49293598532676697, 0.9973582029342651, 0.23330578207969666, -0.0052282256074249744, -0.7711840867996216, 2.3664286136627197, 0.048538535833358765, -1.1856074333190918, -2.5116872787475586, 1.0873463153839111, -0.9342881441116333, -0.45472943782806396, -2.1989662647247314, -1.8145794868469238, -0.0487021766602993, -0.45456525683403015, -1.842651128768921, -0.36996787786483765, 0.47530874609947205, 0.040292806923389435, 0.05678175389766693, 0.906469464302063, 2.9445343017578125, -0.4821600914001465, 1.7219583988189697, 0.7631604671478271, -1.3641464710235596, -1.7053396701812744, -0.27656206488609314, -0.018012627959251404, -0.41588908433914185, 0.6082246899604797, 0.13771162927150726, 0.8303978443145752, 0.18156933784484863, 2.3766846656799316, 0.8972150087356567, 1.2599934339523315, -0.6853657960891724, 1.513370156288147, -0.05126618221402168, 0.07272116839885712, -0.2483731210231781, -0.4918951690196991, 1.1216007471084595, 1.8767266273498535, -0.215352401137352, 0.2258257120847702, 1.1975758075714111, -0.5161462426185608, 0.675403892993927, 0.024801725521683693, 1.3863836526870728, 0.2434971034526825, -0.6461123824119568, -0.08348861336708069, 1.018789291381836, -0.30645355582237244, 0.2535229027271271, 1.1228340864181519, 0.6012938022613525, 0.1608998030424118, 0.03027922287583351, -1.0535048246383667, -0.4022071659564972, 1.0140398740768433, 1.9571774005889893, 0.18244658410549164, 0.01491721160709858, -0.46149617433547974, -0.34801363945007324, 0.284678190946579, 0.39442935585975647, -0.31741535663604736, -1.308922290802002, 0.6139972805976868, -0.8957961797714233, 0.6214712858200073, 0.15173089504241943, 0.8049147725105286, -0.3644113540649414, 1.8389201164245605, -1.3606610298156738, 0.49257951974868774, -2.2826642990112305, -0.3874146640300751, 0.1255122423171997, -1.1059843301773071, 0.8770633339881897, -0.7037184834480286, 0.6472465395927429, -1.9236764907836914, -0.5112924575805664, -0.20932720601558685, 0.020538512617349625, -0.44564640522003174, 1.5388933420181274, 1.115463137626648, 0.5156370401382446, 0.6487607955932617, 0.8377689719200134, 0.5154113173484802, -1.0573654174804688, 0.18533164262771606, -0.18574045598506927, 0.7654458284378052, -0.30366793274879456, 1.4289637804031372, 0.8465811610221863, -0.45875871181488037, -0.5519111156463623, -0.6167958378791809, 1.0350606441497803, -0.12423975765705109, -0.26067250967025757, 1.2152447700500488, -0.46816307306289673, 0.29686370491981506, 0.5159953236579895, 0.7580190896987915, -2.074397087097168, 0.15223680436611176, 0.7658923864364624, -0.519377589225769, -1.0507869720458984, 0.04789748042821884, -0.4775252640247345, -0.05727829784154892, -0.5088071227073669, -0.5029066205024719, 0.24208563566207886, -2.3889734745025635, 1.1238144636154175, -0.5147699117660522, 0.22148625552654266, -0.8553502559661865, 0.7131259441375732, 0.30637821555137634, 0.23624716699123383, 2.0519206523895264, 0.10277776420116425, -0.2775801718235016, -2.3694398403167725, -0.14932455122470856, 0.5193127393722534, -0.5040392279624939, 0.7397110462188721, -0.7377147078514099, -0.7149112224578857, 1.2600177526474, -0.13697241246700287, 1.6579947471618652, 0.44521838426589966, -0.15496498346328735, 0.28556883335113525, -1.3189266920089722, -0.658460259437561, 0.7247448563575745, -0.602219820022583, -0.662958025932312, -0.0017903733532875776, -0.6129655838012695, -0.0028788154013454914, -0.6483567357063293, -0.08402825891971588, 0.3864511549472809, 0.5181079506874084, -0.4296548366546631, -1.6321806907653809, 0.10351040214300156, -1.6391221284866333, 0.736592173576355, -0.6630593538284302, -0.7501785159111023, 0.6209279894828796, -1.5670864582061768, -0.9425997734069824, -1.4866524934768677, -0.10559359192848206, 0.020598575472831726, 1.5282796621322632, -0.3332802355289459, 2.4898364543914795, 0.8931724429130554, -2.530949354171753, -0.11757326126098633, -0.5815740823745728, -0.10289653390645981, 0.31143757700920105, -0.8107699155807495, -0.9463481307029724, -0.836891770362854, -1.9399964809417725, 0.7359875440597534, -0.8021854758262634, -0.9354009032249451, 0.7820250988006592, 0.7877928018569946, -2.15714955329895, 0.6510458588600159, -1.2324477434158325, -0.014474504627287388, 0.5003732442855835, -0.9861807227134705, -2.5584921836853027, -0.10840608179569244, -0.3185281753540039, 0.2912658751010895, 1.2285237312316895, 1.4840338230133057, -1.4376193284988403, -0.5689026713371277, -1.155848741531372, 1.493295669555664, 0.5843324661254883, 0.8792295455932617, -1.6356276273727417, 0.2021581381559372, 1.2925286293029785, -0.7941597700119019, -0.39983993768692017, -1.2536641359329224, 1.6129467487335205, 0.21602243185043335, -1.006224274635315, -1.0093367099761963, -0.7648177146911621, -0.04632950201630592, -0.2910858988761902, 1.3120039701461792, -0.1979549527168274, 0.13905802369117737, 1.0588908195495605, -0.8877323269844055, 1.097905158996582, -1.451515793800354, 1.409050703048706, 0.02540811523795128, 2.2018096446990967, 0.5798464417457581, -1.1491299867630005, 0.3345406651496887, 0.6971997022628784, -0.9880369305610657, 0.7370260953903198, 0.9416533708572388, 1.305216908454895, -0.6213696002960205, 1.0700113773345947, -0.9522796273231506, 0.946902871131897, 0.2089623361825943, -0.25030118227005005, -1.253735899925232, 2.1400656700134277, -2.259258508682251, -0.9725483655929565, -0.30026721954345703, 1.0755219459533691, 1.4238555431365967, -0.3439085781574249, -0.25566214323043823, 0.3615167737007141, -0.8680742979049683, -2.122562885284424, -1.18621027469635, 1.446866750717163, -0.43130189180374146, -0.8699440956115723, -1.076646327972412, -0.08902090042829514, -0.7418789267539978, 1.3446550369262695, -2.7335360050201416, 1.0701764822006226, 0.6484466195106506, -0.488498330116272, -0.8944453001022339, -0.7961670160293579, 2.0018773078918457, 0.8540766835212708, -0.6154617667198181, 0.25225016474723816, 0.2157965749502182, -0.38134515285491943, -0.4276128113269806, 0.34936848282814026, 1.225010633468628, 0.9724259376525879, 1.1301733255386353, -0.28578612208366394, 1.4073381423950195, 0.2138146162033081, 0.48753780126571655, 0.5334545969963074, -0.08722022920846939, -0.70769202709198, -0.6271964907646179, -0.7870214581489563, -0.638127326965332, 2.2246198654174805, 0.16610084474086761, 0.4164329767227173, 0.6920196413993835, -0.38495951890945435, 0.05826980620622635, 0.10269453376531601, -0.1991233229637146, -0.9767840504646301, -1.3602197170257568, 0.17874807119369507, 2.269033908843994, -1.1524525880813599, -1.3642172813415527, -0.09276551008224487, -1.0761228799819946, 0.478342741727829, -0.10052646696567535, -0.8701459765434265, 0.5117788314819336, -0.48007890582084656, 0.41271165013313293, 0.02716677635908127, -0.5943264961242676, -2.0824007987976074, -0.31376147270202637, -0.14388366043567657, 1.2687854766845703, 0.07455378025770187, 0.05949065461754799, -0.41472411155700684, -1.0396020412445068, -0.11016788333654404, 0.8027214407920837, -0.28466856479644775, 0.7575100064277649, -0.6256557106971741, -0.7965415120124817, 0.8025095462799072, -2.0045132637023926, -1.611342191696167, 2.699869155883789, -0.2819569408893585, -0.5223647952079773, 1.3456439971923828, 0.3975849449634552, 0.3441285789012909, -1.5943260192871094, -0.38185155391693115, -1.8044368028640747, -1.2063473463058472, 0.5035054683685303, 1.3856927156448364, -0.4642372131347656, 0.785713255405426, -1.0416885614395142, -0.5650287866592407, 0.055917270481586456, 0.652106761932373, 0.9172443747520447, 1.2331390380859375, 0.9962834119796753, 2.2101285457611084, 0.16705942153930664, -1.2056245803833008, 1.4825454950332642, 0.7742383480072021, 0.6622173190116882, -0.7137402892112732, -0.0981065183877945, -0.2766776382923126, -1.8884357213974, -0.039154380559921265, -0.09162639081478119, -0.6163346171379089, -1.0066380500793457, 1.4819144010543823, -0.12212230265140533, 0.18943654000759125, -0.9295216798782349, -0.5355015993118286, 0.14338178932666779, 1.0415321588516235, 1.235559105873108, 0.6239665150642395, 0.6101263165473938, -1.74593985080719, 0.4902552664279938, 0.4388701021671295, 1.7791742086410522, -1.650238275527954, -0.47209712862968445, 0.9129422307014465, -1.712043285369873, -1.7214950323104858, -0.6630641222000122, -0.20933403074741364, 0.17778274416923523, 0.21683402359485626, 1.2172507047653198, 0.22538532316684723, 0.3059051036834717, 1.61867094039917, -0.7399778962135315, -0.7043738961219788, -1.3927600383758545, 0.25895601511001587, -0.3256711959838867, -0.25616714358329773, -0.010554876178503036, -0.8807501792907715, 0.7992443442344666, -1.2916268110275269, 0.792163074016571, -0.6556578278541565, 1.1123368740081787, -0.9571706652641296, 0.6836025714874268, 0.9474557042121887, -0.19152942299842834, 0.14604578912258148, 0.8564781546592712, 1.041009545326233, -0.43356046080589294, 0.4427105784416199, -0.7511727213859558, -0.1645902544260025, -0.6747676134109497, 0.2627640664577484, -1.3061103820800781, 1.1865967512130737, 0.8396040797233582, -0.6470631957054138, 1.0679864883422852, -0.6582261919975281, -0.05993286892771721, 0.25547948479652405, -0.07666706293821335, -1.5139843225479126, 0.07975304871797562, 1.253289818763733, -1.632860541343689, -0.7369946241378784, -0.3608185052871704, 0.8539577722549438, -1.6727434396743774, 1.8645364046096802, -0.10185866802930832, 1.5062267780303955, -0.8826249837875366, 0.7506617903709412, -0.9691151976585388, -1.4763095378875732, 0.8068882822990417, 0.04302028566598892, -0.6576222777366638, 0.03976903483271599, -0.12379996478557587, -0.8121194839477539, -1.420686960220337, -0.7828558683395386, -0.153545081615448, 0.8364010453224182, -0.2826431393623352, -0.32468119263648987, 0.05313003808259964, 0.549770176410675, -1.964905858039856, 0.006074442062526941, 0.027827467769384384, 0.5707141757011414, 1.5012670755386353, -0.19762487709522247, -0.4916813373565674, -1.6808857917785645, 0.014144673012197018, 0.8915672898292542, 3.379439353942871, 1.7414249181747437, -0.18155702948570251, 1.0313059091567993, 0.20229928195476532, -0.11541416496038437, 0.21663528680801392, -1.573649287223816, -0.7588328719139099, 0.67374587059021, -0.21343965828418732, 0.3851795494556427, 0.7047044038772583, -0.18335109949111938, 0.3281693756580353, 0.5991933941841125, 2.4483249187469482, -0.9978570342063904, -1.527165174484253, 0.1882367879152298, 0.6379150748252869, 0.2708621919155121, -0.5333547592163086, 0.8343546986579895, 0.5650771260261536, 0.23018501698970795, -1.2873315811157227, 1.5052580833435059, 0.672601580619812, -1.0128878355026245, -0.10443062335252762, 2.3782241344451904, -0.8433634042739868, -0.31720709800720215, -0.417174756526947, -0.17907746136188507, 0.6036344170570374, -1.989307165145874, -0.6583099961280823, 0.869554877281189, -0.08614930510520935, -0.8782907128334045, -0.7380071878433228, -1.6456315517425537, -0.7452880144119263, -0.010330303572118282, -1.4673601388931274, -0.2560003995895386, -2.4137113094329834, 0.36446231603622437, -1.1470718383789062, 0.6856071352958679, 0.779076337814331, 1.1394046545028687, 0.9575566053390503, -0.3449642062187195, -1.3002433776855469, -0.8550531268119812, 0.7463597059249878, -0.26812657713890076, 0.21603505313396454, -0.18799695372581482, -1.5431673526763916, 0.2616099715232849, 1.3079313039779663, -0.26485031843185425, 0.794031023979187, 0.09322351217269897, 1.1708124876022339, -1.5966085195541382, -0.6957719326019287, 0.48933959007263184, -0.780993640422821, -0.18128612637519836, -0.7498509883880615, 0.2372562438249588, 1.0957934856414795, 0.5816178321838379, -0.3504882752895355, 1.2764718532562256, -0.8210002779960632, -0.2251296490430832, 1.6224281787872314, -0.032304588705301285, 0.19315947592258453, 0.09965142607688904, 0.30127573013305664, 0.2356749176979065, 0.16212376952171326, -0.606880784034729, 2.194120168685913, 0.5621458888053894, -0.2724168598651886, 1.177215337753296, 2.1822726726531982, 0.6965150237083435, -0.4334923028945923, -1.885002851486206, 1.0962598323822021, -0.8655062317848206, 0.9484094977378845, -1.9338455200195312, 0.1902954876422882, -0.1656722128391266, 0.07347742468118668, 0.23630189895629883, 1.385434865951538, 0.4984746277332306, 1.0027183294296265, -0.4957070052623749, 0.03869328647851944, 0.8383294939994812, -0.013404765166342258, -0.6535835266113281, 0.9764626622200012, -0.8802326917648315, 0.5614188313484192, 0.26359423995018005, -0.5683321356773376, -1.769974946975708, 1.0691698789596558, -0.7924550771713257, -1.0851876735687256, 0.27614638209342957, -0.6833761930465698, -0.9977447390556335, -0.8013976812362671, -0.14051681756973267, -2.062138557434082, 0.4047596752643585, -0.5339661240577698, 2.276944398880005, -0.5110966563224792, -0.35047462582588196, 0.20537976920604706, 0.5462912321090698, 0.8461177945137024, -0.05476876348257065, -0.09380125999450684, -0.4223373830318451, -0.1403016448020935, -0.8208048343658447, 0.06611768901348114, -0.6547694802284241, -0.6718711853027344, 0.45359888672828674, -0.6263741850852966, 0.19000478088855743, 1.0489248037338257, 0.02353947050869465, -0.472735196352005, -1.1846004724502563, -0.7507370114326477, 0.4454098045825958, 0.014949188567698002, 0.3226848840713501, -0.30086731910705566, 2.473712205886841, -0.861501157283783, 0.8221983313560486, 1.7559912204742432, -1.0251646041870117, -1.1759884357452393, -0.41057851910591125, -0.5543307662010193, -0.48309195041656494, -1.2916535139083862, 0.11377933621406555, 0.8983976244926453, -0.4703229069709778, 0.7910788059234619, -1.6544861793518066, -0.6604542136192322, 0.11762940883636475, 0.7603470087051392, -1.13713800907135, -1.43743097782135, -0.1477336883544922, -0.8590441942214966, 1.1464669704437256, 0.6011724472045898, -0.4455704987049103, -0.5668076872825623, -0.15321743488311768, 1.4635673761367798, -1.1039952039718628, -0.5298013091087341, -1.2791876792907715, 2.306480884552002, -1.7408185005187988, -1.6426407098770142, -0.12031055986881256, -0.47062626481056213, -0.6896587610244751, -1.2000417709350586, -0.3297291398048401, 0.1186763271689415, -1.6284971237182617, -1.0198194980621338, 0.22538676857948303, 0.3929479420185089, 1.8302481174468994, -0.3749050498008728, 0.5846941471099854, -0.3210483491420746, -0.5732004046440125, -0.8008492588996887, 0.15491966903209686, -0.7329323291778564, -0.5322358012199402, -0.01812448725104332, 1.6703920364379883, 2.003607988357544, -0.7183907628059387, -0.6644374132156372, 0.49805885553359985, -1.9542639255523682, 0.160128653049469, -0.06272198259830475, -0.6571709513664246, 0.16690592467784882, 0.5004242658615112, -0.5167050361633301, -1.4464678764343262, 0.6806654334068298, -0.35051897168159485, 0.43541643023490906, -1.732351541519165, 0.6052708029747009, -0.10310737788677216, 0.37313079833984375, -1.2561144828796387, 0.1804536134004593, 1.728664755821228, -0.8728537559509277, -0.26018449664115906, 2.28539776802063, -0.8300765752792358, 0.9918261766433716, -0.1308932602405548, 0.3220062255859375, -1.1655293703079224, 0.37102484703063965, 1.7578303813934326, 1.671553611755371, -1.8114995956420898, 0.3158329427242279, -0.6764000058174133, 1.1932100057601929, 0.2728116810321808, -0.38288670778274536, -0.02992948330938816, -0.5303051471710205, 0.47950875759124756, -1.3080178499221802, 0.7075537443161011, 1.1146440505981445, 0.35564082860946655, -0.5068504810333252, 0.4537460505962372, -1.2750704288482666, 1.4395016431808472, 1.396386981010437, 0.3090458810329437, -1.2275145053863525, -0.8098980188369751, 0.27888986468315125, 1.7024810314178467, -1.4932801723480225, -0.48362255096435547, -1.327592134475708, 0.345513254404068, 1.0253995656967163, 0.07451505959033966, 1.1629332304000854, -0.16489659249782562, -1.694322109222412, 0.3491719961166382, -0.7653698921203613, 0.9591158032417297, -2.548076629638672, -0.8810510635375977, 0.21469368040561676, -0.5615262389183044, -0.8680118918418884, -1.3568007946014404, 1.3047720193862915, -0.8690911531448364]], "dtype": "float32"}], "e": {"__nd__": [[-0.30373573303222656, -0.103767991065979, 0.4398961067199707, 1.2954745292663574, -0.8277297616004944, -0.12143445014953613, -2.246039867401123, -0.7268847227096558, 0.996307373046875, 1.3972458839416504, -0.6841690540313721, -0.4228169918060303, 1.4362106323242188, 0.5821990966796875, -0.5519582033157349, -0.06333911418914795, 1.871173620223999, 0.20826983451843262, 0.4483511745929718, 1.3768815994262695, 0.46364420652389526, -1.8185338973999023, 0.9658539295196533, -0.13793599605560303, 0.17838513851165771, 1.785745620727539, 1.5737597942352295, 0.5644521713256836, -1.6057605743408203, -1.8430267572402954, -1.0548095703125, 0.9008935689926147, 2.088393449783325, -0.23128600418567657, 1.9489495754241943, -0.3569455146789551, -1.5042879581451416, -0.7813795208930969, 0.7831371426582336, -0.7023804783821106, -1.5519052743911743, -0.3516019582748413, -1.9829484224319458, 0.6518871188163757, -2.9195475578308105, -0.8634788393974304, 0.47809499502182007, 2.205350399017334, -0.47756725549697876, 1.0611417293548584, -1.5655677318572998, -4.297382831573486, 0.05045771598815918, -0.4568251967430115, 0.6119695901870728, 4.220825672149658, 1.3697633743286133, -1.9643019437789917, 1.7915316820144653, 0.010599851608276367, -1.2596999406814575, -0.7710822820663452, -3.3454699516296387, 0.1999388337135315, 2.19393253326416, -0.9173418283462524, -2.0635781288146973, 0.351232647895813, -1.7070232629776, 2.374019145965576, 0.942575216293335, -0.6198909282684326, 0.9664387106895447, -0.2946854829788208, 0.8592843413352966, -2.188559055328369, 0.92503422498703, -0.1435796022415161, 1.7979984283447266, 0.0017521381378173828, 1.1976499557495117, 0.6959912180900574, 2.587225914001465, -1.850088119506836, 0.37094196677207947, -0.4781469404697418, 0.7409504055976868, 1.2372040748596191, -0.27672314643859863, 0.5523388385772705, 0.7554616332054138, 1.0111547708511353, 2.2096219062805176, -0.5971125364303589, 0.39438262581825256, 0.4225330948829651, 2.1052439212799072, 0.9155697226524353, -0.06793573498725891, -0.33783549070358276, -1.6440114974975586, 1.040637493133545, 0.7131075859069824, -2.440647840499878, -0.3186991214752197, -0.9344656467437744, -1.9229365587234497, -0.9217156171798706, -0.6663204431533813, 0.23873722553253174, 1.5325028896331787, 1.9622430801391602, -2.4691171646118164, 1.8011542558670044, 0.5193631649017334, 1.4609990119934082, -0.7274245023727417, -0.27430224418640137, 1.6063631772994995, -0.8115782737731934, 1.4764904975891113, -1.5769641399383545, 1.226036548614502, -0.2888540029525757, 0.3739706575870514, 1.1983869075775146, 0.42550963163375854, 2.338113784790039, 1.5322519540786743, -0.9367376565933228, -0.5608295202255249, 1.895033359527588, -1.835587978363037, 0.9921458959579468, -3.127594232559204, -0.1679372787475586, 2.9606852531433105, -3.037860870361328, 1.629244327545166, -2.330043315887451, 0.36522436141967773, 2.1386947631835938, -0.6143411993980408, -3.324784517288208, -0.0559161901473999, 1.3793827295303345, -0.8512611985206604, -0.4494936466217041, 1.5401535034179688, 2.471142292022705, 1.5249346494674683, 1.3900189399719238, -0.08703470230102539, 1.9267444610595703, -0.13720351457595825, -0.7207071185112, 0.007243633270263672, 0.48998236656188965, 0.04135013371706009, 1.3739733695983887, 0.20412322878837585, -1.2146577835083008, 1.1764898300170898, -1.1889138221740723, 3.879718542098999, 0.36874932050704956, -0.9876099824905396, 0.4290940761566162, -1.4806559085845947, 0.172919362783432, 0.6157252788543701, -1.9588603973388672, 0.9506197571754456, 1.4183473587036133, -0.24487100541591644, -1.7435771226882935, -0.32918334007263184, 2.2826151847839355, 1.7479785680770874, -0.42290282249450684, -0.23110002279281616, -1.5569695234298706, -0.8618000149726868, 0.4318099021911621, -0.8423667550086975, 1.744434118270874, 0.924163818359375, 0.4504088759422302, 0.7377575635910034, -1.0331958532333374, 0.5938520431518555, -0.31716683506965637, -0.1964735984802246, -0.8883730173110962, -2.1575253009796143, 3.0237512588500977, 3.2900819778442383, -4.489968299865723, 0.1710747331380844, 0.0253790020942688, -0.25252437591552734, 3.188150405883789, -1.3127081394195557, -1.9303936958312988, -1.5978946685791016, -1.2708690166473389, -1.8378009796142578, -1.219039797782898, 0.5344240069389343, -0.21220245957374573, -0.49166518449783325, -0.14023767411708832, 2.227318048477173, -0.28742071986198425, 0.9856707453727722, 1.6660186052322388, -0.04667691886425018, -2.1981284618377686, -0.7677163481712341, -1.2989964485168457, -3.515873432159424, 1.8839869499206543, -2.093738555908203, 1.5014419555664062, -0.9849908947944641, -0.5056651830673218, -1.3458023071289062, 0.7776306867599487, 1.162264347076416, 0.6120049953460693, -0.9280698299407959, -2.0938093662261963, -1.097190022468567, -1.145849585533142, 1.820261001586914, -0.8218133449554443, -0.1462026834487915, 3.022139549255371, 3.923281669616699, 0.5557635426521301, 0.37436649203300476, 1.0706636905670166, 1.1657233238220215, 0.027249231934547424, 1.570960283279419, 0.12647220492362976, -0.7518802285194397, 0.10772569477558136, -1.8365528583526611, 1.448433756828308, 0.6513610482215881, -0.3380083441734314, -0.2891201376914978, -0.7977430820465088, 0.006720460951328278, 0.35676658153533936, 0.9710147976875305, 0.044877588748931885, -0.7691553235054016, 0.2392989993095398, 0.3457695245742798, 0.9573103189468384, -0.43838050961494446, 0.5515723824501038, -0.01204758882522583, 0.5302689075469971, 0.659333348274231, -1.9921677112579346, 0.8584770560264587, 0.5532183051109314, -1.793980360031128, 0.10723745822906494, -4.298266887664795, 0.35819581151008606, -1.3258706331253052, -0.8013050556182861, -2.8522915840148926, 0.8766126036643982, -2.413632869720459, -0.47350937128067017, 1.0142661333084106, -3.143249034881592, 0.9803491830825806, -0.7193635702133179, 0.46098023653030396, 1.3682599067687988, -0.964926540851593, -0.6260430216789246, 1.5512138605117798, -0.7750056982040405, -1.1968728303909302, -0.25411227345466614, 1.1757575273513794, -0.5556947588920593, -0.44414985179901123, -0.9239869117736816, 1.1461358070373535, 0.013565897941589355, 0.11898413300514221, -0.2267947793006897, -1.3196823596954346, 0.9758785963058472, -0.9416384696960449, -3.438784599304199, -1.4269298315048218, 0.1148105338215828, -2.2549145221710205, 0.6775617003440857, -0.7127139568328857, 0.851813018321991, -0.3084569573402405, 0.22951363027095795, 0.02796095609664917, 1.1818898916244507, 0.005368828773498535, 0.2727320194244385, -0.2511324882507324, 1.834850549697876, -2.5628676414489746, 0.8490807414054871, -0.5871840715408325, -2.7110676765441895, 3.6972556114196777, 1.9311671257019043, -1.1481837034225464, -1.6346935033798218, -0.3935009241104126, -0.8669061660766602, -0.5506142973899841, -1.8017070293426514, -1.1956853866577148, 2.887510299682617, 2.5436365604400635, 2.697082996368408, -0.33908700942993164, -1.296972393989563, -1.063737392425537, -2.0546202659606934, -1.7744035720825195, -0.33419251441955566, 0.12270879745483398, 2.5206589698791504, 0.8451600670814514, 0.27858415246009827, 1.3236724138259888, -0.5013402700424194, -0.37527430057525635, -0.2433817982673645, -1.5113661289215088, 3.2381293773651123, -1.0693117380142212, -1.8568470478057861, -0.553585410118103, 0.4131055474281311, -1.7224886417388916, 3.291149139404297, 2.511680841445923, 2.6220319271087646, -2.4489901065826416, -0.7499372363090515, -2.3648695945739746, -1.0528010129928589, -0.16193392872810364, -0.7860227823257446, -1.2462525367736816, 0.8579963445663452, -0.41036611795425415, 0.03421962261199951, 1.5733890533447266, -1.6981174945831299, 0.15643799304962158, -0.3206629753112793, 0.18985193967819214, -0.2870936989784241, 0.13300880789756775, -1.5905039310455322, -0.23736119270324707, -0.37424585223197937, -1.014928936958313, 2.0924816131591797, -0.8749139308929443, -2.231358528137207, -0.3628438115119934, -1.950103998184204, 0.6070289015769958, 1.474687933921814, -0.6354773044586182, 0.8289364576339722, -0.6582438945770264, 0.3971633315086365, -0.10626304149627686, -0.10263717174530029, 0.4444333016872406, 1.6729345321655273, 0.20893235504627228, -2.1337356567382812, -1.4557393789291382, -0.4268566966056824, -1.1766568422317505, -0.5524877905845642, -1.3407924175262451, -1.3035635948181152, -2.601935863494873, -4.903779029846191, -0.07649567723274231, -1.950026035308838, 1.1513667106628418, 0.9902033805847168, -1.8108669519424438, 1.9193215370178223, 0.10723799467086792, 1.6014925241470337, 0.7300312519073486, 0.587125301361084, 0.21467843651771545, 0.2826140224933624, 2.2798752784729004, 0.9731072783470154, 0.0046995580196380615, 1.490630030632019, -1.9702000617980957, 2.4514718055725098, -1.916062831878662, 0.4307555556297302, 1.5355082750320435, 0.25276869535446167, -0.13598692417144775, 3.2768263816833496, 0.7018688917160034, -1.4661102294921875, 0.9086465239524841, -0.5117737054824829, -0.5129104852676392, 1.9537909030914307, -0.3287813663482666, -0.021992415189743042, 2.350329637527466, -1.8858895301818848, 1.4616894721984863, -1.0260244607925415, -1.0694494247436523, 0.6704807877540588, 1.4236259460449219, 2.0195767879486084, 0.7936427593231201, 0.4008758068084717, 0.7951701879501343, 2.156672477722168, 1.3695136308670044, -1.7581861019134521, 0.4489481449127197, 0.12035167217254639, 0.6022641658782959, 0.6329338550567627, -0.2563432455062866, -0.8856175541877747, -0.3278818428516388, 0.5727577805519104, -1.5721969604492188, -0.9757509827613831, -0.3386060297489166, 0.6946171522140503, -2.2071962356567383, 1.903682827949524, 0.9326196312904358, 0.5362516641616821, 0.5089412927627563, -2.5259344577789307, 1.5002261400222778, -2.5779240131378174, -0.3393799364566803, 0.04191720485687256, -0.13236607611179352, -1.5731990337371826, -3.2392351627349854, 1.5066471099853516, 1.2779088020324707, -1.1811171770095825, -1.4927492141723633, 0.5083553791046143, 0.26701468229293823, -0.11713594198226929, 2.7610857486724854, 1.6543047428131104, 1.8818714618682861, -1.1560542583465576, -0.6817784309387207, 1.8333743810653687, 0.3252909183502197, 0.3333110511302948, 0.38235944509506226, 2.228822708129883, 1.6850415468215942, -1.3865262269973755, -2.574859619140625, -1.8719429969787598, -2.114682912826538, -1.4801263809204102, 1.3210562467575073, -0.6675333976745605, 0.5985801219940186, -2.3140945434570312, 1.4942004680633545, -0.8631271719932556, -0.8013529777526855, -0.32763752341270447, 1.352894902229309, -0.5114171504974365, 0.7858190536499023, 1.151257038116455, 0.8443118333816528, -2.242169141769409, 0.6791321039199829, -0.037759557366371155, -1.6258971691131592, -0.40761327743530273, 0.37612736225128174, 0.1232987642288208, -0.6186615228652954, 0.2568877339363098, 0.20782291889190674, 2.2872514724731445, 0.028499692678451538, 1.764415979385376, -1.1513478755950928, -2.020677328109741, 1.8508198261260986, 2.1829938888549805, -1.4725637435913086, 1.2418808937072754, -0.7829179763793945, 2.7619102001190186, 0.17913222312927246, -0.7596297264099121, -0.2062007188796997, -0.4561047852039337, -0.5857095122337341, -0.3913726806640625, -0.12895618379116058, -1.1106834411621094, -1.7010962963104248, -0.6263947486877441, 0.23529592156410217, -1.1352547407150269, 0.09976533055305481, 2.231411933898926, -0.5658395886421204, -0.016727149486541748, -0.6604922413825989, 1.3673374652862549, 1.1978631019592285, 0.07869276404380798, 1.284712553024292, 2.751518726348877, -1.3017586469650269, -2.126352310180664, -0.6367985606193542, 1.246503233909607, -1.0379271507263184, -0.22591406106948853, 0.1558298021554947, 1.1257466077804565, -1.3472009897232056, 0.1988677978515625, -0.9457910060882568, -1.2789720296859741, -0.7810720205307007, 0.4433659315109253, 0.5571797490119934, -2.7231860160827637, 1.7873406410217285, 0.27778348326683044, -0.6888750195503235, 0.16999047994613647, 1.097548246383667, -1.3652613162994385, 0.7942538857460022, -0.6126499176025391, 0.4280056953430176, -1.0927687883377075, -1.2563731670379639, -1.4715001583099365, 1.1266745328903198, 0.850516676902771, -0.9660307168960571, -0.6457922458648682, -2.110532522201538, 2.1042635440826416, -0.3770645558834076, 1.2429050207138062, -1.886855959892273, -1.3617668151855469, 1.2970874309539795, -1.1402949094772339, 0.07177704572677612, -0.5771884918212891, -0.9671657085418701, -0.8300213813781738, 0.035871267318725586, -1.4452086687088013, -0.48475009202957153, 0.8736613988876343, 0.053590089082717896, 0.2678490877151489, -3.2016372680664062, 1.1252485513687134, -0.11343736946582794, 1.58669114112854, -1.2086836099624634, -0.4723585844039917, 2.1658830642700195, -0.7303950786590576, 1.7397592067718506, 0.6904608011245728, 0.8658417463302612, 1.3543779850006104, -0.9057579040527344, -0.1698102355003357, -0.4731665849685669, -2.096473217010498, -1.1607422828674316, 0.643006443977356, 1.5437579154968262, -0.6988148093223572, 0.8489977121353149, 0.31813138723373413, -0.429919958114624, -0.25138115882873535, -0.7996760606765747, -0.13883563876152039, -2.3697195053100586, -0.1428210735321045, -0.9994061589241028, 3.6262545585632324, -0.42865872383117676, -0.4721720516681671, -0.11332186311483383, 0.2160939872264862, 2.217133045196533, -1.3699533939361572, -1.1647861003875732, 0.19241976737976074, 0.6325912475585938, 0.7042950391769409, -0.0698494017124176, -2.9007136821746826, 0.38502365350723267, 1.1881420612335205, -2.7304935455322266, 1.4529954195022583, -0.47764045000076294, 0.7331830263137817, -2.0294394493103027, -1.6509439945220947, -1.647849678993225, -0.3841761350631714, -0.9673858880996704, 1.938186764717102, -0.2932077646255493, -0.9614987373352051, -0.048636168241500854, 1.1700141429901123, -1.976699709892273, -1.2058478593826294, -0.09700596332550049, -2.2237565517425537, -0.9676421284675598, 2.505431652069092, -1.585529088973999, 0.34821000695228577, -2.486945152282715, 2.298449754714966, 1.2586535215377808, -3.0897610187530518, -0.6973506212234497, 1.252741813659668, 1.2897107601165771, -1.9602601528167725, -0.00786086916923523, 0.6996074318885803, 0.8129733800888062, 1.3341374397277832, -0.0725862979888916, -2.125948429107666, -0.9620230793952942, -1.0708626508712769, -1.7897295951843262, -3.0193309783935547, 0.061048030853271484, 2.9216320514678955, -0.05121016502380371, 0.672280490398407, -0.2596592307090759, 1.1569079160690308, 0.031997185200452805, 2.3002357482910156, -1.6497694253921509, 0.4460753798484802, 1.249699354171753, 1.6204402446746826, 1.5381934642791748, -0.07926636934280396, -0.3559741675853729, 0.21887563169002533, 1.4832826852798462, 1.5967273712158203, -0.45541590452194214, 1.6772186756134033, -1.846396803855896, 0.9478289484977722, 0.2560851573944092, 0.1882944107055664, -2.6768434047698975, -0.5665696263313293, -2.468330144882202, 2.5233542919158936, -0.3252653479576111, 0.33067750930786133, 0.8491474390029907, -2.2644803524017334, 0.5253692865371704, -0.60146564245224, 0.701439619064331, 0.17352455854415894, 2.080636501312256, -0.2711326479911804, 0.06972461938858032, -0.6670995950698853, 1.4440641403198242, 1.9374018907546997, -0.8643168210983276, -1.9759743213653564, 1.7275886535644531, 0.5710514187812805, -1.145384669303894, 1.5371097326278687, 2.683605194091797, 0.9557079076766968, 0.8408984541893005, 0.7328683137893677, -0.3553433120250702, 1.472009539604187, -0.10194504261016846, 0.39861154556274414, 0.2364635318517685, -0.11478990316390991, -0.16838529706001282, 0.12894845008850098, -1.454393982887268, -0.31882596015930176, -2.009272813796997, -0.8137807846069336, -1.7706769704818726, -1.6142704486846924, -1.0456948280334473, 2.386359214782715, 1.863115668296814, 0.3825104236602783, 3.1494429111480713, 1.3850717544555664, 1.6025981903076172, -0.4039331078529358, 0.2540251314640045, 1.1142176389694214, -0.4552466869354248, -1.3638036251068115, -2.250664234161377, -1.6676610708236694, 1.622601866722107, -1.1234700679779053, -0.2998063564300537, 0.5131356716156006, 0.33402201533317566, -2.5088376998901367, -2.0341334342956543, -0.142085462808609, -0.871251106262207, -1.479445457458496, 1.4020819664001465, -0.80586177110672, 0.49051839113235474, 0.08205592632293701, 2.432255268096924, 2.2781710624694824, 0.2970714569091797, 0.33076000213623047, 1.144912600517273, 1.5153703689575195, -1.1508963108062744, -1.4451539516448975, 0.11302101612091064, -0.07166627049446106, 0.19285809993743896, -1.828658938407898, -2.0093796253204346, 0.4941442310810089, -1.0443153381347656, -2.940186023712158, -1.9416100978851318, -2.3043603897094727, 0.6460853815078735, -1.1121445894241333, -1.217969298362732, -0.42750513553619385, 0.10270804166793823, 0.22411763668060303, -1.6682476997375488, -0.7359120845794678, 0.5567229390144348, 1.4564369916915894, -0.526465892791748, 0.4554480016231537, -0.5741209387779236, 1.3921438455581665, -2.552443027496338, 1.5137583017349243, 2.396683931350708, 0.010271310806274414, -1.513902187347412, -0.8644912838935852, 1.6632347106933594, -2.417525291442871, -0.02298416942358017, 1.7106354236602783, 2.514646291732788, -2.369243860244751, 0.5702823996543884, 0.05211067199707031, 2.771383762359619, 0.8364232778549194, -1.7990379333496094, -0.03352940082550049, 0.8177033066749573, 2.9048075675964355, 1.152603268623352, -0.21151119470596313, -0.04602992534637451, 0.9111008644104004, 3.615683078765869, -1.7347180843353271, 2.585331439971924, -0.444108784198761, -1.5963692665100098, 0.37092214822769165, 1.6583092212677002, -3.3555383682250977, 0.4701271057128906, 0.990146815776825, -0.24267634749412537, 0.09515001624822617, -2.3049352169036865, -0.4277053773403168, 0.36725351214408875, 0.011318027973175049, 0.047683000564575195, 0.6831878423690796, 1.0030009746551514, -0.1283205896615982, -2.238102674484253, -1.0174517631530762, -4.165457248687744, 0.1387500762939453, -1.7995445728302002, 1.6124063730239868, -0.21517637372016907, -0.13716834783554077, 0.14288204908370972, -1.157492995262146, -0.07377302646636963, -0.20687395334243774, 0.720887303352356, -0.5056661367416382, -1.411913275718689, 0.9005328416824341, 0.5824207663536072, -0.3580327033996582, 0.1587168276309967, -0.94515460729599, 1.9130878448486328, -0.12040604650974274, -1.8062007427215576, -0.685711145401001, -2.7397360801696777, 2.5108695030212402, -0.8311076164245605, 0.6944448351860046, 0.2920219302177429, 0.1142173707485199, -0.760020911693573, -0.1155019998550415, -1.9355239868164062, 0.3633103370666504, 0.42954444885253906, 2.0463967323303223, 1.9168075323104858, 1.9744654893875122, -1.3051373958587646, -1.2624738216400146, 2.134209156036377, -0.8716121912002563, -1.2377705574035645, -0.9356620907783508, 1.5846028327941895, 0.2907833158969879, -1.9393266439437866, -0.8044783473014832, 0.8685009479522705, -2.4833803176879883, 3.192599296569824, -2.972482204437256, -1.610472321510315, -0.164351224899292, -2.218235492706299, 0.9743489623069763, 0.45499691367149353, 2.3332927227020264, 1.2215895652770996, -1.242871880531311, -1.4079068899154663, -3.145735263824463, 1.6970338821411133, -1.6326184272766113, -1.2564632892608643, -1.2636865377426147, 0.5173343420028687, 0.07925620675086975, -0.40799760818481445, -0.3219083547592163, -1.7476797103881836, -0.5320302844047546, -0.6326117515563965, -0.06460762023925781, 1.7494547367095947, 0.43608659505844116, -0.7953176498413086, -1.5862709283828735, 0.7700278759002686, -0.16079610586166382, 1.3438926935195923, 0.37397119402885437, -1.1938886642456055, -2.71110200881958, 1.8017619848251343, -0.8721006512641907, 0.02459627389907837, 0.9668476581573486, -2.521998405456543, 0.22313839197158813, 0.6986548900604248, 0.3303525447845459, -0.19571176171302795, 2.5934808254241943, 0.6185977458953857, 0.3964630365371704, 1.3590128421783447, -2.390864372253418, 2.1779026985168457, -1.8075517416000366, 0.09055078029632568, 0.9819616079330444, 2.8193812370300293, 0.5803016424179077, 0.9887002110481262, 1.1588859558105469, -0.3807080090045929, 1.260957956314087, 1.2586573362350464, -0.6971893906593323, -0.7158153057098389, -2.140777349472046, -0.5449343323707581, 1.5079160928726196, -0.04589809477329254, -0.5531926155090332, -0.6455103754997253, -0.4178205728530884, -0.685013473033905, -0.9467102289199829, -1.3085840940475464, -1.2045577764511108, -2.061286211013794, 3.0841898918151855, -0.5886759161949158, 1.1232600212097168, 0.7141522169113159, -0.34963375329971313, -1.6523982286453247, -0.5927180051803589, 0.4799620509147644, 0.8429077863693237, 1.1084544658660889, -0.027344301342964172, 3.1856374740600586, -0.7525334358215332, 1.3678853511810303, -2.3761682510375977, -0.8051215410232544, 0.12549760937690735, 0.14632657170295715, -0.5112327337265015, -0.885945737361908, 0.7878424525260925, 0.3056761920452118, 0.18144172430038452, -0.6745442152023315, -0.6611717343330383, 0.025777816772460938, 0.7780767679214478, -2.7827930450439453, 0.8552272319793701, 1.0886478424072266, -1.4106695652008057, 0.820522665977478, 1.1558575630187988, 0.7896985411643982, 0.0861101895570755, -1.5235507488250732, -0.6650305390357971, -0.988296389579773, 2.0061826705932617, -0.1340281367301941, 1.134160041809082, 1.5107777118682861, 1.684161901473999, -2.9211337566375732, 0.5239835381507874, -0.8653666973114014, -0.17235153913497925, -1.1384479999542236, 0.0704493373632431, 1.510208010673523, 1.3970105648040771, 1.101859450340271, 0.46174514293670654, -1.6844102144241333, 1.9021269083023071, 0.26186949014663696, -0.08831268548965454, 0.11420011520385742, 0.49961793422698975, 2.1809000968933105, 0.43217772245407104, 3.3339040279388428, 1.1455034017562866, -0.7852506637573242, -1.6653356552124023, -1.1395665407180786, -0.8311288356781006, -1.2938095331192017, 0.6647915244102478, -1.1722288131713867, -0.7780346870422363, -0.40042591094970703, -1.216636061668396, 0.45000433921813965, -0.3666144609451294, 0.47764113545417786, 2.694605827331543, 2.0386710166931152, 1.2506999969482422, 1.5946531295776367, -2.3268420696258545, -0.8840658664703369, -0.7428891062736511, -3.4041855335235596, 0.7937944531440735, 0.8737610578536987, -0.6016116142272949, -1.992525339126587, 1.164640188217163, -1.1521646976470947, 0.5837739109992981, 1.4253268241882324, 0.08355553448200226, 0.23456108570098877, -0.9483516216278076, 1.9744127988815308, 0.8389052152633667, 1.1450960636138916, -3.3140482902526855, 0.4009087383747101, 2.4473085403442383, 0.9429775476455688, -1.7522590160369873, 1.807981014251709, 1.8733863830566406, 1.2915449142456055, 1.2531940937042236, -0.5304321050643921, 1.3843406438827515, 0.25325673818588257, 0.6190979480743408, 3.330963611602783, 1.0350128412246704, 3.1118056774139404, 0.11701476573944092, 1.9716849327087402, 1.629758596420288, -2.8055076599121094, 0.7802637815475464, 0.3888545632362366, -2.434601306915283, -1.2070002555847168, -0.9024789333343506, 1.2533495426177979, -0.5442075729370117, 0.8921602964401245, 1.5626883506774902, -0.16046631336212158, -0.9283298850059509, 4.11387300491333, -0.26339274644851685, -0.30572378635406494, 0.4477476477622986, 2.882197380065918, -0.48416009545326233, 2.141042709350586, -0.12174761295318604, 1.2665035724639893, -1.3606374263763428, -0.41477763652801514, 0.7411840558052063, -0.5491507649421692, -0.869608998298645, 0.4225994646549225, -0.42878028750419617, -2.9296789169311523, -1.5341933965682983, 0.38649970293045044, 0.8733161091804504, 0.5701278448104858, 0.35128992795944214, 0.8362807631492615, 0.7914810180664062, -0.0933823511004448, -0.28165486454963684, 2.0070548057556152, -1.713771939277649, -0.7832221388816833, 1.4848310947418213, 0.27156567573547363, 1.828874111175537, 0.5141470432281494, -0.6463649272918701, -1.146303653717041, 1.8296229839324951, -0.030648767948150635, 3.092489004135132, -1.9286715984344482, -0.36273324489593506, -1.0065110921859741, -0.4115971326828003, 0.16594204306602478, 2.1089489459991455, -0.7484642267227173, 0.19085437059402466, -0.5488013029098511, -0.761873722076416, -3.537424087524414, -1.8244738578796387, 0.2597556710243225, -0.24521967768669128, -1.2829692363739014, 0.945283055305481, -0.5130595564842224, -0.8521124124526978, -1.28206205368042, 0.45782560110092163, 0.2729133069515228, -1.3969746828079224, 0.14037173986434937, -1.6446139812469482, -0.22044488787651062, 1.88426673412323, 2.2209277153015137, 1.8028886318206787, 0.07935632765293121, 2.21580171585083, 0.24901360273361206, 0.07817846536636353, -0.10705828666687012, 0.6919035315513611, 1.7058138847351074, 1.4243693351745605, -1.5373588800430298, -0.11551791429519653, -0.8415935039520264, 2.395524740219116, 0.9184160232543945, -1.3219815492630005, -0.985435426235199, -0.08615563064813614, -2.6542625427246094, -1.4755909442901611, 2.367100954055786, -1.3216402530670166, 0.31549063324928284, 2.0323972702026367, 0.5461352467536926, 0.6843430995941162, -0.28136977553367615, 0.46580442786216736, -1.9105172157287598, -0.5311188697814941, -0.27590322494506836, -0.8011586666107178, 1.1213382482528687, -0.17785823345184326, 2.08150315284729, 0.7114683389663696, -1.1958441734313965, -1.4520936012268066, -0.1857680380344391, 0.5829733610153198, -0.6636922955513, -1.6739267110824585, -1.0961555242538452, 0.1375652551651001, 0.5635062456130981, 0.3133225440979004, -3.4002764225006104, 1.1848851442337036, 0.6178579330444336, 0.6369508504867554, -2.690211296081543, 2.3076210021972656, 0.9915730357170105, 0.8790059089660645, 0.36429738998413086, -2.198539972305298, 1.3956702947616577, 0.4436624348163605, -0.836211085319519, -0.8443984389305115, -1.1051374673843384, -0.15731555223464966, -2.393754720687866, -1.6970316171646118, -1.1165590286254883, -0.9162856340408325, -0.35148441791534424, -0.0692724883556366, 1.1843130588531494, 0.6720802187919617, -2.086759090423584, -0.09543585777282715, -0.1568305492401123, -1.4616127014160156, 2.8803930282592773, -0.21352320909500122, 0.7524885535240173, -1.2244980335235596, 1.8482277393341064, -0.16844066977500916, 2.422180652618408, -0.495319664478302, 1.8394362926483154, -1.3994810581207275, 0.8699208498001099, 1.4194724559783936, 2.0381317138671875, -0.7218937873840332, 0.02086007595062256, -0.5102497935295105, -0.4522436857223511, -0.07420158386230469, 0.8096333742141724, 1.9808930158615112, 1.4572415351867676, -1.3054684400558472, 1.0704820156097412, -1.2011544704437256, 0.9378639459609985, 2.090564250946045, -0.9364305734634399, -0.08940061926841736, 0.2856273353099823, -1.603285789489746, -0.7589784860610962, -2.660146951675415, 0.06470645219087601, 0.8547061681747437, -1.553076982498169, 0.19508130848407745, 0.6854420900344849, 0.34323781728744507, 0.19898931682109833, -1.8814785480499268, -2.3202834129333496, 1.0473088026046753, 1.397645354270935, -0.1880255937576294, -2.6145057678222656, 2.443082809448242, -0.5745339393615723, -1.4467518329620361, 0.0416644811630249, -0.6551274061203003, -1.79731023311615, 2.0117008686065674, 0.1504497528076172, 2.4338979721069336, 1.6479116678237915, 1.4794933795928955, 0.20886266231536865, 1.2244869470596313, -2.0550105571746826, -0.1268901824951172, -0.2274508774280548, 0.6207027435302734, -1.2324817180633545, 1.8244653940200806, 1.0754233598709106, -0.2542578876018524, -0.30132898688316345, -0.9094521403312683, -0.07973937690258026, 1.1589970588684082, 1.255499005317688, 3.2260241508483887, -1.0710701942443848, 0.03186619281768799, 0.860703706741333, 1.84421968460083, 0.8568087816238403, 1.161914348602295, 1.0413100719451904, 0.5047856569290161, 3.1340646743774414, -1.761567234992981, 2.610405445098877, 2.4128575325012207, 1.5212523937225342, 0.2657678723335266, -1.705683946609497, -1.9396456480026245, -1.7498151063919067, -1.2980676889419556, -1.6581006050109863, 0.003075370565056801, -0.869108259677887, 1.999135971069336, 1.8898067474365234, 1.119152307510376, 0.8803603053092957, -1.3289144039154053, -0.5333034992218018, -1.7461048364639282, -1.363248348236084, -0.08889502286911011, -1.3543426990509033, 1.835634469985962, 0.9118427038192749, -1.7988680601119995, -1.065110683441162, 1.857564926147461, 0.7732778191566467, 0.8056066036224365, 2.599238634109497, -0.7584262490272522, 2.140247106552124, 1.274000883102417, -0.09567111730575562, 0.34524285793304443, 0.08507990837097168, -0.30804750323295593, 1.650871753692627, 0.7839093804359436, -2.0433144569396973, -1.1668148040771484, 0.923114538192749, 0.14215993881225586, 1.528394341468811, -1.451648473739624, -1.3588367700576782, -1.71579909324646, -1.4800604581832886, -1.521405816078186, 0.2610403895378113, -0.8612139821052551, -0.724172830581665, 0.3146889805793762, 0.3911359906196594, 0.8018063902854919, -1.4486863613128662, 0.3786238431930542, -2.44213604927063, -0.4433809518814087, 1.2683887481689453, -1.4465004205703735, -0.2236931025981903, -1.5504637956619263, 0.06445753574371338, -0.7942153215408325, 0.36685115098953247, 1.6200354099273682, -1.2337591648101807, 0.03785264492034912, -0.2250782996416092, 3.162409782409668, 0.01397860050201416, -1.1049935817718506, -0.9008933901786804, 0.9531826972961426, 0.4741048216819763, -1.0520269870758057, -2.3656492233276367, 2.3661298751831055, -2.517681837081909, -1.452945351600647, -1.0283063650131226, 1.8778631687164307, 0.49574071168899536, -4.271899223327637, 1.705099105834961, 0.5634722709655762, -2.915309190750122, 2.6730763912200928, 1.2173945903778076, 1.5425156354904175, 0.34134942293167114, -0.12075063586235046, 0.7581985592842102, 1.5039865970611572, 0.8372179269790649, 0.8892607092857361, 3.349355936050415, 0.2900279462337494, -0.593246579170227, 0.7177593111991882, -0.1534034013748169, -0.4420475959777832, 0.7471403479576111, 0.6860642433166504, -0.3811274766921997, 0.39515218138694763, 0.78700852394104, -0.4205784201622009, 0.9852928519248962, 0.0933607667684555, 0.6702861785888672, -2.908583402633667, -1.4182500839233398, 1.7075467109680176, 0.3910636305809021, 1.3669371604919434, 1.096994161605835, 0.22893795371055603, -3.0270535945892334, 0.0922512412071228, -0.02911210060119629, 1.1422098875045776, -1.8782048225402832, -0.2200494408607483, 0.3887794613838196, -0.06585866212844849, -0.3420661389827728, -0.7901605367660522, -0.9370266199111938, 0.33443570137023926, -0.6200335025787354, -0.352297842502594, -0.8166907429695129, 2.4419093132019043, 0.19901669025421143, 0.3477318584918976, 2.789468765258789, -0.29512470960617065, 2.497234582901001, 0.8008125424385071, 0.5866903066635132, -1.0025372505187988, -0.7038891911506653, -2.9203431606292725, 0.8380935788154602, -1.6039516925811768, 0.0729372501373291, -0.9649885296821594, -1.5601227283477783, 3.5974507331848145, -0.38483014702796936, -0.08115875720977783, 0.47842997312545776, -0.48704567551612854, -3.2447831630706787, 0.23285329341888428, 0.5270574688911438, 0.423032283782959, 0.9924647808074951, 1.7474385499954224, -1.7191128730773926, 2.334946870803833, -0.3704538941383362, -1.3753767013549805, -1.428722858428955, 2.5457260608673096, 0.2013784646987915, 2.958583354949951, -1.6713926792144775, 0.8363834619522095], [1.0192795991897583, 0.46889233589172363, 1.9689031839370728, -0.3850370943546295, 0.555429995059967, 0.05134749412536621, 0.6264363527297974, 0.9411157369613647, -0.48339393734931946, 0.9628074765205383, 0.8761822581291199, 0.43862205743789673, -0.04014909267425537, -2.444183349609375, -0.10512322187423706, -0.8383758068084717, 1.9162158966064453, 0.2686072885990143, 0.17870432138442993, -1.3309146165847778, 0.0720670223236084, 0.9024887084960938, 1.7600033283233643, 1.3622524738311768, -0.8343213796615601, 2.3690242767333984, 1.209115982055664, 0.18332666158676147, 0.9983278512954712, 1.6546909809112549, 0.06344974040985107, 0.8315643668174744, 1.4795279502868652, 0.8898974657058716, 0.10573261976242065, 0.9884266257286072, 0.1686210185289383, -1.6059168577194214, 0.43861889839172363, -1.215735912322998, -1.9584661722183228, 0.71700519323349, -2.1408843994140625, 1.9058115482330322, -1.895063877105713, 0.7498798370361328, -0.8883566856384277, 0.9920749664306641, 0.06383073329925537, 0.16641980409622192, 0.28470081090927124, 1.4838638305664062, -2.3358750343322754, 1.9919629096984863, -0.7231839895248413, 0.8366559743881226, 0.667122483253479, -0.10984374582767487, 1.6406810283660889, 1.1884924173355103, -1.5520809888839722, 1.340057134628296, 1.7346094846725464, -2.072507381439209, -0.3246573805809021, 1.9699852466583252, -0.7188186645507812, 1.2790107727050781, 1.956053376197815, 1.859918475151062, -0.3483089506626129, 1.5289970636367798, 0.11760342121124268, -0.783019483089447, -1.6626535654067993, -0.9351716041564941, -0.007817775011062622, 0.22133490443229675, 1.928317666053772, 1.4571255445480347, 0.013947844505310059, 0.023731887340545654, -1.3563885688781738, -0.13062992691993713, -1.053868293762207, -1.64422607421875, -1.4097115993499756, -1.8118926286697388, -0.3056231737136841, -2.8011488914489746, 1.01153564453125, -0.5331121683120728, -1.0692546367645264, 0.42540442943573, 1.6883165836334229, -0.20522111654281616, 1.5458272695541382, 0.8025199174880981, -0.8964940905570984, -0.8198865652084351, -0.16281498968601227, 0.5992149114608765, 2.5138375759124756, 0.9221444725990295, 0.8946170806884766, -0.8860574960708618, -0.2787303626537323, -0.9430546164512634, -1.0580977201461792, -0.4342383146286011, -1.5063817501068115, -0.44532251358032227, 1.9339720010757446, -0.2921057939529419, 0.32054388523101807, -2.4233992099761963, -0.5139511823654175, 1.0745553970336914, -1.9240024089813232, -1.9637937545776367, -0.4712825417518616, 1.040344476699829, -1.883434534072876, 2.4526829719543457, 0.11975318193435669, -1.0965535640716553, -0.3396413326263428, 0.2221449613571167, 1.4650951623916626, -0.32511091232299805, -0.435571551322937, 0.9633678197860718, -0.9580833911895752, 0.7527682185173035, -0.11263886094093323, -0.2823854684829712, -1.7647275924682617, -2.4060373306274414, -1.8781428337097168, -1.037850260734558, 0.3476514220237732, 0.7559314966201782, 2.0141849517822266, 0.28369492292404175, -1.3713984489440918, 2.55448842048645, -0.36429598927497864, 0.11701405048370361, 1.5318193435668945, -0.23085641860961914, -1.4433510303497314, 0.3091263771057129, -1.1286062002182007, -0.6492546796798706, -1.142890214920044, -0.11680081486701965, -1.736553430557251, -0.5306528210639954, 0.9192007184028625, 0.023937225341796875, 0.3741770386695862, -0.40339067578315735, -0.6558314561843872, -0.31839144229888916, 0.7225022315979004, -3.0452446937561035, 0.6240198612213135, 1.0890326499938965, -0.6579912304878235, -2.1656384468078613, -2.7711188793182373, 0.791999340057373, 1.1214663982391357, 1.069894790649414, -1.6613240242004395, -0.47970354557037354, -1.5359206199645996, -1.2956500053405762, 1.6360081434249878, 2.5131947994232178, 0.6299415826797485, -0.7194495797157288, -1.6706081628799438, 1.627570390701294, 0.0735330581665039, 0.6037230491638184, 0.6840439438819885, 0.3640718162059784, 0.26462626457214355, 0.5929272174835205, 1.1165343523025513, 0.27652743458747864, -0.3326466977596283, -0.8385995626449585, -1.5706384181976318, -0.6148477792739868, -0.44842058420181274, -1.8496567010879517, 0.07504504919052124, -1.3476269245147705, 0.8135461807250977, -0.36030369997024536, -3.1721580028533936, 2.2111520767211914, -0.1646273285150528, -0.06647598743438721, -4.806084632873535, 0.44060802459716797, -2.1119611263275146, 1.0833593606948853, -0.465815007686615, 2.6170871257781982, -1.9505665302276611, 1.4476850032806396, -0.3976020812988281, 0.3790212869644165, -0.4752771258354187, 0.8292909264564514, 0.4921778738498688, -1.1468429565429688, -0.30307304859161377, -4.047525882720947, -1.2348393201828003, -1.9668927192687988, 0.8913789391517639, -0.6781684756278992, -0.7242646217346191, -0.07202692329883575, 1.0952551364898682, 0.9091152548789978, -2.148329257965088, 1.0814192295074463, 0.36251163482666016, -0.10410475730895996, -0.8241010904312134, -0.3532097041606903, -0.2064242959022522, -1.366119623184204, 0.2701256275177002, 1.646317958831787, -0.7263734340667725, 2.340641498565674, 0.6480093598365784, 0.14054462313652039, 0.242473304271698, 0.1460845172405243, 0.3818356990814209, 0.8940463066101074, -1.220444679260254, 0.07024793326854706, 2.7353951930999756, -1.1359293460845947, 2.1250672340393066, 0.32253503799438477, -0.5559751391410828, -0.585955798625946, -0.22900012135505676, 2.351154088973999, -1.113990306854248, -0.570191502571106, 2.5104808807373047, 3.7740986347198486, 1.0224412679672241, 0.5293677449226379, 0.5370995998382568, 0.9591109752655029, 3.1561689376831055, 0.014177083969116211, 1.0637824535369873, -0.7203768491744995, 1.026282787322998, 0.918694019317627, -1.2476260662078857, 0.33736300468444824, -0.16702556610107422, -1.1277581453323364, -1.3613851070404053, -0.23835442960262299, -0.5775678157806396, -0.336748331785202, -1.2935237884521484, -1.452304720878601, -0.018380358815193176, 0.889595627784729, -0.6511440873146057, 0.48082250356674194, 0.5950870513916016, -1.8073914051055908, 2.5157041549682617, 1.6934819221496582, -0.5901867151260376, 1.4077556133270264, 0.34901344776153564, 0.13183583319187164, 2.0052099227905273, 0.2809605598449707, -1.5150909423828125, 1.4629976749420166, -1.0245082378387451, -1.4074654579162598, -0.8210495710372925, -2.035276412963867, -1.8629742860794067, -1.9506795406341553, -2.294088840484619, 2.0512211322784424, -4.127971649169922, -0.9971480369567871, 2.643350124359131, 2.396500587463379, -0.8608567714691162, 1.473380446434021, -0.5531231164932251, 1.705698013305664, -1.7970366477966309, 1.7360488176345825, -0.7922291159629822, -0.7927994728088379, 3.8221280574798584, -0.5553627610206604, 0.22826170921325684, 0.8971502780914307, 0.33661580085754395, 1.5386565923690796, -3.3246958255767822, 0.03447145223617554, 0.16806288063526154, -0.32511579990386963, 1.0402991771697998, -0.8149625062942505, 0.0846407413482666, -0.5183880925178528, 0.7133113145828247, -0.21844100952148438, -0.980354905128479, -0.16462478041648865, 0.22553503513336182, -0.36273419857025146, -1.9547499418258667, -0.43206480145454407, -1.7572665214538574, 0.3238416314125061, -2.0225789546966553, -2.657254219055176, -1.1019585132598877, -2.227108955383301, -2.0649008750915527, 2.511784791946411, 0.44730788469314575, 1.8510035276412964, -1.3135453462600708, -0.9475602507591248, 1.8691812753677368, -0.5332887172698975, 0.04067099094390869, -0.943144679069519, -0.8372144103050232, -1.7348220348358154, 2.573725938796997, 0.39745551347732544, -1.8838083744049072, -2.4589104652404785, 0.7567312717437744, 1.6213514804840088, 2.761137008666992, -0.19831520318984985, -1.5031774044036865, 0.4290093183517456, 0.039298295974731445, 3.440110921859741, -0.8815286755561829, 0.14154690504074097, 2.1733150482177734, -0.25533974170684814, -2.3667471408843994, 0.1683814525604248, 0.0007133185863494873, 0.3947485685348511, 1.0818222761154175, 3.2437667846679688, -0.6284220814704895, -1.976299524307251, 0.9483718872070312, 1.3309290409088135, -1.0547817945480347, -1.2576794624328613, -3.2866365909576416, 0.42833489179611206, -1.0967903137207031, 1.070958137512207, -3.3904731273651123, 0.21568584442138672, 0.6630595922470093, -3.1061882972717285, -0.6491588354110718, 1.0750036239624023, -1.8815165758132935, 4.697700023651123, -2.9535751342773438, 0.04560685157775879, -0.45024722814559937, 0.8738843202590942, -0.48089998960494995, -0.8539372086524963, -0.7499656081199646, -0.17322471737861633, -0.5253283977508545, 0.12162768840789795, 1.754050612449646, -0.21626020967960358, -0.9545414447784424, 0.19501996040344238, -0.37508606910705566, -2.3090920448303223, 0.48042935132980347, -2.716519832611084, -1.8447651863098145, -1.2840371131896973, -0.2926780581474304, 1.3153271675109863, -0.5761389136314392, -0.8502089977264404, -0.8892916440963745, -2.2183172702789307, 1.4224663972854614, -0.184303879737854, -2.3201980590820312, -2.0077009201049805, 0.8238592147827148, 2.064570188522339, -1.9615848064422607, -2.013664960861206, -1.2469733953475952, 0.9897381663322449, 1.671911597251892, -2.142796277999878, 0.9420198202133179, 0.8556270599365234, -1.028444528579712, 1.754859447479248, -0.5548120141029358, 0.9559316635131836, -0.24078519642353058, 4.167288780212402, 0.7897788286209106, -0.6963618993759155, 0.20595479011535645, -1.7336359024047852, -1.2040445804595947, 0.07301396131515503, 0.3310353755950928, -0.6255226731300354, -0.16239392757415771, 1.11974036693573, 1.528355360031128, -0.11129820346832275, -0.13003575801849365, 1.7438976764678955, -2.193875312805176, 1.663456678390503, 0.08722828328609467, 0.9564560055732727, -1.4239360094070435, 2.80771541595459, 0.44144749641418457, -1.1996995210647583, 0.6050224304199219, -0.2800890803337097, -3.8986029624938965, 1.713647723197937, -1.5591868162155151, -0.8455849885940552, -0.2525809407234192, 1.9650229215621948, 0.22383582592010498, -0.7349441051483154, 0.06378445029258728, 3.2381653785705566, 0.8909415602684021, 0.3721621632575989, -0.6978381872177124, -0.3488750457763672, 0.7037662267684937, -1.3836547136306763, -0.15370333194732666, -0.4778866767883301, -0.9295871257781982, 0.4444897174835205, 0.6655768156051636, -1.386770248413086, 1.3082103729248047, 0.2940171957015991, 0.699569046497345, -0.7867315411567688, 0.9788303971290588, 0.94910728931427, 1.3398010730743408, -0.5800420641899109, -0.10047677159309387, 1.2927592992782593, -1.7354459762573242, -0.18268442153930664, 1.135005235671997, -0.27154234051704407, -0.44141313433647156, -1.3103303909301758, -0.7387464642524719, 0.33114683628082275, -0.8221440315246582, 1.4729454517364502, -1.4779884815216064, -0.304518461227417, 0.408599317073822, 2.420874834060669, 1.0138972997665405, 2.972280979156494, 0.37471306324005127, -0.3516099154949188, 1.8799258470535278, 0.25867679715156555, 0.2723105847835541, 4.389006614685059, -2.315730094909668, -0.3815405070781708, 0.2859746813774109, -0.4760144352912903, 1.1309949159622192, 1.6748183965682983, -1.2926708459854126, 1.8378376960754395, -1.582919955253601, -0.0586889386177063, -0.12605488300323486, -0.8490839004516602, -1.104637861251831, -0.8474647998809814, 2.132685899734497, 0.023671381175518036, -0.2809789478778839, 0.34407514333724976, 1.7507975101470947, -1.2469706535339355, -0.21475684642791748, -0.40085893869400024, 0.4444270133972168, -1.122480034828186, 1.3170127868652344, 0.1573052704334259, -1.020357608795166, 2.009373903274536, -0.14331796765327454, 2.0442957878112793, -0.4096152186393738, -1.482194185256958, -0.045666396617889404, 0.3336237370967865, 1.5688464641571045, -1.8623130321502686, 0.09030354022979736, -0.5114437937736511, 0.6113531589508057, -0.8235453367233276, -0.6150979399681091, 1.5640592575073242, -0.9975705742835999, 1.1003086566925049, 0.2734377384185791, 3.333800792694092, -0.789341926574707, 0.4399486184120178, 0.47167110443115234, -1.6995385885238647, -0.5901432633399963, 0.5271737575531006, 2.47039794921875, 1.7824724912643433, 2.088348865509033, 0.7899196147918701, 1.3824632167816162, 0.1065472960472107, -0.5824065804481506, 0.18395668268203735, -0.3702775835990906, -3.304772138595581, 1.0679560899734497, -0.9120621681213379, 1.5024417638778687, 2.7743988037109375, 0.6358821392059326, 0.19265729188919067, 1.0613967180252075, -0.4719111919403076, -1.1141526699066162, 1.422830581665039, 0.28429627418518066, 0.03623351454734802, 0.8512493968009949, -0.4647923409938812, 0.27440184354782104, 2.4362571239471436, 0.9905326962471008, -1.652022361755371, -0.9404178857803345, 0.3930751085281372, 0.4340195655822754, -0.8803715109825134, 1.400763988494873, 0.16223156452178955, 1.168905258178711, 1.8112516403198242, 1.2208189964294434, 2.8410427570343018, -0.2545452117919922, -2.6404364109039307, -0.8225598931312561, -0.9365781545639038, 0.17852649092674255, -0.0884946882724762, -0.3007968068122864, -0.6832870244979858, -0.7675659656524658, 0.5877168774604797, 3.2107186317443848, -1.919426679611206, -1.4801702499389648, -2.350970983505249, -0.4372100830078125, -0.7502341270446777, 1.1134344339370728, -0.706052839756012, 0.841079592704773, -1.1786808967590332, -0.6302424073219299, -0.5090057849884033, -1.74494469165802, -0.9102689027786255, -2.2008237838745117, -1.0050380229949951, -0.6367402076721191, 0.20366162061691284, 2.6080451011657715, -1.4985535144805908, -0.7107469439506531, 0.3213125467300415, 1.2335460186004639, -0.04409968852996826, 1.0046024322509766, -0.831770658493042, 1.5600156784057617, 0.31579914689064026, -0.017214983701705933, -0.09413085132837296, -1.5112241506576538, 0.44541892409324646, -1.1001627445220947, 0.7650523781776428, 0.4262089729309082, 1.1892024278640747, 1.341547966003418, 0.4068567752838135, -3.29575777053833, -0.021970480680465698, 1.4486181735992432, -1.3740599155426025, -0.6556934118270874, -1.5637147426605225, 2.332519292831421, 0.9616855382919312, -0.5774743556976318, 2.196347236633301, -1.740911841392517, -0.4564242959022522, 2.2231037616729736, 0.05592013895511627, -0.3954487442970276, -1.111101508140564, 1.7921907901763916, 0.674432098865509, 0.1907738298177719, -1.389129877090454, 1.4124761819839478, -2.387910842895508, 1.0638182163238525, -3.143007278442383, 1.0664067268371582, -0.573875367641449, -0.32013607025146484, -1.5044888257980347, -1.2645387649536133, 2.4041361808776855, 1.2704071998596191, 0.3815864324569702, 1.4846006631851196, 0.9383284449577332, -0.15236115455627441, 1.7298448085784912, -1.459773302078247, 0.4606718420982361, 1.1034908294677734, -1.0652642250061035, 1.932281494140625, 1.0920885801315308, 0.3085898756980896, 0.31729280948638916, 2.313754081726074, -0.5813250541687012, 1.7489368915557861, -2.074389934539795, -2.2593343257904053, -2.7552804946899414, 0.7278608083724976, 0.5671244859695435, -1.1302103996276855, 0.3063999116420746, -1.752002477645874, -2.3490195274353027, 2.315903902053833, -2.551598072052002, 0.014611668884754181, -0.42334526777267456, 2.6102449893951416, -0.7780243158340454, -1.469569444656372, -2.2055206298828125, -1.418372392654419, 1.485551357269287, 1.3935329914093018, -0.408354252576828, -1.3425123691558838, 0.8088822960853577, -1.4784539937973022, -3.561312198638916, 0.6845391988754272, -1.0415840148925781, 0.011620163917541504, 2.067208766937256, 2.7382118701934814, -1.1663607358932495, -1.0389559268951416, 0.11441713571548462, -1.3121428489685059, -0.4915728271007538, 0.026162028312683105, 0.5911396741867065, 1.101068139076233, -1.9942736625671387, -2.9713711738586426, 0.963701069355011, -3.9832897186279297, -0.35600364208221436, -0.1806849092245102, 0.7034084796905518, -3.383565902709961, 1.2796485424041748, -0.5599092245101929, -2.2027251720428467, 0.296735942363739, 0.9671396017074585, -1.2632811069488525, -0.3391682505607605, 2.1347579956054688, 0.060098737478256226, 0.005827799439430237, -2.299797773361206, 0.1252782642841339, -2.56986141204834, -0.2830198407173157, -1.213516354560852, 1.6639273166656494, -0.6025358438491821, -0.0881248190999031, -1.7836064100265503, 1.1408329010009766, 1.4219472408294678, -0.5919976830482483, 1.3311353921890259, -0.4271358847618103, 1.9851219654083252, -2.502882719039917, -2.3383398056030273, 0.3737473785877228, -1.3614604473114014, 1.719804048538208, -0.3643684983253479, 1.473200798034668, -1.2533226013183594, 1.5454387664794922, 0.78693687915802, -1.3928956985473633, 0.4437282979488373, 2.3881282806396484, -0.6931143999099731, -0.2702358365058899, -0.020839393138885498, 0.9820859432220459, 0.6460749506950378, -2.7938570976257324, -0.49020352959632874, 1.6305221319198608, -1.882552146911621, -1.8902339935302734, -1.511514663696289, -1.4475098848342896, -0.29435163736343384, 1.9148790836334229, 1.4305909872055054, -0.09871026873588562, 1.0833063125610352, -0.5761505365371704, -2.272724151611328, 1.5339818000793457, -0.47968852519989014, 1.2020599842071533, 0.6969579458236694, -0.6726385951042175, -1.9657680988311768, 0.10334545373916626, 1.4891945123672485, 0.24180160462856293, -0.20018303394317627, 1.5274295806884766, -0.4341941475868225, -1.149590015411377, -1.7432225942611694, 1.0731102228164673, 0.2388535439968109, -2.131010055541992, 3.5457096099853516, 1.2610414028167725, 1.0157607793807983, -0.9761099815368652, -0.8004096746444702, -1.052293300628662, 1.0337939262390137, -2.2851226329803467, 1.584019660949707, -0.7674238681793213, 1.149554967880249, -0.5284120440483093, -1.6182652711868286, 0.4560704529285431, -0.5766136646270752, 0.9330052733421326, -1.220392107963562, -1.4725927114486694, 1.0067591667175293, 0.7663733959197998, 1.0869275331497192, 0.47916072607040405, 0.21320313215255737, 2.0918195247650146, -0.44248372316360474, 0.25661221146583557, 0.5199648141860962, -1.7915985584259033, -1.8599762916564941, 1.0992276668548584, -0.662826657295227, -1.1353036165237427, -3.677675724029541, -0.4724833369255066, 1.5188993215560913, 0.360046923160553, -0.09599070996046066, 0.5981200933456421, 2.206099033355713, 1.7666711807250977, 1.8810877799987793, -0.08447813987731934, 2.2572970390319824, 0.7896088361740112, 0.5111494064331055, 2.9217333793640137, 2.3068222999572754, 0.4716404676437378, -0.6419222354888916, 0.022774040699005127, 0.23664283752441406, -0.5341196060180664, 0.18408769369125366, -1.5754811763763428, 1.23569655418396, 0.8454685807228088, 1.3162813186645508, -1.0043686628341675, 2.650824546813965, -0.6276606321334839, 1.0854815244674683, 0.05028505623340607, -0.024242281913757324, -2.5001144409179688, -0.9383087754249573, -0.44846785068511963, 0.30944937467575073, 1.5660662651062012, 1.3880987167358398, -2.492846727371216, -1.066897988319397, 1.3888888359069824, -3.289210319519043, 2.1722772121429443, 1.0823129415512085, -0.3941197395324707, -3.630418300628662, -0.05150233209133148, 0.35351139307022095, 0.05856449156999588, -0.7486274242401123, 1.4705336093902588, 0.9821188449859619, 0.5554424524307251, 2.0921683311462402, -3.1242237091064453, -1.2469279766082764, 1.1989710330963135, 1.0370887517929077, -0.7008479237556458, -0.9210036396980286, 0.11779582500457764, 0.7965860366821289, 1.997889518737793, 1.6885771751403809, 2.409034013748169, -0.28569525480270386, 0.6763261556625366, -0.09867435693740845, 2.0471646785736084, 0.11379152536392212, 0.8356362581253052, 2.3590376377105713, -0.7074500322341919, 0.4178552031517029, 0.07995568960905075, 1.149592638015747, 0.9072868227958679, -0.36924564838409424, 0.6084422469139099, -1.166856050491333, 1.582525372505188, -1.784022331237793, 0.9270239472389221, -0.6305851936340332, -1.6020171642303467, -0.43118417263031006, -0.9236968159675598, 0.5553699731826782, 1.264103889465332, 0.056218717247247696, -1.3701633214950562, -0.3184010684490204, 1.9543848037719727, 0.03830951452255249, 0.13334162533283234, 1.6248759031295776, -0.8931618332862854, 0.12905552983283997, 3.2329022884368896, 0.966468870639801, -0.5950801372528076, -0.7939008474349976, 1.0707422494888306, 0.3476794958114624, 0.27918505668640137, 0.7335968017578125, -0.7222371101379395, -0.579275369644165, -0.4693208932876587, 2.5308313369750977, 2.2317214012145996, -4.420052528381348, 1.0205490589141846, 1.0343801975250244, 1.1206135749816895, -2.7251996994018555, 1.1321908235549927, -1.5485820770263672, 2.3623549938201904, 0.32124221324920654, -0.08754202723503113, 0.012016832828521729, 2.1702370643615723, -0.954130232334137, -1.0540037155151367, 0.2777206301689148, -1.6687750816345215, 0.6320751905441284, 0.9099804162979126, 2.5174176692962646, 3.3228116035461426, -2.0921621322631836, -1.0344551801681519, -0.2812967896461487, -0.9847398996353149, 0.6127636432647705, 1.6401365995407104, 0.581693172454834, 1.2675875425338745, 0.07352101802825928, -0.0590742826461792, -1.075196385383606, 0.2623548209667206, 0.11255037784576416, 0.18453805148601532, -1.1371709108352661, 1.1774019002914429, 0.21324461698532104, 1.8053685426712036, 0.31746339797973633, -1.3497461080551147, -0.21408264338970184, 0.18274295330047607, 0.5717337727546692, 2.189391613006592, -2.5212624073028564, 0.1768302619457245, -0.59731125831604, 0.810082197189331, -0.030956149101257324, 0.13687002658843994, -0.049094200134277344, -0.7512032389640808, -1.0318913459777832, -1.2470663785934448, -0.0557248592376709, 0.0764867290854454, 0.38797786831855774, -0.4883105158805847, 1.467985987663269, -0.651146411895752, -1.0830564498901367, -2.9513936042785645, 0.2964441776275635, 0.07862377166748047, -0.147032231092453, 1.3755462169647217, -0.9429054856300354, 1.6003351211547852, 0.8856884241104126, -0.8630027174949646, 0.9249378442764282, 0.14138615131378174, -1.3756617307662964, -0.26541876792907715, -3.4357616901397705, 2.26528263092041, 1.040086030960083, -1.1957082748413086, -0.08205708861351013, 2.323744297027588, 0.15311920642852783, 0.25602012872695923, 0.6507358551025391, -2.2002711296081543, 1.2475653886795044, 1.732527494430542, 2.166694402694702, 0.2460799366235733, 1.8480350971221924, -0.9952164888381958, -0.9162362813949585, -0.7392042875289917, -2.270144462585449, 1.1774555444717407, 0.247083842754364, -0.22852683067321777, -0.47086143493652344, 1.0491952896118164, -2.52677059173584, 0.3657057285308838, -1.0603437423706055, -0.38978755474090576, 2.7460522651672363, 0.5068565607070923, -0.34637272357940674, -2.468876361846924, 4.048605918884277, -1.9907915592193604, 0.08353452384471893, 1.4244728088378906, -0.9477412700653076, 0.19272369146347046, -0.8665980696678162, -0.06457793712615967, -2.4747533798217773, 0.8501827716827393, -3.3211169242858887, -1.157565951347351, -1.1476370096206665, -1.1912723779678345, 0.7196345329284668, 0.7460228204727173, 1.8480733633041382, 0.3434638977050781, 0.5737466216087341, -1.5192866325378418, -1.3541761636734009, -1.8665168285369873, 1.1253116130828857, 0.5701013207435608, -0.22328731417655945, -0.6177383661270142, 0.11427843570709229, -1.245137333869934, 0.1367364078760147, -0.5297685265541077, 1.3299814462661743, -0.07482223212718964, 0.6873388290405273, 0.207889586687088, 1.361682653427124, -1.8314247131347656, 0.489408403635025, 2.244050979614258, 0.46417737007141113, 1.4795737266540527, 1.4430041313171387, 1.1452856063842773, -0.34871459007263184, -1.6301281452178955, -0.38260090351104736, 1.397177815437317, -1.5513203144073486, 0.5266067981719971, -0.5678786039352417, -0.3459375500679016, 2.2359743118286133, 0.8239840269088745, 0.4024592638015747, -0.21191179752349854, 0.6297286748886108, -1.1446019411087036, -0.7248430848121643, 0.16372346878051758, 1.554447889328003, -1.2071336507797241, 1.2700574398040771, 0.8740376830101013, 0.44195491075515747, 1.1803940534591675, -0.6805449724197388, 1.2577369213104248, 1.1135196685791016, -2.7668673992156982, 0.034945547580718994, 2.7147607803344727, -0.14623457193374634, 1.6414451599121094, -0.2360641360282898, 0.34349924325942993, -0.22969336807727814, -0.690634548664093, 0.6609394550323486, -1.4799778461456299, 1.3657829761505127, -0.1325489580631256, 0.7504678964614868, 1.127089023590088, -1.498861312866211, -0.17580178380012512, -0.5205711722373962, -2.7885501384735107, 0.5525113344192505, 0.13870330154895782, 0.9690451622009277, 1.1238832473754883, 0.4786955714225769, 2.120910167694092, -0.601972222328186, -1.630171775817871, -0.8420169353485107, -0.5527311563491821, 1.1924912929534912, 2.0677638053894043, -2.0535974502563477, 1.2394113540649414, 1.2844281196594238, 2.30680251121521, -0.42357301712036133, 4.539917945861816, 1.685381293296814, -2.096442461013794, -0.04095333814620972, -1.0480241775512695, -1.0656790733337402, 2.416872024536133, 0.34861433506011963, 0.18274551630020142, -1.5495306253433228, -1.400003433227539, -0.06513398885726929, 3.391104221343994, -0.29985687136650085, 3.8013105392456055, 0.5083028674125671, 0.22365912795066833, -1.5105514526367188, 2.2583415508270264, 2.922980785369873, -0.35611140727996826, 0.756572961807251, 0.9520007371902466, -0.1852703094482422, -0.09293961524963379, -1.0443037748336792, -1.2218413352966309, 0.7797050476074219, 1.3691396713256836, 0.7305896878242493, -2.2087149620056152, 0.4561924636363983, -0.4720151424407959, 1.5626850128173828, 1.2883477210998535, -1.094202995300293, -4.102901458740234, 0.6447733640670776, -1.8757232427597046, -0.586851179599762, -0.12704718112945557, 0.5029675960540771, 0.7559048533439636, 1.082769751548767, 1.9815839529037476, 1.2901983261108398, 3.894651174545288, -0.9576941728591919, 0.3508288562297821, 0.7453833818435669, 1.2940818071365356, -0.03196358680725098, -0.6918494701385498, 2.0444421768188477, 1.3270902633666992, 3.4440598487854004, 1.1177091598510742, 0.6320950984954834, -2.738344192504883, 1.1310863494873047, 1.918583869934082, 2.675110340118408, 1.3892970085144043, 1.2519960403442383, 0.30594557523727417, 1.0117202997207642, 2.6675283908843994, 0.3088638186454773, -0.3764689266681671, -1.6639277935028076, 0.8052777647972107, 0.07430779933929443, 0.5893543362617493, 0.8416824340820312, 1.3603806495666504, 0.24240326881408691, 0.047045350074768066, 2.061570644378662, 0.16432753205299377, -0.6089738607406616, 0.22483861446380615, -3.645082473754883, 0.014371737837791443, -0.09923899173736572, 1.0942405462265015, -0.9554703831672668, 2.9912030696868896, -2.7037694454193115, 1.818661093711853, 0.05650661885738373, 0.37998008728027344, 0.5627139806747437, 0.08327770233154297, 1.8287160396575928, -3.8767762184143066, -0.24159902334213257, -2.0917210578918457, 0.834447979927063, -0.04597395658493042, -2.258639097213745, 1.133553385734558, -0.07926315069198608, 1.7862529754638672, 0.09006404876708984, 2.676248073577881, -1.6656110286712646, 4.22767448425293, 0.055907994508743286, 0.9870101809501648, 0.10393203794956207, -2.1593990325927734, -0.4884411096572876, -1.8588441610336304, -0.433127760887146, -0.2786819338798523, -0.7516047954559326, -1.019816517829895, 0.3651244342327118, -2.9530951976776123, -2.2718143463134766, -1.364664077758789, -3.141533851623535, -0.03915596008300781, -1.6734709739685059, 0.9885721802711487, 1.102689504623413, 0.5258504152297974, -0.8289390802383423, 0.28128406405448914, 1.393985629081726, 0.02486962080001831, 0.14670930802822113, -3.1253812313079834, -1.412275791168213, 2.0075058937072754, 0.2991313338279724, 1.0411772727966309, 1.9237298965454102, -2.502716541290283, -0.817261278629303, -0.08591675758361816, 1.1641818284988403, -0.10024625062942505, -2.071500778198242, -2.148613929748535, -1.7066956758499146, 0.5233598351478577, 0.547292172908783, -1.5727081298828125, 1.2062962055206299, -0.4449727237224579, -1.5322883129119873, 1.0681581497192383, -1.062982439994812, 1.5937321186065674, -3.5263071060180664, -0.7121471762657166, 0.21410703659057617, 0.9578225016593933, -0.3714207112789154, 0.21465277671813965, 0.32794034481048584, 0.9408525824546814, -1.2501497268676758, 0.3106495141983032, -2.095733165740967, 0.8514341115951538, -2.506356716156006, -0.48330920934677124, -0.4288434088230133, 0.2747189998626709, 1.0353131294250488, -0.3958238661289215, -0.2902491092681885, -0.32226836681365967, 0.8749352693557739, 1.9108351469039917, 0.488847017288208, 2.057955741882324, 0.816495418548584, 0.367877334356308, -1.986665964126587, -1.1622862815856934, -0.33400386571884155, -0.19358813762664795, 0.2157975435256958, 1.1570930480957031, -0.9985219240188599, -0.2856907844543457, 0.8362828493118286, 0.09414494037628174, 0.05809590220451355, 1.8472356796264648, 0.601416289806366, -0.8064329028129578, 0.9041716456413269, 1.5356099605560303, -2.3891050815582275, 1.3599895238876343, 0.23155593872070312, -1.1398954391479492, 0.6592505574226379, -2.932849407196045, -0.7265158891677856, -3.6358375549316406, -0.812293529510498, 1.5100640058517456, 0.49977248907089233, 2.6329898834228516, -0.40133896470069885, -3.2673768997192383, -1.1002624034881592, 0.6722244620323181, 0.6906550526618958, 0.6032686233520508, 0.5975450277328491, 1.727613091468811, -0.41327276825904846, 1.153369665145874, 0.38282307982444763, 0.9495311975479126, 0.04441958665847778, 2.3588898181915283, -0.42079445719718933, 3.325650691986084, 2.1554150581359863, -0.36117881536483765, 0.273257851600647, 3.057511806488037, 0.16729915142059326, -0.06339621543884277, 0.42636537551879883, 2.3222599029541016, 1.020729899406433, -1.8996028900146484, 2.5582480430603027, -0.5285943746566772, -0.9219049215316772, 1.3557288646697998, -2.606724262237549, 1.4761773347854614, 1.5075340270996094, -0.5995009541511536, -3.4838602542877197, 0.17613977193832397, 0.5632344484329224, -0.9501988887786865, -1.2616513967514038, 0.4509718120098114, -2.2871713638305664, -1.4541414976119995, 1.5185842514038086, 0.658535361289978, 2.7555153369903564, -1.495869517326355, 0.15001404285430908, -1.4155542850494385, -3.2281031608581543, -0.5278294682502747, 0.623468279838562, 0.021346066147089005, 0.7547652125358582, 0.9827790856361389, 2.9641273021698, -1.1339521408081055, 1.5628156661987305, -1.371241569519043, 2.416320562362671, 2.21593976020813, -0.37882959842681885, -0.7090416550636292, -1.1149747371673584, -0.31883156299591064, 1.7200560569763184, -0.7591243982315063, 0.3230375647544861, 3.0659542083740234, 1.467244029045105, -2.3705453872680664, -1.985857605934143, -2.197415351867676, -0.6497656106948853, 0.8344958424568176, -0.010843634605407715, 1.6000323295593262, 1.251029372215271, 1.2211843729019165, -0.766422688961029, 1.3980566263198853, -0.6097089052200317, -0.28457504510879517, -3.031515121459961, 1.1324172019958496, -0.38508370518684387, 1.2355681657791138, -0.6630645990371704, 1.064424991607666, -0.7689168453216553, -1.650533676147461, -0.6040792465209961, 0.47976967692375183, -1.2273108959197998, 2.790440559387207, 1.5526893138885498, -1.2048797607421875, 2.3476433753967285, 1.6575945615768433, -1.606177568435669, -0.9297131299972534, -0.33064892888069153, 1.6997003555297852, -0.23251572251319885, 1.1556204557418823, 0.4342770278453827, 2.1050257682800293, -1.1889081001281738, 0.7634328603744507, 1.0282304286956787, 0.2303086519241333, 0.16310833394527435, -1.131522536277771, 1.6915874481201172], [-2.1412384510040283, 0.30975741147994995, 0.782089352607727, -2.0597548484802246, -2.181692600250244, -1.132854700088501, 2.054211139678955, 2.065321207046509, 2.5655064582824707, 0.5539623498916626, 1.8218849897384644, -1.3355867862701416, -0.13743621110916138, -0.5476233959197998, 0.7361334562301636, 1.3687224388122559, 1.5647960901260376, -0.469052791595459, -0.9390644431114197, 0.9254672527313232, 0.3682197630405426, -2.1742422580718994, -0.08655345439910889, -0.690558671951294, -0.35294049978256226, 0.3019259572029114, -0.8144598603248596, 0.5074030160903931, 1.8575630187988281, -0.6400739550590515, 2.3610520362854004, -0.6831727027893066, -0.03399936854839325, 0.29977476596832275, 0.5771347880363464, 0.4113650918006897, 2.246490955352783, 2.047088623046875, -0.3122625946998596, 1.854569435119629, 0.6957173943519592, -0.4680153727531433, 0.2037545144557953, 0.5650241374969482, 0.9744227528572083, -0.2819012403488159, 1.242261290550232, 0.7871487140655518, -1.2383052110671997, -1.0986229181289673, 0.3448430299758911, -0.7099804282188416, -0.7225326299667358, -3.937251091003418, 1.3666448593139648, -0.41888514161109924, 0.7323246598243713, 0.5400905013084412, 0.11383276432752609, 0.40214306116104126, 0.6517642736434937, 1.0866299867630005, -2.3945159912109375, -3.62654447555542, -1.9469521045684814, 0.3993379771709442, -1.67837655544281, -1.2050710916519165, -1.8503752946853638, -1.0535682439804077, -0.9940861463546753, -1.3633885383605957, -2.691622495651245, -0.4555931091308594, -0.16851210594177246, 2.215128183364868, -1.016207218170166, -1.4482975006103516, -2.5980405807495117, -1.5003570318222046, -1.1793639659881592, -0.4890403747558594, -0.2884487211704254, 1.0350703001022339, -0.9807354211807251, -0.28835391998291016, -0.989109992980957, -0.4341466426849365, -0.7257081866264343, 0.24387586116790771, -2.258000373840332, 0.6958326697349548, 0.048698097467422485, -0.15276075899600983, 1.8150944709777832, 0.5791751146316528, -0.45818662643432617, 1.1354128122329712, 2.2357399463653564, 2.997115135192871, -0.30126288533210754, -0.225236177444458, -2.766296625137329, 0.929108738899231, 0.8625714778900146, -0.29066866636276245, 0.8671637177467346, -2.6405227184295654, -0.6333714723587036, 1.5026222467422485, 1.33079195022583, -1.3486552238464355, 0.029712289571762085, 0.41425269842147827, -2.6860175132751465, -0.3131179213523865, -0.48947668075561523, 1.4972931146621704, -1.1487653255462646, 1.6683368682861328, -2.5894880294799805, -0.16446596384048462, 3.4344918727874756, -0.22608435153961182, -1.877469539642334, 1.2200016975402832, 1.4277994632720947, -0.7994537353515625, 1.7764650583267212, -0.8666069507598877, 2.6487560272216797, 0.36783504486083984, -0.18418824672698975, -0.195603609085083, -0.127427339553833, -1.5817618370056152, 2.2421345710754395, -1.118350863456726, -1.119928002357483, -0.9601430892944336, 0.9193155169487, 0.19499993324279785, 0.018202006816864014, -0.12664231657981873, -1.0932250022888184, 0.9751405119895935, 2.8411264419555664, -0.8803037405014038, -2.843014717102051, -1.609329342842102, 0.13941514492034912, 1.4466240406036377, 0.46400970220565796, 0.8759010434150696, -1.2789467573165894, 1.4038487672805786, 0.06984660029411316, 2.205859422683716, 1.0530822277069092, -1.1069601774215698, -0.9332417249679565, -0.7670150995254517, -0.35984891653060913, -2.350499153137207, 0.7120047211647034, -2.6265881061553955, 1.9009171724319458, -1.4636826515197754, 1.3829330205917358, 1.3546260595321655, 1.1327054500579834, 0.028824210166931152, 0.07636773586273193, -1.6470166444778442, 0.1615440398454666, 0.7117266654968262, 0.9688317775726318, -0.6458489298820496, 2.134997844696045, -2.0053086280822754, -0.4882275462150574, 2.094470739364624, -2.303400993347168, 2.699091672897339, 1.1145155429840088, 1.5761330127716064, -2.0880088806152344, -1.175591230392456, 2.309875011444092, 0.45858538150787354, 1.7770179510116577, -1.11636221408844, 0.5680544376373291, -0.7402302622795105, -0.5363974571228027, 1.6171897649765015, 1.5471866130828857, 1.5156443119049072, 0.07683521509170532, 1.497572898864746, -0.08588564395904541, 2.533425807952881, -1.6953030824661255, -0.5290693640708923, 2.4154820442199707, -1.529875636100769, -1.0018768310546875, -1.854827642440796, -1.183946967124939, 2.398244857788086, 0.9779911637306213, 0.4047156870365143, -1.3939670324325562, -0.3477414846420288, 2.1549229621887207, 0.39856380224227905, -1.0490336418151855, -1.7912646532058716, -0.79285728931427, 0.5720527768135071, -2.3433821201324463, 2.020838737487793, 0.42750054597854614, -1.6158695220947266, 1.0958765745162964, 0.3434793949127197, 1.6002930402755737, -1.09637451171875, 0.6686527729034424, -0.23675990104675293, 0.8213284015655518, 0.5326856970787048, -1.0038013458251953, 0.9382795095443726, -1.330291509628296, -0.9624766111373901, 0.7655723094940186, 0.6644729971885681, -1.9563202857971191, -0.5190582275390625, 0.7231515645980835, -0.6496003866195679, -0.7367180585861206, 3.2415387630462646, -2.9440250396728516, 1.3562424182891846, 2.888381004333496, 2.369809150695801, 1.379079818725586, 1.7918380498886108, -0.30985790491104126, -0.6497935056686401, 1.324852466583252, 2.5724239349365234, -0.9213777780532837, -2.410583734512329, -1.5482070446014404, -3.024775981903076, -1.3790569305419922, -2.4298949241638184, -0.15975534915924072, -1.7209906578063965, 0.7779006958007812, 0.8084408044815063, -1.0876424312591553, -0.857785165309906, -0.7884877324104309, 0.8294690251350403, -1.3343470096588135, -0.27131712436676025, 0.290316104888916, -0.637543797492981, 1.180501103401184, 0.9397448301315308, 0.9345361590385437, -1.5712069272994995, 1.3377747535705566, -0.9391666054725647, -0.6853063106536865, 1.5322012901306152, -0.3831546902656555, 0.009858429431915283, -2.9746811389923096, -0.8418599963188171, -1.470656394958496, 0.33455896377563477, 1.0623798370361328, 1.5864338874816895, -1.3270931243896484, -1.2900429964065552, 1.9003329277038574, -0.28016504645347595, 0.6242722868919373, -1.2149583101272583, 1.1095162630081177, 2.8496522903442383, -0.24459561705589294, 0.6313725709915161, 1.1538184881210327, 1.4627598524093628, -0.060216546058654785, 0.8297742605209351, 1.2941476106643677, -2.084397792816162, -2.1113662719726562, -2.780961513519287, 1.3312208652496338, 0.4633811116218567, -1.658928394317627, 0.36702412366867065, -0.21174660325050354, 0.061189621686935425, 0.9923810362815857, 1.7797868251800537, -0.682816743850708, -2.970726728439331, -1.0609636306762695, -0.1740899682044983, 2.0018935203552246, 1.3604601621627808, 0.18642210960388184, 1.0092116594314575, -1.7577537298202515, 0.06482595205307007, -1.879256248474121, 2.5732920169830322, -0.019117116928100586, 0.07998871803283691, 2.106102466583252, -1.982017993927002, 0.40885379910469055, 0.8943798542022705, 0.5353114604949951, 0.5467770099639893, -1.8106038570404053, -0.604656994342804, -1.420844316482544, -1.1120116710662842, -1.196036458015442, -0.7707967162132263, 0.008330345153808594, 1.6963958740234375, 3.4028778076171875, -1.523926019668579, -1.9364173412322998, -0.6391165256500244, -0.3893360495567322, -1.8343687057495117, -1.0034738779067993, -0.7337526082992554, -1.9030072689056396, -0.620348334312439, -0.7869875431060791, -0.3920677900314331, 0.5460321307182312, -0.458368718624115, 1.1637433767318726, 0.13079839944839478, -0.14381027221679688, 0.8141809105873108, -2.7594828605651855, -1.8315796852111816, 2.2257397174835205, -0.836717963218689, 0.5241591930389404, -0.23422740399837494, -1.2133324146270752, -0.6469297409057617, 0.12390375137329102, 0.346513032913208, 0.7960555553436279, -0.08941717445850372, 0.871647834777832, -0.40359488129615784, 1.1519112586975098, 0.2076008915901184, -1.3983067274093628, -1.2891721725463867, -2.1597118377685547, 1.1682488918304443, -3.824977397918701, -3.587928533554077, 0.3169711232185364, -0.09184589982032776, 1.3916200399398804, 1.4270787239074707, -0.6858285665512085, 0.26198673248291016, -2.303107976913452, 1.8033860921859741, -2.885833263397217, -0.8092849254608154, -2.0894346237182617, -0.7263006567955017, 0.5728954076766968, 2.480725049972534, 1.3013134002685547, -1.3090296983718872, -1.0524046421051025, -0.6291224360466003, 0.29257792234420776, -1.630627155303955, 0.07630854845046997, -0.28091323375701904, 1.4996507167816162, -1.395328164100647, 0.930870771408081, -0.7260231971740723, -0.08799612522125244, 2.417585849761963, 1.4297709465026855, 4.182748794555664, 0.970069408416748, -1.7740885019302368, -2.301703691482544, 0.5056357979774475, -0.508616030216217, -1.1702507734298706, -0.09891742467880249, 1.679854154586792, -0.9972532987594604, -0.3136976957321167, 0.784256100654602, 0.3129487931728363, -0.5711990594863892, 1.8264048099517822, 0.09982845187187195, 1.4927706718444824, -0.5284538269042969, 0.3104495406150818, 0.03351849317550659, 1.8686046600341797, 0.06747478246688843, 2.8594424724578857, 2.368305206298828, -0.6722207069396973, -2.390984058380127, -0.16280601918697357, 1.1039087772369385, -1.1276649236679077, -3.127228021621704, -0.24652431905269623, -0.2592509984970093, -1.0452930927276611, 1.6190630197525024, 0.9408451318740845, -2.6004161834716797, 0.6304190158843994, 2.5309717655181885, 1.7032744884490967, 2.0203797817230225, -0.7502241730690002, -0.4984423816204071, -0.17305457592010498, -0.20488743484020233, 3.087587833404541, 2.9609951972961426, -0.22179149091243744, -0.5409328937530518, -0.1949869990348816, -0.7010513544082642, -0.6431920528411865, -0.27687910199165344, -1.5010114908218384, -0.7101843953132629, -3.4640657901763916, 1.2105040550231934, -0.7198199033737183, -0.7805856466293335, 1.9556825160980225, -2.482717990875244, 0.6116546988487244, 0.7587369680404663, 0.6822290420532227, 0.4703690707683563, -1.421328067779541, 0.23537752032279968, -2.3192341327667236, 0.23909765481948853, -0.1810130774974823, 0.6302644610404968, -1.4982519149780273, -1.6697380542755127, 1.7720189094543457, -2.9750704765319824, -1.9659278392791748, -1.0211855173110962, 1.2571513652801514, 1.4301097393035889, 2.829867124557495, -0.7382254004478455, -3.825672149658203, -1.0195283889770508, 0.6244890689849854, 0.039924025535583496, 1.3779770135879517, -1.3157505989074707, 0.3634525239467621, 2.5966076850891113, 0.9824831485748291, -0.8157006502151489, -1.182045340538025, -0.6827602386474609, -2.994980573654175, -1.01210355758667, -0.6299895644187927, -2.3565661907196045, 1.5388425588607788, 1.5305501222610474, -0.22607338428497314, 2.021963119506836, 2.7473039627075195, 0.5369296073913574, 1.6350226402282715, 1.1214022636413574, 0.7086782455444336, 1.7044365406036377, 1.5391544103622437, 1.6252555847167969, 1.1368025541305542, -0.16827979683876038, -0.8529149293899536, -1.6508033275604248, 1.0679742097854614, -2.963139533996582, 1.2332782745361328, -0.49310392141342163, -0.6310536861419678, -0.6069562435150146, 1.5675561428070068, 2.047569751739502, 0.19134056568145752, 0.5383449792861938, 0.47862499952316284, -2.367067813873291, -0.3451230525970459, 0.4885956645011902, -1.3686497211456299, 1.083306908607483, 2.396960496902466, 1.0965038537979126, 0.15860587358474731, -0.49001848697662354, 2.5063185691833496, 2.1335155963897705, 1.2538502216339111, -1.0442475080490112, 0.02992987632751465, 0.3344630002975464, -0.2615540027618408, -0.7237845063209534, 2.273315906524658, 2.293731927871704, -1.8994966745376587, 1.4863040447235107, 0.21081417798995972, -1.9219090938568115, 1.3178346157073975, 1.2387497425079346, 0.12138921022415161, -0.4123212397098541, 0.7420791387557983, -0.8405807018280029, -0.24514222145080566, 0.004392862319946289, -0.5714371800422668, 1.7831649780273438, -1.8683407306671143, -0.7877078652381897, -1.6657986640930176, -1.5272338390350342, -3.277836799621582, -1.4422250986099243, -0.6358644366264343, 0.45630592107772827, -0.09372082352638245, 2.0903375148773193, -2.8435189723968506, -0.6017444729804993, -0.4946938157081604, 0.040847182273864746, -1.0735341310501099, 2.204061508178711, 0.07438807189464569, -0.09572029113769531, -0.1473042070865631, 0.45991599559783936, -0.3037053942680359, -0.725674033164978, 0.7487982511520386, -0.4851759970188141, -0.9748692512512207, 0.6888789534568787, 0.6042279005050659, 2.252488613128662, 0.4446052312850952, -1.120877742767334, 0.9867888689041138, 0.01945817470550537, -0.7813312411308289, 2.161593198776245, -0.28713709115982056, -1.216171383857727, -0.07541131973266602, -1.157408595085144, 1.3429770469665527, -0.3672645390033722, 1.2211902141571045, -0.0667264461517334, 0.4180675148963928, 1.700440764427185, -0.8317467570304871, 1.7327682971954346, -0.3403354287147522, 1.3370368480682373, 1.4316086769104004, 1.4736274480819702, -1.183502435684204, -1.9516080617904663, -2.4147040843963623, -1.5606231689453125, 1.318146824836731, -0.34187811613082886, 0.05495747923851013, -1.1673744916915894, -0.5917676091194153, 1.217345118522644, -0.7108072638511658, 1.831687092781067, 0.7615242600440979, -1.4119017124176025, -1.8856656551361084, -0.5152114629745483, 1.1813204288482666, 0.9086189270019531, 1.1717325448989868, 3.0129315853118896, -1.592549443244934, -0.2337566614151001, -1.1724885702133179, 2.2051634788513184, 1.0375959873199463, 1.2180745601654053, -1.6938862800598145, 3.264087677001953, 3.4036688804626465, 0.8893494606018066, -0.6169471740722656, -0.47797369956970215, -2.5617799758911133, 0.973008930683136, 3.256763458251953, 1.670661211013794, -0.30931615829467773, 2.2280497550964355, -2.08453631401062, -0.2886001765727997, 1.2317231893539429, 1.8931665420532227, 1.571058750152588, -1.267346978187561, 0.8974883556365967, 0.6270580291748047, 2.2584352493286133, -1.2853271961212158, -0.47209632396698, -0.07433879375457764, -0.6064475774765015, -0.846199095249176, 0.05479556322097778, -0.9981611967086792, 3.224226236343384, -3.1337931156158447, 0.7571467757225037, 0.656819760799408, -0.6092046499252319, -0.8923702836036682, -0.5431728959083557, 1.1568305492401123, -0.00218171626329422, 0.09538716077804565, -0.6297607421875, 0.28439217805862427, 1.0926167964935303, 2.4074625968933105, 0.6115744113922119, 1.6642255783081055, 0.6584805846214294, -1.3410817384719849, 2.792269706726074, 1.1744980812072754, 1.7156652212142944, 1.0185233354568481, 1.0688226222991943, 0.1579018235206604, 2.3225810527801514, 0.5819133520126343, 1.5951647758483887, 0.7598268985748291, -3.2799830436706543, -0.3632694184780121, -0.10139137506484985, -2.376430034637451, -1.7294285297393799, -0.3994176685810089, 1.1842052936553955, -1.916165828704834, 0.4317218065261841, 1.726567029953003, -0.6977912187576294, -2.0732314586639404, -1.3880137205123901, -0.9947732090950012, 1.8172101974487305, -1.5660794973373413, -0.8300967812538147, 0.9622931480407715, -1.1446595191955566, -2.5757365226745605, -1.9981764554977417, -0.3038480281829834, 1.0165868997573853, -1.1409461498260498, -1.6760344505310059, 0.9872637987136841, -1.0907033681869507, -0.12988871335983276, 0.08853662014007568, -2.048628330230713, 0.6660100221633911, -1.3596339225769043, 1.1040656566619873, 0.5660428404808044, -2.3840131759643555, -0.2675108015537262, 2.380235195159912, -1.2856160402297974, -0.041226983070373535, 0.2140587419271469, 2.3328609466552734, 0.6527223587036133, 0.4245535135269165, -1.5709580183029175, -1.7344462871551514, 0.47378140687942505, -0.8946511745452881, -0.5338605642318726, -0.9345629215240479, -3.67999267578125, -0.2690938115119934, 0.6516644954681396, -2.750500440597534, -1.9395065307617188, 1.9474979639053345, 1.2622233629226685, -0.25123876333236694, 1.8439698219299316, 4.0006914138793945, -0.14821723103523254, -0.05036783218383789, 0.4395815134048462, -2.559948444366455, -0.44555819034576416, 0.11314895749092102, -0.20016516745090485, 0.3856549859046936, 1.205326795578003, 0.6154003739356995, 1.6958109140396118, -0.12052866816520691, 1.4501231908798218, 0.930448591709137, 1.8254772424697876, 0.2205023169517517, 0.6725892424583435, 0.40512919425964355, -1.289126992225647, 0.49479225277900696, 0.6120004653930664, 3.534554958343506, 4.166879653930664, 1.528577446937561, -1.3606603145599365, 0.5305804014205933, -0.8343063592910767, 0.9763940572738647, 0.7299537658691406, -0.7719732522964478, -0.37680044770240784, -0.96537184715271, 0.616668701171875, -0.6063604354858398, -1.3126250505447388, 0.1644502431154251, 0.47470808029174805, -0.33291178941726685, -0.8029773235321045, 1.4381908178329468, -0.3313843011856079, -0.524573564529419, 1.0539389848709106, 2.6550536155700684, 1.5620288848876953, 0.6112073659896851, 0.6179402470588684, -0.1805196851491928, -1.5167218446731567, 1.2839452028274536, -0.48817580938339233, -2.492393732070923, 0.8817580938339233, 1.5422333478927612, -0.04473656415939331, -0.01231616735458374, 0.21816754341125488, -0.3086789548397064, 3.482785701751709, -1.592132806777954, 1.11215078830719, -3.2794899940490723, -1.4743499755859375, 0.5343614816665649, 0.12117207050323486, 2.098395586013794, -1.3462704420089722, 2.1132171154022217, -1.5930469036102295, -1.5394285917282104, -0.8242673873901367, -0.6549056172370911, -1.0405566692352295, 1.6619495153427124, 2.6242728233337402, 2.123115062713623, -0.0734870433807373, -0.6512113213539124, 2.320531129837036, -1.4482152462005615, 1.105676293373108, -1.3452025651931763, 0.6251903772354126, 1.7470935583114624, 1.4269850254058838, 0.04651486873626709, -0.37743645906448364, -0.5273475050926208, -0.34657424688339233, -1.1141278743743896, -0.1156754195690155, -0.6050989627838135, 2.198789119720459, -0.603294312953949, 0.18956273794174194, 0.1418520212173462, 2.267280101776123, -2.54882550239563, -0.8484117984771729, 1.1352288722991943, -0.39847373962402344, -1.8434438705444336, -0.06759178638458252, -0.9784198999404907, -0.9970583319664001, -0.6392808556556702, -1.853222131729126, -0.18263816833496094, -2.379176139831543, 1.8906137943267822, 0.28315842151641846, 0.16063833236694336, -0.721337080001831, 0.20285439491271973, -1.0254383087158203, 0.8143741488456726, 2.116628885269165, -0.24693216383457184, 0.6152013540267944, -0.9233478307723999, 0.16766072809696198, -0.6786116361618042, -1.352919101715088, 1.1044766902923584, -2.6180613040924072, -1.6976487636566162, -0.33592450618743896, 0.9674032926559448, 2.9611124992370605, 1.5777332782745361, -0.6953123807907104, -1.28507661819458, -1.1809957027435303, -0.0865207314491272, 1.4870796203613281, -1.2431013584136963, -0.0869109034538269, 0.385553240776062, 0.18105018138885498, 1.833008050918579, -2.5200114250183105, -0.03987200930714607, -0.019404947757720947, 1.440739393234253, -1.2732844352722168, -2.5317931175231934, 0.38885292410850525, -2.2323174476623535, 0.746857225894928, -0.9857074022293091, -0.5535321831703186, 0.1881338357925415, -3.2744951248168945, -0.08678442239761353, -1.5872111320495605, -0.6196318864822388, -1.8923969268798828, 1.7250525951385498, 1.0232510566711426, 2.702670097351074, 2.6943867206573486, -3.984917163848877, -0.3045976161956787, 0.6635314226150513, 0.10852422565221786, 1.6364567279815674, 0.012711048126220703, 0.6021367907524109, 1.8775542974472046, -2.0001559257507324, 0.46873223781585693, -2.0822606086730957, -0.6589193940162659, 0.2832696735858917, 3.407073497772217, -1.5945745706558228, 0.8633830547332764, -2.164808750152588, -1.696404218673706, 2.3096680641174316, -1.6609854698181152, -2.844589948654175, -0.6273677945137024, -0.7574515342712402, -0.22887524962425232, 2.0994620323181152, 1.4793041944503784, -1.1603528261184692, -0.029843926429748535, 0.013802886009216309, 0.5894060730934143, 0.4433092176914215, 1.1122492551803589, -1.2532439231872559, -1.477043628692627, 0.5782730579376221, 0.7469462156295776, -0.6918529272079468, -2.5921497344970703, 1.016270399093628, 0.8599857687950134, -1.517425298690796, -1.5642478466033936, -0.6837669014930725, 1.0139020681381226, -1.9237077236175537, 2.8909945487976074, -0.5137467384338379, 0.9374269247055054, 1.957884430885315, -1.2363616228103638, 2.766603469848633, -0.6298248171806335, 0.8855467438697815, -0.5588656067848206, 2.736217498779297, 1.1361045837402344, 0.5691590309143066, 0.656231164932251, 3.3513288497924805, -1.602752685546875, 1.0312217473983765, 0.6987413167953491, 1.0865662097930908, -0.4621692895889282, 1.5224716663360596, -1.651975154876709, 0.4124622344970703, 1.4909236431121826, -0.049251824617385864, -2.937652587890625, 2.2715840339660645, -1.6524767875671387, -1.2132923603057861, -1.5335737466812134, 0.03698074817657471, 1.31393563747406, -0.9945127964019775, -1.653991460800171, 0.8901536464691162, 0.1742696762084961, -2.1148369312286377, -1.5591151714324951, 1.1713364124298096, -0.4317074716091156, -0.5969728231430054, -0.4873085021972656, -0.806351900100708, -1.8072454929351807, 0.9077866673469543, -4.732029438018799, 1.6630539894104004, -0.8345984816551208, -0.018966108560562134, -1.604367971420288, -0.4929938316345215, 2.102796792984009, 0.6798701286315918, -1.0959630012512207, 0.5675314664840698, 1.0892870426177979, 0.8431059122085571, -1.6228692531585693, 1.856475591659546, 0.48461413383483887, 0.7058670520782471, 1.786008596420288, 0.36374226212501526, 3.365534782409668, -0.5031704902648926, -0.2441062331199646, 1.981755018234253, -0.588412344455719, -2.2682228088378906, -0.44997718930244446, -0.7852717638015747, -2.7494516372680664, 2.4036314487457275, -0.14665402472019196, 2.0321152210235596, 2.47556734085083, -0.5129557251930237, -0.971168577671051, -0.09509646147489548, 0.9504254460334778, 0.08792704343795776, 0.5622117519378662, 0.5962647199630737, 1.98334801197052, -0.4620323181152344, -2.581118106842041, -0.897464394569397, 1.4149953126907349, -0.44572892785072327, -0.6371563673019409, -2.4023396968841553, 0.6995706558227539, 1.2493327856063843, 0.14617255330085754, -0.7779878973960876, -1.2050278186798096, -1.9382129907608032, -0.6885942220687866, 0.2571834921836853, -0.003142833709716797, 1.6008849143981934, -3.414484739303589, -0.6350889801979065, -0.9552728533744812, 1.0250506401062012, 0.9860404133796692, -0.7072945833206177, 1.305964708328247, -1.3997877836227417, -1.5814697742462158, 0.7930516004562378, -1.2323722839355469, -1.3248987197875977, 1.948286771774292, -0.2648623287677765, 0.8608784079551697, 1.0785053968429565, -2.27540922164917, 1.6048096418380737, -1.6028980016708374, 1.822107195854187, -1.8575143814086914, -1.1074408292770386, 0.5030337572097778, 2.5174684524536133, -0.30853575468063354, 0.48523032665252686, -0.25517475605010986, -0.41430211067199707, 1.2032965421676636, -0.3033151626586914, 2.738135814666748, 1.3277418613433838, 0.3206416964530945, 1.681919813156128, -0.8890875577926636, -1.2260152101516724, 1.4916179180145264, 0.7139154672622681, -0.2882400155067444, -2.480386734008789, -0.15241235494613647, 0.3341265022754669, -1.2039992809295654, -0.9088724851608276, -0.7303212881088257, 0.9687181115150452, 0.8680804967880249, 1.5333902835845947, -1.2115755081176758, 0.8454806804656982, 0.0752178430557251, -0.30372723937034607, 0.3440370559692383, 0.6242595911026001, 2.0751760005950928, 3.8487887382507324, 0.017145097255706787, -1.180060625076294, -0.3459281623363495, 1.2693040370941162, 1.2227065563201904, -2.5463576316833496, 0.8700858354568481, 1.0860028266906738, 0.15150153636932373, -3.5670881271362305, -1.8493283987045288, -2.2938592433929443, 1.0418238639831543, -0.4969959259033203, 1.3725190162658691, 0.6767458915710449, 0.3110972046852112, 1.8391709327697754, 0.1980043649673462, -1.5773993730545044, -1.7586168050765991, 0.7166358828544617, -0.605430543422699, -0.2247033417224884, 0.7192676663398743, -1.7676951885223389, 1.4851794242858887, -1.526464581489563, -2.224015235900879, -2.340580701828003, 0.06915605068206787, -2.591697931289673, 0.36887192726135254, 1.3558226823806763, 0.39386656880378723, -1.2529791593551636, 1.14320969581604, 0.9435335397720337, 0.08394423127174377, 0.5200821757316589, -1.8003551959991455, 0.7964343428611755, 1.4649714231491089, -0.03654742240905762, -0.14529192447662354, 2.43300199508667, 1.8653056621551514, -1.001564860343933, 0.7434014081954956, -0.9250820875167847, 0.14926956593990326, -1.450891137123108, -1.6493924856185913, -0.203857421875, 1.8833599090576172, 2.5281429290771484, -1.5663254261016846, 0.9214651584625244, -0.6315802335739136, -0.6269931793212891, -1.648006796836853, 2.0619750022888184, 0.6343122720718384, 1.1198590993881226, -2.499743938446045, 1.668793797492981, -0.3960112929344177, -2.743138551712036, 1.0431170463562012, 0.7943605780601501, -1.83955979347229, -0.5999236702919006, -0.698728084564209, -2.923100233078003, -1.2462643384933472, -0.35031458735466003, 2.3325507640838623, 0.6439172625541687, -2.1713194847106934, -0.15437822043895721, 0.42472043633461, 0.36892104148864746, -3.723775863647461, 1.6830580234527588, -0.5160670876502991, -0.40127646923065186, 3.132807493209839, 0.37509918212890625, -1.693938970565796, -2.232372760772705, 0.3077991008758545, -0.2078140377998352, 4.369046211242676, 1.7424299716949463, 0.6370692253112793, 1.4767619371414185, 1.309735894203186, -0.789763867855072, 0.06957559287548065, 0.8295282125473022, -0.3244084119796753, -0.5412315130233765, -0.5280601382255554, 2.71058988571167, 2.221879482269287, 0.6213482618331909, -0.28921929001808167, -0.2706962823867798, 3.3682661056518555, -2.008812427520752, -1.955212116241455, 0.048275649547576904, 0.26868945360183716, -1.4191148281097412, 0.0270082950592041, 1.7952589988708496, 1.5668861865997314, 2.8210549354553223, -0.6900965571403503, 1.6293036937713623, 0.6840877532958984, -1.0173323154449463, -1.39803147315979, 2.6014621257781982, -0.4744863510131836, -1.8350484371185303, -0.08179435133934021, -0.8605421781539917, 1.4120457172393799, -2.226524829864502, -2.1023316383361816, -0.02072054147720337, 0.05197450518608093, -1.5462377071380615, -0.9870449900627136, -1.5643483400344849, -0.5846542716026306, -0.9623898863792419, -1.632262945175171, -0.17209485173225403, -0.9973049163818359, 0.87776780128479, -1.9401969909667969, 2.2059364318847656, 2.058380126953125, -0.008315324783325195, 2.1828360557556152, 0.800510823726654, -1.0506985187530518, -1.0030490159988403, 1.4233726263046265, 0.8824570178985596, -0.18305908143520355, 0.8390011787414551, 0.06417489051818848, 0.056101083755493164, 2.436021327972412, -0.5848153829574585, 1.2044098377227783, 0.7079078555107117, 1.096110463142395, -2.2527759075164795, -0.7161047458648682, 0.32493019104003906, -1.4479529857635498, -1.8868181705474854, -0.11033523082733154, 0.8314059972763062, 1.2276982069015503, -1.0856298208236694, -1.0109070539474487, 2.259716510772705, -1.3627724647521973, 0.7045735716819763, 2.5156922340393066, 0.8374810814857483, -0.7102168798446655, -0.08584277331829071, 0.23966646194458008, 1.5339322090148926, -1.173632025718689, -0.7447669506072998, 0.8569315671920776, 2.062760353088379, -0.35233861207962036, 0.5410271286964417, 2.364954948425293, -0.19538259506225586, 2.2458581924438477, -3.183121681213379, 1.3639731407165527, -0.5111443996429443, 1.4307669401168823, -0.5220174789428711, -1.3135977983474731, -1.1979566812515259, 1.6957595348358154, -2.251532793045044, 0.8566809296607971, -1.320289134979248, 2.610935926437378, -0.03581485152244568, -1.3511360883712769, 0.02745974063873291, 1.5576773881912231, -1.8294477462768555, 1.8953688144683838, -1.2118786573410034, 1.005607008934021, -1.4674575328826904, -1.4073209762573242, -1.6089123487472534, 2.106477737426758, -0.02180767059326172, 0.029816150665283203, 1.5999573469161987, -0.46245694160461426, -0.6004041433334351, -1.6405689716339111, -0.7688212394714355, 0.07252931594848633, 2.141629934310913, -0.2749289572238922, 2.1484124660491943, -0.6531776785850525, -1.7655664682388306, 0.9738287925720215, 1.0760128498077393, 0.49324658513069153, 1.3335387706756592, -1.2274948358535767, -1.4541370868682861, -0.09917383641004562, -1.8251878023147583, 1.117348313331604, 0.1185464859008789, -1.1399005651474, -0.13609465956687927, 2.240966796875, -1.4189831018447876, 2.4256162643432617, -0.41691380739212036, -0.8174529075622559, 0.3870351314544678, -1.6526954174041748, -0.6597274541854858, -1.758790135383606, -0.23101848363876343, 0.9355627298355103, 1.1278108358383179, -2.4237189292907715, 0.21133339405059814, 1.6563138961791992, 0.15395021438598633, -1.836195468902588, -1.4258942604064941, -0.8762872815132141, 0.6157704591751099, -1.8931012153625488, -0.7780609130859375, 0.4612143635749817, -1.0442352294921875, 1.1704504489898682, -3.4872939586639404, -0.910866379737854, -0.06844335794448853, 0.06261098384857178, -1.0625824928283691, -1.3456923961639404, -0.5013424158096313, -0.5890400409698486, 0.017514705657958984, 0.02653968334197998, -0.11427140235900879, -1.7835252285003662, -0.926497220993042, 0.5250195860862732, -1.2755900621414185, -0.5367444753646851, -1.9091911315917969, 1.5333657264709473, -2.0228235721588135, -2.500941753387451, -1.8021268844604492, 1.5078444480895996, 1.0195262432098389, -2.306942939758301, -0.8177671432495117, -0.5264295339584351, -0.8129326701164246, -0.915533721446991, -0.2948261797428131, -2.4406986236572266, 0.8881641626358032, 0.45804136991500854, 0.17317986488342285, -0.5971868634223938, -1.0299715995788574, -1.105919361114502, 2.0190846920013428, -0.361108660697937, -1.129691243171692, 1.6867974996566772, -1.2970612049102783, 2.4876763820648193, -0.9051176309585571, -2.0508923530578613, -1.1144483089447021, -0.6325156688690186, -0.5074882507324219, 1.9965405464172363, -1.58744215965271, 1.6210359334945679, -0.1303078532218933, -1.1056616306304932, -0.23551177978515625, 1.3429811000823975, -1.0935919284820557, 1.2621562480926514, -1.7946032285690308, -0.3653581142425537, 0.9736729264259338, 0.9629085659980774, -1.5603605508804321, 2.2884063720703125, 0.8338909149169922, 0.5190176963806152, -1.0982334613800049, 2.473963737487793, -2.6126928329467773, 3.362316608428955, 0.8578869104385376, -0.165706604719162, -0.07101237773895264, 2.526756763458252, 1.8753881454467773, -0.05466008186340332, -1.2670907974243164, 0.26505881547927856, 0.3436310887336731, 2.0015130043029785, -1.0163531303405762, -1.113653302192688, 0.6380964517593384, -0.7363044619560242, 1.1010949611663818, -1.3825796842575073, 0.4753059446811676, 0.715678334236145, -1.1274991035461426, -0.33087480068206787, 0.5190978050231934, -2.5228943824768066, 1.8212547302246094, 0.9871343374252319, 1.0869165658950806, -3.1245179176330566, -0.5577770471572876, -1.1187149286270142, 1.7315963506698608, -1.0141704082489014, -0.07994991540908813, -0.4393918514251709, -0.6092690229415894, 2.5390961170196533, -0.1669406145811081, 1.4771966934204102, -1.9859230518341064, -2.4119794368743896, 0.7047854661941528, -1.5668227672576904, 0.005881309509277344, -1.5307155847549438, 0.33274877071380615, -1.1103729009628296, 0.498698890209198, -0.9268819093704224, -2.3713223934173584, 2.5209038257598877, -1.5519217252731323]], "dtype": "float32"}}, {"a": [{"__nd__": [1341.0, -159.25, 465.75, 1742.0, 1239.0, -48.46875, 1237.0, -431.75, 154.5, 135.0, -1485.0, 1365.0, 2610.0, 112.125, -461.25, 759.5, 581.5, -410.0, 300.25, 1841.0, 68.6875, -576.5, -921.0, -623.5, -526.0, -1501.0, -293.5, -12.5, 1182.0, 25.671875, 949.0, 314.0, -746.5, 143.25, -494.75, 1188.0, -97.0, -992.0, 923.5, 159.25, 438.5, -171.875, -80.0625, 584.0, -2458.0, 838.0, -1380.0, 1671.0, 1948.0, 1283.0, -308.5, -833.0, 341.25, 605.0, -667.0, -154.5, -250.5, -316.5, -430.25, 78.4375, -906.0, -152.0, 690.0, 386.5, 1708.0, -30.71875, 371.5, -204.375, -1039.0, -186.125, -825.0, -1208.0, -666.5, 303.75, -479.75, -1205.0, -290.5, -1837.0, -1382.0, 1109.0, 418.25, -694.5, 383.5, -22.125, 368.75, -1503.0, 570.0, -44.78125, -170.125, 39.03125, 1228.0, 361.5, 362.75, 1008.0, 2854.0, 933.5, -78.1875, 12.828125, -874.5, 723.0, -22.984375, -264.75, 128.5, -322.0, -2230.0, 854.0, 832.5, -24.390625, -1045.0, -822.5, 925.5, -2120.0, 1506.0, 125.5, -575.0, 722.0, -1578.0, -393.75, 185.875, 732.5, -990.5, 636.0, -1036.0, -342.5, -688.5, -104.4375, 45.40625, 191.375, -254.875, -709.5, -203.25, 157.5, -392.0, 126.3125, 1665.0, -1032.0, -334.5, 196.125, -1647.0, 595.5, -91.5625, -1505.0, -355.75, 448.75, -1138.0, 65.0625, -204.125, -711.5, -889.5, -1888.0, 2450.0, -1000.0, 1201.0, -1663.0, -1312.0, -573.0, 317.0, 189.75, 1062.0, -319.5, -281.25, 179.875, 488.0, 153.625, -32.96875, -107.4375, -986.0, 617.0, -604.0, -436.5, 1154.0, 532.0, 1138.0, 1806.0, 188.75, 1911.0, 1651.0, 381.5, -136.375, -455.25, -50.46875, -1836.0, 463.5, -397.75, -869.5, -690.0, -1557.0, 930.0, 1553.0, 703.5, 204.875, 1469.0, 120.9375, 128.625, -1490.0, -762.5, 232.625, 673.0, -30.078125, 1114.0, -685.0, -422.0, -169.875, 2624.0, 27.078125, -1178.0, 475.75, 1121.0, -628.0, 331.0, 99.1875, 1717.0, -774.5, -915.5, -86.5, -2292.0, 182.5, 1350.0, -1609.0, 902.5, 756.0, 2508.0, -732.5, -139.625, -790.5, 531.0, -824.5, -220.625, -200.375, -178.375, 286.5, -911.0, -422.75, -113.9375, 659.5, 365.0, -102.8125, 459.75, -282.5, -152.125, 2094.0, -106.1875, -422.5, -779.0, 1319.0, 1069.0, 890.0, -1717.0, 930.0, -136.5, 907.0, 395.25, -791.5, -1453.0, 576.5, -424.25, 1780.0, 111.3125, -145.375, -1798.0, -468.75, 1849.0, -172.75, 1083.0, 309.25, 559.5, 74.125, -91.375, 187.125, -31.28125, -248.5, 897.5, 350.25, 233.25, 440.75, 871.0, 182.25, -28.53125, 2198.0, -157.5, 188.75, -710.0, -569.5, 88.0, 623.5, -45.25, -646.0, 107.875, 276.75, 1600.0, -1014.5, -443.25, 542.0, -1410.0, 532.0, -74.9375, -1220.0, 1790.0, 477.25, 1912.0, 2232.0, 209.125, 399.5, -1583.0, -283.25, -139.75, -1193.0, -672.0, 1648.0, 29.65625, 1110.0, -1043.0, -1953.0, 2108.0, -62.3125, -1633.0, 1091.0, 802.0, 1046.0, -1346.0, 45.09375, -1088.0, -1550.0, 334.75, -979.0, 491.25, -409.5, 273.75, -1765.0, 797.5, -233.875, -784.5, 571.5, -2428.0, -1293.0, -361.0, -487.5, 556.0, -575.5, 719.0, -928.0, -81.25, -639.0, -563.5, -489.5, 662.5, 1067.0, -275.0, -1999.0, 470.25, 135.75, -559.0, -1024.0, 1431.0, -2176.0, 303.5, -1257.0, -180.25, -384.0, -766.0, 400.75, -324.25, 800.5, 105.5625, -704.5, 1349.0, -220.0, -155.875, 183.375, 940.0, 1387.0, -118.9375, -796.0, -1428.0, 1001.0, 205.75, 859.5, -89.5625, -585.0, 2094.0, 200.25, -985.5, 1059.0, -264.0, -83.9375, 969.0, 656.5, 302.25, 2031.0, 1009.0, 996.5, -57.96875, -474.0, 710.5, 1498.0, 325.5, -113.875, 142.25, -1174.0, 57.15625, 12.65625, -429.75, -1152.0, 426.75, 366.5, 362.5, 997.5, 403.0, -1146.0, -1069.0, -1121.0, 1650.0, -1429.0, -360.25, -334.5, -2196.0, -39.6875, -1116.0, -552.5, 262.25, 405.25, 508.25, -1397.0, -26.71875, 1375.0, -1085.0, 1005.0, 173.5, 671.5, -1408.0, 169.75, -953.0, 1025.0, 1109.0, -1582.0, -47.375, -1302.0, 491.0, -1599.0, -111.25, -199.375, -685.0, -1089.0, -749.0, 511.25, 469.75, 311.5, -1445.0, -158.875, 1.7705078125, -1347.0, -1005.0, -543.0, 847.0, 1043.0, -603.5, -2442.0, 290.25, -208.5, -128.0, -937.5, -206.0, -890.5, -841.0, -1119.0, -986.5, 1559.0, -18.90625, -1463.0, -397.75, 297.5, -973.5, -1595.0, -58.3125, -94.3125, 3350.0, -1227.0, -1748.0, 349.75, 365.0, -809.5, -771.0, 260.25, 14.15625, 1664.0, -442.25, 676.5, 639.5, 221.125, -792.0, 333.5, 693.0, -71.3125, 304.5, -48.84375, -1208.0, -1109.0, 1956.0, -2330.0, -749.0, 585.5, 1383.0, 766.5, -1190.0, 373.0, 532.0, 654.5, 1646.0, -1791.0, -1458.0, 1342.0, -1207.0, -88.9375, 1055.0, 1273.0, 789.5, -116.9375, 2232.0, 838.5, -1103.0, -1401.0, -286.5, -715.5, -293.75, 1185.0, -1089.0, 58.90625, 1216.0, -1330.0, -1039.0, 216.375, 1016.5, 1159.0, 334.5, 250.25, -1698.0, -423.75, 1826.0, -1390.0, 811.5, 836.0, 799.5, -217.125, 391.25, -110.875, 367.25, -807.0, 475.75, 1161.0, 593.5, 301.75, 183.125, 2822.0, 1650.0, -1426.0, 924.5, 1310.0, -304.0, 476.75, 78.0625, 1001.0, 194.625, -645.5, -486.0, -729.0, 1870.0, 1973.0, 532.0, 1501.0, 863.0, -80.75, -931.0, 1158.0, 1399.0, 349.0, 1271.0, 2468.0, 321.75, -573.5, 535.0, -145.0, 504.0, 119.3125, 326.5, -899.5, 1537.0, 1060.0, 432.25, 1008.0, -577.0, -873.0, -1246.0, 1519.0, 943.0, 1158.0, 353.25, 218.5, 1046.0, -1141.0, 559.0, -202.875, -1414.0, 1519.0, -84.4375, 523.5, -455.25, -1720.0, -229.375, -198.875, 478.75, 267.0, -527.5, -806.5, -278.25, 1687.0, 635.5, 984.5, -36.0, -1483.0, 911.0, -161.5, 1023.5, -302.0, 2066.0, 1913.0, 1335.0, -377.75, 698.0, -1485.0, 746.5, -825.0, 1088.0, 630.5, -1002.0, 903.0, -1837.0, -213.875, 368.25, -469.25, 715.5, -1.4951171875, -35.5, -853.5, 179.75, -2330.0, -994.0, 1320.0, 512.0, -5.08984375, -717.5, 2230.0, 1249.0, 645.5, -1819.0, -121.6875, -904.5, -224.625, -1126.0, -94.8125, 585.0, 705.0, -1071.0, 912.5, -805.0, 214.875, -1027.0, -441.0, 423.0, -231.75, 2008.0, 2294.0, -506.75, 592.5, -1258.0, 10.3125, -1946.0, 456.5, 732.5, -1771.0, 98.3125, -827.5, -110.75, -573.0, 2510.0, -321.75, -1164.0, -605.0, 677.5, 230.75, -1179.0, -340.0, 406.25, -2336.0, 584.0, -888.5, 171.25, 946.5, -854.5, 1066.0, 14.2578125, 494.75, 1267.0, -179.375, 1387.0, 228.875, 537.5, 1085.0, -144.375, 27.875, -1475.0, 293.25, 78.625, 626.0, 75.25, 1073.0, -2506.0, 897.5, 927.0, 1690.0, 50.0625, -216.0, 731.5, 1812.0, -97.625, -270.0, 113.75, 449.75, -3.90234375, 311.75, -628.5, 2164.0, 1582.0, 784.5, -1110.0, -869.0, 1165.0, -1332.0, 1827.0, -754.0, 1084.0, 1900.0, 278.75, 1716.0, 1988.0, -1311.0, 776.5, -1588.0, -543.0, -543.0, 722.5, 1270.0, 518.5, 1272.0, -1937.0, 1923.0, -567.5, -925.0, 43.75, 196.0, -668.5, 295.0, -706.5, 719.0, 168.125, 155.375, -145.0, -541.5, -34.5625, -292.75, 737.0, -169.875, -1664.0, 229.125, -315.0, -753.0, 906.0, 1022.5, -463.75, -229.25, -1182.0, 1447.0, -556.5, -1334.0, 749.5, -491.25, -1545.0, -1582.0, 757.5, -989.0, -702.5, -206.5, -610.0, 1193.0, 34.15625, 155.375, 492.0, -65.625, -1104.0, -1071.0, 1689.0, -148.5, -315.5, -1930.0, -127.3125, 1293.0, -1228.0, 1149.0, -443.5, 259.5, -949.0, -938.5, 665.5, 1499.0, -330.5, -1412.0, -982.5, 601.0, 1140.0, 1431.0, 699.5, -457.25, 977.0, -977.5, 758.0, 652.5, 328.0, -2104.0, -619.0, -69.5625, -426.0, -535.5, 1312.0, 1400.0, -239.625, 892.5, -895.0, -622.0, 341.5, 1300.0, 1095.0, 13.21875, -588.0, -1042.0, 472.75, -733.5, -286.0, 146.625, 522.0, -669.5, -1460.0, 1541.0, 905.5, 1485.0, 1556.0, -219.125, -421.25, -756.0, 1921.0, -856.0, 1077.0, -1451.0, 1442.0, 1226.0, 1337.0, -308.25, 756.0, 782.0, 155.5, -780.5, 733.5, -2156.0, -629.0, 225.875, 920.5, 453.75, -962.0, -116.75, 386.25, -357.5, -1058.0, -260.5, -522.0, -4.41015625, 207.125, 1969.0, -369.25, 736.5, 773.5, -293.25, 1329.0, -1396.0, -619.0, -813.5, -1222.0, 744.5, -100.125, 17.96875, 784.5, 595.5, 1347.0, -836.0, -418.0, -451.75, 646.5, 449.5, -1064.0, 843.0, 651.0, 2234.0, 278.0, 642.0, 465.25, 13.359375, -859.0, -22.890625, -119.125, 1444.0, -345.75, -2300.0, -1357.0, -24.03125, -368.25, 614.5, -811.5, -1519.0, 1010.5, -1656.0, -1589.0, -808.5, 427.75, 104.9375, 463.75, -508.25, 1970.0, -1319.0, 97.625, -683.5, -1022.5, -669.0, -357.75, 626.5, 564.5, -848.0, -1493.0, 855.5, -214.25, 1106.0, -1048.0, 558.0, 171.25, 100.9375, 379.5, -811.0, 358.5, -1134.0, 56.6875, 1501.0, -828.0, 12.90625, -489.25, -1795.0, -213.5, 2372.0, -583.0, -1393.0, -171.875, 579.0, -722.0, -515.5, -1098.0, 951.0, -892.5, -1343.0, 372.0, 750.5, 769.0, -595.5, 533.5, 940.0, 189.625, -69.125, -1232.0, 191.625, 449.0, -430.5, 289.5, -311.0, 832.0, -193.375, 1808.0, 3086.0, 958.0, -551.0, -730.5, -130.5, 118.9375, 327.75, -361.0, -1747.0, -1510.0, -941.0, -1625.0, -1281.0, -1395.0, 454.0, 1071.0, -1593.0, 759.0, 995.0, 148.0, 185.5, 37.125, 308.75, -1090.0, -419.0, 184.625, -1113.0, 102.5, -936.0, 615.0, 150.25, 1332.0, -304.5, 196.375, 644.5, 800.5, 907.0, 1178.0, -2266.0, 901.5, 413.75, -880.0, -257.25, -520.0, -660.0, -225.875, 360.25, 846.0, -325.0, -264.75, 1226.0, 1727.0, 110.125, 884.5, -389.5, 880.5, -498.0, 426.5, -1180.0, 1813.0, -906.0, -281.25, 75.125, -662.5, -580.5, -87.75, -1026.0, 858.5, -456.5, -59.0625, -1107.0, -1475.0, 174.75, 567.0, -1071.0, 119.125, 130.5, 124.6875, 1328.0, 701.5, 1298.0, 352.25, -682.5, 528.0, 1396.0, -676.5, -951.5, -14.6796875, -237.0, -1204.0, -321.75, 1146.0, -306.25, -1010.0, -595.0, -72.875, -203.875, 0.7431640625, 1268.0, 103.125, -214.875, 187.125, -6.125, -369.5, 167.125, 470.5, 481.0, -2828.0, -565.5, 275.5, -224.5, 452.25, 1164.0, -1988.0, 1514.0, -296.75, 1685.0, -671.0, 850.0, 3462.0, 786.5, -252.0, -874.0, 1721.0, 2206.0, -8.1640625, 1561.0, 300.25, -864.0, -54.34375, -661.0, -415.5, 1993.0, 1134.0, 1131.0, -469.5, -271.25, 342.0, -1007.0, 2626.0, -75.3125, -2286.0, 325.0, -781.0, 320.0, 151.0, -1445.0, -1225.0, -876.0, -1503.0, -1885.0, -303.5, -229.125, -486.5, -286.25, 1112.0, 60.5625, 5.6328125, -1965.0, -53.6875, -1214.0, 470.25, 1988.0, 368.75, -1005.0, 1522.0, 221.375, -238.875, -181.625, -135.875, -18.734375, 1584.0, 1051.0, 664.5, 820.0, -2652.0, 548.0, 280.25, -698.0, -1433.0, 39.40625, 1549.0, 899.0, -637.0, 2998.0, -700.5, 1183.0, -561.0, -149.625, -1458.0, -969.0, -168.75, 27.6875, 1288.0, 472.25, 326.5, 1814.0, -657.0, 1402.0, 227.25, -1884.0, 203.875, -1093.0, 215.625, 21.265625, -632.5, -1069.0, 167.125, 106.1875, -1197.0, 1009.0, -137.0, 672.0, -416.5, -172.75, 1173.0, -796.5, 227.0, 1875.0, -617.0, 860.0, 1247.0, 1386.0, -661.0, 1212.0, -1139.0, -1008.0, 1184.0, 233.5, 304.5, 1047.0, -63.78125, 796.5, -993.5, -303.0, -282.0, -396.75, 1865.0, -613.5, 5.9140625, 946.5, -416.25, 1439.0, 20.71875, -540.5, 871.5, -24.0, -840.0, -1574.0, 1327.0, 430.5, -202.625, 251.0, 292.75, 505.0, -515.5, 390.5, 988.0, -1372.0, 501.5, -17.90625, -146.375, -1746.0, 571.0, 1261.0, -630.5, 778.5, 348.25, 636.0, 1027.0, -438.75, 149.375, 786.0, -1133.0, -2130.0, 858.5, -27.953125, 1476.0, -121.375, -1632.0, 1374.0, -438.75, 1774.0, -1049.0, -1209.0, -264.5, -223.5, 1041.0, -791.5, 433.25, -69.0, 550.0, -252.875, 796.0, 1188.0, 1720.0, 573.0, 1809.0, -198.125, -122.125, 345.5, -1569.0, -155.75, -1455.0, -984.0, -766.5, -416.0, 1014.5, 575.0, -1434.0, -401.25, -207.75, -1646.0, 357.25, -260.5, -85.1875, 459.0, 1789.0, -304.5, -366.0, -322.75, 85.0625, -1318.0, 792.0, 1968.0, -931.0, -1181.0, -1017.5, 1707.0, 1590.0, -1312.0, 935.5, -297.5, -311.0, 378.75, 249.5, -211.625, 781.0, -1488.0, -1456.0, -541.5, 1380.0, 1177.0, 420.5, 782.0, -1875.0, -222.75, 1169.0, -252.875, 273.0, 68.25, -366.0, -563.5, 844.0, -265.5, -1474.0, 982.0, -242.0, -158.75, -470.75, 1457.0, -373.25, 1006.0, 771.5, -1212.0, -558.5, -12.5546875, -208.5, 1586.0, -1292.0, 376.25, 367.5, -1601.0, 1107.0, -186.25, 1189.0, -5.1875, -700.0, 45.09375, -1879.0, 908.5, -178.125, 267.0, -1238.0, -882.0, -17.25, 181.5, -2410.0, -763.5, -1711.0, 149.625, 594.0, 1071.0, 165.125, -1896.0, 659.5, 1469.0, 355.25, 845.0, 130.625, 206.875, 46.21875, 846.0, 441.0, -498.75, -228.25, 676.5, -118.75, -215.5, -1008.0, -500.5, -1638.0, 726.0, -1183.0, 335.5, -235.625, -1450.0, -1485.0, -54.8125, -1042.0, 1390.0, 766.5, -286.75, -1658.0, -386.75, 109.875, 2566.0, -295.75, -142.0, 189.875, 328.0, -1149.0, 326.5, -1259.0, 758.0, 2156.0, -548.0, -1129.0, -630.0, 172.875, 1093.0, -934.0, 632.0, 1624.0, 891.5, 774.5, -243.375, 365.0, -872.5, 891.0, 1028.0, 834.5, -895.5, 1018.5, 1433.0, -780.0, 39.96875, -50.75, 1245.0, 323.75, 1097.0, -19.40625, -985.5, -827.0, -150.375, 1182.0, -2910.0, -238.5, 1057.0, -843.0, -279.25, 1959.0, -864.5, -380.75, 1185.0, -2035.0, 1077.0, -421.0, 288.25, -54.15625, -1468.0, -974.5, 1973.0, -1944.0, 0.8193359375, -612.0, 349.75, 747.0, -1951.0, -1517.0, -815.5, 321.0, -3572.0, -1648.0, -558.0, -1429.0, 392.75, -1217.0, 61.15625, -547.0, 595.0, 2798.0, -1081.0, -887.0, 1897.0, -433.5, -941.0, 593.5, -426.25, 889.0, 316.75, 764.0, -860.5, -685.5, 376.75, 994.0, 548.0, -63.28125, -736.5, 42.9375, -704.0, 564.5, -722.0, 125.375, 17.125, -112.4375, 360.75, 147.0, -1258.0, 704.5, -724.0, 310.5, 156.0, -929.5, 393.0, -1574.0, 635.0, -908.0, 1870.0, 856.0, 1247.0, 653.5, 1056.0, -1297.0, -794.5, 751.0, 1092.0, 41.71875, 1648.0, -1099.0, 1420.0, -636.5, 1713.0, -484.0, -736.5, 1275.0, 59.3125, 1658.0, -631.5, 578.5, -819.5, -1096.0, -917.0, -74.9375, 753.5, 869.0, 375.5, -185.875, 901.5, 1585.0, -316.0, 1729.0, 5.078125, 804.0, -605.5, 668.5, 76.6875, -839.5, 775.0, -1261.0, -719.5, 105.9375, 763.5, 1698.0, 1143.0, -1750.0, 1274.0, -1177.0, -241.75, -274.0, 740.0, -944.5, -2002.0, 915.5, -1193.0, -438.5, -957.0, -789.0, 369.25, -572.0, -466.75, 20.234375, 307.25, -2372.0, 565.5, 398.5, -944.5, -1161.0, 2338.0, 219.5, -19.40625, -1537.0, -2068.0, 188.0, 879.0, 991.0, 1318.0, -337.75, 728.0, -366.5, -870.5, -892.5, -1218.0, 497.0, -604.0, 1128.0, 1054.0, 2220.0, 423.0, -1479.0, 1010.0, -732.5, -764.5, 192.75, -932.5, 300.5, -1832.0, -876.5, 1100.0, 892.0, 177.625, 234.75, -866.5, -963.0, -333.0, 429.75, -34.625, -1274.0, 561.0, -984.0, 212.125, 884.5, 943.5, -325.75, -300.75, -963.0, 918.0, -455.25, 64.9375, 1256.0, 1184.0, -636.0, 601.5, 781.5, 2142.0, 1891.0, -23.828125, 1093.0, -127.3125, 485.0, -2200.0, -549.5, 477.25, 177.375, 330.5, 1126.0, 150.75, 2588.0, -661.5, -312.5, -311.0, -1847.0, -1430.0, -168.375, 435.0, 609.0, -900.5, -1147.0, -1706.0, -857.0, -1130.0, 201.375, -1258.0, 1021.5, 1204.0, -161.375, -465.5, 485.75, -984.0, -312.0, -762.0, -314.75, -124.3125, 104.625, 347.5, -1557.0, 1034.0, -1291.0, 872.5, 93.0, 743.0, 211.25, -332.0, -1248.0, -308.75, -3270.0, -328.25, -369.25, 1134.0, 580.0, 1488.0, 6.87890625, 397.75, 549.5, 701.0, 442.75, -655.5, -163.5, 1310.0, 214.75, -1112.0, 1444.0, -721.5, 463.0, 366.5, 950.0, 2444.0, 246.0, 805.0, 1604.0, -1382.0, 1413.0, 1045.0, 1011.5, 206.25, 808.5, 1111.0, -175.75, -1052.0, 1349.0, -435.5, -472.0, -925.5, -1.5888671875, 177.375, 92.9375, -44.4375, 957.5, -764.0, -370.75, -147.0, -656.0, -2508.0, 614.5, -746.0, -686.0, 1532.0, -2152.0, 404.25, 1795.0, 1892.0, -1139.0, 738.0, 1096.0, 2558.0, -1074.0, 845.5, -990.5, -548.5, 1192.0, 661.5, -368.25, 1295.0, 50.6875, -1574.0, 660.0, 552.5, 453.75, -398.5, -465.75, 148.875, -15.3046875, 2080.0, 344.25, 658.5, -238.625, -1230.0, -738.5, 327.0, 1365.0, 505.0, -127.0625, 60.78125, -1125.0, -249.875, -1299.0, -1545.0, -1272.0, 247.625, 103.0, 451.5, -1.318359375, 1274.0, -10.90625, -737.0, 1025.0, 664.0, 282.75, -921.0, -1263.0, -682.0, -21.546875, 810.0, 15.5078125, -1052.0, 1123.0, 1058.0, -1062.0, -1308.0, -49.78125, 713.0, -2268.0, -1610.0, 366.5, 752.5, 655.5, 1193.0, -20.78125, -150.75, -541.5, 1956.0, 1242.0, -549.5, -348.0, 1656.0, -1135.0, 424.0, -433.75, -1225.0, -857.5, -45.03125, 205.75, 428.0, 892.0, -504.5, 664.5, 1624.0, 454.5, -1034.0, -369.75, -643.5, -1821.0, -296.25, 326.5, -2588.0, -298.0, -1666.0, 443.25, -1748.0, -628.5, -1475.0, 505.75, 1663.0, -515.5, 786.0, -1649.0, 585.5, 1375.0, 963.5, 664.0, 640.0, -74.5, 1754.0, -1527.0, 614.5, -108.0625, -429.5, 733.0, -149.25, -1209.0, 375.5, 2128.0, 175.625, 1093.0, -945.5, -380.75, -1019.5, 10.421875, 1030.0, -454.5, 769.5, -185.25, -1022.5, -686.5, 444.0, -2162.0, 447.25, -638.5, -236.0, -502.5, -1138.0, -256.0, -160.25, -1287.0, 1479.0, -2092.0, 88.125, -183.5, -632.0, -367.75, -585.0, 870.0, 1714.0, 653.0, -540.0, 1268.0, 175.0, 41.75, -844.5, -31.015625, 2048.0, -246.875, -239.875, -897.0, 2196.0, 867.5, -1288.0, 186.25, -573.0, -2376.0, 1597.0, 128.5, 767.0, 173.375, 1242.0, 601.0, 124.375, 785.0, 347.25, 207.25, 74.5625, -1249.0, 228.625, -1392.0, -1667.0, 921.5, 509.75, -782.5, 694.5, 538.5, 462.0, -412.75, -118.1875, -1505.0, 1714.0, 2560.0, -765.0, -516.5, -192.5, -127.5625, 1061.0, -502.5, -278.0, -2068.0, -181.5, 769.5, 607.0, -1102.0, -726.5, 670.5, -232.75, 292.75, 489.75, 145.375, 59.875, -1417.0, 983.0, 384.0, 1057.0, 298.0, -621.0, 608.5, -1577.0, -1260.0, -32.4375, -583.0, 1475.0, -408.5, 252.5, 393.0, 880.0, 1693.0, -1309.0, -179.0, 437.5, -1011.5, -2342.0, 47.125, 47.40625, 590.0, 654.5, 1286.0, -74.3125, -22.796875, 379.25, 1097.0, 528.5, -731.5, 836.0, -1040.0, 472.5, -1511.0, 1290.0, 1006.0, 1851.0, -808.5, -711.5, 11.765625, -310.0, 293.75, -541.5, 938.0, 21.265625, 1110.0, -226.75, 330.25, 149.25, 471.75, -723.5, 515.0, -29.15625, -469.5, 662.5, -1434.0, 1473.0, 1132.0, -982.5, -2492.0, 638.5, 869.0, 2276.0, 415.25, -601.5, -83.25, -752.0, 274.25, 16.34375, -1397.0, -815.0, -23.78125, -709.5, 289.5, -771.0, -644.5, -539.5, -550.0, -1794.0, -1555.0, -1551.0, -302.25, -826.5, 1106.0, -924.5, -1385.0, 476.0, 1169.0, 1188.0, 88.0625, -1445.0, 118.6875, -281.5, 230.875, 740.0, 312.5, -64.875, -677.0, -829.5, -569.5, 981.5, 579.0, 250.5, 709.5, 824.5, 125.25, 35.1875, 816.0, 1697.0, 1029.0, 605.0, 1892.0, -1318.0, 1167.0, -867.0, -414.5, -1959.0, 549.0, 994.5, 738.5, -180.875, -134.625, 454.25, -1095.0, 1286.0, -253.625, -1092.0, -620.0, 623.0, 733.5, 1015.5, -1656.0, -169.25, -723.0, -315.0, -1567.0, 306.0, 643.0, 1234.0, 1040.0, -59.40625, -932.0, 115.8125, -931.0, 1162.0, -194.75, 936.5, 466.5, 1253.0, -436.5, 11.15625, 1335.0, -1256.0, -107.25, 87.375, 1223.0, 2756.0, -1489.0, 1950.0, -1700.0, -325.25, 1249.0, 1228.0, -1226.0, 674.5, 165.625, 114.8125, 684.5, 699.5, -330.25, 1149.0, -554.0, -1681.0, -1636.0, -1409.0, 1503.0, -987.5, 935.0, -37.4375, 440.25, 47.75, 1001.0, -782.5, -3158.0, -1695.0, 1836.0, -2220.0, -101.875, -151.625, 414.25, -1319.0, -205.5, -850.5, 1139.0, 1268.0, -2228.0, -1928.0, 990.0, -966.5, 185.625, 11.3515625, -1507.0, -126.3125, -355.75, 36.34375, 881.0, -878.0, 1045.0, -714.0, 387.25, 305.25, 594.0, -2702.0, -1120.0, -1217.0, 862.5, -1192.0, 1091.0, -1525.0, -806.0, 2330.0, 1.46484375, -416.75, 1481.0, -862.5, -146.875, 480.25, -1286.0, 824.0, 138.125, 1698.0, 531.0, 76.6875, 1104.0, -2020.0, -1220.0, 1059.0, 837.0, -494.75, 1388.0, 643.0, 281.5, -625.0, 725.5, 904.0, -57.84375, 310.0, -45.28125, 1089.0, -211.5, -422.0, 312.25, -44.0625, -185.0, 249.875, -956.0, 61.84375, 91.125, -2832.0, -1123.0, -2180.0, -2152.0, -920.0, -876.0, -355.0, -713.0, -1300.0, 267.25, 1782.0, 456.75, 1524.0, -155.25, 198.375, -852.0, -147.125, -1455.0, 82.0625, 729.0, -1575.0, -1272.0, 270.75, 714.0, -62.46875, 343.25, -1530.0, -449.75, 255.875, 568.0, 465.0, 768.0, -772.0, -948.5, -468.25, -1648.0, 738.5, -1487.0, 1097.0, 119.375, 563.0, 548.0, -319.25, -472.25, 1068.0, 555.5, -206.0, 294.25, -30.484375, 318.75, 550.0, 804.5, 619.0, -1798.0, 987.0, 491.25, -624.0, 308.5, 53.5625, 25.65625, 329.0, -224.875, -1260.0, 2472.0, 573.5, -489.5, 1136.0, -1056.0, -359.25, 570.5, -517.5, 1102.0, -236.875, 1165.0, 285.5, -1601.0, -558.5, 673.5, -1823.0, 19.5625, -2276.0, -1837.0, 207.125, 331.75, -656.0, -1484.0, -1845.0, 2458.0, -322.5, 293.0, 1941.0, -60.5, 547.5, -305.25, 108.75, -354.5, 1104.0, -85.1875, -316.25, 280.75, 141.125, -868.5, 427.0, 96.5625, -911.0, -457.0, 1145.0, 143.875, -520.5, -609.5, -1010.5, -746.5, 816.0, 482.0, 412.5, -756.0, -106.3125, -317.75, -807.0, 382.25, 879.5, 441.75, -1111.0, 1267.0, 372.75, 739.5, -443.25, 193.375, 717.0, -1054.0, 1501.0, 875.0, -1409.0, 411.5, 434.0, 457.25, -1333.0, -2064.0, -235.625, -717.0, -546.0, 975.0, 1044.0, -1658.0, 1028.0, -439.25, -1415.0, -223.625, -161.375, -192.75, -84.75, 741.5, 38.03125, -927.5, -838.5, 2582.0, -729.5, 761.5, -30.234375, 524.0, -937.0, 129.875, -169.25, -256.25, 549.5, 136.875, 923.0, 57.5, 1435.0, 60.75, 281.0, 153.25, -1024.0, 2590.0, 991.5, -1657.0, 1193.0, -1316.0, -793.5, 52.03125, 1813.0, -616.5, -40.96875, -361.0, -673.0, -1037.0, 423.5, 620.5, 1471.0, -820.5, -2392.0, -96.25, 1402.0, -36.03125, -4.4765625, -1421.0, 917.5, -345.25, -307.0, -249.5, -236.125, -473.0, -802.5, 1444.0, 1352.0, -1116.0, -1051.0, -316.75, 1696.0, 1184.0, 65.875, 537.5, 568.0, 1659.0, 450.0, 706.0, 359.5, -749.0, -434.25, -40.84375, -1155.0, 371.5, 286.25, 348.25, 438.0, -1651.0, 966.0, -1040.0, 1217.0, -209.25, -883.5, 366.5, -759.5, 698.0, -36.0625, -1112.0, 475.25, 418.0, -82.25, -493.75, 1812.0, 4.23046875, -206.125, -922.0, -1620.0, -2148.0, 312.25, 1012.5, 1812.0, 2564.0, -1950.0, 1036.0, -515.5, 435.25, 137.375, 372.5, -1631.0, -111.125, -1172.0, -1736.0, 18.875, 48.6875, 2606.0, 1478.0, -1419.0, -120.5, -32.21875, 600.5, 1280.0, -261.5, -1676.0, -661.0, 241.75, 502.5, -195.75, -100.125, 434.0, 561.0, 360.25, 375.0, -1373.0, -454.0, -921.5, -656.5, 2031.0, -1730.0, -1929.0, -193.625, -1092.0, 1858.0, -143.125, 474.75, -483.0, -1106.0, -64.375, -906.0, -1392.0, 210.375, -319.5, 1264.0, 1623.0, 164.0, 477.25, 30.9375, 750.5, 59.25, -624.0, 1708.0, -284.5, 601.0, 1003.0, -185.375, -788.5, 322.75, 281.75, 470.75, -124.75, 1093.0, 319.25, -932.0, -1263.0, 319.5, -1006.0, 539.5, 457.0, 1508.0, 1533.0, 628.0, 401.25, -171.75, 159.25, 753.0, 802.0, -699.0, -115.9375, -657.0, 994.5, 1171.0, -200.75, -492.0, 795.0, 138.75, -1198.0, 570.0, 1536.0, 1032.0, 114.75, 531.0, 264.5, 132.125, 316.75, -33.34375, 6.91015625, 755.0, 262.0, 3264.0, -735.0, -880.0, 813.0, -2682.0, 1847.0, 1231.0, 1134.0, 723.5, 1315.0, 2110.0, 812.5, -1782.0, 586.5, 347.25, 1540.0, -959.0, -1132.0, 2448.0, 1140.0, 1090.0, -282.5, -272.25, 544.5, 453.0, 496.0, -1627.0, 900.5, -209.625, 2422.0, 333.0, -145.0, -1569.0, 91.125, 13.578125, -1371.0, 234.875, -1495.0, 2294.0, 628.5, -17.921875, 2226.0, 1001.0, -339.5, 110.75, -451.5, 739.5, 1644.0, 307.0, 784.5, -79.1875, 1915.0, 346.25, -236.5, 253.375, 640.5, -435.0, -830.0, -539.5, 1041.0, -300.75, 257.75, -1473.0, -684.5, -334.5, -490.5, 1571.0, -1677.0, 1932.0, -241.25, 222.25, -277.5, 705.5, -1233.0, -798.0, -1628.0, -1909.0, -2370.0, -1811.0, -49.90625, -403.25, -869.0, 26.8125, 2014.0, -526.5, -1747.0, 380.0, 1127.0, 1386.0, -113.125, -565.5, 2390.0, -304.0, 23.859375, -264.25, 877.0, -1221.0, 164.875, 130.375, -700.5, 818.5, -1514.0, 1432.0, 320.0, 513.0, 609.5, 592.0, -310.0, -817.5, -32.625, 774.5, -141.375, 2522.0, 724.5, -641.5, -1652.0, -417.75, 1087.0, 391.25, -368.5, -33.28125, 308.0, 7.99609375, -37.625, -1081.0, -586.5, 1092.0, 717.0, 212.75, -284.75, -136.0, 1727.0, -1383.0, 1066.0, -1643.0, 145.25, -1751.0, 648.5, -1305.0, 7.6953125, 359.5, 1155.0, 318.0, -165.75, -916.0, 639.5, -703.5, -664.5, -1702.0, -563.0, 643.0, -1807.0, 436.75, -90.6875, 950.5, -535.0, 169.0, 1156.0, 432.25, 2806.0, -1353.0, 1870.0, 1343.0, -486.75, -300.25, -1053.0, -59.8125, -2.98046875, -81.3125, -76.4375, 678.0, 1720.0, -401.0, 21.859375, 694.0, 1780.0, -1011.5, -1283.0, -927.5, -1808.0, -628.0, -57.65625, -2524.0, 821.5, -1001.0, -2336.0, -560.0, 480.0, 1969.0, 789.5, -1285.0, 920.0, 370.25, 1322.0, 239.0, 1222.0, -1353.0, -430.25, 1046.0, -1520.0, -1889.0, -1305.0, 492.25, -569.5, 613.0, 27.328125, -151.75, 846.5, -1262.0, 782.0, -782.5, 1942.0, 1371.0, 622.5, -739.5, -736.0, -453.75, -1219.0, -532.5, -492.25, 5.60546875, -577.0, 1973.0, 916.0, 1107.0, -1063.0, 304.0, -109.3125, 1843.0, 20.8125, 354.5, 421.25, -401.75, -617.0, 193.0, -728.0, 1414.0, -1573.0, -493.75, 1044.0, 789.5, 64.625, -774.5, -48.75, -1352.0, 1359.0, 1103.0, 246.5, -189.0, -1940.0, -640.5, 193.0, -788.5, 1812.0, 1188.0, 289.25, -1692.0, -3048.0, 922.5, 1077.0, -541.0, 1420.0, -818.0, 651.0, -956.5, 265.0, 1357.0, -346.75, -529.0, 559.5, 680.5, -830.0, 1172.0, -686.0, -1217.0, 369.25, 1455.0, 116.625, -242.75, -769.0, -2031.0, 1028.0, -391.75, 27.578125, 132.25, 378.0, -393.75, 368.5, -513.5, -86.9375, -39.34375, -159.0, 62.125, -664.0, -487.25, 565.0, -1405.0, 515.0, -919.0, -813.0, 401.0, 159.125, -669.0, -571.5, -1936.0, -470.75, 226.375, -72.25, -1444.0, -73.4375, -375.25, 1388.0, -677.5, 1041.0, 1200.0, 1248.0, 244.125, -233.75, 13.1875, 739.5, 1701.0, 677.0, -1085.0, -1271.0, 367.0, -1207.0, 690.0, -942.5, 238.625, -335.25, 798.0, 28.8125, 575.0, -573.0, 455.75, 624.0, 996.0, 978.5, 1789.0, -982.5, 427.5, 412.75, -931.5, 795.5, -1180.0, -634.0, 452.0, 433.25, 197.625, -210.0, -1025.0, 380.75, 2076.0, -1903.0, -2027.0, 400.5, -2756.0, 515.0, -56.0, -1966.0, 1275.0, 1230.0, -654.0, -602.0, 1271.0, 631.0, -348.5, -1197.0, -47.4375, 1680.0, 479.0, -407.75, -773.5, -1480.0, -459.0, 80.125, -59.75, -835.5, -871.0, 10.7109375, 299.0, -1512.0, 678.0, 479.5, 993.5, 1441.0, -2476.0, 171.625, -302.75, -657.5, 411.5, -205.375, 1649.0, 213.125, 269.5, -490.25, 1762.0, -159.25, 198.875, -1321.0, 2222.0, 34.0625, 390.5, -314.25, 1842.0, -1358.0, -1037.0, 149.75, 561.0, -282.75, -614.0, 189.75, -762.5, 2090.0, 579.5, 1044.0, -849.0, -315.75, 115.5, -404.25, -952.5, -1783.0, 31.53125, -2014.0, 926.5, 78.625, -155.75, 446.75, -1253.0, -153.25, -799.5, 1216.0, -1554.0, 335.25, -1242.0, -1007.0, 1924.0, 202.375, -1009.0, 891.5, 577.5, 75.0, -777.0, 92.75, 177.25, -1127.0, 59.125, 632.5, -158.125, -450.0, 1664.0, 726.5, -1181.0, 1739.0, 434.75, 1273.0, 2056.0, 1574.0, 279.5, 799.5, -610.0, -2420.0, 265.0, 1290.0, -638.5, -484.0, 183.25, 1807.0, 375.25, 593.5, -326.25, 218.625, -225.625, 670.5, 853.0, 86.5, 532.5, 713.0, -1253.0, 147.5, -700.0, 388.25, 2460.0, 742.5, 1128.0, -3028.0, 1420.0, 772.5, -936.0, -963.0, 126.5625, -608.5, 922.0, -603.5, -191.75, -118.8125, 192.5, -932.5, 613.5, 714.0, 596.0, -228.25, 1116.0, 76.25, -792.0, -652.5, 554.5, -1147.0, 477.5, -1319.0, 116.0625, 477.75, 279.25, -202.625, -319.25, -972.0, -754.5, 1196.0, 285.75, -1488.0, -672.5, 11.21875, -269.5, 1792.0, 194.25, 130.5, -541.5, -1408.0, 211.0, 313.25, -668.0, -582.5, 951.0, -160.5, -664.0, 1051.0, -1698.0, 203.375, 64.6875, -407.25, -264.5, 74.8125, 1049.0, 92.0625, -190.75, 1146.0, 615.0, -442.5, -235.625, -616.5, -1097.0, 375.25, -22.984375, 815.5, -1777.0, 166.25, -116.4375, -779.5, -34.40625, -633.5, -103.75, -32.96875, -449.0, 37.9375, 219.875, 178.875, 274.25, 357.75, 1412.0, -1215.0, -312.75, -117.0, -144.125, 2176.0, -387.75, 556.5, 1123.0, -349.0, -1010.5, -709.5, 393.25, 556.5, -531.5, 165.0, -58.90625, 144.625, 874.0, -928.0, -704.5, 102.9375, -998.5, 213.625, 783.0, -491.25, -322.5, 1367.0, 28.890625, -20.71875, 577.5, 1004.0, 502.5, -16.75, 1310.0, -471.75, 177.5, -250.5, -1218.0, 72.0, 706.5, 487.25, -66.5, -590.0, -355.5, 895.0, 356.75, -98.125, 1105.0, -1275.0, -345.25, 345.75, -714.5, 638.5, 868.0, 1449.0, -1816.0, 146.875, 2358.0, 26.4375, -1433.0, 97.9375, -924.5, 290.75, -412.75, 358.25, 1038.0, 584.0, 29.734375, -1653.0, 202.5, 2684.0, 1569.0, 381.75, 1039.0, -983.0, -661.0, 599.5, 1045.0, -1309.0, -214.125, -604.5, 1144.0, 1409.0, -235.25, 339.5, 1343.0, 148.25, 0.61376953125, -256.25, -13.3828125, 821.0, 296.5, -719.0, 641.5, -982.0, -498.25, 2000.0, -2482.0, 290.75, -435.5, -361.0, -1688.0, 52.3125, 1338.0, -986.0, -574.0, -776.0, 751.5, -544.5, -365.25, -1052.0, -1602.0, -1053.0, 691.0, -1713.0, -1492.0, 1147.0, -141.375, -1095.0, 823.5, 483.0, -644.0, -2086.0, -1867.0, -1456.0, -384.0, 735.0, 1331.0, 1583.0, -148.75, 244.25, -190.625, -257.25, -42.46875, 2906.0, 10.40625, -190.0, 142.75, -1110.0, -474.25, -1737.0, -1465.0, 7.546875, 903.0, 349.0, 76.9375, 40.8125, 43.09375, -688.5, -695.0, 293.5, -1.5048828125, -1959.0, -592.0, 218.625, 767.5, -877.0, -73.3125, -52.53125, 2916.0, 378.0, -201.75, -372.25, -1030.0, 533.5, -317.25, -271.5, -342.0, -510.25, 264.75, 181.25, -842.5, -1078.0, 1431.0, -790.5, 206.75, 1301.0, 56.0625, 311.5, 437.5, 1047.0, -1317.0, -1665.0, 829.5, 220.75, -82.75, 130.375, -1570.0, 986.5, -614.0, -156.125, -950.5, 558.0, -243.875, -1430.0, 418.0, 293.0, 1854.0, -510.75, 2084.0, -521.0, -2106.0, -172.25, 1001.0, -242.0, 69.625, -230.125, -531.0, 1178.0, -883.5, 1059.0, 578.5, -21.234375, 1879.0, 421.25, -520.0, 681.5, 472.5, -411.25, -285.5, 739.5, 160.375, -1051.0, -204.0, 1878.0, -1098.0, -590.0, -109.25, 451.75, 324.25, -1338.0, 1331.0, -368.0, 566.5, 1019.5, -196.0, -1600.0, 539.5, -1028.0, -563.5, -159.375, -971.5, -441.5, -570.0, 224.75, -1245.0, -502.5, -574.5, 1097.0, -626.5, -570.0, 1073.0, 422.25, -1397.0, 935.5, -403.25, 13.78125, 854.0, -459.75, -226.5, 812.0, 952.5, 293.75, -1267.0, -1222.0, 426.75, -801.5, 166.25, -184.75, 1774.0, 1205.0, 1500.0, 292.75, 93.75, 458.5, 1715.0, -1347.0, 75.125, 565.5, -396.5, -577.5, -268.0, -31.125, -243.125, -630.5, 928.0, -1497.0, -1097.0, 927.5, 1638.0, 654.0, -1677.0, 370.25, -298.5, 29.0, 409.25, -1164.0, 45.84375, 1785.0, -212.625, -831.0, -557.5, 297.0, -279.75, 1073.0, -2428.0, -466.5, -175.375, -942.0, -1565.0, 306.75, 1080.0, -545.5, -910.5, 276.0, -1118.0, 1901.0, -1539.0, -1024.0, 713.0, 258.5, -1578.0, -2076.0, 1101.0, 1167.0, 69.9375, 1511.0, -1951.0, 18.890625, 528.5, 529.0, 2172.0, -2454.0, -2220.0, -531.5, 932.5, -326.75, -1128.0, -560.0, 527.5, -1755.0, -878.0, -1627.0, -900.0, -984.5, 1729.0, 1406.0, -42.21875, -1333.0, -70.125, 174.25, 117.5625, 626.5, -619.5, -1488.0, -2752.0, -977.5, -130.0, -712.5, 426.75, 749.0, -877.5, 2136.0, -749.0, -1241.0, 858.0, -247.75, 406.25, 214.125, 2096.0, -664.5, 823.0, -412.0, 318.5, -374.0, 1037.0, -353.5, 66.9375, -1065.0, -30.625, 1586.0, -1260.0, -259.25, -1140.0, -732.5, 1002.0, 10.96875, 1397.0, -986.0, 808.5, 292.5, 1426.0, -211.125, 413.0, 697.5, -686.0, -355.25, 326.0, 60.15625, 2240.0, 1854.0, 580.0, 1380.0, -1130.0, 1512.0, -925.5, -469.25, -1173.0, 1762.0, 269.75, 1099.0, 412.5, 1857.0, -51.21875, -505.25, -37.75, -1898.0, 349.25, 911.5, -719.5, 179.375, 1275.0, 111.625, 2050.0, -211.875, -513.0, 1371.0, -1404.0, -618.5, -111.125, -682.0, 1504.0, 322.0, -59.78125, -82.0, 187.125, 1987.0, -833.0, 624.5, -264.0, 420.75, 65.25, -886.5, 713.0, 286.25, -30.109375, -40.75, -1535.0, 1539.0, 31.15625, -457.25, -987.0, -946.5, -2406.0, 3252.0, 70.5, -800.0, -631.0, 1543.0, -309.25, 324.25, -2012.0, -124.375, -1256.0, 1588.0, 1545.0, -163.25, 2298.0, 313.0, -1907.0, 856.5, -1627.0, -1001.0, -913.5, -191.375, -552.0, -192.75, 1067.0, -555.5, -2352.0, -252.625, 482.25, 1872.0, -8.421875, 464.25, 431.25, -1784.0, -1010.5, 1408.0, -245.0, 866.0, -167.75, -828.5, 577.0, -618.5, 1724.0, 1509.0, -891.5, -119.625, 508.75, -1164.0, 2.15234375, 2372.0, -2612.0, -181.0, -230.75, 1618.0, -43.3125, 920.5, -65.5625, 1388.0, -165.0, 2528.0, 1325.0, -1473.0, 1507.0, -522.0, 1181.0, 1062.0, 950.5, 79.5, -293.75, 1156.0, 273.75, 259.75, -2958.0, 242.875, 1140.0, 1202.0, -3.0625, -732.5, -579.5, -1404.0, 1217.0, 1635.0, 1830.0, -1560.0, 208.875, 198.25, -858.0, 364.75, -141.5, 504.5, -1080.0, 225.75, 561.5, -407.0, -652.5, 785.0, 999.5, 1332.0, 118.125, 503.5, -262.5, -1783.0, -868.5, 1141.0, 908.5, -1037.0, -975.0, -210.375, 775.0, -461.25, 1319.0, 164.125, -1096.0, 150.0, 543.0, -679.5, 619.5, 243.0, -16.09375, -644.0, 1303.0, -2520.0, 91.875, -337.75, 262.25, 2045.0, -17.640625, -128.375, -789.5, -174.875, 614.5, -663.5, -294.5, 1117.0, 320.5, -113.625, -404.25, 1595.0, 179.625, 1074.0, -1254.0, -564.5, -316.75, 1101.0, -90.9375, -1145.0, -537.0, -432.75, 38.1875, 704.5, 813.5, -1194.0, -1073.0, 1073.0, 920.5, 601.5, -1269.0, -224.5, 1236.0, -979.0, -125.0, 288.0, 1734.0, -387.0, 2124.0, -1146.0, -1260.0, -309.25, -1390.0, -913.0, 73.3125, -145.75, 1072.0, -631.5, -433.5, 733.5, -313.75, -304.5, 53.28125, -999.0, 769.5, 494.25, 856.5, -1457.0, -744.5, -730.0, -1961.0, -1496.0, -447.75, 1832.0, -81.5, -552.0, 1984.0, -1080.0, -1407.0, 1617.0, 242.625, -1957.0, -1393.0, -56.375, 938.0, 1369.0, -321.0, -2490.0, 11.984375, -28.671875, 321.5, -940.0, -1771.0, -706.5, 611.0, -255.375, -446.75, -801.0, 2558.0, -369.75, -526.5, 37.28125, 1208.0, 1484.0, 341.0, -377.0, -317.75, 375.75, -826.0, -200.75, -1688.0, 1949.0, 1408.0, 13.3984375, -898.0, 2136.0, 681.5, -730.0, -1271.0, -783.0, 693.0, -615.0, -116.9375, -250.25, 184.375, -352.25, 116.25, -2084.0, -127.1875, 1689.0, -1194.0, -597.0, 1737.0, -793.0, 498.5, 993.0, 748.0, 721.0, -2244.0, -2616.0, -113.875, 411.5, 308.75, -660.5, 1139.0, -605.0, -520.0, 1299.0, -1059.0, 175.0, 331.25, -417.0, 360.5, 457.0, -1178.0, 1776.0, -1178.0, 194.25, -95.6875, 1221.0, -531.0, 1371.0, 397.0, -252.625, -1084.0, 1747.0, 1617.0, 747.5, -220.375, 2138.0, -461.25, 1851.0, 0.92578125, 1373.0, 1063.0, 676.0, -760.5, -1408.0, -241.0, 980.0, 314.5, 485.25, 962.0, 452.5, 2244.0, -711.0, 1034.0, 557.0, -2054.0, -63.15625, -1422.0, -1516.0, -629.0, 1167.0, 592.0, -1482.0, -877.5, 1346.0, 1036.0, 975.5, -857.5, -1311.0, 1350.0, 1013.5, -181.0, 1045.0, 519.5, -276.0, -139.75, -719.5, -405.75, -1294.0, -2542.0, -472.5, -942.5, -190.625, 205.75, 762.0, 802.0, -2098.0, -1551.0, -239.375, 842.5, -911.0, 392.5, -542.0, 605.0, -482.0, 534.0, 577.0, -1751.0, 3322.0, -608.0, -1204.0, -994.5, -74.6875, 1175.0, -258.5, 47.0, 761.5, 235.25, 976.0, -17.296875, 338.5, 642.5, 1646.0, 508.25, -556.0, -393.25, -477.5, -1589.0, 268.0, 35.0, 930.5, 1434.0, -1088.0, -942.5, 760.5, 15.8515625, -524.5, -583.0, 350.25, 1772.0, 633.5, 2004.0, 121.4375, -453.5, -350.75, -573.5, 354.75, 226.5, -1153.0, 74.75, -1717.0, -23.796875, -1425.0, 1626.0, -185.375, -115.8125, -649.0, -1564.0, -1622.0, 288.5, 1591.0, -316.75, 1301.0, 540.0, -536.0, -2764.0, -1161.0, 1148.0, -530.0, 871.5, 629.5, 365.25, 782.0, -1491.0, -641.5, -1854.0, -998.0, -1410.0, 1675.0, 75.125, -58.625, 261.0, -671.5, 172.75, -405.25, -256.5, 83.9375, -1093.0, 512.5, 495.75, -159.5, 156.375, -89.75, 115.0625, 1142.0, 1297.0, 1759.0, -792.0, 1143.0, -89.75, 291.75, 986.5, 1148.0, 1259.0, -182.5, -1826.0, 248.625, 771.5, -182.125, -960.5, 1271.0, -1290.0, 1227.0, 2274.0, -1394.0, -960.0, 888.5, 555.5, -1238.0, -1709.0, 149.875, -1327.0, -1054.0, -499.0, 136.75, 1688.0, -679.0, -1105.0, -1513.0, 150.625, -1895.0, -1097.0, 1257.0, 465.25, -81.6875, -371.5, 1400.0, -2162.0, 901.0, -1680.0, -1228.0, 1188.0, 814.0, 1801.0, -79.4375, 941.0, 595.0, 539.5, -745.0, -1532.0, 912.0, -435.25, 153.25, 13.546875, -286.75, 613.5, -591.5, 1345.0, -336.25, 1413.0, 1199.0, -1764.0, -1181.0, -184.125, 130.75, -9.34375, 1183.0, -154.625, 1254.0, -1746.0, 698.0, -170.0, -424.0, -503.5, -61.21875, -906.0, -54.5, 610.0, 185.625, -845.0, 949.5, 427.5, 830.0, 68.3125, -1521.0, 482.75, 131.875, -1290.0, 363.0, 1547.0, 410.75, 269.25, -548.0, -137.75, 1720.0, -440.25, -931.5, -244.375, -863.5, 581.0, 1308.0, -225.0, -1222.0, 1193.0, -444.5, -1362.0, 1682.0, -1511.0, -2390.0, 910.0, -936.0, -722.5, -97.4375, -19.328125, 154.875, 655.5, 442.75, 1568.0, 193.125, -1086.0, -631.0, 372.25, 1099.0, 437.0, 763.0, 464.5, -324.75, -804.0, 151.625, -408.0, -801.5, 90.9375, 846.5, 421.75, 639.5, -807.5, -1904.0, 300.5, 168.0, -453.5, -1765.0, -1073.0, -856.5, 1840.0, -350.0, -3038.0, -568.5, 362.5, -450.0, 577.0, 333.25, 248.125, -1427.0, -1654.0, 981.5, 1989.0, -1243.0, -568.0, -1954.0, 923.0, 1269.0, -1701.0, -415.0, 728.0, -404.25, -315.5, 284.0, -796.5, -500.0, -451.25, 53.59375, 1059.0, -620.5, 881.5, 1025.0, -1806.0, -480.25, -736.0, 779.5, 951.0, -253.875, 503.0, 445.0, -986.0, -2138.0, -1866.0, 159.875, -1815.0, 757.0, -946.0, -194.625, 1138.0, 799.5, -1762.0, 26.875, -1403.0, 1060.0, -79.6875, -752.5, 132.0, -1018.5, -1092.0, -1479.0, -186.375, 1400.0, -1088.0, -1222.0, -1577.0, -247.25, -201.75, 1868.0, -646.5, 321.75, -328.75, 552.0, -828.5, -526.0, -1383.0, 386.0, -518.5, -422.75, 952.0, 415.75, 718.0, 777.5, 108.875, 550.0, -1838.0, -774.5, -2296.0, -353.25, -721.0, -1542.0, -179.375, -159.375, 283.75, 680.0, -730.5, 557.0, -319.75, -388.25, -947.0, 2338.0, 339.25, -1036.0, -304.0, -1022.5, -1094.0, -245.875, 307.0, -620.5, -622.0, 1063.0, 758.0, 770.0, -870.5, 264.0, -97.125, -18.59375, 122.375, -2116.0, 1524.0, 675.5, 264.0, 1140.0, 1420.0, -898.5, 191.0, 767.0, -815.0, 284.5, 602.0, -822.5, 448.0, 591.5, -1064.0, 163.75, 420.5, 1089.0, -805.0, -195.5, 1722.0, -1346.0, 637.5, 1191.0, 552.0, 677.5, 454.25, 1085.0, -1147.0, -1054.0, -994.5, -1005.0, -294.25, -533.0, 810.5, 657.0, 328.75, 1375.0, -102.125, 3268.0, -1438.0, -1271.0, 618.5, 102.875, -1195.0, 38.59375, 698.0, -161.375, 1528.0, -592.0, -445.5, 951.0, 883.5, -1624.0, -1554.0, 884.0, -943.0, 422.0, 0.471923828125, 1326.0, 332.5, -1124.0, -415.5, 954.0, 183.5, -492.0, 684.0, -276.0, -503.5, 989.5, -229.5, -1128.0, 512.5, 211.75, 1227.0, 1146.0, 1005.0, 238.375, 148.0, 569.5, 753.5, -201.875, -1811.0, 922.0, 2120.0, 262.75, 795.5, 1153.0, 193.625, -1382.0, -1160.0, 738.5, 1979.0, 1199.0, -671.5, -260.75, -823.0, -1856.0, -674.0, -571.0, -1441.0, -710.5, -275.5, 1081.0, 1342.0, 132.875, 802.0, -1689.0, 902.5, 2438.0, 221.75, 70.5625, 1421.0, -603.5, -2080.0, 986.5, -1905.0, 869.5, 1222.0, -138.125, 913.0, 77.9375, 1097.0, 703.5, 28.375, 512.5, 1291.0, -931.0, -1057.0, 183.625, 166.75, -1205.0, 85.75, 10.1796875, 636.0, 1608.0, 1861.0, -1478.0, 1142.0, 12.234375, -866.0, 466.5, 3284.0, 317.5, -42.3125, 947.0, 435.75, 244.125, -547.0, -694.0, 45.1875, 1532.0, 929.0, 1184.0, -397.25, -113.875, -853.0, -604.0, 849.0, 993.5, -885.0, 800.0, 1827.0, -59.90625, 1551.0, 1659.0, 97.75, 1271.0, -606.5, 42.875, -340.75, -995.0, -1382.0, -22.75, -253.125, 1187.0, -1734.0, 331.0, 672.0, 1547.0, 1819.0, -81.0, 117.0625, -380.25, 2606.0, -422.75, -390.0, 105.5, 88.1875, -1060.0, 144.375, -576.0, -1581.0, -1538.0, 182.875, -739.5, 638.0, 308.0, 781.5, 1224.0, 531.5, 227.75, 1850.0, 50.40625, 1542.0, -977.0, -777.0, 39.625, 1007.0, 1908.0, 983.0, -1400.0, 397.5, -401.25, -777.5, 264.5, 105.0, -1005.0, 1125.0, 164.25, -1646.0, -271.25, -248.375, -340.75, -119.5625, -708.0, -874.0, 925.5, 833.0, -1338.0, -909.0, 2572.0, 148.625, -771.5, -1140.0, -79.5625, -1123.0, -289.5, 206.375, -407.25, 224.125, -1052.0, 576.5, -277.25, -252.0, -506.75, 547.5, -15.2734375, -643.5, 747.0, 1023.5, 2444.0, 188.875, 1028.0, 355.75, -209.25, -723.0, 125.875, 573.5, 1641.0, 1077.0, 827.0, -1794.0, -260.5, 347.25, 522.0, 962.0, 1493.0, -315.25, 403.25, -2996.0, 46.8125, 525.5, -735.5, 1526.0, 1190.0, -1970.0, -838.5, -321.0, 1089.0, -1390.0, 913.5, 360.75, 874.5, 1789.0, -1067.0, -1422.0, -54.71875, -707.0, 340.5, 568.5, 836.5, -1131.0, -600.0, -812.5, -72.1875, -531.5, 1732.0, -905.0, -1311.0, -1146.0, 2025.0, 1243.0, -8.90625, -179.375, 615.5, 137.875, -2043.0, -455.75, 79.0, -1361.0, 1715.0, -458.0, 199.875, 1351.0, -380.5, 712.5, 1381.0, -348.25, 823.0, 74.1875, -552.5, 596.0, 1954.0, -1370.0, 359.5, -380.25, -886.5, -343.0, 1067.0, -568.5, -648.0, -1287.0, -1027.0, 280.75, 307.25, -233.0, -2178.0, 65.625, 300.75, -1496.0, 511.25, 945.0, 352.75, 518.5, -1768.0, -934.0, -249.25, -437.0, 406.75, 525.5, 983.5, 808.5, 1470.0, -1002.0, -39.28125, -460.25, 755.0, -543.0, -154.25, 1776.0, 718.5, -230.5, -674.0, -2624.0, 409.75, 1213.0, 1166.0, 395.0, -301.75, -873.5, 1687.0, 2330.0, 1768.0, 1076.0, 582.0, 141.5, -1577.0, -1586.0, 733.5, -308.25, 743.5, -1494.0, -644.0, 108.875, -785.5, 97.625, -234.75, -465.0, -1539.0, 985.5, -1048.0, 573.0, 675.5, 51.59375, 1850.0, -188.375, 245.875, -835.5, 197.75, 30.71875, 1129.0, -254.625, -75.875, 1138.0, -416.5, 594.0, -699.5, -1598.0, 1114.0, -542.5, -153.75, -927.0, 913.0, 633.5, -1290.0, 1679.0, -317.5, 770.0, -190.25, 976.5, -771.0, -698.0, -138.125, -185.125, -864.0, -1852.0, -101.875, 1759.0, 481.25, 46.09375, -17.875, 666.0, -547.5, 620.5, -613.5, 127.0625, -114.875, -760.5, 1029.0, 1817.0, -1103.0, 1072.0, -1741.0, -2196.0, -1111.0, -1854.0, 1119.0, 27.015625, -1110.0, 383.75, 1420.0, -1675.0, -227.125, 270.0, 421.0, -433.75, 2194.0, -889.0, 228.875, -314.25, -1072.0, -479.0, 1166.0, 3276.0, 493.5, -1104.0, -861.5, -251.75, 1355.0, 1086.0, -477.0, 1595.0, -362.75, -1009.0, 673.0, 393.0, 12.703125, 81.6875, 180.625, -1262.0, -876.5, -1067.0, 1281.0, 1589.0, -1185.0, -452.25, 2904.0, -533.0, -213.5, -345.5, 522.5, 473.5, -1122.0, -652.5, 52.96875, 550.5, -70.0625, 535.5, -123.125, 289.75, 990.0, 1142.0, -75.375, -605.0, 182.75, 500.0, 1769.0, -2170.0, -336.0, -128.25, 542.0, 321.75, -1029.0, -1882.0, 455.25, -580.0, -94.625, 1201.0, 1261.0, 24.734375, -726.0, -289.75, -77.375, -694.0, -779.0, -1487.0, 1048.0, 72.875, -234.875, 91.125, 736.5, -864.5, 660.0, 228.0, 1091.0, -384.5, 191.5, 628.0, 126.125, 563.0, -486.75, 412.0, -695.5, -1125.0, -337.25, 183.125, -1147.0, 490.75, 703.0, -611.5, -281.25, -935.0, 457.5, 225.25, -238.0, 211.25, 738.5, -240.625, 1701.0, -606.5, -774.0, -409.75, 835.5, -898.5, 239.125, 360.5, 1577.0, -1236.0, 132.875, -1508.0, -1714.0, 957.5, 427.0, 453.0, -483.75, 1383.0, -1009.0, -1023.5, -2158.0, -357.5, -5.87109375, 454.0, -345.5, 594.5, 1708.0, -803.5, -1183.0, 231.75, 658.5, -768.0, 496.5, 573.0, 611.0, 1801.0, -438.0, -336.0, -19.78125, -1657.0, -391.5, 430.25, 637.5, 415.75, -11.359375, -71.4375, 388.5, 1219.0, -1348.0, -2062.0, -7.625, 315.5, 241.625, -173.0, -1333.0, 294.0, -1936.0, 1989.0, -681.5, 367.0, -62.15625, -170.875, 1763.0, -47.9375, -231.375, 196.25, -1335.0, -1935.0, 235.0, 2021.0, 202.875, 2970.0, -1595.0, -615.0, -817.5, -486.0, 1553.0, -1102.0, 475.75, 380.25, -784.5, 1407.0, 1354.0, 46.75, -1979.0, 660.5, 1566.0, 1288.0, -1632.0, 387.0, -557.5, 567.0, -2268.0, -846.0, -305.75, -927.0, -154.375, 339.5, 229.875, -432.5, -1620.0, -3108.0, 71.625, 1876.0, 911.5, -595.5, 365.25, 827.0, -1086.0, -686.0, 1392.0, -967.5, -692.0, -40.9375, 573.0, 642.5, -626.0, 691.0, 55.59375, 414.5, 254.875, 899.5, -212.375, 511.75, 427.25, -770.0, 1813.0, -424.0], "dtype": "float16"}, {"__nd__": [197.75, -1478.0, 974.5, -85.125, -729.0, -905.0, 189.875, -483.25, 449.25, -329.0, 110.5, 1359.0, 92.5625, -963.0, 100.625, 872.0, -1514.0, -932.0, 441.25, 220.125, 1562.0, 394.0, -840.5, -86.125, 205.625, 1452.0, -1635.0, 1226.0, 423.75, 126.5625, 2250.0, 429.75, 1092.0, 530.5, 909.0, 1077.0, 570.0, -1163.0, -1353.0, -740.0, 121.0625, -745.5, -823.5, -468.75, 1051.0, -1399.0, 366.5, -2104.0, -670.0, -71.5625, 669.5, 1154.0, -802.0, -891.5, 1155.0, 165.625, -568.5, 1106.0, -435.5, 408.25, -971.5, 1107.0, 219.125, 105.5, 752.0, -1968.0, -22.578125, 1516.0, 654.0, 651.5, 384.5, 1290.0, 143.75, 2362.0, -1013.5, 1255.0, 596.5, -242.375, -494.5, -557.5, 533.0, -1543.0, 109.9375, 405.25, 499.25, 527.0, 1430.0, -1057.0, 715.5, -694.5, -967.0, 803.0, -1007.0, 1253.0, 529.0, -1317.0, -212.375, -2035.0, -390.75, -1955.0, 579.0, 1905.0, 76.125, 1015.5, -586.0, -304.25, 151.25, 2218.0, -811.0, -1334.0, -526.5, 1027.0, 752.0, -463.25, 451.5, -11.6328125, 1820.0, -49.0625, -245.0, -556.5, 1815.0, 590.5, 337.0, -1128.0, -1154.0, 1134.0, 1229.0, -418.25, -1622.0, 2188.0, 1077.0, -607.0, 70.4375, -1107.0, -1068.0, 146.875, -495.0, -737.5, 1278.0, -444.0, 559.0, -1615.0, -94.375, 675.0, -1522.0, 1146.0, -679.0, 313.0, 352.5, 92.9375, 414.5, -301.25, 1876.0, 1956.0, 895.0, -1415.0, 657.0, 228.125, -843.0, -537.0, 538.0, 1097.0, 395.0, 535.5, 230.25, 878.5, -199.375, -638.0, -1000.0, 518.0, 1283.0, -842.5, 809.0, -246.375, 548.0, 643.0, 1358.0, -1227.0, -402.25, 944.5, 401.5, 340.25, 884.0, 992.0, -269.25, -1529.0, 461.75, -291.75, 112.125, 2164.0, -1165.0, 429.25, 133.875, -598.0, 1216.0, 747.5, 556.5, 1751.0, 2404.0, 245.125, 671.5, 1588.0, -862.0, -288.5, 386.5, -587.0, -1646.0, 990.5, -752.5, -1512.0, 492.0, -320.75, -1229.0, -309.75, -70.75, 961.0, -322.0, -853.0, 1891.0, 93.4375, -331.75, -1274.0, 1016.5, 860.0, -88.9375, 739.5, -1113.0, -671.0, -4.1015625, 328.5, -879.0, 1046.0, -307.0, 744.5, -282.0, -1533.0, 1295.0, -284.75, 1356.0, 477.25, -132.125, -130.125, 1636.0, 330.5, -1251.0, -1068.0, 616.5, 471.75, 561.0, -648.0, 869.5, -788.5, -566.0, 879.5, -209.125, -1133.0, 342.75, 1084.0, -1003.0, -221.0, 2656.0, 2688.0, -1276.0, 289.75, 763.5, 1540.0, 1393.0, 549.0, -100.8125, -632.5, 1421.0, -669.0, -451.0, 582.5, 470.25, -1152.0, -1915.0, -252.25, -719.5, -1987.0, -1062.0, -1610.0, -522.5, -1479.0, 517.0, 2282.0, 1313.0, 2690.0, 219.375, -1179.0, -1180.0, 344.5, -518.0, 1102.0, 183.125, 520.5, 24.625, 771.0, 2358.0, 1182.0, -1.6552734375, 842.0, 767.0, 1834.0, -3.9453125, -695.5, -767.0, 225.375, -389.75, 450.75, 1358.0, -1290.0, 1036.0, 952.5, -51.4375, 164.25, 86.25, -87.0, 1258.0, -5.078125, -848.5, -1052.0, -1247.0, 62.21875, -706.5, -906.0, -865.0, -882.5, 445.75, 776.5, -2124.0, -174.0, 1396.0, -422.25, -1745.0, -461.75, 359.0, -1334.0, -2122.0, -207.875, 490.0, -345.0, 991.5, 1104.0, 570.0, -181.0, -1446.0, -492.0, 389.0, 2112.0, -629.0, 43.1875, -1178.0, -61.09375, -564.5, -407.75, -1055.0, 571.5, -1543.0, -478.0, -2370.0, -966.5, -290.75, 218.875, 1030.0, -91.9375, 237.125, -1450.0, 1394.0, -181.625, 934.5, -211.125, -771.5, -349.5, 231.5, 76.625, -522.5, 1040.0, -488.75, 276.0, -795.0, -726.5, 407.0, -482.75, 241.25, -2114.0, -254.625, -190.0, -546.5, -1729.0, -864.0, -699.5, 506.25, -346.0, 2446.0, 2016.0, 202.0, 1765.0, -542.5, 1176.0, 1309.0, 673.5, 1023.5, 572.0, 800.0, 1333.0, -697.0, 287.75, 230.0, 305.25, -67.4375, -438.5, -75.25, 236.375, 377.5, -1494.0, 712.5, -956.0, 1213.0, -837.5, 1137.0, 2508.0, -221.0, -472.5, -460.25, -543.0, 1027.0, -790.0, -885.5, 568.5, -23.109375, -27.9375, -1039.0, 1507.0, 408.5, 161.375, 724.0, 613.5, 1010.5, -1527.0, -736.5, 63.40625, -60.21875, -706.0, 1164.0, -583.5, 621.0, 242.75, -486.5, -776.5, -1333.0, -315.75, -499.0, 227.125, -823.0, -812.0, 1007.0, 945.5, -1262.0, -381.5, -1504.0, 475.25, -1642.0, -1564.0, 107.5625, 823.0, 589.5, 509.25, 1724.0, 57.25, 88.0625, 47.59375, 89.5, -336.5, -1850.0, -1194.0, -81.0, 155.625, 1036.0, -810.5, -194.0, 790.0, -102.3125, -981.0, 118.5, 834.0, 1062.0, 379.0, 460.5, -251.25, -1727.0, -1303.0, 541.5, -144.125, -414.5, -481.5, 267.5, -2106.0, 393.75, -263.5, -12.9140625, 2438.0, 1481.0, -310.5, -2282.0, -627.5, -455.25, -568.5, -575.5, -1228.0, -370.0, 1069.0, -1428.0, 605.0, -182.75, -460.0, -1500.0, -1501.0, 1345.0, 724.0, -112.3125, 1070.0, -1170.0, -413.5, 1649.0, -1080.0, 1196.0, 134.5, 273.0, 296.25, 408.5, -1021.5, -715.0, -219.875, -1089.0, 651.0, 743.0, -440.75, -132.125, -481.25, 1028.0, 698.0, 2602.0, 379.5, 858.0, 467.5, 397.0, 1050.0, 1694.0, -380.25, 745.5, 35.78125, -636.0, 676.5, -1813.0, 1632.0, 213.125, -817.0, 278.5, 1386.0, 1612.0, -697.5, -1206.0, -2688.0, -1184.0, -844.0, 1061.0, -170.875, -1275.0, -465.25, -892.5, -1020.5, -152.625, 2080.0, 912.5, -451.25, 1195.0, -386.75, -207.25, -722.5, 179.75, 457.75, 502.5, -561.5, 338.0, 69.3125, 623.0, 1188.0, -943.5, -661.0, 503.5, 481.0, -802.5, -209.0, -202.5, -567.5, -411.5, 121.8125, 1728.0, -1737.0, -247.75, -271.25, -1209.0, -78.4375, 757.5, -491.5, -1271.0, 293.0, 1062.0, 3202.0, 703.5, 370.75, -384.5, 1411.0, -975.0, -1843.0, -51.4375, 538.5, 1100.0, 66.25, 1280.0, 739.0, -464.25, 947.5, 28.953125, 1046.0, 559.5, -1149.0, 290.5, 451.25, 1350.0, 1158.0, 2286.0, 577.0, -146.0, 24.046875, 687.0, -763.5, -1187.0, -798.0, 169.5, -996.5, 1127.0, 537.5, -1237.0, -165.625, -2208.0, -855.5, 614.0, -111.125, -1265.0, 546.0, -1902.0, -1287.0, 1752.0, 1573.0, 894.5, 483.25, 134.375, -645.0, 352.25, 887.5, -948.5, 270.75, -1291.0, 1300.0, -378.0, -466.0, 711.5, 484.5, -61.84375, -1125.0, -140.625, -468.25, 585.5, -548.0, -528.0, -322.0, 357.25, 1218.0, 976.0, -718.5, -385.75, 663.0, 92.0625, 1370.0, 1137.0, -1131.0, -325.25, -479.75, 824.0, 1377.0, 719.0, 1650.0, 296.5, -661.0, -99.8125, 148.0, 51.1875, 388.75, 531.0, -950.5, 546.5, -2346.0, 1433.0, -92.4375, 492.25, -1527.0, 59.59375, 631.0, 884.5, 104.0625, 51.0625, 68.875, -597.5, 738.5, -938.0, -66.8125, 2292.0, 2722.0, -358.25, 204.875, 811.0, -1420.0, -708.5, -347.75, 864.5, -587.0, -1337.0, -687.0, -302.5, 75.75, 196.875, 987.5, -12.015625, -1487.0, -1534.0, 1035.0, 306.75, 760.0, -527.0, 1861.0, -883.5, -646.0, -377.25, 626.0, -489.25, -107.625, 95.8125, 1681.0, -1385.0, 337.25, -1150.0, 1103.0, 1439.0, 603.0, 428.0, -467.75, -1599.0, 665.5, 137.75, 1303.0, -332.0, 224.0, 2122.0, -317.5, -1704.0, -59.9375, -59.625, 294.5, 956.5, -759.5, 388.25, -1366.0, -1704.0, 248.125, -380.25, 592.0, -11.1171875, -628.5, 2.201171875, -367.5, 904.5, -1420.0, -428.5, 223.5, -2110.0, -84.0625, -410.75, 1196.0, 661.5, 773.0, 321.5, 931.0, 186.0, -1478.0, -583.0, -1486.0, 960.0, -1488.0, -2160.0, -1216.0, 1348.0, 946.0, -1882.0, 969.0, 1356.0, -312.75, 1593.0, -22.640625, 926.5, -1388.0, -699.0, -1239.0, -1566.0, 750.5, 317.75, 1572.0, 139.125, -317.75, 353.0, -2029.0, -139.75, -178.5, 1260.0, -44.25, 648.0, 3.224609375, -222.5, -306.75, -1279.0, 1274.0, -823.5, -821.0, 363.5, -691.5, 145.375, -1831.0, -126.6875, 1806.0, -293.25, -263.25, -475.5, 1199.0, 445.25, 163.875, -156.75, -28.703125, 12.859375, -3038.0, 2035.0, -2204.0, 1033.0, -562.0, 1672.0, -364.0, -1610.0, -892.0, 794.5, -1057.0, -713.5, -387.75, 702.5, 1176.0, -417.75, 1317.0, 1304.0, 136.625, 628.0, -1058.0, 817.5, -437.25, 226.75, 788.0, 10.3359375, 1272.0, -405.75, -1873.0, -566.5, 132.75, 123.6875, -201.375, -376.0, 359.75, -515.5, -844.0, 1177.0, 1062.0, 1595.0, 733.0, -243.625, 367.5, -2043.0, -748.5, 379.0, -465.75, -804.0, 114.875, -1421.0, 164.0, 279.25, 149.875, -1075.0, -712.5, 401.5, -1360.0, 14.2109375, 1727.0, 2490.0, -407.0, 1411.0, -221.25, -781.0, 1114.0, -1023.5, -1331.0, 365.75, -263.25, -1458.0, -740.5, -897.5, 174.5, 1300.0, 125.75, 1172.0, 734.5, -675.0, 67.1875, 1078.0, -1267.0, 819.0, 742.5, -603.0, -376.75, 455.5, 1024.0, -1364.0, 1207.0, -35.25, -1371.0, 410.5, -50.3125, -488.25, 220.875, -387.75, -94.5, -806.0, 164.5, 1701.0, -1042.0, 861.5, 203.25, -1427.0, -615.5, 34.9375, 127.0625, 220.0, -562.0, 1416.0, -809.5, 1258.0, -625.5, -1743.0, 1333.0, -557.0, 327.75, 1729.0, 991.5, 1003.0, -1672.0, 824.5, 1115.0, -135.0, -34.28125, -341.5, -2370.0, 306.25, 278.0, -2182.0, -639.0, 1259.0, 1564.0, -1144.0, 1210.0, -1940.0, -1396.0, -25.5625, 1051.0, -654.5, 314.25, 475.75, 1275.0, 885.0, 95.0, -245.5, 162.375, -3116.0, 177.5, -134.125, -444.5, -1019.5, -82.0625, 929.5, 927.0, -1423.0, -36.9375, 678.5, 834.0, 407.5, 222.5, 1240.0, 323.5, -412.5, -463.25, -998.0, 72.4375, 1485.0, -723.5, 492.0, -1195.0, -291.5, -595.0, 1009.0, -478.75, 1213.0, 290.0, -1375.0, 983.0, -1276.0, 886.0, -1019.5, -346.75, -329.75, 308.25, 76.6875, -590.0, -77.875, 1235.0, -28.921875, 1228.0, 103.1875, -497.0, 1856.0, -291.25, -1062.0, 1469.0, 804.0, 1519.0, 602.0, -789.0, -760.5, 161.375, 1135.0, 492.5, -961.5, -156.0, -1572.0, -573.0, -577.5, 164.75, 1495.0, 599.0, -138.125, 44.34375, 1441.0, 209.5, -1146.0, -691.0, -1199.0, 893.0, -488.5, -843.5, -2554.0, -328.25, 607.5, -120.6875, 1604.0, -281.25, -1349.0, -1613.0, -28.28125, 246.625, 946.0, 889.0, -2092.0, -2538.0, -1496.0, -1271.0, 1740.0, 1379.0, -753.5, -1064.0, -766.5, -348.25, 977.0, 117.6875, 609.5, -510.75, -1756.0, 1302.0, -247.875, -233.75, -377.5, 1969.0, -427.75, -860.5, 476.5, 35.46875, 44.8125, -740.0, -1483.0, 1366.0, 491.5, -535.5, 1891.0, 2084.0, 1146.0, 567.5, -730.0, 316.5, 2202.0, 1482.0, -68.1875, 661.5, -1479.0, -121.6875, -420.25, 156.5, 687.0, 114.875, 1256.0, 294.25, -2035.0, -310.5, -284.0, 1779.0, -467.25, 965.5, -1618.0, -1315.0, 368.5, -38.15625, 1833.0, 1495.0, -40.75, 523.0, -209.625, -297.75, -1424.0, 1258.0, 64.0625, -143.125, -135.5, 1893.0, 139.625, -1360.0, -437.5, -1281.0, -25.484375, -870.0, -2.306640625, 1599.0, 461.5, -1680.0, 384.5, 585.0, 16.1875, 266.75, 1478.0, -674.0, -15.0859375, 1045.0, 210.0, 1443.0, -111.9375, 2048.0, 1248.0, -216.125, 355.0, -244.0, -683.5, 1190.0, -1407.0, -195.25, 954.0, -329.0, -1078.0, -976.5, 801.5, -1068.0, 101.75, -1559.0, 945.0, -266.25, 447.5, -577.0, -739.5, -891.0, 744.5, 1710.0, -717.0, -362.0, -600.5, 190.25, -963.0, -369.75, 18.953125, -53.84375, 226.875, -2.7890625, -1136.0, 9.9375, 212.25, -970.5, 955.5, 615.5, -1344.0, 121.9375, -450.5, 390.5, 25.046875, -15.5546875, 560.0, 1791.0, -67.5, 41.90625, -34.1875, -55.9375, 693.5, -126.3125, 561.0, -1116.0, -251.0, 789.0, -1610.0, -1172.0, 1093.0, -1424.0, -494.5, 577.5, -122.875, -1230.0, -119.8125, -844.0, -211.875, 607.0, -455.5, -129.25, -418.5, -240.625, -541.5, -1006.0, 594.5, 494.75, -374.75, -1113.0, 1712.0, 68.875, 192.5, 977.5, 1216.0, -411.25, 46.71875, -553.0, -377.5, -620.5, -888.0, 1103.0, -1184.0, -48.0, 128.125, -393.25, 759.0, -1845.0, 1069.0, -446.25, -1197.0, 1994.0, 1927.0, 109.1875, 647.0, -380.75, 458.25, -343.25, 380.0, -1745.0, 1259.0, -284.75, 598.0, -650.0, -21.078125, -1928.0, -897.0, -70.8125, -653.5, 2212.0, -334.5, -487.0, -188.75, 656.5, -1045.0, -466.25, 558.5, 310.25, -164.875, -583.0, -5.88671875, -730.5, 181.125, -563.5, -975.0, -3072.0, 108.0625, 833.5, 1251.0, -729.5, 100.875, 892.0, 1052.0, 459.25, -746.5, -461.0, -104.375, 475.25, 42.09375, -350.75, 1194.0, -905.0, -958.5, 294.5, 1766.0, -1311.0, 336.5, -783.0, 217.5, -1765.0, -1199.0, 468.75, -2412.0, 117.75, 655.0, 834.0, 891.0, 162.375, 178.75, 752.5, 37.5, 305.5, -698.0, -161.5, 858.5, 2160.0, 217.75, -185.5, -546.5, 1264.0, 1075.0, -671.0, -213.0, 793.0, -1716.0, 46.25, -623.0, -699.0, -1695.0, -498.5, -375.5, 708.5, -426.0, 408.25, 384.25, -1165.0, -520.0, 293.0, 67.0625, 385.5, 454.25, 1211.0, -345.75, 1603.0, 1209.0, -693.0, -302.5, 244.125, -73.1875, 724.5, 284.5, -963.0, -770.0, -1379.0, -1461.0, 1509.0, 79.5625, -1247.0, -876.0, 861.0, 100.6875, 1169.0, -673.0, 278.75, 83.6875, 983.0, -135.0, 330.25, 402.5, -648.5, 559.5, 896.0, -267.5, 518.0, 718.0, 1734.0, -180.5, -481.25, 1073.0, -1068.0, -543.5, 319.75, 302.75, -342.0, -895.0, 644.5, -1880.0, 760.0, -186.25, -1624.0, -573.5, -1110.0, -838.0, 1028.0, -12.1953125, 170.25, -1131.0, -150.25, 267.25, -903.0, -1063.0, -1126.0, -147.875, -1295.0, -1248.0, -664.0, -1222.0, -828.5, 425.0, 456.75, 131.625, -819.0, 1331.0, -1554.0, 156.75, 1375.0, 917.5, 529.0, -750.0, 38.75, -1229.0, 1195.0, 74.625, 1427.0, -1173.0, 1071.0, -751.5, 156.0, 1274.0, 386.75, 822.0, -718.0, -737.0, 1221.0, -1077.0, -788.0, -1451.0, -132.75, -727.5, -3002.0, -1170.0, -573.5, -1463.0, -443.75, -744.0, 1324.0, -898.0, 245.125, -391.25, 764.0, 901.0, -405.75, -902.5, -58.1875, 1429.0, 244.75, 964.0, -888.0, -1534.0, 1073.0, -933.5, 734.5, 550.0, 410.25, 1319.0, 643.5, -252.875, -196.125, -1173.0, 2270.0, 531.0, 575.5, -824.0, -972.5, 243.5, -678.5, 550.0, -55.625, -1368.0, 218.875, -1308.0, -1920.0, 104.875, -262.5, 868.0, 554.0, -1215.0, 436.0, 1335.0, 708.5, 950.5, 555.5, -973.5, -614.0, 337.5, -479.5, -386.25, 1069.0, -1966.0, 856.5, 111.375, -2021.0, -325.5, 1492.0, 792.5, 544.0, -427.25, -0.96875, -62.28125, 12.3046875, 456.75, -99.0625, 1720.0, -2576.0, -479.5, 1481.0, -396.25, 1136.0, -1214.0, 1511.0, -989.5, 634.0, -1386.0, 161.5, -1655.0, 75.5, -599.5, -43.8125, -878.0, -86.5, 104.8125, 963.5, -137.0, 126.6875, -2410.0, 1146.0, -776.5, -783.0, 739.0, 133.0, 1750.0, -247.875, -1488.0, -718.0, 396.0, -134.75, -654.0, -972.5, -1485.0, -982.0, 1180.0, 1478.0, -484.5, 1486.0, 358.0, 1133.0, 255.375, 960.0, -358.0, 625.0, 654.0, -644.5, -247.375, -309.25, -863.0, 1532.0, -209.75, -1761.0, -1573.0, -976.0, -201.375, 1058.0, 1553.0, -2216.0, -576.5, 1315.0, -1442.0, -1085.0, -175.125, 712.0, 75.625, 534.5, -289.0, 200.75, 398.5, -39.6875, -912.5, 788.5, 428.0, 1258.0, -612.0, 321.5, 969.0, -118.0, -933.0, 1136.0, -1213.0, -1088.0, 408.0, 124.3125, 1321.0, 259.0, -332.75, 1402.0, 892.0, -1213.0, -648.5, 898.5, -464.5, -1409.0, -705.0, -1065.0, -635.5, 552.5, -246.875, -2074.0, 850.0, 1032.0, 240.625, -1062.0, -1332.0, -1176.0, 464.25, -1.294921875, -1451.0, -1650.0, -1235.0, -1170.0, -390.5, -1610.0, -1716.0, -1143.0, -1388.0, -1041.0, -150.125, 347.5, 394.5, 1694.0, 1139.0, 686.5, -435.75, -627.5, -1775.0, -1481.0, 116.25, 477.75, 340.5, -2314.0, 1472.0, -637.5, 696.0, 490.25, 1273.0, -850.5, -2406.0, 715.0, 900.5, -556.0, -2708.0, -875.0, 566.0, 264.75, -1246.0, 229.875, -708.5, -464.75, 140.875, 730.5, -608.5, 778.0, -280.5, -363.75, 100.125, -99.5, 593.5, -819.5, -397.5, 51.9375, 1386.0, 337.5, -892.5, 655.0, 946.5, 306.5, 494.75, -1572.0, -1653.0, 1200.0, -895.5, 1366.0, 360.75, -808.0, -646.5, 668.0, -1262.0, 1730.0, 961.5, -1392.0, 1265.0, -2586.0, 362.0, 1489.0, 2640.0, 763.5, -2872.0, 949.0, 342.5, -1266.0, 90.5, -1245.0, -724.0, -1720.0, 581.5, 684.5, 207.125, 612.5, 878.5, -721.0, -136.75, -391.75, 563.0, 86.875, 600.5, 596.5, -1478.0, -1264.0, -200.5, 1551.0, 742.5, -886.0, -901.0, -547.0, 868.0, 327.75, -1758.0, 1038.0, 56.8125, 232.125, 1842.0, 795.5, 1086.0, -844.0, -469.25, 349.0, 60.125, -1287.0, 427.0, 1000.0, 1673.0, -522.5, 223.625, -1059.0, 656.5, -1248.0, -2498.0, 1146.0, -111.75, -15.5078125, 1841.0, 916.0, 1858.0, 1065.0, 222.0, 84.6875, 1343.0, -344.75, -293.75, 317.0, -715.5, 2518.0, 121.1875, -667.5, 2448.0, -785.0, -295.0, -322.25, 636.0, 1198.0, -619.0, 298.75, 2082.0, -530.5, -362.0, 1906.0, 1640.0, -1456.0, -1178.0, 1745.0, 842.0, -1397.0, -1846.0, -2148.0, 1037.0, -1036.0, -462.5, 1121.0, 543.0, -1261.0, -857.0, -894.0, -193.25, -552.5, -79.9375, 193.25, -230.0, -840.5, 112.4375, -598.5, 358.75, 587.0, 603.5, -591.5, 538.0, -1023.5, -30.140625, -529.0, -1972.0, -827.0, -660.5, -797.5, -308.5, -1078.0, 437.5, -1153.0, 415.75, -874.5, -1390.0, -1776.0, 453.25, -48.21875, 1252.0, -791.5, -2732.0, 1078.0, -1119.0, 1223.0, -1582.0, 926.5, 1015.5, 423.75, 939.0, -348.75, 1363.0, -742.0, 50.78125, -659.5, 2380.0, -139.75, -1754.0, -362.75, 1342.0, 28.59375, -997.5, -916.0, 235.125, 692.0, -1083.0, 1064.0, 85.9375, 289.25, -624.0, 948.0, -104.0625, -1369.0, -372.75, 333.75, -1080.0, 971.0, 683.0, 40.9375, 968.0, -23.703125, 60.75, 502.0, 437.25, 460.0, 82.625, -1186.0, 359.0, -333.25, -931.0, -60.90625, 1771.0, -93.8125, 1263.0, 699.5, -1470.0, -25.375, -528.0, -515.0, -585.0, -861.5, -1312.0, 1676.0, -504.0, 1839.0, -336.75, 1536.0, 595.0, 634.5, 254.625, 701.5, -138.125, 959.5, -85.75, 155.125, 365.75, 1513.0, -389.25, 1427.0, 967.5, 190.125, 518.0, 44.25, 1222.0, -2164.0, 1276.0, -1755.0, -232.25, 1554.0, 372.0, -1205.0, 286.75, -204.125, -334.0, -236.75, 1834.0, -611.5, -225.125, -1016.5, -279.25, 1554.0, 840.0, -119.3125, 951.5, -1823.0, 357.75, 723.5, 446.0, 420.5, 191.375, -1913.0, 798.0, 201.875, -1293.0, 818.5, 2200.0, -682.5, -285.5, -965.0, -610.5, 160.75, -337.25, -1435.0, 167.25, 70.5625, 633.5, 538.0, 2198.0, 1502.0, -1717.0, 802.0, -678.5, 1429.0, -462.75, 701.5, 149.0, 651.5, 1422.0, 1626.0, 363.0, -2762.0, -1347.0, -183.0, -692.5, -1275.0, 729.0, 15.1953125, -898.5, -894.0, 1639.0, -705.0, 62.125, -1904.0, -798.5, -231.5, 1704.0, 966.5, 938.0, -717.5, 147.25, -1791.0, 927.0, -2432.0, -184.875, -1319.0, 1699.0, 173.625, 1338.0, -114.0625, -1443.0, -526.5, -129.625, -341.0, -1783.0, 768.5, -346.75, -156.5, -235.625, 798.5, 1541.0, 756.5, -497.5, 72.1875, 172.875, 204.125, 1003.0, -48.71875, -505.75, -52.875, -531.0, -547.5, 571.5, 465.0, 889.5, 260.0, 8.6796875, -616.5, -1187.0, 1331.0, 232.625, 2006.0, -751.5, -605.0, 468.75, -1320.0, -1301.0, 181.625, -1018.5, 872.5, -1876.0, 1760.0, -361.75, 1095.0, 918.5, 1617.0, 14.140625, -179.75, -190.5, 232.375, -447.5, 301.25, -680.5, -449.75, -927.0, -1196.0, -2090.0, 712.0, 990.5, -1186.0, 845.0, 498.5, -1785.0, 560.5, 1438.0, -1293.0, -1897.0, 1558.0, 254.125, -666.0, -823.5, -648.5, 130.875, -1422.0, 386.75, 1216.0, 20.734375, 1334.0, -73.875, -237.375, 1636.0, -229.5, 561.0, 1645.0, 1142.0, 1138.0, 459.5, -1387.0, -956.5, 575.5, 830.0, -1837.0, -999.5, -211.25, -641.0, 391.5, -1816.0, -635.5, -2010.0, -1029.0, 347.25, -972.5, -601.0, 3174.0, -174.0, -1208.0, 772.0, -861.5, -758.0, -616.5, -1432.0, -667.5, -218.625, -1126.0, -1106.0, -286.5, -281.75, -299.0, 948.0, -369.5, 677.0, -664.0, -2650.0, -272.5, -802.0, 1556.0, -1054.0, 2037.0, 1486.0, 936.0, 2224.0, 407.5, 42.5, 633.5, -101.4375, -1687.0, -349.0, -749.0, -187.0, 54.6875, 588.5, 63.21875, 841.0, 1185.0, 262.25, 25.25, 2336.0, 435.75, -862.0, 513.5, -1313.0, -1215.0, 716.0, 942.5, 292.75, 1439.0, 1384.0, -105.3125, 235.125, -766.5, -557.5, 1788.0, -788.0, 664.5, -476.75, -434.0, -545.0, -837.5, 642.5, 1725.0, -1385.0, 1494.0, -503.5, -168.625, 746.5, 2678.0, 1262.0, -1077.0, 504.5, -392.25, -844.0, -551.5, -1363.0, -1332.0, -469.0, 739.5, 2090.0, -322.75, -720.0, -1462.0, -435.5, -42.84375, 1204.0, -301.0, 36.65625, 100.9375, -262.0, 598.0, -695.0, -1257.0, 133.75, -867.5, 286.25, 748.0, 939.0, 877.0, 549.0, 961.0, -992.5, -2358.0, 646.5, -71.6875, -2206.0, -209.375, -4.66015625, -216.625, 978.5, 474.5, 309.5, 134.125, 496.75, -676.5, 2998.0, -349.75, 372.75, -195.75, 1894.0, 1236.0, 167.5, -230.125, -488.25, -2330.0, -605.5, 423.0, 126.6875, -997.0, 793.0, 664.0, -137.625, 488.25, 1460.0, -837.0, 166.0, 719.5, -1002.0, 223.875, -2348.0, -629.0, 2440.0, -1417.0, 1548.0, 219.5, 1030.0, 716.5, -347.5, 518.0, -165.5, 1034.0, -1041.0, -765.0, -595.0, 519.0, -461.0, 902.0, 415.75, -719.0, -898.5, 1076.0, 264.5, 803.0, 730.5, 8.09375, -867.0, -29.359375, -115.625, 1542.0, -55.0625, -407.75, -1860.0, 1200.0, -966.0, -946.0, 1653.0, -1884.0, 334.5, 1.73046875, -1257.0, -362.25, -1023.5, 285.75, 971.0, -1688.0, 155.25, -28.921875, 2124.0, 1844.0, 156.5, 253.625, 485.0, -686.5, -34.125, -125.5, 1501.0, 887.0, 679.5, -1317.0, 754.0, -2414.0, -2380.0, 1706.0, 613.0, -696.0, 1907.0, 192.0, -890.5, -732.5, -2466.0, 241.25, 786.0, 474.25, 68.3125, 478.5, 1203.0, -293.75, 1268.0, -1522.0, 570.5, 1635.0, -716.0, -1234.0, -1874.0, -1746.0, 2368.0, 868.0, 95.5, -896.5, 719.0, 385.75, -1789.0, 1921.0, -2076.0, 1427.0, 963.5, -512.0, -910.0, 624.0, 294.5, -1771.0, 215.875, 2124.0, -1484.0, 1177.0, 775.0, 1676.0, -773.0, -750.5, 568.0, -1613.0, -752.0, -805.0, 358.0, -319.0, 439.25, 1842.0, -107.75, -207.375, 916.0, -1204.0, -167.375, -1114.0, 314.5, 1065.0, -327.25, 493.5, -822.5, -1236.0, -308.25, 7.40234375, -571.5, 95.875, 767.5, 1244.0, -1334.0, -232.625, -685.5, 960.5, 610.5, 830.5, -489.25, 420.75, 1723.0, -1497.0, -779.5, 422.0, -651.5, 341.75, -186.5, -537.5, 1124.0, -1050.0, 2140.0, -62.75, -1391.0, -1073.0, -266.75, 1104.0, 1205.0, 575.5, 1211.0, -1270.0, 24.390625, 83.25, -606.5, -1495.0, -512.0, 985.5, -464.75, -156.625, -1041.0, 271.75, 1581.0, 1622.0, 16.59375, -1203.0, 499.75, 31.3125, 230.125, 150.875, -470.25, 319.75, -2114.0, 521.0, 1253.0, -1449.0, -636.5, 139.125, -131.875, 543.5, 1.5498046875, 836.0, 307.0, -215.375, 498.0, -419.5, -1606.0, 173.375, -569.0, -1225.0, 679.0, 495.75, -1148.0, -408.0, -1746.0, 65.375, 940.0, 635.5, -2027.0, -193.875, -773.0, -833.5, -727.5, -594.0, 522.5, 200.125, -493.75, 71.25, 1360.0, -1817.0, 581.0, -1079.0, -1974.0, -1543.0, 2000.0, -1020.5, -1171.0, -728.0, -644.0, -473.5, -1331.0, -480.0, 572.0, 1421.0, -301.75, -1014.5, 1093.0, 3470.0, -863.5, -684.5, 1188.0, 716.5, -140.75, -167.375, 1173.0, -442.5, 1362.0, -22.625, 361.75, 771.0, -164.75, 154.25, 550.5, -430.75, -296.75, 1947.0, -143.375, 2432.0, -525.5, 1363.0, -170.625, -443.0, -965.5, -653.0, -73.6875, -739.5, -2084.0, -488.75, 460.0, -686.0, 439.0, 492.5, -35.34375, 843.5, 1121.0, 261.5, -451.25, -967.5, -49.78125, -246.75, 600.5, -722.5, -1159.0, 1282.0, 288.0, 1383.0, -823.0, -965.5, -1126.0, -24.625, -805.0, 1115.0, 1094.0, 1194.0, -667.0, -170.75, -525.5, 725.0, -667.0, -377.0, 827.5, -851.5, -1392.0, -6.0703125, 1056.0, 823.0, 793.5, 187.625, 166.0, 1095.0, 342.75, 327.0, -184.25, 1595.0, 54.71875, 3088.0, -1556.0, -45.46875, -709.5, 1749.0, -404.25, -287.5, 596.5, 587.5, 767.0, -1594.0, -1811.0, 148.625, -877.0, -586.0, 1162.0, -380.0, 556.0, 793.5, 1759.0, 1277.0, 3348.0, -120.125, 477.0, 671.5, 768.0, 363.75, -1734.0, 636.5, -955.5, 627.0, 2174.0, 582.0, -846.0, 476.25, -2082.0, -120.125, -1047.0, -92.9375, 1594.0, 1213.0, -991.5, -1954.0, -502.5, -103.625, 283.5, 1461.0, -593.5, -1164.0, -706.5, -472.5, -165.75, -511.25, 156.875, 386.25, 25.59375, -790.5, -559.5, 204.375, -586.5, 245.875, 503.0, 1112.0, -504.5, -194.375, -1716.0, 1081.0, 1883.0, -330.5, 218.75, -413.5, 1275.0, -622.5, 739.5, -1798.0, -209.625, 588.0, -206.0, -993.0, 878.5, 1600.0, 418.25, 230.875, -2634.0, -302.0, 1629.0, -462.5, 390.5, -282.75, 1013.5, 263.5, 940.5, 695.0, -1345.0, -2246.0, -769.0, 1563.0, -514.0, -1245.0, 618.0, -1116.0, -1343.0, -863.5, -1167.0, 1278.0, 683.0, 311.25, -928.5, -163.5, 10.796875, 557.0, -233.875, 431.0, 722.0, 951.0, -2376.0, -456.0, 982.5, 1309.0, -640.5, 220.875, -195.375, -1127.0, 339.5, -1822.0, -734.5, 453.0, -2008.0, 2308.0, -393.5, 916.5, 2260.0, 882.5, 1358.0, 949.0, -29.40625, 183.75, 877.5, 1643.0, -562.0, -644.5, -612.5, -1288.0, 1252.0, 805.0, 89.625, 666.0, 899.0, 491.0, -160.25, -1697.0, -520.0, 197.25, -530.5, -307.0, -515.0, -54.0, -443.5, 60.25, 410.5, -1143.0, -935.0, 537.0, -937.0, -817.0, -39.78125, -547.5, -315.75, -771.0, 596.0, 1671.0, -822.5, -495.25, 182.125, 2340.0, -1557.0, -1975.0, -173.125, -1126.0, 830.5, 476.5, -595.0, 974.0, -339.25, 423.5, -262.25, -953.5, -842.0, 457.25, -1564.0, -1480.0, -1073.0, -259.5, 266.5, -823.0, 796.0, -657.5, 276.5, -611.0, 1982.0, -452.0, 1642.0, -1245.0, -392.25, 422.0, 1711.0, -1301.0, -770.0, -538.5, 1451.0, -647.5, -780.0, -1464.0, 209.625, -700.5, -1986.0, 866.0, -1849.0, -807.5, 19.234375, -1531.0, -713.0, -15.484375, 620.5, 2588.0, 333.0, 1278.0, 807.0, 1708.0, -366.75, -374.25, -885.5, 589.5, -93.3125, 530.0, -663.5, 507.25, -922.0, 1848.0, 1308.0, 854.0, 2004.0, 1528.0, 1089.0, -294.75, -492.25, 181.75, 1283.0, -444.0, -909.5, 491.0, -1258.0, -108.9375, 499.0, 664.0, 143.25, -1037.0, 343.75, -1104.0, -62.40625, 400.75, 947.5, 2094.0, -1216.0, 915.5, -1559.0, 739.0, 333.0, 1567.0, 1028.0, 553.5, 154.75, 2864.0, -46.9375, -319.5, 565.5, -557.5, 866.5, -278.25, -1865.0, 1276.0, -132.5, 594.5, 2932.0, 1146.0, 2528.0, 62.5625, -116.0625, -706.5, 179.75, 3.171875, 1667.0, 277.25, -2824.0, 709.0, -342.25, 979.5, 332.25, -49.5, -949.0, 843.0, 745.5, 284.5, 1003.0, 1584.0, -29.328125, -614.5, -1612.0, 2578.0, -1418.0, -269.75, -703.0, -371.0, 1715.0, -2138.0, -857.5, -978.5, 1247.0, 2944.0, 1081.0, 84.25, -234.5, 969.0, 933.5, -1935.0, 378.25, 317.5, -100.9375, -74.75, -2350.0, 550.0, -138.625, -773.0, 1878.0, -593.0, 264.25, 1046.0, 167.625, 420.25, 981.5, -160.75, 1123.0, 719.5, 1660.0, 843.0, -746.5, 187.875, -244.75, -1137.0, -249.5, -192.25, 809.0, -1998.0, 528.0, 149.125, -587.5, 690.5, -301.0, 850.5, 495.25, -475.25, 1038.0, -660.0, 34.90625, -496.25, 988.5, 342.25, 1308.0, 1817.0, -325.5, -630.0, -535.0, -1257.0, 224.5, 1079.0, -1146.0, -428.5, -1099.0, 30.25, 61.71875, -679.5, -84.4375, 696.5, -252.5, -890.5, 2162.0, 400.75, 1027.0, 350.0, -261.0, -803.5, 1355.0, 1587.0, 1457.0, 1057.0, 58.15625, -101.75, -2120.0, -1477.0, 1060.0, 723.5, 36.125, -2070.0, 895.0, 257.25, 1640.0, 1779.0, 1935.0, -154.5, 935.0, -958.0, -1443.0, -686.0, 1146.0, 1460.0, 591.5, -4.9375, -972.0, 467.25, 321.5, -1442.0, -821.0, -1528.0, 319.5, 126.125, -1216.0, 861.5, -2068.0, 1104.0, 1305.0, 449.25, 663.5, -618.5, 736.0, -126.125, -1250.0, 1230.0, 256.5, -1011.5, 515.5, 242.125, 796.0, 85.75, 1820.0, -415.25, 621.0, -800.0, 1166.0, 941.0, 1189.0, -949.0, 440.75, -2604.0, -729.0, -920.5, -117.875, 10.359375, -453.75, -158.5, 167.875, 1388.0, 1467.0, -224.125, -1152.0, -1381.0, -224.875, -338.5, 1346.0, 803.5, -153.375, -820.5, -444.75, -837.0, -37.84375, -830.0, -321.75, 1793.0, -584.0, 405.0, -1033.0, 185.0, 1288.0, 962.5, 206.75, 188.125, 902.5, -1329.0, -242.125, 912.5, -319.5, -1025.0, -174.25, -1324.0, -263.75, -1925.0, -386.5, -480.0, -1190.0, -1065.0, -612.5, -193.375, -800.0, 807.5, 1268.0, -185.0, -557.5, 194.125, 2080.0, 294.25, -131.125, -533.0, -822.5, -1094.0, 1954.0, -990.0, -851.0, 1755.0, -553.5, -1389.0, -1112.0, -245.375, 1177.0, 494.75, 187.375, -440.0, 595.0, 695.0, 1767.0, -618.5, 856.0, -140.0, -384.0, -1191.0, 317.75, -285.75, 928.0, 298.5, 170.625, 434.5, 312.5, -96.1875, -2540.0, -293.5, -1150.0, -1803.0, 749.0, -1718.0, -144.875, 1692.0, -412.75, 149.125, -215.5, 382.75, -275.75, 163.0, -531.5, 992.0, 171.5, -1850.0, 1142.0, -1489.0, -477.0, -2080.0, -2682.0, 14.7578125, 216.625, -58.9375, -1288.0, -314.75, 839.0, 207.5, -309.0, -195.375, 1448.0, -1549.0, 467.5, 366.25, -1201.0, -1509.0, 968.0, -610.0, -1146.0, -1420.0, -1417.0, -163.875, -1087.0, 1637.0, 2600.0, -143.25, 1761.0, -131.75, -567.5, 957.5, -117.125, -789.0, 819.5, -626.5, 899.0, -288.0, -545.5, -1882.0, -2010.0, -231.875, 225.875, 625.5, -406.25, 2052.0, -8.5078125, 623.5, -1403.0, 22.9375, 1149.0, 2218.0, 729.5, 1351.0, -1962.0, 176.5, 797.0, 1165.0, -1129.0, 612.0, -186.0, -44.53125, 169.125, -693.0, 1977.0, 964.0, 718.5, -675.5, 483.5, 496.5, -517.5, -304.0, -340.75, 503.0, -9.953125, 446.25, 401.5, -992.0, 456.5, -1242.0, 729.0, 1621.0, 1377.0, -1182.0, -0.006259918212890625, 256.25, 124.1875, 215.125, -721.5, -1452.0, -59.6875, -423.5, -34.96875, 1044.0, 1821.0, -865.5, 811.5, -388.25, 115.9375, -1116.0, 1241.0, -983.0, 37.71875, -494.75, 2328.0, 1582.0, -4.84375, -64.6875, -129.125, 243.375, 168.5, -1334.0, 765.5, -1434.0, 928.5, -1276.0, -1422.0, -2037.0, 213.25, 330.75, -1489.0, -90.1875, 850.5, -2840.0, -488.0, -1290.0, -573.5, 264.25, -145.5, 1404.0, 65.125, -419.75, 1661.0, 301.5, -1357.0, 1622.0, 490.5, 91.625, -216.25, 1522.0, 116.375, -255.125, 522.5, 454.5, 1403.0, -171.0, -288.5, -1498.0, -279.25, -1397.0, 1895.0, -166.5, -851.5, 831.0, 1421.0, 242.625, -86.5625, 726.0, -664.0, -637.5, -319.0, 1358.0, -571.0, -1792.0, 1069.0, -467.25, 201.75, 163.375, 508.75, 1630.0, -2156.0, -1445.0, -322.0, 622.5, -1092.0, 452.5, -1426.0, 1179.0, 159.75, -430.5, -21.921875, 397.25, 881.0, 38.96875, 1897.0, -1175.0, 818.0, 247.875, 813.5, -244.625, 190.125, -197.625, -249.75, -276.0, 1511.0, -966.0, -1085.0, 198.5, -77.3125, -684.5, 1728.0, -470.5, 1729.0, 1658.0, -933.0, -178.0, -227.625, 644.0, 1224.0, -1406.0, 163.125, 709.0, -487.5, 1010.0, -2424.0, -1081.0, 83.75, -203.875, 1530.0, 372.5, -921.5, 565.0, -400.75, 2076.0, -967.5, 467.75, 994.5, -774.0, -90.375, 2450.0, 956.0, 86.25, 93.0, 43.0625, -489.75, 513.5, 58.25, -929.5, -211.0, 509.75, 245.125, -212.875, 988.0, 413.75, 190.25, -969.0, 555.5, -200.25, 220.0, -1787.0, 2542.0, -733.5, -596.5, 156.125, 242.375, 1345.0, 887.5, 1042.0, 1137.0, -1155.0, 344.75, 484.5, 52.40625, -200.0, 2228.0, -865.5, -468.0, -2122.0, 161.5, -2162.0, 288.5, -64.9375, 1657.0, -4.5625, 1671.0, 411.0, -572.5, 1573.0, 766.5, -1879.0, 1106.0, 1078.0, -1226.0, -1338.0, -704.5, -1688.0, 577.5, 140.125, 1109.0, 558.0, 109.5, -1745.0, 1161.0, 1543.0, 802.0, -1783.0, 239.5, 1260.0, 2678.0, -1278.0, 880.0, 38.40625, -241.625, 995.5, 712.5, 427.0, 915.0, 215.0, -223.125, -121.6875, 1314.0, -553.0, 1963.0, 1311.0, -125.5, 126.6875, -1287.0, 181.5, -437.5, -1635.0, -746.5, -1123.0, -1309.0, 673.5, 890.5, -706.0, 226.375, -476.0, -1186.0, 844.5, 1639.0, -30.734375, 173.5, -1209.0, 1372.0, 152.25, -764.0, 414.5, 1683.0, 623.5, 376.25, 2578.0, 491.75, 2021.0, -627.5, -1115.0, -188.125, -503.5, -1111.0, 619.0, -472.75, -1623.0, 167.375, 840.5, 1441.0, 447.5, 1128.0, 1028.0, -0.90087890625, -296.5, 174.375, -1129.0, 649.5, -649.5, -831.0, 1192.0, -9.8828125, -335.5, -494.0, -703.0, -622.5, -564.0, 578.5, -1130.0, 850.0, 573.0, 1410.0, -1925.0, 46.84375, -41.03125, 2062.0, -279.25, -825.5, -2852.0, -245.25, 41.1875, -538.0, -1554.0, -432.5, 820.0, 64.6875, 399.0, -387.0, 649.5, -607.5, -2080.0, 273.75, -1409.0, -2788.0, 693.0, 417.25, -546.0, -1435.0, 1669.0, 2047.0, 1452.0, -747.0, 604.0, 1098.0, -147.875, -671.5, -1238.0, 408.0, 1556.0, -747.5, -1416.0, 691.0, -405.75, 851.5, -882.0, -713.0, 752.5, -1337.0, -817.0, -2154.0, 93.1875, -123.3125, -2354.0, 450.75, 145.375, -937.0, 1522.0, 1014.5, 513.5, -653.5, 157.5, -350.5, -1157.0, 582.5, 411.5, -1293.0, -1388.0, 2027.0, -24.3125, -88.75, -325.5, 1170.0, 1582.0, 8.0703125, 524.0, -1312.0, -273.75, -218.375, 203.5, 569.5, -789.0, 21.5, -2654.0, -361.0, 1252.0, 1146.0, 1185.0, -2308.0, 621.5, -533.0, -989.0, 918.5, -1152.0, 1438.0, -372.25, -496.5, 1187.0, 448.5, -718.0, 815.0, -274.75, 635.0, 815.5, -1427.0, -870.5, 468.75, 1101.0, -1067.0, 254.625, 109.8125, -800.0, -574.5, -259.75, 759.0, 1743.0, 15.3671875, -15.578125, -1014.5, 1733.0, -826.0, 1041.0, 1004.0, 623.0, 384.5, 469.25, 1008.0, -1080.0, 524.0, -650.0, -401.25, -229.25, 630.5, 1613.0, -921.5, 1194.0, 1037.0, -1101.0, 596.5, -1012.5, 843.0, -1622.0, -388.0, 441.0, -944.0, -423.0, 540.5, 853.5, 1223.0, -181.375, -246.0, -272.25, -818.5, 265.5, 339.5, 93.375, 1319.0, 1190.0, 1379.0, -2006.0, 1078.0, -165.375, 940.5, 566.0, 603.0, -825.0, -948.0, -699.0, -1345.0, 397.5, -1411.0, -546.5, -993.0, 1794.0, 1199.0, 131.125, 1378.0, -18.875, 856.0, 627.0, -1192.0, -600.0, -146.5, 696.0, 396.0, -1403.0, -1218.0, -80.4375, 576.5, -1117.0, 704.0, 317.25, -1354.0, -1552.0, -238.125, 285.0, 1189.0, -1418.0, -580.0, 155.25, 498.5, 177.0, -1739.0, 453.5, -2148.0, -428.75, -1122.0, 764.0, 2016.0, 1899.0, -40.125, -1105.0, 205.75, 1378.0, -654.5, -373.5, -379.5, 105.5, -1203.0, 339.5, 387.75, -227.5, 240.0, -26.640625, 380.25, -1046.0, -477.5, -383.0, 176.125, -922.0, -364.5, -1165.0, -1337.0, 123.0, -1108.0, 1285.0, 198.375, -863.5, -427.0, 377.5, 395.5, 926.5, 903.0, -898.5, 639.5, -1626.0, -372.0, 27.53125, 234.625, -497.75, -1201.0, -947.5, 808.0, 183.0, 709.0, -509.75, 888.0, 539.5, -955.5, -469.75, 11.859375, 97.25, -515.0, 470.25, 198.625, 1615.0, -188.375, -541.0, -1130.0, 821.5, -6.1484375, -209.5, 114.6875, -929.5, -1008.0, 783.0, -541.0, -1123.0, 847.5, -1956.0, 287.5, -1534.0, 526.5, -523.5, -5.6171875, -1781.0, 175.125, -453.75, 831.0, -148.125, -152.875, 2298.0, 1081.0, -337.75, -109.5625, -800.5, 64.1875, -228.875, 930.5, -882.0, 321.25, 422.75, 1366.0, -226.5, 301.0, -378.5, -1008.0, 142.5, 449.5, 122.9375, 650.0, 1025.0, -1556.0, 944.5, -373.25, -952.5, 95.9375, 435.0, 1652.0, -1225.0, 412.0, 153.875, 1269.0, -1185.0, 509.25, 2008.0, -1588.0, -198.5, 782.5, 694.0, 4.3828125, -441.25, -6.44921875, -551.0, 120.5625, 781.0, 904.5, -450.25, -130.375, 655.0, 1646.0, 252.0, 1212.0, -1423.0, -1938.0, -1491.0, -893.0, 580.0, -1044.0, -1019.5, 1306.0, -1198.0, -653.0, 312.5, 778.5, -695.5, 1623.0, 208.25, 205.875, -775.5, 899.5, 959.0, 1036.0, 531.5, 566.0, -143.125, -958.0, -428.75, -1055.0, 1443.0, 392.25, 664.0, 1513.0, 100.25, -680.5, 1754.0, 148.875, -371.5, -774.0, 1760.0, 1978.0, -2310.0, -491.0, 1796.0, -1127.0, -2914.0, 722.5, 625.5, 263.75, -277.5, -1771.0, -628.5, 702.0, 1331.0, -1903.0, 490.5, -186.25, 418.75, -66.75, -307.5, 785.5, -1109.0, 199.0, -708.0, 1230.0, -260.5, 1089.0, 944.0, -741.0, 30.34375, -1164.0, -1215.0, 675.0, -479.25, 347.5, -177.5, -616.0, -1293.0, -566.0, 1677.0, 477.75, -136.125, 613.0, -1765.0, -977.5, 639.5, -824.5, 3.232421875, 171.0, -155.0, -1607.0, 388.5, 1426.0, 796.5, 125.875, -1211.0, 1961.0, -72.25, -278.25, 1421.0, 187.0, 2258.0, -10.8046875, 494.75, 417.75, -773.0, -795.5, -2308.0, 1479.0, -700.0, 335.0, -26.78125, -57.28125, 457.0, 1357.0, -527.5, 1437.0, 192.625, -1277.0, 2110.0, -1664.0, -287.75, 585.0, 927.5, 186.625, -1435.0, -1164.0, 1000.0, 506.75, -239.0, 808.0, 401.0, 1756.0, 993.0, -160.25, -1774.0, 84.25, 565.0, -340.0, -772.0, 271.75, -496.25, 699.5, -158.5, -273.75, 367.0, 889.0, 143.125, -1099.0, -555.5, -629.5, 1002.0, 946.0, -682.0, -594.5, -164.0, 1299.0, 1538.0, -1167.0, 1795.0, -1056.0, -9.8828125, -701.0, -2234.0, -372.75, -557.5, 927.0, -300.0, 590.0, -321.5, -1156.0, 217.625, -3346.0, -939.5, -325.5, -359.5, -387.5, -232.625, -1548.0, 200.125, 30.015625, -2170.0, 230.125, 537.5, -308.5, -1384.0, 126.125, -662.0, 525.5, -181.0, -585.5, 617.5, 113.5, 486.25, -1182.0, -1058.0, 131.25, -545.5, 1608.0, -1332.0, 318.25, 1518.0, 166.25, 306.5, 1534.0, 2144.0, -3.19140625, 77.875, -411.25, -315.0, -1146.0, 307.0, 929.5, 150.125, -825.5, 689.5, -458.25, 2094.0, -525.0, -798.5, 441.0, 122.375, 348.75, 1925.0, 375.75, -406.5, 946.0, 2448.0, -938.5, -582.5, -444.0, -1268.0, -280.25, -1241.0, -2448.0, 1870.0, -486.75, -360.75, -483.0, 1111.0, 1146.0, -1457.0, -199.25, -222.0, -1869.0, -1411.0, 854.0, -763.5, -291.75, 1094.0, 11.25, -220.875, -1215.0, 1224.0, -1194.0, 211.375, -119.4375, -37.875, -345.0, 285.25, -175.0, 436.75, -672.0, -1317.0, 436.25, -841.0, 1479.0, 589.5, -1279.0, 21.03125, 1170.0, -708.0, -1143.0, -166.125, 2530.0, 398.25, -226.375, 669.0, -127.5625, 1083.0, 506.75, -1791.0, 284.25, 694.5, 1329.0, -903.0, -1559.0, -3.61328125, 690.5, 31.0, -456.75, 655.5, -216.25, -1765.0, 644.0, -844.5, 478.75, -193.0, 253.875, 25.71875, 399.5, -529.5, -1038.0, -297.0, -1198.0, 1843.0, 1373.0, -543.5, -64.9375, -97.0625, 942.5, 204.5, 915.0, 491.0, -913.5, -19.75, -913.0, -1262.0, -1974.0, -1117.0, 1349.0, -906.0, 649.0, -1199.0, -46.8125, -1146.0, -1066.0, -1672.0, 390.75, -321.75, 34.15625, 967.0, 425.0, 224.625, 140.125, 655.0, 1422.0, 2458.0, 1838.0, 661.0, 615.5, 590.5, -1104.0, -293.25, -31.28125, 2094.0, 889.0, -2092.0, 364.25, 2280.0, -887.0, 388.75, 811.0, 1553.0, 151.625, -1229.0, 330.25, 25.875, -449.25, -988.5, -823.0, 365.0, -757.5, 94.125, -14.921875, 400.5, 1282.0, 472.25, -1929.0, 658.5, 280.25, 288.5, 657.5, -116.375, -419.25, -439.25, -684.0, -90.8125, 545.5, -763.0, 562.0, -868.0, -1391.0, -331.0, -148.875, -959.5, -959.0, -752.0, -55.8125, -168.375, -313.5, -551.0, -860.0, 1521.0, 531.0, -563.5, -401.0, -30.859375, -720.0, -191.375, -449.5, 309.75, -216.625, -1378.0, -1309.0, -1504.0, 2638.0, 47.0625, -694.0, -27.828125, 184.875, 274.75, 486.5, -766.5, -245.5, -420.75, -75.0, -921.0, 1016.5, 377.5, 314.0, 2378.0, -562.5, 1146.0, 933.5, 310.25, 454.25, 1901.0, 328.0, 650.5, -865.5, -677.0, -854.0, -875.0, 377.0, 1126.0, -910.0, 678.5, -344.25, -1277.0, 32.875, -468.0, -692.5, 40.375, -997.0, -589.0, 1371.0, 1466.0, 197.625, -754.0, -411.0, 1973.0, 477.5, -801.0, 2316.0, -757.0, -2002.0, -829.5, 1143.0, -1526.0, -1693.0, -138.875, -110.3125, 286.5, -240.625, 507.25, -1230.0, 237.5, -588.5, -308.25, -1184.0, -17.90625, 1907.0, -11.46875, 1335.0, 565.5, -117.4375, -1052.0, -357.0, -209.875, -522.5, -583.5, -553.0, -1512.0, 1549.0, 1354.0, 1042.0, 552.5, -512.5, -117.0, 533.0, -693.5, 2018.0, -993.5, 961.5, 1010.0, 755.0, -402.25, 2108.0, 505.0, -176.25, 787.0, -2450.0, -262.5, 192.625, -312.25, -824.0, 40.375, -1574.0, 519.5, 194.0, 383.5, -1169.0, -53.5, 569.0, 1763.0, 489.0, 1020.5, -1276.0, 163.625, 1352.0, -78.0, -1190.0, 1343.0, 520.0, 780.0, -461.75, -1149.0, -1019.5, -1240.0, 341.75, 480.25, -1325.0, 2514.0, 1355.0, -1571.0, 2204.0, 178.875, 2218.0, 2864.0, -2402.0, -170.25, -857.5, 55.3125, -20.328125, -1602.0, 342.25, 741.0, -546.0, -1229.0, 419.5, -382.25, -118.5625, -1008.0, 268.25, -1370.0, -318.25, -2086.0, 94.0, 580.0, 456.0, -148.5, -521.5, -1564.0, -545.5, -743.0, 1123.0, 689.5, 881.5, 115.375, -1117.0, 173.875, 387.75, 532.5, -1808.0, 858.0, 2120.0, 1421.0, 1011.5, 612.0, 1289.0, 377.0, -102.875, -279.25, -583.5, 801.0, 1350.0, 1473.0, 813.0, -269.25, -1282.0, 775.5, -369.75, -114.5, 702.5, -120.25, -1063.0, -1856.0, -317.75, -170.25, 181.125, 370.25, 1035.0, -595.5, 444.25, -2424.0, -290.0, 721.5, 381.75, 580.5, -1055.0, -96.25, -1064.0, 1378.0, 44.40625, -112.375, 1120.0, 594.5, 206.0, -496.0, -1334.0, -1490.0, -1377.0, -802.5, -137.875, -1995.0, 298.25, 169.25, 234.0, -30.96875, -1090.0, -980.5, 697.0, 60.59375, -2142.0, 1007.0, -597.5, -752.0, 260.0, 359.5, 852.0, -401.75, -931.0, 586.5, 9.6015625, 253.625, -1623.0, 1289.0, 175.0, 265.25, 1585.0, -392.0, 469.0, 108.4375, -853.0, -1782.0, -460.25, -595.0, -1076.0, 836.5, -557.5, 191.0, -227.625, -127.4375, -322.5, -1584.0, -494.25, -797.5, -111.3125, -503.0, -723.0, -277.25, -1700.0, 651.0, -1485.0, 329.0, 1632.0, 343.25, -566.5, -930.0, 615.5, -568.0, -922.5, 313.75, -531.5, -1379.0, 1783.0, -2090.0, 753.5, -307.25, -138.0, 1506.0, -1299.0, 241.75, -6.64453125, -2476.0, -201.125, 1801.0, 1321.0, 1124.0, -730.5, 1080.0, 244.5, -1217.0, -1751.0, 742.5, 324.75, -809.5, -1130.0, -325.0, -850.0, -772.5, -1973.0, 694.0, -736.0, 85.25, 1382.0, -280.75, -1397.0, -479.5, -1327.0, 366.5, 381.0, -407.5, -359.25, 99.25, 3154.0, 418.75, -372.0, -1247.0, 1080.0, 1068.0, 187.75, 1162.0, -1132.0, 568.5, -1079.0, -855.0, -966.0, -744.5, 771.5, 232.125, 636.0, 1322.0, -662.0, -259.25, -45.53125, -939.5, 842.5, -2566.0, 148.5, 224.25, 72.5, -1571.0, 369.5, 677.0, -876.0, 176.625, 39.96875, -1063.0, -390.0, 1323.0, 265.5, 196.25, 385.75, -933.0, 18.953125, 212.25, -1014.5, 1102.0, -799.5, -447.5, -526.0, -432.25, -1314.0, -427.0, 1195.0, -1361.0, 407.5, 1772.0, -2406.0, -2280.0, -349.5, 159.125, 1313.0, -597.0, 147.0, -519.0, -716.5, -279.0, -1187.0, 1350.0, 175.25, 237.75, -1781.0, -867.0, 46.65625, -298.75, 1199.0, -958.5, 1190.0, -943.5, 356.25, -1605.0, -1010.5, 859.5, -785.5, 741.0, -136.125, 358.75, 1448.0, 483.25, -221.0, -757.5, 961.0, 229.75, -355.5, 467.5, 69.5, -327.75, 2154.0, 571.0, 19.484375, -78.6875, 501.0, -1010.5, 908.0, 321.25, 591.0, 404.25, -424.0, 468.25, 790.5, -240.0, 493.5, -19.40625, -553.5, -972.5, 1241.0, 1054.0, 1836.0, -311.25, 971.5, -248.0, -576.5, -390.25, 727.5, -62.5, 1090.0, 328.25, 589.5, 437.5, 1180.0, 1426.0, 2666.0, 1550.0, 105.0, 1461.0, -456.25, -373.0, -531.0, 1513.0, -1519.0, -609.0, 813.5, -296.0, -215.5, -303.5, -2920.0, 689.0, -97.5, -1735.0, 1591.0, 169.375, 358.0, 442.5, -458.25, 1879.0, -116.625, 1068.0, -1228.0, 1231.0, 1062.0, -9.7578125, 444.75, -374.0, -267.5, 945.5, -1982.0, -555.0, 103.25, -643.5, -783.0, 2016.0, -1006.0, 3238.0, -2094.0, -1639.0, -781.5, -906.0, -967.5, 813.5, -1125.0, 112.75, -2228.0, -1094.0, 1963.0, 1592.0, -637.5, 163.25, 448.5, -575.0, -508.25, -338.5, 1548.0, 45.75, 591.0, -3.23046875, 1400.0, -919.0, 33.25, 658.0, -689.0, -426.25, -1606.0, -769.0, -145.375, 410.5, 871.0, 129.375, 881.0, 778.5, 143.5, -230.875, -458.25, -406.5, -321.5, 480.75, 24.984375, 1088.0, 439.0, -1294.0, -1288.0, 956.5, -235.75, -168.375, -91.0625, 1912.0, -226.875, 305.5, 226.625, -1127.0, 74.4375, 592.0, -734.0, 1116.0, -1173.0, -68.5, -568.0, -976.5, 2266.0, -198.375, -221.75, 143.5, 705.0, -669.0, 278.75, 71.75, 1177.0, 380.75, -54.21875, 352.25, -307.0, 301.5, 1316.0, 46.21875, 420.25, -109.75, 699.5, -1935.0, 315.75, -817.5, 1065.0, 574.0, -1200.0, -911.0, -281.25, -1712.0, -872.0, 200.0, -104.125, -726.5, 475.5, -218.75, -1092.0, 844.0, -444.0, 150.625, -458.75, -2182.0, 802.0, -419.25, 412.5, 794.5, -1672.0, -1618.0, -3074.0, 224.125, 1164.0, 790.5, 893.5, 1093.0, 1547.0, -1170.0, -1732.0, -1294.0, 889.5, 158.0, -2248.0, 1438.0, -1468.0, -1900.0, 964.5, -1978.0, 668.0, -392.0, -1273.0, 1272.0, 1453.0, -121.875, 1925.0, 1372.0, -673.5, 797.0, 831.0, -454.0, 660.0, -657.0, 2574.0, -358.25, -252.25, -766.5, -1012.5, 667.0, 1780.0, 213.75, -1374.0, -379.0, 480.75, 75.5, 164.375, 1787.0, -125.375, -252.0, 1465.0, -827.5, 218.0, 1109.0, 323.75, -980.5, 1351.0, 212.125, -819.0, -1858.0, 95.9375, -1904.0, -1888.0, -317.75, -1181.0, -1313.0, -333.25, -617.0, -1005.0, -96.6875, -961.0, -572.5, 1321.0, 618.0, -566.5, 793.5, -760.5, -96.0625, -554.5, -1950.0, -665.0, 493.5, 1064.0, 503.0, -1225.0, 276.5, -1549.0, -818.5, 2023.0, 578.5, -392.0, -948.0, 1397.0, -927.0, 1488.0, -728.0, -705.0, 650.5, -153.75, -1584.0, -373.75, 648.5, -698.0, -702.0, 1618.0, 957.0, 1916.0, 2056.0, 1290.0, 1714.0, -640.0, 847.5, 381.5, 91.25, 1022.5, 327.75, -3034.0, 407.5, -1119.0, -78.625, -1291.0, -2484.0], "dtype": "float16"}], "e": {"__nd__": [1539.0, -1637.0, 1440.0, 1657.0, 510.0, -953.5, 1427.0, -915.0, 604.0, -194.0, -1374.0, 2724.0, 2702.0, -851.0, -360.5, 1632.0, -932.5, -1342.0, 741.5, 2062.0, 1631.0, -182.5, -1762.0, -709.5, -320.5, -49.0, -1928.0, 1214.0, 1606.0, 152.25, 3200.0, 744.0, 345.5, 674.0, 414.25, 2264.0, 473.0, -2156.0, -429.5, -581.0, 559.5, -917.5, -903.5, 115.25, -1407.0, -561.0, -1013.5, -433.0, 1278.0, 1211.0, 361.0, 321.0, -460.75, -286.5, 488.0, 11.125, -819.0, 789.5, -866.0, 486.75, -1878.0, 955.0, 909.0, 492.0, 2460.0, -1999.0, 349.0, 1312.0, -385.0, 465.5, -440.5, 82.0, -523.0, 2666.0, -1493.0, 50.0, 306.0, -2080.0, -1876.0, 551.5, 951.0, -2238.0, 493.5, 383.0, 868.0, -976.0, 2000.0, -1102.0, 545.5, -655.5, 261.0, 1164.0, -644.0, 2260.0, 3384.0, -383.5, -290.5, -2022.0, -1265.0, -1232.0, 556.0, 1640.0, 204.625, 693.5, -2816.0, 550.0, 984.0, 2194.0, -1856.0, -2156.0, 399.0, -1093.0, 2258.0, -337.75, -123.5, 710.5, 242.0, -442.75, -59.125, 176.0, 824.5, 1226.0, -699.0, -1470.0, -1842.0, 1030.0, 1274.0, -226.875, -1877.0, 1478.0, 874.0, -449.5, -321.5, -980.5, 597.0, -885.0, -829.5, -541.5, -369.0, 151.5, 467.5, -3120.0, -450.0, 1124.0, -2660.0, 1211.0, -883.0, -398.5, -537.0, -1795.0, 2864.0, -1301.0, 3076.0, 293.0, -417.0, -1988.0, 974.0, 418.0, 219.0, -856.5, 256.75, 1277.0, 883.0, 689.0, 197.25, 771.0, -1185.0, -21.0, -1604.0, 81.5, 2436.0, -310.5, 1947.0, 1560.0, 737.0, 2554.0, 3008.0, -845.5, -538.5, 489.25, 351.0, -1496.0, 1348.0, 594.0, -1139.0, -2220.0, -1095.0, 638.0, 1665.0, 2868.0, -960.0, 1898.0, 254.75, -469.5, -274.0, -15.0, 789.0, 2424.0, 2374.0, 1359.0, -13.5, 1166.0, -1032.0, 2336.0, 413.5, -1765.0, -1170.0, 2112.0, -1380.0, -1181.0, 591.0, 1396.0, -2004.0, -1225.0, -157.25, -1331.0, -139.5, 497.0, 282.0, 996.0, 424.25, 1234.0, 284.0, 720.5, -879.5, 1270.0, -1938.0, -891.5, -204.5, 150.125, -592.5, 135.0, -730.0, 630.5, 377.5, -1168.0, 1192.0, 175.0, 1074.0, 325.0, 1962.0, -236.25, 1214.0, -448.5, 68.0, 1.0, 1506.0, -1245.0, 1491.0, -784.5, 1776.0, -393.25, -1358.0, -573.5, 367.5, -1557.0, 2122.0, 1195.0, -1148.0, -2019.0, 2188.0, 4536.0, -1449.0, 1373.0, 1073.0, 2100.0, 1467.0, 457.5, 86.3125, -664.0, 1172.0, 228.5, -100.75, 816.0, 911.0, -281.0, -1733.0, -280.75, 1478.0, -2144.0, -873.0, -2320.0, -1092.0, -1391.0, 1140.0, 2236.0, 667.0, 2798.0, 496.0, 421.0, -2194.0, -98.75, 24.0, -308.0, 715.0, 445.5, -1195.0, 2560.0, 2836.0, 3094.0, 2230.0, 1051.0, 1166.0, 251.0, -287.25, -835.0, -1960.0, -446.5, 1258.0, 480.5, 2468.0, -2332.0, -917.0, 3060.0, -113.75, -1469.0, 1177.0, 715.0, 2304.0, -1351.0, -803.5, -2140.0, -2796.0, 397.0, -1686.0, -414.75, -1274.0, -609.0, -1319.0, 1574.0, -2358.0, -958.5, 1968.0, -2850.0, -3038.0, -823.0, -128.5, -778.0, -2698.0, 511.0, -438.0, -426.25, 352.5, 540.5, 80.5, 481.5, -379.0, -767.0, -1610.0, 2582.0, -493.25, -516.0, -2202.0, 1370.0, -2740.0, -104.25, -2312.0, 391.25, -1927.0, -1244.0, -1969.0, -1291.0, 509.75, 324.5, 325.5, 1257.0, 17.125, -1606.0, 1577.0, 758.5, 2322.0, -330.0, -1568.0, -1778.0, 1232.0, 282.5, 337.0, 950.5, -1074.0, 2370.0, -595.0, -1712.0, 1466.0, -747.0, 157.25, -1145.0, 402.0, 112.25, 1484.0, -720.0, 132.5, -757.5, 32.25, 364.5, 3944.0, 2342.0, 88.125, 1907.0, -1716.0, 1233.0, 1322.0, 243.75, -128.5, 999.0, 1166.0, 1696.0, 300.5, 691.0, -916.0, -764.0, -1188.0, 1212.0, -1504.0, -123.875, 43.0, -3690.0, 673.0, -2072.0, 660.5, -575.0, 1542.0, 3016.0, -1618.0, -499.25, 915.0, -1628.0, 2032.0, -616.5, -214.0, -839.5, 146.625, -981.0, -14.0, 2616.0, -1174.0, 114.0, -578.0, 1104.0, -588.5, -1638.0, -936.0, -621.5, -1149.0, -1455.0, 1675.0, -113.75, 932.5, -1202.0, -645.5, -774.5, -2680.0, -1321.0, -1042.0, 1074.0, 220.0, -1416.0, -1435.0, 1236.0, -1470.0, -509.5, -2442.0, 269.25, -2532.0, -2404.0, -1011.5, -163.5, 2148.0, 490.25, 261.0, -340.5, 385.5, -926.0, -1506.0, -394.75, -1944.0, 2156.0, -1308.0, -1592.0, 1386.0, -445.5, -1003.5, 19.0, 158.0, -967.0, 1782.0, 391.75, 1738.0, 1018.5, 681.5, -1043.0, -1394.0, -610.0, 470.25, 160.375, -463.25, -1690.0, -841.5, -150.0, -1936.0, -1012.5, 572.5, 3820.0, 2248.0, -1500.0, -1909.0, -95.5, 199.25, 1078.0, -2366.0, -2686.0, 972.0, -138.0, -1517.0, 1660.0, 1090.0, 329.5, -1617.0, 731.0, 2184.0, -379.0, -1513.0, 783.5, -1886.0, -707.0, 2834.0, -2168.0, 1255.0, 1350.0, -1057.0, -743.0, 625.0, -5.0, 444.0, 114.625, -839.0, -1047.0, 319.25, 1385.0, -1522.0, 330.25, 1864.0, 1498.0, 2384.0, 771.0, 747.0, 835.0, -410.0, 1526.0, 2856.0, 213.25, 1047.0, 218.875, 2186.0, 2326.0, -3240.0, 2556.0, 1523.0, -1121.0, 755.0, 1464.0, 2612.0, -503.0, -1852.0, -3174.0, -1913.0, 1026.0, 3034.0, 361.0, 226.0, 397.75, -973.0, -1952.0, 1005.5, 3480.0, 1262.0, 820.0, 3664.0, -65.0, -781.0, -187.5, 34.75, 962.0, 622.0, -235.0, -561.5, 1606.0, 1683.0, 1620.0, 64.5, -1238.0, -369.5, -765.0, 716.5, 734.0, 955.5, -214.25, -193.0, 1168.0, 587.0, -1178.0, -450.5, -1685.0, 310.0, -162.875, 1281.0, -947.0, -2992.0, 63.625, 863.0, 3680.0, 970.5, -156.75, -1191.0, 1133.0, 712.0, -1208.0, 933.0, 502.5, -383.0, 977.0, 1118.0, 1762.0, -766.0, 3014.0, 1942.0, 2380.0, 181.75, -451.0, -1194.0, 1198.0, 525.0, 2246.0, 2916.0, -425.0, 757.0, -1813.0, 473.0, -395.25, -1656.0, -82.5, 168.0, -1032.0, 273.5, 717.0, -3568.0, -1160.0, -888.0, -343.5, 609.0, -828.5, 965.0, 1795.0, -1256.0, -3106.0, 1630.0, 668.5, 670.0, -643.0, 39.5625, -60.0, 1057.0, -183.5, -36.0, -534.0, -1076.0, 273.0, -819.0, -43.0, 479.75, 2492.0, 2232.0, -1632.0, 452.0, -1726.0, 596.0, -2494.0, -71.5, 410.5, -1414.0, 1316.0, 148.5, -829.0, -959.0, 3172.0, -229.75, 206.0, 532.0, -453.5, -94.5, -1659.0, 484.0, 1783.0, -1617.0, 2234.0, -592.0, -489.75, 846.5, -706.5, 1117.0, 403.0, 1026.0, 316.5, 367.0, -959.0, 1662.0, 445.0, 1577.0, -1671.0, 87.5, -844.0, 1178.0, 182.75, 677.0, 144.125, 475.5, -1768.0, -40.5, 860.0, 3982.0, 2772.0, -574.0, 936.5, 2624.0, -1518.0, -978.5, -234.0, 1314.0, -591.0, -1025.0, -1316.0, 1862.0, 1658.0, 981.5, -122.5, -881.0, -322.0, -2866.0, 2862.0, -447.25, 1844.0, 1373.0, 2140.0, 832.5, 1342.0, -1688.0, 1402.0, -2078.0, -650.5, -447.25, 2404.0, -115.0, 856.0, 122.0, -834.0, 3362.0, 35.5, -497.0, -424.0, -1403.0, -3.0, 432.75, 596.5, 387.0, 392.0, 2278.0, -462.5, -2246.0, -94.5, -352.5, 1032.0, 786.5, -2424.0, 617.5, -1681.0, -2456.0, 1154.0, 642.0, 128.25, -240.375, -1810.0, 1449.0, -924.0, -429.5, -670.5, -920.0, -1322.0, -3692.0, 673.5, -1400.0, 493.5, 455.0, 163.0, 1514.0, 965.0, 341.5, -986.0, -648.5, -2590.0, -111.0, 201.0, -2308.0, -1532.0, -582.0, 818.5, -589.0, -259.0, 2504.0, -756.0, 1852.0, -971.5, -12.0, -722.5, 800.0, -1570.0, -2978.0, -232.0, 919.0, 2712.0, 1570.0, 381.75, -104.25, -1052.0, -1117.0, 579.5, 1912.0, 283.75, -1456.0, -616.0, -292.0, -733.0, -1814.0, 2586.0, 576.5, -1061.0, 1256.0, -1586.0, -476.5, -1490.0, 1173.0, 2900.0, -280.0, -851.0, -1518.0, 1672.0, -288.25, -122.125, -10.125, 493.25, -656.5, -4496.0, 3576.0, -1298.0, 2518.0, 994.0, 1453.0, -785.0, -2366.0, 1029.0, -61.5, 20.0, -2164.0, 1054.0, 1928.0, 2512.0, -726.0, 2072.0, 2086.0, 292.0, -152.5, -324.5, -1338.0, -1066.0, 452.5, 1708.0, 464.0, 310.0, -522.5, -1487.0, -924.0, -925.0, -136.75, -723.5, -380.5, 567.0, 1454.0, -1213.0, 1914.0, 1836.0, 1302.0, 2062.0, -1640.0, -251.5, -2856.0, -1970.0, 1124.0, -566.0, -786.0, 899.5, -825.5, 1511.0, -557.0, -268.0, -1527.0, -66.0, 851.0, -2424.0, 857.0, 2378.0, 4724.0, -129.0, 2052.0, 244.0, -767.5, 255.0, -1046.0, -1450.0, 1810.0, -609.0, -3758.0, -2098.0, -921.5, -193.75, 1914.0, -686.0, -347.0, 1745.0, -2332.0, -1522.0, 269.5, -839.0, 924.0, 1206.0, -1111.0, 1593.0, -863.5, 1122.0, -2048.0, 184.5, -704.0, -1729.0, 1037.0, 514.0, -1336.0, -1272.0, 467.75, -308.75, 300.0, -883.5, 2260.0, -871.0, 962.5, 583.0, -2238.0, -257.0, -1099.0, 183.75, 1721.0, -1390.0, 1429.0, -1299.0, -537.0, -839.0, 629.0, 750.0, -1950.0, 155.875, 2308.0, 269.5, 487.5, -2770.0, 1776.0, 222.5, -1478.0, 337.75, 409.0, -1601.0, -289.25, 811.5, -1242.0, -449.5, 1190.0, 332.0, -952.5, 1659.0, -2370.0, -1106.0, -336.5, 1883.0, -848.0, 2122.0, 3562.0, 2232.0, 334.0, -635.5, -376.0, 281.25, -2788.0, -183.5, -1881.0, -1954.0, -1960.0, -1707.0, -351.5, -468.0, -969.0, 1034.0, -914.5, 1593.0, 1402.0, 370.5, 1426.0, 360.5, -103.75, -1553.0, -1417.0, 257.0, 372.0, -621.0, -444.0, -580.0, -141.25, 737.0, 704.5, -282.5, 1858.0, 1090.0, -468.0, 2160.0, -3542.0, 1788.0, -606.0, -1227.0, -587.0, -211.75, -583.5, -816.0, 282.5, 2080.0, -354.0, 963.0, 1329.0, 1230.0, 1966.0, 593.0, -1452.0, 2350.0, 306.0, 1946.0, -578.0, 1024.0, -1666.0, -119.875, 1210.0, -170.0, -1542.0, -243.75, -2598.0, 285.5, -1034.0, 105.6875, 388.0, -876.0, 36.625, 611.5, 370.0, 328.5, -1015.5, -566.5, 129.0, 1594.0, 809.5, -491.25, -3236.0, 199.75, 2004.0, -797.0, 652.5, -296.0, -1586.0, -2816.0, -350.0, 1393.0, 640.0, -121.0, -2688.0, -2610.0, -1700.0, -1270.0, 3008.0, 1482.0, -968.5, -877.0, -772.5, -718.0, 1144.0, 588.0, 1090.0, -3338.0, -2322.0, 1578.0, -472.5, 218.5, 786.5, -19.0, 1086.0, -1157.0, 2162.0, -635.5, 895.0, 2722.0, -696.5, 1114.0, -382.5, 1186.0, 4096.0, 2076.0, 2708.0, 868.0, -1594.0, 262.25, 1541.0, 1066.0, 1925.0, 1796.0, -348.0, -591.0, -691.5, 498.5, -320.0, 2740.0, 1181.0, -1992.0, -1710.0, -1092.0, 36.0, 1930.0, -1912.0, -259.5, -2494.0, -2818.0, -1516.0, -341.75, 1604.0, 1008.5, -327.0, 1635.0, -149.0, -292.0, -3388.0, 1204.0, -1150.0, 327.0, 1852.0, 2262.0, -865.5, 162.0, -216.125, -1520.0, -207.125, -1006.0, -21.046875, 3184.0, 1512.0, -1015.5, 1204.0, -2068.0, 564.0, 547.0, 780.0, -2108.0, 24.3125, 2594.0, 1109.0, 806.0, 2886.0, 1348.0, 2432.0, -777.0, 205.375, -1702.0, -1652.0, 1021.0, -1379.0, 1093.0, 1426.0, -2.5, 736.0, -1634.0, 2204.0, -841.0, -1782.0, -1355.0, -148.0, -50.625, 468.75, -1210.0, -1808.0, -724.0, 850.5, 513.0, 292.0, -499.0, 71.5, -226.25, -1136.0, 803.0, -777.5, 173.125, 2102.0, -620.0, -276.0, 1257.0, 1598.0, -1632.0, 2168.0, -523.5, -2352.0, 1306.0, -217.0, 695.0, 1072.0, -79.3125, 1356.0, 797.5, -370.5, -240.125, -431.0, 1809.0, 80.0, -120.375, 1508.0, -1532.0, 1188.0, 809.5, -2150.0, -300.5, 1069.0, -2264.0, -2068.0, 1904.0, 307.5, -1433.0, 131.25, -551.0, 293.0, 91.5, -65.0, 859.0, -1790.0, 261.0, -559.5, -1152.0, -1152.0, 1066.0, 886.0, -1744.0, 2490.0, 417.0, 828.5, 2004.0, 777.0, -262.0, 832.5, -1686.0, -2508.0, 238.0, -916.0, 2580.0, -1305.0, -1680.0, 1502.0, -832.0, 2532.0, -2894.0, -140.0, -711.0, -1420.0, 3036.0, 1136.0, 542.5, 578.0, 169.25, 205.375, 452.75, 1568.0, -25.0, 1832.0, 1524.0, 400.0, -772.0, 324.5, -3496.0, -1053.0, -1526.0, -1638.0, 1446.0, -750.5, 527.5, 386.25, -777.5, -1446.0, -674.0, -1088.0, 667.5, -425.5, -668.0, 453.0, 1058.0, -123.375, -929.5, -1298.0, -2986.0, -1210.0, 1626.0, 3220.0, -1660.0, -1080.0, -125.5, 2760.0, 2050.0, -2058.0, 474.5, -402.0, 164.25, 420.75, -101.25, 982.5, -124.0, -2446.0, -1162.0, 1224.0, 69.0, 1514.0, -362.5, 999.5, -3640.0, -1422.0, 1638.0, -2664.0, 390.75, 723.0, 468.0, 327.5, 1006.5, -86.75, -721.5, 1019.5, 63.5, -857.0, -632.0, 2316.0, 1787.0, 1224.0, 586.0, -1758.0, 705.5, 1062.0, -879.5, 1373.0, -499.0, -1340.0, 413.75, -2224.0, 408.0, -1881.0, 690.5, -380.75, 8.5, -381.0, -1471.0, 1293.0, -1343.0, -253.0, -945.0, -815.0, 368.25, 636.0, -1199.0, -1109.0, -108.0, 1359.0, -99.0, 768.5, 409.25, -1969.0, 1384.0, 1754.0, -608.0, 75.0, -1248.0, -1254.0, 1555.0, 925.5, -806.0, -1375.0, 633.0, 777.0, 1050.0, -888.5, -729.0, -416.75, -655.0, 591.0, -853.0, 738.0, -884.0, -890.5, -589.0, -322.25, -524.0, 2108.0, 2500.0, -467.25, -2140.0, 686.0, -958.0, 2022.0, 24.0, 160.75, -152.125, -567.0, -504.5, -1554.0, -499.0, 572.0, 532.0, -1122.0, -2240.0, -1468.0, 1201.0, 1081.0, -764.0, -499.0, 1474.0, 1159.0, -128.5, -1306.0, -761.0, -1020.5, -404.0, -220.0, 170.5, -2118.0, 190.0, 1858.0, -323.25, 171.625, -870.0, 2576.0, -1230.0, 1254.0, 1356.0, -68.0, -298.0, -900.5, 1221.0, -4140.0, 956.5, 1132.0, 584.0, -1452.0, 3030.0, -1616.0, -224.75, 2460.0, -1648.0, 1899.0, -1139.0, -448.75, 1167.0, -2544.0, -1762.0, 522.0, -2076.0, -726.5, -3614.0, -820.0, 173.5, -3414.0, -1961.0, -1560.0, 1645.0, -4472.0, -1403.0, -949.0, -665.0, 1294.0, -1623.0, -841.5, -605.0, 2024.0, 3042.0, -117.0, -1775.0, 363.0, 639.5, -1874.0, 1328.0, 123.75, 1299.0, 1636.0, 1408.0, -1113.0, -881.5, -796.0, 3264.0, 1079.0, 512.0, -1560.0, -929.5, -460.5, -114.0, -172.0, 69.75, -1351.0, 106.4375, -947.0, -1773.0, -1153.0, 442.0, 144.0, 864.5, -1059.0, -493.5, 1728.0, -865.5, 1586.0, -352.5, 896.5, 242.0, 1584.0, 174.0, 670.0, -228.0, -2760.0, 1608.0, 1203.0, -1979.0, 1322.0, 393.0, 2212.0, -92.5, 1286.0, -485.0, -799.0, 1287.0, 516.0, 1559.0, 1088.0, -1998.0, -1299.0, 385.0, -1313.0, 1061.0, -460.5, 2380.0, -614.0, 448.0, -484.5, 1746.0, -1971.0, 1804.0, -594.5, 760.0, -1484.0, 582.0, 181.5, 124.0, 638.0, -1134.0, -3130.0, 1252.0, -13.0, 915.0, 1882.0, -1617.0, 3024.0, -1425.0, -1730.0, -992.0, 1136.0, -1079.0, -2656.0, -57.0, -2678.0, -1420.0, 223.0, 689.0, -115.25, 914.0, -108.75, 1153.0, 562.5, -1412.0, 207.5, 1023.5, -290.5, -1806.0, 2090.0, -89.75, -882.5, -5.0, -2278.0, -1573.0, -694.0, 15.0, 1117.0, 720.0, 2280.0, -2582.0, -1447.0, 422.5, -2660.0, -588.0, -779.0, 1840.0, 1130.0, 2754.0, 134.0, -1278.0, 1408.0, -772.0, -1677.0, 981.0, -504.5, 1558.0, -2444.0, -555.0, 2068.0, 774.0, -755.5, 1371.0, -2080.0, -2052.0, 75.0, 554.0, 1286.0, -1015.0, 228.25, 418.0, 1104.0, -328.5, 295.0, 573.0, -765.0, -2372.0, 213.0, -1520.0, -570.5, 1808.0, 937.0, -2710.0, 1452.0, 1814.0, 2382.0, 829.0, -1356.0, -83.0, 337.0, 483.75, -3652.0, -2200.0, -758.0, -992.5, -60.0, -484.0, -1565.0, 1445.0, -2050.0, -1354.0, -461.0, -1500.0, -1036.0, 1526.0, 1574.0, 1296.0, -1336.0, -1774.0, -3480.0, -2338.0, -1014.0, 679.0, -917.5, -1292.0, 2676.0, -799.0, 230.5, 976.0, 289.0, -1162.0, -3168.0, 400.25, 776.0, -451.5, -2360.0, -2432.0, 1600.0, -1026.0, -373.5, 323.0, 34.5, -253.5, -191.125, -517.5, -917.0, -2492.0, -609.0, -733.0, 1234.0, 480.5, 2082.0, -812.5, 0.25, 601.5, 2088.0, 780.0, -1548.0, 491.5, 2256.0, 521.0, -617.0, -128.0, -2374.0, 1663.0, -529.0, 2316.0, 2804.0, -562.0, 158.5, 2272.0, -2644.0, 3144.0, 2006.0, -380.5, 1471.0, -1778.0, 1473.0, 1313.0, 1588.0, 2112.0, -3308.0, 477.0, -583.0, -1268.0, 268.0, -1152.0, -768.5, -762.5, -182.5, 313.75, 60.125, -43.5, -1630.0, -106.5, -883.0, -1078.0, 2096.0, -2066.0, 1005.0, 2392.0, 414.0, -2404.0, 537.5, 2648.0, 3300.0, -1960.0, -55.5, -1538.0, 319.5, 1520.0, -1096.0, 670.0, 1352.0, 282.75, 268.0, 1456.0, 1638.0, -390.25, -868.0, -116.75, 209.0, -1302.0, 2508.0, 1344.0, 2332.0, -761.0, -1006.5, -1798.0, 983.5, 117.0, -1993.0, 1019.0, -50.96875, -1141.0, 1591.0, -383.0, 313.0, -207.0, 469.5, 187.75, 1794.0, -346.0, 980.0, 306.0, -1452.0, 3544.0, 785.0, -384.75, 1527.0, -2048.0, -977.0, -343.75, 1446.0, 1214.0, -1671.0, 1422.0, 3140.0, -1592.0, -1670.0, 1856.0, 2352.0, -3724.0, -2788.0, 2112.0, 1594.0, -741.5, -653.0, -2168.0, 886.0, -1578.0, 1494.0, 2364.0, -6.5, -1609.0, 799.0, -2029.0, 230.75, -986.0, -1305.0, -664.0, -275.0, -635.0, 540.5, 293.5, -145.75, 1252.0, 2228.0, -137.0, -496.0, -1393.0, -673.5, -2350.0, -2268.0, -500.5, -3248.0, -1096.0, -1974.0, -635.0, -1310.0, -1782.0, -1059.0, -368.75, 273.0, -2292.0, 1239.0, -1697.0, 1838.0, 583.5, -1768.0, 1742.0, -479.0, 1148.0, 172.0, -600.5, 1630.0, 315.75, 509.5, 384.25, 1214.0, -1951.0, 426.25, 1468.0, 2556.0, 953.0, -2700.0, -743.5, 322.5, 39.0, 32.5, -1370.0, 1004.5, 506.75, -2106.0, 377.5, 530.0, -1873.0, -176.75, 309.5, -340.0, -1872.0, -1511.0, 77.75, -1240.0, -316.0, 2162.0, -2052.0, 1056.0, -207.25, -571.0, 134.25, -147.75, 1330.0, 1797.0, -533.0, -181.0, 935.0, -756.0, -19.15625, 926.5, -124.8125, 3312.0, 452.5, -1710.0, -922.5, 1668.0, 352.5, -1873.0, -675.0, -1885.0, -700.0, 1093.0, 1968.0, 430.25, 1709.0, 1837.0, 1236.0, 379.0, 1486.0, 209.125, 1167.0, -11.1875, -1094.0, 594.5, 121.0, -2056.0, 2348.0, 1477.0, -592.5, 1212.0, 583.0, 1684.0, -2576.0, 1158.0, -3260.0, 1482.0, 4112.0, -393.0, -1722.0, 94.25, -331.75, 727.0, -739.0, 1556.0, -2680.0, -406.5, -247.0, 327.75, 452.0, 113.5, 551.0, 719.0, -1530.0, 847.5, 869.0, 506.0, -996.5, 1174.0, -1529.0, 1855.0, 500.0, -1914.0, 1427.0, 623.0, -1942.0, -318.0, -1548.0, 864.5, -247.75, -84.75, -1042.0, 1047.0, 1764.0, -675.5, 359.0, 2636.0, 490.5, -4060.0, 849.0, -631.0, 2019.0, 191.75, 1988.0, 74.6875, 628.5, 1801.0, 2724.0, 891.5, -3494.0, -511.0, -1223.0, -220.0, -2786.0, 2019.0, 1021.0, 952.5, -1702.0, 927.5, -693.0, -247.875, -1610.0, -1340.0, 706.5, 1725.0, 2076.0, 711.0, -387.25, 296.5, -1319.0, 203.5, -1917.0, -214.0, -1788.0, 2362.0, -1260.0, 2812.0, 1018.0, -2426.0, -3018.0, 509.0, 528.0, 493.0, 1184.0, -948.0, -239.75, -987.5, 1073.0, 1557.0, -640.5, -1312.0, 48.40625, -536.5, 493.5, 232.0, -693.0, -1045.0, -603.0, -2324.0, -2102.0, -979.5, 162.75, 63.0, 1366.0, -916.0, -2002.0, -711.0, 2500.0, 1421.0, 2094.0, -2196.0, -486.25, 187.25, -1089.0, -561.0, 494.0, -1083.0, 195.5, -2706.0, 1190.0, 620.0, 1674.0, 1169.0, 2326.0, 838.5, -54.5, -155.25, 1048.0, 1250.0, 1330.0, -75.5, 1442.0, -2244.0, -29.0, -2956.0, 297.5, -968.5, -637.0, 1840.0, 1237.0, -1966.0, 426.0, 1892.0, -2388.0, -611.0, 1304.0, -838.0, -1286.0, -200.5, 85.0, 1146.0, -3078.0, 217.5, 493.0, -294.25, -233.0, 232.125, 405.5, 2870.0, 810.5, 501.5, 713.0, 1258.0, 207.0, 1622.0, -1582.0, -20.0, 1042.0, 2084.0, -2274.0, -988.5, 1124.0, -1897.0, 284.25, -1729.0, 587.5, 746.0, -2518.0, 2298.0, -2672.0, -926.0, 4424.0, 1054.0, -2434.0, 1446.0, -696.0, -643.0, 68.0, -732.5, -998.0, 930.5, -1680.0, -2788.0, -1922.0, -1691.0, 1204.0, -39.5, 565.5, 639.5, -223.75, -2602.0, 728.5, -1584.0, -1602.0, -2748.0, 3872.0, -734.0, 834.0, 2072.0, 822.0, -1276.0, 428.0, -952.0, -548.0, 919.0, -2976.0, -2116.0, 1045.0, -378.0, 248.875, 852.5, -322.0, 136.0, -330.5, 2372.0, 1317.0, -1740.0, 1558.0, -2027.0, -828.0, 1021.0, 1536.0, -2410.0, 319.0, 167.0, 757.0, -957.0, 324.5, -2082.0, 982.0, 1542.0, 666.0, -893.5, 1047.0, -1408.0, -984.5, 1123.0, 439.0, -561.0, 1632.0, 1194.0, 362.5, 823.0, 3782.0, -758.0, -2296.0, 1564.0, 444.75, -1339.0, 836.5, -720.0, -1050.0, -1094.0, 1465.0, 2994.0, -380.5, -410.0, -1507.0, 653.5, -254.375, 782.0, 11.25, -7.40625, -84.0625, -12.125, -358.0, -633.0, -1166.0, -2698.0, -1990.0, -1894.0, -1404.0, 19.0, 1.0, 194.0, 248.0, -2292.0, -2090.0, 2428.0, 385.0, -682.0, -364.5, 193.75, -1069.0, 831.5, -980.5, 391.5, 863.0, -1078.0, -1948.0, 3268.0, 364.25, 310.25, 147.5, 364.0, 786.0, 423.5, 338.0, -23.25, -1562.0, -1378.0, -525.5, -341.5, -2644.0, 1532.0, -823.0, 959.5, 607.5, 2023.0, -289.0, -153.25, 247.25, 66.0, 779.5, -2554.0, -334.75, 2410.0, -1098.0, 2098.0, 1024.0, 1649.0, -1082.0, 639.5, 1009.0, -789.5, 1342.0, -987.5, -739.5, -266.0, 294.0, -1721.0, 3374.0, 989.0, -1208.0, 237.5, 20.0, -94.75, 1374.0, 213.0, 1110.0, -1104.0, 1136.0, 169.875, -59.0, -613.5, 265.75, -3684.0, 1220.0, -3242.0, -2784.0, 1860.0, -1552.0, -321.5, -1482.0, -3102.0, 2096.0, -1346.0, 579.0, 2912.0, -1748.0, 703.0, -334.25, 2232.0, 1490.0, 1260.0, 168.5, 168.75, -405.75, 107.0, -994.0, 1928.0, 983.5, -231.5, -1774.0, 1899.0, -2270.0, -2900.0, 1096.0, -397.5, -1442.0, 2724.0, 674.0, -478.0, -1488.0, -2572.0, -76.5, -21.0, 856.5, 948.0, 920.0, 92.0, 973.0, 1641.0, -782.5, 127.25, 1828.0, 1.0, -2288.0, -373.0, -871.0, 959.0, 1280.0, 529.5, -439.25, -614.0, -1678.0, -2025.0, 1204.0, -2622.0, 2402.0, 2008.0, -2170.0, 118.0, 184.75, -1120.0, -1995.0, 54.5, 1931.0, -1569.0, 1918.0, 813.0, 748.5, -1612.0, 1832.0, -161.5, -851.5, -782.0, -281.0, -579.0, -189.125, 270.0, 1586.0, 441.75, -70.5, 1839.0, -1146.0, 1268.0, -1053.0, 595.5, 1218.0, -1351.0, 3084.0, 169.0, -2892.0, 885.0, -1309.0, -1365.0, 147.875, 2580.0, 627.5, -1375.0, -593.5, -1358.0, -76.5, 1034.0, 1451.0, 982.0, -399.75, -669.0, -1593.0, 622.5, 386.0, -656.0, -1079.0, 731.0, -883.0, 817.0, -1300.0, 1904.0, -536.0, -2194.0, 371.0, 1085.0, -12.0, 154.0, 258.75, 2908.0, -86.0, 90.25, 621.0, -38.5, 164.0, -62.0, 1692.0, -105.25, -905.5, -1475.0, 230.875, 426.0, 1994.0, 302.75, -855.0, 938.0, -1620.0, 1196.0, -889.0, 747.0, 110.5, -2998.0, 887.5, 493.5, -751.0, -672.5, -973.0, 343.5, 961.5, -80.6875, 342.25, 2120.0, -211.125, 292.0, -1342.0, -3226.0, -1975.0, -256.75, -212.5, 2492.0, 3060.0, -3098.0, 628.0, -2262.0, 500.5, 1077.0, 1008.0, -3658.0, -305.0, -1945.0, -2570.0, -708.5, -545.5, 3128.0, 1678.0, -1913.0, -49.25, 1328.0, -1216.0, 1861.0, -1340.0, -3650.0, -2204.0, 2242.0, -518.0, -1367.0, -828.0, -210.0, 87.5, -971.0, -105.0, -801.0, 967.0, -1223.0, -1671.0, 3124.0, 1740.0, -2792.0, -878.0, 96.0, 2574.0, -284.0, 307.5, 690.0, -1548.0, 1298.0, -928.5, -1030.0, 981.5, -484.25, 1418.0, 2174.0, -266.75, 180.5, 1978.0, 607.0, 2492.0, -1150.0, 3072.0, -455.0, 158.0, 37.5, -838.5, -862.0, -416.75, -1802.0, -18.0, 335.25, 407.0, 758.0, -439.5, -1298.0, 1163.0, 115.0, 801.0, 5.75, 540.5, 1483.0, 381.25, 1002.0, -894.0, -1000.0, 2035.0, 1090.0, 684.0, -939.0, -1622.0, -131.5, 1146.0, -1006.0, 623.0, 1889.0, 1333.0, -1865.0, 399.25, 1010.5, 1757.0, -552.0, 154.0, 1092.0, -719.5, -1075.0, -39.40625, 1063.0, 1578.0, 1056.0, 3452.0, -569.0, 215.0, 1156.0, -2356.0, 1663.0, 2826.0, 1189.0, 3812.0, -241.0, 2064.0, 103.0, -33.0, 182.25, 59.75, 2136.0, -371.5, -365.0, 854.0, -671.0, 1239.0, -1160.0, -858.0, 1706.0, 73.0, 1052.0, -833.5, 2660.0, 1067.0, 5768.0, 212.875, 332.0, -897.5, 859.0, 377.25, -3104.0, 871.5, -2450.0, 2920.0, 2802.0, 564.0, 1380.0, 1477.0, -2422.0, -9.375, -1498.0, 646.5, 3238.0, 1520.0, -207.0, -2033.0, 1412.0, 242.625, 47.0, 1714.0, 47.0, -1599.0, -1536.0, -1012.0, 875.0, -812.0, 414.5, -1087.0, -659.0, -1125.0, -1050.0, 1775.0, -2264.0, 2178.0, 261.75, 1334.0, -782.0, 511.0, -2948.0, 283.0, 255.0, -2240.0, -2152.0, -2224.0, 1225.0, -1026.0, -129.5, -1771.0, 1804.0, 61.5, -1953.0, -613.0, 2006.0, 2986.0, 305.0, -334.5, -244.0, -606.0, 1653.0, -727.0, 1268.0, -1504.0, 1178.0, 394.0, 240.0, 1514.0, -2860.0, -814.0, -449.0, 2076.0, 95.5, -653.0, 308.0, -1934.0, -1376.0, -89.0, -1308.0, 3800.0, 1408.0, -330.25, -2580.0, -581.0, 1098.0, 948.0, -602.5, 397.75, 1030.0, 959.0, -2414.0, -1537.0, 396.0, 2400.0, 76.5, 433.5, -480.0, -1263.0, 2066.0, -3204.0, 331.5, -1190.0, -1863.0, 557.0, 255.0, -388.5, 2268.0, 1242.0, 2512.0, 1267.0, -195.125, -732.0, 1517.0, 939.5, -1226.0, -2346.0, -1176.0, -645.0, -555.0, 1242.0, -1.0625, 1616.0, 364.0, 660.0, 996.0, -1265.0, 2286.0, -1156.0, 1340.0, 1036.0, -1002.0, -354.25, -1496.0, 0.4375, 407.5, -1224.0, -1011.5, 1215.0, 783.0, -1218.0, -17.921875, 146.5, 1464.0, -1782.0, -687.0, 743.5, -2630.0, -1123.0, 124.5, -184.0, -735.5, -2976.0, -2510.0, -1686.0, 1310.0, 2446.0, 194.5, -311.0, 581.0, 794.0, 1060.0, -714.5, 380.0, -896.0, -1994.0, -434.0, -2592.0, -2148.0, -1038.0, -330.75, 226.5, -44.5, 303.75, -763.0, 2828.0, -1714.0, 2424.0, -2028.0, 1550.0, 1793.0, 2334.0, -2040.0, -1506.0, -992.0, 232.0, -1180.0, -1272.0, -1458.0, -367.5, 1272.0, -1070.0, 1973.0, -2912.0, -503.5, -90.0625, 312.0, -692.0, 339.0, 1042.0, 2186.0, -284.0, 1471.0, 79.0, 3122.0, -1940.0, -868.0, 158.5, 1379.0, -28.6875, -244.5, -712.0, -845.0, 437.0, 2952.0, 1554.0, 665.0, 64.0, 887.5, 1282.0, -1083.0, 1320.0, 1370.0, 1572.0, -2136.0, -3958.0, 1414.0, -181.0, -650.0, 1919.0, -154.0, 794.0, -1994.0, 609.0, 253.0, -409.25, -128.25, 1507.0, 2774.0, -2046.0, 2088.0, -2244.0, -478.0, 702.0, 3022.0, 1145.0, 310.75, -614.0, 833.0, 981.0, -711.0, 593.0, -425.25, 1244.0, -672.0, -1496.0, 762.5, -219.5, 555.0, 2772.0, 1208.0, 1864.0, -424.75, 449.0, -2112.0, 695.0, -916.0, 854.0, 678.0, -2664.0, 40.0, -914.0, -956.5, -138.5, 176.875, -1021.0, -601.0, 672.0, -90.75, 2392.0, 906.5, 1011.5, 585.5, -364.0, 2822.0, -1652.0, -256.5, 36.5, 1330.0, 2392.0, -3224.0, -2128.0, -611.5, 40.0, 3634.0, 138.5, 323.0, -570.0, 1767.0, 962.5, -1360.0, -194.75, 773.0, 523.0, 921.0, -1372.0, 2340.0, -1121.0, -345.5, 2290.0, -1524.0, 1060.0, -134.0, -466.5, 872.0, 1415.0, 36.875, 913.0, -305.5, 2041.0, 2920.0, -2650.0, -1839.0, 155.75, -3892.0, 265.5, -248.25, -1157.0, -723.0, 1758.0, -505.0, -1190.0, 1962.0, 330.0, 502.0, -702.0, -522.5, 2718.0, -181.0, -372.75, -1270.0, -491.5, -116.75, 1388.0, 1757.0, -1161.0, -1501.0, -524.5, -958.0, -1288.0, 1757.0, -666.5, 565.0, 342.0, -2446.0, 233.375, -982.0, -742.0, 1108.0, -458.0, 758.5, 2376.0, 670.0, 537.0, 2112.0, -420.25, -604.5, 34.0, 3808.0, 1491.0, 1448.0, -256.0, 1740.0, -3478.0, -2514.0, 1210.0, 1284.0, -246.625, -2684.0, 1085.0, -505.25, 3730.0, 2358.0, 2980.0, -1003.5, 619.0, -842.5, -1847.0, -1638.0, -637.0, 1492.0, -1422.0, 921.5, -893.5, 311.5, 768.0, -2696.0, -974.0, -2328.0, 1536.0, -1428.0, -881.0, -380.5, -3076.0, 3028.0, 1507.0, -560.0, 1555.0, -41.0, 811.0, -903.0, -1157.0, 1407.0, -870.5, -952.5, 1148.0, 84.0, 346.0, 1750.0, 2546.0, -1596.0, 2360.0, -365.25, 2440.0, 2996.0, 2764.0, -669.5, 1240.0, -3214.0, -3148.0, -655.5, 1172.0, -628.0, -938.0, 24.75, 1975.0, 1763.0, 2060.0, -550.5, -933.5, -1607.0, 445.5, 514.5, 1432.0, 1336.0, 559.5, -2074.0, -297.25, -1537.0, 350.5, 1630.0, 420.75, 2920.0, -3612.0, 1825.0, -260.5, -751.0, 325.0, 1089.0, -401.75, 1110.0, 299.0, -1521.0, -361.0, 1105.0, -1252.0, -411.5, 540.0, -728.0, -492.0, -809.0, -310.25, -1272.0, -1842.0, -510.5, -1760.0, 284.0, -2120.0, 923.5, 1746.0, 94.25, -760.0, -125.125, 1108.0, -460.25, 1065.0, -247.25, -2310.0, -1766.0, 1965.0, -1260.0, 941.0, 1949.0, -423.0, -1930.0, -2520.0, -34.375, 1490.0, -173.25, -395.0, 511.0, 434.5, 31.0, 2818.0, -2316.0, 1059.0, -75.3125, -791.0, -1456.0, 392.5, 763.0, 1020.0, 107.75, 1317.0, 1050.0, -130.0, -331.75, -3156.0, -1390.0, -775.0, -1826.0, 1564.0, -3496.0, 21.375, 1576.0, -1192.0, 114.75, -849.0, 279.0, -308.75, -286.0, -493.5, 1212.0, 350.5, -1576.0, 1500.0, -77.0, -1692.0, -2392.0, -2800.0, -129.375, 2392.0, -446.75, -731.5, 808.0, 490.0, -803.0, -1018.5, 197.875, 2004.0, -2080.0, 632.5, 307.25, -1056.0, -635.0, 40.0, -1314.0, -1043.0, -2418.0, -1203.0, 619.0, -1578.0, 1314.0, 3968.0, -114.375, 1740.0, 445.75, 436.5, 1460.0, -133.875, 521.0, 347.75, -449.0, 648.5, -1506.0, -473.5, -1176.0, -1523.0, -298.5, -364.0, 270.0, 488.75, 2408.0, -106.625, 1728.0, -2678.0, -322.25, 1495.0, 1504.0, 1368.0, 2220.0, -513.0, -1640.0, 944.0, 3524.0, -1103.0, -821.0, -88.0625, -969.0, 460.0, -1106.0, 2336.0, 2002.0, 1302.0, -646.0, -1170.0, 699.0, 2166.0, 1265.0, 41.0, 1542.0, -993.0, -214.75, 1001.0, 53.0, -852.5, -1456.0, 124.5, 2764.0, 2786.0, -1417.0, 339.5, 1599.0, 272.5, 215.75, -978.0, -1465.0, 761.5, -127.0, -754.0, 1686.0, 839.0, -1364.0, 2812.0, -2870.0, 406.75, -1552.0, 880.0, -2672.0, 90.0, 843.0, 1342.0, 1008.0, -781.0, 687.0, -673.5, -121.875, -883.5, -2936.0, -287.5, -743.0, -784.5, -2768.0, -275.0, -2178.0, -882.0, 1154.0, -1006.0, -734.0, -1236.0, -4708.0, -1944.0, -1674.0, 161.5, 1595.0, 1438.0, 1255.0, 309.5, -610.5, 1404.0, 259.0, 1549.0, 1632.0, 300.5, 234.375, -1326.0, 1048.0, -1621.0, -1720.0, 530.0, 1358.0, 1752.0, -94.0625, -247.75, -1455.0, -968.0, -2092.0, 2188.0, -168.0, -2810.0, 239.0, 1640.0, 1010.0, -963.5, 652.5, -716.5, 2278.0, 59.0, 1156.0, -943.0, -2822.0, 1602.0, -784.5, -69.75, -178.625, -1.5, 1895.0, -1975.0, -2288.0, -1400.0, 2054.0, -1882.0, 659.0, -125.0, 1235.0, 471.25, 7.0, 1025.0, -920.0, -784.0, 868.5, 2118.0, -1258.0, 948.5, -1322.0, 1800.0, -858.5, 34.0, -1148.0, 308.25, -520.0, 81.0, -548.0, -792.0, 2052.0, -588.0, 1400.0, 1207.0, -2576.0, 1557.0, 2660.0, -1175.0, -108.375, -457.75, 113.0, 2402.0, -2290.0, 1222.0, 1288.0, -508.75, 2888.0, -2003.0, -1601.0, 765.0, 268.5, 1119.0, 87.0, -182.0, 725.5, -1452.0, 1872.0, 910.5, -630.0, 404.5, -883.0, 361.5, 2774.0, -382.0, 1417.0, -275.0, 609.5, 530.0, 317.5, -1542.0, -390.0, -1239.0, -53.75, 85.75, -1184.0, 546.5, -156.25, 415.0, -2214.0, 53.0, -775.0, 1317.0, -2414.0, 1972.0, 339.5, -174.25, -1241.0, 1178.0, 942.0, 901.5, 1896.0, 677.0, -1382.0, 1157.0, 1437.0, 346.25, -1467.0, 1006.0, -438.75, -1270.0, -1956.0, -23.25, -388.0, 1494.0, 1435.0, 1950.0, 89.1875, 2130.0, 2126.0, -1920.0, 1648.0, 1332.0, -2276.0, 528.5, 810.0, -1257.0, -1581.0, -1335.0, -760.0, -919.5, -957.0, 2036.0, 2196.0, 763.5, -3422.0, 1531.0, 1244.0, 831.0, -1374.0, -924.5, 1306.0, 4464.0, -1491.0, 49.0, -519.0, 55.375, 716.0, 1786.0, -2001.0, 448.5, 39.625, -1165.0, -1687.0, 1621.0, 527.0, 1418.0, 400.5, 150.5, -991.5, 614.0, -1358.0, -1462.0, -922.0, -488.0, -2700.0, -3384.0, 1774.0, 2058.0, -636.0, 1737.0, -2428.0, -1167.0, 1373.0, 2168.0, 2142.0, -2280.0, -3428.0, 840.5, 1085.0, -1091.0, -713.5, 1123.0, 1151.0, -1379.0, 1700.0, -1135.0, 1121.0, -1612.0, 614.0, 1218.0, -545.5, -2444.0, 549.0, -298.5, -1505.0, 794.0, 221.0, -47.0, -2304.0, 150.5, 898.0, -713.5, 130.25, 923.5, -2006.0, 2786.0, -1398.0, -2072.0, 2050.0, -257.75, 70.75, -280.0, 1393.0, -1287.0, 259.0, 166.5, -811.5, 476.0, 1610.0, 1056.0, -1858.0, -1018.0, -71.625, 3648.0, -1539.0, -1085.0, -3992.0, -978.0, 1043.0, -527.0, -157.0, -1418.0, 1628.0, 357.25, 1825.0, -598.0, 1062.0, 90.0, -2766.0, -81.5, -1083.0, -2728.0, 2932.0, 2272.0, 34.0, -55.0, 539.0, 3560.0, 526.5, -1216.0, -569.0, 2860.0, 121.875, 427.5, -825.5, 2264.0, 1505.0, -1253.0, -1454.0, -1207.0, -56.5, 1763.0, -1602.0, -533.5, 2028.0, -1225.0, 1233.0, -2366.0, -419.75, 1248.0, -3758.0, -167.75, 34.25, -1619.0, 3026.0, 1336.0, 453.75, -735.5, 344.5, 1636.0, -1990.0, 1207.0, 147.5, -872.0, -1323.0, 1140.0, 688.5, 197.5, -355.5, 1129.0, 47.0, 1547.0, 555.0, -1769.0, -1261.0, -1165.0, -2202.0, 3822.0, -718.5, -778.5, -3284.0, 1182.0, 943.0, 1470.0, -827.0, -2432.0, -634.5, 1055.0, 556.0, 755.0, 1146.0, 1751.0, -2280.0, 360.0, -440.0, -552.5, -1632.0, 623.5, -827.0, 442.25, 1882.0, -1982.0, -3222.0, 216.125, 1583.0, 805.0, 246.25, 574.0, -368.75, -2358.0, -1270.0, 2168.0, 1498.0, 881.5, -183.375, -1843.0, 2310.0, -1444.0, 2764.0, 2512.0, -268.5, 265.0, 978.0, -156.0, -1078.0, 2896.0, -3262.0, -582.0, -460.0, 2248.0, 1570.0, -1.0, 1128.0, 2424.0, -1266.0, 3124.0, 312.5, -630.0, -115.0, -910.0, 1622.0, 118.0, 527.5, 620.0, 560.0, 2380.0, 92.375, 13.75, -3230.0, -575.5, 1406.0, 1542.0, 90.3125, 586.5, 610.5, -25.0, -789.0, 2712.0, 1665.0, -619.5, 775.0, 801.0, -1683.0, -583.0, -840.5, -840.5, -682.5, -1185.0, 15.0, -1400.0, 1142.0, 1984.0, 1131.0, 2710.0, 99.25, 1360.0, 364.5, -2976.0, -1468.0, 994.5, 1604.0, -641.0, -2378.0, -1428.0, 694.5, 115.25, 202.0, 868.0, -779.0, -1204.0, -1009.0, -917.5, 904.5, 1432.0, -1434.0, -1224.0, 1458.0, -2022.0, 269.0, -2076.0, 716.0, -103.0, -446.5, -1250.0, -25.5, 1841.0, 2514.0, -703.5, -1400.0, 1323.0, 1698.0, -768.0, -778.0, 1216.0, 285.0, -129.0, -914.5, -176.75, -544.0, 1341.0, -117.5625, -765.0, -1583.0, -910.0, -344.75, 880.5, -108.5, -1558.0, -2238.0, -264.0, 1044.0, -506.5, 16.0, -26.125, 372.5, -1406.0, 252.5, 683.5, 2660.0, 516.0, 1226.0, -506.5, -2886.0, -681.0, -1362.0, -678.5, -424.5, -1347.0, 124.5, 176.5, -250.5, 1442.0, -823.5, 583.5, 593.0, -1954.0, 299.75, 506.0, 954.0, -1972.0, -274.25, -531.5, -346.0, -1684.0, -989.0, 702.0, 740.0, -558.0, 1774.0, -965.5, -2336.0, 609.0, 1026.0, -2498.0, -2516.0, 791.0, -1018.0, 1656.0, -1855.0, -1964.0, -511.5, -34.28125, -1460.0, -765.0, -2224.0, 124.5, 463.0, -408.25, 1851.0, 280.0, 2220.0, -479.25, -1327.0, 101.5, 979.0, 2414.0, -541.0, -55.75, 105.0, 1742.0, -1052.0, 100.25, -2066.0, 941.0, 1550.0, 463.0, -775.0, 2786.0, 1706.0, -2286.0, -326.5, -1156.0, -259.5, -519.0, 318.0, 1402.0, -1041.0, 59.75, 270.0, -815.0, -1312.0, 2198.0, 814.0, -2184.0, 1538.0, -10.5, 1192.0, 997.5, 306.75, 714.5, -2796.0, -2496.0, 667.0, 1316.0, -141.5, -791.0, 1794.0, 1041.0, -268.0, 2512.0, -2482.0, -1763.0, -1160.0, -1310.0, 940.5, -587.0, -2198.0, 3082.0, -2376.0, -458.75, 216.75, 2000.0, -1226.0, 2994.0, 605.0, -46.75, -1860.0, 2646.0, 2576.0, 1784.0, 311.0, 2704.0, -604.5, 893.0, -427.75, 318.0, 2506.0, 1068.0, -96.5, 105.0, -140.75, 299.5, 2068.0, 634.0, 590.5, -321.5, 4004.0, 1267.0, -1276.0, 66.0, -258.0, -1190.0, -4336.0, -793.5, -3.5, 1431.0, 314.5, -3252.0, -1506.0, 2048.0, 2368.0, -927.5, -367.0, -1497.0, 1769.0, 947.0, -488.5, 1830.0, -589.5, -77.0, -848.0, 510.5, -666.0, -205.0, -1598.0, -1214.0, -912.0, -1355.0, -1009.0, 1437.0, 322.75, -1750.0, -1728.0, -855.5, -450.5, -1477.0, 2070.0, -64.25, 469.0, 131.0, -1231.0, -400.5, -1112.0, 2498.0, -605.0, -1033.0, -1150.0, -1682.0, 1564.0, 1168.0, 843.5, 887.5, -976.0, 2936.0, -89.5625, 60.25, 2064.0, 1833.0, 2766.0, -567.0, 101.5, -59.75, -2362.0, -527.5, -2272.0, 2410.0, 734.0, -753.0, -969.5, 703.0, 472.75, 832.5, -1110.0, 1787.0, 1965.0, -643.5, 4112.0, -1543.0, -741.0, 234.25, 354.0, 541.5, -1208.0, -2316.0, 1075.0, -1210.0, -262.75, -617.0, 2027.0, 1571.0, 877.0, -809.0, -3338.0, -1538.0, 853.5, 1251.0, -1089.0, 1573.0, 43.75, 163.5, -2922.0, -1435.0, 1515.0, 359.0, 1014.5, -469.5, -190.25, 152.5, -489.0, 304.5, -2536.0, -1592.0, -1574.0, 2974.0, 1613.0, -1226.0, 2056.0, -1728.0, 162.875, -1106.0, -2490.0, -288.75, -1650.0, 1440.0, 195.75, 430.5, -165.125, -1246.0, 332.75, -2204.0, 357.5, 1434.0, -1152.0, 755.5, -322.5, -1256.0, 1187.0, 1178.0, -911.0, 47.625, -1288.0, -59.875, -612.5, -56.0, -1622.0, 1796.0, -1471.0, 641.5, 2892.0, -1280.0, -473.75, -293.5, -502.5, -1107.0, -2254.0, 1758.0, -2660.0, -736.0, 1019.0, 303.0, 1994.0, 855.0, 1039.0, -1516.0, 228.5, -2306.0, -1412.0, 111.0, 772.0, 848.0, -221.375, 574.5, -1472.0, 442.75, 414.0, -1753.0, 389.5, 1255.0, 1923.0, 269.25, 2866.0, 971.0, 133.0, 201.0, 916.0, -26.5, -1018.0, -290.75, -1254.0, -567.0, -627.5, -3040.0, 3216.0, -823.0, 1052.0, 716.0, -653.0, -35.0, -1641.0, -68.5, -231.375, -686.0, -1566.0, 2108.0, -2510.0, 406.25, 924.0, -412.75, -724.5, -1276.0, 318.0, -1248.0, 821.5, 66.1875, -883.0, 604.5, 713.0, 655.0, 505.0, -2192.0, -834.0, 568.0, -2132.0, 1842.0, 2136.0, -868.0, 290.25, 622.0, -846.0, 577.0, -606.5, 1598.0, 153.875, -1090.0, 1250.0, 1180.0, 858.0, -715.0, -598.0, -160.25, -667.5, 3012.0, -2414.0, -3948.0, 906.5, -245.5, -691.5, -554.0, 636.0, -61.375, -1110.0, 1087.0, 723.5, 672.0, -1279.0, -377.0, 398.0, 1498.0, -92.5, -275.0, 167.5, -1523.0, 1039.0, 1525.0, -951.5, -866.5, -6.125, 1789.0, 626.0, 1554.0, -316.5, -2818.0, 280.75, -745.0, -1716.0, -3740.0, -2190.0, 492.5, 934.0, 299.0, -4236.0, -615.5, -783.5, -1516.0, -1095.0, 724.0, -73.625, -1393.0, -687.0, 1406.0, 2214.0, -1103.0, 87.0, -532.0, 3380.0, 3108.0, -1040.0, 200.5, 1318.0, -1508.0, -609.0, 252.75, 1298.0, 389.0, -2544.0, 417.75, 3340.0, -1508.0, 1270.0, 1836.0, -253.0, -328.5, -1965.0, 1110.0, 977.0, -703.0, -485.5, -378.0, -621.0, -2896.0, -1772.0, 145.0, -1414.0, 2039.0, -473.75, -2124.0, 1796.0, 1080.0, -1474.0, 684.5, -1519.0, 641.0, -519.0, -1436.0, 41.1875, -473.0, -1855.0, -917.0, -1054.0, 9.0, -1419.0, -1371.0, -2536.0, -1206.0, -954.0, 1812.0, -815.0, 8.25, -880.0, -308.0, 692.5, 5.0, -1946.0, -15.0, -549.5, -1143.0, 760.5, -33.75, 1028.0, 561.0, -1269.0, -759.0, -3342.0, 1864.0, -2248.0, -1047.0, -749.0, -1357.0, 95.375, 327.0, -482.75, 434.5, -1151.0, 482.0, -1241.0, 628.0, -569.5, 2652.0, 2718.0, -1598.0, 842.0, -89.0, -784.0, 208.375, 2208.0, -292.5, 28.5, 197.5, 81.0, -84.0, -1746.0, 641.0, 1029.0, -928.5, 801.0, -2460.0, 247.0, 708.5, -204.0, 447.5, 1460.0, -1896.0, -398.0, 2138.0, 651.0, 482.0, -152.0, -1234.0, 2420.0, 1069.0, -1865.0, 2480.0, -336.5, -913.0, -1634.0, 947.5, 196.0, -3040.0, 498.5, 1081.0, 838.5, 437.0, 961.5, -145.0, -909.5, -1642.0, -1303.0, -2188.0, -312.25, 1374.0, 799.0, 1992.0, 894.0, 1258.0, -1154.0, 2912.0, -1648.0, -1794.0, 35.0, -450.0, -2708.0, 1588.0, 2052.0, 880.5, 2080.0, -1104.0, -562.5, 1484.0, 190.0, 394.0, -2548.0, 1846.0, 67.0, 1177.0, -401.75, 3434.0, 837.5, -1300.0, 371.5, -1496.0, -79.0, -299.5, 371.75, -1100.0, -463.0, -584.5, 290.0, -934.0, 896.0, -957.0, 1174.0, 1715.0, 2768.0, 727.5, 1168.0, -706.5, 917.0, 1150.0, -1889.0, -268.0, 3464.0, 783.0, 1576.0, 691.0, -955.5, -2402.0, -2400.0, 1080.0, 2460.0, -126.0, 1842.0, 1094.0, -2394.0, 348.0, -495.0, 1647.0, 1423.0, -3112.0, -445.75, 223.5, 1397.0, 112.5625, -800.0, -1347.0, 1644.0, 1892.0, -1007.0, 490.0, 1039.0, -722.0, -3088.0, 1255.0, -3276.0, 551.0, -864.0, -44.125, 1493.0, 534.0, 948.5, 182.0, -1536.0, -33.0, 548.0, 192.0, -367.5, 1065.0, 282.0, -2322.0, 259.5, 398.0, 1168.0, -200.0, 2720.0, 642.0, 2564.0, 1023.5, -254.0, 1756.0, 3660.0, 214.625, -321.5, 363.5, 1237.0, 1594.0, 926.0, 119.0, -224.0, 250.0, 1704.0, 814.0, -511.75, 588.5, -973.0, -1667.0, -1007.0, 676.0, -1055.0, 981.0, 2198.0, 975.0, 955.5, 2104.0, -2326.0, 981.0, 115.0, 424.5, 239.75, -2050.0, -1478.0, -1087.0, 1125.0, 1231.0, -1846.0, 1451.0, 1266.0, 1753.0, 1323.0, -1415.0, -1373.0, -1757.0, 1804.0, -560.5, -2384.0, 403.75, 257.5, -826.0, 113.375, -1666.0, -2562.0, -841.0, 243.5, -2882.0, 1645.0, -289.5, 29.5, 1484.0, 891.0, 1080.0, 1448.0, -880.5, 2128.0, -967.5, -523.5, -1583.0, 2296.0, 2084.0, 1248.0, 185.0, 5.5, 67.75, -669.0, -588.5, -1677.0, -1465.0, 530.0, -912.0, -809.5, -829.0, -57.375, -568.5, -247.0, -1030.0, -2458.0, 431.25, 35.5, -1449.0, -1412.0, 1849.0, -128.625, -2472.0, -489.0, -1565.0, -794.0, 1342.0, 549.5, -974.0, -706.0, -436.5, 8.5, -1200.0, 61.75, -1038.0, -831.5, 1768.0, -2734.0, 1500.0, 716.0, 2306.0, 1695.0, -271.0, 597.5, -215.875, -3200.0, -75.25, 2374.0, 2962.0, 2200.0, 96.5, -714.0, -16.0, -870.0, -1229.0, 1704.0, 1818.0, -1125.0, -727.0, -3320.0, -803.0, -247.0, -2708.0, 2220.0, 454.0, -1885.0, 543.5, -602.0, -308.0, -1870.0, -413.5, 727.0, 1256.0, 1382.0, -1426.0, -1323.0, 3100.0, -288.25, -31.5, -678.5, 1916.0, -63.0, -412.25, 349.5, -1204.0, 37.0, 653.0, -1760.0, -2276.0, -1890.0, 2796.0, 1475.0, 627.0, 1143.0, -46.5, -121.375, -2088.0, -1395.0, 921.5, -3928.0, 1864.0, -233.75, 272.5, -220.0, -11.0, 1390.0, 505.0, -171.625, 863.0, -989.0, -942.5, 1919.0, 2220.0, -1174.0, 745.0, -1313.0, -867.5, -130.75, 52.5, 533.5, -1448.0, -1734.0, -1553.0, -151.5, -1007.0, -660.0, -983.0, -1295.0, 708.0, 276.0, -1895.0, -1335.0, 3.25, 677.5, -455.0, -1531.0, -102.25, -956.0, -309.75, 246.5, -203.5, 2158.0, 1645.0, -764.0, -1820.0, -1327.0, 801.5, -842.0, 1045.0, 817.5, 1908.0, -1174.0, -317.75, -4228.0, -601.0, 2072.0, 380.5, 1136.0, -438.0, -515.0, 3136.0, 2814.0, 1547.0, 318.5, 1543.0, 371.25, -1932.0, -1118.0, 803.0, -636.0, 2898.0, -923.0, -624.5, 30.1875, -284.5, -913.0, 673.0, -143.75, -948.0, 1390.0, -1472.0, 1041.0, 1466.0, -188.375, 2344.0, -207.75, -307.5, -1808.0, 1439.0, 1085.0, 2964.0, -566.0, 895.5, 890.0, -993.0, 203.75, 28.0, -1660.0, 2204.0, -214.25, 435.75, -489.5, 2092.0, 2060.0, 1376.0, 3228.0, -212.5, 2232.0, -646.5, 603.5, -1302.0, 815.0, -1657.0, -794.0, -50.5, -2148.0, -317.5, 1456.0, -2438.0, 735.0, -115.375, -1069.0, 1044.0, 790.0, -255.5, 569.5, -573.0, 1118.0, 912.5, 2884.0, -2332.0, 2304.0, -679.0, -2206.0, -666.0, -2228.0, 851.5, 972.5, -3092.0, -171.25, 1523.0, -2318.0, -1010.0, 2286.0, -585.0, 2804.0, 100.0, -2528.0, -552.5, -1220.0, -2040.0, 334.5, 41.0, 3388.0, -1734.0, -2198.0, 1102.0, 1340.0, 717.5, 1249.0, -28.5, 1020.0, -871.0, -1348.0, 2220.0, 438.75, 603.5, 78.4375, 1581.0, -2180.0, -843.0, -409.0, 592.0, 1163.0, -2792.0, -1221.0, 2758.0, -122.5, 657.5, -216.125, 1404.0, 1252.0, -978.5, -883.5, -405.25, 144.0, -391.5, 1016.0, -98.125, 1378.0, 1429.0, -152.0, -1363.0, 351.5, -53.0, 331.5, 1678.0, -258.0, -563.0, 177.25, 768.5, -805.0, -954.5, -1290.0, -278.75, 536.0, -1268.0, 1132.0, 693.0, -952.0, 1540.0, -488.0, -299.0, -550.5, -74.0, -2156.0, 1327.0, 144.625, 942.0, 472.0, 682.5, -512.0, 353.0, 529.5, 2408.0, -338.25, 612.0, 518.0, 825.5, -1372.0, -171.0, -405.5, 369.5, -551.0, -1537.0, -728.0, -1428.0, -1221.0, -169.0, -411.5, -385.5, -1662.0, 933.0, 6.5, -1330.0, 1055.0, 294.5, -90.0, 1242.0, -2788.0, 28.0, -829.0, 1248.0, -104.0, -1433.0, -1258.0, -1497.0, -1012.0, 1297.0, -717.5, -820.5, 2050.0, 1974.0, -717.0, -2216.0, 89.0, -119.5, -865.5, -4408.0, 1080.0, -1474.0, -1446.0, 619.0, -1384.0, 2376.0, -1196.0, -2456.0, 1504.0, 2112.0, -890.0, 2422.0, 1945.0, -62.5, 2598.0, 393.0, -790.0, 640.0, -2314.0, 2182.0, 72.0, 385.25, -350.75, -1024.0, 595.5, 2168.0, 1433.0, -2722.0, -2440.0, 473.0, 391.0, 406.0, 1614.0, -1458.0, 42.0, -471.0, 1162.0, -463.5, 1476.0, 261.5, -1151.0, 3114.0, 164.25, -1050.0, -1662.0, -1239.0, -3840.0, -1653.0, 1703.0, -978.0, 1657.0, -1928.0, -1232.0, -1822.0, -582.5, 592.0, -1674.0, 1797.0, 998.0, -1351.0, 2200.0, 593.5, -49.3125, -2534.0, -1290.0, 901.0, 1782.0, -568.0, 890.0, -1782.0, 843.5, -3816.0, -1664.0, 1717.0, -348.5, -546.5, -608.5, 1627.0, -1360.0, -132.0, -3836.0, -633.5, 2526.0, 758.0, -2180.0, -8.5, 1476.0, -1784.0, -1388.0, 3010.0, -10.5, 1224.0, 2015.0, 1863.0, 2356.0, -1266.0, 1538.0, 437.0, 505.75, 1277.0, 1227.0, -3246.0, 919.0, -692.0, -848.5, 522.0, -2908.0], "dtype": "float16"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(vec_add, json.loads(_TESTS), 'triton', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")